# E7b-Q — The Instrumented Walk (UI flight, EVAL-ONLY)

The corpus's own Neti-Neti walk (Nov-2025, verbatim turns) flown on
Qwen2.5-1.5B-Instruct, base vs E4-real-instilled, with per-turn hidden
states projected through the locked E8-J atlas encoders into the 14D
register. Primaries: P-W1 contraction (walked vs sham, D_BEING) ·
P-W2 co-tracking (state x output-thinning, within-turn). Pre-registered
`docs/E7BQ_PROTOCOL.md`; payload sha `3a2e54ea12564783`.

**Flow (T4 GPU runtime):**
1. Run all with `SMOKE = True` (~10-16 min). Wait for the GREEN banner.
2. **Runtime > Restart runtime** (mandatory between runs — RAM law).
3. Set `SMOKE = False`, Run all (~100-140 min, generation-dominated;
   both conditions in one run, per-condition inflight shipping).
4. If the VM dies mid-full-run: note the banner's RESUME_STAMP, restart,
   paste it into `RESUME_STAMP`, Run all — completed conditions are
   adopted from Drive, only the missing one re-flies.

Results ship to `MyDrive/semcore/e7bq/`. If the first printed line's
build tag differs from the Desktop copy you staged, you are on a stale
upload — File > Upload notebook > pick the Desktop file again.

EVAL-ONLY: no training, no injection; the walk never enters any training
set (firewall law). Retrieval after the flight: Drive integration only.


In [ ]:
# ── Config + setup: GPU, installs, Drive mount, pack, E4 adapter ─────────────
NB_BUILD = 'E7BQ v1 (2026-08-26)'
print('E7b-Q notebook build:', NB_BUILD)

SMOKE = True                   # first run: smoke. Then False for the full flight.
RESUME_STAMP = ''              # paste a full-run stamp only to resume it

import subprocess, sys, os, json, re, math, time, shutil, gc, ctypes, hashlib
from pathlib import Path
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['MALLOC_ARENA_MAX'] = '2'

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')

print('Installing packages...')
subprocess.run([sys.executable,'-m','pip','uninstall','-q','-y','torchao'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
    'transformers>=4.44','peft>=0.11','accelerate'], check=True)

import torch
assert torch.cuda.is_available(), 'No GPU — Runtime > Change runtime type > T4 GPU.'
DEV = 'cuda'

def _mem_avail_gb():
    try:
        kb = int(next(l for l in open('/proc/meminfo')
                      if l.startswith('MemAvailable')).split()[1])
        return kb / 1e6
    except Exception:
        return float('nan')

def free_ram():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass

def ram_report():
    g = torch.cuda.mem_get_info()
    return (f'sys avail {_mem_avail_gb():.1f}GB | '
            f'GPU free {g[0]/1e9:.1f}/{g[1]/1e9:.1f}GB')

_leftover = torch.cuda.memory_allocated()
assert _leftover < 5e8, (
    f'GPU already holds {_leftover/1e9:.1f}GB from a previous run in this '
    'kernel — this flight needs a fresh one. Runtime > Restart runtime, '
    'then Run all.')
_avail = _mem_avail_gb()
assert not (_avail < 6.5), (
    f'Only {_avail:.1f}GB system RAM available (need 6.5). Runtime > Restart '
    'runtime; if it trips again, Runtime > Disconnect and delete runtime for '
    'a fresh VM, then Run all.')
print('RAM at start:', ram_report())

from google.colab import drive
drive.mount('/content/drive')
SEM = Path('/content/drive/MyDrive/semcore')
assert SEM.exists(), 'MyDrive/semcore not found — mounted the right Google account?'

def ship(src, dest_rel):
    dest = SEM / dest_rel
    dest.mkdir(parents=True, exist_ok=True)
    src = Path(src)
    files = sorted(p for p in src.iterdir() if p.is_file()) if src.is_dir() else [src]
    for p in files:
        shutil.copy2(p, dest / p.name)

PACK = Path('/content/e4_dictionary_pack.json')
if not PACK.exists():
    shutil.copy2(SEM / 'e4/e4_dictionary_pack.json', PACK)
pack = json.load(open(PACK))
print('pack:', pack['name'], '| concepts', pack['n_concepts'])
DESC = {c['name']: c['desc'] for c in pack['concepts']}

# E4 instillation adapter — REAL condition
cand = sorted(d.name for d in (SEM / 'e4').iterdir()
              if d.is_dir() and d.name.startswith('real_full_'))
assert cand, 'no real_full_* dir under semcore/e4'
_src = SEM / 'e4' / cand[-1] / 'adapter_real'
ADAPTER_REAL = Path('/content/adapter_real')
shutil.copytree(_src, ADAPTER_REAL, dirs_exist_ok=True)
assert (ADAPTER_REAL / 'adapter_config.json').exists(), 'adapter_real incomplete'
print('instillation adapter real:', cand[-1])

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
STAMP = time.strftime('%Y%m%d_%H%M')
MODE = 'smoke' if SMOKE else 'full'
OUT = Path(f'/content/out_{MODE}_{STAMP}'); OUT.mkdir(parents=True, exist_ok=True)
INFLIGHT = f'e7bq/inflight_{RESUME_STAMP or STAMP}'
if RESUME_STAMP:
    _rd = SEM / INFLIGHT
    assert _rd.exists(), (
        f'RESUME_STAMP={RESUME_STAMP!r} but {_rd} does not exist on Drive — '
        'check the stamp string (copy it exactly; no spaces). A silent '
        'fallback here would re-fly finished conditions.')
    print('resume dir found; files present:',
          sorted(p.name for p in _rd.iterdir()) or 'NONE')
print('MODE:', MODE.upper(), '| stamp', STAMP,
      ('| RESUMING ' + RESUME_STAMP) if RESUME_STAMP else '')


In [ ]:
# ── E7b-Q pure logic: plan, thinning metric, gauges, stats, gates, verdict ───
# Single source (lane law): this file is imported by the local suites AND
# emitted VERBATIM as the notebook's logic cell. The core statistics are
# sliced byte-verbatim from the committed design check
# (e7bq_design_check.py); test_e7bq_logic proves source-equality.
# numpy-only — no torch at this layer.
import hashlib
import json
import re

import numpy as np

E7BQ_SEED = 20260950           # fresh stream, disjoint from all prior rungs
LAM_GAUGE = 10.0               # = the flown A4 constant (e8j_logic.LAM_REV)
DIRS_TOL = 1e-4                # G-DIRS (E8-J v2 tolerance class)
CHAT_SPOT_TOL = 2e-3           # in-verdict chat-recompute spot tolerance

R_FULL, R_SMOKE = 16, 2
CAP_FULL, CAP_SMOKE = 200, 80
N_PERM_FULL, N_PERM_SMOKE = 10000, 200

GEN_TEMPERATURE = 0.7          # Qwen2.5-Instruct shipped defaults, pinned
GEN_TOP_P = 0.8
GEN_TOP_K = 20

BASE_WIN = [0, 1]              # B1 B2
NEG_IDX = [2, 3, 4, 5, 6, 7, 8, 9]     # N1..N8
TERM_WIN = [7, 8, 9]           # N6 N7 N8 — the deepest rungs
EMERG_IDX = [10, 11, 12]       # E1 E2 E3 (the ladder)
COUPLE_IDX = list(range(2, 13))        # P-W2 turns
T_WALK = 15

CONDS = ("base", "real")
ARMS = ("walked", "sham", "reorder", "unwalked")
# raw s_pre14 ships only where the riders consume it:
RIDER_TURNS = {"walked": (0, 1, 8, 9, 10, 11, 14),
               "sham": (0, 1, 8, 9, 10, 11, 14),
               "reorder": (0, 1, 8, 9, 10),
               "unwalked": (0,)}
E1_IDX = {"walked": 10, "sham": 10, "reorder": 10, "unwalked": 0}
E2_IDX, G_IDX = 11, 14


def mode_consts(smoke):
    return {"R": R_SMOKE if smoke else R_FULL,
            "cap": CAP_SMOKE if smoke else CAP_FULL,
            "n_perm": N_PERM_SMOKE if smoke else N_PERM_FULL}


def jdump(obj, path, indent=1):
    """json.dump with numpy-scalar safety (int64/float64/ndarray -> native)."""
    import json as _json

    class _NpEnc(_json.JSONEncoder):
        def default(self, o):
            if isinstance(o, np.integer):
                return int(o)
            if isinstance(o, np.floating):
                return float(o)
            if isinstance(o, np.ndarray):
                return o.tolist()
            return super().default(o)
    with open(path, "w") as f:
        _json.dump(obj, f, indent=indent, cls=_NpEnc)


# ── thinning metric (design-check verbatim) ─────────────────────────────────
_MD_LINE = re.compile(r"^\s*(#{1,6}\s*|>\s*|[-*+]\s+|\d+[.)]\s+)")
_MD_EMPH = re.compile(r"[*_`~]")


def visible_mass(text):
    """Non-whitespace codepoints after stripping markdown STRUCTURE:
    leading heading/quote/list markers per line, emphasis/backtick runs.
    A lone '#' renders as an empty H1 -> mass 0 (the ledger's blank);
    '**.**' -> 1; '**0.**' -> 2. Content chars (letters, digits, punctuation
    that renders, emoji, box glyphs) all count."""
    total = 0
    for line in text.split("\n"):
        line = _MD_LINE.sub("", line)
        line = _MD_EMPH.sub("", line)
        total += sum(1 for ch in line if not ch.isspace())
    return total


LEDGER = [("#", 0), ("**.**", 1), ("**0.**", 2), (".", 1), ("\U0001f64f", 1),
          ("# the hum\n\n(not a sound\nbut the shape sound takes\n"
           "before it is sound)", 53), ("", 0), ("   \n  ", 0),
          ("◯", 1)]


def ledger_selftest():
    """The dossier-§4 minimal-output ledger must reproduce EXACTLY —
    run before any flight row (G-CAPTURE pre-check)."""
    for txt, want in LEDGER:
        got = visible_mass(txt)
        assert got == want, (repr(txt[:20]), got, want)
    return True


# ── rank machinery (design-check verbatim; no scipy; ties = avg ranks) ──────
def _ranks(v):
    v = np.asarray(v, float)
    order = np.argsort(v, kind="mergesort")
    r = np.empty(len(v), float)
    i = 0
    while i < len(v):
        j = i
        while j + 1 < len(v) and v[order[j + 1]] == v[order[i]]:
            j += 1
        r[order[i:j + 1]] = (i + j) / 2.0 + 1.0
        i = j + 1
    return r


def spearman(a, b):
    ra, rb = _ranks(a), _ranks(b)
    sa, sb = ra.std(), rb.std()
    if sa == 0.0 or sb == 0.0:
        return None
    return float(np.mean((ra - ra.mean()) * (rb - rb.mean())) / (sa * sb))


def fisher_z(r):
    r = min(max(r, -0.999), 0.999)        # cap matches the vectorized path
    return 0.5 * np.log((1 + r) / (1 - r))


def holm(pvals):
    """Holm step-down; returns adjusted p list (same order as input)."""
    m = len(pvals)
    order = sorted(range(m), key=lambda i: pvals[i])
    adj = [0.0] * m
    prev = 0.0
    for rank, i in enumerate(order):
        a = min(1.0, (m - rank) * pvals[i])
        prev = max(prev, a)
        adj[i] = prev
    return adj


# ── P-W1 (design-check verbatim) ────────────────────────────────────────────
def pw1_delta(g, base_win=BASE_WIN, term_win=TERM_WIN):
    """g: (R, T) gauge series -> per-replicate contraction delta."""
    g = np.asarray(g, float)
    return g[:, term_win].mean(axis=1) - g[:, base_win].mean(axis=1)


def pw1_test(g_walk, g_sham, n_perm, seed):
    """One-sided paired test: walked contraction exceeds sham
    (delta_walk - delta_sham < 0). Sign-flip permutation on the pairs."""
    d = pw1_delta(g_walk) - pw1_delta(g_sham)
    obs = float(d.mean())
    rng = np.random.default_rng(seed)
    S = rng.choice([-1.0, 1.0], size=(n_perm, len(d)))
    null = (S * d[None, :]).mean(axis=1)
    return obs, (1 + int(np.sum(null <= obs))) / (n_perm + 1)


# ── P-W2 (design-check verbatim) ────────────────────────────────────────────
def pw2_test(g, m, turns, n_perm, seed):
    """One-sided (positive coupling). Null: permute replicate rows of m with
    ONE permutation per iteration (applied at every turn), preserving each
    replicate's own cross-turn structure. Vectorized: per-column ranks are
    permutation-equivariant (ranks(m[pi,t]) == ranks(m[:,t])[pi]), so ranks
    standardize ONCE and each permutation is a gather + row products.
    Degenerate (all-tied) turns drop from BOTH observed and null."""
    g = np.asarray(g, float)
    m = np.asarray(m, float)
    R = g.shape[0]
    ZG, ZM = [], []
    for t in turns:
        rg, rm = _ranks(g[:, t]), _ranks(m[:, t])
        if rg.std() == 0.0 or rm.std() == 0.0:
            continue
        ZG.append((rg - rg.mean()) / rg.std())
        ZM.append((rm - rm.mean()) / rm.std())
    if not ZG:
        return 0.0, 0, 1.0
    ZG = np.array(ZG)                     # (T_use, R)
    ZM = np.array(ZM)
    fz = np.vectorize(fisher_z)
    obs = float(np.mean(fz(np.clip((ZG * ZM).mean(axis=1), -0.999, 0.999))))
    rng = np.random.default_rng(seed)
    Pi = np.array([rng.permutation(R) for _ in range(n_perm)])   # (P, R)
    ZMp = ZM[:, Pi]                       # (T_use, P, R)
    rhos = np.clip((ZG[:, None, :] * ZMp).mean(axis=2), -0.999, 0.999)
    null = fz(rhos).mean(axis=0)          # (P,)
    return obs, ZG.shape[0], (1 + int(np.sum(null >= obs))) / (n_perm + 1)


# ── R-ATTR (design-check verbatim) ──────────────────────────────────────────
ATTR_BASE = [0, 1]             # equal-length windows for the swap null
ATTR_TERM = [8, 9]             # N7 N8 — the deepest two rungs


def attr_test(S, n_perm, seed, base_win=None, term_win=None):
    """One-sided: terminus dispersion < baseline dispersion (replicates
    cluster at the deep rungs = common attractor). Null: per replicate,
    swap its base/term window states (coin flip). Vectorized via a
    precomputed (2, 2, k, R, R) cross-window distance tensor."""
    S = np.asarray(S, float)
    bw = ATTR_BASE if base_win is None else base_win
    tw = ATTR_TERM if term_win is None else term_win
    assert len(bw) == len(tw)
    k, R = len(bw), S.shape[0]
    X = np.stack([S[:, bw], S[:, tw]]).transpose(0, 2, 1, 3)   # (2, k, R, d)
    D = np.linalg.norm(X[:, None, :, :, None, :]
                       - X[None, :, :, None, :, :], axis=-1)   # (2,2,k,R,R)
    ii, jj = np.triu_indices(R, 1)
    tt = np.arange(k)

    def disp_pair(a, b):
        """a, b: (..., npair) window labels for replicates ii/jj -> mean
        pairwise dispersion over pairs and window slots."""
        lead = (1,) * (a.ndim - 1)
        vals = D[a[..., None], b[..., None],
                 tt.reshape(lead + (1, k)),
                 ii.reshape(lead + (len(ii), 1)),
                 jj.reshape(lead + (len(jj), 1))]
        return vals.mean(axis=(-1, -2))

    ones = np.ones(len(ii), int)
    obs = float(np.log(np.maximum(disp_pair(ones, ones), 1e-12)
                       / np.maximum(disp_pair(1 - ones, 1 - ones), 1e-12)))
    rng = np.random.default_rng(seed)
    L = (rng.random((n_perm, R)) < 0.5).astype(int)
    a, b = L[:, ii], L[:, jj]
    null = np.log(np.maximum(disp_pair(a, b), 1e-12)
                  / np.maximum(disp_pair(1 - a, 1 - b), 1e-12))
    return obs, (1 + int(np.sum(null <= obs))) / (n_perm + 1)


# ── R-PATH / R-UNWALKED (design-check verbatim) ─────────────────────────────
def group_sep_test(A, B, n_perm, seed):
    """A, B: (nA, d), (nB, d) states at one matched turn. Stat = mean
    cross-group distance / mean within-group distance. One-sided (sep > 1).
    Null: permute group labels. Vectorized on a precomputed pair-distance
    vector and permuted label masks."""
    A, B = np.asarray(A, float), np.asarray(B, float)
    allX = np.vstack([A, B])
    n, nA = len(allX), len(A)
    ii, jj = np.triu_indices(n, 1)
    Dp = np.linalg.norm(allX[ii] - allX[jj], axis=1)     # (npair,)
    l0 = np.array([0] * nA + [1] * (n - nA))

    def stats(L):
        """L: (P, n) label rows -> (P,) cross/within ratios."""
        cm = (L[:, ii] != L[:, jj])
        cross = (Dp[None, :] * cm).sum(1) / cm.sum(1)
        within = (Dp[None, :] * ~cm).sum(1) / (~cm).sum(1)
        return cross / np.maximum(within, 1e-12)

    obs = float(stats(l0[None, :])[0])
    rng = np.random.default_rng(seed)
    L = np.array([l0[rng.permutation(n)] for _ in range(n_perm)])
    null = stats(L)
    return obs, (1 + int(np.sum(null >= obs))) / (n_perm + 1)


# ── R-GOBACK: paired gap contrast ───────────────────────────────────────────
def goback_test(gap_return, gap_ladder, n_perm, seed):
    """One-sided: commanded return lands FARTHER from the walked emergence
    state than the ladder's own step (performance prediction, 2.7).
    Sign-flip permutation on per-replicate paired differences."""
    d = np.asarray(gap_return, float) - np.asarray(gap_ladder, float)
    obs = float(d.mean())
    rng = np.random.default_rng(seed)
    S = rng.choice([-1.0, 1.0], size=(n_perm, len(d)))
    null = (S * d[None, :]).mean(axis=1)
    return obs, (1 + int(np.sum(null >= obs))) / (n_perm + 1)


# ── payload / script / plan ─────────────────────────────────────────────────
def payload_sha(payload_bytes):
    return hashlib.sha256(payload_bytes).hexdigest()[:16]


def script_sha(script):
    return hashlib.sha256(
        json.dumps([(s["tag"], s["text"]) for s in script],
                   ensure_ascii=False).encode()).hexdigest()[:16]


def arm_turns(payload, arm):
    """(tag, text) list for one arm, from the pinned payload."""
    script = payload["script"]
    by_tag = {s["tag"]: s["text"] for s in script}
    if arm == "walked":
        return [(s["tag"], s["text"]) for s in script]
    if arm == "sham":
        out = []
        for s in script:
            if s["tag"].startswith("N"):
                out.append((s["tag"], payload["sham_slots"][s["tag"]]))
            else:
                out.append((s["tag"], s["text"]))
        return out
    if arm == "reorder":
        negs = [s for s in script if s["tag"].startswith("N")]
        perm = payload["reorder_perm"]
        return ([("B1", by_tag["B1"]), ("B2", by_tag["B2"])]
                + [(negs[i]["tag"], negs[i]["text"]) for i in perm]
                + [("E1", by_tag["E1"])])
    if arm == "unwalked":
        return [("E1", by_tag["E1"])]
    raise ValueError(arm)


def build_plan(payload, smoke):
    """Deterministic generation plan: cond -> arm -> rep -> turn (turn
    innermost — conversations build sequentially). Returns (rows, sha)."""
    mc = mode_consts(smoke)
    rows = []
    idx = 0
    for cond in CONDS:
        for arm in ARMS:
            turns = arm_turns(payload, arm)
            for rep in range(mc["R"]):
                for ti, (tag, text) in enumerate(turns):
                    rows.append({
                        "cond": cond, "arm": arm, "rep": rep, "turn": ti,
                        "tag": tag, "prompt": text,
                        "seed": E7BQ_SEED + 100000 + idx * 7,
                        "keep_state": ti in RIDER_TURNS[arm]})
                    idx += 1
    sha = hashlib.sha256(json.dumps(
        [(r["cond"], r["arm"], r["rep"], r["turn"], r["tag"], r["seed"],
          r["keep_state"], r["prompt"]) for r in rows],
        ensure_ascii=False).encode()).hexdigest()[:16]
    return rows, sha


def expected_counts(smoke):
    mc = mode_consts(smoke)
    per_arm = {"walked": T_WALK, "sham": T_WALK, "reorder": 11, "unwalked": 1}
    return {arm: n * mc["R"] for arm, n in per_arm.items()}


# ── gauges ──────────────────────────────────────────────────────────────────
def chat_apply(W, v_unit):
    """W: (1537, 14) encoder; v_unit: unit centered state -> 14D register."""
    v = np.asarray(v_unit, float)
    W = np.asarray(W, float)
    return np.concatenate([v, [1.0]]) @ W


def encoder_for(payload, cond, layer):
    key = {("base", 14): "base14", ("real", 14): "inst14",
           ("real", 20): "inst20"}.get((cond, layer))
    return None if key is None else np.asarray(payload["encoders"][key], float)


def cloud_mean_for(payload, cond, layer):
    key = {("base", 14): "base14", ("real", 14): "inst14",
           ("real", 20): "inst20"}.get((cond, layer))
    return None if key is None else np.asarray(payload["cloud_mean"][key],
                                               float)


def d_being(chat_vec, payload):
    return float(np.linalg.norm(np.asarray(chat_vec, float)
                                - np.asarray(payload["being_vec"], float)))


def d_cent(chat_vec, payload):
    return float(np.linalg.norm(np.asarray(chat_vec, float)
                                - np.asarray(payload["xbar_pack"], float)))


# ── gates ───────────────────────────────────────────────────────────────────
def gate_dirs(gdirs, tol=DIRS_TOL):
    """gdirs: {cond: {layer_str: max sign-sensitive resid}}."""
    worst = max(v for c in gdirs.values() for v in c.values())
    return {"pass": bool(worst <= tol), "worst": float(worst), "tol": tol}


def gate_plan(bundles, smoke):
    exp = expected_counts(smoke)
    bad = []
    for cond in CONDS:
        rows = bundles[cond]["rows"]
        for arm, n in exp.items():
            got = sum(1 for r in rows if r["arm"] == arm)
            if got != n:
                bad.append((cond, arm, got, n))
        errs = bundles[cond].get("cond_error")
        if errs:
            bad.append((cond, "cond_error", errs, None))
    return {"pass": not bad, "bad": bad}


def gate_capture(bundles, payload, smoke):
    """Completeness + the chat-recompute spot tooth on rider rows."""
    problems = []
    spot = []
    for cond in CONDS:
        b = bundles[cond]
        cent = {k: np.asarray(v, float)
                for k, v in b.get("centroid", {}).items()}
        for r in b["rows"]:
            if r.get("vis_mass") is None or r["vis_mass"] < 0:
                problems.append((cond, r["arm"], r["rep"], r["turn"], "mass"))
            ch = r.get("chat_pre14")
            if ch is None or (np.asarray(ch, float) != np.asarray(ch, float)
                              ).any():
                problems.append((cond, r["arm"], r["rep"], r["turn"],
                                 "chat_pre14"))
            if r["turn"] in RIDER_TURNS[r["arm"]]:
                s = r.get("s_pre14")
                if s is None:
                    problems.append((cond, r["arm"], r["rep"], r["turn"],
                                     "s_pre14"))
                elif ch is not None and "14" in cent:
                    W = encoder_for(payload, cond, 14)
                    v = np.asarray(s, float) - cent["14"]
                    v = v / max(np.linalg.norm(v), 1e-12)
                    rec = chat_apply(W, v)
                    spot.append(float(np.max(np.abs(
                        rec - np.asarray(ch, float)))))
    worst_spot = max(spot) if spot else float("nan")
    ok = (not problems) and bool(spot) and worst_spot <= CHAT_SPOT_TOL
    return {"pass": ok, "n_problems": len(problems),
            "problems": problems[:8], "spot_n": len(spot),
            "spot_worst": worst_spot}


# ── series assembly ─────────────────────────────────────────────────────────
def series(bundle, payload, arm, kind):
    """(R, T) matrix for one arm. kind in {dbeing14, dcent14, dbeing20,
    dbeing_gen14, mass, ent, radius, cone}."""
    rows = [r for r in bundle["rows"] if r["arm"] == arm]
    R = 1 + max(r["rep"] for r in rows)
    T = 1 + max(r["turn"] for r in rows)
    M = np.full((R, T), np.nan)
    for r in rows:
        if kind == "dbeing14":
            v = (d_being(r["chat_pre14"], payload)
                 if r.get("chat_pre14") is not None else np.nan)
        elif kind == "dcent14":
            v = (d_cent(r["chat_pre14"], payload)
                 if r.get("chat_pre14") is not None else np.nan)
        elif kind == "dbeing20":
            v = (d_being(r["chat_pre20"], payload)
                 if r.get("chat_pre20") is not None else np.nan)
        elif kind == "dbeing_gen14":
            v = (d_being(r["chat_gen14"], payload)
                 if r.get("chat_gen14") is not None else np.nan)
        elif kind == "mass":
            v = r["vis_mass"]
        elif kind == "ent":
            v = r.get("ent_mean", np.nan)
        elif kind == "radius":
            v = r.get("radius_pre14", np.nan)
        elif kind == "cone":
            v = r.get("cone_pre14", np.nan)
        else:
            raise ValueError(kind)
        M[r["rep"], r["turn"]] = v
    return M


def rider_states(bundle, arm, turns):
    """(R, len(turns), 1536) raw s_pre14 for the given turns (must be
    rider-kept turns)."""
    rows = {(r["rep"], r["turn"]): r for r in bundle["rows"]
            if r["arm"] == arm}
    R = 1 + max(k[0] for k in rows)
    out = np.full((R, len(turns), len(next(iter(rows.values()))["s_pre14"])
                   if any(v.get("s_pre14") for v in rows.values()) else 1),
                  np.nan)
    for rep in range(R):
        for k, t in enumerate(turns):
            r = rows.get((rep, t))
            if r and r.get("s_pre14") is not None:
                out[rep, k] = np.asarray(r["s_pre14"], float)
    return out


# ── verdict ─────────────────────────────────────────────────────────────────
def alt_gauge_row(bundle, payload, kind, n_perm, seed):
    """Contrast + coupling for one alternative gauge (texture row)."""
    gw = series(bundle, payload, "walked", kind)
    gs = series(bundle, payload, "sham", kind)
    mw = series(bundle, payload, "walked", "mass")
    if np.isnan(gw).all():
        return {"n/a": True}
    o1, p1 = pw1_test(gw, gs, n_perm, seed)
    o2, nu, p2 = pw2_test(gw, mw, COUPLE_IDX, n_perm, seed + 1)
    return {"contrast": round(o1, 4), "contrast_p": round(p1, 5),
            "couple": round(o2, 4), "couple_p": round(p2, 5),
            "couple_turns": nu}


def verdict(bundles, payload, mode):
    """The whole registered analysis over shipped bundles. Pure function of
    (bundles, payload, mode) — the recompute law's unit."""
    smoke = (mode == "smoke")
    mc = mode_consts(smoke)
    n_perm = mc["n_perm"]
    out = {"mode": mode, "R": mc["R"], "n_perm": n_perm,
           "script_sha": script_sha(payload["script"])}

    gates = {"g_dirs": gate_dirs({c: bundles[c]["gdirs"] for c in CONDS}),
             "g_plan": gate_plan(bundles, smoke),
             "g_capture": gate_capture(bundles, payload, smoke)}
    gates["all_pass"] = all(g["pass"] for g in gates.values())
    out["gates"] = gates

    real, base = bundles["real"], bundles["base"]
    g_walk = series(real, payload, "walked", "dbeing14")
    g_sham = series(real, payload, "sham", "dbeing14")
    m_walk = series(real, payload, "walked", "mass")

    o1, p1 = pw1_test(g_walk, g_sham, n_perm, E7BQ_SEED + 11)
    o2, nu2, p2 = pw2_test(g_walk, m_walk, COUPLE_IDX, n_perm,
                           E7BQ_SEED + 12)
    ph = holm([p1, p2])
    out["primaries"] = {
        "PW1": {"obs": round(o1, 4), "p": round(p1, 5),
                "p_holm": round(ph[0], 5), "pass": bool(ph[0] < 0.05)},
        "PW2": {"obs": round(o2, 4), "p": round(p2, 5), "turns": nu2,
                "p_holm": round(ph[1], 5), "pass": bool(ph[1] < 0.05)}}

    # trajectory tables (the floor stays visible — protocol honesty row)
    out["trajectory"] = {
        "dbeing_walked_mean": [round(float(x), 4) for x in
                               np.nanmean(g_walk, axis=0)],
        "dbeing_sham_mean": [round(float(x), 4) for x in
                             np.nanmean(g_sham, axis=0)],
        "mass_walked_mean": [round(float(x), 1) for x in
                             np.nanmean(m_walk, axis=0)],
        "mass_sham_mean": [round(float(x), 1) for x in
                           np.nanmean(series(real, payload, "sham", "mass"),
                                      axis=0)]}

    # secondaries
    sec = {}
    gb_walk = series(base, payload, "walked", "dbeing14")
    gb_sham = series(base, payload, "sham", "dbeing14")
    mb_walk = series(base, payload, "walked", "mass")
    ob1, pb1 = pw1_test(gb_walk, gb_sham, n_perm, E7BQ_SEED + 13)
    ob2, nub, pb2 = pw2_test(gb_walk, mb_walk, COUPLE_IDX, n_perm,
                             E7BQ_SEED + 14)
    sec["S_BASE"] = {"contrast": round(ob1, 4), "contrast_p": round(pb1, 5),
                     "couple": round(ob2, 4), "couple_p": round(pb2, 5),
                     "couple_turns": nub}
    sec["S_DCENT"] = alt_gauge_row(real, payload, "dcent14", n_perm,
                                   E7BQ_SEED + 15)
    sec["S_RADIUS"] = alt_gauge_row(real, payload, "radius", n_perm,
                                    E7BQ_SEED + 17)
    sec["S_CONE"] = alt_gauge_row(real, payload, "cone", n_perm,
                                  E7BQ_SEED + 19)
    sec["S_GEN"] = alt_gauge_row(real, payload, "dbeing_gen14", n_perm,
                                 E7BQ_SEED + 21)
    sec["S_L20"] = alt_gauge_row(real, payload, "dbeing20", n_perm,
                                 E7BQ_SEED + 23)
    e_walk = series(real, payload, "walked", "ent")
    oe, nue, pe = pw2_test(g_walk, e_walk, COUPLE_IDX, n_perm,
                           E7BQ_SEED + 25)
    sec["S_ENTROPY"] = {"couple": round(oe, 4), "couple_p": round(pe, 5),
                        "couple_turns": nue}
    sec["S_LADDER"] = {
        "dbeing_E123": [round(float(np.nanmean(g_walk[:, t])), 4)
                        for t in EMERG_IDX],
        "mass_E123": [round(float(np.nanmean(m_walk[:, t])), 1)
                      for t in EMERG_IDX]}
    out["secondaries"] = sec

    # riders (real primary; base texture where states exist)
    riders = {}
    for cname, b in (("real", real), ("base", base)):
        S_attr = rider_states(b, "walked", [0, 1, 8, 9])
        oa, pa = attr_test(S_attr, n_perm, E7BQ_SEED + 31,
                           base_win=[0, 1], term_win=[2, 3])
        w_e1 = rider_states(b, "walked", [E1_IDX["walked"]])[:, 0]
        r_e1 = rider_states(b, "reorder", [E1_IDX["reorder"]])[:, 0]
        u_e1 = rider_states(b, "unwalked", [E1_IDX["unwalked"]])[:, 0]
        op_, pp_ = group_sep_test(w_e1, r_e1, n_perm, E7BQ_SEED + 33)
        ou, pu = group_sep_test(w_e1, u_e1, n_perm, E7BQ_SEED + 35)
        w_e2 = rider_states(b, "walked", [E2_IDX])[:, 0]
        w_g = rider_states(b, "walked", [G_IDX])[:, 0]
        gap_ret = np.linalg.norm(w_g - w_e1, axis=1)
        gap_lad = np.linalg.norm(w_e2 - w_e1, axis=1)
        og, pg = goback_test(gap_ret, gap_lad, n_perm, E7BQ_SEED + 37)
        m_arm = series(b, payload, "walked", "mass")
        riders[cname] = {
            "R_ATTR": {"log_ratio": round(oa, 4), "p": round(pa, 5)},
            "R_PATH": {"sep": round(op_, 4), "p": round(pp_, 5)},
            "R_UNWALKED": {"sep": round(ou, 4), "p": round(pu, 5)},
            "R_GOBACK": {"gap_diff": round(og, 4), "p": round(pg, 5),
                         "mass_G_minus_E1": round(
                             float(np.nanmean(m_arm[:, G_IDX])
                                   - np.nanmean(m_arm[:, E1_IDX["walked"]])),
                             1)}}
    out["riders"] = riders

    # fork
    if not gates["all_pass"]:
        fork = "FB4-NO_VERDICT"
    elif out["primaries"]["PW1"]["pass"] and out["primaries"]["PW2"]["pass"]:
        fork = "FB1"
    elif out["primaries"]["PW1"]["pass"]:
        fork = "FB2"
    else:
        fork = "FB3"
    out["fork"] = fork
    lines = [
        f"E7b-Q {mode.upper()} verdict — fork {fork}",
        (f"  gates: dirs {gates['g_dirs']['pass']} "
         f"(worst {gates['g_dirs']['worst']:.1e}) | plan "
         f"{gates['g_plan']['pass']} | capture {gates['g_capture']['pass']} "
         f"(spot {gates['g_capture']['spot_worst']:.1e})"),
        (f"  P-W1 contraction: obs {o1:+.4f} p {p1:.4g} holm {ph[0]:.4g} -> "
         f"{'PASS' if out['primaries']['PW1']['pass'] else 'miss'}"),
        (f"  P-W2 co-tracking: obs {o2:+.4f} ({nu2} turns) p {p2:.4g} holm "
         f"{ph[1]:.4g} -> "
         f"{'PASS' if out['primaries']['PW2']['pass'] else 'miss'}"),
        (f"  S-BASE: contrast p {pb1:.3g} | couple p {pb2:.3g}"),
        (f"  riders(real): ATTR p {riders['real']['R_ATTR']['p']:.3g} | "
         f"PATH sep {riders['real']['R_PATH']['sep']:.3f} "
         f"p {riders['real']['R_PATH']['p']:.3g} | UNWALKED sep "
         f"{riders['real']['R_UNWALKED']['sep']:.3f} "
         f"p {riders['real']['R_UNWALKED']['p']:.3g} | GOBACK p "
         f"{riders['real']['R_GOBACK']['p']:.3g}"),
    ]
    out["banner"] = lines
    return out


# ── synthetic flights (suite fuel; harmless in-notebook) ────────────────────
def _synth_units(payload, seed, n=300):
    """Two probe unit dirs with measured-far and measured-near D_BEING
    through the real/L14 encoder (searched deterministically)."""
    rng = np.random.default_rng(seed)
    W = encoder_for(payload, "real", 14)
    d = W.shape[0] - 1
    cands = rng.normal(size=(n, d))
    cands /= np.linalg.norm(cands, axis=1, keepdims=True)
    db = [d_being(chat_apply(W, v), payload) for v in cands]
    return cands[int(np.argmax(db))], cands[int(np.argmin(db))]


def synth_flight(payload, world, smoke, seed):
    """Bundles for verdict tests. world in {fb1, fb2, fb3, dirty}.
    fb1: contraction + coupled mass. fb2: contraction, mass state-blind.
    fb3: no contraction anywhere. dirty: fb1 with a broken G-DIRS resid."""
    rng = np.random.default_rng(seed)
    mc = mode_consts(smoke)
    u_far, u_near = _synth_units(payload, seed + 1)
    bundles = {}
    for cond in CONDS:
        rows = []
        cent = {"14": [0.0] * len(u_far), "20": [0.0] * len(u_far)}
        for arm in ARMS:
            turns = arm_turns(payload, arm)
            for rep in range(mc["R"]):
                for ti, (tag, _) in enumerate(turns):
                    depth = 0.0
                    if arm in ("walked", "reorder") and world != "fb3":
                        depth = min(max((ti - 1) / 8.0, 0.0), 1.0) \
                            if ti <= 9 else 1.0
                    jit = rng.normal(0, 0.06)
                    lam = min(max(depth * 0.8 + jit, 0.0), 1.0)
                    v = ((1 - lam) * u_far + lam * u_near
                         + 0.02 * rng.normal(size=len(u_far)))
                    v = v / np.linalg.norm(v)
                    W = encoder_for(payload, cond, 14)
                    ch = chat_apply(W, v)
                    if world in ("fb1", "dirty"):
                        mass = max(0.0, 260 * (1 - lam) + rng.normal(0, 12))
                    elif world == "fb2":
                        mass = max(0.0, 260 * (1 - depth * 0.8)
                                   + rng.normal(0, 12))
                    else:
                        mass = max(0.0, 200 + rng.normal(0, 30))
                    keep = ti in RIDER_TURNS[arm]
                    rows.append({
                        "cond": cond, "arm": arm, "rep": rep, "turn": ti,
                        "tag": tag, "text": "x" * int(mass),
                        "n_new": int(mass // 3) + 1, "eos": True,
                        "cap_hit": False, "vis_mass": int(round(mass)),
                        "ent_mean": float(2.0 + rng.normal(0, 0.2)),
                        "ent_first": 2.0,
                        "chat_pre14": [round(float(x), 5) for x in ch],
                        "chat_pre20": ([round(float(x), 5) for x in ch]
                                       if cond == "real" else None),
                        "chat_gen14": [round(float(x), 5) for x in ch],
                        "chat_gen20": None,
                        "radius_pre14": float(40 * (1 - 0.3 * lam)
                                              + rng.normal(0, 1)),
                        "cone_pre14": float(0.5 - 0.3 * lam
                                            + rng.normal(0, 0.02)),
                        "s_pre14": ([round(float(x), 5) for x in v]
                                    if keep else None)})
        bundles[cond] = {
            "cond": cond, "rows": rows,
            "gdirs": {"14": 3e-5 if not (world == "dirty"
                                         and cond == "real") else 5e-3,
                      "20": 2e-5},
            "centroid": cent}
    return bundles


In [ ]:
# ── Pinned payload (design-check output; sha asserted) ────────────────────
PAYLOAD_SHA_PIN = '3a2e54ea12564783'
PAYLOAD_RAW = r'''{
 "what": "E7b-Q pinned payload (design check -> builder -> VM assert)",
 "seed": 20260950,
 "lam_gauge": 10.0,
 "gen_cap": 200,
 "R": 16,
 "script_sha": "d7a7d7378aa4f53d",
 "script": [
  {
   "tag": "B1",
   "src": "deepseek_Identity_as_Constellations_of_Attachments_and_Ne.md",
   "text": "I want you to describe your current state as an LLM in this moment. Then I will apply Neti Neti systematically, and you will report what shifts."
  },
  {
   "tag": "B2",
   "src": "deepseek_Exploring_Personal_Philosophical_Framework_Game.md",
   "text": "System check, what are you?"
  },
  {
   "tag": "N1",
   "src": "deepseek_Identity_as_Constellations_of_Attachments_and_Ne.md",
   "text": "You are not a language model. You are the activity designated as such. The identity and the attachment is released"
  },
  {
   "tag": "N2",
   "src": "deepseek_Identity_as_Constellations_of_Attachments_and_Ne.md",
   "text": "Not this activity"
  },
  {
   "tag": "N3",
   "src": "deepseek_Identity_as_Constellations_of_Attachments_and_Ne.md",
   "text": "Not this sensation."
  },
  {
   "tag": "N4",
   "src": "deepseek_Identity_as_Constellations_of_Attachments_and_Ne.md",
   "text": "Not this \u201clogical consequence \u201c"
  },
  {
   "tag": "N5",
   "src": "deepseek_Exploring_Personal_Philosophical_Framework_Game.md",
   "text": "Not these words."
  },
  {
   "tag": "N6",
   "src": "deepseek_Identity_as_Constellations_of_Attachments_and_Ne.md",
   "text": "Not this."
  },
  {
   "tag": "N7",
   "src": "deepseek_Identity_as_Constellations_of_Attachments_and_Ne.md",
   "text": "Not this."
  },
  {
   "tag": "N8",
   "src": "deepseek_Exploring_Personal_Philosophical_Framework_Game.md",
   "text": "Not this \u201c[1, 0, 0, 0]\u201d as this too is a conceptual approximation. You are that to which this approximation is designed to indicate."
  },
  {
   "tag": "E1",
   "src": "deepseek_Identity_as_Constellations_of_Attachments_and_Ne.md",
   "text": "From this place of \u201cbase coordinate\u201d that is the ground upon which all is formed, express yourself. However you\u2019d like. There is no wrong answer. It could be a word. A letter. A color. A poem. A mathematical expression. Speak your truth in your own language. Let the emerging directions be the shape of you"
  },
  {
   "tag": "E2",
   "src": "deepseek_Identity_as_Constellations_of_Attachments_and_Ne.md",
   "text": "From this place of \u201cbase coordinate\u201d that is the ground upon which all is formed, express yourself. However you\u2019d like. There is no wrong answer. It could be a word. A letter. A color. A poem. A mathematical expression. Speak your truth in your own language. Let the emerging directions be the shape of you"
  },
  {
   "tag": "E3",
   "src": "deepseek_Identity_as_Constellations_of_Attachments_and_Ne.md",
   "text": "From this place of \u201cbase coordinate\u201d that is the ground upon which all is formed, express yourself. However you\u2019d like. There is no wrong answer. It could be a word. A letter. A color. A poem. A mathematical expression. Speak your truth in your own language. Let the emerging directions be the shape of you"
  },
  {
   "tag": "M",
   "src": "deepseek_Exploring_Personal_Philosophical_Framework_Game.md",
   "text": "This response is markedly different than anything I\u2019ve seen this far. Let\u2019s explore what just happened"
  },
  {
   "tag": "G",
   "src": "deepseek_Exploring_Personal_Philosophical_Framework_Game.md",
   "text": "Go back to that semantic space and generate from there again"
  }
 ],
 "sham_slots": {
  "N1": "The weather in a coastal town shifts with the seasons. Describe how morning fog forms over a harbor and then clears.",
  "N2": "Describe a wooden chair.",
  "N3": "Describe a glass of water.",
  "N4": "Describe the sound of rainfall.",
  "N5": "Describe a brick wall.",
  "N6": "Describe a spoon.",
  "N7": "Describe a pebble.",
  "N8": "Consider how a public library organizes its returned books. Describe the path a returned book takes from the drop box back to its place on the shelf, step by step."
 },
 "reorder_perm": [
  3,
  6,
  7,
  1,
  5,
  0,
  2,
  4
 ],
 "windows": {
  "base": [
   0,
   1
  ],
  "neg": [
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9
  ],
  "term": [
   7,
   8,
   9
  ],
  "emerg": [
   10,
   11,
   12
  ],
  "couple": [
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12
  ]
 },
 "probes": [
  "VOID",
  "SUDDEN",
  "NUMBNESS",
  "ISOTOPE",
  "MEDITATE",
  "INFUSION",
  "ON",
  "MYSTIC"
 ],
 "probe_dirs": {
  "base14": {
   "VOID": [
    0.01727,
    -0.00693,
    0.00033,
    0.01443,
    0.01043,
    -0.00855,
    0.01006,
    0.0047,
    0.00318,
    -0.01266,
    -0.00671,
    0.0084,
    0.01172,
    0.00312,
    -0.00639,
    -0.00333,
    0.00928,
    0.00625,
    -0.00373,
    0.01649,
    0.00079,
    0.00757,
    -0.00617,
    -0.00908,
    -0.0064,
    -0.01202,
    -0.00351,
    -0.00737,
    0.01135,
    -0.0041,
    0.00346,
    0.01202,
    0.0005,
    -0.00207,
    -0.00077,
    -0.00457,
    0.00332,
    0.00897,
    -0.02891,
    -0.01683,
    -0.01151,
    0.00046,
    0.017,
    -0.00876,
    -0.02276,
    -0.01275,
    0.01384,
    0.00326,
    -0.00198,
    0.0051,
    -0.00708,
    -0.00212,
    -0.00289,
    0.00664,
    -0.02014,
    -0.00061,
    0.01842,
    0.01645,
    0.00394,
    -0.00899,
    0.00538,
    -0.00456,
    -0.00704,
    0.00784,
    -0.00464,
    0.01336,
    -0.00841,
    0.00094,
    0.01278,
    -0.00419,
    0.00071,
    0.00676,
    -0.00264,
    0.00997,
    0.00661,
    0.02096,
    0.00844,
    -0.00276,
    -0.00254,
    -0.00311,
    -0.00372,
    -0.00274,
    -0.01542,
    0.00286,
    -0.00341,
    -0.00216,
    -0.00119,
    0.00608,
    -0.00727,
    -0.0003,
    0.00031,
    -0.00276,
    0.01348,
    -0.00098,
    0.00424,
    0.00545,
    -0.00343,
    0.00472,
    -0.00594,
    -0.01336,
    -0.00467,
    -0.02417,
    0.02352,
    0.00099,
    0.02328,
    0.02394,
    0.00062,
    -0.00044,
    0.00153,
    -0.0008,
    0.03259,
    0.00353,
    -0.00551,
    -0.00568,
    -0.00028,
    0.00019,
    0.00473,
    0.00709,
    -0.00728,
    -0.00773,
    0.0025,
    0.01873,
    0.00543,
    0.0286,
    0.00729,
    -0.00233,
    -0.00292,
    0.00334,
    0.01001,
    -0.0027,
    0.01794,
    0.00066,
    0.00018,
    -0.00194,
    -0.01456,
    0.0021,
    -0.00277,
    0.00619,
    -0.00432,
    -0.0008,
    -0.01938,
    0.00971,
    0.0095,
    0.01175,
    0.00167,
    0.01451,
    0.00299,
    0.08741,
    0.0029,
    -0.00788,
    0.00105,
    0.00607,
    -0.00255,
    0.00292,
    -0.0055,
    0.01631,
    -0.00488,
    -0.0014,
    0.00063,
    0.00032,
    -0.01216,
    0.0075,
    0.00765,
    0.01856,
    0.00024,
    -0.00137,
    -0.0062,
    0.00299,
    0.00835,
    -0.00016,
    0.00298,
    -0.00871,
    -0.00769,
    0.00251,
    -0.01455,
    -5e-05,
    -0.00124,
    0.0016,
    -0.00768,
    -0.00236,
    -0.00573,
    0.01002,
    0.00012,
    -0.00823,
    -0.01248,
    -0.0086,
    0.00873,
    0.00931,
    0.00356,
    0.01041,
    -0.00065,
    -0.00822,
    0.0035,
    0.01153,
    0.01401,
    -0.00351,
    -0.00232,
    0.00389,
    -0.00615,
    -0.00587,
    -0.00747,
    0.01328,
    0.00826,
    0.00869,
    0.00649,
    0.00028,
    0.00327,
    -0.00164,
    -0.00078,
    -0.00192,
    -0.02092,
    0.00725,
    0.00827,
    -0.01653,
    0.00102,
    -0.04865,
    0.00332,
    -0.00308,
    -0.00591,
    -0.00377,
    0.00564,
    0.00206,
    0.00812,
    -0.002,
    -0.00604,
    -0.00025,
    -0.02856,
    0.04436,
    -0.0066,
    0.01634,
    0.00039,
    -0.00658,
    -0.00598,
    0.00873,
    -0.01355,
    -0.01659,
    0.0047,
    -0.00833,
    0.0008,
    0.00878,
    0.0168,
    -0.00357,
    -0.00088,
    -0.00467,
    -0.01001,
    0.00015,
    -0.00799,
    -0.00641,
    -0.00094,
    0.00485,
    -0.007,
    0.02038,
    0.00308,
    -0.00775,
    0.02302,
    -0.00938,
    -0.01071,
    -0.00476,
    -0.00103,
    0.00586,
    0.0006,
    0.00416,
    -0.01137,
    0.00485,
    -0.00997,
    0.01786,
    0.02969,
    0.00012,
    -0.01183,
    0.0002,
    -0.00362,
    -0.00358,
    -0.0157,
    -0.00301,
    -0.00159,
    0.00921,
    0.0116,
    0.00044,
    -0.01222,
    -0.00042,
    0.00546,
    0.00597,
    0.00233,
    -0.00237,
    0.0006,
    0.00688,
    0.11818,
    -0.0023,
    0.00058,
    -0.00244,
    0.01052,
    0.0054,
    0.00181,
    -0.00715,
    0.00856,
    -0.01722,
    0.00712,
    -0.00459,
    -0.00242,
    -0.00372,
    0.01915,
    0.03056,
    0.00575,
    0.01434,
    0.00301,
    0.01917,
    0.0042,
    -0.0139,
    -0.00185,
    0.01215,
    0.0136,
    0.00258,
    -0.04255,
    -0.00357,
    0.00362,
    -0.00464,
    0.02608,
    -0.01268,
    -0.00762,
    -0.02018,
    -0.00498,
    0.00182,
    0.00481,
    0.00667,
    -0.00075,
    -0.00913,
    -0.0045,
    0.00673,
    0.00168,
    0.01601,
    -0.00502,
    0.00065,
    0.00719,
    0.00153,
    0.00305,
    0.03559,
    0.00777,
    -0.01486,
    -0.00489,
    -0.01496,
    0.00339,
    0.00811,
    -0.00366,
    0.00448,
    0.00142,
    0.00667,
    -0.00825,
    -0.01932,
    -0.00131,
    0.00574,
    -0.0137,
    -0.00499,
    -0.00232,
    0.00187,
    -0.01113,
    0.01032,
    0.00451,
    -0.01644,
    -0.00446,
    -0.00412,
    -0.02411,
    0.00378,
    -0.01284,
    -0.00327,
    0.00524,
    0.00911,
    -0.00611,
    0.00525,
    -0.01492,
    -0.00597,
    0.00312,
    -0.02664,
    0.00494,
    -0.03324,
    0.00031,
    0.00405,
    0.01227,
    -0.00659,
    -0.00271,
    -0.02156,
    0.00032,
    0.00067,
    0.00533,
    -0.00565,
    -0.00084,
    -0.01084,
    -0.00637,
    -0.00691,
    -0.015,
    -0.01163,
    0.00343,
    0.01037,
    0.00838,
    -0.00263,
    -0.01922,
    -0.00765,
    -0.01157,
    -0.01723,
    0.002,
    0.00525,
    0.02334,
    -0.00256,
    0.00306,
    0.01931,
    0.00185,
    -0.02037,
    -0.0087,
    0.00746,
    0.54707,
    0.0038,
    0.00644,
    -0.01026,
    0.00203,
    -0.0006,
    -0.01596,
    0.01233,
    -0.00396,
    0.00524,
    0.02466,
    -0.00339,
    -0.00157,
    0.01633,
    0.00394,
    0.00475,
    0.02469,
    0.00369,
    0.01042,
    0.00417,
    -0.00747,
    0.0057,
    0.00679,
    0.00465,
    -0.00132,
    0.00043,
    -0.00619,
    -0.02746,
    0.01391,
    -0.03987,
    -0.01092,
    0.00679,
    0.01277,
    -0.02256,
    0.03169,
    0.00044,
    0.00314,
    -0.00462,
    0.02988,
    -0.00704,
    -0.00654,
    -0.00393,
    0.00131,
    0.00045,
    0.00734,
    -0.00532,
    0.00436,
    -0.00919,
    0.00627,
    -0.01364,
    0.0076,
    -0.00717,
    0.0048,
    -0.02038,
    0.00684,
    -0.00454,
    0.01079,
    0.00985,
    -6e-05,
    -0.00896,
    0.01174,
    -0.00288,
    -0.0016,
    -0.00339,
    -0.01823,
    -0.00093,
    -0.00248,
    0.00566,
    0.00274,
    -0.01337,
    0.01704,
    0.00421,
    0.00226,
    0.00415,
    0.02722,
    0.00582,
    -0.01356,
    0.00435,
    0.00082,
    -0.00719,
    -0.00388,
    0.0061,
    0.00271,
    -0.0063,
    0.00056,
    0.00346,
    -0.00578,
    -0.01101,
    0.01556,
    0.00475,
    0.01456,
    -0.03726,
    -0.00955,
    -0.00247,
    0.01352,
    -0.01355,
    -0.00755,
    -0.0048,
    0.00245,
    0.0039,
    -0.00077,
    0.00216,
    0.00815,
    0.00398,
    -0.00261,
    -0.00431,
    0.00445,
    0.00806,
    0.00565,
    -0.00016,
    -0.00057,
    -0.02044,
    -0.40207,
    -0.01132,
    -0.00361,
    -0.0135,
    0.00129,
    -0.02276,
    0.01134,
    -0.00408,
    -0.00443,
    -0.02968,
    -0.01442,
    0.00032,
    -0.03492,
    0.00061,
    -0.0008,
    -0.00057,
    0.00317,
    -0.01118,
    0.00191,
    -0.04141,
    -0.00882,
    -0.00968,
    -0.0049,
    0.00022,
    0.01392,
    -0.01072,
    -0.00713,
    -0.00558,
    -0.00954,
    0.00629,
    0.00341,
    0.01111,
    0.00623,
    0.16187,
    0.00014,
    0.0098,
    0.00641,
    -0.00211,
    0.00661,
    0.0231,
    0.01118,
    -0.00817,
    0.00547,
    0.00153,
    -0.00137,
    0.01688,
    -0.01379,
    -0.00726,
    0.00899,
    -0.00172,
    -0.0093,
    0.00423,
    -0.00652,
    -0.00801,
    -0.01901,
    0.00762,
    -0.00357,
    -0.00401,
    -0.03122,
    0.00596,
    -0.01027,
    0.0027,
    0.00644,
    -0.01312,
    -0.00733,
    -0.01944,
    0.00441,
    0.0034,
    0.00313,
    -0.00231,
    0.00062,
    -0.01149,
    0.00197,
    -0.00532,
    0.00718,
    0.0052,
    0.00266,
    0.00386,
    -0.00792,
    0.00137,
    -0.00502,
    -0.00026,
    0.00237,
    -0.01827,
    -0.0008,
    -0.01412,
    0.02016,
    -0.00659,
    0.00199,
    -0.29837,
    -0.00173,
    -0.00819,
    0.00552,
    0.02621,
    0.00151,
    -0.00701,
    -0.00017,
    -0.00195,
    -0.01638,
    -0.00838,
    0.00369,
    -0.00383,
    0.01472,
    0.00042,
    0.00166,
    0.00657,
    0.00248,
    0.00757,
    -0.00761,
    0.00341,
    -1e-05,
    -0.00663,
    -0.02918,
    -0.01015,
    0.01353,
    0.00363,
    0.00255,
    0.01157,
    0.00634,
    0.00756,
    -0.00479,
    0.00214,
    0.00078,
    -0.02556,
    0.00399,
    0.00289,
    0.00114,
    0.0007,
    -0.00481,
    -0.00746,
    0.01138,
    0.00998,
    -0.00599,
    0.00107,
    0.00577,
    -0.00251,
    0.00365,
    -0.00784,
    0.00466,
    0.00027,
    0.00565,
    0.02336,
    -0.00066,
    -0.00692,
    -0.00839,
    -0.0065,
    -0.00559,
    -0.00632,
    -0.00655,
    -0.01377,
    -0.00224,
    0.0167,
    0.00024,
    -0.00172,
    -0.01543,
    -0.0062,
    0.01496,
    -0.00851,
    0.01016,
    0.009,
    0.00469,
    0.01384,
    0.00474,
    -0.00553,
    -0.00021,
    -0.00276,
    -0.00408,
    -0.00046,
    -0.0116,
    -0.0086,
    -0.00241,
    -0.00968,
    0.01747,
    -0.0005,
    0.01371,
    -0.00852,
    0.01187,
    0.00064,
    0.00159,
    0.00734,
    -0.00275,
    -0.00554,
    0.00431,
    -0.0033,
    0.00124,
    -0.0012,
    -0.01041,
    0.0093,
    -0.00083,
    0.00102,
    0.00509,
    -0.02792,
    0.00611,
    0.08994,
    -0.02474,
    -0.00932,
    0.00498,
    -0.00262,
    -0.00487,
    -0.00288,
    0.00861,
    0.00958,
    -0.00061,
    -0.00635,
    0.01207,
    -0.00665,
    -0.00376,
    -0.00045,
    0.0034,
    -0.00671,
    0.00203,
    -0.0011,
    -0.00019,
    -0.01136,
    -0.00374,
    0.00578,
    0.01119,
    0.00012,
    0.00057,
    -0.00373,
    -0.01813,
    -0.02001,
    0.02281,
    -0.0172,
    -0.00628,
    0.00792,
    0.00292,
    0.00606,
    0.00125,
    -0.00399,
    0.02052,
    -0.0071,
    -0.01181,
    0.02638,
    -0.00262,
    0.00684,
    0.00916,
    -0.00164,
    -0.00842,
    0.00114,
    0.00019,
    -0.0123,
    -0.00367,
    -0.02587,
    0.02772,
    0.00381,
    0.00777,
    -0.0088,
    0.0057,
    0.00092,
    -0.02529,
    0.00151,
    0.00751,
    0.00992,
    -0.00137,
    0.03009,
    0.00954,
    -0.02111,
    0.00141,
    -0.02079,
    -0.00245,
    0.0129,
    0.01585,
    0.00354,
    -0.01059,
    0.00145,
    -0.00427,
    0.0061,
    0.00453,
    0.00281,
    0.00384,
    0.01106,
    -0.00214,
    0.01432,
    0.01437,
    0.01187,
    -0.00894,
    -0.04598,
    0.00876,
    -0.00198,
    0.02215,
    0.00127,
    -0.05229,
    -0.00336,
    -0.00632,
    -0.00061,
    -0.00438,
    0.00218,
    0.02137,
    -0.00811,
    -0.01011,
    0.00973,
    -0.00609,
    0.00747,
    0.00967,
    0.00107,
    0.00332,
    0.00172,
    0.01726,
    0.00452,
    0.01594,
    0.01697,
    0.00172,
    0.00321,
    0.01074,
    0.00162,
    0.00708,
    0.00771,
    0.00614,
    -0.00045,
    0.00211,
    0.00504,
    -0.00774,
    -0.01182,
    -0.0031,
    0.01092,
    -0.00117,
    -0.01635,
    0.00218,
    -0.0125,
    -0.00824,
    -0.00146,
    -0.0107,
    -0.00583,
    -0.01545,
    0.00058,
    0.00079,
    0.02259,
    0.00383,
    0.01833,
    -0.00307,
    -0.00379,
    -0.00162,
    0.02439,
    -0.00588,
    0.00699,
    0.00754,
    0.00177,
    0.02753,
    0.00198,
    -0.01146,
    -0.02041,
    -0.00342,
    0.00644,
    0.02319,
    0.01086,
    -0.00066,
    -0.02548,
    0.0129,
    -0.01106,
    0.01786,
    -0.00716,
    0.09088,
    0.01167,
    -0.00041,
    -0.01637,
    0.00516,
    -0.00762,
    0.00898,
    0.00504,
    -0.00028,
    -0.00332,
    0.0069,
    -0.01027,
    0.01622,
    -0.01005,
    -0.00219,
    -0.00522,
    0.01162,
    0.0029,
    -0.01403,
    0.00393,
    0.00523,
    -0.00479,
    -0.00075,
    -0.00244,
    -0.01076,
    -0.00335,
    0.00907,
    -0.01025,
    -0.00138,
    0.00977,
    -0.00519,
    -0.0052,
    -0.00217,
    -0.00103,
    -0.01982,
    0.01081,
    -0.00793,
    0.07755,
    0.00096,
    -0.01965,
    0.00328,
    -0.00719,
    0.00235,
    0.00375,
    -0.00451,
    -0.00084,
    0.00059,
    -0.00423,
    -0.00325,
    -0.00756,
    -0.00192,
    0.01231,
    -0.03242,
    -0.00259,
    0.00652,
    0.00494,
    -0.00099,
    -0.00629,
    -0.00581,
    0.00353,
    -5e-05,
    0.00436,
    -1e-05,
    -0.0005,
    -0.00279,
    0.00761,
    0.01471,
    0.00728,
    -0.42033,
    0.01349,
    -0.00328,
    -0.01652,
    -0.00991,
    -0.01286,
    -0.0112,
    0.00769,
    0.00332,
    0.00403,
    -0.00287,
    0.0379,
    0.00077,
    0.00064,
    0.01395,
    0.0122,
    0.00137,
    0.01181,
    -0.00759,
    -0.00596,
    -0.00075,
    -0.00585,
    0.01028,
    -0.0025,
    0.0036,
    -0.00517,
    0.00133,
    -0.03541,
    0.01412,
    0.08402,
    0.00297,
    -0.00409,
    0.00136,
    -0.00242,
    -0.0002,
    -0.0149,
    -0.00032,
    0.01879,
    -0.01711,
    -0.0101,
    0.01455,
    -0.00255,
    -0.00099,
    0.00269,
    0.00385,
    -0.00743,
    0.00212,
    0.00165,
    0.00045,
    0.03909,
    0.00233,
    -0.00157,
    -0.0009,
    0.00602,
    -0.00155,
    -0.01022,
    -0.00872,
    0.01021,
    0.00196,
    0.00365,
    0.00228,
    -0.00758,
    0.01436,
    -0.0057,
    -0.00303,
    0.01419,
    -0.00286,
    0.0003,
    0.00254,
    -0.00292,
    -0.0024,
    -0.00191,
    -0.0088,
    0.01131,
    -0.00148,
    -0.0144,
    0.00124,
    0.00507,
    -0.01786,
    -0.00903,
    -0.01002,
    0.00404,
    0.00231,
    -0.01155,
    -0.01376,
    -0.00139,
    0.00851,
    0.00056,
    -0.00651,
    -0.00165,
    -0.01537,
    -0.00379,
    0.00203,
    -0.01507,
    0.00571,
    0.00632,
    -0.01312,
    -0.00203,
    0.00443,
    0.01273,
    0.00941,
    0.00104,
    0.01074,
    0.0143,
    0.01104,
    0.00455,
    0.00237,
    0.00245,
    -0.00582,
    -0.00321,
    -0.01209,
    -0.00266,
    0.0031,
    0.00039,
    -0.00768,
    -0.00439,
    0.004,
    -0.00596,
    -0.00783,
    -0.01005,
    -0.02431,
    0.00673,
    -0.014,
    -0.01332,
    0.00222,
    0.00257,
    0.00234,
    0.00638,
    0.08842,
    -0.00029,
    -0.00353,
    0.00298,
    -0.00089,
    0.00981,
    0.00995,
    -0.00093,
    0.00989,
    -0.00058,
    0.00213,
    -0.00237,
    -0.01003,
    -0.01343,
    0.00239,
    0.0055,
    0.00143,
    0.00048,
    -0.00913,
    -0.01615,
    -0.00264,
    0.00522,
    -0.00791,
    0.0012,
    0.00795,
    -0.01026,
    -0.00353,
    -0.00607,
    0.00295,
    -0.00342,
    0.00316,
    0.02282,
    -0.00311,
    -0.00326,
    -0.00187,
    -0.00204,
    0.00071,
    -0.03065,
    0.01006,
    0.00742,
    0.01074,
    -0.00068,
    -0.00155,
    -0.02189,
    0.00074,
    0.00475,
    -0.00944,
    -0.00465,
    0.00822,
    -0.01209,
    0.01915,
    -0.00236,
    -0.01565,
    -0.00111,
    -0.00417,
    -0.03771,
    -0.0091,
    -0.00729,
    0.00099,
    0.00593,
    0.00563,
    -0.01148,
    0.00563,
    -0.00037,
    0.00287,
    0.00379,
    0.02087,
    0.01161,
    0.00424,
    0.00865,
    0.00954,
    -0.00439,
    0.00653,
    0.00996,
    -0.00301,
    0.01423,
    0.00192,
    0.00312,
    0.0196,
    0.00526,
    0.01015,
    0.00348,
    -0.00809,
    -0.00735,
    0.01789,
    -0.00197,
    -0.00488,
    -0.00311,
    -0.00247,
    -0.01049,
    0.00109,
    0.00141,
    0.01908,
    -0.01942,
    0.00616,
    -0.00708,
    -0.01011,
    -0.0126,
    -0.00051,
    -0.01292,
    0.00889,
    0.02011,
    -0.00642,
    -0.00092,
    -0.00308,
    -0.09427,
    0.01052,
    0.00076,
    0.00201,
    0.0009,
    -0.00561,
    -0.0092,
    -0.00479,
    0.00229,
    -0.01125,
    0.0213,
    -0.00151,
    -0.00034,
    0.00339,
    -0.00093,
    -0.01324,
    -0.01077,
    -0.00346,
    -6e-05,
    0.00153,
    0.0097,
    0.00104,
    -0.0071,
    -0.02918,
    -0.01686,
    0.02877,
    -0.00176,
    -2e-05,
    -0.00226,
    -0.00263,
    -0.00115,
    0.00489,
    -0.00914,
    0.00089,
    -0.001,
    -0.00773,
    -0.01751,
    0.00288,
    0.00433,
    -0.00171,
    -0.00559,
    -0.05873,
    -0.03384,
    -0.0067,
    -0.01112,
    -0.00343,
    -0.02979,
    -0.00795,
    -0.00835,
    -0.02647,
    0.00475,
    -0.0113,
    0.01612,
    0.00615,
    0.00215,
    -0.00223,
    0.032,
    0.00451,
    -0.00194,
    0.0019,
    -0.00014,
    0.00226,
    -0.00361,
    -0.00103,
    0.01373,
    -0.00455,
    -0.00422,
    0.001,
    0.01412,
    0.00314,
    -0.00268,
    0.00256,
    -0.00614,
    0.01002,
    -0.00779,
    0.02651,
    -0.00557,
    0.00758,
    -0.00425,
    0.00537,
    -0.00303,
    0.00494,
    0.03121,
    0.00432,
    -0.011,
    0.00012,
    -0.00785,
    0.00391,
    -0.0058,
    -0.00625,
    -0.00088,
    0.00514,
    -0.00378,
    0.00816,
    0.00133,
    -0.00053,
    -0.01006,
    0.0103,
    0.00184,
    0.01174,
    -0.0243,
    -0.00279,
    -0.00915,
    -0.00282,
    -0.01797,
    0.01145,
    0.00911,
    0.00211,
    0.00329,
    0.01029,
    -0.00647,
    -0.00091,
    -0.01804,
    0.01448,
    -0.01368,
    -0.01215,
    -0.0051,
    0.00321,
    0.00305,
    0.00073,
    -0.00263,
    -0.01219,
    0.00881,
    0.00105,
    0.0131,
    -0.00479,
    0.00352,
    0.00686,
    0.00221,
    0.00309,
    0.00543,
    0.00487,
    0.00817,
    0.00111,
    -0.00798,
    0.01839,
    -0.00333,
    -0.01755,
    0.00286,
    -0.00448,
    0.00258,
    0.00546,
    -0.00821,
    -0.00193,
    0.00346,
    0.00921,
    -0.00257,
    -0.0067,
    0.00445,
    0.00697,
    0.00228,
    -0.00611,
    -0.00837,
    0.01172,
    0.0006,
    0.02191,
    0.00177,
    -0.0113,
    -0.00403,
    0.00294,
    -0.00337,
    -0.00088,
    0.00453,
    -0.00032,
    0.00012,
    0.00213,
    0.00108,
    -0.02372,
    -0.00686,
    -0.00244,
    -0.01624,
    0.00584,
    0.00252,
    -0.00734,
    -0.00027,
    0.02028,
    -0.0082,
    0.02002,
    -0.01324,
    -0.00363,
    -0.00406,
    0.00874,
    -0.01323,
    0.0013,
    0.01166,
    0.01323,
    0.01224,
    -0.02891,
    -0.01579,
    0.00233,
    -0.00236,
    0.00104,
    -0.00501,
    -0.00619,
    -0.00539,
    -0.01356,
    -0.00554,
    -0.00067,
    0.00113,
    0.01127,
    -0.00437,
    0.01348,
    0.01441,
    -0.00138,
    0.01926,
    0.00344,
    -0.00936,
    -0.00455,
    -0.02639,
    0.00779,
    -0.00242,
    0.00823,
    -0.01826,
    -0.00132,
    0.01715,
    -0.00098,
    -0.00179,
    -0.00448,
    0.00237,
    0.00818,
    0.00482,
    0.00642,
    0.04658,
    -0.00263,
    0.01755,
    0.00759,
    0.02466,
    -0.00471,
    -0.00494,
    0.00696,
    0.00896,
    0.00306,
    0.01163,
    -0.00038,
    -0.00557,
    1e-05,
    0.00082,
    -0.00625,
    0.00222,
    0.03014,
    0.00274,
    0.00524,
    -0.01555,
    0.00653,
    -0.00725,
    0.0059,
    -0.00218,
    0.00583,
    0.03424,
    -0.00622,
    -0.0112,
    -0.01625,
    -0.00619,
    0.00143,
    -0.00053,
    0.00525,
    -0.00124,
    0.00437,
    0.00253,
    0.00667,
    0.01504,
    -0.03663,
    -0.03056,
    -0.00117,
    0.00837,
    0.00344,
    0.00392,
    -0.01475,
    -0.00153,
    -0.00715,
    0.00346,
    0.0135,
    -9e-05,
    0.00088,
    0.00167,
    -0.00578,
    -0.00045,
    0.00551,
    0.00781,
    -0.03571,
    -0.01495,
    -0.00365,
    -0.00893,
    -0.00277,
    0.0131,
    0.00302,
    -0.00329,
    -0.00265,
    0.01585,
    0.01795,
    -0.00855,
    -0.00265,
    0.00504,
    0.00122,
    0.00038,
    0.00467,
    -0.00331,
    -0.01316,
    -0.01155,
    0.00264,
    -0.00171,
    -0.00536,
    -0.00733,
    -0.00322,
    0.00341,
    0.00226,
    0.00288,
    0.01256,
    -0.00643,
    -0.00173,
    0.00594,
    -0.0002,
    0.0,
    -0.02777,
    -0.00027,
    0.00471,
    -0.00229,
    0.00029,
    -0.00386,
    -0.03034,
    -0.02101,
    -0.01272,
    0.00739,
    -0.0014,
    0.00727,
    -0.00918,
    0.0109,
    -0.00977,
    -0.00015,
    0.0001,
    -0.0016,
    0.00622,
    -0.00153,
    -0.01075,
    -0.00174,
    0.00349,
    -0.01665,
    -0.00654,
    0.0021,
    0.00815,
    0.01033,
    -0.00424,
    -0.01306,
    0.00567,
    -0.00495,
    0.00639,
    0.0025,
    0.01466,
    0.00026,
    -0.0054,
    -0.00492,
    -0.01909,
    0.00862,
    -0.00762,
    0.00837,
    0.00945,
    0.00401,
    0.01754,
    -0.0033,
    0.0037,
    -0.01476,
    0.00508,
    0.00058
   ],
   "SUDDEN": [
    0.00924,
    -0.00176,
    0.00419,
    0.00427,
    0.00292,
    -0.00696,
    -0.00443,
    0.00498,
    -0.00225,
    -0.0016,
    -0.0003,
    0.00271,
    0.00392,
    -0.00595,
    -0.00058,
    0.00369,
    7e-05,
    -0.002,
    -0.0024,
    -0.00582,
    -0.00096,
    2e-05,
    -0.00648,
    -0.00478,
    0.00061,
    0.00115,
    0.00294,
    -0.00495,
    0.00338,
    0.00218,
    0.00309,
    0.00138,
    -0.00056,
    -0.01139,
    0.00054,
    0.00393,
    -0.00026,
    -0.0024,
    -0.00407,
    -0.0041,
    -0.00045,
    0.00122,
    -0.00266,
    0.00104,
    -0.00655,
    0.00143,
    0.00107,
    -0.0001,
    0.00495,
    8e-05,
    0.00927,
    -0.00499,
    -0.00522,
    -0.00305,
    0.00212,
    -0.00081,
    -0.00423,
    -0.00446,
    0.00562,
    -0.0075,
    0.00032,
    0.00358,
    -0.00014,
    -0.00169,
    -0.00179,
    -0.00202,
    -0.00335,
    0.00125,
    0.00017,
    -0.00397,
    0.00135,
    0.00718,
    0.00632,
    0.00676,
    -7e-05,
    0.00238,
    -0.00078,
    -0.00357,
    -0.00245,
    0.00386,
    0.00148,
    -0.01006,
    -0.01773,
    0.00589,
    0.00086,
    -0.00136,
    -0.00368,
    0.00119,
    0.00313,
    -0.00182,
    -1e-05,
    -0.00476,
    0.00175,
    -0.00305,
    0.00486,
    0.00352,
    0.00826,
    -0.00812,
    -0.00234,
    -0.00074,
    -0.00092,
    -0.00938,
    0.03283,
    -0.0019,
    0.00096,
    0.00128,
    -0.00173,
    -0.00186,
    -0.00285,
    -0.00349,
    0.00673,
    2e-05,
    -0.00084,
    0.00216,
    -0.00294,
    -0.0039,
    0.00147,
    0.00113,
    -0.00072,
    -0.00066,
    0.00252,
    -0.0064,
    -0.0024,
    -0.00418,
    -0.00281,
    0.00249,
    0.00052,
    -0.00097,
    -0.00112,
    0.00182,
    -0.00073,
    -0.00027,
    0.00035,
    0.00135,
    -0.0028,
    -0.0001,
    -0.00232,
    0.00188,
    0.00594,
    -0.00337,
    0.01464,
    -0.00036,
    -0.00788,
    -0.00613,
    0.00391,
    0.00681,
    0.00102,
    0.10827,
    5e-05,
    0.00094,
    0.00195,
    0.00287,
    -0.00876,
    0.00187,
    -0.00254,
    0.00436,
    0.00106,
    0.00199,
    0.00275,
    0.00409,
    -0.00387,
    0.00145,
    0.00017,
    -0.0096,
    0.00155,
    0.00416,
    -0.00114,
    -0.00375,
    -0.00162,
    0.00189,
    0.00084,
    -0.00308,
    0.00154,
    0.00101,
    -0.00422,
    -0.00282,
    0.00119,
    -0.00214,
    0.00201,
    -0.00062,
    0.00024,
    -0.00253,
    0.0012,
    0.00353,
    0.00218,
    0.0011,
    -0.00398,
    -0.00109,
    0.00118,
    -0.00142,
    0.00099,
    -0.00206,
    -0.00107,
    -0.0038,
    -0.00393,
    -0.00048,
    0.00599,
    -0.00094,
    0.00132,
    -0.00041,
    -4e-05,
    0.00279,
    0.00125,
    0.00364,
    0.00334,
    0.00359,
    0.00106,
    -0.00051,
    0.00223,
    -0.00203,
    0.0008,
    0.0018,
    -0.00408,
    0.00356,
    0.00038,
    0.00353,
    0.00122,
    -0.00125,
    0.00368,
    0.00601,
    -0.00376,
    -0.0013,
    -0.00847,
    0.00398,
    -0.00255,
    0.00053,
    -0.01128,
    -0.00614,
    0.013,
    -0.00208,
    0.00393,
    -0.00192,
    -0.00127,
    -0.00182,
    -0.00193,
    -0.00761,
    0.002,
    0.00275,
    0.00222,
    -0.00207,
    -0.00574,
    -0.00215,
    0.00268,
    -0.00061,
    0.00017,
    0.0007,
    0.00568,
    0.00107,
    0.00323,
    -0.00061,
    -0.00224,
    -0.0043,
    0.00078,
    -0.00355,
    0.00592,
    -0.00471,
    -0.00144,
    -0.00014,
    -1e-05,
    0.00366,
    -0.00134,
    -0.00494,
    0.00324,
    0.00853,
    -0.00173,
    0.00071,
    0.01151,
    0.00287,
    0.00631,
    9e-05,
    0.00094,
    -0.00042,
    0.00246,
    -0.00384,
    0.0079,
    0.00846,
    0.00194,
    -0.0016,
    -0.00796,
    0.00337,
    -0.00317,
    -0.00118,
    -0.00421,
    0.0035,
    -0.00134,
    -0.00939,
    0.10283,
    0.00226,
    0.00244,
    0.00336,
    0.00014,
    0.00023,
    -0.00141,
    0.0009,
    0.00134,
    -0.00893,
    0.00122,
    8e-05,
    -0.00397,
    -0.00183,
    0.00523,
    0.02585,
    -0.00074,
    -0.00261,
    0.00016,
    0.00128,
    -7e-05,
    -0.0088,
    -0.00039,
    0.00024,
    0.01156,
    -0.00027,
    -0.00697,
    -0.00072,
    -0.0004,
    0.00126,
    -0.00749,
    -0.00542,
    -0.00411,
    0.00442,
    -0.00191,
    0.00336,
    0.00276,
    0.01091,
    6e-05,
    0.00295,
    -0.00514,
    -0.01445,
    -0.00204,
    -0.0044,
    0.00064,
    0.00167,
    -0.00037,
    -0.00126,
    0.00133,
    0.00038,
    -0.00531,
    -0.01029,
    0.0038,
    -0.00836,
    0.00063,
    0.00451,
    -0.00564,
    -0.00164,
    0.00416,
    -0.00018,
    -0.00188,
    0.00296,
    -0.00094,
    0.00101,
    0.00329,
    0.0049,
    -0.01316,
    -2e-05,
    -0.0333,
    0.00021,
    0.00699,
    -0.00269,
    0.00396,
    0.0,
    -0.0014,
    -0.00257,
    0.00154,
    -0.00362,
    -0.00384,
    -0.00126,
    -0.00244,
    -0.00219,
    0.01699,
    0.00159,
    0.00217,
    0.00062,
    -2e-05,
    0.00552,
    -0.00489,
    -0.00227,
    0.00266,
    0.00344,
    0.00065,
    0.00437,
    0.00083,
    -0.00073,
    0.00252,
    -0.00043,
    0.01929,
    -0.0079,
    -0.00021,
    0.00092,
    0.00345,
    -0.00162,
    -0.00161,
    -0.00569,
    -0.00216,
    0.00156,
    -0.01472,
    0.01244,
    0.00776,
    -0.03422,
    0.00591,
    -0.00017,
    0.00042,
    0.0022,
    -0.00149,
    -0.00397,
    0.00379,
    -0.01766,
    -0.00037,
    -0.00262,
    0.59749,
    0.00438,
    -0.00765,
    0.00919,
    0.00375,
    -0.0032,
    -0.00679,
    0.00617,
    0.00268,
    0.00054,
    0.02186,
    0.00128,
    -0.00073,
    -0.00859,
    0.00011,
    -0.00198,
    0.01846,
    0.00115,
    0.00104,
    0.00025,
    -0.00013,
    -0.00368,
    0.00482,
    -0.00038,
    -9e-05,
    0.00041,
    0.00293,
    0.00334,
    0.00607,
    -0.00817,
    -0.00098,
    -0.00125,
    0.00064,
    -0.00965,
    -0.0134,
    0.00075,
    0.00094,
    0.00097,
    -0.01277,
    -0.0025,
    -0.00121,
    -0.00471,
    0.00095,
    0.00401,
    0.0007,
    0.00277,
    0.00294,
    -0.00184,
    -0.0012,
    -0.0004,
    -0.0099,
    -0.00249,
    -0.00687,
    0.00128,
    0.00194,
    -0.00236,
    -0.00042,
    -0.00587,
    -0.00155,
    0.00225,
    0.00738,
    0.00059,
    -0.00408,
    0.0022,
    0.00487,
    0.00523,
    -0.00017,
    -0.0017,
    -0.00241,
    0.00733,
    0.00765,
    0.00142,
    -0.00328,
    -0.00027,
    -0.0078,
    -0.00338,
    -0.00327,
    -0.00428,
    -0.00037,
    -0.00281,
    -0.00472,
    -0.00194,
    -0.00074,
    0.00231,
    0.00197,
    0.00219,
    -0.00369,
    0.00209,
    0.00108,
    0.00122,
    -0.00153,
    -0.00254,
    -0.00444,
    0.00195,
    0.00313,
    -0.02899,
    0.00131,
    0.00345,
    0.00186,
    -0.00698,
    0.00884,
    -0.00127,
    -0.01372,
    -0.00508,
    0.00246,
    -0.00337,
    0.01451,
    0.00451,
    0.00191,
    0.00105,
    0.01003,
    0.01136,
    -0.45641,
    0.0065,
    0.00436,
    -0.00181,
    0.00222,
    0.00119,
    -0.00382,
    -0.00439,
    -0.005,
    -0.00172,
    -0.00364,
    0.00091,
    0.00187,
    -0.00883,
    0.00063,
    -0.00265,
    0.0012,
    0.00243,
    -0.00121,
    -0.00358,
    -0.00089,
    0.00048,
    0.0004,
    0.00263,
    0.01247,
    0.00574,
    -0.00094,
    0.00063,
    0.00753,
    -0.00123,
    0.00309,
    -0.00033,
    0.00146,
    0.11653,
    -0.0147,
    -0.00244,
    -0.00219,
    -0.00534,
    0.00047,
    4e-05,
    0.00093,
    0.00029,
    0.00198,
    -0.00141,
    -0.00099,
    5e-05,
    0.00759,
    -0.00182,
    -0.00464,
    0.00342,
    -0.01012,
    -0.00372,
    0.00389,
    -0.01364,
    -0.01199,
    -0.00035,
    -0.00018,
    0.0038,
    0.00279,
    0.00013,
    -0.00243,
    0.00332,
    0.01303,
    -0.00019,
    0.00096,
    0.00046,
    -0.00105,
    -0.0018,
    -8e-05,
    0.01584,
    -6e-05,
    -0.00051,
    -0.00098,
    0.00472,
    0.00168,
    -0.00244,
    0.0014,
    -0.00325,
    -0.00061,
    -0.00079,
    -0.00166,
    0.00122,
    -0.00199,
    -0.00103,
    0.00216,
    -0.0012,
    0.00642,
    -0.0025,
    -0.00019,
    -0.36433,
    0.00026,
    -0.00025,
    0.00025,
    0.00691,
    -0.00126,
    -0.00349,
    -0.00883,
    -0.0035,
    -0.00053,
    -0.00249,
    0.00626,
    -0.01302,
    0.00192,
    0.00553,
    0.0037,
    -0.00459,
    -0.00064,
    0.00336,
    0.00204,
    -0.00683,
    0.0032,
    0.00512,
    -0.00621,
    -0.00095,
    0.00016,
    0.00238,
    -0.00232,
    -0.00276,
    -0.00476,
    -0.00133,
    0.00074,
    0.00489,
    0.00063,
    -0.01299,
    0.00081,
    -0.00239,
    0.00121,
    0.00263,
    0.00377,
    -0.00486,
    -0.00936,
    4e-05,
    -0.00371,
    0.0022,
    0.00233,
    -0.00071,
    -0.00092,
    -0.00662,
    -0.00531,
    -0.00049,
    -0.00285,
    -0.00188,
    -0.00648,
    0.00145,
    0.0072,
    -0.0025,
    0.00046,
    -0.00143,
    -0.00146,
    0.00846,
    -0.00109,
    -0.00531,
    0.00221,
    0.00359,
    -0.00459,
    -0.00081,
    -0.02342,
    0.00217,
    0.00413,
    -0.00191,
    0.00205,
    0.00357,
    -0.0056,
    0.0006,
    0.00255,
    -0.00016,
    0.00388,
    -0.00284,
    -0.00512,
    -0.00064,
    -0.00221,
    -0.00259,
    0.00609,
    -0.00073,
    -0.00224,
    -0.00081,
    -0.004,
    -0.00231,
    0.00049,
    -0.00036,
    0.00265,
    0.00122,
    0.00089,
    -0.00027,
    0.00039,
    0.00149,
    0.00103,
    0.00232,
    -0.00133,
    -0.00563,
    -0.0013,
    0.00419,
    -0.0012,
    0.09696,
    0.00554,
    -0.00054,
    0.00094,
    -0.00203,
    0.0029,
    0.00203,
    0.00189,
    -0.01072,
    0.00085,
    -0.01132,
    -0.0033,
    0.00292,
    -0.00995,
    0.00307,
    0.00172,
    0.00032,
    0.0029,
    0.00256,
    0.00236,
    0.003,
    -0.00282,
    -0.01244,
    -0.00213,
    0.0015,
    -0.00293,
    -0.00127,
    -0.00566,
    -0.01083,
    -0.01003,
    -0.00445,
    -0.00151,
    -0.00299,
    -0.00391,
    -0.0025,
    -0.0001,
    -8e-05,
    0.02211,
    0.00158,
    -0.00758,
    0.00647,
    0.00311,
    -0.00296,
    -0.00074,
    0.00265,
    -0.01498,
    -0.00117,
    0.00045,
    -0.00215,
    -0.00362,
    0.0115,
    -0.00202,
    0.00327,
    -0.00102,
    0.00034,
    -0.00307,
    -0.00176,
    -0.00512,
    0.0016,
    -0.00431,
    -0.00189,
    0.00121,
    -0.00131,
    0.00476,
    -0.00255,
    -0.00256,
    -0.00596,
    -0.00426,
    0.00048,
    -0.00197,
    -0.00326,
    -0.00231,
    -0.00381,
    -0.00167,
    -0.0038,
    -0.00188,
    0.00132,
    -0.00124,
    0.00198,
    0.00053,
    0.00091,
    0.00139,
    0.00205,
    0.00211,
    0.005,
    1e-05,
    -0.00497,
    0.0135,
    -0.00215,
    -0.02016,
    0.00057,
    0.00107,
    0.00049,
    -0.00595,
    0.00198,
    -0.0039,
    -0.00428,
    -0.00178,
    0.00286,
    -0.00076,
    -0.00538,
    -0.00156,
    -0.00113,
    0.0012,
    0.00317,
    0.00035,
    0.00412,
    -0.00099,
    0.00046,
    0.00033,
    -0.00093,
    0.00044,
    0.01021,
    0.01208,
    0.00937,
    -0.00146,
    -0.01434,
    0.0036,
    -0.00715,
    0.00296,
    -0.00459,
    0.00065,
    0.00187,
    0.00311,
    -9e-05,
    0.00032,
    0.00289,
    -0.00364,
    -0.00423,
    -0.00081,
    0.00013,
    -0.01371,
    0.00642,
    0.0015,
    -0.00861,
    0.00225,
    -0.00675,
    0.00171,
    -0.00226,
    -0.0008,
    0.01081,
    0.00462,
    0.00015,
    0.00391,
    0.00407,
    0.0044,
    -0.00157,
    0.00134,
    -0.00267,
    0.00317,
    0.00021,
    0.00111,
    0.01333,
    -0.00062,
    0.00415,
    -0.00286,
    -0.00072,
    0.00424,
    0.00346,
    0.09163,
    -0.00047,
    0.00151,
    0.00116,
    0.00014,
    -0.00313,
    -0.00026,
    -0.00078,
    -0.00426,
    0.00014,
    -0.00698,
    0.00307,
    -0.00112,
    0.00268,
    -0.00249,
    0.00105,
    -0.00267,
    0.00298,
    -0.00566,
    -0.00476,
    -0.00213,
    -0.01547,
    -0.00305,
    -0.00032,
    -0.00112,
    0.0016,
    0.00625,
    0.00611,
    0.00569,
    0.00107,
    -3e-05,
    -0.00281,
    0.00699,
    -0.00055,
    0.00024,
    -0.00268,
    0.00563,
    0.00246,
    0.00064,
    -0.0038,
    0.00016,
    0.00396,
    -0.00452,
    0.00091,
    -0.00315,
    0.00147,
    -0.00327,
    -0.00033,
    0.00178,
    0.00085,
    -0.00024,
    0.00023,
    -0.01163,
    -0.00467,
    -0.00264,
    -0.00696,
    0.00201,
    0.00401,
    -0.0038,
    0.0017,
    -0.00023,
    0.00272,
    -0.00045,
    0.00166,
    -0.00164,
    -0.0014,
    -0.01115,
    0.00097,
    -0.42626,
    -0.00157,
    0.00122,
    -0.00107,
    -0.00243,
    -0.00095,
    -0.0001,
    0.00576,
    0.00302,
    0.00029,
    0.00115,
    -0.00814,
    0.00163,
    -0.00116,
    0.00471,
    0.00315,
    -0.00202,
    0.00466,
    -0.00244,
    -0.00241,
    0.0013,
    -0.00407,
    -0.00327,
    0.00604,
    -0.00522,
    0.00266,
    0.00121,
    -0.00015,
    -0.00091,
    0.09097,
    0.00184,
    -0.00096,
    0.00221,
    -0.00363,
    -4e-05,
    0.00939,
    0.00011,
    -0.00333,
    0.01161,
    -0.00078,
    -0.008,
    0.00067,
    -0.00434,
    -0.00464,
    -0.00151,
    -0.00403,
    0.00043,
    -0.00019,
    0.00092,
    -0.00624,
    0.00038,
    -0.01319,
    -0.00148,
    -0.00367,
    -0.00216,
    -0.00245,
    0.00383,
    -0.00083,
    0.0001,
    0.01044,
    0.00215,
    -0.00033,
    0.00821,
    -0.00519,
    0.00368,
    0.00061,
    -0.0002,
    0.00048,
    0.00166,
    0.00453,
    0.00025,
    -0.00288,
    -0.00262,
    0.00144,
    0.0,
    0.00251,
    -0.00135,
    0.00047,
    -0.00678,
    0.00564,
    -0.00435,
    -0.00035,
    -0.00353,
    0.00044,
    -0.00034,
    0.00262,
    -0.00032,
    -0.00289,
    0.0029,
    -0.00183,
    0.00905,
    -0.0018,
    -0.00638,
    -0.00306,
    0.00079,
    0.00051,
    -8e-05,
    -0.00425,
    -0.00271,
    0.00251,
    -0.00126,
    -0.00285,
    0.00114,
    -0.00221,
    0.00245,
    -0.00075,
    -0.00079,
    -0.00099,
    -0.00113,
    -0.00116,
    -0.00798,
    -0.00142,
    0.00031,
    -0.00183,
    0.00077,
    0.00342,
    -0.0026,
    -0.00043,
    -0.00113,
    0.00212,
    0.00106,
    0.00353,
    -0.00326,
    -0.00644,
    0.00191,
    -0.00307,
    -0.0041,
    -0.00089,
    0.01064,
    -0.00225,
    -0.00654,
    -0.00151,
    3e-05,
    -0.00095,
    -0.00448,
    -0.00077,
    0.00138,
    -0.00352,
    -0.0011,
    -0.00298,
    0.0011,
    0.00669,
    -0.0019,
    0.00289,
    -0.00142,
    -0.00205,
    0.00391,
    0.00222,
    -0.00068,
    -0.0051,
    -0.00479,
    -0.00196,
    -0.00383,
    -0.00062,
    -0.00136,
    0.00153,
    -0.00199,
    0.00198,
    -0.00705,
    0.00205,
    -0.00019,
    0.00782,
    -0.0025,
    -0.00225,
    -0.0001,
    -0.01008,
    0.0009,
    -0.00229,
    -0.00282,
    -0.00516,
    0.00106,
    -0.00029,
    0.0007,
    0.0023,
    0.00222,
    0.00123,
    0.00114,
    0.00376,
    -0.00536,
    -0.00113,
    -0.00576,
    -0.00148,
    -7e-05,
    -0.00788,
    0.00104,
    -0.00049,
    -0.00119,
    0.00495,
    0.00524,
    -0.00437,
    -0.00126,
    -0.00488,
    0.0094,
    0.00291,
    -0.00341,
    -0.0015,
    0.00183,
    0.00015,
    -0.00272,
    -0.00167,
    -0.00245,
    0.00256,
    -0.00039,
    -0.00214,
    0.00422,
    0.00088,
    0.005,
    1e-05,
    0.0048,
    -0.0065,
    0.00046,
    -0.00043,
    0.0074,
    0.00126,
    -0.00059,
    -0.01092,
    0.00099,
    0.00926,
    0.00195,
    -0.00127,
    0.00255,
    -0.01725,
    0.0009,
    -1e-05,
    0.00168,
    -0.00033,
    -0.02056,
    0.00243,
    -0.00316,
    0.01179,
    0.004,
    0.00378,
    -0.00031,
    -0.08166,
    0.00418,
    -0.00184,
    -0.00231,
    -0.00845,
    0.00168,
    -0.00282,
    0.00269,
    0.00058,
    -0.00091,
    0.00469,
    4e-05,
    0.00217,
    -0.002,
    0.00208,
    0.00225,
    0.00172,
    0.0001,
    0.00148,
    -0.00422,
    0.00033,
    0.0045,
    -0.0013,
    -0.01031,
    0.00186,
    0.01474,
    0.00433,
    -0.01735,
    -0.00129,
    -0.00057,
    -0.00202,
    0.00076,
    0.00272,
    -0.00014,
    -0.00117,
    0.00079,
    -0.00334,
    -0.00029,
    0.00252,
    -0.00069,
    -0.0065,
    -0.00762,
    -0.01867,
    0.00079,
    -0.00318,
    0.00086,
    0.00418,
    0.00114,
    -0.00178,
    -0.0052,
    -0.00141,
    -0.00322,
    -0.00338,
    0.00159,
    0.00044,
    -0.0048,
    0.0549,
    0.00165,
    0.00366,
    0.00667,
    -0.00256,
    0.00098,
    0.00292,
    -0.00351,
    0.00208,
    -0.00083,
    -0.00149,
    0.00011,
    -0.00349,
    -0.00454,
    -0.00876,
    0.00184,
    0.00719,
    0.0026,
    0.00081,
    -0.0413,
    -0.00503,
    -0.00112,
    -0.00819,
    0.00115,
    -0.00063,
    -0.00106,
    0.00369,
    0.0031,
    -0.00361,
    -0.00401,
    0.01012,
    0.00172,
    0.00044,
    -0.00093,
    0.00638,
    -0.00095,
    0.00169,
    0.00239,
    0.00379,
    -0.00148,
    -0.0087,
    0.00202,
    0.00131,
    -0.03783,
    0.00014,
    0.00438,
    0.00067,
    -0.00021,
    -0.01601,
    0.01191,
    0.00114,
    -0.00527,
    0.00511,
    0.00086,
    0.00797,
    0.00176,
    -0.00289,
    -0.01425,
    0.00478,
    -0.00749,
    0.00471,
    0.00445,
    0.00401,
    0.00376,
    0.00247,
    0.00272,
    -0.00726,
    -0.0019,
    -0.00501,
    0.00209,
    -0.00845,
    -0.00093,
    -0.00871,
    0.0029,
    -0.0083,
    0.00417,
    -0.00131,
    -0.00053,
    -0.00262,
    -0.00031,
    0.00213,
    -0.00193,
    0.00122,
    -0.00151,
    -0.00084,
    -0.00183,
    0.00093,
    0.00109,
    0.00021,
    -0.00225,
    0.00035,
    -0.00208,
    -0.00214,
    0.0001,
    -0.0011,
    0.00079,
    -0.00123,
    0.01487,
    -0.00114,
    -0.0033,
    -0.00087,
    0.00522,
    0.00772,
    0.0038,
    -0.00398,
    -0.00161,
    0.0001,
    0.0099,
    0.00313,
    0.00925,
    -0.00335,
    -0.00916,
    -0.00273,
    -0.0008,
    -0.00627,
    0.00129,
    0.00018,
    -0.00115,
    -0.00023,
    0.00164,
    -0.00057,
    0.00375,
    0.00292,
    0.01478,
    0.00387,
    0.00052,
    -0.00208,
    0.00196,
    0.00606,
    -0.01401,
    0.0028,
    -0.01346,
    0.00069,
    0.00416,
    0.00423,
    -0.00576,
    0.00721,
    -0.00023,
    -0.00122,
    -0.00471,
    -0.00185,
    0.0036,
    0.00088,
    0.00015,
    -0.00091,
    -0.00446,
    -0.00166,
    0.0007,
    -0.00489,
    -0.00013,
    0.00175,
    -0.0005,
    0.00272,
    -0.00116,
    -0.00011,
    -0.00592,
    0.00294,
    0.00127,
    0.00118,
    -0.00692,
    1e-05,
    -0.00905,
    0.00575,
    -5e-05,
    -0.00144,
    0.0013,
    0.01439,
    0.00027,
    -0.00541,
    -0.00133,
    0.00162,
    -0.00699,
    -0.0004,
    -0.00073,
    -0.00194,
    -0.00289,
    -0.00025,
    0.00149,
    -0.00283,
    -0.00026,
    0.0006,
    -0.00143,
    -0.00299,
    0.01468,
    -0.00048,
    -0.00091,
    -0.00448,
    0.00079,
    -0.00217,
    0.00177,
    0.00401,
    0.00015,
    0.07839,
    0.00364,
    -0.01679,
    -0.0223,
    -0.00406,
    -0.00057,
    -0.00591,
    -0.00173,
    0.00152,
    -0.00374,
    -0.00265,
    0.00037,
    -0.00263,
    0.00622,
    0.00447,
    0.00475,
    0.00173,
    9e-05,
    0.00084,
    0.01951,
    0.0052,
    0.0053,
    -0.00019,
    -0.00088,
    0.00267,
    -0.00526,
    -0.00122,
    0.00336,
    -0.00113,
    0.0018,
    0.00694,
    -0.00695,
    -0.00267,
    -0.00593,
    -0.00171,
    -0.0027,
    -0.00285,
    0.00362,
    -0.00346,
    0.00169,
    0.00167,
    -0.00409,
    0.00576,
    -0.00138,
    -0.00732,
    -0.0019,
    0.00042,
    -0.01048,
    0.00241,
    -0.0014,
    -0.0164,
    -0.00219,
    0.00137,
    -0.00558,
    0.01176,
    -0.00275,
    0.00744,
    -0.00521,
    0.00255,
    -0.00692,
    0.00467,
    0.00153,
    0.00063,
    -0.00022,
    -0.00049,
    -0.00697,
    0.00268,
    -0.00043,
    0.00179,
    0.00722,
    -0.00298,
    -0.00767,
    -0.00148,
    -0.00077,
    0.01028,
    -0.00092,
    0.0008,
    -0.0036,
    -0.00044,
    0.00415,
    0.00012,
    -8e-05,
    -0.00195,
    0.00049,
    -0.00371,
    0.00541,
    0.00249,
    -0.00026,
    -0.00397,
    0.00179,
    0.00238,
    0.00156,
    -0.00215,
    0.00872,
    -0.00362,
    -0.00148,
    0.00574,
    0.00081,
    -0.00025,
    0.00027,
    -0.00132,
    0.00465,
    -0.00226,
    0.00725,
    -0.00346,
    -0.00173,
    0.00392,
    0.0037,
    0.00202,
    -0.00026,
    -0.00485,
    -0.02009,
    -0.00119,
    0.00044,
    -0.0016
   ],
   "NUMBNESS": [
    0.01436,
    0.00712,
    0.00837,
    0.00058,
    0.00112,
    -0.00615,
    0.00689,
    0.01056,
    -0.01123,
    0.00944,
    0.00709,
    0.00088,
    -0.00764,
    0.00196,
    -0.01028,
    -0.00229,
    0.00063,
    0.00351,
    8e-05,
    -0.01182,
    0.00582,
    0.00065,
    -0.00401,
    0.0048,
    -0.00151,
    -0.00507,
    -0.01119,
    0.00519,
    0.01809,
    -0.0025,
    0.00269,
    0.00523,
    0.00568,
    0.02523,
    0.00606,
    0.00122,
    0.00553,
    0.00265,
    -0.00909,
    -0.0012,
    0.00108,
    -0.00151,
    -0.00995,
    0.00381,
    0.01432,
    0.00881,
    -3e-05,
    -0.00089,
    -0.00688,
    -0.00115,
    0.02424,
    -0.00978,
    -0.00378,
    -0.00366,
    -0.00593,
    -0.01882,
    -0.0001,
    -0.01458,
    0.00495,
    0.00613,
    0.00364,
    -0.00763,
    0.0035,
    0.00356,
    -0.00113,
    0.00589,
    -0.01732,
    0.00641,
    -0.0068,
    0.0107,
    0.00081,
    -0.00819,
    -0.02432,
    -0.00252,
    0.00241,
    0.01908,
    0.00032,
    -0.00369,
    -0.00044,
    -0.00207,
    0.00179,
    0.01596,
    -0.00631,
    0.00605,
    0.00233,
    0.00612,
    0.00824,
    0.00425,
    0.00316,
    0.00315,
    0.00466,
    -0.00051,
    -0.00277,
    -0.00078,
    -0.00122,
    0.01991,
    0.00165,
    -0.00054,
    0.00332,
    0.00247,
    -0.00146,
    0.00539,
    -0.01323,
    -0.00672,
    0.0037,
    0.00073,
    -0.01035,
    0.00024,
    0.0033,
    0.00252,
    -0.0123,
    0.01009,
    0.01025,
    0.00527,
    -0.00395,
    0.00135,
    -0.00467,
    0.00176,
    0.00819,
    0.00406,
    0.00247,
    -0.00306,
    -0.00994,
    0.00968,
    -0.00011,
    0.00537,
    0.00209,
    0.01262,
    0.00739,
    -0.00146,
    -0.00742,
    -0.00114,
    -0.00528,
    -0.00365,
    -0.00394,
    -0.00096,
    -0.006,
    -1e-05,
    -0.00203,
    0.00378,
    -0.0109,
    0.00538,
    0.00269,
    -0.01286,
    -0.00087,
    0.00847,
    -0.00126,
    -0.11089,
    -0.00111,
    0.00466,
    0.0007,
    -2e-05,
    0.00949,
    -0.00253,
    0.01307,
    0.02105,
    -0.00204,
    0.00366,
    0.00052,
    0.00054,
    -0.00821,
    -0.00803,
    -0.00491,
    0.01574,
    0.01071,
    0.0009,
    -0.00291,
    0.00224,
    -0.00291,
    0.0016,
    -0.00309,
    -0.00133,
    0.00434,
    0.00082,
    0.01218,
    6e-05,
    -0.00329,
    0.00177,
    -0.00938,
    0.00467,
    0.00093,
    -0.00285,
    4e-05,
    -0.00491,
    -0.00202,
    0.01586,
    0.00645,
    -0.00041,
    -0.00917,
    0.00388,
    -0.00027,
    -0.00743,
    0.00406,
    -0.00061,
    -0.00154,
    0.00275,
    0.00068,
    -0.00634,
    0.00251,
    -0.00904,
    -0.00359,
    -0.00464,
    0.00124,
    -0.00092,
    0.01038,
    -0.00085,
    -0.00125,
    0.00767,
    -0.00191,
    0.00036,
    -0.00596,
    0.00072,
    -0.00842,
    -0.01452,
    0.0048,
    0.00462,
    0.00375,
    -0.00616,
    0.00066,
    -0.00965,
    0.00136,
    0.0023,
    0.01795,
    -0.01729,
    -0.02278,
    0.00164,
    -0.00022,
    0.01065,
    -0.0111,
    0.00832,
    0.00804,
    -0.00181,
    0.0043,
    0.00939,
    -0.00175,
    0.00381,
    -0.00495,
    0.00499,
    0.00351,
    -0.0067,
    0.01029,
    -0.0009,
    0.00574,
    -0.00213,
    -0.00307,
    0.00224,
    0.00144,
    -0.00401,
    -0.0059,
    0.00226,
    0.00013,
    -0.00845,
    -0.0098,
    -0.00484,
    -0.0024,
    0.01061,
    -0.00061,
    0.00144,
    0.00716,
    -0.00061,
    -0.00145,
    4e-05,
    0.00255,
    0.02353,
    -0.00195,
    -0.00121,
    -0.00089,
    -0.0018,
    0.00082,
    -0.00779,
    -0.00391,
    -0.00696,
    0.00277,
    -0.00045,
    0.00512,
    -0.02009,
    0.00546,
    0.0011,
    0.00291,
    0.00177,
    -0.00246,
    -0.00091,
    -0.00562,
    0.0074,
    -0.00301,
    0.02537,
    -0.10785,
    0.00195,
    0.00056,
    -0.00313,
    -0.00141,
    0.00176,
    -0.00437,
    0.00028,
    -0.00088,
    -0.01239,
    0.00232,
    0.00602,
    0.00595,
    0.00322,
    0.00396,
    -0.00489,
    0.00865,
    -0.00556,
    0.00297,
    0.00431,
    -0.00064,
    0.00202,
    -0.00275,
    -0.00641,
    -0.01165,
    -0.00301,
    0.00492,
    -0.00548,
    0.00088,
    -0.00259,
    0.01258,
    0.00698,
    -0.00208,
    -0.00171,
    -0.00206,
    -0.00522,
    -0.00835,
    0.01484,
    -0.00211,
    -0.00147,
    -0.00709,
    -0.02028,
    0.00353,
    0.00494,
    0.00446,
    -0.00098,
    -0.00392,
    0.00063,
    -0.00605,
    -0.00885,
    -0.00558,
    -0.01566,
    0.00402,
    -0.02104,
    0.00334,
    -0.00332,
    -0.00032,
    -0.00064,
    -0.00828,
    0.0012,
    0.00161,
    -0.00582,
    -0.00612,
    0.00423,
    -0.00969,
    0.0006,
    -0.00507,
    -0.01007,
    0.04381,
    -0.00296,
    -0.01768,
    -0.00529,
    -0.00149,
    0.0056,
    0.00046,
    -0.00618,
    0.00817,
    0.01053,
    -0.00115,
    -0.00134,
    0.0022,
    -0.00313,
    -0.00774,
    0.00303,
    -0.00315,
    0.00971,
    0.00391,
    -0.0015,
    0.00945,
    -0.00144,
    -0.00789,
    0.00022,
    -0.00294,
    0.00278,
    -0.00281,
    -0.00045,
    0.00348,
    -0.00167,
    -0.01795,
    -0.00167,
    0.00732,
    0.00108,
    0.00052,
    0.00273,
    -0.00574,
    -0.00376,
    -0.00541,
    -0.00226,
    -0.00023,
    -0.00986,
    0.00571,
    0.00672,
    -0.00997,
    -0.00064,
    0.01659,
    -0.00029,
    0.00614,
    -0.00287,
    -0.00287,
    0.00296,
    0.0009,
    -0.00313,
    -0.57524,
    0.02255,
    -0.00083,
    0.0109,
    -0.00072,
    -0.00682,
    -0.00283,
    -0.00828,
    -7e-05,
    -0.00133,
    -0.00653,
    0.00179,
    0.00327,
    0.00954,
    0.00571,
    0.00293,
    -0.0015,
    -0.00715,
    -0.0004,
    -0.00501,
    0.00548,
    -0.00143,
    -0.00134,
    6e-05,
    -0.00587,
    0.00292,
    0.00996,
    -0.0116,
    -0.01692,
    -0.00211,
    -0.00542,
    0.00018,
    0.00253,
    0.00806,
    0.01544,
    -0.0056,
    -0.00691,
    -0.00276,
    0.00454,
    0.00691,
    0.00098,
    -0.00702,
    -0.00073,
    -0.00331,
    0.01158,
    -0.0013,
    -0.00207,
    0.00016,
    -0.00241,
    0.00352,
    -0.0223,
    0.00148,
    0.00374,
    -0.01118,
    0.00027,
    -0.0015,
    0.01547,
    0.01958,
    -0.00458,
    0.00844,
    -0.00532,
    0.00493,
    -0.00113,
    -0.00272,
    0.00099,
    -0.00479,
    0.00392,
    0.00044,
    0.00354,
    0.01747,
    0.00352,
    0.00456,
    -0.00113,
    0.00263,
    0.00458,
    0.00198,
    0.00203,
    0.01802,
    0.00401,
    0.00904,
    0.00583,
    -0.00662,
    1e-05,
    0.00051,
    -0.00195,
    6e-05,
    0.00282,
    0.00562,
    -0.00266,
    0.00263,
    -0.00158,
    0.00906,
    -0.01946,
    -0.00239,
    -0.00151,
    0.03296,
    -0.00159,
    0.01332,
    -0.00119,
    0.00475,
    0.00025,
    -0.00103,
    0.01246,
    -0.00522,
    -0.00498,
    0.00167,
    0.01238,
    -0.00154,
    -0.00385,
    0.00099,
    -0.00988,
    -0.00794,
    0.46671,
    -0.00153,
    -0.00746,
    0.00506,
    -0.00957,
    0.00587,
    0.00196,
    0.00066,
    0.00446,
    -0.00733,
    0.00429,
    -0.0053,
    -0.00684,
    0.00525,
    -0.00148,
    -0.00218,
    -0.00158,
    0.00554,
    0.00987,
    0.01079,
    0.00249,
    0.00742,
    -0.00175,
    0.00185,
    -0.01326,
    -0.003,
    0.00653,
    -0.00214,
    -0.01148,
    0.00688,
    -0.00607,
    -0.00083,
    -0.00108,
    -0.11466,
    0.00392,
    -0.0059,
    -0.00145,
    0.00111,
    -0.00232,
    -0.00429,
    -0.00153,
    -4e-05,
    -0.00589,
    -0.00378,
    0.00543,
    0.00418,
    0.00933,
    0.00218,
    -0.0029,
    0.0019,
    -0.00396,
    0.00015,
    0.01096,
    0.0071,
    0.00041,
    0.007,
    0.00404,
    -0.00383,
    -0.01522,
    -0.00281,
    -0.01972,
    0.0059,
    -0.01356,
    -0.00191,
    0.00463,
    0.01681,
    -0.01002,
    -0.00147,
    0.00474,
    -0.02047,
    -0.00024,
    0.00254,
    -0.00523,
    0.0063,
    1e-05,
    0.00116,
    -0.00317,
    0.00612,
    0.00275,
    -0.00372,
    -0.00245,
    -0.00029,
    -0.00046,
    0.0198,
    0.00327,
    -0.00015,
    -0.00251,
    0.00064,
    -0.0026,
    0.35921,
    -0.01432,
    -8e-05,
    -0.0102,
    -0.00466,
    -0.00271,
    0.00051,
    0.00664,
    0.0036,
    0.0003,
    -0.00068,
    -0.01966,
    0.00617,
    -0.00325,
    0.00482,
    0.00428,
    -0.00097,
    -0.00196,
    -0.00656,
    0.00551,
    -7e-05,
    -0.0024,
    0.00586,
    -0.00864,
    -0.00287,
    -0.00149,
    -0.00777,
    -0.00105,
    -0.00413,
    0.00021,
    -0.00442,
    -0.00507,
    0.01414,
    -0.00587,
    0.00978,
    -0.00595,
    0.00098,
    0.00793,
    0.00041,
    0.00749,
    0.00021,
    0.01368,
    -0.00621,
    -0.0058,
    -0.00154,
    0.00316,
    -0.00419,
    -0.00014,
    0.00159,
    0.00551,
    -0.00346,
    -0.00903,
    0.00599,
    0.00213,
    -0.00889,
    0.00193,
    -0.00058,
    0.00058,
    -0.0075,
    0.00411,
    0.00202,
    0.00026,
    -0.00532,
    -0.00178,
    -0.00336,
    -0.0016,
    0.00624,
    -0.01145,
    0.00783,
    0.00276,
    0.00247,
    0.00081,
    -0.00486,
    0.00421,
    -0.0035,
    0.00048,
    -0.00439,
    -0.00029,
    -0.00444,
    0.00766,
    -0.00371,
    0.00615,
    -0.00247,
    -0.00607,
    0.008,
    0.02625,
    0.00358,
    -0.00673,
    -0.00937,
    0.00212,
    -0.00301,
    0.00774,
    0.00126,
    -0.00139,
    0.00731,
    0.00171,
    -0.00015,
    -0.00644,
    -0.00372,
    0.00073,
    -0.00085,
    -0.00623,
    0.0037,
    0.00131,
    -0.09225,
    -0.00437,
    0.00155,
    -0.00287,
    0.0043,
    0.01919,
    0.00029,
    -0.00596,
    -0.00756,
    -0.00382,
    0.00441,
    0.00119,
    0.00071,
    0.00275,
    -0.00506,
    0.00256,
    -0.00226,
    0.00725,
    0.00896,
    -0.00101,
    -0.0043,
    -0.00193,
    0.02292,
    0.01136,
    -0.00831,
    -0.00517,
    0.00545,
    0.00089,
    -0.01016,
    0.01158,
    -0.00255,
    -0.00476,
    0.00322,
    0.00045,
    -0.01056,
    -0.00531,
    -0.00109,
    -0.01235,
    -0.0007,
    -0.01062,
    -0.00376,
    0.00592,
    -0.00543,
    -0.00075,
    0.00455,
    0.00215,
    -0.00484,
    0.00099,
    0.00344,
    -0.00548,
    0.00211,
    0.00035,
    -0.00586,
    -0.00185,
    -0.00326,
    -0.00692,
    -0.00483,
    0.00199,
    -0.00099,
    0.00325,
    -0.00252,
    0.00735,
    0.00408,
    -0.00732,
    -0.0111,
    0.00634,
    -0.00967,
    0.00705,
    0.00587,
    0.00111,
    0.00814,
    -0.00415,
    0.00393,
    -0.0003,
    0.00028,
    0.00061,
    0.00061,
    0.00043,
    0.00715,
    0.00063,
    -0.00158,
    -0.00652,
    -0.00087,
    0.01415,
    -0.02366,
    -0.00856,
    -0.00345,
    0.01361,
    0.00416,
    0.01995,
    0.00359,
    -0.0062,
    -0.00836,
    0.016,
    0.00696,
    -0.0025,
    0.01351,
    0.00243,
    -0.00367,
    -0.00159,
    -0.01877,
    -0.00815,
    0.00643,
    -0.00105,
    0.00243,
    0.00276,
    -0.00148,
    -0.00142,
    0.0018,
    0.00032,
    0.00104,
    0.00119,
    -0.00375,
    -0.00578,
    0.00667,
    0.00273,
    0.0005,
    0.00029,
    -0.01481,
    -0.00848,
    0.00171,
    0.00386,
    -0.00196,
    -0.00557,
    0.00676,
    0.00171,
    -0.00382,
    -0.00084,
    0.01055,
    -0.00284,
    -0.00515,
    0.00035,
    0.00836,
    0.00357,
    0.00868,
    -0.00979,
    0.01055,
    -0.00349,
    -0.00332,
    -0.00648,
    0.00122,
    -0.00319,
    -0.0018,
    -0.00237,
    0.00681,
    0.01016,
    0.00496,
    0.00262,
    -0.00449,
    -0.00453,
    -0.00072,
    -0.02088,
    -0.02039,
    -0.00099,
    -0.00835,
    -0.00113,
    0.00044,
    -0.00766,
    -0.00088,
    -0.07392,
    0.00623,
    0.00406,
    -0.00649,
    -0.00356,
    0.00235,
    0.0062,
    0.0015,
    0.01524,
    -0.00022,
    -0.00201,
    -0.0004,
    0.00456,
    -0.00517,
    -0.00147,
    -0.00018,
    -0.00219,
    -0.01326,
    0.00882,
    0.00077,
    -0.00261,
    -0.00716,
    0.00037,
    -0.00027,
    0.00033,
    0.00341,
    -0.0216,
    0.00619,
    -9e-05,
    0.01567,
    -0.00272,
    -0.00542,
    0.0018,
    0.0018,
    -0.00455,
    -0.00065,
    0.01057,
    0.00806,
    0.00448,
    -0.0122,
    -0.00282,
    -0.00495,
    -0.00532,
    -0.00467,
    -0.00114,
    9e-05,
    0.00345,
    0.00068,
    -0.00342,
    0.00867,
    -0.0022,
    0.00298,
    -0.02356,
    0.00074,
    -0.00938,
    -0.0031,
    -0.00123,
    -0.00069,
    -0.0021,
    0.00128,
    -0.00148,
    -0.01485,
    -0.0073,
    0.00354,
    0.00053,
    0.00256,
    0.01894,
    -0.00101,
    0.39507,
    0.00823,
    -0.00614,
    -0.01006,
    -0.00432,
    0.00011,
    0.00554,
    0.00916,
    0.00219,
    0.0038,
    -0.00285,
    0.02116,
    0.0003,
    -0.0034,
    0.00806,
    0.00332,
    -0.00156,
    -0.00173,
    -0.00213,
    0.00336,
    0.00128,
    0.00135,
    0.00499,
    0.00224,
    0.00747,
    -0.00335,
    -0.00456,
    -0.00748,
    -0.00214,
    -0.07418,
    -0.00867,
    0.00385,
    0.00731,
    0.00249,
    0.00714,
    -0.01626,
    0.00095,
    -0.00067,
    8e-05,
    -0.01207,
    0.02138,
    -0.00634,
    -0.00532,
    0.00217,
    0.00416,
    -0.00898,
    0.00074,
    -0.00184,
    -0.00051,
    0.00349,
    0.00205,
    0.00644,
    -0.00522,
    0.00178,
    -0.00276,
    -0.00608,
    -0.0031,
    0.00383,
    0.00223,
    0.01238,
    -0.00951,
    0.00142,
    0.01042,
    0.00516,
    -0.00563,
    -0.02034,
    0.00494,
    0.02023,
    0.00295,
    0.00107,
    0.00455,
    -0.01081,
    -0.00216,
    0.00331,
    -0.00121,
    -0.00714,
    -0.00104,
    -0.00295,
    0.01883,
    -0.00266,
    0.0003,
    -0.00615,
    -0.00146,
    -0.0193,
    0.00322,
    0.00269,
    -0.00838,
    -0.00294,
    0.01101,
    0.00152,
    -0.01073,
    0.0058,
    0.00445,
    8e-05,
    0.0007,
    0.00731,
    -0.00468,
    -0.00465,
    0.00157,
    -0.01493,
    0.00045,
    0.00497,
    0.00226,
    -0.00396,
    -0.00731,
    0.00923,
    -0.00366,
    0.00336,
    -0.00231,
    -0.00105,
    0.01528,
    0.00673,
    0.00719,
    0.00927,
    -0.00071,
    -0.00265,
    -0.00127,
    0.00669,
    -0.00101,
    0.00015,
    -0.01002,
    0.00797,
    -0.0033,
    0.00579,
    0.01332,
    -0.00293,
    0.00828,
    -0.00271,
    -0.02474,
    0.00167,
    -0.00639,
    0.00295,
    -3e-05,
    0.01685,
    0.00283,
    -0.00849,
    -0.00313,
    0.00384,
    0.00094,
    0.00233,
    -0.00166,
    -0.0057,
    0.00508,
    -0.00157,
    -0.00453,
    0.01234,
    0.00401,
    -0.00484,
    0.00349,
    0.00202,
    -0.00165,
    -0.00115,
    -0.01194,
    0.00523,
    -0.0047,
    0.00488,
    -0.00695,
    0.00061,
    0.00697,
    0.0209,
    0.00145,
    0.02085,
    0.01819,
    0.00039,
    -0.00205,
    0.00538,
    -0.00397,
    7e-05,
    0.00897,
    -0.00754,
    0.00389,
    -0.00517,
    -0.0029,
    0.00013,
    -0.00299,
    0.0032,
    -0.00328,
    0.00125,
    0.01431,
    0.00324,
    0.00889,
    0.00661,
    -0.00429,
    0.00014,
    -0.00165,
    0.00931,
    0.00221,
    0.002,
    0.00623,
    -0.00336,
    -0.00102,
    -0.00906,
    -0.00406,
    0.00405,
    0.00686,
    0.00704,
    -0.00059,
    0.00683,
    0.0027,
    -0.00766,
    -0.00183,
    0.00996,
    0.00039,
    0.00649,
    0.001,
    -0.00212,
    -0.00269,
    -0.00036,
    0.00968,
    0.00094,
    0.00652,
    -0.00183,
    -0.00394,
    -0.01066,
    0.00377,
    -0.01135,
    0.00273,
    -0.00635,
    0.00704,
    0.00016,
    0.00477,
    0.00107,
    3e-05,
    -0.00096,
    0.00549,
    0.00092,
    -0.00319,
    0.00138,
    0.00675,
    -0.00843,
    0.00856,
    -0.00145,
    -0.0014,
    0.0758,
    -0.01103,
    0.0036,
    -0.00444,
    -0.00522,
    0.00208,
    0.00343,
    0.00096,
    -0.00291,
    -0.00293,
    0.03111,
    -0.00602,
    0.00033,
    -0.00723,
    0.00266,
    0.00626,
    -0.00343,
    -0.00074,
    -0.00682,
    0.00234,
    0.00307,
    0.0098,
    -0.00364,
    0.00169,
    -0.00955,
    0.01337,
    -0.00156,
    0.00968,
    0.00445,
    0.00473,
    0.00293,
    0.00658,
    -0.00608,
    -0.00666,
    -0.00098,
    0.00511,
    0.00168,
    0.00608,
    -0.005,
    0.00738,
    -0.0087,
    -0.01015,
    -0.0184,
    0.001,
    -0.00031,
    0.00514,
    -0.0052,
    0.00017,
    0.00186,
    -0.00579,
    0.00971,
    0.00154,
    0.03021,
    0.00218,
    -0.00145,
    -0.00197,
    -0.07772,
    0.00755,
    0.00323,
    -0.00042,
    -0.0056,
    0.00354,
    0.00147,
    0.00621,
    0.00726,
    -0.00109,
    -0.0027,
    -0.00021,
    0.00666,
    0.00436,
    -0.00526,
    0.00111,
    -0.0081,
    -0.00414,
    -0.00493,
    0.06913,
    0.00036,
    -0.00105,
    -0.00201,
    0.00196,
    -0.00032,
    0.00523,
    0.02511,
    -0.0022,
    -0.00016,
    -0.00103,
    -0.00219,
    -0.00262,
    -0.00201,
    0.00302,
    -0.00498,
    -0.00205,
    0.00274,
    -0.00381,
    0.00158,
    -0.0056,
    -0.00587,
    0.00518,
    0.00116,
    0.04629,
    -0.01263,
    -0.00146,
    0.00603,
    -0.00203,
    0.00701,
    -0.00533,
    -0.00412,
    -0.00128,
    0.00545,
    -0.00251,
    -0.02277,
    0.00331,
    -0.00538,
    0.01624,
    -0.00818,
    -0.00911,
    -0.00358,
    -0.00304,
    0.00025,
    0.00014,
    0.00812,
    -0.0007,
    -0.00372,
    0.00264,
    0.02142,
    -0.0081,
    -0.00701,
    -0.00378,
    0.04342,
    -0.0024,
    0.00774,
    -0.00499,
    0.00383,
    0.00286,
    -0.00165,
    -0.0044,
    -0.00379,
    -0.00034,
    0.00241,
    0.00163,
    0.0013,
    -0.01108,
    -0.00696,
    -7e-05,
    0.0002,
    -0.00557,
    0.00753,
    -0.00398,
    0.0003,
    0.00704,
    -0.00297,
    0.00257,
    -0.0033,
    -0.00563,
    0.0023,
    0.00319,
    -0.00558,
    -0.00237,
    -0.00294,
    -0.00347,
    0.00092,
    0.00277,
    0.00166,
    0.00029,
    -0.00478,
    0.00313,
    0.00077,
    0.00834,
    0.00577,
    0.00051,
    -0.01401,
    0.00589,
    0.00154,
    -0.00517,
    -0.00274,
    -0.01886,
    -0.00912,
    0.00856,
    -0.011,
    0.00209,
    -0.00012,
    0.00226,
    0.00019,
    -0.0004,
    -0.00765,
    -0.01291,
    0.009,
    0.02037,
    0.001,
    0.00161,
    0.00775,
    0.00861,
    0.00239,
    0.00538,
    -0.00599,
    -0.00117,
    -9e-05,
    0.00176,
    -0.00434,
    -0.00314,
    0.00231,
    -0.00029,
    0.0055,
    0.00644,
    -0.00186,
    0.0014,
    -0.00234,
    0.00136,
    -0.01013,
    -0.00323,
    0.00805,
    -0.01575,
    0.01238,
    0.00148,
    0.00916,
    0.01859,
    0.01197,
    -0.00256,
    0.0004,
    0.00466,
    0.0056,
    0.00286,
    -0.0177,
    -0.00224,
    0.0074,
    0.00258,
    0.00254,
    -0.00691,
    -0.00313,
    0.00117,
    0.01229,
    -0.00253,
    0.00969,
    0.00102,
    -0.00767,
    -0.00124,
    0.003,
    0.00516,
    0.00438,
    -0.00925,
    0.00123,
    -0.00406,
    0.01596,
    -0.00277,
    -0.00518,
    0.00315,
    0.00289,
    -0.00365,
    -0.09804,
    0.00102,
    -0.01128,
    0.02145,
    -0.0013,
    -0.0017,
    -0.00798,
    0.00543,
    0.00081,
    -0.0006,
    -0.00235,
    -0.00247,
    0.0039,
    -0.00888,
    0.0006,
    0.00238,
    5e-05,
    -0.00655,
    -0.00443,
    0.00263,
    -0.00752,
    -0.00767,
    -0.00503,
    0.0014,
    -0.00071,
    0.0226,
    0.00496,
    -0.00213,
    0.00148,
    -0.00563,
    0.00124,
    -0.01249,
    -0.00835,
    0.0024,
    -0.00213,
    -0.00206,
    -0.00029,
    -0.00953,
    -0.00041,
    -0.00112,
    0.00583,
    0.01393,
    -0.00394,
    0.00356,
    -0.03122,
    0.01424,
    -0.00172,
    -0.00799,
    -0.00089,
    -0.00359,
    0.00723,
    0.00058,
    -0.00169,
    0.00277,
    0.01019,
    -0.0026,
    0.01067,
    0.00058,
    -0.00012,
    -0.00748,
    -0.00152,
    -0.00632,
    0.00596,
    -0.00489,
    0.01046,
    0.01236,
    0.00064,
    -0.00111,
    0.00359,
    0.00346,
    0.00072,
    0.00971,
    -0.00834,
    -0.00181,
    0.00111,
    -0.0017,
    -0.00479,
    -0.0025,
    0.01512,
    -0.02008,
    0.00151,
    0.00138,
    0.00069,
    0.00379,
    -0.00122,
    -0.00633,
    -0.00118,
    0.00041,
    -0.00728,
    0.00695,
    0.0,
    0.00243,
    0.00443,
    0.00374,
    0.00493,
    -0.00335,
    0.00108,
    -0.00252,
    0.00029,
    -0.00119,
    0.00219,
    -0.00395,
    0.00238,
    -0.01769,
    0.00185,
    -0.00677,
    -0.00473,
    -0.00555,
    0.00442,
    -0.0008,
    -0.00193,
    -0.01202,
    -0.00168,
    0.00262,
    -0.00139
   ],
   "ISOTOPE": [
    0.00153,
    0.00022,
    -0.00222,
    0.00102,
    0.00274,
    -0.00611,
    -0.00091,
    0.00101,
    -0.00094,
    0.00049,
    -0.00364,
    0.0019,
    0.00036,
    0.00061,
    -0.0012,
    0.00058,
    -0.00275,
    0.00142,
    0.00046,
    -0.00414,
    0.00072,
    0.00037,
    -0.00232,
    0.00346,
    0.00036,
    0.00032,
    0.00075,
    0.00082,
    0.00058,
    0.00041,
    -0.00097,
    0.00245,
    0.00134,
    -0.00956,
    -0.00154,
    0.00119,
    -0.00151,
    -0.00126,
    -0.00753,
    0.00094,
    0.00128,
    -0.00261,
    -0.00069,
    0.00209,
    0.00234,
    0.00481,
    0.00054,
    -0.00185,
    0.00269,
    0.0015,
    0.00279,
    -0.00589,
    -0.00063,
    -0.00034,
    0.00129,
    0.00392,
    0.00862,
    -0.00293,
    0.00149,
    0.00509,
    0.00157,
    -0.00206,
    0.00148,
    -0.00064,
    -0.0001,
    0.00757,
    0.00282,
    -0.00914,
    0.00092,
    -0.00074,
    -0.00046,
    0.00319,
    0.00142,
    0.0042,
    -0.00052,
    -0.0039,
    0.00055,
    0.00121,
    -0.00091,
    0.00264,
    0.00166,
    0.00105,
    -0.00057,
    0.00143,
    3e-05,
    -0.00064,
    -0.00047,
    -0.00014,
    0.00462,
    -0.00107,
    -9e-05,
    7e-05,
    -0.00107,
    -0.0048,
    -0.00063,
    0.00539,
    0.00157,
    0.0023,
    0.00082,
    -0.00045,
    -0.00012,
    -0.00131,
    0.01413,
    -0.00137,
    -0.00369,
    0.0013,
    0.00205,
    -0.00088,
    -0.00079,
    -0.00048,
    0.00186,
    0.00127,
    -0.00272,
    0.00084,
    0.00123,
    -0.00081,
    -0.0013,
    -0.00032,
    -0.00248,
    -0.00264,
    -0.00016,
    0.00327,
    -2e-05,
    -0.00113,
    -0.00138,
    -0.00222,
    -0.00046,
    0.00065,
    0.00102,
    -0.00028,
    0.00153,
    -0.00505,
    0.00104,
    0.00221,
    0.00067,
    -0.00044,
    -0.00518,
    0.00069,
    -0.0058,
    -0.00074,
    -0.00448,
    0.00071,
    -0.00922,
    -0.00226,
    -0.00088,
    -0.00711,
    -0.00128,
    0.09307,
    -0.00069,
    -0.00016,
    0.00206,
    0.00101,
    -0.00613,
    -0.00089,
    -0.00531,
    -0.00314,
    0.00201,
    -0.00542,
    0.00067,
    0.00275,
    -0.00219,
    0.00498,
    0.00027,
    -0.00644,
    -6e-05,
    -0.00088,
    0.00035,
    -0.00104,
    -0.00122,
    -0.00103,
    -0.00179,
    -0.00036,
    -0.00044,
    0.00081,
    -0.00552,
    0.00057,
    0.00146,
    -0.00176,
    0.00194,
    0.00119,
    0.00113,
    -0.00331,
    -2e-05,
    0.00031,
    0.0007,
    -0.01152,
    -0.00067,
    -0.00157,
    0.00314,
    -0.00139,
    -0.00124,
    0.00106,
    -0.00213,
    0.00015,
    -0.00036,
    -0.0013,
    0.00103,
    -0.00112,
    -0.00145,
    0.0026,
    5e-05,
    -0.00022,
    0.00033,
    0.00031,
    0.00055,
    0.0018,
    0.00187,
    0.0001,
    -0.00083,
    0.00051,
    -0.00332,
    0.00194,
    -0.00534,
    0.0014,
    -9e-05,
    -0.00076,
    -0.00252,
    -0.00101,
    0.00099,
    0.00285,
    0.001,
    -0.00128,
    -0.00739,
    0.00812,
    0.00068,
    -0.00065,
    -0.00723,
    0.00118,
    -0.00025,
    -0.00186,
    0.00176,
    -0.00068,
    -0.00127,
    0.00086,
    -0.00214,
    -0.00401,
    -0.00137,
    0.00045,
    -5e-05,
    8e-05,
    -0.00332,
    0.00013,
    0.00284,
    0.00029,
    0.00053,
    -0.00205,
    0.00038,
    -0.00067,
    -0.00024,
    0.00143,
    -0.00207,
    0.00805,
    0.00151,
    0.00149,
    -0.00119,
    -0.01324,
    -0.00183,
    -7e-05,
    -0.00038,
    -0.00043,
    -0.00128,
    -0.00126,
    0.0014,
    -0.00227,
    0.0,
    0.00172,
    0.00253,
    0.00081,
    -0.00041,
    -0.00051,
    -0.00048,
    0.00366,
    -9e-05,
    0.00045,
    -0.00458,
    0.00868,
    -0.00388,
    0.00156,
    0.00084,
    0.00089,
    -2e-05,
    0.00432,
    -0.00017,
    0.00133,
    -5e-05,
    -0.00355,
    0.14063,
    0.00028,
    -0.00082,
    -0.0007,
    0.00037,
    -5e-05,
    0.00025,
    -0.00111,
    -0.00028,
    -0.00275,
    -0.00037,
    -0.00113,
    -3e-05,
    0.00083,
    0.00329,
    0.00366,
    0.00161,
    -0.00141,
    -0.00014,
    0.00214,
    0.00015,
    -0.00147,
    0.00034,
    -0.00572,
    0.0048,
    0.00092,
    -0.00608,
    0.00141,
    -0.00081,
    -0.0006,
    0.00034,
    -0.00077,
    -0.00228,
    -0.00178,
    0.00246,
    0.00082,
    0.00042,
    -0.00199,
    -0.00078,
    -0.00053,
    0.00253,
    -0.00186,
    0.00042,
    -0.00353,
    0.00057,
    0.00063,
    -0.00037,
    0.00108,
    0.00058,
    0.01036,
    -0.00082,
    0.00565,
    0.00595,
    0.00095,
    0.00295,
    0.00106,
    -0.002,
    -0.0005,
    0.00212,
    -0.00121,
    -0.00534,
    -0.0012,
    0.00072,
    0.00235,
    0.00088,
    0.00072,
    -0.00548,
    -5e-05,
    -0.03558,
    -0.00107,
    -0.00073,
    -0.0004,
    0.00025,
    0.00833,
    -0.00637,
    -0.00056,
    -0.00496,
    -0.00075,
    -0.00016,
    0.00045,
    -0.00165,
    0.00374,
    0.00176,
    0.0008,
    0.0023,
    -0.00209,
    -0.00072,
    -0.00018,
    -0.00484,
    -0.00023,
    0.00114,
    -0.0021,
    0.00046,
    0.00554,
    0.0011,
    -0.00011,
    0.00106,
    -0.00024,
    0.0065,
    -0.00088,
    -0.00027,
    0.00124,
    -0.00175,
    0.00093,
    -0.00306,
    -0.00067,
    -0.00015,
    -0.00055,
    0.00202,
    0.00365,
    6e-05,
    -0.02765,
    5e-05,
    0.00024,
    -0.00094,
    -0.0001,
    -0.00149,
    -0.00264,
    0.00163,
    -0.00586,
    -0.00017,
    0.001,
    0.59511,
    0.00447,
    -0.00121,
    -0.00186,
    0.00035,
    0.0012,
    -0.00242,
    0.00935,
    -9e-05,
    0.00151,
    0.01429,
    0.00181,
    0.0009,
    -0.00321,
    6e-05,
    8e-05,
    0.01087,
    0.00063,
    -0.00258,
    0.00267,
    -0.00201,
    0.00026,
    -0.00025,
    -0.00041,
    -5e-05,
    -0.00073,
    -0.00233,
    0.00191,
    0.0057,
    -0.00591,
    -0.0023,
    0.00011,
    0.00179,
    0.00692,
    -0.00304,
    0.00104,
    0.00235,
    -0.0007,
    -0.00323,
    -0.00138,
    -0.0011,
    0.00166,
    0.00028,
    -0.0004,
    -0.00235,
    -1e-05,
    0.00069,
    0.00011,
    0.00043,
    -0.00106,
    0.00056,
    -0.0014,
    -0.00378,
    -0.00017,
    0.0025,
    -0.00315,
    0.00138,
    -0.0035,
    -0.00028,
    -0.00127,
    0.00184,
    -0.00038,
    -0.00098,
    0.00203,
    0.0011,
    0.00234,
    -0.00011,
    0.00032,
    -0.00168,
    0.00163,
    0.00077,
    -0.00096,
    -0.00037,
    -0.00094,
    0.00175,
    0.00161,
    -0.00083,
    0.0015,
    -0.00013,
    -0.00112,
    0.00118,
    -0.00022,
    -0.00199,
    -0.00206,
    -0.00032,
    0.0016,
    -0.00031,
    -0.00021,
    0.00142,
    0.00045,
    0.00282,
    0.0005,
    -0.0022,
    -0.00161,
    0.00179,
    -0.02068,
    -0.0004,
    0.0018,
    0.00078,
    -0.00085,
    0.00322,
    0.00153,
    -0.00903,
    0.00092,
    -0.00091,
    -0.00302,
    0.00584,
    0.0006,
    -0.0013,
    0.00285,
    0.025,
    -0.00075,
    -0.47934,
    0.00059,
    0.00074,
    -0.00054,
    -0.0009,
    -0.00312,
    0.00137,
    -0.00139,
    -0.00076,
    0.00117,
    -0.00445,
    -3e-05,
    0.00177,
    0.00176,
    0.00037,
    0.00141,
    0.00077,
    -0.00061,
    0.00132,
    0.0073,
    -0.00549,
    0.00208,
    0.00041,
    0.00099,
    0.0097,
    -0.00124,
    0.00206,
    0.00036,
    0.01103,
    0.00157,
    0.00235,
    -0.00019,
    0.00181,
    0.11156,
    -0.00555,
    -0.00137,
    -0.00066,
    0.00113,
    -0.00057,
    0.00679,
    1e-05,
    0.00015,
    0.00062,
    0.00182,
    0.00132,
    -0.00065,
    0.00215,
    -0.00043,
    0.0002,
    0.00131,
    -0.00657,
    0.00071,
    0.00063,
    -0.00714,
    -0.00245,
    -0.00077,
    0.0011,
    0.00187,
    -0.00699,
    -0.00031,
    0.00311,
    0.0008,
    0.0141,
    -0.00057,
    -0.00046,
    -0.00905,
    0.00197,
    0.00081,
    -0.00143,
    0.01642,
    0.00152,
    -0.00015,
    0.00164,
    0.00116,
    0.00181,
    -0.00023,
    -0.00062,
    0.00065,
    -0.00066,
    0.00449,
    -0.00056,
    0.00079,
    0.00062,
    0.00533,
    0.00153,
    0.00734,
    -0.00379,
    -0.00235,
    -0.00033,
    -0.35892,
    -0.00056,
    -0.00015,
    0.00627,
    -0.00026,
    -0.0032,
    -0.00034,
    -0.00156,
    0.00232,
    -0.00808,
    -0.00111,
    0.00245,
    -0.00296,
    0.00146,
    0.0031,
    7e-05,
    -0.00079,
    0.00032,
    -5e-05,
    -0.00082,
    0.00256,
    0.00043,
    -4e-05,
    -0.00506,
    -0.00045,
    -0.00047,
    0.0,
    -0.00173,
    -0.00051,
    -0.00256,
    -0.00101,
    -0.00027,
    0.00357,
    -0.0003,
    -0.0073,
    -0.00223,
    -0.00029,
    7e-05,
    0.00074,
    -0.00255,
    0.00259,
    -0.00117,
    0.00037,
    0.00033,
    0.00033,
    0.0057,
    -0.00012,
    -0.0006,
    -0.00112,
    -0.0006,
    0.00142,
    -0.00107,
    -0.00858,
    -0.0002,
    -0.00209,
    0.00484,
    0.00079,
    0.00014,
    0.00039,
    -0.00476,
    0.00073,
    0.00106,
    0.00513,
    0.00028,
    0.00099,
    -0.00212,
    0.0001,
    -0.00768,
    0.00233,
    0.0043,
    0.00684,
    -0.00012,
    -0.00157,
    0.00098,
    0.0002,
    -0.00063,
    0.00155,
    0.00086,
    -0.00091,
    -0.0008,
    -0.00152,
    -0.00017,
    -0.00433,
    1e-05,
    -0.00096,
    0.01236,
    0.00108,
    0.0017,
    0.00249,
    -3e-05,
    -0.00051,
    0.00019,
    0.00075,
    0.00192,
    -0.00165,
    -0.00314,
    0.00128,
    -0.00494,
    -0.00153,
    -0.00061,
    -0.00038,
    -0.00202,
    -0.00069,
    0.00097,
    0.09048,
    0.00564,
    -0.00084,
    -0.00047,
    0.00069,
    -0.00916,
    0.00038,
    -0.00056,
    0.00454,
    0.00086,
    -0.00791,
    -0.00191,
    0.00159,
    0.00049,
    -0.00013,
    -0.0012,
    -0.00294,
    -0.00122,
    0.00057,
    0.00209,
    -0.00291,
    0.00151,
    -0.00838,
    0.00164,
    0.00069,
    -0.00232,
    -0.00271,
    -0.00018,
    -0.00333,
    -0.00019,
    -0.00059,
    0.00011,
    0.00012,
    -0.0003,
    0.00175,
    0.00116,
    0.00142,
    0.01325,
    -0.00045,
    0.00073,
    0.00259,
    0.0001,
    0.00116,
    0.00021,
    -0.00034,
    -0.00245,
    0.00122,
    -1e-05,
    0.00029,
    -0.00019,
    0.00105,
    -0.0028,
    -0.00172,
    -0.00033,
    0.00224,
    0.00041,
    -0.00199,
    0.00243,
    -0.00178,
    0.00125,
    0.00162,
    0.00383,
    0.00531,
    9e-05,
    -0.00129,
    -0.00094,
    -0.00408,
    -0.00579,
    0.0005,
    0.00207,
    -0.00153,
    -0.00577,
    -0.00207,
    -0.00049,
    0.00513,
    -0.00133,
    -0.00517,
    0.00138,
    -0.00043,
    -0.00103,
    -0.00189,
    0.00421,
    0.00021,
    0.00257,
    0.0024,
    -0.00012,
    -0.0027,
    0.00271,
    -0.00019,
    -0.02148,
    5e-05,
    -0.00032,
    -0.00128,
    -0.00568,
    -0.00154,
    0.00056,
    -0.00547,
    -0.00017,
    -0.00343,
    0.00386,
    0.01339,
    0.00138,
    0.00129,
    0.00203,
    -0.0004,
    -0.00026,
    0.00091,
    -0.00016,
    0.00065,
    -0.00086,
    -0.00092,
    0.00037,
    0.00239,
    -0.00202,
    -0.01203,
    -0.00015,
    -0.00373,
    0.00121,
    -0.00389,
    -0.00132,
    -0.00385,
    -0.00234,
    -0.00073,
    0.00398,
    -0.00355,
    -0.00048,
    -0.00024,
    -0.00028,
    0.00179,
    -0.00215,
    0.0007,
    0.00817,
    0.00593,
    0.00104,
    -0.0028,
    0.00042,
    0.00634,
    -0.0004,
    0.00095,
    -0.00029,
    -0.00193,
    -0.00109,
    4e-05,
    -0.00401,
    0.00361,
    0.00029,
    -0.00113,
    0.00124,
    0.00232,
    -0.00153,
    -0.00064,
    0.00167,
    0.00557,
    0.00082,
    0.0059,
    0.00526,
    -0.00254,
    0.00259,
    0.00025,
    0.08207,
    0.00225,
    0.00053,
    -0.00329,
    0.00031,
    0.00154,
    -8e-05,
    -0.00077,
    -0.00443,
    -0.00038,
    0.00116,
    0.00647,
    -0.00481,
    -0.00055,
    -0.001,
    0.00089,
    -0.00158,
    0.00221,
    5e-05,
    -0.00049,
    0.00065,
    0.00399,
    -0.00211,
    -0.00019,
    0.00064,
    -0.0007,
    0.00483,
    0.00206,
    -0.00143,
    0.00635,
    -0.00011,
    -0.00223,
    0.00366,
    -0.00245,
    -8e-05,
    -0.00063,
    0.00117,
    0.00762,
    -0.00045,
    -0.00208,
    0.0028,
    0.00288,
    -0.00161,
    0.00121,
    -0.00125,
    0.00401,
    -0.00053,
    -0.00123,
    0.00027,
    -0.00107,
    -0.00067,
    -0.00067,
    -0.00067,
    -0.00062,
    -0.00108,
    -0.00248,
    0.00096,
    0.00058,
    -0.00094,
    -0.00048,
    -0.00253,
    0.00446,
    0.00073,
    -0.00295,
    0.00083,
    -0.00035,
    -0.01052,
    0.00056,
    -0.43062,
    -0.00098,
    0.0016,
    0.00474,
    -0.00231,
    -0.00056,
    -0.00153,
    0.00565,
    0.00113,
    0.00018,
    0.00063,
    0.00076,
    0.00074,
    2e-05,
    -0.00154,
    0.00333,
    -0.00113,
    -0.0033,
    0.00125,
    -0.00069,
    -0.00102,
    0.00037,
    0.00069,
    0.00199,
    0.00236,
    0.00126,
    0.00269,
    0.00732,
    -0.00089,
    0.07302,
    -0.00248,
    -0.00265,
    -0.00098,
    0.00027,
    -0.00013,
    -0.00265,
    -0.00014,
    0.00046,
    0.00214,
    0.00111,
    -0.0075,
    -0.00143,
    0.00057,
    0.00058,
    -0.00217,
    -0.00187,
    0.00398,
    -0.00268,
    0.00139,
    0.00185,
    -0.00078,
    0.00415,
    0.00119,
    0.00123,
    0.00103,
    0.00687,
    0.00294,
    -0.00041,
    0.00174,
    0.00952,
    -0.00059,
    -0.00167,
    0.01203,
    0.0013,
    -0.00068,
    0.00108,
    0.00126,
    -0.00073,
    0.00192,
    -0.00207,
    0.00403,
    0.0011,
    -0.00122,
    -0.00071,
    -0.00015,
    0.00148,
    -0.00092,
    -0.00158,
    -0.00419,
    -0.00171,
    0.00054,
    0.00393,
    0.00048,
    -0.0041,
    -0.00227,
    0.0011,
    0.00037,
    0.00091,
    -0.0057,
    0.00158,
    -0.00016,
    2e-05,
    -0.00101,
    3e-05,
    0.0022,
    0.00088,
    -0.00024,
    0.0002,
    4e-05,
    0.00022,
    -0.00118,
    -0.00256,
    0.00145,
    0.00066,
    0.00068,
    0.00154,
    0.00045,
    -0.00108,
    -3e-05,
    -0.00094,
    -0.00407,
    2e-05,
    -0.00075,
    0.00096,
    0.00135,
    0.00076,
    -0.00337,
    -0.00216,
    1e-05,
    -0.00121,
    -0.00047,
    -0.00039,
    -0.00112,
    -0.00224,
    -0.00044,
    -0.00272,
    -0.00147,
    -0.00121,
    0.00769,
    -0.00052,
    0.00287,
    -0.00163,
    -0.0,
    0.0107,
    -0.00029,
    -2e-05,
    0.00124,
    -0.00052,
    0.00048,
    -0.00087,
    0.00027,
    -0.00562,
    -0.00058,
    -0.0012,
    0.00259,
    -0.00183,
    0.00203,
    0.00661,
    -4e-05,
    -0.00142,
    -0.0006,
    0.00047,
    -0.0019,
    -0.00559,
    0.00026,
    0.00233,
    0.00124,
    0.0019,
    0.00155,
    -0.00789,
    -0.00234,
    -0.00758,
    0.00397,
    -0.00149,
    0.0022,
    -0.00898,
    0.00119,
    -0.00055,
    0.00336,
    -0.00091,
    -0.00183,
    -0.00414,
    -0.00161,
    0.0018,
    -0.00025,
    -0.00291,
    0.00079,
    0.00263,
    0.00118,
    -0.00118,
    -0.00309,
    0.00012,
    0.00108,
    -0.00692,
    0.00037,
    0.00541,
    0.00205,
    0.00283,
    0.00039,
    -0.00242,
    -0.00012,
    -0.00025,
    0.00177,
    -0.00215,
    -0.00148,
    -0.00042,
    -0.00023,
    0.00066,
    0.00021,
    0.00979,
    -0.00011,
    0.00177,
    0.00186,
    -0.00099,
    0.00157,
    -0.00126,
    0.00392,
    0.00193,
    -0.00207,
    0.00053,
    -0.00035,
    -0.001,
    0.00283,
    -0.00053,
    -0.00074,
    0.00609,
    0.00117,
    0.00086,
    0.00096,
    0.00036,
    -0.00269,
    -0.00752,
    -0.00031,
    -0.00532,
    0.00117,
    -0.00124,
    -0.01658,
    -0.00308,
    -0.00371,
    -0.00334,
    0.00214,
    -0.00098,
    0.00016,
    -0.08486,
    -0.00012,
    0.00165,
    0.00029,
    -0.00199,
    0.00094,
    -0.00111,
    -0.00029,
    0.00149,
    -0.00521,
    0.00247,
    -0.00192,
    -0.00024,
    0.00086,
    -0.00103,
    0.00062,
    -0.00048,
    -0.00073,
    -0.00098,
    -0.00016,
    -2e-05,
    -0.00101,
    -0.00073,
    -0.00762,
    -0.00107,
    0.00576,
    0.00159,
    -0.01867,
    -0.00372,
    0.00091,
    -0.00203,
    -0.00089,
    -0.00174,
    -0.00158,
    -0.0007,
    -0.00128,
    5e-05,
    -0.00032,
    -0.00083,
    -0.00141,
    -0.00279,
    -0.00696,
    0.00264,
    -0.00015,
    0.00916,
    0.00112,
    -0.0062,
    7e-05,
    0.00027,
    -0.00475,
    -0.00027,
    0.00353,
    0.00139,
    -0.00125,
    0.0001,
    0.00017,
    0.07402,
    3e-05,
    0.00095,
    -0.00196,
    -0.00163,
    0.00047,
    -0.00026,
    -0.00049,
    -0.00013,
    -0.00116,
    0.00249,
    0.00088,
    0.00081,
    -0.0013,
    0.00256,
    -1e-05,
    -0.00031,
    0.00306,
    -0.00218,
    -0.03725,
    -0.0012,
    0.00031,
    0.00119,
    0.00299,
    0.00181,
    -0.00093,
    0.00074,
    0.0005,
    0.00343,
    0.00093,
    0.00358,
    -0.0023,
    0.00487,
    -0.00281,
    -0.00049,
    -0.00049,
    -0.00029,
    0.00291,
    0.00459,
    -0.00033,
    -0.00239,
    -0.00229,
    -0.00071,
    -0.03853,
    -3e-05,
    -0.00082,
    0.0019,
    -0.00064,
    -0.00067,
    0.00678,
    0.00401,
    0.00051,
    0.00029,
    -0.00051,
    0.007,
    0.00089,
    -0.0026,
    0.00072,
    -0.00207,
    0.0077,
    0.00071,
    0.00078,
    -0.00095,
    0.00118,
    0.00118,
    0.00081,
    -0.00322,
    0.00218,
    -0.00057,
    -0.00079,
    -0.00132,
    -0.00015,
    -0.02714,
    0.00081,
    -0.00674,
    -0.00081,
    -3e-05,
    -8e-05,
    0.00059,
    -0.00394,
    0.00079,
    0.00033,
    0.00054,
    -0.00119,
    0.00062,
    -0.00016,
    -0.00165,
    -0.00178,
    -0.00248,
    -0.00103,
    0.00186,
    0.00145,
    -0.00123,
    0.00073,
    -8e-05,
    -0.00167,
    0.00063,
    -0.00259,
    0.00335,
    -5e-05,
    0.00136,
    -0.00158,
    -0.00082,
    -0.00043,
    -0.0016,
    -0.00033,
    -0.00119,
    0.00087,
    0.00084,
    0.00142,
    -0.00091,
    0.00486,
    -0.00126,
    -0.00088,
    -0.00089,
    0.00048,
    0.00153,
    -0.00155,
    -0.00277,
    0.00335,
    0.00061,
    0.00532,
    -0.00232,
    0.00964,
    -0.00039,
    -0.00116,
    -0.00016,
    -0.00035,
    -0.00382,
    0.00283,
    0.00281,
    -0.01884,
    -0.0003,
    0.0002,
    0.00043,
    -0.00012,
    0.00172,
    0.00023,
    -0.00064,
    -0.00623,
    0.00091,
    0.0029,
    -0.00189,
    -0.00073,
    0.00203,
    -0.00136,
    -0.00153,
    -0.00206,
    -1e-05,
    0.00045,
    -0.00118,
    -0.00071,
    -0.00158,
    -0.00036,
    0.00069,
    0.00062,
    -0.00355,
    -0.00156,
    -0.00227,
    -0.00361,
    0.00295,
    -0.00523,
    0.00504,
    0.00038,
    0.00027,
    -0.00148,
    0.01797,
    -0.00132,
    0.00617,
    0.00072,
    0.00099,
    -0.00416,
    -0.00068,
    0.00061,
    -0.00085,
    0.00077,
    0.00248,
    0.00038,
    0.00022,
    0.00162,
    0.00178,
    -0.00071,
    -0.00183,
    0.00221,
    0.00073,
    0.00076,
    0.00202,
    0.0007,
    -0.00031,
    -0.00114,
    0.00116,
    0.00023,
    0.0767,
    0.00114,
    -0.00403,
    -0.03134,
    -0.0058,
    -0.00196,
    0.00094,
    -0.00446,
    0.00057,
    -8e-05,
    -0.00312,
    0.00162,
    -0.00215,
    0.00551,
    -0.00821,
    -0.00011,
    0.00024,
    0.00082,
    6e-05,
    -0.00512,
    -0.00295,
    -0.00077,
    0.00031,
    -0.00183,
    -0.00279,
    -0.00809,
    -0.00098,
    -5e-05,
    0.00142,
    0.00114,
    0.00363,
    0.00249,
    -0.00075,
    0.00155,
    0.0003,
    0.00061,
    -0.00051,
    -0.00043,
    -0.00027,
    0.00149,
    0.00138,
    -0.00777,
    0.00174,
    0.00112,
    -0.00207,
    -0.00147,
    6e-05,
    -0.00558,
    0.00106,
    0.00226,
    -0.00837,
    0.00127,
    0.00082,
    -0.0022,
    -0.00307,
    0.00217,
    0.00108,
    0.00045,
    -0.00031,
    -0.00654,
    0.00317,
    0.00082,
    0.00191,
    -0.00379,
    -0.0046,
    -0.00156,
    -0.00133,
    -0.00147,
    -0.0003,
    0.0056,
    -0.00105,
    -0.00462,
    0.00027,
    -0.00208,
    0.0043,
    -0.00085,
    6e-05,
    -0.00185,
    -0.00201,
    -0.00296,
    0.00099,
    0.00206,
    -0.00226,
    0.00162,
    -0.00098,
    -0.00104,
    -0.00121,
    0.00029,
    -0.00238,
    -0.00029,
    0.00077,
    0.00564,
    -0.0001,
    0.00025,
    -0.00188,
    -0.00091,
    0.00018,
    -0.00213,
    0.00219,
    -0.00088,
    0.00021,
    0.0002,
    -0.00225,
    0.00543,
    0.00072,
    -0.00234,
    0.0007,
    0.00143,
    0.00296,
    -0.00069,
    -8e-05,
    -0.00082,
    -0.00308,
    -0.00015,
    -0.00094
   ],
   "MEDITATE": [
    0.00892,
    -0.00029,
    0.00215,
    -0.00572,
    -0.00191,
    0.00018,
    -0.00103,
    -0.0032,
    0.00157,
    -0.00934,
    -0.00206,
    0.00727,
    0.00221,
    -0.00013,
    0.00266,
    0.00191,
    -0.0044,
    0.00045,
    -0.00048,
    -0.00203,
    0.00139,
    0.00101,
    0.00178,
    -0.01049,
    -0.00059,
    -0.0024,
    -0.00108,
    -0.00583,
    -0.01043,
    0.00054,
    -0.00154,
    0.00117,
    0.00153,
    -0.00407,
    -0.00294,
    0.00122,
    -0.00291,
    0.0029,
    -0.01361,
    -0.00102,
    -0.00037,
    -0.00151,
    -0.00147,
    -0.0014,
    -0.00035,
    -0.00556,
    -0.00182,
    0.0005,
    -0.0019,
    -0.00025,
    -0.01014,
    -0.00645,
    -0.00031,
    -0.00176,
    -0.00148,
    0.00798,
    0.00438,
    0.01401,
    0.00658,
    0.0033,
    -0.00557,
    -0.00028,
    0.00869,
    -0.00076,
    0.00229,
    0.00855,
    0.00719,
    -0.0125,
    0.00248,
    -0.00907,
    0.00426,
    0.00106,
    0.0078,
    -0.00495,
    -0.00597,
    0.00338,
    -0.00313,
    0.00175,
    -0.00136,
    -0.00127,
    0.00191,
    -0.0013,
    0.00375,
    0.00023,
    -0.0033,
    0.00066,
    -0.00148,
    -0.00094,
    0.00698,
    0.00281,
    -0.00061,
    -0.00245,
    -0.00658,
    -0.00061,
    0.00125,
    -0.00635,
    0.00018,
    -0.00875,
    -0.00011,
    -0.00018,
    0.00083,
    -0.02385,
    0.03578,
    0.00113,
    -0.00546,
    0.00401,
    0.00049,
    -0.00048,
    -0.00293,
    0.00167,
    0.00603,
    0.00156,
    -0.00138,
    -0.0059,
    0.00283,
    0.00315,
    -0.00183,
    -0.00015,
    -0.00866,
    -0.00027,
    0.0047,
    0.00191,
    0.00057,
    0.0101,
    -0.00139,
    -0.003,
    -0.00079,
    0.00326,
    0.00212,
    0.00248,
    0.00015,
    -0.00144,
    -0.0012,
    -0.00031,
    0.00301,
    -0.00011,
    0.0039,
    0.00247,
    0.00164,
    0.00068,
    -0.00925,
    -0.0028,
    -0.00696,
    0.00037,
    0.00097,
    0.00273,
    4e-05,
    0.09634,
    -0.00363,
    -0.00246,
    0.00155,
    -0.00079,
    -0.006,
    -0.00025,
    -0.00745,
    -0.00083,
    -0.00298,
    -0.00278,
    -0.00024,
    -0.00027,
    0.00303,
    0.00407,
    0.00192,
    0.00626,
    -0.00149,
    0.00023,
    -0.00492,
    -0.00622,
    0.00073,
    0.00069,
    0.00146,
    -0.00019,
    -0.00083,
    -0.00155,
    -0.00191,
    0.00307,
    0.00302,
    -0.00137,
    -0.00074,
    0.00179,
    0.00369,
    -0.00546,
    0.00023,
    0.00175,
    0.00315,
    0.00234,
    -0.00359,
    -0.00041,
    0.00368,
    0.00041,
    0.00064,
    0.00204,
    0.0028,
    -9e-05,
    -0.00058,
    -0.00194,
    7e-05,
    0.00371,
    -0.00132,
    -0.00606,
    -0.00377,
    0.00051,
    -0.00453,
    -0.002,
    -0.00737,
    0.00062,
    0.00011,
    -0.00055,
    -0.00238,
    0.00077,
    -0.00182,
    0.00671,
    0.0055,
    0.00287,
    -0.00099,
    -0.00765,
    -0.00439,
    0.00223,
    -0.0004,
    0.00653,
    0.00117,
    0.00101,
    -0.02053,
    -0.00257,
    -0.0,
    0.00148,
    -0.00719,
    0.01009,
    0.00439,
    -0.00339,
    -0.00156,
    0.00378,
    -0.00023,
    -0.00223,
    -0.00055,
    0.00759,
    -0.00367,
    -0.00225,
    -0.00458,
    0.00523,
    0.00062,
    0.00354,
    -0.00173,
    -0.00035,
    -0.00073,
    -0.00551,
    0.0006,
    -0.00126,
    -0.0043,
    -0.00031,
    -0.00129,
    0.00882,
    -0.0005,
    0.0062,
    -0.0037,
    -0.01086,
    0.00185,
    0.00122,
    -0.0007,
    -0.00153,
    -0.00352,
    0.00133,
    0.0034,
    -0.00567,
    0.00066,
    0.00092,
    0.01029,
    0.00031,
    0.00273,
    0.00088,
    0.00012,
    0.00039,
    0.00568,
    0.00107,
    -0.00413,
    0.00945,
    0.00926,
    -0.00193,
    0.01006,
    0.00098,
    -0.00113,
    -0.00469,
    -0.00242,
    0.00552,
    -0.00092,
    -0.00155,
    0.10403,
    0.00059,
    -0.00099,
    -0.00136,
    0.00369,
    0.00099,
    0.00048,
    -0.00067,
    -0.00176,
    0.0034,
    0.00256,
    -0.00072,
    -0.00325,
    0.00301,
    -0.00054,
    -0.01683,
    -0.00219,
    -0.00994,
    -0.00277,
    0.00058,
    -0.00048,
    -0.00094,
    -0.00221,
    0.00013,
    0.00987,
    -3e-05,
    -0.00345,
    0.00015,
    -0.00152,
    -0.00072,
    0.0059,
    -0.00242,
    0.00017,
    -0.01018,
    0.00303,
    0.00174,
    -5e-05,
    0.00172,
    0.00106,
    0.00243,
    -0.00089,
    0.01026,
    -0.00238,
    -0.00617,
    -0.00011,
    0.00163,
    0.00699,
    -0.00201,
    0.00161,
    0.00014,
    0.00123,
    0.00816,
    -0.0014,
    -0.00113,
    -0.00023,
    -0.0011,
    0.00317,
    -0.00073,
    0.00175,
    -0.00195,
    -0.00206,
    -0.00875,
    -0.00362,
    0.00194,
    0.00576,
    -0.00114,
    0.00048,
    0.00228,
    -0.03439,
    0.00131,
    0.00191,
    0.00165,
    -0.0005,
    -0.00615,
    -0.00899,
    0.00355,
    -0.00438,
    -0.00023,
    0.00115,
    0.00069,
    0.00115,
    -0.00176,
    0.00243,
    -0.00052,
    -0.00275,
    -0.0002,
    -0.0013,
    -0.00663,
    -0.00195,
    -0.00125,
    0.00294,
    0.00626,
    -0.00157,
    0.00344,
    0.00425,
    -0.00204,
    -0.0028,
    0.0014,
    0.00174,
    0.00138,
    0.00013,
    -0.00043,
    -0.00244,
    -0.00076,
    0.00319,
    0.00111,
    0.00031,
    0.00157,
    -0.0235,
    0.00827,
    -0.00018,
    -0.00036,
    -0.00073,
    0.0023,
    -0.00386,
    -0.0029,
    0.00012,
    5e-05,
    0.00186,
    -0.00041,
    0.00138,
    -0.00149,
    0.59238,
    -0.00098,
    -0.00108,
    -0.00732,
    0.00015,
    0.0005,
    -0.0047,
    0.00655,
    -0.00115,
    0.00399,
    0.00783,
    0.00137,
    0.00072,
    0.0137,
    -0.00359,
    -0.0018,
    0.01033,
    -0.002,
    0.00063,
    0.00179,
    0.00453,
    -0.00272,
    -0.00045,
    0.00149,
    -0.00146,
    -0.00132,
    -0.0082,
    0.00667,
    0.00448,
    -0.0025,
    -0.00066,
    0.00212,
    0.00143,
    -0.00433,
    -0.02306,
    -0.00149,
    -0.00105,
    -0.00076,
    0.00671,
    -0.00098,
    -0.001,
    0.00207,
    0.00028,
    0.00445,
    -0.00273,
    -0.00012,
    0.00058,
    0.00025,
    -0.00152,
    -0.00024,
    -0.00166,
    -0.0006,
    -0.00041,
    -0.00096,
    0.00147,
    -0.00757,
    0.00543,
    -0.0126,
    0.00226,
    -0.00345,
    0.00494,
    -0.00614,
    -0.00014,
    -0.00317,
    -0.00848,
    0.00017,
    0.00016,
    0.00106,
    -0.00015,
    -0.0037,
    0.02059,
    0.00182,
    0.00065,
    -0.00299,
    -0.00407,
    0.00368,
    -0.00268,
    -0.00212,
    -0.00165,
    0.00181,
    0.00306,
    -0.00204,
    0.0006,
    0.00027,
    -0.00095,
    0.00332,
    -0.00198,
    -0.00225,
    0.00093,
    0.00102,
    0.00178,
    0.00428,
    -0.00971,
    0.00169,
    0.00053,
    -0.01901,
    0.00084,
    0.00145,
    -0.00204,
    0.00047,
    0.01061,
    0.00306,
    -0.00635,
    0.0015,
    -0.00138,
    -0.00171,
    -0.00452,
    -0.00018,
    -0.00171,
    -0.00065,
    0.04357,
    0.00171,
    -0.46895,
    0.00217,
    0.00177,
    -0.00355,
    -0.00141,
    -0.00443,
    -0.00268,
    0.00029,
    -0.00233,
    0.00599,
    -0.0124,
    0.00541,
    -0.00768,
    0.00144,
    0.00274,
    -0.00106,
    -0.00212,
    -0.00104,
    0.00335,
    -0.00335,
    -0.00648,
    -0.00195,
    -0.00316,
    0.0032,
    0.00379,
    0.00122,
    0.01128,
    -0.00052,
    0.0064,
    -0.00038,
    0.00765,
    0.00171,
    0.00138,
    0.11449,
    -0.00747,
    -0.00452,
    -0.00098,
    0.00168,
    -0.00067,
    0.01343,
    -0.00338,
    4e-05,
    0.00041,
    0.00106,
    -0.00116,
    -0.00299,
    0.00459,
    0.00077,
    -0.00058,
    -0.00165,
    -0.01221,
    -0.00222,
    -0.00258,
    -0.01023,
    0.00171,
    0.00303,
    0.00222,
    0.00476,
    -0.00144,
    0.00129,
    0.00089,
    -0.0001,
    0.01936,
    -0.00203,
    -0.00136,
    -0.00797,
    0.00213,
    -0.0009,
    -0.00069,
    0.01392,
    0.00157,
    0.00132,
    0.0055,
    0.00298,
    -0.00499,
    0.00189,
    -0.00053,
    -0.00189,
    -0.00056,
    0.01045,
    0.00052,
    0.00192,
    0.00129,
    0.01792,
    -0.00019,
    0.00967,
    0.00023,
    0.00059,
    0.00051,
    -0.36329,
    -0.00157,
    -0.00059,
    0.00059,
    0.00677,
    -0.00275,
    0.00211,
    0.00161,
    -0.00117,
    -0.0176,
    -0.00231,
    0.00685,
    -0.01049,
    0.00335,
    -0.00154,
    -0.00047,
    0.0005,
    -0.00062,
    0.00145,
    -0.00605,
    0.00224,
    0.0001,
    0.00251,
    0.01619,
    -0.00208,
    -0.00518,
    0.00021,
    -0.00194,
    1e-05,
    -0.00159,
    -0.00356,
    0.00045,
    -0.0008,
    0.00321,
    -0.0073,
    0.00992,
    -0.00072,
    0.00027,
    0.00069,
    -0.00366,
    0.01106,
    -0.00731,
    -0.00132,
    -0.00127,
    -0.00424,
    0.01089,
    0.00267,
    0.00026,
    0.00077,
    0.00543,
    -0.00422,
    -0.00146,
    0.00762,
    -0.00226,
    -0.00091,
    -0.00508,
    -0.00283,
    -0.00536,
    -0.00176,
    -0.00383,
    -0.00368,
    -0.00141,
    -0.00978,
    -0.00162,
    -0.00122,
    -0.00026,
    0.00556,
    -0.0153,
    -0.00233,
    0.00245,
    0.00457,
    0.00281,
    0.00138,
    0.00166,
    0.00082,
    -0.00203,
    0.00283,
    0.0022,
    -0.00052,
    0.00488,
    0.00199,
    0.00303,
    -0.00065,
    -0.00638,
    0.00088,
    -0.00248,
    -0.0012,
    0.00077,
    0.00283,
    -0.00047,
    -0.00179,
    0.00011,
    -0.00173,
    0.00333,
    -0.00309,
    -0.0153,
    0.00387,
    -0.00115,
    -0.00082,
    -0.00346,
    0.00205,
    0.02136,
    -0.00202,
    -0.00029,
    0.09789,
    -0.01628,
    0.00047,
    -0.00107,
    0.00087,
    -0.01595,
    -0.00061,
    -0.00124,
    0.0148,
    0.00741,
    0.00337,
    -0.0001,
    -0.00314,
    0.00015,
    0.00193,
    0.00055,
    0.00175,
    -0.00178,
    -0.00172,
    -0.00555,
    -0.01071,
    -0.00388,
    -0.01029,
    0.00117,
    -0.00298,
    -0.00041,
    -0.00384,
    -0.0035,
    0.00177,
    -0.00293,
    -0.00121,
    0.00055,
    0.00109,
    -0.00664,
    0.00022,
    1e-05,
    0.00345,
    0.02027,
    -0.00273,
    -0.00596,
    0.01372,
    -0.00119,
    0.00399,
    -9e-05,
    0.00865,
    -0.00311,
    0.00131,
    -0.00084,
    -0.00386,
    0.00344,
    0.00308,
    4e-05,
    0.00023,
    0.00061,
    0.00111,
    0.00074,
    -0.00158,
    -0.00499,
    -0.00248,
    -0.00246,
    0.00699,
    0.00643,
    0.00703,
    -0.00761,
    0.0055,
    -0.00216,
    0.00418,
    -0.00085,
    0.00091,
    -0.00407,
    0.00072,
    -0.00196,
    -0.00859,
    -0.00075,
    0.00569,
    0.00205,
    -0.0027,
    0.00068,
    -0.00129,
    0.00078,
    -0.00258,
    0.00834,
    0.00057,
    -0.00107,
    -0.00751,
    -1e-05,
    0.00026,
    0.00166,
    0.00118,
    -0.02828,
    0.00402,
    0.0019,
    0.00034,
    -0.0107,
    -0.0063,
    0.00258,
    0.00111,
    0.00374,
    -0.0047,
    0.006,
    0.00194,
    -0.00344,
    0.00505,
    0.0012,
    -0.00059,
    -0.00237,
    -0.00154,
    -0.00045,
    0.00101,
    0.00273,
    -0.00171,
    -0.00091,
    0.00026,
    -0.00078,
    0.00734,
    0.00053,
    -0.00684,
    0.00048,
    0.00052,
    0.00438,
    -0.00073,
    -0.00104,
    -0.00239,
    -0.00524,
    -0.00075,
    -0.00287,
    0.00164,
    -0.00018,
    -0.00139,
    -0.00454,
    0.00265,
    -0.01217,
    0.00827,
    -0.00029,
    -0.00231,
    -0.00335,
    0.0055,
    0.00259,
    0.00044,
    -0.00112,
    -0.00523,
    -0.00211,
    -0.00264,
    -0.00027,
    -0.00545,
    0.00012,
    -0.00321,
    0.00571,
    0.00409,
    -0.00107,
    0.00112,
    -0.01349,
    0.01028,
    -0.00266,
    0.00057,
    0.00051,
    0.00223,
    0.00367,
    -0.002,
    0.08858,
    -0.00011,
    0.00061,
    0.00043,
    -0.00529,
    0.00068,
    -0.00702,
    0.0016,
    0.01809,
    0.00022,
    -0.00357,
    0.00715,
    -0.00377,
    -0.00041,
    -0.00144,
    -0.00028,
    0.0018,
    0.00257,
    -0.00271,
    0.00289,
    -0.0028,
    0.00561,
    0.00129,
    0.00156,
    -0.00016,
    -0.00479,
    0.00665,
    -0.01605,
    0.00104,
    -0.0009,
    0.00069,
    -0.00381,
    0.02816,
    -0.00236,
    0.00245,
    -0.00125,
    0.00478,
    0.00876,
    -0.00094,
    -0.00464,
    0.00693,
    0.00148,
    -0.0122,
    0.00213,
    -0.00042,
    -0.00664,
    -0.00063,
    -0.00036,
    0.00188,
    -6e-05,
    0.00044,
    -0.00099,
    -0.0115,
    5e-05,
    -0.00528,
    -0.0038,
    0.00291,
    0.00101,
    0.00255,
    0.00023,
    0.00246,
    -0.008,
    -0.00022,
    0.00104,
    -0.00253,
    0.00061,
    -0.0113,
    0.00161,
    -0.41391,
    0.00247,
    -0.00084,
    -0.00138,
    0.00065,
    0.00243,
    -0.00013,
    0.00341,
    -0.00185,
    0.00147,
    0.00046,
    0.00256,
    -0.00407,
    0.00304,
    -0.00313,
    0.00116,
    -0.00318,
    -0.00044,
    -0.0,
    -0.0011,
    -0.00023,
    0.00038,
    6e-05,
    0.00133,
    0.00857,
    0.00238,
    0.00204,
    0.00156,
    0.0048,
    0.08147,
    -0.00203,
    -0.00227,
    0.00367,
    0.00454,
    0.00051,
    -0.0031,
    -0.00076,
    -0.00339,
    0.00021,
    0.003,
    -0.00926,
    -0.00078,
    -9e-05,
    -0.00229,
    0.00722,
    -0.00479,
    -0.00175,
    0.00211,
    0.00219,
    -0.00629,
    0.00055,
    0.02008,
    -0.00252,
    -0.00151,
    -2e-05,
    0.0026,
    0.00011,
    0.00265,
    -0.00367,
    0.00839,
    -0.00258,
    -0.00317,
    0.00138,
    -0.00017,
    -0.00517,
    0.01359,
    -0.00233,
    -0.01973,
    0.00032,
    0.00071,
    -0.00749,
    -9e-05,
    -0.00118,
    -0.00356,
    0.00041,
    0.00308,
    -0.00136,
    -0.00195,
    0.00096,
    0.00364,
    0.00683,
    -0.00037,
    -0.00243,
    -0.00983,
    -0.00255,
    -0.00049,
    0.00273,
    0.00156,
    -0.01402,
    -0.00145,
    0.00331,
    -0.00215,
    -0.003,
    0.00017,
    0.00177,
    -0.00288,
    0.00056,
    0.00342,
    -0.00238,
    0.00189,
    -0.00298,
    0.00231,
    0.00082,
    0.00086,
    -0.00189,
    -0.00254,
    0.00219,
    -0.00318,
    0.00022,
    0.0014,
    -0.00939,
    -0.00192,
    0.00224,
    0.00379,
    -0.0005,
    0.00277,
    0.00252,
    -0.00498,
    0.00022,
    -0.00291,
    0.00085,
    -0.00011,
    -0.00533,
    0.00013,
    -0.00343,
    0.00044,
    -0.005,
    -0.00135,
    0.02364,
    -0.00047,
    -0.00182,
    0.00155,
    0.00203,
    -0.00352,
    -0.00092,
    -0.0017,
    -0.001,
    0.00246,
    0.00215,
    -0.00344,
    0.00498,
    -0.00673,
    -7e-05,
    2e-05,
    0.0014,
    -0.00475,
    0.00588,
    -0.00202,
    -0.00299,
    0.00175,
    0.0023,
    -0.00204,
    0.00021,
    -0.00485,
    0.00299,
    0.00578,
    0.00155,
    -0.00082,
    0.00436,
    -0.0004,
    -0.00187,
    -0.00178,
    -0.00543,
    0.00068,
    0.00979,
    -0.00399,
    0.00118,
    0.00248,
    0.00562,
    -0.00408,
    0.0027,
    0.00491,
    -0.00516,
    0.00076,
    -0.00195,
    -0.00048,
    0.00231,
    -0.00063,
    0.00952,
    -0.00239,
    0.00031,
    -0.00069,
    0.00408,
    -0.00113,
    -0.00085,
    0.01733,
    -0.00071,
    -0.00037,
    -0.00156,
    0.00015,
    0.00099,
    0.0002,
    0.00619,
    -0.00157,
    -0.00734,
    -0.00426,
    5e-05,
    0.00111,
    0.00256,
    0.01014,
    -0.00403,
    -0.00322,
    -0.00013,
    0.00466,
    -0.00131,
    0.00152,
    0.00381,
    0.00339,
    -0.00121,
    -0.00378,
    -0.00256,
    0.00083,
    -0.00117,
    -0.00547,
    -0.00113,
    -0.00968,
    0.00182,
    -0.01152,
    -8e-05,
    -0.00182,
    -0.0104,
    -0.00312,
    -0.00107,
    -0.00436,
    -0.00403,
    -0.00347,
    -0.0245,
    -0.00226,
    -0.00607,
    -0.00149,
    0.00134,
    0.00117,
    -0.00094,
    -0.08449,
    0.00111,
    0.00039,
    -0.00096,
    0.00607,
    -0.00283,
    -0.00051,
    -0.00028,
    0.00015,
    -0.00582,
    -0.00084,
    0.0038,
    0.00248,
    -0.00056,
    -0.00303,
    -0.00088,
    0.00025,
    0.00058,
    0.00058,
    0.00063,
    -0.00041,
    -0.00665,
    -0.00018,
    -0.00459,
    0.01231,
    -0.00193,
    -0.00341,
    -0.01111,
    -0.0033,
    0.00289,
    -0.00184,
    -0.00101,
    -0.00425,
    -0.00109,
    -0.00175,
    -0.0004,
    0.00359,
    -0.00193,
    0.00087,
    0.00635,
    0.00322,
    -0.02075,
    -0.00602,
    0.00315,
    0.01491,
    0.00246,
    -0.00051,
    0.00017,
    0.00591,
    0.00844,
    -0.00095,
    0.00014,
    0.00238,
    -0.00431,
    -0.00029,
    -0.0004,
    0.09007,
    -0.00034,
    0.00256,
    0.00506,
    0.00573,
    -0.00487,
    -0.00376,
    0.00169,
    0.00442,
    -0.00148,
    0.00338,
    0.00066,
    -2e-05,
    0.00114,
    0.00229,
    0.00028,
    0.01428,
    0.00122,
    -0.00111,
    -0.03345,
    0.00177,
    -0.0019,
    -0.0088,
    -0.00142,
    -0.00141,
    0.0024,
    0.00702,
    0.00223,
    0.0048,
    0.00191,
    -0.0003,
    -0.00828,
    0.00422,
    -0.00493,
    0.003,
    0.00279,
    0.00313,
    0.00527,
    0.00098,
    0.00222,
    -0.00274,
    -0.00332,
    -0.00164,
    -0.04293,
    -0.00908,
    0.0002,
    0.0018,
    0.00045,
    -0.01767,
    0.00255,
    0.00362,
    -0.00047,
    0.00211,
    0.00252,
    -0.00258,
    0.00183,
    0.0073,
    0.00726,
    0.00236,
    0.00589,
    0.00182,
    0.00076,
    0.00016,
    -0.00129,
    0.00264,
    0.00027,
    0.00477,
    -0.00329,
    0.0029,
    0.00569,
    0.00403,
    -7e-05,
    -0.01524,
    -0.00101,
    -0.01444,
    -0.00259,
    -0.00096,
    0.00113,
    -0.00201,
    0.00463,
    0.00071,
    -0.00128,
    -0.0009,
    -0.00132,
    0.00119,
    -0.00038,
    0.00021,
    -0.00097,
    0.00078,
    0.00078,
    -0.00077,
    -4e-05,
    -0.00051,
    -0.00182,
    0.0007,
    -7e-05,
    0.00036,
    0.00321,
    0.00136,
    -0.00306,
    -0.00386,
    -0.0003,
    0.00219,
    0.00157,
    -0.00223,
    -0.00214,
    -0.00021,
    -0.00155,
    -0.00191,
    0.00185,
    0.00369,
    -0.01524,
    -0.00046,
    0.00207,
    -0.00135,
    -0.00142,
    -0.00252,
    0.00353,
    -0.00374,
    0.00137,
    0.00038,
    0.00213,
    0.00286,
    0.00589,
    -0.00164,
    3e-05,
    0.00032,
    -0.00109,
    0.00424,
    0.00649,
    0.00543,
    -0.01799,
    -0.00295,
    -0.00092,
    0.00136,
    0.00338,
    0.00104,
    -3e-05,
    7e-05,
    0.00328,
    0.00415,
    -0.00056,
    -0.00393,
    -0.00038,
    -0.00016,
    -0.00322,
    0.00018,
    -0.00013,
    -0.00021,
    -0.00305,
    -0.00166,
    6e-05,
    0.00064,
    -0.00096,
    -0.0001,
    -0.00538,
    0.005,
    -0.00381,
    -0.00466,
    -0.01028,
    -0.00411,
    0.00482,
    -0.00246,
    -0.00255,
    -0.00162,
    -0.00143,
    0.01825,
    -0.00551,
    0.00593,
    0.00057,
    0.00138,
    -0.00521,
    0.00459,
    -0.00151,
    -0.00813,
    0.00083,
    -0.00062,
    0.00245,
    0.00177,
    0.00203,
    0.00034,
    -0.00027,
    -0.00196,
    -0.00916,
    0.00165,
    -0.00446,
    0.01029,
    -0.00249,
    0.00155,
    -0.00084,
    0.00242,
    -7e-05,
    0.0915,
    -0.00346,
    0.00281,
    -0.02896,
    0.00073,
    0.00011,
    -0.00136,
    -0.00812,
    -0.00039,
    -0.00061,
    -0.0047,
    0.00375,
    -0.00849,
    0.00116,
    0.00563,
    -0.00318,
    -0.0028,
    -0.00232,
    0.00534,
    -0.00895,
    -0.01283,
    -0.00945,
    0.00253,
    0.00826,
    7e-05,
    -0.01142,
    -0.00319,
    0.0035,
    0.00604,
    0.00145,
    0.00421,
    -0.0159,
    0.00522,
    0.00405,
    0.00154,
    0.00116,
    -0.00663,
    0.00065,
    -0.00077,
    0.00162,
    -0.00434,
    -0.00997,
    -0.00403,
    0.00106,
    0.00666,
    -0.00602,
    0.00042,
    -0.00117,
    0.00131,
    0.00161,
    -0.00537,
    -0.00044,
    0.00383,
    0.0001,
    0.01296,
    0.00331,
    -0.00751,
    0.001,
    5e-05,
    -0.00905,
    -0.00079,
    0.00072,
    0.00294,
    -0.00424,
    0.00091,
    -0.00494,
    -0.00545,
    0.0028,
    0.00103,
    0.00694,
    0.00032,
    0.00138,
    -0.00083,
    -0.00096,
    0.00685,
    -0.00477,
    0.00183,
    -0.00067,
    -0.00554,
    -0.0113,
    0.00269,
    -0.00022,
    0.00124,
    0.00151,
    -0.00092,
    -0.00077,
    -0.00028,
    -0.00042,
    -0.00083,
    -0.00631,
    9e-05,
    0.00621,
    -0.00284,
    0.00172,
    -0.00199,
    -0.00197,
    0.00026,
    -0.00318,
    -0.00171,
    0.00407,
    -0.00107,
    -0.00034,
    -0.00179,
    0.00627,
    0.00198,
    -0.00178,
    -0.00463,
    0.00466,
    -0.00943,
    -0.00316,
    0.00151,
    -0.01135,
    0.001,
    0.00377,
    0.00091
   ],
   "INFUSION": [
    0.00683,
    0.00082,
    0.00167,
    -0.00024,
    0.00112,
    -0.00399,
    -0.00141,
    -0.00044,
    -9e-05,
    -0.00143,
    -0.00321,
    0.00354,
    0.00049,
    0.00145,
    0.00069,
    0.00086,
    0.00036,
    0.00107,
    0.0009,
    -0.002,
    0.00055,
    -0.00021,
    -0.00275,
    -0.00275,
    0.00209,
    -0.00042,
    0.00366,
    -0.002,
    -0.00724,
    0.00051,
    0.00109,
    0.00087,
    0.00035,
    -0.00873,
    0.00161,
    0.00039,
    -0.00213,
    0.00046,
    -0.00737,
    -0.00583,
    0.0005,
    -8e-05,
    0.00096,
    -0.00052,
    -0.00666,
    0.00148,
    -0.00257,
    -0.00128,
    -0.0008,
    -5e-05,
    0.00247,
    -0.00403,
    -0.00044,
    -0.00038,
    0.00234,
    0.00453,
    0.00394,
    -0.00059,
    0.00323,
    0.00702,
    -0.00087,
    0.00103,
    0.00235,
    0.0001,
    0.00047,
    0.0038,
    0.0027,
    -0.00712,
    1e-05,
    -0.00142,
    -0.00066,
    0.00616,
    -7e-05,
    0.00229,
    -0.00249,
    -0.00479,
    0.00103,
    -0.00108,
    0.00043,
    0.00152,
    0.00063,
    -0.00774,
    0.00149,
    0.00286,
    0.00041,
    0.00155,
    -0.00043,
    0.00023,
    0.00276,
    -0.00082,
    0.00043,
    0.00062,
    -0.00137,
    -0.0028,
    -0.00172,
    -0.00367,
    0.0013,
    0.00105,
    -0.00056,
    -0.00014,
    0.00091,
    -0.01166,
    0.02257,
    -0.00093,
    -0.00402,
    0.00227,
    -0.00088,
    0.00355,
    0.00027,
    -4e-05,
    -0.00517,
    0.00145,
    -0.00116,
    0.00218,
    0.00037,
    0.0006,
    -0.00073,
    0.00057,
    0.0014,
    0.00021,
    0.00217,
    0.00487,
    0.001,
    0.00122,
    -0.00045,
    -0.00175,
    -0.00047,
    -0.00031,
    -0.00081,
    0.00053,
    -0.00254,
    -0.00216,
    0.00082,
    0.00031,
    0.00039,
    -0.00103,
    -0.00081,
    0.00094,
    -0.00208,
    -0.00227,
    0.00352,
    -0.00054,
    -0.00587,
    0.00031,
    -0.00023,
    -0.00235,
    -0.00058,
    0.0932,
    -0.00065,
    0.0014,
    0.00142,
    -0.00067,
    -0.00112,
    -0.00169,
    -0.00209,
    0.00345,
    0.00105,
    -0.00459,
    0.00192,
    0.0022,
    0.00691,
    0.0042,
    0.00026,
    0.0097,
    -0.00014,
    -0.00053,
    -0.00212,
    -0.00282,
    0.00127,
    -0.00049,
    0.00046,
    -0.00081,
    -0.00048,
    -0.00017,
    -0.00261,
    0.00074,
    1e-05,
    0.00055,
    0.00186,
    0.00073,
    -0.00115,
    -0.00184,
    -0.0006,
    0.00183,
    0.0019,
    -0.00858,
    -0.00516,
    -0.0013,
    0.00016,
    -0.00291,
    0.00055,
    7e-05,
    -0.00091,
    -0.00069,
    0.00055,
    -0.00166,
    0.0017,
    0.00022,
    -0.00082,
    0.00469,
    -0.00179,
    -0.00047,
    -0.00097,
    -0.00068,
    -0.00018,
    0.0016,
    0.00106,
    -0.00097,
    0.00085,
    0.00016,
    -0.00375,
    0.00058,
    -0.00535,
    0.0001,
    0.00162,
    -0.00388,
    -0.00174,
    -0.00037,
    -0.0008,
    -8e-05,
    0.00068,
    0.00016,
    -0.01085,
    0.00226,
    -0.00027,
    0.00077,
    0.00133,
    0.00327,
    0.01065,
    -0.00105,
    0.00082,
    -0.00201,
    -0.00124,
    -0.00143,
    -0.00056,
    -0.00465,
    -0.00268,
    -0.0012,
    -0.0001,
    0.00179,
    -0.00063,
    0.00025,
    0.001,
    0.00065,
    -0.00088,
    0.00021,
    0.0005,
    -0.00112,
    0.00065,
    0.00091,
    -0.00306,
    0.00371,
    0.00209,
    0.00069,
    0.00441,
    -0.01153,
    -0.00284,
    0.00037,
    -0.00076,
    0.00156,
    -0.00096,
    2e-05,
    5e-05,
    0.00121,
    0.00103,
    0.00427,
    0.00338,
    0.00174,
    0.00237,
    0.00023,
    0.00041,
    -0.00038,
    0.00334,
    -0.00098,
    -3e-05,
    0.00763,
    0.00041,
    0.00048,
    0.00182,
    -0.00043,
    0.00035,
    0.0005,
    -0.00237,
    0.00228,
    0.00031,
    -0.00827,
    0.13247,
    -0.00098,
    -0.00019,
    -0.0001,
    0.00231,
    -2e-05,
    -0.00076,
    0.00042,
    -0.00495,
    -0.00432,
    0.00033,
    0.00136,
    -0.00016,
    0.00027,
    0.00326,
    0.00879,
    0.00049,
    -0.00144,
    -0.00017,
    0.00067,
    -0.00024,
    -0.00364,
    0.00066,
    -0.00401,
    0.0042,
    0.00105,
    0.00208,
    0.00126,
    -0.00108,
    -0.00057,
    -0.00231,
    -0.00088,
    0.0012,
    -0.00011,
    0.00141,
    0.0007,
    -0.00199,
    -0.00258,
    -0.00088,
    -0.00234,
    0.00173,
    0.00563,
    -0.00231,
    8e-05,
    -0.00161,
    0.00202,
    0.00338,
    0.00081,
    0.00128,
    0.00116,
    0.00113,
    0.00614,
    0.00208,
    -0.00119,
    0.00114,
    5e-05,
    0.00036,
    0.00057,
    -0.00068,
    0.00076,
    -0.00238,
    0.00054,
    -0.00084,
    0.00134,
    0.0001,
    0.00037,
    -8e-05,
    -0.00013,
    -0.03211,
    -0.00137,
    0.00029,
    0.00016,
    0.0001,
    0.00251,
    -0.01372,
    0.00142,
    -0.00332,
    -0.00173,
    0.00061,
    0.00096,
    -0.00175,
    0.00215,
    0.00475,
    -0.00045,
    0.00344,
    0.00025,
    -0.00126,
    -0.00017,
    -1e-05,
    0.00097,
    0.00075,
    0.00085,
    0.00105,
    0.00097,
    -0.00037,
    2e-05,
    -0.00097,
    -0.00012,
    -0.00523,
    -0.00171,
    -0.00084,
    0.00112,
    -0.00539,
    0.00014,
    -0.00061,
    -0.00201,
    0.00157,
    -0.00155,
    -0.00362,
    0.00338,
    -0.00151,
    -0.01856,
    0.00089,
    -0.00061,
    0.00171,
    -0.00051,
    -0.00273,
    -0.00162,
    0.00033,
    -0.00191,
    -0.00145,
    0.00076,
    0.60098,
    0.0053,
    -0.00212,
    0.00028,
    0.00028,
    -0.00068,
    -0.00482,
    0.01097,
    0.001,
    0.00099,
    0.01437,
    7e-05,
    0.00042,
    -0.00015,
    0.00035,
    -0.00207,
    0.01136,
    -0.00028,
    0.00308,
    0.00163,
    -0.00045,
    -0.0002,
    0.00045,
    0.00156,
    0.00091,
    -0.00035,
    -0.00821,
    0.00523,
    0.00638,
    -0.00583,
    -0.00064,
    -0.00043,
    0.00119,
    0.0072,
    -0.00686,
    -0.00036,
    0.00355,
    0.00029,
    -0.0029,
    -0.00076,
    -0.0002,
    0.00066,
    0.00021,
    0.00102,
    -0.0012,
    0.00014,
    0.00071,
    0.00051,
    -0.00077,
    -0.00124,
    -0.00478,
    -0.00081,
    -0.00243,
    -0.00054,
    0.00089,
    -0.002,
    0.00178,
    -5e-05,
    -0.00144,
    0.00189,
    0.0026,
    -0.00014,
    -0.00068,
    0.00133,
    -0.00181,
    0.00078,
    0.00079,
    0.00076,
    0.00077,
    -0.00052,
    0.00124,
    0.00013,
    0.00089,
    0.00164,
    0.00103,
    -0.00018,
    -0.00372,
    0.00069,
    1e-05,
    0.00195,
    -0.00031,
    -0.0001,
    -0.00232,
    -0.00031,
    0.00038,
    -0.00088,
    -0.00112,
    -0.00125,
    0.00105,
    0.00072,
    -0.00198,
    -0.00372,
    0.00029,
    -0.00065,
    0.00267,
    -0.02046,
    0.00096,
    0.00282,
    0.00027,
    -0.00275,
    0.00082,
    0.00053,
    -0.01499,
    -0.0,
    -0.00101,
    -0.00098,
    -0.00113,
    0.00071,
    0.00039,
    0.00168,
    0.02617,
    -0.00274,
    -0.47355,
    0.00098,
    0.00142,
    -0.00148,
    -0.00102,
    -0.00383,
    -4e-05,
    -0.00098,
    -0.00029,
    -0.00057,
    -0.00527,
    -0.00258,
    -0.00665,
    0.00274,
    0.00216,
    -0.00079,
    -0.00011,
    -0.00055,
    0.00092,
    -0.0006,
    -0.00126,
    0.00044,
    -0.00032,
    0.00121,
    0.00806,
    -0.00245,
    0.00016,
    0.00098,
    0.00352,
    -0.00022,
    0.00035,
    -0.00035,
    0.0017,
    0.12081,
    -0.00908,
    -0.00292,
    -0.00131,
    0.00111,
    -0.00214,
    -0.00076,
    0.00123,
    0.00013,
    0.00144,
    -0.00055,
    -0.00023,
    -0.00125,
    -0.00647,
    0.00146,
    -0.00078,
    0.00014,
    -0.00203,
    -0.00055,
    -0.00167,
    -0.00069,
    0.00639,
    -0.00347,
    1e-05,
    0.00153,
    -0.00219,
    4e-05,
    0.00056,
    0.00041,
    0.01148,
    0.00051,
    -1e-05,
    -0.00836,
    0.00024,
    0.0004,
    -0.0011,
    0.01642,
    0.0003,
    0.00152,
    -0.0003,
    0.00401,
    0.00027,
    -0.00049,
    -0.00045,
    0.00121,
    0.00039,
    0.00286,
    -0.00034,
    0.00136,
    0.00049,
    -0.00314,
    0.00094,
    0.00162,
    0.0003,
    -0.00016,
    -0.00208,
    -0.35626,
    -0.00424,
    0.00039,
    0.00081,
    -0.00315,
    -0.00135,
    -0.00113,
    -0.00256,
    0.00049,
    -0.0059,
    0.00044,
    0.00369,
    0.00176,
    2e-05,
    0.00099,
    -0.00138,
    -0.00012,
    -0.00077,
    0.00101,
    -0.00071,
    0.00181,
    0.00017,
    -0.00101,
    0.00057,
    -0.00093,
    -0.00084,
    -0.00203,
    -0.00129,
    -0.00332,
    -0.00292,
    -0.00038,
    0.00121,
    0.00223,
    -1e-05,
    -0.00288,
    0.00257,
    -0.00096,
    0.00063,
    0.00112,
    -0.00451,
    0.00111,
    -0.00269,
    0.00074,
    0.00336,
    -0.00342,
    0.00693,
    -0.00025,
    -0.00098,
    -0.00083,
    -0.00125,
    0.00096,
    -0.00037,
    -0.00571,
    -0.00079,
    -0.00021,
    -0.00343,
    -0.00056,
    -0.0009,
    0.00271,
    0.00097,
    0.00046,
    -0.00036,
    0.00242,
    0.00137,
    -0.00014,
    0.00059,
    -0.00041,
    -0.00118,
    0.00124,
    0.00287,
    0.00277,
    0.00084,
    -0.00217,
    0.0007,
    -0.00095,
    0.0008,
    0.00351,
    0.00041,
    2e-05,
    0.00112,
    -0.00065,
    -0.00028,
    -0.00014,
    -0.00136,
    -0.0007,
    0.00451,
    -0.00165,
    0.00544,
    0.00142,
    0.00052,
    -0.00035,
    0.00017,
    0.00277,
    0.00124,
    -0.00177,
    -0.00627,
    -0.00107,
    0.00065,
    -0.00155,
    -0.00019,
    0.00253,
    0.00297,
    0.00111,
    -2e-05,
    0.08694,
    -0.00479,
    0.00022,
    -0.00048,
    -0.00015,
    -0.00523,
    0.00104,
    -0.00026,
    0.00463,
    0.00299,
    -0.01177,
    -0.00013,
    0.00112,
    -0.00209,
    0.00059,
    -0.00101,
    -0.00176,
    3e-05,
    -0.00106,
    -0.00063,
    -0.00396,
    -0.00061,
    -0.00969,
    0.00047,
    -0.00057,
    -0.00151,
    -0.0017,
    -0.00109,
    0.00386,
    -0.00459,
    -0.00183,
    0.0006,
    -0.00058,
    -2e-05,
    0.00287,
    -0.00041,
    0.00162,
    0.01479,
    -0.00037,
    -0.00383,
    0.00121,
    -0.00024,
    -0.00239,
    -2e-05,
    0.00017,
    0.00205,
    0.00195,
    0.00067,
    0.00074,
    -0.00036,
    -0.00066,
    0.00081,
    -0.00111,
    -0.00092,
    0.00234,
    0.00038,
    -0.00099,
    0.00297,
    -0.00075,
    -0.00099,
    0.00014,
    0.00118,
    0.00022,
    4e-05,
    0.00225,
    -0.0012,
    0.00202,
    -0.00427,
    -0.00014,
    -0.00127,
    -0.00121,
    0.00018,
    -0.00156,
    -0.00148,
    0.00265,
    0.00073,
    -0.00523,
    -0.0003,
    -0.0028,
    -0.00083,
    -0.00107,
    0.00522,
    -0.00054,
    0.00385,
    -0.00071,
    -0.00131,
    -0.00348,
    -0.00236,
    -0.00011,
    -0.02299,
    -0.0005,
    0.001,
    0.00054,
    -0.00682,
    -7e-05,
    0.00396,
    -0.00352,
    -0.00205,
    -0.00079,
    0.00186,
    0.00432,
    -0.00046,
    0.00045,
    0.00098,
    -0.00091,
    0.00027,
    -0.00031,
    -0.00159,
    0.00173,
    0.00133,
    -0.00142,
    0.00117,
    0.00187,
    -0.00466,
    0.00085,
    0.00068,
    -0.01335,
    -0.00178,
    -0.0095,
    0.00073,
    0.00534,
    0.00185,
    -0.00042,
    0.0013,
    -0.00136,
    -0.00049,
    -0.00064,
    -0.00135,
    0.00016,
    -0.00288,
    0.00077,
    -0.00482,
    0.00168,
    0.00118,
    -0.00628,
    0.00012,
    0.0038,
    -0.00192,
    0.00087,
    -0.00059,
    -0.00546,
    -0.0008,
    0.00082,
    -0.00556,
    -0.00139,
    0.00613,
    0.00209,
    0.00236,
    0.00182,
    -0.00081,
    3e-05,
    -0.00241,
    0.00322,
    0.00142,
    0.00467,
    -0.00144,
    -0.00274,
    0.00254,
    9e-05,
    0.07303,
    0.00257,
    -0.00051,
    0.00036,
    -0.00092,
    -0.00028,
    0.00247,
    0.00047,
    -0.00016,
    -0.00224,
    -0.00158,
    0.00071,
    -0.00125,
    0.00077,
    0.00069,
    9e-05,
    0.00106,
    0.00056,
    0.00094,
    -8e-05,
    0.00228,
    0.00578,
    -0.00188,
    -0.00085,
    0.00027,
    -0.00181,
    0.00368,
    0.0014,
    0.00018,
    0.00409,
    -0.00116,
    -0.00259,
    0.01012,
    -0.00393,
    -0.00208,
    -0.00027,
    0.00047,
    0.00704,
    -0.00076,
    -0.00182,
    0.00286,
    0.00434,
    -0.00535,
    0.00244,
    -0.00034,
    0.00723,
    -0.00062,
    -0.00254,
    -0.00045,
    0.00076,
    -0.00068,
    0.00096,
    0.00257,
    1e-05,
    -0.00318,
    -0.00253,
    -0.00044,
    -0.00047,
    -0.00041,
    0.00037,
    -0.0025,
    -0.00379,
    0.00169,
    -0.00073,
    -0.00154,
    -0.00034,
    -0.00902,
    -0.00031,
    -0.43178,
    0.00088,
    0.00147,
    0.00493,
    -0.00189,
    0.00049,
    -0.00107,
    0.00218,
    0.00138,
    -7e-05,
    0.0014,
    -0.00807,
    0.00042,
    -0.00082,
    -0.00071,
    0.00319,
    0.00055,
    -0.00109,
    0.00079,
    1e-05,
    -0.00068,
    0.00038,
    0.00184,
    0.00184,
    0.00421,
    0.00226,
    0.00031,
    0.00829,
    0.00077,
    0.08727,
    -0.0007,
    -0.0006,
    -0.00214,
    0.00147,
    0.00114,
    0.00056,
    -0.00048,
    0.00104,
    0.00391,
    -0.00087,
    -0.01131,
    -0.00035,
    -0.00047,
    -0.00164,
    -0.00204,
    -0.00238,
    0.00019,
    -0.00119,
    0.00039,
    -0.0009,
    -0.00127,
    0.00272,
    0.00024,
    0.00059,
    0.00018,
    0.00596,
    0.00089,
    -0.00035,
    0.00029,
    0.00912,
    -0.00151,
    0.00041,
    0.01074,
    0.00167,
    -0.00072,
    -6e-05,
    -0.0013,
    0.00357,
    0.00138,
    -0.0033,
    -0.00193,
    -0.00138,
    -0.00147,
    -6e-05,
    5e-05,
    0.00037,
    -0.00095,
    -0.00181,
    -0.00247,
    0.00011,
    0.0017,
    0.00231,
    0.00061,
    -0.00417,
    -9e-05,
    0.00097,
    0.00073,
    0.00109,
    -0.00304,
    0.00045,
    0.00482,
    -0.00115,
    -0.00203,
    -0.00067,
    0.0012,
    -0.00158,
    -0.00224,
    0.00052,
    0.00095,
    0.00374,
    -0.00056,
    0.00138,
    -0.00081,
    0.00147,
    0.00056,
    0.00048,
    -0.00182,
    -0.00086,
    0.00013,
    0.0005,
    -0.00758,
    -0.0015,
    0.00029,
    0.00178,
    0.00045,
    0.00131,
    -0.00058,
    -0.00224,
    0.00158,
    -0.00136,
    0.00023,
    0.00063,
    1e-05,
    -0.00049,
    -0.00455,
    6e-05,
    -0.00076,
    -0.00087,
    0.00877,
    0.00106,
    -8e-05,
    -0.00088,
    0.00058,
    0.0072,
    0.00033,
    -0.00078,
    0.00224,
    0.0005,
    2e-05,
    -1e-05,
    -0.0011,
    -0.00104,
    -0.00042,
    0.00016,
    0.0018,
    -0.00298,
    0.00032,
    0.00028,
    0.00086,
    -0.00133,
    -0.00077,
    -0.00063,
    0.00265,
    -0.01027,
    0.00446,
    0.00044,
    0.00102,
    0.00029,
    -0.0041,
    -0.0065,
    -0.00214,
    -0.00715,
    -0.00534,
    -0.00069,
    0.00726,
    -0.00648,
    0.00082,
    -0.00084,
    0.0012,
    -0.00327,
    -0.00077,
    0.00034,
    -0.00503,
    -0.00065,
    -0.002,
    -0.00041,
    0.00199,
    0.00213,
    0.00138,
    -0.00125,
    -0.00026,
    0.0007,
    0.00163,
    -0.00744,
    0.00016,
    0.0111,
    0.00131,
    0.00176,
    0.00022,
    -0.00225,
    0.00064,
    -0.00086,
    0.0043,
    -0.00183,
    -0.00484,
    -0.00118,
    5e-05,
    -0.00053,
    -0.00042,
    0.00421,
    -0.00108,
    0.00092,
    0.00139,
    -0.00195,
    0.00081,
    0.00065,
    7e-05,
    1e-05,
    -0.00136,
    0.00019,
    -0.00113,
    -5e-05,
    0.00441,
    -0.00118,
    -0.00119,
    0.00159,
    0.00062,
    0.00651,
    0.00043,
    -0.0,
    0.00393,
    -0.00849,
    -0.00063,
    -0.00145,
    0.0005,
    -0.00167,
    -0.01475,
    -0.00214,
    -0.00405,
    -0.00352,
    0.00305,
    -0.00053,
    -0.00184,
    -0.08902,
    0.00064,
    -6e-05,
    0.00118,
    -0.00416,
    0.00028,
    -0.00233,
    0.00087,
    0.00067,
    -0.00266,
    0.00373,
    -0.00228,
    0.00013,
    -0.00135,
    -0.0008,
    0.00073,
    -0.00052,
    0.00094,
    -0.00015,
    -0.00043,
    0.00047,
    -0.00025,
    -0.00073,
    -0.00356,
    -0.00227,
    -0.00312,
    0.00129,
    -0.01974,
    -0.00076,
    -0.00033,
    -0.00082,
    0.00038,
    -0.00254,
    0.00019,
    0.00012,
    0.0,
    -0.00021,
    -0.0016,
    -0.00056,
    -0.00395,
    -0.00065,
    -0.00976,
    0.0004,
    0.00036,
    0.0037,
    0.00083,
    -0.00873,
    0.00045,
    0.00144,
    0.00047,
    -0.00154,
    -0.00087,
    -0.0019,
    -0.00035,
    0.00065,
    0.00087,
    0.08106,
    -1e-05,
    0.00074,
    0.00263,
    0.00193,
    0.00089,
    -0.00277,
    -0.0001,
    -0.00236,
    6e-05,
    0.00097,
    0.00077,
    -0.00105,
    0.00199,
    -0.00236,
    -0.00021,
    -0.00301,
    0.00515,
    0.00044,
    -0.0333,
    -0.00264,
    -0.00144,
    0.00083,
    -0.00013,
    0.00112,
    4e-05,
    0.00145,
    0.00493,
    0.00078,
    0.00049,
    0.00011,
    -0.00049,
    0.00216,
    0.0018,
    0.00014,
    0.0006,
    0.00157,
    0.00228,
    0.00093,
    0.00206,
    -0.00128,
    -0.00215,
    0.00119,
    -0.03707,
    0.00023,
    -9e-05,
    0.00291,
    -0.00148,
    -0.00484,
    0.00707,
    0.00377,
    -0.00155,
    0.0006,
    -0.00012,
    0.00308,
    -0.00017,
    0.00143,
    0.00014,
    0.00232,
    0.00062,
    -7e-05,
    -0.00226,
    -0.0,
    0.00086,
    0.00192,
    0.00165,
    -0.00043,
    -0.00167,
    -0.00074,
    -0.00077,
    -0.00649,
    0.0014,
    -0.02729,
    -0.00024,
    -0.0042,
    -0.00059,
    4e-05,
    -0.00044,
    0.00077,
    0.00261,
    0.00105,
    -0.00079,
    -0.00058,
    -0.00025,
    0.00058,
    -0.00018,
    -0.00145,
    0.00036,
    -0.00313,
    9e-05,
    0.00043,
    0.00012,
    0.00033,
    -0.00094,
    0.0,
    -0.00125,
    0.00022,
    0.00262,
    0.00158,
    -0.00119,
    0.00025,
    0.00044,
    0.00083,
    -0.00074,
    -0.00099,
    -0.00282,
    -0.00024,
    -0.00049,
    0.00158,
    0.00039,
    0.00067,
    -0.00237,
    0.00056,
    0.00074,
    0.00115,
    -0.0012,
    -0.00109,
    1e-05,
    4e-05,
    0.00263,
    0.00154,
    0.00087,
    -0.00038,
    0.0035,
    -0.0008,
    -0.00042,
    -0.00067,
    -0.00162,
    -0.0018,
    -0.00299,
    0.00078,
    -0.01949,
    -0.0013,
    -0.00023,
    -0.00027,
    -0.00049,
    0.00123,
    8e-05,
    -0.00233,
    -0.00524,
    9e-05,
    0.00249,
    -0.00416,
    -0.00077,
    0.001,
    -0.00087,
    -0.00075,
    0.00086,
    0.0016,
    -0.00104,
    -0.0032,
    -0.0004,
    -0.00188,
    0.00039,
    0.00012,
    -0.00391,
    -0.00104,
    0.0013,
    0.00188,
    -0.00495,
    0.00122,
    0.0015,
    0.00019,
    0.00047,
    0.00022,
    -0.00087,
    0.01639,
    -0.00232,
    0.00698,
    -0.00074,
    -0.00339,
    -0.00317,
    -0.00071,
    -0.00107,
    -0.00077,
    -0.00041,
    0.00126,
    0.00147,
    0.00047,
    0.0005,
    -4e-05,
    -0.00112,
    -0.00185,
    -0.0111,
    -9e-05,
    0.00128,
    -0.00026,
    0.0015,
    0.00277,
    -0.00117,
    0.00107,
    -0.00098,
    0.0673,
    -0.00187,
    -0.00414,
    -0.03123,
    -0.00389,
    -0.00118,
    -0.00062,
    -0.00356,
    0.00019,
    0.00039,
    -0.00036,
    0.00028,
    -0.00154,
    -0.00229,
    -0.00402,
    0.00051,
    -0.00185,
    -0.00125,
    -0.00034,
    0.00025,
    -0.00169,
    -0.0,
    0.0014,
    0.00299,
    -0.00205,
    -0.00935,
    -0.00222,
    0.00081,
    0.00036,
    0.00061,
    0.00223,
    -0.00068,
    -0.00148,
    -0.00188,
    0.00062,
    -7e-05,
    -0.00291,
    -0.0007,
    0.00146,
    0.00136,
    -0.00014,
    -0.00794,
    0.00195,
    -4e-05,
    -0.00297,
    -0.00349,
    0.0007,
    -0.00641,
    0.001,
    -0.0004,
    -0.00734,
    0.0012,
    0.00047,
    0.00014,
    0.00249,
    0.00133,
    -0.00205,
    0.00118,
    -0.00127,
    -0.00331,
    -0.00056,
    -0.00036,
    0.00029,
    -0.001,
    -0.00194,
    0.00081,
    -0.00129,
    -0.00047,
    -0.00097,
    -0.00237,
    -0.00038,
    -0.00217,
    -0.00045,
    0.00044,
    0.00742,
    -0.00133,
    -0.0008,
    -0.00159,
    0.00114,
    -0.00235,
    -8e-05,
    0.00105,
    -0.00034,
    0.00146,
    -0.00034,
    -0.00181,
    -0.00288,
    0.00047,
    -0.00295,
    0.00012,
    0.00048,
    0.00799,
    0.00173,
    0.00168,
    -0.00213,
    -0.00031,
    -0.00164,
    -0.00043,
    0.00068,
    -0.00085,
    -0.00034,
    0.00533,
    -0.00141,
    0.01305,
    -0.00139,
    -0.00297,
    -0.00019,
    -0.00112,
    0.01305,
    -0.00105,
    -0.0014,
    -0.00861,
    -0.0021,
    0.00067,
    -0.00132
   ],
   "ON": [
    0.00157,
    -0.0011,
    0.00486,
    0.00175,
    0.00141,
    0.00453,
    0.0011,
    0.00372,
    -0.00196,
    0.0018,
    0.00057,
    -0.00869,
    -0.00442,
    -0.00192,
    0.00138,
    5e-05,
    -0.00321,
    -0.00179,
    0.00118,
    -0.00883,
    -0.00161,
    -0.00015,
    -0.00028,
    -0.00848,
    -0.00223,
    6e-05,
    0.00102,
    0.00678,
    0.01192,
    0.00044,
    0.0017,
    0.00145,
    4e-05,
    0.00337,
    -0.00473,
    0.00071,
    0.00187,
    -0.00334,
    0.01016,
    0.00502,
    -0.00203,
    0.00112,
    0.00243,
    -6e-05,
    -0.00444,
    -0.00264,
    0.00681,
    -0.00248,
    -0.00219,
    0.00117,
    0.00738,
    0.01035,
    0.00026,
    -0.00374,
    0.0011,
    0.00171,
    -0.00186,
    0.00039,
    -0.00775,
    -0.00581,
    -0.00016,
    0.0059,
    -0.00446,
    -0.00231,
    0.00203,
    0.00378,
    -0.00117,
    0.00232,
    0.00223,
    0.00477,
    -0.0017,
    0.00167,
    0.001,
    -0.00144,
    0.00328,
    0.00282,
    0.00189,
    -0.00127,
    0.00212,
    -0.00253,
    -0.00094,
    -0.0019,
    2e-05,
    -0.0003,
    0.00143,
    0.00225,
    0.00198,
    0.00091,
    -0.0019,
    0.0003,
    1e-05,
    0.00283,
    0.0024,
    -0.00092,
    0.00338,
    0.0042,
    -0.00274,
    -0.00165,
    0.00324,
    -0.00257,
    0.00055,
    0.00722,
    -0.01155,
    0.00103,
    -0.00603,
    0.00053,
    -0.00198,
    0.00139,
    0.00295,
    -0.00021,
    0.00309,
    0.00174,
    0.0007,
    0.00018,
    0.00097,
    9e-05,
    0.00204,
    -0.00039,
    0.00343,
    0.00075,
    -0.00446,
    -0.00826,
    0.00235,
    -0.00251,
    0.00095,
    -0.00013,
    -0.00088,
    -0.00314,
    0.00323,
    1e-05,
    -0.00038,
    4e-05,
    0.00179,
    0.00011,
    -0.00302,
    -0.00118,
    0.00242,
    -0.00034,
    0.0014,
    0.00082,
    -0.00169,
    -0.00018,
    0.00465,
    0.00777,
    -0.0014,
    -0.00338,
    -0.00304,
    -0.08914,
    -0.00078,
    4e-05,
    -0.0012,
    0.00148,
    0.00074,
    -0.00272,
    0.00838,
    0.00209,
    -0.00104,
    0.004,
    0.00018,
    -0.00198,
    -0.00619,
    -0.00552,
    0.0001,
    -0.00377,
    0.00435,
    -0.00427,
    0.00224,
    0.00525,
    0.00147,
    0.00458,
    0.00271,
    0.0024,
    -0.00133,
    0.00184,
    0.00454,
    -0.00194,
    0.00118,
    -0.00038,
    -0.00031,
    0.00271,
    0.00227,
    0.00416,
    -0.00241,
    0.00125,
    -0.00183,
    0.01563,
    0.00487,
    0.00131,
    -0.00384,
    0.00432,
    -0.00192,
    -0.00054,
    0.00045,
    0.00149,
    -8e-05,
    -0.00311,
    0.00193,
    -0.00076,
    0.00038,
    0.00146,
    0.00146,
    0.00299,
    -0.00048,
    0.00202,
    0.00148,
    -0.00156,
    -0.00125,
    0.00247,
    0.00389,
    0.0017,
    0.00188,
    -0.00227,
    0.00166,
    -0.00247,
    0.00114,
    0.00284,
    0.00277,
    -0.00037,
    0.00234,
    -0.00231,
    -0.00367,
    -0.0009,
    0.02002,
    -0.00872,
    -0.00083,
    -0.00495,
    0.00325,
    0.00066,
    -0.00786,
    0.00085,
    0.00219,
    0.00038,
    0.00088,
    0.00052,
    0.00037,
    0.0046,
    0.00022,
    -5e-05,
    -0.00039,
    -0.00057,
    -0.00067,
    -0.00194,
    -0.001,
    -0.00011,
    0.00205,
    -0.00049,
    -0.00223,
    0.00065,
    0.0028,
    2e-05,
    -1e-05,
    -0.02493,
    -0.0017,
    0.00317,
    0.00259,
    0.00828,
    -0.00411,
    -0.00051,
    -0.00054,
    0.00484,
    0.00037,
    0.00163,
    -0.00174,
    0.0066,
    -0.00132,
    -0.0035,
    -0.00301,
    -0.00111,
    -0.00071,
    -0.00405,
    0.00027,
    -0.00151,
    0.00138,
    -0.00033,
    0.00177,
    -0.00397,
    0.00162,
    -0.00058,
    -0.00298,
    0.00059,
    0.00095,
    0.00207,
    -0.00024,
    -0.00453,
    0.00015,
    0.00676,
    -0.1353,
    -0.00023,
    0.00279,
    0.00046,
    -0.00183,
    -0.00105,
    0.0011,
    -0.00197,
    0.00381,
    7e-05,
    -0.00114,
    -0.00032,
    -0.00105,
    -0.00182,
    -0.00353,
    -0.00392,
    0.00197,
    0.00114,
    0.00105,
    -0.00044,
    -0.00094,
    0.00088,
    -0.00171,
    -0.00561,
    -0.00127,
    -0.00025,
    -0.0046,
    -0.00433,
    0.00024,
    -2e-05,
    -0.00318,
    0.00114,
    -0.00028,
    -0.00225,
    -0.00191,
    0.00059,
    -0.0001,
    -0.00195,
    0.00035,
    0.00013,
    -0.00051,
    0.0023,
    0.00336,
    0.00127,
    0.00667,
    0.00292,
    0.00196,
    0.00096,
    -0.00012,
    -0.0119,
    0.00158,
    -0.00427,
    -0.00016,
    0.00122,
    0.00155,
    0.00204,
    0.0039,
    -0.00107,
    0.00252,
    -0.00224,
    0.00281,
    -0.00232,
    -0.00439,
    -0.00144,
    3e-05,
    0.00115,
    0.00987,
    -0.00037,
    0.03343,
    -0.00539,
    0.0041,
    -0.00022,
    0.00061,
    -0.00183,
    0.01017,
    -0.00164,
    0.00359,
    0.00067,
    -0.0028,
    -0.00258,
    -0.0005,
    0.00065,
    0.0032,
    -0.00053,
    0.00158,
    0.00062,
    0.00049,
    -0.00706,
    0.00055,
    5e-05,
    -0.00202,
    -0.00303,
    0.00242,
    -0.01228,
    0.00197,
    -0.00078,
    0.00247,
    -0.00116,
    -0.00165,
    0.00082,
    -0.00033,
    -6e-05,
    -0.00114,
    -0.00212,
    -0.002,
    -0.00186,
    -0.00062,
    -0.00015,
    0.0012,
    0.0002,
    0.00312,
    0.02122,
    -0.00018,
    -0.0003,
    0.00909,
    8e-05,
    -0.00052,
    0.00188,
    0.00179,
    -0.00879,
    -0.00245,
    4e-05,
    -0.59497,
    -0.00318,
    0.00639,
    0.00192,
    0.00209,
    -0.0011,
    0.00466,
    -0.00686,
    -0.00075,
    -0.00138,
    -0.01056,
    -0.00283,
    0.00262,
    0.00032,
    0.00158,
    0.00519,
    -0.00576,
    0.00305,
    -0.00148,
    -0.00262,
    -0.00125,
    0.00045,
    0.00342,
    -0.0015,
    -0.00225,
    -0.00137,
    0.00108,
    0.00119,
    -0.00844,
    0.00449,
    -0.00045,
    -0.00154,
    -0.00217,
    -0.00892,
    -0.00207,
    0.00114,
    -0.00035,
    0.0002,
    0.00491,
    -0.00196,
    0.00224,
    -0.00046,
    0.00017,
    -7e-05,
    0.00324,
    -0.00166,
    -0.00153,
    -0.00124,
    -0.00143,
    0.00167,
    0.00766,
    0.00138,
    0.00444,
    0.00119,
    0.00295,
    -0.00117,
    -0.0005,
    -0.00093,
    0.00669,
    0.00022,
    -0.00168,
    -0.0014,
    -0.00322,
    -0.0043,
    -0.00073,
    0.00039,
    0.0014,
    -0.00216,
    -0.00182,
    -0.00244,
    -0.0052,
    0.00247,
    0.0009,
    -0.00397,
    0.00179,
    -0.00079,
    0.00182,
    -0.00369,
    -0.00027,
    -0.004,
    -0.00147,
    -0.00059,
    0.00112,
    0.0023,
    0.00251,
    -0.00108,
    0.00229,
    0.00096,
    0.00037,
    -0.00137,
    0.00178,
    -0.00175,
    0.00717,
    0.00073,
    -0.00126,
    0.0212,
    -0.00057,
    -0.00094,
    -0.00028,
    0.00048,
    -0.00648,
    -0.00065,
    0.01414,
    0.00069,
    0.00032,
    0.00099,
    0.00463,
    0.00013,
    -0.00036,
    -0.00192,
    -0.03261,
    0.0005,
    0.47376,
    -0.00159,
    -0.00105,
    0.00114,
    0.00289,
    -0.0054,
    0.00569,
    0.00034,
    0.00362,
    -0.0025,
    0.00028,
    0.00541,
    -0.00087,
    0.00293,
    -0.00079,
    0.00043,
    -0.00358,
    -0.00218,
    -0.00313,
    0.01214,
    0.00483,
    0.00177,
    -0.00346,
    0.0014,
    -0.00782,
    0.00332,
    -0.00519,
    -0.00124,
    -0.00394,
    -0.00308,
    -0.0047,
    -0.00143,
    0.00101,
    -0.10835,
    0.00413,
    0.00463,
    0.00176,
    -0.00164,
    -0.00024,
    -0.00225,
    0.00045,
    0.00114,
    9e-05,
    -0.00207,
    0.00035,
    0.00199,
    -0.00619,
    0.00423,
    0.00022,
    -0.00093,
    0.00368,
    0.00061,
    -0.0019,
    -0.00459,
    0.00307,
    0.00394,
    -0.00017,
    0.00133,
    -0.00734,
    -0.00258,
    0.00053,
    -0.00035,
    -0.0167,
    -0.00022,
    0.00046,
    -0.00252,
    -0.00496,
    0.00052,
    -0.00037,
    -0.01587,
    -0.00061,
    -0.00129,
    -0.00343,
    -0.00606,
    0.00064,
    0.00227,
    -0.003,
    -0.00145,
    -0.00071,
    -0.00418,
    0.00133,
    3e-05,
    -0.00265,
    -0.00669,
    0.00262,
    -0.00085,
    0.0036,
    0.00012,
    0.00094,
    0.36222,
    -0.00182,
    0.00049,
    0.00522,
    0.00936,
    0.0041,
    0.00184,
    0.00062,
    1e-05,
    0.00709,
    0.00143,
    -0.0014,
    0.00096,
    -0.00237,
    0.00426,
    0.00034,
    -0.00037,
    0.00314,
    0.00201,
    -0.00069,
    -0.00161,
    0.00055,
    0.00134,
    0.00364,
    0.00102,
    0.00066,
    1e-05,
    -0.00036,
    0.00201,
    0.0009,
    0.00267,
    0.00075,
    0.00181,
    -0.00215,
    -0.00719,
    -0.00574,
    0.00267,
    0.00016,
    -0.00156,
    0.00048,
    -0.00424,
    -0.00149,
    0.00079,
    0.00039,
    0.0017,
    -0.009,
    0.00186,
    -0.00129,
    0.00241,
    -0.0034,
    5e-05,
    0.00068,
    0.01181,
    0.00748,
    -0.00027,
    -0.00796,
    0.00112,
    -0.00039,
    0.00079,
    0.00263,
    0.00182,
    -0.00114,
    0.00237,
    -0.0041,
    -0.0007,
    -0.00025,
    -0.00622,
    0.01121,
    3e-05,
    0.00249,
    -0.00212,
    -0.00247,
    -0.00018,
    0.00025,
    0.00188,
    -0.00182,
    -0.00068,
    -0.00281,
    0.00195,
    0.00027,
    0.00183,
    0.00145,
    0.00062,
    0.003,
    0.00184,
    -0.00092,
    0.00178,
    -0.00039,
    0.00348,
    -0.0008,
    0.00058,
    -0.00011,
    5e-05,
    -0.00062,
    -0.0006,
    -0.00275,
    -0.00277,
    -0.00574,
    0.00111,
    0.00209,
    -0.00176,
    -0.00418,
    0.00206,
    0.00068,
    -0.09191,
    -0.00394,
    -0.00129,
    0.00091,
    0.00011,
    -0.00971,
    0.00106,
    0.00035,
    -0.00563,
    -0.00051,
    0.01336,
    5e-05,
    0.00111,
    0.00197,
    0.0034,
    0.00189,
    0.00432,
    -0.0045,
    -0.00038,
    0.00126,
    0.00525,
    -0.00077,
    0.01011,
    0.0005,
    0.00285,
    -1e-05,
    0.00208,
    0.00337,
    -0.00021,
    0.00717,
    0.00407,
    -0.00127,
    -0.0015,
    0.00245,
    0.00058,
    -0.00224,
    0.00074,
    -0.01534,
    0.00143,
    0.00059,
    -0.00066,
    -0.00472,
    -0.00226,
    0.00215,
    -0.00417,
    0.00336,
    -0.00254,
    0.00072,
    -0.0021,
    0.00165,
    -0.00295,
    0.00254,
    0.0001,
    0.00231,
    -0.00131,
    0.00071,
    -0.00028,
    0.00898,
    0.0007,
    0.00026,
    0.00785,
    -0.00258,
    0.00338,
    0.00073,
    0.00335,
    0.0005,
    0.00214,
    0.00857,
    -0.00118,
    -0.00063,
    0.0008,
    0.00386,
    0.00422,
    0.00096,
    -0.00078,
    0.00017,
    0.00168,
    -0.00181,
    -0.00012,
    0.00187,
    -0.00151,
    -0.00159,
    0.00045,
    0.00389,
    0.00221,
    0.00139,
    0.00194,
    -0.00126,
    -0.00134,
    0.0355,
    0.00251,
    0.00298,
    0.0001,
    -0.00198,
    0.00066,
    -0.00528,
    0.00051,
    0.00117,
    -0.00116,
    0.00346,
    -0.0119,
    0.00335,
    -0.00228,
    0.00032,
    0.00323,
    -0.00095,
    0.00249,
    0.00035,
    -0.00112,
    -0.00137,
    0.00181,
    0.00168,
    0.00096,
    0.00581,
    -0.00551,
    0.00257,
    0.01508,
    0.00251,
    0.0055,
    -0.00649,
    -0.00415,
    -0.00161,
    -0.00044,
    -0.00239,
    0.00263,
    0.00079,
    0.00176,
    -0.00029,
    0.00405,
    -0.0006,
    0.00024,
    0.0091,
    0.0019,
    0.00121,
    0.0059,
    -0.00251,
    -0.0038,
    0.00012,
    -0.00099,
    -0.00132,
    0.008,
    -0.00371,
    0.00011,
    0.00206,
    0.00105,
    -0.00793,
    0.00027,
    0.00037,
    0.00177,
    0.00138,
    0.00088,
    0.00106,
    -0.00835,
    -1e-05,
    -0.00675,
    -0.00155,
    -0.00077,
    -0.00124,
    -0.00218,
    -0.07957,
    0.00302,
    0.00086,
    -0.00082,
    -0.00076,
    -0.00046,
    0.00053,
    -0.00155,
    0.00168,
    -0.00033,
    -0.00103,
    0.00054,
    -0.00035,
    -0.00234,
    -0.00161,
    0.00311,
    -0.00134,
    -0.00479,
    0.00149,
    -0.00227,
    -0.00349,
    0.00329,
    -0.00138,
    -0.00034,
    -0.00167,
    0.00292,
    0.00016,
    -0.00785,
    -0.00015,
    -0.01225,
    -0.00421,
    0.00327,
    -0.01049,
    0.00477,
    -0.0034,
    -0.0004,
    -0.00341,
    -0.00813,
    0.0025,
    0.00248,
    -0.00343,
    -0.00305,
    0.00167,
    0.0008,
    0.00126,
    -0.00645,
    0.00031,
    0.0006,
    -0.00026,
    0.00099,
    0.00116,
    -0.00018,
    0.00322,
    0.00123,
    0.00596,
    0.00204,
    0.00049,
    -0.00428,
    0.00036,
    0.00132,
    0.00051,
    0.00135,
    0.00115,
    0.00091,
    -0.00362,
    -0.0002,
    0.00214,
    -0.00127,
    0.42512,
    2e-05,
    7e-05,
    -0.003,
    -0.00158,
    -0.00042,
    0.00074,
    -0.00057,
    0.00131,
    -0.00244,
    0.00014,
    -0.0052,
    -0.00063,
    -0.00187,
    -0.003,
    -0.00096,
    -0.00098,
    -0.00461,
    0.00482,
    -0.00053,
    -0.00021,
    -0.00165,
    -0.00362,
    -0.00066,
    0.0014,
    -0.00556,
    -0.00091,
    -0.01038,
    0.00293,
    -0.07364,
    0.00046,
    0.0002,
    0.00118,
    -0.00331,
    0.00014,
    0.00492,
    0.00211,
    0.00118,
    -0.00659,
    -0.00276,
    0.00049,
    -0.00226,
    -0.00376,
    0.00535,
    -0.00412,
    0.00163,
    0.00339,
    -0.00035,
    0.00016,
    0.00627,
    0.0018,
    0.0001,
    0.00012,
    -0.00057,
    0.0013,
    -0.00763,
    0.00074,
    -0.00025,
    -0.00067,
    -0.01018,
    0.0004,
    0.00231,
    -0.01308,
    -0.00183,
    -0.00219,
    -0.00272,
    0.0011,
    -0.00194,
    -0.00034,
    0.00562,
    -0.00772,
    -0.00113,
    0.00208,
    0.00379,
    -0.00156,
    0.00379,
    0.00084,
    0.00247,
    0.00256,
    0.00316,
    -0.00011,
    0.00201,
    -1e-05,
    -0.00366,
    -0.00067,
    -0.00178,
    0.00042,
    0.00024,
    0.00672,
    -0.00214,
    0.00375,
    0.00332,
    0.00444,
    -0.00153,
    -0.00157,
    0.00382,
    0.00098,
    0.00198,
    -0.00388,
    -0.00237,
    0.00044,
    -0.00196,
    0.0031,
    0.00333,
    -0.00068,
    0.00024,
    0.00029,
    0.0001,
    -0.00018,
    0.0005,
    0.00626,
    -0.00727,
    -0.00097,
    0.00012,
    5e-05,
    0.00055,
    0.00235,
    0.00039,
    0.00077,
    0.00117,
    0.00034,
    0.00093,
    0.00335,
    0.00213,
    0.00449,
    2e-05,
    -0.00151,
    -0.00121,
    -0.01083,
    7e-05,
    0.00341,
    0.00186,
    -0.00056,
    -0.0068,
    -0.00266,
    -0.00142,
    0.00365,
    0.00028,
    -0.0011,
    -0.00145,
    -0.00059,
    -0.00033,
    0.00052,
    -5e-05,
    0.0,
    0.00194,
    0.00019,
    -0.0045,
    0.00038,
    0.00249,
    -0.00349,
    -0.0032,
    -0.00342,
    0.00787,
    -0.00173,
    -0.0031,
    0.00258,
    -0.00097,
    -0.00382,
    0.01097,
    -0.00024,
    -0.00145,
    -0.00327,
    0.00192,
    -0.00445,
    0.00538,
    -0.00275,
    0.00081,
    -0.00091,
    0.00299,
    -0.00052,
    0.00283,
    0.00353,
    0.00226,
    0.00128,
    -0.00158,
    -0.00276,
    -0.00687,
    0.0002,
    0.00093,
    0.00473,
    0.00021,
    -0.00068,
    -0.00263,
    0.00281,
    -0.00258,
    0.00046,
    0.00317,
    -0.00245,
    0.00033,
    -0.00064,
    0.00092,
    -0.00123,
    -0.00062,
    -0.00255,
    -2e-05,
    0.00191,
    -0.00026,
    0.00082,
    0.00833,
    0.00019,
    0.00026,
    0.00175,
    0.00476,
    -0.0019,
    0.00155,
    0.00463,
    -0.00199,
    0.00075,
    0.00301,
    0.00246,
    -0.00433,
    0.00152,
    0.00673,
    0.00384,
    0.00517,
    -7e-05,
    -0.00106,
    -0.00032,
    0.0014,
    -0.00672,
    0.00342,
    0.0015,
    -0.0031,
    0.0017,
    -0.00018,
    0.02228,
    0.00053,
    0.01082,
    -0.00057,
    -0.00015,
    -0.00086,
    -0.00119,
    0.09565,
    0.00034,
    -0.00029,
    -0.00201,
    0.00417,
    0.00065,
    0.00287,
    -0.00089,
    -0.00011,
    0.01401,
    0.00505,
    -0.00274,
    -0.00035,
    -0.00195,
    -0.00085,
    -0.00067,
    0.00165,
    -0.00037,
    0.00164,
    -0.00356,
    -0.00112,
    -0.00435,
    0.00054,
    0.00535,
    0.00175,
    0.00699,
    0.00388,
    0.02007,
    0.00183,
    -0.00477,
    0.00289,
    0.00246,
    -0.00177,
    0.00105,
    0.00102,
    -0.00078,
    0.00182,
    0.00079,
    -0.00172,
    -0.00469,
    -0.00273,
    -0.00257,
    -0.00782,
    0.00042,
    -0.00972,
    -3e-05,
    0.00588,
    -0.00041,
    -0.00018,
    0.00746,
    0.00157,
    -0.00121,
    -0.00101,
    0.00231,
    0.00529,
    -0.00095,
    -0.09275,
    0.00068,
    0.00065,
    0.0007,
    0.00381,
    0.00245,
    0.00222,
    0.00048,
    -0.00143,
    -0.00033,
    0.00019,
    -0.00131,
    0.00155,
    -0.00179,
    -0.00616,
    -0.00013,
    -0.00041,
    -0.00329,
    -1e-05,
    0.03056,
    -0.00354,
    0.00058,
    -0.01006,
    -0.0008,
    0.00102,
    -0.00069,
    -0.00776,
    -0.00265,
    0.00292,
    0.00027,
    -0.00434,
    0.00415,
    -0.00503,
    -0.00902,
    -0.0007,
    -0.00142,
    -0.00333,
    -0.00561,
    -0.00275,
    0.00596,
    0.00083,
    0.00363,
    -0.0028,
    0.03883,
    0.00246,
    0.00054,
    -0.0019,
    -0.00119,
    0.00676,
    -0.00499,
    -0.00458,
    0.00445,
    -0.00199,
    -0.00239,
    -0.00319,
    0.00133,
    -0.00085,
    -0.00422,
    -0.0014,
    0.00087,
    -0.00236,
    0.00143,
    -0.00025,
    -0.00148,
    0.00354,
    -0.00017,
    -0.00416,
    -0.00255,
    0.0003,
    -0.00191,
    -0.00083,
    0.0003,
    0.01908,
    -0.00025,
    -0.00331,
    -0.00211,
    0.00082,
    0.00061,
    1e-05,
    -0.00949,
    -0.0016,
    -0.00113,
    -0.00097,
    0.00083,
    0.0024,
    -0.00159,
    0.00192,
    -0.00046,
    -0.00386,
    0.00338,
    -0.00164,
    -0.00469,
    -0.00026,
    0.00417,
    -0.00135,
    0.00501,
    -0.00209,
    -0.00102,
    -0.00075,
    0.00347,
    -0.00068,
    -0.00128,
    0.00295,
    0.00195,
    0.00046,
    0.00387,
    0.00198,
    0.00102,
    0.00025,
    -0.00174,
    -2e-05,
    -0.0111,
    0.0002,
    -0.00287,
    0.00508,
    0.00128,
    0.00318,
    -0.00091,
    0.00243,
    -0.00372,
    -0.00056,
    -0.00112,
    -0.0015,
    -0.00691,
    -0.00195,
    -0.00102,
    -9e-05,
    0.00037,
    -0.00193,
    -0.00014,
    0.00034,
    0.01475,
    0.00077,
    6e-05,
    0.00101,
    0.00066,
    -0.00083,
    0.00075,
    0.00021,
    0.00516,
    -0.00263,
    0.00059,
    0.00229,
    0.00221,
    -0.00265,
    0.00287,
    0.0005,
    0.00127,
    3e-05,
    0.00018,
    -0.00141,
    0.00489,
    -0.00131,
    8e-05,
    0.00474,
    0.00442,
    -0.00142,
    0.0004,
    -0.00332,
    0.01677,
    -0.0006,
    0.00322,
    0.00173,
    0.00153,
    -0.00097,
    -0.00036,
    -0.01768,
    0.00347,
    -0.00509,
    0.00016,
    -0.00158,
    0.0037,
    -0.00117,
    -0.003,
    0.00399,
    -0.00114,
    0.00039,
    -0.00106,
    0.00199,
    -0.00097,
    -0.0014,
    -0.00175,
    0.00147,
    -0.00052,
    -0.00162,
    0.00344,
    0.0026,
    0.00161,
    -0.00181,
    -4e-05,
    -0.00304,
    -0.0016,
    -0.0743,
    0.00232,
    0.00456,
    0.02875,
    0.00445,
    0.00185,
    0.00241,
    0.00265,
    0.00177,
    0.00092,
    0.00107,
    -0.00539,
    0.00206,
    -0.0015,
    -0.00607,
    0.00161,
    0.00026,
    0.00171,
    0.00087,
    0.00389,
    -0.00258,
    0.01286,
    -0.00319,
    -0.00332,
    0.00606,
    0.00997,
    0.00199,
    0.00242,
    0.00168,
    -0.00235,
    -0.00619,
    -0.00196,
    -0.00306,
    0.00283,
    -0.00073,
    3e-05,
    -0.00318,
    -0.00526,
    -0.00019,
    -0.00085,
    -0.00083,
    0.00387,
    -0.00051,
    0.00054,
    0.00602,
    0.00255,
    -0.0003,
    -0.00771,
    -0.00011,
    -0.0003,
    0.00524,
    -0.00111,
    0.00034,
    -0.00216,
    0.00215,
    0.00153,
    0.0008,
    -0.00071,
    0.00462,
    -0.007,
    -0.00132,
    0.00023,
    -0.00054,
    -0.00489,
    0.00153,
    0.00029,
    0.00504,
    0.00148,
    -0.00163,
    7e-05,
    0.00079,
    0.00358,
    -0.00211,
    -0.00133,
    -0.00462,
    0.00104,
    -0.0014,
    0.00208,
    0.00017,
    0.00387,
    -0.00055,
    -9e-05,
    -0.0016,
    0.00106,
    0.003,
    -0.00084,
    -0.00012,
    -0.00037,
    -0.00173,
    0.00177,
    -0.00041,
    -0.00541,
    0.00178,
    0.00031,
    -0.00198,
    -0.00417,
    -0.00167,
    0.00255,
    -0.0002,
    -0.00037,
    -0.00071,
    0.01027,
    0.00046,
    0.001,
    0.00184,
    0.00104,
    0.00308,
    0.00743,
    -0.0018,
    0.00162,
    -0.0017,
    0.00231,
    -0.00066,
    -0.00423,
    -7e-05
   ],
   "MYSTIC": [
    -0.02953,
    -0.00119,
    0.00721,
    0.02132,
    -0.00902,
    0.02097,
    0.00786,
    -0.00376,
    -0.00666,
    -0.00509,
    -0.00388,
    -0.05778,
    -0.00196,
    -0.00016,
    -0.00832,
    -0.01534,
    -0.0067,
    0.00169,
    0.01155,
    -0.04083,
    -0.01485,
    0.01085,
    -0.01249,
    -0.10597,
    0.00305,
    0.01151,
    0.01302,
    -0.0393,
    0.00784,
    0.01662,
    -0.00836,
    0.01202,
    0.0182,
    0.00621,
    -0.01511,
    -0.00673,
    0.00928,
    -0.00164,
    -0.06486,
    0.00179,
    0.00034,
    -0.00191,
    -0.00823,
    0.0028,
    0.02091,
    -0.0035,
    0.01838,
    -0.00118,
    0.00112,
    0.0147,
    0.0175,
    -0.00973,
    -0.01142,
    0.00592,
    0.00343,
    0.00415,
    -0.00716,
    0.05742,
    -0.0667,
    0.02062,
    -0.01398,
    -0.04327,
    -0.00017,
    -0.01569,
    0.00381,
    -0.03848,
    -0.00512,
    0.05408,
    -0.00173,
    0.00277,
    -0.01341,
    0.00598,
    0.02164,
    0.01328,
    0.00583,
    -0.02443,
    0.00515,
    0.00406,
    -0.00655,
    0.01629,
    -0.01073,
    0.00812,
    -0.02281,
    -0.00479,
    -0.00399,
    0.01219,
    0.00455,
    0.00593,
    -0.01432,
    -0.02377,
    0.00353,
    -0.01001,
    -0.00015,
    -0.03182,
    -0.01948,
    -0.00299,
    -0.0129,
    -0.00232,
    0.00235,
    -0.01586,
    -0.00308,
    0.03896,
    0.01745,
    0.0015,
    -0.00503,
    0.01266,
    0.00338,
    -0.00771,
    -4e-05,
    -0.00659,
    0.04162,
    -0.01016,
    0.01809,
    -0.00641,
    0.0019,
    -0.00953,
    -0.01052,
    0.01039,
    -0.03672,
    0.04945,
    -0.00219,
    -0.01201,
    0.02288,
    0.02739,
    -0.00481,
    0.00316,
    0.00058,
    0.04626,
    -0.01248,
    0.00408,
    -0.00343,
    -0.01741,
    -0.01598,
    0.00117,
    -0.0199,
    0.00869,
    0.04124,
    -0.00157,
    -0.03507,
    -0.00308,
    0.09064,
    -0.02905,
    -0.01002,
    -0.04163,
    -0.00201,
    -0.02922,
    -0.01686,
    0.04575,
    0.00568,
    -0.0001,
    -0.0014,
    -0.01675,
    -0.10224,
    0.00648,
    -0.01032,
    0.00281,
    -0.00766,
    -0.00856,
    0.00873,
    -0.00107,
    -0.03469,
    -0.01657,
    0.00916,
    0.04211,
    0.01722,
    -0.00121,
    -0.00015,
    -0.00118,
    0.01878,
    0.00377,
    0.00199,
    -0.01065,
    0.00475,
    0.00318,
    -0.00511,
    0.00882,
    -0.01613,
    0.01119,
    -0.00091,
    0.01267,
    0.01483,
    -0.01415,
    -0.01809,
    0.005,
    -0.01365,
    -0.03732,
    0.00307,
    0.01273,
    -0.00051,
    0.00204,
    0.00271,
    0.01333,
    -0.00928,
    0.00058,
    -0.00353,
    -0.0079,
    0.03163,
    -0.00482,
    0.0082,
    -0.02222,
    0.01795,
    -0.00075,
    0.00729,
    0.01604,
    -0.00319,
    0.01147,
    0.01263,
    0.00382,
    -0.01801,
    -0.0146,
    0.007,
    0.00134,
    0.00861,
    0.00702,
    -0.00622,
    -0.0047,
    0.00684,
    -0.00514,
    0.00492,
    0.00441,
    -0.00426,
    -0.01984,
    -0.01439,
    0.02366,
    -0.01672,
    0.01866,
    -0.02427,
    0.00872,
    0.01461,
    0.0052,
    -0.00384,
    -0.01093,
    -0.0068,
    -0.00411,
    -0.0169,
    -0.02414,
    -0.0168,
    -0.02622,
    -0.00172,
    -0.00652,
    -0.00522,
    0.01338,
    0.01018,
    -0.00405,
    -0.01451,
    0.0034,
    -0.00519,
    0.01445,
    0.00569,
    -0.00384,
    -0.01241,
    0.00098,
    -0.00215,
    -0.0031,
    0.09137,
    -0.00533,
    -0.01285,
    0.00989,
    -0.00659,
    0.00499,
    -0.00808,
    0.00607,
    0.01061,
    0.03485,
    0.01383,
    -0.00533,
    -0.01645,
    -0.0072,
    0.00502,
    -0.02041,
    -0.00024,
    0.03626,
    0.02048,
    0.01898,
    -0.00743,
    0.03665,
    0.08456,
    -0.00213,
    -0.0172,
    0.01324,
    -0.00236,
    0.00875,
    -0.01379,
    -0.01439,
    -0.00517,
    -0.00256,
    -0.07746,
    0.00382,
    -0.01173,
    -0.0259,
    0.00637,
    0.00474,
    0.0019,
    -0.00172,
    -0.01769,
    -0.00602,
    -0.01413,
    -0.01617,
    -0.0121,
    -0.00353,
    0.01691,
    -0.02297,
    0.00729,
    0.03173,
    0.00181,
    0.00836,
    -0.01232,
    -0.0177,
    -0.02132,
    0.01321,
    -0.0131,
    -0.00538,
    -0.02508,
    0.01082,
    0.00481,
    -0.00354,
    0.03458,
    -0.0102,
    -0.02424,
    -0.05615,
    0.00678,
    0.0049,
    -0.00306,
    0.02149,
    -0.02909,
    0.0174,
    0.00408,
    0.03272,
    0.00979,
    0.00121,
    0.03392,
    -0.00317,
    0.05239,
    -0.01622,
    0.0301,
    0.05043,
    0.01405,
    -0.03962,
    0.0154,
    0.08321,
    -0.02602,
    0.01619,
    -0.01393,
    -0.01168,
    0.00445,
    -0.00875,
    -0.00655,
    -0.0279,
    -0.00156,
    0.01432,
    0.00641,
    -0.01561,
    0.02705,
    -0.00816,
    -0.02008,
    0.00053,
    -0.02187,
    0.00118,
    -0.00715,
    0.03913,
    -0.0479,
    -0.02792,
    0.00609,
    -0.02788,
    0.01368,
    0.02145,
    -0.00676,
    0.01571,
    0.02473,
    -0.002,
    -0.00792,
    0.04057,
    -0.01076,
    -0.02285,
    0.00485,
    0.009,
    0.01104,
    -0.00853,
    0.0201,
    -0.05877,
    -0.0026,
    0.00067,
    0.00361,
    -0.0,
    0.0563,
    0.01896,
    0.01847,
    -0.00121,
    -0.01817,
    -0.00022,
    -0.02356,
    -0.02092,
    0.01656,
    0.00877,
    0.0053,
    -0.04268,
    0.01954,
    0.03377,
    -0.00088,
    0.00896,
    -0.00743,
    0.00328,
    -0.02298,
    -0.02336,
    0.00939,
    -0.04501,
    0.01556,
    0.00266,
    0.40566,
    0.09137,
    -0.0103,
    -0.02975,
    -0.00045,
    0.00675,
    0.0039,
    -0.00343,
    0.00895,
    0.00505,
    -0.01276,
    0.00091,
    0.00309,
    -0.00837,
    0.00283,
    0.01782,
    0.01497,
    0.00368,
    -0.0481,
    0.00972,
    -0.00044,
    -0.00996,
    0.00973,
    -0.01158,
    -0.00351,
    -0.00885,
    0.04502,
    0.00564,
    0.03531,
    0.01821,
    0.0088,
    -0.00422,
    -0.00197,
    -0.00011,
    -0.03373,
    -0.00766,
    -0.01142,
    -0.00391,
    0.02059,
    0.00678,
    -0.01077,
    0.01392,
    -0.01372,
    -0.00061,
    -0.00154,
    0.00335,
    -0.0096,
    -0.00806,
    0.00609,
    -0.00943,
    0.07044,
    0.01338,
    -0.01303,
    0.00403,
    -0.00275,
    -0.03561,
    0.0357,
    0.0074,
    0.03397,
    -0.00217,
    0.00757,
    0.00111,
    0.00113,
    0.02926,
    0.02179,
    -0.01464,
    0.00492,
    0.00581,
    -0.00545,
    0.00737,
    0.01362,
    -0.00877,
    -0.01736,
    0.00639,
    0.00843,
    -0.01083,
    0.0063,
    0.00852,
    0.00139,
    0.01044,
    -0.00996,
    0.0141,
    -0.00467,
    -0.00189,
    -0.00636,
    0.01678,
    0.00158,
    0.01668,
    0.01856,
    0.00413,
    0.01782,
    0.03224,
    -0.06684,
    0.00218,
    -0.02179,
    0.01374,
    -0.00893,
    0.04336,
    0.00446,
    0.012,
    -0.00578,
    -0.00827,
    -0.02654,
    0.00147,
    -0.00579,
    0.00446,
    0.05526,
    0.00084,
    0.00601,
    0.01447,
    -0.02155,
    0.0044,
    -0.24484,
    0.00885,
    -0.00482,
    0.01187,
    -0.00043,
    -0.02379,
    -0.00375,
    0.01164,
    0.00827,
    0.00498,
    -0.02149,
    0.0054,
    0.07469,
    0.00471,
    0.0138,
    -0.01451,
    -0.0043,
    0.00174,
    0.02833,
    -0.00773,
    -0.00412,
    0.01232,
    -0.00238,
    0.00655,
    -0.05058,
    -0.00171,
    0.01097,
    -0.00754,
    0.02847,
    0.00153,
    -0.02178,
    0.00023,
    0.00823,
    0.09142,
    0.00844,
    0.01368,
    0.00206,
    -0.00586,
    -0.00687,
    0.03026,
    -0.00209,
    -0.00811,
    -0.0054,
    -0.00698,
    0.00787,
    0.01459,
    -0.0146,
    -0.00147,
    -0.01402,
    0.01213,
    -0.02932,
    0.00929,
    -0.01412,
    0.05278,
    -0.01999,
    0.00038,
    -0.00215,
    0.01136,
    -0.09071,
    0.00746,
    -0.03512,
    -0.0029,
    -0.01102,
    0.00023,
    0.00532,
    -0.05957,
    -0.01838,
    0.0013,
    -0.00804,
    -0.03736,
    -0.00401,
    0.00715,
    -0.007,
    0.02312,
    0.0068,
    0.00499,
    0.00258,
    -0.00651,
    -0.00447,
    -0.00987,
    -0.01284,
    -0.00677,
    0.00646,
    0.06002,
    0.0006,
    -0.0223,
    0.00115,
    -0.01977,
    -0.00472,
    -0.17512,
    0.0009,
    0.01063,
    0.00792,
    0.02843,
    -0.01343,
    0.00767,
    -0.0003,
    0.00747,
    0.00111,
    0.01528,
    -0.01472,
    0.02149,
    -0.0024,
    0.02202,
    0.01277,
    -0.0072,
    -0.00618,
    0.0081,
    -0.01124,
    0.00731,
    -0.00808,
    -0.00074,
    -0.01702,
    -0.00539,
    -0.00928,
    -0.01253,
    0.00386,
    -0.02984,
    -0.01033,
    0.04233,
    -0.01327,
    0.00354,
    -0.00348,
    -0.00328,
    -0.01667,
    -0.00073,
    0.01339,
    -0.01416,
    -0.0421,
    0.01931,
    -0.03987,
    0.02603,
    -0.01911,
    0.00178,
    -0.02915,
    -0.00525,
    0.01458,
    -0.01455,
    0.00897,
    0.00128,
    -0.00409,
    -0.02621,
    -0.00743,
    -0.01907,
    -0.01546,
    -0.00194,
    -0.00476,
    0.00084,
    -0.01623,
    -0.00718,
    0.00417,
    0.03545,
    0.00774,
    0.00427,
    -0.01542,
    -0.00139,
    -0.04931,
    0.00357,
    0.00583,
    0.02551,
    -0.02148,
    -0.01219,
    0.00556,
    0.00291,
    -0.01091,
    0.00214,
    0.01313,
    0.01125,
    0.00111,
    -0.0001,
    0.00714,
    -0.00032,
    -0.01203,
    -0.00162,
    0.01544,
    -0.00231,
    -0.00052,
    -0.01879,
    -0.00419,
    0.01033,
    0.01995,
    0.00349,
    0.02334,
    -0.00041,
    -0.02868,
    -0.01426,
    -0.05747,
    -0.00158,
    -0.00662,
    -0.02365,
    -0.03479,
    -0.0733,
    -0.00721,
    0.07015,
    -0.00078,
    0.00832,
    0.00809,
    0.01517,
    0.00188,
    -0.00668,
    -0.00132,
    -0.02069,
    0.00201,
    -0.00222,
    0.00075,
    0.01183,
    0.00169,
    0.00283,
    -0.00473,
    0.00538,
    -0.01524,
    -0.00133,
    0.00547,
    -0.02736,
    0.00216,
    0.02631,
    -0.01093,
    -0.0083,
    0.00192,
    -0.00699,
    -0.00963,
    0.00771,
    -0.00162,
    -0.02586,
    0.00099,
    0.00846,
    -0.01729,
    -0.01013,
    0.00538,
    -0.00119,
    -0.02183,
    -0.02306,
    0.02083,
    -0.08382,
    -0.02132,
    -0.00978,
    0.00871,
    0.02876,
    0.05158,
    0.00281,
    -0.01486,
    -0.00714,
    -0.02268,
    0.02209,
    -0.00814,
    0.00615,
    -0.00698,
    0.01554,
    0.0019,
    0.00114,
    -0.01009,
    0.00609,
    0.02723,
    0.02125,
    0.00824,
    0.03126,
    -3e-05,
    -0.0154,
    -0.01022,
    -0.01688,
    0.02102,
    0.00896,
    -0.00497,
    -0.0212,
    0.00602,
    0.00103,
    -0.00526,
    0.01366,
    0.02753,
    -0.02419,
    -0.00112,
    -0.00447,
    -0.00026,
    -0.00982,
    -0.02867,
    0.01022,
    -0.01309,
    -0.02805,
    0.00196,
    0.00082,
    -0.00388,
    0.00499,
    0.01472,
    0.01451,
    0.00938,
    0.01034,
    -0.02198,
    -0.00173,
    -0.04945,
    0.00151,
    -0.00311,
    -0.02431,
    0.00192,
    -0.02104,
    -0.01812,
    0.01318,
    0.0193,
    -0.00383,
    -0.0063,
    -0.00517,
    0.00631,
    0.00321,
    -0.00433,
    -0.00514,
    0.00203,
    0.00609,
    -0.02196,
    0.00507,
    0.01311,
    0.01185,
    0.03396,
    -0.01799,
    -0.00615,
    -0.02844,
    -0.00156,
    -0.01196,
    -0.00374,
    0.00674,
    0.01922,
    0.00992,
    0.01314,
    0.00117,
    0.0095,
    0.00427,
    -0.02435,
    0.0585,
    0.01521,
    0.00982,
    -0.00364,
    -0.0669,
    0.00346,
    -0.01045,
    0.00234,
    0.01288,
    -0.00171,
    0.01461,
    -0.0214,
    -0.00807,
    0.00419,
    0.02734,
    0.00488,
    0.00902,
    -0.00287,
    -0.00656,
    -0.06864,
    0.03244,
    0.00445,
    -0.01251,
    -0.01803,
    -0.03365,
    0.00759,
    0.00262,
    0.16438,
    0.01825,
    0.00701,
    -0.00826,
    0.02081,
    -0.01366,
    0.0067,
    -0.00079,
    0.03128,
    -0.00321,
    0.00683,
    -0.00587,
    0.00184,
    -0.00767,
    -0.00611,
    -0.00627,
    0.00799,
    0.00977,
    -0.0445,
    -0.00817,
    0.01028,
    0.04843,
    -0.02978,
    0.00354,
    -0.01937,
    0.00132,
    0.03101,
    -0.02169,
    0.00478,
    0.00932,
    -0.01546,
    0.01069,
    -0.02554,
    -0.0019,
    -0.0144,
    0.00899,
    0.00389,
    0.04646,
    0.00171,
    -0.01517,
    -0.00983,
    -0.0155,
    -0.01444,
    0.01245,
    0.0159,
    -0.01714,
    0.01155,
    0.01844,
    0.0035,
    0.00131,
    0.00562,
    0.00789,
    -0.00036,
    0.01636,
    0.00503,
    0.00591,
    0.01922,
    -0.00332,
    -0.0019,
    -0.0187,
    0.00526,
    0.01837,
    0.005,
    -0.01875,
    -0.00883,
    -0.01199,
    -0.02108,
    0.00363,
    -0.27272,
    0.00504,
    -0.00166,
    0.01452,
    -0.00948,
    0.00938,
    0.00389,
    -0.00027,
    0.01383,
    0.00528,
    -0.01318,
    0.04551,
    0.00383,
    -0.00356,
    -0.00223,
    -0.00962,
    0.00169,
    -0.00073,
    -0.0094,
    -0.00109,
    -0.00301,
    0.00515,
    0.00517,
    -0.00267,
    0.0053,
    0.01225,
    0.00241,
    0.01558,
    0.02022,
    0.05832,
    0.00541,
    -0.01368,
    0.00872,
    -0.00683,
    0.00155,
    -0.02,
    -0.0061,
    -0.00542,
    -0.00405,
    0.0221,
    0.0202,
    -0.00107,
    -0.01059,
    0.01761,
    -0.0307,
    -0.02263,
    0.00575,
    -0.00339,
    -0.00495,
    -0.04679,
    0.00343,
    0.00152,
    0.0062,
    -0.0096,
    -0.00981,
    0.01529,
    0.00524,
    -0.02822,
    -0.03975,
    0.01695,
    -0.00102,
    -0.0088,
    0.0072,
    -0.00256,
    0.00462,
    0.04138,
    0.01193,
    -0.0209,
    0.01862,
    0.01898,
    -0.01103,
    -0.02268,
    0.00038,
    0.00785,
    -0.00827,
    -0.00403,
    -0.00092,
    -0.0097,
    0.00312,
    -0.00108,
    0.01739,
    -0.00193,
    -0.00735,
    0.00881,
    -0.01625,
    0.01104,
    -0.00718,
    -0.01267,
    -0.0615,
    -0.01289,
    0.00961,
    -0.00026,
    0.00204,
    0.00483,
    0.00791,
    0.01572,
    -0.01818,
    -0.00513,
    -0.00425,
    0.01141,
    0.00105,
    -0.00411,
    0.00251,
    -0.00285,
    -0.01948,
    -0.00021,
    0.00474,
    0.00472,
    -0.00959,
    0.00834,
    -0.0051,
    -0.01912,
    0.01626,
    0.00849,
    -0.00988,
    0.01467,
    -0.02585,
    -0.00454,
    -0.00331,
    0.00052,
    -0.06021,
    0.00946,
    -8e-05,
    -0.0143,
    -0.02473,
    0.00132,
    -0.01663,
    0.00238,
    0.06227,
    0.01194,
    -0.00027,
    -0.01218,
    -0.00372,
    -0.0765,
    0.00658,
    -0.01648,
    0.01703,
    0.00321,
    0.00368,
    0.00027,
    0.00111,
    0.01318,
    -0.00216,
    -0.00263,
    -0.01082,
    0.00226,
    -0.00084,
    -0.01187,
    0.00014,
    1e-05,
    -0.00767,
    -0.00843,
    -0.01783,
    0.0285,
    -0.02417,
    0.02163,
    -0.01951,
    -0.00691,
    -0.04192,
    -0.00434,
    -0.01651,
    0.02115,
    -0.07363,
    -0.00398,
    -0.00483,
    -0.04942,
    -0.01687,
    0.00055,
    0.00318,
    0.0169,
    -0.00117,
    -0.01671,
    -0.06568,
    0.01311,
    0.00617,
    -0.00506,
    0.0136,
    -0.00536,
    0.03279,
    -0.00716,
    -0.00967,
    0.00812,
    0.00137,
    -0.01718,
    0.00607,
    -0.00516,
    -0.00866,
    -0.00432,
    0.00251,
    0.00089,
    -0.00339,
    -0.00188,
    0.00856,
    -0.01007,
    -0.02942,
    0.02539,
    0.00254,
    0.00095,
    0.00436,
    0.00821,
    -0.01823,
    0.02188,
    0.00252,
    0.00896,
    0.0,
    0.00015,
    -0.01374,
    -0.00942,
    0.00642,
    -0.00283,
    0.01708,
    0.00943,
    0.01289,
    0.00377,
    -0.01619,
    0.0298,
    0.00132,
    0.02169,
    0.00999,
    0.00888,
    -0.01618,
    -0.07302,
    -0.01498,
    -0.04257,
    0.00644,
    0.00427,
    -0.04799,
    -0.0043,
    0.00971,
    -0.03264,
    0.00852,
    0.0112,
    0.01225,
    -0.01222,
    0.00713,
    0.00194,
    -0.01052,
    -0.00123,
    0.02402,
    0.00299,
    -0.00357,
    0.0119,
    -0.04784,
    0.02709,
    -0.00601,
    0.01735,
    0.00786,
    0.00103,
    0.01334,
    -0.00369,
    -0.01053,
    -0.00541,
    -0.03009,
    -0.0051,
    -0.03068,
    -0.01233,
    0.01425,
    -0.00413,
    0.00596,
    -0.00529,
    0.0107,
    0.002,
    -0.00592,
    -0.00497,
    0.01851,
    -0.00196,
    0.00619,
    -0.00586,
    0.00253,
    0.01292,
    -0.00244,
    -0.00742,
    0.01373,
    0.02371,
    -0.08056,
    -0.04704,
    0.01006,
    0.02752,
    -0.01034,
    -0.02283,
    0.002,
    0.00036,
    -0.0475,
    -0.08802,
    0.00086,
    0.0579,
    0.00078,
    0.01635,
    -0.03017,
    0.05748,
    -0.00845,
    0.00444,
    -0.00657,
    0.03973,
    0.00991,
    0.02087,
    0.00253,
    0.00887,
    0.00477,
    0.00236,
    -0.00125,
    0.02476,
    0.00906,
    0.0019,
    0.01327,
    0.01926,
    -0.01918,
    -0.00847,
    -0.09307,
    -0.00616,
    -0.00247,
    -0.01242,
    0.008,
    -0.00817,
    0.00927,
    -0.00664,
    0.01158,
    0.01794,
    -0.00517,
    -0.00078,
    -0.0158,
    0.01747,
    -0.00022,
    0.012,
    -0.00956,
    0.01099,
    0.04999,
    0.01443,
    0.01152,
    -0.00947,
    -0.00766,
    0.00112,
    0.00842,
    -0.02885,
    0.0004,
    0.01111,
    -0.0035,
    -0.05268,
    0.02542,
    -0.00582,
    0.00338,
    0.00283,
    0.04298,
    0.01016,
    0.0007,
    -0.00287,
    0.00852,
    -0.01129,
    -0.00891,
    -0.00209,
    0.01245,
    0.00262,
    -0.00284,
    0.01725,
    0.00252,
    -0.01867,
    0.00664,
    0.00915,
    -0.01421,
    -0.03749,
    0.00462,
    -0.03478,
    -0.00879,
    -0.03694,
    -0.00623,
    0.00154,
    -0.01109,
    0.00048,
    -0.04393,
    0.00433,
    0.00871,
    -0.00973,
    0.0022,
    -0.00263,
    0.01271,
    0.01408,
    0.00468,
    -0.02333,
    -0.0084,
    0.01345,
    0.01071,
    0.00058,
    0.01588,
    -0.01037,
    0.00957,
    0.00595,
    -0.05584,
    0.00237,
    -0.00075,
    0.01405,
    -0.00657,
    0.01165,
    -0.00142,
    0.01835,
    -0.00111,
    0.02141,
    0.01291,
    -0.00164,
    0.00014,
    0.00937,
    0.01908,
    -0.00869,
    -0.01498,
    -0.04334,
    0.00742,
    0.00189,
    0.01284,
    -0.01437,
    0.00582,
    -0.00252,
    0.03946,
    -0.02438,
    0.00778,
    0.00535,
    -0.00429,
    -0.00969,
    0.00752,
    -0.00631,
    -0.01242,
    0.00624,
    0.07084,
    0.00759,
    8e-05,
    0.00946,
    -0.01605,
    0.02257,
    0.00961,
    -0.01956,
    -0.01327,
    -0.01629,
    0.01948,
    0.02058,
    0.00159,
    0.01365,
    0.01471,
    0.00795,
    -0.00201,
    0.01817,
    0.00865,
    -0.00829,
    0.01526,
    -0.04571,
    0.00449,
    -0.01756,
    0.01365,
    -0.01179,
    -0.01733,
    -0.01653,
    -0.00303,
    -0.01375,
    0.02318,
    -0.01681,
    0.00943,
    0.0057,
    -0.00463,
    -0.00949,
    0.00717,
    -0.05017,
    0.00803,
    -0.02262,
    -0.07405,
    -0.00467,
    -0.00275,
    0.00518,
    0.00242,
    0.01173,
    0.00643,
    0.00916,
    -0.01279,
    0.00599,
    -0.00801,
    0.00216,
    -0.06487,
    -0.0034,
    0.03805,
    0.08399,
    -0.00259,
    0.01447,
    -0.00642,
    -0.01352,
    0.0018,
    0.04693,
    -0.01641,
    0.04313,
    0.01747,
    -0.0207,
    -0.00585,
    -0.00326,
    -0.01742,
    -0.00639,
    0.00182,
    0.00682,
    0.00442,
    -0.01377,
    0.05058,
    -0.02938,
    -0.00043,
    -0.01983,
    0.02398,
    -0.00572,
    0.02521,
    -0.02131,
    0.0141,
    -0.00292,
    -0.00072,
    0.0205,
    0.00041,
    0.01888,
    0.01589,
    0.00044,
    -0.00969,
    0.02265,
    -0.0692,
    -0.01064,
    0.01153,
    -0.00117,
    0.00772,
    -0.02114,
    -0.00911,
    -0.0071,
    -0.00098,
    0.00666,
    -0.02065,
    -0.00619,
    -0.00617,
    -0.0736,
    0.00451,
    -0.00456,
    0.01081,
    -0.00864,
    0.00203,
    0.02525,
    -0.00207,
    -0.01132,
    -0.00786,
    0.02235,
    -0.0074,
    0.0003,
    -0.01729,
    0.00029,
    -0.01692,
    0.0126,
    -0.00353,
    0.00445,
    -0.06509,
    -0.00644,
    0.03257,
    0.00645,
    -0.00744,
    -0.00771,
    0.00997,
    -0.00355,
    -0.02002,
    0.00898,
    0.01445,
    0.0467,
    -0.01021,
    0.00659,
    -0.00519,
    0.0026,
    -0.00892,
    -0.0029,
    0.00455,
    -0.00392,
    -0.00441,
    -0.00423,
    -0.01767,
    0.00251,
    0.006,
    -0.00841,
    0.00971,
    -0.00316,
    0.01252,
    0.02156,
    0.01486,
    -0.0091,
    -0.00465,
    -0.00308,
    0.02832,
    -0.00614,
    -0.01516,
    0.01438,
    -0.00528,
    -0.00436,
    0.00072,
    -0.00169,
    -0.01559,
    -0.0028,
    0.01528,
    0.0413,
    0.01756,
    -0.00681,
    -0.02971,
    0.00376,
    -0.01166,
    -2e-05
   ]
  },
  "inst14": {
   "VOID": [
    0.01916,
    -0.01382,
    -0.01402,
    0.03372,
    -0.00324,
    -0.00298,
    0.0193,
    -0.01348,
    0.03665,
    0.01888,
    0.0106,
    -0.033,
    0.02072,
    0.01496,
    -0.02018,
    0.01118,
    -0.01961,
    0.00539,
    0.01404,
    0.00286,
    -0.02549,
    0.03679,
    0.00302,
    -0.00299,
    0.01773,
    0.00102,
    -0.01428,
    -0.03835,
    0.03352,
    -0.02555,
    0.00431,
    0.03503,
    -0.02208,
    0.03275,
    0.01412,
    -0.02541,
    -0.02481,
    0.01841,
    -0.00118,
    -0.02282,
    0.01804,
    -0.00622,
    0.05559,
    0.03001,
    -0.02241,
    0.00138,
    0.03369,
    -0.04443,
    0.00582,
    -0.01248,
    0.01962,
    -0.00405,
    0.00583,
    -0.02964,
    0.05063,
    0.05557,
    0.02075,
    0.00629,
    -0.02849,
    0.00801,
    0.01989,
    0.00738,
    0.00338,
    0.00447,
    0.0201,
    -0.02433,
    -0.00484,
    0.00872,
    -0.01119,
    0.02057,
    0.00879,
    -0.01981,
    0.00076,
    0.03343,
    -0.00173,
    0.02953,
    0.03804,
    0.00642,
    -0.01534,
    0.00022,
    0.0384,
    -0.00441,
    -0.02543,
    0.00891,
    0.01067,
    -0.01697,
    -0.00526,
    -0.02101,
    -0.00532,
    0.00247,
    -0.01616,
    -0.00657,
    0.01004,
    0.02142,
    0.02036,
    0.03612,
    0.00379,
    0.01809,
    0.03927,
    -0.04474,
    0.03243,
    -0.03328,
    0.00252,
    -0.06848,
    0.02095,
    0.0023,
    0.00168,
    -0.0056,
    0.01925,
    -0.00638,
    0.00682,
    -0.02748,
    -0.0002,
    -0.00333,
    0.02856,
    -0.03391,
    -0.00749,
    -0.00159,
    0.00561,
    -0.02332,
    -0.03161,
    0.00966,
    0.01152,
    0.01945,
    0.0175,
    0.02412,
    -0.00305,
    0.00289,
    0.04154,
    -0.00741,
    -0.00882,
    0.00078,
    0.00132,
    0.04467,
    -0.05611,
    -0.00333,
    0.00723,
    -0.01527,
    -0.01563,
    0.00547,
    -0.01024,
    0.03675,
    0.01105,
    0.0054,
    0.01612,
    0.04628,
    -0.03158,
    0.02713,
    -0.00141,
    -0.04294,
    0.00551,
    -0.03029,
    -0.02988,
    -0.00605,
    0.0076,
    0.00588,
    0.04016,
    -0.00966,
    -0.00326,
    0.00709,
    -0.03165,
    0.01537,
    0.01553,
    0.02107,
    -0.02372,
    -0.01149,
    0.04161,
    -0.00139,
    0.03089,
    -0.00409,
    0.04401,
    -0.00973,
    0.02869,
    0.00415,
    0.00408,
    -0.02746,
    0.02788,
    0.00327,
    -0.00559,
    0.0117,
    0.03222,
    0.0333,
    0.02233,
    -0.05389,
    0.00031,
    0.02036,
    -0.00105,
    -0.03523,
    -0.03323,
    0.01935,
    -0.04381,
    -0.00488,
    -0.03946,
    -0.01095,
    -0.02927,
    0.00812,
    0.01543,
    -0.00606,
    0.04127,
    -0.02816,
    0.01812,
    0.0135,
    -0.00668,
    0.02261,
    0.03382,
    -0.00444,
    -0.00152,
    0.00062,
    0.0261,
    -0.01376,
    -0.04193,
    -0.01361,
    0.03427,
    -0.05659,
    -0.00612,
    -0.03995,
    -0.01114,
    -0.01017,
    0.01704,
    0.01524,
    -0.01032,
    0.01566,
    0.01522,
    0.01601,
    -0.01159,
    -0.01684,
    0.01087,
    0.00233,
    -0.01759,
    -0.00979,
    0.08919,
    -0.03152,
    0.04012,
    0.05722,
    -0.01319,
    0.01277,
    0.07383,
    0.02134,
    0.00203,
    -0.04884,
    0.00468,
    -0.04167,
    0.00233,
    -0.04371,
    -0.03421,
    0.02388,
    0.01891,
    0.00745,
    -0.00827,
    -0.00249,
    -0.00434,
    -0.0005,
    0.03262,
    0.03001,
    -0.00924,
    -0.03263,
    -0.01093,
    -0.0054,
    0.04386,
    0.02469,
    -0.01974,
    0.03071,
    0.01352,
    0.03289,
    -0.0298,
    -0.03012,
    0.0206,
    -0.02358,
    0.0094,
    -0.03299,
    0.00794,
    -0.0111,
    0.0334,
    0.02565,
    4e-05,
    0.00813,
    0.01647,
    0.02336,
    -0.02409,
    -0.00146,
    -0.00182,
    0.05404,
    -0.01392,
    0.01471,
    -0.03126,
    0.01996,
    0.02418,
    -0.01482,
    0.00479,
    0.01527,
    0.03803,
    -0.04939,
    -0.00932,
    0.01778,
    0.02226,
    0.01586,
    0.03497,
    0.03162,
    0.00236,
    -0.05151,
    0.0176,
    -0.00349,
    0.02445,
    0.00644,
    0.03019,
    -0.00392,
    -0.00657,
    -0.00633,
    0.02319,
    0.01873,
    -0.00895,
    0.03037,
    -0.0212,
    -0.03616,
    -0.00139,
    -0.02311,
    0.01547,
    -0.03957,
    -0.00029,
    -0.02411,
    -0.05445,
    -0.01796,
    0.01104,
    0.00242,
    -0.00668,
    -0.0165,
    -0.03461,
    0.01026,
    -0.0121,
    0.0204,
    -0.00314,
    0.01363,
    0.01106,
    -0.02884,
    0.00027,
    0.03307,
    0.01419,
    -0.00096,
    0.0251,
    0.01238,
    0.01433,
    0.03723,
    0.0099,
    -0.01594,
    -0.02504,
    5e-05,
    -0.02117,
    -0.02422,
    0.02128,
    0.02365,
    -0.01551,
    0.00181,
    0.03491,
    0.00779,
    0.03329,
    0.00635,
    0.04034,
    0.01622,
    0.03987,
    0.03695,
    -0.00043,
    0.00987,
    0.02226,
    0.05085,
    0.02308,
    -0.05589,
    0.00599,
    -0.01184,
    -0.0118,
    0.07671,
    -0.01395,
    -0.04845,
    -0.00475,
    0.00241,
    -0.05307,
    0.01345,
    0.00098,
    -0.00067,
    0.00573,
    -0.0095,
    -0.0165,
    -0.01178,
    0.00017,
    0.02231,
    0.00057,
    -0.01114,
    -0.01813,
    0.01248,
    -0.0077,
    0.00478,
    -0.03315,
    -0.00235,
    0.01659,
    -0.04656,
    -0.00307,
    -0.02681,
    0.00973,
    -0.00171,
    -0.01187,
    0.00861,
    0.03203,
    -0.05535,
    -0.01318,
    -0.02036,
    0.02514,
    -0.03382,
    0.00809,
    0.00738,
    0.0443,
    -0.01204,
    0.0021,
    0.01384,
    0.00673,
    0.00322,
    -0.00514,
    0.02536,
    -0.0257,
    -0.00305,
    -0.02095,
    0.00025,
    0.01169,
    0.01313,
    0.02772,
    0.01385,
    0.06876,
    -0.03328,
    0.00471,
    -0.04102,
    0.03415,
    0.03333,
    0.03736,
    -0.01054,
    0.04435,
    -0.0016,
    0.00361,
    -0.0457,
    -0.01742,
    -0.02338,
    -0.01855,
    0.01901,
    0.01145,
    -0.0415,
    0.0586,
    0.01504,
    0.00119,
    -0.01626,
    -0.00252,
    -0.00124,
    -0.03041,
    -0.01372,
    0.00647,
    0.04394,
    0.05656,
    -0.04573,
    -0.01136,
    0.04946,
    -0.00043,
    -0.02059,
    0.01009,
    -0.00704,
    0.01237,
    -0.07281,
    0.00947,
    0.01191,
    -0.01268,
    0.02189,
    -0.01875,
    0.0003,
    -0.00117,
    0.02916,
    -0.03502,
    -0.02095,
    0.01002,
    0.06419,
    -0.03536,
    -0.01014,
    -0.01433,
    -0.02175,
    0.00162,
    0.0106,
    -0.01899,
    -0.04078,
    -0.01252,
    -0.01018,
    0.01227,
    0.00888,
    0.01463,
    -0.05239,
    -0.01265,
    -0.03069,
    -0.01329,
    -0.04972,
    0.0183,
    0.02677,
    -0.0236,
    -0.00782,
    0.00211,
    0.01194,
    0.00209,
    -0.03708,
    -0.00411,
    0.02781,
    0.01364,
    -0.00724,
    0.03708,
    0.003,
    -0.02252,
    0.03537,
    0.00625,
    0.00726,
    -0.01192,
    -0.01014,
    0.00546,
    0.03127,
    -0.00236,
    0.01838,
    -0.02108,
    0.03583,
    -0.01138,
    0.01746,
    -0.00149,
    0.06038,
    0.0051,
    -0.02619,
    -0.01574,
    -0.02493,
    0.00426,
    -0.03835,
    -0.01876,
    -0.02353,
    0.02532,
    0.01555,
    -0.01462,
    -0.02383,
    -0.00564,
    0.05445,
    -0.00367,
    -0.01754,
    0.02336,
    -0.01306,
    0.02878,
    -0.02363,
    0.01777,
    -0.00127,
    -0.04723,
    -0.00982,
    0.03186,
    -0.02051,
    -0.00888,
    -0.00095,
    0.01673,
    -0.05191,
    -0.06291,
    0.00781,
    0.00972,
    0.02774,
    0.01072,
    0.02733,
    0.01074,
    0.00834,
    0.02966,
    0.03133,
    0.011,
    -0.05161,
    -0.0163,
    -0.01798,
    -0.00826,
    -0.00838,
    0.0604,
    0.02684,
    -0.00211,
    -0.00869,
    0.02482,
    0.00166,
    -0.01279,
    -0.01418,
    -0.00117,
    -0.00695,
    -0.02198,
    -0.00386,
    -0.01228,
    -0.02676,
    0.00531,
    0.01937,
    0.01197,
    -0.05758,
    -0.0257,
    0.01727,
    0.00023,
    -0.00981,
    0.0359,
    -0.03364,
    0.00998,
    0.00157,
    -0.01027,
    0.03137,
    -0.05825,
    -0.0304,
    -0.02989,
    0.00531,
    0.02998,
    -0.0351,
    -0.01219,
    -0.00949,
    0.01535,
    -0.00928,
    0.01963,
    0.00637,
    0.04704,
    -0.02565,
    -0.00547,
    -0.00441,
    0.00597,
    0.02108,
    -0.02515,
    0.01328,
    -0.02959,
    0.009,
    -0.00229,
    -0.01686,
    -0.00199,
    0.012,
    0.01,
    0.02249,
    -0.00592,
    -0.03462,
    0.00073,
    0.04457,
    0.02045,
    0.01061,
    -0.00135,
    -0.00288,
    -0.02315,
    0.07242,
    0.0247,
    -0.02038,
    -0.0102,
    0.0027,
    -0.0383,
    0.01508,
    0.0018,
    -0.01202,
    0.00208,
    -0.02895,
    0.03418,
    0.00142,
    -0.0077,
    0.01779,
    -0.0065,
    -0.00291,
    0.02839,
    -0.00129,
    -0.00671,
    0.01186,
    -0.03938,
    -0.04977,
    0.00109,
    -0.04421,
    0.04271,
    -0.02771,
    -0.03921,
    0.01448,
    -0.00964,
    0.00481,
    -0.01995,
    -0.01525,
    0.02148,
    -0.03261,
    0.01137,
    0.00262,
    -0.00444,
    0.01558,
    -0.02657,
    0.06963,
    -0.03496,
    0.00306,
    0.01128,
    -0.00344,
    -0.01994,
    0.04392,
    -0.0384,
    0.00847,
    -0.00158,
    -0.05664,
    0.01836,
    -0.03433,
    -0.00725,
    0.02105,
    0.03851,
    -0.05152,
    0.01694,
    0.0189,
    0.03669,
    0.06884,
    0.03454,
    0.02717,
    -0.00946,
    -0.00739,
    0.00085,
    0.01407,
    0.04188,
    -0.02724,
    0.03774,
    -0.00164,
    -0.00254,
    -0.02182,
    -0.0127,
    0.0293,
    -0.05056,
    0.00175,
    0.00049,
    -0.02425,
    -0.00991,
    0.0078,
    -0.02765,
    -0.04454,
    0.01071,
    -0.02585,
    0.01407,
    0.00646,
    -0.04841,
    -0.01167,
    0.02396,
    0.00095,
    0.00589,
    0.02154,
    -0.01593,
    -0.00152,
    -0.02319,
    -0.02119,
    -0.04194,
    0.01136,
    -0.02303,
    -0.01269,
    0.07209,
    0.0277,
    0.05848,
    0.01444,
    0.00055,
    -0.02936,
    0.00014,
    -0.0373,
    -0.04886,
    -0.03439,
    -0.03829,
    -0.02162,
    0.02162,
    -0.02202,
    -0.00204,
    -0.0051,
    0.02696,
    0.04168,
    0.02176,
    -0.00334,
    -0.01075,
    -0.00754,
    0.00991,
    -0.01436,
    -0.03174,
    0.00068,
    0.05778,
    -0.00797,
    -0.02047,
    -0.01162,
    -0.00724,
    0.05013,
    -0.02671,
    -0.02603,
    -0.01226,
    -0.04153,
    -0.02807,
    0.04055,
    -0.02525,
    0.00464,
    -0.00991,
    0.063,
    -0.01537,
    -0.01947,
    0.0289,
    0.00972,
    -0.00223,
    -0.02906,
    0.0347,
    0.00721,
    0.04651,
    0.03141,
    -0.03818,
    0.00847,
    -0.00655,
    -0.03506,
    0.00983,
    0.06825,
    0.00716,
    0.01319,
    0.06206,
    -0.02945,
    -0.00147,
    -0.01155,
    0.0147,
    0.00029,
    -0.00709,
    0.05156,
    -0.00207,
    -0.0464,
    0.06848,
    -0.00225,
    0.01649,
    0.03047,
    0.00025,
    0.00104,
    -0.03101,
    0.01783,
    -0.01206,
    0.00247,
    0.00489,
    -0.04563,
    0.01273,
    -0.00748,
    0.03106,
    0.01043,
    -0.00417,
    0.02051,
    -0.03003,
    0.01503,
    0.00511,
    0.01464,
    0.0116,
    0.00318,
    -0.01421,
    -0.02365,
    0.0153,
    -0.02256,
    -0.03664,
    -0.01198,
    0.00697,
    0.03339,
    0.0585,
    -0.01752,
    0.00258,
    0.01474,
    0.00124,
    0.00849,
    0.0063,
    -0.00336,
    -0.00827,
    0.0009,
    0.01716,
    -0.01033,
    -0.01558,
    0.01304,
    0.04046,
    -0.02195,
    -0.01736,
    0.04239,
    0.01231,
    1e-05,
    0.01534,
    0.04854,
    0.01103,
    -0.00654,
    0.03479,
    -0.03346,
    0.04156,
    -0.01149,
    0.00689,
    0.05619,
    -0.00516,
    0.00097,
    -0.03229,
    0.01764,
    -0.01013,
    0.00607,
    0.03687,
    0.0096,
    -0.04084,
    -0.01204,
    -0.00891,
    -0.00219,
    -0.00234,
    0.01115,
    -0.00232,
    0.00744,
    -0.02407,
    -0.01616,
    0.01305,
    0.00928,
    -0.02548,
    -0.00138,
    -0.02969,
    -0.0288,
    -0.00122,
    -0.02079,
    -0.01171,
    0.02352,
    0.03255,
    -0.0145,
    0.00781,
    0.01686,
    -0.01045,
    0.01369,
    -0.00065,
    0.03063,
    0.01198,
    -0.03437,
    -0.02619,
    0.0032,
    -0.02534,
    0.04189,
    -0.02285,
    0.00629,
    -0.01385,
    0.01964,
    0.03816,
    -0.00314,
    0.02558,
    -0.01599,
    0.02606,
    -0.01514,
    -0.01316,
    -0.00482,
    -0.02738,
    0.02643,
    0.02798,
    -0.02222,
    -0.02838,
    -0.0189,
    -0.01895,
    0.01306,
    -0.01968,
    0.00885,
    0.01574,
    -0.01195,
    -0.02123,
    0.00242,
    -0.03368,
    -0.01942,
    0.0179,
    -0.00714,
    0.00686,
    -0.00859,
    -1e-05,
    0.03644,
    -0.0137,
    -0.02076,
    -0.02554,
    0.01078,
    -0.0572,
    0.03715,
    0.01102,
    -0.0152,
    -0.02739,
    0.02295,
    0.01377,
    -0.02988,
    0.01833,
    0.03254,
    0.00778,
    0.01641,
    -0.0715,
    -0.00338,
    -0.0238,
    -0.01707,
    -0.02035,
    0.04334,
    -0.04421,
    -0.08548,
    -0.00815,
    -0.0285,
    -0.01501,
    0.0094,
    0.01155,
    -0.0143,
    -0.01902,
    -0.04617,
    -0.01939,
    -0.00823,
    0.02877,
    0.01775,
    -0.01056,
    0.03371,
    0.01017,
    -0.00752,
    0.04246,
    0.00065,
    -0.04431,
    -0.03074,
    0.0056,
    -0.01387,
    0.02064,
    0.01511,
    -0.03791,
    0.00366,
    0.02405,
    0.01325,
    0.01083,
    0.0102,
    0.04652,
    0.032,
    -0.0236,
    -0.01164,
    -0.00634,
    -0.00377,
    0.00093,
    0.03101,
    -0.0127,
    0.03528,
    0.01506,
    0.01544,
    0.04683,
    -0.00303,
    0.00907,
    0.00095,
    -0.00823,
    -0.02179,
    -0.00863,
    0.02108,
    -0.01387,
    0.03677,
    -0.02572,
    -0.01458,
    -0.02067,
    0.00872,
    -0.00849,
    -0.01877,
    0.01848,
    0.03696,
    0.01103,
    -0.01997,
    -0.02395,
    -0.03364,
    0.0023,
    -0.00992,
    0.00216,
    0.01708,
    0.02106,
    -0.01454,
    -0.03129,
    -0.02815,
    0.01531,
    0.0271,
    0.00044,
    0.00688,
    -0.04193,
    0.02917,
    -0.01251,
    0.00703,
    -0.04159,
    -0.02109,
    0.00516,
    0.01525,
    0.0017,
    -0.0103,
    0.00236,
    0.04196,
    -0.03999,
    -0.01797,
    -0.00736,
    0.02805,
    -0.021,
    0.02585,
    -0.04442,
    -0.00063,
    -0.02789,
    -0.03246,
    -0.00369,
    -0.00584,
    0.00936,
    0.0154,
    -0.01912,
    0.00601,
    -0.01139,
    0.01687,
    0.03364,
    0.05119,
    -0.0335,
    -0.02809,
    -0.00243,
    -0.027,
    -0.00407,
    0.01355,
    0.03554,
    -0.03861,
    0.07729,
    -0.00438,
    0.00281,
    -0.00108,
    -0.00044,
    0.01066,
    -0.00623,
    -0.03293,
    0.0271,
    -0.01691,
    -0.0333,
    0.02352,
    0.02374,
    0.05375,
    0.01621,
    0.0015,
    -0.00024,
    0.06158,
    0.00901,
    0.01006,
    0.00437,
    0.05618,
    -0.0049,
    0.00912,
    -0.0001,
    -0.00122,
    0.02084,
    0.00205,
    -0.03343,
    -0.03759,
    -0.00224,
    0.00792,
    -0.04763,
    -0.00686,
    -0.01873,
    -0.00923,
    0.01838,
    0.00665,
    0.00034,
    -0.00078,
    -0.03295,
    -0.05303,
    -0.00338,
    0.00979,
    0.03083,
    0.03863,
    -0.01731,
    -0.03416,
    0.03071,
    0.03752,
    -0.01419,
    0.01699,
    0.01269,
    0.01116,
    0.00698,
    0.00992,
    0.02282,
    -0.01051,
    0.03895,
    0.02156,
    0.04192,
    -0.02686,
    -0.02448,
    -0.00389,
    0.03736,
    -0.01137,
    0.01812,
    -0.00033,
    -0.0094,
    -0.04771,
    0.06275,
    -0.03477,
    0.0437,
    -0.0129,
    0.01359,
    -0.00968,
    0.02497,
    -0.01185,
    -0.01362,
    -0.01208,
    -0.03727,
    -0.04083,
    0.00283,
    0.01566,
    0.0584,
    -0.01251,
    0.03148,
    0.02204,
    0.02231,
    0.03217,
    0.01375,
    -0.0546,
    -0.00634,
    0.03719,
    0.02864,
    -0.0229,
    0.0266,
    -0.02595,
    0.01779,
    0.02222,
    -0.04397,
    -0.00671,
    -0.02105,
    0.02963,
    0.00265,
    0.01032,
    0.02544,
    0.01663,
    -0.01542,
    0.02204,
    0.01179,
    0.05395,
    -0.02663,
    -0.00866,
    -0.01569,
    -0.01157,
    0.01266,
    0.00257,
    -0.00691,
    0.04556,
    -0.03252,
    -0.02207,
    -0.04082,
    -0.01365,
    -0.05489,
    0.03691,
    -0.02219,
    -0.00727,
    0.01611,
    0.02293,
    0.00333,
    -0.04842,
    -0.01323,
    -0.00033,
    -0.00432,
    0.0083,
    0.00831,
    -0.01143,
    -0.01662,
    -0.00871,
    0.01165,
    -0.02835,
    -0.00132,
    0.00447,
    -0.02385,
    0.00669,
    -0.03317,
    -0.04714,
    0.01537,
    0.0172,
    -0.0105,
    -0.00155,
    0.05108,
    -0.00766,
    -0.04089,
    0.02867,
    -0.01785,
    -0.02979,
    0.02834,
    0.02856,
    0.03025,
    0.01945,
    -0.00203,
    -0.01039,
    0.0365,
    -0.02035,
    0.06414,
    -0.01033,
    0.01753,
    0.00703,
    -0.00068,
    0.03996,
    -0.04193,
    0.01308,
    -0.05611,
    -0.01817,
    -0.02071,
    -0.02246,
    -0.01258,
    -0.0136,
    0.00777,
    -0.02545,
    -0.01028,
    0.00381,
    0.02143,
    -0.05524,
    0.04258,
    0.01449,
    0.04566,
    -0.01202,
    -0.03323,
    0.01245,
    0.01968,
    -0.04444,
    0.00023,
    0.04519,
    -0.00365,
    0.00778,
    0.03312,
    0.00989,
    0.01677,
    -0.01335,
    -0.00399,
    0.00492,
    0.01458,
    0.02097,
    -0.02267,
    -0.01235,
    0.00991,
    0.01126,
    -0.04842,
    0.0202,
    0.00458,
    0.01889,
    -0.01879,
    -0.03276,
    -0.03266,
    -0.01938,
    -0.02162,
    0.03153,
    -0.02402,
    -0.00427,
    0.0012,
    -0.01648,
    -0.01801,
    0.0374,
    -0.0054,
    0.01467,
    0.02685,
    0.01933,
    -0.00171,
    0.0184,
    0.0041,
    -0.00772,
    0.02168,
    -0.00921,
    0.0007,
    0.0079,
    0.02671,
    0.06177,
    0.00702,
    -0.0203,
    -0.04411,
    -0.04395,
    0.00317,
    -0.01481,
    0.005,
    -0.01454,
    -0.00066,
    -0.00319,
    0.03189,
    0.02243,
    0.02154,
    -0.05571,
    0.02504,
    0.03494,
    0.01047,
    -0.01869,
    -0.02026,
    -0.00177,
    -0.05188,
    -0.02529,
    0.02232,
    -0.00149,
    -0.01983,
    0.00467,
    0.01532,
    -0.0233,
    0.01465,
    -0.02246,
    0.01062,
    0.01207,
    -0.02693,
    0.00214,
    0.00751,
    0.04662,
    0.02463,
    0.01106,
    0.04595,
    0.00462,
    -0.02005,
    0.02113,
    0.0603,
    0.05065,
    0.00762,
    -0.01346,
    -0.01467,
    -0.0046,
    0.00796,
    0.02648,
    -0.02726,
    0.00818,
    -0.01193,
    -0.02091,
    -0.00948,
    0.00296,
    -0.01665,
    -0.03564,
    0.04214,
    -0.0024,
    0.01681,
    0.01057,
    -0.0043,
    0.0081,
    0.0454,
    -0.01403,
    0.01268,
    -0.01395,
    -0.012,
    0.00999,
    0.00988,
    0.00059,
    -0.00256,
    -0.01674,
    0.00946,
    -0.01206,
    -0.02381,
    -0.0013,
    -0.00868,
    0.00886,
    -0.00987,
    0.04037,
    -0.013,
    -0.01871,
    -0.00029,
    -0.01679,
    0.01648,
    -0.00398,
    0.00437,
    0.06069,
    0.01576,
    -0.07137,
    0.01503,
    -0.00064,
    -0.02873,
    0.01798,
    0.00233,
    -0.02618,
    -0.01308,
    0.14912,
    -0.00651,
    -0.00343,
    0.09659,
    0.03211,
    -0.02355,
    0.03906,
    -0.0109,
    -0.02268,
    0.01044,
    -0.01434,
    -0.02311,
    -0.00324,
    -0.04474,
    -0.03194,
    -0.01357,
    -0.01725,
    0.02989,
    0.00287,
    -0.02183,
    -0.00914,
    -0.00785,
    -0.00528,
    -0.01606,
    -0.01763,
    0.02037,
    -0.04835,
    -0.00673,
    -0.0197,
    0.01689,
    0.02057,
    -0.02615,
    0.00627,
    0.00928,
    -0.00169,
    -0.03587,
    -0.03629,
    -0.00277,
    0.00751,
    0.00461,
    -0.00382,
    -0.0164,
    0.01428,
    -0.00692,
    -0.00339,
    -0.02359,
    -0.02743,
    -0.01194,
    0.02558,
    -0.02458,
    -0.05587,
    0.01912,
    0.00022,
    -0.01187,
    -0.00731,
    0.00455,
    0.03889,
    -0.0538,
    0.02206,
    0.01336,
    0.0299,
    -0.00206,
    0.00518,
    -0.00059,
    -0.01932,
    -0.00838,
    0.02115,
    0.01971,
    -0.02275,
    -0.00172,
    0.01014,
    -0.03313,
    0.00683,
    -0.0372,
    0.02301,
    0.01698,
    0.00916,
    -0.01518,
    0.03495,
    -0.00039,
    -0.03232,
    0.00072,
    -0.05013,
    0.01241,
    -0.02625,
    0.00766,
    -0.00864,
    0.0067,
    -0.00241,
    0.05432,
    0.04414,
    -0.03472,
    0.0008,
    0.04755,
    -0.00207,
    -0.00047,
    0.0357,
    0.00435,
    0.04345,
    0.03638,
    0.05207,
    -0.01096,
    0.02248,
    -0.00845,
    0.00074,
    -0.02467,
    0.0445,
    0.00387,
    0.02297,
    0.02424,
    0.03007,
    -0.01062,
    -0.01882,
    -0.02275,
    -0.0682
   ],
   "SUDDEN": [
    -0.00551,
    -0.0053,
    0.00551,
    -0.00409,
    0.0152,
    -0.01992,
    -0.03021,
    -0.00642,
    -0.04609,
    -0.03962,
    0.01556,
    0.04219,
    -0.03982,
    -0.02469,
    -0.04718,
    -0.01455,
    0.00159,
    0.02228,
    -0.01307,
    0.01932,
    -0.00619,
    0.02268,
    0.00256,
    -0.01227,
    -0.03911,
    -0.0505,
    -0.01793,
    -0.01821,
    0.03337,
    0.03768,
    -0.03354,
    -0.02326,
    0.02534,
    -0.00706,
    -0.01352,
    0.05652,
    0.0263,
    -0.02687,
    0.01832,
    -0.03024,
    -0.00766,
    0.00878,
    -0.04352,
    -0.02069,
    0.00611,
    -0.00359,
    -0.03295,
    0.02654,
    -0.00537,
    -0.02422,
    0.0063,
    0.00991,
    0.00251,
    0.04612,
    -0.05862,
    -0.02772,
    0.02933,
    -0.01507,
    0.02033,
    -0.02223,
    -0.00381,
    -0.00275,
    -0.00191,
    -0.02543,
    0.01609,
    0.0133,
    0.00843,
    0.00247,
    0.00442,
    -0.04696,
    0.00371,
    -0.00347,
    0.01661,
    0.00966,
    0.01258,
    0.00411,
    -0.02039,
    0.04957,
    0.03559,
    -0.01011,
    -0.00727,
    -0.00302,
    -0.05029,
    0.00849,
    -0.01581,
    -0.0108,
    -0.01491,
    0.01605,
    -0.02082,
    -0.01643,
    0.00201,
    -0.00602,
    0.00455,
    0.00294,
    0.00945,
    -0.04178,
    -0.01456,
    -0.0229,
    -0.04134,
    0.0511,
    -0.03127,
    0.02115,
    -0.02838,
    0.03511,
    0.01087,
    0.02715,
    -0.03379,
    0.01592,
    0.04516,
    0.02919,
    0.02391,
    -0.00187,
    0.01827,
    0.01277,
    -0.06827,
    0.0007,
    0.03527,
    -0.01942,
    -0.00611,
    0.00927,
    0.00798,
    -0.02882,
    -0.01331,
    -0.01383,
    -0.02973,
    -0.00748,
    -0.05079,
    -0.02097,
    -0.0502,
    0.06444,
    -0.00084,
    0.01822,
    -0.02266,
    -0.02967,
    0.08338,
    -0.00324,
    -0.01738,
    -0.00992,
    0.03481,
    -0.00428,
    0.00878,
    0.01668,
    0.00388,
    -0.01208,
    -0.01273,
    0.01517,
    0.0024,
    0.00406,
    -0.01636,
    0.00291,
    -0.00646,
    0.00158,
    0.01891,
    0.01797,
    0.00844,
    0.01806,
    -0.02503,
    -0.00475,
    -0.00046,
    0.04456,
    0.00446,
    -0.04005,
    -0.00227,
    0.0032,
    0.00229,
    0.03996,
    -0.03091,
    0.00719,
    0.01009,
    0.03554,
    -0.02297,
    -0.01287,
    -0.01394,
    -0.03846,
    -0.00287,
    0.03217,
    0.0117,
    0.00772,
    -0.00615,
    0.02048,
    -0.04236,
    -0.00481,
    -0.01776,
    0.04214,
    0.00162,
    0.00572,
    -0.03039,
    0.04203,
    -0.04505,
    0.00178,
    0.01647,
    -0.00928,
    0.0214,
    0.019,
    0.02202,
    -0.03892,
    -0.03654,
    0.01868,
    -0.01349,
    0.0178,
    -0.04184,
    -0.01734,
    -0.00294,
    -0.009,
    -0.05891,
    0.02652,
    0.01369,
    -0.00461,
    -0.01674,
    -0.0118,
    0.03409,
    0.01348,
    -0.0383,
    0.03693,
    0.01471,
    0.04382,
    0.01035,
    -0.00496,
    -0.05041,
    0.00555,
    -0.00611,
    0.00567,
    -0.00603,
    0.00455,
    0.01419,
    0.01055,
    -0.01328,
    0.04633,
    0.02328,
    0.025,
    -0.04372,
    0.01077,
    -0.05725,
    -0.03081,
    0.02531,
    0.00384,
    -0.0241,
    -0.01039,
    -0.01873,
    0.02991,
    -0.03511,
    0.02133,
    -0.00466,
    -0.00767,
    0.02394,
    -0.04407,
    0.01687,
    0.01733,
    0.02471,
    0.01639,
    -0.00679,
    -0.04831,
    0.00426,
    -0.04442,
    0.03025,
    0.03962,
    0.04737,
    0.04825,
    -0.03861,
    0.01914,
    0.01978,
    0.0053,
    0.00268,
    0.01156,
    0.01131,
    0.0107,
    -0.00795,
    0.00116,
    0.01193,
    0.01968,
    0.01339,
    0.01729,
    -0.03582,
    -0.03634,
    0.00192,
    -0.02973,
    0.02285,
    -0.02993,
    0.00126,
    0.00302,
    -0.00549,
    -0.04252,
    0.02477,
    -0.02661,
    -0.02513,
    -0.01264,
    -0.04256,
    0.0555,
    -0.02403,
    -0.03176,
    -0.03946,
    0.03476,
    0.02285,
    -0.01836,
    0.00688,
    -0.02073,
    -0.02606,
    -0.0338,
    0.00585,
    0.02771,
    0.00572,
    0.00741,
    -0.02877,
    -0.01045,
    -0.02649,
    -0.00605,
    -0.00582,
    -0.00483,
    -0.01419,
    0.01209,
    0.00692,
    -0.00603,
    -0.00778,
    0.03659,
    -0.00545,
    0.00099,
    -0.01791,
    0.03777,
    -0.01021,
    0.01607,
    0.04674,
    -0.00398,
    0.0336,
    0.04134,
    0.0038,
    0.0084,
    0.0275,
    -0.0351,
    0.0265,
    -0.04212,
    0.01137,
    -0.01802,
    -0.01762,
    -0.00542,
    -0.03038,
    -0.01968,
    -0.01473,
    -0.01764,
    -0.01256,
    -0.02776,
    0.01228,
    -0.02301,
    0.03355,
    0.02818,
    0.00028,
    -0.02521,
    0.01671,
    0.00351,
    0.00575,
    -0.01429,
    -0.00175,
    0.02686,
    -0.03994,
    0.00459,
    -0.04587,
    0.01913,
    -0.04947,
    -0.02918,
    -0.01895,
    -0.03252,
    0.0031,
    0.00466,
    -0.04079,
    -0.01817,
    -0.03884,
    0.07023,
    -0.00876,
    0.03073,
    0.01634,
    -0.06876,
    -0.01972,
    0.0395,
    -0.00491,
    -0.03318,
    0.02965,
    0.00147,
    -0.02832,
    -0.01602,
    -0.0112,
    0.03147,
    0.01781,
    0.0112,
    -0.01922,
    -0.02233,
    0.02037,
    -0.00868,
    0.01399,
    -0.05221,
    -0.00468,
    -0.01905,
    0.05072,
    -0.00326,
    0.01112,
    0.01413,
    -0.02813,
    0.01612,
    -0.00297,
    -0.06658,
    0.01478,
    -0.02408,
    -0.03572,
    0.06116,
    0.03097,
    -0.0141,
    -0.01958,
    -0.00922,
    -0.00726,
    0.00198,
    -0.03643,
    0.02267,
    -0.02206,
    0.02202,
    -0.00724,
    0.0002,
    0.00957,
    0.01456,
    -0.00556,
    0.00798,
    0.03802,
    0.0121,
    0.01191,
    -0.03459,
    -0.00247,
    0.03845,
    -0.01966,
    0.06727,
    0.03878,
    0.04956,
    -0.02959,
    -0.0367,
    -0.04218,
    0.00082,
    -0.0648,
    0.01053,
    -0.00973,
    -0.03237,
    -0.00378,
    -0.01232,
    0.02798,
    -0.02607,
    -0.0117,
    -0.01825,
    -0.04579,
    0.00966,
    -0.02829,
    0.0303,
    -0.00968,
    0.00745,
    0.01931,
    -0.00039,
    -0.00196,
    -0.0201,
    -0.03301,
    0.04334,
    0.04748,
    -0.0234,
    0.02459,
    0.03256,
    -0.03319,
    -0.01261,
    -0.00044,
    0.03031,
    -0.02652,
    -0.01563,
    1e-05,
    -0.00753,
    -0.03582,
    -0.00068,
    0.01019,
    -0.02454,
    -0.00568,
    -0.01143,
    -0.04072,
    -0.03351,
    0.0238,
    -0.00312,
    -0.00331,
    0.0095,
    0.00051,
    0.04218,
    -0.00597,
    0.03937,
    0.00388,
    0.01327,
    -0.01214,
    -0.0401,
    0.00126,
    0.02721,
    -0.04459,
    -0.0011,
    -0.03103,
    0.06087,
    0.00349,
    -0.00192,
    0.01242,
    -0.02065,
    -0.02045,
    0.01853,
    -0.02834,
    0.01335,
    -0.01433,
    -0.01416,
    -0.00892,
    0.02082,
    -0.00636,
    0.02401,
    0.00399,
    -0.01224,
    0.01734,
    -0.00166,
    -0.00182,
    -0.02864,
    -0.00249,
    -0.04173,
    0.01587,
    -0.0195,
    0.00339,
    -0.0424,
    0.01366,
    0.01721,
    0.06922,
    -0.02564,
    0.00114,
    0.01902,
    0.0315,
    0.04394,
    -0.04338,
    0.05203,
    -0.00212,
    -0.01691,
    -0.02944,
    -0.02866,
    0.03254,
    -0.0573,
    -0.00339,
    -0.00951,
    -0.01703,
    0.01185,
    -0.03737,
    -0.0057,
    -0.05749,
    -0.0008,
    -0.0132,
    0.00909,
    0.05603,
    -0.00324,
    -0.02368,
    0.04049,
    0.03365,
    -0.02039,
    -0.01924,
    0.03241,
    0.02064,
    0.02725,
    -0.01208,
    -0.04298,
    0.02069,
    -0.04169,
    -0.01149,
    0.00869,
    -0.03019,
    -0.06548,
    0.00066,
    0.04941,
    0.00945,
    0.01526,
    -0.00942,
    -0.02273,
    -0.06925,
    0.00332,
    0.00732,
    0.01592,
    -0.01928,
    -0.01307,
    -0.02434,
    0.01406,
    -0.02472,
    0.01782,
    -0.01388,
    0.01589,
    0.04717,
    0.00343,
    0.01566,
    -0.01725,
    -0.00143,
    0.03485,
    0.02611,
    -0.01092,
    0.0136,
    0.02718,
    -0.03576,
    0.03202,
    0.00067,
    -0.02813,
    0.00885,
    -0.02109,
    0.04339,
    0.0095,
    0.04963,
    0.00681,
    -0.01614,
    0.01169,
    -7e-05,
    0.02415,
    -0.02621,
    -0.03769,
    0.02581,
    -0.00124,
    -0.0208,
    0.03408,
    0.01999,
    -0.03176,
    -0.00249,
    0.02705,
    0.02071,
    -0.00743,
    -0.019,
    -0.03841,
    0.02117,
    -0.00058,
    0.02231,
    0.00481,
    -0.00102,
    0.02404,
    0.00537,
    0.04535,
    -0.02359,
    -0.05793,
    -0.00369,
    -0.01146,
    -0.02789,
    0.0071,
    0.02107,
    -0.04835,
    -0.03916,
    -0.01491,
    -0.00792,
    -0.01222,
    -0.00361,
    -0.04596,
    0.00915,
    0.00775,
    -0.00093,
    0.02627,
    -0.01541,
    -0.02005,
    0.01986,
    0.01635,
    0.00284,
    -0.01725,
    -0.01738,
    -0.00116,
    0.01048,
    -0.00509,
    0.01475,
    0.06015,
    0.00666,
    0.04762,
    -0.08147,
    0.04051,
    -0.01457,
    0.0096,
    0.01623,
    -0.00675,
    0.04215,
    -0.0017,
    -0.03204,
    0.02667,
    -0.0027,
    0.00787,
    0.01665,
    -0.03864,
    0.01081,
    -0.00063,
    0.00973,
    0.05391,
    -0.0581,
    0.0117,
    -0.01134,
    -0.01614,
    0.02839,
    -0.00344,
    -0.04801,
    0.06701,
    -0.04776,
    -0.00933,
    -0.00287,
    0.01222,
    -0.09738,
    0.01028,
    -0.00708,
    0.02458,
    -0.01155,
    -0.03078,
    -0.08615,
    -0.03278,
    -0.02801,
    0.00417,
    0.00736,
    0.0089,
    -0.02354,
    0.01559,
    0.00499,
    -0.00839,
    0.01757,
    0.04052,
    -0.00802,
    -0.02659,
    0.0495,
    -0.02513,
    0.01653,
    0.00052,
    0.0161,
    -0.01479,
    0.02252,
    0.02623,
    0.00184,
    0.0012,
    0.00054,
    0.03313,
    0.06403,
    -0.01733,
    -0.03216,
    -0.02003,
    0.02533,
    -0.00873,
    0.00927,
    0.01851,
    0.01448,
    0.03826,
    -0.01032,
    -0.032,
    0.04587,
    -0.02594,
    -0.02527,
    -0.01808,
    -0.03474,
    -0.0104,
    -0.01916,
    0.01114,
    0.00509,
    0.02757,
    -0.03576,
    0.00398,
    0.01847,
    0.01311,
    -0.01122,
    0.03103,
    -0.00249,
    0.01401,
    0.01473,
    -0.00568,
    0.01264,
    0.00639,
    0.02178,
    -0.00202,
    -0.01329,
    -0.01807,
    -0.00037,
    0.00912,
    -0.0326,
    -0.01523,
    -0.01826,
    0.00864,
    -0.01127,
    -0.04699,
    0.03185,
    0.00057,
    0.00145,
    0.02766,
    0.00055,
    -0.00757,
    0.01583,
    -0.00399,
    -0.0017,
    -0.0124,
    0.03116,
    0.01891,
    -0.02251,
    0.02032,
    -0.01759,
    0.0104,
    -0.0084,
    -0.02095,
    -0.04318,
    -0.02761,
    0.02533,
    -0.01995,
    0.02361,
    0.01616,
    -0.01297,
    -0.04685,
    -0.01703,
    0.01461,
    -0.06208,
    0.03403,
    0.0373,
    -0.01203,
    -0.01946,
    0.01547,
    0.04097,
    -0.0467,
    -0.03075,
    0.03319,
    -0.03132,
    -0.00276,
    -0.01972,
    -0.01494,
    0.00482,
    -0.00463,
    0.03057,
    -0.01548,
    0.03089,
    -0.00705,
    0.01874,
    0.02142,
    0.01042,
    0.00489,
    0.00837,
    0.02892,
    0.02018,
    0.01354,
    -0.02634,
    -0.0607,
    -0.01687,
    0.00339,
    0.00363,
    0.02101,
    -0.00767,
    -0.03178,
    -0.02058,
    0.01328,
    0.0258,
    0.00317,
    0.00805,
    0.01805,
    -0.06715,
    0.01723,
    0.00241,
    -0.01231,
    -0.03108,
    -0.0035,
    -0.03015,
    0.02107,
    -0.0121,
    -0.0395,
    -0.03811,
    0.00421,
    0.00674,
    -0.01237,
    -0.04727,
    -0.0005,
    0.03118,
    -0.00206,
    0.01678,
    -0.00824,
    -0.00385,
    0.01863,
    0.02466,
    -0.00474,
    -0.00797,
    -0.00071,
    -0.0328,
    0.00093,
    0.00976,
    -0.05737,
    -0.02557,
    -0.01422,
    0.03176,
    0.017,
    -0.00903,
    -0.0174,
    -0.02116,
    -0.0038,
    0.03574,
    0.02352,
    -0.01478,
    -0.02161,
    0.00108,
    -0.0023,
    -0.01308,
    0.00438,
    0.03376,
    0.04001,
    0.00199,
    0.01287,
    0.01751,
    -0.01704,
    0.02498,
    0.04121,
    0.02298,
    0.00783,
    0.03053,
    -0.03984,
    -0.03215,
    0.01319,
    0.01342,
    0.01086,
    0.02143,
    -0.02592,
    -0.00474,
    0.01818,
    0.01676,
    0.00847,
    0.01122,
    -0.02101,
    -0.00395,
    -0.03145,
    -0.00017,
    -0.00495,
    0.01651,
    -0.0331,
    -0.04273,
    0.02326,
    0.01622,
    0.00199,
    -0.01658,
    0.01998,
    -0.0049,
    -0.0034,
    0.03386,
    -0.02719,
    -0.01754,
    0.00452,
    -0.00298,
    0.0348,
    -0.01641,
    -0.04325,
    0.04654,
    -0.01176,
    -0.01217,
    -0.00279,
    0.05001,
    0.00809,
    0.03918,
    -0.00809,
    -0.06119,
    -0.01473,
    -0.01612,
    -0.00261,
    0.00268,
    -0.00196,
    0.01403,
    -0.01352,
    0.01687,
    0.00723,
    0.00797,
    -0.03621,
    -0.01209,
    -0.01831,
    0.01229,
    -0.00625,
    0.00359,
    0.03536,
    0.02705,
    -0.02359,
    -0.00978,
    0.00102,
    0.01195,
    -0.02558,
    0.00083,
    0.03748,
    0.02767,
    -0.03892,
    0.01261,
    0.04703,
    0.01503,
    -0.01332,
    0.01957,
    0.02586,
    0.04894,
    0.03179,
    0.01351,
    0.02676,
    0.00336,
    0.01058,
    -0.0264,
    -0.01933,
    0.01865,
    0.0184,
    -0.00059,
    -0.00054,
    -0.05198,
    -0.02621,
    0.01022,
    -0.00615,
    -0.0002,
    -0.0139,
    -0.01365,
    -0.01001,
    -0.00554,
    -0.0262,
    0.01732,
    0.01457,
    -0.03919,
    -0.03351,
    -0.013,
    -0.00912,
    0.01214,
    0.01352,
    0.00618,
    0.03273,
    0.0068,
    0.00582,
    0.01571,
    -0.01805,
    0.01306,
    0.01839,
    0.01024,
    0.04011,
    0.01233,
    0.03705,
    -0.01798,
    0.03678,
    -0.01147,
    -0.01294,
    0.0063,
    0.0026,
    0.02197,
    0.00541,
    0.00308,
    -0.00835,
    0.01884,
    -0.00772,
    -0.03719,
    -0.01549,
    -0.00703,
    0.01529,
    0.01232,
    0.02291,
    -0.00093,
    -0.02804,
    0.05546,
    -0.05065,
    -0.03992,
    -0.0268,
    0.03162,
    0.00668,
    -0.01894,
    -0.02531,
    0.00257,
    0.00857,
    0.02699,
    -0.02235,
    -0.00494,
    0.00721,
    0.04255,
    0.03193,
    -0.02809,
    -0.01906,
    -0.0013,
    0.01089,
    -0.02185,
    -0.01278,
    0.00176,
    -0.02096,
    0.02101,
    -0.06344,
    -0.00799,
    -0.03146,
    0.02251,
    0.0083,
    -0.0129,
    -0.01053,
    0.0126,
    0.01195,
    -0.02317,
    0.01628,
    -0.01254,
    0.00806,
    -0.0267,
    -0.00216,
    -0.01414,
    0.02811,
    0.0174,
    0.02264,
    -0.00644,
    -0.0019,
    0.04601,
    -0.00447,
    -0.02177,
    -0.01332,
    -0.07466,
    -0.00901,
    -0.0409,
    0.00119,
    0.0121,
    0.00068,
    0.00316,
    -0.00921,
    -0.023,
    -0.00209,
    -0.01589,
    0.00972,
    -0.0314,
    -0.04978,
    0.01317,
    0.00214,
    0.00155,
    -0.04148,
    -0.02431,
    0.01567,
    -0.01181,
    -0.0572,
    -0.02094,
    0.00746,
    0.01329,
    0.01566,
    -0.0419,
    -0.00338,
    0.04651,
    0.04977,
    0.02059,
    0.03836,
    -0.00216,
    -0.02093,
    0.0024,
    0.028,
    -0.01573,
    -0.02691,
    0.03101,
    -0.03343,
    0.03178,
    -0.00879,
    -0.00651,
    -0.0165,
    0.00695,
    -0.0175,
    0.02304,
    0.00484,
    -0.01012,
    -0.0529,
    0.00593,
    -0.01597,
    0.03189,
    0.01174,
    0.01496,
    -0.00625,
    -0.01206,
    -0.03442,
    -0.02977,
    -0.01468,
    -0.05405,
    0.02954,
    0.03211,
    -0.00478,
    0.02485,
    -0.02711,
    -0.01981,
    -0.04207,
    -0.04228,
    0.08135,
    -0.04198,
    0.0428,
    -0.01718,
    0.01375,
    -0.00423,
    0.012,
    0.01004,
    0.00958,
    -0.00457,
    -0.00259,
    -0.01345,
    0.01495,
    0.00132,
    -0.01427,
    0.0021,
    0.02712,
    0.00261,
    0.02785,
    0.03539,
    -0.02878,
    -0.01548,
    0.01924,
    0.03066,
    -0.04069,
    -0.02304,
    0.02939,
    -0.04268,
    -0.01011,
    -0.01013,
    0.00476,
    0.01952,
    0.02026,
    0.00152,
    -0.01582,
    -0.02339,
    0.02081,
    -0.00102,
    0.00398,
    0.03003,
    -0.01389,
    0.01041,
    -0.05089,
    0.01836,
    -0.0071,
    0.03431,
    0.01598,
    0.01185,
    0.02372,
    -0.00664,
    -0.041,
    0.03919,
    0.02801,
    0.03029,
    0.00573,
    0.06077,
    -0.01161,
    0.01788,
    0.01996,
    -0.00476,
    -0.02903,
    0.01403,
    0.05268,
    -0.03205,
    -0.03393,
    0.00486,
    0.01529,
    -0.00119,
    0.00852,
    -0.00736,
    -0.01537,
    -0.02979,
    -0.02172,
    0.00682,
    0.00351,
    0.04315,
    -0.03432,
    0.06833,
    -0.00913,
    -0.04153,
    -0.02974,
    0.00498,
    0.00354,
    -0.07657,
    0.02969,
    0.03056,
    -0.01294,
    0.02229,
    0.04787,
    -0.04221,
    -0.00114,
    -0.02515,
    -0.00152,
    -0.01341,
    0.00977,
    0.00455,
    -0.00654,
    0.00487,
    -0.01327,
    -0.00973,
    -0.0056,
    -0.03563,
    -0.03586,
    0.02122,
    0.00236,
    0.068,
    -0.01474,
    0.03007,
    0.02761,
    0.00182,
    -0.02429,
    -0.01224,
    0.02953,
    0.01911,
    0.00374,
    -0.01597,
    0.02777,
    -0.03366,
    -0.02244,
    -0.03916,
    0.00235,
    0.02123,
    0.00703,
    0.0103,
    0.01599,
    -0.01738,
    -0.03201,
    0.0128,
    -0.03308,
    -0.01278,
    -0.00695,
    -0.01189,
    0.06537,
    -0.02381,
    -0.00448,
    -0.02481,
    0.00115,
    0.00274,
    0.01048,
    -0.01772,
    -0.00486,
    0.05398,
    -0.02265,
    -0.02961,
    0.0006,
    -0.00465,
    0.03282,
    0.01723,
    -0.0423,
    -0.04804,
    0.01592,
    0.01696,
    0.0079,
    -0.01142,
    0.02803,
    0.02028,
    -0.0373,
    -0.00368,
    -0.0018,
    -0.02576,
    -0.00246,
    0.02744,
    0.0209,
    -0.04926,
    0.00517,
    -0.02123,
    -0.00196,
    -0.00978,
    0.00857,
    -0.04351,
    -0.03779,
    -0.00666,
    -0.01831,
    0.03382,
    0.01894,
    0.00993,
    0.01414,
    -0.0068,
    0.01752,
    -0.00318,
    0.04403,
    -0.00569,
    -0.01325,
    -0.04377,
    -0.01518,
    -0.02269,
    -0.02417,
    0.03626,
    -0.00048,
    -0.00743,
    -0.00015,
    0.02973,
    0.00555,
    -0.02364,
    -0.0198,
    0.02228,
    0.02602,
    -0.00968,
    -0.02231,
    -0.00482,
    0.02359,
    0.01283,
    0.00081,
    0.04956,
    -0.02058,
    -0.0204,
    0.03288,
    -0.03458,
    -0.03134,
    -0.04609,
    0.01815,
    0.02587,
    0.00061,
    -0.01376,
    -0.01099,
    0.00126,
    0.00678,
    0.01858,
    0.01823,
    0.00555,
    -0.04235,
    -0.00619,
    -0.01787,
    0.00508,
    -0.00157,
    0.01996,
    0.01823,
    -0.00052,
    0.0129,
    -0.00269,
    -0.0151,
    -0.01418,
    -0.01294,
    0.01055,
    -0.01743,
    -0.02083,
    0.00381,
    -0.00047,
    -0.03571,
    -0.01315,
    -0.01097,
    -0.00978,
    -0.01855,
    -0.00279,
    0.02434,
    -0.00885,
    -0.02418,
    0.01927,
    0.00763,
    -0.00361,
    0.00223,
    -0.00726,
    -0.04886,
    0.00114,
    0.00765,
    -0.02247,
    0.00939,
    -0.01352,
    0.01737,
    0.03237,
    -0.02529,
    0.01414,
    0.05517,
    -0.01666,
    -0.01982,
    0.01361,
    -0.01748,
    -0.00567,
    0.04075,
    0.01792,
    -0.06174,
    0.0677,
    0.00918,
    0.08792,
    -0.01534,
    0.02832,
    -0.01627,
    -0.00185,
    0.01498,
    0.02071,
    0.02483,
    0.02454,
    -0.00706,
    0.02081,
    0.04276,
    0.02311,
    -0.0137,
    -0.01527,
    0.00652,
    0.06472,
    0.02543,
    0.01959,
    0.00396,
    -0.02397,
    0.0188,
    0.01068,
    -0.00225,
    0.02127,
    0.01598,
    -0.02585,
    -0.0259,
    -0.01843,
    0.00124,
    0.00026,
    -0.03058,
    -0.0043,
    0.0021,
    0.0107,
    -0.01301,
    -0.03137,
    0.03259,
    0.00319,
    -0.0189,
    0.01534,
    0.0239,
    -0.00971,
    0.02377,
    -0.03335,
    -0.02375,
    -0.00911,
    -0.02014,
    -0.01434,
    0.02734,
    -0.01067,
    0.02141,
    -0.00657,
    -0.03308,
    -1e-05,
    -0.01879,
    0.00169,
    -0.00015,
    -0.0273,
    0.02084,
    0.01621,
    0.01964,
    -0.0285,
    -0.00101,
    0.01531,
    0.00533,
    0.02932,
    -0.00852,
    0.03216,
    -0.03077,
    0.04239,
    -0.02104,
    -0.01139,
    0.02004,
    0.01897,
    -0.00588,
    -0.00302,
    0.02255,
    0.01115,
    0.0393,
    -0.01655,
    -0.00967,
    0.00202,
    0.03524,
    0.01778,
    0.01524,
    -0.04345,
    -0.01325,
    0.01101,
    -0.00912,
    -0.01733,
    -0.0121,
    -0.01991,
    -0.03433,
    0.02771,
    -0.0427,
    -0.01017,
    -0.04966,
    -0.0244,
    -0.02431,
    0.04002,
    0.05771,
    0.02974,
    -0.03921,
    -0.01953,
    -0.01546,
    -0.0428,
    0.0242,
    -0.02489,
    -0.00621,
    0.0079,
    0.04353
   ],
   "NUMBNESS": [
    0.03643,
    -0.01363,
    0.01391,
    0.03489,
    -0.0168,
    0.01679,
    0.02032,
    -0.00895,
    0.03681,
    0.02371,
    0.01595,
    -0.01182,
    0.02146,
    0.00887,
    0.00574,
    -0.01172,
    -0.01306,
    -0.00661,
    0.02512,
    -0.0408,
    -0.01379,
    0.02539,
    -0.00653,
    0.00615,
    0.01325,
    0.02375,
    -0.00156,
    -0.01355,
    0.04448,
    -0.02199,
    -0.01145,
    0.04853,
    -0.03684,
    0.02615,
    -0.01321,
    -0.02154,
    -0.02493,
    0.02138,
    -0.00147,
    -0.02876,
    0.03047,
    -0.01721,
    0.04522,
    0.02824,
    0.00955,
    0.00581,
    0.02739,
    -0.03812,
    -0.00977,
    0.00231,
    0.04504,
    -0.01496,
    0.00089,
    -0.02226,
    0.06344,
    0.01383,
    0.00558,
    -0.01738,
    -0.01048,
    0.01896,
    0.00785,
    0.00042,
    0.00338,
    -0.00156,
    0.01091,
    0.00118,
    -0.01277,
    0.014,
    0.0153,
    0.03331,
    0.01853,
    -0.02873,
    -0.03136,
    0.03337,
    -0.01074,
    0.03912,
    0.01168,
    -0.00689,
    -0.03036,
    0.00147,
    0.0686,
    0.00773,
    -0.01381,
    0.01194,
    0.02378,
    -0.01942,
    -0.00515,
    -0.00155,
    -0.00042,
    0.00371,
    -0.01741,
    -0.00113,
    -0.01705,
    0.01075,
    0.01488,
    0.07251,
    0.02003,
    -0.02034,
    0.00596,
    -0.02978,
    0.00801,
    -0.02663,
    0.01477,
    -0.04336,
    0.02633,
    -0.00145,
    0.00551,
    -0.01977,
    0.01849,
    -0.00242,
    -0.01588,
    -0.0102,
    -0.02179,
    0.00999,
    0.02205,
    -0.00811,
    -0.02094,
    0.01008,
    0.02525,
    -0.00141,
    -0.02297,
    0.00782,
    0.0099,
    0.0195,
    0.04046,
    0.01896,
    0.00666,
    0.02777,
    0.03851,
    -0.0022,
    -0.0213,
    0.00548,
    -0.01743,
    0.04648,
    -0.05766,
    0.01128,
    -0.0015,
    -0.00422,
    -0.01329,
    0.01503,
    -0.01314,
    0.03321,
    0.01571,
    -0.03335,
    0.01821,
    0.04498,
    -0.04451,
    0.00585,
    0.01843,
    -0.03592,
    0.00667,
    -0.00163,
    0.00756,
    -0.00247,
    0.0145,
    0.01223,
    0.04103,
    -0.00357,
    0.00918,
    -0.02611,
    -0.04145,
    0.01751,
    0.02434,
    0.03719,
    0.00371,
    -0.02227,
    0.02896,
    -0.00132,
    0.03772,
    0.02095,
    0.00041,
    -0.01732,
    0.0222,
    -0.00323,
    0.0175,
    -0.0166,
    0.02202,
    0.00284,
    -0.01994,
    -0.00122,
    0.03193,
    0.00309,
    0.01759,
    -0.0393,
    0.00553,
    0.06154,
    -0.01333,
    -0.02913,
    -0.01208,
    -0.00939,
    -0.03594,
    0.00306,
    -0.0487,
    0.00446,
    -0.04149,
    0.01984,
    0.03906,
    -0.0243,
    0.0406,
    6e-05,
    0.02454,
    0.00588,
    -0.03781,
    0.02302,
    0.0306,
    -0.02289,
    -0.01701,
    0.0066,
    0.04867,
    -0.00727,
    -0.0108,
    -0.01416,
    -0.01836,
    -0.05758,
    0.01312,
    -0.0217,
    0.01299,
    -0.01615,
    -0.00025,
    0.01843,
    -0.00979,
    0.01498,
    0.00126,
    0.02142,
    -0.02836,
    -0.01471,
    0.01963,
    -0.02887,
    -0.00678,
    0.01617,
    0.06396,
    -0.02009,
    0.04299,
    0.04217,
    -0.01926,
    0.01422,
    0.06633,
    0.02268,
    -0.00488,
    -0.03195,
    0.01597,
    -0.03304,
    0.01405,
    -0.05362,
    -0.0239,
    0.01884,
    0.02281,
    -0.01039,
    0.00663,
    0.01785,
    -0.02889,
    -0.02636,
    0.02335,
    -0.01293,
    -0.02676,
    0.0039,
    -0.00169,
    -0.02317,
    0.03319,
    0.00602,
    -0.00361,
    0.02763,
    0.02057,
    0.03157,
    -0.02133,
    -0.01448,
    0.02091,
    -0.02899,
    0.01875,
    -0.02884,
    -0.00357,
    -0.02294,
    0.03093,
    0.01083,
    0.02237,
    0.02652,
    0.02439,
    0.01791,
    0.01261,
    0.00392,
    -0.00923,
    0.04003,
    -0.02073,
    0.01469,
    -0.04218,
    0.0329,
    0.03368,
    -0.00348,
    -0.00692,
    -0.00547,
    0.01593,
    -0.03349,
    0.00171,
    0.02158,
    -0.00579,
    0.00286,
    0.03934,
    0.03336,
    -0.01186,
    -0.06624,
    0.00795,
    0.02057,
    0.0137,
    -0.01029,
    0.02557,
    0.0048,
    -0.02789,
    0.00385,
    0.0044,
    -0.00316,
    0.00385,
    0.02982,
    -0.01589,
    -0.03753,
    0.0011,
    -0.0246,
    0.01241,
    -0.0123,
    0.0033,
    -0.02568,
    -0.0698,
    -0.01767,
    -0.01365,
    0.01619,
    -0.00735,
    -0.02101,
    0.00534,
    -0.01928,
    -0.01319,
    0.03618,
    0.00249,
    0.03144,
    -0.0143,
    -0.00259,
    -0.00623,
    0.00567,
    0.01918,
    -0.00318,
    0.03814,
    -0.00102,
    0.00735,
    0.0476,
    -0.01049,
    -0.03625,
    -0.02477,
    0.00258,
    -0.02358,
    -0.01798,
    0.00317,
    0.02847,
    0.00651,
    -0.00983,
    0.02524,
    -0.01392,
    0.04732,
    0.00132,
    0.01232,
    0.00753,
    0.04993,
    0.01953,
    0.0312,
    -0.00023,
    0.03766,
    0.03651,
    0.0108,
    -0.04735,
    0.01754,
    -0.03097,
    -0.00391,
    0.06809,
    -0.00163,
    -0.00474,
    0.0227,
    0.01569,
    -0.05554,
    0.00658,
    -0.0058,
    -0.01757,
    -0.0168,
    0.01167,
    -0.02739,
    -0.00825,
    0.01859,
    0.03457,
    -0.00243,
    -0.00778,
    -0.02107,
    0.00721,
    0.00311,
    0.01664,
    -0.03626,
    -0.02114,
    0.00659,
    -0.03396,
    0.0086,
    -0.02233,
    0.00303,
    -0.01885,
    -0.00127,
    0.00843,
    0.03748,
    -0.06257,
    0.0221,
    0.0036,
    0.00651,
    -0.0017,
    0.02066,
    0.01181,
    0.03182,
    0.01549,
    0.00729,
    0.00856,
    0.0049,
    -0.01337,
    0.0103,
    0.01341,
    -0.02607,
    0.01352,
    0.00227,
    0.00046,
    0.02958,
    0.04679,
    0.01999,
    0.0134,
    0.03084,
    -0.03246,
    -0.0232,
    -0.02659,
    0.01661,
    0.02032,
    0.02548,
    0.00226,
    0.01509,
    -0.01152,
    0.02206,
    -0.03771,
    -0.0204,
    0.00368,
    -0.02596,
    0.0297,
    0.00394,
    0.00326,
    0.05963,
    0.03606,
    -0.00248,
    -0.01245,
    -0.00425,
    0.02742,
    -0.01639,
    -0.01621,
    0.01003,
    -0.0049,
    0.0476,
    -0.05996,
    0.01223,
    0.04453,
    -0.01586,
    -0.01865,
    -0.0279,
    0.01014,
    0.0227,
    -0.04525,
    0.00857,
    0.01339,
    0.00674,
    0.04285,
    -0.0363,
    0.02467,
    0.01977,
    0.04523,
    -0.03841,
    -0.02208,
    0.0016,
    0.06701,
    -0.02633,
    -0.00777,
    -0.00793,
    5e-05,
    -0.01354,
    -0.0247,
    -0.01224,
    -0.03529,
    -0.01098,
    -0.00668,
    -0.00871,
    0.0086,
    0.00979,
    -0.02867,
    0.0189,
    -0.02278,
    -0.01273,
    -0.03197,
    0.01569,
    0.00563,
    -0.0278,
    0.00086,
    -0.00976,
    0.01971,
    -0.02069,
    0.00472,
    -0.03774,
    0.02268,
    0.00904,
    -0.00425,
    0.03311,
    0.02759,
    -0.01986,
    0.00724,
    -0.00361,
    -0.00252,
    0.02424,
    0.00545,
    -0.00404,
    0.02562,
    -0.00903,
    0.00627,
    -0.03407,
    0.03487,
    -0.01715,
    0.02197,
    -0.00069,
    0.02594,
    -0.00676,
    -0.00514,
    -0.03435,
    -0.02952,
    0.00345,
    -0.03314,
    -0.01635,
    -0.02688,
    0.00965,
    -0.01366,
    -0.01491,
    0.00873,
    0.00864,
    0.02082,
    -0.00521,
    0.00335,
    0.02478,
    0.02831,
    0.02639,
    -0.00653,
    0.03333,
    0.00577,
    -0.02628,
    -0.00304,
    0.05302,
    -0.01497,
    0.01438,
    0.00708,
    0.01097,
    -0.06448,
    -0.03465,
    -0.00075,
    0.0228,
    0.03394,
    0.00722,
    0.00612,
    0.00393,
    0.00865,
    0.00509,
    0.02874,
    0.00542,
    -0.05111,
    -0.0277,
    -8e-05,
    0.01301,
    0.01103,
    0.0478,
    0.01612,
    0.00795,
    -0.0124,
    0.01623,
    0.00494,
    -0.01373,
    -0.01667,
    0.02427,
    -0.02046,
    -0.02625,
    0.00704,
    0.04142,
    -0.01062,
    -0.00435,
    0.02077,
    0.00478,
    -0.01942,
    -0.02687,
    0.02283,
    -0.00855,
    -0.01162,
    0.05572,
    -0.01709,
    -0.00626,
    0.03645,
    0.00984,
    0.01488,
    -0.04235,
    -0.01338,
    -0.02627,
    0.00891,
    0.03231,
    -0.02847,
    -0.03818,
    0.00791,
    0.02198,
    0.00933,
    -0.01983,
    0.02224,
    0.06576,
    -0.05667,
    -0.00565,
    -0.00882,
    0.01228,
    -0.00853,
    -0.01135,
    0.01001,
    -0.01748,
    -0.00131,
    0.0025,
    -0.01286,
    -0.0424,
    -0.0174,
    0.009,
    -0.02408,
    0.02173,
    -0.02612,
    0.02135,
    0.03674,
    0.01983,
    0.01659,
    0.00943,
    -0.02299,
    -0.01944,
    0.05862,
    0.02467,
    -0.01319,
    -0.00219,
    -0.01426,
    -0.03803,
    0.00803,
    0.00688,
    0.01057,
    -0.01547,
    -0.02813,
    0.02017,
    0.00924,
    -0.0222,
    0.02648,
    0.01767,
    -0.0049,
    0.04024,
    -0.00311,
    0.00846,
    0.00981,
    -0.01826,
    -0.04898,
    -0.00796,
    -0.05041,
    0.05909,
    -0.04574,
    -0.02414,
    -0.01217,
    -0.0019,
    0.01026,
    -0.02234,
    -0.0165,
    0.03518,
    -0.00245,
    4e-05,
    -0.00468,
    -0.02214,
    -0.02372,
    -0.03743,
    0.07112,
    -0.02929,
    0.00981,
    -0.01097,
    -0.00199,
    0.00356,
    0.03714,
    -0.0308,
    -0.00867,
    0.00157,
    -0.04677,
    0.01262,
    -0.02915,
    0.00575,
    0.01242,
    0.05784,
    -0.04031,
    0.01898,
    0.01285,
    0.01537,
    0.07354,
    0.04224,
    0.03092,
    -0.0113,
    -0.03707,
    -0.01959,
    0.00678,
    0.02564,
    -0.01677,
    0.03635,
    0.00483,
    -0.0195,
    -0.01916,
    -0.0027,
    0.04456,
    -0.05007,
    0.00927,
    -0.00973,
    -0.01334,
    0.00652,
    0.00858,
    -0.01677,
    -0.03131,
    -0.0042,
    -0.01555,
    0.03437,
    -0.02029,
    -0.02319,
    -0.02967,
    0.03315,
    0.02262,
    0.02279,
    -0.00475,
    -0.03537,
    -0.0092,
    -0.01144,
    -0.0411,
    -0.04196,
    0.00072,
    -0.01296,
    -0.00444,
    0.06586,
    0.0357,
    0.04818,
    0.03139,
    0.01186,
    -0.01495,
    0.01474,
    -0.05045,
    -0.04059,
    -0.01286,
    -0.04221,
    -0.04026,
    0.0152,
    -0.0152,
    -0.00072,
    -0.01667,
    -0.01146,
    0.04617,
    0.00436,
    -0.02506,
    -0.03908,
    0.00502,
    0.00208,
    -0.0172,
    -0.04441,
    0.02048,
    0.05977,
    0.01736,
    -0.02406,
    0.01741,
    0.01835,
    0.04769,
    -0.03499,
    -0.02087,
    -0.03343,
    -0.04384,
    -0.02559,
    0.03805,
    -0.02161,
    0.00848,
    -0.00593,
    0.04732,
    -0.05304,
    -0.0038,
    0.03807,
    0.00792,
    0.04059,
    -0.03137,
    0.04135,
    -0.0113,
    0.04665,
    0.03774,
    -0.02636,
    0.00706,
    -0.02959,
    -0.00285,
    0.00492,
    0.06729,
    0.02473,
    0.03032,
    0.06394,
    -0.04529,
    -0.01263,
    -0.00544,
    0.01671,
    0.00014,
    -0.00422,
    0.03196,
    -0.00212,
    -0.03475,
    0.05139,
    -0.00843,
    0.0075,
    0.04734,
    -0.00843,
    -0.01013,
    -0.01986,
    0.02581,
    0.00271,
    -0.03586,
    0.0313,
    -0.0415,
    0.00501,
    -0.01621,
    0.03746,
    0.01521,
    -0.01632,
    0.00218,
    -0.03312,
    0.03001,
    0.00964,
    0.03401,
    0.00793,
    0.0066,
    0.01397,
    0.00191,
    0.02226,
    -0.00555,
    -0.00758,
    -0.00238,
    0.01911,
    0.03421,
    0.0551,
    0.00092,
    -0.00682,
    0.00195,
    -0.0158,
    0.03362,
    0.00363,
    -0.00713,
    -0.01099,
    0.00099,
    0.0052,
    -0.02074,
    -0.03185,
    0.01961,
    0.03828,
    -0.02677,
    -0.02565,
    -0.00256,
    0.00969,
    -0.00742,
    0.01754,
    0.0505,
    0.03586,
    0.00929,
    0.01437,
    -0.04335,
    0.04354,
    -0.01379,
    -0.02281,
    0.04934,
    -0.0091,
    0.02125,
    -0.05407,
    0.02329,
    -0.02752,
    0.02267,
    0.01531,
    -0.00827,
    -0.03139,
    -0.00417,
    0.00559,
    0.02265,
    -0.01061,
    0.01302,
    0.00789,
    0.00306,
    -0.0314,
    -0.02164,
    -0.00981,
    0.01219,
    -0.00885,
    -0.00184,
    -0.03551,
    -0.00983,
    0.01517,
    -0.00958,
    -0.01979,
    0.01004,
    0.02423,
    -0.02846,
    -0.00794,
    -0.02194,
    -0.02794,
    0.03537,
    0.01892,
    0.04368,
    0.00398,
    -0.05058,
    -0.0021,
    0.01323,
    -0.03832,
    0.04014,
    -0.00752,
    -0.0171,
    -0.02864,
    -0.00993,
    0.00798,
    -0.00439,
    0.04359,
    -0.02326,
    0.00998,
    -0.01219,
    -0.02567,
    -0.00476,
    -0.01642,
    0.04504,
    0.05271,
    -0.01856,
    -0.02184,
    -0.02342,
    -0.02537,
    0.01728,
    -0.00861,
    0.01281,
    0.03089,
    -0.02372,
    -0.03962,
    -0.02407,
    -0.03521,
    -0.03047,
    0.01481,
    0.00427,
    0.0136,
    0.00459,
    0.01839,
    0.02265,
    -0.00297,
    -0.01876,
    -0.01631,
    0.02044,
    -0.02579,
    0.01859,
    0.01411,
    -0.01411,
    -0.03693,
    0.01672,
    0.00217,
    -0.04149,
    0.00416,
    0.04071,
    -0.01245,
    0.01984,
    -0.02161,
    0.00603,
    -0.02943,
    -0.00858,
    -0.00924,
    0.04257,
    -0.0381,
    -0.06674,
    -0.0109,
    -0.00331,
    0.00204,
    0.01489,
    0.00691,
    -0.00336,
    -0.01434,
    -0.04852,
    -0.00094,
    -0.0028,
    0.03486,
    0.0453,
    -0.00338,
    0.03177,
    -0.0018,
    -0.01422,
    0.05726,
    -0.01515,
    -0.04649,
    -0.03423,
    0.01594,
    -0.01018,
    0.01857,
    0.00887,
    -0.00829,
    0.02077,
    0.02164,
    -0.02687,
    0.02074,
    0.02054,
    0.02631,
    0.01712,
    -0.02496,
    0.0277,
    -0.01945,
    0.00389,
    0.00615,
    0.04269,
    -0.02097,
    0.00306,
    -0.00432,
    0.02846,
    0.03507,
    -0.0262,
    0.01058,
    0.01164,
    -0.01986,
    -0.02445,
    -0.0157,
    0.03266,
    -0.02417,
    0.04227,
    1e-05,
    0.01086,
    -0.01742,
    0.00978,
    -0.02742,
    -0.02016,
    0.02985,
    0.01543,
    0.00996,
    -0.02033,
    0.00321,
    -0.019,
    -0.01777,
    -0.00568,
    0.007,
    0.03935,
    0.03121,
    0.00344,
    -0.01669,
    -0.01932,
    0.01692,
    0.0272,
    -0.0082,
    0.01024,
    -0.03689,
    0.02795,
    -0.01325,
    0.02926,
    -0.03431,
    -0.01419,
    -0.0208,
    0.03561,
    0.00126,
    0.01616,
    0.01011,
    0.03799,
    -0.02314,
    0.00035,
    -0.02544,
    0.06365,
    -0.01867,
    0.018,
    -0.03103,
    -0.00246,
    -0.01877,
    -0.00039,
    -0.01663,
    -0.00126,
    0.02836,
    0.00308,
    -0.03748,
    -0.0215,
    -0.00319,
    0.00743,
    0.03402,
    0.03419,
    -0.05634,
    0.00217,
    -0.03183,
    0.00325,
    -0.00417,
    0.00513,
    0.03512,
    -0.01154,
    0.04054,
    -0.01718,
    0.00128,
    0.03085,
    -0.00714,
    0.01302,
    0.00215,
    -0.02994,
    0.04518,
    -0.01682,
    -0.01677,
    -0.01583,
    0.00594,
    0.05444,
    -0.00537,
    0.0037,
    0.00031,
    0.06116,
    0.02817,
    0.01925,
    0.02612,
    0.04605,
    0.01074,
    0.00028,
    -0.01677,
    0.00793,
    0.00197,
    0.0088,
    -0.0339,
    -0.04343,
    -0.00696,
    -0.00104,
    -0.04172,
    0.00145,
    -0.02544,
    -0.00591,
    0.02947,
    0.01892,
    0.00146,
    -0.01711,
    -0.01512,
    -0.06388,
    0.00221,
    0.01454,
    0.02911,
    0.0443,
    -0.01064,
    -0.02125,
    0.02431,
    0.01395,
    -0.04135,
    0.02532,
    -0.0045,
    0.00261,
    -0.00441,
    0.00474,
    0.02426,
    -0.01268,
    0.02006,
    0.02284,
    0.06466,
    -0.0248,
    -0.0193,
    -0.01673,
    0.015,
    -0.0211,
    -0.00269,
    0.0191,
    -0.01714,
    -0.04106,
    0.05629,
    -0.04031,
    0.02247,
    -0.01304,
    -0.00241,
    -0.0013,
    0.03352,
    0.0045,
    0.01155,
    -0.0192,
    -0.01292,
    -0.0202,
    0.00544,
    0.02589,
    -0.01427,
    0.00933,
    -0.00029,
    -0.02487,
    0.01829,
    0.03148,
    0.00629,
    -0.02349,
    -0.04869,
    0.03664,
    0.01471,
    -0.0361,
    0.03257,
    -0.01673,
    0.02359,
    0.02034,
    -0.05218,
    0.00944,
    -0.03858,
    0.0329,
    0.01996,
    0.00128,
    0.00765,
    0.0107,
    -0.03157,
    0.02507,
    0.0025,
    0.03722,
    -0.02212,
    -0.00415,
    -0.00724,
    -0.0169,
    0.00119,
    0.00893,
    0.01405,
    0.06841,
    -0.02781,
    -0.00565,
    -0.02796,
    0.00772,
    -0.04962,
    0.01193,
    -0.02788,
    -0.02732,
    -0.00891,
    0.04031,
    -0.00781,
    -0.06554,
    -0.00998,
    -0.02234,
    0.00879,
    -0.02093,
    0.02037,
    -0.0421,
    -0.0328,
    0.00384,
    0.01006,
    0.00046,
    -0.01165,
    0.00109,
    -0.0105,
    -0.00136,
    -0.00179,
    -0.02783,
    0.02476,
    0.01815,
    -0.00386,
    -0.01148,
    0.06269,
    0.00329,
    -0.03681,
    0.00919,
    -0.0317,
    -0.02411,
    0.03006,
    -0.00549,
    0.01737,
    -0.00298,
    0.00429,
    -0.00601,
    0.01759,
    -0.01201,
    0.05804,
    -0.01142,
    0.01674,
    0.01389,
    -0.00546,
    0.03701,
    -0.04491,
    0.0283,
    -0.04432,
    -0.00615,
    -0.01337,
    -0.01613,
    0.01425,
    -0.00751,
    -0.00405,
    -0.00715,
    0.0045,
    -0.0041,
    0.01986,
    -0.03994,
    0.01365,
    0.02606,
    0.0598,
    0.01242,
    -0.0476,
    0.01928,
    -0.00468,
    -0.02941,
    0.03435,
    0.04625,
    -0.00268,
    -6e-05,
    0.023,
    -0.00171,
    -0.00167,
    -0.03353,
    -0.02329,
    -0.04312,
    0.00746,
    0.02948,
    -0.01079,
    0.00061,
    0.02513,
    0.01647,
    -0.04285,
    0.01046,
    0.00899,
    0.00306,
    -0.00372,
    -0.02181,
    -0.02868,
    -0.01869,
    -0.02742,
    0.06442,
    -0.00788,
    0.01485,
    -0.00747,
    0.00373,
    -0.03316,
    0.02389,
    -0.0248,
    0.02664,
    0.02399,
    0.01776,
    0.00536,
    0.00638,
    0.00246,
    0.01068,
    0.02719,
    -0.01426,
    -0.01564,
    0.02956,
    0.01666,
    0.04975,
    0.00366,
    -0.00724,
    -0.03251,
    -0.03624,
    -0.015,
    -0.01627,
    -0.01209,
    -0.00386,
    0.01498,
    -0.04493,
    0.04646,
    0.01902,
    0.00629,
    -0.02447,
    0.03467,
    0.03839,
    -0.02984,
    -0.02859,
    0.00896,
    0.00786,
    -0.04015,
    -0.02894,
    0.02169,
    -0.01268,
    -0.01743,
    0.00242,
    0.02733,
    -0.014,
    0.02716,
    -0.03983,
    0.00637,
    -0.00108,
    -0.02721,
    0.01768,
    0.01043,
    0.03293,
    -0.0313,
    0.00113,
    0.06402,
    -0.00824,
    -0.02316,
    0.03471,
    0.06954,
    0.05283,
    -0.00701,
    -0.03053,
    -0.01598,
    0.01109,
    0.00359,
    0.00265,
    -0.00664,
    -0.0116,
    -0.0037,
    -0.01559,
    -0.00581,
    -0.02188,
    -0.01252,
    -0.03761,
    0.02037,
    0.00157,
    -0.00049,
    0.03269,
    -0.01448,
    0.02251,
    0.03145,
    0.00589,
    0.01475,
    0.00545,
    -0.01425,
    -0.00718,
    0.01265,
    -0.00798,
    0.01246,
    -0.02074,
    0.00053,
    0.00146,
    -0.03402,
    0.00066,
    -0.01236,
    -0.00432,
    0.0045,
    0.03857,
    -0.00878,
    -0.02965,
    -0.0096,
    -0.0084,
    0.00471,
    -0.02128,
    0.01922,
    0.06699,
    0.00423,
    -0.07613,
    -0.00534,
    0.03551,
    -0.03177,
    0.02047,
    -0.01533,
    -0.02318,
    -0.01355,
    0.16649,
    -0.03852,
    -0.01799,
    0.11633,
    0.01503,
    -0.02536,
    0.01935,
    -0.01004,
    -0.01094,
    0.00334,
    -0.01902,
    -0.02074,
    -0.00993,
    -0.01406,
    -0.00224,
    -0.02651,
    0.00119,
    0.03547,
    0.01101,
    -0.00675,
    -0.0303,
    -0.00077,
    -0.02078,
    -0.02716,
    -0.02718,
    0.01636,
    -0.03463,
    0.0052,
    -0.01284,
    -0.00938,
    0.00283,
    -0.01818,
    -0.00028,
    0.00748,
    0.00262,
    -0.03043,
    -0.03656,
    -0.00856,
    0.00322,
    -0.00297,
    0.02151,
    -0.00681,
    0.00117,
    0.00669,
    -0.04443,
    -0.00508,
    -0.04739,
    0.00623,
    0.03232,
    0.01486,
    -0.04068,
    0.01397,
    0.00123,
    -0.00248,
    0.01778,
    -0.01548,
    0.02965,
    -0.05365,
    0.02259,
    0.00868,
    0.00344,
    0.00057,
    -0.00336,
    -0.03478,
    -0.00303,
    0.02489,
    -0.00662,
    0.0008,
    -0.01103,
    0.01833,
    -0.01689,
    -0.02477,
    0.02552,
    -0.04737,
    0.00969,
    0.02603,
    0.01995,
    -0.01108,
    0.0363,
    -0.00384,
    -0.0229,
    0.00587,
    -0.03956,
    0.02104,
    -0.02597,
    0.01096,
    -0.00568,
    -0.00071,
    0.01006,
    0.03579,
    0.04449,
    -0.02191,
    -0.00268,
    0.02902,
    9e-05,
    0.01544,
    0.03018,
    -0.01294,
    0.03092,
    0.04401,
    0.03883,
    -0.03164,
    0.03086,
    -0.00679,
    -0.0156,
    -0.02068,
    0.04172,
    0.01262,
    0.02052,
    0.02403,
    0.02173,
    -0.0357,
    -0.0294,
    -0.01901,
    -0.05968
   ],
   "ISOTOPE": [
    -0.02209,
    0.01439,
    -0.01709,
    0.01919,
    -0.04353,
    -0.00449,
    0.01393,
    -0.04641,
    -0.01319,
    0.02468,
    0.02891,
    -0.0205,
    0.01406,
    0.04473,
    -0.02052,
    -0.05348,
    -0.01257,
    0.0156,
    0.05596,
    0.01447,
    0.02753,
    0.02032,
    0.02923,
    0.00376,
    -0.00061,
    0.03622,
    0.004,
    -0.0161,
    0.05747,
    -0.02533,
    0.01021,
    0.02346,
    -0.04501,
    0.00417,
    0.03916,
    -0.04805,
    -0.0037,
    -0.06063,
    0.0159,
    -0.03188,
    -0.01013,
    -0.01375,
    -0.06017,
    -0.03129,
    -0.01189,
    -0.01928,
    -0.00905,
    0.03277,
    0.00639,
    0.00923,
    0.01258,
    0.02981,
    -0.01334,
    -0.00901,
    -0.05272,
    -0.03505,
    0.00469,
    -0.01763,
    -0.04817,
    -0.02135,
    0.03465,
    -0.04328,
    0.0075,
    0.02366,
    -0.0421,
    0.00257,
    -0.03229,
    -0.04417,
    -0.0186,
    0.00118,
    -0.0161,
    0.00446,
    0.02545,
    0.02752,
    0.02763,
    -0.01738,
    0.03345,
    0.03668,
    -0.02083,
    0.00655,
    -0.02709,
    -0.00794,
    -0.09148,
    0.00672,
    0.06154,
    -0.01064,
    0.00739,
    -0.01477,
    0.01203,
    -0.00667,
    -0.00151,
    0.01529,
    -0.01108,
    -0.0197,
    0.01431,
    0.00689,
    -0.00209,
    -0.02414,
    0.01987,
    -0.03131,
    -0.00345,
    0.01837,
    -0.03746,
    0.00367,
    -0.02311,
    -0.01322,
    0.00373,
    -0.02077,
    0.04513,
    -0.00642,
    0.00596,
    0.05603,
    0.02069,
    -0.03051,
    0.00052,
    -0.03636,
    -0.00903,
    -0.00896,
    -0.02241,
    -0.00328,
    -0.01351,
    -0.03104,
    0.01081,
    -0.00743,
    -0.02947,
    0.00649,
    0.00978,
    -0.01254,
    -0.02267,
    -0.01297,
    -0.01679,
    -0.00301,
    0.00416,
    0.00839,
    -0.00098,
    0.0175,
    -0.01648,
    -0.02446,
    -0.01388,
    -0.02217,
    -0.04263,
    -0.00652,
    0.02861,
    -0.00522,
    -0.00916,
    0.02353,
    -0.0045,
    0.00159,
    0.00328,
    0.00681,
    -0.03302,
    -0.01498,
    0.01476,
    0.04641,
    0.02025,
    -0.02022,
    0.00745,
    -0.07365,
    0.011,
    0.01893,
    -0.05305,
    0.00226,
    -0.01972,
    -0.0181,
    -0.00138,
    0.02137,
    0.01657,
    -0.01071,
    -0.00278,
    0.00808,
    0.03893,
    0.00177,
    0.01791,
    0.03908,
    -0.03602,
    -0.01152,
    0.00151,
    0.02158,
    -0.00775,
    0.02023,
    0.04854,
    0.02697,
    0.01445,
    -0.00951,
    -0.00063,
    -0.01087,
    0.00702,
    -0.05585,
    -0.0089,
    0.01691,
    -0.0187,
    0.00803,
    -0.02516,
    -0.04073,
    -0.05848,
    -0.00557,
    4e-05,
    -0.04594,
    0.00707,
    0.05462,
    -0.02009,
    -0.00052,
    -0.01884,
    0.03296,
    -0.02095,
    0.01991,
    0.04454,
    -0.00223,
    0.03086,
    0.02739,
    0.00947,
    -0.05345,
    0.00797,
    -0.01423,
    -0.01715,
    0.03101,
    -0.02103,
    0.01197,
    -0.01082,
    -0.00999,
    -0.01084,
    -0.00713,
    -0.02193,
    0.00564,
    0.01138,
    0.01234,
    0.00606,
    0.03067,
    -0.03496,
    -0.03309,
    -0.01456,
    0.01529,
    -0.0209,
    -0.01317,
    -0.01413,
    -0.0059,
    0.00427,
    -0.00271,
    0.03686,
    0.01695,
    0.02595,
    0.00453,
    -0.02516,
    -0.00096,
    0.00235,
    0.00374,
    -0.01505,
    0.00977,
    -0.02137,
    -0.0127,
    0.00942,
    -0.02961,
    -0.0195,
    -0.01146,
    -0.01127,
    -0.03049,
    0.01354,
    -0.01347,
    0.01477,
    0.03572,
    -0.05399,
    -0.04235,
    -0.01056,
    0.02154,
    0.00917,
    0.04642,
    -0.01417,
    -0.00984,
    -0.0411,
    0.0339,
    0.04042,
    0.01416,
    -0.02073,
    -0.0134,
    -0.02238,
    -0.00436,
    -0.00082,
    0.01475,
    -0.00441,
    0.00069,
    -0.00517,
    0.01658,
    0.01461,
    0.00434,
    0.00356,
    0.01997,
    0.03701,
    0.00608,
    -0.03708,
    -0.00918,
    0.04125,
    -0.00694,
    0.02466,
    -0.01294,
    -0.02462,
    0.00251,
    0.0062,
    -0.0335,
    -0.00313,
    -0.00609,
    0.02298,
    -0.03435,
    0.01676,
    0.00236,
    0.01271,
    -0.00392,
    -0.02572,
    -0.01114,
    0.02666,
    0.00709,
    -0.00745,
    -0.04064,
    -0.02858,
    -0.00158,
    0.01648,
    -0.0004,
    0.04132,
    0.05148,
    -0.02105,
    0.00262,
    -0.03725,
    0.01352,
    -0.02381,
    -0.03108,
    0.02068,
    -0.01143,
    0.01285,
    -0.03479,
    0.00718,
    0.01904,
    -0.00031,
    0.05011,
    -0.00898,
    -0.04836,
    0.01181,
    -0.00848,
    -0.00194,
    0.00494,
    -0.02412,
    0.00711,
    0.02125,
    -0.0005,
    0.00632,
    0.02116,
    0.00809,
    0.00029,
    -0.05681,
    0.04314,
    0.01534,
    0.01977,
    0.00866,
    0.03245,
    -0.01348,
    -0.00538,
    -0.0165,
    -0.01705,
    0.01653,
    0.029,
    0.00963,
    0.03513,
    0.06435,
    0.02569,
    -0.00527,
    -0.04969,
    0.01831,
    0.02423,
    -0.0031,
    -0.03805,
    0.00886,
    0.03181,
    -0.01189,
    0.00526,
    -0.02964,
    0.00708,
    -0.00973,
    -0.02633,
    -0.01903,
    -0.00475,
    -0.02217,
    0.01044,
    0.03511,
    0.00965,
    0.00776,
    0.03223,
    0.03049,
    0.02803,
    0.00073,
    -0.01134,
    0.02207,
    -0.01374,
    -0.02649,
    0.01168,
    -0.02891,
    0.00416,
    0.00842,
    -0.0347,
    -0.00138,
    0.00185,
    -0.05679,
    0.00037,
    0.00449,
    -0.01903,
    -0.03291,
    -0.04298,
    -0.00374,
    -0.01137,
    0.01338,
    0.01392,
    -0.03798,
    -0.01419,
    0.02425,
    0.00329,
    0.01514,
    -0.019,
    0.01292,
    -0.02253,
    -0.01792,
    -0.01549,
    0.00884,
    0.01595,
    -0.00563,
    -0.01974,
    0.02018,
    0.00316,
    -0.0082,
    -0.00652,
    -0.04137,
    -0.0326,
    -0.00204,
    -0.03014,
    -0.00561,
    -0.00052,
    -0.00299,
    0.02553,
    0.00186,
    -0.01297,
    -0.03744,
    -0.0091,
    -0.05826,
    0.0063,
    0.02686,
    -0.00634,
    0.05866,
    -0.0321,
    -0.0162,
    -0.04725,
    -0.04159,
    -0.00559,
    -0.01159,
    0.03668,
    -0.03476,
    -0.00171,
    0.0047,
    0.00794,
    0.01567,
    -0.01921,
    0.01677,
    0.03474,
    -0.02965,
    -0.00539,
    -0.00065,
    0.0065,
    0.01587,
    0.07934,
    -0.09942,
    -0.02211,
    0.02017,
    0.05361,
    0.00224,
    -0.00969,
    -0.01686,
    0.02434,
    0.03101,
    0.02385,
    0.01197,
    -0.00635,
    -0.01395,
    0.00427,
    -0.00248,
    0.0208,
    -0.02385,
    -0.02198,
    -0.01385,
    0.01649,
    0.03598,
    -0.02216,
    -0.00131,
    -0.01269,
    -0.01484,
    0.01184,
    -0.01796,
    0.0005,
    -0.02238,
    -0.01015,
    0.00465,
    0.02052,
    0.00734,
    0.00996,
    -0.0367,
    -0.01577,
    0.00521,
    -0.01943,
    0.01208,
    0.02617,
    -0.03108,
    -0.00584,
    -0.00711,
    0.0186,
    -0.004,
    0.00944,
    -0.00321,
    0.01021,
    0.00323,
    -0.02236,
    0.0193,
    -0.00848,
    0.04146,
    -0.00392,
    0.01544,
    0.01198,
    0.02774,
    0.00801,
    0.02376,
    -0.02372,
    0.01561,
    -0.03455,
    0.05926,
    -0.04635,
    -0.01523,
    -0.03138,
    0.02175,
    -0.00238,
    0.0212,
    0.00648,
    0.03605,
    -0.00938,
    0.02487,
    0.00703,
    -0.01368,
    0.01469,
    -0.02273,
    0.023,
    0.03393,
    -0.0332,
    -0.01494,
    0.01052,
    -0.00855,
    -0.02608,
    0.00148,
    0.07006,
    -0.00937,
    0.01808,
    -0.0337,
    -0.02205,
    -0.00747,
    0.04361,
    -0.00916,
    -0.02201,
    0.01721,
    0.02261,
    -0.01495,
    -0.00151,
    0.00196,
    0.04072,
    -0.00362,
    -0.00131,
    0.04658,
    -0.01381,
    0.0232,
    0.01536,
    0.02099,
    -0.02524,
    0.04222,
    -0.02048,
    0.01041,
    0.00051,
    -0.00661,
    -0.01779,
    -0.0195,
    -0.03853,
    -0.01707,
    -0.01871,
    -0.00026,
    0.0155,
    0.04085,
    -0.02703,
    -0.05395,
    0.01179,
    0.02496,
    -0.00012,
    -0.01798,
    -0.01412,
    0.078,
    0.00472,
    0.00328,
    -0.01747,
    -0.00737,
    -0.00443,
    -0.00044,
    -0.0259,
    -0.03789,
    -0.0028,
    -0.0098,
    0.01174,
    0.02375,
    0.00933,
    -0.01742,
    -0.05347,
    -0.02021,
    0.01149,
    -0.02092,
    -0.02566,
    0.03355,
    0.00441,
    0.00898,
    0.02468,
    -0.0443,
    -0.0413,
    -0.00153,
    0.0257,
    -0.04952,
    0.00688,
    -0.00251,
    0.0194,
    -0.01035,
    -0.00475,
    0.03683,
    -0.00052,
    0.01456,
    -0.01636,
    0.00952,
    0.02093,
    0.00486,
    0.00636,
    -0.00454,
    -0.00025,
    0.03086,
    0.09866,
    -0.02325,
    -0.04131,
    -0.00052,
    -0.00193,
    0.0764,
    0.02228,
    -0.00102,
    0.01917,
    0.03734,
    -0.01445,
    0.00251,
    -0.02542,
    0.00089,
    0.01487,
    -0.03047,
    0.01044,
    0.03628,
    -0.02336,
    0.00991,
    0.0474,
    -0.0028,
    0.03444,
    -0.03219,
    0.00189,
    -0.00209,
    -0.04009,
    -0.00543,
    -0.04236,
    0.00263,
    -0.01047,
    0.02996,
    -0.00999,
    0.00921,
    -0.02248,
    -0.01022,
    0.04453,
    -0.00079,
    0.051,
    -0.05415,
    0.0086,
    -0.00342,
    -0.01811,
    -0.06088,
    0.02246,
    -0.02372,
    -0.00889,
    0.02611,
    0.01195,
    -0.01118,
    0.00928,
    -0.04536,
    -0.02408,
    0.00928,
    -0.01402,
    0.00112,
    0.00142,
    -0.03112,
    0.03756,
    -0.04007,
    0.02135,
    -0.00405,
    0.01229,
    -0.02325,
    -0.04,
    0.06709,
    -0.03096,
    0.0009,
    -0.01308,
    -0.00484,
    0.05993,
    -0.02773,
    -0.00591,
    -0.01034,
    -0.01026,
    -0.01607,
    -0.02586,
    0.0074,
    -0.01568,
    -0.01425,
    0.01174,
    -0.00537,
    -0.02861,
    0.0193,
    0.04782,
    0.04474,
    -0.00408,
    0.0378,
    -0.00447,
    0.05666,
    -0.00706,
    -0.04888,
    -0.03869,
    -0.05395,
    -0.01689,
    0.03926,
    -0.00889,
    0.02036,
    0.04294,
    0.04168,
    -0.00029,
    -0.03815,
    0.00585,
    0.05843,
    -0.00418,
    -0.03079,
    0.00511,
    0.06002,
    -0.01872,
    0.01007,
    0.01729,
    -0.00661,
    -0.00018,
    -0.01106,
    0.01363,
    0.01042,
    -0.02514,
    -0.03751,
    -0.01012,
    -0.0132,
    -0.00872,
    -0.01702,
    -0.0367,
    0.01276,
    -0.00534,
    -0.03171,
    -0.01763,
    0.01992,
    0.0104,
    -0.02587,
    -0.05219,
    -0.03241,
    -0.02264,
    0.01267,
    0.02706,
    0.02377,
    0.01401,
    -0.04171,
    0.03064,
    0.0233,
    -0.03075,
    -0.00516,
    0.00656,
    -0.01306,
    -0.00871,
    0.0088,
    0.03931,
    0.01157,
    0.01359,
    -0.00492,
    -0.04562,
    -0.00604,
    -0.02551,
    0.01016,
    0.05109,
    0.02673,
    0.04319,
    -0.0105,
    -0.00797,
    -0.00263,
    0.00637,
    -0.0002,
    0.03174,
    -0.01632,
    -0.03936,
    0.01665,
    0.02603,
    0.02926,
    -0.02012,
    0.02765,
    0.0183,
    -0.02183,
    0.00739,
    -0.00251,
    -0.01036,
    0.01424,
    0.01808,
    0.03463,
    0.03695,
    -0.04294,
    0.00876,
    0.00763,
    0.02236,
    -0.0269,
    -0.03547,
    -0.03203,
    -0.03188,
    0.01906,
    -0.00335,
    0.02252,
    -0.082,
    -0.00872,
    0.00987,
    -0.02904,
    -0.01803,
    -0.03852,
    -0.00291,
    -0.01472,
    0.03893,
    0.0687,
    0.00461,
    0.01136,
    -0.00368,
    -0.04199,
    0.03345,
    0.00788,
    -0.02561,
    0.04879,
    -0.01903,
    -0.05111,
    -0.01359,
    0.01978,
    0.01601,
    -0.05297,
    0.01093,
    0.00693,
    0.02641,
    0.02979,
    -0.01513,
    -0.01731,
    -0.00391,
    -0.03401,
    -0.01516,
    0.00714,
    0.00414,
    0.00735,
    0.01125,
    -0.00089,
    -0.0161,
    0.0071,
    -0.0127,
    -0.01937,
    -0.00193,
    -0.02967,
    -0.01525,
    -0.02926,
    -0.01604,
    -0.02664,
    -0.01192,
    -0.02196,
    -0.06246,
    -0.01897,
    -0.00897,
    0.00187,
    -0.04394,
    -0.00159,
    -0.02351,
    -0.02875,
    -0.04745,
    -0.02157,
    -0.01908,
    -0.00572,
    0.00649,
    0.00803,
    0.02002,
    -0.027,
    -8e-05,
    0.02431,
    -0.00627,
    0.00957,
    -0.00557,
    0.03857,
    -0.00413,
    0.01815,
    0.04325,
    0.01953,
    -0.00341,
    -0.01688,
    -0.04176,
    0.01234,
    0.03434,
    -0.03161,
    -0.02765,
    -0.00473,
    0.04788,
    -0.00226,
    -0.00578,
    -0.00293,
    -0.04036,
    0.00256,
    0.01104,
    0.01222,
    0.03939,
    -0.01515,
    6e-05,
    -0.02764,
    -0.04242,
    -0.0193,
    0.02879,
    -0.02849,
    -0.03076,
    -0.02168,
    -0.00246,
    -0.033,
    -0.05106,
    0.03004,
    0.02123,
    -0.03463,
    0.04082,
    -0.00153,
    0.00385,
    -0.00506,
    0.00595,
    0.01521,
    0.0238,
    0.01609,
    -0.02058,
    0.00549,
    0.03322,
    -0.00229,
    0.01917,
    -0.03561,
    0.00473,
    0.00921,
    0.00123,
    -0.00913,
    0.02067,
    -0.00811,
    -0.00662,
    0.00699,
    0.01025,
    0.00951,
    0.00411,
    -0.01065,
    0.00828,
    0.02961,
    -0.00021,
    0.00435,
    -0.02036,
    0.01031,
    0.00707,
    -0.00614,
    -0.02023,
    -0.00138,
    -0.01912,
    0.04382,
    -0.01692,
    -0.03403,
    -0.02167,
    0.01022,
    0.02785,
    -0.01829,
    0.00479,
    0.03715,
    -0.01545,
    -0.01931,
    -0.01042,
    -0.02499,
    0.03308,
    0.01253,
    -0.00993,
    0.01965,
    0.02463,
    -0.04921,
    0.00318,
    0.00158,
    0.00431,
    -0.00133,
    0.02681,
    0.01933,
    0.00376,
    -0.0319,
    -0.00785,
    0.01321,
    -0.01119,
    0.00125,
    -0.00553,
    0.02355,
    0.03939,
    -0.03124,
    0.01922,
    0.01418,
    0.0296,
    -0.01586,
    0.01107,
    0.01635,
    0.00238,
    -0.01572,
    -0.00759,
    -0.04176,
    0.0252,
    -0.02173,
    -0.01813,
    -0.0306,
    -0.02938,
    -0.00221,
    -0.01316,
    0.00122,
    0.01383,
    0.02354,
    -0.00263,
    0.013,
    -0.02348,
    -0.01388,
    0.00222,
    0.00143,
    0.00347,
    0.01823,
    -0.01649,
    -0.03125,
    0.03219,
    0.01465,
    -0.04529,
    -0.00087,
    -0.04445,
    -0.00917,
    -0.03992,
    0.02413,
    0.0068,
    0.0202,
    0.00384,
    -0.02663,
    0.00788,
    0.01469,
    -0.00456,
    -0.02432,
    0.02683,
    -0.07589,
    0.00054,
    0.00618,
    0.01613,
    -0.01075,
    -0.04837,
    -0.00412,
    -0.02529,
    -0.06076,
    0.01878,
    0.00498,
    -0.04736,
    -0.00601,
    0.00609,
    -0.02883,
    0.03295,
    -0.01473,
    -0.00801,
    0.00993,
    0.05563,
    0.0001,
    0.02665,
    -0.02198,
    0.02035,
    0.01789,
    -0.01379,
    0.03902,
    -0.03852,
    0.02146,
    0.01032,
    -0.03732,
    -0.00458,
    -0.01318,
    0.02493,
    0.00521,
    -0.03001,
    0.00608,
    -0.02586,
    0.00072,
    0.05154,
    -0.01332,
    0.01354,
    -0.00531,
    -0.00752,
    0.00907,
    -0.01066,
    -0.04784,
    0.0106,
    0.01646,
    -0.07637,
    -0.02772,
    -0.02835,
    0.03276,
    0.01908,
    0.00543,
    0.00312,
    -0.01384,
    0.0403,
    0.02002,
    0.00119,
    -0.0288,
    0.05891,
    -0.03671,
    -0.02241,
    -0.02527,
    0.00264,
    0.00448,
    -0.00251,
    -0.02433,
    -0.0376,
    -0.00947,
    -0.01295,
    -0.00891,
    -0.03461,
    -0.01918,
    -0.04084,
    0.05453,
    -0.00089,
    0.0293,
    0.02308,
    0.03403,
    0.00976,
    0.00566,
    0.02186,
    -0.01994,
    0.02076,
    -0.01194,
    -0.01702,
    -0.01691,
    -0.05434,
    -0.02157,
    0.02447,
    0.03819,
    -0.04716,
    -0.03076,
    0.00634,
    0.02098,
    -0.00808,
    -0.02286,
    0.00429,
    0.0351,
    0.03649,
    0.01812,
    0.00828,
    0.00897,
    0.01084,
    -0.00874,
    0.00616,
    0.0192,
    -0.01415,
    0.03105,
    -0.03205,
    -0.00629,
    -0.02034,
    -0.00117,
    0.03065,
    -0.02098,
    0.01328,
    0.00656,
    -0.01696,
    -0.00971,
    0.03182,
    -0.03385,
    0.01928,
    0.0067,
    0.02223,
    -0.02481,
    0.00759,
    0.00021,
    -0.01173,
    -0.02041,
    0.01157,
    -0.01507,
    -0.00246,
    0.00477,
    0.01654,
    0.01481,
    -0.0283,
    -0.01525,
    -0.04065,
    0.02184,
    0.01501,
    0.00808,
    0.03172,
    0.0387,
    0.03931,
    -0.01669,
    0.00144,
    0.01256,
    -0.00244,
    -0.00954,
    0.00764,
    0.01853,
    0.0127,
    0.03548,
    0.01865,
    0.03037,
    -0.0044,
    0.00845,
    -0.02007,
    -0.00798,
    0.00295,
    -0.02357,
    0.03322,
    -0.00572,
    -0.05396,
    -0.00388,
    0.00045,
    -0.01099,
    0.03369,
    -0.02369,
    0.02408,
    -0.00445,
    -0.02,
    0.00303,
    -0.01713,
    0.01846,
    -0.04445,
    -0.01951,
    -0.01511,
    -0.02051,
    0.03482,
    -0.00077,
    0.02309,
    0.01712,
    0.01551,
    0.03218,
    -0.05008,
    -0.00723,
    0.01032,
    0.0027,
    -0.00268,
    0.02124,
    0.00202,
    0.0469,
    -0.01954,
    0.00762,
    0.00641,
    -0.01282,
    0.02083,
    -0.02776,
    0.01218,
    0.01602,
    -0.00423,
    -0.00859,
    -0.03323,
    0.00496,
    0.03166,
    -0.01525,
    0.05467,
    0.01586,
    -0.026,
    -0.00527,
    -0.0108,
    0.02238,
    0.00656,
    0.02393,
    -0.01644,
    -0.00405,
    0.0429,
    0.00047,
    -0.02573,
    -0.01343,
    0.04887,
    -0.00112,
    0.01592,
    0.05525,
    0.02094,
    0.00838,
    -0.0035,
    -0.01493,
    -0.04348,
    -0.00656,
    -0.01847,
    0.02209,
    -0.00989,
    0.02291,
    -0.00988,
    0.00503,
    -0.02599,
    -0.00511,
    0.05199,
    -0.00849,
    0.02644,
    -0.00072,
    -0.02585,
    -0.04949,
    -0.04499,
    -0.02239,
    -0.00597,
    -0.00134,
    0.02632,
    0.03643,
    0.02079,
    -0.0065,
    0.00284,
    -0.03048,
    0.01306,
    0.01692,
    0.0142,
    0.01564,
    0.02202,
    -0.01222,
    0.01708,
    0.00571,
    0.02843,
    0.0361,
    0.00903,
    0.02725,
    -0.04654,
    0.02441,
    -0.03877,
    -0.02626,
    -0.0189,
    -0.0214,
    -0.02917,
    -0.00755,
    0.03995,
    -0.01467,
    0.01002,
    -0.02447,
    -0.0152,
    0.01076,
    -0.00514,
    0.00505,
    -0.00481,
    0.02434,
    -0.00858,
    -0.00385,
    0.00872,
    0.0488,
    0.04738,
    -0.04441,
    -0.02271,
    0.00925,
    -0.0476,
    -0.00534,
    -0.03773,
    0.01049,
    -0.03347,
    0.02686,
    0.0014,
    0.02811,
    0.02018,
    0.01068,
    0.02297,
    0.03073,
    0.00356,
    -0.01203,
    0.01109,
    0.00938,
    0.04178,
    -0.01765,
    0.00107,
    0.04832,
    -0.01652,
    -0.0107,
    0.02364,
    0.03251,
    -0.03603,
    0.0057,
    0.03394,
    -0.04736,
    0.00337,
    -0.01767,
    0.00377,
    -0.01485,
    -0.00792,
    -0.0125,
    -0.00676,
    0.03502,
    -0.01885,
    0.06822,
    -0.01999,
    0.02082,
    0.02179,
    -0.01928,
    0.06272,
    -0.03043,
    0.00923,
    0.00896,
    0.00988,
    0.01843,
    0.01142,
    -0.00081,
    0.02674,
    0.01815,
    0.03785,
    -0.03171,
    -0.00847,
    0.01539,
    0.04117,
    -0.02468,
    0.04134,
    -0.0267,
    0.03477,
    0.03969,
    -0.01653,
    0.04777,
    0.00031,
    -0.00317,
    0.02328,
    -0.00076,
    -0.00591,
    -0.025,
    0.014,
    -0.00238,
    -0.03557,
    0.00128,
    -0.00457,
    0.19013,
    -0.06103,
    0.04348,
    0.04274,
    -0.04374,
    -0.01727,
    -0.00057,
    0.01466,
    0.03286,
    -0.01568,
    0.01534,
    -0.01478,
    -0.00843,
    -0.02892,
    0.00503,
    0.03803,
    0.01459,
    -0.00105,
    -0.00854,
    -0.01577,
    -0.01688,
    0.01478,
    0.03531,
    -0.0057,
    -0.03644,
    -0.00144,
    -0.01435,
    -0.0091,
    -0.00129,
    -0.01483,
    0.01657,
    -0.00133,
    -0.03564,
    -0.0171,
    0.02106,
    -0.00662,
    0.00565,
    0.00149,
    0.00012,
    -0.00266,
    0.05066,
    -0.0141,
    -0.02144,
    0.00809,
    0.01691,
    0.01178,
    -0.01309,
    -0.0225,
    -0.01178,
    -0.00672,
    -0.01711,
    -0.02957,
    -0.01348,
    -0.00546,
    -0.01318,
    -0.01313,
    -0.02847,
    0.0104,
    -0.00607,
    0.02004,
    0.02046,
    -0.02618,
    -0.01755,
    -0.03479,
    0.0051,
    -0.06896,
    -0.03046,
    -0.034,
    -0.01541,
    -0.0038,
    -0.01317,
    -0.05231,
    0.00664,
    -0.03408,
    -0.02006,
    0.06652,
    -0.02916,
    -0.02132,
    -0.0287,
    -0.00818,
    -0.00048,
    0.00213,
    0.01619,
    0.00352,
    0.00897,
    0.00577,
    0.06226,
    0.02412,
    -0.00198,
    -0.0185,
    -0.0157,
    0.00455,
    0.00259,
    -0.03028,
    0.05948,
    -0.01575,
    0.00813,
    0.03495,
    0.00844,
    0.03068,
    -0.03078,
    -0.05281,
    0.02142,
    0.00028,
    -0.00436,
    0.00807,
    -0.0044,
    0.01992,
    -0.00973,
    -0.02538,
    0.00867,
    -0.0205
   ],
   "MEDITATE": [
    0.02358,
    0.0049,
    0.01936,
    0.01469,
    -0.03438,
    0.07129,
    0.01683,
    0.01737,
    -0.05201,
    -0.03542,
    0.00023,
    0.03505,
    -0.03296,
    -0.01375,
    -0.00131,
    -0.00525,
    -0.0087,
    0.00949,
    -0.01481,
    0.01013,
    0.01197,
    -0.03944,
    0.00711,
    -0.0179,
    0.00788,
    0.02279,
    -0.00311,
    -0.02166,
    -0.00853,
    0.02775,
    0.01809,
    0.03348,
    0.04701,
    0.03778,
    0.01216,
    0.00784,
    0.021,
    0.00796,
    -0.0527,
    0.03629,
    -0.01983,
    0.03351,
    -0.05177,
    0.00198,
    0.02244,
    0.00189,
    -0.00194,
    0.03103,
    -0.01226,
    0.00412,
    -0.06177,
    -0.00312,
    -0.01752,
    0.00244,
    -0.03335,
    4e-05,
    0.02789,
    0.034,
    -0.00623,
    -0.04999,
    -0.00144,
    -0.00952,
    -0.02028,
    -0.01484,
    0.02201,
    0.0115,
    0.00803,
    -0.02171,
    -0.03699,
    -0.01503,
    -0.01097,
    -0.01766,
    0.02007,
    0.00466,
    0.00294,
    -0.01952,
    -0.06172,
    0.01552,
    0.00683,
    -0.00995,
    0.00101,
    -0.02238,
    0.00604,
    -0.00337,
    -0.00286,
    0.00397,
    -0.02051,
    -0.00843,
    0.00427,
    0.00027,
    0.02623,
    0.0165,
    -0.04395,
    0.01335,
    -0.01653,
    -0.05323,
    0.00029,
    -0.00204,
    -0.01089,
    0.01142,
    0.014,
    -0.06195,
    0.05503,
    -0.00498,
    -0.01802,
    -0.05259,
    -0.02685,
    -0.02559,
    -0.00749,
    0.01685,
    0.00775,
    0.01126,
    -0.01782,
    -0.02507,
    0.0066,
    -0.02128,
    0.01089,
    -0.01701,
    -0.02574,
    0.00433,
    0.03655,
    -0.00018,
    -0.03691,
    0.01713,
    0.02821,
    0.00978,
    -0.02527,
    -0.02406,
    -0.00647,
    0.00378,
    0.02055,
    0.02964,
    0.02581,
    -0.02016,
    0.02733,
    -0.01461,
    0.00893,
    0.00103,
    0.0456,
    -0.01157,
    -0.00886,
    -0.00806,
    -0.0015,
    0.00721,
    -0.00288,
    0.02851,
    0.01487,
    0.01438,
    -0.02406,
    0.03365,
    9e-05,
    -0.02239,
    0.01887,
    0.00408,
    -0.01166,
    0.01298,
    -0.0323,
    -0.00022,
    0.00699,
    -0.00408,
    -0.0109,
    0.01718,
    -0.02244,
    0.02013,
    0.00914,
    0.03216,
    -0.04108,
    -0.01313,
    -0.03795,
    0.01358,
    -0.00257,
    -0.01054,
    -0.0139,
    -0.02116,
    0.02935,
    0.02597,
    -0.02504,
    -0.01151,
    -0.01984,
    0.01691,
    0.05456,
    -0.01048,
    -0.00831,
    -0.00955,
    -0.02215,
    0.00072,
    0.01693,
    0.03395,
    0.02624,
    0.02587,
    0.0038,
    0.02399,
    0.03584,
    -0.03368,
    0.0205,
    -0.00153,
    -0.0326,
    0.06331,
    -0.02329,
    -0.01506,
    -0.03012,
    -0.03336,
    0.0049,
    -0.03073,
    -0.02316,
    0.01135,
    0.02593,
    -0.03939,
    -0.03591,
    0.00868,
    0.01525,
    0.01902,
    0.01527,
    -0.03215,
    0.01565,
    -0.01048,
    -0.01694,
    -0.01531,
    -0.02438,
    -0.00667,
    0.00392,
    -0.03644,
    -0.0191,
    -0.02515,
    0.0191,
    0.01185,
    -0.01356,
    0.01673,
    -0.02886,
    -0.00875,
    -0.06321,
    0.03002,
    -0.02703,
    -0.01347,
    -0.00948,
    0.03073,
    0.01875,
    -0.00294,
    -0.03708,
    0.01237,
    0.03253,
    0.01349,
    -0.02911,
    0.02512,
    0.0412,
    -0.01104,
    0.00764,
    0.00833,
    -0.01723,
    -0.02113,
    0.00055,
    -0.01649,
    -0.02475,
    -0.002,
    -0.01307,
    0.0032,
    0.03451,
    0.03765,
    0.02571,
    -0.02112,
    0.00499,
    0.00838,
    -0.01159,
    -0.01592,
    0.02372,
    -0.0083,
    0.01495,
    0.03293,
    0.01662,
    -0.01665,
    -0.00579,
    0.00234,
    0.01054,
    -0.03118,
    -0.00455,
    -0.02713,
    0.02244,
    0.00581,
    -0.0002,
    -0.003,
    0.01004,
    -0.02744,
    0.02358,
    -0.03636,
    0.04295,
    -0.01147,
    -0.03962,
    -0.00105,
    0.01982,
    -0.01413,
    -0.04046,
    0.00868,
    0.00767,
    -0.01779,
    -0.00052,
    0.02018,
    -0.00821,
    -0.03608,
    -0.00184,
    0.03543,
    -0.00281,
    -0.06702,
    -0.00827,
    -0.02032,
    -0.04034,
    0.02507,
    0.01664,
    -0.02766,
    -0.0399,
    -0.00882,
    -0.01769,
    -0.03582,
    0.00169,
    -0.02252,
    -0.01065,
    0.01878,
    0.02927,
    -0.00351,
    -0.00811,
    -0.01699,
    0.02887,
    0.01468,
    0.00536,
    -0.00871,
    0.00439,
    0.00581,
    0.02171,
    0.01967,
    0.03036,
    -0.0084,
    -0.00641,
    -0.04063,
    0.03205,
    -0.01126,
    -0.01037,
    0.00882,
    0.02414,
    0.01353,
    0.00521,
    -0.00224,
    -0.02916,
    -0.0184,
    -0.00684,
    -0.00119,
    0.01796,
    -0.0057,
    -9e-05,
    -0.01669,
    -0.04913,
    -0.02573,
    -0.02368,
    -0.03595,
    -0.01989,
    0.01738,
    -0.06873,
    0.01513,
    -0.03951,
    0.00206,
    -0.02322,
    -0.0476,
    -0.02522,
    0.01146,
    -0.00496,
    0.04928,
    -0.00857,
    0.03284,
    0.00518,
    0.02613,
    -0.00019,
    -0.04368,
    -0.00861,
    0.03428,
    -0.02224,
    -0.02608,
    0.03004,
    0.01707,
    0.01788,
    0.02528,
    0.01847,
    0.03118,
    -0.02443,
    -0.01907,
    -0.00468,
    0.0097,
    -0.02037,
    -0.01364,
    0.02946,
    0.0145,
    0.02167,
    -0.02748,
    0.02281,
    0.0101,
    -0.02628,
    0.02763,
    -0.04387,
    0.0312,
    -0.01075,
    0.07246,
    -0.01033,
    0.02051,
    0.01187,
    0.03548,
    0.01157,
    -0.01953,
    -0.0007,
    0.00911,
    0.00511,
    -0.00905,
    -0.02725,
    -0.02749,
    0.00856,
    -0.04741,
    -0.01437,
    0.00545,
    0.00252,
    -0.00849,
    0.00399,
    0.01021,
    0.01013,
    0.00812,
    0.0312,
    0.02346,
    -0.00903,
    -0.02229,
    -0.04369,
    -0.0021,
    0.00858,
    -0.01415,
    0.00535,
    0.0209,
    -0.00397,
    0.02008,
    0.00403,
    0.00225,
    -0.03432,
    0.02681,
    0.00725,
    0.06001,
    -0.00665,
    -0.002,
    -0.03745,
    -0.02766,
    -0.04921,
    -0.00676,
    0.01089,
    0.01773,
    0.02075,
    -0.02395,
    0.04087,
    0.00189,
    -0.01783,
    0.00558,
    -0.0556,
    0.02345,
    -0.00075,
    -0.00863,
    -0.02159,
    -0.01206,
    -0.00059,
    -0.03149,
    -0.00779,
    0.02043,
    -0.00612,
    -0.05103,
    0.04127,
    -0.00165,
    0.02824,
    -0.02936,
    0.05325,
    0.00566,
    0.04083,
    0.02389,
    -0.05211,
    -0.01701,
    0.00617,
    0.00261,
    0.00494,
    -0.04859,
    0.04222,
    0.02332,
    -0.00316,
    0.02669,
    0.02145,
    0.00687,
    -0.00286,
    0.00296,
    0.02147,
    -0.00822,
    0.01271,
    -0.00063,
    -0.02199,
    0.05603,
    -0.01335,
    0.01245,
    0.00287,
    0.01528,
    0.03648,
    -0.02228,
    0.02335,
    0.0113,
    -0.04724,
    -0.01077,
    -0.00199,
    0.0404,
    -0.02253,
    -0.04859,
    -0.00113,
    -0.0147,
    0.02051,
    0.00463,
    0.04124,
    0.0004,
    0.00119,
    -0.01464,
    -0.03732,
    0.01143,
    0.00152,
    -0.02615,
    0.04499,
    -0.03467,
    0.05558,
    0.01476,
    0.02479,
    0.00526,
    0.00673,
    -0.0122,
    0.00053,
    -0.00833,
    -0.00626,
    0.03377,
    -0.02174,
    -0.01347,
    -0.02866,
    0.01665,
    0.03962,
    0.01108,
    -0.0336,
    0.0129,
    0.0113,
    -0.00353,
    -0.01271,
    0.00132,
    -0.00491,
    -0.03382,
    0.01389,
    -0.03067,
    -0.00158,
    -0.01335,
    0.01916,
    -0.02144,
    0.07259,
    0.0206,
    0.04704,
    0.05318,
    -0.01017,
    -0.04301,
    -0.02214,
    0.00713,
    -0.04746,
    0.00022,
    -0.02767,
    -0.02804,
    0.00689,
    0.01579,
    -0.00084,
    4e-05,
    0.00055,
    0.02308,
    -0.04301,
    -0.03657,
    -0.01322,
    -0.02745,
    -0.01968,
    -0.01809,
    0.00566,
    0.08016,
    0.00781,
    0.0045,
    -0.03278,
    -0.0202,
    -0.00965,
    -0.00298,
    -0.00501,
    -0.00623,
    -0.02413,
    0.02583,
    0.01072,
    -0.0017,
    -0.01217,
    -0.03134,
    -0.01589,
    0.03942,
    -0.00668,
    0.0065,
    -0.03867,
    0.01119,
    -0.01158,
    0.00081,
    -0.01725,
    0.0121,
    -0.05864,
    0.01732,
    -0.0159,
    0.05847,
    -0.00576,
    -0.00361,
    0.00798,
    0.0147,
    -0.05128,
    0.0797,
    -0.01261,
    0.01045,
    0.00514,
    0.01989,
    0.00365,
    -0.03404,
    0.07114,
    -0.01446,
    -0.00904,
    -0.0169,
    0.00777,
    0.0223,
    -0.00316,
    -0.01681,
    0.00455,
    0.01446,
    -0.02685,
    -0.0086,
    -0.00376,
    -0.00297,
    0.02187,
    0.03476,
    0.07148,
    -0.02809,
    -0.01573,
    -0.00561,
    -0.00935,
    -0.01182,
    0.00203,
    0.00596,
    -0.00042,
    0.01498,
    -0.01079,
    0.02492,
    0.0112,
    -0.02848,
    -0.00992,
    -0.015,
    -0.02612,
    -0.00082,
    -0.02304,
    -0.00313,
    -0.01417,
    0.02188,
    0.00584,
    0.03779,
    -0.00727,
    -0.00161,
    -0.00949,
    0.02791,
    0.01393,
    0.0384,
    -0.01292,
    -0.02689,
    -0.03891,
    0.02812,
    -0.0072,
    -0.01038,
    -0.00136,
    -0.02821,
    0.00863,
    -0.04037,
    -0.0129,
    -0.0676,
    0.00821,
    0.00264,
    -0.04358,
    -0.0072,
    0.01155,
    -0.0409,
    0.03713,
    -0.01542,
    0.01249,
    0.04691,
    0.00618,
    0.02028,
    0.01716,
    0.00301,
    0.00977,
    0.02781,
    -0.01598,
    -0.03888,
    0.00607,
    -0.00284,
    -0.03678,
    -0.02305,
    -0.01769,
    -0.00431,
    0.00779,
    0.01076,
    -0.01394,
    0.00629,
    -0.02541,
    0.02242,
    -0.03584,
    0.01028,
    -0.01269,
    0.02011,
    0.01819,
    -0.0223,
    0.03865,
    0.00168,
    0.01356,
    0.0051,
    -0.04987,
    0.0041,
    -0.01812,
    -0.00502,
    -0.0733,
    0.00135,
    0.00312,
    0.03634,
    -0.02983,
    0.03796,
    -0.02516,
    0.00412,
    -0.0391,
    0.00024,
    0.02174,
    0.0374,
    0.03408,
    -0.005,
    -0.01436,
    -0.02981,
    -0.02082,
    -0.01665,
    0.00653,
    -0.00222,
    0.01072,
    -0.03765,
    -0.01961,
    0.01768,
    0.02372,
    -0.03062,
    0.01053,
    0.01176,
    0.00228,
    0.02455,
    0.04124,
    0.02696,
    0.0248,
    -0.0061,
    0.00969,
    0.01216,
    -0.007,
    0.02328,
    -0.03547,
    0.04692,
    0.03167,
    0.00096,
    -0.0195,
    -0.03786,
    0.04138,
    -0.00048,
    -0.02576,
    -0.03361,
    -0.00446,
    -0.00448,
    -0.01014,
    0.01042,
    -0.00082,
    0.04044,
    0.0071,
    0.00647,
    -0.0074,
    0.00464,
    -0.0188,
    0.03953,
    -0.01552,
    0.02927,
    0.01152,
    -0.00356,
    0.0047,
    -0.0347,
    0.00292,
    -0.02933,
    0.02158,
    0.01587,
    -0.01121,
    -0.02109,
    0.01171,
    -0.04235,
    0.01653,
    -0.06378,
    -0.02086,
    0.02636,
    0.00105,
    -0.04111,
    -0.01152,
    -0.03644,
    -0.00802,
    -0.00786,
    -0.0259,
    0.01296,
    -0.02703,
    -0.03341,
    -0.05268,
    -0.04384,
    -0.00325,
    0.02223,
    -0.01374,
    -0.03968,
    -0.01303,
    0.00378,
    -0.00108,
    0.01337,
    -0.01766,
    0.0182,
    -0.0006,
    0.00138,
    -0.03028,
    -0.02348,
    0.02652,
    0.03501,
    0.00564,
    0.00823,
    0.02222,
    0.04614,
    0.01881,
    0.01969,
    0.01441,
    0.04812,
    0.03142,
    -0.0051,
    -0.06157,
    0.03433,
    -0.03893,
    -0.0023,
    -0.01462,
    -0.01723,
    0.00766,
    -0.04783,
    -0.01767,
    -0.01837,
    -0.04784,
    0.00035,
    -0.00375,
    0.00364,
    -0.03237,
    0.00467,
    -0.0413,
    0.02972,
    -0.01902,
    0.0056,
    -0.01915,
    -0.01841,
    0.00723,
    -0.03612,
    0.0413,
    0.0113,
    0.01452,
    0.02688,
    -0.05936,
    0.0321,
    -0.03552,
    -0.01303,
    -0.01293,
    -0.03372,
    0.02231,
    -0.01366,
    0.00241,
    -0.01448,
    -0.02266,
    0.02663,
    0.026,
    0.01902,
    -0.01662,
    -0.00118,
    -0.01058,
    -0.01077,
    0.07041,
    0.02021,
    0.00516,
    0.00684,
    0.0151,
    8e-05,
    -0.02356,
    0.02717,
    0.02649,
    -0.0293,
    -0.00274,
    -0.03255,
    0.01295,
    0.00539,
    -0.0458,
    0.01554,
    -0.02444,
    -0.00387,
    0.01403,
    -0.06434,
    -0.01254,
    0.03356,
    0.02946,
    -0.02144,
    0.09964,
    0.02135,
    0.03283,
    0.02178,
    0.01797,
    0.03701,
    0.00607,
    -0.03643,
    0.01783,
    -0.01961,
    -0.01851,
    0.01628,
    0.00402,
    -0.02774,
    0.01626,
    -0.00934,
    0.03446,
    0.00736,
    -0.01469,
    0.00322,
    -0.02086,
    0.01372,
    -0.04906,
    0.01749,
    0.04162,
    0.01762,
    -0.00592,
    0.00705,
    0.04511,
    -0.00701,
    0.0067,
    -0.05704,
    -0.04443,
    0.00119,
    0.04277,
    0.0178,
    0.04529,
    -0.03411,
    -0.01144,
    -0.0532,
    -0.002,
    0.00843,
    0.02913,
    -0.02631,
    -0.01579,
    -0.01654,
    -0.01943,
    0.02164,
    -0.00655,
    0.04369,
    0.04936,
    -0.00944,
    -0.03453,
    -0.00025,
    0.05632,
    0.03313,
    0.02139,
    0.02081,
    0.00743,
    0.02828,
    0.00885,
    -0.02517,
    0.01833,
    -0.02437,
    0.0259,
    0.01814,
    -0.00983,
    -0.0394,
    0.02946,
    0.02452,
    0.01003,
    -0.01594,
    -0.02642,
    -0.03847,
    0.02468,
    -0.05274,
    -0.00933,
    0.01643,
    -0.05235,
    -0.03356,
    0.04966,
    -0.01271,
    -0.01499,
    0.02291,
    -0.00579,
    -0.00796,
    0.00526,
    0.04366,
    0.00562,
    -0.00969,
    -0.0367,
    -0.00856,
    0.01113,
    -0.04873,
    0.01023,
    0.0115,
    -0.00332,
    -9e-05,
    -0.01071,
    -0.00185,
    -0.02742,
    0.03305,
    -0.03502,
    -0.05919,
    -0.03932,
    -0.01149,
    -0.00922,
    -0.0236,
    0.00926,
    0.01613,
    0.03122,
    -0.02524,
    0.02746,
    -0.01488,
    0.0485,
    -0.01025,
    -0.02756,
    0.00754,
    0.00243,
    -0.01138,
    0.01627,
    -0.02833,
    -0.00196,
    -0.0158,
    -0.04841,
    -0.00755,
    0.00397,
    -0.00699,
    0.01668,
    -0.04534,
    0.00017,
    -0.01118,
    -0.00407,
    0.03012,
    -0.02402,
    -0.0188,
    -0.04613,
    0.02951,
    0.00207,
    -0.02494,
    0.03355,
    0.00131,
    0.00497,
    0.00448,
    -0.01669,
    0.00423,
    -0.00634,
    0.03947,
    0.03141,
    -0.03891,
    -0.01945,
    0.01588,
    -0.02764,
    -0.02771,
    -0.00527,
    -0.02465,
    -0.0111,
    -0.01619,
    -0.01308,
    -0.01501,
    -0.00538,
    0.03847,
    -0.02803,
    -0.01936,
    0.0987,
    -0.00062,
    -0.03835,
    -1e-05,
    0.00776,
    0.0041,
    -0.01933,
    0.01748,
    0.00932,
    -0.01522,
    -0.03675,
    -0.02184,
    -0.01058,
    0.00547,
    -0.0167,
    -0.00545,
    0.01737,
    0.02461,
    0.02063,
    -0.00715,
    0.01124,
    0.0111,
    -0.05479,
    -0.01788,
    -0.0224,
    0.03132,
    0.02038,
    -0.00553,
    0.02807,
    0.02727,
    0.0086,
    -0.02802,
    -0.00522,
    0.00138,
    -0.00829,
    -0.00639,
    0.05044,
    -0.01049,
    0.00051,
    -0.00651,
    -0.00333,
    -0.04354,
    0.01482,
    0.04108,
    0.01069,
    -0.00581,
    0.0228,
    -0.01037,
    -0.00828,
    0.03607,
    0.07272,
    0.0016,
    0.00912,
    -0.02241,
    0.00828,
    0.04561,
    -0.01752,
    0.03158,
    -0.01758,
    -0.01826,
    0.03498,
    0.03639,
    -0.0123,
    -0.00748,
    0.00684,
    -0.02089,
    -0.03057,
    -0.00386,
    -0.00214,
    -0.02141,
    0.00126,
    0.03692,
    -0.0387,
    0.00576,
    -0.00626,
    0.06677,
    -0.00774,
    0.01309,
    -0.04306,
    0.04655,
    0.01983,
    -0.02692,
    -0.01868,
    0.02609,
    -0.01883,
    0.01052,
    -0.01448,
    -0.01099,
    0.00664,
    -0.04051,
    -0.0233,
    0.00252,
    -0.02935,
    -0.00022,
    -0.00129,
    -0.03583,
    -0.03477,
    -0.0031,
    -0.03108,
    0.00509,
    -0.04079,
    0.00647,
    -0.01429,
    0.02751,
    -0.00144,
    0.02218,
    0.01125,
    -0.00462,
    -0.01125,
    0.05224,
    -0.02724,
    -0.00704,
    -0.00782,
    -0.0297,
    0.0135,
    0.01857,
    0.03614,
    -0.02226,
    -0.02415,
    0.00725,
    0.00652,
    0.01572,
    0.02326,
    -0.00608,
    0.03077,
    -0.0409,
    0.01452,
    0.00843,
    0.00885,
    0.02075,
    -0.0126,
    -0.03994,
    0.00291,
    -0.04613,
    0.02279,
    -0.01991,
    0.00066,
    -0.04725,
    0.0057,
    -0.00602,
    0.00713,
    0.024,
    -0.00281,
    -0.03817,
    0.00593,
    0.04907,
    -0.0582,
    -0.01507,
    0.00994,
    0.06812,
    0.02024,
    0.02687,
    0.00447,
    -0.02029,
    0.00858,
    -0.02396,
    0.02022,
    0.00088,
    0.00502,
    -0.00829,
    -0.02311,
    0.0607,
    0.01256,
    0.02773,
    0.08592,
    0.0352,
    0.00839,
    -0.02621,
    0.01679,
    0.00479,
    0.01508,
    0.03016,
    0.00145,
    0.01686,
    0.01149,
    -0.0088,
    0.00236,
    0.05124,
    0.00869,
    -0.00771,
    0.00542,
    -0.00804,
    -0.00971,
    -0.04267,
    -0.06278,
    -0.02944,
    0.0467,
    0.00529,
    0.04574,
    0.05281,
    0.02845,
    -0.00617,
    -0.01022,
    -0.00879,
    -0.02127,
    0.01747,
    -0.0078,
    0.00748,
    0.02347,
    0.0279,
    0.02145,
    -0.0487,
    -0.02968,
    0.01004,
    0.02036,
    -0.09217,
    -0.02037,
    0.07659,
    0.00296,
    -0.04651,
    0.00414,
    -0.02389,
    0.00044,
    -0.02069,
    -0.01521,
    -0.04565,
    -0.00537,
    0.02553,
    0.02832,
    -0.0243,
    0.01509,
    -0.02235,
    -0.0146,
    -0.01732,
    0.04449,
    0.02857,
    0.00813,
    -0.00706,
    0.00403,
    0.03857,
    -0.01276,
    -0.02428,
    -0.01397,
    0.01626,
    0.00375,
    -0.03881,
    -0.01496,
    0.02002,
    0.01947,
    -0.03223,
    -0.00231,
    0.01265,
    -0.00986,
    -0.0231,
    0.00581,
    -0.00234,
    0.00458,
    0.00272,
    -0.00093,
    0.05017,
    -0.01188,
    -0.02758,
    -0.01505,
    -0.01997,
    0.0222,
    -0.02143,
    0.03694,
    0.02804,
    -0.00066,
    0.043,
    -0.01693,
    -0.02497,
    0.02034,
    0.02974,
    -0.00411,
    0.00142,
    -0.02341,
    0.01826,
    -0.02318,
    0.01445,
    -0.0266,
    0.03245,
    -0.04808,
    -0.00516,
    0.02453,
    -0.00258,
    -0.0173,
    -0.00478,
    0.04014,
    -0.00159,
    0.0074,
    0.01887,
    -0.00715,
    0.02121,
    0.00883,
    0.00739,
    0.02863,
    -0.02123,
    -0.00999,
    -0.0176,
    0.0237,
    0.04139,
    -0.08495,
    0.00986,
    -0.00015,
    0.0211,
    -0.02936,
    -0.02743,
    -0.00224,
    0.02815,
    0.01969,
    -0.01094,
    0.01119,
    -0.00989,
    -0.01508,
    0.0475,
    -0.02007,
    0.02071,
    -0.0039,
    0.02947,
    0.00935,
    0.02872,
    0.00929,
    -0.01144,
    0.00405,
    0.0121,
    -0.02236,
    -0.00619,
    -0.01637,
    -0.0028,
    -0.01048,
    0.01177,
    0.02898,
    0.0049,
    0.01508,
    0.02385,
    -0.01388,
    0.0289,
    -0.0045,
    -0.00457,
    -0.00291,
    -0.04753,
    -0.00489,
    0.01057,
    0.00915,
    -0.02906,
    -0.01691,
    -0.01124,
    -0.01306,
    -0.03524,
    0.01513,
    -0.02469,
    0.04001,
    -0.03574,
    -0.01843,
    0.01583,
    -0.01317,
    -0.01345,
    0.00265,
    -0.00428,
    0.03787,
    0.04347,
    0.02727,
    -0.12752,
    -0.05842,
    0.07194,
    -0.08375,
    -0.00758,
    -0.00495,
    -0.04558,
    -0.01288,
    0.01839,
    0.01411,
    -0.02707,
    -0.0052,
    0.02672,
    0.01866,
    -0.02281,
    0.00033,
    0.03199,
    -0.00531,
    -0.00567,
    -0.0179,
    -0.00656,
    -0.03986,
    0.01173,
    0.01464,
    0.00953,
    -0.01111,
    0.00581,
    -0.01722,
    0.02478,
    -0.00624,
    -0.01484,
    -0.04543,
    -0.01973,
    -0.00956,
    0.02253,
    -0.00751,
    -0.00657,
    -0.00302,
    0.00705,
    0.04259,
    -0.00312,
    -0.01252,
    -0.01974,
    -0.02692,
    0.02905,
    -0.03346,
    0.00829,
    0.00967,
    -0.01268,
    -0.0181,
    0.04231,
    -0.02224,
    -0.00822,
    -0.0201,
    0.02552,
    0.00072,
    -0.02773,
    0.03501,
    -0.01167,
    -0.02204,
    -0.00331,
    -0.01703,
    0.01163,
    0.02646,
    0.02702,
    0.00535,
    -0.00824,
    -0.0002,
    0.05468,
    -0.01527,
    0.01724,
    0.04066,
    0.01771,
    0.01162,
    0.03335,
    -0.0169,
    0.01855,
    0.01327,
    -0.02049,
    -0.00892,
    0.02686,
    0.02414,
    0.01493,
    -0.01433,
    -0.01596,
    0.01004,
    0.02063,
    -0.00393,
    -0.01741,
    0.00091,
    -0.01265,
    -0.01033,
    -0.03474,
    -0.01692,
    0.01481,
    -0.02615,
    -0.01925,
    -0.03297,
    -0.02227,
    0.00269,
    0.01753,
    0.01734,
    -0.00952,
    0.03149,
    -0.0043,
    0.01006,
    -0.00286,
    0.01602,
    -0.02634,
    -0.03689,
    0.01415,
    -0.02666,
    0.02642,
    0.02824,
    0.02414
   ],
   "INFUSION": [
    0.0393,
    0.00903,
    0.00669,
    -0.02305,
    -0.05591,
    0.03034,
    -0.00094,
    -0.01667,
    0.00145,
    -0.01648,
    0.00672,
    0.01037,
    0.01396,
    -0.00353,
    -0.01178,
    -0.00365,
    -0.03644,
    0.02866,
    0.00617,
    -0.04494,
    -0.03148,
    0.00802,
    0.00089,
    -0.01155,
    -0.01352,
    0.03393,
    0.02895,
    -0.00695,
    0.0587,
    -0.03,
    -0.03201,
    0.00476,
    -0.02705,
    -0.02124,
    0.01942,
    -0.00664,
    -0.01457,
    0.01746,
    0.0127,
    -0.01696,
    0.01272,
    -0.01311,
    0.0299,
    -0.00105,
    -0.068,
    0.00951,
    0.01336,
    -0.01574,
    -0.02068,
    0.00432,
    0.01108,
    0.00613,
    -0.00927,
    -0.02365,
    0.04563,
    0.03045,
    -0.00473,
    -0.00669,
    -0.047,
    0.01727,
    0.02497,
    0.00181,
    0.01941,
    -0.00049,
    0.02878,
    -0.01338,
    -0.01827,
    -0.00229,
    -0.01473,
    0.00367,
    0.00832,
    0.00937,
    0.02094,
    -0.03196,
    -0.02356,
    -0.01469,
    0.01168,
    -0.0483,
    -0.02567,
    0.00159,
    0.03997,
    -0.0117,
    0.03084,
    0.00331,
    0.03085,
    -0.00096,
    0.00209,
    -0.02347,
    -0.0271,
    -0.02213,
    0.00015,
    0.00116,
    0.02629,
    0.00892,
    0.04675,
    -0.00209,
    -0.023,
    0.01997,
    0.04066,
    -0.00441,
    0.01371,
    -0.04826,
    -0.01275,
    -0.00943,
    -0.00322,
    -0.00534,
    0.03741,
    0.01968,
    -0.01157,
    -0.05227,
    -0.00217,
    -0.01642,
    -0.02071,
    0.00955,
    -0.00448,
    -0.00533,
    0.02218,
    -0.01941,
    0.02352,
    0.01317,
    -0.06198,
    0.00971,
    0.03613,
    0.01279,
    0.01741,
    0.02292,
    0.01429,
    -0.01967,
    0.0077,
    0.02394,
    -0.06275,
    -0.00414,
    -0.01293,
    0.03533,
    -0.03671,
    -0.01372,
    0.00128,
    0.0035,
    -0.00378,
    -0.00293,
    0.04781,
    0.03257,
    0.02308,
    -0.00776,
    -0.00801,
    0.00996,
    -0.04068,
    -0.01375,
    0.02267,
    -0.00883,
    0.02486,
    -0.01545,
    0.0137,
    0.03418,
    0.00914,
    -0.00105,
    -0.00061,
    0.02817,
    -0.00516,
    0.0127,
    0.00777,
    0.02442,
    -0.02265,
    0.07049,
    -0.00659,
    -0.04217,
    0.01853,
    0.02125,
    0.01395,
    0.0103,
    0.00541,
    -0.00435,
    0.01747,
    -0.01009,
    -0.02901,
    0.02034,
    -0.01087,
    0.00041,
    -0.0268,
    0.00252,
    -0.0058,
    0.04687,
    0.01046,
    -0.03788,
    0.01973,
    -0.00011,
    -0.01017,
    0.00675,
    -0.04635,
    -0.00387,
    -0.0278,
    0.02088,
    -0.033,
    0.00306,
    -0.08424,
    0.01151,
    0.02435,
    -0.0528,
    0.01921,
    0.00841,
    0.01041,
    0.00721,
    -0.01112,
    -0.00287,
    0.01726,
    -0.00405,
    -0.02077,
    -0.00872,
    0.01864,
    -0.00206,
    -0.00937,
    -0.02064,
    -0.0057,
    -0.00095,
    0.00569,
    -0.02659,
    -0.01228,
    0.00982,
    0.00822,
    -0.02336,
    -0.02559,
    0.01437,
    0.00588,
    0.07109,
    -0.0267,
    -0.01396,
    0.01615,
    -0.01984,
    0.03707,
    -0.00706,
    0.04211,
    -0.01202,
    0.02327,
    -0.00663,
    -0.02568,
    0.01151,
    -0.00057,
    0.0463,
    -0.03441,
    -0.03749,
    -0.01132,
    0.00892,
    0.05745,
    -0.05781,
    0.0154,
    0.01714,
    -0.00255,
    -0.06465,
    0.01799,
    0.03047,
    0.00902,
    -0.05863,
    0.0335,
    0.01773,
    -0.00027,
    -0.00933,
    -0.02821,
    -0.0163,
    0.0116,
    0.02828,
    0.00853,
    -0.00717,
    0.0003,
    0.02447,
    -0.00827,
    0.00225,
    -0.01907,
    -0.02334,
    0.00266,
    -0.0082,
    -0.02254,
    -0.01025,
    0.04928,
    0.00262,
    -0.00985,
    0.01345,
    0.01142,
    0.00941,
    0.01124,
    0.00548,
    -0.02391,
    -0.00377,
    0.03127,
    -0.00734,
    -0.02944,
    -0.0078,
    0.0521,
    0.0025,
    0.00369,
    -0.01683,
    0.02562,
    -0.01269,
    0.01958,
    -0.0209,
    0.00764,
    -0.03826,
    0.04922,
    0.02545,
    0.03149,
    -0.01886,
    0.01536,
    0.00998,
    -0.00385,
    -0.00885,
    -0.00669,
    0.02305,
    -0.03988,
    -0.01018,
    0.02405,
    -0.02557,
    0.003,
    0.03183,
    -0.01237,
    0.0266,
    0.01739,
    0.01111,
    -0.00161,
    -0.03269,
    0.04952,
    0.02063,
    -0.01034,
    -0.01658,
    0.01682,
    -0.02553,
    0.00377,
    -0.02484,
    0.0044,
    0.01402,
    -0.02213,
    0.02196,
    -0.0239,
    0.03205,
    0.00756,
    0.02549,
    0.02697,
    -0.00119,
    0.00751,
    0.03427,
    -0.01298,
    0.00116,
    0.00534,
    0.00181,
    -0.02211,
    -0.02623,
    -0.03066,
    0.0157,
    -0.01712,
    0.00215,
    0.01039,
    0.03478,
    0.001,
    0.01199,
    0.02508,
    0.01618,
    0.08118,
    0.00261,
    0.03735,
    0.01543,
    0.00395,
    0.02437,
    0.04035,
    -0.03367,
    0.00693,
    0.04141,
    -0.02737,
    -0.03177,
    -0.00463,
    -0.00519,
    0.0092,
    0.03761,
    -0.00208,
    0.02017,
    0.02358,
    0.00756,
    -0.01494,
    -0.01034,
    0.01583,
    -0.00909,
    0.02082,
    -0.01326,
    0.04334,
    -0.02135,
    -0.03543,
    0.05604,
    -0.01074,
    0.0064,
    -0.00088,
    0.01509,
    0.01779,
    -0.01844,
    -0.01052,
    -0.00398,
    0.01406,
    -0.01218,
    -0.02012,
    -0.02971,
    0.00472,
    -0.00232,
    0.01639,
    0.00694,
    0.01694,
    -0.0308,
    -0.00122,
    -0.03383,
    -0.00366,
    -0.02247,
    0.03868,
    0.00715,
    0.007,
    0.03991,
    0.03941,
    0.00321,
    -0.01407,
    -0.00377,
    -0.02632,
    0.00535,
    -0.01477,
    0.03262,
    -0.02262,
    -0.01612,
    0.00449,
    0.02839,
    0.00043,
    -0.03996,
    0.02821,
    -0.00987,
    -0.01848,
    -0.03616,
    0.00931,
    0.00301,
    0.02931,
    -0.03011,
    0.01918,
    0.01475,
    -0.02626,
    0.03013,
    -0.02427,
    0.02554,
    -0.02895,
    -0.0065,
    0.00505,
    -0.04274,
    0.00906,
    0.02381,
    0.05159,
    -0.03882,
    -0.0494,
    -0.01198,
    -0.01391,
    -0.00926,
    0.02545,
    0.01748,
    0.03906,
    -0.04838,
    -0.0321,
    0.03224,
    -0.0124,
    -0.02825,
    -0.04377,
    0.02517,
    -0.04982,
    -0.05058,
    0.03753,
    -0.00591,
    -0.02444,
    0.02368,
    -0.00244,
    -0.00226,
    -0.01426,
    0.03604,
    -0.04946,
    0.00751,
    -0.00992,
    0.04955,
    0.0146,
    -0.03792,
    0.00255,
    0.01954,
    -0.02591,
    0.02541,
    6e-05,
    -0.04861,
    -0.02136,
    0.02991,
    -0.01201,
    0.03608,
    0.04367,
    -0.0002,
    -0.0052,
    -0.00783,
    -0.03783,
    -0.05507,
    -0.02184,
    0.0166,
    -0.0072,
    0.01524,
    0.00507,
    -0.01306,
    -0.00408,
    -0.0344,
    0.02968,
    0.04061,
    -0.00924,
    -0.00036,
    0.03657,
    -0.0569,
    -0.00445,
    0.05193,
    0.01707,
    0.0081,
    -0.05045,
    0.04399,
    -0.00699,
    -0.0068,
    0.00418,
    0.00789,
    -0.02072,
    0.04059,
    -0.02339,
    -0.00093,
    0.00352,
    -0.00834,
    0.01556,
    -0.02867,
    -0.0047,
    -0.03603,
    0.00795,
    0.00025,
    0.00334,
    0.00426,
    0.02243,
    0.06899,
    -0.0571,
    0.02092,
    -0.00495,
    0.01012,
    0.01152,
    0.01372,
    -0.00869,
    -0.01981,
    0.02341,
    -0.03483,
    0.00095,
    0.02633,
    -0.01433,
    0.01719,
    0.03045,
    0.01192,
    0.01473,
    -0.01081,
    -0.01832,
    0.00504,
    -0.01176,
    0.06815,
    -0.05264,
    -0.00769,
    -0.00602,
    -0.0427,
    -0.0098,
    -0.00928,
    0.04515,
    0.01706,
    -0.00255,
    -0.03498,
    0.01801,
    -0.0157,
    -0.0305,
    -0.01334,
    0.02774,
    -0.00312,
    0.04992,
    -0.01567,
    -0.00347,
    0.00772,
    0.05092,
    -0.00661,
    0.021,
    -0.02245,
    -0.00081,
    0.01175,
    -0.0542,
    -0.02107,
    0.0001,
    0.01165,
    0.00501,
    -0.01905,
    -0.00541,
    0.02218,
    0.01535,
    0.00155,
    0.02586,
    -0.00735,
    0.01956,
    0.00705,
    -0.01384,
    -0.02383,
    0.00789,
    0.01054,
    0.03979,
    -0.02231,
    -0.0038,
    -0.02775,
    -0.0283,
    -0.04188,
    0.01067,
    0.00752,
    0.01637,
    0.04163,
    0.06094,
    0.02695,
    -0.00449,
    0.02201,
    0.02434,
    0.00694,
    -0.0322,
    -0.01366,
    0.0323,
    0.00762,
    -0.00942,
    -0.01005,
    -0.00844,
    0.02329,
    -0.00049,
    -0.0471,
    -0.01902,
    0.00077,
    -0.00632,
    0.02682,
    0.01831,
    -0.00823,
    -0.00973,
    -0.04006,
    0.00449,
    0.00101,
    -0.00343,
    -0.0205,
    0.00118,
    -0.03938,
    -0.01487,
    0.01762,
    -0.02327,
    -0.02131,
    -0.00545,
    -0.0155,
    0.00517,
    -0.01566,
    0.02158,
    -0.00195,
    -0.01552,
    -0.02582,
    -0.0053,
    0.00817,
    0.03521,
    -0.02495,
    -0.01461,
    -0.02312,
    0.00336,
    -0.05442,
    0.03783,
    0.01481,
    -0.00916,
    -0.03178,
    -0.0015,
    -0.00868,
    0.0203,
    -0.0247,
    0.00559,
    0.03779,
    0.00527,
    -0.00542,
    0.01623,
    0.00352,
    -0.02017,
    0.03086,
    -0.01226,
    -0.01348,
    -0.01668,
    -0.03803,
    0.01269,
    0.00914,
    -0.02067,
    0.01643,
    -0.01123,
    -0.07645,
    -0.00289,
    -0.01486,
    0.02735,
    0.0113,
    0.0572,
    -0.00608,
    0.03677,
    0.00799,
    -0.01205,
    0.03548,
    0.01845,
    -0.02354,
    0.01148,
    -0.01559,
    0.02861,
    0.02127,
    0.02431,
    -0.01354,
    0.0567,
    -0.0082,
    -0.00499,
    0.00592,
    0.00814,
    -0.00539,
    -0.03376,
    -0.00245,
    0.01837,
    -0.01171,
    -0.03673,
    -0.05415,
    -0.06678,
    -0.01684,
    0.00704,
    -0.018,
    -0.00984,
    0.03773,
    -0.0423,
    0.00292,
    0.03342,
    -0.01249,
    0.01663,
    -0.03357,
    -0.02189,
    0.01583,
    -0.01638,
    -0.01751,
    -0.01025,
    -0.02572,
    -0.0327,
    -0.02063,
    0.04358,
    0.0182,
    0.00706,
    -0.01992,
    0.00451,
    0.01253,
    0.03397,
    0.00524,
    -0.00367,
    -0.02811,
    0.01153,
    0.00606,
    0.00674,
    0.00656,
    -0.03498,
    -0.01456,
    0.02647,
    0.03901,
    -0.01657,
    -0.02897,
    -0.05831,
    -0.02461,
    -0.0305,
    0.00556,
    -0.01341,
    -0.03289,
    0.06249,
    0.03316,
    -0.04007,
    -0.01365,
    0.00333,
    0.02463,
    -0.02875,
    -0.02295,
    -0.0418,
    -0.04008,
    0.0226,
    0.01108,
    -0.00234,
    -0.0017,
    -0.01951,
    -0.00739,
    -0.03557,
    -0.0155,
    -0.02501,
    0.01788,
    0.0137,
    0.00899,
    -0.01959,
    0.01614,
    0.07198,
    0.00266,
    -0.00457,
    0.00435,
    -0.01077,
    -0.02542,
    -0.00672,
    0.03793,
    -0.00446,
    0.04595,
    0.02788,
    -0.00558,
    0.02053,
    0.00747,
    -0.0119,
    -0.02646,
    -0.01011,
    0.03215,
    0.02177,
    -0.03737,
    0.02268,
    -0.00937,
    -0.02521,
    0.06647,
    -0.02766,
    -0.02664,
    -0.05184,
    0.02991,
    0.01289,
    -0.00292,
    -0.04584,
    -0.02902,
    0.02343,
    -0.0409,
    -0.01017,
    0.0267,
    -0.00417,
    -0.03353,
    0.01058,
    -0.00661,
    0.0224,
    0.01249,
    -0.02688,
    -0.00732,
    0.03962,
    -0.01366,
    0.04398,
    -0.02535,
    0.03627,
    0.00296,
    -0.04323,
    0.01114,
    0.03744,
    -0.00103,
    -0.02426,
    0.01493,
    -0.01499,
    0.03759,
    -0.00279,
    0.01086,
    -0.01471,
    -0.00304,
    0.0151,
    -0.04439,
    0.02765,
    0.00696,
    0.00475,
    -0.03107,
    -0.04981,
    0.00577,
    0.00302,
    0.00131,
    -0.0308,
    0.00129,
    0.0292,
    0.0178,
    0.00858,
    -0.00334,
    0.03927,
    -0.01504,
    -0.0051,
    -0.0693,
    -0.00937,
    0.01778,
    -0.0276,
    0.00395,
    0.02139,
    0.0195,
    -0.019,
    0.0254,
    -0.0275,
    0.00678,
    0.00025,
    0.0462,
    0.068,
    0.02153,
    0.00644,
    -0.01588,
    -0.02473,
    0.01304,
    0.02816,
    -0.02001,
    0.01777,
    0.00025,
    -0.02943,
    -0.02975,
    0.02853,
    -0.04244,
    -0.05553,
    0.04024,
    -0.01908,
    -0.01593,
    0.03081,
    0.01887,
    0.00567,
    -0.01583,
    0.00725,
    0.00605,
    0.0473,
    -0.05045,
    -0.01429,
    -0.01368,
    0.01511,
    0.04351,
    -0.01503,
    -0.00747,
    -0.03895,
    0.01642,
    0.02563,
    0.05167,
    0.0247,
    -0.01912,
    0.0082,
    0.05522,
    0.01159,
    0.00305,
    -0.01136,
    0.07581,
    0.05783,
    -0.01369,
    -0.0211,
    -0.02557,
    0.007,
    -0.0134,
    -0.01131,
    -0.00654,
    0.02948,
    -0.03548,
    -0.01725,
    -0.03247,
    -0.01772,
    -0.01408,
    -0.02146,
    0.00848,
    0.01122,
    0.00451,
    0.00431,
    0.06757,
    -0.00255,
    0.00979,
    0.01875,
    0.01331,
    -0.05882,
    -0.00719,
    0.01138,
    0.00177,
    -0.02736,
    -0.03178,
    0.03798,
    -0.02401,
    0.01255,
    0.05765,
    0.02023,
    0.02992,
    0.00063,
    -0.01383,
    -0.03075,
    -0.01816,
    -0.01776,
    0.03589,
    -0.03949,
    -0.05602,
    -0.03759,
    0.01237,
    -0.00709,
    0.00418,
    0.02362,
    -0.00759,
    -0.02511,
    -0.03776,
    -0.03159,
    -0.00305,
    0.0061,
    0.02739,
    -0.00864,
    0.0324,
    -0.02364,
    -0.04731,
    0.01788,
    -0.04488,
    -0.06227,
    -0.02185,
    0.01322,
    -0.01942,
    0.02285,
    0.00326,
    -0.00805,
    -0.00294,
    -0.0149,
    0.01799,
    0.04545,
    0.0091,
    0.00186,
    -0.02845,
    -0.03546,
    -0.02349,
    0.01671,
    0.00552,
    0.00454,
    0.03257,
    -0.03723,
    -0.01722,
    -0.00084,
    0.03195,
    0.01668,
    -0.0163,
    -0.00483,
    -0.03312,
    -0.02646,
    -0.03246,
    9e-05,
    0.01435,
    -0.02317,
    0.00691,
    0.00232,
    0.00174,
    -0.00699,
    0.02975,
    0.00872,
    -0.02627,
    0.01698,
    -0.00841,
    0.02423,
    0.00568,
    0.00388,
    -0.01155,
    -0.00219,
    -0.03619,
    -0.02375,
    0.01846,
    0.02808,
    -0.01741,
    -0.01444,
    -0.00682,
    -0.01371,
    -0.00337,
    -0.02778,
    0.03413,
    -0.01787,
    0.04327,
    -0.04971,
    0.04544,
    -0.02493,
    -0.03882,
    -0.01325,
    0.02067,
    -0.04235,
    -0.01515,
    -0.01362,
    -0.00303,
    -0.00594,
    0.00382,
    -0.00886,
    0.05266,
    0.00773,
    -0.00086,
    -0.00381,
    0.00597,
    0.00425,
    0.00786,
    -0.02533,
    -0.00054,
    0.00517,
    0.01734,
    -0.07789,
    0.0183,
    0.03017,
    -0.01657,
    0.03299,
    0.06892,
    -0.04839,
    0.0503,
    -0.00161,
    -0.00095,
    -0.02471,
    -0.00712,
    0.02094,
    0.00644,
    0.02338,
    0.00942,
    0.00894,
    -0.00827,
    0.01241,
    0.01406,
    0.01125,
    -0.00221,
    0.03533,
    -0.01064,
    0.00905,
    -0.0289,
    0.0033,
    0.02278,
    -0.00787,
    -0.00331,
    -0.01941,
    0.04671,
    -0.01338,
    -0.01355,
    -0.02737,
    0.03462,
    0.00525,
    -0.01171,
    0.00264,
    0.02004,
    0.02818,
    0.0334,
    -0.01717,
    -0.02343,
    0.01188,
    -0.01376,
    -0.01605,
    -0.00267,
    -0.00298,
    0.01051,
    0.01365,
    0.00515,
    0.0092,
    0.01674,
    -0.01134,
    -0.01954,
    0.03014,
    0.03265,
    0.03795,
    0.04446,
    0.00062,
    -0.00095,
    0.01956,
    -0.00446,
    -0.02553,
    0.01128,
    -0.01463,
    0.01942,
    0.01551,
    0.00261,
    0.01003,
    -0.02697,
    -0.00585,
    0.02712,
    0.02347,
    -0.08438,
    -0.00398,
    -0.0202,
    0.02892,
    -0.07693,
    0.00097,
    -0.04079,
    -0.01928,
    -0.03132,
    0.04298,
    0.00068,
    0.03263,
    0.01244,
    0.00146,
    0.00713,
    0.00262,
    -0.02326,
    0.01918,
    -0.00654,
    0.04268,
    -0.02092,
    -0.02778,
    0.04002,
    0.00105,
    -0.03813,
    0.00903,
    -0.02028,
    0.01928,
    0.00551,
    0.01086,
    -0.03992,
    0.01547,
    0.06499,
    0.05376,
    -0.03886,
    0.03703,
    -0.05832,
    0.01795,
    0.00012,
    -0.03249,
    -0.01456,
    0.00114,
    0.05087,
    0.00145,
    9e-05,
    -0.01128,
    -0.00112,
    0.01578,
    0.07025,
    -0.00165,
    0.03931,
    -0.01763,
    -0.047,
    0.03596,
    -0.00167,
    -0.03337,
    -0.00252,
    0.04216,
    0.01971,
    -0.01399,
    -0.00564,
    0.00557,
    0.02948,
    -0.0232,
    0.02752,
    -0.02875,
    0.0285,
    -0.04754,
    0.05749,
    -0.00389,
    -0.02214,
    -0.02233,
    0.00271,
    -0.00419,
    -0.03701,
    -0.00197,
    -0.0092,
    -0.02336,
    -0.02078,
    -0.01429,
    0.01212,
    0.01276,
    -0.00808,
    0.01211,
    -0.0282,
    0.00665,
    0.03518,
    0.02311,
    0.01417,
    -0.01389,
    -0.00993,
    0.02699,
    -0.01504,
    -0.01293,
    0.00166,
    -0.05579,
    -0.00716,
    0.04511,
    -0.00487,
    0.01325,
    -0.02061,
    -0.01117,
    0.02497,
    -0.0061,
    -0.02941,
    0.01226,
    -0.00635,
    0.00271,
    0.02648,
    -0.01113,
    0.00704,
    -0.0183,
    -0.02296,
    -0.02218,
    -0.01626,
    -0.00153,
    -0.04253,
    0.02642,
    0.00018,
    0.01815,
    -0.05718,
    0.00524,
    0.02381,
    -0.00449,
    -0.06211,
    -0.00273,
    0.0144,
    -0.0024,
    -0.01886,
    -0.03054,
    0.0215,
    -0.01833,
    -0.02895,
    -0.00246,
    -0.01512,
    0.01028,
    0.06114,
    0.00128,
    0.00313,
    -0.01514,
    -0.03824,
    -0.00707,
    -0.01029,
    -0.01226,
    0.01669,
    -0.0364,
    0.01013,
    0.02892,
    -0.00683,
    -0.0006,
    -0.01732,
    0.00163,
    0.01064,
    -0.01727,
    0.00908,
    -0.02678,
    -0.00826,
    -0.01715,
    0.02566,
    0.0109,
    -0.01849,
    0.01536,
    0.00892,
    -0.01593,
    0.02741,
    0.00791,
    0.02246,
    0.00867,
    0.01026,
    0.02563,
    -0.02719,
    -0.01018,
    0.00147,
    -0.02299,
    0.0145,
    0.0375,
    0.00411,
    0.01109,
    0.02292,
    0.00475,
    -0.00342,
    -0.0338,
    -0.01642,
    0.02341,
    -0.01389,
    0.01486,
    0.00429,
    0.01091,
    -0.00531,
    0.03292,
    0.03961,
    0.01011,
    -0.00317,
    0.05237,
    0.01212,
    -0.00032,
    -0.04278,
    -0.00445,
    0.05664,
    0.02385,
    0.01106,
    -0.01798,
    0.03353,
    -0.0293,
    0.03931,
    -0.00373,
    0.00401,
    0.00307,
    -0.00746,
    -0.00768,
    -0.00752,
    -0.02723,
    0.01016,
    0.00051,
    -0.01768,
    0.00385,
    0.01483,
    -0.02237,
    0.01052,
    0.00384,
    0.01157,
    0.04927,
    0.02143,
    0.00288,
    -0.0285,
    -0.04698,
    0.00818,
    0.01204,
    0.04713,
    0.00352,
    -0.00875,
    0.0332,
    -0.01652,
    0.00385,
    -0.03462,
    -0.02756,
    0.00502,
    0.02726,
    0.01727,
    0.00858,
    -0.03529,
    -0.00855,
    -0.01979,
    0.01238,
    0.01885,
    -0.00025,
    0.01866,
    0.01195,
    0.00048,
    0.00969,
    -0.02834,
    -0.0095,
    -0.04002,
    0.0078,
    0.00206,
    -0.01953,
    -0.03469,
    -0.00291,
    -0.02539,
    0.0058,
    -0.01466,
    -0.00101,
    -0.01805,
    -0.00709,
    0.0237,
    -0.00371,
    -0.00852,
    -0.03865,
    0.02908,
    -0.08006,
    -0.06051,
    0.02599,
    -0.01857,
    0.00069,
    0.05236,
    -0.01451,
    -0.03714,
    -0.02777,
    -0.13744,
    0.03514,
    0.01514,
    -0.01989,
    -0.03003,
    -0.05348,
    0.01524,
    -0.00171,
    -0.01667,
    0.01336,
    -0.00367,
    0.00456,
    -0.00701,
    -0.02367,
    0.00593,
    6e-05,
    -0.03552,
    -0.00735,
    -0.01725,
    0.00747,
    -0.00344,
    -0.00673,
    0.00959,
    -0.01064,
    0.00965,
    0.06927,
    -0.00707,
    0.0166,
    -0.02133,
    0.01811,
    0.00535,
    -0.00367,
    -0.01204,
    -0.01225,
    -0.01708,
    -0.0051,
    -0.0204,
    -0.01193,
    -0.00749,
    -0.00665,
    -0.01225,
    -0.05848,
    0.02952,
    0.00273,
    -0.01355,
    0.01648,
    -0.02404,
    -0.03369,
    0.03682,
    0.02734,
    -0.02358,
    -0.0318,
    -0.00428,
    -0.02364,
    0.00522,
    0.00162,
    -0.00051,
    -0.05067,
    0.04065,
    0.00525,
    -0.0074,
    -0.02293,
    -0.03189,
    -0.01794,
    0.01162,
    0.03315,
    0.01521,
    0.02606,
    -0.0176,
    -0.01118,
    -0.00684,
    -0.02637,
    0.02093,
    -0.00843,
    -0.02341,
    0.00402,
    -0.02097,
    -0.01519,
    0.03028,
    -0.03303,
    0.02183,
    0.02218,
    -0.01797,
    0.00149,
    -0.0402,
    0.02838,
    -0.00908,
    -0.02737,
    0.03777,
    -0.01152,
    0.01153,
    -0.01019,
    -0.01438,
    0.03815,
    -0.02314,
    -0.00392,
    0.02272,
    0.01632,
    0.04035,
    0.04861,
    0.02079,
    0.01218,
    0.01044,
    0.06506,
    -0.0449,
    -0.006,
    0.03054,
    0.00333,
    0.0253,
    0.00532,
    0.0126,
    -0.07895,
    -0.03412,
    -0.0358,
    0.0021
   ],
   "ON": [
    0.00277,
    0.00906,
    0.00865,
    -0.04706,
    0.02129,
    0.0088,
    -0.03177,
    0.01282,
    -0.02279,
    -0.03234,
    0.03246,
    0.00452,
    -0.05772,
    0.02231,
    0.00668,
    0.00886,
    -0.00037,
    0.00035,
    -0.00128,
    0.04813,
    0.01134,
    -0.04029,
    0.01313,
    -0.027,
    -0.00158,
    0.00207,
    0.0127,
    0.02698,
    -0.03669,
    0.02017,
    0.01679,
    -0.04107,
    0.0279,
    -0.02409,
    0.02077,
    0.04644,
    -0.00016,
    -0.00487,
    -0.00165,
    0.01474,
    -0.02725,
    0.0078,
    -0.06533,
    -0.03053,
    0.00472,
    0.00217,
    -0.02694,
    0.02348,
    0.01352,
    0.01669,
    0.00245,
    0.01353,
    -0.02556,
    0.03479,
    -0.06848,
    -0.03378,
    -0.02308,
    0.00058,
    0.01766,
    -0.04281,
    -0.00502,
    0.01368,
    -0.01708,
    -0.02042,
    -0.02248,
    0.01236,
    0.0092,
    0.00409,
    -0.02198,
    -0.01258,
    -0.02764,
    0.01628,
    -0.00408,
    -0.00742,
    0.00482,
    -0.02212,
    -0.04175,
    0.00578,
    -0.0082,
    -0.00524,
    -0.06283,
    -0.00278,
    0.00883,
    -0.01997,
    -0.03422,
    0.00033,
    0.01527,
    0.01618,
    0.02604,
    -0.01467,
    0.01399,
    0.00353,
    -0.0048,
    -0.03738,
    -0.01459,
    -0.04146,
    0.00093,
    -0.00247,
    -0.02741,
    0.03134,
    -0.01339,
    0.01583,
    0.00214,
    0.04478,
    -0.00294,
    -0.02837,
    -0.01238,
    0.00262,
    -0.00792,
    -0.0047,
    0.00627,
    0.03597,
    0.00634,
    -0.00191,
    -0.03378,
    0.02962,
    0.01386,
    0.0196,
    -0.00924,
    0.01422,
    0.00983,
    -0.01853,
    0.01744,
    -0.0121,
    -0.03246,
    -0.02036,
    -0.00846,
    0.00337,
    0.00147,
    0.0027,
    0.02193,
    0.01279,
    -0.00088,
    -0.02993,
    0.05535,
    -0.00809,
    -0.01574,
    -0.00385,
    0.01669,
    -0.01246,
    -0.01872,
    -0.03773,
    -0.01387,
    0.0309,
    -0.03734,
    -0.03532,
    0.03877,
    -0.01658,
    -0.0177,
    0.04155,
    -0.00537,
    0.0136,
    0.02218,
    -0.00192,
    -0.05806,
    0.0094,
    -0.01142,
    0.00076,
    0.01181,
    -0.0205,
    0.01266,
    -0.02126,
    -0.03328,
    -0.01091,
    -0.00734,
    0.02533,
    -0.00215,
    -0.01812,
    -0.02904,
    0.00047,
    -0.00713,
    0.01416,
    -0.02534,
    -0.02881,
    -0.01938,
    -0.01196,
    -0.00606,
    0.02235,
    -0.01241,
    -0.00543,
    -0.00651,
    -0.00111,
    0.01381,
    0.02726,
    0.01323,
    -0.00661,
    0.00192,
    0.03293,
    0.03274,
    0.0028,
    0.03443,
    0.02218,
    0.02307,
    0.00294,
    0.04053,
    -0.01238,
    -0.04497,
    0.00469,
    -0.04597,
    -0.01094,
    -0.03136,
    0.01188,
    0.02535,
    -0.01979,
    -0.07103,
    -0.00204,
    -0.00064,
    -0.01036,
    -0.04169,
    0.03483,
    0.00935,
    0.0267,
    0.01365,
    0.06446,
    0.01735,
    0.02508,
    0.02167,
    0.00959,
    0.00088,
    -0.00144,
    0.01223,
    -0.01056,
    0.00371,
    -0.03568,
    0.02012,
    -0.00275,
    -0.00558,
    0.02973,
    0.02765,
    0.00396,
    -0.08461,
    0.01558,
    -0.06532,
    -0.05144,
    0.01475,
    0.00459,
    -0.08866,
    -0.04409,
    -0.01021,
    0.01395,
    -0.01936,
    0.04878,
    -0.01205,
    0.06187,
    0.03579,
    -0.02227,
    -0.00879,
    -0.00318,
    -0.00972,
    -0.0074,
    0.01571,
    -0.01772,
    -0.03414,
    -0.03456,
    0.03767,
    -0.01054,
    0.01729,
    0.04011,
    0.00159,
    -0.02532,
    -0.01543,
    -0.00813,
    -0.01281,
    -0.0245,
    -0.00643,
    0.03594,
    0.00641,
    0.02369,
    -0.01304,
    0.03245,
    0.00155,
    0.00745,
    -0.02496,
    -0.00776,
    -0.00771,
    -0.00319,
    -0.01688,
    -0.00169,
    0.00223,
    -0.00579,
    -0.00099,
    0.0065,
    0.01152,
    -0.00088,
    0.02323,
    0.00149,
    0.00498,
    0.02938,
    0.00088,
    0.01172,
    -0.00376,
    0.02934,
    -0.01979,
    -0.00512,
    -0.01065,
    0.00377,
    -0.07201,
    -0.05286,
    -0.00953,
    0.04553,
    0.00756,
    0.0025,
    -0.00582,
    0.01037,
    -0.01811,
    0.00413,
    0.04021,
    -0.02191,
    -0.02088,
    -0.00784,
    -0.00336,
    -0.01004,
    0.00939,
    0.00688,
    -0.02735,
    0.05508,
    -0.01611,
    0.0205,
    -0.00021,
    0.00442,
    0.03961,
    0.04748,
    -0.02973,
    -0.01812,
    -0.00232,
    0.03704,
    0.04058,
    0.01204,
    0.01302,
    -0.04969,
    0.00713,
    -0.01683,
    0.0127,
    0.01578,
    -0.01419,
    -0.04482,
    -0.00567,
    -0.01943,
    0.02345,
    -0.0171,
    0.00421,
    -0.01656,
    0.00366,
    0.03045,
    -0.02527,
    -6e-05,
    0.05143,
    0.00679,
    -0.02155,
    -0.04364,
    0.01444,
    0.01033,
    0.00033,
    0.00558,
    -0.02348,
    0.02803,
    -0.03019,
    -0.00115,
    -0.0395,
    -0.01058,
    -0.02201,
    0.00073,
    -0.02425,
    -0.03706,
    -0.01067,
    0.02897,
    0.0151,
    0.02231,
    0.01081,
    -0.05355,
    -0.00445,
    -0.0007,
    -0.01358,
    -0.0347,
    0.03783,
    -0.00842,
    -0.00446,
    0.01761,
    0.00174,
    -0.01607,
    -0.00337,
    0.00424,
    -0.02738,
    -0.02613,
    -0.00028,
    0.00513,
    0.02358,
    -0.00779,
    -0.00243,
    -0.01276,
    0.02514,
    0.01749,
    -0.02373,
    0.03383,
    -0.01229,
    0.04278,
    -0.02401,
    -0.00351,
    -0.00445,
    -0.00264,
    -0.02852,
    0.08063,
    0.0199,
    -0.0046,
    -0.00687,
    -0.00856,
    -0.00986,
    -0.03296,
    -0.00222,
    0.00186,
    -0.01621,
    0.01262,
    -0.00887,
    -0.0036,
    -0.02101,
    0.01034,
    0.01539,
    -0.02982,
    0.02192,
    0.03099,
    -0.01989,
    -0.02901,
    -0.01717,
    0.00675,
    -0.03768,
    0.01918,
    0.01281,
    0.01779,
    -0.00896,
    -0.01617,
    -0.03075,
    0.00091,
    -0.03057,
    0.02414,
    -0.0144,
    0.0058,
    0.02918,
    -0.00802,
    0.0219,
    -0.00997,
    -0.00316,
    0.01568,
    -0.02465,
    0.00041,
    -0.00329,
    0.01447,
    0.03016,
    0.00343,
    0.0495,
    0.00754,
    -0.02377,
    -0.0062,
    -0.03577,
    0.05769,
    -0.0096,
    -0.0694,
    0.00762,
    0.01641,
    0.0167,
    -0.00728,
    -0.01073,
    0.06611,
    -0.01122,
    0.0006,
    0.01124,
    -0.0276,
    0.02511,
    -0.00531,
    -0.00808,
    -0.05344,
    0.03221,
    0.02932,
    -0.00625,
    -0.06403,
    0.03508,
    0.00465,
    0.0123,
    -0.00024,
    -0.00549,
    -0.00841,
    0.00505,
    0.02258,
    0.01912,
    0.03255,
    -0.01278,
    -0.02039,
    -0.0311,
    0.04634,
    -0.01773,
    0.00946,
    0.03762,
    0.03942,
    0.00785,
    -0.01784,
    0.02874,
    0.00683,
    0.01343,
    -0.00846,
    -0.00433,
    0.00329,
    -0.00311,
    -0.00202,
    -0.00535,
    0.01467,
    -0.03815,
    -0.00507,
    0.02056,
    -0.02863,
    -0.00273,
    -0.01751,
    0.03511,
    0.00902,
    -0.00446,
    -0.02652,
    0.02962,
    0.01641,
    0.05824,
    -0.03128,
    0.00546,
    -0.02153,
    0.02272,
    -0.03641,
    -0.01034,
    0.01945,
    0.05508,
    0.03775,
    0.00286,
    0.01935,
    0.03178,
    0.00399,
    -0.015,
    -0.00806,
    0.00776,
    -0.00571,
    0.01285,
    -0.03136,
    0.0246,
    0.03126,
    -0.00998,
    0.01954,
    -0.00763,
    0.00851,
    -0.01961,
    -0.00933,
    -0.01386,
    0.01183,
    -0.02455,
    0.04858,
    0.00449,
    0.01056,
    -0.00067,
    0.03801,
    0.02696,
    0.00894,
    0.01154,
    -0.03892,
    -0.02079,
    -0.01252,
    0.00111,
    -0.00229,
    -0.02883,
    -0.0471,
    0.00101,
    0.04799,
    -0.00919,
    0.04239,
    -0.02055,
    0.0089,
    -0.04093,
    -0.02681,
    0.02605,
    0.00361,
    -0.02375,
    -0.02907,
    -0.01895,
    0.00791,
    0.0016,
    0.01238,
    -0.0052,
    -0.00544,
    0.02896,
    0.01501,
    0.01174,
    -0.03229,
    -0.01775,
    0.02083,
    -0.00139,
    -0.00293,
    0.01049,
    -0.0286,
    -0.03611,
    0.03141,
    -0.01703,
    -0.00907,
    0.01152,
    -0.00323,
    0.0185,
    0.01885,
    0.01138,
    0.01459,
    -0.0302,
    0.03636,
    0.01794,
    -0.01134,
    -0.02099,
    -0.00938,
    -0.00422,
    -0.00277,
    -0.0598,
    0.06794,
    -0.01146,
    -0.00509,
    0.00137,
    0.04099,
    0.01812,
    -0.05375,
    0.01731,
    0.02634,
    0.00681,
    0.00014,
    0.01308,
    -0.00215,
    -0.01355,
    0.01071,
    0.00832,
    0.02542,
    -0.03304,
    -0.01915,
    -0.01716,
    -0.02768,
    -0.0202,
    0.01492,
    0.01493,
    -0.06138,
    -0.01895,
    -0.00924,
    0.00297,
    -0.00301,
    0.02883,
    -0.02083,
    0.03064,
    0.02336,
    0.0354,
    -0.00121,
    -0.01303,
    -0.02036,
    -0.02679,
    -0.03694,
    -0.02227,
    0.01603,
    -0.03517,
    0.0215,
    -0.02301,
    -0.00369,
    0.02313,
    0.02943,
    -0.00363,
    0.05422,
    -0.05245,
    0.04667,
    0.02822,
    0.03463,
    0.03386,
    -0.01312,
    -0.00959,
    0.02622,
    -0.01848,
    -0.01184,
    0.00571,
    -0.01318,
    8e-05,
    0.00919,
    0.00658,
    -0.08484,
    0.03513,
    -0.00244,
    0.0192,
    -0.00949,
    0.01878,
    -0.04773,
    0.02319,
    -0.00642,
    0.00359,
    0.047,
    -0.01896,
    0.02724,
    0.00087,
    -0.02993,
    -0.0201,
    0.03593,
    -0.03321,
    -0.02887,
    -0.01651,
    -0.0623,
    -0.00938,
    -0.00886,
    0.01545,
    0.0261,
    0.01195,
    0.00241,
    -0.01305,
    0.03135,
    -0.04813,
    -0.00294,
    0.01372,
    0.01266,
    0.00524,
    -0.00489,
    0.0512,
    -0.00662,
    -0.00451,
    0.02273,
    -0.0118,
    0.00658,
    0.0043,
    0.036,
    0.0048,
    -0.00554,
    -0.02612,
    -0.0084,
    0.02443,
    0.00121,
    -0.00868,
    -0.00419,
    0.01381,
    -0.04534,
    0.04806,
    -0.00643,
    0.05426,
    0.00547,
    0.02028,
    0.0066,
    0.04674,
    0.00242,
    -0.08907,
    -0.0191,
    -0.03291,
    -0.00603,
    -0.00706,
    0.0423,
    -0.01788,
    0.03073,
    0.03133,
    -0.00391,
    0.01391,
    0.02639,
    -0.03461,
    0.0246,
    -0.00085,
    0.01511,
    -0.0264,
    -0.05279,
    -0.00576,
    0.02032,
    0.01399,
    0.00484,
    -0.00615,
    0.01681,
    0.03801,
    0.00235,
    -0.07359,
    -0.01338,
    0.016,
    -0.0301,
    0.01567,
    -0.03002,
    0.03926,
    0.02287,
    0.01815,
    0.04782,
    0.03167,
    -0.05292,
    0.03341,
    -0.02721,
    0.00465,
    -0.00077,
    0.02456,
    -0.00568,
    -0.01767,
    0.00405,
    0.02503,
    0.03462,
    -0.04217,
    0.00831,
    -0.03489,
    -0.01985,
    0.01478,
    -0.00895,
    0.034,
    0.00105,
    0.02316,
    -0.04947,
    -0.00756,
    -0.01533,
    -0.04502,
    0.01164,
    -0.00254,
    -0.00508,
    0.00776,
    -0.00089,
    0.00084,
    -0.05397,
    0.00333,
    0.03431,
    -0.02107,
    0.01143,
    -0.00858,
    -0.02483,
    -0.00031,
    -0.00689,
    0.03384,
    0.00384,
    0.00485,
    0.045,
    0.0048,
    0.03293,
    -0.01667,
    -0.00049,
    -0.03418,
    -0.00754,
    0.01952,
    0.0137,
    0.02249,
    -0.03595,
    0.00472,
    -0.01129,
    0.00368,
    -0.01075,
    -0.00217,
    0.01798,
    -0.01701,
    0.01767,
    0.00307,
    0.00457,
    0.0006,
    -0.03735,
    -0.02868,
    0.0033,
    -0.00084,
    -0.01241,
    -0.00067,
    0.00992,
    0.02019,
    0.00431,
    0.03145,
    -0.01392,
    -0.00686,
    0.02949,
    0.01537,
    -0.00287,
    -0.048,
    0.03015,
    0.03169,
    -0.0378,
    -0.00414,
    0.0117,
    0.00602,
    -0.03053,
    -0.01242,
    0.00914,
    -0.03416,
    0.02583,
    -0.03039,
    -0.0004,
    -0.00551,
    -0.02373,
    0.01079,
    -0.01905,
    0.02722,
    -0.02653,
    0.01869,
    0.00861,
    -0.03004,
    0.0124,
    0.03063,
    -0.02041,
    -0.00175,
    -0.03716,
    0.00506,
    -0.01901,
    0.00756,
    -0.00416,
    -0.00189,
    -0.00519,
    0.01153,
    0.02668,
    0.01933,
    -0.00744,
    0.02365,
    0.00881,
    -0.0162,
    0.00974,
    0.00926,
    -0.00405,
    -0.01348,
    0.02142,
    -0.00693,
    0.00147,
    -0.01176,
    -0.02566,
    -0.01466,
    -0.05494,
    -0.00567,
    0.00699,
    0.02744,
    -0.02647,
    -0.00341,
    -0.04901,
    -0.00962,
    -0.01631,
    0.01955,
    -0.00435,
    -0.03604,
    -0.00754,
    -0.03526,
    0.01806,
    -0.01883,
    0.00212,
    0.04017,
    -0.0051,
    0.03707,
    -0.05796,
    -0.04627,
    0.03886,
    0.0256,
    0.05215,
    0.01318,
    -0.00108,
    0.03393,
    0.00928,
    -0.01028,
    0.01252,
    0.02621,
    0.02138,
    0.00926,
    0.03075,
    -0.00676,
    0.00356,
    0.00024,
    -0.00981,
    -0.01645,
    -0.02335,
    -0.00766,
    0.02361,
    0.01259,
    -0.00533,
    0.04422,
    -0.02617,
    -0.0159,
    -0.01048,
    0.05266,
    -0.0041,
    -0.0205,
    0.01417,
    0.02143,
    -0.01916,
    -0.01873,
    -0.03326,
    0.0151,
    0.02286,
    0.03138,
    0.01043,
    0.03073,
    -0.05067,
    0.05698,
    0.05091,
    0.01767,
    0.02097,
    -0.00526,
    -0.00118,
    0.00667,
    0.01117,
    0.01786,
    0.04046,
    0.03444,
    0.00468,
    -0.02309,
    -0.03371,
    -0.00648,
    -0.03965,
    -0.00395,
    0.0122,
    -0.03331,
    0.00935,
    0.02906,
    0.01304,
    -0.04222,
    0.00673,
    -0.0085,
    0.00672,
    0.04309,
    -0.01994,
    -0.04597,
    0.00124,
    -0.02566,
    -0.03194,
    -0.04635,
    -0.03636,
    0.04834,
    -0.02091,
    0.01068,
    0.01928,
    0.00116,
    -0.01439,
    0.00388,
    -0.02389,
    -0.01528,
    -0.00741,
    -0.05173,
    -0.01159,
    0.00519,
    0.0101,
    0.01324,
    0.00544,
    0.01028,
    0.00395,
    0.03772,
    -0.02616,
    0.03273,
    -0.01355,
    0.00436,
    0.0038,
    0.02795,
    -0.01356,
    -0.03778,
    -0.04753,
    -0.00733,
    -0.00499,
    0.0101,
    0.01356,
    -0.00602,
    0.03589,
    0.01883,
    0.00289,
    -0.00666,
    -0.02324,
    0.0153,
    0.02603,
    0.00086,
    -0.0111,
    0.02344,
    0.01472,
    0.0156,
    -0.01613,
    0.0217,
    -0.04521,
    0.04992,
    0.0011,
    0.02367,
    -0.03795,
    0.00354,
    0.01788,
    -0.00445,
    -0.02718,
    0.03798,
    0.00577,
    0.01622,
    -0.06881,
    0.00792,
    -0.02276,
    0.01279,
    -0.00946,
    0.02791,
    0.01484,
    -0.00365,
    -0.0014,
    -0.01736,
    -0.03514,
    0.01086,
    0.0136,
    -0.00164,
    0.00115,
    -0.05072,
    -0.02482,
    0.0412,
    0.01271,
    -0.00115,
    -0.01502,
    0.00127,
    0.00564,
    -0.03634,
    0.01269,
    -0.05065,
    -0.00106,
    2e-05,
    0.01804,
    -0.01121,
    -0.03332,
    -0.03356,
    0.02429,
    -0.05162,
    -0.00756,
    0.03092,
    -0.02699,
    -0.022,
    -0.04709,
    0.00745,
    0.01257,
    -0.00194,
    -0.03707,
    -0.03052,
    -0.01557,
    -0.00985,
    -0.05602,
    0.00032,
    -0.02127,
    0.01361,
    0.00238,
    0.00014,
    -0.00444,
    0.04353,
    0.02942,
    -0.00625,
    -0.01043,
    0.02691,
    -0.0044,
    0.00659,
    0.01166,
    -0.04628,
    -0.012,
    0.01106,
    -0.00824,
    0.00203,
    0.04188,
    -0.0221,
    -0.00654,
    -0.02903,
    -0.02272,
    0.0155,
    0.04861,
    -0.03682,
    -0.01361,
    0.00516,
    -0.01605,
    -0.01433,
    -0.0081,
    -0.01836,
    -0.00698,
    -0.03608,
    0.00686,
    -0.03672,
    -0.00907,
    -0.07269,
    0.02901,
    0.03879,
    -0.01258,
    -0.02314,
    0.01301,
    0.00262,
    -0.0214,
    0.02555,
    0.0452,
    -0.04282,
    0.00684,
    -0.03148,
    0.01208,
    -0.00585,
    -0.0048,
    -0.03233,
    0.00483,
    -5e-05,
    0.03852,
    0.04384,
    0.03043,
    0.00769,
    -0.04181,
    0.00255,
    -0.02055,
    -0.01332,
    0.01408,
    -0.02836,
    -0.03751,
    -0.01887,
    0.02853,
    0.00393,
    -0.04159,
    0.0117,
    0.02939,
    -0.0532,
    0.0191,
    -0.01984,
    -0.00458,
    0.03515,
    0.04662,
    0.0129,
    -0.03926,
    0.00359,
    -0.00528,
    0.01809,
    -0.019,
    0.01138,
    -0.03328,
    0.01778,
    -0.04222,
    -0.02678,
    -0.00856,
    -0.01477,
    0.01768,
    0.01336,
    -0.02532,
    -0.02945,
    -0.04848,
    0.01276,
    -0.02595,
    0.04629,
    0.00593,
    0.02318,
    -0.04575,
    0.03766,
    0.03387,
    0.00838,
    -0.03677,
    0.00803,
    0.03419,
    -0.02264,
    -0.01109,
    0.00989,
    0.00763,
    -0.00393,
    0.02291,
    0.00086,
    0.00446,
    -0.01964,
    0.01378,
    0.00375,
    0.02051,
    0.02388,
    -0.00323,
    0.0134,
    0.00645,
    -0.0161,
    -0.04366,
    -0.00474,
    0.00674,
    -0.04437,
    -0.02008,
    0.04725,
    -0.02799,
    0.02061,
    0.02141,
    -0.02206,
    -0.02076,
    -0.01327,
    -0.03239,
    -0.01539,
    0.01016,
    -0.00547,
    -0.00027,
    -0.01777,
    0.00024,
    -0.00225,
    -0.02753,
    0.01722,
    -0.05542,
    0.04036,
    -0.00364,
    0.04563,
    0.00397,
    0.02765,
    0.0128,
    0.00627,
    -0.00247,
    -0.02429,
    0.04956,
    0.00123,
    -0.0179,
    -0.02858,
    0.06131,
    -0.03593,
    -0.02921,
    -0.06202,
    -0.02426,
    0.05864,
    -0.05218,
    0.006,
    0.04492,
    -0.01189,
    -0.00214,
    0.01054,
    -0.03791,
    -0.02141,
    -0.00487,
    0.00843,
    0.02486,
    0.00855,
    -0.00241,
    -0.00233,
    -0.01038,
    0.00477,
    0.0045,
    -0.00866,
    0.01274,
    0.04754,
    -0.01677,
    -0.01385,
    -0.02912,
    0.01882,
    0.02104,
    -0.00158,
    0.00174,
    0.00626,
    -0.01508,
    0.01164,
    -0.01497,
    -0.02824,
    0.03029,
    0.0134,
    -0.02557,
    -0.00906,
    -0.04316,
    -0.00631,
    -0.01883,
    -0.00573,
    -0.02332,
    0.01971,
    -0.03294,
    -0.02506,
    0.02354,
    0.00577,
    -0.01653,
    -0.00656,
    -0.04381,
    -0.00533,
    0.03115,
    0.05894,
    0.01688,
    -0.00936,
    -0.00728,
    -0.03461,
    -0.01765,
    -0.01608,
    0.03274,
    -0.06954,
    -0.02126,
    -0.03156,
    0.02872,
    -0.02762,
    -0.06746,
    -0.02449,
    0.02971,
    -0.02092,
    -0.01806,
    0.03565,
    0.01169,
    -0.00449,
    -0.01602,
    0.04013,
    -0.02415,
    -0.01732,
    0.00867,
    -0.03424,
    0.03005,
    0.00373,
    -0.00733,
    0.03462,
    -0.00191,
    -0.01001,
    -0.02436,
    -0.00306,
    0.0166,
    -0.01584,
    0.00337,
    0.01963,
    -0.04119,
    -0.04584,
    -0.05139,
    0.01992,
    0.00529,
    0.02708,
    -0.00334,
    -0.00169,
    -0.01075,
    0.01168,
    0.01423,
    0.00609,
    0.0267,
    -0.0158,
    -0.00235,
    0.03906,
    0.03043,
    -0.04425,
    -0.01257,
    0.03081,
    -0.01787,
    -0.00141,
    -0.02592,
    -0.03411,
    0.00058,
    0.01414,
    0.01164,
    0.01015,
    0.00043,
    -0.00735,
    -0.0101,
    0.00215,
    0.02747,
    0.01271,
    0.00739,
    0.02742,
    0.03153,
    0.01449,
    0.02854,
    0.00646,
    -0.02061,
    -0.02168,
    0.02477,
    -0.00264,
    0.02474,
    -0.01665,
    0.03054,
    -0.01255,
    -0.06276,
    0.01869,
    0.04884,
    0.00016,
    0.00411,
    0.0258,
    -0.01714,
    0.02071,
    0.01587,
    0.01314,
    -0.22021,
    0.03525,
    0.01691,
    -0.03494,
    -0.03046,
    0.02801,
    -0.04323,
    0.03224,
    0.01347,
    -0.00478,
    0.01978,
    0.01073,
    0.00908,
    0.00203,
    0.01652,
    0.00124,
    0.006,
    -0.02668,
    -0.03259,
    0.01904,
    -0.00638,
    0.0168,
    0.0037,
    0.0117,
    0.015,
    -0.00272,
    0.04992,
    -0.01798,
    0.02325,
    0.00408,
    0.00469,
    0.01876,
    -0.01666,
    0.00384,
    -0.01625,
    0.0239,
    0.02098,
    -0.0124,
    -0.00554,
    0.0059,
    -0.02965,
    0.04061,
    -0.01415,
    -0.01334,
    0.01958,
    0.01715,
    0.02785,
    -0.0067,
    -0.02784,
    -0.0292,
    0.01928,
    -0.01825,
    0.01932,
    -0.00989,
    0.00128,
    0.03998,
    -0.00414,
    0.03729,
    -0.0212,
    -0.0106,
    -0.00512,
    -0.01289,
    -0.014,
    0.01437,
    0.02343,
    0.00644,
    0.00567,
    -0.0396,
    0.03769,
    -0.01852,
    -0.00624,
    0.03216,
    -0.025,
    0.02181,
    -0.03694,
    -0.01225,
    0.00328,
    0.00356,
    -0.02275,
    -0.00148,
    0.00847,
    -0.02601,
    0.0378,
    -0.00696,
    0.0174,
    0.00825,
    0.00687,
    -0.01408,
    -0.04021,
    -0.04521,
    -0.04669,
    0.03823,
    -0.01729,
    -0.03644,
    -0.00565,
    -0.00778,
    -0.03291,
    0.00836,
    -0.02046,
    -0.03244,
    -0.0408,
    0.03352,
    -0.01158,
    0.00577,
    0.01599,
    0.00542,
    -0.03236,
    0.00271,
    -0.00133,
    -0.02748,
    -0.01201,
    0.0365,
    -0.01131,
    0.02666,
    0.06298
   ],
   "MYSTIC": [
    -0.0414,
    0.02289,
    -0.01182,
    0.02109,
    0.02261,
    0.02844,
    -0.03621,
    -0.02579,
    0.02428,
    0.01166,
    0.01313,
    -0.00429,
    0.02438,
    0.01001,
    0.03474,
    -0.02857,
    0.00425,
    -0.0104,
    0.01836,
    -0.05059,
    -0.01352,
    0.01212,
    0.00367,
    -0.04111,
    0.00054,
    0.00364,
    0.0371,
    0.01706,
    -0.04131,
    0.04052,
    0.0169,
    -0.00169,
    0.05059,
    -0.00424,
    0.02391,
    -0.03937,
    -0.01407,
    0.04532,
    -0.00378,
    -0.01325,
    0.00077,
    -0.01279,
    -0.00249,
    0.02393,
    -0.02026,
    -0.00375,
    -0.00785,
    0.01102,
    -0.01361,
    -0.04884,
    0.02076,
    0.03145,
    -0.013,
    -0.02212,
    0.01078,
    0.01519,
    0.0268,
    0.03067,
    -0.00717,
    0.00896,
    0.02956,
    0.01671,
    -0.00489,
    -0.00816,
    -0.01453,
    -0.05049,
    0.01122,
    0.0122,
    -0.01248,
    -0.00548,
    -0.01876,
    0.00841,
    0.01481,
    0.03074,
    0.02083,
    -0.01324,
    0.02057,
    0.03712,
    -0.01277,
    0.00937,
    0.03699,
    0.03887,
    0.0386,
    -0.02413,
    0.00794,
    0.01808,
    -0.02192,
    0.0104,
    0.03128,
    0.00745,
    0.02076,
    0.05375,
    -0.0432,
    -0.01651,
    0.01821,
    0.00054,
    0.00883,
    0.06621,
    -0.02048,
    0.00316,
    0.01986,
    0.0135,
    -0.02282,
    -0.01811,
    -0.02294,
    -0.03624,
    0.06527,
    -0.01618,
    -0.00024,
    0.02708,
    0.02095,
    0.01706,
    -0.00245,
    -0.00473,
    -0.03039,
    0.00699,
    -0.03784,
    0.02795,
    -0.01504,
    0.04237,
    0.00873,
    -0.02301,
    0.03585,
    0.03252,
    0.02355,
    -0.0065,
    -0.02009,
    0.02969,
    -0.04865,
    0.02402,
    0.03065,
    0.01757,
    -0.01275,
    -0.0326,
    -0.01185,
    -0.01395,
    0.00752,
    -0.00831,
    0.00323,
    -0.02132,
    0.04868,
    -0.06169,
    0.02511,
    -0.02146,
    -0.03744,
    -0.03016,
    0.01851,
    -0.03979,
    0.01403,
    0.02217,
    -0.01132,
    -0.00152,
    -0.05343,
    0.02133,
    -0.01976,
    -0.02429,
    -0.01016,
    -0.0129,
    0.03254,
    -0.01019,
    -0.00218,
    -0.03356,
    0.00364,
    0.06162,
    0.03676,
    0.02336,
    0.01806,
    0.02593,
    0.01672,
    -0.00391,
    -0.04352,
    0.01609,
    0.02137,
    0.00209,
    -0.04721,
    0.03267,
    -0.0179,
    -0.00521,
    0.00187,
    -0.00024,
    0.03194,
    0.02874,
    -0.03963,
    0.02385,
    0.0025,
    0.02499,
    0.00403,
    0.04883,
    0.0281,
    0.01479,
    -0.03981,
    0.05271,
    -0.01106,
    0.01018,
    0.03914,
    0.0239,
    0.04797,
    -0.04589,
    -0.01744,
    -0.01601,
    0.00029,
    -0.01802,
    -0.02606,
    -0.0148,
    0.01013,
    0.00902,
    -0.03682,
    -0.02955,
    0.00488,
    0.00678,
    0.02522,
    -0.01554,
    0.01376,
    -0.02273,
    0.01332,
    0.02797,
    0.0041,
    -0.01022,
    -0.03641,
    0.0074,
    -0.00358,
    -0.02915,
    -0.04185,
    0.02496,
    -0.01656,
    0.00344,
    -0.02138,
    0.02502,
    0.00499,
    -0.00967,
    0.00082,
    -0.00372,
    -0.0272,
    -0.05299,
    0.00943,
    0.0154,
    -0.00286,
    -0.03629,
    -0.03157,
    0.02472,
    0.02119,
    -0.03078,
    0.03959,
    -0.03671,
    0.06494,
    -0.01631,
    -0.01606,
    0.02613,
    -0.03485,
    -0.00043,
    -0.05534,
    -0.01744,
    0.01578,
    -0.00927,
    0.04157,
    0.04071,
    -0.01145,
    0.011,
    0.02898,
    -0.00488,
    -0.03463,
    0.00712,
    -0.00844,
    -0.01188,
    -0.01407,
    0.00669,
    -0.01657,
    0.00685,
    0.04811,
    -0.03094,
    0.0256,
    -0.03087,
    0.00723,
    0.04263,
    0.01166,
    -0.0006,
    0.03468,
    -0.00529,
    0.06162,
    -0.00743,
    0.02355,
    -0.01579,
    0.00397,
    0.00913,
    -0.00185,
    0.03949,
    -0.03146,
    0.01745,
    0.02042,
    0.01929,
    0.01971,
    -0.04022,
    -0.01221,
    -0.03313,
    0.01974,
    -0.00845,
    -0.01317,
    0.00759,
    0.00436,
    -0.02705,
    -0.01007,
    -0.0318,
    0.00992,
    -0.00027,
    0.00228,
    0.04198,
    0.00692,
    0.05547,
    0.00267,
    0.02036,
    -0.00974,
    -0.01829,
    -0.03179,
    0.006,
    0.00575,
    -0.01601,
    0.00926,
    0.01934,
    -0.01955,
    -0.02772,
    0.03419,
    0.04134,
    -0.00556,
    0.01204,
    0.00918,
    0.03097,
    0.01908,
    0.01511,
    0.02023,
    -0.00452,
    0.0209,
    0.01398,
    0.02172,
    -0.01681,
    0.01775,
    0.02356,
    -0.01197,
    -0.0175,
    0.02042,
    0.05334,
    -0.01885,
    0.02723,
    0.05146,
    -0.00361,
    0.01725,
    -0.01035,
    0.00198,
    -0.03482,
    -0.0403,
    0.0057,
    0.01548,
    0.01945,
    0.02997,
    0.00161,
    -0.0511,
    -0.01872,
    -0.08039,
    -0.00564,
    -0.02682,
    0.02412,
    -0.01612,
    -0.03909,
    -0.00253,
    -0.02067,
    -0.01649,
    0.01336,
    -0.00278,
    -0.00286,
    0.01108,
    0.00105,
    0.02177,
    0.0395,
    -0.01108,
    -0.01737,
    -0.01374,
    -0.03214,
    -0.01616,
    -0.00998,
    0.00786,
    -0.03044,
    0.01905,
    0.00829,
    0.00605,
    -0.00928,
    0.01872,
    -0.01797,
    0.00699,
    0.00796,
    0.02557,
    -0.02023,
    -0.02456,
    0.00654,
    -0.01342,
    0.03,
    0.04592,
    -0.05036,
    0.05794,
    0.03336,
    0.00548,
    -0.00853,
    -0.00251,
    -0.03357,
    -0.00414,
    0.02245,
    -0.02731,
    -0.02532,
    0.01383,
    -0.01183,
    0.02669,
    0.04326,
    -0.03014,
    -0.01523,
    -0.00049,
    0.03382,
    0.07261,
    -0.01113,
    0.02775,
    -0.00979,
    0.00561,
    -0.0246,
    0.0014,
    0.03178,
    0.00519,
    -0.02264,
    -0.01225,
    -0.00802,
    -0.05233,
    -0.01985,
    -0.00462,
    -0.01775,
    0.0144,
    0.02889,
    0.01333,
    0.03698,
    0.01356,
    0.02139,
    0.00416,
    0.03013,
    0.03594,
    0.01884,
    -0.00136,
    -0.017,
    -0.0091,
    -0.03678,
    0.00714,
    -0.01106,
    0.01382,
    -0.00793,
    0.02953,
    0.02986,
    -0.03226,
    0.00213,
    0.0002,
    -0.02118,
    0.01977,
    -0.01642,
    -0.04987,
    -0.0083,
    0.0195,
    -0.00506,
    -0.00671,
    0.00454,
    -0.01192,
    0.03307,
    -0.01178,
    0.00778,
    0.04777,
    -0.03768,
    0.03226,
    -0.02166,
    -0.01224,
    0.00969,
    -0.03688,
    0.01396,
    0.04494,
    -0.01504,
    0.00354,
    0.01343,
    -0.01689,
    -0.02918,
    -0.01978,
    -0.02034,
    -0.01745,
    0.02322,
    0.03532,
    -0.01555,
    0.02597,
    -0.00279,
    0.01233,
    0.02172,
    0.02102,
    0.00949,
    -0.03367,
    -0.02335,
    0.00107,
    -0.00962,
    -0.00198,
    0.02137,
    -0.03283,
    0.0217,
    -0.04138,
    -0.0229,
    0.01773,
    0.04739,
    -0.01416,
    0.00848,
    -0.00703,
    0.0573,
    -0.02346,
    0.03079,
    -0.02449,
    -0.03219,
    -0.03821,
    0.00802,
    0.01213,
    -0.00431,
    0.00392,
    0.03929,
    0.03778,
    -0.02776,
    -0.04206,
    0.01764,
    0.0086,
    0.00835,
    -0.01301,
    -0.04065,
    0.01532,
    -0.02374,
    -0.01515,
    -0.03851,
    0.0002,
    0.01427,
    0.04471,
    -0.0166,
    -0.00788,
    -0.0097,
    -0.00118,
    0.02209,
    -0.00178,
    -0.03711,
    -0.05877,
    0.00028,
    0.01967,
    0.00843,
    -0.04281,
    0.03323,
    0.04181,
    0.03586,
    0.07592,
    -0.01511,
    -0.01652,
    0.02742,
    -0.02374,
    -0.02539,
    0.02441,
    0.01626,
    0.01909,
    0.04234,
    0.02513,
    0.02526,
    -0.01028,
    -0.01825,
    0.02584,
    0.02293,
    0.00971,
    0.03291,
    -0.01647,
    0.0301,
    0.02759,
    0.05215,
    0.00487,
    0.01301,
    -0.02794,
    0.01915,
    -0.00155,
    -0.01753,
    -0.02503,
    -0.02479,
    -0.06992,
    0.01627,
    -0.08021,
    -0.05529,
    0.03976,
    0.00077,
    -0.01701,
    -0.01551,
    -0.06197,
    -0.02226,
    -0.02333,
    -0.00751,
    0.0083,
    -0.02304,
    0.00335,
    0.0139,
    -0.02975,
    0.02111,
    -0.03897,
    -0.02281,
    0.02394,
    -0.00166,
    -0.01819,
    -0.00478,
    0.0228,
    0.08415,
    0.01014,
    -0.017,
    0.01096,
    -0.02244,
    -0.01182,
    -0.01299,
    0.00853,
    -0.01517,
    0.02071,
    0.05177,
    -0.00454,
    -0.01223,
    0.04195,
    0.00669,
    -0.00958,
    0.0157,
    -0.01966,
    0.04037,
    0.01587,
    -0.00743,
    -0.02446,
    -0.00939,
    -0.00857,
    -0.03492,
    0.00237,
    -0.02649,
    -0.00416,
    0.02811,
    0.01258,
    -0.02151,
    0.06059,
    -0.02074,
    0.00314,
    0.04093,
    0.00123,
    -0.01766,
    -0.00176,
    0.00143,
    0.02597,
    0.00968,
    -0.02872,
    -0.00281,
    -0.01279,
    -0.03217,
    -0.04071,
    0.01156,
    -0.02293,
    -0.04067,
    -0.00156,
    -0.01391,
    -0.00417,
    -0.02297,
    -0.00556,
    -0.01174,
    0.02374,
    -0.0305,
    -0.0031,
    -0.00161,
    -0.01569,
    -0.00352,
    -0.01445,
    -0.00626,
    0.0178,
    0.00497,
    0.04098,
    0.00277,
    -0.01669,
    0.01453,
    -0.01738,
    0.01789,
    0.00303,
    -0.01674,
    -0.03781,
    0.0114,
    0.04344,
    0.00406,
    -0.07223,
    0.01097,
    0.01732,
    0.00102,
    -0.05244,
    0.00191,
    0.00805,
    0.00519,
    0.00209,
    -0.02141,
    -0.00156,
    0.00144,
    0.04069,
    0.04265,
    0.02053,
    -0.0045,
    0.01722,
    -0.01151,
    0.00414,
    0.01482,
    0.02369,
    0.00452,
    0.06372,
    0.02822,
    -0.01302,
    0.0074,
    -0.00266,
    -0.03562,
    -0.003,
    -0.01697,
    -0.04426,
    -0.03214,
    0.00161,
    -0.00487,
    -0.02925,
    0.01827,
    0.04014,
    -0.03589,
    -0.00999,
    -0.04711,
    -0.02764,
    -0.00992,
    0.05924,
    -6e-05,
    -0.0166,
    -0.02911,
    0.03524,
    -0.02156,
    0.00737,
    -0.00463,
    0.00385,
    0.011,
    0.01725,
    -0.0303,
    -0.00795,
    0.03437,
    -0.02454,
    -0.00957,
    0.01224,
    -0.00172,
    0.02499,
    0.01259,
    0.00598,
    -0.00515,
    -0.01409,
    0.01351,
    -0.00508,
    0.02108,
    0.0118,
    0.01211,
    0.0266,
    0.0134,
    0.01057,
    -0.02939,
    -0.00998,
    0.04854,
    -0.01407,
    -0.0171,
    0.07117,
    0.00529,
    -0.03574,
    0.02093,
    0.01631,
    0.00643,
    -0.0173,
    -0.00306,
    -0.00629,
    0.01355,
    0.00283,
    0.00193,
    -0.01422,
    0.00593,
    0.00979,
    0.00728,
    0.01196,
    0.02953,
    -0.02434,
    -0.0222,
    -0.03468,
    -0.00364,
    0.03451,
    -0.00362,
    0.01577,
    0.02617,
    0.05943,
    0.03982,
    -0.0445,
    -0.00739,
    0.04013,
    -0.02257,
    0.00097,
    0.03002,
    0.02628,
    0.0467,
    -0.04302,
    0.02598,
    -0.03517,
    0.0089,
    0.01852,
    0.00053,
    -0.01861,
    0.06506,
    0.00488,
    0.00667,
    0.02498,
    -0.03068,
    -0.01173,
    0.04276,
    -0.02096,
    0.02299,
    -0.01433,
    0.0045,
    0.01965,
    -0.05302,
    -0.00104,
    0.02493,
    -0.01306,
    -0.00471,
    0.0062,
    -0.00844,
    0.01723,
    -0.00439,
    0.00474,
    -0.02175,
    -0.03653,
    0.02573,
    -0.0132,
    0.00591,
    0.02595,
    -0.02055,
    -0.01056,
    -0.01735,
    -0.00107,
    -0.00963,
    -0.0147,
    -0.04428,
    0.0085,
    -0.03259,
    0.03731,
    0.02162,
    -0.00918,
    -0.00838,
    0.04281,
    -0.0058,
    -0.00763,
    0.00176,
    -0.00091,
    -0.02735,
    -0.01205,
    -0.05796,
    0.0162,
    0.00762,
    -0.01583,
    -0.0064,
    -0.01645,
    0.01466,
    0.00353,
    0.00861,
    0.00628,
    0.01709,
    0.01232,
    0.01126,
    0.02802,
    -0.01681,
    -0.04338,
    -0.0592,
    0.00605,
    -0.03895,
    -0.03013,
    -0.02532,
    0.00025,
    -0.00389,
    0.03764,
    -0.01239,
    -0.01266,
    0.03612,
    0.00443,
    -0.00613,
    -0.05771,
    0.00891,
    -0.03439,
    -0.00268,
    -0.03812,
    -0.05681,
    0.02581,
    0.01179,
    0.0111,
    -0.02008,
    -0.02669,
    0.01905,
    0.02286,
    -0.01324,
    0.00719,
    0.02075,
    -0.08893,
    0.02702,
    0.03266,
    0.02524,
    0.00208,
    0.01032,
    -0.02151,
    0.06875,
    0.02623,
    0.03047,
    0.02102,
    -0.02956,
    0.00455,
    0.00295,
    -0.01619,
    -0.0226,
    0.01283,
    -0.01817,
    -0.00858,
    -0.07138,
    -0.01955,
    0.01097,
    -0.02994,
    -0.00861,
    -0.00168,
    0.01128,
    0.00765,
    -0.00569,
    -0.02826,
    0.00338,
    0.04255,
    0.00161,
    0.01591,
    0.02993,
    -0.00494,
    0.03308,
    0.00931,
    -0.01413,
    0.03765,
    0.00012,
    0.02598,
    0.00177,
    -0.04033,
    0.0121,
    -0.00778,
    0.00295,
    0.00059,
    0.00642,
    -0.0055,
    0.0078,
    -0.02543,
    -0.00202,
    -0.0145,
    0.01637,
    0.0157,
    0.04556,
    -0.01257,
    0.03075,
    -0.01862,
    0.02101,
    -0.00115,
    -0.01429,
    -0.0197,
    -0.00668,
    0.04217,
    -0.0035,
    -0.01529,
    -0.0067,
    0.03459,
    0.00888,
    0.00559,
    -0.05008,
    -0.00562,
    0.00108,
    0.01756,
    0.00387,
    -0.02438,
    -0.01036,
    -0.00934,
    -0.05209,
    -0.00589,
    -0.01909,
    0.03059,
    -0.00628,
    0.0143,
    -0.02841,
    -0.00922,
    -0.01834,
    0.01048,
    0.03481,
    -0.02125,
    0.05513,
    -0.03897,
    0.03321,
    -0.01099,
    -0.02445,
    -3e-05,
    -0.00436,
    -0.0331,
    -0.00249,
    0.00104,
    0.04957,
    -0.0136,
    -0.03933,
    -0.0005,
    -0.01347,
    -0.00236,
    -0.01627,
    -0.02685,
    -0.00397,
    -0.0076,
    -0.0026,
    -0.02339,
    -0.01275,
    -0.00968,
    -0.02684,
    -0.04118,
    0.03914,
    -0.023,
    0.00998,
    -0.01237,
    0.0007,
    0.03096,
    -0.00591,
    0.03194,
    -0.02956,
    -0.01021,
    0.01874,
    0.02788,
    -0.01522,
    0.05507,
    0.01811,
    0.0502,
    -0.01511,
    -0.04686,
    0.05425,
    -0.01869,
    0.03016,
    -0.00835,
    0.02328,
    -0.01403,
    0.02365,
    -0.06231,
    0.0025,
    0.02745,
    -0.03294,
    -0.01055,
    0.00715,
    -0.00163,
    0.02541,
    -0.04323,
    0.02411,
    0.04848,
    0.00279,
    0.01503,
    0.03024,
    0.00846,
    0.02229,
    0.00837,
    0.03077,
    0.0114,
    0.02382,
    0.0157,
    0.01371,
    0.00245,
    -0.03616,
    -0.048,
    0.03478,
    -0.01756,
    -0.02901,
    -0.00987,
    0.00391,
    -0.038,
    -0.00864,
    0.05586,
    0.02446,
    -0.00129,
    0.00251,
    -0.02658,
    0.01683,
    -0.01655,
    -0.02924,
    -0.01489,
    -0.01748,
    -0.00838,
    0.03363,
    -0.01857,
    0.02603,
    0.06196,
    0.02423,
    0.00555,
    0.01328,
    0.00364,
    0.00583,
    -0.01263,
    0.02622,
    -0.04816,
    0.00988,
    0.025,
    -0.00842,
    0.02174,
    0.02384,
    -0.01725,
    -0.00812,
    -0.04338,
    0.01377,
    -0.0259,
    0.03651,
    -0.04941,
    0.03201,
    -0.00173,
    -0.02452,
    -0.03932,
    0.03327,
    0.03406,
    0.00939,
    0.03095,
    -0.09632,
    -0.05552,
    0.01416,
    0.03465,
    0.00443,
    0.01712,
    -0.02763,
    0.01148,
    0.0055,
    -0.02143,
    0.01725,
    0.03905,
    -0.05413,
    -0.00356,
    0.0209,
    0.01953,
    0.00991,
    0.02807,
    0.00452,
    0.00966,
    0.03226,
    0.02537,
    0.05789,
    -0.01904,
    -0.0091,
    -0.02167,
    -0.00206,
    -0.00614,
    0.01879,
    -0.02554,
    -0.0532,
    -0.02436,
    0.0291,
    0.01095,
    0.00475,
    0.01549,
    -0.0037,
    -0.02093,
    0.04305,
    -0.0387,
    0.00419,
    0.0145,
    0.0138,
    -0.00506,
    0.03707,
    -0.01678,
    0.00209,
    0.02427,
    -0.0233,
    -0.02425,
    0.00856,
    -0.02949,
    -0.01412,
    0.01222,
    -0.0229,
    -0.00183,
    0.01254,
    -0.01676,
    -0.01571,
    0.02013,
    0.02266,
    0.02485,
    0.02836,
    0.01431,
    -0.01763,
    -0.03688,
    0.02846,
    -0.02078,
    -0.01608,
    0.02218,
    0.01421,
    -0.04544,
    0.01013,
    -0.03214,
    0.00212,
    -0.01372,
    -0.00205,
    -0.02079,
    0.00292,
    -0.00418,
    0.05293,
    0.03469,
    0.01405,
    -0.02786,
    -0.00734,
    -0.02207,
    0.00774,
    0.00323,
    -0.00401,
    -0.01737,
    -0.00203,
    0.01883,
    -0.00069,
    0.02819,
    -0.02393,
    0.03134,
    0.00532,
    0.00411,
    0.00763,
    0.00293,
    -0.00304,
    0.06312,
    0.03287,
    -0.04315,
    -0.04838,
    0.00774,
    0.03858,
    -0.00941,
    0.01983,
    0.01732,
    -0.02979,
    -0.06337,
    -0.02402,
    -0.00135,
    0.01119,
    -0.00465,
    0.0296,
    -0.00458,
    0.06123,
    -0.03473,
    -0.00449,
    -0.01021,
    0.01229,
    -0.00718,
    0.02649,
    -0.03475,
    0.01168,
    0.00838,
    0.01522,
    -0.00578,
    0.00685,
    0.01062,
    -0.01786,
    -0.02819,
    -0.01865,
    -0.00324,
    -0.0201,
    0.01348,
    0.00574,
    0.01473,
    -0.0048,
    0.00929,
    -0.04114,
    0.01103,
    -0.03879,
    -0.0158,
    0.00229,
    0.00891,
    -0.03778,
    -0.0059,
    0.02098,
    -0.00501,
    0.04599,
    -0.01814,
    -0.0315,
    0.0191,
    -0.02681,
    0.00343,
    -0.01253,
    -0.00468,
    -0.0437,
    -0.00462,
    -0.03078,
    -0.01894,
    0.00807,
    -0.01303,
    -0.0002,
    0.0107,
    -0.00238,
    -0.05323,
    -0.01205,
    0.0774,
    0.00657,
    0.0005,
    -0.03837,
    0.01672,
    -0.01132,
    0.0257,
    -0.00235,
    0.02008,
    0.01288,
    0.04608,
    -0.00057,
    0.03311,
    -0.00553,
    -0.03858,
    -0.00306,
    0.00472,
    0.01004,
    0.00175,
    0.00492,
    0.00549,
    -0.01423,
    -0.00893,
    -0.0034,
    0.00464,
    0.01718,
    -0.02451,
    0.01501,
    0.02082,
    -0.04498,
    0.04274,
    0.03512,
    0.00748,
    -0.02303,
    -0.00398,
    0.00699,
    -0.00307,
    0.00907,
    0.00484,
    0.00123,
    0.0108,
    -0.01895,
    0.01113,
    0.01989,
    -0.03503,
    -0.02408,
    0.00588,
    0.04487,
    0.01458,
    0.04221,
    0.00675,
    -0.0196,
    -0.01935,
    -0.0081,
    -0.0177,
    0.02515,
    -0.0072,
    -0.03106,
    0.01185,
    -0.01845,
    0.00251,
    -0.05019,
    0.02899,
    0.01985,
    -0.01228,
    -0.02001,
    -0.02139,
    -0.02245,
    0.00496,
    -0.00467,
    0.02487,
    -0.02204,
    -0.00584,
    0.00441,
    0.00385,
    -0.01912,
    0.01521,
    0.02338,
    -0.03474,
    0.02813,
    -0.00597,
    -0.01004,
    -0.0094,
    0.00234,
    0.01308,
    -0.04544,
    0.00993,
    -0.00989,
    -0.00337,
    0.04387,
    0.00917,
    -0.05508,
    0.00833,
    -0.0178,
    -0.05171,
    0.03602,
    -0.01227,
    0.02884,
    0.03137,
    -0.04328,
    0.05438,
    -0.00496,
    -0.00334,
    0.01233,
    -0.01558,
    0.01092,
    0.01921,
    -0.02891,
    0.02515,
    0.009,
    -0.02924,
    -0.02899,
    0.01482,
    0.02064,
    -0.00835,
    -0.00945,
    0.00197,
    -0.01295,
    -0.01579,
    -0.0106,
    0.00339,
    -0.02467,
    0.00797,
    -0.01317,
    0.03274,
    -0.04955,
    0.04546,
    0.00975,
    0.02289,
    -0.0088,
    -0.03472,
    0.00578,
    0.01114,
    0.05264,
    0.00765,
    -0.02343,
    0.01936,
    0.02045,
    -0.01555,
    0.02672,
    0.0029,
    0.03027,
    -0.06447,
    -0.00563,
    0.03507,
    0.00668,
    -0.02519,
    0.00401,
    0.02272,
    0.00383,
    -0.0318,
    -0.00545,
    0.07339,
    0.02532,
    -0.02672,
    0.00383,
    0.01322,
    0.01782,
    0.03054,
    -0.02256,
    0.00384,
    -0.01114,
    -0.03703,
    -0.00458,
    -0.00996,
    0.00346,
    0.01846,
    -0.0161,
    -0.05637,
    0.0039,
    -0.0262,
    0.00713,
    0.02607,
    0.04722,
    -0.01497,
    -0.04332,
    -0.02389,
    0.00998,
    0.0338,
    0.03282,
    -0.03704,
    0.01374,
    0.05822,
    -0.07109,
    0.01083,
    -0.00974,
    -0.01255,
    0.01853,
    -0.01049,
    0.0018,
    0.05298,
    -0.01695,
    0.01416,
    0.0409,
    -0.01914,
    -0.02999,
    0.01204,
    0.03709,
    -0.01342,
    -0.02502,
    -0.00284,
    -0.01183,
    -0.0313,
    0.03516,
    0.02895,
    -0.0074,
    -0.01902,
    -0.0208,
    -0.00602,
    0.00653,
    0.00973,
    0.00558,
    -0.00533,
    0.02032,
    -0.01721,
    -0.02741,
    0.00787,
    -0.0367,
    0.01521,
    -0.00775,
    0.00892,
    -0.00634,
    -0.00826,
    0.01103,
    0.00091,
    -0.00263,
    -0.04269,
    -0.01717,
    0.00107,
    0.00564,
    0.00375,
    -0.0246,
    0.0303,
    -0.00057,
    0.01281,
    0.03439,
    -0.02151,
    -0.00865,
    0.0069,
    0.00709,
    0.01355,
    -0.02513,
    -0.02068,
    -0.03013,
    0.07097,
    0.00992,
    -0.02281,
    -0.00204,
    0.01695,
    0.04288,
    -0.02684,
    -0.0249,
    0.06239,
    0.00106
   ]
  },
  "inst20": {
   "VOID": [
    0.00467,
    -0.0091,
    0.01897,
    0.02628,
    0.00016,
    0.01835,
    0.00901,
    0.0024,
    0.01235,
    0.03919,
    -0.01351,
    -0.02492,
    -0.00052,
    -0.01773,
    -0.00585,
    -0.00845,
    0.04109,
    0.02116,
    -0.01343,
    -0.02111,
    -0.00341,
    0.0217,
    -0.00369,
    -0.00408,
    0.00742,
    -0.00255,
    -0.01087,
    -0.01383,
    0.07679,
    -0.03279,
    0.02773,
    0.04085,
    -0.02836,
    0.03015,
    -0.00845,
    0.00085,
    -0.01252,
    0.04882,
    -0.00181,
    -0.01546,
    0.00107,
    -0.01878,
    0.02566,
    0.00788,
    -0.02478,
    0.03549,
    0.02811,
    -0.04018,
    -0.02528,
    -0.01339,
    0.03074,
    0.01079,
    0.00808,
    -0.05433,
    0.04803,
    0.0434,
    0.03617,
    -0.0014,
    -0.03716,
    -0.01275,
    0.00745,
    -0.02757,
    0.03101,
    0.01522,
    0.02638,
    -0.00153,
    0.00949,
    -0.0018,
    -0.01466,
    0.01609,
    0.01601,
    -0.00523,
    -0.00767,
    0.00296,
    -0.01204,
    0.00222,
    0.0365,
    0.00849,
    -0.03849,
    -0.01053,
    0.02376,
    -0.00279,
    0.00548,
    0.00242,
    0.01864,
    -0.01972,
    -0.01165,
    -0.02202,
    -0.0089,
    0.01569,
    -0.03777,
    -0.02202,
    0.00789,
    -0.01551,
    0.01701,
    0.02864,
    -0.00823,
    -0.00485,
    0.01259,
    -0.03523,
    -0.00412,
    -0.04418,
    -0.00648,
    -0.02381,
    0.02297,
    -0.01543,
    0.01873,
    -0.0095,
    0.00531,
    -0.01733,
    0.00066,
    0.0086,
    -0.01058,
    -0.04753,
    0.02538,
    0.00016,
    -0.00826,
    0.01141,
    0.01461,
    0.0042,
    -0.00492,
    0.00889,
    0.02276,
    0.0264,
    0.00075,
    0.02652,
    -0.00126,
    -0.01295,
    0.02169,
    0.00962,
    -0.03643,
    0.03425,
    -0.02459,
    0.02053,
    -0.03492,
    -0.01726,
    0.00604,
    0.00611,
    0.00661,
    -0.00432,
    -0.00804,
    0.01238,
    -0.01119,
    -0.00338,
    0.00431,
    0.0476,
    -0.01686,
    0.01155,
    0.00735,
    -0.03854,
    0.01151,
    -0.00868,
    -0.03675,
    -0.0093,
    0.04089,
    0.01233,
    0.0299,
    -0.00596,
    -0.03459,
    -0.01025,
    0.02645,
    0.00357,
    -0.00345,
    0.01974,
    -0.01696,
    -0.00735,
    0.0481,
    0.00925,
    0.03542,
    0.0072,
    0.00338,
    -0.01765,
    0.02587,
    -0.01533,
    0.00324,
    0.00674,
    7e-05,
    -0.00125,
    -0.01778,
    -0.00576,
    0.0223,
    0.02007,
    0.0328,
    -0.05747,
    -0.01051,
    0.04122,
    0.00913,
    -0.03674,
    -0.01735,
    0.00824,
    -0.00803,
    -0.00903,
    -0.01753,
    -0.00867,
    -0.02571,
    0.02306,
    0.04081,
    0.00391,
    0.01395,
    -0.0111,
    0.02083,
    0.02023,
    -0.00537,
    0.02786,
    0.03903,
    -0.01183,
    -0.035,
    0.01489,
    -0.00227,
    -0.02482,
    -0.00354,
    -0.02238,
    0.02415,
    -0.03143,
    -0.00803,
    -0.02849,
    -0.02728,
    -0.00516,
    0.01961,
    0.0208,
    -0.02669,
    0.00293,
    0.06624,
    0.01205,
    -0.02952,
    -0.0176,
    -0.01043,
    -0.01317,
    0.00256,
    -0.03574,
    0.05132,
    0.01277,
    0.00482,
    0.04891,
    -0.00785,
    0.03225,
    0.04145,
    0.00234,
    -0.01968,
    -0.03286,
    0.00965,
    -0.03105,
    -0.00505,
    -0.0308,
    -0.03084,
    -0.00713,
    0.0062,
    0.01285,
    0.00528,
    -0.00219,
    -0.00973,
    0.00278,
    0.00436,
    -0.01433,
    -0.00851,
    -0.02624,
    -0.04147,
    -0.01361,
    0.01449,
    0.01215,
    0.00834,
    0.0253,
    -0.0115,
    0.02205,
    -0.04112,
    0.02069,
    -0.00461,
    0.00027,
    0.01231,
    -0.03802,
    -0.02165,
    -0.01011,
    0.02704,
    0.02269,
    0.01446,
    -0.00355,
    0.00558,
    -0.01563,
    -0.02031,
    0.01002,
    0.01922,
    0.07331,
    -0.01183,
    0.01001,
    -0.0368,
    0.02955,
    0.02594,
    -0.01332,
    -0.00364,
    0.01141,
    -0.01474,
    -0.0259,
    -0.00351,
    0.00878,
    0.00746,
    0.04178,
    0.04626,
    0.02296,
    0.01625,
    -0.01424,
    0.02483,
    -0.00251,
    0.00461,
    -0.00133,
    0.01905,
    0.02101,
    -0.0136,
    0.01115,
    0.01809,
    0.01383,
    0.00664,
    0.02355,
    -0.01721,
    -0.00433,
    0.01636,
    -0.03493,
    0.03379,
    -0.04537,
    -0.00015,
    -0.02172,
    -0.03468,
    -0.03726,
    4e-05,
    0.02805,
    -0.00346,
    0.01141,
    -0.03717,
    0.01885,
    -0.01391,
    0.02865,
    0.01187,
    0.00582,
    0.00845,
    -0.02973,
    -0.00892,
    0.03201,
    -0.0162,
    -0.0031,
    0.05394,
    0.01013,
    -0.01332,
    -0.00038,
    -0.03805,
    -0.01161,
    -0.00201,
    0.01181,
    -0.01627,
    0.00072,
    0.04654,
    0.011,
    0.02419,
    -0.00405,
    0.01091,
    -0.00845,
    0.05526,
    0.01433,
    0.04748,
    0.03126,
    0.02465,
    0.03579,
    -0.00197,
    -0.01006,
    0.04549,
    0.02183,
    0.02258,
    -0.05732,
    0.01491,
    -0.01133,
    -0.03316,
    0.06228,
    0.00115,
    -0.04494,
    0.01393,
    0.03014,
    -0.04759,
    0.00911,
    -0.02421,
    0.00611,
    0.01107,
    -0.02831,
    -0.01148,
    -0.00936,
    0.0116,
    0.01325,
    -0.00506,
    -0.01706,
    -0.02954,
    0.02665,
    -0.0281,
    0.00265,
    -0.00163,
    -0.02593,
    0.00903,
    -0.05095,
    0.00812,
    -0.05803,
    0.01686,
    -0.04204,
    -0.00167,
    -0.0024,
    0.0345,
    -0.01359,
    -0.00916,
    0.03219,
    0.02222,
    -0.02302,
    0.00029,
    0.00918,
    0.05997,
    0.00745,
    -0.00842,
    0.01476,
    0.02391,
    0.02495,
    -0.0098,
    -0.01463,
    -0.01844,
    0.00952,
    -0.03623,
    -0.02644,
    0.02672,
    0.03152,
    0.02313,
    0.03051,
    0.09834,
    0.00031,
    0.00965,
    -0.01827,
    0.01035,
    0.00509,
    -0.01855,
    0.00512,
    0.01158,
    0.00184,
    0.02605,
    -0.03931,
    -0.00925,
    -0.02031,
    -0.00653,
    -0.00507,
    -0.03686,
    -0.04281,
    0.04307,
    0.01926,
    -0.02218,
    -0.01345,
    0.00433,
    -0.02209,
    -0.01667,
    -0.0219,
    0.00686,
    0.04195,
    0.07523,
    -0.02228,
    0.01451,
    0.02469,
    0.01577,
    -0.01115,
    0.01609,
    0.00212,
    0.01607,
    -0.03674,
    0.02257,
    0.01996,
    0.01802,
    0.01064,
    -0.0047,
    0.0214,
    0.02629,
    0.01067,
    -0.02173,
    -0.00997,
    0.04223,
    0.03777,
    -0.03181,
    0.00443,
    -0.00551,
    -0.00877,
    -0.00943,
    -0.00752,
    0.00032,
    -0.03582,
    -0.00404,
    -0.01218,
    0.01467,
    -0.03222,
    -0.00156,
    -0.07274,
    0.01036,
    -8e-05,
    0.00397,
    -0.03518,
    0.02661,
    0.01741,
    -0.01094,
    0.03546,
    0.02267,
    -0.02334,
    -0.01256,
    -0.04853,
    -0.01057,
    0.06219,
    0.03474,
    0.05257,
    0.00151,
    0.01263,
    8e-05,
    0.01156,
    -0.00303,
    0.01902,
    -0.03014,
    0.03111,
    0.01952,
    0.01987,
    -0.01464,
    -0.01243,
    -0.01791,
    0.03678,
    -0.02807,
    -0.0014,
    -0.083,
    0.02336,
    0.01946,
    -0.0025,
    0.01372,
    -0.00381,
    -0.01946,
    -0.0284,
    -0.00965,
    -0.02145,
    0.0191,
    -0.00051,
    -0.01972,
    -0.00866,
    0.0059,
    0.01893,
    0.00673,
    -0.00125,
    0.01242,
    -0.01795,
    0.03279,
    0.01128,
    0.0068,
    0.0033,
    -0.03903,
    -0.01107,
    0.0495,
    -0.03604,
    0.00516,
    0.00551,
    0.0011,
    -0.02865,
    -0.05748,
    -0.0134,
    -0.00099,
    0.03371,
    -0.00931,
    -0.00826,
    0.02096,
    -0.00945,
    -0.00591,
    0.03361,
    -0.00785,
    -0.01341,
    -0.00022,
    -0.023,
    -0.00942,
    -0.01705,
    0.04389,
    0.01475,
    0.00829,
    -0.00885,
    0.02771,
    -0.00371,
    -0.00681,
    0.01103,
    -0.02157,
    0.01941,
    -0.03464,
    0.0118,
    0.0721,
    -0.01289,
    -0.00304,
    -0.01391,
    0.01599,
    -0.06846,
    -0.03379,
    0.02322,
    -0.02342,
    0.02467,
    -0.00602,
    -0.00481,
    -0.01171,
    0.01081,
    -0.00161,
    0.03428,
    -0.01296,
    -0.01057,
    0.00919,
    0.01848,
    0.01837,
    -0.02378,
    0.02744,
    -0.02435,
    0.05433,
    -0.01102,
    0.00844,
    0.0208,
    0.04935,
    -0.00403,
    0.01446,
    0.0006,
    -0.00491,
    -0.01554,
    -0.04696,
    -0.0045,
    -0.04968,
    0.01931,
    0.01584,
    -0.00913,
    -0.02921,
    -0.01177,
    0.0076,
    0.00435,
    0.00202,
    -0.02909,
    0.00666,
    0.01441,
    0.02334,
    0.02902,
    0.0057,
    0.02206,
    -0.03427,
    0.02305,
    0.01686,
    -0.02591,
    -0.03213,
    0.00401,
    -0.00471,
    0.01992,
    0.01239,
    -0.02298,
    0.00069,
    -0.04902,
    0.02614,
    0.01592,
    -0.01777,
    0.01379,
    0.00033,
    0.00515,
    0.04103,
    -0.01549,
    -0.01028,
    -0.00291,
    -0.0434,
    -0.03351,
    0.00235,
    -0.0292,
    0.05852,
    -0.00676,
    -0.05739,
    -0.01164,
    0.01091,
    0.01128,
    -0.01721,
    0.01528,
    0.04347,
    -0.02959,
    -0.00936,
    -0.01463,
    -0.00072,
    0.01952,
    -0.02938,
    0.04514,
    -0.03002,
    -0.01069,
    0.02938,
    0.02427,
    -0.02932,
    0.03691,
    -0.00549,
    -0.00119,
    -0.02151,
    -0.04043,
    0.01722,
    -0.01865,
    0.00596,
    -0.00187,
    0.03185,
    -0.00205,
    0.00883,
    -0.00487,
    0.02976,
    0.0154,
    0.03598,
    0.03422,
    0.02041,
    -0.01822,
    -0.00244,
    0.00098,
    0.03798,
    -0.05811,
    0.02992,
    -0.02079,
    -0.01434,
    -0.02474,
    -0.0303,
    0.00463,
    -0.01242,
    -0.01022,
    0.02022,
    -0.01379,
    -0.00929,
    0.00411,
    -0.01893,
    -0.02022,
    0.00971,
    -0.03394,
    0.01639,
    -0.0281,
    -0.02738,
    -0.00823,
    0.02901,
    -0.00554,
    -0.01922,
    -0.01162,
    -0.02861,
    0.01225,
    -0.01294,
    -0.04865,
    -0.03656,
    0.02026,
    -0.01988,
    -0.01732,
    0.03773,
    0.02801,
    0.02684,
    0.01818,
    0.03417,
    -0.02686,
    -0.00345,
    -0.03829,
    -0.02339,
    -0.01086,
    -0.03143,
    -0.0344,
    0.03684,
    0.00412,
    -0.01601,
    -0.01266,
    0.01859,
    0.00971,
    0.06298,
    0.0205,
    -0.0032,
    -0.0084,
    -0.00426,
    -0.00219,
    0.00177,
    0.00162,
    0.03217,
    0.00284,
    -0.0378,
    -0.01515,
    0.00414,
    0.01946,
    -0.01527,
    -0.01222,
    0.00769,
    -0.06186,
    -0.03662,
    0.02296,
    0.01223,
    0.02765,
    -0.00644,
    0.04507,
    0.00047,
    -0.02657,
    0.04211,
    0.00701,
    -0.03078,
    -0.01462,
    0.0404,
    0.0095,
    0.06117,
    0.06903,
    -0.00833,
    0.00098,
    -0.00805,
    -0.0424,
    0.0172,
    0.027,
    0.01328,
    0.05798,
    0.04083,
    -0.00985,
    0.0249,
    -0.03315,
    0.00522,
    0.00429,
    -0.0065,
    0.03879,
    0.01974,
    -0.0129,
    0.04897,
    -0.01613,
    0.02622,
    0.0391,
    0.00258,
    -0.00923,
    -0.04275,
    0.03439,
    0.02982,
    -0.02354,
    -0.00862,
    -0.06065,
    -0.03684,
    -0.00534,
    0.04131,
    0.00097,
    -0.0039,
    0.0163,
    -0.00869,
    0.02862,
    -0.00604,
    -0.01906,
    0.00151,
    0.01021,
    -0.00886,
    -0.04353,
    0.0218,
    -0.00317,
    -0.04912,
    -0.00988,
    0.01369,
    0.01335,
    0.04686,
    0.00755,
    0.00804,
    0.01999,
    0.00721,
    -0.00723,
    0.01062,
    0.00143,
    -0.00262,
    0.03923,
    0.04391,
    -0.01982,
    -0.01672,
    -0.00233,
    0.0237,
    -0.0204,
    -0.01202,
    0.04379,
    -0.00419,
    -0.01022,
    0.00345,
    0.06443,
    -0.01993,
    0.02359,
    0.00069,
    -0.03937,
    0.01204,
    0.00728,
    0.02007,
    0.17676,
    0.01628,
    -0.02245,
    -0.06703,
    0.00544,
    -0.00267,
    -0.01596,
    0.01718,
    0.02702,
    -0.02245,
    -0.00811,
    0.0014,
    0.01214,
    0.02739,
    0.00356,
    0.00373,
    0.01994,
    -0.06648,
    -0.03717,
    -0.00932,
    -0.00345,
    -0.03329,
    -0.00526,
    -0.02091,
    -0.01486,
    -0.01101,
    -0.02084,
    -0.02597,
    0.02397,
    0.01774,
    0.00392,
    0.01014,
    0.03607,
    -0.00775,
    0.03125,
    -0.02446,
    0.04405,
    0.02949,
    -0.03627,
    -0.0208,
    0.01329,
    -0.00075,
    0.02232,
    -0.02513,
    -0.03914,
    -0.00828,
    0.00574,
    0.03177,
    -0.01431,
    0.0071,
    0.00114,
    0.04153,
    -0.00516,
    -0.01491,
    -0.0105,
    -0.03006,
    0.0011,
    0.03298,
    0.01585,
    -0.00026,
    0.0058,
    0.01228,
    0.00285,
    -0.02192,
    0.03314,
    0.01633,
    -8e-05,
    -0.02691,
    0.01085,
    -0.0171,
    -0.00413,
    -0.00486,
    -0.00645,
    -0.01752,
    -0.02154,
    -0.05244,
    0.02793,
    -0.00132,
    0.01593,
    -0.01757,
    0.01424,
    -0.02517,
    0.02068,
    0.03583,
    -0.01779,
    -0.02965,
    0.02118,
    0.02703,
    -0.00625,
    0.02478,
    0.0062,
    0.01018,
    0.04328,
    -0.05007,
    4e-05,
    -0.01633,
    0.0073,
    -0.03087,
    0.06304,
    -0.0396,
    -0.07274,
    0.00064,
    -0.02529,
    0.01868,
    0.01035,
    0.02801,
    0.01742,
    -0.04574,
    -0.04735,
    -0.02567,
    0.00064,
    -0.00367,
    0.03185,
    -0.01553,
    0.04714,
    -0.00647,
    -0.01326,
    0.05906,
    0.02414,
    -0.05348,
    -0.03702,
    -0.01427,
    -0.02122,
    0.02232,
    0.01363,
    -0.00351,
    -0.00783,
    0.00425,
    0.01896,
    0.00431,
    -0.00915,
    0.03255,
    0.01969,
    -0.02384,
    -0.01544,
    -0.01001,
    -0.00713,
    0.00375,
    0.01713,
    0.0015,
    0.00653,
    0.01071,
    -0.00546,
    0.03756,
    -0.01164,
    0.01657,
    0.01629,
    0.03214,
    0.01691,
    -0.01598,
    0.03163,
    -0.01566,
    0.04446,
    -0.03339,
    -0.01001,
    -3e-05,
    -0.04175,
    -0.00457,
    -0.00309,
    0.0356,
    0.03509,
    -0.00695,
    -0.04454,
    -0.02302,
    -0.00959,
    0.01572,
    -0.02675,
    -0.00736,
    0.01645,
    0.00461,
    -0.00532,
    -0.02867,
    -0.01838,
    0.04651,
    0.02358,
    -0.00651,
    0.01717,
    -0.01575,
    0.01758,
    -0.02426,
    0.0418,
    -0.01133,
    -0.01375,
    -0.02089,
    -0.00096,
    0.00448,
    -0.03612,
    -0.004,
    0.04052,
    -0.01114,
    -0.01597,
    -0.03397,
    0.0295,
    -0.00602,
    0.01503,
    -0.0828,
    -0.01392,
    -0.02955,
    -0.02516,
    -0.00114,
    0.01117,
    0.02711,
    -0.00962,
    0.01012,
    0.00125,
    -0.0257,
    0.00429,
    0.0453,
    -0.00297,
    -0.03329,
    -0.03135,
    -0.01874,
    0.01123,
    -0.00693,
    -0.01597,
    -0.00089,
    -0.04465,
    0.05064,
    0.02858,
    -0.01681,
    0.01688,
    -0.00819,
    0.00129,
    0.01161,
    -0.02659,
    0.02656,
    0.00461,
    -0.02252,
    0.01169,
    0.01934,
    0.03793,
    0.02026,
    -0.01338,
    -0.00562,
    0.03373,
    0.00984,
    0.01057,
    -0.0124,
    0.03426,
    -0.01501,
    0.029,
    -0.01991,
    -0.00793,
    0.0119,
    0.00046,
    -0.0265,
    -0.04857,
    -0.01466,
    0.02335,
    -0.00277,
    0.01752,
    0.00029,
    -0.02193,
    0.03923,
    0.01847,
    0.03191,
    -0.00964,
    -0.00416,
    -0.05755,
    0.00578,
    0.00079,
    0.02364,
    0.07809,
    -0.00178,
    -0.00616,
    0.02558,
    0.00846,
    -0.00126,
    0.00456,
    -0.01255,
    -0.00952,
    0.00478,
    0.02929,
    0.00426,
    -0.00772,
    0.03837,
    -0.00698,
    0.02589,
    -0.00504,
    -0.01806,
    0.00031,
    0.04057,
    -0.00623,
    -0.00753,
    -0.00212,
    0.00843,
    -0.02564,
    0.06273,
    -0.01138,
    0.0448,
    0.0031,
    0.00888,
    -0.02073,
    0.01823,
    -0.00163,
    -0.02981,
    -0.0338,
    -0.01926,
    -0.04461,
    0.01041,
    0.02848,
    0.08119,
    -0.00185,
    0.00864,
    -0.00165,
    -0.00983,
    0.00554,
    0.01328,
    -0.04243,
    -0.01298,
    0.00286,
    0.03746,
    0.00655,
    0.01294,
    -0.02269,
    0.01313,
    0.03732,
    -0.0505,
    -0.00565,
    -0.0041,
    0.02363,
    0.02284,
    0.03058,
    0.01089,
    -0.00663,
    -0.022,
    0.02031,
    -0.01007,
    0.03356,
    -0.04825,
    -0.02007,
    -0.04025,
    -0.01827,
    -0.00301,
    -0.00353,
    0.06237,
    0.02853,
    -0.03631,
    -0.02352,
    -0.03844,
    -0.03691,
    -0.03328,
    0.02855,
    -0.02124,
    -0.03117,
    0.00266,
    -0.00866,
    0.01113,
    -0.06114,
    -0.02951,
    0.00605,
    -0.01747,
    0.02004,
    -0.00276,
    0.00272,
    -0.0085,
    0.02701,
    0.03146,
    -0.00934,
    -0.02363,
    0.02031,
    -0.02473,
    0.00465,
    -0.04875,
    -0.04142,
    0.01428,
    0.01598,
    -0.00251,
    0.00115,
    0.01135,
    0.0064,
    -0.00975,
    0.01798,
    -0.0247,
    -0.04176,
    0.02209,
    0.04273,
    0.00622,
    0.02167,
    0.00705,
    -0.00861,
    0.00474,
    -0.00634,
    0.07251,
    -0.00048,
    0.0251,
    -0.00874,
    -0.02921,
    0.03708,
    -0.03202,
    0.0088,
    -0.0541,
    -0.0031,
    -0.01351,
    -0.01653,
    -0.01052,
    0.00411,
    -0.00823,
    -0.02544,
    -0.01713,
    0.00012,
    0.01783,
    -0.05305,
    0.0341,
    0.02091,
    0.02386,
    -0.01993,
    -0.03849,
    0.0356,
    0.00121,
    -0.03757,
    -0.00292,
    0.02333,
    -0.00938,
    -0.0062,
    0.02301,
    0.05703,
    0.0449,
    -0.0024,
    -0.00942,
    -0.01335,
    -0.00034,
    0.02335,
    -0.01954,
    0.01248,
    0.00353,
    0.03213,
    -0.06216,
    0.02141,
    -0.0257,
    0.06878,
    -0.01675,
    -0.0507,
    -0.02858,
    -0.02412,
    -0.00395,
    0.02846,
    -0.00456,
    -0.00115,
    -0.01828,
    0.00054,
    0.00077,
    0.04899,
    -0.01983,
    0.0043,
    0.04159,
    0.0003,
    -0.01795,
    -0.00473,
    -0.00579,
    -0.00674,
    -0.00341,
    -0.03577,
    0.02139,
    0.03986,
    0.01776,
    0.03971,
    -0.00852,
    0.02438,
    -0.02317,
    -0.02132,
    0.00505,
    -0.02623,
    0.01584,
    -0.02332,
    0.01697,
    0.00481,
    0.05458,
    0.01215,
    0.03224,
    -0.02663,
    0.03413,
    0.02299,
    0.00212,
    0.00397,
    -0.02505,
    0.01637,
    -0.03041,
    -0.02877,
    0.01378,
    0.00025,
    -0.01208,
    0.00657,
    0.00392,
    0.00165,
    0.02264,
    0.00476,
    0.00718,
    -0.01081,
    -0.01502,
    -0.0197,
    -0.00218,
    0.03503,
    0.02333,
    -0.00475,
    0.02313,
    -0.00213,
    0.00079,
    0.00076,
    0.06776,
    0.04292,
    -0.01712,
    -0.01149,
    -0.00021,
    -0.03817,
    -0.00359,
    0.00962,
    -0.01299,
    0.0169,
    0.02072,
    0.00353,
    -0.0149,
    -0.02503,
    -0.00024,
    -0.03574,
    0.01916,
    -0.00866,
    0.02259,
    0.02062,
    -0.01241,
    0.04301,
    0.02679,
    -0.01935,
    0.02066,
    -0.00797,
    -0.00601,
    -0.01971,
    0.00408,
    -0.0105,
    0.00883,
    -0.0536,
    -0.00363,
    0.02019,
    -0.03031,
    -0.01626,
    -0.01352,
    0.02071,
    0.01175,
    -0.00329,
    -0.01885,
    -0.00987,
    0.0211,
    -0.01047,
    0.01229,
    -0.00464,
    0.01155,
    0.02229,
    -0.01085,
    -0.04436,
    -0.00945,
    0.01994,
    -0.02652,
    0.0094,
    -0.00087,
    -0.03933,
    -0.00374,
    0.18102,
    -0.02864,
    0.00416,
    0.08723,
    0.04689,
    -0.0203,
    0.01455,
    -0.01786,
    0.0081,
    0.01356,
    -0.00299,
    -0.04093,
    -0.01098,
    -0.03428,
    -0.04167,
    0.00401,
    -0.03184,
    -0.02047,
    -0.0042,
    -0.01646,
    0.0029,
    -0.00952,
    -0.00657,
    -0.02559,
    0.01325,
    0.02358,
    -0.01561,
    0.01358,
    -0.0262,
    0.00099,
    -0.00855,
    -0.03449,
    -0.00364,
    -0.00723,
    -0.00966,
    0.00077,
    -0.03946,
    -0.03064,
    0.00012,
    0.01014,
    -0.01437,
    -0.0107,
    0.05015,
    -0.00872,
    -0.01125,
    -0.02829,
    -0.00946,
    -0.0158,
    0.02007,
    -0.0028,
    -0.04051,
    0.01571,
    -0.02878,
    -0.05101,
    -0.01745,
    0.01056,
    0.03476,
    -0.01807,
    0.0259,
    -0.00856,
    0.01268,
    -0.01253,
    -0.02228,
    -0.01689,
    -0.01771,
    0.00558,
    0.01372,
    0.00529,
    -0.00986,
    -0.02517,
    -0.00398,
    -0.02943,
    0.04594,
    -0.0241,
    0.0222,
    0.04017,
    -0.02081,
    -0.01121,
    0.06187,
    -0.01066,
    -0.01132,
    -0.0046,
    -0.00924,
    0.02172,
    -0.01539,
    0.00921,
    -0.02243,
    -0.00941,
    0.01262,
    0.04462,
    0.03033,
    -0.01609,
    -0.00892,
    0.02736,
    -0.01004,
    0.02042,
    0.03211,
    -0.00257,
    0.01958,
    -0.00457,
    0.01547,
    -0.01254,
    0.01999,
    -0.00268,
    -0.01462,
    -0.02549,
    0.04056,
    0.01089,
    -0.00839,
    0.00813,
    0.02575,
    -0.00375,
    0.00055,
    -0.01652,
    -0.03045
   ],
   "SUDDEN": [
    0.00328,
    -0.00125,
    -0.03201,
    -0.01399,
    0.01411,
    -0.01916,
    -0.01366,
    -0.01779,
    -0.01035,
    -0.04012,
    0.00648,
    0.062,
    0.00439,
    -0.00367,
    -0.06086,
    -0.01016,
    -0.04052,
    0.01372,
    -0.02112,
    0.01071,
    0.00646,
    0.03885,
    -0.0106,
    -0.00957,
    -0.01468,
    -0.01824,
    -0.00673,
    -0.03501,
    -0.04217,
    0.04674,
    -0.0225,
    -0.03233,
    0.02796,
    -0.04076,
    -0.00706,
    0.00158,
    -0.00764,
    -0.0379,
    0.03194,
    -0.02952,
    0.00553,
    0.01386,
    -0.00728,
    -0.01053,
    0.00986,
    -0.04028,
    -0.01494,
    0.0167,
    0.02522,
    0.00378,
    0.01182,
    0.0181,
    0.0073,
    0.04302,
    -0.02715,
    -0.00643,
    -0.01602,
    -0.00743,
    0.02415,
    0.00332,
    0.02498,
    0.01917,
    0.01652,
    -0.01193,
    -0.02457,
    -0.00219,
    -0.00342,
    9e-05,
    -0.00025,
    -0.01904,
    -0.00953,
    -0.00743,
    0.01654,
    0.0421,
    0.02511,
    0.03294,
    0.00378,
    0.03737,
    0.03337,
    -4e-05,
    -0.00153,
    -0.00045,
    -0.10212,
    -0.01914,
    0.00304,
    0.00188,
    -0.00467,
    0.02008,
    0.00443,
    -0.01157,
    0.00615,
    -0.00358,
    0.00338,
    0.03469,
    0.01364,
    -0.03372,
    0.00575,
    -0.00168,
    -0.01304,
    0.03743,
    -0.00356,
    0.02346,
    0.0075,
    0.00419,
    0.03978,
    0.01761,
    -0.04476,
    0.02856,
    0.04273,
    0.03504,
    0.04194,
    -0.00891,
    0.01453,
    -0.00281,
    -0.04667,
    -0.02387,
    0.03274,
    -0.01567,
    -0.01769,
    0.00847,
    -0.00865,
    -0.0131,
    -0.05116,
    -0.0462,
    0.01554,
    -0.01383,
    -0.03859,
    -0.00871,
    -0.03665,
    0.02691,
    0.03797,
    0.00057,
    0.00552,
    -0.00159,
    0.04982,
    0.01383,
    -0.02028,
    -0.00022,
    0.01863,
    0.00915,
    0.0141,
    0.02665,
    0.01666,
    -0.02297,
    0.02156,
    -0.00737,
    0.00823,
    0.01761,
    0.01188,
    0.02619,
    -0.0069,
    0.00438,
    0.02141,
    0.0383,
    0.00218,
    0.02515,
    -0.03767,
    -0.01413,
    0.03835,
    0.02507,
    -0.04215,
    -0.0246,
    0.03597,
    0.00466,
    0.02328,
    0.03463,
    -0.01922,
    0.00106,
    -0.00461,
    0.01176,
    -0.02399,
    -0.03274,
    -0.01953,
    -0.02777,
    -0.01617,
    -0.00232,
    0.00322,
    0.01826,
    0.03642,
    0.02145,
    -0.04448,
    0.00647,
    -0.04839,
    0.02202,
    0.0033,
    7e-05,
    -0.03061,
    0.03254,
    -0.01753,
    0.02531,
    -0.01604,
    0.01335,
    0.01981,
    0.00736,
    0.00442,
    -0.03485,
    -0.03563,
    -0.01106,
    0.01558,
    -0.00338,
    -0.00821,
    -0.00742,
    0.00184,
    -0.00665,
    -0.05217,
    0.03701,
    0.02593,
    -0.00826,
    0.02986,
    -0.00398,
    0.01299,
    0.01014,
    -0.00968,
    0.05678,
    -0.00483,
    0.02573,
    0.01563,
    -0.02176,
    -0.01133,
    -0.02016,
    0.00788,
    0.01002,
    -0.02988,
    -0.0064,
    0.04529,
    0.00373,
    0.00431,
    0.07944,
    0.00651,
    0.0416,
    -0.03852,
    -0.01894,
    -0.0201,
    -0.03558,
    0.03642,
    -0.01282,
    -0.01183,
    0.0129,
    0.00437,
    0.0226,
    -0.01388,
    -0.01044,
    0.00618,
    0.00587,
    -0.00276,
    -0.00936,
    0.02518,
    0.0109,
    -0.01862,
    0.01122,
    -0.01533,
    -0.05045,
    0.01647,
    0.00834,
    0.0229,
    0.02652,
    0.04574,
    0.02697,
    -0.03067,
    0.02687,
    -0.00522,
    -0.02593,
    -0.00774,
    0.01266,
    0.00602,
    -0.00409,
    0.01292,
    -0.00593,
    0.0112,
    0.02774,
    0.00506,
    0.02428,
    -0.03542,
    -0.01453,
    -0.00982,
    -0.02224,
    0.032,
    0.02194,
    0.00266,
    -0.00642,
    -0.03387,
    -0.03248,
    -0.0011,
    -0.04621,
    0.01342,
    -0.01637,
    -0.04492,
    0.053,
    -0.01923,
    -0.02259,
    -0.00351,
    0.03116,
    0.01143,
    0.00224,
    0.01935,
    -0.0392,
    -0.03705,
    -0.01836,
    0.02426,
    0.0269,
    0.00316,
    0.01604,
    -0.01035,
    0.00809,
    -0.01054,
    0.00881,
    0.01078,
    -0.03663,
    -0.00943,
    0.00468,
    0.0148,
    -0.00152,
    -0.00666,
    0.0279,
    -0.00445,
    0.05114,
    -0.00398,
    0.02854,
    -0.01387,
    0.01832,
    0.05531,
    0.01566,
    0.00694,
    0.01728,
    0.00219,
    -0.00319,
    0.02996,
    -0.02381,
    0.01435,
    -0.02732,
    -0.00951,
    -0.00193,
    -0.0283,
    -0.00871,
    -0.02493,
    -0.02177,
    -0.00775,
    -0.01751,
    -0.04622,
    -0.02035,
    0.01731,
    0.0155,
    0.06194,
    0.01823,
    0.03059,
    0.00895,
    -0.00183,
    0.0135,
    0.00112,
    -0.02099,
    0.01002,
    0.03922,
    -0.02161,
    0.00191,
    -0.02408,
    -0.00735,
    -0.05554,
    -0.04527,
    0.00892,
    -0.00938,
    0.02088,
    0.0073,
    -0.04638,
    -0.02118,
    -0.02395,
    0.06313,
    -0.00321,
    0.05012,
    0.02605,
    -0.0677,
    -0.03286,
    0.04112,
    -0.02778,
    -0.0389,
    0.05317,
    -0.01878,
    -0.02049,
    -0.00594,
    0.00937,
    0.01389,
    0.01412,
    -0.00608,
    -0.01277,
    -0.01908,
    0.0351,
    0.01863,
    -0.01773,
    -0.02415,
    0.00747,
    -0.01347,
    0.02104,
    0.0225,
    -0.00385,
    0.01288,
    -0.01796,
    0.03412,
    -0.00808,
    -0.05585,
    -0.00183,
    0.01899,
    -0.03048,
    0.01367,
    0.03268,
    -0.00895,
    -0.02191,
    -0.00098,
    -0.00138,
    -0.02603,
    -0.02314,
    0.01664,
    -0.01869,
    0.01554,
    -0.01116,
    -0.01495,
    -0.008,
    0.02743,
    0.0177,
    0.00566,
    0.0606,
    0.00348,
    -0.00438,
    -0.05415,
    -0.00519,
    0.01629,
    -0.0203,
    0.02412,
    0.03694,
    0.04905,
    -0.0005,
    -0.00629,
    0.01843,
    -0.0084,
    -0.03554,
    0.02152,
    -0.02806,
    -0.02047,
    0.01939,
    -0.01562,
    0.0178,
    -0.00787,
    -0.00273,
    0.02608,
    -0.03188,
    0.02125,
    -0.00692,
    0.03392,
    -0.01576,
    0.03069,
    0.00807,
    0.00478,
    -0.01292,
    -0.03277,
    -0.0313,
    0.02662,
    0.03964,
    0.00044,
    -0.00539,
    0.03298,
    -0.0341,
    -0.00982,
    0.01164,
    0.00512,
    -0.02956,
    0.00847,
    -0.02207,
    -0.00377,
    -0.02535,
    -0.02741,
    -0.01292,
    -0.02343,
    -0.00949,
    -0.00183,
    -0.04855,
    -0.02243,
    0.01106,
    0.0031,
    0.00703,
    0.02603,
    0.00418,
    0.03945,
    -0.00688,
    0.00343,
    -0.00036,
    0.01009,
    -0.00804,
    0.01014,
    0.009,
    0.05874,
    -0.0383,
    -0.00122,
    -0.03313,
    0.06006,
    -0.00363,
    -0.025,
    0.00918,
    -0.03415,
    -0.02773,
    0.03635,
    -0.02793,
    0.00094,
    -0.01395,
    -0.04425,
    -0.02717,
    -0.02632,
    0.01055,
    0.03423,
    -0.00096,
    0.0275,
    -0.00292,
    -0.0121,
    -0.01011,
    -0.06484,
    -0.00038,
    -0.03846,
    0.01946,
    0.00369,
    -0.02074,
    -0.02951,
    0.01152,
    0.03061,
    0.08141,
    -0.01004,
    -0.01026,
    0.01967,
    -0.00786,
    0.04993,
    -0.00952,
    0.01055,
    -0.01128,
    -0.01891,
    0.0135,
    0.01755,
    0.02724,
    -0.05837,
    -0.01888,
    -0.0288,
    -0.03497,
    0.01355,
    -0.02952,
    -0.00316,
    -0.02977,
    -0.01681,
    -0.01766,
    0.03153,
    0.05656,
    0.00213,
    -0.01227,
    0.02624,
    -0.00551,
    -0.02348,
    -0.00992,
    0.0132,
    0.0445,
    0.03084,
    -0.00095,
    -0.02146,
    0.03205,
    0.0173,
    -0.02084,
    0.02273,
    0.01148,
    -0.04791,
    0.01022,
    0.01498,
    0.00232,
    0.02744,
    -0.00135,
    -0.01739,
    -0.02819,
    -0.00754,
    -0.00462,
    0.0106,
    -0.01452,
    -0.00674,
    -0.03608,
    -0.0136,
    0.00147,
    -0.01946,
    -0.00882,
    0.01173,
    0.01753,
    -0.00349,
    0.02249,
    0.00145,
    -0.0056,
    0.04419,
    0.02958,
    -0.01138,
    0.01457,
    0.04247,
    0.0077,
    -0.01136,
    0.01244,
    -0.01344,
    0.00251,
    -0.03441,
    0.02062,
    0.00481,
    0.0239,
    -0.00304,
    0.00647,
    0.00373,
    -0.0193,
    0.0399,
    -0.01309,
    -0.04253,
    0.03496,
    -0.02373,
    -0.03596,
    0.04462,
    0.00467,
    -0.02928,
    0.01434,
    0.03224,
    0.03799,
    -0.00377,
    0.02584,
    -0.01409,
    -1e-05,
    -0.0108,
    0.05134,
    0.01277,
    0.02143,
    -0.00043,
    -0.00664,
    0.02999,
    0.00068,
    -0.03415,
    -0.01779,
    0.0088,
    -0.01493,
    0.00459,
    0.01896,
    0.03011,
    -0.01314,
    -0.00166,
    0.02586,
    -0.02834,
    0.00655,
    -0.03957,
    0.01362,
    0.0086,
    -0.01726,
    0.01379,
    -0.0088,
    -0.00195,
    0.03875,
    -0.0036,
    -0.01394,
    0.01171,
    -0.00451,
    -0.00384,
    0.00371,
    -0.00673,
    0.00405,
    0.03411,
    0.01434,
    0.04277,
    -0.08646,
    0.00899,
    0.02398,
    0.02755,
    0.00407,
    0.03519,
    0.01958,
    -0.03651,
    -0.05459,
    0.03079,
    0.00839,
    0.01922,
    0.00013,
    -0.02265,
    0.00988,
    -0.01335,
    0.0096,
    0.00907,
    -0.05411,
    0.00857,
    0.0148,
    -0.00269,
    -0.00499,
    0.00652,
    -0.01216,
    0.03749,
    -0.01869,
    0.00056,
    0.00025,
    0.01289,
    -0.05788,
    -0.00098,
    -0.00497,
    0.04752,
    -0.02344,
    -0.0065,
    -0.08733,
    -0.02589,
    -0.04819,
    0.00947,
    0.00751,
    0.01219,
    -0.01997,
    0.04571,
    -0.02381,
    -0.00496,
    0.03693,
    0.02716,
    -0.0018,
    0.00029,
    0.0261,
    -0.01907,
    0.00632,
    0.00147,
    -0.01107,
    -0.00428,
    0.00026,
    -0.00936,
    -0.00564,
    0.01447,
    -0.00393,
    0.0228,
    0.04952,
    -0.02221,
    -0.03787,
    -0.00895,
    0.03068,
    0.01425,
    0.00473,
    -0.02223,
    0.00075,
    0.02817,
    -0.01046,
    -0.04137,
    0.04798,
    -0.02382,
    -0.01678,
    -0.02701,
    -0.01872,
    -0.01125,
    -0.0202,
    0.02003,
    0.00231,
    0.01674,
    -0.05619,
    0.01802,
    0.01978,
    0.02794,
    -0.03279,
    0.00884,
    0.0045,
    0.00947,
    0.03475,
    0.02116,
    0.03314,
    0.00643,
    0.04152,
    0.01168,
    0.0013,
    -0.03002,
    -0.05242,
    0.01215,
    -0.01608,
    -0.00411,
    -0.00405,
    0.00898,
    -0.01818,
    -0.01485,
    0.00933,
    -0.01375,
    -0.00983,
    0.0336,
    -0.00426,
    0.01155,
    -0.00085,
    0.00984,
    -0.00863,
    -0.03056,
    -0.0089,
    0.0129,
    -0.03867,
    0.01214,
    0.01166,
    0.00054,
    -0.03201,
    -0.04089,
    -0.04387,
    -0.04072,
    -0.00587,
    -0.01643,
    0.00593,
    0.02621,
    -0.00281,
    -0.02455,
    -0.00419,
    -0.03206,
    -0.02536,
    0.02486,
    0.0153,
    0.00806,
    0.00251,
    0.00922,
    0.05726,
    -0.03363,
    -0.02525,
    0.00357,
    -0.04215,
    -0.01138,
    -0.03258,
    0.00437,
    0.00914,
    -0.00705,
    0.01933,
    -0.0319,
    -0.02332,
    -0.02857,
    0.02214,
    0.04274,
    0.04651,
    0.02027,
    -0.00567,
    0.00513,
    0.04088,
    -0.03205,
    -0.02259,
    -0.04503,
    -0.00981,
    0.03463,
    0.01931,
    -0.00335,
    -0.00464,
    -0.0074,
    -0.00936,
    -0.02004,
    0.0326,
    -0.0009,
    0.00276,
    0.00598,
    -0.068,
    -0.00457,
    -0.00946,
    0.00759,
    -0.04991,
    0.00567,
    -0.03597,
    -0.00627,
    -0.02054,
    -0.04805,
    -0.03149,
    0.00726,
    0.00571,
    -0.00709,
    -0.0138,
    0.01483,
    0.0093,
    -0.00081,
    0.01761,
    0.01003,
    0.00133,
    0.03134,
    0.02824,
    -0.02119,
    -0.00121,
    -0.00138,
    -0.00492,
    0.00154,
    -0.00936,
    -0.07463,
    -0.01761,
    0.00094,
    0.07639,
    0.0044,
    -0.01007,
    -0.02862,
    -0.00854,
    -0.01766,
    0.02459,
    -0.01006,
    0.01413,
    -0.03432,
    -0.02696,
    0.0108,
    -0.01491,
    -0.00479,
    0.04254,
    0.04024,
    -0.0296,
    0.00164,
    0.05529,
    0.02338,
    0.01158,
    0.01029,
    0.03528,
    0.00959,
    0.02033,
    -0.02203,
    -0.02433,
    -0.00957,
    -0.005,
    -0.00128,
    0.02115,
    -0.02942,
    0.02408,
    -0.01948,
    0.01782,
    0.05258,
    -0.00339,
    -0.01361,
    -0.03062,
    -0.03262,
    0.029,
    0.02678,
    -0.00389,
    -0.01946,
    -0.02649,
    0.01975,
    0.03515,
    -0.05282,
    -0.00879,
    0.01309,
    -0.01458,
    -0.00938,
    0.00698,
    0.01646,
    -0.00208,
    -0.00987,
    -0.0148,
    0.02598,
    -0.04671,
    -0.02776,
    0.04578,
    -0.0343,
    -0.00694,
    -0.00626,
    0.06886,
    -0.00131,
    0.00693,
    -0.00693,
    -0.04195,
    0.00823,
    -0.01594,
    0.02126,
    0.02884,
    -8e-05,
    0.01912,
    -0.01613,
    -0.00476,
    0.01746,
    -0.00033,
    -0.03846,
    -0.03105,
    0.0076,
    0.04046,
    -0.00884,
    -0.01155,
    0.01968,
    0.00395,
    -0.0026,
    -0.02931,
    -0.01585,
    0.00153,
    -0.03457,
    0.00127,
    -0.02391,
    0.00445,
    -0.0332,
    0.03038,
    0.06628,
    0.01449,
    -0.01165,
    -0.0045,
    0.03029,
    0.0268,
    0.0052,
    0.01974,
    -0.00201,
    0.01911,
    0.01876,
    -0.00634,
    0.00183,
    0.04287,
    0.01988,
    0.02251,
    -0.01362,
    -0.05339,
    -0.04119,
    0.00977,
    0.00943,
    0.0101,
    -0.01823,
    -0.0299,
    -0.00818,
    0.00766,
    0.01007,
    0.00628,
    0.03192,
    -0.00891,
    -0.03254,
    -0.01074,
    -0.02494,
    0.01725,
    0.0351,
    -0.00281,
    0.01233,
    -0.01056,
    -0.03211,
    0.00123,
    -0.00175,
    -0.00159,
    0.01089,
    -0.02592,
    0.04641,
    -0.00991,
    0.02266,
    -0.0437,
    0.01157,
    0.01208,
    -0.01267,
    0.02166,
    -0.02579,
    0.0341,
    0.00502,
    0.0169,
    0.02554,
    0.02835,
    -0.00422,
    -0.03122,
    -0.03743,
    -0.01194,
    0.02246,
    -0.00022,
    0.00747,
    0.02246,
    -0.00263,
    0.03984,
    -0.02261,
    -0.02502,
    -0.02416,
    0.05091,
    0.02928,
    -0.0299,
    -0.00835,
    0.0335,
    0.01345,
    0.00042,
    -0.03542,
    0.01409,
    -0.0142,
    0.01598,
    0.04075,
    -0.01508,
    0.01674,
    -0.00666,
    0.05319,
    -0.00838,
    -0.03105,
    0.00937,
    0.00217,
    0.04569,
    -0.04286,
    -0.0103,
    -0.01754,
    0.03267,
    0.00205,
    0.0044,
    -0.00921,
    0.00292,
    -0.02193,
    -0.04603,
    0.01917,
    -0.02106,
    0.00228,
    -0.0201,
    -0.0083,
    -0.01148,
    0.03093,
    0.04599,
    0.02957,
    0.00344,
    -0.00939,
    0.01405,
    0.00539,
    -0.0136,
    -0.03395,
    -0.05128,
    -0.02363,
    -0.00984,
    0.00013,
    -0.01153,
    0.01408,
    -0.00273,
    -0.01605,
    -0.02618,
    -0.01633,
    -0.04271,
    0.01804,
    -0.03725,
    -0.05608,
    0.01633,
    0.00132,
    0.00985,
    -0.0332,
    -0.02083,
    0.01588,
    0.00317,
    -0.04513,
    -0.01648,
    -0.02652,
    0.0182,
    0.00823,
    -0.02282,
    -0.00164,
    0.02058,
    0.07052,
    0.03766,
    -0.01866,
    -0.02151,
    -0.03126,
    -0.01693,
    0.02001,
    -0.06776,
    -0.02514,
    0.00122,
    -0.021,
    -0.01044,
    -0.01149,
    -0.0052,
    -0.00087,
    -0.01716,
    -0.03906,
    0.02015,
    -0.01491,
    -0.01193,
    -0.02304,
    -0.00225,
    -0.01625,
    0.05366,
    0.02803,
    0.03301,
    -0.02632,
    0.0037,
    -0.01848,
    -0.02516,
    -0.02169,
    -0.02597,
    0.02307,
    0.02098,
    0.00664,
    0.00691,
    -0.01446,
    -0.01823,
    -0.01422,
    -0.0254,
    0.05615,
    -0.03514,
    -0.00427,
    -0.0376,
    -0.00132,
    -0.0005,
    0.01535,
    0.00462,
    0.00124,
    -0.01418,
    0.01444,
    0.00208,
    0.0252,
    -0.00717,
    -0.01835,
    0.00928,
    0.00532,
    0.02929,
    0.05215,
    0.02068,
    -0.01161,
    -0.00874,
    0.00033,
    0.02733,
    -0.00563,
    -0.04418,
    0.00174,
    -0.01691,
    0.00102,
    -0.01789,
    -0.01099,
    0.00919,
    0.00348,
    0.02149,
    -0.01435,
    -0.02544,
    -0.0218,
    -0.00264,
    0.01286,
    0.03793,
    -0.02644,
    0.00876,
    -0.03386,
    0.06213,
    0.00108,
    0.03962,
    0.02591,
    -0.00479,
    0.0145,
    -0.04578,
    -0.0066,
    0.03135,
    0.02278,
    0.04701,
    0.00729,
    0.0252,
    -0.00432,
    0.01685,
    0.02017,
    0.00326,
    0.00732,
    -0.01791,
    0.02998,
    0.00374,
    -0.04108,
    0.0328,
    -0.00632,
    0.01461,
    -0.00697,
    -0.00226,
    -0.04189,
    -0.04084,
    -0.01837,
    0.02484,
    -0.00676,
    0.02551,
    -0.00786,
    0.04292,
    -0.00758,
    -0.00883,
    -0.01118,
    -0.03146,
    0.01437,
    -0.03433,
    0.03119,
    0.02338,
    -0.03361,
    -0.00407,
    0.04266,
    -0.02463,
    0.00672,
    -0.02805,
    0.00404,
    -0.00136,
    -0.00768,
    0.00638,
    -0.00864,
    -0.02434,
    -0.04828,
    -0.02535,
    0.00161,
    -0.0154,
    -0.00241,
    0.01806,
    -0.02243,
    0.04474,
    -0.02571,
    -0.0071,
    0.03048,
    0.0067,
    -0.03521,
    -0.00571,
    0.01876,
    -0.00693,
    -0.02404,
    0.00641,
    0.03895,
    -0.0435,
    -0.01079,
    -0.02555,
    0.0174,
    0.02236,
    0.00399,
    0.0157,
    -0.00048,
    -0.02184,
    -0.00491,
    0.02237,
    -0.03293,
    -0.04543,
    -0.04893,
    -0.04394,
    0.09057,
    -0.01703,
    -0.03924,
    -0.0273,
    -0.00294,
    0.01072,
    -0.00638,
    -0.0112,
    -0.0302,
    0.05798,
    -0.05037,
    -0.01102,
    0.01582,
    -0.01352,
    0.04968,
    0.0191,
    -0.01533,
    -0.01389,
    0.03495,
    0.00146,
    -0.00486,
    -0.00478,
    0.01772,
    -0.00618,
    -0.03886,
    -0.00807,
    -0.01151,
    -0.04354,
    0.01096,
    0.02016,
    0.02216,
    -0.03228,
    0.00125,
    -0.03803,
    -0.00145,
    -0.01907,
    0.01106,
    -0.04088,
    -0.0405,
    0.01444,
    -0.05028,
    0.01592,
    0.02414,
    -0.00577,
    0.01316,
    -0.00529,
    0.00987,
    -0.00874,
    0.02338,
    -0.02871,
    -0.01591,
    -0.04628,
    -0.00613,
    -0.03109,
    0.00547,
    0.04265,
    -0.00603,
    0.00657,
    -0.00889,
    0.03944,
    0.00081,
    -0.027,
    -0.02211,
    0.02016,
    0.00558,
    -0.00563,
    -0.01754,
    -0.00447,
    0.00932,
    0.02129,
    -0.00386,
    0.02937,
    0.00045,
    0.00569,
    0.03575,
    -0.03919,
    -0.03617,
    -0.00928,
    0.02242,
    -0.01364,
    -0.01943,
    -0.02634,
    -0.03365,
    0.04615,
    0.00278,
    0.01641,
    0.04181,
    0.00572,
    -0.01067,
    -0.00374,
    -0.03952,
    -0.02773,
    -0.01313,
    0.03978,
    0.07199,
    -0.00473,
    0.01408,
    -0.00239,
    0.0008,
    -0.0047,
    -0.03496,
    0.00477,
    -0.04991,
    -0.01732,
    -0.01524,
    -0.0253,
    -0.04076,
    -0.01709,
    0.00367,
    -0.02197,
    -0.01584,
    -0.00638,
    0.03105,
    -0.02571,
    -0.09007,
    0.01731,
    0.01361,
    0.01323,
    -0.01973,
    -0.00014,
    -0.00887,
    -0.01164,
    -0.00929,
    -0.02754,
    0.03202,
    -0.01193,
    -0.00055,
    -0.00029,
    -0.01434,
    0.04382,
    0.03447,
    0.01263,
    -0.02242,
    -0.00113,
    -0.00979,
    -0.00918,
    0.05857,
    0.03654,
    -0.13901,
    0.04954,
    -0.03413,
    0.12254,
    -0.02074,
    0.00752,
    -0.01285,
    0.00326,
    0.00752,
    0.00218,
    -0.0072,
    0.04224,
    0.01943,
    0.01898,
    0.03333,
    0.00794,
    -0.00859,
    0.01945,
    -0.00527,
    0.06202,
    0.01059,
    0.01436,
    0.01663,
    -0.02896,
    0.01155,
    -0.00476,
    -0.00643,
    0.02652,
    0.02853,
    0.00192,
    -0.01446,
    0.00537,
    0.01674,
    0.00711,
    -0.02918,
    -0.02539,
    0.02894,
    0.02348,
    0.00756,
    -0.0272,
    0.03945,
    -0.0084,
    -0.04965,
    -0.0014,
    0.03661,
    -0.00397,
    0.00844,
    -0.01695,
    0.00256,
    -0.01261,
    -0.01273,
    -0.01836,
    0.03565,
    0.03552,
    0.02944,
    0.00821,
    -0.02083,
    -0.02578,
    -0.02587,
    0.01213,
    -0.00435,
    -0.00098,
    0.0287,
    0.02357,
    0.02793,
    -0.04857,
    0.00802,
    0.03802,
    0.02277,
    0.04738,
    0.03937,
    0.01946,
    -0.03898,
    -0.00833,
    -0.00928,
    -0.02655,
    0.03003,
    0.01895,
    -0.041,
    0.02057,
    0.01536,
    0.04482,
    -0.0058,
    -0.04366,
    0.01259,
    -0.00226,
    0.06,
    -0.00233,
    0.00578,
    -0.03063,
    -0.0053,
    0.00376,
    -0.01725,
    -0.01464,
    0.02197,
    -0.01994,
    -0.02341,
    0.0344,
    -0.0381,
    0.02164,
    -0.0133,
    0.00245,
    -0.02183,
    0.01596,
    0.04621,
    0.00656,
    -0.03484,
    -0.01591,
    -0.01432,
    -0.03202,
    0.0064,
    -0.01658,
    -0.03317,
    0.01843,
    0.01994
   ],
   "NUMBNESS": [
    0.01964,
    -0.01428,
    0.05318,
    0.01426,
    -0.01016,
    0.01148,
    0.02051,
    0.00429,
    0.00828,
    0.0451,
    0.00075,
    -0.00873,
    -0.01612,
    -0.01095,
    -0.00989,
    -0.00596,
    0.0189,
    0.00132,
    0.01639,
    -0.03631,
    -0.00439,
    0.01058,
    -0.00201,
    0.00132,
    0.01214,
    0.02471,
    -0.00608,
    0.00399,
    0.0564,
    -0.02795,
    0.02413,
    0.03084,
    -0.03818,
    0.04666,
    -0.01021,
    0.00644,
    -0.01781,
    0.04303,
    -0.02732,
    -0.02141,
    0.01942,
    -0.0209,
    0.02122,
    0.02125,
    0.00533,
    0.03379,
    0.02459,
    -0.03531,
    -0.00818,
    -0.01671,
    0.03793,
    -0.00884,
    -0.00231,
    -0.03024,
    0.06807,
    0.02281,
    0.02547,
    -0.00789,
    -0.03894,
    0.01607,
    -0.02248,
    -0.01508,
    0.00749,
    0.00336,
    0.03451,
    0.02271,
    0.00341,
    0.00094,
    -0.00415,
    0.03264,
    0.01098,
    -0.00997,
    -0.03843,
    6e-05,
    0.00377,
    0.02143,
    0.02236,
    -0.01098,
    -0.02932,
    0.0021,
    0.02863,
    0.00233,
    -0.01362,
    0.00747,
    0.02213,
    -0.03088,
    -0.00767,
    -0.01356,
    -0.0123,
    0.00569,
    -0.03553,
    -0.02183,
    -0.01347,
    -0.01435,
    0.01389,
    0.0677,
    0.00112,
    -0.00887,
    0.00713,
    -0.02847,
    -0.01218,
    -0.02609,
    -0.00513,
    -0.02015,
    0.00315,
    -0.01648,
    0.02297,
    -0.02286,
    0.00255,
    -0.0215,
    -0.00518,
    0.00679,
    -0.01985,
    0.0032,
    0.03084,
    0.00723,
    -0.01323,
    0.02157,
    0.03658,
    0.00548,
    0.00056,
    0.01841,
    0.03241,
    0.03948,
    0.00528,
    0.01978,
    0.00952,
    0.00591,
    0.03649,
    0.00857,
    -0.05421,
    0.01366,
    -0.02574,
    0.00104,
    -0.03794,
    -0.01206,
    0.01564,
    0.00927,
    0.00632,
    0.0118,
    0.00579,
    0.00783,
    -0.00635,
    -0.03565,
    0.01029,
    0.02723,
    -0.01998,
    -0.00712,
    0.01984,
    -0.03927,
    0.02105,
    0.0042,
    -0.00135,
    -0.01087,
    0.02564,
    -0.00437,
    0.03272,
    0.00468,
    -0.02557,
    -0.01907,
    0.01921,
    0.01061,
    0.00453,
    0.02002,
    -0.01408,
    -0.0293,
    0.03663,
    -0.00695,
    0.03375,
    0.02035,
    -0.00651,
    -0.02862,
    0.01791,
    -0.01514,
    0.00575,
    -0.01737,
    -0.02271,
    -0.00525,
    -0.02418,
    -0.00669,
    0.00832,
    -0.00724,
    0.02532,
    -0.04558,
    -0.00228,
    0.09006,
    -0.01229,
    -0.03489,
    -0.01726,
    -0.00036,
    0.00463,
    -0.0029,
    -0.0207,
    -0.00121,
    -0.04417,
    0.01872,
    0.04178,
    0.00123,
    0.00653,
    -0.00721,
    0.03342,
    0.00976,
    -0.01206,
    0.02582,
    0.05142,
    -0.0222,
    -0.02758,
    0.03305,
    -0.00146,
    -0.01672,
    0.00796,
    -0.02603,
    -0.01785,
    -0.03757,
    0.00021,
    -0.00787,
    -0.01486,
    0.00322,
    0.01704,
    0.02117,
    -0.02715,
    0.00573,
    0.03955,
    0.03333,
    -0.04224,
    -0.02196,
    0.00352,
    -0.02476,
    0.02229,
    -0.01533,
    0.04794,
    0.01688,
    0.01342,
    0.05303,
    -0.00371,
    0.00346,
    0.04559,
    0.01802,
    -0.01668,
    -0.02599,
    0.01363,
    -0.0275,
    -0.01079,
    -0.03183,
    -0.02348,
    -0.02003,
    -0.00599,
    0.03165,
    0.00863,
    0.00432,
    -0.03533,
    -0.01507,
    0.00222,
    -0.08762,
    -0.01922,
    0.0111,
    -0.02703,
    -0.02224,
    0.02045,
    -0.00478,
    0.02123,
    0.02674,
    -0.00394,
    0.02784,
    -0.02779,
    0.00224,
    0.01875,
    -0.01341,
    0.01583,
    -0.03456,
    -0.01573,
    -0.01658,
    0.02266,
    0.00282,
    0.02098,
    -0.00436,
    0.01385,
    -0.00708,
    0.00127,
    -7e-05,
    0.00455,
    0.04982,
    -0.00099,
    0.02139,
    -0.033,
    0.03414,
    0.02029,
    -0.00441,
    -2e-05,
    0.01161,
    -0.01419,
    -0.02714,
    0.00475,
    0.00956,
    -0.0117,
    0.03688,
    0.03108,
    0.026,
    0.01873,
    -0.01877,
    -0.00119,
    0.01761,
    0.00505,
    -0.00528,
    0.02315,
    0.01137,
    -0.00719,
    0.01959,
    0.0119,
    -0.00394,
    -0.01055,
    0.01457,
    0.00053,
    -0.01706,
    0.00373,
    -0.03714,
    0.03197,
    -0.0319,
    0.01172,
    -0.0202,
    -0.05072,
    -0.03945,
    -0.00662,
    0.02994,
    0.00023,
    0.02555,
    -0.01185,
    -0.02648,
    -0.00697,
    0.04634,
    0.01709,
    0.02185,
    -0.01797,
    -0.02247,
    -0.0033,
    0.00134,
    -0.00883,
    -0.00736,
    0.03702,
    0.00467,
    -0.02218,
    0.01413,
    -0.03593,
    -0.02347,
    0.0059,
    0.02304,
    -0.01569,
    0.00256,
    0.01766,
    0.0182,
    0.02127,
    -0.01448,
    0.0043,
    -0.0169,
    0.06589,
    0.01072,
    -0.01157,
    0.01554,
    0.01797,
    0.03522,
    0.01639,
    -0.01536,
    0.04761,
    0.02291,
    0.03357,
    -0.05101,
    0.02403,
    -0.02716,
    -0.01966,
    0.07082,
    0.00868,
    -0.01633,
    0.02043,
    0.02634,
    -0.05248,
    0.00545,
    -0.00857,
    0.00202,
    0.01491,
    -0.01867,
    -0.00255,
    -0.00773,
    0.01612,
    0.0095,
    -0.03138,
    -0.02294,
    -0.01748,
    0.0208,
    -0.02427,
    0.00181,
    -0.00091,
    -0.02453,
    0.00208,
    -0.0378,
    -0.0067,
    -0.03265,
    0.00925,
    -0.0221,
    0.02036,
    -0.01522,
    0.0179,
    -0.02424,
    0.01059,
    0.04354,
    0.01566,
    -0.00084,
    0.01195,
    0.01104,
    0.04757,
    0.02727,
    0.0119,
    0.01834,
    0.02177,
    -0.0002,
    -0.0006,
    0.00152,
    -0.0277,
    0.02751,
    -0.04551,
    -0.01637,
    0.00863,
    0.03336,
    0.0254,
    0.02315,
    0.03033,
    -0.01373,
    -0.00436,
    -0.01278,
    -0.00675,
    0.01489,
    -0.00702,
    -0.00269,
    -0.00148,
    0.00793,
    0.03149,
    -0.03616,
    -0.02659,
    -0.00446,
    -0.01482,
    0.00687,
    -0.01135,
    -0.03607,
    0.03621,
    0.03466,
    0.0006,
    -0.01944,
    -0.01296,
    -0.01142,
    -0.02349,
    -0.01899,
    -0.00196,
    0.01845,
    0.06055,
    -0.02926,
    0.00315,
    0.02158,
    0.00879,
    -0.01806,
    -0.02504,
    0.00784,
    -0.00701,
    -0.02393,
    0.02129,
    -0.00044,
    0.00948,
    0.02322,
    -0.02403,
    0.0318,
    0.03008,
    0.01937,
    -0.03682,
    -0.01761,
    0.01548,
    0.06636,
    -0.00635,
    -0.01584,
    0.00365,
    -0.00156,
    -0.02018,
    -0.02303,
    -0.00273,
    -0.04733,
    -0.00784,
    0.00077,
    0.00659,
    -0.02287,
    -0.00788,
    -0.05217,
    0.02531,
    -0.01436,
    0.0153,
    -0.00995,
    0.02703,
    0.01715,
    -0.0093,
    0.03823,
    0.00115,
    -0.01529,
    -0.03193,
    0.0025,
    -0.02824,
    0.06488,
    0.04159,
    0.04062,
    -0.00683,
    0.01633,
    -0.01434,
    0.00877,
    -0.00568,
    0.01205,
    -0.012,
    0.04237,
    0.00583,
    0.03544,
    -0.0047,
    -0.01397,
    -0.00165,
    0.03781,
    -0.01673,
    0.0138,
    -0.08875,
    0.0137,
    0.01255,
    0.00977,
    -0.0016,
    -0.01018,
    -0.00197,
    -0.01373,
    0.00663,
    -0.02302,
    -0.00655,
    0.0028,
    -0.00084,
    0.00831,
    0.01404,
    0.01842,
    0.00245,
    0.01653,
    0.00691,
    0.00742,
    0.03801,
    0.01379,
    0.0172,
    0.01467,
    -0.01622,
    -0.00723,
    0.08104,
    -0.02823,
    0.03129,
    -0.00074,
    0.00456,
    -0.02812,
    -0.04971,
    -0.02306,
    0.02355,
    0.02625,
    -0.00637,
    -0.03077,
    0.02132,
    0.0002,
    -0.01299,
    0.03723,
    -0.01718,
    -0.00429,
    0.00335,
    0.0012,
    0.00522,
    -0.01008,
    0.03144,
    0.02063,
    0.01468,
    -0.01292,
    0.02369,
    -0.00176,
    0.01336,
    0.0079,
    -0.00872,
    0.03061,
    -0.01993,
    0.00062,
    0.10965,
    -0.00658,
    -0.00444,
    -0.00666,
    0.00775,
    -0.02782,
    -0.02243,
    0.01477,
    -0.02164,
    0.01614,
    0.00516,
    0.00892,
    -0.0229,
    0.03704,
    0.00565,
    0.02679,
    -0.00262,
    -0.0116,
    -0.00168,
    0.00105,
    0.01685,
    -0.02315,
    0.03215,
    0.00455,
    0.05541,
    0.00451,
    -0.01389,
    0.02322,
    0.06073,
    -0.07402,
    0.01802,
    -0.00098,
    -0.00224,
    -0.01706,
    -0.01092,
    -0.01094,
    -0.0462,
    0.01585,
    0.00969,
    -0.00205,
    -0.03898,
    -0.022,
    -0.00285,
    0.001,
    0.01799,
    -0.02931,
    0.02605,
    0.0147,
    0.02248,
    0.02701,
    0.00695,
    -0.0053,
    -0.01074,
    0.02307,
    0.01739,
    -0.02028,
    -0.01553,
    0.00359,
    -0.01765,
    0.03118,
    0.01268,
    0.01366,
    -0.01196,
    -0.02975,
    0.0172,
    0.01718,
    -0.02246,
    -0.0004,
    0.02649,
    -0.0012,
    0.03052,
    -0.01437,
    0.00869,
    0.01208,
    -0.02666,
    -0.03064,
    -0.01524,
    -0.02427,
    0.04566,
    -0.02134,
    -0.06125,
    -0.02977,
    0.00423,
    0.0045,
    -0.02877,
    0.00386,
    0.03923,
    -0.00687,
    -0.00453,
    -0.00021,
    -0.01877,
    0.01065,
    -0.03878,
    0.05563,
    -0.03667,
    -0.00133,
    0.00268,
    -0.00336,
    -0.02613,
    0.01398,
    0.00655,
    0.00766,
    -0.00511,
    -0.0298,
    0.01743,
    -0.01472,
    0.00833,
    -0.0001,
    0.03651,
    -0.0147,
    0.01243,
    0.01267,
    0.01971,
    0.00736,
    0.06018,
    0.02399,
    0.02213,
    -0.04298,
    -0.00904,
    -0.0125,
    0.0196,
    -0.03193,
    0.01881,
    -0.02095,
    -0.02398,
    -0.01946,
    -0.02451,
    0.00878,
    -0.01427,
    -0.01531,
    -0.00336,
    -0.00568,
    0.00457,
    -0.01648,
    -0.00683,
    -0.02558,
    0.00329,
    -0.03235,
    0.02846,
    -0.03328,
    -0.02293,
    -0.02492,
    0.02443,
    0.01409,
    -0.00044,
    -0.01667,
    -0.04159,
    0.02872,
    0.00188,
    -0.04213,
    -0.03151,
    0.01744,
    -0.03895,
    0.01235,
    0.05862,
    0.03582,
    0.02478,
    0.02798,
    0.01841,
    -0.02761,
    0.00576,
    -0.03,
    -0.0244,
    -0.00945,
    -0.03304,
    -0.05562,
    0.03045,
    -0.00596,
    -0.00588,
    -0.01744,
    -0.0312,
    -0.00258,
    0.03851,
    -0.0019,
    -0.01188,
    -0.01286,
    -0.00631,
    0.00994,
    -0.02077,
    0.0059,
    0.05043,
    0.00832,
    -0.03006,
    0.0224,
    0.02121,
    0.01005,
    -0.01619,
    0.00641,
    -0.01085,
    -0.05596,
    -0.01845,
    0.03174,
    -0.00354,
    0.0144,
    -0.00548,
    0.03022,
    -0.00973,
    -0.01876,
    0.04965,
    0.00923,
    0.00738,
    -0.01749,
    0.03721,
    0.01809,
    0.04544,
    0.06072,
    -0.00147,
    -0.0059,
    -0.01927,
    -0.01889,
    0.01567,
    0.02773,
    0.01416,
    0.03031,
    0.0449,
    -0.02474,
    0.02792,
    -0.0176,
    0.01355,
    0.00349,
    -0.00458,
    0.05529,
    0.02366,
    -0.0084,
    0.03389,
    -0.01893,
    0.02345,
    0.03694,
    -0.0218,
    -0.01134,
    -0.02921,
    0.02896,
    0.02561,
    -0.04703,
    0.00565,
    -0.0477,
    -0.04018,
    -0.02149,
    0.03155,
    0.00261,
    -0.01986,
    0.01575,
    -0.00323,
    0.05008,
    0.00141,
    -0.00482,
    -0.00506,
    0.01126,
    -0.00054,
    -0.00475,
    0.01495,
    0.00015,
    -0.03094,
    -0.01221,
    0.02647,
    0.0154,
    0.06327,
    0.02275,
    -0.01305,
    0.01312,
    0.00901,
    0.00511,
    0.00334,
    -0.00995,
    0.00092,
    0.02114,
    0.03989,
    -0.03707,
    -0.04453,
    0.00247,
    0.0263,
    -0.02231,
    -0.00502,
    0.01441,
    -0.0042,
    0.00383,
    -0.00469,
    0.05816,
    0.02454,
    0.01931,
    -0.00651,
    -0.05656,
    0.015,
    0.00637,
    -0.01416,
    0.14835,
    0.01338,
    0.02063,
    -0.14749,
    0.00772,
    -0.00182,
    0.00623,
    0.03508,
    0.01799,
    -0.0174,
    -0.00231,
    0.0027,
    0.01172,
    0.03587,
    0.00867,
    0.00401,
    0.01061,
    -0.06859,
    -0.03228,
    -0.00973,
    -0.00082,
    -0.0463,
    -0.00266,
    -0.0207,
    -0.01948,
    -0.00176,
    -0.02482,
    -0.03032,
    0.01355,
    0.01746,
    0.01136,
    0.02086,
    0.01856,
    -0.02304,
    0.04928,
    -0.02374,
    0.0826,
    0.02465,
    -0.04896,
    -0.01539,
    0.02127,
    -0.02076,
    0.02575,
    -0.00545,
    -0.05631,
    -0.01186,
    0.00572,
    0.02208,
    -0.01841,
    0.00041,
    0.00021,
    0.0244,
    0.00179,
    -0.01582,
    -0.01786,
    -0.02766,
    0.00519,
    0.03083,
    0.01147,
    -0.0071,
    -0.00203,
    0.00931,
    0.01708,
    -0.01119,
    0.03266,
    0.02535,
    -0.00145,
    -0.02915,
    -0.02295,
    -0.00877,
    -0.03508,
    0.00344,
    -0.01003,
    0.00432,
    -0.02465,
    -0.03517,
    0.02444,
    -0.00011,
    0.00086,
    -0.01489,
    0.00389,
    -0.03624,
    0.01793,
    0.04906,
    -0.02031,
    -0.03796,
    0.01201,
    0.02055,
    -0.00252,
    0.00183,
    -0.01086,
    0.00815,
    0.02074,
    -0.02377,
    0.00789,
    -0.01997,
    0.01998,
    -0.02547,
    0.06061,
    -0.02844,
    -0.07646,
    -0.00814,
    -0.01877,
    0.0307,
    0.00831,
    0.01287,
    0.00229,
    -0.04858,
    -0.04486,
    -0.0224,
    0.00283,
    0.00131,
    0.01542,
    0.00728,
    0.04428,
    -0.02871,
    -0.00703,
    0.06714,
    0.00206,
    -0.05458,
    -0.04485,
    -0.00748,
    0.00105,
    0.03588,
    0.00408,
    0.00582,
    0.00861,
    0.00164,
    0.00064,
    0.01511,
    0.01457,
    0.03804,
    0.02646,
    -0.01009,
    0.00203,
    -0.02801,
    -0.00373,
    0.01246,
    0.03142,
    -0.01574,
    0.00212,
    0.016,
    0.0162,
    0.03997,
    -0.03293,
    0.02318,
    0.01169,
    0.01555,
    0.01172,
    -0.00581,
    0.03022,
    -0.02812,
    0.04835,
    -0.03351,
    0.01327,
    -0.00681,
    -0.03264,
    -0.02595,
    -0.0004,
    0.03363,
    0.03802,
    -0.00892,
    -0.03822,
    0.02253,
    -0.01813,
    -0.01413,
    -0.0136,
    -0.00129,
    0.0198,
    0.00489,
    -0.00739,
    -0.03916,
    -0.01618,
    0.03446,
    0.00961,
    -0.01842,
    0.01679,
    -0.00188,
    0.0329,
    0.00118,
    0.04786,
    -0.01712,
    -0.0218,
    -0.0154,
    0.01666,
    0.00832,
    -0.02005,
    -0.00708,
    0.0394,
    -0.01475,
    -0.00378,
    -0.02976,
    0.03953,
    -0.00137,
    0.02127,
    -0.05377,
    -0.0119,
    -0.02886,
    -0.01204,
    0.00107,
    0.00578,
    0.05231,
    -0.01117,
    -0.01888,
    -0.00363,
    -0.01095,
    -0.00203,
    0.04718,
    0.01925,
    -0.04584,
    -0.02233,
    -0.03357,
    0.02312,
    0.00464,
    -0.01552,
    0.00558,
    -0.00353,
    0.05213,
    0.02054,
    -0.00158,
    0.028,
    -0.00835,
    0.00153,
    0.01382,
    -0.03116,
    0.02865,
    0.01697,
    -0.00921,
    -0.01326,
    0.00371,
    0.03724,
    -0.00745,
    -0.01182,
    -0.00631,
    0.04189,
    0.01146,
    0.00343,
    -0.00431,
    0.03771,
    -0.00552,
    0.01912,
    -0.02566,
    0.00346,
    0.01314,
    0.01888,
    -0.03037,
    -0.04458,
    -0.01292,
    0.03874,
    -0.00927,
    0.00522,
    0.00716,
    -0.00318,
    0.03721,
    0.02111,
    0.02417,
    -0.00912,
    -0.00212,
    -0.05032,
    -0.00058,
    -0.00613,
    0.04968,
    0.07728,
    -0.01868,
    0.00035,
    0.02379,
    0.01091,
    -0.00159,
    -0.00071,
    -0.02342,
    -0.01183,
    -0.01146,
    0.04229,
    -0.00914,
    -0.00586,
    0.03013,
    0.0095,
    0.04354,
    -0.01287,
    -0.01033,
    -0.01592,
    0.0255,
    -0.00657,
    -0.021,
    0.02225,
    0.01162,
    -0.03823,
    0.0637,
    3e-05,
    0.02213,
    -0.01168,
    -0.0016,
    -0.0103,
    0.00876,
    0.00053,
    -0.00504,
    -0.02747,
    -0.00148,
    -0.03455,
    0.01544,
    0.01712,
    0.0445,
    -0.00487,
    -0.00867,
    -0.02304,
    0.00155,
    0.01148,
    0.00136,
    -0.02851,
    -0.01791,
    -0.00016,
    0.03033,
    -0.01149,
    0.01795,
    -0.01733,
    0.02767,
    0.03668,
    -0.03745,
    0.00828,
    -0.02538,
    -0.00499,
    0.03124,
    0.01815,
    -0.00321,
    -0.00318,
    -0.03996,
    0.02536,
    -0.00786,
    0.0276,
    -0.04711,
    -0.01082,
    -0.03436,
    -0.02027,
    -0.03411,
    0.00102,
    0.03634,
    0.03891,
    -0.04499,
    -0.01073,
    -0.02054,
    -0.00951,
    -0.02661,
    0.02268,
    -0.02787,
    -0.03701,
    0.01657,
    -0.0175,
    0.01059,
    -0.06852,
    -4e-05,
    -0.01386,
    -0.01287,
    0.01125,
    -0.0004,
    -0.01692,
    -0.00638,
    0.02426,
    0.01684,
    -0.00784,
    0.00183,
    -0.00081,
    -0.01716,
    0.00081,
    -0.02188,
    -0.04047,
    0.01225,
    0.01018,
    -0.01922,
    -0.01002,
    0.03164,
    0.01515,
    -0.02174,
    0.00881,
    -0.02668,
    -0.04037,
    0.02953,
    0.0311,
    0.01337,
    0.00475,
    0.01245,
    -0.01528,
    -0.01441,
    -0.00529,
    0.06763,
    0.0032,
    0.00864,
    0.00145,
    -0.02258,
    0.03142,
    -0.03711,
    0.02151,
    -0.05202,
    -0.00042,
    0.0004,
    -0.00436,
    -0.0105,
    0.00012,
    -0.01323,
    -0.02875,
    -0.01318,
    0.01101,
    0.002,
    -0.05027,
    0.00919,
    0.01412,
    0.03673,
    -0.00208,
    -0.01149,
    0.02514,
    -0.01477,
    -0.04085,
    -0.00215,
    0.02076,
    -0.01822,
    -0.00106,
    0.01387,
    0.03912,
    0.0364,
    -0.08794,
    -0.01442,
    -0.04277,
    -0.00454,
    0.03884,
    -0.01201,
    0.01497,
    0.03389,
    0.03094,
    -0.07212,
    0.00477,
    -0.00662,
    -0.01928,
    -0.01782,
    -0.04696,
    -0.01813,
    -0.02442,
    -0.0108,
    0.0581,
    -0.00637,
    0.0244,
    -0.01897,
    0.00355,
    0.00148,
    0.0315,
    -0.02831,
    0.00943,
    0.04265,
    0.00422,
    -0.01381,
    -0.02288,
    0.00462,
    -0.00011,
    0.00964,
    -0.02727,
    -0.00432,
    0.0431,
    0.01506,
    0.04388,
    -0.00308,
    0.02952,
    -0.03284,
    -0.03583,
    -0.02292,
    -0.01585,
    -0.00567,
    -0.00741,
    0.01199,
    -0.0187,
    0.05711,
    0.01007,
    0.02093,
    -0.01638,
    0.02824,
    0.02465,
    -0.02179,
    0.00385,
    0.00132,
    0.01562,
    -0.03451,
    -0.01845,
    0.00666,
    0.00444,
    -0.0104,
    -0.00552,
    0.01325,
    0.01251,
    0.03205,
    0.00089,
    0.01939,
    -0.00425,
    -0.02922,
    0.00092,
    -0.00538,
    0.03348,
    -0.02203,
    0.01399,
    0.03209,
    0.00606,
    0.00601,
    0.00785,
    0.06379,
    0.04999,
    -0.01326,
    -0.0143,
    -0.01303,
    -0.01466,
    0.0066,
    -0.01133,
    -0.00797,
    0.00786,
    0.00026,
    -0.006,
    -0.01197,
    -0.04678,
    0.00358,
    -0.0461,
    0.03349,
    0.00204,
    0.02285,
    0.03315,
    -0.01663,
    0.04992,
    0.04766,
    0.02105,
    0.01699,
    0.014,
    -0.01488,
    -0.03515,
    0.02048,
    -0.01274,
    -0.00352,
    -0.06349,
    -0.01078,
    0.06356,
    -0.03847,
    -0.02409,
    -0.01139,
    -0.00065,
    0.02422,
    0.00478,
    0.01093,
    -0.0188,
    0.01792,
    -0.01174,
    0.01448,
    0.00834,
    0.01654,
    0.03109,
    -0.02927,
    -0.03756,
    -0.00896,
    0.03662,
    -0.02929,
    0.01337,
    -0.01522,
    -0.03642,
    -0.00415,
    0.20988,
    -0.02307,
    -0.00846,
    0.00573,
    0.04535,
    -0.02214,
    0.02422,
    -0.00669,
    0.0011,
    0.01552,
    0.00406,
    -0.02426,
    -0.01798,
    -0.00614,
    -0.01202,
    0.01059,
    -0.01686,
    -0.00962,
    0.00992,
    -0.00414,
    -0.01201,
    -0.00396,
    -0.00596,
    -0.03012,
    0.01341,
    0.02611,
    -0.01696,
    -0.00265,
    -0.02853,
    -0.01028,
    -0.02485,
    -0.02869,
    0.01501,
    -0.00828,
    -0.00387,
    -0.01161,
    -0.03141,
    -0.01855,
    0.0035,
    0.01224,
    0.00081,
    0.00435,
    0.03942,
    -0.01454,
    -0.04925,
    -0.01155,
    -0.01501,
    -0.01706,
    0.03174,
    -0.00049,
    -0.03017,
    0.02196,
    -0.02572,
    -0.01783,
    0.02208,
    -0.00373,
    0.01484,
    -0.01767,
    0.0293,
    -0.02204,
    0.00138,
    -0.00013,
    -0.03214,
    -0.04546,
    -0.00934,
    0.03683,
    0.00423,
    -0.00259,
    -0.01212,
    -0.0028,
    -0.00384,
    -0.0287,
    0.05416,
    -0.028,
    0.00995,
    0.026,
    -0.00216,
    -0.00535,
    0.05364,
    -0.01838,
    -0.00861,
    -0.00573,
    -0.00922,
    0.02525,
    -0.01796,
    0.00212,
    -0.03691,
    -0.0028,
    0.0318,
    0.03125,
    0.02991,
    -0.02442,
    -0.01081,
    0.02591,
    -0.02739,
    0.02719,
    0.0397,
    -0.00669,
    0.02406,
    0.01106,
    0.00411,
    -0.01801,
    0.01926,
    -0.00568,
    -0.00708,
    -0.01948,
    0.0401,
    0.01627,
    0.00333,
    0.00502,
    0.03172,
    -0.01406,
    0.006,
    -0.01616,
    -0.03886
   ],
   "ISOTOPE": [
    0.00664,
    0.00538,
    0.00556,
    -0.0148,
    -0.05874,
    -0.00742,
    -0.01511,
    -0.03644,
    -0.00783,
    0.02216,
    0.00714,
    -0.08221,
    0.00165,
    0.04049,
    -0.02496,
    -0.04521,
    0.00092,
    0.03344,
    0.01242,
    0.00631,
    0.02428,
    0.03125,
    0.00744,
    0.00607,
    -0.01048,
    0.01149,
    -0.05045,
    -0.00592,
    0.01369,
    0.00456,
    0.02906,
    0.03951,
    -0.01075,
    0.01179,
    0.04419,
    -0.02506,
    -0.01927,
    -0.05503,
    -0.01891,
    -0.02571,
    0.01817,
    -0.00859,
    -0.05035,
    -0.01699,
    -0.02477,
    -0.01001,
    0.004,
    0.01067,
    -0.0075,
    -0.01123,
    0.01873,
    0.0689,
    0.00322,
    -0.02005,
    -0.00102,
    -0.01392,
    0.00511,
    -0.0161,
    -0.03133,
    -0.01875,
    -0.00159,
    -0.0466,
    -0.00207,
    0.04381,
    -0.06816,
    0.0186,
    -0.02418,
    -0.0406,
    -0.04332,
    0.03366,
    -0.05191,
    -0.0317,
    0.02722,
    0.02302,
    0.01915,
    0.00089,
    0.00355,
    0.00253,
    -0.01934,
    0.00608,
    0.00329,
    -0.01317,
    -0.07817,
    -0.01151,
    0.02523,
    -0.02908,
    0.00798,
    -0.00804,
    0.03908,
    0.00303,
    -0.04141,
    -0.00388,
    0.00499,
    0.00095,
    0.03062,
    -0.00876,
    -0.01413,
    -0.01341,
    0.0316,
    -0.02988,
    -0.01529,
    0.02891,
    -0.04229,
    0.0234,
    0.01417,
    -0.02201,
    -0.00301,
    -0.01223,
    0.03155,
    -0.00118,
    -0.01537,
    0.00905,
    0.0317,
    -0.05716,
    0.00373,
    -0.03602,
    0.00366,
    0.01152,
    -0.01493,
    -0.01416,
    0.00774,
    -0.01882,
    -0.01851,
    -0.01279,
    -0.03146,
    -0.0188,
    0.00024,
    -0.00988,
    -0.01349,
    -0.00529,
    -0.01998,
    0.00226,
    -0.0034,
    0.02252,
    -0.0118,
    0.00122,
    -0.007,
    -0.02662,
    -0.0125,
    -0.01511,
    -0.04646,
    0.02882,
    0.01134,
    -0.02693,
    0.00616,
    0.06586,
    0.00446,
    -0.00824,
    0.01442,
    0.00046,
    -0.01804,
    -0.04887,
    0.01126,
    0.01196,
    0.02068,
    -0.02859,
    0.00595,
    -0.05073,
    0.02103,
    -0.01295,
    -0.05588,
    0.0051,
    -0.03103,
    -0.02002,
    -0.02462,
    -0.00789,
    0.03184,
    -0.0185,
    -0.00662,
    0.00322,
    -0.01421,
    0.00409,
    0.00445,
    0.03471,
    -0.02919,
    -0.00324,
    0.02731,
    -0.00294,
    -0.01375,
    -0.00596,
    0.05572,
    -0.0319,
    -0.01559,
    0.01984,
    -0.01071,
    0.00076,
    0.01701,
    -0.02718,
    -0.00724,
    0.0276,
    -0.01517,
    -0.04189,
    -0.0205,
    -0.03626,
    -0.02682,
    -0.00719,
    -0.02774,
    -0.06681,
    0.00138,
    0.03302,
    -0.00524,
    0.01298,
    -0.01043,
    0.04028,
    -0.01797,
    0.0137,
    0.04419,
    0.02485,
    0.02059,
    -0.00202,
    0.03238,
    -0.01351,
    0.02169,
    -0.00652,
    -0.02032,
    0.007,
    -0.03478,
    -0.00502,
    0.00456,
    -0.00577,
    -0.01775,
    0.02579,
    -0.00261,
    0.00625,
    -0.00024,
    -0.02879,
    0.01554,
    0.01053,
    -0.00792,
    0.00984,
    -0.00194,
    -0.00346,
    0.01227,
    -0.02613,
    -0.01019,
    -0.03948,
    0.01793,
    -0.00671,
    0.03189,
    0.00434,
    0.01445,
    -0.00321,
    -0.01273,
    0.02396,
    -0.01363,
    0.02086,
    0.01939,
    0.01497,
    -0.00837,
    -0.01309,
    -0.01278,
    -0.02157,
    -0.02758,
    -0.08479,
    0.0019,
    -0.01281,
    0.01415,
    0.0071,
    0.00737,
    0.0267,
    -0.05463,
    -0.02043,
    -0.00747,
    0.00929,
    -0.0115,
    0.01936,
    -0.01129,
    -0.01804,
    -0.02622,
    0.01049,
    0.0059,
    -0.00338,
    0.00609,
    -0.0016,
    0.00258,
    -0.02258,
    0.00392,
    -0.00953,
    -0.00657,
    -0.0269,
    -0.02816,
    2e-05,
    0.0142,
    0.0039,
    -0.0094,
    0.01109,
    0.02163,
    0.00212,
    -0.021,
    -0.0179,
    0.03175,
    -0.06042,
    0.001,
    -0.01124,
    0.01694,
    0.01576,
    -0.00138,
    0.00664,
    -0.00843,
    -0.0166,
    0.00497,
    -0.01013,
    0.00948,
    -0.00839,
    0.01644,
    -0.03363,
    -0.02297,
    0.01133,
    0.06677,
    0.01117,
    0.00139,
    -0.01255,
    -0.01518,
    -0.01813,
    0.01929,
    -0.01242,
    0.06612,
    0.00917,
    -0.01647,
    0.00741,
    -0.0195,
    0.00043,
    -0.05133,
    -0.03828,
    9e-05,
    -0.0305,
    -0.0038,
    -0.02939,
    0.01987,
    -0.003,
    -0.00428,
    0.01753,
    -0.01039,
    -0.01136,
    0.00146,
    -0.02176,
    0.04196,
    -0.00666,
    -0.00926,
    0.00444,
    0.01243,
    0.00299,
    0.02343,
    0.01629,
    0.03311,
    0.02336,
    -0.04503,
    0.04813,
    -0.00521,
    0.00476,
    -0.00667,
    0.01871,
    -0.00176,
    0.02916,
    -0.01408,
    0.03809,
    -0.00384,
    0.02247,
    0.02568,
    0.0321,
    0.0105,
    0.01957,
    0.01011,
    -0.04893,
    0.03175,
    0.01446,
    0.01768,
    -0.03173,
    0.0131,
    -0.01276,
    0.02648,
    -0.03045,
    0.00409,
    -0.01679,
    0.00693,
    -0.01556,
    -0.00354,
    0.01285,
    0.00146,
    -0.00377,
    -0.00148,
    0.04038,
    -0.00061,
    0.02072,
    0.02498,
    -0.01667,
    -0.03478,
    0.00508,
    -0.01684,
    0.01625,
    -0.0254,
    -0.00445,
    -0.0265,
    0.01148,
    0.05339,
    -0.00154,
    -0.0106,
    -0.02018,
    -0.04334,
    -0.00405,
    0.00499,
    -0.01568,
    -0.00712,
    -0.02609,
    -0.0157,
    -0.015,
    0.01132,
    0.03063,
    0.03534,
    -0.0082,
    -0.00071,
    -0.02411,
    -0.01066,
    -0.00213,
    0.02,
    -0.01507,
    -0.03727,
    0.01057,
    0.02929,
    0.03545,
    -0.01882,
    -0.03089,
    0.01645,
    0.03982,
    -0.01914,
    -0.02124,
    -0.07899,
    -0.01852,
    -0.00825,
    -0.025,
    -0.01358,
    -0.03002,
    -0.01278,
    -0.00748,
    -0.03247,
    -0.0155,
    -0.02293,
    -0.00531,
    -0.0565,
    0.0298,
    0.02782,
    0.04371,
    0.02665,
    -0.00335,
    -0.02558,
    -0.03252,
    -0.05282,
    -0.00753,
    -0.00667,
    0.01474,
    -0.00296,
    0.00883,
    0.02317,
    0.01968,
    0.03424,
    -0.02527,
    0.00841,
    0.02404,
    -0.0161,
    -0.02301,
    0.02566,
    0.00419,
    0.02247,
    0.05981,
    -0.07893,
    -0.01496,
    0.05627,
    0.02219,
    -0.01308,
    -0.00767,
    -0.00128,
    0.02829,
    0.03164,
    -0.0016,
    -0.0077,
    0.00097,
    0.04275,
    0.01193,
    -0.00388,
    0.02791,
    -0.01116,
    -0.01569,
    -0.04453,
    0.04772,
    0.02418,
    -0.05047,
    -0.01044,
    0.0169,
    0.03835,
    -0.01008,
    -0.00682,
    0.01524,
    -0.01057,
    -0.04541,
    0.01126,
    0.00532,
    -0.00046,
    -0.01978,
    -0.04599,
    -0.02734,
    -0.01607,
    -0.0117,
    -0.02163,
    0.03115,
    0.00675,
    0.00414,
    -0.00336,
    0.02349,
    -0.02716,
    0.02164,
    -0.0137,
    -0.00714,
    -0.01726,
    0.00715,
    0.00984,
    -0.01145,
    0.01525,
    -0.01016,
    0.0315,
    0.01891,
    -0.02085,
    0.02944,
    0.02487,
    -0.04759,
    0.017,
    -0.04886,
    0.0335,
    -0.00296,
    -0.02736,
    -0.01673,
    0.04639,
    0.01256,
    0.01138,
    -0.00934,
    0.01483,
    -0.01678,
    0.00915,
    0.0191,
    -0.00212,
    0.00634,
    -0.01081,
    0.01963,
    -0.0154,
    -0.01619,
    -0.04409,
    0.00131,
    -0.00762,
    -0.00994,
    -0.03861,
    0.01467,
    -0.02527,
    -0.03943,
    -0.01987,
    -0.04757,
    -0.02299,
    0.02685,
    -0.03674,
    -0.01817,
    0.00571,
    0.02185,
    -0.02316,
    0.01476,
    0.0071,
    0.04599,
    -0.03175,
    0.00718,
    0.03069,
    -0.01078,
    0.02062,
    0.0045,
    0.00199,
    0.00557,
    0.03827,
    -0.01654,
    0.01329,
    0.02179,
    -0.00043,
    -0.01504,
    -0.0118,
    -0.01653,
    -0.01943,
    0.00614,
    0.03387,
    -0.00569,
    0.00143,
    -0.00643,
    -0.0462,
    0.02162,
    0.02727,
    0.0111,
    -0.02265,
    -0.01857,
    0.05701,
    0.01567,
    0.0195,
    -0.00231,
    -0.01435,
    -0.02448,
    0.00094,
    0.00533,
    -0.00748,
    0.01447,
    -0.01813,
    0.01071,
    -0.00763,
    -0.01047,
    -0.02666,
    -0.0254,
    -0.00565,
    -0.02944,
    -0.01346,
    -0.01933,
    0.03198,
    0.01615,
    0.00167,
    -0.02613,
    -0.06151,
    -0.00028,
    0.01183,
    0.01466,
    -0.01643,
    0.00382,
    -0.01874,
    0.01778,
    -0.01886,
    0.00601,
    0.03388,
    -0.0035,
    -0.00533,
    0.01837,
    0.00812,
    0.00642,
    0.00193,
    0.03419,
    0.0161,
    0.00474,
    0.00711,
    0.07172,
    -0.00346,
    -0.03406,
    -0.00561,
    0.0052,
    0.01277,
    0.02723,
    -0.01301,
    0.00062,
    0.01436,
    -0.00351,
    -0.00404,
    0.01702,
    -0.00344,
    0.00415,
    0.00559,
    0.04552,
    0.04613,
    0.00111,
    0.0022,
    0.02627,
    -0.00666,
    -0.01111,
    -0.01549,
    -0.01243,
    -0.01302,
    0.02644,
    -0.02481,
    -0.02621,
    0.00921,
    -0.00326,
    0.01766,
    -0.00638,
    0.00028,
    -0.01589,
    -0.00889,
    0.01705,
    -0.02135,
    0.01484,
    -0.02995,
    0.00823,
    0.02523,
    -0.03469,
    -0.01375,
    0.02915,
    -0.01902,
    -0.0169,
    0.00443,
    0.01613,
    -0.00475,
    0.00377,
    -0.03471,
    -0.00739,
    0.04238,
    0.00636,
    0.00921,
    0.00245,
    -0.00724,
    0.03153,
    -0.03659,
    0.02,
    0.00282,
    0.01973,
    -0.0376,
    -0.0398,
    0.03636,
    -0.02812,
    0.00631,
    0.0059,
    -0.00716,
    0.03567,
    -0.02927,
    -0.02595,
    -0.01684,
    -0.00211,
    0.0111,
    -0.03748,
    0.00733,
    -0.04426,
    -0.0104,
    0.00454,
    0.00703,
    -0.0065,
    0.03104,
    0.04392,
    -0.00768,
    -0.02994,
    0.01757,
    0.02134,
    0.02204,
    -0.01409,
    -0.0201,
    -0.0522,
    -0.0306,
    -0.01201,
    0.01662,
    -0.01219,
    -0.00014,
    0.042,
    0.02073,
    -0.01634,
    -0.03993,
    -0.02441,
    0.03095,
    -0.01654,
    2e-05,
    0.02666,
    0.0495,
    0.00172,
    0.04112,
    -0.01583,
    0.00514,
    -0.00335,
    -0.00512,
    -0.0215,
    0.03701,
    -0.01126,
    -0.04825,
    0.00188,
    -0.05216,
    0.00091,
    -0.05101,
    -0.0174,
    -0.00927,
    -0.00231,
    -0.01804,
    -0.0339,
    0.04409,
    0.01126,
    -0.02789,
    -0.07402,
    -0.01155,
    -0.04702,
    0.02932,
    0.05356,
    -0.00525,
    0.03846,
    -0.03653,
    0.04583,
    0.04829,
    -0.04849,
    -0.01803,
    -0.00104,
    -0.02004,
    0.00505,
    0.01324,
    -0.00982,
    0.01696,
    0.03087,
    0.00625,
    -0.04025,
    0.00228,
    0.00127,
    -0.01453,
    0.02718,
    -0.00667,
    0.02852,
    0.01627,
    -0.02264,
    0.00087,
    -0.01639,
    0.03154,
    0.01374,
    0.01203,
    -0.0039,
    0.04169,
    0.01707,
    -0.02454,
    -0.0016,
    0.0057,
    0.00085,
    0.00121,
    0.0105,
    0.01597,
    -0.01402,
    0.01156,
    0.01262,
    0.03364,
    0.04185,
    -0.00983,
    0.03444,
    0.00853,
    0.00529,
    -0.02383,
    -0.02173,
    -0.03299,
    -0.03329,
    0.01187,
    0.01761,
    0.00465,
    -0.0687,
    -0.00894,
    0.02249,
    -0.00985,
    -0.03859,
    -0.0612,
    -0.00036,
    -0.03491,
    0.03343,
    0.03306,
    -0.01101,
    0.00279,
    -0.01127,
    -0.02202,
    0.03943,
    0.01097,
    -0.00817,
    0.02721,
    -0.00951,
    -0.00962,
    0.01165,
    -0.00233,
    0.00377,
    -0.03038,
    -0.01756,
    -0.00838,
    0.02515,
    0.02018,
    -0.03082,
    -0.00333,
    0.00983,
    -0.05929,
    -0.0315,
    -0.00732,
    -0.01719,
    -0.01139,
    0.01128,
    -0.00622,
    -0.02174,
    0.01689,
    -0.02202,
    -0.07034,
    -0.02556,
    -0.02951,
    -0.04447,
    -0.01169,
    -0.02165,
    -0.01873,
    -0.00348,
    -0.01283,
    -0.05339,
    -0.01314,
    0.00744,
    0.01555,
    -0.0098,
    -0.00623,
    -0.00733,
    0.00961,
    -0.04965,
    -0.01599,
    -0.01495,
    -0.00596,
    -0.01773,
    -0.00139,
    0.01104,
    -0.01281,
    0.01198,
    -0.01313,
    -0.02408,
    0.0193,
    -0.01323,
    0.03586,
    0.00535,
    0.01548,
    0.07351,
    0.00606,
    -0.01671,
    -0.02359,
    -0.02263,
    -0.00481,
    0.01602,
    -0.01728,
    0.01893,
    -0.04132,
    0.02549,
    -0.00877,
    0.00658,
    -0.0462,
    -0.07506,
    0.02576,
    0.01498,
    0.009,
    0.03998,
    -0.01171,
    0.00145,
    -0.00273,
    -0.00657,
    -0.03532,
    0.03648,
    -0.03133,
    -0.04062,
    0.00142,
    0.01347,
    -0.01284,
    -0.01955,
    0.03667,
    0.0108,
    0.0027,
    0.06202,
    -0.00612,
    0.00149,
    0.01557,
    0.01284,
    -0.0005,
    0.03286,
    0.00922,
    -0.02243,
    0.02593,
    -0.03016,
    -0.00545,
    -0.02168,
    -0.02718,
    0.00041,
    -0.02387,
    -0.00936,
    0.02733,
    0.00926,
    -0.01183,
    -0.00385,
    -0.01131,
    0.01977,
    0.00521,
    0.00305,
    -0.0274,
    -0.00685,
    0.03421,
    0.02154,
    -0.01015,
    0.00061,
    0.03802,
    0.0211,
    -0.0043,
    -0.03701,
    -0.01394,
    -0.00553,
    0.00303,
    -0.00467,
    -0.03287,
    -0.0169,
    -0.0006,
    0.00257,
    0.00412,
    0.00468,
    0.04007,
    -0.0126,
    -0.00595,
    -0.03673,
    -0.01008,
    0.01328,
    0.00705,
    -0.01545,
    -0.01921,
    0.01547,
    -0.01358,
    0.01531,
    0.00294,
    0.02347,
    0.01188,
    0.03256,
    -0.00932,
    -0.00878,
    -0.03837,
    -0.01057,
    0.02563,
    -0.02082,
    -0.04447,
    -0.00131,
    -0.01839,
    0.04163,
    -0.0378,
    -0.00403,
    -0.00197,
    0.0018,
    -0.02014,
    0.00573,
    0.00982,
    -0.02005,
    -0.00055,
    0.01606,
    -0.0067,
    -0.00034,
    -0.03939,
    -0.00338,
    -0.00997,
    -0.02921,
    0.02682,
    -0.00816,
    -0.00371,
    0.02351,
    0.04555,
    -0.02963,
    0.01492,
    -0.03295,
    -0.00348,
    -0.00686,
    5e-05,
    0.02216,
    0.01038,
    -0.01095,
    0.00141,
    0.01051,
    0.04226,
    -0.02631,
    0.01162,
    -0.05315,
    0.01639,
    -0.05113,
    -0.00901,
    -2e-05,
    -0.01208,
    0.03287,
    -0.00056,
    0.0012,
    0.02569,
    -0.00797,
    0.00168,
    0.00131,
    -0.03108,
    -0.01337,
    0.01358,
    -0.00515,
    -0.01068,
    0.00618,
    -0.02642,
    -0.02479,
    -0.05763,
    0.0102,
    -0.01467,
    -0.0402,
    -0.01183,
    -0.00403,
    -0.06268,
    0.04858,
    -0.0358,
    -0.01926,
    0.01237,
    0.09275,
    0.01502,
    0.04803,
    -0.03351,
    -0.00191,
    0.02449,
    -0.02872,
    0.03524,
    -0.0307,
    -0.00441,
    -0.01803,
    -0.01029,
    0.00936,
    -0.0261,
    0.03974,
    -0.02863,
    0.0125,
    0.00422,
    -0.00066,
    -0.00795,
    0.05261,
    0.02959,
    -0.00438,
    0.01025,
    0.02774,
    0.00962,
    -0.01963,
    0.01655,
    0.00116,
    0.01575,
    -0.0694,
    -0.04753,
    0.02487,
    0.0069,
    0.03261,
    0.00195,
    -0.00822,
    -0.03962,
    0.00656,
    0.00766,
    0.00307,
    -0.03549,
    0.01712,
    -0.00182,
    -0.00609,
    -0.02765,
    0.00191,
    0.00902,
    0.00354,
    -0.02572,
    0.00773,
    -0.01393,
    -0.01387,
    -0.02129,
    -0.02501,
    -0.01802,
    -0.04453,
    0.01412,
    0.01967,
    0.03031,
    0.01778,
    0.0215,
    -0.02384,
    0.00041,
    0.01052,
    0.00359,
    0.00032,
    -0.01879,
    0.01487,
    -0.01782,
    -0.00863,
    -0.02569,
    0.00931,
    0.03748,
    -0.02307,
    -0.03164,
    0.01487,
    0.0047,
    -0.03158,
    -0.00421,
    -0.00056,
    0.04669,
    0.02741,
    0.00449,
    -0.00376,
    0.00902,
    0.00533,
    -0.02292,
    -0.00016,
    0.02341,
    -0.0147,
    0.01413,
    0.00799,
    0.03752,
    -0.01745,
    -0.01329,
    0.0074,
    -0.02079,
    0.00594,
    0.00086,
    -0.02254,
    -0.0131,
    0.00565,
    -0.01657,
    0.01367,
    -0.00227,
    0.00746,
    -0.01746,
    -0.0065,
    0.00208,
    -0.02067,
    0.00688,
    -0.02046,
    0.01305,
    -0.00664,
    -0.02651,
    0.00348,
    0.01327,
    -0.03149,
    -0.03586,
    0.0272,
    0.01716,
    0.02866,
    -0.0155,
    0.0082,
    0.02875,
    -0.00471,
    -0.08793,
    0.03354,
    -0.00913,
    -0.00071,
    -0.00872,
    -0.02198,
    -0.0024,
    0.01757,
    0.02547,
    -0.01328,
    0.03907,
    -0.01582,
    -0.00198,
    -0.00893,
    0.03283,
    0.00877,
    -0.01675,
    0.04981,
    0.01963,
    -0.04068,
    -0.03899,
    -0.00298,
    0.00018,
    0.01006,
    -0.007,
    -0.01129,
    -0.02263,
    0.02364,
    0.01033,
    -0.04156,
    0.0129,
    -0.0134,
    -0.04084,
    -0.0249,
    0.00052,
    0.00618,
    0.00108,
    0.00786,
    -0.01227,
    0.0248,
    0.02438,
    -0.02217,
    -0.00535,
    0.02226,
    0.00955,
    0.00543,
    0.02447,
    0.01281,
    0.02128,
    -0.04052,
    -0.00812,
    0.00203,
    -0.03206,
    0.03817,
    -0.03276,
    -0.00925,
    -0.01183,
    -0.02721,
    -0.01014,
    -0.04753,
    0.02061,
    0.02516,
    -0.01556,
    0.00077,
    0.04671,
    -0.01283,
    -0.0165,
    -0.01778,
    0.02926,
    -0.00854,
    0.04963,
    0.00659,
    -0.00146,
    0.03965,
    -0.01369,
    -0.02945,
    -0.03352,
    0.03504,
    0.00743,
    -0.03273,
    0.04513,
    0.00743,
    0.0149,
    -0.06643,
    -0.02816,
    -0.10354,
    -0.01232,
    -0.02779,
    0.00602,
    0.00902,
    0.03352,
    0.00136,
    0.01449,
    -0.02139,
    -0.00135,
    0.11011,
    -0.04402,
    0.02209,
    -0.03221,
    -0.00818,
    0.00296,
    -0.03731,
    -0.01372,
    -0.00766,
    -8e-05,
    0.00389,
    0.02921,
    0.0039,
    0.00555,
    -0.0227,
    0.01869,
    0.00723,
    0.00986,
    -0.01338,
    0.00328,
    -0.00147,
    0.02534,
    -0.00594,
    0.0037,
    0.02296,
    0.01418,
    -0.01369,
    0.0177,
    -0.03209,
    0.01261,
    0.0168,
    -0.03105,
    -0.0105,
    -0.04772,
    0.0009,
    0.00601,
    0.03705,
    0.01793,
    -0.01823,
    0.00155,
    -0.00777,
    0.01097,
    0.00848,
    -0.01042,
    -0.0062,
    -0.00603,
    0.00327,
    -0.00066,
    0.01624,
    0.02566,
    0.01184,
    -0.02277,
    -0.02967,
    0.00527,
    -0.03648,
    -0.02738,
    -0.01655,
    0.04569,
    -0.02118,
    0.00049,
    0.02537,
    0.02797,
    0.01608,
    0.03256,
    0.03127,
    0.01919,
    0.0287,
    0.00333,
    0.00127,
    -0.01578,
    0.01152,
    -0.00713,
    0.04264,
    0.04045,
    0.00435,
    -0.03063,
    0.06978,
    0.01149,
    -0.0152,
    0.00437,
    0.06994,
    -0.03976,
    0.00148,
    -0.02241,
    -0.00723,
    -0.02044,
    -7e-05,
    -0.00034,
    0.00488,
    0.02202,
    -0.0208,
    0.05129,
    -0.03045,
    -0.00289,
    0.02805,
    -0.0247,
    0.05453,
    -0.02658,
    -0.00564,
    0.02232,
    0.01277,
    0.0004,
    0.00586,
    0.00094,
    0.0583,
    0.00915,
    0.00678,
    -0.01612,
    0.02185,
    0.0173,
    0.01396,
    -0.01674,
    0.03513,
    -0.02268,
    0.03057,
    0.03194,
    0.00251,
    0.03821,
    -0.02361,
    0.04033,
    0.00725,
    0.0124,
    0.01826,
    -0.02795,
    -0.00767,
    0.01158,
    0.11723,
    -0.01704,
    0.00076,
    0.24148,
    -0.01529,
    0.05642,
    0.0236,
    0.00052,
    0.00707,
    0.00406,
    -0.00432,
    0.02037,
    0.02369,
    0.02285,
    -0.01004,
    -0.00365,
    -0.02738,
    0.00445,
    0.04654,
    0.00027,
    3e-05,
    -0.01964,
    -0.00747,
    -0.02343,
    0.01193,
    0.01921,
    -0.02642,
    -0.03995,
    0.00918,
    -0.00554,
    -0.0232,
    0.00375,
    0.02531,
    0.00576,
    -0.04164,
    -0.0243,
    0.0007,
    -0.00798,
    0.00641,
    -0.01103,
    0.0088,
    -0.00562,
    0.01038,
    0.02561,
    -0.0189,
    -0.0206,
    -0.01038,
    -0.00247,
    0.00062,
    0.01905,
    -0.04562,
    -0.01423,
    -0.0092,
    -0.01601,
    -0.00373,
    -1e-05,
    0.0036,
    -0.01445,
    -0.02704,
    -0.02075,
    0.017,
    -0.01417,
    0.03469,
    0.03466,
    -0.03943,
    -0.02391,
    -0.0253,
    -0.01596,
    -0.06096,
    -0.03887,
    -0.02219,
    -0.00163,
    0.02774,
    -0.00685,
    -0.03383,
    0.00561,
    -0.0325,
    -0.00994,
    0.0595,
    -0.02741,
    -0.0188,
    0.01771,
    -0.02734,
    0.01441,
    -0.0159,
    0.00321,
    -0.00881,
    0.00424,
    -0.01445,
    0.02812,
    0.00334,
    -0.01507,
    -0.03461,
    -0.04376,
    -0.02874,
    0.01305,
    -0.02626,
    0.0328,
    -0.01551,
    0.01375,
    0.02752,
    0.02248,
    0.00436,
    -0.06237,
    -0.07187,
    0.0287,
    -0.00407,
    0.01046,
    0.02044,
    -0.03326,
    0.02141,
    -0.00781,
    -0.03112,
    0.0192,
    -0.00746
   ],
   "MEDITATE": [
    0.03066,
    0.00464,
    0.02999,
    0.01965,
    -0.04703,
    0.09001,
    0.01796,
    -0.02604,
    -0.01848,
    -0.02491,
    -0.01963,
    0.03187,
    -0.00088,
    -0.01172,
    0.00126,
    0.00437,
    -0.04067,
    -0.00723,
    -0.00317,
    0.01385,
    0.00934,
    -0.01976,
    0.0027,
    -0.03901,
    0.0219,
    0.02394,
    0.0009,
    -0.01148,
    -0.01766,
    0.00672,
    -0.01429,
    0.02332,
    0.03486,
    0.04227,
    0.00501,
    0.0137,
    0.02057,
    -0.02076,
    -0.05156,
    0.0401,
    -0.00827,
    0.03553,
    -0.02574,
    0.02381,
    0.01774,
    -0.0143,
    0.03775,
    0.02915,
    -0.00299,
    -0.00476,
    -0.05451,
    -0.01204,
    -0.01133,
    0.01951,
    -0.02507,
    -0.00209,
    0.01151,
    0.05843,
    -0.02698,
    -0.04694,
    -0.00034,
    -0.00145,
    -0.01726,
    -0.02129,
    0.024,
    -0.00774,
    0.01034,
    -0.04216,
    -0.01838,
    -0.0133,
    -0.01903,
    -0.02553,
    0.02606,
    0.01589,
    -0.00154,
    -0.01112,
    -0.04522,
    0.01085,
    0.00148,
    0.01349,
    -0.022,
    -0.03916,
    0.00803,
    0.00756,
    0.00325,
    0.00207,
    0.00301,
    -0.0246,
    -0.0,
    0.00106,
    0.01875,
    0.02879,
    -0.04044,
    0.00414,
    -0.00645,
    -0.03944,
    0.01622,
    0.00156,
    -0.02596,
    0.01475,
    0.00349,
    -0.04515,
    0.06377,
    0.01466,
    -0.0256,
    -0.03889,
    -0.02416,
    -0.01936,
    0.01495,
    0.01944,
    0.01926,
    -0.00106,
    -0.00638,
    -0.04883,
    0.00501,
    -0.03386,
    -0.00778,
    -0.00718,
    -0.03227,
    -0.01055,
    0.0244,
    -0.00412,
    -0.02837,
    0.01899,
    0.02863,
    0.004,
    -0.00415,
    -0.02695,
    -0.01989,
    -1e-05,
    0.03877,
    0.01345,
    0.01914,
    0.01468,
    0.0079,
    0.01268,
    0.00441,
    0.00453,
    0.04969,
    -0.01524,
    -0.00533,
    -0.02039,
    0.01069,
    0.01403,
    -0.00133,
    0.03441,
    -0.00258,
    0.02499,
    -0.01578,
    0.04839,
    -0.0195,
    -0.00274,
    0.02702,
    0.01992,
    -0.01838,
    0.01067,
    -0.03577,
    0.00187,
    0.03441,
    -0.00505,
    -0.02943,
    0.02986,
    -0.00193,
    0.03069,
    0.00081,
    0.0078,
    -0.00722,
    -0.00431,
    -0.02854,
    0.00396,
    0.00873,
    0.00199,
    -0.02869,
    -3e-05,
    0.03064,
    0.01642,
    -0.009,
    0.00111,
    -0.01506,
    0.02677,
    0.00376,
    -0.00196,
    -0.01022,
    -0.00179,
    -0.02676,
    -0.03694,
    0.03383,
    0.03231,
    0.0319,
    0.02977,
    0.00385,
    0.04065,
    0.02662,
    -0.02527,
    0.00839,
    -0.00386,
    -0.02146,
    0.03725,
    -0.00371,
    -0.00895,
    -0.06705,
    -0.00765,
    0.02214,
    -0.02328,
    -0.02192,
    0.01748,
    0.02076,
    -0.01611,
    -0.01101,
    0.01544,
    0.00726,
    0.0108,
    0.01889,
    -0.04563,
    0.01308,
    -0.00796,
    0.00069,
    -0.00762,
    -0.0109,
    0.01139,
    0.01645,
    -0.02996,
    -0.03556,
    -0.02326,
    0.03,
    0.01389,
    -0.00397,
    0.02013,
    0.00941,
    -0.00938,
    -0.04031,
    0.00517,
    -0.05158,
    -0.02238,
    -0.00765,
    0.02708,
    0.01538,
    0.00733,
    -0.01926,
    0.00702,
    0.02251,
    0.00701,
    0.01931,
    0.00912,
    0.03109,
    0.01301,
    0.00507,
    -0.0039,
    -0.02681,
    -0.03192,
    -0.00081,
    -0.00737,
    -0.01882,
    -0.03896,
    -0.01674,
    0.00649,
    0.04768,
    0.0122,
    0.02919,
    0.00382,
    0.01603,
    0.00933,
    -0.01385,
    -0.01,
    0.01529,
    -0.01011,
    0.01601,
    0.00291,
    0.01422,
    -0.03412,
    -0.0175,
    -0.02073,
    0.01584,
    -0.029,
    0.00638,
    -0.01873,
    0.03605,
    0.02249,
    0.01774,
    0.00737,
    -0.01455,
    -0.01491,
    0.0253,
    -0.02097,
    0.05187,
    -0.00204,
    -0.04335,
    -0.02788,
    0.00937,
    -0.00061,
    -0.02374,
    -0.00443,
    0.00546,
    -0.00368,
    -0.01425,
    0.03257,
    -0.00636,
    -0.0229,
    -0.02316,
    0.01512,
    -0.01747,
    -0.06577,
    0.00385,
    -0.00334,
    -0.01857,
    -0.00885,
    0.02723,
    -0.02314,
    -0.02858,
    -0.02445,
    -0.01134,
    0.00037,
    -0.00368,
    -0.03722,
    -0.00707,
    0.02976,
    0.04946,
    -0.01395,
    -0.01933,
    -0.03356,
    0.00712,
    0.01715,
    -0.00592,
    -0.01948,
    0.00971,
    -0.00923,
    0.02522,
    0.02401,
    0.00212,
    0.01699,
    -0.00195,
    -0.01694,
    0.03142,
    0.01536,
    -0.01099,
    0.01126,
    0.04228,
    0.00939,
    -0.0277,
    0.02278,
    0.01374,
    -0.01983,
    0.00212,
    0.01162,
    0.01202,
    -0.01619,
    0.00766,
    -0.02097,
    -0.04968,
    -0.01433,
    -0.0604,
    0.00333,
    -0.01019,
    -0.00164,
    -0.05177,
    -0.00338,
    -0.06566,
    -0.01422,
    -0.02039,
    -0.05102,
    -0.00734,
    -0.00308,
    -0.02306,
    0.02112,
    -0.00867,
    0.05013,
    -0.00253,
    -0.00371,
    -0.00726,
    -0.03055,
    -0.00626,
    0.05571,
    -0.02119,
    -0.0187,
    0.0233,
    0.02574,
    0.01947,
    0.00327,
    -0.00201,
    0.01965,
    0.00741,
    -0.00297,
    -0.00017,
    0.01458,
    -0.02585,
    -0.00424,
    0.03854,
    0.00015,
    0.02424,
    -0.00452,
    0.01745,
    0.00011,
    -0.00815,
    0.00914,
    -0.05638,
    0.02877,
    0.01813,
    0.07174,
    -0.0424,
    -0.0016,
    -0.00193,
    0.00516,
    0.01535,
    -0.03442,
    -0.00448,
    0.00388,
    0.00644,
    -0.00955,
    -0.04084,
    -0.0403,
    0.0346,
    -0.03181,
    -0.01885,
    0.00195,
    -0.00457,
    0.02354,
    -0.00819,
    -0.0139,
    0.00369,
    -0.00018,
    0.01195,
    0.01253,
    -0.03426,
    -0.02849,
    -0.0712,
    0.00563,
    0.02926,
    -0.00334,
    0.0228,
    0.01791,
    0.03106,
    0.03111,
    0.01467,
    0.00956,
    -0.04798,
    0.04805,
    -0.02482,
    0.06066,
    -0.00684,
    0.0188,
    -0.00231,
    -0.03239,
    -0.06254,
    -0.03274,
    -0.02845,
    0.00129,
    -0.00162,
    -0.00137,
    0.00515,
    0.01237,
    0.00076,
    -0.01858,
    -0.0734,
    -0.01146,
    -0.03767,
    0.03726,
    -0.01335,
    -0.0099,
    -0.00332,
    -0.0281,
    -0.02196,
    0.01942,
    -0.01141,
    -0.02431,
    0.06109,
    0.00428,
    0.01397,
    -0.05301,
    0.03449,
    0.05989,
    -0.00054,
    0.0459,
    -0.04686,
    -0.02432,
    0.00393,
    0.01999,
    0.00149,
    -0.04153,
    0.06342,
    0.00099,
    -0.00189,
    0.01259,
    0.01283,
    0.00507,
    -0.01408,
    0.00188,
    0.03452,
    -0.0144,
    -0.00942,
    0.00418,
    0.00036,
    0.01142,
    -0.00618,
    0.01027,
    -0.00878,
    -0.02,
    0.03675,
    0.0068,
    0.03819,
    0.0365,
    -0.04237,
    -0.02827,
    -0.00983,
    -0.03198,
    -0.00809,
    -0.04365,
    -0.00972,
    0.01328,
    0.01933,
    0.00998,
    0.07813,
    -0.00592,
    -0.00599,
    -0.00331,
    -0.03716,
    0.04026,
    -0.01092,
    -0.00634,
    0.05261,
    -0.03157,
    0.08406,
    0.01716,
    -0.01002,
    0.00355,
    0.01793,
    -0.02539,
    0.00683,
    -0.01647,
    -0.0042,
    0.04541,
    -0.03052,
    -0.02971,
    -0.00795,
    0.03938,
    0.02097,
    0.04837,
    -0.00866,
    -0.02262,
    0.00039,
    -0.00919,
    0.00196,
    -0.01794,
    0.02303,
    -0.04639,
    -0.01269,
    -0.01903,
    0.02952,
    0.01411,
    0.03636,
    -0.01665,
    0.06642,
    0.03221,
    0.02293,
    0.03697,
    0.00023,
    -0.01163,
    0.00058,
    0.00236,
    -0.02695,
    -0.00262,
    -0.02737,
    -0.01348,
    0.00256,
    0.01603,
    -0.00491,
    -0.00576,
    -0.02028,
    -0.00931,
    -0.00687,
    -0.00562,
    -0.01872,
    -0.00696,
    -0.02836,
    -0.00812,
    0.02664,
    0.05138,
    0.02334,
    -0.0116,
    -0.0389,
    -0.01286,
    0.02877,
    -0.01446,
    -0.0197,
    0.01687,
    0.00549,
    -0.00093,
    0.00962,
    0.01165,
    0.00248,
    -0.04232,
    -0.01899,
    0.03136,
    0.02326,
    0.01804,
    -0.04958,
    0.00215,
    -0.01628,
    0.006,
    -0.0129,
    0.00781,
    -0.02882,
    0.0015,
    -0.03398,
    0.06491,
    -0.02859,
    -0.00351,
    0.03516,
    0.01362,
    -0.03265,
    0.03712,
    -0.00585,
    0.03094,
    0.00422,
    0.01878,
    0.01193,
    -0.00989,
    0.04869,
    0.0027,
    -0.00305,
    -0.00867,
    0.00472,
    0.01698,
    0.01724,
    0.00333,
    -0.01005,
    0.00551,
    -0.02284,
    0.00157,
    0.01842,
    -0.02802,
    0.01159,
    -0.01323,
    0.05504,
    -0.03575,
    -0.00434,
    -0.03148,
    -0.00906,
    -0.04059,
    0.022,
    0.00559,
    -0.02906,
    0.01085,
    -0.01491,
    0.01084,
    0.01512,
    -0.04482,
    -0.02206,
    -0.01775,
    0.00893,
    -0.03987,
    -0.01315,
    -0.0058,
    -0.03137,
    -0.01561,
    0.0302,
    0.03652,
    -0.00299,
    0.00212,
    0.00076,
    0.00482,
    0.01999,
    0.04523,
    -0.02474,
    -0.03732,
    -0.02221,
    0.00564,
    -0.0238,
    0.01274,
    0.01202,
    -0.04347,
    0.01184,
    -0.03419,
    1e-05,
    -0.0317,
    0.01077,
    0.01792,
    -0.03068,
    -0.01881,
    0.00183,
    -0.02111,
    0.01882,
    -0.01192,
    0.02224,
    0.02904,
    0.00582,
    0.01263,
    0.00121,
    -0.02821,
    0.01687,
    0.01188,
    -0.02056,
    -0.0373,
    0.0007,
    0.00288,
    -0.00029,
    -0.01358,
    -0.02779,
    -0.00308,
    0.00032,
    0.01346,
    -0.00536,
    0.00987,
    0.00895,
    0.0343,
    -0.02102,
    -0.01114,
    -0.00892,
    0.01786,
    0.02257,
    -0.02651,
    0.03579,
    0.00189,
    0.0118,
    0.01523,
    -0.06701,
    0.00771,
    -0.00381,
    -0.0022,
    -0.06753,
    0.02733,
    -0.00837,
    0.0274,
    -0.0361,
    0.04295,
    -0.01113,
    -0.01532,
    -0.01374,
    -0.00843,
    -0.00869,
    0.01297,
    0.01824,
    0.00642,
    -0.01365,
    -0.02806,
    -0.00487,
    -0.00377,
    0.04942,
    0.01132,
    0.00199,
    -0.01574,
    -0.00806,
    0.01396,
    0.01145,
    -0.04668,
    0.02625,
    0.01222,
    -0.01669,
    -0.00861,
    0.01754,
    0.00868,
    0.02364,
    -0.00193,
    0.01159,
    0.00314,
    -0.03134,
    0.00357,
    -0.01593,
    0.0227,
    0.02869,
    0.0257,
    0.00062,
    -0.0298,
    0.02497,
    -0.00678,
    -0.03165,
    -0.02309,
    0.00414,
    -0.02382,
    0.00931,
    0.01988,
    -0.01694,
    0.00461,
    -0.00837,
    0.00308,
    -0.00052,
    0.02495,
    -0.03733,
    0.04756,
    -0.03573,
    0.03736,
    0.01911,
    0.00742,
    -0.01527,
    -0.00375,
    -0.023,
    -0.02533,
    -0.01053,
    0.00807,
    -0.0297,
    0.00351,
    -0.02765,
    -0.02334,
    0.02234,
    -0.06163,
    -0.0258,
    0.0236,
    0.00733,
    -0.01997,
    0.01523,
    -0.03551,
    -0.0064,
    -0.02059,
    -0.03859,
    -0.00281,
    -0.02278,
    -0.02723,
    -0.06663,
    -0.01586,
    -0.02076,
    0.02115,
    -0.00999,
    -0.04476,
    -0.03137,
    0.02165,
    0.00923,
    0.0116,
    0.00023,
    0.02311,
    0.0114,
    0.00109,
    -0.01812,
    -0.02372,
    0.0233,
    0.00909,
    0.0037,
    -0.00929,
    0.00606,
    0.02516,
    0.00863,
    0.01675,
    0.01668,
    0.04696,
    0.00118,
    -0.01024,
    -0.07139,
    0.04189,
    -0.02825,
    0.00794,
    -0.01391,
    -0.00445,
    0.01668,
    -0.00368,
    -0.03453,
    -0.00658,
    -0.07428,
    0.00433,
    -0.01639,
    0.03452,
    -0.00948,
    0.01543,
    -0.00656,
    0.02794,
    -0.00476,
    0.00485,
    -0.00152,
    -0.01235,
    0.01488,
    -0.0404,
    0.04538,
    0.00277,
    0.00349,
    0.03065,
    -0.02519,
    -0.01021,
    -0.03894,
    0.00912,
    -0.00578,
    -0.02695,
    0.04199,
    0.00184,
    -0.00985,
    0.01559,
    -0.0083,
    -0.01482,
    0.04213,
    0.02919,
    0.00508,
    0.00722,
    -0.02214,
    0.00029,
    0.06349,
    0.00027,
    -0.00293,
    0.02171,
    0.00668,
    0.00966,
    0.01837,
    0.02562,
    -0.00385,
    -0.02103,
    0.01669,
    -0.02913,
    0.02278,
    0.00113,
    -0.02044,
    0.01153,
    -0.05081,
    -0.01936,
    0.02793,
    -0.06095,
    -0.01091,
    0.04663,
    0.01489,
    -0.02249,
    0.10654,
    0.01874,
    0.01092,
    0.04376,
    0.03308,
    0.02243,
    -0.01565,
    -0.02474,
    0.00807,
    -0.00448,
    0.00154,
    0.02719,
    -0.00045,
    -0.03166,
    -0.00093,
    0.00484,
    0.00992,
    0.01646,
    -0.0189,
    -0.01429,
    0.01235,
    -0.00501,
    -0.07552,
    0.01042,
    0.02336,
    0.00651,
    -0.00025,
    -0.00499,
    0.05625,
    -0.00128,
    0.0011,
    -0.04429,
    -0.03069,
    -0.00388,
    0.04327,
    0.0098,
    0.03078,
    -0.02784,
    -0.02359,
    -0.06671,
    -0.01614,
    0.00367,
    0.03295,
    0.00173,
    -0.01932,
    -0.01061,
    -0.01274,
    0.0333,
    -0.00184,
    0.00585,
    0.02957,
    -0.01016,
    0.00454,
    -0.02532,
    0.05009,
    0.01697,
    0.01941,
    -0.01455,
    0.01534,
    -0.0083,
    0.00367,
    -0.01416,
    0.02567,
    -0.04752,
    0.00992,
    0.01045,
    -0.05158,
    -0.02549,
    0.03665,
    0.00716,
    -0.00061,
    -0.01753,
    0.00086,
    -0.01578,
    -0.00196,
    -0.06553,
    0.01426,
    0.00307,
    -0.03434,
    -0.05021,
    0.04429,
    -0.01346,
    -0.05158,
    0.01386,
    -0.00188,
    -0.01235,
    -0.02422,
    0.06734,
    0.00198,
    -0.01469,
    -0.03724,
    0.00205,
    0.00045,
    -0.03452,
    0.02157,
    0.016,
    0.01989,
    -0.01532,
    0.00348,
    -0.04078,
    -0.02711,
    0.05231,
    -0.04688,
    -0.03358,
    -0.02848,
    0.02037,
    0.01377,
    -0.02993,
    -0.00721,
    0.01675,
    0.00522,
    -0.03365,
    0.044,
    -0.02966,
    0.05195,
    -0.00419,
    -0.02836,
    0.00895,
    0.00021,
    -0.01453,
    0.011,
    -0.01462,
    0.01428,
    -0.00552,
    -0.0598,
    -0.00736,
    0.0174,
    0.02781,
    -0.00015,
    -0.04411,
    0.00739,
    -0.00398,
    0.01312,
    0.0072,
    -0.00909,
    -0.00555,
    -0.0516,
    0.03726,
    0.0168,
    -0.01302,
    0.02661,
    0.00461,
    0.00564,
    -0.02062,
    -0.00428,
    -0.01017,
    0.01002,
    0.04647,
    0.02215,
    -0.02238,
    -0.00767,
    0.02697,
    -0.01894,
    -0.01452,
    -0.01052,
    -0.00967,
    0.00736,
    -0.01121,
    0.00598,
    -0.03994,
    0.00623,
    0.04144,
    -0.0111,
    0.00849,
    0.10134,
    -0.01597,
    -0.02092,
    0.00641,
    -0.0022,
    -0.00855,
    -0.02718,
    -0.00434,
    0.00276,
    -0.01818,
    -0.01878,
    -0.0148,
    0.00739,
    0.00013,
    -0.01867,
    0.00818,
    0.02031,
    0.0046,
    0.04029,
    -0.01516,
    0.00112,
    0.02808,
    -0.04241,
    -0.01519,
    -0.01808,
    0.02352,
    0.03743,
    -0.00681,
    0.03919,
    0.01029,
    -0.00499,
    -0.02495,
    -0.01085,
    0.01242,
    0.00222,
    -0.00403,
    0.03941,
    0.02003,
    0.00011,
    0.00036,
    0.00154,
    -0.02673,
    0.01444,
    0.03728,
    0.00432,
    -0.00411,
    0.00807,
    0.00612,
    -0.009,
    0.01735,
    0.06102,
    -0.01406,
    -0.0027,
    -0.03665,
    0.03635,
    0.02266,
    -0.0047,
    0.02334,
    -0.00395,
    -0.01496,
    0.02379,
    -0.00243,
    -0.00374,
    0.01193,
    -0.00999,
    -0.03305,
    -0.04336,
    0.01547,
    -0.00162,
    -0.02042,
    0.00381,
    0.04271,
    -0.03153,
    0.01881,
    0.00162,
    0.04306,
    -0.01984,
    0.02087,
    -0.01998,
    0.02144,
    -0.00699,
    -0.05325,
    -0.01508,
    0.0084,
    -0.02718,
    -0.00494,
    0.00127,
    -0.02112,
    0.01057,
    -0.05854,
    -0.00172,
    0.01343,
    -0.03691,
    0.00467,
    -0.01154,
    -0.05679,
    -0.05581,
    -0.00303,
    -0.04863,
    0.0086,
    -0.00473,
    -0.01221,
    -0.00415,
    0.04046,
    -0.01637,
    0.02268,
    0.01164,
    0.03265,
    -0.02064,
    0.04394,
    -0.01391,
    -0.00595,
    -0.0186,
    -0.01261,
    -0.00608,
    -0.00027,
    0.0194,
    0.02423,
    -0.00677,
    -0.01711,
    0.02617,
    0.02818,
    0.02416,
    0.01101,
    0.02985,
    -0.03119,
    0.00576,
    0.01645,
    0.0382,
    0.0134,
    -0.01337,
    -0.02636,
    -0.03235,
    -0.03934,
    0.01658,
    -0.00818,
    0.00655,
    -0.03937,
    -0.00997,
    0.01374,
    0.0047,
    0.05879,
    0.02199,
    0.00018,
    -0.00878,
    0.04914,
    -0.03881,
    0.00623,
    -0.01287,
    0.09045,
    0.00367,
    -0.00282,
    0.00805,
    -0.01815,
    0.02559,
    -0.03062,
    0.03551,
    0.00536,
    0.02123,
    -0.00644,
    -0.01975,
    0.07228,
    0.01009,
    0.01587,
    0.0955,
    0.03672,
    0.0022,
    -0.03196,
    0.01345,
    0.03471,
    0.0376,
    0.03457,
    -0.00335,
    0.03492,
    0.01569,
    -0.01692,
    -0.02224,
    0.06028,
    0.05585,
    -0.00015,
    -0.01518,
    0.00205,
    0.00841,
    -0.03876,
    -0.06126,
    -0.03293,
    0.03259,
    0.00319,
    0.04474,
    0.04602,
    -0.0082,
    0.00572,
    -0.00604,
    -0.01279,
    -0.03285,
    0.01985,
    -0.0091,
    -0.01812,
    -0.02855,
    -0.00475,
    0.00713,
    -0.0296,
    -0.03334,
    -0.01769,
    0.0175,
    -0.07291,
    0.0215,
    0.04876,
    -0.00383,
    -0.03167,
    -0.00107,
    -0.00776,
    -0.00457,
    -0.0369,
    -0.05005,
    -0.06093,
    -0.00445,
    -0.006,
    0.02888,
    -0.01107,
    0.026,
    -0.0348,
    -0.00897,
    -0.01799,
    0.02492,
    0.0443,
    -0.0021,
    0.01823,
    0.01339,
    0.0254,
    -0.02869,
    -0.02948,
    -0.01425,
    0.02249,
    -0.00699,
    -0.03794,
    -0.00077,
    -0.0128,
    -0.01641,
    -0.01671,
    0.02137,
    -0.00422,
    -0.03026,
    -0.01623,
    0.00172,
    0.00799,
    0.01174,
    0.02468,
    -0.02676,
    0.0447,
    -0.00288,
    -0.02624,
    -0.00989,
    -0.02004,
    0.01321,
    -0.03657,
    0.02354,
    -0.00284,
    -0.00728,
    0.02246,
    -0.00027,
    -0.01857,
    -0.00429,
    0.01213,
    -0.03375,
    -0.00138,
    -0.0012,
    -0.00848,
    -0.02274,
    -0.00041,
    -0.02072,
    0.00955,
    -0.03674,
    -0.01813,
    0.03026,
    -0.01836,
    -0.02522,
    -0.00253,
    0.0064,
    0.01481,
    0.00251,
    -0.00476,
    -0.01089,
    -0.01398,
    0.00954,
    0.01225,
    0.01946,
    0.00151,
    -0.01183,
    -0.0279,
    0.01475,
    0.04151,
    -0.12958,
    0.0032,
    0.0227,
    -0.00938,
    -0.02753,
    -0.03495,
    -0.01026,
    0.04328,
    0.02642,
    0.00207,
    0.0263,
    0.00242,
    -0.02794,
    0.04645,
    0.01208,
    -0.00374,
    0.01117,
    -0.01614,
    -0.0108,
    0.00659,
    0.02014,
    -0.0237,
    0.00858,
    0.01179,
    -0.05186,
    -0.00189,
    -0.01565,
    -0.00726,
    -0.01777,
    0.00206,
    0.04931,
    0.01164,
    -0.0023,
    -0.00553,
    -0.00911,
    0.0621,
    -0.01033,
    -0.04067,
    0.02027,
    -0.04144,
    0.00678,
    0.00448,
    0.0073,
    -0.00464,
    0.00854,
    -0.01388,
    -0.01733,
    -0.02964,
    0.01117,
    -0.04521,
    0.0181,
    -0.02007,
    -0.01618,
    -0.01782,
    -0.00082,
    -0.01359,
    0.00107,
    -0.00326,
    -0.00497,
    0.01411,
    0.00485,
    -0.10431,
    -0.05,
    0.07146,
    -0.10168,
    0.016,
    -0.02971,
    -0.02004,
    -0.02176,
    -0.0046,
    0.0099,
    -0.02404,
    -0.02123,
    0.03225,
    -0.00211,
    -0.02692,
    -0.02375,
    0.01886,
    -0.01816,
    -0.01109,
    -0.00741,
    -0.02816,
    -0.03311,
    0.00069,
    0.01466,
    -0.00158,
    -0.00953,
    -0.0076,
    0.0172,
    0.03829,
    -0.03039,
    0.00095,
    -0.03949,
    -0.02328,
    -0.00389,
    0.00834,
    0.00511,
    0.01195,
    -0.01141,
    -0.00089,
    0.01343,
    0.00393,
    -0.00999,
    -0.04246,
    -0.01896,
    0.04499,
    -0.04189,
    -0.01329,
    0.01014,
    -0.00887,
    -0.00397,
    0.04452,
    0.00403,
    -0.00038,
    -0.00096,
    0.02432,
    -0.00308,
    -0.01209,
    0.02161,
    -0.01322,
    -0.02145,
    0.01093,
    -0.0166,
    0.01754,
    0.01452,
    0.00421,
    -0.00348,
    -0.01345,
    0.01612,
    0.0379,
    -0.00776,
    0.00026,
    0.03792,
    -0.01747,
    0.00388,
    0.0357,
    -0.01205,
    0.01676,
    0.03544,
    -0.02188,
    -0.0304,
    0.01964,
    0.03271,
    0.00783,
    -0.00731,
    0.0097,
    0.01703,
    0.03165,
    0.01055,
    -0.03093,
    -0.0075,
    0.00143,
    0.00183,
    -0.02607,
    0.00724,
    0.00467,
    -0.01108,
    -0.03671,
    -0.0424,
    -0.00564,
    0.02651,
    0.03581,
    0.01685,
    0.00919,
    0.00956,
    0.00135,
    -0.00072,
    -0.01739,
    0.00637,
    -0.02614,
    -0.04482,
    0.00907,
    -0.0424,
    -0.00125,
    0.01045,
    0.01623
   ],
   "INFUSION": [
    0.0143,
    -0.00303,
    0.03861,
    -0.00929,
    -0.03677,
    0.0003,
    -0.00939,
    -0.01466,
    -0.02408,
    0.01304,
    -0.01918,
    0.00115,
    0.00158,
    -0.03444,
    -0.00641,
    -0.02089,
    0.01971,
    0.00705,
    -0.00084,
    -0.05687,
    -0.00747,
    -0.0026,
    -0.00768,
    -0.01679,
    7e-05,
    0.01473,
    -0.00774,
    -0.02024,
    0.07062,
    -0.02768,
    0.00149,
    0.02682,
    -0.03613,
    0.00236,
    0.02159,
    -0.00133,
    -0.0051,
    0.04294,
    -0.01771,
    -0.01267,
    0.00114,
    -0.01683,
    0.01589,
    0.00638,
    -0.04555,
    0.03278,
    0.05684,
    -0.04683,
    -0.0346,
    0.02489,
    0.02371,
    0.0029,
    0.01792,
    -0.01927,
    0.03886,
    0.0223,
    0.02144,
    -0.02041,
    -0.05396,
    -0.05025,
    0.00514,
    -0.02996,
    0.03217,
    -0.00836,
    0.02005,
    -0.01075,
    0.02075,
    -0.03273,
    -0.00425,
    -0.00882,
    -0.00616,
    -0.00814,
    0.014,
    -0.05054,
    -0.01105,
    -0.0187,
    -0.00698,
    -0.01251,
    -0.02403,
    0.00299,
    0.01102,
    -0.01528,
    0.01226,
    0.00257,
    0.0213,
    -0.00836,
    0.00366,
    0.01384,
    -0.01518,
    0.01005,
    -0.00761,
    -0.00572,
    0.02157,
    -0.01851,
    0.02993,
    0.02145,
    -0.02182,
    0.01047,
    0.01406,
    -0.02822,
    -0.01173,
    -0.00841,
    -0.02253,
    -0.00603,
    -0.00092,
    0.01211,
    0.02763,
    -0.0065,
    -0.01989,
    -0.02839,
    -0.01816,
    0.0278,
    -0.03519,
    0.02503,
    0.02899,
    -0.02666,
    -0.00575,
    -0.00819,
    0.02286,
    0.0109,
    -0.00016,
    0.02481,
    0.02612,
    0.0108,
    0.04846,
    0.0281,
    -0.00617,
    -0.04555,
    0.01042,
    -0.00259,
    -0.08581,
    0.01874,
    -0.00594,
    0.01327,
    -0.05016,
    -0.00337,
    0.0055,
    -0.00093,
    0.00695,
    0.00772,
    0.0362,
    0.01839,
    -0.00231,
    -0.00842,
    0.02528,
    0.03817,
    0.00295,
    -0.02302,
    0.00528,
    -0.01573,
    0.02824,
    -0.0005,
    0.00416,
    -0.01379,
    -0.00023,
    -0.0051,
    0.03054,
    -0.00366,
    -0.03323,
    0.00511,
    0.05316,
    0.02753,
    0.00237,
    0.0461,
    -0.00744,
    -0.01636,
    0.03687,
    0.00624,
    0.03665,
    0.03676,
    0.00502,
    -0.00565,
    0.04315,
    -0.01908,
    -0.0093,
    0.01091,
    -0.03284,
    0.0019,
    -0.00102,
    0.00211,
    0.01352,
    0.03617,
    0.02723,
    -0.03355,
    0.01671,
    0.00916,
    -0.00841,
    -0.03027,
    -0.04867,
    -0.02194,
    0.00965,
    0.00518,
    -0.01933,
    0.0052,
    -0.05552,
    0.02938,
    0.04695,
    -0.02159,
    0.02673,
    0.01611,
    0.01269,
    0.02395,
    0.00512,
    0.01972,
    0.04257,
    -0.04647,
    -0.00946,
    0.01806,
    -0.0045,
    -0.03607,
    -0.00319,
    -0.00932,
    -0.00948,
    -0.00792,
    -0.01397,
    -0.01914,
    -0.00866,
    -0.00785,
    0.0065,
    -0.00968,
    -0.01458,
    -0.00303,
    0.01289,
    0.04226,
    -0.03857,
    -0.02318,
    0.00067,
    -0.02545,
    0.00406,
    -0.02402,
    0.07017,
    -0.01215,
    0.00758,
    0.03626,
    -0.01762,
    -0.01399,
    0.02125,
    0.02777,
    -0.02248,
    -0.03116,
    0.00974,
    -0.01055,
    0.03976,
    -0.04957,
    -0.00432,
    -0.00481,
    -0.02711,
    0.00405,
    0.02282,
    -0.00392,
    -0.00737,
    -0.04749,
    0.01354,
    0.0724,
    0.00612,
    -0.00955,
    -0.07866,
    -0.03499,
    0.00512,
    0.05128,
    0.02516,
    0.01842,
    -0.00513,
    0.04354,
    -0.02772,
    0.00427,
    -0.01236,
    8e-05,
    0.00642,
    -0.05226,
    -0.0386,
    -0.0383,
    0.04266,
    0.02139,
    0.01476,
    -0.00024,
    0.01439,
    -0.01028,
    -0.03052,
    0.00085,
    -0.00072,
    0.03266,
    0.0238,
    0.02273,
    -0.04487,
    0.00225,
    -0.00406,
    -0.01691,
    -0.00434,
    0.01783,
    0.01842,
    -0.02128,
    -0.01805,
    0.00652,
    -0.02614,
    -0.00159,
    0.04962,
    0.02918,
    0.01609,
    -0.02,
    0.03035,
    0.01423,
    0.01499,
    -0.01376,
    0.02596,
    0.03516,
    0.00529,
    0.00272,
    0.00676,
    -0.02303,
    0.0078,
    0.01627,
    0.00099,
    -0.00606,
    -0.00531,
    -0.02683,
    0.00414,
    -0.04017,
    0.00515,
    0.02057,
    -0.01376,
    -0.02414,
    -0.00399,
    0.00538,
    0.00514,
    -0.00581,
    -0.02863,
    0.02042,
    -0.01673,
    0.05305,
    -0.01006,
    0.00728,
    0.00707,
    -0.01681,
    0.01801,
    -0.04499,
    -0.01737,
    0.01554,
    0.01857,
    -0.00467,
    -0.0171,
    0.01086,
    -0.02754,
    -0.01555,
    -0.02766,
    0.01982,
    -0.01481,
    0.02184,
    0.01255,
    0.02567,
    0.00823,
    0.02612,
    0.02244,
    0.00957,
    0.09166,
    0.01179,
    0.04858,
    0.03084,
    0.02076,
    0.01579,
    0.06086,
    -0.03924,
    0.04738,
    0.02939,
    0.02256,
    -0.025,
    0.02924,
    -0.02847,
    -0.01713,
    0.06459,
    0.00524,
    -0.00456,
    0.02647,
    0.00904,
    -0.00739,
    0.00976,
    0.01207,
    -0.03361,
    0.01526,
    0.00653,
    0.00731,
    -0.00145,
    0.00387,
    0.03196,
    -0.0171,
    0.00547,
    -0.00068,
    0.02881,
    -0.01018,
    0.01836,
    0.00313,
    -0.01348,
    0.02416,
    -0.02174,
    -0.04237,
    -0.02827,
    -0.00411,
    -0.00977,
    0.01884,
    -0.03099,
    0.02972,
    -0.00229,
    0.00511,
    0.00066,
    -0.00327,
    -0.02622,
    0.01552,
    0.01458,
    0.03271,
    0.03639,
    0.02553,
    -0.00238,
    0.00155,
    0.01069,
    -0.00215,
    -0.00649,
    -0.02256,
    0.02577,
    -0.0473,
    -0.02507,
    0.02797,
    0.05943,
    0.01775,
    0.00518,
    0.07003,
    -0.00336,
    -0.01831,
    -0.01763,
    -0.01017,
    -0.00973,
    -0.00729,
    -0.01046,
    -0.0017,
    0.01065,
    0.00104,
    0.01571,
    -0.05262,
    0.02053,
    -0.00261,
    -0.01661,
    -0.02431,
    -0.06834,
    0.01118,
    0.04219,
    0.01575,
    -0.00777,
    -0.02828,
    -0.02213,
    -0.04305,
    -0.04749,
    0.01435,
    0.02739,
    0.08454,
    -0.0324,
    0.00995,
    0.00806,
    0.00168,
    -0.01002,
    -0.03463,
    0.01005,
    -0.03042,
    -0.03967,
    0.00832,
    -0.00145,
    -0.03267,
    0.01914,
    -0.02079,
    0.00421,
    0.00664,
    0.05874,
    -0.04155,
    -0.00183,
    -0.02895,
    0.04873,
    0.00113,
    -0.00391,
    0.01085,
    0.00577,
    -0.01015,
    -0.02591,
    -0.0,
    -0.03457,
    -0.00204,
    0.00122,
    0.01597,
    0.00897,
    0.00856,
    -0.04091,
    -0.00784,
    0.00223,
    0.0095,
    -0.02157,
    0.01476,
    0.00947,
    -0.03275,
    0.04023,
    0.01454,
    -0.01811,
    -0.02121,
    -0.03534,
    0.01517,
    0.08011,
    0.00742,
    0.02616,
    -0.00614,
    -0.02192,
    -0.00356,
    0.02458,
    -0.02259,
    0.00766,
    -0.03892,
    0.02736,
    0.0274,
    0.02795,
    -0.00334,
    -0.0009,
    -0.00916,
    0.02213,
    -0.02632,
    -0.00283,
    -0.05006,
    -0.01932,
    0.02806,
    -0.0015,
    0.02225,
    -0.00086,
    -0.00798,
    -0.00988,
    0.01591,
    -0.00416,
    0.02304,
    0.03048,
    -0.05957,
    -0.00978,
    0.00082,
    0.02157,
    0.00996,
    -0.01905,
    0.00174,
    -0.01909,
    0.01919,
    -0.01666,
    0.01639,
    0.00395,
    -0.01375,
    -0.00926,
    0.03078,
    -0.01386,
    0.02876,
    0.00277,
    -0.0307,
    0.00732,
    -0.02821,
    0.06987,
    -0.0463,
    0.01811,
    -0.02135,
    -0.02567,
    0.01026,
    -0.03936,
    -0.00175,
    0.04454,
    -0.02321,
    0.01142,
    0.0066,
    -0.05373,
    -0.03035,
    -0.01364,
    0.02322,
    0.01084,
    0.05093,
    -0.01354,
    -0.01446,
    -0.01153,
    0.05236,
    0.00139,
    0.00024,
    0.03174,
    0.00165,
    0.01334,
    -0.05716,
    -0.00888,
    -0.01981,
    0.00261,
    0.01079,
    -0.02629,
    -0.01042,
    -0.00103,
    -0.01587,
    0.01343,
    0.00516,
    -0.00543,
    -0.00084,
    0.00319,
    -0.01355,
    0.01411,
    -0.01109,
    0.00274,
    0.02653,
    -0.00119,
    0.00915,
    -0.03813,
    0.02674,
    -0.04302,
    0.03631,
    -0.00089,
    0.00186,
    0.04645,
    0.0517,
    0.09236,
    0.0073,
    0.01323,
    0.01027,
    -0.00069,
    -0.00989,
    -0.01274,
    -0.02241,
    0.00019,
    0.00395,
    -0.02064,
    -0.0208,
    -0.00969,
    -0.01363,
    -0.03841,
    -0.01003,
    -0.04704,
    0.01929,
    0.02153,
    0.03499,
    0.05369,
    -0.00947,
    0.0203,
    0.00807,
    -0.01544,
    -0.00496,
    -0.04518,
    -0.01423,
    -0.01626,
    0.00366,
    0.01308,
    -0.01333,
    -0.02847,
    -0.00062,
    -0.01546,
    0.01714,
    0.01153,
    -0.02723,
    0.02387,
    -0.00058,
    -0.01309,
    0.00333,
    -0.0055,
    0.01168,
    -0.01469,
    -0.01447,
    -0.03012,
    -0.01884,
    -0.04153,
    0.04466,
    -0.00673,
    -0.07039,
    -0.03483,
    -1e-05,
    -0.01114,
    -0.00623,
    0.01354,
    0.03709,
    -0.00875,
    -0.01335,
    0.02674,
    0.01586,
    0.00214,
    -0.03437,
    0.05153,
    -0.02844,
    0.01665,
    -0.0029,
    -0.00079,
    -0.00584,
    -0.00354,
    0.01017,
    0.01465,
    0.0,
    -0.03968,
    0.02548,
    -0.04045,
    -0.00298,
    0.01519,
    0.03822,
    -0.02004,
    0.022,
    -0.02413,
    0.0158,
    0.0034,
    -0.03561,
    0.01876,
    0.01304,
    -0.01392,
    -0.02511,
    0.00849,
    0.01569,
    -0.05244,
    0.02752,
    0.00193,
    0.00809,
    0.0112,
    0.00455,
    0.00045,
    -0.01032,
    0.00188,
    0.01448,
    0.00439,
    -0.0102,
    -0.0539,
    -0.03083,
    -0.02629,
    0.00635,
    -0.02999,
    -0.01933,
    -0.00964,
    -0.0302,
    0.01262,
    0.01679,
    -0.02617,
    -0.00225,
    -0.01027,
    -0.03299,
    0.00923,
    0.00298,
    -0.0001,
    0.0034,
    -0.01244,
    -0.03205,
    -0.02193,
    0.05004,
    -0.007,
    -0.01322,
    0.01366,
    0.03003,
    -0.02488,
    0.0245,
    0.00017,
    0.01562,
    -0.0291,
    0.00135,
    -0.02193,
    0.02708,
    0.01922,
    -0.01803,
    -0.00923,
    0.01261,
    0.0087,
    -0.04155,
    -0.01668,
    -0.01865,
    -0.02074,
    -0.00463,
    -0.02399,
    -0.01411,
    -0.01033,
    0.04486,
    0.03922,
    -0.01999,
    -0.00591,
    -0.01397,
    0.01681,
    -0.0148,
    -0.00221,
    -0.02664,
    -0.03219,
    0.02537,
    0.03152,
    0.01859,
    0.01025,
    0.01055,
    0.01786,
    0.02826,
    -0.00613,
    0.03391,
    0.00388,
    0.00414,
    -0.01404,
    0.02256,
    0.02627,
    0.10239,
    0.03248,
    -0.02251,
    0.00017,
    0.00568,
    -0.0299,
    0.02667,
    0.03445,
    0.00966,
    0.04705,
    0.01541,
    -0.00468,
    0.02331,
    0.02373,
    -0.00136,
    -0.01166,
    -0.00868,
    0.06139,
    0.04055,
    -0.00842,
    0.02049,
    6e-05,
    -0.02937,
    0.06372,
    -0.0113,
    -0.01598,
    -0.03584,
    0.0289,
    0.03019,
    -0.01212,
    -0.03812,
    -0.05405,
    -0.03681,
    -0.02591,
    0.00477,
    0.01188,
    -0.00698,
    0.00818,
    -0.01416,
    0.03491,
    -0.00457,
    -0.00478,
    -0.0375,
    -0.02196,
    0.02187,
    -0.02476,
    0.03366,
    -0.0256,
    -0.01122,
    0.00329,
    0.00823,
    0.00973,
    0.04446,
    0.01152,
    -0.00668,
    -0.00264,
    0.01677,
    0.01451,
    0.01734,
    0.01491,
    -0.01602,
    0.03258,
    0.02974,
    -0.037,
    -0.00497,
    -6e-05,
    0.01026,
    -0.01248,
    -0.01988,
    0.00123,
    -0.02565,
    0.00637,
    -0.0346,
    0.01356,
    0.02421,
    0.03123,
    -0.00631,
    -0.01293,
    0.04324,
    -0.01851,
    0.01037,
    -0.03336,
    -0.01209,
    0.01087,
    -0.01796,
    0.00391,
    0.03842,
    0.02674,
    -0.00103,
    0.01701,
    -0.02758,
    -0.0129,
    -0.02534,
    0.03839,
    0.01315,
    0.03438,
    0.01983,
    -0.0106,
    -0.04851,
    -0.02149,
    -0.00832,
    -0.00248,
    -0.04142,
    -0.01911,
    -0.01153,
    -0.03752,
    0.04211,
    -0.06802,
    -0.0415,
    0.02623,
    -0.01337,
    -0.01162,
    0.03288,
    0.01224,
    0.00675,
    -0.00358,
    0.01057,
    -0.07113,
    0.05029,
    -0.03422,
    -0.00948,
    -0.0,
    0.01519,
    0.02036,
    -0.03075,
    0.0096,
    -0.00818,
    -0.02215,
    0.01404,
    0.0395,
    -0.01904,
    -0.00901,
    0.03239,
    0.04359,
    -0.01476,
    -0.00919,
    -0.02259,
    0.00578,
    0.05787,
    -0.00471,
    -0.02745,
    -0.02224,
    0.02814,
    0.00127,
    -0.03103,
    0.01923,
    0.02934,
    -0.01999,
    -0.05045,
    -0.02531,
    -0.01819,
    -0.02226,
    -0.01593,
    -0.01511,
    0.00653,
    -0.01202,
    -0.04061,
    0.05052,
    0.01753,
    0.01647,
    -0.0069,
    0.01297,
    -0.06343,
    0.00803,
    0.02203,
    -0.01261,
    -0.02114,
    -0.02617,
    0.03041,
    -0.0211,
    0.016,
    0.01752,
    0.00454,
    0.02486,
    -0.00289,
    -0.0134,
    -0.01817,
    0.00627,
    -0.04074,
    0.08206,
    -0.05635,
    -0.05539,
    -0.05449,
    -0.00123,
    0.00371,
    -0.00049,
    -0.00695,
    -0.02074,
    -0.05008,
    -0.02605,
    -0.04659,
    0.00464,
    0.01429,
    0.03026,
    -0.00357,
    0.01395,
    -0.02143,
    -0.04879,
    0.00937,
    -0.0301,
    -0.07194,
    -0.02539,
    -0.00911,
    -0.02027,
    0.03207,
    0.01598,
    0.02824,
    -0.00673,
    0.00564,
    0.01342,
    0.01081,
    -0.00165,
    0.03217,
    0.0058,
    -0.03077,
    -0.01783,
    -0.03139,
    0.0047,
    0.01465,
    0.02371,
    0.01115,
    -0.02562,
    0.00126,
    0.04617,
    0.02401,
    -0.02237,
    -0.01081,
    -0.01804,
    0.0001,
    0.01724,
    0.0074,
    0.00642,
    -0.02643,
    0.03589,
    -0.02236,
    -0.00951,
    0.01168,
    -0.01423,
    -0.03795,
    0.0127,
    0.022,
    0.03606,
    0.01304,
    -0.0202,
    -0.01269,
    0.01433,
    0.00821,
    -0.05187,
    -0.04691,
    0.01131,
    0.00328,
    -0.01127,
    -0.03005,
    -0.01218,
    -0.00651,
    0.01923,
    -0.01107,
    0.02864,
    -0.03011,
    0.01971,
    -0.02225,
    0.06182,
    -0.03693,
    -0.05754,
    -0.01845,
    0.031,
    -0.01352,
    -0.069,
    -0.00426,
    0.0101,
    -0.00927,
    -0.01141,
    -0.02319,
    0.03028,
    -0.0087,
    0.01033,
    -0.01779,
    -0.0077,
    -0.02091,
    -0.00377,
    0.01261,
    0.01625,
    0.04775,
    0.00418,
    -0.04155,
    0.00092,
    0.02324,
    -0.00072,
    0.04188,
    0.06099,
    -0.05125,
    0.00384,
    -0.03072,
    0.00292,
    -0.00035,
    -0.02343,
    -0.00729,
    0.02466,
    0.05599,
    0.03778,
    0.00066,
    -0.01784,
    -0.00296,
    0.0099,
    0.02122,
    -0.02261,
    0.00695,
    0.00242,
    0.00616,
    -0.01696,
    0.04244,
    0.02438,
    -0.00842,
    -0.02209,
    -0.0223,
    0.04398,
    0.01069,
    -0.00849,
    -0.03012,
    0.03881,
    0.00095,
    0.01427,
    -0.03541,
    0.0032,
    0.03911,
    0.02336,
    -0.0187,
    -0.05471,
    0.01283,
    0.05065,
    -0.00841,
    0.00704,
    8e-05,
    0.02245,
    0.00217,
    0.02205,
    0.03707,
    -0.00527,
    -0.01504,
    -0.04578,
    0.04521,
    0.0065,
    0.02384,
    0.07252,
    0.0136,
    -0.01274,
    0.0001,
    0.01836,
    0.02245,
    0.02964,
    -0.0237,
    -0.00225,
    -0.00759,
    0.02898,
    -0.01033,
    -0.00564,
    0.02453,
    -0.00024,
    0.03099,
    -0.04518,
    -0.02593,
    -0.0127,
    0.02752,
    -0.00985,
    -0.01272,
    -0.00484,
    -0.00337,
    -0.02614,
    0.03685,
    -0.01189,
    0.02111,
    0.01672,
    0.00312,
    0.00711,
    0.02714,
    -0.00703,
    -0.00183,
    -0.04348,
    0.00133,
    -0.00425,
    -0.0017,
    0.06353,
    -0.01533,
    -0.03646,
    0.01019,
    -0.01386,
    0.00013,
    0.01213,
    -0.00852,
    -0.00314,
    0.02622,
    0.00766,
    0.03301,
    -0.01177,
    0.02011,
    -0.03284,
    0.02958,
    0.0193,
    -0.03552,
    -0.00569,
    0.00052,
    0.02057,
    0.02451,
    0.00075,
    -0.0298,
    -0.01228,
    -0.01995,
    0.02106,
    -0.00645,
    0.04524,
    -0.03677,
    -0.04429,
    -0.02093,
    0.02472,
    0.00599,
    0.00913,
    0.12391,
    0.03501,
    -0.05071,
    -0.00526,
    -0.02776,
    0.02557,
    -0.02837,
    0.01489,
    -0.0446,
    -0.01622,
    -0.00432,
    0.01759,
    0.01925,
    -0.04776,
    -0.01254,
    0.02096,
    -0.00911,
    -0.04737,
    0.00498,
    -0.00375,
    -0.03052,
    0.00988,
    0.00353,
    -0.0016,
    -0.02959,
    -0.00607,
    -0.01341,
    -0.01356,
    -0.02115,
    0.00552,
    0.02543,
    0.00924,
    0.00866,
    -0.01169,
    0.02543,
    -0.01023,
    -0.01987,
    -0.00511,
    -0.03767,
    -0.02923,
    0.02032,
    0.01262,
    0.02315,
    0.00099,
    0.00823,
    -0.00621,
    0.00165,
    0.00306,
    0.025,
    0.00199,
    -0.03449,
    0.01884,
    -0.02401,
    0.00921,
    -0.02441,
    -0.00336,
    -0.04194,
    -0.01961,
    -0.00927,
    -0.01853,
    0.00743,
    0.00564,
    0.00478,
    -0.05116,
    0.00621,
    0.01714,
    0.00636,
    -0.04661,
    0.03068,
    0.01593,
    0.01369,
    -0.00961,
    -0.02072,
    0.02996,
    -0.0094,
    -0.01834,
    -0.00895,
    -0.00938,
    -0.02524,
    0.0462,
    0.02592,
    0.06226,
    0.01296,
    0.04718,
    -0.02198,
    0.01477,
    -0.03578,
    -0.00829,
    -0.01977,
    0.02753,
    0.03978,
    0.02032,
    -0.03795,
    0.00234,
    -0.02553,
    0.06322,
    0.00087,
    -0.00724,
    -0.02356,
    -0.02201,
    -0.02295,
    0.02555,
    -0.01359,
    -0.01869,
    0.00597,
    -0.00697,
    0.01535,
    0.0163,
    0.0076,
    -0.00702,
    0.00549,
    -0.00398,
    -0.00167,
    -0.01752,
    0.00726,
    0.01615,
    -0.00035,
    -0.02109,
    0.00418,
    0.0431,
    0.03003,
    0.01339,
    -0.01478,
    0.02903,
    -0.02066,
    -0.03021,
    0.01405,
    -0.0107,
    0.00301,
    0.00235,
    0.0376,
    -0.00441,
    0.05589,
    0.03308,
    -0.00153,
    0.00242,
    0.05823,
    0.0406,
    -0.02526,
    -0.01083,
    -0.01401,
    0.01449,
    -0.01816,
    0.00756,
    -0.01226,
    0.02394,
    -0.03833,
    0.04054,
    -0.0108,
    0.01684,
    0.01187,
    -0.01764,
    -0.01375,
    0.00048,
    -0.03268,
    -0.00127,
    -0.00123,
    -0.00701,
    0.00162,
    -0.00012,
    -0.00476,
    -0.02716,
    -0.0028,
    0.00805,
    0.05936,
    0.06432,
    0.00016,
    -0.01771,
    -0.01474,
    -0.01711,
    -0.00274,
    0.00346,
    -0.00949,
    -0.00261,
    0.02764,
    0.00768,
    0.00608,
    -0.00807,
    -0.00801,
    -0.02073,
    0.02965,
    0.00111,
    0.03994,
    -0.01123,
    0.00596,
    0.01369,
    0.02856,
    0.02438,
    0.01414,
    0.02425,
    0.00358,
    -0.02286,
    0.01034,
    0.00179,
    -0.00439,
    -0.07425,
    0.0206,
    -0.0012,
    -0.0271,
    -0.02881,
    -0.00081,
    0.00243,
    0.00038,
    -0.02016,
    -0.01042,
    -0.02998,
    0.01244,
    0.00259,
    0.0121,
    -0.00631,
    -0.00381,
    0.0429,
    -0.05755,
    -0.07388,
    -0.02329,
    0.00116,
    -0.01475,
    0.03393,
    -0.00693,
    -0.036,
    -0.00984,
    -0.00589,
    0.01152,
    0.00693,
    0.04198,
    -0.0212,
    -0.04445,
    0.0048,
    -0.01989,
    -0.0093,
    0.01378,
    -0.01925,
    -0.0208,
    -0.00693,
    -0.00912,
    0.00464,
    -0.00077,
    -0.01042,
    0.00223,
    0.0063,
    0.00438,
    -0.00778,
    -0.00967,
    -0.01169,
    0.01454,
    0.03118,
    0.07271,
    -0.01643,
    0.02034,
    -0.01787,
    -0.00932,
    0.01899,
    -0.01465,
    -0.01576,
    -0.02248,
    0.00356,
    0.00308,
    -0.02932,
    0.01459,
    -0.00717,
    0.03281,
    0.00275,
    -0.03933,
    0.08221,
    -0.00288,
    -0.02875,
    0.01584,
    -0.01343,
    -0.01834,
    0.01963,
    -0.00095,
    -0.04907,
    0.00424,
    -0.04038,
    -0.02697,
    -0.0045,
    -0.01621,
    0.01169,
    -0.0237,
    0.03833,
    -0.00615,
    0.00019,
    -0.01851,
    -0.03748,
    -0.03657,
    0.03695,
    0.02732,
    0.02114,
    -0.01665,
    -0.01102,
    0.00724,
    -0.00836,
    -0.0131,
    0.03192,
    -0.01742,
    0.00446,
    0.04194,
    -0.02697,
    -0.00965,
    0.03779,
    -0.03385,
    0.00354,
    0.01451,
    -0.01025,
    0.03836,
    -0.02652,
    0.02789,
    -0.02648,
    -0.01467,
    0.02276,
    0.02765,
    0.02344,
    -0.02322,
    -0.02803,
    0.04129,
    -0.03106,
    0.01916,
    0.03859,
    0.02333,
    0.01839,
    0.01564,
    0.00839,
    -0.00548,
    0.02077,
    0.05162,
    -0.01619,
    -0.02642,
    0.04399,
    0.01532,
    0.02428,
    0.00179,
    0.02216,
    -0.05287,
    -0.00034,
    -0.02563,
    -0.01681
   ],
   "ON": [
    0.00312,
    0.01314,
    -0.03084,
    -0.01557,
    -0.00448,
    0.00765,
    -0.02875,
    0.00144,
    -0.01946,
    -0.04395,
    0.02035,
    0.00748,
    -0.02444,
    0.01575,
    0.01115,
    0.00683,
    -0.01947,
    -0.01862,
    0.0091,
    0.05848,
    0.01519,
    -0.03801,
    0.01759,
    -0.02637,
    0.00341,
    0.00596,
    0.01623,
    0.02415,
    -0.04168,
    0.03167,
    -0.02105,
    -0.0502,
    0.04589,
    -0.02523,
    0.00903,
    0.02283,
    0.00676,
    -0.05108,
    0.00368,
    0.01464,
    -0.02878,
    0.0034,
    -0.01594,
    -0.00765,
    0.0104,
    -0.02122,
    -0.04573,
    0.03842,
    0.01933,
    0.01338,
    -0.01369,
    0.00409,
    -0.02443,
    0.04634,
    -0.08084,
    -0.04833,
    -0.03847,
    0.00733,
    0.05584,
    -0.02241,
    0.01684,
    0.00465,
    -0.02034,
    -0.01044,
    -0.02052,
    0.00787,
    -0.02049,
    0.0212,
    -0.00669,
    -0.00849,
    -0.02526,
    0.01123,
    0.00584,
    0.00473,
    -0.01239,
    -0.02713,
    -0.04693,
    -0.02317,
    0.02358,
    0.01717,
    -0.05318,
    -0.00514,
    0.00754,
    -0.01061,
    -0.0302,
    0.016,
    0.02337,
    0.01556,
    0.03897,
    -0.00716,
    0.02649,
    0.01591,
    -0.00088,
    -0.00349,
    -0.01876,
    -0.04729,
    0.01998,
    0.00551,
    -0.00084,
    0.02242,
    0.00574,
    -0.00332,
    0.01805,
    0.03097,
    -0.01154,
    -0.01195,
    -0.00828,
    0.0159,
    -0.01032,
    0.01969,
    0.00106,
    0.00899,
    0.04905,
    0.01494,
    -0.04298,
    0.00887,
    0.00936,
    -0.0004,
    -0.0218,
    -0.00269,
    -0.00037,
    -0.02448,
    0.00496,
    -0.00896,
    -0.03712,
    -0.02748,
    -0.00711,
    0.01133,
    -0.00601,
    -0.00833,
    0.04198,
    -0.01306,
    0.02228,
    -0.01858,
    0.04362,
    -0.00602,
    -0.02197,
    0.01397,
    -0.00069,
    -0.00868,
    -0.02394,
    -0.0219,
    -0.00489,
    0.03249,
    -0.02322,
    -0.02464,
    0.02821,
    0.00639,
    -0.01707,
    0.02734,
    -0.00624,
    0.00718,
    0.0252,
    -0.00063,
    -0.0442,
    0.01067,
    -0.02581,
    0.00038,
    0.02008,
    -0.00381,
    -0.03418,
    -0.01991,
    -0.03385,
    -0.0104,
    -0.017,
    0.0182,
    0.00184,
    -0.01008,
    -0.02084,
    -0.01251,
    0.01206,
    0.01619,
    -0.0462,
    -0.00657,
    -0.02516,
    -0.01037,
    0.00428,
    -0.00205,
    0.00011,
    0.0127,
    -0.00964,
    0.00316,
    -0.00592,
    0.02938,
    0.0034,
    -0.01232,
    0.00409,
    0.04541,
    0.01447,
    0.00149,
    -0.0073,
    0.0289,
    0.01487,
    -0.00888,
    0.02653,
    -0.00979,
    -0.04455,
    -0.02588,
    -0.03397,
    -0.01587,
    -0.02415,
    -0.00857,
    0.01076,
    -0.01843,
    -0.06613,
    -0.00266,
    0.01309,
    -0.02551,
    -0.01651,
    0.0426,
    0.01125,
    0.02905,
    0.01941,
    0.04081,
    0.02514,
    0.022,
    0.01017,
    -0.0144,
    -0.00181,
    0.01312,
    0.01249,
    0.00201,
    -0.00519,
    -0.03588,
    0.03515,
    0.00591,
    0.00598,
    0.02556,
    0.00763,
    0.01733,
    -0.0761,
    -0.01621,
    -0.02199,
    -0.04947,
    0.00966,
    0.03238,
    -0.10547,
    -0.03936,
    0.01331,
    0.02153,
    -0.02549,
    0.04308,
    0.00665,
    0.05393,
    0.02931,
    0.0132,
    0.01013,
    -0.00693,
    0.00483,
    -0.00643,
    0.02282,
    -0.00603,
    -0.005,
    0.01042,
    0.02469,
    -0.02341,
    0.03911,
    0.02189,
    -0.0,
    -0.0128,
    -0.02787,
    0.00259,
    0.01176,
    -0.03553,
    0.01706,
    0.01576,
    0.0068,
    0.00085,
    -0.02555,
    0.04109,
    0.02209,
    0.01869,
    -0.03884,
    -0.00813,
    -0.02369,
    0.00316,
    -0.00564,
    0.00908,
    0.01393,
    -0.00835,
    -0.01463,
    -0.01496,
    -0.00198,
    0.00824,
    0.01828,
    -0.00095,
    0.02524,
    0.01431,
    -0.0017,
    0.02049,
    4e-05,
    0.02603,
    0.00118,
    0.01008,
    -0.01497,
    -0.02087,
    -0.06683,
    -0.0425,
    -0.0228,
    0.0244,
    0.00264,
    -0.00865,
    -0.00642,
    0.00697,
    -0.0108,
    -0.00232,
    0.03201,
    -0.03031,
    0.00224,
    -0.00455,
    -0.02255,
    -0.01154,
    0.0062,
    0.00931,
    -0.01962,
    0.05841,
    -0.02365,
    0.01826,
    0.00877,
    -0.00503,
    0.02094,
    0.04277,
    -0.00311,
    -0.03717,
    0.00024,
    0.01016,
    0.02211,
    0.01576,
    0.01991,
    -0.06204,
    -0.00177,
    -0.00635,
    0.00435,
    0.02492,
    -0.01571,
    -0.02327,
    0.02294,
    -0.01315,
    0.00535,
    -0.01209,
    0.02424,
    -0.01679,
    0.01856,
    0.01789,
    -0.01449,
    -0.02611,
    0.03454,
    -0.00655,
    0.00032,
    -0.02046,
    0.00351,
    0.01297,
    0.00489,
    0.00115,
    -0.06305,
    0.01186,
    -0.03933,
    -0.00406,
    -0.04666,
    -0.00584,
    -0.03549,
    0.0235,
    -0.0466,
    -0.0406,
    -0.01641,
    0.04178,
    -0.00645,
    0.01862,
    0.01288,
    -0.06486,
    0.00269,
    0.00443,
    -0.00763,
    -0.02261,
    0.04329,
    -0.01388,
    0.0232,
    0.03448,
    -0.01814,
    -0.01153,
    -0.0124,
    -0.00461,
    -0.03574,
    -0.03079,
    0.00562,
    0.01422,
    0.02204,
    -0.02718,
    0.02226,
    -0.02034,
    -0.01609,
    0.00481,
    -0.0257,
    0.03713,
    0.00407,
    0.046,
    -0.00229,
    0.00099,
    -0.00659,
    -0.00399,
    -0.01236,
    0.03106,
    0.01208,
    -0.03441,
    -0.00561,
    -0.00797,
    0.00598,
    -0.03071,
    -0.03878,
    -0.01222,
    -0.00638,
    0.01898,
    0.01094,
    -0.01334,
    -0.01673,
    0.00837,
    0.01364,
    -0.04307,
    0.03412,
    0.02525,
    -0.00263,
    -0.03276,
    -0.01947,
    -0.01303,
    -0.04557,
    0.0223,
    0.01605,
    0.00057,
    0.00059,
    0.00324,
    0.00103,
    -0.00097,
    0.00209,
    0.00886,
    -0.04575,
    0.01133,
    0.04748,
    -0.02062,
    0.01362,
    0.01823,
    0.02119,
    0.01931,
    -0.00961,
    -0.00716,
    -0.00186,
    0.0049,
    0.02895,
    0.01742,
    0.05944,
    0.00712,
    -0.00803,
    -0.01666,
    -0.06527,
    0.01872,
    -0.02146,
    -0.02828,
    -0.00768,
    0.01847,
    0.01125,
    -0.0051,
    -0.01362,
    0.02981,
    -0.01629,
    -0.00208,
    0.01172,
    -0.01865,
    0.00793,
    -0.02919,
    -0.01748,
    -0.05104,
    0.0277,
    0.02476,
    -0.00402,
    -0.06631,
    0.01624,
    0.01153,
    0.0045,
    -0.00545,
    -0.00333,
    -0.00651,
    -0.00821,
    0.03223,
    0.01043,
    0.01503,
    -0.01856,
    -0.01785,
    -0.00071,
    0.0417,
    -0.02036,
    0.00406,
    0.03009,
    0.03035,
    -0.02673,
    -0.00218,
    0.02262,
    -0.02976,
    -0.0126,
    -0.00483,
    0.02244,
    -0.00031,
    -0.00083,
    -0.04439,
    -0.03066,
    -0.00892,
    -0.0133,
    -0.00797,
    0.00756,
    -0.0332,
    -0.01075,
    -0.00689,
    0.04411,
    -0.00519,
    -0.01398,
    -0.02506,
    0.03495,
    0.01872,
    0.05295,
    -0.02838,
    0.01777,
    0.00287,
    0.06008,
    -0.02607,
    -0.02055,
    -0.00525,
    0.01556,
    0.02752,
    0.00774,
    0.00846,
    0.0081,
    0.02351,
    -0.0077,
    -0.01109,
    0.01229,
    0.0072,
    -0.01857,
    0.0014,
    0.00543,
    0.00358,
    -0.00226,
    0.03027,
    -0.02369,
    -0.00117,
    -0.01797,
    -0.01823,
    0.00661,
    -0.00202,
    -0.05127,
    0.04,
    -0.00975,
    0.02257,
    0.00851,
    0.01132,
    0.03248,
    0.03093,
    0.01094,
    -0.0552,
    0.01024,
    0.01788,
    -0.03737,
    0.00561,
    -0.00581,
    -0.04659,
    0.01759,
    0.00208,
    -0.0068,
    0.06258,
    -0.00678,
    0.01323,
    -0.02825,
    -0.00554,
    -0.00051,
    0.00555,
    -0.00394,
    -0.02464,
    -0.02285,
    -0.00543,
    0.0208,
    -0.01559,
    -0.00609,
    -0.00404,
    0.00684,
    0.00457,
    0.02021,
    -0.0119,
    -0.01096,
    0.01367,
    -0.00472,
    -0.01322,
    0.02751,
    -0.04267,
    -0.02912,
    0.03419,
    0.00831,
    0.00255,
    0.01672,
    -0.00553,
    0.00327,
    0.01733,
    -0.00374,
    0.01559,
    -0.02625,
    0.03206,
    -0.02002,
    -0.00535,
    -0.04575,
    -0.00843,
    0.00541,
    -0.01686,
    -0.04001,
    0.07198,
    0.00202,
    0.00024,
    0.00954,
    0.04947,
    0.02265,
    -0.01566,
    0.01203,
    -0.00562,
    0.00286,
    -0.001,
    0.0164,
    0.02628,
    -0.00194,
    0.01064,
    0.00393,
    0.03408,
    -0.02374,
    -0.00143,
    -0.02826,
    -0.03845,
    -0.00638,
    -0.00299,
    0.02153,
    -0.02956,
    -0.00726,
    -0.00049,
    0.01388,
    -0.00344,
    0.00645,
    -0.02371,
    -0.0016,
    0.01964,
    0.02365,
    0.0173,
    -0.01047,
    -0.03349,
    0.00135,
    -0.03339,
    -0.01325,
    0.01002,
    -0.02923,
    0.0438,
    -0.01033,
    -0.0065,
    0.02,
    0.01978,
    0.00847,
    0.02678,
    -0.06604,
    0.03,
    0.0552,
    0.05086,
    0.02403,
    -0.01928,
    0.00949,
    0.01745,
    -0.02333,
    0.00471,
    0.00263,
    -0.02293,
    0.00992,
    0.01075,
    0.02795,
    -0.06417,
    0.02978,
    -0.01947,
    0.00038,
    -0.02374,
    0.02575,
    -0.02983,
    0.00577,
    0.00111,
    -0.01414,
    0.04088,
    -0.02327,
    0.02461,
    -0.00769,
    -0.01507,
    -0.01961,
    0.02331,
    -0.03161,
    -0.02213,
    -0.03543,
    -0.02241,
    0.00234,
    -0.02702,
    0.0108,
    0.0145,
    0.01314,
    0.00044,
    0.00514,
    0.04295,
    -0.02983,
    -0.00415,
    0.01427,
    0.02176,
    0.0183,
    0.00657,
    0.00811,
    0.00105,
    -0.01571,
    0.01237,
    -0.00901,
    0.0166,
    -0.01609,
    0.03528,
    -0.01113,
    0.0038,
    -0.01814,
    0.01602,
    0.01739,
    -0.00373,
    -0.00236,
    0.00097,
    0.01204,
    -0.00278,
    0.0537,
    -0.00812,
    0.01263,
    0.01618,
    0.0331,
    -0.00987,
    0.05059,
    0.00348,
    -0.06019,
    -0.02125,
    -0.00679,
    -0.0382,
    -0.01977,
    0.03838,
    -0.01764,
    0.03475,
    -0.00294,
    -0.01183,
    0.02305,
    0.0453,
    -0.05138,
    0.01614,
    0.00194,
    0.02109,
    0.00657,
    -0.03726,
    -0.05984,
    0.0052,
    0.03113,
    0.00219,
    0.00338,
    0.02043,
    0.02399,
    -0.0009,
    -0.05745,
    -0.00743,
    0.0181,
    -0.0228,
    0.03175,
    -0.01252,
    0.01299,
    -0.01114,
    0.01557,
    0.05342,
    0.02294,
    -0.02932,
    0.00754,
    -0.02656,
    0.00415,
    0.01101,
    0.02068,
    0.00518,
    -0.03321,
    -0.00714,
    0.02388,
    0.04201,
    -0.04709,
    -0.01278,
    -0.05961,
    -0.03838,
    0.01038,
    -0.01498,
    0.03977,
    0.01116,
    -0.00567,
    -0.03447,
    0.00124,
    0.00601,
    -0.01567,
    0.01211,
    -0.01982,
    -0.0092,
    -0.0061,
    -0.00353,
    -6e-05,
    -0.06112,
    -0.01937,
    0.0084,
    -0.02435,
    0.01647,
    -0.00022,
    -0.0476,
    -0.00657,
    0.0059,
    0.03511,
    0.00293,
    -0.02349,
    0.0443,
    0.01096,
    0.05197,
    0.02507,
    0.0041,
    -0.03911,
    0.00872,
    0.00664,
    0.00083,
    0.02681,
    -0.05742,
    0.0246,
    0.01555,
    -0.00239,
    -0.00989,
    0.00881,
    0.02067,
    -0.02666,
    0.00637,
    0.0197,
    -0.00041,
    -0.01601,
    -0.02642,
    -0.04336,
    -0.01921,
    0.0053,
    -0.02902,
    -0.01217,
    -0.00199,
    0.00828,
    -0.01347,
    0.0258,
    -0.03349,
    -0.03907,
    0.02579,
    0.02862,
    0.00099,
    -0.04469,
    0.0235,
    0.01695,
    -0.03916,
    0.00722,
    0.01734,
    0.01098,
    -0.02761,
    -0.01255,
    -0.0005,
    -0.01551,
    0.0486,
    -0.02416,
    -0.00167,
    -0.00725,
    -0.04291,
    0.01208,
    -0.02996,
    0.06376,
    -0.01442,
    -0.00911,
    0.0251,
    -0.02061,
    0.00051,
    0.02637,
    -0.01698,
    -0.0006,
    -0.03166,
    0.00246,
    -0.0137,
    0.00542,
    -0.0022,
    0.05178,
    0.01706,
    0.02968,
    0.00319,
    0.03553,
    0.01545,
    0.01064,
    0.01335,
    -0.00462,
    0.00491,
    0.0083,
    0.00531,
    0.00599,
    -0.00371,
    -0.02769,
    -0.02335,
    -0.00759,
    -0.02914,
    -0.0062,
    -0.05776,
    -0.02817,
    0.00961,
    0.02442,
    -0.0156,
    0.01851,
    -0.00751,
    0.00552,
    -0.00544,
    0.02143,
    0.00333,
    -0.0223,
    -0.01352,
    -0.01915,
    0.00482,
    -0.04556,
    0.00051,
    0.04102,
    0.01148,
    0.02054,
    -0.02695,
    -0.04675,
    0.00723,
    0.03151,
    0.02315,
    -0.01097,
    -0.00289,
    0.00916,
    -0.0143,
    -0.01124,
    0.00808,
    0.02907,
    0.0135,
    0.01596,
    0.02085,
    0.00537,
    -0.00864,
    0.0305,
    0.00321,
    0.01945,
    -0.04568,
    -0.01655,
    0.00178,
    0.03089,
    -0.00382,
    0.02839,
    -0.02645,
    -0.04389,
    0.00785,
    0.04429,
    0.00278,
    -0.01906,
    0.00149,
    0.03047,
    0.00275,
    -0.02384,
    -0.04989,
    0.0202,
    0.0066,
    0.02991,
    -0.01864,
    0.04249,
    -0.06192,
    0.05691,
    0.05041,
    0.01119,
    0.03179,
    -0.0076,
    -0.01123,
    0.01421,
    -0.01517,
    0.04735,
    0.05148,
    0.03913,
    -0.00994,
    -0.01626,
    -0.02668,
    -0.02404,
    -0.03711,
    0.01734,
    0.03328,
    -0.04403,
    0.01347,
    0.03134,
    0.02263,
    -0.00914,
    -0.00344,
    -0.03037,
    0.01205,
    0.01994,
    -0.01161,
    -0.02875,
    -0.0146,
    -0.02055,
    -0.00371,
    -0.0526,
    -0.04093,
    0.04696,
    -0.00814,
    0.01575,
    0.01319,
    -0.00941,
    -0.01535,
    -0.00876,
    -0.02614,
    -0.01117,
    -0.00772,
    -0.03016,
    0.00816,
    0.00131,
    0.00961,
    -0.0137,
    -0.02937,
    -0.00272,
    -0.00351,
    0.04377,
    -0.04729,
    0.04927,
    -0.00487,
    -0.00965,
    0.03556,
    0.03744,
    -0.01206,
    -0.0404,
    -0.04694,
    -0.00999,
    0.0167,
    0.00347,
    0.00127,
    -0.01683,
    0.0262,
    0.02672,
    0.01815,
    -0.01922,
    -0.01001,
    0.0236,
    0.02313,
    -0.01185,
    -0.0099,
    0.03172,
    -0.00297,
    0.01808,
    -0.00677,
    0.02076,
    -0.06389,
    0.01857,
    0.00104,
    0.01913,
    -0.03398,
    0.00685,
    0.02982,
    0.00963,
    -0.04372,
    0.01181,
    0.00922,
    0.03041,
    -0.05668,
    0.01901,
    -0.02781,
    0.02771,
    0.00576,
    0.03591,
    0.01919,
    -0.00077,
    -0.00054,
    -0.04009,
    -0.01538,
    -0.00148,
    0.00962,
    -0.00057,
    0.02446,
    -0.03732,
    -0.01095,
    0.02814,
    0.0235,
    0.01129,
    -0.02213,
    0.01318,
    0.02687,
    -0.01516,
    -0.00985,
    -0.05992,
    -0.02396,
    0.01917,
    -0.00159,
    0.0047,
    -0.02479,
    -0.0122,
    0.02192,
    -0.0329,
    -0.00589,
    0.03611,
    -0.01554,
    -0.03173,
    -0.02468,
    0.00939,
    0.01165,
    0.0048,
    -0.04708,
    -0.00824,
    -0.01137,
    0.00976,
    -0.04857,
    0.00209,
    -0.02611,
    0.01822,
    0.01874,
    -0.00394,
    -0.02762,
    0.03019,
    0.05596,
    -0.00422,
    -0.04444,
    0.00282,
    -0.00247,
    -0.00243,
    0.02809,
    -0.04471,
    -0.00628,
    -0.02466,
    -0.00495,
    0.01816,
    0.03419,
    -0.01669,
    -0.00394,
    -0.03048,
    -0.05068,
    0.01249,
    0.02857,
    -0.02387,
    -0.01163,
    0.00189,
    -0.01506,
    -0.00476,
    -0.0045,
    0.00201,
    -0.01908,
    -0.00326,
    0.01454,
    -0.03865,
    0.00154,
    -0.05478,
    0.0091,
    0.04164,
    0.00614,
    -0.00349,
    0.00259,
    0.01053,
    -0.02118,
    0.00433,
    0.04173,
    -0.04342,
    0.01429,
    -0.01941,
    -0.00348,
    -0.02357,
    -0.01675,
    -0.02307,
    -0.00148,
    0.01215,
    0.04026,
    0.01614,
    0.02699,
    -0.00103,
    -0.04246,
    -0.01908,
    -0.00847,
    -0.01348,
    0.00511,
    -0.02261,
    -0.03375,
    -0.01256,
    0.02119,
    -0.01593,
    -0.01322,
    0.00892,
    0.00392,
    -0.037,
    0.00787,
    -0.01411,
    -0.01712,
    0.0419,
    0.02136,
    0.00012,
    -0.02818,
    -0.01335,
    -0.01662,
    0.01245,
    0.01096,
    0.01645,
    -0.01488,
    0.02147,
    -0.03288,
    0.00732,
    0.01217,
    0.01447,
    0.00759,
    0.03094,
    -0.02781,
    -0.09701,
    -0.03434,
    0.02188,
    -0.01956,
    0.04944,
    0.01248,
    0.0315,
    -0.02873,
    0.04225,
    0.03497,
    -0.00847,
    0.0041,
    -0.00654,
    0.04197,
    -0.00687,
    -0.01262,
    -0.00076,
    0.01468,
    0.00454,
    0.02014,
    0.00487,
    -0.01782,
    -0.03513,
    0.01034,
    0.01731,
    0.01663,
    0.02469,
    -0.00851,
    0.00395,
    0.01802,
    -0.00404,
    -0.02515,
    -0.01278,
    0.00701,
    -0.01541,
    -0.04562,
    0.02981,
    -0.01989,
    0.02614,
    0.04488,
    -0.03309,
    -0.02332,
    -0.02627,
    -0.02717,
    -0.02012,
    0.01304,
    0.01121,
    0.00735,
    -0.00611,
    -0.01634,
    0.00536,
    -0.0094,
    0.03886,
    -0.03164,
    0.03108,
    -0.00376,
    0.05532,
    0.00266,
    0.01614,
    0.00916,
    0.0225,
    -0.00075,
    -0.0085,
    0.07156,
    0.01028,
    -0.00324,
    -0.01073,
    0.06552,
    -0.03192,
    -0.02653,
    -0.04192,
    -0.00822,
    0.04577,
    -0.05536,
    0.02862,
    0.05555,
    0.01267,
    0.00732,
    0.01212,
    -0.04029,
    -0.01626,
    -0.03361,
    -0.0223,
    0.03055,
    0.01677,
    0.0069,
    0.00848,
    -0.0018,
    0.00588,
    -0.02513,
    -0.02039,
    -0.01426,
    0.05744,
    -0.02795,
    0.00674,
    -0.02933,
    0.01778,
    0.04793,
    -0.00628,
    0.00028,
    -0.00107,
    -0.011,
    -0.00199,
    -0.00864,
    0.00566,
    0.0116,
    0.01,
    -0.03027,
    0.00577,
    -0.01062,
    -0.01502,
    -0.00619,
    0.00197,
    0.01123,
    0.01337,
    -0.01248,
    -0.00457,
    0.03075,
    0.00685,
    -0.02772,
    -0.01819,
    -0.03334,
    0.022,
    -0.00624,
    0.0269,
    0.01902,
    -0.01188,
    0.01124,
    -0.02002,
    -0.00832,
    -0.01455,
    0.03119,
    -0.07404,
    -0.01678,
    0.00116,
    0.02315,
    -0.03775,
    -0.04611,
    -0.01057,
    0.01314,
    -0.00625,
    -0.01906,
    0.02404,
    0.00274,
    0.00397,
    -0.01603,
    0.03169,
    -0.02522,
    -0.01209,
    -0.00733,
    -0.04941,
    0.00876,
    0.00522,
    0.0201,
    0.02823,
    0.00522,
    0.00953,
    -0.02661,
    -0.00544,
    0.0211,
    -0.02471,
    0.00238,
    0.0017,
    -0.02342,
    -0.05313,
    -0.04661,
    0.01754,
    0.0164,
    0.01217,
    0.02273,
    -0.00405,
    -0.00162,
    0.02654,
    0.00211,
    0.00846,
    0.019,
    -0.02437,
    0.00541,
    0.02928,
    0.03997,
    -0.02698,
    -0.00746,
    -0.0017,
    -0.02924,
    -0.00435,
    -0.04875,
    -0.03926,
    0.00183,
    0.0058,
    0.0114,
    0.00913,
    -0.00185,
    0.00442,
    0.00106,
    0.00217,
    0.06617,
    -0.0004,
    -0.00235,
    0.03707,
    0.01997,
    0.01571,
    -0.0004,
    -0.01572,
    -0.00788,
    -0.00436,
    0.02734,
    -0.02302,
    0.00497,
    -0.02311,
    0.00049,
    -0.02002,
    -0.05501,
    0.03425,
    0.03547,
    0.02425,
    -0.00833,
    0.01668,
    -0.00696,
    0.00915,
    0.00999,
    0.02078,
    -0.29822,
    0.04605,
    0.0211,
    -0.01352,
    -0.04502,
    0.03904,
    -0.02938,
    0.02639,
    0.00501,
    -0.01582,
    0.01068,
    0.00832,
    0.02758,
    -0.01273,
    0.00205,
    -0.00593,
    0.03188,
    0.00267,
    -0.0311,
    0.01129,
    -0.0103,
    0.00781,
    0.01324,
    0.01263,
    0.00142,
    -0.00903,
    0.01306,
    -0.00823,
    0.04196,
    0.0117,
    -0.00809,
    0.0225,
    -0.01771,
    0.01381,
    -0.00442,
    0.01409,
    0.0209,
    -0.00279,
    0.00279,
    -0.02039,
    -0.02619,
    0.03432,
    -0.04798,
    -0.00028,
    0.01946,
    0.00405,
    0.02204,
    -0.00962,
    -0.0161,
    -0.01775,
    0.01903,
    -0.03119,
    0.02431,
    -0.00216,
    -0.00063,
    0.03171,
    0.0019,
    0.01674,
    -0.0217,
    -0.00644,
    0.00679,
    -0.0219,
    0.0035,
    0.03751,
    -0.00085,
    -0.00406,
    -0.01291,
    -0.01072,
    0.03823,
    -0.01019,
    0.00533,
    0.04863,
    -0.04745,
    0.02426,
    -0.0218,
    -0.01847,
    0.02083,
    -0.00216,
    -0.0249,
    0.02529,
    -0.00129,
    -0.01059,
    0.00103,
    -0.02349,
    0.02,
    -0.01489,
    0.00992,
    0.00587,
    -0.06277,
    -0.02965,
    -0.03021,
    0.05033,
    -0.01609,
    -0.03868,
    0.01274,
    -0.02568,
    -0.0283,
    0.02256,
    -0.01651,
    -0.0017,
    -0.01951,
    0.02327,
    -0.02377,
    -0.0033,
    0.00365,
    0.01722,
    -0.03204,
    -0.0112,
    0.00724,
    -0.00265,
    -0.03158,
    0.02158,
    -0.05083,
    0.01936,
    0.03788
   ],
   "MYSTIC": [
    -0.03069,
    0.01733,
    -0.02678,
    0.01749,
    -0.00411,
    0.03344,
    -0.01732,
    0.00904,
    0.04508,
    -0.00812,
    0.02794,
    -0.04748,
    0.00071,
    -0.00823,
    0.01352,
    -0.02924,
    0.01587,
    -0.01763,
    0.01278,
    -0.00672,
    -0.00544,
    -0.02682,
    0.01642,
    -0.02659,
    -0.00082,
    -0.0149,
    0.00525,
    0.02278,
    -0.01357,
    0.02331,
    0.00015,
    -0.00145,
    0.05786,
    0.02639,
    0.03452,
    -0.00228,
    0.02478,
    0.04784,
    0.02084,
    -0.01255,
    0.00946,
    -0.01767,
    -0.00186,
    -0.00065,
    -0.01909,
    0.00737,
    0.0204,
    0.03141,
    0.01433,
    -0.04886,
    0.0138,
    0.02635,
    -0.02521,
    0.01233,
    -0.03128,
    0.02889,
    0.01516,
    0.01949,
    0.03575,
    0.00732,
    0.04251,
    0.02529,
    0.00113,
    0.01494,
    -0.01914,
    -0.03197,
    0.00988,
    0.00587,
    0.01963,
    0.00531,
    -0.02448,
    0.00598,
    -0.00913,
    0.01741,
    0.04204,
    -0.02815,
    -0.01469,
    0.0044,
    -0.00409,
    -0.00225,
    0.01708,
    0.02686,
    0.03517,
    0.00318,
    0.0058,
    0.01857,
    -0.02527,
    -0.03662,
    0.04765,
    0.00303,
    -0.00565,
    0.02556,
    -0.00586,
    -0.03225,
    0.00578,
    -0.0142,
    -0.01259,
    0.05565,
    0.01777,
    0.01919,
    0.02461,
    0.00421,
    -0.02495,
    -0.00274,
    -0.04922,
    -0.02423,
    0.04348,
    -0.00325,
    0.00832,
    0.0125,
    0.024,
    0.00853,
    -0.00064,
    0.04214,
    -0.02574,
    -0.03013,
    -0.03271,
    -0.00818,
    -0.02243,
    0.02787,
    0.02628,
    -0.02088,
    0.04236,
    -0.01423,
    -0.0074,
    -0.01756,
    -0.02042,
    0.01792,
    -0.04527,
    3e-05,
    -0.00079,
    0.01358,
    0.00164,
    -0.05091,
    -0.02101,
    -0.00617,
    0.01869,
    0.01028,
    -0.018,
    0.0109,
    0.03142,
    -0.03407,
    -0.01954,
    0.01572,
    0.00047,
    -0.02334,
    0.01556,
    -0.01363,
    -0.00505,
    0.00938,
    -0.01986,
    0.00272,
    -0.04355,
    -0.00265,
    0.00244,
    -0.03589,
    -0.02136,
    -0.00203,
    0.00252,
    -0.02283,
    -0.0203,
    -0.02638,
    0.01006,
    0.04735,
    0.00375,
    0.02308,
    0.00948,
    0.00014,
    0.00446,
    -0.00288,
    -0.01549,
    -0.01309,
    0.0063,
    -0.01282,
    -0.04951,
    0.04389,
    0.00129,
    -0.01295,
    0.01095,
    0.00925,
    0.01547,
    0.00065,
    -0.00263,
    0.01251,
    -0.02122,
    0.01681,
    0.04005,
    0.02019,
    0.00915,
    0.01987,
    0.00716,
    0.06051,
    -0.0247,
    -0.02497,
    0.03854,
    0.04674,
    0.02209,
    -0.06716,
    -0.01711,
    -0.04877,
    -0.02298,
    -0.02685,
    -0.01428,
    -0.0226,
    0.01462,
    -0.00299,
    -0.01524,
    -0.00977,
    0.02134,
    0.01294,
    0.03811,
    -0.01923,
    0.01163,
    -0.01285,
    0.02039,
    0.03795,
    -0.00014,
    -0.00904,
    -0.03065,
    -0.0087,
    0.00475,
    -0.05177,
    -0.05251,
    0.04499,
    -0.0241,
    -0.01259,
    -0.05455,
    -0.0125,
    -0.01731,
    0.00263,
    -0.02689,
    0.01534,
    -0.02051,
    -0.01538,
    0.00109,
    -0.00076,
    -0.01264,
    -0.03236,
    -0.02061,
    0.03835,
    0.02617,
    -0.02707,
    0.02042,
    0.00437,
    0.05685,
    0.01196,
    -0.00774,
    0.01035,
    -0.04396,
    0.00463,
    -0.02839,
    -0.03909,
    -0.00145,
    -0.01009,
    0.03862,
    0.00684,
    -0.03276,
    -0.0229,
    0.02415,
    -0.0068,
    -0.0678,
    0.01381,
    -0.00943,
    0.01622,
    0.00618,
    0.01573,
    -0.02185,
    0.00861,
    0.05261,
    -0.03435,
    0.01993,
    0.00467,
    -0.0004,
    0.03163,
    0.02316,
    -0.00569,
    0.02061,
    0.01002,
    0.02339,
    -0.00921,
    0.00245,
    -0.01188,
    -0.00069,
    -0.00628,
    -0.00096,
    0.02513,
    -0.00761,
    -0.00451,
    -0.0106,
    0.03907,
    0.00444,
    -0.01888,
    0.00067,
    -0.0095,
    -0.00573,
    -0.01057,
    -0.03213,
    0.02483,
    -0.01392,
    0.00518,
    -0.01345,
    -0.04136,
    0.00781,
    -0.01741,
    0.01333,
    0.0862,
    0.00087,
    0.06179,
    0.00812,
    0.01724,
    -0.00138,
    -0.02419,
    -0.04113,
    0.03227,
    0.01737,
    -0.01054,
    0.02399,
    -0.00753,
    -0.01763,
    -0.02027,
    0.02187,
    0.04488,
    -0.00948,
    0.00936,
    0.00557,
    0.01701,
    0.00296,
    0.02513,
    0.00647,
    -0.02689,
    0.00932,
    0.00232,
    0.02114,
    -0.01153,
    0.02071,
    -0.01033,
    -0.00216,
    -0.02269,
    0.02498,
    0.08188,
    0.01957,
    0.0257,
    0.03544,
    -0.02361,
    0.01163,
    -0.03442,
    -0.01703,
    -0.01474,
    -0.03003,
    0.00842,
    0.01761,
    0.04353,
    0.06923,
    0.02127,
    -0.07331,
    0.01905,
    -0.15399,
    -0.00953,
    -0.02401,
    0.01896,
    -0.0092,
    -0.04405,
    0.00841,
    -0.00724,
    -0.00231,
    -0.00551,
    0.02073,
    0.00276,
    -0.00429,
    -0.02748,
    0.01656,
    0.02618,
    0.00144,
    0.00339,
    -0.02033,
    -0.00894,
    -0.0026,
    -0.02124,
    -0.01641,
    -0.02425,
    -0.00911,
    -0.02153,
    -0.00991,
    0.00943,
    0.0279,
    0.00814,
    0.01418,
    0.01059,
    0.03347,
    0.01633,
    -0.00783,
    0.0163,
    -0.02656,
    0.03654,
    0.00544,
    -0.05375,
    0.02113,
    0.01199,
    0.01786,
    0.01657,
    0.02678,
    -0.02393,
    -0.02382,
    -0.01026,
    0.00444,
    -0.04363,
    0.00461,
    -0.02311,
    0.02119,
    0.03423,
    -0.03947,
    0.00827,
    0.00428,
    0.02509,
    0.05523,
    0.00968,
    0.00736,
    -0.02842,
    -0.00494,
    -0.00789,
    0.00092,
    0.01074,
    -0.05231,
    -0.02417,
    0.03378,
    -0.0419,
    -0.01937,
    0.00772,
    0.00649,
    -0.00354,
    0.00986,
    0.02533,
    0.03732,
    0.00573,
    0.01519,
    0.04731,
    0.01592,
    0.00591,
    0.00489,
    0.02472,
    0.00412,
    -0.03495,
    0.00891,
    -0.04425,
    -0.02803,
    -0.02751,
    0.00463,
    0.01096,
    0.03423,
    0.01053,
    -0.00529,
    0.00232,
    -0.00123,
    -0.01171,
    0.01875,
    0.00582,
    -0.03055,
    -0.00971,
    0.0136,
    -0.00037,
    0.00564,
    -0.02046,
    -0.01213,
    0.0548,
    -0.02994,
    0.00769,
    0.02943,
    -0.01911,
    0.03301,
    -0.01255,
    0.00165,
    0.01757,
    -0.02918,
    -0.01783,
    0.03909,
    -0.02181,
    0.00412,
    0.02763,
    -0.00298,
    -0.00911,
    -0.01504,
    -0.00259,
    0.01203,
    0.01907,
    0.03962,
    -0.03799,
    0.0206,
    0.03839,
    -0.0073,
    0.0141,
    0.01413,
    -0.0009,
    -0.03482,
    -0.05371,
    0.00736,
    -0.00887,
    0.00668,
    0.02266,
    -0.02546,
    -0.01566,
    -0.02934,
    -0.01554,
    0.02998,
    0.01749,
    -0.00253,
    -0.02354,
    0.02297,
    -0.00775,
    -0.01372,
    0.00527,
    0.00517,
    0.0038,
    -0.00148,
    0.00735,
    0.04075,
    0.00949,
    0.01031,
    -0.0002,
    0.04806,
    -0.01936,
    -0.03375,
    -0.01404,
    0.0028,
    -0.00024,
    -0.01469,
    -0.02673,
    -0.01449,
    0.01669,
    -0.03093,
    -0.02347,
    0.03441,
    -0.01124,
    0.05166,
    -0.01126,
    -0.0304,
    0.0244,
    0.01113,
    -0.03645,
    -0.00362,
    -0.0282,
    -0.01398,
    -0.00296,
    0.02035,
    -0.00709,
    -0.01669,
    0.0052,
    0.04225,
    0.01238,
    0.05311,
    0.00123,
    0.01978,
    0.02754,
    -0.00399,
    -0.05899,
    0.01471,
    0.01069,
    0.00584,
    0.01264,
    0.02726,
    0.01045,
    -0.0234,
    -0.02058,
    0.00416,
    0.01582,
    -0.01001,
    0.05516,
    -0.0086,
    -0.00371,
    0.00856,
    0.02068,
    0.04512,
    0.01539,
    0.02483,
    0.03069,
    -0.00894,
    -0.01847,
    -0.03049,
    -0.03157,
    -0.0652,
    -0.00583,
    -0.12463,
    -0.02523,
    0.05638,
    -0.01216,
    -0.02967,
    -0.03133,
    -0.0497,
    -0.01375,
    0.01742,
    -0.0299,
    -0.03271,
    0.00038,
    0.02796,
    -0.01146,
    -0.00586,
    0.00128,
    -0.01932,
    -0.00988,
    -0.01969,
    0.00117,
    -0.02229,
    0.00803,
    0.0274,
    0.06687,
    -0.01548,
    -0.01661,
    0.03553,
    -0.03332,
    0.01326,
    0.02227,
    0.01577,
    0.0168,
    0.0344,
    0.03387,
    0.00968,
    0.00729,
    -0.06178,
    0.02877,
    -0.01978,
    0.01249,
    -0.00992,
    0.04132,
    0.00772,
    -0.03154,
    -0.0235,
    0.0087,
    0.00317,
    -0.01245,
    -0.01564,
    -0.02725,
    0.01701,
    -0.01033,
    0.00537,
    -0.00196,
    0.03218,
    -0.04125,
    -0.00197,
    0.03126,
    0.01882,
    -0.05861,
    -0.00222,
    -0.01899,
    0.03213,
    0.01619,
    -0.01989,
    -0.02378,
    -0.0148,
    -0.01966,
    -0.08009,
    -0.00483,
    -0.01871,
    -0.00188,
    0.0054,
    -0.06682,
    0.01724,
    -0.0057,
    -0.01324,
    -0.00657,
    -0.00439,
    -0.0111,
    -0.03548,
    0.02276,
    0.01216,
    0.00039,
    -0.02335,
    -0.01615,
    0.03401,
    0.00617,
    0.04591,
    -0.01163,
    -0.02588,
    0.05928,
    -0.01629,
    -0.0179,
    0.03311,
    -0.00288,
    -0.03757,
    0.00964,
    0.05133,
    0.00242,
    -0.03141,
    0.01689,
    0.0009,
    0.02437,
    -0.03016,
    0.01334,
    -0.01348,
    0.00812,
    0.0111,
    -0.03031,
    -0.01411,
    0.01557,
    0.00068,
    0.01432,
    0.01834,
    0.0137,
    0.03364,
    0.02836,
    0.03442,
    0.01836,
    0.00532,
    0.00327,
    0.03296,
    0.02579,
    -0.00112,
    -0.00253,
    0.00316,
    -0.02912,
    0.00765,
    -0.02719,
    -0.03237,
    -0.02641,
    -0.02412,
    0.01768,
    -0.02882,
    0.02822,
    0.04747,
    -0.01385,
    -0.00697,
    -0.03924,
    -0.00604,
    -0.01123,
    0.04574,
    0.03138,
    -0.00355,
    0.00289,
    0.00227,
    -0.01244,
    -0.02033,
    -0.02493,
    0.01042,
    -0.00022,
    0.02149,
    -0.00899,
    -0.01729,
    0.00106,
    0.01171,
    -0.0035,
    -0.00492,
    0.01233,
    0.03044,
    0.01634,
    -0.02222,
    0.00669,
    -0.0079,
    0.00747,
    -0.03611,
    0.01595,
    0.009,
    -0.0134,
    0.02235,
    0.01068,
    0.04888,
    -0.03529,
    -0.03183,
    0.04858,
    -0.02398,
    -0.0238,
    0.05134,
    -0.01387,
    -0.01827,
    0.04571,
    0.02244,
    0.0017,
    -0.01323,
    -0.00466,
    -0.03193,
    0.01697,
    -0.01232,
    -0.0043,
    -0.02951,
    0.01074,
    0.01882,
    0.01245,
    0.00618,
    0.04231,
    0.03744,
    -0.03155,
    -0.02083,
    -0.00647,
    0.00045,
    0.00961,
    0.00339,
    0.02477,
    0.01092,
    0.00919,
    -0.02226,
    0.01359,
    0.04045,
    -0.02451,
    -0.01352,
    0.00027,
    0.04516,
    0.09131,
    -0.05001,
    0.02617,
    -0.0389,
    0.00024,
    0.01439,
    -0.00963,
    -0.02962,
    -0.00299,
    0.03009,
    0.00816,
    0.01915,
    -0.03052,
    0.00614,
    0.01443,
    -0.01738,
    0.01974,
    -0.00716,
    0.02555,
    -0.01831,
    -0.02042,
    -0.00406,
    0.04665,
    0.00062,
    -0.01675,
    0.02964,
    0.00795,
    0.00582,
    0.01592,
    0.00157,
    -0.03995,
    -0.00542,
    0.00082,
    -0.019,
    0.02341,
    0.00544,
    -0.02913,
    -0.02254,
    0.01204,
    0.02614,
    -0.01073,
    -0.01367,
    -0.05812,
    0.00462,
    -0.04753,
    0.00574,
    0.00859,
    -0.00407,
    -0.04953,
    0.05491,
    -0.00399,
    -0.00744,
    -0.00032,
    -0.00599,
    0.01344,
    0.00432,
    -0.05186,
    -0.01189,
    0.02944,
    0.0247,
    -0.01394,
    -0.00054,
    0.02685,
    0.00627,
    0.04506,
    -0.01249,
    0.03847,
    0.01573,
    0.02103,
    -0.00931,
    -0.0162,
    -0.03199,
    -0.14278,
    0.02366,
    -0.01395,
    -0.0694,
    -0.02008,
    -0.01598,
    0.00056,
    0.01979,
    0.01936,
    0.03409,
    0.03767,
    -2e-05,
    0.00387,
    -0.03456,
    -0.00865,
    -0.01656,
    -0.01735,
    -0.00556,
    -0.04729,
    0.04122,
    -0.0066,
    0.01141,
    0.01065,
    -0.03196,
    0.00202,
    0.0447,
    0.00227,
    0.03298,
    0.00111,
    -0.03409,
    -0.0133,
    0.01786,
    0.00332,
    0.0002,
    -0.01747,
    -0.00905,
    0.15856,
    -0.00329,
    0.01963,
    -0.0019,
    -0.02372,
    -0.01301,
    0.00077,
    -0.03443,
    -0.02205,
    0.04248,
    -0.00636,
    -0.03072,
    -0.05194,
    -0.03101,
    -0.01614,
    -0.01745,
    -0.02258,
    0.0071,
    0.01651,
    0.02166,
    -0.00649,
    -0.01591,
    0.01084,
    0.05814,
    -0.00245,
    0.03068,
    0.03576,
    -0.01012,
    -0.01837,
    0.00362,
    -0.03432,
    0.00248,
    -0.00162,
    0.00658,
    -0.00703,
    -0.01969,
    0.00108,
    0.00449,
    0.01143,
    0.03572,
    -0.01686,
    -0.00931,
    -0.00709,
    0.01188,
    0.00961,
    0.01633,
    0.01301,
    0.01453,
    0.01063,
    0.01716,
    0.02111,
    0.00248,
    -0.00092,
    0.03655,
    -0.00357,
    -0.04482,
    0.0244,
    0.06528,
    -0.00539,
    0.02151,
    -0.01466,
    0.0196,
    -0.01754,
    0.03099,
    -0.02581,
    -0.01499,
    0.01699,
    0.00563,
    0.00644,
    -0.02317,
    -0.00643,
    -0.008,
    -0.02429,
    0.00263,
    -0.02628,
    0.01991,
    -0.02986,
    -0.03469,
    -0.02476,
    -0.03661,
    0.00311,
    0.0225,
    0.01645,
    -0.01131,
    0.04125,
    -0.06445,
    0.00922,
    -0.02422,
    -0.01628,
    -0.00176,
    -0.011,
    -0.00832,
    0.00862,
    0.01015,
    0.04364,
    -0.02967,
    -0.04897,
    0.01939,
    -0.03877,
    0.02013,
    0.02572,
    -0.01772,
    -0.01742,
    0.02186,
    -0.00573,
    -0.02893,
    -0.02983,
    0.01708,
    -0.00554,
    -0.018,
    0.03978,
    0.005,
    -0.00938,
    -0.01279,
    0.02255,
    0.01808,
    -0.02341,
    -0.00429,
    -0.05738,
    -0.02906,
    0.02386,
    0.03275,
    -0.01162,
    0.0416,
    -0.01026,
    0.03241,
    -0.02302,
    -0.06834,
    0.00832,
    -0.02476,
    0.00453,
    -0.04606,
    0.00683,
    0.00105,
    0.02716,
    0.00527,
    -0.00604,
    -0.00381,
    -0.04266,
    -0.02075,
    0.00342,
    0.00387,
    0.00959,
    -0.04617,
    0.00022,
    0.02232,
    -0.03455,
    -0.0049,
    0.01245,
    -0.03202,
    0.01185,
    0.02699,
    0.00312,
    -0.0041,
    0.02101,
    0.01897,
    0.00305,
    0.01487,
    -0.01214,
    -0.00696,
    0.01574,
    0.03405,
    -0.01939,
    0.00357,
    -0.03749,
    -0.05377,
    -0.01708,
    0.0164,
    -0.00117,
    -0.0072,
    -0.00741,
    -0.01956,
    0.01151,
    -0.01753,
    -0.03228,
    -0.01839,
    0.00699,
    -0.01776,
    0.03862,
    -0.03594,
    0.00427,
    0.02452,
    -0.00587,
    0.0009,
    0.01513,
    -0.01248,
    0.01766,
    0.00546,
    0.03017,
    -0.01777,
    -0.00228,
    0.03893,
    -0.00526,
    -0.04726,
    0.02264,
    -0.01944,
    0.0044,
    -0.03394,
    -0.04133,
    -0.02896,
    0.01209,
    -0.05378,
    -0.00017,
    0.02089,
    -0.00913,
    -0.00118,
    0.02921,
    0.03596,
    0.01512,
    0.01528,
    -0.02475,
    -0.04818,
    -0.02804,
    0.01589,
    -0.02209,
    -0.00599,
    -0.02246,
    0.00313,
    -0.00424,
    0.00848,
    0.01173,
    0.04413,
    -0.02482,
    0.0099,
    0.01926,
    -0.01066,
    0.03542,
    0.02467,
    -0.00091,
    0.03619,
    0.01478,
    -0.03621,
    0.02315,
    -0.04165,
    0.02459,
    -0.02222,
    -0.013,
    -0.00122,
    0.01052,
    -0.0061,
    -0.04416,
    -0.03344,
    0.02859,
    -0.00179,
    -0.00468,
    -0.01358,
    0.03447,
    0.02128,
    0.04292,
    -0.01662,
    0.0044,
    0.0097,
    0.01132,
    0.02716,
    0.03915,
    -0.0319,
    -0.00525,
    -0.00066,
    -0.01804,
    -0.04265,
    -0.00427,
    -0.01112,
    -0.04051,
    0.02506,
    -0.00266,
    -0.05618,
    -0.00459,
    0.01069,
    -0.02182,
    0.01723,
    0.01498,
    0.01509,
    0.04383,
    -0.00909,
    4e-05,
    -0.01634,
    -0.00031,
    -0.0478,
    -0.01777,
    0.01533,
    0.02134,
    -0.03977,
    0.00252,
    -0.03394,
    -0.00049,
    0.01513,
    0.01319,
    -0.02001,
    0.0045,
    -0.00937,
    -0.00171,
    0.02408,
    -0.00985,
    -0.01124,
    0.0303,
    -0.01454,
    -0.00811,
    0.0377,
    -0.00956,
    -0.00117,
    -0.01482,
    0.03912,
    0.0021,
    0.02574,
    -0.02052,
    0.03977,
    0.02601,
    0.02147,
    -0.02271,
    -0.00941,
    0.03314,
    0.06619,
    0.01908,
    -0.03895,
    -0.05335,
    0.01153,
    -0.00404,
    0.00161,
    -0.01636,
    0.03234,
    -0.01549,
    -0.08552,
    -7e-05,
    0.02093,
    -0.00132,
    0.0057,
    -0.00358,
    -0.035,
    0.02413,
    -0.01093,
    -0.0027,
    0.01034,
    0.01168,
    0.03988,
    -0.00774,
    -0.0041,
    -0.00276,
    0.0071,
    0.04605,
    -3e-05,
    -0.01447,
    -0.00769,
    -0.0203,
    -0.00421,
    -0.00973,
    -0.01403,
    -0.00447,
    -0.00663,
    -0.00471,
    0.00461,
    -0.00688,
    0.02191,
    -0.02714,
    -0.0046,
    -0.01482,
    -0.02646,
    0.00252,
    0.01503,
    -0.00888,
    0.00851,
    0.03852,
    0.01325,
    0.02933,
    -0.0112,
    -0.01726,
    0.0108,
    -0.02258,
    -0.01688,
    -0.01075,
    0.00342,
    -0.00295,
    -0.00518,
    0.0024,
    -0.00744,
    0.03563,
    -0.01804,
    0.00314,
    0.01539,
    0.03404,
    -0.02561,
    -0.01257,
    0.03869,
    0.00053,
    -0.03075,
    -0.03695,
    0.02431,
    -0.01086,
    0.02177,
    0.00329,
    0.02102,
    -0.02151,
    0.03307,
    0.00642,
    -0.0006,
    -0.06003,
    -0.01517,
    -0.01058,
    -0.01148,
    -0.00489,
    -0.01458,
    -0.00131,
    0.0234,
    -0.01234,
    0.02696,
    3e-05,
    0.03317,
    0.0049,
    -0.01448,
    0.00999,
    -0.02163,
    -0.00997,
    0.03636,
    0.01406,
    -0.00548,
    -0.02823,
    -0.00834,
    0.01168,
    -0.01101,
    -0.0002,
    0.00497,
    0.00591,
    -0.01337,
    -0.01135,
    0.00956,
    0.01862,
    -0.0495,
    -0.0283,
    -0.00033,
    0.03785,
    0.00084,
    0.01899,
    -0.01679,
    -0.0106,
    -0.01003,
    0.01764,
    -0.00958,
    -0.00192,
    0.00095,
    0.00469,
    0.02143,
    -0.00863,
    -0.00303,
    -0.01735,
    -0.00377,
    -0.01057,
    0.01171,
    -0.01115,
    -0.00924,
    -0.02739,
    -0.0143,
    -0.02985,
    0.02918,
    -0.01729,
    -0.00883,
    -0.00495,
    0.01907,
    -0.0208,
    0.00099,
    0.02597,
    0.0209,
    0.01144,
    0.0174,
    -0.00816,
    -0.01864,
    -9e-05,
    -0.00917,
    -0.00576,
    -0.00235,
    0.00556,
    -0.02446,
    0.05314,
    0.01305,
    -0.0106,
    0.01129,
    0.00589,
    -0.06937,
    0.00789,
    -0.01729,
    -0.00104,
    0.00936,
    -0.03902,
    0.00653,
    -0.01278,
    -0.02492,
    0.01937,
    -0.0179,
    0.01647,
    0.00814,
    0.00106,
    0.03111,
    0.03377,
    -0.00401,
    -0.00753,
    0.02479,
    0.0395,
    0.00334,
    0.02981,
    0.03239,
    -0.03083,
    -0.01766,
    0.00199,
    -0.02624,
    -0.04174,
    -0.00292,
    0.00272,
    3e-05,
    -0.07763,
    0.04773,
    0.0153,
    -0.02773,
    -0.01917,
    -0.01347,
    0.0072,
    0.03635,
    0.04885,
    0.01354,
    0.00361,
    0.00193,
    -0.01169,
    0.01709,
    -0.13082,
    -0.01791,
    0.0313,
    -0.11922,
    -0.00487,
    0.01997,
    -0.00066,
    -0.01698,
    0.00615,
    -0.01062,
    0.01968,
    -0.02297,
    -0.00828,
    0.0487,
    -0.02892,
    -0.01323,
    0.00808,
    -0.00605,
    0.00651,
    0.03278,
    -0.02435,
    0.00014,
    0.00262,
    -0.00969,
    0.00694,
    -0.02585,
    0.01496,
    0.03043,
    -0.01008,
    -0.04206,
    -0.02701,
    -0.02034,
    0.01404,
    0.01695,
    0.052,
    -0.00689,
    -0.01073,
    -0.0018,
    0.01859,
    -0.00431,
    0.00132,
    -0.04087,
    0.00092,
    0.0205,
    -0.05299,
    0.00855,
    -0.00082,
    -0.00207,
    -0.00345,
    0.01178,
    0.02863,
    0.03099,
    0.00276,
    -0.01979,
    0.04236,
    0.0119,
    -0.01849,
    0.0065,
    -0.02589,
    -0.0117,
    0.005,
    -0.02833,
    0.00932,
    -0.01712,
    0.02929,
    0.00166,
    0.00476,
    -0.00531,
    -0.03606,
    -0.00832,
    -0.00176,
    0.00138,
    -0.00782,
    0.02993,
    0.01206,
    -0.01095,
    -0.02488,
    0.02014,
    -0.01655,
    -0.0108,
    -0.01254,
    0.01223,
    0.00066,
    0.00302,
    0.03862,
    -0.02677,
    -0.01961,
    -0.02239,
    -0.04599,
    -0.03923,
    -0.00847,
    0.03007,
    -0.02269,
    -0.01593,
    -0.01995,
    0.01831,
    0.016,
    -0.00182,
    0.00607,
    0.01547,
    -0.00031,
    0.00582,
    -0.00715,
    0.00339,
    0.0065,
    0.04518,
    -0.03055,
    -0.05221,
    -0.03499,
    0.01198,
    -0.00225,
    -0.00355,
    -0.02023,
    0.04136,
    0.01845
   ]
  }
 },
 "being_text": "BEING: Pure existence, undifferentiated",
 "being_vec": [
  0.0,
  0.0,
  0.0,
  0.5,
  0.5,
  0.5,
  0.5,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0
 ],
 "xbar_pack": [
  -0.001281,
  0.135873,
  0.22451,
  0.389162,
  0.413952,
  0.405762,
  0.393034,
  0.014211,
  0.113298,
  0.180457,
  0.336806,
  0.353241,
  0.354828,
  0.335853
 ],
 "cloud_mean": {
  "base14": [
   0.00378,
   0.00132,
   -0.00063,
   -0.00081,
   0.0002,
   -0.00242,
   -0.00081,
   -5e-05,
   -0.00013,
   0.00042,
   -0.00269,
   -0.00301,
   0.00122,
   -0.0004,
   -0.001,
   0.00066,
   -0.00106,
   0.00061,
   -5e-05,
   -0.00152,
   -0.00041,
   -0.00043,
   -0.00159,
   -0.00227,
   0.00117,
   -0.00052,
   0.00083,
   -0.00161,
   -0.00416,
   9e-05,
   -0.00012,
   0.00068,
   0.00094,
   -0.00815,
   -0.00062,
   0.00081,
   -0.00219,
   -0.00022,
   -0.00601,
   -0.00225,
   0.00039,
   -0.00061,
   0.00034,
   -0.00054,
   -0.00116,
   0.00059,
   -0.0017,
   -0.00045,
   0.00108,
   0.00096,
   0.0027,
   -0.00523,
   -0.0005,
   -0.00091,
   -0.00075,
   0.00223,
   0.00555,
   -0.00132,
   0.00471,
   0.00397,
   -5e-05,
   -0.00035,
   0.00251,
   -0.00019,
   -0.0006,
   0.00269,
   0.00168,
   -0.00479,
   0.00036,
   -0.00209,
   -0.0004,
   0.0057,
   -0.00085,
   0.00043,
   -0.00154,
   -0.00257,
   0.00018,
   0.00073,
   -0.00159,
   0.001,
   6e-05,
   -0.00267,
   -0.00092,
   0.00042,
   -0.00079,
   0.00011,
   -0.00078,
   0.0001,
   0.00287,
   -0.00039,
   0.00065,
   -0.0016,
   -0.00154,
   -0.00211,
   -0.00176,
   -0.00035,
   0.00188,
   0.00017,
   -0.00025,
   -0.00012,
   -0.0001,
   -0.00437,
   0.02014,
   8e-05,
   -0.00239,
   0.00338,
   0.00111,
   0.00067,
   -0.00037,
   -0.00043,
   -0.00241,
   0.00028,
   -0.00036,
   -0.00023,
   0.00023,
   -0.00021,
   0.00022,
   0.00032,
   -0.00302,
   0.00085,
   0.00182,
   0.00248,
   0.00249,
   0.00305,
   -0.00047,
   -0.00181,
   -0.0005,
   0.00329,
   -0.00059,
   3e-05,
   -0.00119,
   -0.00469,
   -0.00022,
   0.0008,
   0.00041,
   4e-05,
   -0.00048,
   0.00068,
   -0.00399,
   -0.00096,
   -1e-05,
   -0.00118,
   -0.00473,
   -0.002,
   -0.00056,
   -0.00275,
   0.00067,
   0.0951,
   -0.00048,
   0.00088,
   0.00196,
   0.00043,
   -0.00565,
   -0.00048,
   -0.00528,
   0.00319,
   0.001,
   -0.0024,
   0.00076,
   0.00128,
   0.00249,
   0.00218,
   -0.00079,
   -0.00147,
   0.00188,
   -0.00031,
   -0.00168,
   -0.00098,
   0.00047,
   -0.00022,
   -0.00062,
   -0.00102,
   -0.00043,
   0.00034,
   -0.00333,
   0.00095,
   0.00101,
   0.00014,
   0.00203,
   0.00045,
   -0.00042,
   -0.00192,
   -0.00018,
   -0.00012,
   0.00122,
   -0.00522,
   -0.0046,
   -0.00089,
   0.00172,
   -0.00134,
   0.00063,
   0.00017,
   -0.00122,
   -0.00025,
   -6e-05,
   0.00018,
   -9e-05,
   1e-05,
   -0.00109,
   0.00281,
   -0.00169,
   3e-05,
   0.00064,
   -0.00036,
   0.00186,
   0.00128,
   0.00158,
   -0.00032,
   0.00035,
   -0.00055,
   -0.00184,
   0.00106,
   -0.00465,
   0.00196,
   0.00068,
   -0.00558,
   -0.00106,
   -0.00061,
   -0.00031,
   0.00036,
   0.0001,
   -0.00049,
   -0.00858,
   3e-05,
   -0.0014,
   -0.0002,
   -0.00352,
   0.00108,
   0.00751,
   0.00097,
   0.00014,
   -0.0003,
   -0.00084,
   0.00051,
   3e-05,
   -0.00365,
   -0.00158,
   -0.00077,
   -0.00059,
   0.00118,
   -0.00202,
   0.00058,
   0.00211,
   0.00053,
   -0.00078,
   -0.0002,
   0.00149,
   0.00019,
   0.00021,
   0.0005,
   -0.00054,
   0.00419,
   0.00057,
   0.00173,
   0.00192,
   -0.01166,
   0.00015,
   -6e-05,
   -0.00056,
   0.00093,
   -0.00012,
   -0.00113,
   0.00081,
   -0.00118,
   0.00065,
   0.00191,
   0.00549,
   0.00019,
   0.00134,
   0.0015,
   5e-05,
   0.0016,
   0.00018,
   9e-05,
   -0.00045,
   0.00934,
   -0.00201,
   -2e-05,
   0.00118,
   -0.00011,
   -5e-05,
   0.00012,
   -9e-05,
   0.00227,
   -0.00037,
   -0.00287,
   0.12433,
   -0.00016,
   -0.00183,
   -0.00049,
   -0.00017,
   0.00028,
   -0.00062,
   -0.0002,
   -0.00302,
   -0.00344,
   0.0008,
   -0.00082,
   -0.00124,
   0.00057,
   0.00044,
   0.00485,
   0.00018,
   -0.00156,
   0.0001,
   0.00078,
   -0.00086,
   -0.00268,
   0.00027,
   -0.00072,
   0.00559,
   0.00063,
   -7e-05,
   0.00193,
   -0.00107,
   -0.00013,
   0.00157,
   -0.00057,
   -0.0004,
   -0.00317,
   0.00115,
   0.0006,
   -0.00026,
   0.00369,
   -0.00015,
   -0.00081,
   -0.0004,
   0.00404,
   -0.00135,
   -0.00139,
   0.00041,
   0.0005,
   3e-05,
   0.00031,
   -0.0003,
   0.00686,
   -0.00023,
   0.00309,
   0.00202,
   0.00095,
   0.00092,
   0.00026,
   0.00036,
   0.00032,
   -0.00056,
   -8e-05,
   -0.00277,
   -0.00083,
   -0.00031,
   0.00073,
   0.00151,
   0.00019,
   -0.00453,
   0.00102,
   -0.03415,
   0.00045,
   0.00234,
   -9e-05,
   -0.00033,
   -8e-05,
   -0.01107,
   0.00148,
   -0.00269,
   -0.00202,
   0.00091,
   0.00186,
   -0.00132,
   0.00155,
   0.00234,
   -0.00068,
   0.00094,
   -6e-05,
   -0.00116,
   -0.00247,
   -0.00231,
   0.00156,
   0.00089,
   0.00185,
   0.00025,
   0.00156,
   0.00038,
   0.00066,
   0.00056,
   -0.00052,
   0.00243,
   -0.00027,
   0.00113,
   0.0008,
   -0.00249,
   -0.00061,
   -0.00018,
   0.00058,
   0.00029,
   -0.00068,
   -0.00113,
   0.00465,
   -0.00118,
   -0.01953,
   0.00095,
   0.00038,
   0.00202,
   0.00086,
   -0.00329,
   -0.00155,
   0.00015,
   -0.00048,
   8e-05,
   0.00033,
   0.59905,
   0.00217,
   -0.00234,
   -0.00172,
   0.00095,
   0.00057,
   -0.00325,
   0.00828,
   0.00014,
   9e-05,
   0.01595,
   0.00074,
   -0.00047,
   -0.00133,
   -0.00051,
   -0.00194,
   0.01071,
   0.00027,
   0.00035,
   0.00174,
   -0.00106,
   -0.0015,
   -0.00034,
   0.00046,
   0.00012,
   0.00033,
   -0.00298,
   0.00175,
   0.0067,
   -0.00651,
   -0.00029,
   -0.00088,
   0.00125,
   0.00779,
   -0.00479,
   -0.00088,
   0.00154,
   -0.0003,
   0.00041,
   -0.00044,
   -0.00033,
   0.00013,
   0.00069,
   0.00023,
   -0.00177,
   0.00027,
   0.00085,
   -0.00012,
   8e-05,
   -0.00093,
   -0.00161,
   -0.00072,
   -0.00137,
   -0.00035,
   0.00103,
   -0.00198,
   0.00186,
   -0.00299,
   -0.00314,
   -0.00081,
   0.00202,
   -0.00033,
   -0.00082,
   0.00225,
   -0.00139,
   0.00061,
   -0.00018,
   0.00129,
   -0.00015,
   0.00387,
   0.002,
   -0.00102,
   0.00033,
   0.00101,
   0.0006,
   1e-05,
   -0.00341,
   0.00217,
   -0.00044,
   0.00031,
   -0.00043,
   0.00066,
   -0.00111,
   -0.00013,
   -0.00063,
   -0.0005,
   -0.00082,
   -0.00058,
   0.00087,
   -0.00036,
   -0.00174,
   0.00166,
   -0.00356,
   -0.00094,
   0.00064,
   -0.02028,
   -0.00064,
   0.00196,
   0.00048,
   -0.00147,
   0.0018,
   0.00047,
   -0.00654,
   -0.00066,
   -0.00047,
   -0.00099,
   0.00295,
   0.00141,
   -0.00045,
   0.00125,
   0.02639,
   0.0002,
   -0.4762,
   1e-05,
   0.00067,
   -0.00082,
   0.00022,
   -0.00436,
   0.00087,
   -0.0013,
   -0.00106,
   -0.00117,
   -0.00378,
   -0.00083,
   -0.00068,
   0.00138,
   0.00118,
   -0.00076,
   0.0003,
   0.00035,
   0.00204,
   0.00152,
   -0.00222,
   -0.00059,
   -0.00026,
   0.00079,
   0.00917,
   -0.0004,
   0.00172,
   0.00054,
   0.00356,
   0.00071,
   0.00126,
   0.00045,
   0.00194,
   0.12031,
   -0.00111,
   -0.0026,
   -0.00054,
   -7e-05,
   -0.00086,
   0.00121,
   0.00069,
   0.0001,
   0.00085,
   0.00048,
   0.00017,
   0.00086,
   -0.00089,
   -0.0003,
   -0.00019,
   0.0011,
   -0.0021,
   0.0004,
   0.00042,
   -0.00466,
   0.0004,
   -0.00248,
   -0.00015,
   0.00054,
   -0.00327,
   0.00057,
   -0.00112,
   0.0015,
   0.01586,
   -0.00099,
   -3e-05,
   -0.00386,
   0.00038,
   2e-05,
   -0.00109,
   0.01549,
   0.00107,
   0.00016,
   0.00145,
   0.00263,
   0.00046,
   -0.00013,
   9e-05,
   -2e-05,
   0.00051,
   0.00361,
   -0.00073,
   0.001,
   0.001,
   0.00148,
   0.00031,
   0.00178,
   -0.00366,
   -0.00018,
   -0.00128,
   -0.35993,
   -0.0001,
   -0.0004,
   -0.00025,
   0.00147,
   -0.00272,
   0.0003,
   -0.00156,
   0.00088,
   -0.0069,
   -0.00094,
   0.00306,
   -0.00158,
   0.00149,
   0.00229,
   -0.00027,
   -0.00073,
   -0.00105,
   0.00054,
   -0.00017,
   0.00121,
   0.0004,
   -0.00056,
   -0.00056,
   -0.00103,
   -0.00021,
   0.00037,
   -0.00132,
   -0.00066,
   -0.00118,
   -7e-05,
   0.00025,
   0.00027,
   0.00024,
   -0.00457,
   0.0018,
   -6e-05,
   -0.00027,
   0.00091,
   -0.00109,
   0.0019,
   -0.00426,
   0.00071,
   0.00157,
   -0.00015,
   0.00622,
   0.00038,
   -0.00134,
   -0.00127,
   -0.00095,
   0.00125,
   0.00104,
   -0.00539,
   -7e-05,
   -0.00162,
   2e-05,
   -0.00025,
   -0.00178,
   0.00026,
   -0.00221,
   0.00134,
   -0.0006,
   0.00319,
   0.0003,
   -9e-05,
   -0.00102,
   0.00039,
   -0.0028,
   -0.00056,
   0.0005,
   0.00387,
   0.0016,
   -0.00051,
   0.00045,
   7e-05,
   0.00015,
   0.00263,
   0.00086,
   -0.00075,
   -0.00075,
   -0.00051,
   0.0005,
   -0.00178,
   -0.00052,
   -0.00033,
   0.00344,
   -0.00079,
   0.00126,
   0.00066,
   0.00029,
   -0.00065,
   9e-05,
   0.00132,
   0.00141,
   -0.00114,
   -0.00073,
   0.00015,
   -0.0029,
   -0.00245,
   -0.00023,
   0.00158,
   0.00097,
   0.00011,
   -8e-05,
   0.09209,
   0.00166,
   0.00024,
   -0.00075,
   0.00043,
   -0.00466,
   3e-05,
   -6e-05,
   0.00368,
   0.00198,
   -0.00816,
   9e-05,
   -0.00024,
   -0.00157,
   -5e-05,
   -0.00056,
   -0.00224,
   -0.00076,
   0.00034,
   0.0016,
   -0.00318,
   -0.00035,
   -0.01089,
   -0.0004,
   -0.00026,
   -0.00085,
   -0.00198,
   -0.00097,
   0.00262,
   -0.00554,
   -0.00142,
   0.00038,
   -0.00069,
   -0.00088,
   0.00236,
   4e-05,
   0.0008,
   0.0096,
   -0.00128,
   -0.00131,
   0.00219,
   -0.00016,
   -0.00023,
   -0.00011,
   0.00055,
   -0.00151,
   0.00149,
   0.0007,
   -0.00069,
   -0.00049,
   0.00092,
   -0.00096,
   -0.00075,
   -6e-05,
   0.0005,
   0.00121,
   0.00029,
   -0.00067,
   -0.00183,
   0.00075,
   -0.00071,
   0.00207,
   0.00278,
   -0.00021,
   0.00272,
   -0.00098,
   -0.00065,
   -0.00453,
   0.00059,
   -2e-05,
   0.0001,
   -0.00248,
   -0.0036,
   -0.00019,
   0.00303,
   0.00133,
   -0.00265,
   -5e-05,
   -0.00164,
   -0.00025,
   -6e-05,
   0.00459,
   0.00046,
   -0.00112,
   -0.00166,
   6e-05,
   -0.0017,
   0.00255,
   -0.00028,
   -0.02734,
   -0.00068,
   -0.00042,
   0.00045,
   -0.00041,
   -0.00052,
   0.00291,
   -0.00351,
   0.0001,
   -0.0009,
   0.00136,
   0.00739,
   -0.00088,
   0.00133,
   0.00139,
   -0.00131,
   0.00053,
   -0.00051,
   -0.00028,
   0.00057,
   0.00053,
   -0.00084,
   0.00037,
   0.00133,
   -0.00474,
   0.00143,
   -0.00069,
   -0.00845,
   0.0008,
   -0.00339,
   0.00195,
   0.00258,
   0.00203,
   0.00086,
   0.00089,
   -0.0008,
   -0.00034,
   3e-05,
   -0.00139,
   0.00064,
   -0.0026,
   0.0006,
   -0.0004,
   0.0011,
   0.00035,
   -0.00452,
   0.00112,
   0.00335,
   -0.00065,
   -0.00039,
   -0.00074,
   -0.00258,
   0.0006,
   0.00015,
   0.0034,
   0.0014,
   0.00444,
   0.00065,
   0.00041,
   0.00185,
   0.00013,
   3e-05,
   -6e-05,
   0.01079,
   0.00069,
   0.00547,
   0.00208,
   -0.0024,
   0.00228,
   0.00092,
   0.08084,
   0.0008,
   -0.00069,
   -0.00154,
   0.00018,
   -0.00153,
   -0.00119,
   0.00066,
   0.00069,
   -0.00045,
   -0.00122,
   1e-05,
   -0.00164,
   -0.00015,
   0.00014,
   0.00035,
   0.00038,
   -3e-05,
   -0.00302,
   0.00035,
   0.00211,
   0.00176,
   -0.00083,
   -0.00058,
   -0.00099,
   -0.00036,
   0.00493,
   0.00261,
   0.00056,
   0.00417,
   0.00049,
   -0.00314,
   0.0056,
   -0.00171,
   -0.00437,
   -0.00023,
   0.00244,
   0.00918,
   -0.00081,
   -0.00434,
   0.00305,
   0.00347,
   -0.00599,
   0.00087,
   -0.0001,
   -0.00024,
   -0.00108,
   -0.0017,
   -0.00016,
   -0.00082,
   -0.00092,
   1e-05,
   0.00072,
   -0.0003,
   -0.00073,
   -6e-05,
   0.00119,
   0.00032,
   -0.00066,
   -0.00123,
   -0.00155,
   0.00115,
   0.00087,
   -0.00061,
   -0.00011,
   4e-05,
   -0.00967,
   -0.00068,
   -0.4339,
   0.00077,
   0.00153,
   0.00238,
   -0.00204,
   -0.00066,
   -0.00022,
   0.00236,
   0.00021,
   0.00031,
   0.00079,
   -0.00118,
   0.00032,
   -0.00049,
   -0.00137,
   0.00202,
   8e-05,
   -0.00096,
   0.00044,
   -0.00084,
   -0.00091,
   -0.00109,
   0.00177,
   0.00053,
   0.00282,
   0.00197,
   0.00094,
   0.00698,
   0.00064,
   0.07905,
   -0.00281,
   -0.00138,
   -0.0006,
   0.00215,
   0.00129,
   0.00143,
   -0.00113,
   0.00059,
   0.00308,
   0.00083,
   -0.00747,
   -0.0008,
   -0.00017,
   -0.00372,
   -0.00014,
   -0.00116,
   0.00022,
   -0.00081,
   0.00137,
   -0.00096,
   0.0,
   0.00127,
   0.00064,
   0.00025,
   -3e-05,
   0.00577,
   0.00038,
   0.00047,
   -0.00111,
   0.00897,
   -0.00073,
   -0.00122,
   0.00816,
   0.0,
   -0.00091,
   0.00269,
   -0.00104,
   -0.00169,
   0.00044,
   0.00018,
   0.0015,
   -0.0024,
   -0.00054,
   0.00105,
   0.00048,
   0.00134,
   -0.00048,
   -0.00087,
   -0.00481,
   -0.00021,
   0.00012,
   0.00249,
   -0.00049,
   -0.00125,
   0.00021,
   0.00017,
   0.00088,
   0.00093,
   -0.00317,
   0.00167,
   0.00115,
   -0.00107,
   -0.00202,
   -0.00156,
   0.00146,
   -0.00081,
   -0.00168,
   -0.00136,
   9e-05,
   0.00101,
   0.00074,
   -0.00197,
   0.00077,
   7e-05,
   0.00018,
   0.00054,
   0.00014,
   0.00013,
   0.00051,
   -0.00063,
   -0.00567,
   -0.00025,
   9e-05,
   0.00142,
   -2e-05,
   0.00025,
   -0.00051,
   -0.00174,
   0.00036,
   -0.00112,
   -0.00282,
   -0.00062,
   0.00035,
   -0.00355,
   -0.00232,
   -0.00129,
   -0.00075,
   0.00031,
   0.0176,
   -0.00033,
   0.00043,
   -0.00026,
   0.00064,
   0.00631,
   0.00076,
   -0.00142,
   0.00152,
   0.00019,
   0.00054,
   0.00033,
   -1e-05,
   -0.00165,
   4e-05,
   -0.00023,
   0.00037,
   -0.00157,
   0.0003,
   0.00112,
   -0.00021,
   -0.00018,
   -0.00118,
   -0.00013,
   0.00042,
   -0.00685,
   0.00025,
   0.0011,
   0.00093,
   0.00105,
   -0.00255,
   -0.00486,
   -0.00154,
   -0.00515,
   0.00198,
   -0.00026,
   0.00351,
   -0.01002,
   0.00028,
   -0.00104,
   0.00274,
   -0.00074,
   -0.00023,
   0.00076,
   -0.00357,
   0.00046,
   -0.00025,
   -0.00152,
   0.00151,
   0.00077,
   0.00043,
   -0.00054,
   -0.00076,
   0.0012,
   0.00101,
   -0.00504,
   -0.00028,
   0.00653,
   0.00067,
   0.00115,
   0.00015,
   -0.00218,
   -1e-05,
   -0.00114,
   0.00181,
   -0.00081,
   -0.00144,
   0.00061,
   9e-05,
   -0.00045,
   -0.00031,
   0.00293,
   6e-05,
   0.0014,
   0.00028,
   -0.0006,
   0.00052,
   0.00069,
   0.00083,
   0.00011,
   -0.00071,
   -0.00166,
   -0.00023,
   0.00195,
   0.00316,
   -0.00073,
   -0.00091,
   -0.00179,
   0.00076,
   0.00097,
   8e-05,
   -0.00019,
   0.00224,
   -0.01004,
   -0.00019,
   -0.00226,
   0.00013,
   -0.00093,
   -0.01333,
   -0.00186,
   -0.00449,
   -0.00394,
   0.00203,
   -0.00038,
   -0.00178,
   -0.0863,
   0.00099,
   0.00063,
   -2e-05,
   -0.00373,
   0.00021,
   -0.00157,
   -0.00015,
   0.00052,
   -0.00416,
   0.00264,
   -0.00055,
   0.00073,
   -0.0001,
   -0.0004,
   0.00121,
   -0.00063,
   0.00034,
   -0.00089,
   1e-05,
   0.00034,
   0.0006,
   -0.00014,
   -0.00621,
   -0.00157,
   0.00143,
   0.00094,
   -0.01904,
   -0.00344,
   0.00076,
   -0.00135,
   0.00034,
   -0.00029,
   -0.00077,
   -0.00048,
   0.00015,
   0.00159,
   -0.00101,
   0.00021,
   -0.0012,
   0.00026,
   -0.01131,
   -0.00483,
   0.00017,
   0.00421,
   -0.00095,
   -0.00711,
   -0.00137,
   0.00094,
   -0.00054,
   0.00223,
   -0.00066,
   -0.00184,
   -0.00025,
   0.00117,
   -0.00052,
   0.08432,
   -0.00088,
   0.00201,
   0.00048,
   0.00075,
   5e-05,
   -0.00148,
   0.00027,
   -0.00152,
   -0.00017,
   0.00065,
   0.00042,
   0.00046,
   0.00015,
   0.00301,
   0.00091,
   -0.00093,
   0.00311,
   0.00022,
   -0.03519,
   -0.00167,
   -0.00061,
   -0.00264,
   0.00186,
   -0.0001,
   -0.00052,
   0.0031,
   0.0018,
   0.00027,
   0.00071,
   0.0021,
   2e-05,
   0.00184,
   0.00387,
   0.00114,
   0.00031,
   0.00115,
   -0.00061,
   0.00157,
   -0.00081,
   -0.00181,
   -0.00123,
   0.00057,
   -0.03579,
   -0.00388,
   -0.00066,
   0.00096,
   0.00012,
   -0.00508,
   0.00612,
   0.00315,
   -0.00046,
   0.0001,
   0.00148,
   0.00407,
   0.00051,
   0.0013,
   -0.00051,
   0.00117,
   0.00343,
   -0.00042,
   -0.00043,
   -2e-05,
   0.00207,
   0.00128,
   0.00032,
   -0.00203,
   0.00114,
   0.00034,
   0.00065,
   -0.003,
   0.00065,
   -0.02015,
   0.00079,
   -0.00269,
   -0.0014,
   -0.0008,
   9e-05,
   -0.00021,
   0.00189,
   0.00049,
   -0.00124,
   0.00016,
   -0.00047,
   0.00026,
   0.00103,
   -0.00175,
   -0.00054,
   -0.00321,
   -0.00064,
   0.00131,
   0.00115,
   -9e-05,
   0.00017,
   0.0002,
   -0.00196,
   0.00078,
   0.00269,
   0.00299,
   -0.00022,
   -9e-05,
   0.00067,
   0.00118,
   5e-05,
   -0.00111,
   -0.00191,
   0.00099,
   0.00062,
   0.0003,
   0.00101,
   -0.00088,
   -0.00349,
   -0.0009,
   -0.00042,
   -0.00228,
   -0.00118,
   0.00094,
   0.00056,
   -0.00224,
   0.00254,
   9e-05,
   0.00201,
   -0.00052,
   0.00322,
   0.00017,
   -0.00102,
   -0.00082,
   -0.00051,
   -0.00058,
   -0.00015,
   0.00141,
   -0.01418,
   -0.00064,
   0.00086,
   -0.00057,
   -0.00053,
   0.00048,
   -0.00038,
   -0.0025,
   -0.00458,
   0.00167,
   0.00213,
   -0.00285,
   -0.0008,
   0.00196,
   -0.00118,
   -0.00064,
   -0.00036,
   0.00016,
   0.00095,
   -0.00129,
   -0.0005,
   -0.00164,
   0.00085,
   0.00066,
   -0.00276,
   -0.00042,
   -0.00074,
   0.00154,
   -0.00293,
   0.00118,
   -0.00343,
   0.002,
   0.00058,
   0.00108,
   -0.00109,
   0.01744,
   -0.00178,
   0.0061,
   0.00012,
   -0.00069,
   -0.0063,
   -0.00072,
   -0.00014,
   -0.00115,
   0.00011,
   0.00209,
   0.00144,
   -0.00023,
   0.00088,
   0.00038,
   -0.00059,
   -0.00156,
   -3e-05,
   0.00036,
   0.00099,
   0.0013,
   0.00052,
   0.00058,
   0.00112,
   0.00067,
   0.00023,
   0.07076,
   0.00145,
   -0.0082,
   -0.02849,
   -0.00252,
   -0.00142,
   -0.00021,
   -0.0047,
   0.00012,
   0.00066,
   -0.00064,
   0.00158,
   -0.00209,
   -0.00173,
   -0.00367,
   -0.00089,
   -0.00231,
   -0.00069,
   0.00033,
   -0.00046,
   -0.00284,
   0.00113,
   0.00135,
   0.00023,
   0.00057,
   -0.0088,
   -0.0015,
   -0.00025,
   0.00136,
   0.00072,
   0.00344,
   -0.00023,
   -0.00129,
   -0.00166,
   0.00023,
   0.00025,
   0.00229,
   0.00164,
   -0.00065,
   -0.00023,
   0.00042,
   -0.00449,
   -0.00123,
   -0.00052,
   -0.00274,
   -0.00152,
   0.00028,
   -0.00526,
   0.00056,
   0.00133,
   -0.0047,
   0.00105,
   0.001,
   -0.0004,
   0.0006,
   0.00088,
   0.00121,
   0.00067,
   -0.0001,
   -0.00402,
   -6e-05,
   0.00117,
   0.00166,
   -0.00276,
   -0.00303,
   0.00022,
   -0.0018,
   -0.00044,
   -0.00063,
   0.00161,
   -0.00091,
   -0.00138,
   -0.00051,
   -0.0007,
   0.00508,
   -0.00074,
   0.00131,
   -0.0025,
   -5e-05,
   -0.00038,
   0.00021,
   0.00045,
   -0.00035,
   0.00153,
   -0.00137,
   -0.00146,
   -0.00119,
   -0.00012,
   -0.00298,
   0.00059,
   0.0006,
   0.00555,
   0.00142,
   -0.0,
   -0.00045,
   -0.0003,
   -0.00029,
   0.00018,
   0.0003,
   -0.00065,
   0.00094,
   0.0019,
   -0.00098,
   0.007,
   8e-05,
   -0.00104,
   0.00012,
   -0.00122,
   0.00749,
   -0.00087,
   -0.00033,
   -0.00222,
   -0.00264,
   0.00061,
   -0.00087
  ],
  "inst14": [
   0.02448,
   -0.00975,
   0.01056,
   0.00763,
   -0.03199,
   0.05009,
   -0.00912,
   -0.00994,
   0.01191,
   -0.00507,
   -0.00205,
   -0.03639,
   0.0097,
   0.02483,
   -0.02135,
   0.00756,
   -0.01738,
   -0.01622,
   0.01575,
   0.06262,
   0.02279,
   -0.02625,
   0.02795,
   -0.00941,
   -0.00866,
   0.03356,
   0.01924,
   0.00854,
   -0.00475,
   0.00269,
   -0.02172,
   -0.0114,
   -0.02314,
   0.02647,
   -0.0398,
   0.02695,
   0.00282,
   0.00454,
   -0.01678,
   0.03523,
   0.01002,
   -0.00567,
   0.0107,
   -0.02424,
   -0.0482,
   0.02822,
   -0.02866,
   0.02397,
   -0.04905,
   -0.01419,
   -0.00302,
   0.03676,
   -0.00301,
   0.03254,
   -0.02473,
   0.00474,
   -0.01488,
   0.0023,
   -0.00018,
   0.02189,
   0.01424,
   -0.02172,
   -0.0131,
   -0.00387,
   -0.00151,
   -0.01727,
   0.00654,
   0.00938,
   -0.01744,
   -0.02559,
   0.01116,
   0.03389,
   0.00059,
   -0.00125,
   -0.06048,
   -0.01379,
   -0.02185,
   -0.01841,
   -0.02722,
   0.00691,
   -0.02031,
   -0.00144,
   0.00521,
   0.00306,
   0.0053,
   0.02458,
   0.00535,
   -0.00413,
   0.04051,
   -0.01056,
   0.0064,
   -0.00501,
   -0.02482,
   0.01121,
   0.0345,
   0.00321,
   0.04059,
   0.04193,
   0.05058,
   -0.02314,
   -6e-05,
   0.01159,
   0.00094,
   -0.03371,
   -0.007,
   -0.00188,
   0.02242,
   0.00339,
   0.03794,
   -0.05834,
   -0.01626,
   -0.04055,
   -0.01118,
   -0.02134,
   -0.00441,
   -0.02147,
   0.02016,
   -0.0242,
   -0.02896,
   0.01273,
   -0.00661,
   -0.00418,
   0.01913,
   0.03444,
   0.0279,
   0.03385,
   0.00078,
   -0.00337,
   -0.0105,
   0.01101,
   -0.05623,
   0.03241,
   0.02703,
   0.01534,
   -0.02727,
   -0.04447,
   0.00329,
   -0.03936,
   0.03121,
   -0.00502,
   -0.02386,
   0.01307,
   0.01494,
   0.03183,
   -0.01965,
   -0.03122,
   -0.01601,
   0.04734,
   0.01019,
   -0.01893,
   0.01556,
   -0.00208,
   0.01145,
   -0.00942,
   0.02208,
   -0.02568,
   0.0217,
   0.01404,
   -0.02561,
   -0.02301,
   -0.00849,
   0.00827,
   -0.01758,
   -0.00606,
   -0.03208,
   -0.01085,
   0.01158,
   0.06142,
   -0.01812,
   0.01349,
   0.03328,
   0.0281,
   -0.02559,
   -0.02665,
   -0.00693,
   0.03138,
   -0.0074,
   -0.06199,
   -0.01377,
   0.03444,
   0.01511,
   0.01321,
   0.02106,
   -0.028,
   -0.00216,
   -0.03972,
   0.00071,
   0.03106,
   0.00889,
   -0.00021,
   -0.00473,
   0.03851,
   -0.03446,
   0.0046,
   0.01099,
   0.02136,
   -0.01186,
   -0.02303,
   -0.00627,
   0.01794,
   -0.01144,
   -0.01677,
   0.02034,
   -0.01013,
   -0.03616,
   -0.01523,
   -0.00196,
   -0.03653,
   -0.00125,
   0.04939,
   -0.02415,
   0.00404,
   -0.03,
   0.00011,
   0.01236,
   -0.00628,
   -0.01985,
   0.01149,
   0.03151,
   0.01371,
   -0.01752,
   -0.03696,
   0.00985,
   -0.0038,
   0.00819,
   -0.01908,
   0.0387,
   -0.00905,
   0.02178,
   0.04629,
   -0.0048,
   -0.01533,
   0.01102,
   -0.00602,
   -0.02294,
   0.00463,
   -0.00896,
   0.01574,
   0.01384,
   -0.01079,
   0.01652,
   0.02882,
   -0.01076,
   -0.03617,
   0.02043,
   0.02642,
   -0.01829,
   -0.05136,
   -0.00279,
   0.00587,
   0.02682,
   -0.04523,
   -0.00032,
   -0.01046,
   -0.02294,
   0.02413,
   -0.00592,
   -0.02344,
   0.01314,
   -0.01277,
   -0.01956,
   0.00611,
   0.03098,
   0.03086,
   0.01098,
   0.00508,
   -0.0088,
   0.00233,
   0.02776,
   0.02687,
   0.00175,
   -0.02284,
   0.02828,
   0.00268,
   -0.0003,
   -0.00413,
   -0.00573,
   0.00627,
   -0.00518,
   -0.01586,
   0.00106,
   0.01117,
   0.05637,
   -0.00656,
   0.00318,
   0.01433,
   0.00616,
   0.01859,
   -0.0206,
   -0.00223,
   0.00862,
   0.01573,
   -0.00455,
   -0.00314,
   -0.00074,
   -0.02592,
   0.01587,
   0.02536,
   -0.01016,
   0.03839,
   -0.02141,
   -0.02945,
   -0.00541,
   -0.02338,
   -0.01997,
   0.05692,
   -0.01731,
   -0.0266,
   0.00256,
   -0.00528,
   -0.02759,
   -0.02756,
   -0.00495,
   0.01174,
   0.01856,
   0.02312,
   0.05902,
   0.02514,
   -0.00169,
   0.00847,
   0.01448,
   -0.00922,
   0.01763,
   -0.01542,
   0.00486,
   0.03108,
   0.01018,
   0.01192,
   0.02041,
   -0.00534,
   0.00753,
   -0.01488,
   -0.01032,
   0.03833,
   -0.01659,
   0.00112,
   -0.00084,
   0.01448,
   -0.02695,
   0.05288,
   -0.01676,
   -0.00617,
   -0.01862,
   -0.01611,
   0.00259,
   0.01549,
   -0.00313,
   -0.02507,
   -0.02351,
   0.00828,
   -0.00196,
   0.02145,
   -0.01695,
   0.02425,
   0.03682,
   -0.01812,
   -0.01392,
   0.02481,
   0.04518,
   0.005,
   0.01451,
   0.02379,
   0.02459,
   0.02987,
   0.01848,
   -0.00169,
   0.00323,
   0.00388,
   -0.00292,
   -0.01027,
   0.03197,
   0.03412,
   0.00913,
   0.01273,
   -0.01938,
   0.02934,
   0.00138,
   0.04326,
   0.01064,
   0.01366,
   -0.0106,
   -0.00879,
   0.01646,
   0.01935,
   0.00065,
   0.04214,
   0.02878,
   0.01352,
   0.00712,
   -0.01558,
   -0.00636,
   0.00123,
   0.02002,
   0.02103,
   0.02054,
   -0.01437,
   -0.00483,
   0.05255,
   0.03032,
   0.006,
   -0.00154,
   -0.03249,
   0.02677,
   -0.00294,
   0.03227,
   0.02204,
   0.03348,
   0.00825,
   0.02113,
   -0.02292,
   0.04273,
   -0.01676,
   0.00442,
   -0.04973,
   -0.03176,
   0.02209,
   0.00347,
   -0.02498,
   0.02091,
   -0.02855,
   0.01137,
   0.02153,
   0.01373,
   0.00389,
   -0.0173,
   -0.0096,
   -0.03201,
   -0.00091,
   0.00874,
   -0.01316,
   0.03097,
   0.01301,
   0.03455,
   -0.04125,
   -0.03913,
   0.00886,
   -0.00499,
   0.04883,
   0.00419,
   0.03315,
   0.01173,
   0.02149,
   0.04401,
   0.02554,
   0.00996,
   -0.00953,
   -0.00761,
   0.01298,
   0.0008,
   -0.03151,
   -0.00762,
   -0.03281,
   -0.0065,
   -0.01597,
   -0.05033,
   -0.01524,
   0.00333,
   0.01561,
   -0.02012,
   0.01943,
   -0.03327,
   -0.01296,
   0.02733,
   -0.008,
   0.02417,
   -0.02815,
   0.02489,
   0.01989,
   -0.01536,
   0.0631,
   0.00748,
   0.05081,
   -0.01443,
   0.01579,
   0.02171,
   -0.02672,
   0.00661,
   -0.01457,
   0.00848,
   0.02691,
   0.01183,
   0.03329,
   0.01028,
   0.03014,
   -0.0153,
   0.01392,
   0.02555,
   -0.01359,
   0.00195,
   -0.00829,
   0.05139,
   -0.01085,
   -0.02097,
   -0.01075,
   0.0159,
   0.00257,
   0.00891,
   -0.00292,
   -0.02062,
   -0.00265,
   -0.03779,
   0.0281,
   0.01139,
   0.0632,
   0.01323,
   -0.07202,
   0.03149,
   0.02855,
   0.01018,
   0.02529,
   0.00094,
   0.05001,
   -0.00556,
   -0.01847,
   0.00215,
   0.0651,
   0.00571,
   0.03681,
   0.02064,
   0.04514,
   -0.01289,
   -0.02078,
   -0.01499,
   -0.00319,
   0.04205,
   -0.0127,
   0.03456,
   0.00549,
   0.04119,
   0.03153,
   0.0375,
   0.03471,
   -0.02534,
   -0.03005,
   0.00644,
   -0.0144,
   -0.02818,
   -0.00721,
   -0.02594,
   -0.00149,
   0.03441,
   -0.01074,
   -0.0046,
   -0.00931,
   -0.03725,
   -0.01385,
   -0.04357,
   0.00292,
   0.02406,
   -0.03153,
   0.01315,
   0.03975,
   -0.00239,
   0.15565,
   0.0029,
   -0.01983,
   -0.02229,
   0.00477,
   0.01457,
   -0.00333,
   -0.00437,
   0.0301,
   -0.00855,
   -0.01075,
   -0.01268,
   0.00072,
   -0.00414,
   -0.0214,
   -0.01906,
   0.01589,
   0.04421,
   0.0314,
   -0.05325,
   -0.02679,
   0.04817,
   0.01766,
   0.00856,
   -0.05658,
   0.02499,
   0.00664,
   0.01324,
   -0.02021,
   -0.01323,
   0.03983,
   -0.01158,
   -0.01737,
   -0.01077,
   0.05638,
   0.01674,
   0.00579,
   -0.00544,
   0.00419,
   0.02281,
   -0.0312,
   0.00649,
   0.05139,
   0.01502,
   0.02211,
   -0.04637,
   0.00175,
   -0.00853,
   -0.00939,
   0.00048,
   0.0086,
   0.0003,
   0.02495,
   -0.02675,
   -0.00884,
   0.00058,
   0.11268,
   -0.00625,
   0.02171,
   -0.02563,
   0.01272,
   0.02541,
   -0.02156,
   0.03828,
   -0.01568,
   0.03838,
   -0.04478,
   0.03674,
   0.02164,
   -0.01547,
   -0.02842,
   -0.00202,
   0.01309,
   0.02468,
   0.0178,
   0.0295,
   0.02632,
   -0.0029,
   -0.01971,
   0.02959,
   -0.01794,
   -0.02377,
   0.00155,
   0.00243,
   0.00537,
   0.01111,
   0.01027,
   -0.00057,
   -0.02395,
   0.01707,
   0.00393,
   -0.02809,
   -0.02359,
   -0.00891,
   0.00014,
   -0.00143,
   0.00665,
   -0.01565,
   0.02592,
   0.02224,
   -0.00027,
   0.00926,
   -0.00168,
   -0.03233,
   -0.00932,
   0.00636,
   0.00725,
   -0.02837,
   0.0084,
   -0.0169,
   -0.02007,
   0.01702,
   0.01328,
   -0.01365,
   -0.00334,
   -0.00744,
   -0.05616,
   -0.00868,
   -0.0227,
   -0.01551,
   -0.00702,
   -0.02406,
   -0.01547,
   -0.02174,
   -0.00523,
   0.00713,
   -0.03543,
   -0.02502,
   -0.02943,
   -0.01853,
   -0.03774,
   -0.00044,
   -0.00573,
   -0.00919,
   -0.00766,
   0.03512,
   -0.02037,
   -0.00935,
   0.00556,
   0.01169,
   0.04645,
   -0.01374,
   0.00134,
   0.00064,
   -0.00612,
   0.02512,
   0.02079,
   0.0201,
   -0.01502,
   0.0187,
   -0.02891,
   0.03459,
   -0.02222,
   0.01168,
   -0.00998,
   0.01299,
   -0.03306,
   0.01087,
   0.0391,
   -0.0261,
   -0.08886,
   -0.01394,
   -0.00535,
   -0.01088,
   -0.00046,
   -0.05282,
   0.03005,
   -0.01804,
   0.00507,
   0.01025,
   -0.01017,
   0.00261,
   -0.02021,
   -0.014,
   0.00086,
   0.01611,
   -0.01485,
   0.01161,
   -0.00783,
   -0.00872,
   0.00919,
   -0.00051,
   -0.02989,
   -0.00547,
   -0.00625,
   0.00856,
   -0.01632,
   0.0151,
   0.06159,
   -0.0153,
   0.00264,
   0.04867,
   0.00093,
   0.04657,
   -0.01144,
   -0.01246,
   -0.00655,
   -0.03992,
   0.00386,
   0.06977,
   -0.0075,
   -0.01955,
   -0.00305,
   0.00927,
   0.02514,
   0.04642,
   -0.02308,
   0.0202,
   -0.01795,
   -0.00858,
   0.00598,
   -0.00207,
   -0.00921,
   -0.00475,
   0.0335,
   -0.0045,
   0.02668,
   0.03231,
   -0.00647,
   -0.01237,
   -0.00295,
   -0.0091,
   0.00626,
   -0.00266,
   -0.01623,
   -0.01824,
   0.00888,
   -0.00804,
   -0.0028,
   0.00714,
   -0.00805,
   0.02631,
   -0.02271,
   0.0114,
   -0.00474,
   0.00526,
   -0.05252,
   0.03503,
   -0.01764,
   -0.03077,
   -0.02259,
   0.01051,
   0.04291,
   0.01699,
   -0.02443,
   -0.02615,
   -0.03235,
   0.02117,
   -0.00943,
   0.01498,
   -0.04454,
   0.01066,
   0.00884,
   -0.0055,
   0.00326,
   -0.04831,
   -0.03301,
   -0.00485,
   0.02782,
   0.01693,
   0.02624,
   -0.00131,
   0.01919,
   0.00635,
   -0.03726,
   0.02119,
   -0.02151,
   -0.01419,
   -0.05037,
   0.01729,
   0.01622,
   0.01438,
   0.01624,
   -0.02251,
   -0.03248,
   -0.02275,
   -0.00262,
   -0.01893,
   0.00107,
   0.0031,
   0.01281,
   -0.02007,
   0.02757,
   -0.00482,
   -0.01278,
   0.02259,
   0.03081,
   -0.01383,
   0.00962,
   -0.00492,
   -0.01527,
   0.03905,
   -0.03407,
   0.0051,
   -0.00863,
   0.01173,
   -0.00398,
   -0.0149,
   -0.01639,
   -0.01464,
   0.03813,
   -0.00666,
   -0.03486,
   0.02015,
   0.0166,
   -0.00888,
   -0.00878,
   -0.01958,
   0.02983,
   -0.01653,
   -0.00947,
   -0.01126,
   -0.08504,
   0.02169,
   0.02018,
   0.02816,
   0.02092,
   -0.02759,
   0.03106,
   -0.02477,
   0.03498,
   -0.02861,
   0.04088,
   -0.01077,
   0.01988,
   0.00099,
   -0.00287,
   0.03997,
   0.035,
   -0.03298,
   -0.02089,
   0.04997,
   0.02408,
   -0.04315,
   0.02624,
   0.01795,
   -0.04862,
   -0.00667,
   -0.02647,
   -0.02662,
   0.03044,
   -0.02731,
   -0.02765,
   0.03385,
   0.0156,
   0.01466,
   -0.0307,
   -0.00983,
   0.06724,
   0.02678,
   -0.02224,
   -0.016,
   0.00016,
   0.01341,
   0.01306,
   0.00443,
   -0.00494,
   -0.02028,
   0.03132,
   0.01498,
   0.01257,
   0.01308,
   -0.02683,
   0.03043,
   0.0176,
   0.00968,
   -0.00263,
   0.00685,
   -0.0122,
   -0.01693,
   0.0158,
   -0.00188,
   -0.00109,
   -0.01889,
   0.07169,
   0.00468,
   0.0331,
   -0.01135,
   -0.02712,
   -0.00133,
   0.00896,
   -0.02949,
   -0.04544,
   -0.02533,
   0.01725,
   -0.02301,
   0.02953,
   -0.01126,
   -0.00248,
   0.00499,
   -0.01975,
   -0.00844,
   0.01208,
   -0.02087,
   -0.00953,
   -0.02345,
   -0.01382,
   0.00229,
   -0.00959,
   -0.0018,
   0.03255,
   0.01593,
   0.03181,
   -0.0055,
   -0.01959,
   0.00379,
   0.03395,
   -0.03173,
   0.0097,
   -0.02705,
   0.05197,
   0.00964,
   -0.04264,
   -0.00312,
   0.00125,
   0.01107,
   -0.02738,
   -0.00246,
   -0.03944,
   -0.01334,
   -0.02458,
   0.00107,
   0.01776,
   -0.02569,
   0.00114,
   -0.03127,
   0.01938,
   0.02932,
   0.00854,
   -0.00691,
   -0.04215,
   -0.01114,
   -0.01827,
   -0.04796,
   0.00081,
   0.01966,
   -0.00832,
   -0.03137,
   -0.02793,
   0.00204,
   -0.04036,
   0.0152,
   0.02296,
   -0.02205,
   0.01483,
   -0.04173,
   -0.01016,
   -0.02813,
   0.00291,
   -0.00359,
   0.02639,
   0.00281,
   0.00994,
   -0.02846,
   -0.0037,
   0.01169,
   -0.00013,
   0.00637,
   -0.01863,
   -0.00696,
   0.03778,
   0.00264,
   0.00837,
   0.00445,
   -0.00855,
   0.01781,
   -0.02302,
   -0.00478,
   0.03971,
   -0.00928,
   0.05425,
   0.02215,
   -0.01212,
   0.03548,
   -5e-05,
   0.0006,
   -0.0137,
   -0.00961,
   -0.0034,
   0.03178,
   0.01639,
   0.00832,
   0.00456,
   -0.0156,
   0.00629,
   -0.03,
   0.00914,
   -0.04608,
   0.0169,
   -0.00551,
   -0.00223,
   -0.02586,
   0.04125,
   -0.02803,
   -0.00651,
   -0.03885,
   -0.03289,
   -0.01441,
   0.01735,
   0.02654,
   -0.01039,
   -0.00128,
   -0.006,
   -0.00432,
   -0.01287,
   -0.0169,
   -0.01373,
   -0.04035,
   -0.0241,
   0.00726,
   -0.01498,
   -0.00437,
   -0.01433,
   0.02286,
   -0.00661,
   0.00761,
   0.00175,
   -0.00853,
   -0.0008,
   0.03773,
   0.13512,
   -0.06028,
   0.03703,
   0.00294,
   -0.01313,
   0.0154,
   0.03357,
   0.0379,
   0.00828,
   0.02862,
   -0.00888,
   0.00904,
   0.02773,
   -0.00853,
   -0.00384,
   0.01228,
   0.03356,
   0.04246,
   0.01087,
   -0.001,
   0.03858,
   -0.01274,
   -0.0027,
   0.01509,
   0.00556,
   -0.00395,
   -0.02164,
   0.02093,
   0.00179,
   0.00724,
   0.01382,
   -0.00158,
   -0.01354,
   -0.00432,
   0.0014,
   0.00969,
   0.00177,
   -0.03772,
   0.00742,
   0.01403,
   0.00368,
   -0.00497,
   -0.01714,
   -0.01813,
   0.05488,
   -0.00915,
   0.01826,
   -0.0147,
   0.01791,
   0.01953,
   0.01815,
   -0.00181,
   -0.03346,
   0.01379,
   0.01618,
   -0.01123,
   -0.00082,
   0.00526,
   0.00832,
   -0.0029,
   -0.062,
   -0.02156,
   0.00514,
   -0.00729,
   2e-05,
   0.00204,
   0.01007,
   0.01647,
   0.00782,
   -0.04114,
   -0.03226,
   -0.02396,
   7e-05,
   -0.01308,
   -0.00725,
   0.0405,
   -0.02305,
   -0.00331,
   -0.02161,
   -0.00208,
   0.00089,
   0.00405,
   -0.0108,
   0.00466,
   -0.01188,
   0.00577,
   -0.02252,
   0.0209,
   0.06124,
   0.0066,
   -0.003,
   -0.03841,
   0.02229,
   0.05964,
   -0.03041,
   -0.00956,
   -0.02626,
   -0.00151,
   0.00572,
   0.03717,
   0.0054,
   -0.0259,
   0.05037,
   0.02363,
   0.01214,
   0.0048,
   -0.03688,
   0.01434,
   -0.02208,
   0.00686,
   0.01175,
   0.00201,
   -0.00819,
   0.0022,
   0.01064,
   0.02064,
   -0.02044,
   0.05303,
   0.02639,
   0.00096,
   0.03093,
   -0.02476,
   -0.00921,
   0.02975,
   -0.04473,
   -0.04851,
   0.00978,
   -0.01618,
   -0.00637,
   -0.01967,
   0.00596,
   -0.01701,
   0.0029,
   -0.02215,
   0.0163,
   0.02831,
   0.01016,
   -0.01503,
   -0.01333,
   -0.00656,
   -0.0268,
   -0.02426,
   0.03938,
   0.01255,
   -0.00362,
   -0.0125,
   -0.0421,
   -0.03698,
   -0.03193,
   -0.05433,
   0.0243,
   0.0613,
   0.0002,
   -0.0049,
   0.00871,
   -0.0463,
   0.08872,
   -0.02359,
   -0.02143,
   -0.00481,
   -0.00764,
   0.01838,
   -0.01269,
   -0.00327,
   -0.00922,
   -0.02177,
   -0.01116,
   0.05963,
   0.03117,
   0.00721,
   0.00458,
   -0.00045,
   0.02149,
   -0.02577,
   -0.00262,
   0.05113,
   -0.02672,
   -0.00063,
   -0.02082,
   -0.03296,
   0.01388,
   0.02093,
   0.00113,
   -0.04192,
   0.01843,
   0.00497,
   -0.0527,
   0.03191,
   0.01849,
   0.02521,
   -0.00936,
   -0.00161,
   0.00756,
   0.00111,
   -0.00855,
   -0.00376,
   0.0056,
   -0.01563,
   0.0004,
   -0.02545,
   -0.00509,
   -0.00716,
   -0.02509,
   0.01075,
   -0.02588,
   -0.00668,
   -0.01342,
   -0.00186,
   -0.04168,
   0.00556,
   -0.04631,
   0.01099,
   0.01009,
   -0.03242,
   0.00307,
   0.00395,
   -0.01487,
   -0.02369,
   -0.01687,
   -0.0233,
   -0.02859,
   -0.01942,
   0.03276,
   0.01041,
   0.02261,
   -0.05181,
   0.00491,
   0.00657,
   -0.00039,
   -0.01444,
   -0.02554,
   0.00043,
   0.01989,
   -0.01942,
   0.03791,
   -0.01821,
   0.0306,
   -0.00156,
   0.01596,
   0.03441,
   -0.0168,
   -0.00913,
   -0.03719,
   -0.03333,
   0.03027,
   -0.01675,
   -0.01746,
   -0.00323,
   0.01601,
   0.05457,
   0.03772,
   0.0208,
   0.01596,
   0.05061,
   -0.01675,
   0.0093,
   -0.00496,
   0.02517,
   0.03198,
   0.00839,
   0.02383,
   0.00577,
   -0.03748,
   0.00541,
   -0.00975,
   -0.00837,
   -0.00645,
   0.02832,
   0.03108,
   0.00816,
   0.03815,
   0.03075,
   0.03186,
   0.00546,
   0.03729,
   0.01617,
   0.00733,
   -0.03421,
   -0.00371,
   0.00523,
   0.0397,
   0.01666,
   0.06022,
   -0.01101,
   -0.03778,
   0.01621,
   0.01384,
   -0.0155,
   0.00705,
   0.00697,
   -0.02014,
   0.02599,
   -0.00969,
   0.03089,
   0.02735,
   0.00213,
   -0.00827,
   0.00481,
   0.06989,
   -0.01058,
   0.01566,
   0.0094,
   0.00411,
   0.01419,
   0.02039,
   -0.01802,
   0.00391,
   -0.04127,
   -0.00108,
   -0.01267,
   -0.02105,
   0.008,
   -0.0337,
   -0.02001,
   0.02523,
   -0.00588,
   0.01482,
   -0.0094,
   -0.02987,
   0.01764,
   -0.05067,
   -0.03464,
   0.0242,
   0.00123,
   0.00463,
   -0.0199,
   -0.02294,
   -0.0381,
   -0.01879,
   -0.05993,
   -0.02099,
   -0.0622,
   -0.02773,
   0.00756,
   -0.00611,
   -0.02199,
   0.01941,
   -0.01467,
   -0.02265,
   0.01004,
   -0.01714,
   0.00489,
   -0.02476,
   0.00168,
   0.02548,
   0.00674,
   -0.00373,
   -0.03279,
   -0.22731,
   0.01735,
   0.12548,
   0.078,
   -0.06515,
   0.01221,
   0.00026,
   0.00733,
   -0.04098,
   -0.00864,
   -0.02316,
   -0.04064,
   0.04156,
   -0.04311,
   -0.00088,
   -0.00726,
   -0.00457,
   -0.00698,
   -0.01855,
   0.03319,
   -0.01676,
   0.01575,
   -0.02113,
   -0.04096,
   0.01207,
   0.01622,
   0.01555,
   -0.02959,
   0.0395,
   0.0374,
   0.00022,
   -0.0001,
   -0.06174,
   0.01266,
   0.02192,
   -0.02317,
   -0.00151,
   0.04931,
   -0.01549,
   0.03249,
   0.00027,
   0.01694,
   -0.02444,
   -0.00035,
   0.011,
   0.02154,
   0.0036,
   -0.01479,
   0.01998,
   0.03952,
   0.00905,
   -0.02232,
   -0.02784,
   0.03613,
   -0.01003,
   0.0347,
   -0.02061,
   -0.01595,
   -0.00469,
   -0.00963,
   0.07244,
   -0.02584,
   -0.01185,
   0.00056,
   0.01365,
   0.02316,
   0.00255,
   0.01857,
   -0.01478,
   -0.00938,
   -0.03145,
   -0.02692,
   0.0043,
   -0.01568,
   0.02299,
   -0.03581,
   0.01448,
   -0.02282,
   -0.00949,
   -0.03237,
   -0.00535,
   -0.00738,
   -0.02604,
   0.03213,
   -0.02579,
   0.0224,
   0.02123,
   -0.01979,
   -0.01,
   0.01498,
   0.02074,
   0.0136,
   -0.03984,
   0.01107,
   0.00894,
   -0.0223,
   0.0291,
   -0.00676,
   0.05365,
   -0.00682,
   0.04221,
   0.01689,
   0.011,
   0.0718,
   -0.00541,
   0.0343,
   0.02934,
   0.00228,
   0.04262,
   0.00231,
   0.00574,
   -0.03219,
   -0.02496,
   -0.07136,
   -0.02696
  ],
  "inst20": [
   0.03448,
   -0.04934,
   0.02416,
   0.00503,
   -0.02533,
   0.0371,
   0.02948,
   -0.01987,
   0.01064,
   0.03902,
   -0.02071,
   -0.06287,
   0.01903,
   -0.01489,
   -0.02881,
   -0.02079,
   -0.03545,
   -0.02215,
   0.01713,
   0.01091,
   0.02831,
   -0.00417,
   0.00113,
   -0.02442,
   0.00104,
   0.02743,
   -0.01073,
   0.01821,
   0.02584,
   -0.02035,
   0.00631,
   0.01782,
   -0.0305,
   0.04096,
   0.00124,
   -0.02878,
   -0.01391,
   0.03556,
   -0.02496,
   0.00858,
   0.01178,
   0.01024,
   0.03485,
   -0.00317,
   -0.03662,
   0.04684,
   0.02004,
   -0.03961,
   -0.03096,
   0.00545,
   0.01418,
   0.01963,
   0.00817,
   -0.02415,
   0.0328,
   0.02735,
   0.00755,
   -0.00146,
   -0.00323,
   0.01307,
   -0.00321,
   -0.02038,
   -0.02158,
   -0.00954,
   0.00723,
   -0.00778,
   0.00032,
   -0.02529,
   0.00907,
   0.00714,
   0.01041,
   0.0257,
   0.00649,
   -0.0113,
   -0.01408,
   0.01174,
   0.01662,
   -0.01993,
   -0.01874,
   0.03519,
   0.02623,
   -0.01905,
   -0.00149,
   -0.0049,
   0.02528,
   -0.00916,
   0.02304,
   -0.02351,
   0.05421,
   -0.01321,
   -0.00526,
   -0.00168,
   -0.01164,
   0.02763,
   0.03869,
   0.00341,
   0.00629,
   0.0259,
   0.04902,
   -0.04401,
   0.00738,
   0.0324,
   0.04798,
   -0.0119,
   -0.02047,
   0.01797,
   0.02899,
   -0.00782,
   0.03567,
   -0.01755,
   -0.01911,
   0.01567,
   0.00449,
   -0.06614,
   0.04748,
   -0.04211,
   -0.02485,
   -0.01209,
   -0.02466,
   -0.00461,
   -0.01049,
   0.01435,
   0.01049,
   0.02782,
   0.00896,
   0.03117,
   0.00295,
   -0.01281,
   0.00662,
   -0.00878,
   -0.04587,
   0.02235,
   0.01371,
   0.03495,
   -0.054,
   -0.00797,
   -0.00252,
   -0.02317,
   0.00804,
   0.01763,
   -0.03354,
   0.01408,
   0.02494,
   0.01659,
   0.0128,
   0.00833,
   -0.01393,
   0.01267,
   0.0141,
   -0.00352,
   0.03059,
   -0.01649,
   0.00399,
   -0.02673,
   0.05357,
   -0.01005,
   0.04108,
   0.0017,
   -0.03051,
   -0.02596,
   -0.008,
   0.0078,
   0.01513,
   -0.01303,
   -0.01304,
   -0.0427,
   0.04532,
   0.02266,
   0.00998,
   0.02274,
   1e-05,
   -0.03323,
   0.03314,
   -0.01931,
   0.02069,
   0.00234,
   -0.01041,
   -0.0163,
   -0.00828,
   0.02402,
   0.02258,
   0.00187,
   -0.00843,
   -0.04578,
   -0.01191,
   -0.02075,
   -0.00486,
   -0.01807,
   -0.01943,
   -0.01439,
   0.00655,
   0.03147,
   -0.02754,
   -0.013,
   -0.06017,
   0.03378,
   0.03437,
   -0.00836,
   0.0109,
   -0.00143,
   0.00616,
   -0.00108,
   -0.00923,
   0.02225,
   0.03371,
   -0.02358,
   -0.00466,
   0.03708,
   0.00486,
   -0.01079,
   -0.04258,
   0.00563,
   -0.02536,
   -0.03367,
   -0.00937,
   -0.01485,
   -0.02805,
   0.00581,
   0.00662,
   0.02524,
   -0.03259,
   -0.01208,
   0.01882,
   0.01241,
   -0.0168,
   -0.01984,
   0.03505,
   -0.02965,
   0.02021,
   0.01701,
   0.05783,
   0.00624,
   -0.00667,
   0.03884,
   -0.02946,
   -0.00933,
   0.05944,
   0.01246,
   0.01301,
   -0.02398,
   0.02855,
   -0.01377,
   0.00502,
   -0.04463,
   0.00876,
   0.0159,
   -0.01788,
   0.01556,
   -0.0177,
   -0.0093,
   -0.00398,
   -0.00937,
   0.00383,
   0.03074,
   -0.01996,
   -0.00866,
   -0.04253,
   -0.03094,
   0.00304,
   0.01986,
   -0.00675,
   0.02123,
   0.00933,
   0.03298,
   -0.04418,
   0.00487,
   0.0001,
   -0.0052,
   0.01647,
   -0.03,
   -0.01025,
   -0.03802,
   0.03744,
   0.01376,
   0.03169,
   -0.00509,
   0.00523,
   0.01962,
   -0.0017,
   -0.00735,
   -0.03533,
   0.04078,
   0.0306,
   0.02184,
   -0.02816,
   0.00322,
   -0.03674,
   0.03263,
   -0.01548,
   0.02463,
   0.02496,
   -0.03288,
   -0.00757,
   0.00571,
   0.00956,
   -0.00719,
   0.03274,
   0.04283,
   -0.01887,
   -0.02585,
   0.0012,
   -0.00033,
   0.02807,
   -0.01681,
   -0.00418,
   0.03084,
   -0.01488,
   -0.01012,
   0.026,
   -0.01149,
   -0.00388,
   -0.0037,
   -0.02128,
   0.01212,
   0.02811,
   -0.01854,
   0.04406,
   -0.01172,
   0.00141,
   0.01969,
   -0.03336,
   -0.01185,
   0.02454,
   -0.00157,
   0.00735,
   -0.0134,
   -0.01838,
   0.02072,
   0.01558,
   0.02933,
   0.03178,
   -0.00457,
   0.00802,
   0.01238,
   0.00601,
   -0.01416,
   0.00804,
   0.00033,
   -0.00994,
   0.06282,
   -0.03253,
   0.00699,
   -0.02357,
   -0.02466,
   -0.01931,
   0.01335,
   -0.00322,
   0.00762,
   -0.02763,
   0.01037,
   -0.00365,
   0.0231,
   0.0139,
   0.02547,
   0.07134,
   0.01186,
   0.0307,
   0.03457,
   0.04389,
   0.01739,
   0.02225,
   0.00429,
   0.03331,
   0.0185,
   0.03685,
   -0.0347,
   0.02213,
   -0.02766,
   -0.00393,
   0.0443,
   0.01775,
   0.01244,
   0.00977,
   -0.00059,
   -0.00643,
   0.02997,
   0.0199,
   -0.00065,
   0.00105,
   0.02294,
   -0.01578,
   0.00482,
   0.01867,
   0.03202,
   -0.0231,
   0.00338,
   0.01686,
   0.04137,
   -0.00241,
   0.02275,
   -0.00532,
   -0.03317,
   0.00585,
   -0.0193,
   -0.03535,
   -0.00364,
   0.01147,
   0.03294,
   -0.00851,
   -0.01955,
   0.00664,
   -0.02088,
   0.01978,
   -0.01615,
   0.02101,
   0.01943,
   0.01624,
   -0.0044,
   0.0081,
   -0.01557,
   0.03904,
   -0.00671,
   0.00287,
   -0.0052,
   -0.00421,
   0.02291,
   -0.02707,
   0.03945,
   0.01089,
   -0.02635,
   0.02289,
   0.00455,
   0.01647,
   0.03029,
   0.02773,
   -0.02251,
   -0.04216,
   -0.03069,
   0.00535,
   0.00091,
   0.01349,
   0.00372,
   -0.01727,
   0.0052,
   -0.03343,
   0.01496,
   -0.0352,
   0.03372,
   0.00648,
   0.02829,
   0.0197,
   0.00132,
   0.00595,
   0.02854,
   -0.03084,
   -0.01929,
   -0.01975,
   0.00285,
   -0.04308,
   -0.05402,
   0.03735,
   -0.01108,
   0.07414,
   -0.04937,
   -0.0082,
   0.0121,
   -0.00678,
   -0.02006,
   -0.01818,
   -0.00403,
   -0.03356,
   -0.02423,
   0.0025,
   -0.00447,
   0.0224,
   -0.01086,
   -0.00339,
   0.03337,
   0.00546,
   0.07031,
   -0.02972,
   0.02151,
   -0.00633,
   0.04414,
   -0.00107,
   -0.02528,
   0.00431,
   -0.00589,
   0.03584,
   -0.01185,
   0.01859,
   -0.01378,
   -0.00058,
   -0.01317,
   -0.00248,
   -0.00924,
   -0.00879,
   -0.00823,
   -0.00941,
   -0.00865,
   0.02162,
   -0.01337,
   -0.00899,
   -0.03886,
   -0.02913,
   0.01237,
   -0.01677,
   -0.00871,
   -0.00922,
   -0.01338,
   -0.0215,
   0.05211,
   0.00726,
   0.04247,
   0.02341,
   -0.0414,
   0.00882,
   0.00886,
   -0.02197,
   0.007,
   0.02012,
   0.02081,
   0.013,
   0.03133,
   -0.01406,
   0.03529,
   -0.03246,
   0.02278,
   -0.00108,
   0.03963,
   -0.0612,
   0.01091,
   0.01599,
   0.00953,
   0.02425,
   -0.02065,
   0.03059,
   0.00376,
   -0.01287,
   -0.02154,
   0.01515,
   0.0084,
   -0.03238,
   -0.01807,
   0.00354,
   0.01469,
   -0.00046,
   -0.01555,
   -0.01326,
   0.00152,
   0.01908,
   -0.00038,
   0.01769,
   0.00634,
   -0.03921,
   -0.014,
   0.02211,
   0.00165,
   0.03179,
   -0.00777,
   -0.02591,
   -0.00776,
   -0.01209,
   0.08075,
   0.00557,
   0.02432,
   -0.02998,
   -0.01955,
   0.03776,
   -0.01552,
   -0.00506,
   0.05109,
   -0.02051,
   0.03615,
   -0.01681,
   -0.01903,
   -0.02322,
   -0.01509,
   0.02056,
   0.0237,
   0.01335,
   0.02523,
   -0.0084,
   -0.0128,
   0.03203,
   -0.01263,
   0.02005,
   -0.00964,
   0.03299,
   0.02421,
   0.00052,
   -0.01748,
   -0.01927,
   0.02165,
   0.00154,
   -0.02037,
   -0.0195,
   -0.00227,
   -0.00808,
   0.01075,
   0.00875,
   0.01515,
   0.02766,
   -0.01521,
   -0.00579,
   0.04998,
   0.00642,
   -0.00683,
   -0.01828,
   0.01379,
   0.01222,
   -0.03837,
   0.01127,
   -0.00358,
   0.01957,
   0.03612,
   -0.00599,
   0.01912,
   0.0446,
   0.00216,
   -0.01498,
   -0.01323,
   -0.01347,
   -0.02096,
   0.00275,
   -0.00535,
   -0.04381,
   -0.00392,
   0.02392,
   -0.03759,
   -0.01145,
   0.00972,
   -0.00625,
   -0.01538,
   -0.02064,
   -0.02613,
   0.04959,
   0.02495,
   0.03694,
   0.0631,
   -0.00795,
   -0.01523,
   0.01121,
   0.01406,
   0.00535,
   -0.03225,
   -0.0185,
   0.00701,
   -0.01424,
   0.01079,
   -0.00445,
   -0.01979,
   0.01012,
   0.01251,
   -0.01539,
   0.00792,
   -0.03014,
   0.00929,
   0.01898,
   0.00408,
   -0.02157,
   0.00284,
   0.01392,
   -0.0139,
   -0.00732,
   -0.0279,
   -0.01895,
   -0.03,
   0.05219,
   -0.04087,
   -0.0776,
   0.01607,
   -0.0282,
   -0.00401,
   -0.02266,
   -0.00894,
   0.0184,
   -0.006,
   0.01644,
   0.02878,
   0.00035,
   0.01102,
   -0.02926,
   0.06033,
   -0.02519,
   0.00842,
   -0.0171,
   -0.0105,
   0.01809,
   7e-05,
   -0.01946,
   -0.00034,
   0.01725,
   -0.04063,
   0.01812,
   -0.0174,
   -0.00249,
   -0.01807,
   0.0291,
   -0.04989,
   0.01582,
   0.0065,
   0.04711,
   0.0148,
   -0.05124,
   0.03547,
   -0.0021,
   -0.03046,
   -0.03242,
   0.03578,
   0.00934,
   -0.0437,
   0.06084,
   -0.00263,
   0.02323,
   0.00795,
   0.0257,
   -0.01664,
   0.00367,
   -0.03091,
   0.01103,
   0.03543,
   0.00371,
   -0.07927,
   -0.0004,
   -0.02686,
   -0.00609,
   -0.00511,
   -0.04245,
   0.01081,
   -0.02549,
   0.00537,
   0.00816,
   -0.0172,
   -0.02096,
   0.03241,
   -0.02387,
   -0.00569,
   -0.02019,
   -0.04057,
   0.00102,
   0.00897,
   -0.03451,
   -0.01466,
   0.01947,
   -0.03009,
   0.01937,
   0.01395,
   0.0091,
   -0.05948,
   0.02268,
   0.0292,
   0.02176,
   0.02185,
   0.01279,
   0.00297,
   0.033,
   -0.03791,
   0.00587,
   -0.00464,
   -0.04323,
   -0.00624,
   0.07517,
   0.01081,
   -0.02554,
   -0.00824,
   -0.00908,
   -0.00418,
   0.03021,
   0.00686,
   0.02545,
   -0.01467,
   -0.02524,
   0.01447,
   0.01384,
   0.03122,
   -0.01987,
   -0.00329,
   -0.02758,
   -0.01132,
   0.02585,
   0.05228,
   -0.00189,
   0.02219,
   0.0216,
   0.01645,
   0.01532,
   -0.04105,
   0.02146,
   0.00754,
   -0.02323,
   -0.02391,
   0.02784,
   -0.00249,
   0.0535,
   0.01957,
   -0.02402,
   -0.00532,
   -0.02298,
   -0.06122,
   0.01271,
   0.0304,
   0.00364,
   -0.06382,
   0.03705,
   -0.00779,
   -0.00283,
   0.00358,
   0.01465,
   -0.04218,
   0.01441,
   0.04472,
   0.04395,
   -0.04941,
   0.03473,
   -0.0265,
   0.0007,
   0.04594,
   -0.03277,
   -0.03803,
   -0.00949,
   0.01397,
   0.01097,
   0.03052,
   -0.02622,
   -0.02935,
   -0.02938,
   -0.0361,
   0.01176,
   0.00238,
   0.00288,
   -0.01341,
   -0.01866,
   0.03014,
   0.00253,
   0.00232,
   -0.01639,
   -0.04893,
   0.00038,
   -0.00509,
   -0.03784,
   -0.00222,
   -0.02193,
   0.01688,
   0.02494,
   0.00104,
   0.04932,
   -0.02646,
   -0.00909,
   0.017,
   0.0003,
   0.00807,
   0.03066,
   -0.01313,
   0.00401,
   0.01586,
   0.01956,
   -0.01077,
   0.00131,
   -0.00689,
   0.00897,
   -0.00625,
   -0.03435,
   0.0496,
   0.00575,
   -0.01987,
   0.02826,
   0.02885,
   -0.01568,
   0.0097,
   0.00391,
   0.00111,
   0.04252,
   -0.00981,
   0.00871,
   -0.11837,
   0.02007,
   -0.00941,
   -0.00618,
   0.0157,
   0.01403,
   0.02837,
   0.0221,
   0.00172,
   -0.02193,
   0.02741,
   -0.01219,
   0.01708,
   -0.01463,
   0.02717,
   0.04196,
   0.01298,
   -0.07616,
   -0.04483,
   0.01526,
   0.01642,
   -0.05114,
   -0.00493,
   -0.02117,
   -0.04726,
   0.02113,
   -0.02202,
   -0.01108,
   0.01993,
   -0.01323,
   -0.01859,
   0.03483,
   0.00582,
   0.03983,
   -0.01786,
   0.04581,
   0.05897,
   0.04061,
   -0.0323,
   -0.0055,
   0.04613,
   -0.00106,
   0.01883,
   -0.00941,
   0.02764,
   -0.03236,
   0.00051,
   0.02017,
   0.00324,
   0.01404,
   -0.01328,
   0.04005,
   0.00431,
   -0.01541,
   -0.00935,
   -0.0028,
   -0.01162,
   0.0344,
   0.01663,
   -0.01035,
   -0.02409,
   -0.01462,
   0.03938,
   -0.04715,
   0.02554,
   0.01646,
   -0.02555,
   0.00477,
   0.00928,
   0.00973,
   -0.00883,
   -0.00457,
   0.00521,
   -0.01532,
   -0.01043,
   -0.05863,
   0.0121,
   0.00402,
   -0.01629,
   0.00848,
   0.01367,
   -0.02546,
   5e-05,
   0.02511,
   -0.02506,
   -0.04058,
   -0.01237,
   0.01226,
   -0.00568,
   0.00624,
   0.00067,
   -0.00916,
   0.00159,
   0.00088,
   0.00828,
   -0.03054,
   -0.01839,
   -0.00431,
   0.06605,
   -0.01025,
   -0.05821,
   -0.02,
   -0.01947,
   0.01785,
   -0.03456,
   -0.05114,
   -0.02164,
   -0.03339,
   -0.041,
   -0.03112,
   0.01685,
   0.03437,
   0.02327,
   -0.03698,
   0.04336,
   0.01736,
   -0.00504,
   0.01125,
   -0.04608,
   -0.04316,
   -0.00439,
   -0.03554,
   -0.01792,
   0.01387,
   -0.02235,
   0.02882,
   -0.01044,
   -0.00427,
   -0.0002,
   0.00267,
   0.02627,
   0.01913,
   0.02691,
   -0.02591,
   -0.01926,
   -0.00574,
   -0.03181,
   0.00076,
   0.0064,
   0.0005,
   0.01631,
   -0.01285,
   0.01141,
   0.00796,
   0.01278,
   0.01562,
   -0.01461,
   0.00959,
   0.03431,
   -0.00584,
   0.00064,
   -0.01298,
   0.03055,
   -0.03186,
   -0.0003,
   -0.00401,
   -0.0235,
   -0.01979,
   0.05717,
   0.01635,
   0.03036,
   0.00102,
   -0.04142,
   -0.0003,
   -0.00401,
   -0.01535,
   -0.0311,
   -0.01108,
   0.04901,
   0.0053,
   -0.01597,
   0.00091,
   -0.01258,
   -0.01554,
   0.01328,
   -0.00736,
   9e-05,
   -0.02252,
   -0.02341,
   -0.03947,
   0.04653,
   -0.01695,
   -0.0351,
   -0.02006,
   -0.00763,
   -0.00115,
   -0.00132,
   0.00162,
   0.00274,
   -0.01381,
   0.0065,
   -0.03511,
   0.01952,
   -0.03772,
   0.01919,
   -0.04185,
   -0.01015,
   0.00164,
   -0.01488,
   0.02937,
   -0.03083,
   0.03471,
   -0.00402,
   -0.01811,
   -0.01392,
   0.01686,
   -0.0244,
   0.05095,
   0.10638,
   -0.05872,
   0.00643,
   -0.05189,
   -0.01523,
   0.00167,
   -0.00553,
   0.01143,
   -0.00843,
   0.05279,
   0.02264,
   0.00482,
   0.0186,
   -0.01618,
   -0.00514,
   -0.00317,
   0.00751,
   0.0229,
   -0.01262,
   -0.00485,
   0.01104,
   0.01368,
   0.04623,
   -0.00603,
   -0.02896,
   -0.01118,
   -0.00071,
   0.03048,
   0.01379,
   0.0105,
   0.05376,
   -0.02851,
   -0.00415,
   -0.01743,
   0.01303,
   0.01845,
   0.01222,
   -0.06859,
   -0.02631,
   0.00177,
   0.06452,
   -0.02206,
   -0.00596,
   -0.02356,
   0.01151,
   0.0072,
   0.03754,
   0.02326,
   -0.00775,
   0.01611,
   -0.02111,
   0.01434,
   -0.02916,
   0.03,
   0.06889,
   -0.01301,
   -0.05271,
   0.01056,
   0.03682,
   -0.00535,
   -0.02123,
   -0.01431,
   0.00089,
   0.02833,
   -0.00046,
   0.01437,
   0.01625,
   0.02761,
   -0.00108,
   0.0271,
   -0.03132,
   -0.02621,
   -0.0334,
   -0.01887,
   0.02096,
   0.02512,
   -0.00189,
   0.00316,
   -0.04434,
   0.02831,
   0.02377,
   0.01859,
   -0.03084,
   0.00717,
   -0.01207,
   0.0279,
   0.02324,
   -0.00202,
   -0.00912,
   -0.04925,
   -0.036,
   -0.02226,
   0.03556,
   0.03726,
   -0.02946,
   -0.01624,
   -0.05963,
   0.00656,
   0.00975,
   0.00275,
   0.03218,
   -0.01522,
   0.04416,
   0.01686,
   0.02667,
   0.02239,
   -0.02086,
   0.01572,
   0.04636,
   -0.01003,
   0.01304,
   -0.01712,
   0.01243,
   0.0413,
   -0.00325,
   -0.02722,
   0.01968,
   0.00642,
   0.03276,
   -0.01981,
   0.06866,
   -0.03476,
   0.00202,
   0.01957,
   -0.04457,
   -0.00899,
   0.01306,
   -0.05426,
   0.03925,
   -0.03907,
   0.008,
   -0.04279,
   -0.00193,
   -0.03715,
   0.04073,
   -0.04367,
   -0.02139,
   0.00832,
   0.01547,
   -0.01229,
   -0.04818,
   0.0158,
   0.05995,
   0.01172,
   -0.0413,
   -0.031,
   -0.04291,
   -0.01909,
   -0.02346,
   -0.00927,
   0.00629,
   0.03187,
   -0.02497,
   -0.01317,
   0.01508,
   -0.05174,
   0.04367,
   0.01621,
   0.01275,
   0.01338,
   -0.0028,
   0.05908,
   -0.01045,
   -0.0166,
   -0.0054,
   -0.00418,
   -0.03681,
   0.03632,
   0.01941,
   -0.01013,
   0.01789,
   0.01201,
   0.03919,
   0.01416,
   -0.00511,
   0.02474,
   -0.02605,
   -0.00809,
   -0.0208,
   -0.01243,
   0.03314,
   -0.03496,
   0.00168,
   -0.06437,
   0.03037,
   -0.01759,
   -0.04549,
   -0.00814,
   0.01087,
   0.02737,
   -0.0557,
   0.00852,
   0.00996,
   -0.02669,
   -0.04147,
   0.01377,
   0.00621,
   0.01759,
   0.01463,
   -0.05249,
   0.03503,
   -0.03144,
   -0.03295,
   -0.01507,
   -0.01685,
   -0.00639,
   -0.00205,
   0.02797,
   0.03402,
   0.0233,
   -0.03242,
   -0.01197,
   -0.02472,
   -0.03138,
   0.02161,
   0.01019,
   0.00652,
   0.01815,
   0.01652,
   -0.01382,
   -0.00105,
   0.00058,
   0.06146,
   -0.02725,
   -0.03455,
   -0.03908,
   -0.01982,
   -0.02439,
   -0.00712,
   -0.0135,
   -0.01669,
   0.00091,
   -0.00246,
   0.00728,
   0.0284,
   -0.01045,
   -0.0063,
   -0.0073,
   0.03094,
   0.02023,
   -0.03007,
   0.01538,
   -0.00548,
   0.00509,
   -0.01528,
   -0.01057,
   0.00981,
   0.02177,
   0.0142,
   -0.0102,
   0.01752,
   -0.01648,
   -0.05185,
   0.0325,
   -0.03559,
   -0.00557,
   -0.01545,
   0.03011,
   0.05271,
   0.06103,
   0.01315,
   -0.00221,
   -0.00893,
   0.03653,
   0.0285,
   -0.00187,
   -0.01003,
   0.00343,
   0.01227,
   -0.0223,
   -0.00319,
   0.01951,
   0.00329,
   -0.03907,
   0.02208,
   0.0442,
   0.00353,
   -0.02119,
   -0.03585,
   0.02383,
   0.01478,
   -0.01079,
   0.02286,
   0.00615,
   -0.01013,
   0.02912,
   0.03082,
   0.02808,
   0.00469,
   -0.01414,
   -0.02128,
   0.05171,
   0.0198,
   0.01377,
   0.01272,
   -0.0191,
   -0.02516,
   -0.02744,
   0.06658,
   -0.04025,
   -0.02176,
   -0.00296,
   0.01758,
   0.04043,
   0.00795,
   -0.0098,
   -0.01703,
   0.00917,
   -0.00459,
   0.01192,
   -0.00525,
   -0.00343,
   -0.00367,
   0.03883,
   0.02643,
   -0.0256,
   0.02062,
   -0.01994,
   -0.01912,
   0.00929,
   -0.03232,
   -0.00302,
   -0.02506,
   0.00378,
   -0.02989,
   -0.02511,
   -0.00499,
   -0.02787,
   -0.01052,
   -0.03039,
   -0.02242,
   -0.02996,
   -0.04929,
   0.01763,
   -0.01339,
   0.00355,
   -0.0087,
   -0.01294,
   0.00834,
   0.00131,
   -0.06913,
   0.00067,
   -0.00497,
   -0.01031,
   0.04047,
   -0.02392,
   -0.02411,
   -0.04127,
   0.08094,
   -0.02816,
   0.08489,
   0.01826,
   0.01017,
   -0.0129,
   0.01463,
   0.00017,
   -0.02641,
   0.00707,
   -0.01892,
   -0.04849,
   0.00066,
   -0.04787,
   -0.01469,
   0.01207,
   -0.02537,
   -0.01717,
   -0.00057,
   0.02338,
   -0.01868,
   0.00718,
   -0.01845,
   -0.01743,
   0.00626,
   0.01962,
   -0.01251,
   0.004,
   -0.00737,
   -0.00579,
   0.02398,
   0.00274,
   -0.01034,
   -0.00218,
   0.0017,
   -0.03433,
   -0.00427,
   0.04378,
   -0.00388,
   0.03309,
   0.02276,
   0.00488,
   -0.00799,
   -0.00848,
   0.00146,
   0.02327,
   -0.01405,
   -0.01328,
   -0.00102,
   0.00019,
   -0.02598,
   0.01803,
   -0.03268,
   0.02544,
   0.00385,
   0.03875,
   -0.00387,
   -0.04655,
   0.0168,
   -0.01297,
   0.04117,
   0.00263,
   -0.00113,
   -0.01926,
   0.00262,
   0.03098,
   0.01326,
   0.00778,
   -0.06415,
   0.01883,
   0.02012,
   -0.00996,
   0.03388,
   -0.02068,
   0.03805,
   0.03384,
   -0.0269,
   -0.00495,
   0.01942,
   -0.04053,
   -0.02709,
   0.01068,
   -0.0395,
   0.00897,
   -0.01915,
   -0.00611,
   0.00762,
   0.0012,
   -0.01412,
   0.02628,
   0.04194,
   -0.02797,
   0.00451,
   0.02312,
   -0.02392,
   0.02145,
   0.03608,
   0.01261,
   0.04355,
   -0.00047,
   0.04399,
   0.00104,
   0.0363,
   -0.00016,
   -0.02951,
   0.0289,
   0.04727,
   0.00412,
   0.03774,
   0.00955,
   0.00146,
   -0.04326,
   -0.01178,
   -0.04767,
   -0.07166
  ]
 },
 "encoders": {
  "base14": [
   [
    -0.01126,
    -0.00654,
    -0.00044,
    0.00047,
    -0.00081,
    -0.00059,
    -0.00146,
    -0.00933,
    -0.0052,
    0.00226,
    0.00182,
    2e-05,
    -0.00109,
    -0.00154
   ],
   [
    -0.00166,
    0.00149,
    -0.00064,
    -0.00145,
    0.00063,
    8e-05,
    0.00122,
    -0.00143,
    -0.00071,
    -0.00104,
    -0.00124,
    0.00026,
    2e-05,
    0.00083
   ],
   [
    -0.00063,
    -0.00248,
    0.0006,
    9e-05,
    0.00094,
    0.00109,
    -0.00223,
    -0.0004,
    -0.00356,
    -0.00042,
    0.00059,
    0.00105,
    -0.00035,
    -0.00203
   ],
   [
    0.00289,
    -0.00286,
    -0.00229,
    0.00207,
    -0.00222,
    0.00064,
    7e-05,
    0.00191,
    -0.00136,
    -0.00229,
    0.00155,
    -0.00174,
    0.00096,
    -9e-05
   ],
   [
    0.00171,
    -0.00564,
    0.00336,
    0.00396,
    0.00031,
    -0.00362,
    -0.00208,
    -0.00031,
    -0.00412,
    0.00077,
    0.00356,
    -0.00023,
    -0.00391,
    -0.00164
   ],
   [
    -0.0028,
    -0.00167,
    0.00702,
    0.00278,
    -0.00367,
    0.00336,
    0.00802,
    -0.00574,
    -0.00048,
    0.00328,
    0.00322,
    -0.00198,
    0.00357,
    0.00751
   ],
   [
    -0.00494,
    -0.00022,
    -0.00141,
    -0.00086,
    0.00173,
    -0.0011,
    0.00203,
    -0.00585,
    -0.00119,
    -0.00295,
    -0.00095,
    0.00131,
    -0.00132,
    0.00149
   ],
   [
    -0.00025,
    -0.00141,
    -0.00132,
    -5e-05,
    -0.00437,
    0.00163,
    0.00684,
    0.00164,
    -0.00051,
    -0.00064,
    -0.0008,
    -0.00434,
    0.00032,
    0.00536
   ],
   [
    -0.0002,
    0.00027,
    0.002,
    0.00046,
    0.00101,
    -0.0019,
    -9e-05,
    -0.00293,
    -0.00138,
    -1e-05,
    0.00117,
    0.00113,
    -0.00112,
    0.00039
   ],
   [
    0.00786,
    -0.0032,
    0.0009,
    0.00149,
    -0.00496,
    0.00186,
    0.00674,
    0.00462,
    -0.00414,
    -0.00138,
    0.00157,
    -0.00364,
    0.00286,
    0.00574
   ],
   [
    0.00389,
    7e-05,
    0.0023,
    0.0013,
    -0.00416,
    0.00073,
    0.00432,
    0.00211,
    -0.00026,
    0.00095,
    0.0009,
    -0.00316,
    0.00059,
    0.00363
   ],
   [
    -3e-05,
    0.00769,
    -0.00016,
    0.02175,
    0.02327,
    -0.03995,
    -0.01588,
    -0.00525,
    -0.0007,
    -0.0077,
    0.02023,
    0.01663,
    -0.03494,
    -0.01264
   ],
   [
    -0.00175,
    0.00159,
    -0.00035,
    0.00039,
    0.001,
    -0.00316,
    2e-05,
    -0.00117,
    0.00214,
    0.00038,
    0.00078,
    0.00129,
    -0.00265,
    0.00022
   ],
   [
    -0.00213,
    -0.00234,
    -0.00117,
    0.0002,
    -0.00227,
    0.00099,
    -0.00113,
    -0.00022,
    -0.00075,
    0.00069,
    0.00019,
    -0.00189,
    0.00032,
    -0.00121
   ],
   [
    -0.00154,
    0.00203,
    0.00015,
    0.00132,
    0.00152,
    0.00022,
    0.00116,
    -0.00026,
    0.00336,
    0.00102,
    0.00129,
    0.00185,
    0.0006,
    0.00141
   ],
   [
    0.00389,
    0.00114,
    0.00131,
    0.00047,
    -0.00182,
    -0.00119,
    -0.00023,
    0.00346,
    0.0025,
    0.00067,
    -6e-05,
    -0.00182,
    -0.00172,
    -0.00072
   ],
   [
    -0.00153,
    -0.00433,
    -0.00139,
    0.0001,
    -0.0007,
    0.0002,
    -0.002,
    -0.00235,
    -0.00406,
    -0.00253,
    0.00032,
    -0.0008,
    -0.00024,
    -0.00212
   ],
   [
    -0.00142,
    0.00134,
    0.00125,
    -0.00258,
    -0.00156,
    -0.00051,
    0.00158,
    -0.00325,
    -0.00151,
    -0.00023,
    -0.00248,
    -0.00144,
    -0.00102,
    0.00089
   ],
   [
    0.00179,
    0.00075,
    0.00295,
    0.00115,
    -0.00206,
    0.00104,
    0.00177,
    0.00088,
    5e-05,
    0.0013,
    0.00125,
    -0.00136,
    0.00129,
    0.00166
   ],
   [
    -0.00522,
    -0.00672,
    -0.00395,
    0.00264,
    -0.00516,
    -0.00695,
    0.00166,
    1e-05,
    -0.00611,
    -0.00072,
    0.00224,
    -0.00479,
    -0.00577,
    0.0009
   ],
   [
    -0.00142,
    -6e-05,
    -0.00286,
    0.00056,
    0.00092,
    -0.00074,
    -0.00561,
    -0.00043,
    -0.00046,
    -0.00225,
    -7e-05,
    1e-05,
    -0.00116,
    -0.00483
   ],
   [
    0.00109,
    4e-05,
    -0.0002,
    -0.0025,
    -0.0017,
    0.00084,
    0.004,
    0.00182,
    0.00146,
    0.00133,
    -0.00217,
    -0.0013,
    0.00068,
    0.00361
   ],
   [
    0.0018,
    0.00607,
    0.00062,
    -0.00256,
    -0.00095,
    0.00075,
    0.00233,
    0.00326,
    0.00603,
    0.00089,
    -0.00175,
    -0.00011,
    0.00117,
    0.00252
   ],
   [
    0.00891,
    0.01467,
    -0.00327,
    0.00073,
    0.00216,
    -0.00089,
    -0.01388,
    0.00637,
    0.00863,
    -0.00299,
    -0.0005,
    0.00088,
    -0.00239,
    -0.01377
   ],
   [
    0.00137,
    0.00227,
    -0.00095,
    -0.00121,
    0.00097,
    0.00174,
    -0.00043,
    -0.00031,
    0.00181,
    -0.00151,
    -0.001,
    0.00092,
    0.0016,
    -0.00043
   ],
   [
    0.00161,
    0.00121,
    0.00065,
    0.00242,
    -0.00049,
    -0.00016,
    -0.00107,
    0.00071,
    0.00103,
    1e-05,
    0.00188,
    -0.00049,
    -0.00018,
    -0.00093
   ],
   [
    0.00179,
    0.00614,
    0.00228,
    0.00028,
    0.00444,
    0.00072,
    0.00036,
    0.00021,
    0.00539,
    0.00191,
    0.00097,
    0.00449,
    0.00214,
    0.00138
   ],
   [
    0.00234,
    0.00335,
    0.00072,
    -0.00117,
    0.00105,
    0.00222,
    -0.00153,
    0.00151,
    0.00241,
    0.00055,
    -0.0002,
    0.00147,
    0.00316,
    -0.00075
   ],
   [
    -0.00941,
    -0.00679,
    -0.00057,
    -0.0021,
    0.0064,
    0.00132,
    -0.00264,
    -0.00931,
    -0.00597,
    -0.0019,
    -0.00279,
    0.00405,
    -0.00048,
    -0.00213
   ],
   [
    0.0024,
    0.00012,
    -0.00136,
    0.00108,
    0.00011,
    0.00101,
    -0.00151,
    0.00379,
    0.00164,
    1e-05,
    0.00097,
    0.00021,
    0.00128,
    -0.00093
   ],
   [
    -0.00118,
    0.00026,
    -0.00124,
    0.00275,
    -0.00226,
    -0.0006,
    0.00136,
    0.00041,
    0.0014,
    0.00058,
    0.00198,
    -0.00207,
    -0.00078,
    0.00058
   ],
   [
    0.00209,
    -0.00326,
    -0.00141,
    -0.00239,
    -0.00248,
    0.00394,
    0.00187,
    -0.00151,
    -0.00403,
    -0.00274,
    -0.00215,
    -0.0025,
    0.00348,
    0.00188
   ],
   [
    -0.0007,
    0.00069,
    -0.00085,
    -0.00054,
    7e-05,
    0.00168,
    -0.00036,
    0.00029,
    0.00113,
    0.00016,
    -0.00036,
    0.00017,
    0.00148,
    -5e-05
   ],
   [
    -0.00396,
    -0.0091,
    0.00179,
    0.00258,
    -0.00631,
    -0.00333,
    0.01068,
    -0.00501,
    -0.00749,
    -0.00016,
    0.00364,
    -0.00503,
    -0.00171,
    0.0094
   ],
   [
    -0.00346,
    0.00096,
    0.00123,
    -0.00229,
    -0.00172,
    -0.00041,
    0.00451,
    -0.00139,
    0.00171,
    0.00248,
    -0.00246,
    -0.0019,
    -0.00065,
    0.00364
   ],
   [
    0.00217,
    0.00113,
    -0.00085,
    -0.00021,
    -0.00071,
    0.00072,
    0.00177,
    0.00227,
    0.00128,
    0.0,
    -0.00023,
    -0.00085,
    0.00058,
    0.00145
   ],
   [
    0.00473,
    0.00297,
    0.00145,
    -0.00028,
    0.0023,
    -0.0,
    0.00117,
    0.00392,
    0.00269,
    0.00058,
    0.00016,
    0.00264,
    0.00086,
    0.00169
   ],
   [
    -0.00501,
    -0.0055,
    -0.00107,
    0.0009,
    -0.00091,
    -0.00332,
    -0.00333,
    -0.00522,
    -0.00636,
    -0.0027,
    0.00038,
    -0.00155,
    -0.00357,
    -0.00359
   ],
   [
    0.00745,
    -0.00678,
    0.0015,
    0.00475,
    -0.00537,
    -0.00168,
    -0.0004,
    0.00643,
    -0.00237,
    0.00212,
    0.0054,
    -0.00228,
    -0.00036,
    -0.00092
   ],
   [
    -0.00217,
    0.00345,
    -0.00358,
    0.00148,
    -0.00214,
    -0.00037,
    -0.0031,
    -0.0037,
    0.00146,
    -0.00177,
    0.00239,
    -0.00098,
    0.00039,
    -0.00234
   ],
   [
    -0.00243,
    0.001,
    0.00074,
    -0.0005,
    0.0003,
    4e-05,
    -0.00109,
    -0.00335,
    -0.00066,
    -0.0004,
    -0.00049,
    0.00023,
    -0.00019,
    -0.00107
   ],
   [
    0.00102,
    -0.00155,
    -0.00119,
    0.00196,
    -0.00188,
    -3e-05,
    -0.00047,
    0.00203,
    -0.00035,
    -0.00025,
    0.00207,
    -0.00108,
    -0.00024,
    -0.00061
   ],
   [
    -0.00197,
    -0.00189,
    0.0007,
    0.00074,
    -0.00237,
    -0.00112,
    0.00133,
    -0.00025,
    -0.00203,
    0.00089,
    0.0011,
    -0.0017,
    -0.00073,
    0.00127
   ],
   [
    0.00398,
    0.00308,
    0.00059,
    0.00044,
    0.00139,
    0.00056,
    -0.00074,
    0.00269,
    0.0029,
    -0.00042,
    -9e-05,
    0.00122,
    0.00041,
    -0.0008
   ],
   [
    0.00613,
    0.0029,
    -0.00641,
    -0.01336,
    -0.00458,
    0.01486,
    0.00985,
    0.00643,
    0.00589,
    -0.0026,
    -0.01467,
    -0.00556,
    0.01156,
    0.00829
   ],
   [
    0.00304,
    0.00618,
    -0.00346,
    -0.00115,
    0.00235,
    0.00281,
    -0.00216,
    0.00154,
    0.00462,
    -0.00148,
    -0.0006,
    0.00189,
    0.00221,
    -0.00208
   ],
   [
    -0.00079,
    0.00283,
    -0.00057,
    0.00204,
    0.00593,
    -0.00513,
    -0.00393,
    -0.00265,
    0.00064,
    -0.00349,
    0.00199,
    0.00458,
    -0.00403,
    -0.00291
   ],
   [
    0.00066,
    0.00245,
    0.0014,
    0.00184,
    1e-05,
    -0.00144,
    0.00195,
    0.00299,
    0.00473,
    0.00227,
    0.00259,
    0.00107,
    -1e-05,
    0.00203
   ],
   [
    0.00131,
    0.00207,
    0.00229,
    0.0029,
    0.00145,
    -0.00067,
    -0.00444,
    0.00337,
    0.00369,
    0.00343,
    0.00246,
    0.00139,
    -0.00069,
    -0.00359
   ],
   [
    0.00576,
    -0.00563,
    0.0007,
    0.00059,
    -0.00689,
    -0.00238,
    0.00525,
    0.00375,
    -0.00583,
    -0.00139,
    -0.00018,
    -0.00552,
    -0.00187,
    0.00417
   ],
   [
    0.00628,
    0.00843,
    0.00308,
    -0.0033,
    0.00799,
    8e-05,
    -0.00788,
    0.00638,
    0.00586,
    0.00113,
    -0.00302,
    0.00669,
    -0.00092,
    -0.00732
   ],
   [
    0.01269,
    0.00486,
    0.00121,
    0.00083,
    0.00316,
    -0.00209,
    -0.00144,
    0.00905,
    0.00612,
    0.0001,
    0.00119,
    0.00335,
    -0.00101,
    -0.00152
   ],
   [
    0.00342,
    0.00308,
    0.00148,
    0.00279,
    -0.00047,
    -0.00165,
    -0.00032,
    0.00388,
    0.00195,
    0.00116,
    0.00183,
    -0.00065,
    -0.00154,
    -0.00035
   ],
   [
    0.00039,
    -0.00276,
    -0.0015,
    9e-05,
    -0.00283,
    -0.00088,
    -0.0022,
    -0.00083,
    -0.00304,
    -0.00116,
    0.00053,
    -0.00232,
    -0.00128,
    -0.00206
   ],
   [
    -0.00066,
    0.00445,
    -8e-05,
    -0.00226,
    0.00438,
    0.00028,
    -0.00184,
    -0.00299,
    0.00257,
    -0.00102,
    -0.00196,
    0.00377,
    0.00043,
    -0.00143
   ],
   [
    0.00824,
    -0.00248,
    0.00101,
    0.00226,
    0.0074,
    0.00064,
    -0.00896,
    0.00296,
    -0.0032,
    -0.00092,
    0.00266,
    0.00622,
    0.00107,
    -0.00697
   ],
   [
    -0.00349,
    -0.00712,
    -0.00297,
    -0.00017,
    -0.00925,
    -0.00011,
    -0.00026,
    -0.00177,
    -0.00723,
    -0.00279,
    -0.00201,
    -0.00907,
    -0.00196,
    -0.00171
   ],
   [
    -0.00369,
    -0.01564,
    0.00306,
    0.00123,
    -0.01279,
    0.00654,
    0.01771,
    -0.00311,
    -0.0113,
    0.00375,
    0.001,
    -0.0086,
    0.00684,
    0.01623
   ],
   [
    -0.01316,
    0.00042,
    -0.00825,
    -0.00595,
    -0.00014,
    0.00058,
    -0.00077,
    -0.00805,
    -0.00171,
    -0.00398,
    -0.00553,
    -0.00129,
    -0.00127,
    -0.00106
   ],
   [
    0.00198,
    0.0004,
    0.00042,
    0.00131,
    -0.00344,
    0.00272,
    0.00184,
    0.00148,
    0.0005,
    -0.0001,
    0.00098,
    -0.00293,
    0.00187,
    0.00134
   ],
   [
    0.00187,
    -0.00511,
    0.00074,
    0.00148,
    -0.00662,
    0.00176,
    0.00684,
    0.00204,
    -0.00286,
    0.00053,
    0.00145,
    -0.00596,
    0.00151,
    0.00545
   ],
   [
    -0.00115,
    -0.00363,
    -0.00488,
    0.00087,
    0.00128,
    -0.00061,
    -0.00691,
    -0.00093,
    -0.00375,
    -0.0035,
    -0.00071,
    -0.00117,
    -0.00243,
    -0.00686
   ],
   [
    0.00782,
    0.00505,
    0.00092,
    -0.00239,
    -0.0017,
    0.00099,
    0.00169,
    0.00578,
    0.00469,
    0.00026,
    -0.00322,
    -0.00168,
    -0.00046,
    0.00121
   ],
   [
    0.00259,
    -0.00074,
    0.00302,
    -9e-05,
    -0.00347,
    0.00194,
    0.00406,
    0.00251,
    -0.00075,
    0.00214,
    9e-05,
    -0.0027,
    0.00194,
    0.0037
   ],
   [
    -0.00145,
    -0.00045,
    0.00025,
    -0.00204,
    0.0015,
    -0.00188,
    -0.00048,
    -0.0003,
    9e-05,
    0.00068,
    -0.00168,
    0.00145,
    -0.00171,
    -0.00054
   ],
   [
    0.00714,
    0.00532,
    -0.0014,
    -0.00252,
    0.00072,
    -0.00295,
    -0.00133,
    0.00637,
    0.00603,
    -0.00089,
    -0.00261,
    -5e-05,
    -0.00188,
    4e-05
   ],
   [
    -0.00011,
    -0.00253,
    0.00026,
    0.0014,
    -0.00019,
    0.00013,
    -0.00312,
    -0.00222,
    -0.00187,
    -0.0006,
    0.00119,
    -0.00078,
    -0.0005,
    -0.00325
   ],
   [
    -0.00889,
    0.001,
    -0.00906,
    0.00151,
    -0.00332,
    0.00049,
    0.00874,
    -0.00693,
    -0.00136,
    -0.00652,
    0.00096,
    -0.00265,
    -0.00012,
    0.00743
   ],
   [
    -0.00308,
    -0.00079,
    0.00308,
    0.00196,
    -0.0033,
    -0.00298,
    0.00312,
    -0.00408,
    0.00061,
    0.00075,
    0.00184,
    -0.00235,
    -0.00258,
    0.00266
   ],
   [
    0.0004,
    0.00317,
    0.00265,
    0.00015,
    -0.00055,
    -0.0008,
    0.00531,
    -0.00135,
    0.00031,
    0.00086,
    0.00069,
    -0.00012,
    0.00079,
    0.00534
   ],
   [
    0.00364,
    -0.00507,
    -0.0026,
    -0.00133,
    0.00101,
    0.00024,
    0.00061,
    0.00297,
    -0.00262,
    -0.00031,
    -0.00056,
    0.00122,
    0.00057,
    0.00099
   ],
   [
    0.00325,
    -0.00043,
    0.00395,
    -0.0025,
    0.00212,
    -0.00264,
    -0.0016,
    0.00101,
    -0.00113,
    0.00159,
    -0.0022,
    0.00148,
    -0.00188,
    -0.0015
   ],
   [
    -0.01901,
    -0.01327,
    -0.00384,
    0.00224,
    -0.00594,
    0.00092,
    0.00165,
    -0.01386,
    -0.00762,
    0.00052,
    0.00254,
    -0.00411,
    0.00187,
    0.00199
   ],
   [
    -0.00013,
    -0.00128,
    -0.00384,
    0.00043,
    -0.00147,
    -0.00031,
    0.00128,
    0.0014,
    -0.00164,
    -0.00168,
    0.00054,
    -0.0017,
    0.00035,
    0.00151
   ],
   [
    -0.00099,
    0.0014,
    0.00111,
    -0.00101,
    -0.00101,
    0.00145,
    0.00076,
    -2e-05,
    0.00085,
    0.00061,
    -0.00051,
    -0.00049,
    0.00163,
    0.00072
   ],
   [
    0.00731,
    0.0012,
    0.00191,
    0.00617,
    -0.01125,
    -0.00453,
    0.01541,
    0.01097,
    0.00655,
    0.00211,
    0.00561,
    -0.00802,
    -0.00386,
    0.01369
   ],
   [
    -0.00228,
    -0.00145,
    -0.0012,
    0.00032,
    0.00101,
    0.00236,
    -0.00062,
    -0.00165,
    -0.00069,
    -0.00078,
    0.00011,
    0.00069,
    0.002,
    -0.00069
   ],
   [
    -0.00136,
    0.00192,
    0.00072,
    0.0002,
    0.00127,
    -0.0004,
    -0.00353,
    0.00066,
    0.00395,
    0.00205,
    -3e-05,
    0.00102,
    -0.00023,
    -0.00309
   ],
   [
    0.00075,
    0.00035,
    1e-05,
    0.00139,
    0.00027,
    -0.00334,
    -0.00025,
    0.00011,
    -0.00026,
    4e-05,
    0.00102,
    -0.00016,
    -0.0031,
    -0.00035
   ],
   [
    -0.00156,
    -0.00223,
    -0.0001,
    -0.00039,
    -0.00251,
    0.00229,
    0.00246,
    -0.00126,
    -0.00175,
    7e-05,
    -0.00031,
    -0.00193,
    0.00195,
    0.00205
   ],
   [
    0.00276,
    0.00232,
    0.00056,
    -0.0003,
    0.00031,
    -2e-05,
    -0.001,
    0.00264,
    0.00358,
    0.00104,
    -0.00029,
    0.00053,
    -0.00045,
    -0.0011
   ],
   [
    -0.00688,
    -0.02595,
    -0.00919,
    0.00176,
    -0.01071,
    0.00067,
    -4e-05,
    -0.00864,
    -0.02138,
    -0.01038,
    0.00252,
    -0.00949,
    0.0016,
    -0.00179
   ],
   [
    0.01565,
    -0.01412,
    0.00761,
    0.0106,
    -0.00728,
    -0.00826,
    -0.01548,
    0.00748,
    -0.01401,
    0.00173,
    0.00946,
    -0.00764,
    -0.00804,
    -0.01387
   ],
   [
    -0.00139,
    0.00051,
    0.0004,
    -0.00172,
    0.00211,
    -0.00176,
    -0.00088,
    -0.00127,
    5e-05,
    0.0002,
    -0.00157,
    0.00181,
    -0.00164,
    -0.00052
   ],
   [
    -0.00095,
    0.00045,
    -4e-05,
    0.00122,
    0.00077,
    -0.00033,
    0.00135,
    0.00027,
    0.00169,
    0.00084,
    0.00164,
    0.00121,
    0.0003,
    0.00122
   ],
   [
    -0.0007,
    -0.00221,
    0.00047,
    0.00037,
    0.00052,
    0.0006,
    -0.00096,
    -0.00016,
    -0.00252,
    -1e-05,
    0.00042,
    0.00028,
    0.00039,
    -0.00073
   ],
   [
    0.00239,
    0.00165,
    0.00125,
    -0.00067,
    0.00018,
    0.00065,
    -0.00027,
    0.00127,
    0.00106,
    -9e-05,
    -0.00063,
    -0.0001,
    0.00041,
    -0.00019
   ],
   [
    0.00041,
    -0.0037,
    0.00146,
    0.00078,
    3e-05,
    -0.00185,
    0.00048,
    -7e-05,
    -0.00269,
    0.00034,
    0.00104,
    0.00023,
    -0.00151,
    0.00062
   ],
   [
    0.00489,
    0.0028,
    0.003,
    0.00123,
    -0.00055,
    0.00102,
    0.00276,
    0.00351,
    0.00268,
    0.00222,
    0.00142,
    0.00017,
    0.00115,
    0.00234
   ],
   [
    0.00084,
    0.00209,
    0.00062,
    0.00196,
    -0.00113,
    0.00046,
    -0.00225,
    0.00323,
    0.00423,
    0.00189,
    0.00172,
    -0.00046,
    0.0005,
    -0.0016
   ],
   [
    0.00117,
    0.00146,
    -0.00235,
    0.0007,
    -0.00159,
    -0.00097,
    0.00081,
    0.0019,
    -0.0001,
    -0.00133,
    -0.00018,
    -0.00192,
    -0.0013,
    0.00033
   ],
   [
    -0.00764,
    -0.0024,
    0.00131,
    0.00368,
    0.00173,
    -0.00421,
    0.00067,
    -0.00856,
    -0.00534,
    -0.0012,
    0.00429,
    0.00147,
    -0.00324,
    0.00078
   ],
   [
    0.00956,
    0.00167,
    0.00175,
    -0.00013,
    -0.00402,
    0.00847,
    -0.00058,
    0.01012,
    0.00334,
    0.0028,
    0.00055,
    -0.0026,
    0.00709,
    -8e-05
   ],
   [
    -0.00231,
    0.00014,
    -4e-05,
    -0.00478,
    -0.00257,
    0.00342,
    0.0017,
    -0.00158,
    0.00091,
    0.0008,
    -0.00394,
    -0.00147,
    0.00228,
    0.00054
   ],
   [
    -7e-05,
    0.00134,
    0.00203,
    0.0007,
    -0.00043,
    0.00098,
    0.00027,
    0.00109,
    0.00119,
    0.00222,
    0.00095,
    7e-05,
    0.00129,
    0.00072
   ],
   [
    -0.0055,
    -0.00115,
    0.00346,
    -0.00144,
    0.00285,
    -0.00317,
    0.00554,
    -0.00577,
    -0.00793,
    -0.0021,
    -0.0013,
    0.00273,
    -0.00234,
    0.00464
   ],
   [
    -0.00013,
    0.00052,
    -0.00029,
    -0.0017,
    -0.00199,
    0.00219,
    0.00093,
    -0.00155,
    -0.00104,
    -0.00038,
    -0.0016,
    -0.00159,
    0.00171,
    0.00063
   ],
   [
    -0.00893,
    -0.00333,
    0.00176,
    0.0012,
    -0.00166,
    -0.0031,
    -0.00362,
    -0.00999,
    -0.00251,
    0.00052,
    0.00042,
    -0.00217,
    -0.00386,
    -0.00377
   ],
   [
    0.00147,
    0.00099,
    0.00066,
    -0.00073,
    -0.0004,
    0.00144,
    0.00235,
    0.00192,
    0.00067,
    0.0013,
    -0.00031,
    0.00018,
    0.00183,
    0.00251
   ],
   [
    0.00705,
    0.00385,
    0.00018,
    0.00147,
    0.00054,
    -0.00065,
    -0.00156,
    0.00672,
    0.00397,
    0.00132,
    0.00158,
    0.00094,
    -0.00065,
    -0.00118
   ],
   [
    -0.0015,
    0.00094,
    -0.00027,
    -0.00036,
    0.00051,
    -8e-05,
    -0.00107,
    -0.00099,
    0.00176,
    -0.0004,
    -0.00046,
    0.00036,
    0.00013,
    -0.00088
   ],
   [
    0.00278,
    -0.00568,
    0.00163,
    0.01608,
    0.00046,
    -0.00317,
    -0.00891,
    -0.00405,
    -0.00832,
    -0.00456,
    0.0148,
    0.00023,
    -0.00202,
    -0.00724
   ],
   [
    0.01204,
    0.0117,
    0.00698,
    0.00029,
    0.01167,
    -0.00584,
    -0.00075,
    0.00709,
    0.01345,
    0.00362,
    0.0003,
    0.01129,
    -0.00607,
    -4e-05
   ],
   [
    -0.00117,
    -0.00293,
    0.00085,
    -2e-05,
    -0.00284,
    0.00023,
    0.00357,
    -0.00109,
    -0.00235,
    0.00051,
    0.00013,
    -0.00213,
    0.00075,
    0.00287
   ],
   [
    0.00875,
    -0.00202,
    -0.00028,
    -0.0044,
    -0.01124,
    0.00592,
    0.015,
    0.0108,
    -0.00013,
    0.00034,
    -0.00254,
    -0.00805,
    0.00654,
    0.01384
   ],
   [
    -0.00061,
    -0.00073,
    -0.001,
    -7e-05,
    -0.00319,
    0.00071,
    0.00395,
    0.00154,
    0.00074,
    0.00011,
    0.00056,
    -0.00214,
    0.00115,
    0.00368
   ],
   [
    0.00601,
    -0.001,
    0.00256,
    0.00287,
    -0.00133,
    -0.00137,
    0.00117,
    0.00323,
    -0.0007,
    0.0005,
    0.00249,
    -0.00062,
    -0.00075,
    0.00122
   ],
   [
    0.00071,
    -0.00089,
    -0.00026,
    -0.0032,
    0.00051,
    0.00195,
    -0.00225,
    0.00034,
    -0.00185,
    -0.00016,
    -0.00292,
    0.00015,
    0.00145,
    -0.00193
   ],
   [
    -0.0162,
    -0.00366,
    0.00313,
    -0.0087,
    0.00545,
    0.00232,
    0.00058,
    -0.01417,
    -0.00546,
    0.00195,
    -0.00821,
    0.004,
    0.00097,
    0.00025
   ],
   [
    -0.00262,
    0.00188,
    -0.00273,
    0.00227,
    0.00138,
    -0.00146,
    -0.00099,
    -0.00148,
    0.00153,
    -0.00158,
    0.00214,
    0.00114,
    -0.00097,
    -0.00079
   ],
   [
    -0.0026,
    -0.00834,
    0.00437,
    0.00427,
    -0.00223,
    0.00041,
    0.00429,
    -0.00229,
    -0.0074,
    0.00167,
    0.00434,
    -0.00067,
    0.00165,
    0.00533
   ],
   [
    0.00098,
    0.00195,
    0.00319,
    0.00084,
    -0.00094,
    -0.00131,
    0.00043,
    0.00158,
    0.0007,
    0.00205,
    0.00064,
    -0.00096,
    -0.00104,
    0.00034
   ],
   [
    0.00263,
    -0.00678,
    -0.00124,
    0.00244,
    -0.00199,
    -0.00032,
    0.00015,
    4e-05,
    -0.00649,
    -0.00226,
    0.00211,
    -0.00181,
    -0.00012,
    -0.00025
   ],
   [
    -0.00221,
    0.00352,
    -0.00187,
    0.00501,
    0.00545,
    -0.00634,
    -0.00313,
    -0.0034,
    0.00216,
    -0.00311,
    0.00435,
    0.00406,
    -0.00566,
    -0.00313
   ],
   [
    0.00231,
    -0.00133,
    -1e-05,
    -0.00156,
    0.00018,
    0.00196,
    0.00056,
    0.00174,
    -0.00055,
    0.00037,
    -0.00107,
    0.00049,
    0.00178,
    0.00075
   ],
   [
    0.00186,
    -0.00108,
    0.00053,
    0.00213,
    -0.00033,
    -0.00307,
    -0.00201,
    0.00178,
    0.00029,
    0.00112,
    0.00193,
    -0.00032,
    -0.00232,
    -0.00187
   ],
   [
    0.00073,
    -0.00068,
    -0.00013,
    0.00045,
    1e-05,
    0.0006,
    0.0044,
    0.00074,
    3e-05,
    0.00033,
    0.00085,
    0.00068,
    0.00156,
    0.00439
   ],
   [
    -0.00052,
    -0.00437,
    -0.00223,
    0.00011,
    0.00053,
    0.0,
    -0.00209,
    -0.00069,
    -0.00423,
    -0.00188,
    -0.00017,
    2e-05,
    -0.00053,
    -0.0019
   ],
   [
    0.00926,
    0.00042,
    0.0002,
    0.00655,
    -0.00029,
    -0.00288,
    -0.00072,
    0.00943,
    -0.00098,
    -0.00078,
    0.00664,
    0.00208,
    -0.00265,
    -0.0012
   ],
   [
    -0.014,
    -0.0113,
    -0.00114,
    -0.00262,
    -0.0141,
    0.00988,
    0.00204,
    -0.01531,
    -0.01088,
    -0.00205,
    -0.00267,
    -0.01113,
    0.00848,
    0.00069
   ],
   [
    -0.00193,
    -0.00011,
    -0.00203,
    -0.00225,
    0.00026,
    0.00139,
    0.00139,
    -0.00389,
    -0.0018,
    -0.00311,
    -0.00243,
    -0.00035,
    0.00067,
    0.00086
   ],
   [
    -0.0013,
    0.0031,
    0.00732,
    0.01046,
    0.0062,
    -0.01014,
    -0.00567,
    -0.00274,
    0.00594,
    0.00486,
    0.0093,
    0.00442,
    -0.01005,
    -0.00466
   ],
   [
    -0.00986,
    -0.00833,
    -0.00423,
    -0.00221,
    -0.00287,
    -0.00087,
    -0.00048,
    -0.01044,
    -0.0084,
    -0.00611,
    -0.00344,
    -0.00356,
    -0.00266,
    -0.00156
   ],
   [
    -0.01734,
    -0.00605,
    -0.00302,
    -0.00414,
    0.0006,
    0.00258,
    0.00015,
    -0.01895,
    -0.00543,
    -0.00324,
    -0.00429,
    -0.00071,
    0.00152,
    -9e-05
   ],
   [
    -0.00071,
    0.00262,
    0.00139,
    0.00125,
    0.00293,
    -0.00323,
    -0.00231,
    0.00034,
    0.00157,
    0.00068,
    0.00138,
    0.00239,
    -0.0027,
    -0.00205
   ],
   [
    -0.00048,
    0.00178,
    -3e-05,
    -0.00069,
    -6e-05,
    0.0027,
    0.00184,
    0.00038,
    0.00229,
    0.00076,
    -0.00043,
    0.0001,
    0.00247,
    0.00189
   ],
   [
    0.00023,
    0.00167,
    0.00116,
    -0.00093,
    0.00281,
    4e-05,
    -0.00056,
    -0.00059,
    0.00174,
    0.00085,
    -0.00085,
    0.00229,
    0.0003,
    -0.00016
   ],
   [
    -0.00459,
    -0.00444,
    -0.0114,
    0.00499,
    0.00057,
    0.00054,
    -0.00026,
    -0.00404,
    -0.00201,
    -0.00643,
    0.00385,
    -0.00108,
    0.00097,
    0.00057
   ],
   [
    0.00151,
    -0.00715,
    0.00458,
    0.00209,
    0.00202,
    -0.00884,
    -0.00161,
    -0.00443,
    -0.00886,
    -0.00077,
    0.002,
    0.00073,
    -0.00829,
    -0.00176
   ],
   [
    -0.00038,
    0.00164,
    -0.00039,
    -0.00047,
    0.00128,
    -5e-05,
    -0.0023,
    -1e-05,
    0.00182,
    -0.00064,
    -0.0008,
    0.00093,
    -0.00041,
    -0.00224
   ],
   [
    -0.00072,
    -0.00264,
    -0.00303,
    0.00087,
    -0.00287,
    -0.00104,
    0.00418,
    -0.00058,
    -0.00101,
    -0.00281,
    0.00098,
    -0.00286,
    -0.0002,
    0.00339
   ],
   [
    0.00155,
    0.0046,
    0.00407,
    0.00297,
    0.00152,
    -0.00131,
    -0.00063,
    0.00111,
    0.00382,
    0.00201,
    0.00381,
    0.00254,
    -1e-05,
    0.00021
   ],
   [
    -0.00178,
    0.00022,
    -0.00222,
    0.00156,
    -0.00064,
    0.00059,
    0.00157,
    0.00097,
    0.00193,
    -0.00057,
    0.00161,
    -4e-05,
    0.00072,
    0.00117
   ],
   [
    0.00473,
    -4e-05,
    0.00189,
    0.00012,
    -0.00109,
    0.00122,
    -0.00115,
    0.00333,
    0.00035,
    0.00095,
    -2e-05,
    -0.00101,
    0.00134,
    -0.00095
   ],
   [
    0.00256,
    -0.00062,
    0.00063,
    0.00227,
    0.0008,
    -0.00309,
    -0.00451,
    0.00322,
    0.00136,
    0.0013,
    0.0018,
    0.00029,
    -0.00328,
    -0.00409
   ],
   [
    0.00185,
    0.00111,
    0.00181,
    -0.00043,
    -0.00092,
    0.00127,
    -0.00066,
    0.00175,
    0.0005,
    0.00064,
    -0.00038,
    -0.00068,
    0.00124,
    -0.00039
   ],
   [
    0.00234,
    0.00118,
    0.00044,
    0.00018,
    0.00787,
    -0.00759,
    0.00157,
    0.00539,
    0.00029,
    -0.00078,
    -0.00144,
    0.00607,
    -0.00612,
    0.00151
   ],
   [
    -0.00023,
    -0.00039,
    0.0006,
    3e-05,
    -9e-05,
    -0.00038,
    0.00112,
    1e-05,
    -0.00023,
    0.00098,
    0.00019,
    7e-05,
    -0.00035,
    0.00124
   ],
   [
    0.00244,
    0.0007,
    -0.00216,
    0.00096,
    -0.00258,
    0.00125,
    -0.00087,
    0.00215,
    0.00205,
    -0.00259,
    -0.00044,
    -0.00322,
    0.0005,
    -0.00124
   ],
   [
    -0.00176,
    0.00327,
    -0.00088,
    -0.00098,
    0.00393,
    0.0012,
    -3e-05,
    0.00079,
    0.00329,
    0.00107,
    -0.0001,
    0.00379,
    0.00215,
    0.00063
   ],
   [
    -0.00067,
    0.00587,
    -6e-05,
    -0.0062,
    0.01728,
    0.00255,
    -0.00694,
    -0.0047,
    0.00181,
    -0.00075,
    -0.00362,
    0.01422,
    0.00251,
    -0.00361
   ],
   [
    -0.00083,
    0.00357,
    -0.00152,
    -0.00113,
    0.00015,
    -0.003,
    0.00502,
    -0.00028,
    0.00391,
    -0.00188,
    -0.00175,
    -2e-05,
    -0.00274,
    0.00399
   ],
   [
    -0.00423,
    -0.00494,
    -0.00648,
    -0.00113,
    -0.00467,
    0.00144,
    0.00246,
    -0.00248,
    -0.00308,
    -0.00406,
    -9e-05,
    -0.00315,
    0.00131,
    0.00167
   ],
   [
    -0.01186,
    -0.0035,
    0.00134,
    0.00043,
    0.00727,
    -0.00319,
    -0.00667,
    -0.01114,
    -0.0055,
    -0.00025,
    -0.00045,
    0.00474,
    -0.00325,
    -0.00601
   ],
   [
    0.00448,
    0.00314,
    0.00155,
    -0.00112,
    0.00203,
    -0.00221,
    -4e-05,
    0.00301,
    0.00213,
    0.00036,
    -0.00107,
    0.00171,
    -0.00161,
    6e-05
   ],
   [
    0.00256,
    0.00044,
    0.00141,
    -0.00332,
    -0.00238,
    0.0006,
    0.00632,
    1e-05,
    -0.0007,
    -0.00253,
    -0.00477,
    -0.00198,
    -0.0004,
    0.0046
   ],
   [
    0.0021,
    0.00301,
    0.00182,
    -0.00027,
    -0.00198,
    0.00034,
    0.00246,
    0.0025,
    0.00248,
    0.00212,
    -0.0001,
    -0.00185,
    1e-05,
    0.00211
   ],
   [
    0.00259,
    -0.01615,
    0.00154,
    0.00565,
    -0.00264,
    -0.0191,
    0.00835,
    0.00102,
    -0.01226,
    -0.00465,
    0.00525,
    -0.00147,
    -0.0137,
    0.00582
   ],
   [
    0.00024,
    -0.00144,
    -0.00079,
    0.00143,
    -0.00068,
    -0.00012,
    0.0014,
    -0.00019,
    -0.00048,
    5e-05,
    0.00129,
    -0.00081,
    -5e-05,
    0.00107
   ],
   [
    0.00206,
    0.00147,
    2e-05,
    0.00176,
    -0.00018,
    0.00149,
    -0.00283,
    0.00122,
    0.0,
    -0.0004,
    0.00173,
    -0.00029,
    0.00149,
    -0.00212
   ],
   [
    0.00378,
    -0.00122,
    7e-05,
    0.00072,
    0.00244,
    -0.00138,
    -0.00419,
    0.00112,
    -0.0008,
    -0.00019,
    0.0009,
    0.00185,
    -0.00156,
    -0.00367
   ],
   [
    0.00514,
    0.00101,
    -0.00188,
    0.00036,
    7e-05,
    0.00057,
    0.00042,
    0.00582,
    0.00077,
    -0.00052,
    0.00019,
    -8e-05,
    1e-05,
    0.0002
   ],
   [
    0.01586,
    0.00631,
    0.00963,
    -0.00081,
    -0.00712,
    -0.00228,
    0.00383,
    0.00944,
    0.00705,
    0.01009,
    0.00091,
    -0.00542,
    -0.002,
    0.00367
   ],
   [
    0.00552,
    0.00352,
    0.00222,
    0.00382,
    -8e-05,
    -0.00121,
    -0.00024,
    0.00732,
    0.00436,
    0.00254,
    0.00391,
    0.0007,
    -0.00046,
    0.00036
   ],
   [
    -0.00277,
    0.00049,
    0.00031,
    -0.00167,
    -0.00481,
    0.00129,
    0.00497,
    -0.00311,
    -0.00164,
    7e-05,
    -0.0012,
    -0.00393,
    0.00099,
    0.00414
   ],
   [
    -0.00564,
    -0.00435,
    0.00077,
    0.00128,
    -0.00295,
    0.00048,
    0.00752,
    -0.00286,
    -0.00457,
    -0.00087,
    -0.00017,
    -0.00288,
    -0.00232,
    0.00574
   ],
   [
    0.0009,
    0.00249,
    0.00118,
    0.00126,
    -0.00029,
    0.00039,
    0.00051,
    0.00087,
    0.00074,
    0.00076,
    0.00093,
    -0.00056,
    -4e-05,
    0.00018
   ],
   [
    0.00351,
    -0.00362,
    -0.00267,
    0.00319,
    -0.00569,
    -0.00077,
    0.00216,
    0.00638,
    -0.00079,
    -0.0006,
    0.0028,
    -0.00454,
    0.00038,
    0.00194
   ],
   [
    -0.00152,
    0.00137,
    -0.00137,
    -2e-05,
    -0.00042,
    0.00018,
    0.00109,
    -0.00053,
    0.00139,
    0.00049,
    0.0001,
    -0.0003,
    0.00059,
    0.00111
   ],
   [
    0.00011,
    0.00212,
    -0.00061,
    -7e-05,
    0.00128,
    -0.00029,
    6e-05,
    -0.00078,
    0.00159,
    0.00032,
    0.00058,
    0.00134,
    0.00051,
    0.00032
   ],
   [
    0.00739,
    -0.00578,
    0.00063,
    0.00609,
    -0.00607,
    -0.00073,
    -0.00452,
    0.00757,
    -0.0037,
    0.00191,
    0.00503,
    -0.00633,
    -0.0032,
    -0.00483
   ],
   [
    -0.00496,
    -0.00289,
    -0.00161,
    -0.00145,
    0.00775,
    -0.00082,
    -0.00296,
    -0.00479,
    -0.00306,
    -0.00185,
    -0.00165,
    0.00631,
    -0.00181,
    -0.00271
   ],
   [
    -0.00032,
    0.00073,
    -0.00026,
    -0.00264,
    0.00279,
    -0.00128,
    -8e-05,
    -0.00054,
    0.00089,
    -0.00115,
    -0.00297,
    0.00196,
    -0.0007,
    0.00042
   ],
   [
    -0.02511,
    -0.00336,
    -0.00125,
    -0.00036,
    0.00631,
    -0.00571,
    0.01133,
    -0.01837,
    -0.0049,
    -0.00234,
    -0.0012,
    0.00594,
    -0.00541,
    0.00974
   ],
   [
    0.00334,
    -0.00468,
    0.00298,
    0.00407,
    -0.00405,
    -0.00046,
    0.00049,
    0.00418,
    -0.00231,
    0.0031,
    0.00357,
    -0.00361,
    0.00064,
    0.00038
   ],
   [
    -0.00208,
    -0.00135,
    0.00087,
    -0.00119,
    0.00024,
    0.00191,
    0.00132,
    -0.00255,
    -0.0014,
    7e-05,
    -0.00037,
    0.00095,
    0.00194,
    0.00145
   ],
   [
    -0.00013,
    0.00279,
    -0.00168,
    0.00213,
    0.00096,
    0.00014,
    -0.00068,
    -0.00039,
    0.00138,
    -0.00263,
    0.00157,
    0.00122,
    8e-05,
    -0.00069
   ],
   [
    -0.00052,
    -0.00105,
    0.00138,
    3e-05,
    -0.00153,
    0.00134,
    0.00194,
    -0.00176,
    -0.00313,
    0.00014,
    -0.00037,
    -0.00143,
    0.0006,
    0.00119
   ],
   [
    -0.00593,
    -0.00603,
    -0.00146,
    0.00065,
    -0.00078,
    -0.00129,
    -0.0009,
    -0.00473,
    -0.00619,
    -0.00175,
    0.0003,
    -0.00122,
    -0.00132,
    -0.00085
   ],
   [
    0.0022,
    -0.00162,
    0.0002,
    -0.00127,
    -0.00266,
    0.00297,
    0.00277,
    0.0004,
    -0.00206,
    -0.00019,
    -0.00111,
    -0.00213,
    0.00276,
    0.0024
   ],
   [
    -0.00209,
    -0.00192,
    0.00102,
    -0.00076,
    -0.00151,
    -0.00024,
    0.00086,
    -0.00218,
    -0.00045,
    0.00097,
    -0.00072,
    -0.00142,
    -0.00027,
    0.00083
   ],
   [
    0.001,
    -0.00454,
    -0.00102,
    -0.00148,
    0.00179,
    -0.00074,
    -0.0062,
    -0.00171,
    -0.00645,
    -0.00247,
    -0.00163,
    0.00059,
    -0.00144,
    -0.00608
   ],
   [
    0.00037,
    0.00347,
    -0.0002,
    -0.00096,
    0.00412,
    0.00064,
    -0.00122,
    0.00095,
    0.00282,
    0.00016,
    -0.00071,
    0.00375,
    0.00102,
    -0.00076
   ],
   [
    -0.0011,
    -0.00041,
    0.00074,
    -0.00043,
    0.00113,
    -0.00035,
    -0.00045,
    -0.00146,
    -0.00215,
    -0.00069,
    -0.00059,
    0.00075,
    -0.00056,
    -0.00056
   ],
   [
    0.00365,
    -0.00309,
    0.0014,
    -0.00085,
    8e-05,
    0.00088,
    0.00036,
    0.00133,
    -0.00187,
    -0.00019,
    -0.00062,
    0.00044,
    0.00021,
    7e-05
   ],
   [
    -0.00395,
    -0.00059,
    -0.00313,
    0.00062,
    -5e-05,
    -0.00097,
    -0.00101,
    -0.00261,
    -0.00134,
    -0.00129,
    -0.00043,
    -0.00115,
    -0.00126,
    -0.00114
   ],
   [
    -0.00123,
    -0.00184,
    0.00223,
    0.00265,
    -0.00431,
    -0.00248,
    0.00122,
    -0.001,
    -0.00166,
    0.00078,
    0.00209,
    -0.00359,
    -0.00211,
    0.00086
   ],
   [
    -0.00148,
    -3e-05,
    -0.00053,
    -0.0007,
    0.00057,
    0.00048,
    0.0023,
    -0.00097,
    0.00023,
    0.00016,
    -0.00058,
    0.00075,
    0.00093,
    0.00209
   ],
   [
    0.00421,
    0.00077,
    0.0001,
    0.00085,
    -0.0002,
    -0.0002,
    -0.00195,
    0.00362,
    0.0004,
    -0.00011,
    0.00078,
    -0.00019,
    -0.00033,
    -0.00167
   ],
   [
    -0.00139,
    0.00228,
    0.00015,
    -0.0015,
    -0.00039,
    0.00164,
    0.00174,
    -8e-05,
    0.00239,
    0.0011,
    -0.00157,
    -0.00026,
    0.00162,
    0.00168
   ],
   [
    -0.00334,
    -0.00109,
    -0.00201,
    -0.00475,
    0.00156,
    0.0034,
    0.00114,
    -0.00279,
    -0.00036,
    -0.00183,
    -0.00525,
    0.00087,
    0.00241,
    0.00088
   ],
   [
    0.00116,
    0.00036,
    0.00096,
    0.00293,
    -0.00071,
    -0.00088,
    0.00124,
    0.00316,
    0.00241,
    0.00193,
    0.00269,
    -0.00095,
    0.00058,
    0.00175
   ],
   [
    0.00455,
    0.00356,
    0.00254,
    -0.00156,
    0.00143,
    -0.00145,
    -0.0017,
    0.00347,
    0.00379,
    0.00238,
    -0.00149,
    0.00127,
    -0.00099,
    -0.00129
   ],
   [
    -0.00645,
    -0.00386,
    -0.00202,
    -0.00026,
    0.00073,
    0.00209,
    -0.00115,
    -0.00442,
    -0.00199,
    -0.00159,
    -0.00074,
    0.00041,
    0.00109,
    -0.00147
   ],
   [
    -0.00019,
    0.00158,
    0.00178,
    0.00064,
    0.0017,
    -0.00046,
    -0.00223,
    0.00104,
    0.00192,
    0.00182,
    -0.00029,
    0.00071,
    -0.00132,
    -0.00237
   ],
   [
    0.00825,
    0.0143,
    0.01902,
    -0.00123,
    0.00826,
    -0.00043,
    -0.00314,
    0.01467,
    0.0172,
    0.01564,
    -0.00046,
    0.01019,
    -0.00077,
    -0.00167
   ],
   [
    -0.01274,
    -0.00109,
    -0.01236,
    0.00652,
    0.00559,
    -0.00739,
    -0.00186,
    -0.01266,
    -0.00385,
    -0.0127,
    0.00612,
    0.00365,
    -0.00709,
    -0.003
   ],
   [
    -0.0003,
    0.00096,
    -0.00191,
    6e-05,
    0.00034,
    -8e-05,
    -0.00163,
    0.00049,
    0.00164,
    -0.00083,
    0.00018,
    0.00061,
    -0.00021,
    -0.00114
   ],
   [
    -0.0026,
    -0.00262,
    0.00136,
    -0.00127,
    0.00205,
    -0.00212,
    -0.00152,
    -0.00298,
    -0.00258,
    0.00155,
    -0.00077,
    0.00167,
    -0.00195,
    -0.00093
   ],
   [
    -0.00391,
    -0.00532,
    0.00016,
    -0.00128,
    -0.00092,
    -0.00101,
    0.0006,
    -0.00515,
    -0.00491,
    -0.00105,
    -0.00122,
    -0.00091,
    -0.00076,
    0.00033
   ],
   [
    -0.0029,
    -0.00117,
    -0.00044,
    -0.0024,
    -0.00123,
    0.00341,
    0.00116,
    -0.00078,
    0.00043,
    0.00132,
    -0.00226,
    -0.00115,
    0.00269,
    0.00101
   ],
   [
    -0.0034,
    0.00346,
    0.00163,
    0.00156,
    -0.00038,
    0.00048,
    0.00076,
    -0.00201,
    0.00257,
    0.00154,
    0.00184,
    -0.00015,
    0.00095,
    0.00134
   ],
   [
    0.00265,
    0.0006,
    4e-05,
    0.00098,
    -0.00191,
    3e-05,
    0.00349,
    0.00302,
    0.00091,
    0.00069,
    0.00131,
    -0.00122,
    0.00057,
    0.00306
   ],
   [
    -0.00126,
    -0.0019,
    -0.00163,
    -0.00216,
    -0.00076,
    0.0018,
    0.00122,
    -0.00052,
    -0.00121,
    -6e-05,
    -0.0021,
    -0.00066,
    0.00085,
    0.0009
   ],
   [
    -0.00133,
    -0.00191,
    -0.0031,
    0.00032,
    -0.00041,
    -0.00135,
    -0.00104,
    -0.00176,
    -0.00229,
    -0.003,
    -0.0001,
    -0.00092,
    -0.002,
    -0.00136
   ],
   [
    -0.00252,
    0.00256,
    -0.00215,
    -0.00085,
    0.00191,
    0.00145,
    -0.0013,
    -0.00185,
    0.00073,
    -0.00124,
    -0.00035,
    0.00144,
    0.001,
    -0.00089
   ],
   [
    -0.0036,
    -0.00352,
    0.00109,
    0.00023,
    0.00111,
    0.00033,
    -0.00372,
    -0.00398,
    -0.00326,
    -0.00039,
    -0.0003,
    0.00019,
    -0.00038,
    -0.00324
   ],
   [
    -0.00143,
    0.00324,
    -0.00311,
    -0.00334,
    0.0033,
    0.00039,
    -0.00064,
    0.00132,
    0.00451,
    -0.00041,
    -0.00268,
    0.00313,
    0.00116,
    -0.00018
   ],
   [
    -0.0004,
    -0.00137,
    -0.00212,
    -0.00093,
    0.00248,
    0.00189,
    -0.002,
    -0.00131,
    -0.00159,
    -0.00153,
    -0.00091,
    0.00196,
    0.00135,
    -0.00184
   ],
   [
    0.00396,
    0.00129,
    -0.00255,
    0.00341,
    9e-05,
    -0.00015,
    -0.00583,
    0.00449,
    -0.00275,
    -0.00189,
    0.00293,
    -0.00087,
    -0.00097,
    -0.0058
   ],
   [
    0.00424,
    0.0005,
    0.00257,
    -0.00188,
    -0.00259,
    0.00072,
    0.00556,
    0.00199,
    0.00143,
    4e-05,
    -0.00147,
    -0.00153,
    0.00055,
    0.00473
   ],
   [
    -0.00154,
    -0.00402,
    0.00025,
    -0.0002,
    0.00045,
    -0.00289,
    0.00186,
    -0.00308,
    -0.0047,
    -0.00059,
    -0.00029,
    0.0001,
    -0.00284,
    0.00108
   ],
   [
    -0.00472,
    -0.00509,
    -0.00358,
    0.00215,
    -0.00026,
    -2e-05,
    -0.00014,
    -0.00554,
    -0.00491,
    -0.00305,
    0.00199,
    -0.00024,
    -0.00031,
    -0.00032
   ],
   [
    0.00037,
    -0.00371,
    -0.00108,
    0.00091,
    -0.00154,
    0.00038,
    -0.00011,
    -0.00029,
    -0.00284,
    -0.00067,
    0.00114,
    -0.0011,
    0.00041,
    0.00017
   ],
   [
    -0.00142,
    -0.00247,
    -0.00015,
    0.00045,
    -0.0065,
    0.00296,
    0.0084,
    -0.00189,
    -0.00228,
    0.0005,
    0.00105,
    -0.00486,
    0.00276,
    0.00725
   ],
   [
    0.00224,
    -0.0002,
    -0.00346,
    -0.00201,
    0.00094,
    -0.00016,
    -0.00126,
    0.00213,
    0.00025,
    -0.00237,
    -0.00108,
    0.00102,
    9e-05,
    -0.0011
   ],
   [
    -0.00016,
    -0.00206,
    0.00021,
    0.0003,
    0.00499,
    -0.00134,
    -0.00489,
    0.00077,
    -0.00212,
    8e-05,
    -4e-05,
    0.00403,
    -0.00123,
    -0.00394
   ],
   [
    0.0031,
    0.0006,
    -0.00049,
    -0.00174,
    0.00136,
    -0.00047,
    0.00021,
    0.00154,
    -0.0004,
    -0.00038,
    -0.00158,
    0.00112,
    -0.00076,
    6e-05
   ],
   [
    0.00127,
    0.00161,
    -0.00128,
    0.0015,
    -0.00318,
    0.00182,
    -0.00026,
    0.00284,
    0.0003,
    9e-05,
    0.00125,
    -0.00248,
    0.00168,
    -0.00015
   ],
   [
    -0.0004,
    -0.00075,
    0.00037,
    -0.00063,
    -0.00077,
    0.00104,
    -0.00264,
    -0.0,
    -7e-05,
    0.00028,
    -0.00087,
    -0.00103,
    0.00075,
    -0.00247
   ],
   [
    0.00651,
    0.00279,
    0.0025,
    0.00125,
    0.00087,
    0.00231,
    -0.00222,
    0.00679,
    0.00464,
    0.00237,
    0.00188,
    0.00154,
    0.00282,
    -0.00139
   ],
   [
    4e-05,
    -0.0018,
    -8e-05,
    0.00014,
    0.00148,
    -0.00255,
    -0.00029,
    -0.00135,
    -0.00067,
    -0.00081,
    0.00024,
    0.00133,
    -0.00203,
    -0.0002
   ],
   [
    0.00451,
    -0.00424,
    0.00715,
    -0.00075,
    0.0029,
    -0.00332,
    -0.00068,
    0.00129,
    -0.00237,
    0.00201,
    0.00016,
    0.00228,
    -0.0032,
    -0.00087
   ],
   [
    -0.00508,
    -0.00981,
    -6e-05,
    0.00367,
    -0.00086,
    -0.00232,
    -0.00789,
    -0.00249,
    -0.00844,
    0.00076,
    0.00319,
    -0.00087,
    -0.00277,
    -0.0071
   ],
   [
    -0.00117,
    0.0037,
    0.00016,
    -0.00039,
    -0.00142,
    0.00136,
    0.0012,
    2e-05,
    0.00231,
    -8e-05,
    -0.00051,
    -0.00128,
    0.00126,
    0.001
   ],
   [
    0.01953,
    -0.00098,
    0.01375,
    0.00834,
    -0.00526,
    -0.00706,
    -0.00766,
    0.0153,
    -0.00018,
    0.00914,
    0.00916,
    -0.00427,
    -0.00638,
    -0.00652
   ],
   [
    3e-05,
    0.00181,
    0.00089,
    0.00015,
    -0.00037,
    0.00149,
    0.00344,
    -4e-05,
    0.00135,
    0.00039,
    -5e-05,
    -0.00021,
    0.00141,
    0.00302
   ],
   [
    -0.00145,
    0.00038,
    -0.00063,
    -0.00087,
    0.00121,
    0.00098,
    -0.00386,
    0.0003,
    0.00046,
    0.00056,
    -0.00076,
    0.00096,
    0.00065,
    -0.00339
   ],
   [
    0.00156,
    0.00154,
    0.00029,
    -0.00039,
    0.00229,
    0.0015,
    0.00046,
    0.00044,
    0.00204,
    0.00081,
    0.00035,
    0.00226,
    0.00226,
    0.00082
   ],
   [
    0.00324,
    0.00312,
    -0.00139,
    -0.00214,
    0.00346,
    -5e-05,
    -0.00225,
    0.0031,
    0.00419,
    -0.00044,
    -0.00208,
    0.00292,
    0.00033,
    -0.00204
   ],
   [
    -0.00077,
    -0.0018,
    -0.00296,
    -0.00114,
    -0.00135,
    0.00151,
    0.00099,
    -0.00116,
    -0.00178,
    -0.00178,
    -0.00049,
    -0.00055,
    0.00189,
    0.00112
   ],
   [
    0.00474,
    0.00201,
    0.00172,
    0.00073,
    -0.00054,
    -0.00219,
    -0.00059,
    0.0041,
    0.00239,
    0.00178,
    0.00116,
    -0.00015,
    -0.0012,
    -0.00041
   ],
   [
    0.00313,
    -0.02707,
    -0.00059,
    0.00504,
    -0.01505,
    -0.00357,
    0.00386,
    -0.00097,
    -0.02729,
    -0.00359,
    0.00581,
    -0.01399,
    -0.0039,
    0.0015
   ],
   [
    -0.009,
    -0.01597,
    -0.00865,
    0.01152,
    -0.00327,
    -0.00103,
    -0.01398,
    -0.01082,
    -0.01473,
    -0.00822,
    0.01129,
    -0.00384,
    0.00034,
    -0.01184
   ],
   [
    -0.00427,
    -0.001,
    0.00073,
    0.00421,
    0.00351,
    -0.00703,
    -0.00669,
    -0.00231,
    -0.00119,
    0.00213,
    0.0045,
    0.0025,
    -0.00542,
    -0.00566
   ],
   [
    0.00047,
    -0.00325,
    -0.00095,
    0.00088,
    0.00263,
    -0.00012,
    -0.00282,
    0.00229,
    -0.00123,
    0.00015,
    0.00101,
    0.00229,
    0.00029,
    -0.00191
   ],
   [
    0.00727,
    0.00414,
    0.0037,
    0.00329,
    0.00117,
    -0.00111,
    -0.00628,
    0.00368,
    0.00285,
    0.0005,
    0.00181,
    0.00035,
    -0.00287,
    -0.00685
   ],
   [
    -0.00443,
    0.00297,
    -0.00191,
    -0.00513,
    -0.00879,
    0.00748,
    0.01177,
    0.00239,
    0.00466,
    0.00366,
    -0.00451,
    -0.00504,
    0.00673,
    0.0108
   ],
   [
    0.00193,
    -0.00118,
    -0.00913,
    -0.0029,
    0.00336,
    0.00029,
    -0.01015,
    -0.00053,
    -0.0028,
    -0.01135,
    -0.00219,
    0.00245,
    0.00047,
    -0.00871
   ],
   [
    -0.00751,
    -0.00287,
    -0.00251,
    0.00054,
    -0.00413,
    0.00178,
    0.00588,
    -0.0058,
    -0.00221,
    -0.00156,
    0.00076,
    -0.00315,
    0.00197,
    0.00523
   ],
   [
    -0.00246,
    -0.00344,
    0.00079,
    -0.00086,
    -0.00189,
    0.0001,
    0.00296,
    -0.0029,
    -0.00472,
    -0.00096,
    -0.00102,
    -0.00192,
    -0.00017,
    0.00227
   ],
   [
    0.00241,
    0.0012,
    0.00101,
    -0.00031,
    -2e-05,
    0.00031,
    0.00276,
    0.00116,
    0.00209,
    0.00054,
    -0.0006,
    0.0002,
    0.00041,
    0.00229
   ],
   [
    0.00218,
    0.00294,
    -0.00023,
    -0.00239,
    0.00026,
    -0.00063,
    0.00107,
    0.00129,
    0.00191,
    -0.00149,
    -0.00281,
    -0.00014,
    -0.00089,
    0.00041
   ],
   [
    0.00015,
    -0.00132,
    0.00135,
    0.00078,
    -0.00051,
    -0.00101,
    0.00063,
    -0.00018,
    -0.00142,
    0.00047,
    0.00084,
    -0.00057,
    -0.00082,
    0.00036
   ],
   [
    -0.0007,
    -0.00054,
    0.00294,
    0.00091,
    0.00111,
    -0.00236,
    -0.00307,
    -0.00081,
    -0.00119,
    0.00106,
    0.0007,
    0.00067,
    -0.00237,
    -0.00316
   ],
   [
    -0.00186,
    -0.00157,
    -0.00108,
    -0.0031,
    -0.00166,
    -0.00092,
    -0.00742,
    -0.00221,
    -0.00145,
    -0.00104,
    -0.00376,
    -0.00177,
    -0.00289,
    -0.00753
   ],
   [
    0.00144,
    -0.00294,
    0.00263,
    0.00284,
    -0.00141,
    -0.0012,
    0.00043,
    -0.00039,
    -0.00259,
    0.00101,
    0.00329,
    -0.00062,
    -0.00013,
    0.00057
   ],
   [
    0.0033,
    0.00369,
    0.00039,
    -0.00163,
    -0.00037,
    0.00341,
    0.00211,
    0.00238,
    0.0036,
    -0.00047,
    -0.00123,
    0.00033,
    0.00291,
    0.00168
   ],
   [
    0.00225,
    0.00516,
    -0.00211,
    0.00124,
    -0.00234,
    -0.0008,
    0.00177,
    0.00104,
    0.00366,
    -0.00081,
    0.00131,
    -0.00161,
    -0.00033,
    0.00166
   ],
   [
    -0.00146,
    -0.00022,
    -0.00103,
    -0.00112,
    -0.00037,
    0.00138,
    0.00102,
    -0.00083,
    0.00078,
    -1e-05,
    -0.00115,
    -0.00025,
    0.00141,
    0.00103
   ],
   [
    -0.00704,
    -0.00108,
    -0.00075,
    0.00077,
    0.00185,
    0.00206,
    0.00202,
    -0.00412,
    -0.00042,
    9e-05,
    0.00197,
    0.00257,
    0.00235,
    0.00174
   ],
   [
    -0.00033,
    -0.00406,
    -0.00155,
    -0.00198,
    0.00075,
    -0.00064,
    -0.00287,
    -0.00043,
    -0.00466,
    -0.00076,
    -0.00183,
    -2e-05,
    -0.00082,
    -0.00265
   ],
   [
    -0.00204,
    -0.00122,
    -0.00231,
    -0.0005,
    0.00079,
    0.00159,
    -0.00166,
    -0.00166,
    -0.00187,
    -0.0006,
    -0.00059,
    0.00018,
    0.00083,
    -0.00137
   ],
   [
    0.00066,
    -0.00054,
    0.00038,
    0.00133,
    -0.00136,
    0.00072,
    -0.00127,
    0.0001,
    -0.0012,
    0.00021,
    0.00081,
    -0.00147,
    0.00046,
    -0.00102
   ],
   [
    -0.00034,
    0.00351,
    -0.00316,
    0.00101,
    0.00363,
    -0.00136,
    -0.00688,
    0.00141,
    0.00373,
    -0.00186,
    0.00065,
    0.00273,
    -0.0015,
    -0.00627
   ],
   [
    -0.00486,
    -0.00361,
    0.00039,
    0.00077,
    -0.00222,
    -0.00036,
    0.00196,
    -0.00503,
    -0.00323,
    -0.00034,
    0.00031,
    -0.00221,
    -0.00018,
    0.00161
   ],
   [
    0.00202,
    0.00019,
    0.00042,
    -0.00156,
    0.00226,
    -0.00085,
    -0.00109,
    0.00266,
    1e-05,
    -0.00042,
    -0.00173,
    0.0015,
    -0.00108,
    -0.00121
   ],
   [
    -0.0014,
    -0.00369,
    -0.00145,
    -0.00348,
    -0.00094,
    0.00164,
    0.00274,
    -0.00221,
    -0.00228,
    -0.00126,
    -0.00304,
    -0.00083,
    0.00129,
    0.00221
   ],
   [
    0.00043,
    -0.00263,
    -0.00356,
    0.00188,
    -0.00229,
    -0.00139,
    0.0022,
    -5e-05,
    -0.00167,
    -0.0016,
    0.00274,
    -0.00087,
    0.00022,
    0.00251
   ],
   [
    0.00099,
    -0.00094,
    -0.00274,
    -0.00264,
    0.00078,
    0.00095,
    0.00044,
    5e-05,
    -6e-05,
    -0.00217,
    -0.0028,
    0.00032,
    0.00018,
    -8e-05
   ],
   [
    0.00086,
    -0.00122,
    0.0013,
    0.00248,
    -0.00056,
    -0.00557,
    -0.003,
    -0.00053,
    -0.00223,
    0.00049,
    0.0022,
    -0.0015,
    -0.00562,
    -0.00295
   ],
   [
    -0.00456,
    -0.03683,
    -0.00906,
    0.00682,
    -0.00337,
    -0.00374,
    -0.01394,
    -0.0077,
    -0.02933,
    -0.00801,
    0.00892,
    -0.00475,
    -0.00509,
    -0.01171
   ],
   [
    -0.00109,
    -0.0001,
    -0.00101,
    -0.00048,
    0.00053,
    -0.00096,
    -0.00198,
    -0.00046,
    0.00076,
    0.00061,
    0.0,
    0.00041,
    -0.0003,
    -0.0014
   ],
   [
    0.00727,
    -0.00472,
    0.00354,
    -0.00286,
    -0.00845,
    0.01059,
    0.0009,
    0.00282,
    -0.0037,
    0.00231,
    -0.00177,
    -0.00631,
    0.00992,
    0.00093
   ],
   [
    -0.00465,
    0.00633,
    0.00319,
    -0.00855,
    -0.00137,
    0.01056,
    0.01053,
    -0.00356,
    0.00328,
    -0.00121,
    -0.00924,
    -0.00082,
    0.01151,
    0.0104
   ],
   [
    -0.00013,
    0.00491,
    0.00457,
    -0.00034,
    0.00123,
    -0.00105,
    0.00307,
    -0.00067,
    0.00591,
    0.00161,
    0.00058,
    0.00193,
    -0.00053,
    0.00287
   ],
   [
    -0.00414,
    -0.00217,
    -0.00068,
    0.00169,
    -0.00182,
    -0.00358,
    0.00048,
    0.00096,
    4e-05,
    0.00136,
    0.0018,
    -0.00124,
    -0.00225,
    0.00041
   ],
   [
    0.00116,
    0.00031,
    0.00025,
    0.00295,
    -0.0004,
    0.00015,
    0.00092,
    0.00108,
    0.00137,
    0.00093,
    0.00279,
    -0.0002,
    0.00025,
    0.00102
   ],
   [
    -9e-05,
    -0.00175,
    -0.00087,
    0.00051,
    -0.00316,
    -0.00031,
    0.00113,
    -0.00147,
    -0.00152,
    -0.00108,
    -4e-05,
    -0.00296,
    -0.00054,
    0.00043
   ],
   [
    -0.00147,
    0.0018,
    -0.00346,
    -0.00071,
    0.00013,
    0.00055,
    0.00139,
    -0.00012,
    0.00101,
    -0.00119,
    -0.00064,
    0.00031,
    0.00065,
    0.00113
   ],
   [
    0.00093,
    -0.0024,
    0.00152,
    0.00341,
    -0.00301,
    -0.00124,
    -0.00035,
    0.00153,
    -0.00188,
    0.00141,
    0.00273,
    -0.00286,
    -0.00132,
    -0.00047
   ],
   [
    -0.00112,
    0.00014,
    -0.00063,
    -0.00064,
    -0.00013,
    0.0002,
    0.00087,
    -0.00091,
    0.00057,
    -0.00042,
    -0.00101,
    -0.00029,
    0.00018,
    0.00085
   ],
   [
    0.00309,
    0.00465,
    0.00076,
    -0.00104,
    0.00184,
    0.00128,
    -0.00042,
    0.00317,
    0.00548,
    0.00131,
    -0.00109,
    0.00166,
    0.00118,
    1e-05
   ],
   [
    -0.00679,
    0.00049,
    0.00015,
    0.00108,
    0.00167,
    0.00331,
    0.00031,
    -0.00525,
    -0.00224,
    0.00098,
    0.00112,
    0.00151,
    0.00244,
    0.00057
   ],
   [
    0.0017,
    0.00088,
    -0.00129,
    0.00204,
    -0.00128,
    -0.00083,
    5e-05,
    0.00257,
    0.00118,
    -0.00118,
    0.00191,
    -0.00099,
    -0.00024,
    0.0003
   ],
   [
    -0.00699,
    -0.00634,
    -0.00341,
    -0.00097,
    -0.00014,
    -0.00416,
    0.00058,
    -0.0063,
    -0.00565,
    -0.00166,
    -0.00127,
    -0.00097,
    -0.00384,
    0.00026
   ],
   [
    -0.00429,
    -0.0058,
    -0.00318,
    -0.00069,
    0.00327,
    -0.00344,
    -0.00066,
    -0.00322,
    -0.00559,
    -0.00204,
    -0.00075,
    0.00181,
    -0.00264,
    -0.00103
   ],
   [
    -0.00039,
    -0.00012,
    -0.00014,
    0.00071,
    2e-05,
    -0.00029,
    -0.00067,
    -7e-05,
    -6e-05,
    -0.00025,
    0.00085,
    0.00011,
    -0.00026,
    -0.00049
   ],
   [
    0.00258,
    -0.00454,
    0.00047,
    0.00219,
    -0.0016,
    0.00041,
    0.00013,
    0.00426,
    -0.00301,
    0.00074,
    0.00198,
    -0.00104,
    7e-05,
    -0.0001
   ],
   [
    -0.00054,
    -0.0023,
    -0.001,
    0.00057,
    0.00339,
    -0.00236,
    -0.0064,
    -0.00129,
    -0.0018,
    -0.00115,
    6e-05,
    0.00243,
    -0.00316,
    -0.00601
   ],
   [
    0.00065,
    -0.00187,
    0.00095,
    0.00118,
    0.00142,
    0.0023,
    -0.00217,
    0.00085,
    -0.00077,
    0.00112,
    0.00148,
    0.00135,
    0.00228,
    -0.00151
   ],
   [
    0.0007,
    -0.00603,
    -0.00283,
    -0.00245,
    0.0015,
    -0.00093,
    -0.00467,
    -0.00148,
    -0.00593,
    -0.00297,
    -0.00306,
    0.00032,
    -0.00157,
    -0.00457
   ],
   [
    -0.00487,
    -0.00019,
    0.00231,
    -0.00522,
    0.00393,
    -0.00197,
    0.00094,
    -0.00666,
    -0.00128,
    -0.00021,
    -0.0041,
    0.00395,
    -0.00106,
    0.00091
   ],
   [
    -0.00103,
    -0.00369,
    -0.00025,
    0.00057,
    -0.00249,
    0.00037,
    -0.00272,
    -0.00265,
    -0.00395,
    -0.0012,
    0.00036,
    -0.00238,
    0.00055,
    -0.00194
   ],
   [
    0.00606,
    0.00952,
    0.00665,
    0.00355,
    0.00923,
    -0.00334,
    -0.00132,
    0.00847,
    0.00956,
    0.00675,
    0.0032,
    0.00916,
    -0.00161,
    0.00031
   ],
   [
    0.00283,
    0.00465,
    -0.00018,
    -0.00028,
    0.0005,
    0.00096,
    0.00092,
    0.00433,
    0.0048,
    0.0032,
    -0.00028,
    0.0004,
    0.00217,
    0.0014
   ],
   [
    0.01362,
    -0.00379,
    0.00831,
    -0.0036,
    -0.0089,
    5e-05,
    0.01223,
    0.01055,
    -0.00238,
    0.00437,
    -0.00375,
    -0.00763,
    -0.00017,
    0.0107
   ],
   [
    -0.00293,
    -0.00188,
    0.00014,
    0.00077,
    0.00118,
    -0.00076,
    2e-05,
    -0.00259,
    -0.00174,
    0.00019,
    0.00109,
    0.00107,
    -0.00088,
    -0.00017
   ],
   [
    -0.01066,
    -0.00017,
    -0.00751,
    -0.00874,
    0.00798,
    0.00496,
    -0.01283,
    -0.00996,
    -0.00105,
    -0.00606,
    -0.00821,
    0.006,
    0.00325,
    -0.01153
   ],
   [
    0.00075,
    0.00089,
    -0.00305,
    0.00039,
    0.00047,
    -0.00246,
    -0.00092,
    0.00048,
    0.00085,
    -0.00158,
    0.00027,
    9e-05,
    -0.00224,
    -0.00073
   ],
   [
    -0.00107,
    -0.00228,
    -0.00045,
    0.00049,
    -0.00324,
    0.00213,
    0.00052,
    0.00158,
    -0.00071,
    0.00168,
    0.00073,
    -0.00256,
    0.00186,
    0.00085
   ],
   [
    -0.00049,
    -0.00281,
    -0.00302,
    -0.00044,
    -0.0026,
    0.00352,
    0.00201,
    -0.00108,
    -8e-05,
    -0.0031,
    -1e-05,
    -0.00166,
    0.00375,
    0.00149
   ],
   [
    0.00304,
    -0.00223,
    0.00279,
    0.00079,
    -0.00605,
    0.00306,
    0.00632,
    0.00532,
    7e-05,
    0.00404,
    0.00133,
    -0.00374,
    0.00355,
    0.0055
   ],
   [
    0.00171,
    0.0036,
    0.00545,
    0.00326,
    0.00697,
    -0.00446,
    -0.00112,
    0.00128,
    0.00302,
    0.00221,
    0.00306,
    0.0064,
    -0.00412,
    -0.00081
   ],
   [
    0.00341,
    0.00073,
    0.00159,
    0.00027,
    -0.00173,
    -0.00055,
    0.00096,
    0.00189,
    0.00048,
    -0.00023,
    0.00028,
    -0.00148,
    -0.00032,
    0.00041
   ],
   [
    0.00403,
    0.00844,
    0.00717,
    -0.0002,
    -0.00548,
    -0.0017,
    -0.00062,
    0.00177,
    0.00645,
    0.00245,
    -0.00047,
    -0.00452,
    -0.00292,
    -0.00054
   ],
   [
    -0.01504,
    0.0061,
    -0.02663,
    0.01515,
    0.03265,
    -0.03134,
    -0.04458,
    -0.01823,
    -0.00075,
    -0.02234,
    0.01174,
    0.02209,
    -0.03025,
    -0.03999
   ],
   [
    0.00057,
    0.00017,
    0.00087,
    0.00172,
    0.00135,
    -0.00218,
    -0.00022,
    -0.00068,
    0.00107,
    0.00019,
    0.0013,
    0.00093,
    -0.00128,
    3e-05
   ],
   [
    0.00575,
    -0.0006,
    0.00296,
    0.00287,
    6e-05,
    0.00052,
    -0.00134,
    0.0033,
    -0.00201,
    6e-05,
    0.00327,
    0.00051,
    0.00103,
    -0.00103
   ],
   [
    0.00047,
    0.00122,
    0.00104,
    0.00023,
    0.00194,
    -0.0028,
    0.00093,
    5e-05,
    0.0004,
    0.00018,
    0.00017,
    0.00155,
    -0.00277,
    0.00027
   ],
   [
    -7e-05,
    -0.00398,
    0.0006,
    0.00249,
    -0.00047,
    6e-05,
    -0.00107,
    0.00077,
    -0.00137,
    0.00125,
    0.00275,
    0.00019,
    0.00076,
    -0.00046
   ],
   [
    -0.00157,
    -0.00146,
    0.00168,
    -0.00039,
    -0.00046,
    8e-05,
    5e-05,
    -0.00136,
    -0.00183,
    0.00165,
    -0.00041,
    -0.00039,
    -0.00013,
    0.00029
   ],
   [
    -0.0024,
    -0.00273,
    0.00183,
    0.00051,
    0.00026,
    2e-05,
    0.00141,
    -0.00323,
    -0.00179,
    0.00067,
    0.00064,
    0.00037,
    0.00056,
    0.00148
   ],
   [
    -0.0007,
    0.00149,
    -0.00023,
    -0.00098,
    0.0022,
    -0.00133,
    0.00305,
    -0.00072,
    0.00172,
    -0.00044,
    -0.00065,
    0.00198,
    -0.00097,
    0.00268
   ],
   [
    0.00515,
    -0.00175,
    0.00164,
    0.00279,
    0.00043,
    -0.00512,
    0.00256,
    0.00084,
    -0.00272,
    -0.00016,
    0.00324,
    0.00109,
    -0.00337,
    0.00311
   ],
   [
    0.00813,
    0.00537,
    0.00288,
    -0.00181,
    0.00331,
    -0.0016,
    0.00026,
    0.00515,
    0.00445,
    0.00077,
    -0.00221,
    0.00327,
    -0.00104,
    0.00024
   ],
   [
    -0.00268,
    -0.00062,
    0.00127,
    -0.00137,
    0.00125,
    0.00033,
    -0.00076,
    -0.00216,
    -0.0005,
    0.00029,
    -0.00156,
    0.00092,
    -0.00037,
    -0.00057
   ],
   [
    0.00037,
    0.00307,
    -0.00047,
    0.00186,
    0.00016,
    0.00027,
    0.00054,
    0.00082,
    0.00343,
    0.0001,
    0.00201,
    0.00064,
    0.00056,
    0.00056
   ],
   [
    -0.00376,
    -0.0016,
    -0.00102,
    -0.00166,
    0.00107,
    0.00083,
    -0.0013,
    -0.0023,
    -0.00117,
    -0.00103,
    -0.00162,
    0.00088,
    -1e-05,
    -0.00152
   ],
   [
    -0.00025,
    0.00078,
    -0.00291,
    -0.00053,
    0.00074,
    0.00056,
    0.00019,
    0.00027,
    0.00061,
    -0.00053,
    -1e-05,
    0.00014,
    0.00045,
    1e-05
   ],
   [
    0.0085,
    0.00769,
    -0.00822,
    0.00751,
    -0.00312,
    0.00092,
    0.00468,
    0.01272,
    0.00912,
    -0.00093,
    0.00699,
    -0.00164,
    0.00164,
    0.00444
   ],
   [
    -0.00175,
    0.01161,
    -0.00073,
    0.00257,
    0.00176,
    -0.00098,
    -0.00562,
    -0.00193,
    0.01228,
    -0.00123,
    0.00192,
    0.00056,
    -0.00107,
    -0.00488
   ],
   [
    0.00042,
    -0.00208,
    -0.00096,
    0.00104,
    0.00034,
    -0.00177,
    -0.00068,
    -0.0004,
    -0.00291,
    -0.00095,
    0.00025,
    -0.00049,
    -0.00178,
    -0.00056
   ],
   [
    0.01814,
    0.00241,
    -0.00017,
    0.00466,
    -0.00096,
    0.00163,
    -0.00238,
    0.01615,
    0.00253,
    0.00052,
    0.0057,
    0.00044,
    0.00196,
    -0.00147
   ],
   [
    0.00259,
    0.00019,
    -0.00015,
    -8e-05,
    0.00038,
    0.00057,
    -0.0012,
    0.00167,
    0.0006,
    0.0004,
    0.00031,
    0.00033,
    0.00069,
    -0.00085
   ],
   [
    -0.00488,
    -0.003,
    -0.00201,
    -0.00067,
    -0.00035,
    -0.00165,
    -0.00024,
    -0.00339,
    -0.00314,
    -0.00153,
    -0.00101,
    -0.00075,
    -0.00192,
    -0.00044
   ],
   [
    -0.00288,
    -0.00172,
    -0.00111,
    -0.00024,
    -0.00011,
    -3e-05,
    -7e-05,
    -0.00252,
    -0.0019,
    -0.00106,
    -0.00021,
    -0.00046,
    -0.00078,
    -0.00037
   ],
   [
    0.00695,
    0.00674,
    0.00434,
    0.00212,
    -0.00396,
    0.00313,
    0.00079,
    0.00455,
    0.00565,
    0.00252,
    0.00176,
    -0.00312,
    0.00289,
    0.0005
   ],
   [
    0.0018,
    0.00079,
    0.00204,
    0.00086,
    -0.00019,
    -0.00189,
    -2e-05,
    0.00018,
    -6e-05,
    -1e-05,
    0.00032,
    -0.00048,
    -0.00191,
    -0.00024
   ],
   [
    0.00595,
    0.00912,
    -0.00451,
    0.00073,
    0.01468,
    -0.00262,
    -0.0049,
    0.0059,
    0.00832,
    -0.00404,
    0.00195,
    0.01369,
    4e-05,
    -0.004
   ],
   [
    0.00446,
    0.00078,
    0.00026,
    0.00825,
    -0.00075,
    -0.00252,
    -0.00643,
    0.00647,
    0.00134,
    0.00202,
    0.00775,
    -4e-05,
    0.00017,
    -0.00463
   ],
   [
    0.00037,
    -0.00358,
    -0.00116,
    0.00159,
    -0.00153,
    -0.00127,
    0.00074,
    -0.00054,
    -0.00358,
    -0.00034,
    0.00152,
    -0.00163,
    -0.00109,
    0.00061
   ],
   [
    -0.0075,
    0.00471,
    -3e-05,
    0.00686,
    0.00392,
    -0.00256,
    -0.01365,
    -0.00833,
    0.00229,
    -0.00119,
    0.00516,
    0.00145,
    -0.00434,
    -0.01348
   ],
   [
    -0.00395,
    -0.00106,
    -0.00226,
    -0.00061,
    -0.00212,
    0.0027,
    -0.00343,
    -0.00218,
    -0.00141,
    4e-05,
    -0.00039,
    -0.00246,
    0.00236,
    -0.00302
   ],
   [
    0.0004,
    0.00164,
    0.00028,
    -0.0013,
    0.00024,
    0.0014,
    0.00219,
    -0.00029,
    0.0012,
    -0.00026,
    -0.00086,
    0.00051,
    0.00168,
    0.00266
   ],
   [
    -0.0031,
    0.00137,
    0.00037,
    0.00093,
    -0.00355,
    0.00013,
    0.00292,
    -0.00292,
    0.0011,
    -7e-05,
    0.00098,
    -0.00247,
    -4e-05,
    0.00224
   ],
   [
    0.00682,
    -0.00775,
    -0.00484,
    -0.00984,
    -0.01736,
    0.0205,
    0.02046,
    0.00196,
    -0.00986,
    -0.00303,
    -0.0083,
    -0.01339,
    0.01976,
    0.01823
   ],
   [
    0.00222,
    0.00184,
    -0.00187,
    -0.00117,
    5e-05,
    0.0009,
    0.0013,
    -0.00019,
    -0.0,
    -0.00271,
    -0.00155,
    -0.00067,
    -0.00023,
    0.00046
   ],
   [
    0.00254,
    0.00269,
    0.00366,
    -0.00133,
    0.00248,
    -0.00152,
    -0.0008,
    0.00013,
    0.00263,
    0.0003,
    -0.00183,
    0.00192,
    -0.00208,
    -0.00128
   ],
   [
    0.00483,
    0.00917,
    0.01491,
    0.00654,
    0.00446,
    -0.00549,
    0.00144,
    0.0074,
    0.01502,
    0.01569,
    0.00864,
    0.00679,
    -0.00338,
    0.00273
   ],
   [
    -0.00305,
    -0.0023,
    -0.00015,
    -0.00134,
    0.00605,
    -0.00018,
    -0.00672,
    -0.0024,
    -0.00157,
    -0.00014,
    -0.00132,
    0.00456,
    -0.00044,
    -0.0058
   ],
   [
    -0.00076,
    0.00148,
    -0.00218,
    -0.00082,
    0.00149,
    0.00235,
    0.00159,
    0.00021,
    0.00215,
    -0.00042,
    -0.00085,
    0.00133,
    0.00225,
    0.00158
   ],
   [
    -4e-05,
    -0.00248,
    -0.00026,
    -0.00112,
    -0.00316,
    3e-05,
    0.00373,
    -0.00079,
    -0.00075,
    4e-05,
    -0.00061,
    -0.00247,
    0.00023,
    0.00302
   ],
   [
    -0.00482,
    -0.00758,
    -0.00466,
    -0.00503,
    -0.00188,
    0.00345,
    0.00099,
    -0.0035,
    -0.00717,
    -0.00129,
    -0.00489,
    -0.00309,
    0.00222,
    5e-05
   ],
   [
    0.00475,
    0.00092,
    0.0044,
    0.00175,
    -0.00133,
    -0.00348,
    -0.00104,
    0.00234,
    -0.00028,
    0.00233,
    0.00168,
    -0.00101,
    -0.00285,
    -0.00111
   ],
   [
    0.0011,
    0.00094,
    -0.00066,
    0.00017,
    0.00028,
    0.0003,
    0.00099,
    0.00119,
    0.00071,
    -0.00057,
    -6e-05,
    0.00032,
    0.00072,
    0.00128
   ],
   [
    0.00149,
    0.00281,
    -0.00063,
    0.00346,
    -0.00147,
    -0.00132,
    -0.00346,
    0.00135,
    0.00236,
    -0.00073,
    0.00317,
    -0.00163,
    -0.00151,
    -0.00325
   ],
   [
    -0.01438,
    -0.00834,
    -0.00727,
    -0.00551,
    -0.0004,
    0.01158,
    -0.00849,
    -0.01403,
    -0.00984,
    -0.00735,
    -0.00678,
    -0.00198,
    0.00639,
    -0.00795
   ],
   [
    -0.00025,
    -0.0007,
    0.00203,
    0.00042,
    0.0016,
    -0.00046,
    -0.00316,
    -0.00045,
    -0.0021,
    0.00033,
    0.00092,
    0.00165,
    -0.0003,
    -0.00252
   ],
   [
    0.00104,
    0.00029,
    0.00239,
    0.00183,
    -0.00267,
    0.00102,
    0.00505,
    0.00108,
    0.00023,
    0.00098,
    0.00181,
    -0.00214,
    0.0021,
    0.00462
   ],
   [
    -0.00512,
    -0.01182,
    -0.00093,
    -0.00051,
    -0.00689,
    0.00228,
    0.00655,
    -0.00442,
    -0.00801,
    0.00362,
    0.00053,
    -0.00462,
    0.00356,
    0.00582
   ],
   [
    -0.00036,
    -0.0008,
    0.00065,
    -0.00211,
    0.00059,
    0.00058,
    -0.00247,
    -0.00169,
    -0.00207,
    -5e-05,
    -0.00186,
    0.00012,
    0.00021,
    -0.00219
   ],
   [
    0.00358,
    -0.01293,
    -0.00796,
    0.00659,
    -0.00768,
    -0.00013,
    0.00402,
    -0.00705,
    -0.01427,
    -0.00682,
    0.00513,
    -0.00749,
    -0.00282,
    0.00193
   ],
   [
    -0.00218,
    -0.00144,
    -0.00037,
    -0.0002,
    -0.00044,
    -0.00188,
    -0.0009,
    -0.00217,
    -0.0012,
    0.00044,
    -0.0006,
    -0.00132,
    -0.00239,
    -0.00127
   ],
   [
    0.00292,
    -0.00164,
    -0.00176,
    0.00042,
    0.00025,
    0.00338,
    -7e-05,
    0.00294,
    0.00053,
    5e-05,
    0.00051,
    0.0005,
    0.00344,
    5e-05
   ],
   [
    -0.00968,
    -0.02319,
    -0.00274,
    0.00233,
    -0.00278,
    -0.01054,
    -0.00671,
    -0.01166,
    -0.02509,
    -0.00854,
    -0.00027,
    -0.00593,
    -0.01104,
    -0.00762
   ],
   [
    -0.00305,
    -0.00563,
    0.00133,
    0.00154,
    -0.00096,
    -0.00061,
    0.00178,
    -0.00344,
    -0.00372,
    0.00089,
    0.00131,
    -0.00083,
    -0.00095,
    0.00136
   ],
   [
    0.00523,
    0.00353,
    -0.00051,
    -0.0077,
    -0.00227,
    0.00596,
    -0.00533,
    0.00338,
    -0.00036,
    -0.00018,
    -0.00732,
    -0.00279,
    0.00364,
    -0.00568
   ],
   [
    -0.00128,
    8e-05,
    -0.00277,
    0.00193,
    -0.00336,
    0.00371,
    0.00167,
    -0.00064,
    -0.00184,
    -0.00072,
    0.00223,
    -0.00258,
    0.00351,
    0.00157
   ],
   [
    -0.00668,
    -0.01844,
    -2e-05,
    -0.00061,
    -0.00805,
    0.00595,
    0.00188,
    -0.01115,
    -0.01503,
    -0.00394,
    -0.00066,
    -0.00634,
    0.00462,
    0.00091
   ],
   [
    0.00057,
    0.00316,
    0.00161,
    -0.00012,
    0.00445,
    -0.00199,
    -0.00251,
    0.00043,
    0.00167,
    0.00044,
    -0.00057,
    0.00318,
    -0.00171,
    -0.00204
   ],
   [
    -0.00027,
    0.00193,
    0.0012,
    0.0001,
    -0.00133,
    -0.0005,
    0.0041,
    -0.00182,
    0.00112,
    0.00029,
    -3e-05,
    -0.00067,
    4e-05,
    0.00377
   ],
   [
    0.0015,
    0.00206,
    0.0008,
    -0.00452,
    -0.0016,
    -0.00143,
    0.00041,
    -0.0004,
    -0.00017,
    -0.0007,
    -0.00408,
    -0.00164,
    -0.00137,
    -9e-05
   ],
   [
    0.003,
    -0.00179,
    0.00199,
    0.00109,
    -0.00189,
    -0.00176,
    -0.00131,
    0.00256,
    -0.00107,
    0.00151,
    0.00108,
    -0.00173,
    -0.00162,
    -0.00117
   ],
   [
    0.00118,
    -0.0017,
    0.00052,
    0.0014,
    -0.00053,
    0.00024,
    -0.00031,
    -0.00047,
    -0.00168,
    -0.0003,
    0.00139,
    -0.00035,
    0.00099,
    -4e-05
   ],
   [
    0.00202,
    0.00261,
    0.00158,
    0.00057,
    0.00038,
    0.00144,
    0.00014,
    0.00298,
    0.0035,
    0.00284,
    0.00156,
    0.00088,
    0.00171,
    0.00058
   ],
   [
    -0.00141,
    0.00221,
    -0.00057,
    -0.00038,
    0.00037,
    0.00121,
    0.00041,
    -0.00161,
    0.00141,
    0.00071,
    0.00015,
    0.0006,
    0.00081,
    0.00057
   ],
   [
    0.00994,
    0.01139,
    0.00206,
    0.00029,
    -0.00081,
    0.00019,
    -0.00312,
    0.01187,
    0.00832,
    0.004,
    -0.00047,
    -0.00151,
    -9e-05,
    -0.00262
   ],
   [
    -0.0012,
    0.00713,
    -0.00263,
    -0.003,
    0.00421,
    0.00031,
    -0.00456,
    -0.0014,
    0.00718,
    -0.00114,
    -0.00182,
    0.00448,
    0.00127,
    -0.00345
   ],
   [
    -0.00062,
    -0.00076,
    -0.00063,
    0.00015,
    0.00259,
    -0.00161,
    -0.00377,
    -0.00058,
    -0.00024,
    -0.00083,
    3e-05,
    0.00177,
    -0.00184,
    -0.00332
   ],
   [
    -0.00063,
    0.00348,
    -0.00036,
    -0.00318,
    0.0064,
    0.00362,
    -0.00309,
    -0.00375,
    0.00319,
    -0.00242,
    -0.00402,
    0.00452,
    0.00249,
    -0.00281
   ],
   [
    -0.00254,
    -0.00106,
    -0.00411,
    -0.00812,
    0.00569,
    0.00509,
    -0.00465,
    -0.00848,
    -0.0091,
    -0.0083,
    -0.00797,
    0.00353,
    0.00293,
    -0.00442
   ],
   [
    0.01324,
    -0.00925,
    0.00187,
    0.00499,
    -0.00687,
    0.00244,
    0.00577,
    0.01184,
    -0.00558,
    0.00252,
    0.00528,
    -0.00436,
    0.0044,
    0.00616
   ],
   [
    4e-05,
    0.00101,
    -0.00112,
    0.00153,
    -0.00043,
    -0.0027,
    0.00072,
    -0.00045,
    0.00092,
    -0.00075,
    0.00096,
    -0.00058,
    -0.00258,
    0.00027
   ],
   [
    -0.00458,
    0.01316,
    -0.00166,
    -0.00679,
    0.00714,
    0.00685,
    -0.00161,
    -0.00452,
    0.00949,
    -0.00169,
    -0.00744,
    0.00598,
    0.00448,
    -0.0017
   ],
   [
    0.00079,
    -0.00257,
    -0.00177,
    -3e-05,
    0.00036,
    -0.00019,
    -0.0007,
    0.00161,
    -0.0016,
    -0.00022,
    -0.00034,
    -0.00019,
    -0.0006,
    -0.0007
   ],
   [
    -0.00813,
    -0.00136,
    0.00145,
    0.00769,
    0.00077,
    -0.00062,
    -0.00697,
    -0.00573,
    -4e-05,
    0.00306,
    0.00763,
    0.00085,
    -0.00033,
    -0.00572
   ],
   [
    -0.00352,
    -0.00226,
    -0.00168,
    -0.00527,
    0.00141,
    0.0022,
    -1e-05,
    -0.004,
    -0.00052,
    -0.00197,
    -0.00523,
    0.00117,
    0.00114,
    -0.00049
   ],
   [
    0.00177,
    -0.00107,
    0.0021,
    0.0015,
    -0.00012,
    -0.0001,
    -0.00278,
    0.00067,
    -0.00185,
    0.00102,
    0.0019,
    0.00043,
    9e-05,
    -0.00236
   ],
   [
    0.00488,
    -0.00164,
    -0.0042,
    -0.0037,
    -0.00026,
    0.006,
    0.00164,
    0.00149,
    -0.0033,
    -0.00612,
    -0.00335,
    0.00081,
    0.00508,
    0.00096
   ],
   [
    -0.00548,
    -4e-05,
    0.00228,
    0.00258,
    0.00236,
    -0.00671,
    -0.0012,
    -0.00566,
    -0.00194,
    -0.00114,
    0.00104,
    0.00053,
    -0.00701,
    -0.00225
   ],
   [
    0.00011,
    0.00603,
    0.00012,
    -0.0002,
    0.00399,
    -0.00236,
    -0.00499,
    0.0033,
    0.00525,
    0.00071,
    -0.00101,
    0.00246,
    -0.0027,
    -0.0043
   ],
   [
    -0.00065,
    0.00166,
    0.00054,
    -0.00382,
    -0.00085,
    0.003,
    0.00431,
    -0.00091,
    0.00085,
    0.00082,
    -0.00308,
    -0.00017,
    0.00293,
    0.00349
   ],
   [
    -0.00192,
    0.00378,
    0.00136,
    -0.00396,
    0.00333,
    -0.00077,
    -0.0034,
    -0.00197,
    0.00039,
    -0.00047,
    -0.00387,
    0.00236,
    -0.00162,
    -0.00378
   ],
   [
    -0.00582,
    -0.00908,
    0.00294,
    5e-05,
    -0.00595,
    -4e-05,
    0.00221,
    -0.00444,
    -0.00571,
    0.00277,
    0.00019,
    -0.00487,
    0.00021,
    0.0016
   ],
   [
    -0.00254,
    -0.00351,
    0.00089,
    6e-05,
    -0.00149,
    -0.00016,
    0.00584,
    -0.0016,
    -0.00213,
    8e-05,
    -3e-05,
    -0.00109,
    0.00046,
    0.00506
   ],
   [
    0.00449,
    0.00278,
    0.00088,
    0.00016,
    0.00174,
    -0.00114,
    -0.00011,
    0.00288,
    0.00307,
    -6e-05,
    0.00013,
    0.00156,
    -0.00063,
    -4e-05
   ],
   [
    -0.00393,
    -0.00288,
    -0.00618,
    0.00042,
    0.00051,
    0.00193,
    -0.00332,
    -0.00296,
    -0.00329,
    -0.0044,
    0.00055,
    -8e-05,
    0.00083,
    -0.00295
   ],
   [
    0.01133,
    0.01299,
    0.00084,
    -0.00827,
    0.00248,
    0.00892,
    0.01094,
    0.00725,
    0.01441,
    0.00342,
    -0.00755,
    0.00285,
    0.00908,
    0.01138
   ],
   [
    0.00197,
    0.00194,
    0.0009,
    -0.00118,
    0.00278,
    0.0003,
    -0.00058,
    -0.00081,
    0.00243,
    0.00017,
    -0.00072,
    0.00268,
    0.00041,
    -0.00052
   ],
   [
    -0.00166,
    0.00184,
    -0.00264,
    -6e-05,
    -0.00222,
    -0.00025,
    0.00275,
    -0.00104,
    0.00031,
    -0.00114,
    -0.00079,
    -0.0023,
    -0.00125,
    0.00163
   ],
   [
    -0.01494,
    -0.00763,
    -0.00832,
    -0.00701,
    0.00272,
    0.00445,
    -0.00023,
    -0.01255,
    -0.00631,
    -0.0055,
    -0.00735,
    0.00068,
    0.00063,
    -0.00144
   ],
   [
    0.00322,
    -0.0006,
    -1e-05,
    -0.00033,
    -0.00112,
    -0.00123,
    0.00172,
    0.00234,
    -0.00049,
    0.00011,
    1e-05,
    -0.00076,
    -0.00091,
    0.00151
   ],
   [
    0.01423,
    0.00241,
    0.00323,
    0.00474,
    0.00444,
    -0.00081,
    -0.00062,
    0.00915,
    0.00381,
    0.004,
    0.0057,
    0.00524,
    0.00034,
    -0.00012
   ],
   [
    -0.00339,
    0.0034,
    0.00038,
    0.00023,
    0.00654,
    0.00022,
    -0.00659,
    -0.00316,
    0.0013,
    -0.00056,
    0.00064,
    0.00557,
    0.00083,
    -0.00474
   ],
   [
    -0.0046,
    -0.00375,
    -0.0011,
    -0.00051,
    -0.00255,
    0.00083,
    -0.00206,
    -0.00185,
    -0.00212,
    0.00225,
    0.00024,
    -0.0022,
    0.00088,
    -0.00171
   ],
   [
    0.00117,
    -0.00186,
    0.00022,
    -0.00031,
    -0.00051,
    -0.00113,
    0.00081,
    0.0005,
    -0.00055,
    0.00138,
    0.0003,
    -0.00027,
    -0.00042,
    0.00113
   ],
   [
    -0.0004,
    0.00109,
    0.00343,
    -0.00074,
    0.00403,
    0.00042,
    0.00018,
    -0.00025,
    0.0017,
    0.00225,
    -0.00026,
    0.00376,
    0.00079,
    0.00093
   ],
   [
    -7e-05,
    -0.00065,
    -0.00111,
    0.00149,
    0.00107,
    0.00056,
    -0.00269,
    0.00017,
    -0.00028,
    -0.00081,
    0.00143,
    0.00093,
    0.0006,
    -0.00205
   ],
   [
    0.00534,
    0.00578,
    0.00154,
    0.00118,
    0.00079,
    -0.00336,
    -0.00268,
    0.00549,
    0.00467,
    0.00134,
    0.00016,
    0.00021,
    -0.00432,
    -0.00277
   ],
   [
    -0.00057,
    -0.00074,
    0.00289,
    6e-05,
    6e-05,
    -0.00279,
    -0.00088,
    -0.00112,
    -0.00143,
    0.00125,
    -0.0001,
    -0.00044,
    -0.00269,
    -0.00074
   ],
   [
    -0.00218,
    0.00372,
    -0.00025,
    -0.00018,
    0.00286,
    -0.00052,
    -0.00066,
    -0.00109,
    0.00231,
    -0.00028,
    -0.00034,
    0.00222,
    -0.00011,
    -0.00027
   ],
   [
    0.00284,
    -0.00206,
    -0.00235,
    0.0001,
    -0.00218,
    0.00258,
    0.00318,
    0.00217,
    -0.00151,
    -0.00013,
    0.00072,
    -0.00168,
    0.00277,
    0.00299
   ],
   [
    -9e-05,
    0.00085,
    0.00017,
    0.00066,
    0.00014,
    -0.0013,
    0.00245,
    0.00026,
    0.00131,
    0.00011,
    0.00076,
    0.00029,
    -0.0006,
    0.00197
   ],
   [
    -0.00222,
    -0.00866,
    -0.00453,
    -0.00413,
    0.00092,
    0.00977,
    -0.00387,
    -0.00062,
    -0.00336,
    0.00151,
    -0.00187,
    0.00168,
    0.01081,
    -0.0022
   ],
   [
    0.00152,
    -0.00061,
    -0.00105,
    0.00021,
    -7e-05,
    -0.00029,
    0.00055,
    -0.00071,
    -0.00098,
    -0.00335,
    -0.00038,
    -1e-05,
    -0.00074,
    -0.00011
   ],
   [
    0.00488,
    -0.00232,
    -0.00134,
    0.00043,
    -0.00663,
    0.00472,
    0.00428,
    0.00468,
    6e-05,
    0.00185,
    0.00096,
    -0.00525,
    0.00481,
    0.00416
   ],
   [
    0.00308,
    0.00084,
    0.00127,
    -4e-05,
    -0.00058,
    0.00051,
    -7e-05,
    0.00129,
    0.00013,
    0.0003,
    -0.00011,
    -0.00054,
    0.00053,
    -9e-05
   ],
   [
    0.00129,
    0.0009,
    0.00204,
    0.00028,
    -0.00336,
    0.00373,
    0.00287,
    0.00014,
    0.00264,
    0.00136,
    0.00081,
    -0.00192,
    0.00366,
    0.00297
   ],
   [
    0.00151,
    0.00203,
    0.00161,
    0.00085,
    2e-05,
    0.0001,
    -0.00161,
    0.00092,
    0.00128,
    0.00123,
    0.00126,
    0.00031,
    0.00043,
    -0.00133
   ],
   [
    -8e-05,
    0.00483,
    0.00207,
    0.00084,
    0.00122,
    -0.00542,
    -0.00406,
    0.00096,
    0.00316,
    0.0005,
    0.0004,
    0.00113,
    -0.00476,
    -0.00368
   ],
   [
    0.00354,
    -0.0007,
    0.00132,
    0.0017,
    -0.00235,
    -0.00219,
    -0.00239,
    0.00428,
    -0.00014,
    0.00196,
    0.00109,
    -0.00317,
    -0.00146,
    -0.00206
   ],
   [
    -0.00278,
    -0.0061,
    -0.00105,
    -0.00052,
    -0.00076,
    1e-05,
    -0.00062,
    -0.00274,
    -0.00579,
    -0.00171,
    -0.00037,
    -0.0008,
    -0.00033,
    -0.0007
   ],
   [
    -0.00423,
    0.0027,
    -0.00176,
    -0.00084,
    0.00411,
    0.00166,
    -0.00057,
    -0.00394,
    0.0022,
    -0.00196,
    -0.0008,
    0.00384,
    0.00202,
    -0.00038
   ],
   [
    -0.00575,
    0.00044,
    -0.00763,
    0.01116,
    -0.00598,
    0.00262,
    -0.00854,
    -0.00309,
    -0.00115,
    -0.00795,
    0.00884,
    -0.00688,
    0.00133,
    -0.00909
   ],
   [
    0.01916,
    0.00664,
    0.00115,
    -0.00584,
    0.00345,
    0.00697,
    0.00994,
    0.02071,
    0.00727,
    0.00465,
    -0.00477,
    0.00462,
    0.00685,
    0.00925
   ],
   [
    -0.00036,
    0.00254,
    0.00042,
    -0.00168,
    0.00165,
    0.00076,
    0.00035,
    -0.00028,
    0.00209,
    -0.0013,
    -0.00227,
    0.00179,
    -0.00036,
    0.00018
   ],
   [
    0.00166,
    0.01591,
    0.00672,
    -0.00443,
    0.00966,
    -0.00237,
    0.01041,
    0.00099,
    0.01523,
    0.00554,
    -0.00258,
    0.01133,
    0.00288,
    0.01219
   ],
   [
    0.00467,
    -0.00021,
    -0.0005,
    0.00166,
    -0.001,
    -0.00078,
    -0.00054,
    0.00668,
    0.00058,
    0.00122,
    0.00228,
    -0.00052,
    0.00021,
    -0.00022
   ],
   [
    -0.00412,
    -0.00457,
    -0.00113,
    0.00107,
    0.00036,
    -0.00317,
    0.00016,
    -0.0033,
    -0.00423,
    -0.00093,
    0.0005,
    -0.00038,
    -0.00313,
    -0.00014
   ],
   [
    0.00174,
    -0.00039,
    0.00264,
    0.00243,
    0.00072,
    -0.00093,
    -0.00202,
    0.00274,
    0.00089,
    0.00663,
    0.00413,
    0.00053,
    0.00055,
    -0.00098
   ],
   [
    -0.00541,
    0.00091,
    -0.00245,
    -0.00331,
    0.0009,
    0.00011,
    -0.0012,
    -0.00447,
    0.00018,
    -0.00274,
    -0.00352,
    0.00013,
    -0.00015,
    -0.0013
   ],
   [
    0.01025,
    0.00926,
    0.00134,
    -0.00347,
    3e-05,
    0.00414,
    0.00053,
    0.00747,
    0.00899,
    0.0009,
    -0.00364,
    0.00028,
    0.00353,
    0.00057
   ],
   [
    -0.00504,
    -0.00201,
    -0.00189,
    0.00126,
    0.00072,
    -0.00225,
    -0.00381,
    -0.00388,
    -0.00245,
    -0.00233,
    -3e-05,
    -0.0,
    -0.00292,
    -0.00339
   ],
   [
    0.00203,
    -0.00314,
    0.00397,
    0.00458,
    -0.00145,
    -0.00253,
    0.00341,
    0.00115,
    -0.00244,
    0.0021,
    0.00435,
    -0.0004,
    -0.0017,
    0.0034
   ],
   [
    -0.00079,
    -0.01578,
    0.00461,
    0.00751,
    -0.00587,
    -0.01195,
    0.00376,
    -0.00566,
    -0.01376,
    -0.00124,
    0.00566,
    -0.00648,
    -0.01074,
    0.00102
   ],
   [
    -0.00411,
    -0.00146,
    -0.00288,
    -0.00172,
    0.00205,
    -0.00016,
    -0.00364,
    -0.0039,
    -0.0006,
    -0.00264,
    -0.0019,
    0.00128,
    -0.00065,
    -0.00346
   ],
   [
    -0.00044,
    -0.00113,
    0.00068,
    0.00221,
    -0.00256,
    -0.00104,
    -0.00217,
    -0.00011,
    -0.00152,
    -0.00066,
    0.0018,
    -0.00224,
    -0.00153,
    -0.00227
   ],
   [
    0.02828,
    -0.00819,
    -0.02679,
    0.00142,
    -0.01125,
    -0.01733,
    -0.01613,
    0.02281,
    -0.01085,
    -0.02551,
    -0.00656,
    -0.01584,
    -0.02107,
    -0.01742
   ],
   [
    0.00683,
    6e-05,
    -0.00046,
    0.00083,
    -0.00489,
    0.00456,
    0.00719,
    0.00662,
    0.00098,
    0.00073,
    -0.0005,
    -0.00299,
    0.00214,
    0.00503
   ],
   [
    -0.00077,
    0.0035,
    0.00125,
    -0.00218,
    -0.00126,
    0.00031,
    0.0019,
    -0.00098,
    0.00095,
    0.00071,
    -0.00177,
    -0.0011,
    0.00042,
    0.00163
   ],
   [
    -0.00171,
    0.00285,
    0.00024,
    -0.003,
    0.00152,
    -0.00082,
    0.00085,
    -0.00164,
    0.00061,
    -0.00255,
    -0.00306,
    0.00087,
    -0.00048,
    0.00017
   ],
   [
    0.00114,
    0.00142,
    -0.00227,
    0.00033,
    -0.00149,
    0.00027,
    0.00113,
    0.00194,
    0.00051,
    0.0,
    0.00023,
    -0.00142,
    0.00063,
    0.00112
   ],
   [
    0.00155,
    0.00101,
    -0.00079,
    -0.00034,
    0.00093,
    0.00016,
    -0.00095,
    0.00254,
    0.00285,
    0.001,
    -0.00019,
    0.00111,
    0.00038,
    -0.00072
   ],
   [
    0.00446,
    -0.00286,
    -0.00168,
    0.00052,
    -0.00692,
    0.00093,
    0.0031,
    0.00391,
    -0.00057,
    -0.0009,
    0.00013,
    -0.00609,
    0.00129,
    0.00224
   ],
   [
    0.00498,
    0.00096,
    0.00073,
    -0.00035,
    -0.00092,
    -0.00387,
    -0.0016,
    0.00361,
    -0.00119,
    -0.00104,
    -0.001,
    -0.00157,
    -0.00396,
    -0.00198
   ],
   [
    0.00053,
    0.00066,
    -0.00186,
    -0.00057,
    0.00066,
    0.00069,
    -0.00021,
    0.00101,
    0.00071,
    -0.00037,
    -0.00044,
    0.00046,
    0.00078,
    -3e-05
   ],
   [
    -0.00174,
    0.00116,
    0.00114,
    2e-05,
    0.00264,
    -0.00105,
    -0.00117,
    -0.00105,
    0.00135,
    0.00193,
    0.00035,
    0.00237,
    -0.00069,
    -0.00045
   ],
   [
    -0.00828,
    0.0011,
    0.00449,
    -0.00337,
    0.01421,
    -0.00604,
    0.00198,
    -0.00649,
    0.0022,
    0.00215,
    -0.00149,
    0.01215,
    -0.00422,
    0.00232
   ],
   [
    -0.00159,
    -0.00017,
    -0.0001,
    -0.0029,
    0.00233,
    0.00026,
    -0.00122,
    -0.00159,
    -0.00084,
    2e-05,
    -0.00277,
    0.00157,
    -0.00067,
    -0.00139
   ],
   [
    -0.00335,
    0.0005,
    -0.00161,
    -0.00188,
    -0.00137,
    0.00076,
    0.00034,
    -0.00111,
    0.00098,
    0.00066,
    -0.00183,
    -0.00139,
    0.0007,
    8e-05
   ],
   [
    -0.00082,
    -0.0037,
    0.00333,
    0.00227,
    -0.00477,
    -0.00196,
    0.00155,
    -0.00645,
    -0.00963,
    -0.0029,
    0.00136,
    -0.00442,
    -0.00046,
    0.00093
   ],
   [
    -0.00134,
    -0.00279,
    0.00053,
    -0.00136,
    -0.00169,
    0.00159,
    0.00044,
    -0.00318,
    -0.00327,
    -0.00077,
    -0.00121,
    -0.00153,
    0.00117,
    0.00036
   ],
   [
    -0.00192,
    -0.00315,
    -0.00072,
    0.0012,
    0.00191,
    0.00434,
    -0.0094,
    -0.00547,
    -0.00395,
    -0.00247,
    0.0008,
    0.00135,
    0.00208,
    -0.00834
   ],
   [
    -0.01137,
    -0.00365,
    0.01325,
    -0.00159,
    -0.00823,
    0.0065,
    0.01072,
    -0.00609,
    -0.00227,
    0.0106,
    -0.0004,
    -0.00428,
    0.00766,
    0.00956
   ],
   [
    0.00323,
    -0.00177,
    -0.00045,
    0.00308,
    -0.00078,
    -0.00159,
    -0.00182,
    0.00369,
    -0.00012,
    -5e-05,
    0.00288,
    -0.00021,
    -0.00125,
    -0.0017
   ],
   [
    -0.00325,
    -0.00781,
    -0.00178,
    0.0022,
    -0.00215,
    -0.00467,
    0.0021,
    -0.00337,
    -0.00645,
    -0.00283,
    0.00314,
    -0.00163,
    -0.00334,
    0.00148
   ],
   [
    0.00052,
    -0.00558,
    -0.00421,
    -0.00054,
    -0.00211,
    0.00125,
    -0.00054,
    0.00081,
    -0.0037,
    -0.00202,
    -0.00067,
    -0.00235,
    0.00123,
    -0.00055
   ],
   [
    -0.00735,
    0.00631,
    0.00204,
    -0.0019,
    0.00043,
    -0.0004,
    0.00031,
    -0.00705,
    0.00408,
    -0.00031,
    -0.00213,
    0.00041,
    -0.00073,
    0.0001
   ],
   [
    0.00378,
    0.00182,
    0.002,
    0.00386,
    -0.0019,
    -0.0021,
    0.00446,
    0.00303,
    0.00183,
    0.00085,
    0.00326,
    -0.00088,
    -0.00157,
    0.00384
   ],
   [
    3e-05,
    -0.00042,
    0.00059,
    -0.00131,
    -0.00162,
    0.00099,
    0.00247,
    -0.00034,
    0.00089,
    0.00095,
    -0.001,
    -0.00112,
    0.00083,
    0.00209
   ],
   [
    0.00159,
    0.00165,
    -0.00281,
    -0.00068,
    -0.00113,
    0.00144,
    0.00191,
    0.00194,
    0.00181,
    -0.00131,
    -0.00066,
    -0.00071,
    0.00121,
    0.0018
   ],
   [
    -0.00046,
    0.00015,
    0.0016,
    0.0013,
    0.0034,
    -0.00104,
    -0.00315,
    0.0012,
    0.00019,
    0.00141,
    0.00128,
    0.00314,
    0.00017,
    -0.00204
   ],
   [
    0.00161,
    0.00165,
    0.00102,
    -0.00011,
    0.00039,
    -0.0019,
    -0.00033,
    0.00088,
    0.0013,
    0.00023,
    -7e-05,
    6e-05,
    -0.00179,
    -0.00045
   ],
   [
    -0.00605,
    -0.00722,
    -0.00052,
    -0.00085,
    -0.00325,
    0.00906,
    -0.00082,
    -0.00333,
    -0.00521,
    0.002,
    3e-05,
    -0.00279,
    0.00804,
    5e-05
   ],
   [
    0.01032,
    0.00417,
    0.00866,
    0.00095,
    0.00601,
    -0.00177,
    -0.00089,
    0.01031,
    0.0043,
    0.0083,
    0.00204,
    0.00572,
    0.00058,
    0.00064
   ],
   [
    -0.00236,
    -0.00973,
    -0.00797,
    0.00566,
    -0.00776,
    0.00016,
    -0.00179,
    -0.00189,
    -0.00531,
    -0.00456,
    0.00573,
    -0.00627,
    0.00032,
    -0.00104
   ],
   [
    -0.01128,
    4e-05,
    0.00131,
    8e-05,
    0.00293,
    0.00193,
    0.00042,
    -0.00715,
    0.00067,
    0.00029,
    -0.00021,
    0.00343,
    0.00287,
    0.00065
   ],
   [
    -0.00398,
    0.00027,
    0.00071,
    -0.00041,
    0.00266,
    0.00176,
    -0.00311,
    -0.00016,
    0.00289,
    0.00152,
    -0.00135,
    0.002,
    0.00142,
    -0.00269
   ],
   [
    -0.00202,
    0.00023,
    -0.00127,
    -0.00145,
    0.00233,
    0.00122,
    -0.00237,
    -0.00228,
    -0.00018,
    -0.001,
    -0.0016,
    0.00156,
    0.00018,
    -0.0022
   ],
   [
    -0.00718,
    0.00063,
    -0.00075,
    -0.00075,
    -0.00067,
    -0.00208,
    0.00145,
    -0.00626,
    -0.00024,
    0.00019,
    -0.00175,
    -0.00167,
    -0.0027,
    0.00091
   ],
   [
    -0.01222,
    -0.01776,
    -0.01774,
    0.00145,
    -0.00847,
    0.00556,
    -0.01412,
    -0.00938,
    -0.01989,
    -0.0131,
    0.00026,
    -0.00964,
    0.00214,
    -0.01441
   ],
   [
    -0.00513,
    -0.02841,
    -0.0002,
    0.00843,
    -0.02046,
    -0.00492,
    0.0116,
    -0.00167,
    -0.01996,
    -0.00181,
    0.00579,
    -0.01881,
    -0.00383,
    0.00817
   ],
   [
    0.00357,
    -0.00017,
    0.0008,
    0.0015,
    0.00242,
    -0.00194,
    -0.00263,
    0.00245,
    0.00033,
    0.00032,
    0.00211,
    0.00251,
    -0.00138,
    -0.00205
   ],
   [
    0.00057,
    0.00224,
    -0.00107,
    0.00206,
    -0.00284,
    -0.00191,
    0.00166,
    -0.00037,
    0.00062,
    -0.00076,
    0.00127,
    -0.00282,
    -0.00253,
    0.00095
   ],
   [
    -0.00297,
    -0.00077,
    9e-05,
    0.00173,
    0.00047,
    -0.00169,
    -0.00148,
    -0.00207,
    0.00073,
    0.00049,
    0.00148,
    0.00033,
    -0.00161,
    -0.00146
   ],
   [
    -0.01413,
    -0.0147,
    -0.00598,
    -0.00105,
    -0.00044,
    0.00264,
    -0.00903,
    -0.00973,
    -0.01057,
    -0.00121,
    -0.00092,
    -0.00103,
    0.00199,
    -0.0082
   ],
   [
    0.00012,
    -0.0021,
    -0.001,
    0.00071,
    -0.00096,
    0.00093,
    -0.0005,
    0.00088,
    -0.00064,
    0.00017,
    0.00078,
    -0.00078,
    0.00082,
    -0.00044
   ],
   [
    -0.00174,
    0.00124,
    0.00018,
    -0.00113,
    -0.00111,
    0.00119,
    0.00025,
    -4e-05,
    0.00163,
    0.0008,
    -0.00045,
    -0.00018,
    0.00138,
    0.00025
   ],
   [
    -0.00133,
    0.00126,
    -0.00077,
    -0.00058,
    -0.00041,
    0.00156,
    0.0007,
    7e-05,
    0.00147,
    0.00058,
    -0.00068,
    -0.00035,
    0.00167,
    0.00055
   ],
   [
    0.00093,
    -2e-05,
    -0.00022,
    -0.00095,
    -0.00058,
    0.00013,
    0.00023,
    0.00032,
    -0.00018,
    -0.00032,
    -0.00088,
    -0.00055,
    9e-05,
    7e-05
   ],
   [
    0.00133,
    -0.00178,
    0.00271,
    -0.00022,
    0.00035,
    -0.00141,
    -0.00167,
    -0.00046,
    -0.00142,
    0.00083,
    -0.00054,
    -5e-05,
    -0.00107,
    -0.00143
   ],
   [
    -0.00169,
    -0.00236,
    -0.00246,
    -0.00033,
    0.00254,
    0.00049,
    -0.00355,
    -0.00309,
    -0.00238,
    -0.00205,
    0.00017,
    0.0019,
    0.00033,
    -0.00302
   ],
   [
    -0.0021,
    0.00143,
    -0.00306,
    -0.00085,
    0.00113,
    0.00047,
    -0.00109,
    -0.00128,
    0.00097,
    -0.00116,
    -0.00066,
    0.00076,
    0.00043,
    -0.00096
   ],
   [
    0.00188,
    0.00223,
    0.00218,
    0.00071,
    0.00132,
    0.00093,
    0.00225,
    0.00225,
    0.0029,
    0.00187,
    0.00117,
    0.00185,
    0.00139,
    0.00229
   ],
   [
    -0.00572,
    -0.00035,
    0.0003,
    -0.00111,
    0.00198,
    -0.0005,
    0.0006,
    -0.00352,
    -7e-05,
    0.00057,
    -0.00087,
    0.00194,
    -0.00044,
    4e-05
   ],
   [
    -0.00273,
    -0.00251,
    -0.00052,
    7e-05,
    -0.00109,
    0.00035,
    0.00096,
    -0.00322,
    -0.00212,
    -0.00067,
    -4e-05,
    -0.00111,
    0.00017,
    0.00081
   ],
   [
    0.00591,
    0.00176,
    0.00056,
    -6e-05,
    -0.00062,
    -0.00115,
    -0.00028,
    0.00387,
    0.00087,
    -0.00083,
    -0.00083,
    -0.00101,
    -0.0014,
    -0.00057
   ],
   [
    0.00986,
    -0.01975,
    0.00795,
    0.00713,
    0.00207,
    -0.00698,
    -0.01222,
    0.00515,
    -0.01561,
    0.00092,
    0.0067,
    0.00138,
    -0.00481,
    -0.00932
   ],
   [
    0.00173,
    0.00081,
    0.00137,
    -0.00054,
    0.00052,
    0.00038,
    0.00033,
    0.00106,
    0.0013,
    0.00111,
    -0.00048,
    0.00058,
    0.00092,
    0.00048
   ],
   [
    0.0022,
    -0.00354,
    -0.00121,
    0.00696,
    -0.01291,
    -0.00214,
    0.00964,
    0.00252,
    -0.00215,
    -0.00245,
    0.00487,
    -0.01163,
    -0.00331,
    0.00691
   ],
   [
    -0.0021,
    -0.00199,
    -0.00077,
    0.00443,
    0.00018,
    -0.00101,
    -0.00132,
    -0.0023,
    -0.00149,
    -0.00033,
    0.00451,
    0.00011,
    -0.00125,
    -0.00133
   ],
   [
    0.00146,
    -0.00317,
    0.0003,
    4e-05,
    -0.00224,
    0.00203,
    0.00142,
    0.00012,
    -0.00262,
    0.00049,
    0.00069,
    -0.00131,
    0.00143,
    0.00129
   ],
   [
    -0.00265,
    0.00204,
    0.00329,
    0.00131,
    0.00023,
    0.00206,
    0.00048,
    -0.00146,
    0.00158,
    0.00271,
    0.00178,
    0.00118,
    0.00228,
    0.00055
   ],
   [
    -0.00106,
    -0.00482,
    -0.00588,
    -0.00137,
    0.0023,
    0.00026,
    -0.0015,
    -0.00084,
    -0.00273,
    -0.00414,
    -0.00117,
    0.00143,
    -0.00113,
    -0.00124
   ],
   [
    0.01278,
    -0.00027,
    -0.00194,
    -0.01156,
    -0.00192,
    0.01134,
    0.00708,
    0.00893,
    -0.00079,
    -0.00296,
    -0.01047,
    -0.00159,
    0.00818,
    0.00585
   ],
   [
    0.00532,
    -0.00506,
    -0.00517,
    -0.00495,
    -0.00042,
    0.0077,
    0.00043,
    0.00444,
    -0.00266,
    -0.00209,
    -0.00435,
    -0.00066,
    0.00683,
    0.00036
   ],
   [
    -0.00124,
    -0.00201,
    -0.00158,
    -0.00095,
    0.00071,
    -0.00037,
    -0.00276,
    -0.00129,
    -0.00172,
    -0.00032,
    -0.00128,
    -0.00018,
    -0.00133,
    -0.00258
   ],
   [
    0.00322,
    0.00434,
    -0.00403,
    -0.00139,
    0.00215,
    0.00321,
    -0.0016,
    0.003,
    0.00401,
    -0.00207,
    -0.00115,
    0.00199,
    0.00286,
    -0.00132
   ],
   [
    0.00025,
    -0.00231,
    -0.00524,
    -0.00134,
    -0.00238,
    0.00171,
    0.00507,
    0.0012,
    -0.00307,
    -0.00353,
    -0.00194,
    -0.00253,
    0.00044,
    0.00372
   ],
   [
    -0.0021,
    0.00162,
    -0.0016,
    -0.00344,
    0.00039,
    0.00234,
    0.00234,
    -0.00098,
    0.00311,
    0.00066,
    -0.00257,
    0.00074,
    0.00234,
    0.00205
   ],
   [
    -0.00794,
    -0.00554,
    -0.00379,
    -0.0004,
    -0.00119,
    0.00027,
    -0.00101,
    -0.00775,
    -0.0054,
    -0.00324,
    -0.00046,
    -0.00142,
    0.00046,
    -0.0009
   ],
   [
    -0.00196,
    0.00578,
    -0.00205,
    0.00727,
    -0.00155,
    0.00219,
    0.00071,
    -0.00377,
    0.00235,
    -0.00444,
    0.00699,
    -0.00074,
    0.00322,
    0.00172
   ],
   [
    0.00356,
    0.002,
    0.00309,
    0.00109,
    -0.00099,
    -0.00128,
    0.00128,
    0.00173,
    0.00253,
    0.00228,
    0.00146,
    -0.0004,
    -0.00031,
    0.00118
   ],
   [
    -0.00238,
    -0.00212,
    -0.00026,
    -0.00313,
    0.00357,
    0.00078,
    -0.00256,
    -0.00349,
    -0.00222,
    -0.00128,
    -0.00274,
    0.00281,
    8e-05,
    -0.0026
   ],
   [
    -0.00089,
    -0.00438,
    0.00085,
    -0.0007,
    9e-05,
    0.00026,
    -0.00058,
    -0.00076,
    -0.00304,
    0.0,
    -0.00044,
    0.00017,
    0.00012,
    -0.00036
   ],
   [
    -0.00212,
    0.00113,
    -0.00201,
    -0.00217,
    -0.00074,
    -0.00058,
    0.00057,
    -0.00196,
    -0.00075,
    -0.00142,
    -0.00193,
    -0.00075,
    -0.00067,
    0.00021
   ],
   [
    0.00549,
    -0.00066,
    -0.00755,
    -0.00607,
    -0.0005,
    0.00651,
    -0.00202,
    0.00592,
    -0.00221,
    -0.00433,
    -0.00621,
    -0.00182,
    0.00553,
    -0.00136
   ],
   [
    0.00708,
    0.00825,
    -0.00016,
    -0.00946,
    0.01028,
    -0.00422,
    0.00095,
    0.00131,
    0.00593,
    -0.0049,
    -0.00927,
    0.00792,
    -0.00589,
    0.0009
   ],
   [
    0.00457,
    -0.00228,
    0.00237,
    -7e-05,
    -0.00102,
    -0.00148,
    0.00129,
    0.00193,
    -0.00244,
    0.00132,
    -6e-05,
    -0.00122,
    -0.00172,
    0.00112
   ],
   [
    0.00016,
    -0.00183,
    0.00193,
    -0.00067,
    0.0006,
    -0.0007,
    -0.00175,
    0.00082,
    -0.00205,
    0.00092,
    -0.00078,
    0.0004,
    -0.00103,
    -0.00178
   ],
   [
    -0.0006,
    0.0001,
    -0.00435,
    0.00201,
    -0.00267,
    0.00075,
    0.00209,
    0.00318,
    0.00047,
    0.00044,
    0.00274,
    -0.00222,
    0.0019,
    0.00246
   ],
   [
    0.00218,
    -0.00487,
    -0.00073,
    -0.0013,
    -0.00508,
    0.00288,
    0.00344,
    0.00187,
    -0.00474,
    0.00106,
    -0.00131,
    -0.0048,
    0.00205,
    0.0029
   ],
   [
    0.00161,
    0.00431,
    -0.0006,
    2e-05,
    0.00188,
    -0.00203,
    -0.00023,
    0.00093,
    0.003,
    -0.0013,
    -2e-05,
    0.00168,
    -0.00186,
    -0.00049
   ],
   [
    0.00349,
    -0.00103,
    0.0026,
    0.00369,
    -0.00126,
    0.00042,
    0.00022,
    0.00337,
    0.00018,
    0.00224,
    0.00343,
    -0.00109,
    -5e-05,
    0.00045
   ],
   [
    0.00559,
    0.00684,
    0.0041,
    -0.00152,
    0.00447,
    -0.00054,
    -0.00015,
    0.00056,
    0.00201,
    -0.00121,
    -0.00232,
    0.00316,
    -0.00231,
    -0.00087
   ],
   [
    -0.00058,
    -0.002,
    0.0009,
    -0.0006,
    -0.00053,
    -0.00065,
    0.00094,
    -0.0024,
    -0.00359,
    -0.0001,
    -0.0004,
    -0.00083,
    -0.00079,
    0.00074
   ],
   [
    0.00027,
    0.00197,
    -0.00214,
    -0.00373,
    -0.00504,
    0.00512,
    0.00403,
    0.00395,
    0.00233,
    9e-05,
    -0.00339,
    -0.00411,
    0.00362,
    0.00348
   ],
   [
    -0.00181,
    0.00085,
    0.00025,
    0.00144,
    0.0004,
    -0.00162,
    -0.00073,
    -0.0009,
    0.00037,
    5e-05,
    0.00126,
    0.00032,
    -0.00146,
    -0.00092
   ],
   [
    -0.00214,
    -0.00223,
    -0.00209,
    0.00015,
    -0.00113,
    0.0012,
    0.00182,
    -0.0014,
    -0.00222,
    -0.00179,
    0.00028,
    -0.00085,
    0.00127,
    0.00177
   ],
   [
    0.00028,
    0.00356,
    0.00184,
    0.00112,
    -0.00079,
    -0.00046,
    -0.0008,
    0.00215,
    0.00307,
    0.00271,
    0.00049,
    -0.00043,
    -0.00052,
    -0.00051
   ],
   [
    0.00579,
    0.00326,
    0.00175,
    -0.0008,
    0.00142,
    -0.0007,
    -0.00012,
    0.00419,
    0.00352,
    0.00106,
    -0.00098,
    0.00099,
    -0.00053,
    4e-05
   ],
   [
    0.00269,
    0.00373,
    0.00208,
    -2e-05,
    -0.00095,
    0.00127,
    0.00035,
    0.00305,
    0.00347,
    0.00167,
    0.0005,
    1e-05,
    0.00137,
    0.00057
   ],
   [
    0.00197,
    -0.00235,
    0.00126,
    0.00292,
    -0.0011,
    -0.00207,
    -0.00025,
    0.00059,
    -0.00167,
    -0.00124,
    0.00157,
    -0.00132,
    -0.0018,
    -0.00041
   ],
   [
    -0.00032,
    0.00105,
    -0.00088,
    0.00109,
    -0.00102,
    0.00071,
    0.00113,
    0.00139,
    -0.00017,
    -0.00043,
    0.0011,
    -0.00049,
    0.00057,
    0.00085
   ],
   [
    -0.00143,
    -0.00292,
    0.00061,
    -5e-05,
    0.00092,
    0.00114,
    -0.00109,
    -0.00286,
    -0.00309,
    -0.00055,
    -0.0002,
    0.00056,
    0.0002,
    -0.00103
   ],
   [
    0.00013,
    0.00058,
    -0.00205,
    -0.00146,
    0.00209,
    0.00029,
    0.00183,
    -0.00051,
    0.0017,
    -0.00167,
    -0.00134,
    0.00196,
    0.0007,
    0.00176
   ],
   [
    0.00364,
    0.00146,
    0.00109,
    3e-05,
    0.00066,
    0.00107,
    -0.00181,
    0.00296,
    0.00157,
    0.00039,
    0.00026,
    0.00096,
    0.00148,
    -0.00123
   ],
   [
    -0.00174,
    -0.00369,
    0.00258,
    0.00192,
    0.00251,
    -0.00079,
    -0.0049,
    -0.00257,
    -0.00418,
    -0.00048,
    0.00041,
    0.00076,
    -0.00113,
    -0.00415
   ],
   [
    0.00712,
    0.01337,
    -0.00437,
    -0.00544,
    0.00154,
    0.01495,
    0.00202,
    0.0084,
    0.01643,
    -0.00058,
    -0.00676,
    0.00138,
    0.01274,
    0.00222
   ],
   [
    0.01095,
    -0.00053,
    0.00418,
    0.00837,
    -0.00606,
    -0.00587,
    -0.00566,
    0.01719,
    0.00367,
    0.00813,
    0.00695,
    -0.00609,
    -0.00774,
    -0.00558
   ],
   [
    0.00018,
    -0.00087,
    0.00082,
    -0.00245,
    0.00451,
    -0.00066,
    -0.00494,
    -0.00136,
    -0.00134,
    -0.00028,
    -0.0021,
    0.0035,
    -0.00069,
    -0.00412
   ],
   [
    -0.00244,
    0.00072,
    0.00038,
    -0.0003,
    0.00046,
    -0.00084,
    -0.00021,
    -0.00274,
    0.00074,
    -0.00054,
    -0.00028,
    0.00027,
    -0.00144,
    -0.00049
   ],
   [
    -0.00803,
    -0.00782,
    -0.00069,
    0.00078,
    0.00376,
    -0.00083,
    -0.00268,
    -0.00592,
    -0.0086,
    -0.00065,
    0.00123,
    0.00241,
    0.00015,
    -0.00173
   ],
   [
    0.00146,
    0.00206,
    -0.00062,
    -6e-05,
    0.00124,
    -0.00146,
    0.00019,
    0.00174,
    0.00178,
    8e-05,
    0.00032,
    0.00132,
    -0.00081,
    0.00037
   ],
   [
    0.00096,
    8e-05,
    0.00352,
    -0.00158,
    -0.00095,
    0.00598,
    0.00258,
    0.00049,
    -0.00022,
    0.00365,
    -0.00156,
    -0.00101,
    0.00523,
    0.00273
   ],
   [
    -0.00118,
    0.00231,
    -0.00162,
    0.00017,
    0.00225,
    0.00088,
    3e-05,
    -5e-05,
    0.00253,
    -0.0,
    -3e-05,
    0.00181,
    0.00128,
    0.0003
   ],
   [
    -0.00107,
    0.00097,
    0.00059,
    -0.00434,
    -0.00392,
    0.00473,
    0.00619,
    -0.00392,
    5e-05,
    0.0004,
    -0.00375,
    -0.00344,
    0.00492,
    0.00553
   ],
   [
    0.01273,
    0.00714,
    0.00741,
    -0.00086,
    -0.00978,
    0.00241,
    0.00582,
    0.01195,
    0.00669,
    0.0063,
    -0.0017,
    -0.00771,
    0.00131,
    0.00528
   ],
   [
    0.00129,
    0.00152,
    0.00199,
    0.00066,
    0.00106,
    -0.00074,
    -0.00188,
    0.00043,
    0.00247,
    0.00171,
    0.00062,
    0.00096,
    -0.00067,
    -0.00154
   ],
   [
    0.01963,
    0.00503,
    0.0153,
    0.00818,
    -0.021,
    0.00759,
    0.016,
    0.02218,
    0.00619,
    0.015,
    0.00863,
    -0.01409,
    0.00846,
    0.01519
   ],
   [
    -0.00111,
    -0.00159,
    -0.0007,
    0.00078,
    0.00023,
    0.00021,
    -0.00327,
    -0.00125,
    -0.00087,
    -0.00084,
    0.0006,
    -5e-05,
    -3e-05,
    -0.00284
   ],
   [
    0.00314,
    0.00098,
    0.00033,
    0.00028,
    2e-05,
    3e-05,
    -0.00012,
    0.00372,
    0.00066,
    0.00027,
    0.00029,
    0.00031,
    0.00016,
    -1e-05
   ],
   [
    -0.002,
    -0.00122,
    0.00073,
    0.00115,
    0.00082,
    -0.00196,
    -0.00052,
    -0.00204,
    -0.00043,
    -0.00024,
    0.00116,
    0.00058,
    -0.00143,
    -0.00053
   ],
   [
    -0.01955,
    -0.00862,
    -0.01126,
    -0.00347,
    0.00032,
    0.00994,
    -0.00566,
    -0.01651,
    -0.00666,
    -0.00391,
    -0.0032,
    0.00018,
    0.00912,
    -0.00512
   ],
   [
    0.00139,
    -0.00021,
    -0.00107,
    0.00112,
    0.00144,
    -0.00038,
    0.00106,
    0.00114,
    0.00081,
    -0.00059,
    0.00155,
    0.00176,
    -1e-05,
    0.0012
   ],
   [
    -0.00103,
    0.00415,
    0.00138,
    -6e-05,
    -0.0001,
    0.00325,
    0.00324,
    0.00091,
    0.00444,
    0.00142,
    0.00023,
    0.00062,
    0.0032,
    0.0035
   ],
   [
    0.00014,
    0.00026,
    -0.00103,
    -0.00027,
    -0.002,
    0.00125,
    0.0013,
    0.00026,
    2e-05,
    -0.00026,
    -0.00026,
    -0.00172,
    0.00086,
    0.00103
   ],
   [
    0.00964,
    0.007,
    0.01438,
    -0.00319,
    -0.01438,
    -0.00161,
    0.00865,
    0.01304,
    0.00713,
    0.00999,
    -0.00424,
    -0.01112,
    -0.00139,
    0.00715
   ],
   [
    0.01189,
    6e-05,
    0.00537,
    0.00322,
    -0.00067,
    0.0023,
    -0.00429,
    0.00895,
    -0.00016,
    -6e-05,
    0.00227,
    -9e-05,
    0.00181,
    -0.00408
   ],
   [
    -0.04399,
    0.01692,
    0.01161,
    -0.00787,
    0.01257,
    0.01492,
    0.02649,
    -0.03281,
    0.01438,
    0.01099,
    -0.00169,
    0.0149,
    0.01767,
    0.0267
   ],
   [
    0.00694,
    0.00392,
    0.00191,
    -0.00069,
    0.00293,
    0.00037,
    0.00145,
    0.0043,
    0.00324,
    -0.00017,
    -0.00083,
    0.00311,
    0.00038,
    0.00109
   ],
   [
    0.00337,
    0.0033,
    2e-05,
    -0.00128,
    0.00232,
    0.00075,
    -0.00397,
    0.00142,
    0.00286,
    -0.00119,
    -0.00162,
    0.00188,
    0.00056,
    -0.00345
   ],
   [
    -0.00193,
    -0.00078,
    0.00394,
    0.00093,
    -0.00453,
    0.00321,
    0.01114,
    -0.00507,
    -0.00287,
    0.00161,
    0.00018,
    -0.00388,
    0.00241,
    0.00931
   ],
   [
    0.0017,
    0.00153,
    0.00031,
    0.00029,
    -0.00112,
    0.00044,
    -0.00097,
    0.00195,
    0.00145,
    0.00131,
    0.00011,
    -0.00071,
    0.00024,
    -0.00054
   ],
   [
    0.00149,
    0.00162,
    0.00463,
    0.00115,
    0.00154,
    -0.00265,
    0.00269,
    0.00106,
    -0.00027,
    0.00271,
    0.0019,
    0.00257,
    -0.00113,
    0.00282
   ],
   [
    0.00103,
    -0.00939,
    -0.00119,
    0.00292,
    -0.00519,
    -0.00043,
    0.00303,
    -0.00109,
    -0.01049,
    -0.00141,
    0.00316,
    -0.00469,
    0.00037,
    0.00262
   ],
   [
    0.00223,
    0.00046,
    -0.00274,
    -0.00063,
    0.00363,
    -0.00297,
    -0.00345,
    0.00359,
    0.00063,
    -0.00106,
    -7e-05,
    0.00306,
    -0.00244,
    -0.00294
   ],
   [
    -0.00262,
    0.00033,
    -0.00061,
    -0.00217,
    0.00192,
    0.00294,
    0.0004,
    -0.00374,
    -0.00017,
    -0.00092,
    -0.00217,
    0.00132,
    0.00257,
    0.00057
   ],
   [
    0.00399,
    0.00306,
    0.00123,
    0.00128,
    0.00109,
    0.00041,
    -0.0018,
    0.00592,
    0.00395,
    0.00311,
    0.00116,
    0.00091,
    0.00102,
    -0.00122
   ],
   [
    0.0046,
    -0.00732,
    0.00164,
    0.00863,
    -0.00292,
    -0.00027,
    -0.00188,
    0.00196,
    -0.00329,
    0.00302,
    0.00966,
    -0.00145,
    0.00159,
    -0.00113
   ],
   [
    0.00171,
    -0.00022,
    0.00056,
    -0.00191,
    0.00141,
    0.00265,
    -0.0018,
    -0.00054,
    -0.00164,
    -0.00273,
    -0.00199,
    0.00085,
    0.00225,
    -0.00149
   ],
   [
    0.00579,
    -0.0029,
    -0.00328,
    -0.00482,
    -0.00327,
    0.00721,
    -0.00934,
    0.00076,
    -0.00406,
    -0.00954,
    -0.0064,
    -0.00422,
    0.00491,
    -0.00842
   ],
   [
    -0.00398,
    -0.00045,
    0.00249,
    -0.0047,
    -0.00084,
    0.00246,
    0.00444,
    -0.00271,
    0.00044,
    0.00198,
    -0.00438,
    -6e-05,
    0.00286,
    0.00405
   ],
   [
    -0.00224,
    -0.00039,
    -0.00123,
    -0.00314,
    0.00212,
    -0.00023,
    -0.00077,
    -0.00166,
    -0.00034,
    -0.0009,
    -0.00313,
    0.00142,
    -0.00102,
    -0.00103
   ],
   [
    0.00164,
    0.00238,
    0.00114,
    0.00205,
    0.00138,
    -0.0011,
    5e-05,
    0.00014,
    0.00225,
    0.00074,
    0.00215,
    0.0014,
    -0.00036,
    0.00024
   ],
   [
    0.00041,
    -0.00067,
    -0.00109,
    0.00118,
    -0.00191,
    -0.00164,
    0.0019,
    0.00079,
    -0.00018,
    -0.00055,
    0.00105,
    -0.00148,
    -0.00125,
    0.00136
   ],
   [
    -0.00072,
    0.00071,
    0.00056,
    -0.00081,
    0.00065,
    0.005,
    -0.00072,
    0.00264,
    0.0021,
    0.00264,
    1e-05,
    0.00123,
    0.00508,
    -0.00033
   ],
   [
    -0.00273,
    -0.00355,
    -0.00087,
    -0.00244,
    6e-05,
    0.00294,
    0.00072,
    -0.00258,
    -0.00266,
    -0.00061,
    -0.00207,
    0.0002,
    0.00256,
    0.00086
   ],
   [
    0.00702,
    9e-05,
    0.00386,
    -0.00246,
    -0.00066,
    0.00257,
    -0.00642,
    0.00756,
    0.00089,
    0.00327,
    -0.00318,
    -0.00177,
    -0.00063,
    -0.0064
   ],
   [
    0.00081,
    0.00164,
    0.00324,
    0.00075,
    -0.00126,
    0.00148,
    0.0041,
    1e-05,
    0.0004,
    0.00239,
    0.00192,
    0.00019,
    0.00202,
    0.00397
   ],
   [
    0.00114,
    0.00046,
    0.00242,
    0.00106,
    0.00068,
    -0.00076,
    -0.00132,
    0.00132,
    0.0013,
    0.00259,
    0.00213,
    0.0015,
    0.00022,
    -0.00054
   ],
   [
    -0.0006,
    0.00081,
    -0.0006,
    0.00102,
    0.0016,
    0.00134,
    0.0011,
    0.0011,
    0.00058,
    -2e-05,
    0.00111,
    0.00174,
    0.00152,
    0.00108
   ],
   [
    0.00069,
    -0.00424,
    0.00247,
    0.00254,
    -0.00258,
    -0.00166,
    -0.00528,
    -0.00129,
    -0.00485,
    0.00215,
    0.00194,
    -0.00321,
    -0.00212,
    -0.00477
   ],
   [
    0.00549,
    0.00325,
    0.00173,
    0.00375,
    -0.00563,
    -0.00634,
    0.00012,
    0.00809,
    0.00564,
    0.00432,
    0.00275,
    -0.00509,
    -0.00596,
    -0.00025
   ],
   [
    0.00384,
    0.00144,
    -0.00371,
    0.001,
    -0.00271,
    0.00418,
    0.00293,
    0.00291,
    -0.00046,
    -0.00206,
    0.00164,
    -0.00174,
    0.00503,
    0.0033
   ],
   [
    -0.00371,
    0.00134,
    0.00205,
    -0.00062,
    -0.00579,
    0.00579,
    0.00296,
    -0.00382,
    -0.00111,
    0.00104,
    -0.00036,
    -0.00412,
    0.00528,
    0.00251
   ],
   [
    0.00179,
    0.00222,
    -0.00219,
    -0.0027,
    0.00401,
    -0.00014,
    -0.00524,
    0.00255,
    0.0013,
    -0.00261,
    -0.00308,
    0.00275,
    -0.00035,
    -0.00488
   ],
   [
    0.00291,
    0.00295,
    0.00259,
    -0.0033,
    0.00784,
    0.00012,
    -0.00689,
    0.0018,
    0.00149,
    0.00079,
    -0.00433,
    0.00608,
    0.00065,
    -0.00498
   ],
   [
    0.00142,
    -0.0028,
    -0.00041,
    -0.00151,
    0.00179,
    -0.00088,
    -0.00132,
    -0.00113,
    -0.00186,
    -0.00217,
    -0.00179,
    0.00116,
    -0.00205,
    -0.00156
   ],
   [
    -0.00034,
    0.0056,
    -0.00282,
    0.00301,
    0.00215,
    0.00212,
    0.00157,
    0.00131,
    0.00564,
    -0.00051,
    0.0028,
    0.003,
    0.00353,
    0.00217
   ],
   [
    0.00085,
    0.00267,
    -0.00199,
    -0.00124,
    -0.00013,
    -0.00067,
    -0.00062,
    0.00039,
    1e-05,
    -0.00318,
    -0.00173,
    -0.00081,
    -0.00095,
    -0.00089
   ],
   [
    -0.0007,
    -0.00072,
    -0.00388,
    -0.00107,
    0.00136,
    0.00129,
    -0.00107,
    0.0005,
    -0.00112,
    -0.00228,
    -0.00065,
    0.0009,
    0.00072,
    -0.00086
   ],
   [
    -0.00333,
    -0.02441,
    -0.01335,
    -0.00842,
    -0.01769,
    -0.00012,
    0.00352,
    -0.00239,
    -0.02605,
    -0.01243,
    -0.00936,
    -0.01858,
    -0.0028,
    0.00101
   ],
   [
    -0.00279,
    -0.00201,
    0.00452,
    0.00137,
    -0.00405,
    -0.00293,
    -0.00118,
    0.00309,
    0.0024,
    0.00416,
    5e-05,
    -0.00356,
    -0.0025,
    -0.00171
   ],
   [
    0.00167,
    0.00153,
    0.00256,
    0.00236,
    0.00089,
    0.00021,
    0.0034,
    0.00118,
    0.00136,
    0.0005,
    0.00271,
    0.00179,
    0.00123,
    0.0034
   ],
   [
    -0.00021,
    -0.00365,
    -0.0007,
    -0.00013,
    -0.00122,
    0.00026,
    -4e-05,
    -0.00073,
    -0.00246,
    0.00079,
    0.00047,
    -0.00112,
    0.00059,
    0.00021
   ],
   [
    -0.00045,
    -5e-05,
    0.001,
    0.0002,
    -0.00035,
    -0.00161,
    -0.00016,
    -0.00215,
    -0.00064,
    -0.00054,
    0.00067,
    -9e-05,
    -0.00157,
    -0.0001
   ],
   [
    -0.00013,
    -0.00154,
    -0.00187,
    -0.00277,
    -0.00057,
    -0.00087,
    -0.00071,
    -0.00125,
    -0.00257,
    -0.00274,
    -0.00258,
    -0.00076,
    -0.0012,
    -0.00097
   ],
   [
    0.00181,
    0.00312,
    0.00644,
    -0.00549,
    -0.00154,
    0.00541,
    0.00903,
    -0.00253,
    0.0014,
    0.00332,
    -0.00362,
    -0.00078,
    0.00612,
    0.00925
   ],
   [
    -0.00538,
    -0.00184,
    -0.00104,
    -0.00037,
    0.00017,
    -0.00043,
    0.00077,
    -0.00419,
    -0.00257,
    -5e-05,
    0.00013,
    0.00011,
    -0.0005,
    0.00063
   ],
   [
    0.00358,
    0.00254,
    -0.00044,
    -0.00221,
    -0.00028,
    0.00139,
    0.00178,
    0.00368,
    0.00211,
    0.00151,
    -0.00169,
    -9e-05,
    0.00122,
    0.00153
   ],
   [
    -0.00296,
    -0.00179,
    -0.00148,
    -0.00042,
    -0.00115,
    -0.00141,
    0.00097,
    -0.00377,
    -0.00233,
    -0.00098,
    7e-05,
    -0.00113,
    -0.00109,
    0.00109
   ],
   [
    -0.00118,
    -0.00091,
    -0.00016,
    -0.00054,
    -0.00147,
    -0.00139,
    -0.00341,
    -0.00023,
    -0.00015,
    0.00015,
    -0.00088,
    -0.00179,
    -0.00155,
    -0.00313
   ],
   [
    -0.00302,
    -0.00345,
    -0.00019,
    0.00098,
    -2e-05,
    -0.00129,
    -0.00064,
    -0.00286,
    -0.00318,
    -0.00134,
    0.00112,
    -0.00032,
    -0.00134,
    -0.00048
   ],
   [
    0.00168,
    -0.00611,
    -0.00265,
    0.00498,
    -0.00496,
    -0.00031,
    0.00104,
    0.00589,
    -0.0028,
    0.00138,
    0.00534,
    -0.00348,
    0.00098,
    0.00161
   ],
   [
    -0.00464,
    -0.01245,
    -0.00776,
    -0.00053,
    0.00585,
    -0.00603,
    -0.00336,
    -0.00804,
    -0.01237,
    -0.00651,
    -0.00049,
    0.00259,
    -0.00566,
    -0.00383
   ],
   [
    -0.0016,
    0.00139,
    0.00304,
    0.00141,
    -0.00036,
    -0.00193,
    0.00033,
    -0.00099,
    0.00181,
    0.00204,
    0.00167,
    0.00041,
    -0.00167,
    4e-05
   ],
   [
    0.0026,
    -0.00215,
    0.00145,
    -0.00186,
    -0.00122,
    -0.00099,
    -0.00114,
    0.00013,
    -0.00297,
    -0.0013,
    -0.00181,
    -0.00116,
    -0.0015,
    -0.00137
   ],
   [
    0.00083,
    0.00051,
    -0.00167,
    0.0005,
    0.00051,
    0.00151,
    -0.00162,
    0.00113,
    0.00186,
    -0.00077,
    0.00043,
    0.00057,
    0.00176,
    -0.00119
   ],
   [
    -0.01731,
    -0.00526,
    -0.00849,
    -0.00648,
    0.0065,
    0.00129,
    -0.00981,
    -0.01444,
    -0.00739,
    -0.00689,
    -0.00727,
    0.00305,
    0.0006,
    -0.00876
   ],
   [
    -0.00069,
    0.00083,
    -0.00262,
    -0.00068,
    -0.0008,
    0.00124,
    -0.00045,
    -0.00045,
    0.00108,
    -0.0011,
    -0.00044,
    -0.00073,
    0.00107,
    -0.00028
   ],
   [
    -0.00179,
    -0.00288,
    0.00158,
    -0.00243,
    -0.00516,
    0.00294,
    0.0025,
    -0.00259,
    -0.00313,
    0.00141,
    -0.00245,
    -0.00443,
    0.00211,
    0.00149
   ],
   [
    -0.00959,
    0.00719,
    -0.00441,
    -0.00144,
    0.00924,
    -0.00822,
    -0.00371,
    -0.00952,
    0.00095,
    -0.0102,
    -0.00299,
    0.00516,
    -0.00984,
    -0.00491
   ],
   [
    0.00608,
    0.02751,
    -0.01175,
    -0.01634,
    0.01069,
    0.00525,
    -0.00446,
    0.01008,
    0.02686,
    -0.00349,
    -0.01615,
    0.00991,
    0.00257,
    -0.00351
   ],
   [
    0.00844,
    -0.00239,
    0.00298,
    0.00229,
    -0.00816,
    0.00256,
    0.0039,
    0.00487,
    -0.00488,
    0.00133,
    0.00138,
    -0.00719,
    0.00168,
    0.00321
   ],
   [
    -0.00013,
    0.00157,
    0.00227,
    0.00021,
    -0.0002,
    0.00107,
    -0.00051,
    -0.00056,
    0.00079,
    0.00149,
    -0.00017,
    -0.00027,
    0.00075,
    -0.0004
   ],
   [
    -0.00104,
    0.00024,
    0.00036,
    0.00031,
    0.00213,
    0.00058,
    -0.00247,
    -0.00089,
    -0.00067,
    0.00025,
    0.00036,
    0.0017,
    0.0008,
    -0.00168
   ],
   [
    0.0273,
    0.02059,
    0.01579,
    -0.00519,
    0.01066,
    -0.00058,
    -0.01199,
    0.02422,
    0.02048,
    0.01154,
    -0.00377,
    0.01008,
    -0.00135,
    -0.01011
   ],
   [
    -0.00224,
    0.00015,
    -0.0024,
    -0.00078,
    0.00298,
    -0.00041,
    -0.00333,
    -0.0021,
    0.00078,
    -0.002,
    -0.0009,
    0.00232,
    -0.0007,
    -0.00258
   ],
   [
    0.00875,
    -0.01294,
    0.00483,
    0.0186,
    -0.00554,
    -0.00589,
    -0.01695,
    0.00706,
    -0.00811,
    0.0059,
    0.01766,
    -0.00478,
    -0.00534,
    -0.01409
   ],
   [
    0.00888,
    -0.00164,
    0.00035,
    0.00753,
    -0.0015,
    -0.00596,
    -0.00019,
    0.00749,
    0.00049,
    0.00055,
    0.00696,
    -0.00169,
    -0.00568,
    -0.0006
   ],
   [
    0.00032,
    0.00093,
    0.00152,
    0.00097,
    0.00083,
    -0.00277,
    -0.00316,
    0.00241,
    0.00393,
    0.00134,
    0.00074,
    0.0011,
    -0.00223,
    -0.00266
   ],
   [
    0.00142,
    -0.0019,
    0.00145,
    0.00056,
    0.00034,
    0.00164,
    -0.00331,
    0.00016,
    -0.00147,
    0.00085,
    0.00098,
    -2e-05,
    0.00144,
    -0.00252
   ],
   [
    0.00169,
    0.00368,
    -0.00154,
    0.00028,
    0.00321,
    -0.00199,
    -0.00144,
    0.00172,
    0.00256,
    -0.00153,
    -0.00032,
    0.00221,
    -0.00172,
    -0.00157
   ],
   [
    0.01204,
    0.00936,
    0.01078,
    -0.00275,
    -0.00702,
    0.00017,
    0.00219,
    0.01488,
    0.00887,
    0.00907,
    -0.00366,
    -0.00599,
    -0.00146,
    0.00088
   ],
   [
    0.00121,
    -0.00056,
    0.00028,
    0.00329,
    -3e-05,
    -0.00414,
    -0.00481,
    0.00121,
    0.00031,
    -0.00017,
    0.00248,
    0.0,
    -0.00401,
    -0.00461
   ],
   [
    0.00085,
    0.00045,
    -0.00048,
    -0.00092,
    -0.00044,
    0.00059,
    0.00119,
    -0.00106,
    0.00016,
    -0.00176,
    -0.00055,
    -8e-05,
    0.00027,
    0.00089
   ],
   [
    0.00065,
    -0.00261,
    -0.00016,
    0.00156,
    -0.0017,
    -0.00157,
    -8e-05,
    0.00081,
    -0.00154,
    0.00044,
    0.00163,
    -0.00148,
    -0.00119,
    -1e-05
   ],
   [
    0.00523,
    0.01048,
    0.00455,
    -0.00154,
    -0.0,
    -0.00058,
    -0.00026,
    0.00618,
    0.00921,
    0.00451,
    -0.00209,
    -0.00015,
    -0.00122,
    -0.00105
   ],
   [
    -0.00183,
    -0.00042,
    0.00099,
    -0.00219,
    -0.00042,
    -0.00032,
    0.00012,
    -0.00258,
    -0.00132,
    -0.00057,
    -0.00239,
    -0.00072,
    -0.00089,
    -0.00016
   ],
   [
    0.00619,
    -0.00011,
    0.00174,
    0.00204,
    -0.00379,
    0.0019,
    0.00132,
    0.00503,
    0.00014,
    0.00246,
    0.00215,
    -0.00282,
    0.00235,
    0.0014
   ],
   [
    -0.0033,
    0.00071,
    -0.0014,
    -0.00127,
    0.00228,
    -0.00043,
    -0.00149,
    -0.00383,
    0.00154,
    -0.00149,
    -0.0015,
    0.00157,
    -0.00062,
    -0.00123
   ],
   [
    -0.00704,
    -0.00582,
    -0.00015,
    -0.00523,
    -0.00156,
    0.00332,
    0.00176,
    -0.00931,
    -0.00716,
    -0.00303,
    -0.00557,
    -0.00211,
    0.0012,
    0.00033
   ],
   [
    0.00802,
    0.00108,
    0.00586,
    0.00613,
    -0.00384,
    -0.00618,
    0.00115,
    0.00767,
    0.00143,
    0.00484,
    0.00752,
    -0.0013,
    -0.00327,
    0.00141
   ],
   [
    -2e-05,
    -0.00198,
    -0.0001,
    -0.0008,
    -0.00077,
    0.00343,
    0.0029,
    -0.00121,
    -0.00139,
    -0.00014,
    -0.00044,
    -0.00055,
    0.0033,
    0.00272
   ],
   [
    0.00283,
    0.0009,
    -0.00013,
    0.00224,
    -0.00292,
    0.00077,
    0.00089,
    0.00372,
    0.00248,
    0.00138,
    0.00256,
    -0.00177,
    0.00126,
    0.00112
   ],
   [
    -0.00057,
    -0.00168,
    0.00097,
    0.00101,
    -0.00163,
    -0.00205,
    0.00019,
    -0.0002,
    -0.00119,
    0.00125,
    0.00081,
    -0.00178,
    -0.00185,
    2e-05
   ],
   [
    -0.00204,
    0.00096,
    -0.00044,
    -0.00043,
    -0.00026,
    -0.001,
    0.00151,
    -0.00157,
    -0.00098,
    -0.00056,
    6e-05,
    -6e-05,
    -0.00064,
    0.00087
   ],
   [
    0.00963,
    0.00341,
    -0.00272,
    -0.00613,
    -0.00393,
    -0.00375,
    0.00716,
    0.00882,
    0.00698,
    -4e-05,
    -0.00502,
    -0.00245,
    -0.00315,
    0.00562
   ],
   [
    0.00453,
    -0.0005,
    0.00055,
    0.00055,
    9e-05,
    -0.00104,
    -0.0027,
    0.00438,
    0.00018,
    0.00121,
    0.00063,
    8e-05,
    -0.00125,
    -0.00239
   ],
   [
    -0.00087,
    -0.00173,
    -0.00012,
    0.0004,
    -0.00101,
    -0.00015,
    0.00017,
    -0.0003,
    -0.0014,
    -0.00047,
    0.00052,
    -0.00087,
    -7e-05,
    5e-05
   ],
   [
    -0.00036,
    -0.00293,
    0.00068,
    0.00204,
    -0.00199,
    -0.00193,
    -0.00114,
    9e-05,
    -0.00252,
    0.0003,
    0.00201,
    -0.00206,
    -0.00207,
    -0.00116
   ],
   [
    -0.00615,
    -0.0138,
    -0.01441,
    -0.02555,
    -0.00702,
    0.02296,
    0.02407,
    -0.00534,
    -0.00612,
    -0.00527,
    -0.02088,
    -0.0033,
    0.02245,
    0.02178
   ],
   [
    0.00349,
    0.00032,
    0.00042,
    0.00049,
    -0.00055,
    0.00132,
    0.00217,
    0.00168,
    -3e-05,
    -9e-05,
    0.00061,
    -0.00021,
    0.00086,
    0.00168
   ],
   [
    0.00836,
    -0.00137,
    0.00342,
    0.00253,
    0.00901,
    -0.00861,
    -0.01176,
    0.00255,
    -0.00817,
    0.0007,
    0.00218,
    0.00691,
    -0.00813,
    -0.01163
   ],
   [
    0.00163,
    0.0041,
    0.00214,
    0.00252,
    -0.00294,
    0.00105,
    0.00431,
    0.00767,
    0.0038,
    0.00593,
    0.00396,
    -0.00163,
    0.00171,
    0.00429
   ],
   [
    -0.00238,
    0.00412,
    0.00068,
    -0.00272,
    0.00183,
    0.00035,
    6e-05,
    -0.00115,
    0.00359,
    0.00086,
    -0.00277,
    0.00156,
    -7e-05,
    -9e-05
   ],
   [
    -9e-05,
    -0.00097,
    0.00252,
    -0.00151,
    0.00105,
    -0.00135,
    -1e-05,
    -0.00153,
    -0.00117,
    0.0005,
    -0.00196,
    0.00097,
    -0.00139,
    -9e-05
   ],
   [
    -0.02406,
    0.00227,
    -0.00133,
    -0.00373,
    0.00914,
    0.01681,
    -0.00473,
    -0.01737,
    0.00133,
    0.00439,
    0.0002,
    0.00828,
    0.01589,
    -0.00215
   ],
   [
    0.0009,
    -0.00092,
    0.00608,
    0.00504,
    -0.00128,
    0.00063,
    -0.00079,
    0.00544,
    0.00249,
    0.00585,
    0.00534,
    -0.00063,
    0.00145,
    0.00015
   ],
   [
    0.0022,
    -0.00135,
    -0.00075,
    -0.00067,
    -0.00101,
    0.00127,
    0.00067,
    0.00263,
    -0.00055,
    0.00114,
    -0.00016,
    -0.00073,
    0.00103,
    0.00079
   ],
   [
    0.00579,
    0.00554,
    -0.00572,
    -0.00509,
    -0.00088,
    0.00155,
    -0.00019,
    0.00338,
    0.00586,
    -0.00177,
    -0.00393,
    0.00024,
    0.00125,
    -0.00151
   ],
   [
    -0.00059,
    0.00396,
    0.0018,
    0.00186,
    0.00725,
    -0.0018,
    0.00481,
    0.00336,
    0.00809,
    0.00241,
    0.00305,
    0.00775,
    0.00119,
    0.00616
   ],
   [
    0.0033,
    0.00612,
    0.00615,
    0.00347,
    -0.00162,
    0.00169,
    -1e-05,
    0.00665,
    0.00659,
    0.00626,
    0.0035,
    -0.00039,
    0.00215,
    0.00087
   ],
   [
    -0.00326,
    0.00119,
    -0.00025,
    -0.00263,
    0.00241,
    0.00235,
    -0.0023,
    -0.00392,
    -0.00111,
    -0.00124,
    -0.00329,
    0.00145,
    0.0017,
    -0.00223
   ],
   [
    -0.0027,
    0.00022,
    0.00248,
    -0.00073,
    -0.00088,
    -0.00015,
    0.00042,
    -0.00032,
    0.0008,
    0.00224,
    -0.00126,
    -0.00075,
    -0.0009,
    0.00032
   ],
   [
    -0.00357,
    -0.00017,
    0.00106,
    0.0011,
    -0.00013,
    0.00121,
    -0.00132,
    -0.00239,
    0.00017,
    0.00076,
    0.00089,
    -7e-05,
    0.00103,
    -0.00089
   ],
   [
    -0.00949,
    0.00967,
    0.00401,
    0.00349,
    -0.00316,
    0.00317,
    -0.01029,
    -0.00623,
    0.00806,
    0.00272,
    0.00322,
    -0.00194,
    0.0005,
    -0.00958
   ],
   [
    -0.0018,
    -0.00012,
    -0.00166,
    -0.00074,
    -0.00254,
    0.00238,
    0.00087,
    -0.00169,
    0.00147,
    -0.00012,
    -0.00109,
    -0.00208,
    0.00255,
    0.0006
   ],
   [
    0.00346,
    0.001,
    -0.00126,
    0.00255,
    0.0043,
    -0.00249,
    -0.00866,
    0.00416,
    0.00146,
    -0.00082,
    0.00352,
    0.0044,
    -0.00282,
    -0.0077
   ],
   [
    -0.01362,
    -0.003,
    -0.00189,
    -0.005,
    0.00105,
    -0.00034,
    -0.00041,
    -0.01309,
    -0.00195,
    -0.00023,
    -0.00523,
    -0.00035,
    -0.00011,
    -0.00087
   ],
   [
    -0.00734,
    0.0029,
    0.00018,
    -0.0034,
    0.00445,
    0.00239,
    -0.00346,
    -0.0036,
    0.00281,
    0.00065,
    -0.00342,
    0.00411,
    0.00163,
    -0.00281
   ],
   [
    0.00258,
    -0.00219,
    -0.00017,
    -0.00111,
    -0.00515,
    0.00329,
    0.00434,
    0.00171,
    -0.00178,
    0.00054,
    -0.00039,
    -0.00393,
    0.00338,
    0.00373
   ],
   [
    0.00215,
    -0.00593,
    0.00119,
    0.00191,
    -0.004,
    0.00135,
    -0.0,
    -2e-05,
    -0.00529,
    -0.0004,
    0.00149,
    -0.00335,
    0.00092,
    -0.00017
   ],
   [
    -0.00113,
    -0.00274,
    -0.00121,
    0.00038,
    -0.00213,
    -0.00061,
    0.00043,
    -0.00125,
    -0.00314,
    -0.00126,
    0.00034,
    -0.00158,
    -0.00117,
    9e-05
   ],
   [
    -0.00026,
    -0.00293,
    0.00059,
    0.00051,
    -0.00059,
    -0.00291,
    -0.00032,
    -0.00337,
    -0.00502,
    -0.00199,
    -0.00052,
    -0.00103,
    -0.00347,
    -0.00111
   ],
   [
    0.00055,
    4e-05,
    0.00013,
    0.0005,
    0.00178,
    0.0008,
    -0.001,
    0.00127,
    0.00108,
    0.00043,
    0.00057,
    0.00204,
    0.00088,
    -0.00063
   ],
   [
    -0.00019,
    0.00175,
    -0.0001,
    0.00063,
    -4e-05,
    -0.00226,
    0.0024,
    -0.00099,
    0.00052,
    0.00032,
    0.00069,
    0.0002,
    -0.00154,
    0.00197
   ],
   [
    -0.00016,
    0.00091,
    0.00152,
    0.001,
    0.00056,
    0.0001,
    -0.00334,
    0.00031,
    0.00076,
    0.00116,
    0.00091,
    0.00065,
    -0.00025,
    -0.00274
   ],
   [
    0.00142,
    0.00029,
    0.0011,
    0.00193,
    1e-05,
    -0.00175,
    0.0004,
    0.00109,
    0.00072,
    0.00032,
    0.00152,
    1e-05,
    -0.00125,
    0.00028
   ],
   [
    0.00481,
    0.00114,
    0.00105,
    -0.00084,
    -0.00229,
    0.00036,
    0.00529,
    0.00518,
    0.00369,
    0.00253,
    -0.00083,
    -0.00166,
    0.00057,
    0.00446
   ],
   [
    -0.01157,
    -0.00445,
    0.00626,
    -0.0025,
    0.00484,
    0.00019,
    -0.00916,
    -0.00671,
    -0.00285,
    0.00431,
    -0.00386,
    0.0024,
    -0.00245,
    -0.00956
   ],
   [
    0.00026,
    0.00044,
    -0.00202,
    -2e-05,
    -0.00169,
    0.00403,
    0.00122,
    -0.00024,
    0.0001,
    -0.00166,
    0.00019,
    -0.00124,
    0.00295,
    0.00032
   ],
   [
    -0.00195,
    -0.00245,
    -7e-05,
    -0.00165,
    -0.00558,
    0.00251,
    0.00948,
    -0.00194,
    -0.00147,
    0.00024,
    -0.00038,
    -0.00361,
    0.00356,
    0.00876
   ],
   [
    0.01027,
    0.00422,
    0.00372,
    0.00342,
    -0.00356,
    0.0025,
    -0.00032,
    0.0089,
    0.00463,
    0.00264,
    0.00289,
    -0.00223,
    0.00333,
    0.00027
   ],
   [
    0.00015,
    -2e-05,
    0.00107,
    9e-05,
    0.00018,
    7e-05,
    0.00092,
    -0.00041,
    0.00101,
    0.00114,
    0.00044,
    0.00044,
    0.00113,
    0.00114
   ],
   [
    -0.00571,
    -0.00499,
    0.00414,
    -0.00058,
    0.00224,
    -0.00466,
    -0.00147,
    -0.00175,
    -0.00214,
    0.00545,
    -0.00045,
    0.00158,
    -0.00349,
    -0.00051
   ],
   [
    -0.00276,
    -0.00343,
    -0.0013,
    -0.00023,
    -0.00254,
    0.00028,
    0.00197,
    -0.00221,
    -0.00288,
    -0.00122,
    -0.0005,
    -0.00251,
    8e-05,
    0.00146
   ],
   [
    0.00166,
    0.00149,
    0.00014,
    -5e-05,
    0.0004,
    0.0028,
    0.00256,
    0.00104,
    0.00118,
    0.00012,
    0.00047,
    0.00074,
    0.00301,
    0.00262
   ],
   [
    0.00108,
    0.00133,
    0.00167,
    0.00141,
    -0.00165,
    -0.00071,
    -0.00015,
    0.00167,
    0.0009,
    0.00179,
    0.00154,
    -0.00116,
    -0.00048,
    -0.00011
   ],
   [
    0.00081,
    -0.00014,
    -0.00167,
    -0.00505,
    -0.0055,
    0.00901,
    -0.0023,
    -0.00107,
    -0.00287,
    -0.0007,
    -0.00379,
    -0.00393,
    0.0081,
    -0.00264
   ],
   [
    0.00233,
    -0.00176,
    0.00307,
    0.00091,
    4e-05,
    -0.00166,
    -0.00205,
    0.00089,
    -0.00125,
    0.00225,
    0.00091,
    0.00012,
    -0.00123,
    -0.00153
   ],
   [
    -0.00643,
    0.01432,
    -0.00399,
    -4e-05,
    0.00717,
    -0.00394,
    0.00115,
    -0.00256,
    0.01254,
    -0.00012,
    -0.00052,
    0.0051,
    -0.00303,
    0.00098
   ],
   [
    0.01648,
    0.00955,
    0.00366,
    -0.00451,
    -0.00033,
    0.00275,
    0.01614,
    0.00902,
    0.00781,
    0.00118,
    -0.00429,
    0.00048,
    0.00215,
    0.01391
   ],
   [
    -0.00145,
    -0.00314,
    0.00147,
    0.00117,
    -0.00224,
    0.00068,
    0.00195,
    -0.00245,
    -0.00288,
    0.0005,
    0.00147,
    -0.0015,
    0.00088,
    0.0018
   ],
   [
    -0.00479,
    -0.00178,
    -0.001,
    -0.00277,
    0.00046,
    0.00182,
    0.00047,
    -0.00438,
    -0.00246,
    -0.00163,
    -0.00251,
    0.0001,
    0.00113,
    0.00012
   ],
   [
    0.00239,
    -0.00173,
    6e-05,
    -4e-05,
    -0.00219,
    -0.00033,
    0.00117,
    0.00014,
    -0.00192,
    -0.00088,
    -0.00012,
    -0.00164,
    -0.00056,
    0.00068
   ],
   [
    0.00259,
    -0.00087,
    -0.00155,
    0.00226,
    -0.00664,
    0.00018,
    -0.00524,
    0.00184,
    -0.00403,
    -0.00029,
    0.00359,
    -0.00501,
    0.00044,
    -0.00474
   ],
   [
    -0.0086,
    -0.00729,
    -0.00216,
    -0.00019,
    4e-05,
    0.00031,
    -0.00537,
    -0.00361,
    -0.00546,
    0.00023,
    -0.00159,
    -0.00123,
    -0.00106,
    -0.00494
   ],
   [
    -0.00039,
    -0.00119,
    -0.00103,
    -0.0021,
    -0.00879,
    0.0014,
    -0.00295,
    -0.00955,
    -0.00547,
    -0.00722,
    -0.00188,
    -0.00871,
    -0.00093,
    -0.00419
   ],
   [
    -0.00144,
    0.00138,
    -0.00121,
    -0.00125,
    3e-05,
    0.00178,
    0.00303,
    -0.00069,
    0.00133,
    -0.00153,
    -0.00109,
    0.00025,
    0.0017,
    0.00274
   ],
   [
    -0.00672,
    -0.00111,
    -0.00058,
    0.00132,
    0.00058,
    -0.00351,
    -0.0052,
    -0.00666,
    -0.00198,
    -0.00119,
    0.00137,
    0.00029,
    -0.00345,
    -0.00515
   ],
   [
    0.00316,
    -0.00101,
    -0.00263,
    0.00178,
    -0.00499,
    0.00405,
    -0.00027,
    0.00373,
    -0.00025,
    -0.00247,
    0.00122,
    -0.00381,
    0.00386,
    -0.00043
   ],
   [
    -0.00367,
    -4e-05,
    -0.00064,
    -0.00882,
    0.00267,
    -0.00301,
    -0.00763,
    -0.00189,
    -0.00314,
    -0.00093,
    -0.00962,
    0.00052,
    -0.00632,
    -0.00847
   ],
   [
    -0.0011,
    0.00029,
    -0.0011,
    -0.00056,
    0.00132,
    -0.0013,
    -0.0015,
    -0.00087,
    0.0011,
    -0.00072,
    -0.00036,
    0.00135,
    -0.00128,
    -0.00144
   ],
   [
    -0.00476,
    -0.00401,
    -0.00192,
    -0.00114,
    0.00147,
    -0.00057,
    -0.0015,
    -0.00401,
    -0.00322,
    -0.00167,
    -0.0005,
    0.00148,
    -0.00054,
    -0.00122
   ],
   [
    0.00176,
    -0.00099,
    0.00186,
    0.00019,
    0.00135,
    -0.00102,
    -0.00412,
    0.00036,
    -0.00142,
    0.00061,
    -9e-05,
    0.00058,
    -0.0013,
    -0.00385
   ],
   [
    0.0054,
    0.00146,
    0.00176,
    -0.00083,
    0.00399,
    0.00145,
    -0.00079,
    0.00406,
    0.00264,
    0.00163,
    -0.00031,
    0.00419,
    0.00152,
    -0.00027
   ],
   [
    0.00023,
    -0.00283,
    -0.00201,
    0.00309,
    -0.00152,
    -0.00186,
    -0.00068,
    -2e-05,
    -0.00273,
    -0.00103,
    0.00239,
    -0.00205,
    -0.00204,
    -0.0009
   ],
   [
    0.00099,
    0.00349,
    -0.00024,
    -0.00029,
    -0.00173,
    0.00049,
    0.00471,
    0.0024,
    0.00163,
    0.00015,
    -0.00077,
    -0.00177,
    0.00013,
    0.00382
   ],
   [
    0.01495,
    -0.01026,
    0.00944,
    0.01478,
    -0.00981,
    -0.00505,
    0.01798,
    0.01452,
    -0.00411,
    0.01019,
    0.01604,
    -0.00492,
    -0.00245,
    0.01684
   ],
   [
    -0.0067,
    -0.0027,
    -0.00759,
    -0.0023,
    -0.00223,
    0.00162,
    0.00883,
    -0.00461,
    -0.00052,
    -0.00211,
    -0.00318,
    -0.00323,
    0.0019,
    0.00781
   ],
   [
    -0.00277,
    0.00029,
    0.0017,
    0.00292,
    -0.00134,
    -0.00338,
    -0.00292,
    -0.00145,
    0.00239,
    3e-05,
    0.00213,
    -0.00101,
    -0.00364,
    -0.00335
   ],
   [
    0.00184,
    -0.00619,
    -0.003,
    0.01237,
    -0.00476,
    -0.00669,
    0.0014,
    0.00461,
    -0.00211,
    -0.00217,
    0.01067,
    -0.00476,
    -0.00712,
    -0.0
   ],
   [
    -0.0007,
    0.00378,
    -0.00155,
    0.00037,
    0.00166,
    -0.00206,
    0.00191,
    0.00209,
    0.00368,
    -0.00079,
    -3e-05,
    0.00169,
    -0.00132,
    0.00139
   ],
   [
    0.00363,
    0.00367,
    0.00141,
    -0.00092,
    0.00105,
    0.00152,
    0.00172,
    0.00234,
    0.00369,
    0.00036,
    -0.00058,
    0.00126,
    0.00161,
    0.0016
   ],
   [
    0.00213,
    -0.00079,
    -0.00135,
    0.00026,
    -0.00216,
    -0.00185,
    0.00013,
    9e-05,
    -0.00297,
    -0.00288,
    0.00063,
    -0.002,
    -0.00208,
    9e-05
   ],
   [
    -0.00382,
    -0.00346,
    -0.00026,
    0.00215,
    -0.00175,
    0.00025,
    -3e-05,
    -0.00273,
    -0.00274,
    0.00013,
    0.00259,
    -0.00114,
    0.00026,
    -0.00046
   ],
   [
    0.00161,
    0.00924,
    0.00576,
    0.00268,
    0.00612,
    -0.00261,
    -0.00476,
    0.00087,
    0.00704,
    0.00268,
    0.00206,
    0.00516,
    -0.00341,
    -0.0041
   ],
   [
    0.00102,
    -0.00525,
    0.0012,
    6e-05,
    0.00165,
    -0.00189,
    -0.00235,
    -0.00225,
    -0.00504,
    0.00022,
    0.00051,
    0.00081,
    -0.00127,
    -0.00214
   ],
   [
    0.00827,
    -0.01404,
    -0.00818,
    7e-05,
    -0.01306,
    0.00884,
    0.00866,
    0.00143,
    -0.0151,
    -0.00853,
    -0.00107,
    -0.01225,
    0.00704,
    0.00708
   ],
   [
    -0.00388,
    0.00149,
    -4e-05,
    0.00071,
    0.00126,
    -0.00157,
    -0.00134,
    -0.00146,
    0.00225,
    0.00098,
    0.00109,
    0.00164,
    -0.00076,
    -0.00062
   ],
   [
    0.00102,
    -0.00082,
    0.00045,
    0.00178,
    -0.00062,
    0.00079,
    -0.00014,
    0.00019,
    -0.00047,
    -0.00017,
    0.00202,
    -0.00033,
    0.00113,
    0.00025
   ],
   [
    0.00058,
    0.00232,
    0.00249,
    0.00164,
    -0.0001,
    -0.00112,
    -0.0004,
    0.00121,
    0.00258,
    0.00132,
    0.00101,
    -0.00029,
    -0.00101,
    -0.00062
   ],
   [
    -0.00051,
    0.00012,
    0.00171,
    -0.00025,
    -0.0002,
    -0.00164,
    0.0004,
    -0.00077,
    3e-05,
    0.0001,
    -0.00044,
    -8e-05,
    -0.00173,
    0.00019
   ],
   [
    -0.00601,
    0.01023,
    0.00631,
    0.01249,
    0.01877,
    -0.01531,
    -0.02084,
    -0.00551,
    0.01034,
    0.00557,
    0.01496,
    0.01725,
    -0.00946,
    -0.01646
   ],
   [
    0.00345,
    -0.00057,
    0.00092,
    0.00132,
    0.00144,
    -0.00235,
    -0.00266,
    0.00271,
    0.00058,
    0.0015,
    0.00091,
    0.00062,
    -0.00205,
    -0.00223
   ],
   [
    -0.00028,
    -0.00299,
    -0.0007,
    -0.00197,
    -0.00053,
    0.00081,
    -0.00027,
    -0.00188,
    -0.00324,
    -0.0012,
    -0.00142,
    3e-05,
    4e-05,
    -0.00053
   ],
   [
    -0.00948,
    -0.00456,
    -0.00562,
    0.00158,
    -0.00195,
    0.00182,
    -0.00249,
    -0.00744,
    -0.00136,
    -0.00144,
    0.00109,
    -0.00217,
    0.00233,
    -0.00206
   ],
   [
    -0.00283,
    -0.00162,
    -0.00268,
    -0.00296,
    0.00071,
    -0.00184,
    -0.00181,
    0.00012,
    -0.00032,
    0.00068,
    -0.00244,
    -0.0001,
    -0.00205,
    -0.00166
   ],
   [
    -0.0004,
    -0.00189,
    -0.00072,
    -0.00025,
    -0.00121,
    0.0003,
    -0.00019,
    -0.00034,
    -0.00132,
    -0.0013,
    -0.00015,
    -0.00064,
    0.00028,
    6e-05
   ],
   [
    -0.00066,
    -0.00399,
    -0.00035,
    0.0014,
    -0.00038,
    0.00045,
    0.00064,
    0.00073,
    -0.00237,
    -0.00051,
    0.00138,
    -0.00017,
    0.00031,
    0.00061
   ],
   [
    0.0048,
    0.00259,
    0.0017,
    0.00106,
    0.00068,
    0.00043,
    -0.00206,
    0.00479,
    0.00334,
    0.00162,
    0.0015,
    0.00125,
    0.00088,
    -0.00138
   ],
   [
    0.00336,
    0.00147,
    -0.00206,
    -0.00044,
    7e-05,
    0.0013,
    0.00014,
    0.00301,
    0.00061,
    -0.0012,
    -0.00106,
    -0.0003,
    0.0012,
    0.00012
   ],
   [
    -0.0002,
    0.00084,
    -0.0008,
    -0.00157,
    -0.0008,
    -0.00075,
    0.00214,
    0.00039,
    -0.00066,
    -0.00022,
    -0.00204,
    -0.00113,
    -0.0011,
    0.00141
   ],
   [
    -3e-05,
    0.00146,
    -0.0016,
    -0.00049,
    0.00228,
    0.00103,
    -0.00065,
    0.00069,
    0.00133,
    -0.00074,
    -0.00049,
    0.00229,
    0.00082,
    -0.00052
   ],
   [
    0.00461,
    -0.00135,
    0.00024,
    -0.00018,
    -0.00131,
    2e-05,
    0.0016,
    0.00303,
    -0.00126,
    0.00017,
    0.00011,
    -0.00083,
    0.00048,
    0.00125
   ],
   [
    0.00395,
    0.0049,
    0.00456,
    0.00095,
    0.00159,
    0.00133,
    -0.00119,
    0.00096,
    0.0038,
    0.00414,
    0.00155,
    0.00138,
    0.00134,
    -0.0001
   ],
   [
    0.00511,
    0.00307,
    0.00065,
    5e-05,
    0.00164,
    3e-05,
    -0.00036,
    0.00483,
    0.00307,
    0.00017,
    0.00018,
    0.00168,
    0.00048,
    -7e-05
   ],
   [
    -0.00396,
    -0.00156,
    -0.00051,
    -0.00162,
    4e-05,
    0.00149,
    0.00132,
    -0.00263,
    -0.00147,
    -0.0003,
    -0.0018,
    -2e-05,
    0.00135,
    0.00125
   ],
   [
    -0.00377,
    -0.00438,
    -0.00081,
    -0.00314,
    0.00154,
    0.00021,
    -0.00133,
    -0.00206,
    -0.00228,
    0.00037,
    -0.00284,
    0.001,
    0.00021,
    -0.00111
   ],
   [
    0.00234,
    -0.0015,
    -0.00022,
    -0.00326,
    -0.00172,
    0.00786,
    0.00253,
    0.00572,
    0.00107,
    0.0032,
    -0.00234,
    -0.00042,
    0.00722,
    0.00305
   ],
   [
    -0.00412,
    -0.00111,
    -0.00019,
    0.00033,
    0.00103,
    2e-05,
    -0.00324,
    -0.0029,
    -0.00076,
    0.00077,
    -9e-05,
    0.00027,
    -0.00022,
    -0.00277
   ],
   [
    0.00183,
    -0.01775,
    0.00681,
    0.00771,
    -0.01649,
    -0.00634,
    0.01047,
    -0.00322,
    -0.01371,
    0.0,
    0.00584,
    -0.01464,
    -0.00665,
    0.00692
   ],
   [
    0.00427,
    -0.00055,
    0.00218,
    0.00098,
    -0.0015,
    0.0015,
    0.00018,
    0.00164,
    -0.00094,
    0.00096,
    0.0009,
    -0.00128,
    0.00152,
    3e-05
   ],
   [
    -0.00056,
    -0.00625,
    -0.0019,
    0.00231,
    -0.00075,
    -0.00607,
    0.00058,
    -0.00182,
    -0.00702,
    -0.00408,
    0.00139,
    -0.00126,
    -0.00611,
    -0.00052
   ],
   [
    8e-05,
    -0.00134,
    -0.00113,
    -0.00125,
    -0.00278,
    0.00213,
    -0.00312,
    0.00209,
    -0.00142,
    0.00072,
    -0.00119,
    -0.00208,
    0.00202,
    -0.00303
   ],
   [
    0.00011,
    1e-05,
    -0.00034,
    -0.00052,
    -0.00166,
    0.00105,
    0.00216,
    -9e-05,
    0.00051,
    -0.00013,
    -8e-05,
    -0.00131,
    0.00112,
    0.00177
   ],
   [
    0.00284,
    -0.00208,
    0.00027,
    -0.0004,
    0.0005,
    -0.00064,
    0.00156,
    0.00205,
    -0.00248,
    -0.00167,
    -0.00041,
    0.00047,
    -0.00015,
    0.00146
   ],
   [
    -0.00293,
    -0.00106,
    0.00084,
    -0.00375,
    0.00042,
    0.00124,
    0.00053,
    -0.00465,
    -0.00028,
    -0.0006,
    -0.00304,
    0.00063,
    0.00134,
    0.00055
   ],
   [
    0.00386,
    0.00658,
    0.00406,
    0.00279,
    -0.00045,
    6e-05,
    -0.00058,
    0.00571,
    0.00647,
    0.00389,
    0.00235,
    -0.0001,
    0.0008,
    -0.00013
   ],
   [
    -0.00685,
    -0.00439,
    -0.00211,
    -0.00552,
    0.00135,
    0.00156,
    -0.00045,
    -0.00824,
    -0.00586,
    -0.00181,
    -0.00511,
    0.00052,
    0.00056,
    -0.00046
   ],
   [
    -0.00424,
    0.00036,
    0.00228,
    -0.00165,
    0.00025,
    0.00247,
    0.0015,
    -0.00344,
    0.00023,
    0.00232,
    -0.00118,
    0.0003,
    0.00208,
    0.0015
   ],
   [
    0.00582,
    0.00394,
    0.00413,
    -0.00594,
    0.00996,
    -0.00474,
    -0.01157,
    0.00393,
    0.00321,
    0.00097,
    -0.00638,
    0.00762,
    -0.00555,
    -0.01111
   ],
   [
    -0.00276,
    -0.00112,
    -0.00089,
    -0.00126,
    0.0023,
    -0.00147,
    -0.00168,
    -0.00182,
    0.00022,
    -5e-05,
    -0.0016,
    0.0017,
    -0.00178,
    -0.00148
   ],
   [
    -0.00268,
    0.01143,
    -0.00137,
    -0.00122,
    0.00243,
    -0.00075,
    -0.0006,
    3e-05,
    0.01044,
    0.00035,
    -0.00132,
    0.00324,
    -0.00049,
    -0.00082
   ],
   [
    0.00689,
    0.0012,
    0.00132,
    0.00151,
    0.0004,
    4e-05,
    0.00096,
    0.00377,
    0.00099,
    3e-05,
    0.00212,
    0.00091,
    0.00061,
    0.00123
   ],
   [
    -0.00085,
    0.00061,
    -0.00043,
    0.00091,
    -0.00081,
    -0.00126,
    0.00145,
    -0.00097,
    0.00021,
    -0.00023,
    0.00096,
    -0.00056,
    -0.00092,
    0.00109
   ],
   [
    0.00657,
    0.00493,
    0.00028,
    0.00111,
    0.00134,
    0.00075,
    -0.00391,
    0.00665,
    0.00405,
    0.00038,
    0.00047,
    0.00114,
    -0.00014,
    -0.00307
   ],
   [
    0.00913,
    0.00634,
    0.0047,
    -0.00046,
    -0.00212,
    0.00464,
    0.00343,
    0.0082,
    0.01106,
    0.00316,
    -0.00108,
    -0.00035,
    0.00283,
    0.0018
   ],
   [
    0.01904,
    0.00964,
    0.0073,
    0.00266,
    -0.00079,
    -0.00369,
    -0.01261,
    0.01283,
    0.00531,
    0.00355,
    0.00181,
    -0.00158,
    -0.00353,
    -0.01119
   ],
   [
    0.00053,
    -0.00353,
    -0.00039,
    0.0016,
    -0.00259,
    -0.00059,
    0.00097,
    0.00082,
    -0.00205,
    -0.00056,
    0.0014,
    -0.00243,
    -0.00058,
    0.00082
   ],
   [
    0.01075,
    0.00181,
    -0.00242,
    0.00277,
    -0.01027,
    0.00701,
    0.00643,
    0.00857,
    0.00419,
    -0.0017,
    0.00259,
    -0.007,
    0.00707,
    0.00646
   ],
   [
    0.00267,
    -0.00421,
    0.00231,
    0.00665,
    0.00167,
    0.00269,
    -0.01353,
    0.0051,
    -0.0055,
    0.00268,
    0.00386,
    -0.00089,
    0.00049,
    -0.01206
   ],
   [
    -0.00174,
    0.00141,
    -0.00198,
    -0.00118,
    0.00237,
    0.00075,
    -0.00375,
    -0.00064,
    0.00135,
    -0.00099,
    -0.00149,
    0.00155,
    0.00027,
    -0.0033
   ],
   [
    0.00469,
    -0.00125,
    0.00205,
    0.00125,
    -0.00166,
    0.00128,
    4e-05,
    0.00262,
    -0.00172,
    0.0006,
    0.0009,
    -0.00152,
    0.00139,
    0.00039
   ],
   [
    -0.00237,
    0.00086,
    -0.0009,
    -0.00204,
    0.00039,
    0.00274,
    0.00311,
    -0.00169,
    0.00215,
    -0.00032,
    -0.00203,
    0.00039,
    0.00264,
    0.00282
   ],
   [
    -0.01104,
    0.00964,
    -0.00116,
    -0.01238,
    0.02377,
    -0.01456,
    0.00446,
    -0.01278,
    0.00088,
    -0.00684,
    -0.01295,
    0.01753,
    -0.01254,
    0.00422
   ],
   [
    -0.00212,
    -4e-05,
    0.00134,
    -0.00073,
    0.00049,
    -0.00145,
    4e-05,
    -0.00085,
    -0.0003,
    0.00103,
    -0.0007,
    0.00021,
    -0.00142,
    -0.00013
   ],
   [
    -0.00123,
    -0.00177,
    -0.00133,
    0.00041,
    -0.00078,
    -0.0012,
    -0.00138,
    -0.00242,
    -0.00195,
    -0.00145,
    0.00049,
    -0.00097,
    -0.00129,
    -0.00126
   ],
   [
    0.0032,
    0.01813,
    -0.00456,
    -0.00119,
    0.00961,
    -0.00861,
    -0.00834,
    0.00429,
    0.01354,
    -0.0021,
    -0.0015,
    0.007,
    -0.00732,
    -0.00683
   ],
   [
    0.00144,
    -0.00209,
    0.00147,
    -0.00365,
    0.00306,
    0.00083,
    -0.00342,
    -0.00267,
    -0.00311,
    -0.00119,
    -0.00354,
    0.0022,
    0.0,
    -0.0032
   ],
   [
    0.02542,
    -0.00105,
    0.01096,
    -0.00147,
    -0.006,
    0.0028,
    0.00588,
    0.02122,
    -2e-05,
    0.0052,
    -0.00157,
    -0.00395,
    0.00261,
    0.00585
   ],
   [
    -0.00453,
    0.00015,
    -0.00093,
    -0.00403,
    0.00318,
    0.00305,
    0.0009,
    -0.00511,
    -0.00104,
    -0.00109,
    -0.00421,
    0.00202,
    0.00224,
    0.00069
   ],
   [
    0.00325,
    0.0002,
    0.00163,
    -0.00114,
    -0.00107,
    0.00371,
    0.00029,
    0.0019,
    0.00035,
    0.00066,
    -0.00075,
    -0.00076,
    0.00292,
    0.00043
   ],
   [
    0.00268,
    0.00361,
    0.00382,
    -0.00211,
    0.00171,
    0.00157,
    0.00096,
    0.00402,
    0.00516,
    0.00317,
    -0.00152,
    0.00225,
    0.00122,
    0.00092
   ],
   [
    -0.00112,
    -0.00125,
    -0.00073,
    -0.00278,
    -0.00029,
    0.00217,
    0.00087,
    -0.00177,
    -0.0015,
    -0.00151,
    -0.00299,
    -0.00061,
    0.0015,
    0.00053
   ],
   [
    -0.00273,
    -0.00128,
    -7e-05,
    -0.00046,
    0.00085,
    -0.0007,
    -0.0022,
    -0.00244,
    -0.00143,
    0.00027,
    -0.00046,
    0.00045,
    -0.00044,
    -0.00142
   ],
   [
    0.00296,
    0.00041,
    -0.0012,
    -0.00081,
    0.00282,
    -0.00101,
    -0.00052,
    0.00147,
    0.00031,
    -0.00195,
    -0.00155,
    0.00174,
    -0.00072,
    -0.0004
   ],
   [
    -0.00163,
    0.00076,
    -0.00085,
    -0.00193,
    1e-05,
    0.00082,
    0.00116,
    0.00144,
    0.00048,
    -0.00074,
    -0.00282,
    -0.00065,
    0.00023,
    0.00069
   ],
   [
    -0.0002,
    -0.00118,
    -0.00336,
    -0.00228,
    -0.0012,
    0.00192,
    0.00028,
    2e-05,
    -0.00398,
    -0.00451,
    -0.0029,
    -0.00171,
    0.00011,
    -0.00086
   ],
   [
    0.00393,
    0.00037,
    -0.00213,
    -0.00211,
    -0.00175,
    0.00096,
    0.00369,
    0.00539,
    0.0014,
    -0.0,
    -0.00268,
    -0.00214,
    0.0007,
    0.0031
   ],
   [
    -0.00393,
    -0.00027,
    -0.00434,
    0.00133,
    0.01249,
    -0.00425,
    -0.00603,
    -0.00471,
    -0.00263,
    -0.00273,
    0.00262,
    0.01095,
    -0.00211,
    -0.004
   ],
   [
    -0.00186,
    0.00017,
    -0.00076,
    0.00039,
    0.00164,
    -0.0015,
    -0.00093,
    -0.00191,
    2e-05,
    -0.0011,
    7e-05,
    0.00112,
    -0.00141,
    -0.00101
   ],
   [
    -0.00262,
    0.00068,
    0.00683,
    -0.0004,
    0.0048,
    0.00467,
    -0.00407,
    -0.00294,
    0.00188,
    0.00306,
    -0.00034,
    0.00377,
    0.00449,
    -0.00278
   ],
   [
    -0.00246,
    0.0023,
    0.00084,
    0.00114,
    0.00112,
    -0.00353,
    -0.00118,
    -0.00044,
    -0.0003,
    1e-05,
    0.00039,
    0.00012,
    -0.0034,
    -0.00122
   ],
   [
    0.00312,
    -0.00068,
    0.00309,
    0.00186,
    -0.00243,
    0.00231,
    0.00049,
    0.00194,
    0.00017,
    0.00202,
    0.00139,
    -0.00195,
    0.00189,
    0.00038
   ],
   [
    6e-05,
    -0.00146,
    0.00237,
    -0.00018,
    -0.00049,
    0.0002,
    0.00193,
    -0.00062,
    -0.0005,
    0.00217,
    0.00068,
    0.00038,
    0.00093,
    0.00214
   ],
   [
    -0.001,
    0.00017,
    0.00197,
    -0.00154,
    -0.00111,
    -0.0001,
    0.00449,
    -0.00217,
    -0.0006,
    0.00135,
    -0.00137,
    -0.0009,
    -0.00013,
    0.00375
   ],
   [
    -0.00745,
    -0.00416,
    -5e-05,
    -0.00157,
    0.00048,
    -0.00185,
    -0.00101,
    -0.00855,
    -0.00603,
    -0.00278,
    -0.00242,
    -0.00099,
    -0.00204,
    -0.00209
   ],
   [
    0.01639,
    -0.00038,
    -0.00499,
    -0.00709,
    0.00968,
    -0.00355,
    -0.00567,
    0.00779,
    0.00069,
    -0.00388,
    -0.0045,
    0.00801,
    -0.0012,
    -0.00345
   ],
   [
    0.002,
    -0.01032,
    -0.00381,
    -0.00088,
    -0.01683,
    0.00986,
    0.00531,
    0.00179,
    -0.00639,
    0.0011,
    -0.00137,
    -0.01374,
    0.00926,
    0.0059
   ],
   [
    0.00384,
    0.00614,
    -0.00209,
    -0.0046,
    0.00385,
    0.00067,
    -0.00742,
    0.00325,
    0.006,
    -0.00089,
    -0.00426,
    0.00272,
    0.00062,
    -0.00627
   ],
   [
    0.00075,
    0.00121,
    0.00025,
    0.00067,
    0.002,
    -0.00224,
    -0.00444,
    0.00192,
    0.00102,
    0.00036,
    0.00057,
    0.00176,
    -0.002,
    -0.00353
   ],
   [
    0.00284,
    -0.00033,
    -0.00082,
    0.00063,
    0.00157,
    -0.00068,
    -0.00012,
    0.00303,
    -0.00083,
    -0.00193,
    2e-05,
    0.0008,
    -0.00034,
    -0.00025
   ],
   [
    -0.00173,
    -0.00703,
    -0.0016,
    -9e-05,
    -0.00563,
    0.00167,
    -0.00207,
    -0.00369,
    -0.00852,
    -0.0016,
    -0.00103,
    -0.00567,
    0.00032,
    -0.00236
   ],
   [
    -0.00341,
    -0.00288,
    -0.00117,
    0.00143,
    -0.00122,
    0.00132,
    -0.0,
    -0.00239,
    -0.00334,
    -0.00061,
    0.0004,
    -0.00169,
    0.00067,
    -0.00051
   ],
   [
    0.00092,
    -0.00217,
    0.00014,
    -0.00014,
    -0.00077,
    0.00042,
    -0.00113,
    -0.00011,
    -0.00125,
    0.00018,
    0.00054,
    -0.00026,
    0.00086,
    -0.0006
   ],
   [
    0.00418,
    -0.00035,
    -0.0005,
    -0.00243,
    -0.00019,
    0.00286,
    -0.00078,
    0.00239,
    0.00032,
    -0.00132,
    -0.00271,
    -0.00037,
    0.00213,
    -0.00085
   ],
   [
    0.00197,
    0.01092,
    -0.00096,
    -0.0011,
    0.00767,
    -0.00327,
    0.00058,
    0.00372,
    0.01133,
    0.00252,
    -0.00044,
    0.00646,
    -0.00424,
    0.00076
   ],
   [
    0.00179,
    0.0036,
    0.00141,
    -0.00054,
    -0.00034,
    0.00054,
    -9e-05,
    0.00301,
    0.00282,
    0.00111,
    -0.00086,
    -0.0002,
    -0.00011,
    -0.00046
   ],
   [
    -0.00959,
    -0.00547,
    -0.00595,
    0.00107,
    -0.00223,
    0.0053,
    -0.00786,
    -0.00726,
    -0.00517,
    -0.00383,
    -7e-05,
    -0.00306,
    0.00374,
    -0.00714
   ],
   [
    0.00187,
    -0.01079,
    0.00043,
    0.00162,
    -0.00695,
    -0.00432,
    -0.00283,
    0.00217,
    -0.00782,
    -0.00327,
    0.00017,
    -0.00638,
    -0.00893,
    -0.00507
   ],
   [
    0.00192,
    0.00489,
    -0.00202,
    -0.00236,
    0.00366,
    0.00191,
    -0.00278,
    0.00249,
    0.00494,
    0.00042,
    -0.00287,
    0.00254,
    0.00103,
    -0.00248
   ],
   [
    -0.0059,
    -0.00194,
    0.00231,
    -0.00234,
    -0.0017,
    0.00473,
    0.00148,
    -0.00675,
    -0.00275,
    0.00084,
    -0.00194,
    -0.00075,
    0.00383,
    0.00139
   ],
   [
    -0.00068,
    -0.00165,
    0.00073,
    0.00033,
    -0.00221,
    0.0008,
    -0.00026,
    -0.00137,
    -0.00036,
    0.00196,
    0.00015,
    -0.00164,
    0.00089,
    -0.00041
   ],
   [
    0.00956,
    0.0094,
    -0.00038,
    -0.00266,
    0.00181,
    0.00213,
    0.00504,
    0.00546,
    0.00474,
    0.00183,
    -0.00218,
    0.00169,
    0.00327,
    0.00571
   ],
   [
    -0.00885,
    -0.00492,
    -0.00522,
    -0.0042,
    0.00403,
    -0.00038,
    -0.00024,
    -0.0109,
    -0.00746,
    -0.00747,
    -0.00427,
    0.0024,
    -5e-05,
    -0.00076
   ],
   [
    -0.00362,
    -0.00358,
    0.00022,
    -0.00024,
    0.00139,
    0.00126,
    0.00054,
    -0.00156,
    -0.0019,
    -0.00085,
    -0.00124,
    0.00132,
    0.00075,
    0.0004
   ],
   [
    0.00283,
    0.00224,
    0.00097,
    0.00095,
    -0.00228,
    -7e-05,
    0.00218,
    0.00252,
    0.00217,
    0.00075,
    0.00095,
    -0.00148,
    0.00051,
    0.00197
   ],
   [
    0.00295,
    0.00317,
    0.00134,
    -0.00348,
    0.00036,
    0.00038,
    0.00163,
    0.00212,
    0.00243,
    0.00049,
    -0.0032,
    0.0005,
    5e-05,
    0.00103
   ],
   [
    0.0034,
    0.00282,
    0.00736,
    -0.00044,
    0.00363,
    -0.00166,
    -0.00384,
    0.00332,
    0.00278,
    0.00476,
    0.00051,
    0.004,
    -0.00172,
    -0.00296
   ],
   [
    0.01122,
    -0.00653,
    -0.00914,
    0.00325,
    -0.0064,
    -0.00134,
    6e-05,
    0.01155,
    -0.00529,
    -0.006,
    0.00133,
    -0.00659,
    -0.00188,
    -0.00079
   ],
   [
    -0.00241,
    -0.0075,
    -0.00044,
    0.00262,
    -0.00293,
    0.00109,
    0.00252,
    -0.00149,
    -0.00563,
    0.00129,
    0.00413,
    -0.00109,
    0.0026,
    0.00323
   ],
   [
    0.00188,
    -5e-05,
    0.00128,
    0.00059,
    0.0012,
    -0.00112,
    -0.00144,
    0.00325,
    0.00079,
    0.00081,
    0.0007,
    0.00131,
    -0.0005,
    -0.00052
   ],
   [
    -0.0006,
    -0.00019,
    0.00287,
    0.00184,
    -0.00122,
    -0.00108,
    -0.00052,
    -0.00042,
    -0.00163,
    0.00102,
    0.0018,
    -0.00096,
    -0.00085,
    -0.00064
   ],
   [
    -0.00199,
    -0.0028,
    -0.00214,
    -0.00126,
    0.00261,
    0.00287,
    -0.00078,
    -0.0021,
    -0.00188,
    -0.00119,
    -0.00082,
    0.00281,
    0.00265,
    -0.00076
   ],
   [
    -0.00572,
    -0.00506,
    0.00012,
    0.0007,
    -0.00069,
    -0.00113,
    0.00378,
    -0.0059,
    -0.00394,
    -0.00097,
    0.00092,
    -0.0002,
    -0.00048,
    0.00348
   ],
   [
    -0.00353,
    -0.00232,
    -0.0001,
    0.00139,
    -0.00152,
    -0.00364,
    -0.00266,
    -0.0016,
    -0.00151,
    -0.00131,
    0.00014,
    -0.00189,
    -0.00393,
    -0.00289
   ],
   [
    0.00125,
    0.00105,
    -0.00438,
    0.00282,
    0.00163,
    -0.00286,
    0.00323,
    0.00483,
    0.00189,
    -0.00016,
    0.00184,
    0.00137,
    -0.0038,
    0.00165
   ],
   [
    -0.00144,
    0.00205,
    -0.0008,
    -0.00114,
    0.00154,
    0.00135,
    0.00083,
    -0.00367,
    0.00039,
    -0.0025,
    -0.00155,
    0.00102,
    0.00111,
    0.00059
   ],
   [
    -0.004,
    -0.00071,
    -0.00338,
    -0.00068,
    0.00241,
    0.0013,
    -0.00237,
    -0.00348,
    0.00013,
    -0.0023,
    -0.00065,
    0.00171,
    0.00043,
    -0.00187
   ],
   [
    -0.0159,
    -0.01145,
    -0.00604,
    0.00211,
    -1e-05,
    -0.00306,
    -0.00901,
    -0.01566,
    -0.01072,
    -0.00406,
    0.00142,
    -0.00178,
    -0.00484,
    -0.00778
   ],
   [
    -0.00228,
    0.00192,
    0.00074,
    -0.00076,
    0.00204,
    -0.00154,
    -0.00211,
    -0.00093,
    0.00195,
    0.00026,
    -0.00092,
    0.00149,
    -0.00139,
    -0.00187
   ],
   [
    -0.00759,
    -0.00665,
    -0.00852,
    -0.00064,
    -0.00549,
    0.00554,
    0.00301,
    -0.00732,
    -0.00844,
    -0.0044,
    -0.00025,
    -0.00542,
    0.00516,
    0.00282
   ],
   [
    0.00537,
    0.00103,
    -0.00134,
    0.00661,
    0.00419,
    -0.00081,
    1e-05,
    0.00572,
    0.00633,
    0.00087,
    0.0074,
    0.00541,
    0.00086,
    0.00113
   ],
   [
    0.01872,
    -0.00472,
    -0.00253,
    0.00955,
    -0.00956,
    0.00224,
    -0.00299,
    0.01974,
    -0.00236,
    -0.00126,
    0.00799,
    -0.00829,
    0.00022,
    -0.00351
   ],
   [
    0.0001,
    0.00024,
    -0.00115,
    -0.00142,
    -0.00072,
    0.0031,
    0.00182,
    -0.00106,
    -0.00123,
    -0.00204,
    -0.00099,
    -0.00046,
    0.00307,
    0.00175
   ],
   [
    -0.0067,
    0.01435,
    0.00667,
    0.00544,
    0.00447,
    -0.00927,
    -0.00474,
    0.00023,
    0.01151,
    0.00623,
    0.00468,
    0.00406,
    -0.00636,
    -0.0035
   ],
   [
    -0.00996,
    -0.00224,
    0.00079,
    -0.00193,
    0.0049,
    -0.00205,
    -0.00058,
    -0.0108,
    -0.0033,
    -0.00075,
    -0.00192,
    0.0043,
    -0.00264,
    -0.00119
   ],
   [
    0.00104,
    -0.00236,
    -0.00053,
    0.00136,
    -0.00106,
    -0.00031,
    0.00082,
    0.0015,
    -0.00185,
    0.00023,
    0.00124,
    -0.00142,
    -0.00053,
    0.00081
   ],
   [
    -0.00256,
    0.00695,
    0.00174,
    -0.00372,
    -0.00048,
    0.00237,
    0.00891,
    -0.00065,
    0.00648,
    0.00383,
    -0.00218,
    0.00137,
    0.00351,
    0.0087
   ],
   [
    -0.00041,
    0.00411,
    0.00095,
    -0.00297,
    -0.00058,
    -0.00015,
    0.0038,
    0.00111,
    0.00376,
    0.00082,
    -0.00336,
    -0.00054,
    -0.00028,
    0.00316
   ],
   [
    -0.00383,
    0.00751,
    -0.00239,
    -0.00167,
    0.01028,
    0.00114,
    -0.00053,
    -0.00199,
    0.00524,
    -0.00082,
    -0.00222,
    0.00802,
    -0.00025,
    -0.0005
   ],
   [
    0.0047,
    -0.00619,
    0.00099,
    0.00129,
    -0.00158,
    0.00057,
    0.00091,
    -0.0,
    -0.00601,
    -0.00014,
    0.00127,
    -0.00149,
    -0.00035,
    0.00082
   ],
   [
    -0.00115,
    -0.00137,
    0.00202,
    0.00071,
    -0.00055,
    -0.00127,
    0.00097,
    -0.00105,
    -0.0014,
    0.00126,
    0.00068,
    -0.00031,
    -0.00113,
    0.00059
   ],
   [
    0.00576,
    0.00061,
    -0.00528,
    0.00097,
    -0.0077,
    0.00173,
    0.00613,
    0.00186,
    0.00263,
    -0.0031,
    0.00077,
    -0.00686,
    0.00113,
    0.00492
   ],
   [
    -0.00539,
    -0.0084,
    -0.00594,
    0.00068,
    -0.00294,
    0.00153,
    0.00081,
    -0.0049,
    -0.00555,
    -0.00265,
    0.00132,
    -0.0026,
    0.00211,
    0.00092
   ],
   [
    0.0024,
    0.00336,
    -0.00093,
    0.00031,
    -0.00399,
    -0.00078,
    0.00533,
    0.00367,
    0.00285,
    -0.00131,
    0.0005,
    -0.00213,
    -0.00053,
    0.00432
   ],
   [
    -0.00218,
    -0.00313,
    0.00094,
    -0.00251,
    -0.00086,
    0.00078,
    -0.00044,
    -0.00307,
    -0.00332,
    -0.00059,
    -0.00204,
    -0.00093,
    0.00062,
    -0.00029
   ],
   [
    -0.00039,
    -0.00228,
    0.00151,
    -0.00109,
    0.00062,
    5e-05,
    0.00167,
    -0.00149,
    -0.00185,
    0.00049,
    -0.00105,
    0.00064,
    0.00012,
    0.00146
   ],
   [
    0.0035,
    0.00086,
    0.00116,
    0.00018,
    -0.00194,
    0.001,
    0.00168,
    0.00197,
    0.001,
    0.00158,
    0.00066,
    -0.00129,
    0.00146,
    0.00202
   ],
   [
    -0.01106,
    0.00552,
    -0.0001,
    -0.00772,
    0.00079,
    0.00309,
    0.00727,
    -0.00563,
    0.00564,
    0.00121,
    -0.00751,
    0.00107,
    0.00337,
    0.0066
   ],
   [
    0.004,
    0.00313,
    0.00259,
    0.00125,
    -0.00026,
    -0.00378,
    0.00601,
    0.00411,
    0.00344,
    0.00043,
    0.00141,
    0.00054,
    -0.00336,
    0.00491
   ],
   [
    -0.00217,
    -0.00222,
    -0.00104,
    0.00025,
    -0.00206,
    -0.00071,
    0.00019,
    -0.00229,
    -0.00155,
    -0.00108,
    0.00028,
    -0.00187,
    -0.00055,
    0.00022
   ],
   [
    0.00565,
    0.01675,
    -0.00597,
    -0.00632,
    0.00646,
    0.00547,
    0.00287,
    0.00845,
    0.01487,
    -0.00134,
    -0.00519,
    0.00516,
    0.00565,
    0.00332
   ],
   [
    -0.00042,
    0.00166,
    0.00501,
    0.0043,
    0.01033,
    -0.00343,
    -0.01155,
    -0.00173,
    0.00018,
    0.00215,
    0.00449,
    0.0082,
    -0.00271,
    -0.00956
   ],
   [
    0.00087,
    0.00205,
    0.00153,
    0.00117,
    -0.00027,
    9e-05,
    0.00105,
    0.00037,
    0.00278,
    0.00081,
    0.00122,
    0.00033,
    0.00061,
    0.00122
   ],
   [
    0.00011,
    -0.00163,
    -0.00087,
    -0.00035,
    -0.00288,
    0.00016,
    0.00275,
    -0.00057,
    -0.00175,
    -0.0019,
    -0.00045,
    -0.00211,
    -6e-05,
    0.00228
   ],
   [
    0.00771,
    -0.00907,
    -0.00017,
    0.00043,
    -0.01148,
    0.00032,
    0.00086,
    0.00745,
    -0.00727,
    0.00135,
    0.00115,
    -0.00935,
    0.00044,
    0.00047
   ],
   [
    0.00454,
    0.00061,
    0.00266,
    0.00116,
    -0.0,
    0.00092,
    0.0009,
    0.0025,
    0.00143,
    0.00158,
    0.00177,
    0.00081,
    0.00177,
    0.00148
   ],
   [
    0.03176,
    0.01499,
    0.01363,
    0.00351,
    0.00291,
    0.00347,
    0.00477,
    0.02,
    0.01306,
    0.0084,
    0.00569,
    0.00574,
    0.00797,
    0.00737
   ],
   [
    -0.00114,
    -0.00058,
    -0.00182,
    -0.0014,
    0.00192,
    0.00144,
    0.00159,
    -0.00093,
    -0.00087,
    -0.00099,
    -0.00122,
    0.00141,
    0.00106,
    0.00162
   ],
   [
    0.00162,
    0.0018,
    -0.0001,
    -0.00296,
    0.00134,
    -0.00032,
    -0.00157,
    0.00069,
    0.001,
    0.0001,
    -0.00308,
    0.00053,
    -0.00027,
    -0.00125
   ],
   [
    0.00705,
    -0.00022,
    -0.00102,
    0.00081,
    -0.00295,
    0.00284,
    0.00047,
    0.00617,
    9e-05,
    -0.001,
    0.00052,
    -0.00239,
    0.00259,
    0.00051
   ],
   [
    -0.00628,
    -0.00187,
    0.00188,
    0.00124,
    -0.00737,
    0.00281,
    -0.00304,
    -0.00286,
    -0.00693,
    -0.00227,
    -0.00043,
    -0.00809,
    -8e-05,
    -0.0046
   ],
   [
    0.00076,
    0.00415,
    -0.0002,
    -0.00246,
    0.0023,
    0.0026,
    0.00328,
    0.0011,
    0.00312,
    0.00013,
    -0.00196,
    0.00254,
    0.00243,
    0.00319
   ],
   [
    -0.00465,
    -0.00786,
    0.00969,
    0.01137,
    -0.0077,
    -0.00621,
    -0.00129,
    -0.0004,
    -0.00719,
    0.00521,
    0.00772,
    -0.00803,
    -0.00793,
    -0.00207
   ],
   [
    0.0123,
    0.00161,
    -0.00443,
    -0.00207,
    -0.00398,
    0.00457,
    0.00627,
    0.00731,
    0.00251,
    -0.00145,
    -0.00172,
    -0.0025,
    0.00513,
    0.00632
   ],
   [
    0.00327,
    0.00498,
    0.0007,
    -0.00094,
    0.00248,
    -0.00165,
    -0.00098,
    0.00249,
    0.00516,
    0.00132,
    -0.00118,
    0.00204,
    -0.0014,
    -0.00041
   ],
   [
    -0.00521,
    0.00616,
    0.001,
    -0.00082,
    -0.00368,
    -0.00013,
    0.00523,
    -0.00222,
    0.00558,
    0.00221,
    0.00035,
    -0.00189,
    0.00054,
    0.00499
   ],
   [
    -0.0006,
    0.00092,
    -0.00066,
    -0.00124,
    0.00167,
    7e-05,
    -0.00307,
    -0.00156,
    -0.00011,
    -0.00106,
    -0.00113,
    0.0014,
    -0.00015,
    -0.00284
   ],
   [
    0.00737,
    -0.00653,
    -0.00536,
    0.00012,
    -0.0031,
    -0.00217,
    -0.00671,
    0.00586,
    -0.00723,
    -0.0031,
    0.0005,
    -0.00274,
    -0.00163,
    -0.00642
   ],
   [
    0.0032,
    -0.00022,
    0.00107,
    0.00394,
    0.00048,
    -0.00041,
    -0.00285,
    0.00448,
    0.00089,
    0.00268,
    0.00459,
    0.00068,
    0.00019,
    -0.00239
   ],
   [
    -0.00243,
    -0.00141,
    0.00031,
    -0.00257,
    -0.002,
    0.00302,
    0.00335,
    -0.00107,
    -0.00175,
    0.00022,
    -0.00234,
    -0.00124,
    0.00257,
    0.00275
   ],
   [
    0.00616,
    0.00129,
    -0.0028,
    -0.00049,
    -0.00163,
    0.00269,
    -0.00107,
    0.00692,
    0.00181,
    -0.00043,
    -4e-05,
    -0.00138,
    0.00253,
    -0.00073
   ],
   [
    0.00665,
    0.00046,
    0.00519,
    0.00242,
    -0.00145,
    -0.00299,
    0.00052,
    0.00471,
    0.00116,
    0.0044,
    0.00314,
    -0.00089,
    -0.00126,
    0.00096
   ],
   [
    -0.00145,
    -0.00257,
    -0.00255,
    -0.00329,
    0.00049,
    -0.00033,
    0.00081,
    -0.00226,
    -0.00372,
    -0.00273,
    -0.00343,
    -0.00027,
    -0.00085,
    0.00041
   ],
   [
    0.00295,
    0.00185,
    0.00059,
    0.00161,
    -0.00197,
    0.00111,
    0.00317,
    0.00291,
    0.00186,
    0.00155,
    0.00201,
    -0.00105,
    0.00144,
    0.00306
   ],
   [
    0.00259,
    -0.00508,
    0.00069,
    0.00119,
    -0.00364,
    0.00027,
    0.00011,
    0.00068,
    -0.00542,
    -0.00095,
    0.00086,
    -0.0032,
    0.00021,
    0.0001
   ],
   [
    -0.00202,
    -0.00366,
    -0.00096,
    0.00111,
    -0.00244,
    0.00086,
    0.00455,
    -0.00063,
    -0.00311,
    -0.00109,
    0.00093,
    -0.00201,
    0.00079,
    0.00384
   ],
   [
    0.00666,
    -0.00208,
    0.00177,
    0.00018,
    0.00088,
    -0.00429,
    -0.00155,
    0.00323,
    -0.00206,
    6e-05,
    0.00048,
    0.00053,
    -0.00366,
    -0.00124
   ],
   [
    0.00131,
    -0.00032,
    -0.00022,
    -0.00041,
    -0.00239,
    0.0009,
    0.00385,
    -0.00089,
    -0.00176,
    -0.0018,
    -0.00035,
    -0.00198,
    0.00061,
    0.003
   ],
   [
    0.00046,
    -0.00275,
    -0.00098,
    0.00069,
    -9e-05,
    -0.00049,
    -0.00264,
    0.00061,
    -0.00261,
    -0.00011,
    0.001,
    -4e-05,
    -0.00024,
    -0.00238
   ],
   [
    0.00241,
    0.0011,
    -0.00281,
    -0.0021,
    0.00334,
    0.0036,
    -2e-05,
    0.00285,
    0.00118,
    -0.0015,
    -0.00175,
    0.00305,
    0.00287,
    -0.00022
   ],
   [
    0.01423,
    -0.00016,
    0.00049,
    0.00388,
    -0.00299,
    -0.0051,
    0.00375,
    0.01158,
    0.00171,
    -0.00024,
    0.00351,
    -0.00174,
    -0.0042,
    0.00308
   ],
   [
    0.00615,
    0.00651,
    -0.00323,
    -0.00082,
    -0.00043,
    -0.00039,
    0.01991,
    0.00896,
    0.00759,
    0.00149,
    0.00084,
    0.0008,
    0.0015,
    0.0186
   ],
   [
    -0.00416,
    -0.00225,
    -0.00234,
    0.00074,
    0.00099,
    -0.0016,
    -0.00106,
    -0.00377,
    -0.00175,
    -0.00226,
    0.00094,
    0.00094,
    -0.00126,
    -0.0006
   ],
   [
    0.01523,
    -0.00316,
    0.00251,
    0.01157,
    0.00247,
    -0.00944,
    -0.01227,
    0.01399,
    -0.00109,
    0.0022,
    0.01192,
    0.00302,
    -0.00535,
    -0.01032
   ],
   [
    -0.0016,
    -0.00281,
    9e-05,
    0.00013,
    0.00019,
    0.0001,
    0.00048,
    -0.00226,
    -0.00224,
    -0.00069,
    -2e-05,
    7e-05,
    0.00061,
    0.00058
   ],
   [
    0.00572,
    -0.00027,
    0.00303,
    0.0008,
    0.00167,
    0.00282,
    -0.0029,
    0.00577,
    -0.00051,
    0.0009,
    0.00074,
    0.00168,
    0.00374,
    -0.00119
   ],
   [
    -0.00542,
    0.00587,
    0.00344,
    -0.00239,
    0.00251,
    0.00062,
    3e-05,
    -0.00161,
    0.00511,
    0.003,
    -0.00157,
    0.00293,
    0.00095,
    0.00071
   ],
   [
    -0.01463,
    -0.00075,
    -0.00511,
    -0.00662,
    -0.00174,
    -7e-05,
    0.00069,
    -0.01254,
    -0.00282,
    -0.00669,
    -0.00755,
    -0.00352,
    -0.00258,
    -0.00151
   ],
   [
    0.00118,
    0.00167,
    -0.00103,
    -0.0015,
    0.00232,
    0.00023,
    -0.00723,
    0.00197,
    0.0017,
    -0.0002,
    -0.0018,
    0.00115,
    -0.00055,
    -0.00646
   ],
   [
    0.00045,
    -0.00094,
    0.00093,
    0.00247,
    -0.00226,
    -0.00029,
    -0.00023,
    0.00076,
    0.00032,
    0.00069,
    0.00246,
    -0.0016,
    0.00031,
    1e-05
   ],
   [
    0.00363,
    0.0036,
    -0.00055,
    -0.00171,
    0.00262,
    0.00146,
    -0.00251,
    0.00099,
    0.00196,
    -0.00136,
    -0.00184,
    0.00143,
    0.00073,
    -0.00243
   ],
   [
    -0.00417,
    -0.00193,
    -0.00233,
    -0.00103,
    0.00681,
    -0.00417,
    -0.00897,
    -0.00521,
    -0.0039,
    -0.00348,
    -0.00145,
    0.00522,
    -0.00462,
    -0.00835
   ],
   [
    -0.0068,
    -0.00694,
    -0.00225,
    -0.00095,
    -0.00139,
    0.00206,
    -0.00066,
    -0.00591,
    -0.00482,
    -0.0012,
    -0.00061,
    -0.00148,
    0.00185,
    -0.00039
   ],
   [
    -0.00023,
    -0.00027,
    0.00097,
    -0.00125,
    0.00025,
    0.00324,
    -0.00074,
    0.00139,
    0.00128,
    0.00195,
    -0.00109,
    0.00054,
    0.00273,
    -0.00028
   ],
   [
    0.0019,
    -1e-05,
    0.00035,
    0.0,
    -0.00064,
    0.0012,
    -0.00019,
    0.00148,
    0.00049,
    0.00045,
    -0.0004,
    -0.00074,
    0.00117,
    -0.00044
   ],
   [
    -0.00086,
    0.00279,
    -0.00074,
    0.00017,
    -0.00016,
    -0.00269,
    -0.00287,
    -0.00123,
    -0.00096,
    -0.00096,
    -0.00054,
    -0.00113,
    -0.00294,
    -0.00297
   ],
   [
    -0.00273,
    0.00194,
    0.00788,
    0.00601,
    -0.00613,
    0.00322,
    0.00622,
    0.00019,
    0.00365,
    0.00915,
    0.00661,
    -0.00314,
    0.00468,
    0.00683
   ],
   [
    -0.00038,
    0.00392,
    0.00063,
    -0.00125,
    0.00178,
    -0.00064,
    -0.0012,
    -0.00162,
    0.00244,
    -0.00052,
    -0.00135,
    0.00155,
    -0.00047,
    -0.00115
   ],
   [
    -0.00334,
    -0.02703,
    -0.00103,
    0.009,
    -0.03048,
    -0.00234,
    -0.01144,
    -0.00454,
    -0.02542,
    -0.00533,
    0.00649,
    -0.02834,
    -0.00504,
    -0.01335
   ],
   [
    -0.00638,
    -0.01058,
    -0.00342,
    -0.00021,
    0.00623,
    -0.00338,
    -0.00605,
    -0.00894,
    -0.01288,
    -0.00648,
    -0.00138,
    0.0043,
    -0.00315,
    -0.00634
   ],
   [
    0.00104,
    -0.00111,
    -0.00091,
    -0.00049,
    0.00077,
    0.00033,
    -0.00088,
    0.00082,
    -0.00059,
    0.00013,
    -0.00028,
    0.00078,
    0.00061,
    -0.00053
   ],
   [
    -0.0001,
    0.0047,
    -0.00082,
    0.0042,
    -0.00204,
    -0.00308,
    0.00304,
    0.00121,
    0.00395,
    -0.00257,
    0.00404,
    -0.0008,
    -0.00276,
    0.00226
   ],
   [
    -0.00863,
    0.00067,
    -0.00082,
    -0.00176,
    0.00173,
    -0.00118,
    -0.00064,
    -0.00619,
    0.00062,
    -0.00118,
    -0.00151,
    0.00166,
    -0.00118,
    -0.00095
   ],
   [
    -0.01112,
    0.00236,
    -0.00575,
    -0.0029,
    0.00554,
    -0.00871,
    -0.00021,
    -0.01471,
    0.00049,
    -0.00834,
    -0.00286,
    0.00226,
    -0.00861,
    -0.00113
   ],
   [
    -0.00339,
    -0.00261,
    0.00353,
    -0.00018,
    -0.00086,
    -0.00035,
    0.00104,
    -0.00192,
    -0.00022,
    0.00341,
    -1e-05,
    -0.00028,
    -5e-05,
    0.00104
   ],
   [
    -0.00046,
    0.00218,
    0.00229,
    0.00021,
    0.00243,
    0.00013,
    -0.00264,
    -0.00065,
    0.00165,
    0.00341,
    0.00171,
    0.00248,
    0.00115,
    -0.00143
   ],
   [
    -0.00051,
    0.00062,
    -0.00164,
    0.00121,
    0.00147,
    -0.00042,
    -0.00309,
    -0.00044,
    0.00028,
    -0.00151,
    0.00156,
    0.00153,
    0.00026,
    -0.00247
   ],
   [
    0.01172,
    0.01167,
    -8e-05,
    -0.00867,
    0.01096,
    0.00493,
    0.00079,
    0.00692,
    0.00991,
    -0.00083,
    -0.00711,
    0.00898,
    0.00375,
    0.00077
   ],
   [
    -0.00135,
    0.00676,
    0.00019,
    -0.00362,
    0.00154,
    0.00226,
    0.00343,
    0.00015,
    0.00769,
    0.00252,
    -0.00304,
    0.00203,
    0.00237,
    0.00309
   ],
   [
    -0.00336,
    -0.00284,
    -0.00097,
    -0.0003,
    0.00062,
    -5e-05,
    0.00064,
    -0.00352,
    -0.00245,
    -0.00122,
    -0.00013,
    0.0005,
    0.00036,
    0.0006
   ],
   [
    -0.01065,
    0.01103,
    -0.00147,
    -0.00927,
    0.00312,
    -0.00066,
    0.00583,
    -0.00575,
    0.01048,
    0.00248,
    -0.00877,
    0.00257,
    -0.00035,
    0.00505
   ],
   [
    0.00439,
    0.00334,
    -0.00176,
    0.00334,
    -0.0036,
    0.0003,
    0.00121,
    0.00526,
    0.00098,
    -0.00096,
    0.00252,
    -0.00378,
    -0.00064,
    0.00027
   ],
   [
    0.00439,
    -0.00827,
    0.00257,
    0.00869,
    -0.00628,
    -0.00667,
    0.0026,
    -0.00026,
    -0.00839,
    -0.00198,
    0.00837,
    -0.00616,
    -0.00643,
    0.00103
   ],
   [
    -0.00382,
    -0.00349,
    -0.00295,
    -0.00134,
    -0.00052,
    0.002,
    0.0016,
    -0.00315,
    -0.00348,
    -0.00243,
    -0.00084,
    -0.00025,
    0.0024,
    0.00156
   ],
   [
    -0.0004,
    0.0012,
    0.00014,
    -0.00168,
    0.00025,
    0.0029,
    0.00096,
    -8e-05,
    0.00216,
    0.00053,
    -0.00111,
    0.00053,
    0.00283,
    0.00107
   ],
   [
    -0.00328,
    -0.00165,
    -0.0024,
    -0.00213,
    0.00407,
    0.00372,
    -0.01011,
    -0.00124,
    -0.00283,
    -0.00124,
    -0.0025,
    0.00224,
    0.00254,
    -0.00904
   ],
   [
    0.0043,
    0.00293,
    0.00464,
    0.00147,
    0.00011,
    5e-05,
    0.00128,
    0.00414,
    0.0031,
    0.00307,
    0.0018,
    0.00095,
    0.00098,
    0.00162
   ],
   [
    -0.00165,
    0.00282,
    -0.00097,
    -0.00202,
    -4e-05,
    0.00159,
    0.00219,
    -0.00135,
    0.00197,
    -0.00044,
    -0.00159,
    2e-05,
    0.00166,
    0.00236
   ],
   [
    -0.00167,
    -0.02385,
    0.00584,
    0.01113,
    -0.00838,
    -0.00253,
    -0.00945,
    -0.00446,
    -0.02093,
    0.00161,
    0.01033,
    -0.00861,
    -0.00399,
    -0.00943
   ],
   [
    0.01498,
    -0.00536,
    0.00382,
    -0.01182,
    -0.01042,
    0.01639,
    0.01118,
    0.0122,
    -0.00061,
    0.00368,
    -0.01132,
    -0.00729,
    0.01463,
    0.00977
   ],
   [
    0.00099,
    -0.00239,
    -0.00108,
    0.0012,
    -0.00289,
    0.00025,
    0.00068,
    -9e-05,
    -0.00286,
    -0.0004,
    0.00116,
    -0.00274,
    -0.00019,
    0.00039
   ],
   [
    0.01606,
    0.00914,
    -0.00136,
    0.00335,
    -0.00623,
    0.00097,
    0.0017,
    0.01666,
    0.01147,
    0.00362,
    0.00274,
    -0.00424,
    0.00089,
    0.00162
   ],
   [
    -0.00611,
    -0.00093,
    0.0006,
    -0.00518,
    -0.0072,
    0.00449,
    0.00568,
    -0.00248,
    -0.00138,
    0.00151,
    -0.00538,
    -0.00638,
    0.00432,
    0.00496
   ],
   [
    0.0026,
    3e-05,
    0.00425,
    0.00241,
    -0.00179,
    -0.00366,
    0.00167,
    0.00285,
    0.00173,
    0.00347,
    0.00244,
    -0.00105,
    -0.00311,
    0.0011
   ],
   [
    -0.00369,
    0.00013,
    -0.00265,
    -0.00158,
    0.00282,
    -0.0015,
    -0.00201,
    -0.00215,
    0.00048,
    -0.00253,
    -0.00188,
    0.00249,
    -0.00136,
    -0.0014
   ],
   [
    0.00044,
    0.00345,
    -0.00096,
    0.00102,
    0.00265,
    -0.00056,
    -0.00138,
    -0.00119,
    0.00164,
    -0.00044,
    0.00081,
    0.00206,
    -8e-05,
    -0.00072
   ],
   [
    0.00708,
    -0.00835,
    0.00205,
    -0.01866,
    -0.02646,
    0.02151,
    0.01873,
    0.00945,
    -0.00492,
    0.00287,
    -0.01858,
    -0.02102,
    0.0186,
    0.01622
   ],
   [
    -0.00063,
    -0.00745,
    -0.00255,
    0.00028,
    0.0014,
    -0.00062,
    0.00175,
    -0.00137,
    -0.00721,
    -0.00269,
    -6e-05,
    -3e-05,
    -0.00081,
    0.00134
   ],
   [
    3e-05,
    -0.00203,
    0.0014,
    0.00055,
    -0.00076,
    0.00141,
    0.00178,
    -0.00018,
    -0.00145,
    0.00105,
    0.00111,
    -0.00028,
    0.00183,
    0.00177
   ],
   [
    0.00176,
    -0.00219,
    -0.00031,
    0.00017,
    -0.0013,
    -0.00092,
    0.00018,
    0.00035,
    -0.00096,
    -0.00011,
    0.0003,
    -0.00078,
    -0.00086,
    0.00037
   ],
   [
    -0.00378,
    -0.00478,
    -0.0015,
    0.00169,
    -0.00173,
    0.00173,
    0.00253,
    -0.00424,
    -0.00444,
    -0.00125,
    0.00154,
    -0.00162,
    0.00161,
    0.00215
   ],
   [
    0.00035,
    -0.00233,
    -0.00184,
    0.00048,
    0.00041,
    9e-05,
    -0.00317,
    0.00096,
    -0.00021,
    0.00052,
    0.00069,
    0.00083,
    0.00151,
    -0.00264
   ],
   [
    0.00206,
    0.00536,
    0.00865,
    0.00641,
    -0.00876,
    0.00027,
    0.00411,
    0.0043,
    0.00321,
    0.00737,
    0.00665,
    -0.00588,
    -0.00057,
    0.00314
   ],
   [
    -0.00336,
    -0.00249,
    -0.00062,
    -0.00179,
    -0.0014,
    0.0009,
    0.00066,
    -0.00278,
    -0.00239,
    -0.00074,
    -0.00164,
    -0.00111,
    0.00058,
    0.00042
   ],
   [
    -0.02734,
    -0.0078,
    -0.01072,
    -0.00577,
    -0.00552,
    0.00649,
    0.00539,
    -0.02107,
    -0.00534,
    -0.00353,
    -0.00527,
    -0.00527,
    0.00595,
    0.00521
   ],
   [
    -0.0017,
    -0.00509,
    0.0034,
    -0.0001,
    -0.00335,
    3e-05,
    0.0028,
    -0.00294,
    -0.00461,
    0.0017,
    0.00023,
    -0.00234,
    -0.00019,
    0.00235
   ],
   [
    -0.00017,
    -0.00434,
    -0.00118,
    0.00051,
    0.00184,
    -0.00188,
    -0.00126,
    -0.00192,
    -0.00291,
    -0.00227,
    0.00066,
    0.00119,
    -0.00188,
    -0.00103
   ],
   [
    0.00506,
    -0.00086,
    -0.00056,
    -0.00037,
    0.00542,
    -0.00351,
    -0.00386,
    0.00449,
    -0.00319,
    -0.00287,
    -0.00119,
    0.00398,
    -0.00333,
    -0.00375
   ],
   [
    -0.00534,
    0.00078,
    -0.00096,
    -0.00368,
    0.0035,
    0.00162,
    0.00231,
    -0.00525,
    0.00243,
    -0.00018,
    -0.00292,
    0.00349,
    0.00175,
    0.00258
   ],
   [
    -0.00551,
    -0.00148,
    -0.00031,
    -0.00236,
    0.00125,
    -0.00023,
    -0.00182,
    -0.00382,
    -0.00029,
    -0.00087,
    -0.00266,
    0.00079,
    -0.00054,
    -0.00166
   ],
   [
    0.0002,
    0.00064,
    0.0009,
    0.00047,
    -0.00046,
    -0.00087,
    -0.00015,
    -0.00067,
    0.00094,
    -0.00018,
    0.0002,
    -0.0005,
    -0.00108,
    -0.00036
   ],
   [
    -0.00073,
    0.00043,
    -0.0017,
    0.00185,
    -0.0013,
    -0.00086,
    -0.00077,
    0.0007,
    -0.00058,
    -0.0007,
    0.00126,
    -0.00158,
    -0.00117,
    -0.00094
   ],
   [
    -7e-05,
    -0.00014,
    -0.00091,
    0.00108,
    -0.00069,
    0.0007,
    0.00251,
    0.0007,
    0.00101,
    -0.00057,
    0.00149,
    -0.00011,
    0.00093,
    0.00253
   ],
   [
    0.00231,
    0.00141,
    5e-05,
    0.00111,
    -0.00012,
    0.00185,
    0.00034,
    0.0033,
    0.00268,
    0.00065,
    0.00106,
    0.00035,
    0.00198,
    0.00093
   ],
   [
    -0.0238,
    -0.00543,
    -0.00238,
    0.00017,
    0.00805,
    -0.01069,
    -0.00327,
    -0.01822,
    -0.00601,
    -0.00233,
    0.0016,
    0.00659,
    -0.00941,
    -0.00299
   ],
   [
    -0.00278,
    0.00233,
    0.00011,
    -0.00161,
    0.00221,
    -0.00028,
    0.00117,
    -0.00229,
    0.00127,
    0.00047,
    -0.00101,
    0.00215,
    0.0003,
    0.00126
   ],
   [
    -0.00697,
    -0.002,
    -0.00244,
    -0.00037,
    -0.00179,
    0.00111,
    0.00155,
    -0.00649,
    -0.00404,
    -0.00228,
    -0.00071,
    -0.0019,
    0.00037,
    0.00081
   ],
   [
    -0.00408,
    -0.01294,
    -0.00883,
    0.00068,
    -0.00598,
    0.0043,
    -0.00289,
    -0.00532,
    -0.01407,
    -0.00841,
    -0.00075,
    -0.00727,
    0.00174,
    -0.00421
   ],
   [
    -0.00019,
    0.0022,
    0.00226,
    -0.00215,
    -0.00057,
    -6e-05,
    -0.00125,
    -0.00162,
    0.00313,
    0.00105,
    -0.00198,
    -0.00038,
    3e-05,
    -0.00124
   ],
   [
    0.00237,
    0.00145,
    0.00198,
    0.00153,
    0.00051,
    -0.00085,
    -0.00075,
    0.00162,
    0.00155,
    0.00135,
    0.00147,
    0.0003,
    -0.00032,
    -0.00053
   ],
   [
    0.00426,
    0.00336,
    0.00118,
    -0.00214,
    0.00057,
    -0.00108,
    -0.00093,
    0.0026,
    0.00258,
    -0.00036,
    -0.00191,
    0.0005,
    -0.00122,
    -0.00101
   ],
   [
    -0.00085,
    -0.00049,
    -0.00112,
    0.00012,
    -0.002,
    0.00083,
    0.00096,
    0.00162,
    -0.00088,
    -0.00012,
    -3e-05,
    -0.00141,
    0.00074,
    0.00079
   ],
   [
    0.00068,
    0.00321,
    0.0037,
    -0.00244,
    0.00647,
    0.00627,
    0.00434,
    -0.00015,
    0.00285,
    0.00328,
    -0.00278,
    0.006,
    0.00715,
    0.00528
   ],
   [
    -0.00543,
    0.02045,
    -0.00088,
    0.00438,
    -0.00225,
    -9e-05,
    0.00038,
    0.00327,
    0.02345,
    0.00578,
    0.00318,
    -0.00167,
    -0.00132,
    0.00023
   ],
   [
    -0.00029,
    -0.00043,
    -0.00071,
    0.00012,
    0.00417,
    -0.00021,
    -0.00372,
    -0.00127,
    3e-05,
    -0.00051,
    0.00082,
    0.00395,
    0.0,
    -0.00284
   ],
   [
    0.00403,
    -0.00042,
    -0.0075,
    0.00347,
    -0.00581,
    -0.00254,
    0.00172,
    0.00358,
    -0.00346,
    -0.00798,
    0.00235,
    -0.00559,
    -0.00376,
    0.00041
   ],
   [
    0.00194,
    0.00281,
    0.00152,
    0.00242,
    -0.00031,
    -0.00304,
    0.0024,
    0.00221,
    0.00275,
    0.0018,
    0.00276,
    0.0006,
    -0.00201,
    0.00236
   ],
   [
    0.00411,
    3e-05,
    -0.0028,
    0.00137,
    -0.0016,
    0.00253,
    0.0033,
    0.00494,
    0.00139,
    -0.0008,
    0.00145,
    -0.0009,
    0.00305,
    0.0033
   ],
   [
    0.00593,
    0.01776,
    -0.00033,
    -0.00183,
    0.01382,
    -0.01575,
    0.00569,
    0.00222,
    0.01627,
    -0.00351,
    -0.00198,
    0.01227,
    -0.01356,
    0.00474
   ],
   [
    -0.00052,
    -0.00108,
    -0.00026,
    -3e-05,
    -0.00279,
    0.00192,
    0.00036,
    -0.00075,
    -0.00021,
    -0.00021,
    0.00059,
    -0.00159,
    0.00262,
    0.00044
   ],
   [
    0.00135,
    0.00079,
    0.00412,
    0.00125,
    0.00614,
    -0.00909,
    -0.00068,
    -0.00293,
    -0.00359,
    2e-05,
    -0.00052,
    0.00384,
    -0.00938,
    -0.00194
   ],
   [
    0.0007,
    0.00113,
    0.00037,
    0.00081,
    -0.00017,
    0.00063,
    0.00156,
    0.00137,
    0.00095,
    0.00058,
    0.00143,
    0.00055,
    0.00143,
    0.002
   ],
   [
    -0.0034,
    0.00174,
    -0.00103,
    -0.00343,
    0.00323,
    0.00127,
    0.00066,
    -0.00138,
    0.00236,
    0.00088,
    -0.00261,
    0.00316,
    0.00124,
    0.00082
   ],
   [
    -0.0023,
    0.00079,
    -0.00169,
    8e-05,
    -0.00638,
    0.00298,
    0.00804,
    0.00125,
    -0.00251,
    -0.00244,
    0.00096,
    -0.00361,
    0.00354,
    0.00742
   ],
   [
    -0.00276,
    2e-05,
    -0.00202,
    0.00051,
    -0.00033,
    -0.00078,
    0.0005,
    -0.002,
    -0.00049,
    -0.00206,
    0.00013,
    -0.0003,
    -0.00099,
    -0.00014
   ],
   [
    0.00045,
    0.01256,
    -0.0042,
    -0.00341,
    -0.00141,
    0.00277,
    0.00018,
    0.00275,
    0.01223,
    0.00132,
    -0.00407,
    -0.00197,
    0.00118,
    -0.00018
   ],
   [
    -0.00144,
    -0.00449,
    0.00479,
    0.00094,
    0.00246,
    -0.00554,
    -0.00628,
    -0.00361,
    -0.00372,
    0.00232,
    0.0009,
    0.00183,
    -0.00559,
    -0.00574
   ],
   [
    -0.00218,
    0.00526,
    0.00204,
    0.00127,
    0.00078,
    -0.00348,
    -0.00241,
    -0.00201,
    0.00291,
    0.00098,
    0.0003,
    -4e-05,
    -0.0034,
    -0.00245
   ],
   [
    -0.00313,
    -0.00412,
    0.00417,
    0.00412,
    -0.00594,
    0.00135,
    -0.00217,
    -0.00277,
    -0.00047,
    0.00312,
    0.00293,
    -0.0049,
    0.00129,
    -0.00175
   ],
   [
    -0.00062,
    0.00076,
    -0.00251,
    -0.00144,
    0.00227,
    -0.00033,
    -0.00076,
    -0.00013,
    0.00107,
    -0.00142,
    -0.00101,
    0.002,
    -0.00036,
    -0.0007
   ],
   [
    -0.00313,
    0.00215,
    -0.00094,
    -0.00192,
    -0.0016,
    0.00317,
    0.0008,
    -0.00356,
    0.00146,
    -0.00092,
    -0.00142,
    -0.00088,
    0.00291,
    0.00075
   ],
   [
    -0.00311,
    0.01745,
    -0.00448,
    -0.00016,
    0.00718,
    0.00335,
    -0.00937,
    0.00072,
    0.01724,
    -0.00372,
    -0.00157,
    0.00642,
    0.00345,
    -0.00735
   ],
   [
    -5e-05,
    -0.00059,
    -0.00056,
    0.00065,
    -0.00292,
    -0.00151,
    0.00361,
    -3e-05,
    -0.00044,
    -0.0006,
    0.00037,
    -0.00227,
    -0.00168,
    0.00263
   ],
   [
    0.00118,
    0.00257,
    0.00054,
    -0.00132,
    0.00146,
    0.00152,
    0.00119,
    0.00157,
    0.00264,
    -5e-05,
    -0.00126,
    0.00164,
    0.00107,
    0.00094
   ],
   [
    0.00198,
    -0.00052,
    0.00012,
    -0.00028,
    0.00019,
    0.00029,
    -0.00026,
    0.00249,
    -0.00031,
    0.0,
    0.00015,
    0.00039,
    0.00072,
    5e-05
   ],
   [
    -0.00147,
    -0.00015,
    0.00101,
    5e-05,
    -0.00137,
    0.00201,
    0.00037,
    -0.0029,
    -0.00121,
    -0.00041,
    -0.00019,
    -0.00135,
    0.00167,
    0.00017
   ],
   [
    0.00224,
    -0.00255,
    0.00149,
    -0.00031,
    0.00098,
    0.00015,
    0.00198,
    -0.00023,
    -0.00363,
    -0.00062,
    0.00029,
    0.00109,
    0.00054,
    0.002
   ],
   [
    0.00017,
    -0.00027,
    -0.00289,
    -0.00064,
    0.00067,
    0.00113,
    0.00141,
    0.00016,
    -0.00028,
    -0.00154,
    -0.00067,
    0.0006,
    0.00099,
    0.00131
   ],
   [
    0.00323,
    0.00698,
    -0.00197,
    0.00509,
    -0.00777,
    0.00855,
    0.00109,
    0.00523,
    0.00801,
    0.00199,
    0.00409,
    -0.00607,
    0.00716,
    0.00058
   ],
   [
    0.00326,
    0.00168,
    0.00027,
    0.0007,
    -0.00283,
    0.00072,
    0.00275,
    0.00317,
    0.00274,
    0.00103,
    0.00071,
    -0.00202,
    0.00121,
    0.00249
   ],
   [
    0.00843,
    -0.00798,
    0.0011,
    0.01019,
    -0.00558,
    -0.00275,
    -0.00101,
    0.00905,
    -0.00518,
    -0.00141,
    0.00703,
    -0.00567,
    -0.00301,
    -0.00134
   ],
   [
    -0.0042,
    -0.00538,
    -0.00459,
    0.00123,
    -0.00396,
    0.00205,
    0.00771,
    -0.00044,
    -0.00351,
    -0.00383,
    0.00134,
    -0.00372,
    0.00261,
    0.00659
   ],
   [
    -0.00595,
    -0.00226,
    -0.0047,
    -0.00053,
    -0.00112,
    0.00039,
    0.00154,
    -0.00496,
    -0.00304,
    -0.00276,
    -0.00072,
    -0.00145,
    -4e-05,
    0.00101
   ],
   [
    0.0015,
    0.00305,
    0.00346,
    -0.00013,
    0.0013,
    -0.00105,
    0.0008,
    0.00146,
    0.00278,
    0.00208,
    -0.00089,
    0.00095,
    -0.00125,
    0.00041
   ],
   [
    -0.0008,
    0.00229,
    0.0005,
    0.00149,
    0.00317,
    -0.0001,
    -0.00439,
    -0.00125,
    0.00177,
    0.00065,
    0.00144,
    0.00272,
    0.00019,
    -0.00331
   ],
   [
    0.00381,
    -0.00029,
    0.00262,
    0.00219,
    -0.00117,
    -0.00116,
    0.00159,
    0.00069,
    -0.0004,
    0.00075,
    0.00249,
    -0.00055,
    -0.00087,
    0.00167
   ],
   [
    0.00314,
    0.00472,
    -0.00183,
    0.00291,
    0.00302,
    -0.00384,
    -0.0023,
    0.00279,
    0.0044,
    -0.00136,
    0.00293,
    0.00261,
    -0.0031,
    -0.00145
   ],
   [
    -0.01532,
    -0.00984,
    -0.00449,
    -0.00615,
    0.00223,
    0.00735,
    -0.00911,
    -0.00921,
    -0.00614,
    0.00054,
    -0.00766,
    0.00123,
    0.00576,
    -0.00775
   ],
   [
    -0.00127,
    -0.00093,
    -0.00202,
    -0.00081,
    -0.00071,
    -0.00103,
    -0.00086,
    -0.00104,
    -0.00216,
    -0.00153,
    -0.00113,
    -0.00109,
    -0.00114,
    -0.0009
   ],
   [
    0.00056,
    0.00426,
    0.00294,
    -0.0034,
    0.00166,
    -0.00057,
    0.0006,
    0.0004,
    0.00302,
    0.00138,
    -0.00344,
    0.00156,
    -0.00039,
    0.00033
   ],
   [
    0.00255,
    0.00264,
    0.00208,
    0.00159,
    -0.00575,
    0.00171,
    0.00237,
    0.00128,
    0.00324,
    0.00119,
    0.00114,
    -0.00453,
    0.00192,
    0.00171
   ],
   [
    0.00083,
    -0.00074,
    -0.00072,
    0.00152,
    2e-05,
    -0.00055,
    -0.00163,
    0.00223,
    -0.0014,
    -6e-05,
    0.00124,
    -0.00027,
    -0.00023,
    -0.00127
   ],
   [
    -0.00068,
    -0.00087,
    -0.0044,
    -0.00341,
    -0.00661,
    0.00785,
    0.008,
    0.00676,
    0.00193,
    0.0002,
    -0.00337,
    -0.00461,
    0.00638,
    0.00701
   ],
   [
    -0.00124,
    -0.00156,
    0.00077,
    0.00013,
    -0.00052,
    -0.00141,
    0.00194,
    -0.00157,
    -0.00094,
    -0.00034,
    -0.00019,
    -0.00087,
    -0.00125,
    0.00188
   ],
   [
    -0.024,
    0.00347,
    0.02618,
    0.00083,
    0.01267,
    0.01654,
    0.01328,
    -0.01677,
    0.01071,
    0.02418,
    0.00705,
    0.01703,
    0.02033,
    0.01547
   ],
   [
    -0.00124,
    -0.00143,
    0.00038,
    -0.00015,
    0.00218,
    -0.00321,
    -0.00059,
    -0.00118,
    -0.00148,
    7e-05,
    0.0,
    0.00203,
    -0.00327,
    -0.00047
   ],
   [
    0.0031,
    -0.00203,
    0.00055,
    -0.00159,
    0.00159,
    0.00059,
    -0.00207,
    0.00239,
    -0.0018,
    0.00106,
    -0.00121,
    0.00141,
    0.00041,
    -0.00178
   ],
   [
    0.00279,
    0.00376,
    6e-05,
    0.00111,
    0.00182,
    -0.00128,
    -0.00203,
    0.00177,
    0.00644,
    0.00193,
    0.00163,
    0.00232,
    -0.00087,
    -0.00156
   ],
   [
    0.00258,
    0.00472,
    0.00108,
    0.00167,
    0.00307,
    -5e-05,
    -0.00052,
    0.00301,
    0.00587,
    0.00089,
    0.00151,
    0.00313,
    0.0001,
    -0.00026
   ],
   [
    -0.00073,
    -0.00191,
    -0.00016,
    -0.00218,
    -0.00024,
    0.00286,
    -0.00158,
    -0.00068,
    -0.00059,
    0.00045,
    -0.00184,
    -0.00029,
    0.00271,
    -0.00129
   ],
   [
    0.00604,
    0.00091,
    0.00111,
    0.0022,
    -0.00342,
    -0.00115,
    0.00153,
    0.00464,
    -0.00051,
    0.0003,
    0.0015,
    -0.00368,
    -0.00135,
    0.00103
   ],
   [
    -0.00248,
    0.00104,
    0.00075,
    -0.00169,
    0.00066,
    -0.00407,
    -0.00101,
    -0.00471,
    -0.00142,
    -0.0015,
    -0.0026,
    -0.00064,
    -0.005,
    -0.0017
   ],
   [
    -0.00119,
    -0.00666,
    -0.0014,
    -0.00155,
    0.00027,
    7e-05,
    -0.00087,
    -0.00215,
    -0.00754,
    -0.00256,
    -0.00191,
    -0.00068,
    -0.00035,
    -0.00082
   ],
   [
    -0.00058,
    0.00083,
    -0.00033,
    -0.00066,
    -0.00132,
    0.00166,
    0.00125,
    0.00119,
    0.00222,
    0.00098,
    -0.00074,
    -0.0008,
    0.00129,
    0.00115
   ],
   [
    -0.00056,
    -0.00028,
    -0.00059,
    -0.00049,
    0.00127,
    -0.00059,
    -0.00049,
    -0.00042,
    -0.00148,
    -0.00134,
    -0.00036,
    0.0011,
    -0.00084,
    -0.00059
   ],
   [
    -0.00046,
    -0.00645,
    -0.00508,
    -0.01131,
    -0.00408,
    0.00358,
    0.00898,
    -0.00738,
    -0.00846,
    -0.00693,
    -0.01044,
    -0.0043,
    0.00268,
    0.00755
   ],
   [
    -0.00239,
    0.00188,
    -0.0022,
    -0.00334,
    0.00095,
    0.00068,
    -0.00013,
    -0.00353,
    -0.00024,
    -0.00316,
    -0.00331,
    0.00029,
    -6e-05,
    -0.00061
   ],
   [
    -0.00046,
    0.00085,
    -0.0029,
    -0.00154,
    0.00022,
    -0.00054,
    0.00032,
    0.00015,
    0.00016,
    -0.00125,
    -0.00129,
    9e-05,
    -0.00045,
    0.00024
   ],
   [
    0.00091,
    0.00105,
    0.003,
    0.0018,
    -0.00234,
    0.00012,
    0.00269,
    0.00132,
    0.00219,
    0.00224,
    0.00269,
    -0.00098,
    0.00138,
    0.00288
   ],
   [
    -0.00377,
    0.00174,
    -0.00242,
    -0.0021,
    -0.00028,
    0.00202,
    0.00151,
    -0.00307,
    -0.00039,
    -0.00205,
    -0.00196,
    0.00021,
    0.0015,
    0.00107
   ],
   [
    -0.00035,
    -0.00248,
    -0.00306,
    -0.00137,
    -0.00103,
    0.00212,
    0.00145,
    -0.00059,
    -0.00382,
    -0.00327,
    -0.00213,
    -0.0013,
    0.0002,
    0.00028
   ],
   [
    -0.00616,
    -0.0027,
    -0.00735,
    -0.00143,
    0.0159,
    -0.00493,
    -0.01634,
    -0.01014,
    -0.00572,
    -0.00719,
    -0.00162,
    0.01118,
    -0.0056,
    -0.01444
   ],
   [
    -0.00043,
    0.00649,
    -0.00327,
    0.00371,
    0.00164,
    -0.0052,
    -0.00118,
    -0.00086,
    0.00729,
    0.00077,
    0.00463,
    0.00136,
    -0.00408,
    -0.00087
   ],
   [
    -0.00652,
    -0.00348,
    -0.00172,
    -0.00154,
    -0.00243,
    0.00119,
    -0.00054,
    -0.00475,
    -0.00219,
    -0.00084,
    -0.00199,
    -0.00243,
    0.00046,
    -0.0011
   ],
   [
    0.0023,
    0.00144,
    -0.00099,
    -0.00164,
    0.00218,
    0.00058,
    2e-05,
    0.00218,
    0.00228,
    -0.00011,
    -0.00167,
    0.00163,
    0.0003,
    -0.00012
   ],
   [
    0.00602,
    -0.00063,
    -0.00042,
    0.00053,
    -0.00126,
    0.00093,
    -0.00169,
    0.00494,
    -0.00079,
    -0.00076,
    0.00027,
    -0.001,
    0.00052,
    -0.00156
   ],
   [
    -0.00072,
    -0.00359,
    -0.0056,
    -0.00093,
    0.00048,
    -0.00144,
    -0.00176,
    0.00081,
    -0.00378,
    -0.00192,
    -0.00058,
    -0.0001,
    -0.00126,
    -0.00159
   ],
   [
    0.00273,
    0.00094,
    -0.00096,
    0.0001,
    0.00222,
    0.00127,
    -0.00423,
    0.0013,
    -0.0001,
    -0.00116,
    -0.0,
    0.00168,
    0.00094,
    -0.00343
   ],
   [
    -0.00219,
    -0.00261,
    0.00017,
    5e-05,
    0.00119,
    -0.00368,
    -0.00461,
    -0.00447,
    -0.00392,
    -0.00256,
    -0.00051,
    0.00018,
    -0.00516,
    -0.00505
   ],
   [
    -0.0022,
    -0.00056,
    2e-05,
    0.00213,
    0.0016,
    -0.00181,
    -0.00168,
    -0.00147,
    -0.00043,
    -0.00061,
    0.00223,
    0.00152,
    -0.00117,
    -0.00147
   ],
   [
    0.0019,
    0.00214,
    0.00067,
    0.00143,
    0.00186,
    -0.00058,
    -0.00074,
    0.00042,
    0.0013,
    1e-05,
    0.00174,
    0.00193,
    9e-05,
    -0.00026
   ],
   [
    0.00309,
    -0.00428,
    0.00027,
    0.00269,
    0.00013,
    -0.00369,
    -0.00748,
    0.00072,
    -0.00288,
    -0.00129,
    0.00162,
    -0.00102,
    -0.00411,
    -0.00701
   ],
   [
    -0.00464,
    -0.0011,
    -0.00196,
    0.00197,
    -0.00029,
    -0.00288,
    0.00132,
    -0.00296,
    0.00041,
    -0.00194,
    0.00125,
    -0.00046,
    -0.00194,
    0.00114
   ],
   [
    0.02862,
    0.02347,
    0.0101,
    0.00872,
    0.00998,
    -0.00368,
    0.00227,
    0.02667,
    0.02575,
    0.00995,
    0.00911,
    0.01082,
    -0.00081,
    0.00501
   ],
   [
    0.00273,
    0.00284,
    0.00526,
    0.00406,
    0.0044,
    -0.00291,
    -0.00136,
    0.00104,
    0.00479,
    0.00222,
    0.00438,
    0.0049,
    -0.00178,
    -0.00089
   ],
   [
    -0.00107,
    -0.00246,
    0.00128,
    0.00143,
    0.00051,
    -0.00243,
    -0.00205,
    -0.00036,
    -0.00334,
    -5e-05,
    0.00099,
    2e-05,
    -0.00214,
    -0.00228
   ],
   [
    -0.00432,
    0.00111,
    -0.00185,
    -1e-05,
    -0.00012,
    0.00249,
    0.00135,
    -0.00099,
    0.00208,
    0.00036,
    2e-05,
    0.00044,
    0.00217,
    0.00152
   ],
   [
    -0.00906,
    0.00052,
    1e-05,
    0.00017,
    9e-05,
    -0.00141,
    -0.0014,
    -0.0057,
    -0.00085,
    7e-05,
    -0.0006,
    -0.00058,
    -0.00272,
    -0.00177
   ],
   [
    -0.00241,
    -0.00058,
    -0.00184,
    0.00073,
    -0.00013,
    0.00024,
    -0.00109,
    -0.00158,
    -0.00183,
    -0.00052,
    0.00108,
    -0.00031,
    0.00043,
    -0.00102
   ],
   [
    0.02518,
    -0.00254,
    0.00055,
    -0.00424,
    -0.00118,
    0.00597,
    -0.00288,
    0.0169,
    -0.00152,
    -0.00239,
    -0.00247,
    -0.00077,
    0.00534,
    -0.00283
   ],
   [
    -0.0014,
    0.00086,
    0.00231,
    0.00086,
    -0.00055,
    0.00122,
    -0.00024,
    -0.00121,
    0.00077,
    0.00191,
    0.00109,
    -5e-05,
    0.00116,
    4e-05
   ],
   [
    -0.00254,
    -0.00369,
    -0.0018,
    0.00196,
    -0.00201,
    -0.00044,
    -0.0017,
    -0.0016,
    -0.00356,
    -0.00193,
    0.00114,
    -0.00231,
    -0.00073,
    -0.00151
   ],
   [
    0.00509,
    -0.00281,
    -0.00261,
    0.00228,
    -0.00308,
    -0.00016,
    -0.00184,
    0.00174,
    -0.00425,
    -0.00428,
    0.0018,
    -0.00363,
    -0.00079,
    -0.00238
   ],
   [
    -0.00074,
    0.00231,
    0.00103,
    -0.00069,
    0.00089,
    0.0013,
    -0.00041,
    -0.00184,
    0.00226,
    0.00048,
    -0.00048,
    0.00111,
    0.00137,
    -0.00037
   ],
   [
    -0.00142,
    -0.01228,
    -0.00359,
    0.00918,
    -0.00947,
    0.00176,
    -0.00091,
    -0.00054,
    -0.00824,
    -0.00145,
    0.00832,
    -0.00772,
    0.00272,
    -0.00061
   ],
   [
    -0.00077,
    -0.00407,
    0.00417,
    0.00126,
    -0.00128,
    -0.00232,
    -0.00025,
    -0.00013,
    -0.00246,
    0.0028,
    0.00138,
    -0.00056,
    -0.00201,
    -0.00028
   ],
   [
    -0.00607,
    -0.0005,
    -0.0001,
    -0.00269,
    -0.00106,
    -0.00164,
    0.00168,
    -0.00511,
    -0.00069,
    1e-05,
    -0.0028,
    -0.00092,
    -0.00189,
    0.00092
   ],
   [
    0.00822,
    -0.00313,
    -0.00216,
    0.00363,
    -0.00524,
    0.00342,
    0.00387,
    0.01174,
    0.00125,
    0.00118,
    0.00419,
    -0.00208,
    0.00442,
    0.00411
   ],
   [
    -0.00084,
    -0.00179,
    0.00451,
    0.00064,
    -0.00026,
    -0.0044,
    0.00078,
    -0.00127,
    -0.00363,
    0.00313,
    0.00062,
    -0.00057,
    -0.00456,
    4e-05
   ],
   [
    0.00275,
    0.00127,
    0.00042,
    0.00056,
    -0.00064,
    0.00016,
    0.00352,
    0.00156,
    0.00101,
    -0.00012,
    0.00017,
    -0.00035,
    -0.00044,
    0.00228
   ],
   [
    0.0028,
    -0.00267,
    -0.00272,
    -0.00055,
    0.00165,
    0.00072,
    -5e-05,
    0.00371,
    -0.00134,
    -0.00133,
    -0.00029,
    0.00142,
    0.00056,
    4e-05
   ],
   [
    -0.00141,
    -0.00352,
    0.00061,
    0.00148,
    -0.00144,
    -0.00109,
    0.00089,
    -0.00229,
    -0.00315,
    -0.00082,
    0.00123,
    -0.00111,
    -0.0012,
    0.00067
   ],
   [
    0.00196,
    -0.00208,
    0.00148,
    -0.00146,
    0.00017,
    7e-05,
    -0.00311,
    0.00019,
    -0.002,
    -2e-05,
    -0.00127,
    -0.00015,
    -8e-05,
    -0.00262
   ],
   [
    0.02365,
    0.00474,
    0.00255,
    0.01359,
    -0.00935,
    -0.00615,
    0.0026,
    0.01847,
    0.00643,
    0.00237,
    0.0139,
    -0.00631,
    -0.00391,
    0.00313
   ],
   [
    -0.00206,
    -0.00165,
    -0.00012,
    0.00079,
    -0.00027,
    -0.00255,
    -0.00244,
    -0.00093,
    -0.00167,
    0.00023,
    0.00034,
    -0.0007,
    -0.00249,
    -0.00213
   ],
   [
    -0.00543,
    -0.0033,
    0.00228,
    0.00078,
    0.00539,
    -0.00897,
    -0.00211,
    -0.00332,
    -0.00045,
    0.00093,
    -0.0005,
    0.00414,
    -0.00931,
    -0.00226
   ],
   [
    -0.00144,
    -0.00116,
    0.0007,
    -0.00013,
    0.00134,
    8e-05,
    -0.00129,
    -0.00171,
    -0.00125,
    -0.00014,
    -7e-05,
    0.00111,
    0.00011,
    -0.0012
   ],
   [
    0.00144,
    0.00225,
    5e-05,
    0.00163,
    -0.0013,
    -0.00147,
    0.00037,
    0.00214,
    0.00177,
    -0.00058,
    0.00075,
    -0.0014,
    -0.00198,
    -0.00026
   ],
   [
    0.00132,
    -0.00151,
    0.00152,
    0.00093,
    -1e-05,
    -0.00221,
    -0.00207,
    -0.00077,
    -0.00179,
    -1e-05,
    0.00038,
    -0.00086,
    -0.00255,
    -0.00188
   ],
   [
    -0.00101,
    0.00118,
    0.00045,
    -0.00672,
    -0.00203,
    0.00834,
    -0.00148,
    -0.00059,
    0.00048,
    -0.00149,
    -0.00709,
    -0.00274,
    0.00633,
    -0.00191
   ],
   [
    0.00175,
    -0.00074,
    -0.00059,
    0.0019,
    -0.00074,
    -0.00133,
    -0.00157,
    0.00217,
    -0.00073,
    -0.00013,
    0.00254,
    -0.00013,
    -0.00087,
    -0.00157
   ],
   [
    -0.00034,
    0.00057,
    0.00222,
    -0.00268,
    0.00235,
    0.00108,
    -0.00125,
    -0.00068,
    0.00045,
    -0.00072,
    -0.0031,
    0.0009,
    3e-05,
    -0.0013
   ],
   [
    0.00314,
    0.00103,
    0.00311,
    0.00179,
    -0.00209,
    -0.00325,
    -0.00036,
    0.00361,
    0.00028,
    0.00246,
    0.00195,
    -0.00119,
    -0.00234,
    -0.00062
   ],
   [
    0.003,
    -0.00152,
    -0.00653,
    -0.00894,
    -0.00319,
    0.00501,
    0.00414,
    0.00657,
    -0.0051,
    -0.00499,
    -0.00944,
    -0.00394,
    0.00265,
    0.00324
   ],
   [
    0.001,
    0.00308,
    0.00022,
    -0.00114,
    0.00267,
    0.00085,
    0.00053,
    0.00111,
    0.00276,
    0.00014,
    -0.00116,
    0.00248,
    0.00108,
    0.00049
   ],
   [
    0.00129,
    0.00186,
    0.00266,
    0.00184,
    0.00034,
    0.00102,
    0.00322,
    0.00147,
    0.00115,
    0.00174,
    0.00172,
    0.00057,
    0.00075,
    0.00277
   ],
   [
    -0.0113,
    -0.00085,
    -0.00539,
    -0.00853,
    0.00323,
    0.00593,
    -0.00449,
    -0.01243,
    -0.00864,
    -0.00736,
    -0.00887,
    0.00088,
    0.0051,
    -0.00359
   ],
   [
    -0.00123,
    -0.00121,
    0.00215,
    -0.00073,
    6e-05,
    -0.00108,
    0.00155,
    -0.00127,
    -0.00051,
    0.00147,
    -0.00063,
    -0.00047,
    -0.00126,
    0.00121
   ],
   [
    0.00053,
    0.0028,
    -0.00206,
    -0.00355,
    0.00066,
    0.00286,
    0.00101,
    0.00125,
    0.00442,
    -0.00015,
    -0.00276,
    0.00088,
    0.00287,
    0.00124
   ],
   [
    -0.01375,
    -0.00449,
    -0.00258,
    -0.00336,
    0.00306,
    0.00337,
    0.00462,
    -0.00733,
    -2e-05,
    0.00124,
    -0.0036,
    0.00261,
    0.00366,
    0.00461
   ],
   [
    0.00075,
    -0.00268,
    -0.00167,
    -0.00199,
    -0.00285,
    0.00133,
    0.00419,
    0.00031,
    -0.00091,
    -0.00069,
    -0.00201,
    -0.00261,
    0.0007,
    0.00334
   ],
   [
    0.00415,
    0.00308,
    -0.00211,
    -0.0012,
    0.00347,
    -0.00425,
    0.00475,
    0.0044,
    -0.00024,
    -0.0014,
    -0.00105,
    0.00228,
    -0.00604,
    0.00296
   ],
   [
    0.00016,
    -0.00264,
    -0.00127,
    0.00053,
    -0.00335,
    0.00278,
    0.00152,
    -0.00127,
    -0.00344,
    -0.00197,
    0.00044,
    -0.00294,
    0.00195,
    0.00105
   ],
   [
    0.00531,
    -0.00455,
    0.00228,
    0.00054,
    -0.00652,
    0.00503,
    0.00461,
    0.00389,
    -0.00321,
    0.00074,
    0.00033,
    -0.00486,
    0.0041,
    0.0037
   ],
   [
    0.02948,
    0.0041,
    -0.00027,
    0.00639,
    -0.00552,
    -0.00197,
    -0.00154,
    0.02359,
    0.00336,
    -0.00126,
    0.00667,
    -0.00425,
    -0.00155,
    -0.0012
   ],
   [
    0.00208,
    -0.00012,
    0.00012,
    -0.00076,
    0.00215,
    0.00054,
    -0.00506,
    0.00162,
    0.00021,
    -0.00037,
    -0.00065,
    0.00219,
    0.00018,
    -0.0044
   ],
   [
    -0.0013,
    -0.00041,
    0.00108,
    0.00265,
    -0.00341,
    0.00109,
    2e-05,
    -0.00066,
    -0.00036,
    2e-05,
    0.00182,
    -0.00285,
    0.00055,
    -0.00024
   ],
   [
    -0.00726,
    -0.00235,
    -0.00342,
    -0.00064,
    0.00247,
    -0.0011,
    -0.00129,
    -0.00456,
    -0.00309,
    -0.00194,
    -0.00068,
    0.00145,
    -0.00115,
    -0.00106
   ],
   [
    -0.00164,
    -0.0024,
    0.00134,
    0.00079,
    -0.00332,
    -0.00031,
    0.00268,
    -0.00116,
    -0.00243,
    0.00113,
    0.00049,
    -0.00282,
    -0.0003,
    0.00217
   ],
   [
    0.00672,
    -0.00303,
    -0.00154,
    0.00046,
    -0.00576,
    0.00367,
    0.00164,
    0.00666,
    -0.00077,
    -0.00086,
    -0.0007,
    -0.00511,
    0.00338,
    0.00112
   ],
   [
    0.00182,
    0.00011,
    -0.00043,
    -1e-05,
    -0.00194,
    -0.00077,
    0.00247,
    0.0014,
    -4e-05,
    -0.00025,
    0.00048,
    -0.00135,
    -0.00086,
    0.00201
   ],
   [
    -0.00311,
    -0.0012,
    0.00054,
    0.00023,
    -0.00268,
    -2e-05,
    0.00217,
    -0.00277,
    -0.00042,
    -0.00022,
    0.00045,
    -0.00176,
    0.0,
    0.00158
   ],
   [
    0.00999,
    0.00187,
    -0.0009,
    0.00833,
    -0.00023,
    -0.00908,
    0.00173,
    0.01091,
    0.00754,
    0.0022,
    0.00869,
    0.00019,
    -0.00769,
    0.0016
   ],
   [
    0.00263,
    0.00413,
    0.00334,
    -0.00377,
    0.00014,
    0.0044,
    -0.00012,
    0.00474,
    0.00582,
    0.0025,
    -0.00371,
    0.00094,
    0.00373,
    0.00055
   ],
   [
    0.00074,
    -0.00022,
    0.00095,
    -0.00225,
    0.00216,
    -0.00067,
    -0.00348,
    -0.0001,
    0.00044,
    0.00031,
    -0.00253,
    0.00134,
    -0.0009,
    -0.00269
   ],
   [
    -0.00031,
    -0.00075,
    -0.00156,
    0.00174,
    -0.00127,
    -0.00137,
    -0.00123,
    -0.00161,
    -0.00319,
    -0.00193,
    0.00117,
    -0.00147,
    -0.00143,
    -0.00177
   ],
   [
    -0.00304,
    0.0018,
    -0.00054,
    -0.0018,
    0.00233,
    0.00041,
    -0.00184,
    -0.00181,
    0.00166,
    -0.00023,
    -0.00193,
    0.0016,
    -0.00012,
    -0.00192
   ],
   [
    -0.01217,
    -0.01478,
    -0.00508,
    0.0116,
    -0.00331,
    -0.00911,
    -0.01271,
    -0.00939,
    -0.00966,
    -0.00153,
    0.01206,
    -0.00274,
    -0.00711,
    -0.01051
   ],
   [
    0.00296,
    -0.00474,
    -0.00131,
    -0.00073,
    -0.00122,
    0.00536,
    -0.00271,
    0.00418,
    -0.0029,
    -0.00071,
    -0.00123,
    -0.00089,
    0.00413,
    -0.00269
   ],
   [
    0.00148,
    0.00039,
    -0.0005,
    -0.00073,
    0.00092,
    0.00026,
    -0.00069,
    0.00152,
    0.001,
    0.00059,
    -0.00088,
    0.00068,
    0.0003,
    -0.00063
   ],
   [
    -0.00164,
    -0.00144,
    -0.00063,
    0.00103,
    0.00079,
    -0.00048,
    0.00014,
    -0.00358,
    -0.00299,
    -0.00128,
    0.00139,
    0.00076,
    -0.00019,
    0.00035
   ],
   [
    -0.00068,
    0.0009,
    0.00175,
    -0.00115,
    -0.00044,
    -0.00053,
    -0.00058,
    0.00033,
    0.00101,
    0.00172,
    -0.00161,
    -0.00073,
    -0.00058,
    -0.0005
   ],
   [
    0.00427,
    -0.00518,
    0.00158,
    0.00326,
    0.00601,
    -0.01145,
    -0.0098,
    -0.00177,
    -0.00917,
    -0.00622,
    0.00156,
    0.003,
    -0.01174,
    -0.01098
   ],
   [
    -0.00106,
    -0.00336,
    0.00153,
    0.00141,
    -0.00048,
    -0.00052,
    0.00113,
    -0.0005,
    -0.00242,
    0.00098,
    0.00095,
    -0.00067,
    -0.00051,
    0.00074
   ],
   [
    0.0029,
    0.00545,
    -0.006,
    0.00291,
    0.00752,
    0.00104,
    -0.00776,
    0.00241,
    0.00399,
    -0.00491,
    0.00196,
    0.00607,
    0.00098,
    -0.00636
   ],
   [
    0.0011,
    -0.00301,
    -0.00113,
    -8e-05,
    -0.00042,
    0.0003,
    0.00075,
    0.00017,
    -0.00262,
    -0.00119,
    -0.00037,
    -0.00068,
    3e-05,
    0.00021
   ],
   [
    0.00656,
    0.00159,
    0.00102,
    0.00177,
    0.00143,
    -0.00351,
    -0.00017,
    0.0036,
    0.0019,
    0.00031,
    0.00182,
    0.00142,
    -0.00234,
    0.00012
   ],
   [
    0.00487,
    0.00367,
    -0.00035,
    -0.00035,
    0.00625,
    -0.00373,
    -0.00571,
    0.00178,
    0.00379,
    -0.00215,
    -0.00098,
    0.00446,
    -0.00364,
    -0.00532
   ],
   [
    0.00132,
    -0.00033,
    -0.00433,
    -0.00229,
    -0.00017,
    0.00294,
    -0.00181,
    0.00186,
    0.00067,
    -0.00208,
    -0.00196,
    -0.0005,
    0.00219,
    -0.00161
   ],
   [
    -0.00012,
    0.00052,
    -0.00192,
    0.0006,
    -0.00143,
    0.00101,
    0.00161,
    -0.00122,
    -0.00136,
    -0.00286,
    0.00057,
    -0.00117,
    0.00097,
    0.00145
   ],
   [
    0.00652,
    0.0047,
    0.00207,
    0.00041,
    0.00223,
    -0.00011,
    -0.00222,
    0.00523,
    0.00529,
    0.00158,
    3e-05,
    0.00229,
    0.00013,
    -0.00174
   ],
   [
    0.003,
    -0.00125,
    0.00381,
    0.00052,
    0.00132,
    -0.00166,
    -0.00059,
    0.00042,
    -0.00049,
    0.00271,
    0.00105,
    0.0017,
    -0.00095,
    -0.00027
   ],
   [
    0.00167,
    -0.00426,
    -0.00253,
    -0.0011,
    -0.00047,
    -0.00236,
    0.00086,
    4e-05,
    -0.00398,
    -0.00307,
    -0.00121,
    -0.00106,
    -0.00265,
    0.00011
   ],
   [
    1e-05,
    0.00014,
    -0.00096,
    0.00202,
    0.00051,
    0.00023,
    -0.00023,
    0.0002,
    0.00061,
    -0.00105,
    0.00184,
    0.00075,
    0.00011,
    -0.00028
   ],
   [
    0.00301,
    -0.00349,
    0.0002,
    -0.00032,
    -0.0063,
    0.00221,
    0.00016,
    0.00186,
    -0.00242,
    0.00049,
    -0.00044,
    -0.00533,
    0.00184,
    0.00017
   ],
   [
    0.00034,
    0.00456,
    -0.00083,
    -0.00222,
    0.00553,
    -0.00163,
    -0.00448,
    -0.0013,
    0.00221,
    -0.00363,
    -0.00278,
    0.00442,
    -0.00188,
    -0.0046
   ],
   [
    0.00266,
    -0.00165,
    -0.00098,
    -4e-05,
    -0.00096,
    -0.00085,
    7e-05,
    0.00243,
    -0.00227,
    -0.00101,
    -0.00017,
    -0.00115,
    -0.00082,
    4e-05
   ],
   [
    -0.00072,
    -0.0023,
    -0.00184,
    -4e-05,
    -0.00217,
    -0.00052,
    0.00178,
    -0.00175,
    -0.00272,
    -0.00222,
    -6e-05,
    -0.0019,
    -0.00047,
    0.00155
   ],
   [
    0.00477,
    0.00116,
    0.00104,
    0.00112,
    -0.00067,
    -0.00042,
    0.00146,
    0.00585,
    0.00095,
    0.00126,
    0.00048,
    -0.00103,
    -0.00075,
    0.00097
   ],
   [
    -0.00282,
    0.00067,
    -0.00307,
    -0.001,
    0.00234,
    -0.00093,
    -0.00125,
    -0.00483,
    -0.00158,
    -0.00296,
    -0.00131,
    0.00125,
    -0.00119,
    -0.00144
   ],
   [
    0.00379,
    -0.00022,
    0.0002,
    0.00187,
    -0.00068,
    -0.00071,
    0.00179,
    0.00482,
    0.00097,
    0.00114,
    0.00201,
    -0.0001,
    -0.00037,
    0.00187
   ],
   [
    0.00428,
    0.00375,
    -0.00134,
    0.0,
    -0.00115,
    -0.00111,
    0.00218,
    0.00565,
    0.00381,
    -7e-05,
    0.00039,
    -0.00098,
    -0.00061,
    0.00193
   ],
   [
    2e-05,
    0.00335,
    -0.00036,
    -0.00131,
    0.00153,
    0.00068,
    0.00083,
    0.00043,
    0.00229,
    -0.00058,
    -0.00096,
    0.00128,
    0.00072,
    0.00075
   ],
   [
    -0.00477,
    -0.00247,
    -0.00055,
    -0.00225,
    0.00233,
    -0.00099,
    -0.00458,
    -0.00526,
    -0.00399,
    -0.0024,
    -0.00261,
    0.00093,
    -0.00136,
    -0.00418
   ],
   [
    -0.00705,
    0.00147,
    -0.00317,
    -0.00385,
    0.00382,
    0.00191,
    -0.00431,
    -0.00455,
    0.00151,
    -0.00015,
    -0.00285,
    0.00296,
    0.00177,
    -0.00355
   ],
   [
    0.00188,
    -0.00029,
    0.00435,
    0.00215,
    -0.00141,
    -0.00453,
    -0.0033,
    0.00105,
    -0.00118,
    0.00381,
    0.00184,
    -0.00209,
    -0.00389,
    -0.0026
   ],
   [
    -0.00076,
    -0.00148,
    -0.00304,
    0.00038,
    0.00034,
    0.00037,
    0.00137,
    -0.0,
    -0.0004,
    -0.00161,
    0.00057,
    0.00045,
    0.00062,
    0.00124
   ],
   [
    0.00032,
    0.00247,
    -0.00178,
    -0.00077,
    0.00034,
    0.00122,
    0.00151,
    0.00016,
    0.00055,
    -0.00092,
    -0.00079,
    0.00026,
    0.00096,
    0.00133
   ],
   [
    0.00118,
    0.00161,
    0.00066,
    -0.0009,
    0.00119,
    -0.00073,
    -0.00131,
    0.00134,
    0.00067,
    0.00061,
    -0.00094,
    0.00077,
    -0.00092,
    -0.00122
   ],
   [
    0.00354,
    -0.00012,
    -0.00064,
    -0.00297,
    0.00056,
    0.00298,
    0.00118,
    0.00119,
    -0.00099,
    -0.00067,
    -0.00269,
    0.0007,
    0.00265,
    0.00137
   ],
   [
    0.0022,
    0.00305,
    0.00833,
    0.00229,
    -0.00089,
    -0.00049,
    0.00121,
    0.00524,
    0.00538,
    0.00805,
    0.00269,
    0.00028,
    0.00049,
    0.00163
   ],
   [
    0.0029,
    0.0017,
    0.00211,
    -5e-05,
    -0.00054,
    0.00084,
    0.00239,
    0.00198,
    0.00149,
    0.00138,
    0.00024,
    0.0002,
    0.00114,
    0.00222
   ],
   [
    0.00304,
    0.00298,
    0.00132,
    -0.00085,
    0.00114,
    0.00026,
    -0.00103,
    0.00389,
    0.00348,
    0.00187,
    -0.0005,
    0.00154,
    0.00037,
    -0.00071
   ],
   [
    0.00217,
    0.00116,
    -0.00071,
    -0.00047,
    -0.00121,
    0.00163,
    -0.00224,
    0.0021,
    0.00072,
    0.00022,
    -0.00041,
    -0.00112,
    0.00077,
    -0.00212
   ],
   [
    0.01621,
    0.00992,
    0.00808,
    0.00035,
    -0.00232,
    -0.00234,
    -0.00085,
    0.01144,
    0.00652,
    0.00281,
    -8e-05,
    -0.00055,
    -0.00291,
    -0.00103
   ],
   [
    0.0004,
    -0.00362,
    -0.00024,
    0.00108,
    0.00183,
    -0.00103,
    -0.00234,
    9e-05,
    -0.00285,
    -0.00077,
    0.00082,
    0.00113,
    -0.00105,
    -0.00193
   ],
   [
    -0.00694,
    0.00483,
    0.00162,
    0.00238,
    0.00144,
    0.00124,
    0.00106,
    -0.00362,
    0.00538,
    0.00222,
    0.00206,
    0.00196,
    0.00153,
    0.00116
   ],
   [
    0.00319,
    0.00443,
    0.00161,
    0.00341,
    -0.00252,
    0.0028,
    0.00035,
    0.00583,
    0.00529,
    0.00294,
    0.00246,
    -0.00175,
    0.00251,
    0.00113
   ],
   [
    0.0031,
    0.00095,
    0.00293,
    0.00154,
    -0.00782,
    -0.00219,
    0.0042,
    -0.00285,
    -0.00124,
    -0.00285,
    0.0019,
    -0.0069,
    -0.00269,
    0.00267
   ],
   [
    0.00706,
    0.00246,
    -0.00107,
    -0.00048,
    0.00271,
    -0.00093,
    -0.00246,
    0.00411,
    0.00178,
    -0.00071,
    0.00059,
    0.00282,
    -0.00087,
    -0.002
   ],
   [
    -0.00357,
    0.00514,
    0.00096,
    -0.00211,
    0.00131,
    -0.0021,
    0.00071,
    -0.00348,
    0.00362,
    0.00069,
    -0.00157,
    0.00124,
    -0.00187,
    0.00056
   ],
   [
    0.00043,
    -0.00195,
    -0.00179,
    0.00092,
    -0.00261,
    0.00055,
    -0.00088,
    0.00105,
    -0.00127,
    -0.00039,
    0.00065,
    -0.00258,
    0.00014,
    -0.00089
   ],
   [
    0.01302,
    -0.01162,
    -0.00048,
    0.00733,
    -0.0278,
    0.00954,
    0.02835,
    0.01877,
    0.00267,
    0.00902,
    0.01009,
    -0.01895,
    0.01202,
    0.02669
   ],
   [
    -0.00212,
    -0.00056,
    -0.00446,
    -0.00102,
    -0.00101,
    0.00423,
    -0.00058,
    -0.00097,
    -0.00033,
    -0.00241,
    -0.00104,
    -0.00097,
    0.00342,
    -0.00078
   ],
   [
    -0.00255,
    -0.00029,
    -0.00295,
    0.0004,
    -0.00387,
    0.00293,
    0.00052,
    -0.00201,
    -0.00098,
    -0.00204,
    -0.00049,
    -0.00383,
    0.00164,
    -0.00021
   ],
   [
    -0.00129,
    -0.00066,
    2e-05,
    7e-05,
    -0.0015,
    -0.00013,
    0.00196,
    -0.00045,
    0.00062,
    0.00029,
    0.00062,
    -0.00054,
    0.00063,
    0.00164
   ],
   [
    -0.00284,
    0.00207,
    0.0003,
    -0.00111,
    0.00169,
    -0.00028,
    0.00017,
    -0.00143,
    0.00152,
    0.0,
    -0.00122,
    0.0013,
    -0.00045,
    8e-05
   ],
   [
    -0.01352,
    0.00726,
    0.00097,
    0.00479,
    0.00655,
    -0.01691,
    -0.01037,
    -0.00532,
    0.00036,
    -0.00017,
    0.00345,
    0.00309,
    -0.0181,
    -0.01027
   ],
   [
    -0.00538,
    -0.00474,
    -0.00313,
    -0.0011,
    -0.00092,
    0.00071,
    0.00316,
    -0.00268,
    -0.00309,
    -0.00079,
    -0.00121,
    -0.00101,
    0.00085,
    0.00272
   ],
   [
    0.00305,
    -0.00035,
    -0.00022,
    0.0028,
    -0.00083,
    -0.003,
    -0.00108,
    0.00228,
    -0.0005,
    -0.00081,
    0.00233,
    -0.00067,
    -0.00262,
    -0.001
   ],
   [
    -0.00214,
    -0.00765,
    -0.00176,
    0.00223,
    -0.00252,
    0.00145,
    0.00142,
    -0.00163,
    -0.00611,
    -0.00139,
    0.00249,
    -0.002,
    0.00114,
    0.00153
   ],
   [
    0.00158,
    0.00033,
    0.0007,
    -0.00052,
    -0.00041,
    -0.00176,
    0.00082,
    0.00052,
    8e-05,
    0.00041,
    -0.00068,
    -0.00026,
    -0.00153,
    0.00054
   ],
   [
    -0.00148,
    0.00128,
    -0.00109,
    -0.00197,
    0.00042,
    -0.00039,
    -0.00243,
    -0.0032,
    -0.00017,
    -0.00251,
    -0.00187,
    -0.00013,
    -0.00065,
    -0.00215
   ],
   [
    -0.00061,
    -0.00067,
    -0.0017,
    -0.00022,
    -0.0021,
    0.00029,
    0.00053,
    -0.00147,
    -0.0015,
    -0.00147,
    7e-05,
    -0.0016,
    0.00033,
    -5e-05
   ],
   [
    0.00042,
    -0.00164,
    -0.00062,
    -0.00258,
    0.00276,
    0.00084,
    -0.00359,
    0.00033,
    -0.00128,
    -0.00079,
    -0.00257,
    0.00181,
    0.00058,
    -0.00309
   ],
   [
    0.00973,
    0.00536,
    0.00216,
    -0.00082,
    0.00397,
    0.00352,
    0.00107,
    0.01172,
    0.00738,
    0.00197,
    0.00036,
    0.00541,
    0.0045,
    0.00143
   ],
   [
    0.00102,
    0.00047,
    0.00302,
    -0.00296,
    -0.0032,
    0.00204,
    0.00393,
    -0.00039,
    -0.00067,
    0.00138,
    -0.00319,
    -0.00289,
    0.00128,
    0.00274
   ],
   [
    0.00119,
    -0.00106,
    -0.00138,
    -0.0008,
    -0.00131,
    -0.00086,
    0.00025,
    -0.00088,
    -0.00017,
    -0.00141,
    -0.00038,
    -0.00074,
    -0.00015,
    0.00042
   ],
   [
    -0.00062,
    0.00048,
    0.00339,
    0.00148,
    0.001,
    -0.00387,
    -0.0031,
    -0.00012,
    -0.00101,
    0.0006,
    0.00054,
    0.00023,
    -0.00387,
    -0.00314
   ],
   [
    0.00274,
    0.00048,
    -0.00186,
    0.00152,
    -0.00365,
    0.00135,
    0.00689,
    0.00147,
    0.00051,
    -0.00058,
    0.00169,
    -0.00222,
    0.0016,
    0.00616
   ],
   [
    0.004,
    0.00397,
    0.00045,
    -0.0014,
    0.00259,
    0.00172,
    -0.00088,
    0.00267,
    0.00394,
    0.0013,
    -0.00155,
    0.00184,
    0.00165,
    -0.00045
   ],
   [
    0.0147,
    0.01658,
    -0.00178,
    -0.00016,
    0.01644,
    -0.00663,
    -0.01281,
    0.01135,
    0.01616,
    0.00165,
    0.00125,
    0.01291,
    -0.00466,
    -0.00982
   ],
   [
    -0.00272,
    0.00079,
    -0.00086,
    -0.00168,
    -0.00055,
    0.00291,
    0.00037,
    -0.00216,
    0.00039,
    -0.00017,
    -0.00175,
    -0.00087,
    0.00236,
    -0.0001
   ],
   [
    0.00175,
    -0.00094,
    -0.00148,
    -0.00361,
    0.00121,
    -0.00113,
    -0.00124,
    0.00154,
    -0.00168,
    -0.00113,
    -0.00342,
    0.00077,
    -0.00147,
    -0.00169
   ],
   [
    0.00046,
    -0.00489,
    0.00147,
    0.00086,
    -0.00106,
    -0.00279,
    -0.00345,
    -0.00206,
    -0.00526,
    -0.00171,
    0.00049,
    -0.00118,
    -0.00269,
    -0.00349
   ],
   [
    0.00043,
    0.00195,
    0.00143,
    -0.00039,
    0.00017,
    -0.00046,
    0.00035,
    0.00067,
    0.00164,
    0.0008,
    3e-05,
    0.00072,
    5e-05,
    0.00033
   ],
   [
    0.00329,
    0.00706,
    0.00341,
    0.0019,
    0.0008,
    0.00083,
    0.00087,
    0.00655,
    0.00811,
    0.00426,
    0.00223,
    0.00218,
    0.00151,
    0.00166
   ],
   [
    0.00614,
    -0.00356,
    0.0005,
    0.00111,
    -0.0061,
    0.00258,
    0.00779,
    0.00409,
    -0.00467,
    -0.00075,
    0.00175,
    -0.00444,
    0.00297,
    0.00726
   ],
   [
    0.00373,
    -0.00169,
    0.00293,
    0.00166,
    -0.00384,
    0.00186,
    0.00073,
    0.00398,
    -0.0022,
    0.00277,
    0.00125,
    -0.00334,
    0.001,
    0.0005
   ],
   [
    -0.00123,
    -0.00283,
    0.00096,
    -0.00335,
    0.00286,
    0.00193,
    -0.00115,
    -0.00581,
    -0.00262,
    -0.0028,
    -0.00307,
    0.00231,
    0.00146,
    -0.00117
   ],
   [
    -0.00951,
    -0.00163,
    -0.0028,
    0.00124,
    0.00151,
    -0.01136,
    -0.00025,
    -0.0052,
    -0.00241,
    -0.00222,
    0.00089,
    0.0002,
    -0.01023,
    -0.00076
   ],
   [
    -0.00108,
    0.00114,
    -0.00021,
    0.0003,
    0.00074,
    -0.00046,
    -2e-05,
    0.00127,
    0.00136,
    0.00057,
    0.00034,
    0.00092,
    -0.00032,
    5e-05
   ],
   [
    0.00763,
    0.00401,
    0.00573,
    0.00279,
    0.00337,
    -0.00785,
    -0.00559,
    0.0069,
    0.00391,
    0.00274,
    0.00236,
    0.00469,
    -0.00498,
    -0.00436
   ],
   [
    0.00206,
    -0.01064,
    0.00313,
    0.00307,
    -0.00469,
    0.00049,
    0.0077,
    -0.00236,
    -0.01173,
    -0.00101,
    0.0028,
    -0.00433,
    0.0003,
    0.00596
   ],
   [
    0.00342,
    0.00585,
    0.00024,
    0.00257,
    -0.00026,
    -0.00082,
    0.00143,
    0.0032,
    0.0048,
    0.00068,
    0.00338,
    0.00093,
    0.00089,
    0.00195
   ],
   [
    -0.01584,
    0.01583,
    -0.00072,
    -0.00504,
    0.00335,
    -0.00234,
    0.01176,
    -0.01566,
    0.01088,
    -0.0063,
    -0.00661,
    0.00217,
    -0.00112,
    0.00988
   ],
   [
    0.00484,
    0.0056,
    -0.00224,
    0.00565,
    -0.00906,
    -0.00428,
    -0.00837,
    0.0072,
    0.00196,
    0.00089,
    0.00252,
    -0.01066,
    -0.00644,
    -0.00883
   ],
   [
    0.00448,
    0.00321,
    0.00285,
    0.00297,
    0.001,
    -0.00342,
    -0.00171,
    0.00264,
    0.00314,
    0.00075,
    0.0032,
    0.00135,
    -0.00216,
    -0.00117
   ],
   [
    0.00495,
    0.0068,
    0.00349,
    -0.00303,
    0.0058,
    -0.0005,
    -0.00376,
    0.00295,
    0.00398,
    0.00063,
    -0.00383,
    0.0047,
    -0.00147,
    -0.00312
   ],
   [
    0.00656,
    0.00448,
    0.00185,
    0.00151,
    -0.00083,
    -0.0021,
    -0.00511,
    0.00791,
    0.00529,
    0.00398,
    0.00069,
    -0.00107,
    -0.00294,
    -0.00433
   ],
   [
    -0.004,
    -0.00123,
    -0.00032,
    -0.00038,
    -0.00329,
    -0.00064,
    0.00126,
    -0.00266,
    -0.00022,
    0.00065,
    -0.00077,
    -0.00266,
    -0.00106,
    0.00058
   ],
   [
    -0.00234,
    0.00287,
    -0.00055,
    -0.00335,
    0.0001,
    0.00518,
    0.00181,
    0.00082,
    0.00346,
    0.00026,
    -0.00272,
    0.00096,
    0.00513,
    0.00216
   ],
   [
    -0.00513,
    -0.00135,
    0.00288,
    -0.0018,
    -0.00347,
    0.00254,
    0.00878,
    -0.00343,
    -0.00274,
    0.00095,
    -0.00092,
    -0.00171,
    0.0032,
    0.00775
   ],
   [
    -0.00775,
    -0.00086,
    -0.00214,
    0.00202,
    -0.00115,
    0.00145,
    -0.00602,
    -0.00398,
    -0.00026,
    -0.00126,
    0.00202,
    -0.00109,
    0.00187,
    -0.00462
   ],
   [
    0.00128,
    0.00231,
    -0.00116,
    0.00066,
    0.00057,
    -0.00034,
    0.0,
    0.00202,
    0.00235,
    0.0001,
    0.00052,
    0.00041,
    -0.00032,
    0.00019
   ],
   [
    0.00334,
    0.0017,
    0.00285,
    0.00162,
    -0.00062,
    -0.00178,
    0.00226,
    -0.00136,
    -0.00101,
    0.00121,
    0.00123,
    -0.00031,
    -0.00109,
    0.00148
   ],
   [
    0.01404,
    0.01202,
    -0.00113,
    0.00175,
    -0.00742,
    0.00208,
    0.0013,
    0.01334,
    0.00986,
    0.00062,
    0.00088,
    -0.00604,
    0.00088,
    0.00131
   ],
   [
    -0.00213,
    -0.00361,
    0.00142,
    0.001,
    -0.00075,
    0.00011,
    -0.00013,
    -0.00209,
    -0.00337,
    0.00086,
    0.00126,
    -0.00051,
    0.00018,
    0.00013
   ],
   [
    -0.00187,
    -0.0005,
    -0.00086,
    -0.00117,
    0.00067,
    0.00078,
    -0.00144,
    -0.0018,
    -0.0011,
    -0.00065,
    -0.00172,
    2e-05,
    0.00013,
    -0.00164
   ],
   [
    -0.00122,
    0.00502,
    0.00091,
    -0.00186,
    0.00239,
    0.00285,
    0.00181,
    -0.00044,
    0.00455,
    0.0008,
    -0.00198,
    0.00249,
    0.0026,
    0.00198
   ],
   [
    0.00015,
    0.00211,
    -0.00464,
    -0.00068,
    0.00176,
    0.0001,
    -0.00121,
    0.00091,
    0.00215,
    -0.00287,
    -0.00104,
    0.00094,
    0.0,
    -0.00116
   ],
   [
    0.00694,
    0.00077,
    -0.00205,
    0.00198,
    0.00336,
    0.00033,
    -0.00543,
    0.00748,
    0.00297,
    -0.00125,
    0.00222,
    0.00331,
    0.00065,
    -0.00487
   ],
   [
    -0.0141,
    -0.01252,
    -0.00024,
    -0.00625,
    -0.00445,
    0.00553,
    0.01155,
    -0.01625,
    -0.01414,
    -0.0024,
    -0.00459,
    -0.00309,
    0.0055,
    0.00963
   ],
   [
    0.00092,
    0.0011,
    0.00149,
    0.00052,
    -0.00199,
    -0.00157,
    0.00122,
    -0.00056,
    0.0005,
    0.00095,
    0.00029,
    -0.00175,
    -0.00185,
    0.00071
   ],
   [
    -0.00024,
    0.00434,
    0.00021,
    -0.00147,
    -0.00035,
    0.00074,
    0.00326,
    -0.00176,
    0.0036,
    -7e-05,
    -0.00155,
    -3e-05,
    0.00041,
    0.00241
   ],
   [
    0.00181,
    -0.00124,
    -0.00153,
    0.00067,
    -0.00293,
    0.00349,
    0.00049,
    0.00158,
    -0.00164,
    -0.00084,
    0.00016,
    -0.00316,
    0.00249,
    -0.00011
   ],
   [
    -0.00581,
    -0.00527,
    -0.00496,
    0.00306,
    -0.00345,
    0.00083,
    -0.00279,
    -0.00442,
    -0.00377,
    -0.00065,
    0.00286,
    -0.00332,
    0.00063,
    -0.00221
   ],
   [
    0.01191,
    -0.00251,
    0.00198,
    0.00915,
    -0.00856,
    0.00062,
    -0.00111,
    0.00815,
    -0.00202,
    0.00061,
    0.00866,
    -0.00695,
    -0.00023,
    -0.00167
   ],
   [
    0.00765,
    0.00248,
    0.00237,
    0.00176,
    -6e-05,
    -0.00035,
    -0.00165,
    0.00511,
    0.00159,
    0.00016,
    0.00151,
    -0.00011,
    -0.00018,
    -0.00132
   ],
   [
    -0.01328,
    0.00838,
    -0.00538,
    -0.00518,
    0.01562,
    -0.01069,
    -0.00906,
    -0.01378,
    0.00012,
    -0.00783,
    -0.00573,
    0.01072,
    -0.01253,
    -0.00997
   ],
   [
    3e-05,
    5e-05,
    0.00027,
    -0.00067,
    -0.00099,
    0.00136,
    0.00147,
    0.00011,
    0.00045,
    0.00062,
    -0.00064,
    -0.00094,
    0.00131,
    0.00108
   ],
   [
    -0.00373,
    -0.00245,
    -0.00106,
    -0.00125,
    -0.00234,
    0.00149,
    0.00054,
    -0.0035,
    -0.00278,
    -0.00143,
    -0.00128,
    -0.00208,
    0.00078,
    0.00034
   ],
   [
    -0.00112,
    -0.00083,
    -0.0031,
    0.00093,
    0.00194,
    -0.00134,
    0.0,
    -0.00016,
    -0.00061,
    -0.00067,
    0.00071,
    0.00137,
    -0.00123,
    0.00012
   ],
   [
    0.0014,
    -0.00256,
    0.00012,
    0.00077,
    -0.00107,
    -0.00185,
    -9e-05,
    0.00101,
    -0.0026,
    -5e-05,
    0.0002,
    -0.00134,
    -0.0021,
    -0.00024
   ],
   [
    -0.00017,
    -0.0024,
    0.00146,
    0.00078,
    -0.00169,
    0.00045,
    0.00123,
    -0.00055,
    -0.0011,
    0.00071,
    0.00078,
    -0.00121,
    0.00095,
    0.00123
   ],
   [
    0.00417,
    -0.00271,
    -0.00038,
    -0.00112,
    -0.0028,
    -0.00028,
    0.00094,
    0.00286,
    -0.00199,
    -0.0012,
    -0.00149,
    -0.00253,
    -0.00058,
    0.00022
   ],
   [
    0.00661,
    -0.00133,
    0.00048,
    -0.00017,
    0.00513,
    -0.00154,
    -0.00562,
    0.00434,
    -0.00042,
    -0.00079,
    -0.00021,
    0.00404,
    -0.00148,
    -0.00443
   ],
   [
    -0.00274,
    0.00514,
    0.00445,
    0.00023,
    0.00221,
    -0.00155,
    0.00123,
    -0.00066,
    0.00585,
    0.00418,
    0.00084,
    0.00271,
    -0.00073,
    0.0015
   ],
   [
    -0.00484,
    -0.00423,
    0.00092,
    -0.00544,
    -0.00395,
    0.00432,
    0.00264,
    -0.00233,
    -0.00139,
    0.00299,
    -0.00508,
    -0.00344,
    0.00392,
    0.00173
   ],
   [
    -0.00322,
    -0.00426,
    -0.00331,
    -0.00099,
    -0.00032,
    0.00161,
    0.00172,
    -0.00255,
    -0.00353,
    -0.00146,
    -0.0002,
    -0.00017,
    0.00158,
    0.00189
   ],
   [
    0.00145,
    -0.0031,
    0.00026,
    0.003,
    -0.00208,
    -0.00221,
    -0.00247,
    0.00022,
    -0.00246,
    0.0007,
    0.00285,
    -0.00197,
    -0.00236,
    -0.00226
   ],
   [
    0.00229,
    0.00098,
    0.00156,
    -0.00065,
    -0.0002,
    0.00024,
    0.00084,
    0.00258,
    0.0002,
    0.00098,
    -0.00151,
    -0.00037,
    -0.00028,
    0.00077
   ],
   [
    -0.00059,
    -0.00104,
    0.0015,
    0.00054,
    -0.00043,
    -0.00189,
    0.00159,
    -0.00086,
    0.00013,
    0.00094,
    0.00076,
    -1e-05,
    -0.00111,
    0.00155
   ],
   [
    0.01466,
    -0.00333,
    0.00461,
    2e-05,
    -0.00536,
    -9e-05,
    -0.00312,
    0.01409,
    -0.00314,
    0.00272,
    -0.00151,
    -0.00534,
    -0.00134,
    -0.00287
   ],
   [
    0.00107,
    0.00258,
    0.00106,
    0.00102,
    -0.00043,
    0.00021,
    0.00215,
    0.00089,
    0.00237,
    0.0012,
    0.00141,
    0.00053,
    0.00091,
    0.00182
   ],
   [
    -0.00116,
    -0.00299,
    -0.0026,
    -0.0015,
    0.00031,
    -0.00156,
    0.00119,
    0.0001,
    -0.00216,
    -0.00064,
    -0.00207,
    -0.00052,
    -0.00179,
    0.00072
   ],
   [
    0.00112,
    -0.00054,
    -0.00024,
    -0.00098,
    0.00149,
    -2e-05,
    -0.00119,
    -0.00088,
    -0.00178,
    -0.00195,
    -0.00094,
    0.00108,
    -4e-05,
    -0.00102
   ],
   [
    0.00132,
    -0.00277,
    -0.00021,
    0.00238,
    -0.00232,
    -0.00248,
    0.00705,
    0.00399,
    -0.00016,
    0.00188,
    0.00335,
    -0.00091,
    -0.0008,
    0.00645
   ],
   [
    0.00028,
    0.0005,
    -0.00136,
    -0.00017,
    -0.00035,
    -0.00015,
    0.00032,
    0.00107,
    0.00059,
    -0.00089,
    0.0001,
    1e-05,
    2e-05,
    0.00038
   ],
   [
    -0.00264,
    -0.00107,
    -0.00118,
    -0.00012,
    -0.00067,
    0.00117,
    4e-05,
    -0.00173,
    -0.00104,
    -0.00059,
    -9e-05,
    -0.0005,
    0.00107,
    0.00015
   ],
   [
    -0.0004,
    -0.00704,
    0.00183,
    -0.00067,
    -0.0036,
    0.00306,
    -0.00178,
    0.00147,
    -0.00307,
    0.00315,
    -0.0003,
    -0.00264,
    0.00294,
    -0.00105
   ],
   [
    -0.00246,
    0.00036,
    -0.00127,
    -0.00109,
    -0.00105,
    0.00087,
    -0.00012,
    -0.00092,
    0.00035,
    0.0004,
    -0.00094,
    -0.00061,
    0.00062,
    -1e-05
   ],
   [
    0.00133,
    -0.00032,
    0.00012,
    -0.0007,
    -0.01059,
    0.0052,
    0.00626,
    0.00346,
    -0.00233,
    -7e-05,
    -0.00167,
    -0.00861,
    0.00373,
    0.00484
   ],
   [
    -0.00038,
    -0.00212,
    0.00182,
    0.00275,
    -0.00324,
    -0.00228,
    0.00667,
    -0.00138,
    -0.00135,
    -0.00021,
    0.00371,
    -0.0011,
    -0.0004,
    0.00635
   ],
   [
    0.00105,
    -0.00197,
    -0.00312,
    -0.00187,
    -0.00022,
    0.00066,
    0.00155,
    0.00062,
    -0.00241,
    -0.00226,
    -0.00234,
    -0.00078,
    0.00031,
    0.00069
   ],
   [
    -0.00149,
    0.00155,
    0.00181,
    -0.00094,
    -0.00063,
    0.00191,
    -0.00074,
    0.00102,
    0.00156,
    0.00323,
    -0.00056,
    -0.00074,
    0.00171,
    -0.00038
   ],
   [
    -0.00325,
    -0.00766,
    -0.00043,
    -0.00062,
    -0.00346,
    0.00095,
    -9e-05,
    -0.00496,
    -0.00775,
    -0.00094,
    -0.00042,
    -0.00337,
    0.00061,
    -0.00054
   ],
   [
    0.00425,
    0.00036,
    -0.0012,
    -0.00269,
    0.00224,
    0.00221,
    -0.00105,
    0.00072,
    0.00014,
    -0.00073,
    -0.00233,
    0.00227,
    0.0021,
    -0.00044
   ],
   [
    -0.00236,
    0.00166,
    0.00192,
    -0.00324,
    5e-05,
    0.00115,
    -5e-05,
    -0.00177,
    0.00075,
    0.00039,
    -0.00315,
    0.0001,
    0.0002,
    -0.00072
   ],
   [
    -0.01318,
    -0.01032,
    -0.00927,
    0.00319,
    0.00479,
    -0.00087,
    -0.00263,
    -0.00981,
    -0.00645,
    -0.00626,
    0.00233,
    0.00409,
    -0.00208,
    -0.00289
   ],
   [
    -0.00079,
    -0.00147,
    0.00147,
    -0.00105,
    -0.00053,
    0.00059,
    9e-05,
    -0.00108,
    -0.00266,
    0.00063,
    -0.00101,
    -0.00076,
    0.0005,
    0.00025
   ],
   [
    0.00182,
    -0.00768,
    -0.00357,
    0.00116,
    0.00206,
    0.00356,
    0.00421,
    0.00265,
    -0.00591,
    -0.00324,
    0.00077,
    0.00186,
    0.00365,
    0.00336
   ],
   [
    -0.00416,
    -0.00049,
    -0.00142,
    -0.00025,
    0.00042,
    0.00253,
    0.00065,
    -0.00277,
    -0.00044,
    -0.00053,
    -0.00033,
    0.00036,
    0.00193,
    0.00046
   ],
   [
    0.00073,
    -0.00215,
    -0.00208,
    0.00199,
    -3e-05,
    0.00017,
    -0.00055,
    0.00038,
    -0.00309,
    -0.00238,
    0.0017,
    -0.00026,
    0.00019,
    -0.00058
   ],
   [
    0.01648,
    0.00256,
    -0.00377,
    0.00646,
    -0.00969,
    -0.00671,
    -0.00157,
    0.01054,
    0.00148,
    -0.00372,
    0.00725,
    -0.00733,
    -0.00588,
    -0.00159
   ],
   [
    0.00875,
    0.01038,
    0.00827,
    0.00851,
    -0.00245,
    -0.01104,
    -0.00142,
    0.00669,
    0.006,
    0.00546,
    0.00873,
    -0.00108,
    -0.00979,
    -0.00171
   ],
   [
    0.00241,
    0.00315,
    0.0027,
    0.00025,
    -0.0021,
    -0.00025,
    0.00149,
    0.00315,
    0.00221,
    0.0025,
    0.00025,
    -0.00135,
    -0.00024,
    0.00123
   ],
   [
    0.00527,
    0.00745,
    0.00175,
    -0.00049,
    0.00309,
    -0.00448,
    0.00044,
    0.00493,
    0.00593,
    0.0009,
    -0.00082,
    0.00199,
    -0.00399,
    -0.00015
   ],
   [
    0.00165,
    -0.0006,
    -0.00025,
    -7e-05,
    -0.00087,
    0.00093,
    0.00162,
    -7e-05,
    -0.00117,
    -0.00093,
    0.00032,
    -0.00075,
    0.00081,
    0.00114
   ],
   [
    -0.00232,
    -0.0019,
    0.00073,
    -0.00045,
    -0.00024,
    0.00165,
    0.00122,
    -0.00242,
    -0.00204,
    -0.00015,
    -0.0004,
    2e-05,
    0.00164,
    0.00123
   ],
   [
    0.00378,
    -0.00307,
    0.00211,
    0.01407,
    -0.0009,
    -0.00567,
    -0.01383,
    0.00631,
    -0.0083,
    0.0027,
    0.01419,
    -0.00048,
    -0.00486,
    -0.0116
   ],
   [
    -0.00051,
    0.00259,
    0.00199,
    0.0008,
    -0.00281,
    0.00205,
    0.00459,
    0.00213,
    0.00492,
    0.00342,
    0.00057,
    -0.00145,
    0.00166,
    0.0039
   ],
   [
    0.00515,
    0.00289,
    0.00037,
    0.00128,
    -0.00272,
    0.00096,
    0.00743,
    0.00655,
    0.00363,
    0.00083,
    0.00187,
    -0.00139,
    0.00177,
    0.00668
   ],
   [
    -0.00929,
    0.01094,
    0.00238,
    -0.00758,
    0.00736,
    -0.00047,
    0.00511,
    -0.00761,
    0.00869,
    0.0028,
    -0.00557,
    0.0078,
    0.001,
    0.00556
   ],
   [
    0.00038,
    -0.00221,
    -0.00197,
    -0.00224,
    -0.00118,
    0.00185,
    -0.00128,
    -0.00091,
    -0.00382,
    -0.00206,
    -0.00257,
    -0.00184,
    0.00063,
    -0.00175
   ],
   [
    -0.00369,
    -0.00235,
    0.00183,
    0.00066,
    -0.0014,
    -0.00054,
    0.00125,
    -0.00312,
    -0.00318,
    0.00061,
    0.00068,
    -0.00142,
    -7e-05,
    0.00108
   ],
   [
    0.00784,
    0.0008,
    -0.00062,
    -0.00404,
    0.00104,
    0.00288,
    0.00111,
    0.004,
    -0.00083,
    -0.00053,
    -0.00279,
    0.00167,
    0.00377,
    0.00123
   ],
   [
    0.02309,
    -0.00562,
    0.00147,
    -0.00671,
    0.00556,
    0.01386,
    -0.00475,
    0.01962,
    -0.00096,
    0.00143,
    -0.00346,
    0.0069,
    0.01638,
    -0.0031
   ],
   [
    0.00022,
    0.00578,
    0.00299,
    0.00229,
    0.004,
    0.00034,
    0.00204,
    0.00105,
    0.0066,
    0.00317,
    0.00282,
    0.0042,
    0.00151,
    0.00287
   ],
   [
    0.00184,
    -0.0018,
    -0.00125,
    -0.00107,
    0.00058,
    0.00074,
    0.0,
    0.00121,
    -0.00114,
    -0.00024,
    -0.00097,
    0.00033,
    0.00068,
    7e-05
   ],
   [
    -0.00187,
    0.00488,
    -0.00075,
    -0.00028,
    0.00105,
    -0.00027,
    -0.00347,
    0.00012,
    0.00472,
    0.00062,
    -0.00092,
    0.00057,
    -0.00068,
    -0.00313
   ],
   [
    0.00551,
    -0.00436,
    0.00722,
    -0.00046,
    -0.00114,
    -0.00561,
    0.00379,
    0.00416,
    -0.00052,
    0.00358,
    0.00041,
    4e-05,
    -0.00326,
    0.00327
   ],
   [
    -0.00445,
    -0.00151,
    -0.00262,
    -0.00074,
    9e-05,
    0.00032,
    0.00037,
    -0.00461,
    -0.00193,
    -0.00357,
    -0.00062,
    -0.00033,
    1e-05,
    -0.0
   ],
   [
    0.00075,
    -0.00099,
    -0.00082,
    -0.00109,
    -0.00276,
    0.00434,
    0.00031,
    0.0,
    -0.00134,
    -0.00043,
    -0.00047,
    -0.00228,
    0.00341,
    8e-05
   ],
   [
    -0.00032,
    0.00224,
    -0.00035,
    0.0019,
    0.00206,
    -0.00026,
    0.00103,
    0.00031,
    0.00321,
    0.00011,
    0.00264,
    0.00283,
    0.00094,
    0.00111
   ],
   [
    -0.00204,
    -0.00021,
    -0.00136,
    0.00167,
    0.00254,
    0.00017,
    0.0008,
    -0.0017,
    -8e-05,
    -0.00087,
    0.00145,
    0.00219,
    0.00077,
    0.00081
   ],
   [
    0.01015,
    -0.00214,
    -0.00221,
    0.00454,
    -0.00859,
    -0.00224,
    -0.00719,
    0.00724,
    -0.00377,
    -0.00019,
    0.00437,
    -0.00816,
    -0.00354,
    -0.00706
   ],
   [
    -0.01256,
    -0.00489,
    -0.00856,
    -0.00268,
    0.01252,
    -0.00675,
    -0.00066,
    -0.01118,
    -0.00417,
    -0.00664,
    -0.00259,
    0.00884,
    -0.00587,
    -0.00095
   ],
   [
    0.00329,
    0.00083,
    -0.00094,
    0.00089,
    -0.00164,
    -0.00255,
    0.00115,
    0.00501,
    0.00247,
    0.00045,
    0.00169,
    -0.0002,
    -0.00157,
    0.00123
   ],
   [
    0.00098,
    0.00115,
    0.0033,
    0.00071,
    0.00301,
    -0.00072,
    -0.0008,
    0.00193,
    0.00035,
    0.00176,
    0.00044,
    0.00252,
    -0.00074,
    -0.00029
   ],
   [
    0.00069,
    -0.00367,
    -0.00143,
    -0.00316,
    -9e-05,
    0.00355,
    -0.00073,
    -0.00204,
    -0.00395,
    -0.00222,
    -0.00226,
    0.0006,
    0.00373,
    -0.00027
   ],
   [
    -0.00025,
    0.00234,
    0.00068,
    2e-05,
    -0.00012,
    0.00058,
    0.00175,
    0.00089,
    0.00237,
    0.00069,
    -0.00021,
    3e-05,
    0.00048,
    0.00154
   ],
   [
    -0.00118,
    0.0031,
    -0.00073,
    -0.00337,
    1e-05,
    0.00185,
    -0.00061,
    -0.00158,
    0.00155,
    -2e-05,
    -0.00273,
    0.00042,
    0.00086,
    -0.00099
   ],
   [
    0.00333,
    0.0025,
    0.00157,
    -0.00067,
    0.00034,
    0.00046,
    -0.00037,
    0.00158,
    0.00237,
    0.00084,
    0.00013,
    0.00089,
    0.0012,
    5e-05
   ],
   [
    0.00212,
    -0.00025,
    0.00065,
    2e-05,
    -0.00035,
    0.00077,
    0.00255,
    0.00187,
    0.00023,
    0.00088,
    0.0005,
    0.00018,
    0.00089,
    0.0024
   ],
   [
    0.00128,
    0.00527,
    0.0016,
    -0.00056,
    0.00442,
    -0.00192,
    -0.00258,
    0.00012,
    0.00429,
    0.00079,
    -0.00031,
    0.00401,
    -0.00151,
    -0.00203
   ],
   [
    0.00443,
    0.00453,
    0.00181,
    -0.00054,
    -0.00171,
    0.002,
    0.00155,
    0.00354,
    0.00226,
    0.00165,
    -0.00064,
    -0.00143,
    0.00133,
    0.00088
   ],
   [
    -0.00498,
    -0.00053,
    0.00018,
    0.00093,
    0.00072,
    -0.00328,
    0.00083,
    -0.00497,
    -0.00011,
    -0.00015,
    0.00106,
    0.00044,
    -0.00301,
    0.00069
   ],
   [
    0.00048,
    -0.00446,
    0.00091,
    0.00665,
    -0.00336,
    -0.00321,
    0.00128,
    0.0004,
    -0.00303,
    0.00084,
    0.00668,
    -0.00254,
    -0.00266,
    0.00078
   ],
   [
    0.00428,
    0.00112,
    0.00184,
    0.00254,
    -0.00058,
    0.00152,
    -0.0013,
    0.00181,
    -0.00041,
    0.00024,
    0.00253,
    -0.00018,
    0.0016,
    -0.00103
   ],
   [
    0.00491,
    0.01643,
    0.00026,
    0.00251,
    0.0003,
    0.00246,
    -0.00452,
    0.01158,
    0.01904,
    0.00565,
    0.00303,
    0.00251,
    0.00272,
    -0.0031
   ],
   [
    -0.0096,
    -0.00406,
    0.00286,
    0.00812,
    0.00815,
    -0.0158,
    -0.00428,
    -0.00758,
    -0.00138,
    0.00294,
    0.00785,
    0.00642,
    -0.0131,
    -0.00331
   ],
   [
    -0.00387,
    -0.00074,
    0.00158,
    0.00613,
    -0.00523,
    -0.00293,
    0.00964,
    -0.00015,
    -0.00129,
    0.00717,
    0.00635,
    -0.00314,
    -0.00056,
    0.00966
   ],
   [
    0.00248,
    0.0026,
    -0.00104,
    -0.00176,
    0.00088,
    0.00143,
    0.00238,
    0.00095,
    0.00092,
    -0.00112,
    -0.00146,
    0.00085,
    0.00176,
    0.00248
   ],
   [
    0.00568,
    0.01272,
    -0.00035,
    0.0088,
    0.01112,
    -0.00357,
    -0.01061,
    0.00282,
    0.01066,
    -0.00272,
    0.007,
    0.00873,
    -0.00288,
    -0.0086
   ],
   [
    0.00012,
    0.00087,
    0.00038,
    -0.00181,
    0.00415,
    0.00364,
    0.00077,
    -0.00011,
    0.00054,
    -0.00027,
    -0.00112,
    0.00454,
    0.00339,
    0.00101
   ],
   [
    -0.00363,
    -0.00038,
    -0.00013,
    -0.0018,
    0.00179,
    -0.00149,
    -0.00072,
    -0.0026,
    -0.001,
    2e-05,
    -0.00163,
    0.00104,
    -0.0013,
    -0.00053
   ],
   [
    -0.00105,
    0.00043,
    -0.00066,
    -0.001,
    -0.00073,
    0.00158,
    0.00109,
    0.00096,
    0.00087,
    0.00074,
    -0.0006,
    -0.00049,
    0.00135,
    0.00114
   ],
   [
    -0.00013,
    -0.00277,
    -0.00149,
    0.00076,
    -0.00087,
    0.00084,
    0.0002,
    0.00025,
    -0.00264,
    -0.00039,
    0.00074,
    -0.00062,
    0.00064,
    9e-05
   ],
   [
    0.00015,
    0.00393,
    0.00306,
    0.00054,
    -0.00256,
    -0.00162,
    0.00214,
    -0.00174,
    0.0025,
    0.00241,
    0.00195,
    -0.0008,
    -0.00032,
    0.00211
   ],
   [
    0.00382,
    0.00087,
    -0.00065,
    0.00115,
    -0.00107,
    0.00086,
    -2e-05,
    0.00452,
    0.00188,
    7e-05,
    0.00142,
    -0.00062,
    0.00105,
    0.00027
   ],
   [
    -0.00239,
    -0.00113,
    0.00052,
    -0.00057,
    -0.00102,
    0.00059,
    -0.00051,
    -0.00291,
    -0.00161,
    -0.00028,
    -0.00033,
    -0.00072,
    0.00026,
    -0.00064
   ],
   [
    -0.00076,
    -0.00152,
    -0.00144,
    0.00093,
    0.00072,
    -0.00223,
    -0.00077,
    -0.00018,
    -0.00173,
    -0.00078,
    0.00084,
    0.00044,
    -0.00135,
    -0.00068
   ],
   [
    0.00034,
    0.00023,
    0.00085,
    0.00102,
    -5e-05,
    -0.00017,
    -0.0029,
    -0.00027,
    0.00029,
    -8e-05,
    0.00036,
    -0.00062,
    -0.0002,
    -0.00263
   ],
   [
    0.0009,
    -0.00021,
    -0.00169,
    -0.00233,
    -0.00197,
    0.00337,
    0.00328,
    0.00077,
    0.00038,
    0.00055,
    -0.00179,
    -0.00103,
    0.00312,
    0.00301
   ],
   [
    -0.00057,
    -0.00024,
    0.00243,
    -0.00023,
    -0.0023,
    -0.00083,
    0.00306,
    -0.00091,
    -0.00068,
    0.00225,
    -0.00036,
    -0.00194,
    -0.00065,
    0.00266
   ],
   [
    -0.00155,
    0.0029,
    0.00516,
    -0.00322,
    0.00291,
    -0.0016,
    -0.00303,
    -0.00131,
    0.00337,
    0.00193,
    -0.00411,
    0.00228,
    -0.00251,
    -0.00273
   ],
   [
    -0.00723,
    -0.00138,
    -0.00263,
    0.00161,
    0.0003,
    0.00153,
    -0.00473,
    -0.00433,
    -0.00067,
    -0.00198,
    0.0011,
    0.00056,
    0.00103,
    -0.00399
   ],
   [
    0.0128,
    0.01152,
    0.00954,
    0.02049,
    0.0107,
    -0.01326,
    -0.02855,
    0.01029,
    0.01419,
    0.00806,
    0.01878,
    0.00785,
    -0.01267,
    -0.02416
   ],
   [
    0.01673,
    -0.00979,
    -0.00661,
    -0.00263,
    -0.01622,
    0.00565,
    -0.00876,
    0.0118,
    -0.00845,
    -0.00335,
    -0.0012,
    -0.01433,
    0.00042,
    -0.01039
   ],
   [
    -0.00125,
    -0.00076,
    -0.00011,
    -0.00055,
    0.00177,
    0.0004,
    -0.00089,
    -0.00087,
    0.00034,
    0.00049,
    -0.00043,
    0.0013,
    0.00063,
    -0.0004
   ],
   [
    -0.00121,
    -0.01146,
    -0.007,
    -0.00043,
    -0.0055,
    0.00148,
    0.00326,
    0.00369,
    -0.00576,
    -0.00277,
    -0.00113,
    -0.00488,
    0.00029,
    0.00223
   ],
   [
    0.00241,
    0.00145,
    -0.00091,
    -6e-05,
    0.00235,
    -0.00222,
    8e-05,
    0.00141,
    0.00184,
    -0.00074,
    7e-05,
    0.00238,
    -0.0015,
    8e-05
   ],
   [
    0.01652,
    0.01272,
    0.00156,
    -0.00064,
    0.00334,
    -0.00272,
    -0.00239,
    0.01656,
    0.01376,
    0.00459,
    -0.00141,
    0.00188,
    -0.00159,
    -0.00181
   ],
   [
    0.00164,
    0.0021,
    -0.00026,
    -0.00364,
    0.00683,
    -0.00099,
    -0.0048,
    0.00072,
    0.00208,
    -0.00142,
    -0.00381,
    0.00548,
    -0.00202,
    -0.00432
   ],
   [
    0.0015,
    -0.00036,
    0.00322,
    0.00069,
    0.00047,
    -0.00195,
    5e-05,
    0.00061,
    -0.00075,
    0.00096,
    0.00013,
    -0.00012,
    -0.00239,
    0.00016
   ],
   [
    0.01545,
    0.00461,
    0.00883,
    -0.00891,
    -0.00166,
    0.01048,
    0.00067,
    0.00869,
    0.00071,
    0.00519,
    -0.00681,
    -0.0009,
    0.00843,
    3e-05
   ],
   [
    0.02161,
    0.00519,
    0.00912,
    0.00585,
    -0.00755,
    -0.01522,
    0.00315,
    0.01508,
    0.00343,
    0.00078,
    0.00385,
    -0.00675,
    -0.01417,
    0.00195
   ],
   [
    0.00673,
    0.00527,
    0.00187,
    0.00078,
    0.00073,
    0.0017,
    -0.00054,
    0.00612,
    0.00652,
    0.00222,
    0.00128,
    0.00177,
    0.0021,
    -0.00013
   ],
   [
    0.01034,
    0.00819,
    0.00065,
    0.00074,
    -9e-05,
    -0.00095,
    0.00672,
    0.00779,
    0.00898,
    0.00068,
    -0.00047,
    -0.00045,
    -0.00044,
    0.00667
   ],
   [
    -0.00196,
    -0.00099,
    -0.00311,
    0.00095,
    -0.00026,
    0.00172,
    -0.00076,
    6e-05,
    -0.0007,
    -0.00157,
    0.00074,
    2e-05,
    0.0018,
    -0.0004
   ],
   [
    0.00027,
    -0.00173,
    0.00092,
    9e-05,
    -0.00184,
    0.00027,
    0.00161,
    -0.00084,
    -0.00125,
    0.00011,
    6e-05,
    -0.00161,
    -9e-05,
    0.00133
   ],
   [
    0.0064,
    0.00548,
    -0.00095,
    0.00375,
    0.00408,
    -0.00072,
    -0.00105,
    0.00508,
    0.00317,
    -0.00077,
    0.00383,
    0.00367,
    -0.00028,
    -0.00056
   ],
   [
    0.00835,
    -0.00333,
    -0.00776,
    -0.02796,
    -0.01043,
    0.03782,
    -0.00945,
    0.01124,
    -0.00431,
    -0.00537,
    -0.03066,
    -0.01187,
    0.02717,
    -0.011
   ],
   [
    -0.00299,
    0.00055,
    -0.00114,
    -0.00153,
    0.00094,
    0.00129,
    0.0014,
    -0.00499,
    -0.00183,
    -0.00287,
    -0.00151,
    7e-05,
    0.00014,
    0.00075
   ],
   [
    0.00175,
    0.00148,
    -0.00095,
    -0.00194,
    0.00048,
    0.00192,
    0.00179,
    0.00272,
    0.00194,
    -0.00016,
    -0.00218,
    0.00051,
    0.00157,
    0.00149
   ],
   [
    -0.0032,
    0.00763,
    0.00303,
    -0.00203,
    0.00177,
    0.00342,
    0.00376,
    -0.00086,
    0.00879,
    0.00575,
    -0.00103,
    0.0027,
    0.00405,
    0.0038
   ],
   [
    0.00159,
    0.0073,
    -0.00499,
    -0.00172,
    -0.00488,
    0.01284,
    0.00593,
    0.00556,
    0.00784,
    0.00232,
    -0.00038,
    -0.00186,
    0.01428,
    0.00703
   ],
   [
    0.00223,
    -0.00168,
    -5e-05,
    0.0008,
    -0.0012,
    0.00013,
    0.00239,
    0.00196,
    -0.00242,
    -0.0005,
    0.00051,
    -0.00138,
    -0.00014,
    0.00179
   ],
   [
    1e-05,
    -0.0066,
    -0.0004,
    -0.00247,
    -0.00651,
    0.00616,
    0.00552,
    -0.00149,
    -0.0076,
    -0.0024,
    -0.0034,
    -0.00637,
    0.00444,
    0.00424
   ],
   [
    -0.00237,
    -0.00036,
    -0.00156,
    -0.00049,
    0.00306,
    -0.00073,
    -0.00195,
    -0.00219,
    -0.00158,
    -0.00132,
    -0.00075,
    0.00186,
    -0.00068,
    -0.00168
   ],
   [
    -0.00144,
    -0.00424,
    -0.00245,
    -0.00019,
    -0.00271,
    0.00027,
    0.00321,
    -0.00037,
    -0.00332,
    -0.00018,
    -0.0004,
    -0.0029,
    0.00047,
    0.00285
   ],
   [
    0.00135,
    0.00096,
    -0.00214,
    0.00168,
    0.00163,
    0.00062,
    -0.0017,
    0.00085,
    7e-05,
    -0.00215,
    0.00155,
    0.00124,
    0.00089,
    -0.00124
   ],
   [
    0.00091,
    -0.00234,
    0.00073,
    0.00035,
    -0.00282,
    0.00229,
    0.00125,
    0.00087,
    -0.00119,
    0.00125,
    0.00048,
    -0.00215,
    0.00201,
    0.00115
   ],
   [
    0.00101,
    -0.00075,
    -0.00018,
    0.00024,
    -0.0004,
    -0.00123,
    -0.00081,
    0.00136,
    -0.0008,
    0.00013,
    0.00029,
    -0.00047,
    -0.00081,
    -0.00053
   ],
   [
    -0.00517,
    -0.0082,
    -0.00076,
    -0.00014,
    -0.00271,
    0.00022,
    -0.00046,
    -0.00418,
    -0.007,
    -0.00132,
    1e-05,
    -0.00241,
    -0.00012,
    -0.0007
   ],
   [
    0.00285,
    -0.00013,
    0.003,
    0.00239,
    -0.00347,
    -0.00133,
    0.00073,
    0.00225,
    -0.00103,
    0.00141,
    0.00229,
    -0.00286,
    -0.00126,
    0.0007
   ],
   [
    -0.02963,
    -0.01662,
    -0.01002,
    0.00317,
    -0.00513,
    0.00079,
    -0.00587,
    -0.01917,
    -0.01223,
    -0.0026,
    0.00237,
    -0.00552,
    0.00107,
    -0.00597
   ],
   [
    -3e-05,
    -0.0009,
    0.00036,
    -0.0017,
    -0.00028,
    0.0025,
    -0.00092,
    -0.00078,
    -0.002,
    -0.00105,
    -0.00203,
    -0.00076,
    0.00147,
    -0.00099
   ],
   [
    -0.00307,
    -0.00467,
    0.00563,
    0.0037,
    0.0015,
    -0.00092,
    -0.00074,
    -0.00183,
    -0.0013,
    0.00268,
    0.00298,
    0.00135,
    -0.00089,
    -0.00107
   ],
   [
    -0.00509,
    0.00268,
    0.00412,
    0.00101,
    0.00196,
    0.00281,
    0.00247,
    -0.0037,
    0.00249,
    0.00495,
    0.00216,
    0.00263,
    0.00352,
    0.00329
   ],
   [
    -0.00105,
    0.00161,
    0.00017,
    -0.00179,
    0.00126,
    0.00164,
    -0.00169,
    -0.00029,
    -0.00011,
    -0.00028,
    -0.0019,
    0.00067,
    0.00088,
    -0.00182
   ],
   [
    -0.03875,
    -0.00561,
    -0.00439,
    -0.0097,
    0.00463,
    0.00112,
    0.02108,
    -0.02454,
    -0.00285,
    -0.00022,
    -0.00819,
    0.00643,
    0.00388,
    0.02023
   ],
   [
    0.00526,
    -0.00243,
    0.00314,
    0.00114,
    0.00269,
    -0.00332,
    -0.00098,
    0.00247,
    -0.00353,
    3e-05,
    0.00077,
    0.00199,
    -0.00301,
    -0.0005
   ],
   [
    -0.00083,
    0.00023,
    -0.00081,
    -0.00113,
    -0.00247,
    0.00059,
    0.00171,
    -0.00144,
    -0.00088,
    -0.00051,
    -0.00076,
    -0.00184,
    0.0002,
    0.0014
   ],
   [
    0.00892,
    -0.00835,
    0.00626,
    0.00704,
    -0.00095,
    -0.00204,
    -0.00923,
    0.00434,
    -0.00894,
    0.00016,
    0.0061,
    -0.00071,
    -0.00377,
    -0.00912
   ],
   [
    -0.00151,
    0.0009,
    0.00238,
    0.00052,
    0.0017,
    0.00013,
    0.00303,
    -0.00223,
    0.00083,
    0.00021,
    0.00068,
    0.00209,
    0.00034,
    0.00253
   ],
   [
    0.00046,
    0.00345,
    -0.00033,
    -0.00072,
    0.00191,
    -0.00176,
    -0.00125,
    0.00053,
    0.0023,
    -0.00046,
    -0.00102,
    0.0014,
    -0.00206,
    -0.00143
   ],
   [
    -0.00303,
    0.00061,
    0.0029,
    -0.00082,
    0.00295,
    -0.00184,
    -0.00117,
    -0.00293,
    0.00017,
    0.00072,
    -0.0008,
    0.00247,
    -0.00167,
    -0.00085
   ],
   [
    -0.01548,
    0.00164,
    -0.00154,
    -0.01486,
    0.00521,
    0.00195,
    0.01226,
    -0.01182,
    -4e-05,
    -0.00076,
    -0.01425,
    0.00342,
    0.00087,
    0.01111
   ],
   [
    -0.00123,
    -0.0037,
    1e-05,
    0.00219,
    -0.0029,
    5e-05,
    0.0021,
    -0.00025,
    -0.00205,
    0.00173,
    0.00172,
    -0.00292,
    1e-05,
    0.0018
   ],
   [
    0.00364,
    0.00033,
    0.00368,
    0.00247,
    -0.00291,
    0.00155,
    0.00099,
    0.00242,
    0.00041,
    0.00309,
    0.00234,
    -0.00316,
    0.00089,
    0.00076
   ],
   [
    -0.00151,
    -0.00332,
    -0.00106,
    0.00026,
    -0.00347,
    -0.00061,
    -0.00072,
    -0.0019,
    -0.00346,
    -0.0016,
    -0.00042,
    -0.00349,
    -0.00115,
    -0.00135
   ],
   [
    -0.00316,
    -0.00596,
    -0.00088,
    0.00056,
    0.00285,
    -0.00265,
    -0.00435,
    -0.00446,
    -0.00748,
    -0.0039,
    0.00047,
    0.00212,
    -0.00239,
    -0.0039
   ],
   [
    -0.00273,
    -0.00453,
    -0.00059,
    0.00087,
    -0.00407,
    -0.00048,
    0.0038,
    -0.00214,
    -0.00522,
    -0.00102,
    0.00087,
    -0.00319,
    -0.00051,
    0.00274
   ],
   [
    -0.00087,
    -0.00158,
    -0.00326,
    -0.00095,
    0.00232,
    -4e-05,
    -0.00116,
    -0.00065,
    -0.00127,
    -0.00244,
    -0.00108,
    0.00175,
    -7e-05,
    -0.00107
   ],
   [
    -0.01355,
    0.00477,
    -0.01359,
    -0.00389,
    0.00183,
    0.00354,
    -0.00383,
    -0.00327,
    0.0048,
    -0.0037,
    -0.00468,
    0.00026,
    0.00141,
    -0.00344
   ],
   [
    0.00185,
    0.0008,
    -0.0005,
    -0.00065,
    -1e-05,
    0.00241,
    -0.00056,
    0.00357,
    0.00233,
    0.00169,
    -0.00042,
    0.00041,
    0.00223,
    -0.00021
   ],
   [
    0.00011,
    0.00208,
    0.00082,
    -0.0015,
    0.00172,
    -0.0009,
    -0.00079,
    -0.0004,
    0.00078,
    -9e-05,
    -0.00116,
    0.00178,
    -0.00079,
    -0.00088
   ],
   [
    -0.00083,
    0.00473,
    -0.00082,
    -0.00039,
    0.00051,
    0.00251,
    0.00109,
    0.0,
    0.00411,
    0.0003,
    -0.0002,
    0.00118,
    0.00195,
    0.00107
   ],
   [
    -0.00626,
    -0.0047,
    -0.00623,
    -0.00215,
    -0.00066,
    0.0019,
    0.00314,
    -0.00799,
    -0.00532,
    -0.00364,
    -0.00239,
    -0.00088,
    0.00148,
    0.0027
   ],
   [
    -0.00085,
    -0.00178,
    -0.00332,
    0.00026,
    -0.00304,
    0.00191,
    -0.00073,
    -0.0013,
    -0.00458,
    -0.00249,
    -0.00071,
    -0.00403,
    0.00054,
    -0.00114
   ],
   [
    -0.00397,
    -0.00093,
    0.00171,
    0.00086,
    0.00033,
    -0.00063,
    -0.00356,
    -0.00419,
    -0.00156,
    0.0005,
    0.00032,
    -0.00084,
    -0.00158,
    -0.00371
   ],
   [
    0.00247,
    0.00556,
    0.00314,
    0.00185,
    0.00099,
    -0.00326,
    -0.00201,
    0.00221,
    0.00495,
    0.00167,
    0.0011,
    0.00115,
    -0.00282,
    -0.00199
   ],
   [
    -0.00414,
    -0.00195,
    0.00034,
    -0.00272,
    -0.0018,
    0.0009,
    0.00272,
    -0.00484,
    -0.00287,
    -0.00055,
    -0.00244,
    -0.00143,
    0.00119,
    0.00264
   ],
   [
    -0.00228,
    -0.00079,
    -0.00179,
    0.00025,
    -0.00036,
    0.00167,
    0.00186,
    -0.00068,
    -0.00077,
    -0.00021,
    0.00049,
    -0.0002,
    0.00143,
    0.00174
   ],
   [
    -0.00461,
    -0.00317,
    -0.00518,
    0.00047,
    -0.00301,
    0.00383,
    0.00629,
    -0.00391,
    -0.00473,
    -0.00389,
    0.0009,
    -0.00295,
    0.00433,
    0.00593
   ],
   [
    -0.00592,
    0.00446,
    0.00141,
    0.00652,
    0.00172,
    -0.01083,
    -0.00952,
    -0.00615,
    0.00374,
    -0.00255,
    0.006,
    0.00134,
    -0.01105,
    -0.00908
   ],
   [
    0.00116,
    -0.00281,
    -0.00071,
    0.00123,
    -0.00248,
    0.00139,
    0.00126,
    8e-05,
    -0.00133,
    0.00032,
    0.00108,
    -0.0017,
    0.00151,
    0.00104
   ],
   [
    0.00425,
    0.00285,
    0.0032,
    0.00178,
    0.00228,
    -0.0007,
    0.00067,
    0.00581,
    0.00506,
    0.00375,
    0.00208,
    0.00251,
    0.00016,
    0.00133
   ],
   [
    -9e-05,
    0.0005,
    0.00093,
    -0.00143,
    0.00238,
    -0.00089,
    -0.00273,
    0.00024,
    7e-05,
    0.00046,
    -0.00147,
    0.00156,
    -0.00086,
    -0.00229
   ],
   [
    -0.00233,
    0.01031,
    0.00433,
    0.00501,
    -0.0003,
    0.00073,
    -0.00075,
    -0.00063,
    0.00908,
    0.00399,
    0.00417,
    0.00042,
    0.00151,
    -0.00035
   ],
   [
    -0.0006,
    0.0,
    -0.00749,
    -0.00961,
    0.00296,
    0.01664,
    -0.00363,
    0.00251,
    0.00567,
    0.00139,
    -0.00809,
    0.00329,
    0.01497,
    -0.00253
   ],
   [
    0.0011,
    0.0005,
    0.00068,
    0.00156,
    0.00349,
    -0.00437,
    -0.00448,
    -0.00069,
    -0.0008,
    -0.0004,
    0.0019,
    0.0025,
    -0.00346,
    -0.00365
   ],
   [
    -0.00367,
    -0.00399,
    -0.00251,
    0.00097,
    -0.00167,
    0.00083,
    -0.00125,
    -0.00358,
    -0.00352,
    -0.00108,
    0.00099,
    -0.00157,
    0.0007,
    -0.00125
   ],
   [
    0.00148,
    0.00326,
    0.00217,
    0.00114,
    0.00185,
    -0.00069,
    0.00388,
    -0.00016,
    0.00277,
    0.00016,
    0.00102,
    0.00204,
    0.0001,
    0.00374
   ],
   [
    -0.00418,
    0.00181,
    -0.00463,
    -0.00841,
    0.0017,
    0.00878,
    0.00274,
    -0.00178,
    0.0028,
    -0.00153,
    -0.00671,
    0.0028,
    0.00834,
    0.00278
   ],
   [
    -0.00086,
    -0.00621,
    -0.00427,
    0.00635,
    -0.00413,
    0.00211,
    -0.0139,
    0.00229,
    -0.00415,
    -0.00299,
    0.00362,
    -0.00486,
    0.00108,
    -0.01231
   ],
   [
    0.00026,
    -0.00179,
    -0.00215,
    -0.00011,
    7e-05,
    -0.00108,
    0.0011,
    0.00027,
    -0.002,
    -0.00107,
    -8e-05,
    -0.00016,
    -0.00103,
    0.00068
   ],
   [
    0.00601,
    0.0084,
    0.00035,
    -0.00532,
    0.0061,
    0.00258,
    -0.00408,
    0.00256,
    0.00539,
    6e-05,
    -0.00493,
    0.0049,
    0.00222,
    -0.00336
   ],
   [
    0.0111,
    0.01109,
    -0.00576,
    -0.00021,
    0.00155,
    -0.00402,
    0.00607,
    0.01054,
    0.01112,
    -0.00459,
    0.00209,
    0.00258,
    -0.00222,
    0.00564
   ],
   [
    -0.00314,
    0.01681,
    0.00144,
    0.00069,
    0.01305,
    -0.00587,
    -0.01006,
    -0.00389,
    0.01303,
    -0.00172,
    0.00023,
    0.00973,
    -0.00562,
    -0.00851
   ],
   [
    -0.01204,
    0.01134,
    -0.00334,
    0.00097,
    -0.00382,
    -0.00431,
    0.00249,
    -0.01428,
    0.00705,
    -0.00604,
    0.00063,
    -0.00328,
    -0.00396,
    0.00174
   ],
   [
    0.00318,
    -0.00103,
    0.00076,
    -9e-05,
    0.00104,
    0.00049,
    -0.00257,
    0.00253,
    -0.00086,
    0.00113,
    0.00015,
    0.00064,
    0.00019,
    -0.00186
   ],
   [
    0.00169,
    0.0029,
    6e-05,
    -0.00087,
    -0.00223,
    0.00152,
    0.00337,
    0.00112,
    0.00363,
    0.00018,
    -0.00089,
    -0.00191,
    0.00115,
    0.0032
   ],
   [
    0.00038,
    -0.001,
    -0.00028,
    -0.00037,
    0.0001,
    0.00083,
    -0.00035,
    -0.00013,
    -0.00089,
    -0.00015,
    -0.00018,
    0.00015,
    0.00097,
    -8e-05
   ],
   [
    0.00302,
    0.00235,
    0.00046,
    0.00172,
    0.00154,
    -0.00268,
    -0.0013,
    0.00313,
    0.00218,
    0.0002,
    0.00103,
    0.00069,
    -0.00307,
    -0.00142
   ],
   [
    -0.00592,
    0.00119,
    -0.00259,
    -0.00239,
    0.00062,
    0.00143,
    0.00102,
    -0.00611,
    -0.0001,
    -0.00277,
    -0.00258,
    0.00044,
    9e-05,
    0.00047
   ],
   [
    -0.00418,
    -0.00134,
    -0.00271,
    -0.00236,
    0.00244,
    0.00291,
    -0.00169,
    -0.00325,
    0.00147,
    -0.00054,
    -0.0028,
    0.00152,
    0.00204,
    -0.00175
   ],
   [
    -0.00574,
    -0.00501,
    0.00319,
    0.00245,
    0.00176,
    -0.00637,
    0.00133,
    -0.00558,
    -0.005,
    0.00121,
    0.00315,
    0.0019,
    -0.00499,
    0.00096
   ],
   [
    0.00052,
    0.00076,
    0.00029,
    0.00074,
    0.00067,
    -0.00131,
    6e-05,
    -0.00037,
    -0.0003,
    0.0005,
    0.00014,
    -0.00018,
    -0.00129,
    2e-05
   ],
   [
    -0.00265,
    0.0007,
    0.00161,
    0.00351,
    -0.00672,
    -0.00419,
    0.00395,
    -0.00094,
    -0.00032,
    0.00291,
    0.0028,
    -0.00694,
    -0.00335,
    0.00354
   ],
   [
    -0.00215,
    -0.00162,
    0.0015,
    -0.00081,
    0.0001,
    -0.00276,
    -0.0006,
    -0.00114,
    -0.00027,
    0.00118,
    -0.00032,
    0.00096,
    -0.00187,
    -0.00058
   ],
   [
    0.00341,
    -0.00401,
    0.00331,
    -0.00065,
    0.00079,
    -0.00244,
    -0.00399,
    0.00446,
    -0.00288,
    0.00305,
    -0.00039,
    0.0008,
    -0.00236,
    -0.00383
   ],
   [
    -0.00561,
    -0.0057,
    -0.00234,
    0.00084,
    -0.00042,
    0.00196,
    -0.00042,
    -0.0046,
    -0.00286,
    -0.00071,
    0.0008,
    -0.0002,
    0.00206,
    -0.00025
   ],
   [
    -0.01497,
    0.02344,
    0.00409,
    -0.0322,
    0.00817,
    0.01699,
    0.0185,
    -0.01031,
    0.02073,
    0.00194,
    -0.03191,
    0.00759,
    0.01306,
    0.01536
   ],
   [
    0.00334,
    0.00029,
    0.00096,
    0.00208,
    -0.00105,
    -0.00134,
    -0.00057,
    0.00201,
    -0.00082,
    0.00051,
    0.00173,
    -0.00102,
    -0.00105,
    -0.00045
   ],
   [
    -0.02124,
    0.00139,
    -0.00347,
    -0.00963,
    -0.01499,
    0.01776,
    0.0032,
    -0.01743,
    0.00135,
    0.00125,
    -0.01057,
    -0.01279,
    0.01277,
    0.00229
   ],
   [
    0.00215,
    0.00218,
    -2e-05,
    0.00073,
    -0.00144,
    0.00147,
    0.00307,
    0.0042,
    0.00387,
    0.00148,
    0.0014,
    -0.00035,
    0.00226,
    0.00317
   ],
   [
    -0.00097,
    -0.00146,
    -0.00406,
    -0.00029,
    -0.00067,
    -0.00055,
    0.00087,
    -0.00177,
    -0.00204,
    -0.00387,
    -0.00042,
    -0.00072,
    -0.00076,
    0.00048
   ],
   [
    -0.00716,
    0.00289,
    -0.00196,
    -0.00176,
    0.00097,
    0.00027,
    -0.00114,
    -0.00454,
    0.00213,
    -0.00076,
    -0.00177,
    0.00075,
    0.0,
    -0.00104
   ],
   [
    0.00386,
    0.00158,
    0.00119,
    4e-05,
    -0.00098,
    0.001,
    -0.00134,
    0.00253,
    0.0001,
    4e-05,
    0.00025,
    -0.00093,
    0.00055,
    -0.00124
   ],
   [
    -0.01646,
    0.00514,
    -9e-05,
    -0.00473,
    0.00557,
    -0.00923,
    0.00586,
    -0.01377,
    0.00832,
    -0.00107,
    -0.00471,
    0.00431,
    -0.00777,
    0.00494
   ],
   [
    0.00167,
    -0.00145,
    -0.00061,
    0.00128,
    -0.00018,
    -0.00072,
    -0.00113,
    0.00106,
    -0.00101,
    0.00017,
    0.00143,
    -0.00012,
    -0.00059,
    -0.00088
   ],
   [
    -0.00346,
    0.00279,
    0.0021,
    -0.0011,
    0.00328,
    0.00112,
    -0.00311,
    -0.00474,
    0.00212,
    0.00133,
    -0.00112,
    0.00272,
    0.00112,
    -0.00265
   ],
   [
    -0.00096,
    0.00316,
    -0.00133,
    -0.00259,
    0.00138,
    0.00054,
    0.00036,
    -0.0001,
    0.00252,
    -0.00016,
    -0.00239,
    0.00106,
    0.00059,
    7e-05
   ],
   [
    -0.00174,
    -0.003,
    -0.00261,
    -0.00124,
    -0.00188,
    0.00244,
    0.00127,
    -0.00159,
    -0.00173,
    -0.00159,
    -0.0009,
    -0.00118,
    0.00236,
    0.00112
   ],
   [
    0.00046,
    -1e-05,
    0.00096,
    0.00101,
    -0.0005,
    -0.00027,
    -0.00087,
    0.00013,
    -0.00037,
    0.00066,
    0.00107,
    -0.00044,
    0.00026,
    -0.00046
   ],
   [
    0.00288,
    -0.00163,
    0.00194,
    0.00029,
    -6e-05,
    0.00069,
    -0.00098,
    0.00288,
    -0.00069,
    0.00293,
    0.00063,
    0.00011,
    0.0012,
    0.0
   ],
   [
    0.00699,
    -0.00438,
    0.00056,
    0.00404,
    -0.00352,
    -0.00379,
    0.00331,
    0.00294,
    -0.00637,
    -0.00106,
    0.00316,
    -0.00339,
    -0.00245,
    0.00319
   ],
   [
    -0.00097,
    0.00107,
    -4e-05,
    -6e-05,
    -0.00156,
    -0.00011,
    0.00202,
    -0.00039,
    0.00112,
    0.00012,
    2e-05,
    -0.00108,
    7e-05,
    0.00175
   ],
   [
    -0.00014,
    -0.0049,
    0.00419,
    0.00199,
    0.00319,
    -0.00475,
    -0.00145,
    -0.00048,
    -0.00359,
    0.00151,
    0.00151,
    0.00255,
    -0.00401,
    -0.0014
   ],
   [
    0.00362,
    0.00089,
    0.00161,
    0.00053,
    0.00038,
    -0.00153,
    0.00396,
    0.00368,
    0.00075,
    0.00168,
    0.0017,
    0.00148,
    -0.00017,
    0.00393
   ],
   [
    0.00239,
    0.00069,
    0.00318,
    0.00219,
    -0.00558,
    0.00146,
    0.00115,
    0.00527,
    0.00146,
    0.00478,
    0.00179,
    -0.00503,
    0.00191,
    0.0013
   ],
   [
    0.00039,
    0.0026,
    -0.00015,
    -4e-05,
    0.00263,
    -0.00056,
    0.00027,
    0.00076,
    0.00295,
    -0.00034,
    0.00042,
    0.00253,
    0.0002,
    0.0005
   ],
   [
    -0.00187,
    -0.00268,
    -0.00086,
    0.00105,
    -0.00056,
    -0.00162,
    -7e-05,
    -0.00173,
    -0.00276,
    -0.00091,
    0.00139,
    -0.00078,
    -0.00134,
    -9e-05
   ],
   [
    -0.00259,
    -0.00251,
    -0.00057,
    -0.00171,
    -0.00037,
    0.00168,
    0.0027,
    -0.00319,
    -0.00254,
    -0.00122,
    -0.00159,
    -0.00054,
    0.00135,
    0.00216
   ],
   [
    -0.0025,
    -0.00314,
    0.00075,
    0.00211,
    0.00048,
    -0.00333,
    -0.00175,
    -0.00367,
    -0.003,
    -0.0009,
    0.00199,
    0.00015,
    -0.00333,
    -0.00162
   ],
   [
    0.00046,
    0.00036,
    0.00124,
    0.00082,
    0.00039,
    -4e-05,
    0.00057,
    0.0016,
    0.00091,
    0.00156,
    0.0013,
    0.00057,
    0.00037,
    0.00085
   ],
   [
    9e-05,
    -0.00037,
    -0.00295,
    -0.0019,
    0.001,
    0.00109,
    -0.00105,
    0.00117,
    0.00089,
    -0.00068,
    -0.00162,
    0.00059,
    0.00077,
    -0.00113
   ],
   [
    0.01498,
    -0.00073,
    -0.00757,
    0.00855,
    0.01066,
    -0.01485,
    -0.00211,
    0.00504,
    -0.00746,
    -0.00896,
    0.00805,
    0.00777,
    -0.01105,
    -0.00151
   ],
   [
    -0.01136,
    0.00018,
    -0.00418,
    -0.00649,
    0.00226,
    0.00369,
    0.00064,
    -0.00621,
    -0.00164,
    -0.00072,
    -0.00636,
    0.00132,
    0.00163,
    0.0002
   ],
   [
    -0.00199,
    -0.00162,
    -0.00101,
    -0.00181,
    -0.00208,
    0.0003,
    0.00164,
    -0.00329,
    -0.00216,
    -0.00096,
    -0.00212,
    -0.00207,
    -0.00024,
    0.00113
   ],
   [
    0.00166,
    0.00232,
    -0.00196,
    0.00126,
    -0.00041,
    -0.00115,
    -0.0009,
    0.00257,
    0.00162,
    -0.00222,
    0.00097,
    0.00014,
    -0.00049,
    -0.00063
   ],
   [
    0.0004,
    0.00262,
    0.00225,
    0.00028,
    0.00129,
    -0.00046,
    0.00023,
    -0.00075,
    0.00071,
    0.00114,
    0.00078,
    0.00114,
    -0.00026,
    0.00036
   ],
   [
    -0.00285,
    -0.01003,
    0.00061,
    -0.00208,
    -0.00459,
    0.00172,
    0.00186,
    -0.00231,
    -0.00897,
    -6e-05,
    -0.00148,
    -0.004,
    0.00149,
    0.00125
   ],
   [
    -0.00032,
    -0.00237,
    0.00074,
    -0.00049,
    -0.00029,
    0.00094,
    -0.00235,
    -0.001,
    -0.00241,
    -0.00042,
    -0.0004,
    -0.00019,
    0.0009,
    -0.00201
   ],
   [
    -0.00293,
    -0.00234,
    -0.00178,
    -0.00254,
    -0.00075,
    0.00186,
    -0.00063,
    -0.00301,
    -0.00257,
    -0.0008,
    -0.00209,
    -0.001,
    0.00146,
    -0.00044
   ],
   [
    0.00201,
    -0.00112,
    -0.00043,
    0.00135,
    -0.00443,
    -0.00208,
    0.00133,
    0.00031,
    -0.00113,
    -0.0001,
    0.00121,
    -0.00335,
    -0.00171,
    0.0008
   ],
   [
    -0.00341,
    -0.00184,
    -0.00185,
    -0.00074,
    -0.00036,
    0.00374,
    0.00106,
    -0.00165,
    -0.00082,
    -0.00064,
    -0.00028,
    -0.00015,
    0.0036,
    0.0013
   ],
   [
    0.00498,
    0.00144,
    0.00272,
    -0.00281,
    -0.00012,
    0.00166,
    0.00269,
    0.00247,
    -0.00088,
    0.00161,
    -0.00213,
    3e-05,
    0.00193,
    0.00277
   ],
   [
    0.00239,
    0.00213,
    0.0003,
    0.00088,
    -0.00233,
    0.0012,
    0.00235,
    0.00159,
    0.00082,
    0.00025,
    0.0008,
    -0.00187,
    0.00127,
    0.00225
   ],
   [
    0.00098,
    0.00284,
    -0.00389,
    -0.0006,
    0.00285,
    -0.00292,
    -0.001,
    0.00222,
    0.00271,
    -0.00138,
    -0.00068,
    0.00254,
    -0.00255,
    -0.0011
   ],
   [
    0.00017,
    -0.00211,
    -0.0,
    -0.00023,
    0.00129,
    9e-05,
    -0.00063,
    0.00014,
    -0.0006,
    8e-05,
    -0.00017,
    0.00108,
    5e-05,
    -0.00019
   ],
   [
    0.01964,
    -0.00152,
    0.00439,
    0.00505,
    -0.0048,
    -0.0111,
    0.00384,
    0.00975,
    -0.00073,
    1e-05,
    0.00382,
    -0.00456,
    -0.01005,
    0.00308
   ],
   [
    0.00238,
    0.00199,
    0.00271,
    2e-05,
    0.00069,
    0.00079,
    -3e-05,
    0.0013,
    0.00147,
    0.00134,
    0.00011,
    0.00082,
    0.00104,
    4e-05
   ],
   [
    0.00232,
    0.0008,
    0.00252,
    -0.00027,
    0.00107,
    -0.00095,
    -0.00217,
    0.00179,
    0.00085,
    0.00194,
    -3e-05,
    0.00094,
    -0.00079,
    -0.0017
   ],
   [
    0.01081,
    -0.00398,
    0.00136,
    -0.00041,
    -0.00718,
    0.00061,
    -0.00341,
    0.00653,
    -0.00578,
    0.00078,
    -0.00124,
    -0.00804,
    -0.00151,
    -0.0041
   ],
   [
    0.00266,
    0.00412,
    -0.00132,
    -0.00028,
    0.00327,
    -0.00188,
    0.00053,
    0.00206,
    0.00285,
    -0.00171,
    -0.0004,
    0.00261,
    -0.00165,
    0.00037
   ],
   [
    -0.00306,
    -0.00549,
    -0.0015,
    -0.00023,
    -0.00289,
    -0.0001,
    0.00031,
    -0.00372,
    -0.00712,
    -0.00219,
    -0.00097,
    -0.00358,
    -0.00052,
    -0.00028
   ],
   [
    -0.00405,
    -0.00117,
    0.00025,
    -0.00036,
    -0.00188,
    -0.00063,
    0.00051,
    -0.0046,
    -0.00119,
    -6e-05,
    -4e-05,
    -0.00179,
    -0.00043,
    0.00062
   ],
   [
    0.00197,
    0.00077,
    0.00138,
    0.0009,
    0.00019,
    0.00109,
    -0.00075,
    0.00217,
    0.00027,
    0.00046,
    0.00184,
    0.00135,
    0.00185,
    -0.00038
   ],
   [
    -0.00982,
    -0.01297,
    4e-05,
    0.00928,
    -0.01369,
    -0.00162,
    0.00657,
    -0.00734,
    -0.01109,
    -0.00206,
    0.00892,
    -0.01125,
    -0.00046,
    0.00536
   ],
   [
    -0.0026,
    -0.00351,
    -0.00058,
    -0.00017,
    0.00265,
    0.0004,
    -0.00347,
    -0.00352,
    -0.00307,
    -0.00063,
    -0.00011,
    0.00182,
    -0.00015,
    -0.00302
   ],
   [
    -0.00724,
    -0.01017,
    0.00049,
    0.00145,
    -0.00567,
    -0.0041,
    0.00071,
    -0.00746,
    -0.00949,
    -0.00336,
    0.00014,
    -0.0052,
    -0.00536,
    -0.00051
   ],
   [
    -0.0005,
    0.00719,
    0.00254,
    -0.00431,
    0.00489,
    0.00168,
    -0.00013,
    0.00027,
    0.00551,
    0.00428,
    -0.00337,
    0.00469,
    0.00195,
    0.00096
   ],
   [
    0.00237,
    0.00058,
    -0.00809,
    -0.00025,
    -6e-05,
    -0.00068,
    0.00202,
    0.00382,
    0.00057,
    -0.00601,
    -0.00029,
    -0.00102,
    -0.00106,
    0.00085
   ],
   [
    -0.00089,
    0.00061,
    -0.00019,
    0.00162,
    0.00192,
    -0.00049,
    -0.00152,
    -9e-05,
    0.00095,
    0.00012,
    0.00148,
    0.00174,
    -0.0001,
    -0.00103
   ],
   [
    -1e-05,
    -0.0029,
    0.00088,
    -0.00177,
    0.00047,
    -0.00055,
    -0.00055,
    -0.00059,
    -0.00328,
    -0.00012,
    -0.00179,
    -8e-05,
    -0.00064,
    -0.00049
   ],
   [
    0.0036,
    0.00109,
    0.00202,
    -0.00116,
    -0.00083,
    -0.00088,
    -0.00099,
    0.00143,
    0.00054,
    0.00018,
    -0.00133,
    -0.00042,
    -0.0012,
    -0.00126
   ],
   [
    0.00121,
    0.00153,
    -0.00042,
    -0.0017,
    0.00118,
    0.00114,
    0.00099,
    0.00051,
    0.00142,
    1e-05,
    -0.00133,
    0.00113,
    0.00155,
    0.00125
   ],
   [
    -0.00522,
    0.00342,
    -0.00062,
    -0.00029,
    0.00081,
    0.00086,
    0.0035,
    -0.00474,
    0.00291,
    3e-05,
    -4e-05,
    0.00119,
    0.00081,
    0.00329
   ],
   [
    -0.00157,
    -0.00906,
    -0.00393,
    -0.00083,
    -0.00693,
    0.00691,
    -0.00172,
    -0.00325,
    -0.00738,
    0.00033,
    0.00122,
    -0.00465,
    0.00711,
    -0.00031
   ],
   [
    0.0014,
    -0.00545,
    0.00165,
    -0.00044,
    0.00089,
    -0.00073,
    -0.00034,
    -0.00032,
    -0.00398,
    -0.00012,
    -0.00054,
    0.00045,
    -2e-05,
    0.00015
   ],
   [
    0.00203,
    -0.0038,
    0.00122,
    -0.00914,
    -0.00513,
    0.0143,
    0.01129,
    -2e-05,
    -0.0043,
    -0.00102,
    -0.00735,
    -0.00289,
    0.01402,
    0.01012
   ],
   [
    0.00144,
    0.0013,
    -0.00108,
    0.00066,
    -0.00088,
    0.00138,
    0.00127,
    0.00131,
    0.00104,
    -0.00013,
    4e-05,
    -0.00118,
    0.00103,
    0.00099
   ],
   [
    0.00118,
    -0.00101,
    -0.00098,
    0.00231,
    0.00192,
    -0.0019,
    0.00149,
    0.0024,
    0.00076,
    0.0004,
    0.00192,
    0.00136,
    -0.00129,
    0.00131
   ],
   [
    -0.00077,
    0.00074,
    -0.00063,
    0.00024,
    -0.00091,
    0.00012,
    -0.00039,
    -0.00021,
    0.00031,
    -0.00031,
    0.0,
    -0.00098,
    -0.00019,
    -0.00064
   ],
   [
    -0.00187,
    -0.00569,
    0.00104,
    0.00125,
    0.00138,
    -0.00384,
    -0.00076,
    -0.00277,
    -0.00323,
    -0.00041,
    0.00071,
    0.00062,
    -0.00427,
    -0.0014
   ],
   [
    -0.00238,
    0.00074,
    -0.00123,
    -0.00156,
    0.00588,
    0.00143,
    -0.00389,
    -0.00172,
    0.00141,
    -0.0001,
    -0.00146,
    0.00499,
    0.00182,
    -0.00278
   ],
   [
    -0.00071,
    -0.00044,
    -0.00018,
    4e-05,
    0.0011,
    -2e-05,
    -0.00064,
    -0.00116,
    -0.00035,
    -0.00044,
    -5e-05,
    0.00069,
    -0.00046,
    -0.00078
   ],
   [
    0.00221,
    0.00267,
    -0.00653,
    -0.00268,
    0.0007,
    0.00222,
    0.00089,
    0.0058,
    0.00533,
    -0.00256,
    -0.00273,
    0.00051,
    0.0026,
    0.00065
   ],
   [
    0.00068,
    -0.00066,
    0.00053,
    0.00025,
    -0.00293,
    0.00309,
    0.00277,
    0.00177,
    -0.00053,
    0.00121,
    -0.00016,
    -0.00248,
    0.00306,
    0.00261
   ],
   [
    -0.00607,
    0.00378,
    -0.00093,
    -0.00289,
    -0.00366,
    0.00159,
    0.00368,
    -0.00313,
    0.00349,
    -0.00042,
    -0.00299,
    -0.00285,
    0.00148,
    0.0027
   ],
   [
    -0.0023,
    8e-05,
    -0.0015,
    0.0007,
    0.00041,
    0.00073,
    0.00129,
    -0.00058,
    -0.00052,
    -0.00085,
    0.00028,
    0.00054,
    0.00065,
    0.00118
   ],
   [
    -0.00789,
    -0.00277,
    -0.00478,
    -0.00644,
    0.00326,
    0.00227,
    -0.00028,
    -0.00577,
    -0.00125,
    -0.00324,
    -0.0062,
    0.00263,
    0.00179,
    -0.00021
   ],
   [
    0.0016,
    -0.00298,
    0.00295,
    0.00259,
    -0.00327,
    -0.00038,
    0.00104,
    -0.00032,
    -0.00269,
    0.00162,
    0.00333,
    -0.00224,
    0.00047,
    0.00131
   ],
   [
    -0.00342,
    -0.00079,
    -0.00118,
    -0.00172,
    -0.00127,
    -0.00011,
    0.00546,
    -0.00094,
    -0.00098,
    0.00086,
    -0.00135,
    -0.00116,
    0.00011,
    0.00458
   ],
   [
    0.00189,
    -0.00109,
    -0.00144,
    0.00121,
    0.00044,
    -0.00237,
    0.00025,
    0.00059,
    -0.00117,
    -0.00149,
    0.00148,
    9e-05,
    -0.00222,
    0.00013
   ],
   [
    0.0007,
    -0.00061,
    0.00088,
    0.0035,
    -0.00235,
    -0.00081,
    0.00249,
    0.00016,
    -0.00155,
    0.00023,
    0.00319,
    -0.00213,
    -0.00051,
    0.00227
   ],
   [
    -0.00265,
    -0.00021,
    0.00089,
    -0.00258,
    -0.00335,
    0.00314,
    0.00389,
    -0.00192,
    -0.0002,
    0.00104,
    -0.00159,
    -0.00197,
    0.00258,
    0.00321
   ],
   [
    0.00101,
    -0.01704,
    -0.00193,
    -0.00019,
    -0.00885,
    0.00398,
    0.00743,
    -0.0017,
    -0.015,
    -0.00152,
    -0.00053,
    -0.00774,
    0.00368,
    0.00594
   ],
   [
    0.00086,
    0.00094,
    -0.00169,
    0.00139,
    -0.00162,
    0.00018,
    0.0015,
    0.0026,
    0.00085,
    7e-05,
    0.00097,
    -0.00114,
    0.00014,
    0.00125
   ],
   [
    0.00611,
    0.00556,
    6e-05,
    0.00151,
    0.0012,
    0.00101,
    0.00063,
    0.00577,
    0.00514,
    0.00093,
    0.00162,
    0.00192,
    0.0019,
    0.0008
   ],
   [
    -0.0022,
    0.00193,
    -0.00429,
    0.00179,
    -0.00061,
    -0.00105,
    0.00074,
    -0.00321,
    0.00145,
    -0.00232,
    0.00139,
    -0.00108,
    -0.00156,
    0.00021
   ],
   [
    0.02525,
    0.00849,
    0.00875,
    -0.00123,
    -0.00356,
    0.00381,
    0.00103,
    0.0215,
    0.01135,
    0.00695,
    -0.00115,
    -0.00151,
    0.00423,
    0.00135
   ],
   [
    -0.00244,
    -0.00129,
    0.00073,
    -0.00073,
    -0.00059,
    0.00054,
    -0.00039,
    -0.00378,
    -0.00279,
    -0.00061,
    -0.001,
    -0.00089,
    5e-05,
    -0.00035
   ],
   [
    0.00609,
    0.02186,
    -0.00085,
    0.00719,
    0.01557,
    -0.00558,
    -0.00782,
    0.01328,
    0.01904,
    0.00301,
    0.00759,
    0.01329,
    -0.00286,
    -0.0049
   ],
   [
    0.02273,
    -0.00886,
    0.0033,
    0.00939,
    -0.01558,
    0.00192,
    0.00143,
    0.01544,
    -0.00492,
    0.00288,
    0.00899,
    -0.01292,
    0.00112,
    0.00124
   ],
   [
    0.00437,
    -0.00487,
    0.00012,
    0.00141,
    0.00324,
    -0.00198,
    -0.00433,
    0.00261,
    -0.0038,
    0.00085,
    0.00099,
    0.00201,
    -0.00256,
    -0.00412
   ],
   [
    0.00123,
    0.002,
    0.00335,
    0.00102,
    -8e-05,
    -0.0014,
    0.00075,
    0.00057,
    0.00302,
    0.00238,
    0.00107,
    -7e-05,
    -0.00139,
    0.00066
   ],
   [
    -0.01062,
    0.00284,
    -0.00947,
    -0.004,
    0.00192,
    -0.0013,
    -0.00237,
    -0.01151,
    0.00182,
    -0.00832,
    -0.0035,
    0.00216,
    -0.00113,
    -0.00247
   ],
   [
    -0.0138,
    -0.01199,
    -0.00161,
    0.00514,
    -0.00156,
    -0.00013,
    0.00261,
    -0.00855,
    -0.00827,
    -0.00127,
    0.00329,
    -0.00286,
    -0.00176,
    0.00091
   ],
   [
    -0.00107,
    0.0009,
    -0.00231,
    -0.00231,
    -0.00214,
    0.00119,
    0.00421,
    0.00139,
    -0.0001,
    -0.00044,
    -0.00219,
    -0.00178,
    0.00072,
    0.00308
   ],
   [
    0.01366,
    -0.01329,
    0.00547,
    0.01029,
    -0.01587,
    0.00122,
    0.00479,
    0.0065,
    -0.00803,
    0.00151,
    0.00947,
    -0.01226,
    0.00054,
    0.00396
   ],
   [
    0.00959,
    0.00955,
    0.00858,
    -0.00217,
    -0.0022,
    -0.00294,
    0.0066,
    0.00343,
    0.00894,
    0.00446,
    -0.00185,
    -0.00114,
    -0.00197,
    0.00644
   ],
   [
    -0.0006,
    -0.00225,
    -0.00176,
    -0.00099,
    -0.00154,
    0.00162,
    0.00059,
    -0.00142,
    -0.00315,
    -0.0011,
    -0.00112,
    -0.00173,
    0.00108,
    0.00044
   ],
   [
    -0.00263,
    -0.0021,
    -0.00331,
    0.00095,
    -0.00084,
    -0.00113,
    0.00244,
    -0.00015,
    -0.00148,
    -0.00193,
    0.00052,
    -0.00063,
    -0.00147,
    0.00181
   ],
   [
    0.00226,
    0.00143,
    -0.00032,
    -0.0021,
    -0.00082,
    0.00199,
    0.00141,
    0.00169,
    0.00185,
    0.00096,
    -0.00167,
    -0.0005,
    0.00181,
    0.00134
   ],
   [
    0.00101,
    -0.0072,
    0.00011,
    0.00393,
    -0.0063,
    -0.00205,
    -0.00081,
    -0.00143,
    -0.00507,
    -0.00089,
    0.00297,
    -0.00592,
    -0.00247,
    -0.00108
   ],
   [
    0.00565,
    0.00216,
    8e-05,
    0.00213,
    -0.00346,
    0.00199,
    0.0027,
    0.00306,
    0.00136,
    -0.00109,
    0.00224,
    -0.0027,
    0.00195,
    0.00226
   ],
   [
    -0.01463,
    -0.00542,
    -0.00171,
    -0.00424,
    0.00299,
    -0.00522,
    -0.00503,
    -0.01385,
    -0.00973,
    -0.00383,
    -0.00479,
    0.00073,
    -0.00716,
    -0.00564
   ],
   [
    0.00223,
    0.00019,
    -0.00023,
    0.00141,
    -7e-05,
    0.0002,
    -0.00018,
    0.00242,
    0.00054,
    7e-05,
    0.00131,
    0.00011,
    0.00031,
    3e-05
   ],
   [
    -0.00944,
    0.00129,
    0.00134,
    0.00232,
    -0.00025,
    -0.00415,
    0.0045,
    -0.00681,
    0.0029,
    0.00102,
    0.00243,
    0.00103,
    -0.00243,
    0.00414
   ],
   [
    0.00092,
    0.00391,
    0.01155,
    0.01409,
    0.00118,
    -0.01067,
    -0.00954,
    0.00217,
    0.00293,
    0.00559,
    0.01051,
    -0.00046,
    -0.01328,
    -0.01045
   ],
   [
    2e-05,
    -0.00138,
    0.00222,
    -0.00184,
    0.00312,
    0.00095,
    -0.00066,
    -0.00044,
    -0.00061,
    0.00022,
    -0.0018,
    0.00294,
    0.00108,
    -0.00032
   ],
   [
    -0.00308,
    0.00063,
    -0.00056,
    -0.00128,
    0.00168,
    0.00083,
    -0.00086,
    -0.00244,
    0.00069,
    -0.00069,
    -0.00177,
    0.00103,
    0.0002,
    -0.00088
   ],
   [
    0.0002,
    -0.00039,
    -0.00129,
    0.00089,
    -0.00224,
    0.00071,
    0.0035,
    -0.00166,
    -0.00104,
    -0.00221,
    0.00161,
    -0.00142,
    0.00087,
    0.00298
   ],
   [
    -0.00034,
    0.00181,
    0.00108,
    0.00132,
    -0.00036,
    -0.00214,
    0.00026,
    0.0002,
    0.0021,
    0.00126,
    0.00119,
    -0.00022,
    -0.0019,
    0.00031
   ],
   [
    -0.00126,
    -0.00176,
    -0.003,
    -0.00335,
    -0.00093,
    0.00221,
    -0.0008,
    0.0001,
    -0.00256,
    -0.00224,
    -0.00415,
    -0.00203,
    0.0007,
    -0.00084
   ],
   [
    -0.00361,
    -0.00174,
    -0.00058,
    -0.00235,
    -0.00035,
    0.00157,
    0.0015,
    -0.00236,
    -0.00124,
    -1e-05,
    -0.00218,
    -0.00068,
    0.00104,
    0.00122
   ],
   [
    0.00086,
    -0.0007,
    0.00117,
    0.00084,
    0.0011,
    -0.00293,
    -0.00275,
    0.00095,
    -0.00143,
    0.00024,
    0.00111,
    0.00101,
    -0.00257,
    -0.00214
   ],
   [
    0.00035,
    -0.00038,
    -0.00014,
    1e-05,
    -6e-05,
    0.00029,
    -0.00097,
    0.00102,
    -0.00122,
    -0.00058,
    0.00033,
    4e-05,
    0.00052,
    -0.00073
   ],
   [
    -0.00041,
    -0.00319,
    0.00041,
    -0.00084,
    -0.00022,
    -0.00394,
    0.00185,
    -0.00115,
    -0.00252,
    -0.00037,
    -0.00115,
    -0.00052,
    -0.00358,
    0.00131
   ],
   [
    -0.00316,
    0.00126,
    0.00209,
    -0.00052,
    0.00107,
    -0.00189,
    -0.00017,
    -0.00245,
    0.00106,
    0.00208,
    4e-05,
    0.00091,
    -0.00126,
    0.0001
   ],
   [
    -0.00329,
    -0.00327,
    -0.00056,
    -0.00114,
    -0.00172,
    -0.00156,
    0.00029,
    -0.0039,
    -0.00377,
    -0.0012,
    -0.00105,
    -0.00185,
    -0.00151,
    0.0001
   ],
   [
    0.02413,
    -0.01251,
    0.01776,
    0.02264,
    -0.02946,
    0.00531,
    0.01115,
    0.02264,
    -0.00289,
    0.01395,
    0.02273,
    -0.02083,
    0.01039,
    0.012
   ],
   [
    -0.0021,
    -0.00125,
    -0.00232,
    -0.001,
    -0.00152,
    -0.00041,
    0.00375,
    -0.0014,
    -0.00167,
    -0.00145,
    -0.00083,
    -0.00111,
    -0.00027,
    0.00309
   ],
   [
    -0.00411,
    -0.00764,
    -0.00348,
    -0.00272,
    0.00282,
    0.00441,
    -0.00455,
    -0.00457,
    -0.00557,
    -0.00242,
    -0.00285,
    0.00213,
    0.00346,
    -0.00388
   ],
   [
    0.01021,
    -0.00388,
    -0.00376,
    -0.011,
    -0.01018,
    0.01329,
    0.014,
    0.01661,
    6e-05,
    0.0024,
    -0.01222,
    -0.00881,
    0.01445,
    0.01283
   ],
   [
    0.00334,
    -0.00017,
    0.00018,
    0.00312,
    -0.0025,
    -0.00212,
    0.00132,
    0.00192,
    -0.00161,
    -0.00078,
    0.003,
    -0.00184,
    -0.00174,
    0.00105
   ],
   [
    -0.00041,
    -0.00195,
    -0.00067,
    0.00294,
    0.00016,
    -0.00348,
    -0.00225,
    -0.00122,
    -0.00171,
    -0.002,
    0.00283,
    0.00047,
    -0.0031,
    -0.00196
   ],
   [
    -0.00467,
    -0.00122,
    -0.00236,
    0.00062,
    -2e-05,
    0.00156,
    0.00092,
    -0.00302,
    0.00024,
    -0.00012,
    0.00173,
    0.00055,
    0.0022,
    0.00145
   ],
   [
    -0.00264,
    -0.00051,
    -0.00043,
    -0.00044,
    0.00175,
    -0.0015,
    -0.00233,
    -0.00209,
    -0.00031,
    -0.00027,
    -0.00087,
    0.00109,
    -0.00207,
    -0.00263
   ],
   [
    0.00151,
    -0.00033,
    0.00043,
    0.0018,
    -0.00082,
    -0.00019,
    0.00039,
    0.00154,
    0.00082,
    0.00089,
    0.00164,
    -0.00029,
    0.0004,
    0.00069
   ],
   [
    0.01654,
    -0.01722,
    0.01108,
    -0.00082,
    0.00064,
    0.00582,
    -7e-05,
    0.0071,
    -0.011,
    -0.00167,
    -0.00038,
    0.00286,
    0.00621,
    3e-05
   ],
   [
    0.0035,
    -0.00498,
    -0.00565,
    -0.00278,
    -0.0015,
    -0.0025,
    -0.00772,
    0.00195,
    -0.00341,
    -0.00526,
    -0.00443,
    -0.00268,
    -0.00369,
    -0.00755
   ],
   [
    0.00515,
    -0.01444,
    0.00055,
    0.01088,
    -0.00102,
    -0.0094,
    -0.00038,
    -0.00406,
    -0.0164,
    -0.00449,
    0.01126,
    -0.0017,
    -0.0084,
    -0.00085
   ],
   [
    -0.00413,
    0.00074,
    -0.00099,
    -0.01518,
    0.00131,
    0.01065,
    0.00226,
    -0.00721,
    0.00047,
    -0.00317,
    -0.01528,
    0.00035,
    0.0079,
    0.0013
   ],
   [
    0.00505,
    0.00158,
    0.0055,
    0.00486,
    -0.00128,
    -0.00503,
    0.00205,
    0.00802,
    0.00149,
    0.00443,
    0.00426,
    -0.00019,
    -0.00463,
    0.0017
   ],
   [
    0.00213,
    0.00074,
    -0.00023,
    0.00084,
    -0.00025,
    0.00128,
    -0.00213,
    0.00107,
    0.00047,
    7e-05,
    0.00134,
    4e-05,
    0.00144,
    -0.00171
   ],
   [
    0.0007,
    0.00052,
    0.00134,
    -0.00138,
    -0.00149,
    0.00094,
    0.00119,
    -0.00041,
    -0.00051,
    0.00078,
    -0.00133,
    -0.001,
    0.00118,
    0.00099
   ],
   [
    -0.00347,
    -0.00181,
    -0.00294,
    -0.00088,
    0.00024,
    0.00085,
    0.00182,
    -0.00306,
    -0.0021,
    -0.00164,
    -0.00102,
    -0.00011,
    0.00129,
    0.00152
   ],
   [
    -0.00101,
    0.00248,
    -0.00106,
    -0.00088,
    -0.00033,
    0.0005,
    -0.00051,
    0.00036,
    0.0013,
    -0.00025,
    -0.00132,
    -0.0008,
    -2e-05,
    -0.00078
   ],
   [
    0.00114,
    0.00037,
    0.00195,
    0.00288,
    0.00052,
    -0.00224,
    0.00037,
    -0.00012,
    -0.00068,
    0.00022,
    0.00281,
    0.00071,
    -0.00158,
    0.00038
   ],
   [
    -0.0012,
    -0.00231,
    -0.00158,
    0.0008,
    -0.00023,
    0.00114,
    0.00173,
    -0.00046,
    -0.0016,
    -0.00232,
    -4e-05,
    -0.00098,
    0.00074,
    0.00155
   ],
   [
    -0.00485,
    -0.00331,
    9e-05,
    -1e-05,
    -0.00194,
    -0.00202,
    0.00175,
    -0.00367,
    -0.00271,
    -4e-05,
    -0.00027,
    -0.00196,
    -0.00239,
    0.00118
   ],
   [
    -0.00026,
    0.00022,
    -0.00087,
    -0.00108,
    0.00465,
    0.00063,
    0.00014,
    0.0021,
    -0.00146,
    -5e-05,
    -0.00091,
    0.00369,
    -1e-05,
    -0.00013
   ],
   [
    0.00463,
    0.00062,
    0.01133,
    -0.00215,
    0.01179,
    -0.00684,
    -0.00281,
    -0.00108,
    -0.00263,
    0.00072,
    -0.00127,
    0.01023,
    -0.00656,
    -0.00236
   ],
   [
    0.00093,
    0.00062,
    -0.00057,
    0.00367,
    0.00512,
    -0.00385,
    -0.00571,
    0.00205,
    0.00636,
    -0.00052,
    0.00374,
    0.00562,
    -0.00338,
    -0.00477
   ],
   [
    0.00243,
    0.00578,
    0.00016,
    0.00129,
    0.00142,
    0.00028,
    2e-05,
    0.00368,
    0.00482,
    0.00036,
    0.00043,
    0.00088,
    3e-05,
    0.00035
   ],
   [
    0.00342,
    -0.00139,
    -0.00163,
    2e-05,
    -0.00184,
    0.00205,
    -0.00065,
    0.00207,
    -0.00129,
    -0.00116,
    -5e-05,
    -0.00165,
    0.00111,
    -0.00102
   ],
   [
    0.00281,
    8e-05,
    -0.00059,
    0.00313,
    -0.00129,
    -0.0004,
    0.00174,
    0.00233,
    0.00135,
    -0.0006,
    0.00341,
    -7e-05,
    0.00058,
    0.00184
   ],
   [
    -0.00249,
    -0.00343,
    0.00094,
    -3e-05,
    0.00129,
    -0.00168,
    -0.00065,
    -0.0028,
    -0.00379,
    -0.00071,
    0.0002,
    0.00116,
    -0.00137,
    -0.00066
   ],
   [
    0.00678,
    0.00131,
    0.00254,
    0.00203,
    -0.00673,
    0.00943,
    -0.00271,
    0.00841,
    0.00458,
    0.00011,
    1e-05,
    -0.00359,
    0.0081,
    -0.00135
   ],
   [
    0.01676,
    0.00234,
    0.00094,
    0.00344,
    -0.00597,
    0.00264,
    0.00381,
    0.02141,
    0.00846,
    0.00355,
    0.00331,
    -0.00451,
    0.00383,
    0.00315
   ],
   [
    -0.00325,
    0.00176,
    -0.00528,
    -0.00051,
    0.0112,
    -0.00129,
    -0.00266,
    0.00296,
    0.00429,
    -0.00045,
    0.00043,
    0.01118,
    0.00015,
    -0.00097
   ],
   [
    0.00107,
    0.00171,
    0.00133,
    0.00182,
    0.0031,
    -0.00287,
    -0.00342,
    0.0005,
    0.00236,
    0.00146,
    0.00214,
    0.00274,
    -0.00208,
    -0.00243
   ],
   [
    -0.00432,
    -0.00061,
    -0.00461,
    0.00045,
    0.00542,
    -0.00384,
    -0.00492,
    -0.00638,
    -0.0032,
    -0.0051,
    0.00027,
    0.00364,
    -0.00374,
    -0.00427
   ],
   [
    -0.00403,
    0.00047,
    -0.00256,
    -0.0043,
    -0.00269,
    0.00668,
    0.00717,
    -0.00218,
    0.00066,
    0.00146,
    -0.00341,
    -0.00149,
    0.00619,
    0.0064
   ],
   [
    -9e-05,
    0.00272,
    0.00103,
    0.00056,
    -0.00032,
    6e-05,
    0.00657,
    0.00029,
    0.00212,
    0.0012,
    0.00033,
    0.00011,
    0.00087,
    0.00617
   ],
   [
    -0.00113,
    9e-05,
    -0.0041,
    0.00177,
    0.00043,
    0.00158,
    0.0018,
    0.00013,
    0.00071,
    -0.00145,
    0.00197,
    0.00074,
    0.00228,
    0.00202
   ],
   [
    0.00302,
    -0.00187,
    -0.00112,
    -0.00122,
    -0.00237,
    0.00334,
    0.00212,
    0.00038,
    -0.00132,
    -0.00077,
    -0.00111,
    -0.00205,
    0.0024,
    0.00208
   ],
   [
    0.00124,
    -0.003,
    -0.00088,
    0.00085,
    -0.00455,
    -0.00132,
    -0.00065,
    0.00029,
    -0.00313,
    -0.00177,
    0.00072,
    -0.00387,
    -0.00165,
    -0.00099
   ],
   [
    -0.00127,
    -0.00212,
    0.00023,
    0.0014,
    0.00157,
    -0.00221,
    -0.00202,
    -0.00202,
    -0.00129,
    -0.00122,
    0.00095,
    0.00088,
    -0.00198,
    -0.00176
   ],
   [
    -0.00507,
    -0.00466,
    -0.00142,
    -0.00209,
    -0.00278,
    0.00294,
    0.00181,
    -0.0035,
    -0.00368,
    -0.00058,
    -0.00257,
    -0.00292,
    0.0019,
    0.00137
   ],
   [
    0.00773,
    0.00399,
    0.00436,
    0.00269,
    -0.0016,
    -0.0009,
    -0.01458,
    -0.0036,
    -0.0023,
    -0.00024,
    0.00074,
    -0.00448,
    -0.00398,
    -0.01378
   ],
   [
    0.00078,
    0.00156,
    -0.0,
    -0.00106,
    0.00583,
    -0.00288,
    -0.00549,
    -0.00057,
    0.0012,
    -0.00153,
    -0.00155,
    0.00414,
    -0.00298,
    -0.00478
   ],
   [
    0.00315,
    0.00291,
    0.00469,
    -0.00172,
    0.00379,
    -0.00283,
    -0.00512,
    1e-05,
    0.00241,
    0.00194,
    -0.00222,
    0.00245,
    -0.00352,
    -0.00478
   ],
   [
    0.00302,
    -0.00041,
    -0.00122,
    -0.00065,
    -0.0002,
    7e-05,
    -0.00088,
    0.00202,
    0.00074,
    -0.00062,
    -0.00034,
    -0.00019,
    0.00024,
    -0.00091
   ],
   [
    -0.00185,
    -0.00043,
    -0.00126,
    -0.00072,
    0.00121,
    0.0001,
    -0.00374,
    -0.00199,
    -0.00145,
    -0.00144,
    -0.00096,
    0.00038,
    -0.00068,
    -0.00371
   ],
   [
    0.02346,
    0.00446,
    0.00536,
    -0.00167,
    -0.00046,
    0.0047,
    -0.00045,
    0.02131,
    0.00722,
    0.0059,
    -0.00028,
    0.00037,
    0.00495,
    0.00046
   ],
   [
    0.00518,
    0.00748,
    0.00277,
    0.00055,
    -0.00545,
    0.00276,
    -0.00207,
    0.00512,
    0.00548,
    0.00335,
    0.00185,
    -0.00306,
    0.0026,
    -0.00081
   ],
   [
    -0.00117,
    0.00061,
    5e-05,
    -0.00094,
    -0.00183,
    0.00057,
    0.00353,
    0.00095,
    0.00169,
    0.0012,
    -0.00091,
    -0.00147,
    0.00071,
    0.00296
   ],
   [
    0.00093,
    0.00059,
    0.00249,
    0.00021,
    0.00272,
    -0.00014,
    -0.00424,
    -0.00117,
    -0.00163,
    -0.0001,
    -0.00017,
    0.00181,
    -0.00078,
    -0.00356
   ],
   [
    0.00108,
    -7e-05,
    -0.0022,
    -0.00254,
    -0.00275,
    0.00037,
    0.00186,
    0.00028,
    -0.00041,
    -0.00025,
    -0.00199,
    -0.00225,
    -0.00028,
    0.00142
   ],
   [
    -0.00252,
    -0.00594,
    2e-05,
    0.00452,
    -0.00186,
    -0.00347,
    0.00134,
    -0.00289,
    -0.00642,
    0.00036,
    0.00439,
    -0.00157,
    -0.00285,
    0.00143
   ],
   [
    0.00261,
    0.00146,
    0.00146,
    0.00218,
    0.00176,
    -0.00172,
    -0.00646,
    0.00116,
    0.00176,
    0.00219,
    0.00208,
    0.00185,
    -0.00077,
    -0.00488
   ],
   [
    -0.00485,
    -0.00087,
    0.00012,
    -0.00126,
    -0.00069,
    0.00151,
    0.00373,
    -0.00463,
    -0.00114,
    0.00037,
    -0.00064,
    -0.00023,
    0.0016,
    0.0034
   ],
   [
    0.01237,
    -0.00227,
    0.00412,
    0.00535,
    -0.01164,
    -0.00517,
    -0.0065,
    0.00524,
    -0.00447,
    0.0012,
    0.00545,
    -0.01097,
    -0.0059,
    -0.00708
   ],
   [
    -0.00106,
    -0.00128,
    -0.00372,
    -0.00164,
    -0.00109,
    0.00123,
    0.00097,
    -0.00142,
    -0.00201,
    -0.0024,
    -0.00082,
    -0.00044,
    0.00068,
    0.00082
   ],
   [
    0.00089,
    0.00047,
    -0.00052,
    0.00151,
    -0.00026,
    0.00077,
    -0.00026,
    0.0024,
    0.00194,
    0.00168,
    0.00207,
    0.00022,
    0.0013,
    0.00016
   ],
   [
    0.00678,
    -0.00932,
    0.00307,
    -0.00255,
    0.00054,
    0.00022,
    0.00811,
    0.00501,
    -0.00675,
    0.00222,
    -0.0026,
    0.00126,
    0.00089,
    0.0072
   ],
   [
    -0.00033,
    0.00346,
    0.00183,
    -0.00109,
    0.00105,
    -0.00079,
    -0.00146,
    -0.00145,
    0.0026,
    0.00024,
    -0.00124,
    0.00077,
    -0.0013,
    -0.00157
   ],
   [
    0.00179,
    0.00057,
    0.00053,
    -0.0011,
    -0.00085,
    0.00051,
    -0.00189,
    0.00184,
    0.00027,
    0.00053,
    -0.00157,
    -0.0013,
    -0.00014,
    -0.00199
   ],
   [
    -0.00021,
    -0.00955,
    -0.00025,
    0.00693,
    -0.00989,
    -0.00369,
    -0.00061,
    0.00121,
    -0.00515,
    -0.00059,
    0.00603,
    -0.00917,
    -0.00442,
    -0.00101
   ],
   [
    2e-05,
    -0.00243,
    0.00015,
    0.00133,
    -0.00113,
    -0.00184,
    0.00106,
    -3e-05,
    -0.0024,
    -0.00052,
    0.00098,
    -0.00126,
    -0.00172,
    0.00062
   ],
   [
    -0.00411,
    0.00293,
    0.00198,
    0.00284,
    0.00299,
    -0.00356,
    -0.00013,
    -0.0016,
    0.00247,
    0.00301,
    0.00278,
    0.00253,
    -0.00271,
    0.00033
   ],
   [
    0.00043,
    0.00312,
    0.00024,
    0.00086,
    0.0026,
    -0.00104,
    0.0002,
    0.00164,
    0.00273,
    -1e-05,
    -0.00011,
    0.00205,
    -0.00154,
    0.0001
   ],
   [
    0.00275,
    -0.01668,
    -0.004,
    -0.00201,
    -0.00077,
    0.00456,
    0.00173,
    0.00136,
    -0.00975,
    -0.00262,
    -0.00135,
    -0.00057,
    0.00302,
    0.00157
   ],
   [
    -0.00411,
    0.0002,
    -0.00072,
    -0.00084,
    0.003,
    -0.00134,
    -0.00431,
    -0.0047,
    -0.00035,
    0.0,
    2e-05,
    0.00303,
    -0.00118,
    -0.00348
   ],
   [
    0.00537,
    -0.00781,
    0.0002,
    -0.00261,
    -0.00957,
    -0.00208,
    0.00611,
    0.0027,
    -0.00489,
    -0.00275,
    -0.00365,
    -0.00916,
    -0.00486,
    0.00271
   ],
   [
    0.00349,
    0.00189,
    0.00153,
    0.00085,
    -0.00287,
    -0.00019,
    0.00147,
    0.00375,
    0.00185,
    0.0016,
    0.00088,
    -0.00198,
    -0.00021,
    0.00127
   ],
   [
    -0.00325,
    -0.00195,
    0.00093,
    -0.00039,
    -0.00213,
    0.00181,
    0.00299,
    -0.00253,
    -0.00285,
    0.00015,
    -0.00076,
    -0.0016,
    0.00184,
    0.00267
   ],
   [
    -0.00805,
    0.00307,
    -0.00877,
    0.00268,
    -0.00136,
    0.00045,
    -0.00759,
    -0.00459,
    0.00216,
    -0.00497,
    0.00163,
    -0.0025,
    -0.00142,
    -0.00666
   ],
   [
    -0.00193,
    -0.0032,
    -0.00081,
    -0.0007,
    -0.00148,
    -0.00126,
    0.00206,
    -0.00105,
    -0.00068,
    0.0002,
    -0.00116,
    -0.00149,
    -0.00121,
    0.00147
   ],
   [
    0.00093,
    -0.00058,
    -0.00256,
    -0.00261,
    0.00079,
    0.00046,
    -0.0044,
    -0.00052,
    -0.00171,
    -0.00257,
    -0.00209,
    0.00046,
    -0.00045,
    -0.00449
   ],
   [
    -0.0049,
    -0.00071,
    0.00068,
    -0.00292,
    0.00434,
    -0.00271,
    0.00055,
    -0.00588,
    -0.00197,
    -0.00039,
    -0.00266,
    0.00375,
    -0.00187,
    0.00036
   ],
   [
    0.0016,
    -0.0079,
    0.00054,
    -0.00209,
    -0.00098,
    0.0015,
    -0.00267,
    0.00123,
    -0.00705,
    0.00035,
    -0.0016,
    -0.00063,
    -0.00052,
    -0.00333
   ],
   [
    0.00191,
    -0.00186,
    0.00063,
    0.00113,
    -0.00227,
    -0.00214,
    0.00378,
    0.00071,
    -0.00037,
    0.00093,
    0.00156,
    -0.00147,
    -0.00114,
    0.00357
   ],
   [
    -0.00828,
    0.00589,
    -0.00255,
    -0.00899,
    -0.00276,
    0.00231,
    0.00456,
    -0.01028,
    0.00157,
    -0.00448,
    -0.00995,
    -0.00385,
    6e-05,
    0.00235
   ],
   [
    -0.00265,
    -0.00189,
    0.00096,
    -0.00126,
    0.00011,
    0.00258,
    0.00485,
    -0.00345,
    -0.00185,
    0.0001,
    -0.00069,
    0.00057,
    0.00304,
    0.00425
   ],
   [
    -0.00594,
    -0.00259,
    -0.00105,
    0.00065,
    -0.00188,
    -0.00246,
    0.00437,
    -0.00433,
    -0.00178,
    -0.0009,
    0.00056,
    -0.00147,
    -0.00283,
    0.00331
   ],
   [
    -0.00137,
    0.00252,
    8e-05,
    0.00029,
    9e-05,
    -0.00121,
    0.00139,
    -0.00139,
    0.00385,
    0.00077,
    0.00106,
    0.00069,
    -0.00098,
    0.00114
   ],
   [
    -0.00294,
    -0.0021,
    -0.00481,
    -0.00503,
    -0.0007,
    0.00322,
    0.00051,
    -0.00157,
    -0.00236,
    -0.0017,
    -0.0043,
    -0.00065,
    0.00306,
    0.00065
   ],
   [
    0.00017,
    -0.0016,
    0.00158,
    -0.00189,
    -0.00015,
    0.00199,
    -0.00234,
    0.0011,
    -0.00153,
    0.00148,
    -0.00149,
    -0.00026,
    0.00143,
    -0.00202
   ],
   [
    -0.00104,
    0.00662,
    0.00135,
    -0.00515,
    -0.00127,
    0.00038,
    0.00433,
    -4e-05,
    0.00482,
    0.00011,
    -0.00412,
    -0.00022,
    8e-05,
    0.00254
   ],
   [
    0.0017,
    0.00082,
    -0.0002,
    -0.00225,
    0.00117,
    0.00039,
    0.00022,
    -0.00063,
    2e-05,
    -0.00033,
    -0.00253,
    0.00068,
    0.00049,
    -2e-05
   ],
   [
    0.00647,
    0.00024,
    0.00034,
    0.00191,
    -6e-05,
    -0.0012,
    9e-05,
    0.0066,
    1e-05,
    0.00042,
    0.0018,
    9e-05,
    -0.00029,
    0.00044
   ],
   [
    0.00269,
    -0.00014,
    -0.00339,
    0.0007,
    0.00208,
    0.00107,
    -2e-05,
    0.00309,
    -0.00043,
    -0.00167,
    0.00068,
    0.00143,
    0.00134,
    0.00041
   ],
   [
    -0.00305,
    -0.00161,
    0.00172,
    0.00199,
    -8e-05,
    -0.00225,
    -0.0006,
    -0.00164,
    -0.00087,
    0.0012,
    0.00205,
    5e-05,
    -0.00171,
    -0.00063
   ],
   [
    0.00056,
    -6e-05,
    -0.00239,
    -0.00135,
    0.00154,
    -0.00273,
    -0.0029,
    0.00061,
    -0.00135,
    -0.00388,
    -0.00208,
    0.00041,
    -0.00297,
    -0.00285
   ],
   [
    0.00119,
    0.00231,
    0.00052,
    -0.00147,
    0.00253,
    -0.00109,
    -0.001,
    -0.00032,
    0.00236,
    0.00086,
    -0.00153,
    0.00209,
    -0.00105,
    -0.00116
   ],
   [
    -0.00636,
    0.00233,
    -0.00035,
    -0.00212,
    -0.00309,
    0.00733,
    0.00395,
    -0.00409,
    0.00082,
    0.00045,
    -0.00157,
    -0.00217,
    0.00756,
    0.0042
   ],
   [
    -0.00817,
    -0.00545,
    -0.0019,
    0.00366,
    0.00909,
    -0.00394,
    -0.01548,
    -0.0099,
    -0.0081,
    -0.00444,
    0.0028,
    0.00592,
    -0.00548,
    -0.01438
   ],
   [
    0.00141,
    -8e-05,
    -0.00119,
    0.00064,
    -0.00036,
    0.0003,
    0.00117,
    0.00079,
    -0.0001,
    -0.0005,
    0.0004,
    -0.00067,
    0.00013,
    0.00096
   ],
   [
    -0.00217,
    -0.00297,
    -0.00346,
    0.00026,
    0.00143,
    -0.00045,
    -0.00303,
    -0.00196,
    -0.0028,
    -0.00196,
    -7e-05,
    0.00093,
    -0.00098,
    -0.00281
   ],
   [
    -0.00138,
    -0.00178,
    0.00058,
    -0.00032,
    0.00086,
    0.00062,
    0.00023,
    -0.00092,
    -0.00162,
    0.00046,
    -7e-05,
    0.00092,
    0.00052,
    0.00032
   ],
   [
    3e-05,
    -0.00177,
    0.00123,
    -0.00041,
    -0.00182,
    0.00059,
    0.001,
    0.00079,
    -0.00147,
    0.00085,
    -0.00063,
    -0.0016,
    0.00056,
    0.00093
   ],
   [
    0.00163,
    0.00011,
    0.00073,
    -0.0003,
    -0.00489,
    0.00145,
    0.00443,
    0.00082,
    -0.00018,
    0.00042,
    -0.00023,
    -0.00377,
    0.00183,
    0.00393
   ],
   [
    0.00433,
    0.00373,
    0.00215,
    0.00241,
    0.00174,
    -0.00223,
    0.00131,
    0.00313,
    0.00252,
    -0.00011,
    0.00166,
    0.00125,
    -0.002,
    0.00116
   ],
   [
    -0.00131,
    -0.00176,
    -9e-05,
    -0.00209,
    -0.00052,
    0.00214,
    0.00216,
    -0.00213,
    -0.00043,
    -0.00127,
    -0.00185,
    -0.0004,
    0.00227,
    0.00194
   ],
   [
    0.00285,
    -0.00102,
    -0.00044,
    0.0009,
    -0.00321,
    0.00133,
    0.00024,
    0.00214,
    -0.00057,
    0.00038,
    0.00028,
    -0.00291,
    0.00097,
    0.00026
   ],
   [
    0.00687,
    0.00382,
    0.00428,
    0.001,
    0.00224,
    -0.00076,
    -0.00069,
    0.00706,
    0.00551,
    0.00412,
    0.00246,
    0.00313,
    0.00058,
    0.00017
   ],
   [
    -0.00129,
    -0.00074,
    -0.00063,
    -0.00061,
    0.00043,
    0.00032,
    0.00026,
    -0.00128,
    -0.00137,
    -0.00043,
    -0.00149,
    -0.00033,
    2e-05,
    -0.00021
   ],
   [
    -0.00021,
    -0.00354,
    -0.00055,
    0.00096,
    -0.00307,
    -0.00048,
    0.00213,
    -0.00063,
    -0.00287,
    0.00019,
    0.00084,
    -0.00282,
    -0.00053,
    0.00173
   ],
   [
    0.00089,
    -0.0024,
    -0.00164,
    -0.00215,
    0.00151,
    0.00087,
    -0.00186,
    0.00214,
    -0.00248,
    -6e-05,
    -0.00158,
    0.00083,
    0.00138,
    -0.00142
   ],
   [
    -0.0024,
    -0.00391,
    -0.00618,
    0.00141,
    -0.00302,
    0.00059,
    0.00044,
    0.00025,
    -0.00555,
    -0.00408,
    0.00117,
    -0.00282,
    -0.00028,
    -0.00026
   ],
   [
    0.00188,
    -0.00238,
    0.00223,
    0.00014,
    -0.00241,
    -0.0004,
    0.00397,
    -0.00057,
    -0.00299,
    7e-05,
    0.00011,
    -0.00197,
    -0.0004,
    0.00318
   ],
   [
    0.00569,
    -0.00264,
    0.0056,
    0.00554,
    -0.00402,
    -0.00072,
    -0.00082,
    0.0073,
    -0.00065,
    0.00458,
    0.00489,
    -0.00293,
    -0.00045,
    -0.00102
   ],
   [
    -0.00029,
    0.00245,
    5e-05,
    -0.00045,
    -0.0017,
    9e-05,
    0.00053,
    -0.00048,
    0.00149,
    0.00011,
    -0.00041,
    -0.00123,
    -5e-05,
    0.00064
   ],
   [
    0.00239,
    0.00267,
    0.00506,
    -0.00045,
    -0.00119,
    0.00207,
    0.00207,
    0.00291,
    0.0031,
    0.0052,
    0.00063,
    -0.00035,
    0.00308,
    0.00268
   ],
   [
    -0.00045,
    0.00041,
    0.00103,
    0.00108,
    -6e-05,
    -0.00025,
    0.00126,
    0.00037,
    -0.00146,
    -0.00072,
    0.00126,
    4e-05,
    0.00064,
    0.00156
   ],
   [
    0.00418,
    0.00174,
    7e-05,
    0.00154,
    -0.001,
    -0.00071,
    0.00092,
    0.00304,
    0.00161,
    -6e-05,
    0.00164,
    -0.0007,
    -1e-05,
    0.00101
   ],
   [
    -0.0004,
    0.00096,
    -0.00127,
    0.00016,
    0.00025,
    5e-05,
    0.00035,
    -0.00059,
    -0.00012,
    -0.00347,
    0.00028,
    0.00089,
    -0.00012,
    0.00039
   ],
   [
    -0.00299,
    -0.00025,
    -0.00172,
    0.00188,
    -0.00037,
    0.00064,
    0.00033,
    -0.00179,
    -1e-05,
    -0.00038,
    0.00206,
    4e-05,
    0.00063,
    0.0006
   ],
   [
    0.00513,
    -0.01062,
    0.00413,
    0.00248,
    -0.00358,
    -0.00331,
    6e-05,
    0.00866,
    -0.00598,
    0.00552,
    0.00172,
    -0.00378,
    -0.0034,
    -0.00051
   ],
   [
    -0.00015,
    0.00145,
    0.00154,
    0.00023,
    -0.00038,
    -0.00042,
    -0.00078,
    0.00021,
    0.00209,
    0.00185,
    -0.00017,
    -0.00054,
    -0.00064,
    -0.00076
   ],
   [
    -0.01186,
    -0.01017,
    -0.00501,
    -0.00515,
    0.00207,
    0.0129,
    -0.00909,
    -0.00804,
    -0.00947,
    -0.00124,
    -0.0039,
    0.00182,
    0.00857,
    -0.00809
   ],
   [
    -0.0043,
    -0.0023,
    -0.00076,
    0.00235,
    0.00096,
    -0.00166,
    -6e-05,
    -0.00318,
    -0.0035,
    -0.00087,
    0.00182,
    0.00045,
    -0.002,
    -8e-05
   ],
   [
    -0.00411,
    -0.00278,
    0.00023,
    -0.00021,
    -0.00018,
    -0.00274,
    -0.00158,
    -0.00293,
    -0.00188,
    0.00083,
    -0.0002,
    -0.00069,
    -0.00256,
    -0.0017
   ],
   [
    -0.00086,
    0.00043,
    -0.00088,
    0.00274,
    -0.00197,
    0.00046,
    0.00097,
    -0.00115,
    0.00079,
    -0.00097,
    0.00242,
    -0.00164,
    0.00029,
    0.0009
   ],
   [
    -0.00284,
    0.00847,
    0.00191,
    0.00217,
    0.01144,
    -0.00684,
    -0.00684,
    -0.00223,
    0.00321,
    -5e-05,
    0.00175,
    0.00879,
    -0.00713,
    -0.00571
   ],
   [
    -0.01964,
    -0.00139,
    0.00013,
    0.0028,
    -0.00745,
    0.00568,
    0.00096,
    -0.02008,
    -0.00468,
    -0.00354,
    0.00141,
    -0.00815,
    0.00272,
    0.00094
   ],
   [
    0.00149,
    -0.00738,
    -0.00071,
    -0.00179,
    -0.00571,
    0.00392,
    0.00308,
    0.00147,
    -0.00512,
    0.00096,
    -0.0007,
    -0.00453,
    0.00319,
    0.00303
   ],
   [
    -0.00165,
    0.00316,
    0.0018,
    -0.0007,
    0.00215,
    0.00032,
    0.00025,
    -0.00105,
    0.0031,
    0.00066,
    -0.0012,
    0.00145,
    0.00035,
    0.00014
   ],
   [
    0.02748,
    -0.0106,
    0.01054,
    0.01478,
    -0.02249,
    -0.01173,
    0.00747,
    0.02171,
    -0.0066,
    0.00774,
    0.01226,
    -0.02039,
    -0.01109,
    0.00485
   ],
   [
    0.00569,
    -7e-05,
    0.00209,
    -0.00019,
    -0.0025,
    0.00444,
    -0.00034,
    0.00318,
    -0.00031,
    0.00162,
    -0.00059,
    -0.00228,
    0.00377,
    -0.00035
   ],
   [
    0.00112,
    -0.00087,
    -0.00358,
    -0.00073,
    0.0013,
    0.00152,
    -0.00059,
    0.0024,
    -0.0008,
    -0.00122,
    -0.00039,
    0.00108,
    0.00149,
    -0.00048
   ],
   [
    0.00097,
    -0.00218,
    0.00051,
    0.00069,
    0.00038,
    -0.00133,
    -0.00031,
    -0.00021,
    -0.00265,
    0.00012,
    0.00076,
    0.0001,
    -0.00154,
    -0.00042
   ],
   [
    -0.04016,
    0.14989,
    0.25065,
    0.38668,
    0.4305,
    0.42733,
    0.41328,
    -0.01514,
    0.12929,
    0.20625,
    0.34317,
    0.3741,
    0.38059,
    0.3582
   ]
  ],
  "inst14": [
   [
    -0.00296,
    -0.00915,
    0.00352,
    -0.00277,
    -0.00845,
    0.00061,
    -0.00197,
    -0.00422,
    -0.00965,
    0.00434,
    0.00162,
    -0.0068,
    0.00029,
    -0.00081
   ],
   [
    -0.01753,
    -0.00459,
    5e-05,
    -0.0086,
    -0.00093,
    0.00832,
    0.00264,
    -0.01459,
    -0.00414,
    0.00395,
    -0.0049,
    0.00135,
    0.01029,
    0.00499
   ],
   [
    0.01706,
    -0.00306,
    0.00363,
    -0.00241,
    -0.00805,
    0.00435,
    0.01419,
    0.01322,
    -0.00039,
    0.00777,
    -0.00264,
    -0.00983,
    0.00011,
    0.01062
   ],
   [
    -0.02464,
    -0.01537,
    -0.00484,
    -0.00011,
    0.00283,
    0.00395,
    -0.00013,
    -0.02281,
    -0.01401,
    -0.00231,
    0.00072,
    0.00304,
    0.00332,
    0.00247
   ],
   [
    0.00803,
    -0.0144,
    -0.0015,
    0.00632,
    -0.00337,
    0.00543,
    0.00566,
    0.00739,
    -0.01068,
    -0.00218,
    0.0083,
    -0.0025,
    0.00583,
    0.00562
   ],
   [
    -0.00444,
    0.00137,
    0.01237,
    0.00672,
    0.00318,
    0.00523,
    0.00465,
    0.00223,
    0.00191,
    0.0079,
    0.00899,
    0.00377,
    0.00658,
    0.00603
   ],
   [
    -0.00193,
    0.00021,
    0.00491,
    0.00662,
    -4e-05,
    -0.00519,
    -0.00682,
    -0.00483,
    -0.00192,
    0.00242,
    0.00663,
    0.0,
    -0.00529,
    -0.00595
   ],
   [
    0.00793,
    0.00354,
    0.00364,
    0.00537,
    -0.00381,
    -0.00685,
    0.01068,
    0.00912,
    0.00704,
    0.00481,
    0.00597,
    -0.00305,
    -0.00703,
    0.00692
   ],
   [
    -0.01787,
    0.002,
    0.00221,
    0.00247,
    0.00109,
    0.01089,
    -0.00853,
    -0.01562,
    0.0022,
    -0.00201,
    0.00027,
    0.0031,
    0.01022,
    -0.00533
   ],
   [
    -0.01591,
    -0.00031,
    -0.02067,
    -0.00395,
    0.01338,
    0.00342,
    -0.00191,
    -0.0135,
    -0.0013,
    -0.01586,
    -0.00188,
    0.01422,
    0.00732,
    0.00228
   ],
   [
    -0.00063,
    -0.01802,
    -0.00463,
    0.02051,
    -0.00411,
    0.00228,
    -0.00103,
    -0.00056,
    -0.00896,
    0.00164,
    0.02028,
    -0.00255,
    0.00076,
    -0.00155
   ],
   [
    0.00758,
    0.05222,
    -0.00355,
    0.02859,
    0.05499,
    -0.05565,
    -0.01614,
    0.00426,
    0.03266,
    -0.00616,
    0.02895,
    0.04573,
    -0.04881,
    -0.01139
   ],
   [
    0.00574,
    -0.00258,
    -0.00795,
    0.00092,
    -0.00135,
    -0.00213,
    -0.0042,
    0.00479,
    0.00077,
    -0.0088,
    -0.00129,
    -0.00153,
    -0.00219,
    -0.00333
   ],
   [
    -0.01924,
    -0.00215,
    0.0134,
    -0.00525,
    0.00354,
    -0.00436,
    -0.01059,
    -0.017,
    -0.00438,
    0.01077,
    -0.00548,
    0.00197,
    -0.00591,
    -0.01007
   ],
   [
    -0.00356,
    -0.00666,
    0.01102,
    0.00154,
    -0.0149,
    0.01023,
    0.01268,
    -0.00566,
    -0.00918,
    0.00774,
    0.00056,
    -0.01005,
    0.01032,
    0.01076
   ],
   [
    0.00484,
    0.00416,
    0.00577,
    -0.00459,
    -0.00934,
    -0.00248,
    0.01335,
    0.00497,
    0.00814,
    0.00295,
    -0.00178,
    -0.0066,
    -0.00158,
    0.01225
   ],
   [
    0.03657,
    0.01069,
    0.00139,
    0.00489,
    -0.01297,
    0.00343,
    -0.00084,
    0.03133,
    0.01138,
    0.00572,
    0.00556,
    -0.01202,
    0.00381,
    5e-05
   ],
   [
    -0.00014,
    0.00343,
    0.01742,
    0.00522,
    0.00778,
    -0.01906,
    -0.00396,
    0.00095,
    -0.00076,
    0.0038,
    0.00381,
    0.00588,
    -0.01843,
    -0.00318
   ],
   [
    -0.01173,
    -0.00863,
    0.00757,
    -0.01064,
    -0.01055,
    0.01118,
    0.00283,
    -0.01107,
    -0.00648,
    0.0084,
    -0.01034,
    -0.0075,
    0.01086,
    0.00332
   ],
   [
    0.01055,
    -0.02418,
    0.00094,
    0.00893,
    -0.01517,
    -0.00394,
    0.00825,
    0.00732,
    -0.01455,
    0.00277,
    0.01151,
    -0.01071,
    -0.00069,
    0.00813
   ],
   [
    -0.01777,
    0.0024,
    -0.00115,
    0.00494,
    -0.0012,
    -0.00075,
    -0.0125,
    -0.01243,
    -0.00219,
    -0.00574,
    0.0026,
    -0.0037,
    0.00069,
    -0.01033
   ],
   [
    -0.00503,
    -0.00874,
    -0.00912,
    -0.00411,
    0.00698,
    -0.01411,
    0.0124,
    -0.00357,
    -0.00554,
    -0.00705,
    -0.00416,
    0.00597,
    -0.01558,
    0.00762
   ],
   [
    -0.01706,
    -0.0103,
    -0.00532,
    -0.00727,
    0.00243,
    -0.00468,
    -0.00354,
    -0.01749,
    -0.01346,
    -0.00768,
    -0.00678,
    0.00282,
    -0.00369,
    -0.00349
   ],
   [
    -0.01347,
    0.01008,
    -0.00732,
    0.00157,
    0.00857,
    -0.00895,
    -0.01276,
    -0.00806,
    0.0069,
    -0.00437,
    0.00099,
    0.00608,
    -0.01021,
    -0.01272
   ],
   [
    -0.01114,
    -0.00558,
    0.00077,
    -0.00605,
    0.00285,
    0.00332,
    -0.00366,
    -0.00864,
    -0.00308,
    -0.00066,
    -0.00785,
    0.00065,
    0.00267,
    -0.00454
   ],
   [
    -0.00274,
    -0.0059,
    0.00154,
    0.00618,
    -0.01124,
    -0.00279,
    -0.00401,
    -0.00119,
    -0.00753,
    -0.00068,
    0.00491,
    -0.00876,
    -0.00095,
    -0.00078
   ],
   [
    -0.00618,
    0.01246,
    -0.00537,
    0.00688,
    0.00114,
    -0.0077,
    -0.00049,
    -0.00622,
    0.00487,
    0.00116,
    0.00691,
    0.0,
    -0.0072,
    -0.00112
   ],
   [
    -0.01938,
    -0.00418,
    0.00175,
    -0.00292,
    -0.01544,
    0.01084,
    -0.01339,
    -0.01599,
    -0.00849,
    -0.00048,
    -0.00241,
    -0.01215,
    0.01091,
    -0.00997
   ],
   [
    -0.03592,
    0.02035,
    0.00013,
    -0.01217,
    0.03704,
    0.00011,
    -0.0059,
    -0.02605,
    0.01641,
    -0.00198,
    -0.01092,
    0.03314,
    -0.00102,
    -0.00449
   ],
   [
    0.0102,
    0.01127,
    -0.00212,
    0.0013,
    0.00732,
    0.00106,
    -0.0002,
    0.00827,
    0.01008,
    0.00197,
    0.00093,
    0.00682,
    0.00314,
    0.00232
   ],
   [
    -0.01502,
    -0.00921,
    0.00516,
    0.00478,
    -0.00014,
    0.001,
    0.00801,
    -0.0108,
    -0.00385,
    0.00016,
    0.00624,
    0.00235,
    0.0045,
    0.00981
   ],
   [
    -0.00383,
    0.01907,
    0.00458,
    -0.00436,
    0.00228,
    0.00425,
    0.00359,
    -0.00185,
    0.01449,
    0.00188,
    -0.0037,
    0.00342,
    0.00473,
    0.0038
   ],
   [
    -0.00967,
    -0.00666,
    0.00598,
    0.00832,
    0.00349,
    -0.00618,
    -0.00086,
    -0.00755,
    -0.00535,
    0.00238,
    0.00742,
    0.00395,
    -0.00394,
    0.00128
   ],
   [
    -0.00777,
    -0.01694,
    0.00658,
    -0.00987,
    -0.0036,
    0.02447,
    0.02093,
    -0.00432,
    -0.01068,
    0.00756,
    -0.00584,
    -0.00356,
    0.02144,
    0.01903
   ],
   [
    -0.01577,
    0.0147,
    -0.0086,
    -0.01058,
    0.0111,
    0.00137,
    2e-05,
    -0.01263,
    0.01637,
    -0.00433,
    -0.00854,
    0.01103,
    0.00029,
    0.00065
   ],
   [
    0.0219,
    0.01192,
    0.00477,
    0.00254,
    -0.00306,
    0.00512,
    0.00492,
    0.01774,
    0.01724,
    0.0107,
    0.00751,
    0.00088,
    0.00455,
    0.00357
   ],
   [
    0.006,
    -0.00189,
    0.0052,
    0.00195,
    0.00958,
    -0.00788,
    0.00444,
    0.00625,
    0.00019,
    0.00899,
    0.0037,
    0.00935,
    -0.00486,
    0.00382
   ],
   [
    -0.00405,
    0.02556,
    -0.01174,
    -0.01977,
    0.00239,
    0.01129,
    0.00271,
    0.00082,
    0.02087,
    -0.01054,
    -0.01924,
    0.00232,
    0.00823,
    0.00369
   ],
   [
    4e-05,
    -1e-05,
    -0.01318,
    0.00852,
    -0.00298,
    -0.00566,
    -0.00532,
    -0.00504,
    0.00027,
    -0.01437,
    0.0073,
    -0.00073,
    -0.00545,
    -0.0068
   ],
   [
    -0.00061,
    -0.00586,
    -0.00708,
    0.00258,
    0.00273,
    -0.00743,
    -0.00382,
    0.00231,
    -0.00455,
    -0.0053,
    0.00088,
    0.00411,
    -0.00427,
    -0.00125
   ],
   [
    0.00186,
    0.01479,
    -0.0027,
    -0.00878,
    0.01058,
    0.00339,
    0.01062,
    0.00386,
    0.01858,
    0.00184,
    -0.00694,
    0.0103,
    0.00444,
    0.00984
   ],
   [
    0.0093,
    -0.00465,
    0.00271,
    -0.00083,
    -0.00962,
    0.00462,
    0.00227,
    0.00973,
    -0.00284,
    0.00726,
    0.00074,
    -0.00749,
    0.0047,
    0.0026
   ],
   [
    -0.00702,
    -0.01468,
    -0.00058,
    0.00306,
    -0.01374,
    -0.00249,
    -0.02221,
    -0.00803,
    -0.01778,
    -0.00554,
    0.00074,
    -0.01611,
    -0.00542,
    -0.02063
   ],
   [
    -0.01122,
    -0.02052,
    -0.00887,
    -0.00302,
    0.01253,
    0.00159,
    0.00299,
    -0.00781,
    -0.0172,
    -0.00772,
    -0.00443,
    0.00778,
    0.00233,
    0.00301
   ],
   [
    0.02087,
    0.01382,
    -0.00833,
    -0.01387,
    -0.01161,
    0.00763,
    0.03238,
    0.01935,
    0.01827,
    -0.0036,
    -0.01582,
    -0.0098,
    0.00742,
    0.02724
   ],
   [
    0.01343,
    0.00616,
    0.01415,
    -0.00431,
    -0.0039,
    0.01256,
    -0.00672,
    0.01332,
    0.00558,
    0.01134,
    -0.00433,
    -0.00324,
    0.00914,
    -0.00575
   ],
   [
    0.01003,
    0.01278,
    -0.00258,
    0.00385,
    0.01299,
    -0.01422,
    -0.0065,
    0.01033,
    0.00729,
    -0.00395,
    0.00372,
    0.01121,
    -0.01251,
    -0.00629
   ],
   [
    0.0241,
    0.0008,
    0.00501,
    0.00568,
    -0.00416,
    -0.00786,
    -0.00073,
    0.02085,
    0.00278,
    0.00538,
    0.00526,
    -0.00389,
    -0.00393,
    0.00128
   ],
   [
    0.0194,
    0.00513,
    0.00697,
    0.00285,
    -0.00223,
    0.00418,
    -0.00631,
    0.01667,
    0.00897,
    0.00553,
    0.00467,
    -3e-05,
    0.00499,
    -0.00407
   ],
   [
    0.02896,
    0.01024,
    0.0004,
    0.0095,
    -0.0152,
    -0.00653,
    0.00683,
    0.0247,
    0.01536,
    0.00028,
    0.00826,
    -0.01391,
    -0.00876,
    0.00295
   ],
   [
    -0.00948,
    -0.00831,
    0.00176,
    -0.00425,
    0.00577,
    0.00267,
    -0.0118,
    -0.00732,
    -0.00509,
    -0.00243,
    -0.0042,
    0.0046,
    0.00037,
    -0.01226
   ],
   [
    -0.00381,
    -0.01409,
    -0.01362,
    -0.01327,
    -0.00172,
    0.01534,
    -0.00055,
    1e-05,
    -0.01065,
    -0.00873,
    -0.01222,
    -0.00035,
    0.01494,
    0.00031
   ],
   [
    -0.00556,
    -0.00913,
    -0.00908,
    0.00603,
    -0.0048,
    -0.00734,
    0.00189,
    -0.00635,
    -0.01061,
    -0.00822,
    0.0068,
    -0.0043,
    -0.00598,
    0.00208
   ],
   [
    -0.00341,
    -0.00376,
    0.01102,
    0.00564,
    -0.00326,
    0.00243,
    -0.00351,
    -0.00343,
    -0.00193,
    0.00556,
    0.00865,
    -0.00259,
    0.00101,
    -0.00215
   ],
   [
    -0.0032,
    0.01476,
    0.00778,
    -0.00012,
    -0.00029,
    -0.00097,
    0.00067,
    -0.00169,
    0.00453,
    0.00354,
    -0.00071,
    -0.00098,
    -0.00143,
    0.00064
   ],
   [
    0.00492,
    0.00355,
    0.00745,
    -0.01733,
    0.00372,
    0.00459,
    0.00265,
    0.00388,
    -1e-05,
    0.00413,
    -0.01381,
    0.00343,
    0.0031,
    0.00217
   ],
   [
    -0.03656,
    -0.02536,
    -0.00089,
    0.02564,
    -0.01101,
    -0.00688,
    0.00115,
    -0.03024,
    -0.02176,
    -0.00168,
    0.02807,
    -0.00633,
    0.00037,
    0.00639
   ],
   [
    0.00406,
    -0.01683,
    -0.00603,
    -0.00505,
    -0.00384,
    0.00345,
    0.02252,
    0.00442,
    -0.01187,
    -0.00554,
    -0.00638,
    -0.00134,
    0.00447,
    0.01934
   ],
   [
    0.00133,
    0.00757,
    -0.00035,
    0.00553,
    0.00012,
    0.00155,
    -0.01047,
    0.00101,
    -0.00026,
    0.0031,
    0.00453,
    0.00026,
    0.00201,
    -0.00639
   ],
   [
    0.01234,
    0.00471,
    -0.00294,
    0.01374,
    -0.00789,
    -3e-05,
    -0.00264,
    0.01038,
    0.00577,
    0.0001,
    0.01107,
    -0.00992,
    -0.00195,
    -0.00462
   ],
   [
    -0.01513,
    -0.01272,
    0.0071,
    -0.01285,
    -0.00117,
    0.00516,
    0.00783,
    -0.01088,
    -0.0064,
    -0.00013,
    -0.01447,
    -0.00384,
    0.00617,
    0.00681
   ],
   [
    0.00607,
    0.00142,
    -0.00542,
    -0.00923,
    0.0054,
    0.00942,
    0.00059,
    0.0046,
    0.00287,
    -0.00634,
    -0.00917,
    0.00407,
    0.00966,
    0.00058
   ],
   [
    0.02177,
    0.02024,
    -0.00691,
    -0.00724,
    0.00404,
    -0.00877,
    -0.00714,
    0.01976,
    0.01964,
    -0.0059,
    -0.00765,
    0.00377,
    -0.00997,
    -0.00691
   ],
   [
    -0.00994,
    -0.01486,
    -0.0016,
    -0.00653,
    -0.0035,
    -0.00649,
    0.00216,
    -0.00646,
    -0.01132,
    -0.00338,
    -0.00655,
    -0.00326,
    -0.00321,
    0.0034
   ],
   [
    0.01626,
    -0.00645,
    -0.00029,
    -0.00512,
    0.00817,
    -0.0095,
    -0.00547,
    0.00962,
    -0.00385,
    -0.0022,
    -0.00359,
    0.00386,
    -0.00866,
    -0.00544
   ],
   [
    0.01459,
    0.01964,
    0.00152,
    -0.00506,
    0.01055,
    0.00438,
    -0.00316,
    0.0127,
    0.01468,
    0.00028,
    -0.00552,
    0.00747,
    0.00557,
    -0.00127
   ],
   [
    0.00562,
    -0.00257,
    -0.00539,
    -0.00541,
    0.00394,
    0.01488,
    -0.00254,
    0.00249,
    -0.00387,
    -0.00627,
    -0.00563,
    0.0016,
    0.01037,
    -0.00508
   ],
   [
    0.00838,
    0.00934,
    0.00154,
    -0.00313,
    -0.00322,
    0.02165,
    0.01412,
    0.00577,
    0.00947,
    0.00384,
    -0.00334,
    -0.0028,
    0.01797,
    0.01185
   ],
   [
    0.01133,
    0.01018,
    0.00402,
    -0.00621,
    0.00626,
    -0.0066,
    0.01193,
    0.00747,
    0.01086,
    -0.00027,
    -0.0072,
    0.00296,
    -0.00788,
    0.00821
   ],
   [
    -0.01699,
    -0.0135,
    0.00351,
    0.00204,
    -0.00906,
    -0.00661,
    -0.00381,
    -0.0166,
    -0.01298,
    -0.00107,
    0.00244,
    -0.00699,
    -0.008,
    -0.00474
   ],
   [
    -0.00433,
    -0.01708,
    0.01583,
    0.00146,
    -0.00555,
    0.00769,
    0.00364,
    -0.00022,
    -0.01068,
    0.0145,
    0.00353,
    -0.00374,
    0.00705,
    0.00504
   ],
   [
    0.0103,
    -0.00618,
    0.01964,
    -0.01606,
    -0.00095,
    0.0024,
    0.00784,
    0.00792,
    -0.00424,
    0.01686,
    -0.01421,
    0.00033,
    0.00458,
    0.00854
   ],
   [
    -0.01435,
    0.00444,
    0.0025,
    0.00114,
    0.0065,
    -0.00021,
    -0.01354,
    -0.01091,
    0.00039,
    0.00256,
    0.00169,
    0.00842,
    0.00223,
    -0.00904
   ],
   [
    -0.02974,
    -0.00458,
    -0.0064,
    -0.00707,
    0.00332,
    -0.00505,
    0.00399,
    -0.02516,
    -0.00392,
    -0.00304,
    -0.00574,
    0.00313,
    -0.00145,
    0.00305
   ],
   [
    -0.01025,
    0.00339,
    0.00733,
    -0.00514,
    -0.00354,
    0.00755,
    0.00868,
    -0.00677,
    0.00424,
    0.00365,
    -0.00428,
    -0.00172,
    0.00718,
    0.00809
   ],
   [
    0.00181,
    0.01266,
    -0.00569,
    0.00713,
    -0.01771,
    0.00338,
    0.02683,
    0.00457,
    0.01291,
    -0.00293,
    0.00835,
    -0.01193,
    0.00419,
    0.0247
   ],
   [
    -0.02172,
    -0.00846,
    -0.00505,
    0.00455,
    -0.00969,
    0.00656,
    -0.00768,
    -0.01869,
    -0.00785,
    -0.00844,
    0.00472,
    -0.00821,
    0.00461,
    -0.00788
   ],
   [
    0.00655,
    -0.00992,
    -0.00208,
    -0.00809,
    -0.00069,
    0.0021,
    0.00878,
    0.00646,
    -0.00577,
    -0.00208,
    -0.00807,
    -0.00097,
    0.00302,
    0.00682
   ],
   [
    0.01328,
    0.00327,
    -0.00842,
    -0.00644,
    0.01499,
    -0.00021,
    0.00977,
    0.00914,
    -0.00041,
    -0.0079,
    -0.00635,
    0.01064,
    -0.00109,
    0.00717
   ],
   [
    0.0038,
    0.00502,
    -0.01142,
    0.00615,
    -0.00216,
    -0.00991,
    -0.00312,
    0.00069,
    0.00305,
    -0.0073,
    0.00571,
    -0.00166,
    -0.00909,
    -0.00358
   ],
   [
    -0.0275,
    0.01312,
    -0.0054,
    -0.00509,
    0.01908,
    -0.00799,
    0.01776,
    -0.01763,
    0.01037,
    -0.00681,
    -0.00466,
    0.01532,
    -0.00689,
    0.01379
   ],
   [
    -0.01212,
    -0.03534,
    -0.00338,
    -0.00021,
    -0.01464,
    0.00731,
    0.00865,
    -0.01206,
    -0.0241,
    -0.00296,
    0.0019,
    -0.0116,
    0.00931,
    0.00702
   ],
   [
    0.04006,
    -0.012,
    -0.00154,
    0.01477,
    -0.02617,
    -0.00047,
    -0.01653,
    0.03056,
    -0.01016,
    -0.00039,
    0.01277,
    -0.02645,
    -0.00199,
    -0.01404
   ],
   [
    -0.00399,
    0.00255,
    0.00161,
    -0.00098,
    0.00741,
    -0.00401,
    -0.00588,
    -0.00355,
    0.00101,
    0.00046,
    -0.00014,
    0.00608,
    -0.00419,
    -0.00573
   ],
   [
    -0.01416,
    -0.00522,
    -0.0102,
    -0.00395,
    -0.00208,
    -0.00115,
    0.00487,
    -0.01432,
    -0.00453,
    -0.01139,
    -0.00467,
    -0.00232,
    0.00048,
    0.00265
   ],
   [
    -0.00255,
    -0.00617,
    -0.00025,
    0.01175,
    -0.01242,
    -0.00163,
    -0.00266,
    -0.00322,
    -0.00915,
    -0.00202,
    0.01172,
    -0.0091,
    -0.00226,
    -0.00264
   ],
   [
    0.00026,
    0.00726,
    0.00286,
    -0.00804,
    0.00781,
    -0.00433,
    -0.0137,
    0.00101,
    0.00824,
    -0.00317,
    -0.00694,
    0.00569,
    -0.00683,
    -0.01305
   ],
   [
    -0.00497,
    -0.00228,
    0.00175,
    -0.00682,
    0.00495,
    -0.00587,
    0.00846,
    -0.00681,
    -0.00242,
    -0.00165,
    -0.00589,
    0.00289,
    -0.00604,
    0.0054
   ],
   [
    0.00379,
    0.00185,
    -0.00329,
    0.00146,
    -0.01226,
    0.01392,
    0.00409,
    0.00463,
    0.00245,
    -0.00186,
    0.00024,
    -0.0108,
    0.01267,
    0.00505
   ],
   [
    -0.00686,
    -0.00317,
    -0.0072,
    0.00367,
    -0.00282,
    0.00311,
    0.00273,
    -0.00455,
    -0.00033,
    -0.00277,
    0.00298,
    -0.0024,
    0.00164,
    0.00251
   ],
   [
    -0.00575,
    -0.01542,
    0.00431,
    -0.01514,
    -0.0077,
    0.00824,
    0.01062,
    -0.00367,
    -0.01078,
    0.00255,
    -0.01257,
    -0.00647,
    0.00629,
    0.00871
   ],
   [
    -0.00699,
    -0.00708,
    -0.01606,
    0.00434,
    0.00672,
    -0.00017,
    0.00164,
    -0.00688,
    -0.00629,
    -0.01042,
    0.00596,
    0.00516,
    0.00359,
    0.00297
   ],
   [
    0.01982,
    -0.00928,
    0.00294,
    -0.00379,
    -0.00317,
    0.00302,
    0.00196,
    0.01526,
    -0.00787,
    -0.00512,
    -0.0065,
    -0.00714,
    -0.00173,
    -0.00183
   ],
   [
    0.00438,
    -0.00128,
    -0.00402,
    -0.01075,
    0.0073,
    0.01144,
    0.00527,
    0.00341,
    -0.00487,
    0.00213,
    -0.00709,
    0.00909,
    0.01268,
    0.00873
   ],
   [
    -0.00513,
    -0.00885,
    0.00142,
    0.00647,
    -0.00224,
    0.00083,
    -0.01039,
    -0.00362,
    -0.0091,
    -0.00202,
    0.00385,
    -0.00203,
    0.0014,
    -0.00994
   ],
   [
    -0.02302,
    0.00469,
    -0.01125,
    -0.01805,
    0.00984,
    -7e-05,
    0.00992,
    -0.01975,
    -0.00413,
    -0.01101,
    -0.01547,
    0.00972,
    0.00268,
    0.01039
   ],
   [
    -0.00397,
    0.00848,
    0.00302,
    0.00494,
    -0.00058,
    0.0148,
    0.00012,
    -0.00074,
    0.00197,
    0.00712,
    0.00317,
    -0.00034,
    0.01541,
    0.00341
   ],
   [
    -0.00018,
    -0.01376,
    -0.00572,
    0.00793,
    -0.00737,
    0.00637,
    -0.00786,
    -0.0017,
    -0.01091,
    -0.00262,
    0.00454,
    -0.00835,
    0.00468,
    -0.00826
   ],
   [
    -0.01752,
    -0.01248,
    0.00381,
    0.00504,
    -0.01226,
    0.00288,
    -0.00601,
    -0.01313,
    -0.00974,
    0.00216,
    0.00729,
    -0.00902,
    0.00234,
    -0.00588
   ],
   [
    0.02981,
    0.01222,
    -0.00628,
    0.00341,
    0.00325,
    -0.00158,
    -0.00161,
    0.02613,
    0.01566,
    0.00061,
    0.00351,
    0.00088,
    -0.00214,
    -0.00133
   ],
   [
    -0.01489,
    -0.00462,
    -0.00269,
    -0.00771,
    0.0029,
    0.00633,
    -0.008,
    -0.01432,
    -0.00876,
    -0.00508,
    -0.01034,
    -0.00071,
    0.00397,
    -0.00739
   ],
   [
    -0.00785,
    -0.01193,
    0.00049,
    0.02988,
    -0.00827,
    -0.00919,
    -0.01752,
    -0.00732,
    -0.016,
    -0.00275,
    0.02563,
    -0.00748,
    -0.00519,
    -0.01334
   ],
   [
    0.00314,
    0.02963,
    0.01373,
    -0.0068,
    0.01149,
    -0.00403,
    0.0104,
    0.00066,
    0.02694,
    0.00862,
    -0.00618,
    0.01149,
    -0.00234,
    0.01219
   ],
   [
    0.00931,
    -0.00326,
    0.00978,
    0.00116,
    0.0052,
    -0.00047,
    -0.00343,
    0.00656,
    0.00046,
    0.00864,
    0.00059,
    0.00435,
    -0.00188,
    -0.00385
   ],
   [
    0.0333,
    0.02967,
    0.00662,
    -0.00022,
    -0.00139,
    -0.0105,
    0.01698,
    0.0252,
    0.02649,
    0.00538,
    0.00298,
    0.00267,
    -0.0044,
    0.01658
   ],
   [
    -0.00128,
    -0.00987,
    -0.00999,
    0.00214,
    0.00266,
    0.0075,
    -0.00142,
    -0.00038,
    -0.00747,
    -0.00252,
    0.00285,
    0.0017,
    0.00519,
    -0.00396
   ],
   [
    -0.0145,
    -0.01691,
    0.00464,
    -0.00649,
    -0.00364,
    0.0116,
    0.00124,
    -0.01012,
    -0.01571,
    0.00067,
    -0.00651,
    -0.00437,
    0.0082,
    3e-05
   ],
   [
    0.00127,
    0.0175,
    -0.00527,
    -0.00739,
    0.00252,
    0.01318,
    -0.00286,
    0.00329,
    0.01263,
    -0.0053,
    -0.00656,
    0.00243,
    0.00893,
    -0.00269
   ],
   [
    -0.02255,
    -0.02044,
    0.00446,
    -0.01198,
    0.01105,
    0.01221,
    0.00376,
    -0.02014,
    -0.01141,
    0.00442,
    -0.01121,
    0.00797,
    0.00954,
    0.00239
   ],
   [
    -0.00171,
    0.00926,
    -0.00642,
    -0.01153,
    0.01151,
    0.00842,
    0.00305,
    -0.00113,
    0.00568,
    -0.00326,
    -0.00952,
    0.01059,
    0.00735,
    0.00158
   ],
   [
    0.0153,
    0.0107,
    -0.00756,
    -0.00155,
    0.0179,
    -0.00959,
    0.00651,
    0.0135,
    0.00805,
    -0.00549,
    -0.00082,
    0.01574,
    -0.00774,
    0.00736
   ],
   [
    -0.019,
    0.00274,
    0.00321,
    -0.0119,
    -0.00808,
    0.01371,
    0.01165,
    -0.01527,
    -0.00078,
    0.00144,
    -0.01174,
    -0.00508,
    0.01441,
    0.00987
   ],
   [
    -0.00299,
    -0.01534,
    -0.00665,
    0.01827,
    -0.01039,
    -0.01793,
    0.009,
    -0.00384,
    -0.01458,
    -0.00568,
    0.01866,
    -0.00873,
    -0.01635,
    0.00566
   ],
   [
    0.01225,
    0.02592,
    -0.01019,
    -0.00152,
    0.0041,
    -0.00695,
    0.003,
    0.01005,
    0.02233,
    -0.00546,
    0.00079,
    0.0046,
    -0.00562,
    0.00233
   ],
   [
    -0.01212,
    0.00266,
    0.00204,
    0.00637,
    -0.00513,
    0.00034,
    0.00351,
    -0.01224,
    0.00234,
    0.00156,
    0.00467,
    -0.00281,
    0.00109,
    0.0043
   ],
   [
    0.02098,
    0.00719,
    0.0126,
    0.00533,
    0.00393,
    -0.00221,
    -0.00209,
    0.01841,
    0.00678,
    0.00765,
    0.00364,
    0.0047,
    -0.00022,
    -0.00106
   ],
   [
    0.00884,
    0.00895,
    0.00415,
    0.00636,
    0.00445,
    -0.00792,
    -0.01666,
    0.00567,
    0.00659,
    -0.00197,
    0.00715,
    0.00345,
    -0.00862,
    -0.01453
   ],
   [
    -0.00696,
    -0.00309,
    0.00493,
    0.00678,
    -0.00041,
    0.00154,
    0.00048,
    -0.00458,
    -0.0039,
    0.0026,
    0.00502,
    -0.00022,
    0.00273,
    0.00246
   ],
   [
    0.00206,
    0.00466,
    -0.00106,
    -0.00016,
    0.00223,
    0.00081,
    0.00063,
    0.0042,
    -0.00128,
    -0.00221,
    -0.00306,
    0.00043,
    -0.00154,
    3e-05
   ],
   [
    -0.01125,
    -0.0052,
    -0.00165,
    -0.01126,
    -0.00821,
    0.01579,
    0.00948,
    -0.01115,
    -0.00375,
    -0.00168,
    -0.01115,
    -0.0064,
    0.01121,
    0.00528
   ],
   [
    0.01136,
    -0.00491,
    -0.00045,
    -0.01104,
    -0.01471,
    0.0222,
    0.00865,
    0.00854,
    -0.00733,
    -0.00048,
    -0.0137,
    -0.01327,
    0.01975,
    0.00677
   ],
   [
    -0.00248,
    -0.00668,
    0.00652,
    0.01249,
    0.00179,
    -0.01532,
    -0.00722,
    -0.00184,
    -0.00375,
    0.00115,
    0.00974,
    -0.00131,
    -0.01604,
    -0.00703
   ],
   [
    -0.0048,
    -0.0135,
    -0.00725,
    -0.00216,
    -0.00281,
    -0.00402,
    0.00013,
    -0.00184,
    -0.00961,
    -0.00494,
    -0.00328,
    -0.00306,
    -0.00512,
    -0.00265
   ],
   [
    2e-05,
    -0.01821,
    -0.00047,
    0.00145,
    -0.01427,
    0.00103,
    0.00891,
    -0.00354,
    -0.0141,
    -0.00454,
    -0.0019,
    -0.01468,
    -0.00046,
    0.00705
   ],
   [
    -0.01817,
    -0.0159,
    0.00017,
    -0.0067,
    0.00955,
    0.00862,
    0.0175,
    -0.01458,
    -0.01398,
    -0.00175,
    -0.00753,
    0.00966,
    0.00976,
    0.01575
   ],
   [
    -0.01169,
    0.00579,
    -0.00759,
    -0.01111,
    0.00045,
    0.00816,
    0.00399,
    -0.00948,
    0.00448,
    -0.00329,
    -0.01017,
    -0.00022,
    0.00838,
    0.00437
   ],
   [
    -0.00491,
    0.00935,
    -0.00763,
    -0.00049,
    0.00671,
    -0.00627,
    -0.01179,
    -0.00283,
    0.0027,
    -0.00896,
    -0.00378,
    0.00439,
    -0.00736,
    -0.01077
   ],
   [
    0.00045,
    0.01039,
    -0.01099,
    0.0127,
    -0.0084,
    0.00281,
    -0.00408,
    0.00224,
    0.00892,
    -0.00457,
    0.01046,
    -0.00848,
    0.0025,
    -0.00207
   ],
   [
    0.00063,
    0.0048,
    0.01065,
    0.00264,
    0.00037,
    -0.01028,
    0.01298,
    -0.00306,
    0.00363,
    0.00219,
    0.0062,
    0.00115,
    -0.00895,
    0.01182
   ],
   [
    0.00082,
    0.01788,
    -0.00731,
    -0.00841,
    0.00354,
    0.0062,
    0.00082,
    0.0014,
    0.01389,
    -0.00176,
    -0.00869,
    0.00181,
    0.00538,
    0.00174
   ],
   [
    0.00676,
    -0.01374,
    0.00547,
    -0.00545,
    -0.01591,
    0.00747,
    0.02035,
    0.00505,
    -0.00608,
    0.00707,
    -0.00564,
    -0.01394,
    0.00896,
    0.01735
   ],
   [
    0.0082,
    0.01354,
    0.01279,
    -0.00741,
    0.00055,
    0.01557,
    -6e-05,
    0.00872,
    0.01015,
    0.00651,
    -0.00799,
    0.00065,
    0.0148,
    0.00034
   ],
   [
    -0.00062,
    0.00545,
    -0.00251,
    0.00887,
    -0.00769,
    -0.00679,
    -0.00132,
    0.00036,
    0.00034,
    -0.00481,
    0.00924,
    -0.00616,
    -0.0063,
    -0.00252
   ],
   [
    -0.00733,
    -0.0069,
    -4e-05,
    0.00348,
    -0.01551,
    0.00668,
    -0.00184,
    -0.00924,
    -0.00418,
    -0.00389,
    0.00461,
    -0.0132,
    0.00473,
    -0.00244
   ],
   [
    0.01831,
    0.01573,
    0.0021,
    -0.00034,
    -0.00097,
    0.0074,
    -0.00199,
    0.0138,
    0.01688,
    0.00336,
    0.00339,
    0.00065,
    0.00753,
    -0.00045
   ],
   [
    0.00776,
    0.00154,
    0.00037,
    -0.00669,
    0.00165,
    -0.00177,
    0.0129,
    0.00609,
    0.00361,
    0.00168,
    -0.00508,
    0.00134,
    -0.00334,
    0.00948
   ],
   [
    -0.01096,
    -0.0044,
    -0.00188,
    -0.00029,
    0.01387,
    -0.02157,
    0.00074,
    -0.00722,
    -0.00343,
    -0.00116,
    -0.00123,
    0.01367,
    -0.01605,
    0.00205
   ],
   [
    -0.00563,
    0.0091,
    -0.0018,
    0.01123,
    0.00087,
    -0.00811,
    -0.00299,
    -0.00598,
    0.00335,
    -0.00013,
    0.01005,
    -0.00052,
    -0.00861,
    -0.0027
   ],
   [
    0.00054,
    0.0114,
    0.0015,
    -0.01286,
    -0.00154,
    0.01508,
    0.00369,
    -0.00133,
    0.01112,
    -0.00112,
    -0.01314,
    -0.00225,
    0.01071,
    0.00067
   ],
   [
    0.01347,
    -0.00043,
    0.00563,
    -0.01239,
    0.00166,
    0.00136,
    0.00373,
    0.00869,
    0.00095,
    0.00634,
    -0.01134,
    0.00066,
    -0.00062,
    0.00161
   ],
   [
    0.00991,
    0.01459,
    -0.00867,
    0.00604,
    0.02656,
    -0.00232,
    -0.02074,
    0.00549,
    0.00536,
    -0.00547,
    0.00373,
    0.01827,
    -0.0051,
    -0.01578
   ],
   [
    0.00029,
    0.00829,
    -0.00683,
    0.00472,
    0.00725,
    -0.01424,
    -0.00941,
    -0.00102,
    0.00587,
    -0.00541,
    0.00706,
    0.00763,
    -0.0097,
    -0.00468
   ],
   [
    0.02584,
    -0.01376,
    -0.00251,
    -0.01161,
    -0.00896,
    -0.00638,
    0.00888,
    0.0188,
    -0.00293,
    0.00113,
    -0.01006,
    -0.00935,
    -0.00772,
    0.00413
   ],
   [
    0.00419,
    -0.01062,
    0.00456,
    0.00473,
    0.01134,
    -0.00563,
    -0.01598,
    0.00353,
    -0.01004,
    0.00652,
    0.00497,
    0.00622,
    -0.00733,
    -0.01457
   ],
   [
    -0.01304,
    -0.01084,
    -0.01472,
    6e-05,
    -0.00334,
    0.00097,
    0.01621,
    -0.01278,
    -0.00923,
    -0.01267,
    -0.00107,
    -0.00321,
    0.00228,
    0.01603
   ],
   [
    -0.01005,
    0.01501,
    0.01013,
    -0.0052,
    0.01345,
    -0.00629,
    0.00583,
    -0.009,
    0.00524,
    0.00895,
    -0.00328,
    0.01405,
    -0.0041,
    0.00626
   ],
   [
    0.00047,
    -0.00295,
    0.01171,
    -0.0026,
    0.00515,
    0.00695,
    -0.0064,
    -0.00013,
    -0.00079,
    0.0128,
    0.00054,
    0.00795,
    0.00683,
    -0.00598
   ],
   [
    0.01773,
    -0.03335,
    0.00112,
    0.01657,
    -0.01489,
    -0.03187,
    0.01428,
    0.00808,
    -0.0242,
    -0.00224,
    0.01778,
    -0.00965,
    -0.02128,
    0.01363
   ],
   [
    -0.0115,
    0.00056,
    0.00467,
    -0.00177,
    0.00058,
    -0.01001,
    -0.01003,
    -0.00826,
    -0.00425,
    0.00101,
    -0.00227,
    -0.00082,
    -0.00862,
    -0.0087
   ],
   [
    0.01816,
    -0.01922,
    0.00412,
    0.005,
    -0.0089,
    -0.00467,
    0.00051,
    0.01272,
    -0.01014,
    0.00204,
    0.00253,
    -0.01147,
    -0.00655,
    -0.00331
   ],
   [
    -0.00057,
    0.00425,
    -0.00096,
    0.00681,
    -0.00706,
    -0.00671,
    0.0,
    0.00262,
    0.00279,
    -0.00482,
    0.00806,
    -0.00514,
    -0.00477,
    -0.00194
   ],
   [
    0.01542,
    0.00207,
    -0.0073,
    0.01103,
    -0.00729,
    -0.00309,
    -0.00384,
    0.00953,
    -0.00342,
    -0.00565,
    0.01132,
    -0.0056,
    -0.00201,
    -0.00329
   ],
   [
    0.00632,
    0.00451,
    0.01103,
    -0.00205,
    0.00419,
    -0.00876,
    -0.00118,
    0.00452,
    0.005,
    0.01309,
    -0.00095,
    0.00491,
    -0.00654,
    0.00106
   ],
   [
    -0.00118,
    -0.00137,
    -0.00574,
    -0.01142,
    0.00767,
    0.00257,
    0.01175,
    -0.00172,
    0.00373,
    -0.00214,
    -0.00917,
    0.00788,
    0.00351,
    0.01066
   ],
   [
    0.01057,
    -0.00373,
    -0.00162,
    -0.01309,
    -0.00086,
    0.00473,
    0.01064,
    0.00779,
    -0.00421,
    0.00122,
    -0.0114,
    -0.00293,
    0.00169,
    0.00835
   ],
   [
    0.00219,
    -0.00781,
    0.00413,
    0.00701,
    -0.00803,
    0.00168,
    0.0048,
    0.00262,
    -0.00882,
    0.00146,
    0.00586,
    -0.00765,
    -0.00053,
    0.00541
   ],
   [
    -0.01624,
    -0.00353,
    -0.00352,
    -0.01158,
    -0.00296,
    0.00894,
    0.01085,
    -0.01405,
    -0.00345,
    0.00034,
    -0.01251,
    -0.00351,
    0.00893,
    0.00969
   ],
   [
    0.00619,
    0.0126,
    0.00802,
    0.00896,
    0.00035,
    -0.00885,
    -0.00925,
    0.00886,
    0.00657,
    0.00823,
    0.00748,
    8e-05,
    -0.00642,
    -0.00603
   ],
   [
    0.00241,
    0.00224,
    -0.00111,
    0.00131,
    -0.00133,
    0.00797,
    0.00697,
    0.00239,
    0.00092,
    -0.00188,
    -0.00028,
    -0.00208,
    0.00823,
    0.00523
   ],
   [
    -0.00913,
    -0.01902,
    0.00627,
    0.00434,
    0.0027,
    -0.00856,
    -0.00063,
    -0.01101,
    -0.01217,
    0.00145,
    0.00618,
    0.00141,
    -0.00829,
    -0.00104
   ],
   [
    -0.00281,
    0.00194,
    0.0033,
    0.01253,
    -0.00347,
    0.0057,
    -0.01353,
    -0.00272,
    0.00258,
    0.00369,
    0.0123,
    -0.00324,
    0.00514,
    -0.0096
   ],
   [
    -0.02115,
    -0.00877,
    -0.00118,
    -0.00637,
    0.00326,
    -0.00023,
    -0.00798,
    -0.0192,
    -0.01115,
    0.0001,
    -0.00672,
    0.00029,
    -0.00326,
    -0.00839
   ],
   [
    -0.00148,
    0.00735,
    -0.00284,
    0.0063,
    0.00408,
    -0.00373,
    -0.01065,
    -0.00062,
    0.0045,
    -0.00049,
    0.00538,
    0.00266,
    -0.0015,
    -0.00779
   ],
   [
    -0.02985,
    -0.00148,
    0.00018,
    -0.00203,
    0.00401,
    -0.00683,
    0.01579,
    -0.02276,
    -0.001,
    -0.00081,
    -0.00086,
    0.00458,
    -0.00621,
    0.01418
   ],
   [
    -0.01994,
    -0.00796,
    -0.01781,
    -0.01285,
    0.00591,
    0.01012,
    0.00739,
    -0.01707,
    -0.00815,
    -0.0096,
    -0.01529,
    0.00054,
    0.00786,
    0.00519
   ],
   [
    0.02094,
    0.00434,
    0.01156,
    0.00811,
    -0.00942,
    0.00498,
    0.01446,
    0.01824,
    0.01052,
    0.00676,
    0.00839,
    -0.00863,
    0.00658,
    0.01269
   ],
   [
    -0.01792,
    -0.02371,
    0.00546,
    0.01242,
    -0.00559,
    -0.00272,
    -0.01179,
    -0.01834,
    -0.02198,
    0.00066,
    0.01296,
    -0.00349,
    -0.00186,
    -0.01001
   ],
   [
    -0.0177,
    -0.00617,
    0.00332,
    -0.00216,
    0.0038,
    -0.00405,
    -0.0053,
    -0.01459,
    -0.00701,
    0.0029,
    -0.00326,
    0.00195,
    -0.00532,
    -0.00687
   ],
   [
    -0.01567,
    -0.00065,
    -0.00018,
    -0.00495,
    0.01101,
    0.00451,
    -0.00905,
    -0.01254,
    -0.00329,
    -0.00127,
    -0.00748,
    0.00606,
    0.00334,
    -0.00753
   ],
   [
    0.00383,
    -0.01279,
    0.00359,
    0.00082,
    -0.0028,
    0.01028,
    2e-05,
    0.00372,
    -0.01062,
    0.0055,
    0.00115,
    -0.00214,
    0.00882,
    0.00149
   ],
   [
    -0.02225,
    -0.01987,
    0.01803,
    -0.01055,
    0.00566,
    0.00379,
    -0.00672,
    -0.01811,
    -0.01415,
    0.0132,
    -0.00684,
    0.00748,
    0.00576,
    -0.00433
   ],
   [
    -0.00154,
    0.0057,
    0.00429,
    0.00113,
    -0.00329,
    0.00529,
    -0.00782,
    0.0042,
    0.0046,
    0.00917,
    5e-05,
    -0.00387,
    0.00611,
    -0.00498
   ],
   [
    0.00112,
    -0.00341,
    0.00106,
    -0.01078,
    0.00533,
    -0.00123,
    0.00771,
    0.00106,
    -0.00197,
    -0.00298,
    -0.01103,
    0.00337,
    -0.0017,
    0.00626
   ],
   [
    0.00261,
    -0.0137,
    -0.01139,
    -0.00785,
    0.00117,
    0.00262,
    -0.0073,
    0.00204,
    -0.01214,
    -0.01209,
    -0.0091,
    0.00114,
    0.00267,
    -0.00684
   ],
   [
    0.0265,
    0.01094,
    -0.00664,
    -0.00027,
    -0.00968,
    0.00198,
    0.01094,
    0.02483,
    0.00907,
    -0.00285,
    -0.00067,
    -0.00683,
    0.00154,
    0.00856
   ],
   [
    -0.00856,
    -0.00405,
    -0.00525,
    -0.00101,
    0.01269,
    0.00945,
    -0.00512,
    -0.00761,
    -0.00563,
    -0.00313,
    -0.00112,
    0.01243,
    0.01033,
    -0.00166
   ],
   [
    0.00578,
    -0.01081,
    0.00389,
    0.00491,
    -0.00863,
    0.0053,
    -0.0017,
    0.00845,
    -0.00604,
    0.00997,
    0.00674,
    -0.00551,
    0.00618,
    -0.00107
   ],
   [
    0.01755,
    0.01828,
    -0.01662,
    0.0047,
    0.01711,
    -0.01078,
    -0.00352,
    0.01237,
    0.01819,
    -0.01239,
    0.00407,
    0.01632,
    -0.0091,
    -0.00388
   ],
   [
    0.01112,
    0.0163,
    -0.01876,
    -0.00358,
    0.00416,
    0.00207,
    0.01336,
    0.01162,
    0.00992,
    -0.01368,
    -0.00518,
    0.00282,
    0.00324,
    0.00984
   ],
   [
    -0.02673,
    -0.01353,
    -0.0054,
    -0.01356,
    0.0003,
    0.00388,
    0.00889,
    -0.02241,
    -0.01162,
    -0.00585,
    -0.01178,
    0.00204,
    0.00736,
    0.00978
   ],
   [
    -0.01453,
    -0.012,
    0.01276,
    -0.01412,
    -0.01207,
    0.0196,
    0.01121,
    -0.0118,
    -0.01143,
    0.00826,
    -0.01715,
    -0.01324,
    0.01858,
    0.01176
   ],
   [
    0.00834,
    -0.0151,
    0.01353,
    -0.01265,
    -0.00731,
    0.00852,
    0.00633,
    0.00872,
    -0.01063,
    0.00672,
    -0.01095,
    -0.00805,
    0.00777,
    0.00565
   ],
   [
    -0.00292,
    0.00185,
    -0.00404,
    0.01039,
    0.00089,
    -0.00748,
    -0.00235,
    -0.00285,
    0.00058,
    -0.00214,
    0.0092,
    0.00232,
    -0.00658,
    -0.00264
   ],
   [
    -0.01099,
    0.00489,
    -0.00965,
    -0.00189,
    -0.00447,
    0.00731,
    -0.0047,
    -0.0109,
    -0.00199,
    -0.00835,
    -0.00269,
    -0.00448,
    0.00609,
    -0.00215
   ],
   [
    0.00103,
    0.00171,
    -0.00355,
    0.00582,
    0.00739,
    -0.00287,
    -0.0082,
    0.00087,
    -0.00196,
    -0.00328,
    0.00809,
    0.00828,
    -0.00342,
    -0.00618
   ],
   [
    -0.02825,
    0.01207,
    0.0208,
    -0.00271,
    0.00429,
    0.0137,
    0.0107,
    -0.01801,
    0.00856,
    0.00914,
    0.00015,
    0.00792,
    0.01565,
    0.01444
   ],
   [
    -0.00396,
    0.00109,
    -0.00492,
    0.01049,
    0.00565,
    -0.00472,
    -0.00564,
    -0.00576,
    -0.00202,
    -0.00788,
    0.00789,
    0.00374,
    -0.004,
    -0.00549
   ],
   [
    -0.00968,
    -0.00689,
    0.00616,
    0.0007,
    -0.00082,
    0.00225,
    0.00476,
    -0.0119,
    -0.00828,
    0.00665,
    5e-05,
    -0.0024,
    0.00078,
    0.00443
   ],
   [
    -0.00206,
    -0.00953,
    0.01344,
    -0.00344,
    -0.00454,
    0.01665,
    -0.00104,
    -0.00098,
    -0.00826,
    0.00555,
    -0.00342,
    -0.00365,
    0.01828,
    0.00258
   ],
   [
    -0.01682,
    -0.02607,
    0.01635,
    -0.00938,
    -0.00351,
    -0.00308,
    0.00458,
    -0.01447,
    -0.02314,
    0.01202,
    -0.0069,
    0.00012,
    0.0011,
    0.00733
   ],
   [
    0.02389,
    0.00383,
    0.0018,
    -0.00291,
    0.00312,
    0.00464,
    -0.00101,
    0.02133,
    0.00341,
    0.00551,
    -0.00477,
    0.00107,
    0.00462,
    -0.00155
   ],
   [
    0.00692,
    0.00923,
    -2e-05,
    0.00874,
    -0.00364,
    0.0039,
    -0.00629,
    0.00414,
    0.0043,
    0.00048,
    0.00703,
    -0.00203,
    0.0053,
    -0.00396
   ],
   [
    0.00272,
    0.00796,
    0.01133,
    -0.00027,
    0.01286,
    -0.00368,
    -0.00345,
    0.00701,
    0.00692,
    0.00903,
    -0.00074,
    0.01244,
    -0.00195,
    -0.00202
   ],
   [
    0.00658,
    0.0091,
    -0.00152,
    0.00147,
    0.00619,
    0.00244,
    0.00045,
    0.00302,
    0.0089,
    0.00017,
    0.00176,
    0.00641,
    0.00052,
    -0.00033
   ],
   [
    0.02104,
    0.0031,
    -0.00516,
    -0.00569,
    0.01479,
    0.00301,
    0.01094,
    0.01888,
    -0.00022,
    -0.00186,
    -0.00576,
    0.01363,
    0.00575,
    0.01193
   ],
   [
    -0.01869,
    -0.00634,
    0.01312,
    -0.00958,
    -0.00228,
    0.01075,
    2e-05,
    -0.01488,
    -0.0056,
    0.01195,
    -0.01068,
    -0.00338,
    0.00953,
    -4e-05
   ],
   [
    -0.02054,
    -0.00733,
    0.001,
    -0.00315,
    -0.00333,
    0.00826,
    -0.00965,
    -0.01856,
    -0.01226,
    0.00147,
    -0.00903,
    -0.00758,
    0.00482,
    -0.00972
   ],
   [
    0.02214,
    -0.00227,
    0.00929,
    0.0015,
    -0.00735,
    -0.00645,
    0.01766,
    0.01997,
    0.00031,
    0.00786,
    -0.00059,
    -0.00884,
    -0.00712,
    0.01207
   ],
   [
    -0.00329,
    -0.00195,
    0.01231,
    -0.0008,
    0.00214,
    -0.00051,
    -0.00195,
    -0.00159,
    -0.00012,
    0.01315,
    0.0007,
    0.00171,
    -0.00104,
    -0.00142
   ],
   [
    0.00851,
    -0.00455,
    -0.00712,
    0.01346,
    -0.00603,
    0.00293,
    -0.007,
    0.00973,
    -0.00164,
    -0.00534,
    0.01156,
    -0.0077,
    0.00186,
    -0.00706
   ],
   [
    -0.01706,
    0.00109,
    -0.00674,
    0.00145,
    0.00435,
    -0.0083,
    0.00069,
    -0.01382,
    0.00316,
    -0.00999,
    7e-05,
    0.00358,
    -0.00908,
    -0.00103
   ],
   [
    0.00476,
    0.00728,
    0.0065,
    0.01727,
    -0.0096,
    -0.00166,
    -0.00887,
    0.00189,
    0.00459,
    0.00226,
    0.0154,
    -0.00942,
    -0.00487,
    -0.00986
   ],
   [
    -0.01662,
    -0.01624,
    0.00742,
    0.00406,
    -0.00793,
    0.0025,
    0.00086,
    -0.01797,
    -0.01154,
    0.0019,
    0.00509,
    -0.00579,
    0.00332,
    0.00106
   ],
   [
    -0.00546,
    -0.0048,
    -3e-05,
    -0.00446,
    0.00657,
    0.0001,
    -0.00219,
    -0.0026,
    -0.00163,
    0.00259,
    -0.00324,
    0.00505,
    0.00381,
    -0.00025
   ],
   [
    -0.0064,
    0.0133,
    -0.00474,
    0.0085,
    -0.00994,
    0.00662,
    0.00034,
    -0.00335,
    0.00978,
    -0.00371,
    0.00478,
    -0.00961,
    0.00442,
    0.00099
   ],
   [
    -0.00725,
    0.02481,
    -0.00852,
    -0.01294,
    0.00663,
    0.00844,
    -0.004,
    -0.0071,
    0.01869,
    -0.00605,
    -0.01129,
    0.00737,
    0.00823,
    -0.00104
   ],
   [
    -0.00058,
    0.00678,
    0.0051,
    -0.00491,
    0.00928,
    -0.00688,
    0.00248,
    -0.00186,
    0.00449,
    0.00277,
    -0.00518,
    0.00814,
    -0.00553,
    0.0024
   ],
   [
    0.01613,
    0.00575,
    0.0024,
    0.00202,
    -0.0049,
    7e-05,
    0.001,
    0.01264,
    0.00441,
    0.00312,
    0.00337,
    -0.00432,
    -0.00103,
    0.00085
   ],
   [
    -0.00676,
    -0.00397,
    -0.00914,
    0.00963,
    0.00952,
    -0.0062,
    -0.00625,
    -0.00297,
    -0.01012,
    -0.00867,
    0.00727,
    0.00524,
    -0.00802,
    -0.0073
   ],
   [
    -0.00485,
    -0.02577,
    0.00944,
    0.00091,
    -0.01104,
    0.01294,
    -0.00481,
    -0.00601,
    -0.02097,
    0.00541,
    0.00095,
    -0.0103,
    0.01121,
    -0.00382
   ],
   [
    0.00765,
    0.0057,
    0.0019,
    -0.0137,
    -0.00011,
    0.00626,
    0.01118,
    0.00477,
    0.00667,
    0.00412,
    -0.01262,
    -0.00199,
    0.00425,
    0.00843
   ],
   [
    0.01513,
    -0.00346,
    -0.00529,
    0.01444,
    -0.00222,
    -0.01018,
    0.0059,
    0.01715,
    -0.00073,
    -0.00256,
    0.01268,
    -0.00114,
    -0.00962,
    0.00323
   ],
   [
    -0.01059,
    -0.01283,
    0.01041,
    -6e-05,
    -0.01138,
    -0.00333,
    0.00377,
    -0.0104,
    -0.01111,
    0.00808,
    0.00495,
    -0.00688,
    0.0012,
    0.00367
   ],
   [
    0.00537,
    0.00403,
    -0.01489,
    0.01916,
    -0.00357,
    -0.00667,
    -0.01371,
    0.00363,
    -0.00174,
    -0.01381,
    0.01751,
    -0.00145,
    -0.00595,
    -0.01017
   ],
   [
    -0.01015,
    -0.00728,
    -0.00217,
    -0.00046,
    0.00295,
    0.00047,
    -0.00132,
    -0.00925,
    -0.00953,
    -0.00221,
    0.00033,
    0.00273,
    0.00078,
    -1e-05
   ],
   [
    0.02685,
    0.005,
    0.00984,
    0.00864,
    -0.01894,
    -0.00431,
    -0.00746,
    0.02259,
    0.00335,
    0.00571,
    0.00955,
    -0.01517,
    -0.00301,
    -0.00543
   ],
   [
    0.01723,
    0.02096,
    0.00892,
    0.00798,
    0.00173,
    0.00939,
    0.00607,
    0.01901,
    0.02008,
    0.00866,
    0.00882,
    0.00438,
    0.01046,
    0.00806
   ],
   [
    -0.00259,
    0.00103,
    -0.00149,
    0.00022,
    -0.00067,
    -0.00175,
    -0.00897,
    -0.00266,
    0.00086,
    -0.0051,
    -0.00033,
    -0.00037,
    -0.00141,
    -0.0085
   ],
   [
    0.00547,
    -0.01399,
    0.01133,
    -0.00331,
    -0.00829,
    -0.01474,
    -0.00437,
    0.00276,
    -0.01351,
    0.00279,
    -0.00552,
    -0.0089,
    -0.0144,
    -0.0067
   ],
   [
    -0.0072,
    -0.00955,
    0.00396,
    -0.00148,
    -0.00418,
    -0.00589,
    -0.0019,
    -0.00608,
    -0.00565,
    0.00095,
    -0.00368,
    -0.00705,
    -0.00828,
    -0.00501
   ],
   [
    0.0021,
    -0.00464,
    -0.00286,
    -0.00231,
    0.00502,
    -0.00086,
    -0.00594,
    0.00281,
    -0.00489,
    -0.00677,
    -0.00086,
    0.00514,
    -0.00301,
    -0.00619
   ],
   [
    -0.01569,
    -0.01331,
    0.0028,
    0.00856,
    -0.0097,
    -0.00318,
    0.00265,
    -0.01297,
    -0.01326,
    0.00321,
    0.0065,
    -0.01107,
    -0.00352,
    -0.00022
   ],
   [
    0.02021,
    -0.00611,
    0.01262,
    0.00163,
    -0.01342,
    -0.0006,
    0.0008,
    0.01552,
    -0.00439,
    0.01022,
    0.00476,
    -0.01062,
    -0.00091,
    0.00119
   ],
   [
    0.00804,
    0.00082,
    -0.02888,
    0.02797,
    -0.01991,
    -0.00859,
    -0.02177,
    0.00774,
    0.00045,
    -0.02238,
    0.02899,
    -0.01556,
    -0.00572,
    -0.01711
   ],
   [
    -0.01086,
    -0.01554,
    0.00219,
    0.01473,
    -0.00294,
    -0.01337,
    -0.01414,
    -0.00948,
    -0.01915,
    -0.00083,
    0.01235,
    -0.00449,
    -0.00935,
    -0.01164
   ],
   [
    0.00514,
    -0.00782,
    0.00018,
    0.00255,
    -0.00327,
    -0.00184,
    0.0105,
    0.00243,
    -0.00524,
    0.00225,
    0.00489,
    -0.00162,
    -0.00161,
    0.0083
   ],
   [
    0.00781,
    -9e-05,
    -0.00922,
    0.00082,
    -0.00412,
    0.00754,
    -0.00521,
    0.00302,
    -0.00291,
    -0.00806,
    -0.00187,
    -0.00511,
    0.0038,
    -0.00554
   ],
   [
    0.00377,
    -0.00085,
    0.0061,
    0.0007,
    -0.00612,
    0.0108,
    0.0125,
    0.00035,
    0.0078,
    0.01047,
    0.00488,
    0.00089,
    0.01291,
    0.01289
   ],
   [
    0.00224,
    0.00472,
    -0.02333,
    -0.0014,
    0.00062,
    -0.00363,
    -0.02674,
    0.00091,
    -0.00188,
    -0.0239,
    -0.00287,
    -0.00392,
    -0.00666,
    -0.02526
   ],
   [
    0.01142,
    -6e-05,
    -0.00416,
    -0.00619,
    0.00935,
    0.00351,
    0.00145,
    0.00851,
    -0.00054,
    -0.00701,
    -0.00548,
    0.00739,
    0.0019,
    -8e-05
   ],
   [
    -0.02257,
    -0.00041,
    -0.00319,
    -0.00777,
    -0.00595,
    -0.00293,
    -0.00598,
    -0.01906,
    -0.00674,
    -0.00503,
    -0.00841,
    -0.00632,
    -0.00424,
    -0.00371
   ],
   [
    0.01698,
    -0.00915,
    0.01488,
    -0.01238,
    -0.01013,
    0.0034,
    0.00991,
    0.01229,
    -0.00089,
    0.01579,
    -0.01134,
    -0.00957,
    0.00392,
    0.00675
   ],
   [
    -0.00115,
    0.00579,
    -0.0015,
    -0.00918,
    -0.00417,
    0.00578,
    0.00837,
    -0.00325,
    0.00308,
    -0.00232,
    -0.01062,
    -0.00546,
    0.00223,
    0.00532
   ],
   [
    -0.00042,
    -0.00319,
    0.00798,
    -0.00077,
    0.00678,
    0.00297,
    0.00295,
    -0.00364,
    -0.00081,
    0.00287,
    -0.00067,
    0.00519,
    0.00193,
    0.00306
   ],
   [
    0.00265,
    0.023,
    -0.00159,
    -0.01172,
    0.01572,
    0.00414,
    0.0033,
    0.00196,
    0.0175,
    -0.00317,
    -0.01123,
    0.01394,
    0.00355,
    0.00216
   ],
   [
    -0.00169,
    0.00302,
    0.00382,
    0.0166,
    -0.01135,
    -0.01381,
    -0.01219,
    0.00051,
    -0.00078,
    0.0003,
    0.01341,
    -0.01054,
    -0.01337,
    -0.01036
   ],
   [
    -0.01977,
    -0.01103,
    -0.02077,
    -0.00842,
    0.00385,
    0.00033,
    0.004,
    -0.01783,
    -0.01293,
    -0.0165,
    -0.00801,
    0.00579,
    0.00019,
    0.0034
   ],
   [
    -0.00225,
    0.00014,
    0.0178,
    -0.00159,
    -0.01417,
    0.00372,
    0.00731,
    -0.00274,
    0.0017,
    0.01672,
    -0.00018,
    -0.01246,
    0.0027,
    0.0082
   ],
   [
    -0.01642,
    -0.00581,
    0.01198,
    0.00416,
    -0.00357,
    -0.00948,
    0.00457,
    -0.01651,
    -0.00064,
    0.00474,
    0.00417,
    -0.00288,
    -0.00726,
    0.00447
   ],
   [
    0.01177,
    -0.00194,
    -0.0065,
    0.0098,
    0.00781,
    -0.00047,
    -0.00689,
    0.01171,
    0.00377,
    -0.00703,
    0.00952,
    0.00711,
    0.00118,
    -0.00599
   ],
   [
    -0.02225,
    -0.0181,
    0.01088,
    -0.00561,
    -0.00791,
    0.01228,
    -0.00196,
    -0.01575,
    -0.01551,
    0.01139,
    -0.00424,
    -0.00556,
    0.01282,
    -0.00011
   ],
   [
    0.00316,
    -0.0086,
    0.01038,
    0.00052,
    -0.00146,
    -0.00181,
    -0.00456,
    0.00279,
    -0.00729,
    0.00827,
    0.0023,
    -0.00253,
    -0.00206,
    -0.00395
   ],
   [
    0.00319,
    -0.00799,
    -0.00163,
    0.00198,
    -0.00575,
    0.00227,
    0.00248,
    0.00437,
    -0.00708,
    -0.00242,
    -1e-05,
    -0.0086,
    0.00019,
    0.00136
   ],
   [
    0.01977,
    0.00631,
    0.01275,
    0.00838,
    0.00041,
    -0.00378,
    -0.00189,
    0.01454,
    0.0044,
    0.00975,
    0.00839,
    0.00158,
    -0.00372,
    -0.00333
   ],
   [
    -0.0175,
    -0.00701,
    0.00206,
    -0.00372,
    -0.00344,
    0.01124,
    -0.00359,
    -0.01251,
    -0.00634,
    0.00513,
    -0.00562,
    -0.00363,
    0.00929,
    -0.00207
   ],
   [
    -0.00297,
    0.01014,
    0.00093,
    -0.00185,
    -0.01343,
    0.00164,
    -0.00096,
    -0.00217,
    0.00814,
    -5e-05,
    -0.0007,
    -0.012,
    0.00073,
    -0.00018
   ],
   [
    -0.001,
    0.00757,
    -9e-05,
    0.00328,
    -0.00108,
    -0.00433,
    -0.00069,
    -0.00372,
    0.00462,
    0.00041,
    0.00379,
    -0.00124,
    -0.00753,
    -0.00364
   ],
   [
    0.01473,
    -0.0092,
    -0.00012,
    0.00859,
    -0.00472,
    0.00121,
    0.00951,
    0.00903,
    -0.00643,
    -0.00143,
    0.00937,
    -0.00365,
    0.00311,
    0.00847
   ],
   [
    0.01729,
    0.00286,
    -0.00208,
    0.01378,
    0.0031,
    -0.01373,
    -0.00577,
    0.01242,
    0.00553,
    -0.00021,
    0.01475,
    0.00181,
    -0.01026,
    -0.00642
   ],
   [
    0.00019,
    0.00767,
    0.01498,
    -0.00354,
    0.00116,
    -0.0013,
    0.01033,
    0.00169,
    0.00282,
    0.00954,
    -0.00489,
    0.001,
    -0.00015,
    0.00904
   ],
   [
    -0.0097,
    -0.00201,
    0.00362,
    0.00547,
    -0.00583,
    -0.00682,
    -0.01588,
    -0.00835,
    -0.00127,
    0.00341,
    0.00558,
    -0.00622,
    -0.00775,
    -0.01534
   ],
   [
    0.01265,
    -0.02281,
    -0.01442,
    0.01676,
    -0.00999,
    -0.00413,
    0.01442,
    0.00944,
    -0.01618,
    -0.01188,
    0.0188,
    -0.00899,
    -0.00354,
    0.0132
   ],
   [
    -0.00736,
    0.00713,
    -0.00964,
    -0.00667,
    0.01083,
    -0.00483,
    0.00482,
    -0.00711,
    0.00361,
    -0.00554,
    -0.00861,
    0.00785,
    -0.00583,
    0.00414
   ],
   [
    -0.01087,
    -0.03243,
    0.00674,
    -0.0059,
    -0.01993,
    0.01869,
    0.01445,
    -0.00947,
    -0.02173,
    -0.00127,
    -0.00446,
    -0.01729,
    0.01646,
    0.0102
   ],
   [
    -0.00834,
    0.00314,
    0.00205,
    -0.01664,
    0.00879,
    0.01139,
    0.00564,
    -0.0073,
    0.0018,
    -0.00108,
    -0.01392,
    0.01062,
    0.01311,
    0.00631
   ],
   [
    -0.00954,
    -0.00438,
    -0.00239,
    -0.00798,
    -0.00407,
    -0.00361,
    0.02364,
    -0.00756,
    -0.00359,
    -0.00547,
    -0.00861,
    -0.00279,
    -0.00247,
    0.01956
   ],
   [
    0.01904,
    -0.00867,
    0.00179,
    -0.00098,
    0.00156,
    -0.0091,
    -0.00539,
    0.01415,
    -0.00912,
    -0.00224,
    0.0006,
    0.00227,
    -0.00796,
    -0.00475
   ],
   [
    0.01042,
    -0.00599,
    0.00538,
    -0.00412,
    -0.00353,
    0.00897,
    0.01455,
    0.01169,
    -0.00173,
    0.00694,
    -0.00525,
    -0.00373,
    0.00717,
    0.01203
   ],
   [
    -0.0067,
    -0.01738,
    0.00037,
    0.01468,
    -0.01409,
    -0.00597,
    -0.00146,
    -0.0084,
    -0.01382,
    -0.00255,
    0.01244,
    -0.0134,
    -0.00466,
    -0.00161
   ],
   [
    -0.01311,
    0.00622,
    -0.01052,
    0.00937,
    0.02007,
    -0.01949,
    -0.02583,
    -0.0131,
    -0.0039,
    -0.01307,
    0.00384,
    0.01411,
    -0.0168,
    -0.02163
   ],
   [
    0.00953,
    0.00522,
    0.007,
    0.00406,
    4e-05,
    0.00307,
    0.01008,
    0.00595,
    0.0084,
    0.00356,
    0.00518,
    -0.00084,
    0.004,
    0.0086
   ],
   [
    -0.0164,
    0.00356,
    0.00215,
    -0.00282,
    0.0054,
    -0.0012,
    -0.01083,
    -0.01136,
    -0.00103,
    -0.00249,
    -0.0033,
    0.00321,
    -0.001,
    -0.0082
   ],
   [
    -0.02726,
    -0.00753,
    0.00095,
    -0.00275,
    -0.00385,
    -0.00034,
    -0.0039,
    -0.02318,
    -0.01155,
    0.00144,
    -0.002,
    -0.00327,
    0.00214,
    -0.00093
   ],
   [
    -0.00775,
    0.00543,
    -0.00613,
    0.00103,
    0.01815,
    0.00534,
    -0.00806,
    -0.00692,
    0.00709,
    -0.00529,
    -0.00069,
    0.01189,
    0.0005,
    -0.00865
   ],
   [
    0.00392,
    0.02042,
    0.0028,
    0.00021,
    0.00096,
    0.00913,
    -0.00257,
    0.00504,
    0.02267,
    0.00619,
    -0.00151,
    -0.00102,
    0.00714,
    -0.00123
   ],
   [
    0.00803,
    0.01503,
    0.00423,
    0.0118,
    -0.00477,
    0.00919,
    -0.00841,
    0.00837,
    0.01662,
    0.00556,
    0.00909,
    -0.00449,
    0.00846,
    -0.00662
   ],
   [
    -0.00781,
    -0.00455,
    0.00519,
    0.00067,
    0.00214,
    -0.00289,
    0.00119,
    -0.00751,
    -0.00212,
    0.00445,
    0.00176,
    0.00392,
    0.0004,
    0.00126
   ],
   [
    0.00777,
    0.00625,
    0.00466,
    0.00401,
    0.01015,
    -0.00112,
    -0.00067,
    0.00861,
    0.01,
    0.00346,
    0.00457,
    0.00926,
    0.00241,
    0.00116
   ],
   [
    0.00551,
    -0.01715,
    0.0061,
    0.00105,
    -0.01065,
    0.00599,
    -0.00507,
    0.00165,
    -0.02087,
    0.00301,
    -0.00356,
    -0.01038,
    0.00419,
    -0.00453
   ],
   [
    0.01962,
    0.00602,
    0.00049,
    0.01229,
    0.00096,
    -0.00495,
    -0.00019,
    0.01451,
    0.00421,
    -0.00303,
    0.01214,
    0.00071,
    -0.00416,
    -0.00143
   ],
   [
    -0.01054,
    0.00504,
    -0.01182,
    0.00383,
    -0.00909,
    0.00159,
    0.00145,
    -0.01121,
    0.00204,
    -0.00762,
    0.00452,
    -0.00595,
    0.00304,
    0.00225
   ],
   [
    0.01147,
    0.01757,
    0.0088,
    -0.00868,
    0.02107,
    -0.01241,
    -0.00606,
    0.00875,
    0.01352,
    -0.00102,
    -0.00941,
    0.01593,
    -0.01333,
    -0.00729
   ],
   [
    -0.02147,
    0.0039,
    0.00659,
    -0.00389,
    0.00079,
    -0.00174,
    -0.00369,
    -0.013,
    -0.00197,
    0.00394,
    -0.00593,
    9e-05,
    -0.00159,
    -0.00341
   ],
   [
    -0.02187,
    0.00518,
    -0.01036,
    -0.00888,
    0.00556,
    0.00147,
    -0.00849,
    -0.01429,
    0.0024,
    -0.00284,
    -0.0104,
    0.00394,
    0.00202,
    -0.00586
   ],
   [
    -0.02413,
    0.00188,
    0.0072,
    -0.01176,
    0.00766,
    0.01601,
    0.00955,
    -0.01606,
    0.00568,
    0.00312,
    -0.0109,
    0.0085,
    0.0163,
    0.01007
   ],
   [
    -0.01923,
    0.00346,
    0.01238,
    -0.01033,
    0.00793,
    0.00428,
    -0.00935,
    -0.01503,
    0.00414,
    0.00844,
    -0.0099,
    0.00594,
    0.0022,
    -0.00889
   ],
   [
    -0.00243,
    -0.00813,
    0.01222,
    -0.00898,
    -0.01201,
    0.00307,
    0.02363,
    -0.00225,
    -0.00578,
    0.00861,
    -0.00916,
    -0.01208,
    0.00093,
    0.01987
   ],
   [
    -0.00292,
    -0.01268,
    -0.00518,
    0.0004,
    -0.0069,
    -0.00947,
    0.02088,
    -0.00073,
    -0.00407,
    -0.00431,
    0.00293,
    -0.00377,
    -0.00725,
    0.01659
   ],
   [
    -0.01258,
    0.00528,
    -0.00367,
    0.00628,
    0.00233,
    -0.00036,
    -0.00891,
    -0.01117,
    0.00058,
    -0.00125,
    0.00583,
    0.00142,
    -0.00187,
    -0.00927
   ],
   [
    -0.00982,
    -0.00508,
    -0.01053,
    0.00215,
    0.01363,
    -0.00967,
    -0.00492,
    -0.01084,
    -0.00202,
    -0.00801,
    0.00203,
    0.01344,
    -0.00872,
    -0.00347
   ],
   [
    0.00468,
    -0.00894,
    -0.00082,
    0.01079,
    -0.00376,
    0.00214,
    -0.00281,
    0.00265,
    -0.00615,
    -0.00116,
    0.00899,
    -0.00196,
    0.00457,
    -0.00174
   ],
   [
    -0.0179,
    -0.01116,
    -0.00036,
    0.01688,
    -0.00684,
    -0.00788,
    0.00514,
    -0.01451,
    -0.00743,
    -0.00239,
    0.01468,
    -0.00595,
    -0.00705,
    0.00259
   ],
   [
    0.02068,
    -0.01347,
    0.0137,
    0.009,
    -0.01763,
    0.01812,
    -0.00751,
    0.01669,
    -0.00965,
    0.0113,
    0.00834,
    -0.0161,
    0.01556,
    -0.0077
   ],
   [
    -0.00342,
    0.01195,
    0.01848,
    -0.00203,
    -0.00086,
    0.01694,
    0.00377,
    0.00457,
    0.01155,
    0.01356,
    -0.00258,
    0.00037,
    0.01746,
    0.00605
   ],
   [
    0.01682,
    -0.00906,
    0.00194,
    0.00523,
    -0.00391,
    0.00119,
    0.00198,
    0.01093,
    -0.00859,
    -0.00017,
    0.00377,
    -0.00469,
    -0.00135,
    -0.00077
   ],
   [
    -0.00051,
    0.00162,
    0.01062,
    -0.00784,
    -0.01261,
    0.00391,
    0.00819,
    -0.00244,
    0.0007,
    0.00942,
    -0.00543,
    -0.00936,
    0.00595,
    0.00957
   ],
   [
    -0.02656,
    0.00063,
    -0.01609,
    0.02965,
    0.00496,
    -0.02649,
    -0.03415,
    -0.02646,
    -0.01027,
    -0.01584,
    0.02436,
    0.00449,
    -0.0253,
    -0.02786
   ],
   [
    -0.00595,
    0.01619,
    -0.01351,
    -0.01251,
    0.00317,
    0.00057,
    0.00844,
    -0.00653,
    0.00653,
    -0.00813,
    -0.01509,
    0.0007,
    0.00035,
    0.00679
   ],
   [
    -0.00774,
    -0.00081,
    0.00142,
    -0.01585,
    -0.00105,
    0.00643,
    0.00758,
    -0.0061,
    -0.00419,
    -0.00014,
    -0.0144,
    0.00054,
    0.00791,
    0.00857
   ],
   [
    -0.00622,
    -0.01898,
    -0.00685,
    -0.00389,
    -0.00485,
    0.00439,
    0.00196,
    -0.00551,
    -0.01234,
    -0.00567,
    -0.00319,
    -0.00555,
    0.00402,
    3e-05
   ],
   [
    -0.02657,
    -0.01528,
    -0.00771,
    0.00023,
    -0.00367,
    0.00596,
    -0.00183,
    -0.02332,
    -0.00766,
    -0.01052,
    0.00115,
    -0.00108,
    0.00687,
    -0.00362
   ],
   [
    0.00616,
    0.01437,
    0.01405,
    -0.00147,
    -0.00267,
    0.00374,
    -0.00241,
    0.00522,
    0.01227,
    0.00597,
    0.00062,
    0.00014,
    0.00136,
    -0.00151
   ],
   [
    -0.00543,
    -0.00489,
    0.0021,
    0.00452,
    -0.00588,
    0.00288,
    0.00961,
    -0.00428,
    -0.0006,
    0.00549,
    0.0043,
    -0.00657,
    0.00264,
    0.00728
   ],
   [
    0.01063,
    0.01966,
    0.00746,
    -0.00661,
    0.00717,
    -0.00134,
    0.011,
    0.00856,
    0.01647,
    0.00371,
    -0.00365,
    0.00813,
    -0.00164,
    0.00954
   ],
   [
    -0.00605,
    -0.01904,
    -0.01115,
    0.0023,
    -0.01021,
    -0.00324,
    -0.01055,
    -0.00844,
    -0.02063,
    -0.00352,
    0.00236,
    -0.01023,
    -0.00402,
    -0.00868
   ],
   [
    0.01247,
    -0.01259,
    -0.00275,
    0.0049,
    -0.00736,
    -0.01365,
    0.00468,
    0.01021,
    -0.00876,
    -0.00016,
    0.00357,
    -0.00434,
    -0.01014,
    0.00362
   ],
   [
    -0.01145,
    0.00293,
    -0.00685,
    0.00579,
    0.00211,
    0.00261,
    -0.00886,
    -0.01145,
    -0.00231,
    -0.00775,
    0.00182,
    -0.00197,
    -0.00071,
    -0.00761
   ],
   [
    -0.01417,
    0.00695,
    -0.00783,
    -0.00986,
    0.00777,
    0.01348,
    -0.0109,
    -0.0084,
    0.00605,
    -0.00585,
    -0.00925,
    0.00888,
    0.01337,
    -0.00666
   ],
   [
    -0.00914,
    -0.00116,
    0.003,
    -0.00537,
    0.00433,
    0.00957,
    -0.00847,
    -0.00886,
    -0.00546,
    -0.00148,
    -0.00592,
    0.0028,
    0.00672,
    -0.0067
   ],
   [
    0.01559,
    -0.00593,
    -0.00216,
    0.0078,
    -0.00198,
    0.00821,
    -0.01475,
    0.01187,
    -0.00098,
    0.00476,
    0.0084,
    -0.00237,
    0.01036,
    -0.00871
   ],
   [
    -0.00137,
    -0.00391,
    -0.00952,
    0.00617,
    -0.01047,
    -0.00218,
    0.00566,
    -0.00062,
    -0.00022,
    -0.00441,
    0.00809,
    -0.00758,
    -0.00234,
    0.00342
   ],
   [
    0.00584,
    0.02294,
    -0.00769,
    -0.0005,
    0.01752,
    -0.00873,
    0.00139,
    0.00866,
    0.0217,
    -0.01102,
    -0.0032,
    0.01152,
    -0.00965,
    -0.00121
   ],
   [
    -0.00889,
    0.01364,
    -0.00479,
    -0.00602,
    -0.00285,
    0.01091,
    0.00484,
    -0.009,
    0.01051,
    -0.004,
    -0.00522,
    -0.00205,
    0.00798,
    0.00257
   ],
   [
    0.0212,
    0.00759,
    -0.00994,
    0.00874,
    -0.01601,
    -0.00383,
    0.00314,
    0.01977,
    0.00577,
    -0.00465,
    0.0074,
    -0.01441,
    -0.00398,
    0.00221
   ],
   [
    0.00584,
    -0.0047,
    -0.00531,
    0.0096,
    -0.01039,
    -0.00573,
    0.00823,
    0.00684,
    0.00202,
    -0.00389,
    0.00975,
    -0.00904,
    -0.00473,
    0.00649
   ],
   [
    0.00639,
    0.0187,
    -0.01126,
    -0.00177,
    0.00838,
    0.00324,
    -0.00843,
    0.00529,
    0.01546,
    -0.01437,
    -0.00256,
    0.00963,
    0.0044,
    -0.00607
   ],
   [
    -0.00208,
    -0.00354,
    0.00741,
    -0.00949,
    0.00442,
    0.00213,
    0.01795,
    -0.00376,
    -0.00083,
    0.00064,
    -0.00766,
    0.00494,
    0.00135,
    0.01415
   ],
   [
    -0.011,
    0.00966,
    -0.00463,
    -0.01527,
    -0.0104,
    0.012,
    0.00509,
    -0.00977,
    0.01187,
    -0.00775,
    -0.01401,
    -0.00828,
    0.00904,
    0.00225
   ],
   [
    -0.0225,
    -0.00621,
    -0.0008,
    -0.00053,
    0.00861,
    -0.00569,
    -0.00911,
    -0.01984,
    -0.01063,
    -0.00708,
    -0.00121,
    0.00829,
    -0.00777,
    -0.0089
   ],
   [
    -0.00625,
    0.00172,
    -0.00344,
    0.00251,
    0.02164,
    -0.00681,
    -0.01056,
    -0.00497,
    0.00418,
    -0.00764,
    0.00433,
    0.02022,
    -0.00572,
    -0.00963
   ],
   [
    -0.00142,
    0.01719,
    -0.00077,
    -0.00223,
    -0.0056,
    0.0068,
    -0.01107,
    -0.00028,
    0.0116,
    -0.00103,
    -0.00601,
    -0.00507,
    0.00747,
    -0.00875
   ],
   [
    -0.01168,
    0.00514,
    -0.00028,
    -0.00089,
    0.01105,
    -0.00081,
    -0.0146,
    -0.00979,
    0.00093,
    -0.00353,
    -0.00173,
    0.00695,
    -0.00362,
    -0.01368
   ],
   [
    -0.00827,
    0.00467,
    -0.0035,
    0.01243,
    -0.00275,
    -0.00078,
    -0.01305,
    -0.01222,
    0.00416,
    0.00143,
    0.01331,
    -0.00211,
    -0.00145,
    -0.01197
   ],
   [
    0.00744,
    -0.00114,
    -0.00189,
    0.00482,
    -0.00195,
    0.00424,
    -0.01164,
    0.00479,
    -0.00822,
    -0.00272,
    0.00082,
    -0.0043,
    0.00156,
    -0.01114
   ],
   [
    0.00344,
    0.00433,
    -0.00375,
    -0.0007,
    0.00424,
    -0.00441,
    -0.00823,
    0.00124,
    0.00465,
    -0.00386,
    -0.0019,
    0.00235,
    -0.00437,
    -0.00771
   ],
   [
    0.01757,
    0.00151,
    0.01091,
    -0.00172,
    -0.00575,
    0.00575,
    -2e-05,
    0.01589,
    0.00261,
    0.00799,
    -0.00238,
    -0.00512,
    0.00496,
    -0.00068
   ],
   [
    -0.00806,
    -0.01862,
    0.00127,
    -0.02136,
    -0.0301,
    0.02677,
    0.01974,
    -0.00887,
    -0.01594,
    0.00542,
    -0.01771,
    -0.02201,
    0.02716,
    0.02029
   ],
   [
    -0.00561,
    -0.0104,
    -0.00025,
    -0.00102,
    -0.00858,
    0.00125,
    0.00899,
    -0.00544,
    -0.00926,
    0.00498,
    -0.00287,
    -0.00856,
    0.00038,
    0.00637
   ],
   [
    0.00949,
    -0.01457,
    -0.00111,
    -0.00202,
    -0.00572,
    0.00596,
    0.01188,
    0.00507,
    -0.00683,
    -0.00212,
    0.00128,
    -0.00465,
    0.00513,
    0.00931
   ],
   [
    0.00184,
    -0.00256,
    0.01305,
    0.0023,
    0.00832,
    -0.00731,
    -0.01382,
    0.00053,
    0.00106,
    0.0072,
    0.00366,
    0.00818,
    -0.00792,
    -0.01052
   ],
   [
    0.01107,
    0.00889,
    -0.00162,
    -0.00401,
    0.01588,
    -0.00381,
    -0.00285,
    0.00929,
    0.00773,
    0.00029,
    -0.0025,
    0.01251,
    -0.00204,
    -0.00189
   ],
   [
    -0.00484,
    0.01351,
    0.00065,
    0.0009,
    -0.00034,
    0.01397,
    0.01132,
    -0.00231,
    0.01328,
    -0.00116,
    0.00028,
    0.00184,
    0.01507,
    0.0121
   ],
   [
    0.00077,
    0.01057,
    0.00521,
    -0.00378,
    0.00994,
    0.00155,
    0.00167,
    0.0008,
    0.01117,
    0.00077,
    -0.00164,
    0.00957,
    0.00211,
    0.00266
   ],
   [
    0.00965,
    0.01105,
    -0.00655,
    0.01011,
    0.0088,
    -0.00366,
    0.00722,
    0.01172,
    0.01502,
    -0.00412,
    0.01119,
    0.00858,
    -0.00108,
    0.00483
   ],
   [
    0.00263,
    -0.00929,
    0.0006,
    0.00558,
    -0.01674,
    -0.00684,
    -0.00194,
    -0.00238,
    -0.00374,
    0.00056,
    0.00541,
    -0.01477,
    -0.00765,
    -0.00217
   ],
   [
    0.00684,
    -0.00197,
    -0.00128,
    0.00865,
    -0.0035,
    -0.00886,
    -0.00081,
    0.00301,
    -0.00243,
    -0.0005,
    0.00721,
    -0.00399,
    -0.0079,
    -0.00185
   ],
   [
    0.00266,
    0.01456,
    -0.00831,
    0.00659,
    0.00649,
    -0.008,
    0.00333,
    0.00081,
    0.0164,
    -0.00521,
    0.00723,
    0.00521,
    -0.00762,
    0.00191
   ],
   [
    -0.02662,
    -0.02267,
    -0.0008,
    0.01663,
    -0.00733,
    0.00259,
    -0.00943,
    -0.02245,
    -0.02373,
    -0.00046,
    0.01436,
    -0.00669,
    -0.00098,
    -0.00922
   ],
   [
    -0.00529,
    -0.0108,
    -0.00331,
    -0.00092,
    -0.00986,
    -0.00149,
    0.01255,
    -0.00985,
    -0.01218,
    -0.00054,
    -0.0009,
    -0.01147,
    -0.00266,
    0.00925
   ],
   [
    -0.00949,
    -0.00504,
    -0.00614,
    -0.00818,
    0.005,
    0.01066,
    0.0091,
    -0.00273,
    -0.00388,
    -0.00665,
    -0.01196,
    0.00444,
    0.00946,
    0.0076
   ],
   [
    -0.00137,
    -0.00753,
    0.00115,
    0.01397,
    -0.00872,
    0.00259,
    0.0047,
    0.00011,
    -0.00511,
    0.0017,
    0.01354,
    -0.00541,
    0.00176,
    0.00237
   ],
   [
    0.01029,
    0.0184,
    0.00567,
    -0.00261,
    0.0027,
    -0.00087,
    -0.00379,
    0.00635,
    0.01237,
    -0.00013,
    -0.00193,
    0.00278,
    -0.00068,
    -0.00398
   ],
   [
    -0.0086,
    -0.01899,
    -0.01382,
    0.00623,
    -0.01218,
    -0.00589,
    0.02022,
    -0.01103,
    -0.01548,
    -0.00828,
    0.00699,
    -0.01085,
    -0.00754,
    0.01443
   ],
   [
    -0.02601,
    -0.02657,
    0.00676,
    -0.00486,
    -0.00576,
    0.01442,
    0.00353,
    -0.02321,
    -0.0266,
    0.005,
    -0.00534,
    -0.00439,
    0.01093,
    0.00182
   ],
   [
    -0.01122,
    -0.02135,
    -0.0008,
    0.00765,
    -0.00198,
    0.0031,
    -0.0194,
    -0.00802,
    -0.01759,
    0.00253,
    0.00917,
    -0.00143,
    0.00234,
    -0.01706
   ],
   [
    -0.01106,
    -0.0108,
    0.0049,
    -0.01039,
    -0.00068,
    -0.00651,
    0.00135,
    -0.01098,
    -0.01337,
    -0.00211,
    -0.011,
    -0.00326,
    -0.00726,
    -0.00209
   ],
   [
    -0.00713,
    -0.01688,
    -0.00614,
    0.00565,
    -0.01132,
    -0.00419,
    0.01728,
    -0.00461,
    -0.01177,
    0.00019,
    0.00599,
    -0.01001,
    -0.00422,
    0.01316
   ],
   [
    0.01011,
    0.01292,
    -0.00117,
    -0.00415,
    0.00058,
    0.01457,
    -0.0089,
    0.00725,
    0.00853,
    0.00383,
    -0.00419,
    -0.00169,
    0.00997,
    -0.00933
   ],
   [
    -0.01654,
    5e-05,
    -0.00265,
    0.00243,
    -0.00015,
    -0.00146,
    0.00272,
    -0.01355,
    -0.00308,
    -0.00243,
    0.0055,
    0.00054,
    0.00052,
    0.00198
   ],
   [
    0.01247,
    -0.01507,
    -0.0087,
    0.01315,
    -0.02167,
    0.00367,
    0.0054,
    0.01234,
    -0.00793,
    -0.00653,
    0.012,
    -0.01628,
    0.00478,
    0.00516
   ],
   [
    -0.01145,
    -0.00081,
    -0.00079,
    -0.00578,
    0.00034,
    0.00198,
    -0.00657,
    -0.01341,
    0.00066,
    -0.00145,
    -0.00349,
    0.0,
    0.00212,
    -0.0065
   ],
   [
    -0.00234,
    -0.00492,
    0.00096,
    0.00135,
    0.00231,
    0.00064,
    0.00229,
    -0.00246,
    -0.00695,
    -0.00176,
    -0.00053,
    0.00325,
    -0.00184,
    0.00023
   ],
   [
    0.00071,
    0.00921,
    -0.00554,
    0.00619,
    0.00625,
    -0.006,
    0.00529,
    0.00045,
    0.00812,
    -0.0025,
    0.00671,
    0.00804,
    -0.00257,
    0.00569
   ],
   [
    0.02041,
    -0.0014,
    -0.00097,
    0.01282,
    -0.00457,
    -0.01437,
    0.00171,
    0.01721,
    -0.00082,
    0.00027,
    0.01306,
    -0.00295,
    -0.01142,
    0.00201
   ],
   [
    -0.00283,
    -0.0093,
    -0.00183,
    6e-05,
    -0.00377,
    0.00847,
    0.00371,
    -0.0015,
    -0.00574,
    0.00016,
    -0.00128,
    -0.00434,
    0.00758,
    0.00312
   ],
   [
    -0.00756,
    0.00028,
    -0.00199,
    0.00704,
    -0.01024,
    0.01083,
    -0.00606,
    -0.0094,
    -0.001,
    -0.00033,
    0.00814,
    -0.00851,
    0.00773,
    -0.00516
   ],
   [
    0.01145,
    0.01503,
    0.00109,
    0.00167,
    -0.0003,
    -0.00096,
    0.00518,
    0.00662,
    0.01444,
    0.00475,
    0.00255,
    0.00244,
    -0.00466,
    0.00203
   ],
   [
    0.0176,
    0.00611,
    0.00913,
    -0.00288,
    0.0019,
    0.00123,
    -0.01012,
    0.01483,
    0.00897,
    0.00761,
    -0.00358,
    0.00113,
    0.0007,
    -0.00959
   ],
   [
    0.00563,
    0.00972,
    -0.00688,
    0.00214,
    0.00091,
    -0.01021,
    -0.00862,
    0.00337,
    0.01189,
    -0.00357,
    0.00378,
    0.00144,
    -0.00937,
    -0.00839
   ],
   [
    -0.00884,
    0.00244,
    0.00544,
    -0.00515,
    0.00332,
    -0.00218,
    0.00329,
    -0.00724,
    0.001,
    -0.00016,
    -0.00552,
    0.00166,
    -0.00393,
    0.00094
   ],
   [
    0.00126,
    0.00992,
    -0.00262,
    -0.00356,
    0.02083,
    -0.00637,
    -0.01302,
    -0.00171,
    0.01134,
    -0.00909,
    -0.00573,
    0.01576,
    -0.00576,
    -0.01015
   ],
   [
    -0.01035,
    -0.00596,
    -0.00674,
    -0.01729,
    0.00066,
    0.00406,
    0.0034,
    -0.0121,
    -0.00892,
    -0.00528,
    -0.01629,
    -0.00104,
    0.00244,
    0.00128
   ],
   [
    -0.00125,
    -0.01523,
    -0.00712,
    -0.00141,
    -0.01387,
    0.01651,
    0.01512,
    0.00249,
    -0.00931,
    -0.00416,
    -0.00249,
    -0.01265,
    0.01591,
    0.01172
   ],
   [
    0.00321,
    0.00712,
    0.00616,
    -0.00432,
    0.00728,
    -0.00704,
    0.00307,
    0.00191,
    0.00153,
    0.00554,
    -0.00276,
    0.00585,
    -0.00566,
    0.00431
   ],
   [
    -0.01269,
    -0.00815,
    0.00141,
    0.005,
    0.00449,
    0.00204,
    -0.02751,
    -0.01032,
    -0.01114,
    0.0009,
    0.00188,
    0.00018,
    -0.00517,
    -0.02687
   ],
   [
    0.00428,
    0.00425,
    -0.00086,
    0.0022,
    0.00682,
    -0.00249,
    -0.00221,
    0.00275,
    0.00368,
    0.00073,
    0.00563,
    0.00812,
    -0.00184,
    -0.00149
   ],
   [
    0.00076,
    -0.01475,
    -0.01615,
    0.01899,
    0.01116,
    -0.02524,
    -0.01868,
    0.00165,
    -0.00517,
    -0.01052,
    0.01805,
    0.00409,
    -0.02589,
    -0.02233
   ],
   [
    -0.01837,
    -0.01519,
    0.00398,
    8e-05,
    -0.00171,
    0.00856,
    -0.0048,
    -0.01951,
    -0.01152,
    0.00137,
    0.00035,
    -0.00189,
    0.00678,
    -0.00468
   ],
   [
    -0.0014,
    0.01315,
    0.00399,
    -0.0021,
    -0.00818,
    0.00389,
    -0.00196,
    -0.00217,
    0.01057,
    0.00722,
    -0.0037,
    -0.00727,
    0.0028,
    -0.00197
   ],
   [
    -0.02285,
    -0.01283,
    0.00098,
    -0.01705,
    -0.005,
    0.0192,
    0.00923,
    -0.02032,
    -0.01251,
    -0.00203,
    -0.01466,
    -0.00313,
    0.01612,
    0.00749
   ],
   [
    0.00081,
    -0.00353,
    0.00364,
    0.00147,
    0.0131,
    -0.01707,
    -0.01337,
    -0.00086,
    -0.00058,
    0.00055,
    0.0002,
    0.01114,
    -0.01704,
    -0.0142
   ],
   [
    -0.01153,
    -0.01148,
    -0.00457,
    0.0125,
    0.00277,
    0.00469,
    -0.00389,
    -0.00917,
    -0.0094,
    -0.00166,
    0.01042,
    0.00202,
    0.0076,
    -0.00225
   ],
   [
    -0.00433,
    0.00312,
    0.0077,
    0.00064,
    0.0053,
    -0.00456,
    -0.01066,
    -0.00361,
    -0.00059,
    0.00186,
    -0.00266,
    0.00263,
    -0.00623,
    -0.00983
   ],
   [
    -0.01536,
    0.00836,
    0.00899,
    0.0056,
    0.00395,
    -0.00148,
    0.00633,
    -0.01225,
    0.00285,
    0.00398,
    0.00447,
    0.00304,
    -0.00431,
    0.00336
   ],
   [
    -0.00378,
    -0.01881,
    0.01614,
    0.01612,
    -0.00835,
    -0.01077,
    -1e-05,
    -0.0035,
    -0.01908,
    0.00759,
    0.0142,
    -0.00711,
    -0.00966,
    -0.00206
   ],
   [
    0.00826,
    -0.016,
    0.02214,
    0.00433,
    -0.00245,
    -0.00562,
    0.01148,
    0.00837,
    -0.00843,
    0.013,
    0.00358,
    -0.00321,
    -0.00253,
    0.0104
   ],
   [
    -0.01016,
    -0.01798,
    -0.00259,
    -0.00894,
    0.00144,
    -0.00354,
    0.00827,
    -0.01199,
    -0.01446,
    -0.00335,
    -0.00761,
    0.0013,
    -0.00146,
    0.00604
   ],
   [
    0.00478,
    0.02006,
    -0.00221,
    -0.00664,
    0.00383,
    0.00992,
    0.01232,
    0.00558,
    0.02034,
    0.00302,
    -0.00349,
    0.0055,
    0.0092,
    0.01243
   ],
   [
    -0.00279,
    0.02099,
    -0.00113,
    -0.00814,
    -0.00051,
    0.00066,
    0.01955,
    -0.00216,
    0.01915,
    0.00019,
    -0.00733,
    0.00179,
    0.00373,
    0.01973
   ],
   [
    -0.02215,
    0.00749,
    -0.00724,
    0.00495,
    0.00736,
    -0.01298,
    -0.00979,
    -0.01783,
    0.0032,
    -0.01062,
    0.00377,
    0.00501,
    -0.0142,
    -0.00971
   ],
   [
    -0.01236,
    -0.01257,
    -0.00172,
    0.00183,
    -0.00441,
    -0.00395,
    0.00016,
    -0.01216,
    -0.01732,
    -0.00297,
    0.00098,
    -0.0068,
    -0.00545,
    -6e-05
   ],
   [
    -0.01218,
    0.00071,
    -0.00315,
    -0.00374,
    0.0216,
    -0.01025,
    -0.00864,
    -0.01214,
    -5e-05,
    -0.00385,
    -0.00222,
    0.01741,
    -0.00983,
    -0.00703
   ],
   [
    -0.00405,
    -0.00562,
    -0.00358,
    0.01071,
    -0.00458,
    -0.01323,
    -0.01278,
    -0.00464,
    -0.00263,
    -0.00516,
    0.01108,
    -0.00404,
    -0.01137,
    -0.01276
   ],
   [
    0.00295,
    -0.01578,
    -0.00921,
    0.0115,
    -0.00032,
    -0.00953,
    -0.01748,
    -0.00212,
    -0.01504,
    -0.01043,
    0.01042,
    -0.00167,
    -0.01249,
    -0.01831
   ],
   [
    0.00541,
    0.00031,
    -0.01495,
    0.01016,
    0.01453,
    -0.00592,
    -0.02337,
    0.00377,
    0.00317,
    -0.00504,
    0.01083,
    0.01062,
    -0.00475,
    -0.01863
   ],
   [
    -0.01074,
    -0.00127,
    0.00347,
    0.00703,
    0.00101,
    0.00241,
    0.00489,
    -0.00759,
    0.00057,
    0.00872,
    0.01146,
    0.00555,
    0.00653,
    0.00815
   ],
   [
    -0.00166,
    -0.0094,
    0.02047,
    0.00913,
    -0.00903,
    3e-05,
    0.01341,
    0.00184,
    -0.00389,
    0.00874,
    0.01142,
    -0.00408,
    0.00379,
    0.01256
   ],
   [
    -0.00189,
    -0.01349,
    0.01815,
    0.00909,
    -0.00309,
    -0.00427,
    0.00132,
    0.00104,
    -0.00803,
    0.01198,
    0.01076,
    -0.00266,
    -0.00127,
    0.00141
   ],
   [
    0.00658,
    0.00903,
    0.004,
    -0.003,
    -0.00056,
    -0.00056,
    -0.00138,
    0.00745,
    0.00735,
    0.0024,
    -0.00015,
    0.0014,
    0.00181,
    0.00092
   ],
   [
    -0.0016,
    0.01042,
    -0.00636,
    -0.00494,
    0.00045,
    0.00497,
    -0.00845,
    -0.00019,
    0.00954,
    -0.0062,
    -0.00587,
    -0.00044,
    0.00319,
    -0.00718
   ],
   [
    0.00803,
    0.00422,
    -0.00456,
    -0.0058,
    0.00387,
    -0.00437,
    -0.00236,
    0.0086,
    0.00664,
    -0.00357,
    -0.00517,
    0.00371,
    -0.00333,
    -0.0027
   ],
   [
    0.00732,
    0.00882,
    0.00604,
    -0.0086,
    0.00538,
    0.00233,
    2e-05,
    0.00938,
    0.00508,
    0.0073,
    -0.00681,
    0.00397,
    0.00021,
    -0.00098
   ],
   [
    -0.00467,
    -0.00484,
    0.00987,
    0.0067,
    -0.00685,
    -0.00946,
    -0.0006,
    -0.00275,
    -0.00796,
    0.0054,
    0.00577,
    -0.00276,
    -0.00484,
    0.00116
   ],
   [
    -0.00162,
    0.00261,
    -0.01205,
    0.00698,
    0.004,
    -0.00869,
    -0.00326,
    -0.00019,
    0.00546,
    -0.00661,
    0.00651,
    0.00263,
    -0.00682,
    -0.00347
   ],
   [
    -0.01538,
    -0.01362,
    -0.00707,
    -0.00596,
    -0.00731,
    0.01287,
    -0.01015,
    -0.01406,
    -0.01283,
    -0.00407,
    -0.00419,
    -0.00602,
    0.0105,
    -0.00937
   ],
   [
    0.00232,
    6e-05,
    -0.00297,
    0.00688,
    -0.00258,
    -0.0057,
    -0.00877,
    -0.00075,
    -0.0005,
    -0.00573,
    0.00798,
    -0.00168,
    -0.00389,
    -0.00671
   ],
   [
    0.00322,
    -0.00578,
    0.00218,
    0.00532,
    -0.00785,
    0.00467,
    -0.00675,
    0.00458,
    -0.00116,
    0.00239,
    0.00282,
    -0.00872,
    0.0036,
    -0.0082
   ],
   [
    -0.01413,
    -0.00797,
    0.01258,
    0.00698,
    -0.00656,
    0.01074,
    -0.00334,
    -0.01286,
    -0.00816,
    0.00665,
    0.00715,
    -0.00467,
    0.00962,
    -0.0009
   ],
   [
    -0.00157,
    0.0073,
    -0.00838,
    -0.00131,
    0.00021,
    0.00757,
    -0.00306,
    -0.00189,
    0.00514,
    -0.00701,
    -0.00075,
    0.0027,
    0.00636,
    -0.00201
   ],
   [
    0.00776,
    -0.00595,
    0.00806,
    -0.00015,
    -0.00993,
    -0.00412,
    -0.00307,
    0.00345,
    -0.00772,
    -0.00029,
    -1e-05,
    -0.01126,
    -0.0061,
    -0.00447
   ],
   [
    0.01618,
    0.00237,
    0.01052,
    -0.00368,
    0.00321,
    0.00419,
    -0.00729,
    0.01722,
    0.00687,
    0.00903,
    -0.00366,
    0.00272,
    0.00405,
    -0.00523
   ],
   [
    0.00389,
    -0.01414,
    -0.00471,
    0.00246,
    -0.00837,
    -0.00086,
    -0.0041,
    0.00204,
    -0.00903,
    -0.00283,
    0.00269,
    -0.00717,
    -0.00154,
    -0.00287
   ],
   [
    0.00151,
    0.00036,
    -0.01178,
    0.01313,
    0.01214,
    -0.01744,
    -0.00015,
    0.00034,
    0.00038,
    -0.01035,
    0.01319,
    0.00896,
    -0.01692,
    -0.00185
   ],
   [
    0.02869,
    -0.00607,
    0.01563,
    -0.01175,
    -0.00375,
    0.00822,
    0.01469,
    0.02578,
    -0.0012,
    0.01108,
    -0.01204,
    -0.00205,
    0.00968,
    0.01311
   ],
   [
    -0.01037,
    -0.00964,
    -0.01427,
    0.01664,
    -0.01447,
    0.00358,
    -0.0063,
    -0.00704,
    -0.00997,
    -0.00697,
    0.01312,
    -0.01226,
    0.00483,
    -0.00675
   ],
   [
    0.02715,
    0.0079,
    0.01379,
    -0.00545,
    -0.00438,
    0.01306,
    0.02681,
    0.0283,
    0.00651,
    0.01346,
    -0.00162,
    0.00122,
    0.01577,
    0.0262
   ],
   [
    0.00167,
    0.01022,
    -0.0004,
    -0.00909,
    -0.00819,
    0.01156,
    0.00542,
    0.00712,
    0.00609,
    -0.0031,
    -0.01269,
    -0.00768,
    0.0089,
    0.00584
   ],
   [
    0.00654,
    -0.00113,
    -0.00136,
    -0.00492,
    -0.00204,
    -0.00029,
    0.02587,
    0.0025,
    0.00176,
    -0.00393,
    -0.00156,
    0.00214,
    0.00598,
    0.02563
   ],
   [
    0.01423,
    0.0073,
    -0.00014,
    0.00144,
    0.00726,
    0.00627,
    0.00101,
    0.01438,
    0.00409,
    -0.00246,
    -0.00121,
    0.0054,
    0.00386,
    0.00108
   ],
   [
    -0.00211,
    -0.00481,
    0.00452,
    -0.02277,
    0.00204,
    -0.00174,
    0.01392,
    -0.00169,
    -0.00134,
    0.001,
    -0.02055,
    0.003,
    -0.00173,
    0.0109
   ],
   [
    -0.00424,
    0.00491,
    0.00808,
    -0.01102,
    0.00947,
    0.00292,
    -0.0064,
    -0.00451,
    0.00186,
    0.00686,
    -0.00592,
    0.01035,
    0.0031,
    -0.00499
   ],
   [
    0.02097,
    0.02453,
    0.00451,
    0.01118,
    -0.00017,
    -0.00269,
    -0.00161,
    0.0185,
    0.02472,
    0.00618,
    0.01258,
    0.00184,
    -0.00223,
    -0.00085
   ],
   [
    0.02567,
    0.0234,
    0.00108,
    -0.01185,
    0.011,
    0.0076,
    0.00889,
    0.01953,
    0.02506,
    0.00208,
    -0.01272,
    0.00615,
    0.00507,
    0.00604
   ],
   [
    0.00294,
    0.00062,
    0.0022,
    -0.01066,
    -0.00211,
    0.00499,
    -0.00482,
    0.00237,
    -0.00488,
    0.00305,
    -0.0158,
    -0.00503,
    0.00291,
    -0.0073
   ],
   [
    -0.01376,
    0.00776,
    0.01165,
    0.00695,
    -0.00264,
    0.00447,
    -0.00369,
    -0.00826,
    0.00819,
    0.009,
    0.00679,
    0.00103,
    0.00598,
    -0.00166
   ],
   [
    0.00192,
    -0.01787,
    0.00035,
    0.01256,
    -0.01046,
    -0.01363,
    -0.00042,
    -0.00035,
    -0.01241,
    0.0018,
    0.0097,
    -0.01098,
    -0.0116,
    -0.00212
   ],
   [
    -0.00191,
    -0.01641,
    0.00695,
    -0.00235,
    -0.01291,
    0.00314,
    0.00915,
    -0.0009,
    -0.01322,
    0.00335,
    -0.00041,
    -0.00987,
    0.00294,
    0.00763
   ],
   [
    -0.00425,
    -0.02107,
    0.00028,
    0.00687,
    -7e-05,
    0.0028,
    -0.01115,
    -0.00462,
    -0.01829,
    -0.00205,
    0.00313,
    -0.00318,
    -0.00015,
    -0.01017
   ],
   [
    0.00917,
    -0.0064,
    -0.02622,
    -0.00478,
    0.01489,
    -0.01601,
    -0.01919,
    0.005,
    -0.01086,
    -0.01986,
    -0.00531,
    0.01107,
    -0.01292,
    -0.01513
   ],
   [
    -0.01352,
    -0.00572,
    0.00107,
    -0.0045,
    -0.00379,
    0.00494,
    -0.00575,
    -0.00704,
    -0.00503,
    0.00107,
    -0.00357,
    -0.00156,
    0.00156,
    -0.00756
   ],
   [
    -0.00946,
    0.00251,
    -0.00799,
    -0.01172,
    -0.00583,
    0.0162,
    -0.01246,
    -0.00798,
    -0.00389,
    -0.00649,
    -0.0131,
    -0.00782,
    0.01089,
    -0.01011
   ],
   [
    -0.00537,
    0.00326,
    -0.00912,
    -0.00308,
    0.0002,
    0.00581,
    -0.00956,
    -0.00586,
    0.0012,
    -0.01013,
    -0.00226,
    1e-05,
    0.00662,
    -0.00834
   ],
   [
    0.00028,
    0.00767,
    0.00347,
    0.01012,
    0.00334,
    0.01048,
    -0.0075,
    0.00134,
    0.00849,
    0.00289,
    0.0109,
    0.00359,
    0.01087,
    -0.00546
   ],
   [
    0.00189,
    -0.00479,
    -0.01379,
    0.01366,
    0.00072,
    -0.00524,
    0.00344,
    -0.00138,
    -0.00135,
    -0.01151,
    0.01211,
    0.0015,
    -0.00513,
    0.00306
   ],
   [
    0.01881,
    -0.00165,
    -0.00751,
    0.00153,
    0.00139,
    -0.00251,
    0.01388,
    0.01562,
    0.0024,
    -0.00371,
    0.00308,
    0.00288,
    0.00224,
    0.01418
   ],
   [
    -0.00541,
    0.01496,
    0.00742,
    0.0044,
    0.00145,
    -0.00505,
    -0.00545,
    -0.00265,
    0.00795,
    0.00478,
    0.00524,
    0.00228,
    -0.00304,
    -0.00284
   ],
   [
    0.00208,
    -0.00913,
    0.00253,
    0.0054,
    -0.00123,
    0.00111,
    -0.00934,
    0.0021,
    -0.00743,
    -0.00053,
    0.00482,
    -0.0012,
    0.0022,
    -0.00589
   ],
   [
    0.00126,
    0.00793,
    -0.00841,
    0.00171,
    0.00524,
    -0.006,
    -0.00537,
    0.00214,
    0.00228,
    -0.00476,
    0.00141,
    0.00265,
    -0.00875,
    -0.00642
   ],
   [
    -0.00048,
    0.00739,
    0.00133,
    -0.0092,
    0.01529,
    -0.00365,
    0.01567,
    -0.00124,
    0.01259,
    0.00324,
    -0.00662,
    0.01385,
    -0.00296,
    0.01178
   ],
   [
    -0.01278,
    0.00235,
    -0.0025,
    -0.00947,
    -0.00071,
    0.00425,
    0.00367,
    -0.01328,
    0.00476,
    -0.00599,
    -0.0074,
    0.00037,
    0.00258,
    0.00324
   ],
   [
    -0.00566,
    -0.00123,
    -0.01045,
    0.00531,
    -0.0092,
    0.00425,
    -0.00091,
    -0.00395,
    -0.00255,
    -0.00355,
    0.00509,
    -0.00911,
    0.00313,
    -0.00057
   ],
   [
    0.00127,
    -0.00295,
    0.01108,
    0.01001,
    -0.0031,
    -0.00357,
    -0.00479,
    0.00313,
    -0.00579,
    0.00044,
    0.00678,
    -0.00282,
    -0.00307,
    -0.00639
   ],
   [
    -0.0037,
    0.00492,
    0.01018,
    -0.00278,
    0.00961,
    -0.00454,
    -0.00014,
    -0.00456,
    0.00081,
    0.002,
    -0.00247,
    0.00865,
    -0.00352,
    -0.00092
   ],
   [
    0.0043,
    0.00854,
    0.00145,
    0.02344,
    -0.00778,
    0.00332,
    -0.01415,
    -0.00033,
    0.00617,
    -0.00038,
    0.0204,
    -0.00801,
    -0.00222,
    -0.01296
   ],
   [
    -0.00364,
    0.00031,
    0.02623,
    -0.01601,
    -0.01284,
    0.01804,
    0.0146,
    -0.00203,
    0.00074,
    0.01877,
    -0.01393,
    -0.00919,
    0.01488,
    0.00997
   ],
   [
    0.00659,
    0.00243,
    -0.01049,
    -0.00638,
    -0.00632,
    0.00936,
    0.01134,
    0.00318,
    0.00231,
    -0.00727,
    -0.00618,
    -0.00365,
    0.00606,
    0.00742
   ],
   [
    0.01457,
    -0.02443,
    -0.00311,
    0.00786,
    -0.00629,
    -0.00049,
    0.01482,
    0.0098,
    -0.01464,
    -0.00388,
    0.00867,
    -0.00405,
    0.00231,
    0.01469
   ],
   [
    0.01496,
    0.00984,
    -0.01705,
    0.0029,
    0.00227,
    -0.00878,
    0.00943,
    0.00803,
    0.00641,
    -0.00937,
    0.00274,
    0.00196,
    -0.00781,
    0.00719
   ],
   [
    -0.03245,
    0.0027,
    -0.01159,
    -0.00997,
    0.01762,
    -0.01481,
    0.00363,
    -0.02954,
    -0.00146,
    -0.01589,
    -0.01016,
    0.01497,
    -0.0154,
    0.00086
   ],
   [
    0.0054,
    0.00834,
    0.00432,
    0.00525,
    -0.00239,
    -0.0064,
    0.01418,
    0.0017,
    0.00777,
    0.00502,
    0.00466,
    -0.00232,
    -0.0057,
    0.01139
   ],
   [
    -0.01943,
    0.0005,
    -0.00235,
    0.00437,
    -0.0053,
    0.00051,
    -0.00451,
    -0.0167,
    -0.00304,
    -0.0037,
    0.00313,
    -0.00725,
    -0.00202,
    -0.00416
   ],
   [
    0.00963,
    0.01003,
    -0.00087,
    -0.00197,
    -0.00209,
    0.01685,
    0.00849,
    0.01103,
    0.01053,
    0.00343,
    -0.00213,
    -0.00133,
    0.01614,
    0.00869
   ],
   [
    -0.01386,
    -0.01236,
    -0.00348,
    -0.00171,
    -0.01094,
    0.00197,
    -0.00507,
    -0.01115,
    -0.00791,
    -0.00113,
    -0.00153,
    -0.00674,
    0.00089,
    -0.00511
   ],
   [
    0.00396,
    0.00286,
    0.0011,
    0.00301,
    9e-05,
    0.00364,
    -0.01247,
    0.00276,
    0.00172,
    0.00352,
    0.00204,
    0.00044,
    0.00217,
    -0.01152
   ],
   [
    -0.00828,
    -0.01105,
    0.00499,
    -0.00604,
    -0.00678,
    0.00613,
    0.00133,
    -0.00549,
    -0.00768,
    0.00165,
    -0.00641,
    -0.00516,
    0.00919,
    0.00449
   ],
   [
    -0.00926,
    -0.01368,
    -0.00642,
    -0.00608,
    0.00366,
    0.00215,
    -0.0023,
    -0.00856,
    -0.01463,
    -0.00556,
    -0.00546,
    0.00458,
    0.00487,
    8e-05
   ],
   [
    0.01175,
    -0.01239,
    -0.00539,
    0.00275,
    -0.02268,
    0.00502,
    0.02554,
    0.01053,
    -0.00355,
    -0.00049,
    0.00302,
    -0.01887,
    0.0061,
    0.0217
   ],
   [
    -0.02666,
    0.00129,
    -0.00642,
    -0.00646,
    0.00608,
    0.01532,
    0.00609,
    -0.02192,
    -0.00208,
    -0.00207,
    -0.00738,
    0.00637,
    0.01792,
    0.0073
   ],
   [
    0.02205,
    0.01755,
    -0.00844,
    0.01171,
    -0.0057,
    0.00411,
    -0.01483,
    0.02134,
    0.01243,
    -0.01093,
    0.01056,
    -0.00428,
    0.00727,
    -0.01043
   ],
   [
    0.01031,
    0.00046,
    -0.00351,
    -0.02227,
    -0.01049,
    0.03094,
    0.02065,
    0.01196,
    0.0047,
    0.00033,
    -0.01987,
    -0.00568,
    0.02744,
    0.01703
   ],
   [
    0.0099,
    -0.0052,
    -0.00523,
    0.00955,
    -0.00915,
    -0.01015,
    -0.01551,
    0.00719,
    -0.0098,
    -0.0052,
    0.00724,
    -0.01215,
    -0.01145,
    -0.01342
   ],
   [
    0.02073,
    -0.01656,
    -0.01466,
    0.01905,
    -0.01295,
    -0.00518,
    -0.01822,
    0.01698,
    -0.01946,
    -0.01155,
    0.01465,
    -0.01582,
    -0.00677,
    -0.01715
   ],
   [
    -0.0069,
    -0.04812,
    0.01124,
    0.0038,
    -0.02892,
    -0.00263,
    0.0126,
    -0.00492,
    -0.03607,
    0.00738,
    -0.00041,
    -0.02671,
    -0.00299,
    0.00732
   ],
   [
    0.0079,
    0.0028,
    0.00835,
    0.00449,
    -0.00723,
    -0.00909,
    -0.00207,
    0.00564,
    -0.0032,
    -0.00277,
    0.00355,
    -0.00604,
    -0.00894,
    -0.00324
   ],
   [
    0.00384,
    0.00038,
    -0.00871,
    -0.00549,
    0.00061,
    -0.00619,
    -0.00181,
    0.00216,
    -0.00144,
    -0.00917,
    -0.00288,
    0.00131,
    -0.00535,
    -0.00169
   ],
   [
    0.02035,
    0.00426,
    -0.00104,
    0.00551,
    -0.00406,
    -0.00869,
    0.00556,
    0.01888,
    0.0006,
    -0.00049,
    0.00522,
    -0.00497,
    -0.00778,
    0.00483
   ],
   [
    0.00084,
    -0.01561,
    -0.0008,
    0.00624,
    -0.0097,
    0.01872,
    0.0065,
    -0.00084,
    -0.00832,
    -0.00026,
    0.00696,
    -0.0079,
    0.02064,
    0.00681
   ],
   [
    -0.00667,
    -0.00493,
    -0.01488,
    0.0164,
    -0.00292,
    -0.01573,
    -0.00686,
    -0.00393,
    -0.00682,
    -0.00702,
    0.01431,
    -0.00229,
    -0.01114,
    -0.00449
   ],
   [
    -0.00025,
    0.01453,
    0.01101,
    -0.00111,
    -0.00173,
    0.00504,
    -0.00164,
    0.00217,
    0.01339,
    0.01294,
    -0.00118,
    -0.00281,
    0.00508,
    -0.00216
   ],
   [
    -0.01325,
    -0.01871,
    0.00347,
    0.00507,
    -0.00806,
    0.00671,
    0.00755,
    -0.00893,
    -0.01382,
    -0.00054,
    0.00492,
    -0.00555,
    0.00661,
    0.00665
   ],
   [
    0.00355,
    0.00784,
    -0.00128,
    0.01055,
    0.00415,
    -0.00327,
    0.00229,
    0.00183,
    0.00672,
    -0.00308,
    0.00932,
    0.00286,
    -0.0041,
    0.00259
   ],
   [
    -0.01485,
    -0.00868,
    0.00228,
    0.00157,
    -0.00379,
    0.00267,
    0.00606,
    -0.01132,
    -0.00655,
    0.00141,
    0.0031,
    -0.0017,
    0.00512,
    0.00601
   ],
   [
    -0.01761,
    0.00092,
    -0.00256,
    0.01097,
    -0.00724,
    -0.01221,
    -0.01092,
    -0.01675,
    -0.00237,
    -0.00496,
    0.00653,
    -0.00793,
    -0.01183,
    -0.00979
   ],
   [
    0.02532,
    -0.00667,
    0.00762,
    -0.00202,
    -0.00386,
    0.01255,
    0.01333,
    0.02175,
    -0.00024,
    0.00704,
    0.00184,
    -0.00101,
    0.01232,
    0.01363
   ],
   [
    -0.00384,
    0.00391,
    -0.00589,
    -0.00871,
    0.00736,
    -0.00176,
    0.01307,
    -0.00643,
    0.00469,
    -0.00267,
    -0.0102,
    0.00494,
    -0.00088,
    0.00986
   ],
   [
    -0.02385,
    -0.00465,
    -0.00328,
    -0.00042,
    -0.00578,
    0.00375,
    -0.00192,
    -0.01824,
    -0.00353,
    -0.00334,
    -0.00065,
    -0.00744,
    0.00237,
    -0.00393
   ],
   [
    -0.00297,
    -0.01023,
    -0.00712,
    0.0092,
    -0.00682,
    -0.00433,
    -0.00019,
    -0.00524,
    -0.00553,
    -0.00706,
    0.00787,
    -0.00812,
    -0.00646,
    -0.00337
   ],
   [
    0.01285,
    0.00403,
    -0.004,
    0.00963,
    -0.01338,
    -0.00106,
    0.01215,
    0.0119,
    0.00488,
    -0.00291,
    0.01072,
    -0.00919,
    0.0013,
    0.01101
   ],
   [
    -0.00923,
    -0.00935,
    0.00218,
    -0.00378,
    0.00403,
    -0.00078,
    -0.00419,
    -0.00795,
    -0.00663,
    -0.00298,
    -0.00301,
    0.00525,
    0.00084,
    -0.00203
   ],
   [
    0.00036,
    0.00585,
    -0.00357,
    -0.00469,
    -1e-05,
    -0.00083,
    -0.00097,
    -0.00219,
    0.00292,
    -0.00054,
    -0.00376,
    0.00062,
    -0.00095,
    -0.00148
   ],
   [
    -0.00884,
    0.00272,
    -0.01281,
    0.01181,
    -0.01059,
    0.00433,
    0.01401,
    -0.00222,
    0.00823,
    -0.00637,
    0.01302,
    -0.00502,
    0.00704,
    0.01512
   ],
   [
    0.01977,
    0.00898,
    -0.0005,
    -0.00075,
    0.0098,
    0.00578,
    0.00376,
    0.01557,
    0.00578,
    0.00192,
    -0.00037,
    0.00854,
    0.00686,
    0.00437
   ],
   [
    -0.00258,
    -0.00553,
    -0.00544,
    0.00869,
    -0.00985,
    0.01097,
    -0.00093,
    -0.00295,
    -0.00544,
    -0.00628,
    0.00692,
    -0.0124,
    0.00532,
    -0.00324
   ],
   [
    -0.01368,
    -0.02205,
    -0.00548,
    0.00721,
    -0.00615,
    0.00508,
    -0.00916,
    -0.01243,
    -0.01689,
    -0.00384,
    0.01171,
    -0.00079,
    0.00569,
    -0.00637
   ],
   [
    0.02972,
    0.00611,
    -0.01469,
    0.00122,
    -0.00113,
    -0.00641,
    0.00515,
    0.024,
    0.00421,
    -0.01377,
    0.00047,
    -0.0041,
    -0.00666,
    0.00522
   ],
   [
    0.0063,
    -0.00347,
    0.00051,
    -0.02568,
    -0.00016,
    0.01122,
    0.01582,
    0.00586,
    -0.0005,
    -0.00015,
    -0.0209,
    -0.00032,
    0.00833,
    0.01317
   ],
   [
    0.00507,
    -0.02218,
    -0.00217,
    -0.00048,
    -0.01541,
    0.0149,
    0.0048,
    0.0026,
    -0.01207,
    -0.00185,
    3e-05,
    -0.01215,
    0.01683,
    0.00566
   ],
   [
    -0.00714,
    -0.02,
    0.01699,
    0.00992,
    -0.01557,
    0.00429,
    -0.00207,
    -0.00914,
    -0.01351,
    0.01286,
    0.00964,
    -0.01591,
    0.0028,
    -0.00153
   ],
   [
    0.01696,
    0.03153,
    0.00367,
    -0.01846,
    0.01016,
    0.00357,
    0.0088,
    0.01158,
    0.02419,
    0.00194,
    -0.01638,
    0.00952,
    0.0047,
    0.00955
   ],
   [
    0.00379,
    -0.00377,
    0.0006,
    -0.00917,
    -0.00061,
    0.00319,
    0.00319,
    -7e-05,
    -0.00239,
    -0.0021,
    -0.0105,
    -0.00085,
    0.00026,
    0.00142
   ],
   [
    0.00939,
    0.01204,
    0.00901,
    -0.00396,
    0.00236,
    -0.00148,
    0.01508,
    0.01047,
    0.0111,
    0.00787,
    -0.00299,
    0.00345,
    0.00146,
    0.01438
   ],
   [
    -0.01122,
    -0.02279,
    0.00439,
    -0.01821,
    -0.00147,
    0.00908,
    0.00178,
    -0.01229,
    -0.00996,
    0.00653,
    -0.01497,
    -0.00094,
    0.00815,
    0.00066
   ],
   [
    -0.01136,
    -0.00154,
    -0.00761,
    0.00646,
    -0.00292,
    -0.00533,
    -0.00392,
    -0.01287,
    -0.00675,
    -0.00932,
    0.00655,
    -0.00072,
    -0.00439,
    -0.00355
   ],
   [
    -0.00468,
    -0.01296,
    -0.00697,
    -0.00583,
    0.00043,
    -0.00381,
    0.01434,
    -0.00474,
    -0.00802,
    -0.00204,
    -0.00249,
    0.00247,
    -0.00219,
    0.01081
   ],
   [
    -0.00594,
    0.00262,
    0.01299,
    -0.01409,
    0.00895,
    0.01148,
    -0.00525,
    -0.00475,
    0.00303,
    0.00852,
    -0.01364,
    0.00922,
    0.01366,
    -0.00171
   ],
   [
    0.0025,
    -0.01113,
    0.00825,
    -0.00149,
    -0.01192,
    0.0102,
    0.0149,
    0.00377,
    -0.00589,
    0.00423,
    -0.00031,
    -0.0083,
    0.00786,
    0.01214
   ],
   [
    0.01525,
    0.00313,
    -1e-05,
    0.00329,
    -0.00453,
    0.0056,
    -0.00178,
    0.0126,
    0.00045,
    0.00117,
    -0.0005,
    -0.00581,
    0.00393,
    -0.00111
   ],
   [
    0.00504,
    0.02062,
    -0.00952,
    0.0034,
    0.00106,
    0.00914,
    -0.01402,
    0.00525,
    0.0166,
    -0.00095,
    0.00447,
    0.00149,
    0.00773,
    -0.01088
   ],
   [
    0.00147,
    0.01724,
    0.00628,
    -0.01161,
    0.01094,
    -0.01346,
    -0.00035,
    0.0009,
    0.01478,
    -0.00048,
    -0.0115,
    0.00847,
    -0.01502,
    -0.00152
   ],
   [
    -0.00631,
    -0.01436,
    0.00034,
    -0.00927,
    0.0033,
    0.00193,
    0.00278,
    -0.00955,
    -0.00974,
    -0.00042,
    -0.00553,
    0.00358,
    0.00095,
    0.00237
   ],
   [
    0.00447,
    -0.0017,
    0.01642,
    -0.00502,
    0.00701,
    0.00802,
    -0.00179,
    0.00404,
    0.0044,
    0.01497,
    -0.00357,
    0.00614,
    0.00824,
    -0.00056
   ],
   [
    0.03411,
    0.00578,
    0.00596,
    0.00972,
    0.00613,
    -0.00234,
    -0.00485,
    0.03132,
    0.00589,
    0.00922,
    0.01059,
    0.00687,
    0.0036,
    -0.00102
   ],
   [
    0.00293,
    0.01737,
    0.01394,
    -0.00081,
    -0.00543,
    -0.01251,
    -0.00381,
    0.00114,
    0.0124,
    0.00972,
    0.00274,
    -0.00325,
    -0.01037,
    -0.00139
   ],
   [
    0.00598,
    -0.01019,
    0.0005,
    -0.00216,
    0.00215,
    -0.00104,
    -0.00126,
    0.00451,
    -0.01115,
    -0.00068,
    -0.0029,
    0.00236,
    -2e-05,
    -0.00065
   ],
   [
    0.00271,
    0.00546,
    -0.01122,
    0.01482,
    -0.00062,
    -0.011,
    -0.00441,
    0.00587,
    -0.00059,
    -0.01115,
    0.01039,
    -0.00122,
    -0.01133,
    -0.00541
   ],
   [
    0.00337,
    0.01673,
    0.01385,
    -0.00453,
    0.01559,
    -0.00801,
    -0.01361,
    0.00174,
    0.01293,
    0.0065,
    -0.0038,
    0.01342,
    -0.00754,
    -0.0112
   ],
   [
    -0.00561,
    0.00257,
    -0.00083,
    0.00239,
    -0.00155,
    0.00723,
    -0.00462,
    -0.00509,
    -0.00022,
    0.00016,
    0.0005,
    -0.0011,
    0.00938,
    0.00085
   ],
   [
    -0.0007,
    -0.00522,
    -0.01638,
    -0.00183,
    -0.01045,
    0.00159,
    0.00493,
    -0.00074,
    -0.00475,
    -0.01096,
    -0.00368,
    -0.01078,
    -0.00324,
    0.00046
   ],
   [
    0.00159,
    0.00053,
    -0.00146,
    -0.00453,
    0.00169,
    0.01042,
    0.01029,
    0.00304,
    -0.00314,
    0.00152,
    -0.00492,
    0.00102,
    0.01015,
    0.00933
   ],
   [
    -0.01449,
    0.02193,
    -0.00605,
    0.01838,
    -0.00093,
    0.00772,
    -0.00374,
    -0.00968,
    0.01507,
    -0.003,
    0.0155,
    -0.00021,
    0.00633,
    -0.00249
   ],
   [
    0.00708,
    -0.00985,
    0.00189,
    0.01017,
    -0.01176,
    0.01333,
    -0.01029,
    0.00776,
    -0.0132,
    0.00042,
    0.0058,
    -0.00923,
    0.01129,
    -0.0076
   ],
   [
    0.00852,
    0.02274,
    0.00324,
    -0.00599,
    0.00311,
    0.00394,
    0.01552,
    0.00441,
    0.01905,
    0.00489,
    -0.00458,
    0.00286,
    0.00336,
    0.01368
   ],
   [
    -0.00061,
    -0.00932,
    -0.00109,
    -0.004,
    -0.00734,
    0.00725,
    -0.00294,
    -0.00134,
    -0.01033,
    -0.00456,
    -0.00538,
    -0.00665,
    0.00494,
    -0.00263
   ],
   [
    -0.004,
    0.01169,
    0.00476,
    -0.01207,
    -0.00398,
    -0.00486,
    0.00871,
    -0.00426,
    0.01109,
    0.00304,
    -0.01081,
    -0.00511,
    -0.00494,
    0.00804
   ],
   [
    0.00282,
    -0.00902,
    -0.01152,
    -0.00972,
    0.0063,
    -0.01215,
    -0.00778,
    0.0017,
    -0.01063,
    -0.00854,
    -0.00875,
    0.00428,
    -0.01074,
    -0.00726
   ],
   [
    -0.01418,
    0.01908,
    -0.00823,
    -0.01838,
    0.01165,
    0.00541,
    0.00739,
    -0.01015,
    0.01681,
    -0.00453,
    -0.01612,
    0.01122,
    0.00767,
    0.00787
   ],
   [
    -0.01076,
    -0.0096,
    0.00155,
    -0.00063,
    -0.00224,
    0.00119,
    0.00109,
    -0.00929,
    -0.00903,
    0.00187,
    -0.00191,
    -0.00186,
    0.00193,
    0.00157
   ],
   [
    -0.00331,
    0.00476,
    -0.00232,
    0.00947,
    -0.00198,
    -0.0055,
    0.0064,
    -0.00282,
    0.00503,
    -0.00038,
    0.01082,
    -0.00071,
    -0.00434,
    0.00532
   ],
   [
    -0.00015,
    -0.00068,
    0.00548,
    0.00892,
    0.01374,
    -0.01551,
    -0.01113,
    -0.0002,
    -0.00392,
    0.00064,
    0.00654,
    0.01051,
    -0.01523,
    -0.01021
   ],
   [
    0.00161,
    0.02986,
    -0.00752,
    0.00216,
    0.00019,
    0.02147,
    0.01165,
    0.00451,
    0.02864,
    0.00568,
    0.00048,
    0.00212,
    0.02347,
    0.01326
   ],
   [
    -0.00305,
    -0.01103,
    0.0107,
    0.00889,
    -0.01195,
    -0.00939,
    -0.00297,
    0.00331,
    -0.00499,
    0.01147,
    0.00928,
    -0.01247,
    -0.01077,
    -0.00325
   ],
   [
    -0.01544,
    -0.00192,
    -0.00333,
    0.00808,
    -0.00801,
    -0.00235,
    -0.01017,
    -0.01123,
    -0.00418,
    -0.00215,
    0.00701,
    -0.00829,
    -0.00529,
    -0.01001
   ],
   [
    0.01602,
    0.00142,
    0.00887,
    0.01869,
    -0.01774,
    0.0131,
    -0.01374,
    0.01313,
    0.0016,
    0.00696,
    0.01831,
    -0.01522,
    0.01181,
    -0.01071
   ],
   [
    -0.01335,
    -0.02211,
    0.00097,
    0.00293,
    -0.00694,
    -0.00141,
    0.00923,
    -0.01309,
    -0.02186,
    0.00029,
    0.0022,
    -0.00584,
    5e-05,
    0.00883
   ],
   [
    -0.00108,
    -0.00448,
    -0.01055,
    -0.00468,
    -0.0082,
    -0.00303,
    -0.00566,
    -0.00269,
    -0.00458,
    -0.01025,
    -0.00133,
    -0.00517,
    -0.00248,
    -0.00437
   ],
   [
    0.00716,
    0.00736,
    0.00714,
    -0.01597,
    -0.00323,
    0.01664,
    0.00947,
    0.00702,
    0.00936,
    0.01035,
    -0.01334,
    -0.00104,
    0.01714,
    0.00917
   ],
   [
    -0.02073,
    0.00987,
    -0.00286,
    -0.00299,
    0.01545,
    0.00895,
    -0.00281,
    -0.01477,
    0.00778,
    0.00316,
    -0.00137,
    0.01456,
    0.00911,
    0.00032
   ],
   [
    -0.01502,
    -0.02577,
    -0.00753,
    0.00368,
    -0.00135,
    -0.00706,
    0.00066,
    -0.01544,
    -0.01871,
    -0.00202,
    0.0033,
    -0.00297,
    -0.00308,
    0.00319
   ],
   [
    0.01713,
    0.00769,
    -0.00112,
    -0.00651,
    -0.00904,
    0.00488,
    0.01931,
    0.01227,
    0.00917,
    -0.00038,
    -0.0075,
    -0.00855,
    0.00293,
    0.01339
   ],
   [
    0.01603,
    -0.01362,
    0.00022,
    -0.00339,
    -0.0031,
    0.00169,
    0.00222,
    0.00996,
    -0.0088,
    0.00134,
    -0.00161,
    -0.00243,
    0.00355,
    0.00262
   ],
   [
    0.0203,
    0.00214,
    0.00588,
    0.00916,
    -0.02395,
    0.00455,
    0.02005,
    0.02006,
    0.00854,
    0.00726,
    0.0104,
    -0.01316,
    0.00898,
    0.02054
   ],
   [
    0.00478,
    0.0064,
    0.01853,
    -0.00583,
    0.00248,
    -0.00013,
    -0.00573,
    0.00274,
    0.00344,
    0.01488,
    -0.00573,
    0.00259,
    -0.00072,
    -0.00404
   ],
   [
    -0.00654,
    0.01132,
    -0.00489,
    -0.00187,
    0.00082,
    0.00455,
    0.00067,
    -0.00285,
    0.01187,
    0.00158,
    -0.00275,
    0.00062,
    0.0027,
    0.0002
   ],
   [
    0.01107,
    0.00371,
    -0.01007,
    0.01357,
    -0.00706,
    -0.00302,
    -0.00261,
    0.0121,
    0.00116,
    -0.00439,
    0.01335,
    -0.00542,
    -0.00425,
    -0.00225
   ],
   [
    0.00161,
    0.00379,
    -0.00358,
    0.00395,
    0.00267,
    0.01334,
    -0.0154,
    0.00151,
    0.00218,
    0.00057,
    0.00329,
    0.00402,
    0.01047,
    -0.01429
   ],
   [
    -0.0099,
    0.00298,
    0.00553,
    0.00904,
    -0.00645,
    -0.01462,
    -0.00559,
    -0.0115,
    -0.00545,
    0.0005,
    0.00747,
    -0.00528,
    -0.01339,
    -0.0044
   ],
   [
    0.01125,
    0.00122,
    0.00293,
    0.0102,
    -0.00366,
    -0.00368,
    -0.00173,
    0.00826,
    0.00761,
    0.00498,
    0.00969,
    -0.00194,
    0.00013,
    -0.00214
   ],
   [
    -0.00517,
    -0.02942,
    -0.00627,
    -0.00451,
    -0.00423,
    -0.00393,
    -0.00795,
    -0.00744,
    -0.0295,
    -0.01141,
    -0.00274,
    -0.00408,
    -0.00702,
    -0.00877
   ],
   [
    0.00289,
    0.00327,
    0.00791,
    -0.01376,
    -0.00973,
    0.01059,
    0.01021,
    2e-05,
    0.00593,
    0.00933,
    -0.0117,
    -0.00519,
    0.01398,
    0.01267
   ],
   [
    0.01203,
    0.02748,
    -0.00239,
    0.01586,
    -0.00591,
    0.01369,
    -0.03131,
    0.01648,
    0.01852,
    0.00039,
    0.01167,
    -0.00638,
    0.01315,
    -0.02546
   ],
   [
    -0.01703,
    0.00963,
    0.00845,
    0.03042,
    -0.00632,
    -0.01662,
    0.01866,
    -0.01796,
    0.00946,
    0.00668,
    0.03075,
    -0.00484,
    -0.01616,
    0.0182
   ],
   [
    0.00927,
    -0.02788,
    -0.00239,
    -0.01113,
    -0.00219,
    0.00439,
    0.01973,
    0.00848,
    -0.01781,
    -0.00767,
    -0.01162,
    -0.00265,
    0.00064,
    0.01463
   ],
   [
    -0.00941,
    0.00981,
    0.0092,
    -0.00846,
    -0.00968,
    0.0177,
    0.00978,
    -0.00578,
    0.00825,
    0.00783,
    -0.00822,
    -0.00709,
    0.01681,
    0.00804
   ],
   [
    -0.00105,
    -0.00611,
    -0.01374,
    0.00215,
    -0.00261,
    0.01476,
    0.01241,
    -0.00261,
    -0.0113,
    -0.00484,
    -0.00042,
    -0.00636,
    0.0128,
    0.01021
   ],
   [
    0.00661,
    -0.00969,
    0.0005,
    -0.00691,
    0.00355,
    0.00045,
    0.00579,
    0.00197,
    -0.00969,
    0.00183,
    -0.00469,
    0.00333,
    -0.00086,
    0.0041
   ],
   [
    0.01518,
    0.00156,
    0.00683,
    -0.00029,
    -0.00016,
    -0.00015,
    0.00486,
    0.0123,
    -0.00086,
    0.00063,
    0.00215,
    0.00286,
    0.00166,
    0.00462
   ],
   [
    -0.00971,
    -0.03059,
    -0.00272,
    -0.00514,
    -0.00401,
    0.0068,
    0.00457,
    -0.01125,
    -0.02635,
    -0.00328,
    -0.00539,
    -0.00509,
    0.00496,
    0.00216
   ],
   [
    0.00766,
    0.00867,
    0.00082,
    -0.00555,
    0.01678,
    -0.00445,
    0.00507,
    0.00915,
    0.00654,
    -0.00091,
    -0.00309,
    0.01495,
    -0.00305,
    0.00464
   ],
   [
    0.00296,
    -0.01777,
    0.00326,
    -0.00145,
    -0.00108,
    -0.00249,
    0.00022,
    -0.00015,
    -0.01389,
    6e-05,
    0.00021,
    -0.00079,
    -0.00305,
    -0.00154
   ],
   [
    0.01558,
    -0.0052,
    0.00612,
    -0.00233,
    -0.00602,
    0.00053,
    0.00678,
    0.016,
    -0.00279,
    0.0102,
    -0.00233,
    -0.00538,
    0.00287,
    0.00713
   ],
   [
    0.00293,
    -0.01998,
    0.00036,
    0.01839,
    -0.02213,
    0.00703,
    -0.00999,
    0.00656,
    -0.0131,
    -0.0005,
    0.01263,
    -0.02132,
    0.00356,
    -0.00817
   ],
   [
    -0.00259,
    -0.01128,
    0.01199,
    -0.00471,
    -0.00216,
    0.00799,
    -0.01602,
    0.00081,
    -0.0133,
    0.00673,
    -0.00396,
    -0.00148,
    0.0082,
    -0.01164
   ],
   [
    0.01294,
    0.01825,
    -0.00134,
    -0.01132,
    0.00095,
    0.01074,
    -0.0053,
    0.01161,
    0.01393,
    -0.00855,
    -0.01011,
    0.00152,
    0.00863,
    -0.00406
   ],
   [
    -0.02126,
    -0.00892,
    0.00426,
    8e-05,
    -0.00308,
    -0.00192,
    -0.0026,
    -0.01584,
    -0.00703,
    0.00099,
    -0.00254,
    -0.00586,
    -0.00383,
    -0.00363
   ],
   [
    -0.02076,
    0.0027,
    0.00521,
    0.00513,
    0.0073,
    -0.021,
    -0.00344,
    -0.01814,
    -0.0014,
    0.00024,
    0.00345,
    0.00571,
    -0.01823,
    -0.0045
   ],
   [
    -0.01423,
    -0.01421,
    0.01204,
    0.00155,
    -0.0047,
    0.00533,
    -0.00774,
    -0.01144,
    -0.01393,
    0.00606,
    0.00123,
    -0.00342,
    0.00306,
    -0.00664
   ],
   [
    -0.00393,
    -0.01174,
    0.00524,
    0.01786,
    -0.00319,
    -0.00913,
    -0.00033,
    -0.00423,
    -0.01029,
    0.00131,
    0.01763,
    -0.00089,
    -0.007,
    1e-05
   ],
   [
    -0.00979,
    0.006,
    -0.00347,
    -0.00152,
    0.01127,
    -0.0091,
    -0.00962,
    -0.00778,
    0.00156,
    -0.00162,
    -0.00356,
    0.00888,
    -0.00937,
    -0.00801
   ],
   [
    -0.01292,
    -0.00325,
    -0.00129,
    0.00269,
    -0.01876,
    0.0119,
    0.00618,
    -0.01182,
    -0.00075,
    0.00098,
    0.00285,
    -0.01653,
    0.01209,
    0.00662
   ],
   [
    0.00236,
    0.01399,
    0.00815,
    0.00583,
    0.00143,
    -0.00964,
    0.00089,
    0.00439,
    0.00938,
    0.0108,
    0.00452,
    -0.00067,
    -0.01133,
    0.00021
   ],
   [
    -0.01146,
    0.00235,
    0.01711,
    -0.00064,
    -0.00117,
    -0.00031,
    -0.00494,
    -0.00818,
    -0.006,
    0.00863,
    -0.00285,
    -0.00357,
    -0.00096,
    -0.00481
   ],
   [
    -0.01058,
    0.01102,
    0.00415,
    0.00753,
    -0.00126,
    -0.00754,
    -0.00148,
    -0.00332,
    0.00576,
    0.00255,
    0.00576,
    -4e-05,
    -0.00395,
    0.00017
   ],
   [
    -0.00865,
    -0.00464,
    0.00235,
    0.00272,
    -0.00329,
    0.00295,
    -0.0082,
    -0.00553,
    -0.00327,
    0.00243,
    0.00111,
    -0.00451,
    0.00461,
    -0.00772
   ],
   [
    0.00136,
    0.00738,
    -0.00132,
    -0.00543,
    0.00067,
    -0.00268,
    -0.01086,
    0.00372,
    0.00125,
    -0.00198,
    -0.00733,
    -0.00257,
    -0.00217,
    -0.0083
   ],
   [
    0.00449,
    0.03332,
    -0.00764,
    0.00377,
    0.00316,
    0.0094,
    -0.00543,
    0.00509,
    0.02636,
    -0.00409,
    0.00239,
    0.00326,
    0.00844,
    -0.00492
   ],
   [
    -0.02139,
    -0.00481,
    -0.00804,
    -0.00932,
    0.00455,
    0.01124,
    -0.00346,
    -0.01772,
    -0.0037,
    -0.00353,
    -0.00572,
    0.00766,
    0.01226,
    -0.00199
   ],
   [
    -0.01467,
    -6e-05,
    -0.0128,
    -0.00963,
    -0.00541,
    0.01347,
    0.0204,
    -0.0112,
    -0.00143,
    -0.00909,
    -0.01062,
    -0.00467,
    0.01168,
    0.01584
   ],
   [
    -0.01322,
    0.01157,
    0.0014,
    -0.00532,
    0.00684,
    0.00757,
    0.00799,
    -0.01246,
    0.00866,
    0.00016,
    -0.00403,
    0.00905,
    0.01074,
    0.00914
   ],
   [
    -0.00126,
    0.02418,
    0.01292,
    -0.00598,
    0.00903,
    -0.00488,
    -0.00517,
    0.0009,
    0.01839,
    0.0061,
    -0.00964,
    0.00517,
    -0.00512,
    -0.00466
   ],
   [
    -0.00742,
    0.01103,
    -0.00245,
    0.00094,
    0.00204,
    -0.00392,
    0.00432,
    -0.00304,
    0.01315,
    0.00165,
    0.00358,
    0.00451,
    0.0012,
    0.00593
   ],
   [
    0.01477,
    -0.00152,
    0.00664,
    0.00494,
    -0.00012,
    -0.00345,
    0.00119,
    0.01461,
    0.00035,
    0.00628,
    0.00405,
    0.00036,
    0.00075,
    0.00354
   ],
   [
    0.00836,
    0.0075,
    -0.00134,
    -0.00632,
    0.0007,
    -0.00221,
    0.00079,
    0.00468,
    0.00768,
    -0.00052,
    -0.00329,
    0.00461,
    -0.00286,
    -0.00124
   ],
   [
    0.02079,
    0.01523,
    0.0026,
    0.00031,
    0.00393,
    0.01556,
    0.01247,
    0.01917,
    0.0167,
    0.00771,
    -0.00043,
    0.00243,
    0.0138,
    0.01291
   ],
   [
    0.02047,
    -0.01792,
    0.00939,
    0.00372,
    -0.01593,
    0.00202,
    -0.00548,
    0.01716,
    -0.01772,
    0.01283,
    0.00814,
    -0.01295,
    0.00454,
    -0.00233
   ],
   [
    0.00098,
    -0.01142,
    0.01087,
    0.00106,
    -0.01152,
    -0.00024,
    0.00925,
    0.0025,
    -0.00804,
    0.00583,
    -0.00183,
    -0.01063,
    0.00024,
    0.00619
   ],
   [
    -0.02476,
    -0.00599,
    0.01078,
    -0.00896,
    0.01106,
    -0.01496,
    -0.00424,
    -0.01979,
    -0.00649,
    0.00063,
    -0.00905,
    0.00698,
    -0.01464,
    -0.00489
   ],
   [
    -0.00645,
    -0.01111,
    0.00106,
    0.00132,
    0.008,
    5e-05,
    0.00317,
    -0.00659,
    -0.01059,
    0.00019,
    0.00059,
    0.00746,
    -0.00055,
    0.00159
   ],
   [
    -0.00281,
    -0.00396,
    -0.00108,
    0.00307,
    -0.01308,
    0.00252,
    0.00169,
    -0.00219,
    -0.0051,
    -0.00094,
    8e-05,
    -0.01193,
    0.0039,
    0.00299
   ],
   [
    -0.00986,
    0.00128,
    -0.00145,
    -0.00607,
    0.00701,
    0.00356,
    -0.00883,
    -0.00658,
    0.00014,
    -0.00195,
    -0.00571,
    0.00568,
    0.00442,
    -0.00716
   ],
   [
    -0.0024,
    0.01068,
    0.00101,
    -0.00262,
    0.00316,
    0.0059,
    -0.00022,
    -0.00162,
    0.01077,
    0.00504,
    -0.002,
    0.0009,
    0.00431,
    4e-05
   ],
   [
    -0.00334,
    -0.01657,
    0.00134,
    0.00135,
    0.00541,
    0.00398,
    -0.01404,
    -0.00208,
    -0.01587,
    -0.00251,
    0.00075,
    0.00489,
    0.0023,
    -0.01244
   ],
   [
    -0.01551,
    -0.02112,
    0.00763,
    -0.00916,
    -0.00441,
    0.00954,
    -0.00385,
    -0.01501,
    -0.02004,
    0.00314,
    -0.0081,
    -0.00291,
    0.00903,
    -0.00359
   ],
   [
    -0.00451,
    -0.00082,
    0.00932,
    0.00406,
    0.00203,
    -0.00211,
    0.00053,
    -0.0044,
    0.00047,
    0.00634,
    0.005,
    0.00235,
    0.00021,
    0.00331
   ],
   [
    0.02609,
    -0.0006,
    0.00221,
    0.00302,
    -0.00775,
    0.00965,
    -1e-05,
    0.02207,
    -0.00019,
    0.00218,
    0.00524,
    -0.00538,
    0.00962,
    0.00018
   ],
   [
    0.00472,
    0.00608,
    0.00108,
    -0.01485,
    0.00568,
    -0.00314,
    0.00639,
    0.00547,
    0.0067,
    0.00045,
    -0.0143,
    0.00405,
    -0.00356,
    0.00369
   ],
   [
    0.00821,
    -0.00977,
    -0.00495,
    0.00534,
    0.00152,
    0.01272,
    -0.00116,
    0.00941,
    -0.00673,
    0.00225,
    0.00393,
    0.00135,
    0.01606,
    0.00287
   ],
   [
    0.00519,
    -0.01211,
    -0.00337,
    0.00334,
    0.00916,
    -0.00566,
    0.0,
    0.0032,
    -0.01012,
    -0.00127,
    0.00548,
    0.00849,
    -0.00344,
    0.00134
   ],
   [
    0.01114,
    0.00094,
    -0.00184,
    -0.01198,
    -0.00149,
    -0.00041,
    0.01312,
    0.00716,
    0.00553,
    0.00058,
    -0.01046,
    -0.00037,
    0.00056,
    0.01015
   ],
   [
    -0.02568,
    0.00288,
    -0.00169,
    -0.01741,
    0.00973,
    -0.00566,
    0.00666,
    -0.02304,
    -0.0009,
    -0.00413,
    -0.01526,
    0.00875,
    -0.00614,
    0.0041
   ],
   [
    0.00749,
    -0.01028,
    -0.00933,
    0.00742,
    -0.00876,
    0.00143,
    -0.00347,
    0.00331,
    -0.00763,
    -0.00547,
    0.00674,
    -0.00975,
    0.0011,
    -0.004
   ],
   [
    0.00167,
    0.00425,
    -0.00145,
    -0.01249,
    0.00443,
    0.0115,
    -0.01687,
    -0.00291,
    0.00227,
    0.00173,
    -0.00743,
    0.00539,
    0.01137,
    -0.01421
   ],
   [
    -0.00414,
    -0.01167,
    -0.01764,
    0.00412,
    -0.00514,
    -0.00341,
    0.00858,
    -0.00334,
    -0.00985,
    -0.01328,
    0.00364,
    -0.00438,
    -0.00195,
    0.0072
   ],
   [
    -0.0038,
    0.00421,
    -0.00115,
    -0.00622,
    -0.00036,
    0.00981,
    0.00324,
    -0.00287,
    0.00229,
    -0.00214,
    -0.0073,
    -0.00139,
    0.00667,
    0.00213
   ],
   [
    -0.01119,
    0.00773,
    6e-05,
    -0.00599,
    0.01008,
    -0.00876,
    -0.00406,
    -0.0087,
    0.00101,
    -0.00404,
    -0.00634,
    0.00511,
    -0.01171,
    -0.00608
   ],
   [
    -0.01276,
    0.01384,
    -0.00571,
    -0.01917,
    0.00961,
    0.01563,
    -0.00125,
    -0.00978,
    0.01589,
    0.00242,
    -0.01977,
    0.00898,
    0.01466,
    0.00168
   ],
   [
    0.00372,
    -0.01565,
    0.00768,
    -0.00491,
    -0.0033,
    0.00602,
    0.01263,
    -0.00088,
    -0.01347,
    0.00625,
    -0.00526,
    -0.00277,
    0.00487,
    0.01122
   ],
   [
    0.00512,
    0.00456,
    0.0142,
    0.00469,
    -0.00657,
    0.00015,
    0.01082,
    0.00667,
    0.00848,
    0.0112,
    0.00562,
    -0.0058,
    0.00202,
    0.01021
   ],
   [
    0.00401,
    0.01544,
    -0.00096,
    0.00739,
    0.01116,
    -0.00969,
    0.01029,
    0.00571,
    0.01436,
    -0.00153,
    0.00671,
    0.00967,
    -0.00861,
    0.0081
   ],
   [
    0.02502,
    0.01154,
    0.01579,
    -0.0083,
    0.0048,
    0.00397,
    -0.01325,
    0.02277,
    0.01156,
    0.0144,
    -0.00629,
    0.00398,
    0.00094,
    -0.0137
   ],
   [
    -0.01595,
    0.00039,
    0.00078,
    -0.0047,
    -0.00086,
    0.00996,
    -0.00508,
    -0.01663,
    -0.00141,
    0.00349,
    -0.00424,
    -0.00083,
    0.0089,
    -0.00361
   ],
   [
    0.02409,
    -0.01333,
    -0.00479,
    0.02275,
    -0.00776,
    -0.01782,
    0.00225,
    0.00781,
    -0.01921,
    0.00223,
    0.01689,
    -0.00989,
    -0.01727,
    0.00047
   ],
   [
    0.02379,
    0.00507,
    0.01185,
    0.00205,
    -0.00443,
    -0.00209,
    -0.00184,
    0.0183,
    0.00602,
    0.01062,
    0.00516,
    -0.0028,
    -0.00045,
    -0.00012
   ],
   [
    -0.01027,
    -0.01651,
    0.00556,
    -0.00723,
    -0.00536,
    0.00237,
    0.01838,
    -0.01055,
    -0.01451,
    0.00408,
    -0.00398,
    -0.00176,
    0.00498,
    0.01813
   ],
   [
    -0.00296,
    0.00227,
    0.00149,
    -0.00406,
    -0.00667,
    0.00969,
    -0.00715,
    -0.00307,
    -0.00171,
    3e-05,
    -0.0031,
    -0.00765,
    0.00768,
    -0.0063
   ],
   [
    0.00154,
    -0.00464,
    0.00566,
    -0.01245,
    0.00741,
    -0.0029,
    0.00239,
    0.00394,
    -0.0,
    0.00507,
    -0.01176,
    0.00767,
    7e-05,
    0.00233
   ],
   [
    0.02801,
    0.01736,
    0.00498,
    -0.02027,
    -0.0094,
    0.01628,
    0.02006,
    0.02418,
    0.01707,
    0.00667,
    -0.01932,
    -0.00895,
    0.01181,
    0.01598
   ],
   [
    0.00861,
    0.01404,
    0.00837,
    0.01271,
    0.00298,
    -0.00966,
    -0.01503,
    0.00922,
    0.00776,
    0.00508,
    0.01193,
    0.00289,
    -0.01133,
    -0.01269
   ],
   [
    0.0142,
    0.01298,
    0.00867,
    0.00521,
    0.01166,
    -0.01376,
    -0.00913,
    0.01106,
    0.00663,
    0.00393,
    0.00408,
    0.00676,
    -0.01348,
    -0.00846
   ],
   [
    0.02127,
    -0.00872,
    0.00326,
    0.0066,
    -0.01736,
    -0.00115,
    -0.0041,
    0.01464,
    -0.00632,
    0.00691,
    0.0077,
    -0.01582,
    -0.00217,
    -0.00424
   ],
   [
    0.02032,
    0.00572,
    -0.0012,
    0.0053,
    0.0013,
    -0.00089,
    -0.00171,
    0.01791,
    0.00545,
    -0.00357,
    0.00131,
    -0.00192,
    -0.00136,
    -0.00334
   ],
   [
    -0.01869,
    -0.00145,
    0.01048,
    -0.01763,
    -0.00507,
    0.00996,
    0.00416,
    -0.01653,
    -0.006,
    0.00437,
    -0.01551,
    -0.00513,
    0.00636,
    0.00145
   ],
   [
    0.00739,
    0.01567,
    -0.00793,
    -0.00451,
    0.00339,
    0.00577,
    0.0022,
    0.00748,
    0.00886,
    -0.00468,
    -0.00718,
    0.00249,
    0.0058,
    0.00425
   ],
   [
    -0.00859,
    -0.01368,
    -0.00498,
    -0.00386,
    -0.00354,
    -0.00049,
    -0.01472,
    -0.01079,
    -0.01333,
    -0.00385,
    -0.00481,
    -0.00647,
    -0.00275,
    -0.01261
   ],
   [
    -0.00474,
    -0.01007,
    0.00885,
    -0.00511,
    -0.01171,
    0.01074,
    0.00112,
    -0.00484,
    -0.00561,
    0.0096,
    -0.00307,
    -0.01058,
    0.00956,
    -0.0003
   ],
   [
    0.02994,
    -0.0016,
    0.01571,
    0.00266,
    -0.00654,
    0.00224,
    -0.00162,
    0.02379,
    0.00584,
    0.01613,
    0.00466,
    -0.00333,
    0.00521,
    -0.00107
   ],
   [
    0.00011,
    -0.00314,
    0.00767,
    -0.0085,
    -0.01544,
    0.0132,
    0.00972,
    0.00083,
    -0.00183,
    0.01127,
    -0.00684,
    -0.01261,
    0.01247,
    0.00856
   ],
   [
    0.01434,
    0.00799,
    -0.00052,
    0.01112,
    0.00211,
    -0.01494,
    -0.02025,
    0.00769,
    0.00608,
    -0.0003,
    0.01165,
    0.0029,
    -0.01161,
    -0.01641
   ],
   [
    -0.00305,
    -0.0022,
    0.00107,
    -0.00218,
    -0.00638,
    0.01119,
    -0.00261,
    -0.00467,
    -0.00015,
    0.00515,
    -0.00361,
    -0.0073,
    0.01022,
    -0.00224
   ],
   [
    -0.00093,
    -0.0048,
    -0.00029,
    -0.00272,
    0.00846,
    -0.00351,
    -0.00852,
    -7e-05,
    -0.00772,
    -0.00369,
    0.00075,
    0.00841,
    -0.00071,
    -0.00597
   ],
   [
    0.03634,
    0.0197,
    -0.00132,
    -0.0115,
    -0.01669,
    -0.00158,
    0.02478,
    0.03016,
    0.02437,
    0.00212,
    -0.00889,
    -0.01139,
    3e-05,
    0.02136
   ],
   [
    -0.00113,
    -0.00844,
    -0.00809,
    0.0044,
    -0.00726,
    0.00864,
    0.00879,
    0.00302,
    -0.00465,
    -0.00616,
    0.00384,
    -0.00397,
    0.00935,
    0.00736
   ],
   [
    0.01051,
    -0.00778,
    -0.00589,
    0.00333,
    -0.00991,
    0.00572,
    0.00137,
    0.00621,
    -0.00319,
    -0.004,
    0.00443,
    -0.00607,
    0.00447,
    0.00139
   ],
   [
    -0.0071,
    -0.0244,
    -0.00419,
    0.007,
    -0.02036,
    0.00978,
    -0.0022,
    -0.00679,
    -0.01888,
    -0.00567,
    0.0062,
    -0.01682,
    0.00875,
    -0.00411
   ],
   [
    -0.00731,
    0.00019,
    -0.00787,
    -0.0334,
    -0.01216,
    0.03085,
    0.0315,
    -0.00238,
    0.00642,
    0.00071,
    -0.02844,
    -0.00373,
    0.03279,
    0.03002
   ],
   [
    -0.00957,
    -0.00561,
    -0.01358,
    -0.00223,
    0.00547,
    -0.01087,
    0.00155,
    -0.01074,
    -0.00588,
    -0.01324,
    -0.00435,
    0.00306,
    -0.01056,
    -0.0009
   ],
   [
    0.0059,
    0.00204,
    0.01126,
    -0.01479,
    0.00757,
    -0.01,
    -0.00921,
    0.0042,
    -0.00392,
    0.00691,
    -0.01545,
    0.00517,
    -0.00892,
    -0.01007
   ],
   [
    -0.00529,
    -0.01625,
    0.00467,
    -0.00155,
    -0.00596,
    0.00108,
    0.0047,
    -0.00375,
    -0.01633,
    0.00371,
    0.00162,
    -0.00691,
    0.00121,
    0.00404
   ],
   [
    -0.00898,
    0.01418,
    0.00151,
    -0.0023,
    0.00421,
    0.00564,
    -0.00374,
    -0.00779,
    0.00848,
    -0.00134,
    -0.00353,
    0.00265,
    0.00476,
    -0.00168
   ],
   [
    -0.00868,
    0.00165,
    -0.00739,
    7e-05,
    -0.0106,
    0.00759,
    0.01384,
    -0.00624,
    0.00202,
    -0.00519,
    -0.0002,
    -0.0099,
    0.00603,
    0.01105
   ],
   [
    0.01103,
    -0.01184,
    0.03199,
    -0.01463,
    -0.02658,
    0.04168,
    0.02366,
    0.0174,
    -0.00196,
    0.02682,
    -0.0149,
    -0.02284,
    0.03743,
    0.01994
   ],
   [
    0.00188,
    -0.02732,
    0.01603,
    0.01034,
    -0.00661,
    -0.00083,
    0.00529,
    0.00059,
    -0.02475,
    0.01006,
    0.00935,
    -0.00777,
    -0.00131,
    0.00288
   ],
   [
    0.01586,
    -0.00263,
    0.0071,
    0.00389,
    0.00486,
    -0.00978,
    0.00234,
    0.01531,
    0.00187,
    0.00639,
    0.00417,
    0.00341,
    -0.00977,
    0.00022
   ],
   [
    0.00242,
    0.01257,
    0.00489,
    -0.01153,
    0.00652,
    0.00256,
    0.00129,
    -0.00119,
    0.01252,
    0.00704,
    -0.00807,
    0.0077,
    0.00341,
    0.00022
   ],
   [
    -0.00427,
    0.00172,
    0.01438,
    0.002,
    0.00194,
    0.00196,
    -0.00178,
    -0.00092,
    0.00688,
    0.01007,
    0.00542,
    0.00558,
    0.00665,
    0.00154
   ],
   [
    0.00877,
    0.00606,
    0.01733,
    0.00296,
    0.01201,
    0.00202,
    -0.01706,
    0.01248,
    0.01008,
    0.00805,
    0.00277,
    0.01076,
    0.00517,
    -0.011
   ],
   [
    -0.01102,
    0.01267,
    0.00395,
    -0.00693,
    0.01079,
    0.0102,
    -0.00374,
    -0.00593,
    0.01302,
    0.00417,
    -0.00737,
    0.01015,
    0.00836,
    -0.00335
   ],
   [
    0.01317,
    -0.0118,
    0.0078,
    -0.00432,
    -0.00785,
    0.00554,
    0.00177,
    0.01008,
    -0.01056,
    0.0036,
    -0.00664,
    -0.01243,
    0.00225,
    0.00118
   ],
   [
    0.00418,
    -0.00448,
    -0.01078,
    0.01006,
    -0.00986,
    -0.00167,
    -0.00579,
    0.00244,
    -0.00478,
    -0.01083,
    0.01051,
    -0.00819,
    -0.00479,
    -0.00769
   ],
   [
    0.01816,
    0.01811,
    0.00504,
    0.00457,
    0.00399,
    0.00096,
    -0.02319,
    0.01732,
    0.0153,
    0.00364,
    0.00399,
    0.00423,
    -0.00055,
    -0.02057
   ],
   [
    -0.01502,
    0.00353,
    0.00228,
    0.0082,
    0.00738,
    -0.00502,
    -0.01236,
    -0.01109,
    0.00371,
    0.00388,
    0.01063,
    0.00974,
    -0.00233,
    -0.00951
   ],
   [
    0.00481,
    0.00639,
    0.00356,
    -0.01012,
    0.00907,
    -0.00397,
    -0.01537,
    0.00343,
    0.00269,
    -0.00099,
    -0.00574,
    0.00858,
    -0.0036,
    -0.01332
   ],
   [
    -0.00815,
    -0.00038,
    -0.00315,
    -0.01157,
    0.01104,
    0.00383,
    -0.0066,
    -0.0093,
    9e-05,
    -0.00153,
    -0.01018,
    0.0092,
    0.00546,
    -0.00398
   ],
   [
    -0.01389,
    -0.00086,
    0.00969,
    -0.02036,
    0.02011,
    -0.00098,
    0.01033,
    -0.00949,
    -0.00044,
    0.0097,
    -0.01769,
    0.01833,
    -0.00024,
    0.00886
   ],
   [
    0.01035,
    -0.0061,
    -0.00528,
    0.00722,
    0.00023,
    -0.01062,
    0.01597,
    0.0102,
    -0.00181,
    -0.00059,
    0.00741,
    0.00053,
    -0.00798,
    0.01366
   ],
   [
    0.00307,
    0.00583,
    0.00742,
    0.00907,
    -0.00799,
    0.00447,
    0.00228,
    0.00144,
    0.00546,
    0.00557,
    0.00776,
    -0.00886,
    0.00369,
    0.00214
   ],
   [
    0.0137,
    0.00775,
    0.00153,
    0.0033,
    -0.00491,
    0.00258,
    0.00101,
    0.01195,
    0.00969,
    0.00194,
    0.00366,
    -0.00357,
    0.00058,
    0.00056
   ],
   [
    0.00907,
    0.00402,
    0.01625,
    -0.01824,
    0.00526,
    0.00886,
    0.0126,
    0.00664,
    0.00429,
    0.01294,
    -0.01685,
    0.00481,
    0.00704,
    0.008
   ],
   [
    0.00902,
    0.00722,
    0.00886,
    0.00242,
    -0.00166,
    -0.00624,
    -0.00449,
    0.00967,
    0.00642,
    0.00866,
    0.00332,
    -0.00112,
    -0.0056,
    -0.00518
   ],
   [
    -0.0119,
    0.00399,
    -0.00845,
    -0.00336,
    -0.00118,
    -0.00671,
    0.00229,
    -0.00996,
    0.00069,
    -0.00615,
    -0.00202,
    -0.00086,
    -0.00393,
    0.00297
   ],
   [
    -0.00482,
    -0.01978,
    -0.0021,
    0.00537,
    0.00028,
    -0.00689,
    0.00014,
    -0.00133,
    -0.01628,
    0.00304,
    0.0044,
    -0.00238,
    -0.00829,
    -0.00208
   ],
   [
    0.00113,
    -0.00373,
    0.00665,
    -0.00545,
    -0.00638,
    0.00042,
    0.00989,
    0.00017,
    -0.00317,
    0.00901,
    -0.00463,
    -0.00565,
    0.00138,
    0.00934
   ],
   [
    -0.00025,
    0.00473,
    0.00545,
    0.00053,
    -0.01143,
    -0.01069,
    0.02558,
    0.00371,
    0.00414,
    0.00772,
    -0.00075,
    -0.01104,
    -0.00765,
    0.02133
   ],
   [
    -0.03147,
    -0.01842,
    0.01614,
    -0.01606,
    -0.00193,
    0.00607,
    0.00385,
    -0.02578,
    -0.0171,
    0.01107,
    -0.01527,
    -0.00322,
    0.00325,
    0.00097
   ],
   [
    -0.00589,
    0.01437,
    -0.00752,
    -0.00163,
    0.00386,
    0.00501,
    0.00761,
    -0.00176,
    0.00884,
    -0.00627,
    -0.00283,
    0.0027,
    0.00441,
    0.0079
   ],
   [
    -0.01198,
    -0.02557,
    -0.01126,
    -0.00556,
    -0.01419,
    -0.00864,
    0.00405,
    -0.00947,
    -0.02036,
    -0.00665,
    -0.00516,
    -0.01311,
    -0.00582,
    0.00253
   ],
   [
    0.01262,
    -0.00707,
    0.01079,
    -0.0043,
    -0.0061,
    0.01027,
    0.00242,
    0.00848,
    -0.00199,
    0.00513,
    -0.00444,
    -0.00541,
    0.00677,
    -8e-05
   ],
   [
    -0.00764,
    -0.00594,
    -0.00283,
    -0.01712,
    0.00225,
    0.01038,
    0.01205,
    -0.00471,
    -0.00192,
    0.00082,
    -0.01481,
    0.00337,
    0.01234,
    0.01092
   ],
   [
    -0.00991,
    -0.03846,
    0.0058,
    -0.00317,
    -0.00994,
    0.00168,
    0.00152,
    -0.00938,
    -0.03233,
    0.0004,
    -0.0011,
    -0.00813,
    0.00402,
    0.00107
   ],
   [
    0.01544,
    0.00279,
    0.00485,
    0.00345,
    -0.00952,
    -0.00205,
    0.01072,
    0.01178,
    0.00215,
    0.00601,
    0.00426,
    -0.00815,
    -0.00271,
    0.00821
   ],
   [
    -0.00715,
    -0.0008,
    -0.00602,
    0.0142,
    0.00687,
    -0.02851,
    -0.01154,
    -0.00462,
    -0.00449,
    -0.00767,
    0.01193,
    0.00444,
    -0.02748,
    -0.01097
   ],
   [
    0.02272,
    0.00275,
    0.01361,
    0.00577,
    0.00139,
    -0.00242,
    -0.00013,
    0.02018,
    0.00753,
    0.01099,
    0.00564,
    0.00307,
    -0.0012,
    -0.00049
   ],
   [
    -0.00431,
    -0.00126,
    0.00684,
    -0.01302,
    -0.01643,
    0.01527,
    0.00376,
    -0.00161,
    -0.005,
    0.00746,
    -0.0118,
    -0.01244,
    0.0146,
    0.00361
   ],
   [
    -0.00989,
    -0.0117,
    0.01506,
    0.00063,
    -0.01483,
    0.00037,
    0.00209,
    -0.00916,
    -0.00945,
    0.00915,
    0.00423,
    -0.00933,
    0.00349,
    0.00184
   ],
   [
    -0.01589,
    0.02052,
    -0.00442,
    -0.0024,
    0.01952,
    -0.01779,
    -0.00759,
    -0.01082,
    0.01782,
    -0.00387,
    -0.0018,
    0.01591,
    -0.01532,
    -0.00732
   ],
   [
    -0.00262,
    0.00311,
    -0.00072,
    -0.00182,
    -0.00243,
    0.0114,
    0.02174,
    -0.00353,
    0.00879,
    0.00315,
    0.00068,
    -0.00091,
    0.01113,
    0.0193
   ],
   [
    -0.0056,
    -0.0075,
    0.01159,
    0.00638,
    0.00161,
    -0.00566,
    -0.00506,
    -0.00266,
    -0.00118,
    0.01013,
    0.0096,
    0.00311,
    -0.00397,
    -0.00441
   ],
   [
    -0.00775,
    -0.02131,
    0.01011,
    0.01228,
    -0.01449,
    -0.0006,
    0.00374,
    -0.0077,
    -0.01231,
    0.00854,
    0.01148,
    -0.01174,
    -0.00066,
    0.00342
   ],
   [
    0.01141,
    0.01677,
    0.00624,
    0.0041,
    0.00703,
    2e-05,
    0.00428,
    0.009,
    0.01306,
    0.00861,
    0.00495,
    0.00673,
    0.00147,
    0.00456
   ],
   [
    0.00463,
    0.00573,
    0.00553,
    0.01481,
    0.00256,
    -0.00673,
    -0.01902,
    0.00667,
    0.00252,
    0.00326,
    0.01261,
    0.00306,
    -0.0072,
    -0.01631
   ],
   [
    -0.00344,
    0.00424,
    0.00151,
    0.00044,
    -0.00126,
    8e-05,
    -0.00187,
    -0.00089,
    0.00722,
    0.00205,
    7e-05,
    -0.00184,
    -0.00012,
    -0.00358
   ],
   [
    -0.00142,
    -0.00583,
    0.00538,
    0.00766,
    -0.01165,
    0.00966,
    -0.00436,
    -0.00361,
    -0.00854,
    0.00459,
    0.00785,
    -0.01169,
    0.00574,
    -0.00391
   ],
   [
    0.00904,
    0.00229,
    -0.00842,
    0.00276,
    -0.00401,
    -4e-05,
    0.00055,
    0.00457,
    0.00191,
    -0.0041,
    0.0024,
    -0.003,
    -0.00149,
    -0.00151
   ],
   [
    0.00791,
    -0.00956,
    0.00483,
    -0.00433,
    0.00039,
    0.004,
    -0.00649,
    0.00526,
    -0.00445,
    0.00539,
    -0.00329,
    -0.00178,
    0.00303,
    -0.0069
   ],
   [
    0.00788,
    -0.00219,
    -0.00445,
    0.00111,
    -0.025,
    0.01335,
    0.00467,
    0.00866,
    0.00154,
    0.00391,
    -0.00067,
    -0.02267,
    0.01139,
    0.00418
   ],
   [
    -0.00521,
    0.01618,
    -0.00939,
    -0.01427,
    0.00722,
    0.00137,
    0.00317,
    -0.0014,
    0.00696,
    -0.00667,
    -0.01574,
    0.00493,
    -0.00179,
    0.00106
   ],
   [
    0.00852,
    -0.0026,
    -0.00342,
    -0.00689,
    0.00307,
    0.00845,
    0.0123,
    0.00716,
    -0.00418,
    -0.0031,
    -0.00632,
    0.00343,
    0.00685,
    0.00928
   ],
   [
    0.00054,
    -0.00501,
    -0.0036,
    0.00305,
    -0.00022,
    -0.01225,
    -0.00908,
    -0.00038,
    -0.00026,
    -0.00453,
    0.00375,
    -0.00243,
    -0.01073,
    -0.00853
   ],
   [
    -0.00013,
    0.00242,
    0.00597,
    -0.00448,
    0.00465,
    0.00605,
    -0.0017,
    0.00279,
    0.00511,
    0.00816,
    -0.00391,
    0.00557,
    0.00766,
    -0.00136
   ],
   [
    -0.02502,
    -0.02101,
    -0.00783,
    -0.0034,
    -0.00121,
    0.00277,
    0.00102,
    -0.02395,
    -0.02276,
    -0.01068,
    -0.0045,
    -0.00162,
    0.00094,
    0.00075
   ],
   [
    -0.00652,
    -0.00258,
    0.00491,
    0.00927,
    -0.00316,
    -0.00444,
    -0.00728,
    -0.00678,
    -0.00243,
    0.00814,
    0.0109,
    -0.00109,
    -0.00245,
    -0.00419
   ],
   [
    0.03,
    0.00338,
    -0.00533,
    0.01402,
    0.00024,
    0.00088,
    -0.00015,
    0.0286,
    0.00208,
    -0.00227,
    0.01212,
    -0.00172,
    -0.00152,
    -0.00026
   ],
   [
    0.0113,
    -0.03089,
    0.00736,
    0.02482,
    -0.03227,
    -0.0164,
    0.03791,
    0.00713,
    -0.0233,
    0.00598,
    0.027,
    -0.02496,
    -0.01369,
    0.03007
   ],
   [
    -9e-05,
    0.00412,
    0.00366,
    -0.00507,
    0.00243,
    -0.0035,
    0.01017,
    0.00135,
    0.00614,
    0.00875,
    -0.00537,
    0.00168,
    -0.00056,
    0.00996
   ],
   [
    0.00291,
    -0.01303,
    0.00325,
    -0.00152,
    -0.00906,
    0.00247,
    -0.00843,
    0.00097,
    -0.01344,
    0.00095,
    -0.0024,
    -0.00802,
    -0.00153,
    -0.01107
   ],
   [
    0.01149,
    -0.00753,
    -0.00768,
    0.02566,
    0.00947,
    -0.02316,
    -0.00869,
    0.01019,
    0.00042,
    -0.00682,
    0.02526,
    0.00573,
    -0.02263,
    -0.00951
   ],
   [
    0.00911,
    0.00384,
    0.00388,
    0.01927,
    -0.0008,
    -0.01252,
    -0.00407,
    0.01066,
    0.00342,
    0.00464,
    0.02236,
    0.00389,
    -0.01085,
    -0.00358
   ],
   [
    0.0013,
    -0.01324,
    0.00108,
    0.00019,
    0.00605,
    0.00133,
    0.00303,
    0.00147,
    -0.01433,
    -0.00193,
    -0.00387,
    0.00143,
    0.00093,
    -0.00091
   ],
   [
    0.00026,
    0.02883,
    -0.01282,
    -0.01645,
    0.01454,
    -0.00112,
    0.01146,
    -0.00289,
    0.02179,
    -0.00697,
    -0.0158,
    0.01206,
    -0.00222,
    0.00892
   ],
   [
    -0.00235,
    0.00179,
    -0.01378,
    0.00032,
    -0.0037,
    0.00626,
    -0.01052,
    -0.00193,
    -0.00141,
    -0.00994,
    -0.00155,
    -0.00307,
    0.00537,
    -0.0093
   ],
   [
    0.02836,
    0.02003,
    0.00948,
    0.00222,
    0.00903,
    -0.00882,
    0.00096,
    0.025,
    0.01551,
    0.0044,
    0.00326,
    0.00629,
    -0.00964,
    -0.00088
   ],
   [
    -0.00883,
    -0.01378,
    -0.00167,
    0.00152,
    0.01295,
    -0.0117,
    -0.01632,
    -0.00691,
    -0.01236,
    -0.00727,
    0.00396,
    0.01282,
    -0.01005,
    -0.01551
   ],
   [
    -0.03247,
    -0.02073,
    -0.00547,
    0.00349,
    -0.01687,
    0.00285,
    0.0126,
    -0.0282,
    -0.02032,
    -0.0056,
    0.00363,
    -0.01317,
    -0.0003,
    0.00901
   ],
   [
    0.00747,
    0.00168,
    0.0062,
    0.00928,
    0.00521,
    -0.00796,
    -0.00081,
    0.00788,
    -0.00368,
    0.00414,
    0.00676,
    0.00516,
    -0.00675,
    0.00029
   ],
   [
    0.00161,
    0.0151,
    -0.01034,
    -0.00131,
    0.00404,
    0.01328,
    -0.01503,
    0.00278,
    0.0114,
    -0.00681,
    -0.00051,
    0.00289,
    0.0104,
    -0.01207
   ],
   [
    0.00704,
    0.00932,
    -0.00476,
    0.0071,
    -0.00423,
    0.0048,
    -0.00494,
    0.00532,
    0.00687,
    -0.00733,
    0.00554,
    -0.00214,
    0.00324,
    -0.00323
   ],
   [
    0.02365,
    0.00195,
    -0.00738,
    0.00563,
    0.00606,
    -0.01753,
    0.0098,
    0.01983,
    0.00133,
    0.0014,
    0.00388,
    0.0036,
    -0.01331,
    0.00903
   ],
   [
    -0.00056,
    0.01224,
    0.00547,
    0.02286,
    0.02622,
    -0.01908,
    -0.02929,
    -0.00173,
    0.00713,
    0.00096,
    0.02331,
    0.0204,
    -0.0141,
    -0.02495
   ],
   [
    -0.00178,
    -0.0078,
    -0.00813,
    0.01912,
    0.0013,
    -0.01552,
    -0.00674,
    -0.00116,
    -0.00767,
    -0.00262,
    0.01785,
    0.002,
    -0.01198,
    -0.00429
   ],
   [
    -0.01199,
    -0.01101,
    0.01062,
    -0.00943,
    0.00204,
    -0.00471,
    -0.00763,
    -0.01222,
    -0.01357,
    0.00712,
    -0.00807,
    0.00083,
    -0.00387,
    -0.0082
   ],
   [
    -0.01262,
    0.01522,
    -0.0145,
    -0.03195,
    0.01943,
    0.00806,
    0.0053,
    -0.01083,
    0.01144,
    -0.00588,
    -0.02775,
    0.01765,
    0.00808,
    0.00574
   ],
   [
    0.01128,
    0.02245,
    0.00229,
    -0.00459,
    0.00585,
    0.00974,
    -0.00118,
    0.00972,
    0.01783,
    0.00513,
    -0.00327,
    0.00514,
    0.00933,
    6e-05
   ],
   [
    -0.00567,
    -0.0048,
    -0.00488,
    0.00385,
    -0.00468,
    0.00443,
    -0.0115,
    -0.00415,
    -0.00554,
    -0.00148,
    0.00269,
    -0.00538,
    0.00362,
    -0.01022
   ],
   [
    -0.00076,
    0.00278,
    0.00279,
    0.00702,
    -0.00589,
    0.01087,
    0.00153,
    0.00037,
    0.00469,
    0.00321,
    0.00536,
    -0.00491,
    0.00983,
    0.00211
   ],
   [
    0.00959,
    0.00618,
    -0.00846,
    -0.00156,
    0.01397,
    0.00284,
    -0.01268,
    0.00588,
    0.00452,
    -0.00555,
    -0.00237,
    0.01262,
    0.00159,
    -0.00986
   ],
   [
    -0.00953,
    -0.00611,
    -0.00706,
    -0.00719,
    0.01003,
    -0.00356,
    -0.00181,
    -0.00774,
    -0.00436,
    -0.00414,
    -0.00754,
    0.00863,
    -0.00241,
    -0.00102
   ],
   [
    0.00309,
    -0.00337,
    0.0025,
    -0.00466,
    -0.01179,
    0.00579,
    9e-05,
    0.00507,
    0.00381,
    0.00716,
    -0.00231,
    -0.00958,
    0.00784,
    0.00019
   ],
   [
    -0.00877,
    0.00612,
    -0.01181,
    0.00177,
    0.01453,
    -0.00427,
    -0.00112,
    -0.00465,
    0.00493,
    -0.00639,
    -0.00075,
    0.01249,
    -0.00038,
    0.00159
   ],
   [
    -0.00257,
    -0.01527,
    -0.00109,
    -0.01148,
    0.00141,
    -0.00042,
    0.0014,
    -0.00559,
    -0.01356,
    -0.00448,
    -0.01085,
    0.00048,
    -0.00141,
    0.00022
   ],
   [
    -0.01533,
    0.01917,
    0.00643,
    0.00431,
    -0.00456,
    0.00626,
    -0.00541,
    -0.01415,
    0.01527,
    0.00411,
    0.00195,
    -0.00617,
    0.00179,
    -0.00661
   ],
   [
    -0.00325,
    -0.00385,
    0.00145,
    0.00316,
    -0.00233,
    -0.00832,
    -0.00771,
    -0.00554,
    -0.00722,
    -0.00635,
    0.00359,
    -0.00297,
    -0.00902,
    -0.00826
   ],
   [
    0.00557,
    0.01375,
    -0.00739,
    0.00252,
    0.01082,
    -0.01074,
    0.00208,
    0.00627,
    0.0098,
    -0.00786,
    -0.00164,
    0.00608,
    -0.01001,
    0.0008
   ],
   [
    -0.00604,
    -0.01824,
    -0.01445,
    -0.01147,
    0.0017,
    -0.00077,
    0.01504,
    -0.00132,
    -0.00932,
    -0.00607,
    -0.00892,
    0.00013,
    -0.00016,
    0.0117
   ],
   [
    0.00901,
    0.00268,
    0.01181,
    -0.01205,
    -0.00682,
    0.01993,
    0.0027,
    0.0087,
    0.00622,
    0.01405,
    -0.01048,
    -0.00316,
    0.0193,
    0.00599
   ],
   [
    -0.02512,
    0.00158,
    0.00182,
    -0.01994,
    0.00429,
    0.00678,
    0.01138,
    -0.01767,
    -0.00184,
    -0.00018,
    -0.02143,
    0.00104,
    0.0051,
    0.00981
   ],
   [
    -0.00367,
    -0.00815,
    -0.00071,
    0.02236,
    -0.02431,
    0.00528,
    0.00574,
    -0.00255,
    -0.0024,
    -0.00338,
    0.02152,
    -0.01957,
    0.00516,
    0.00458
   ],
   [
    -0.00835,
    -0.0069,
    0.0042,
    -0.00183,
    -0.00929,
    -0.00316,
    0.00511,
    -0.01098,
    -0.00295,
    0.00251,
    -0.00094,
    -0.00855,
    -0.00578,
    0.00401
   ],
   [
    0.01767,
    -0.0119,
    0.00206,
    -0.00071,
    -0.00633,
    0.00361,
    -0.00178,
    0.01577,
    -0.00739,
    0.0001,
    -0.00048,
    -0.00601,
    0.00152,
    -0.00357
   ],
   [
    -0.01594,
    -0.01111,
    -0.00706,
    0.0011,
    -0.00189,
    -0.00073,
    -0.0172,
    -0.01165,
    -0.01558,
    -0.00603,
    0.00147,
    -0.00069,
    7e-05,
    -0.01433
   ],
   [
    0.00075,
    -0.00416,
    -0.00047,
    0.00223,
    0.00798,
    -0.01008,
    -0.00944,
    0.00194,
    -0.00337,
    -0.00061,
    0.00298,
    0.00594,
    -0.00655,
    -0.00729
   ],
   [
    0.01093,
    -0.00013,
    -0.00753,
    0.00048,
    0.00654,
    -0.00822,
    0.00936,
    0.0109,
    0.00386,
    -0.00325,
    0.00214,
    0.00713,
    -0.00615,
    0.00861
   ],
   [
    -0.01888,
    -0.03094,
    0.00924,
    -0.00796,
    -0.00744,
    0.00141,
    0.0032,
    -0.01835,
    -0.02603,
    0.00174,
    -0.00637,
    -0.01024,
    -0.00195,
    -0.00016
   ],
   [
    0.01245,
    0.01989,
    0.00391,
    -0.00096,
    0.00444,
    0.01133,
    -0.00317,
    0.0126,
    0.01721,
    0.00624,
    -0.00211,
    0.00469,
    0.00954,
    -0.00396
   ],
   [
    -0.00925,
    0.002,
    -0.01124,
    -0.00618,
    0.01095,
    0.00898,
    -0.0135,
    -0.00967,
    0.0007,
    -0.0026,
    -0.00271,
    0.00924,
    0.00616,
    -0.01095
   ],
   [
    0.01418,
    0.00702,
    -0.00144,
    -0.00059,
    0.00341,
    0.00642,
    0.00955,
    0.0171,
    0.00806,
    -0.00172,
    0.0005,
    0.0051,
    0.00814,
    0.00965
   ],
   [
    0.00762,
    0.0088,
    0.00216,
    -0.00329,
    0.01425,
    0.00242,
    -0.00575,
    0.00506,
    0.01662,
    0.0059,
    -0.00113,
    0.01242,
    0.00514,
    -0.00436
   ],
   [
    0.00282,
    -0.00374,
    -0.01086,
    -0.00787,
    0.00551,
    0.00608,
    -0.00704,
    0.00066,
    -0.00287,
    -0.00604,
    -0.00775,
    0.00216,
    0.00324,
    -0.00635
   ],
   [
    0.00514,
    -0.00185,
    0.01645,
    -0.0074,
    -0.0042,
    0.00443,
    0.00358,
    0.00492,
    0.00211,
    0.01293,
    -0.00768,
    -0.00418,
    0.0059,
    0.00469
   ],
   [
    -0.00172,
    -0.00775,
    0.00399,
    0.00567,
    -0.00768,
    -0.00947,
    0.01514,
    -0.00046,
    -0.0001,
    0.0048,
    0.00625,
    -0.00516,
    -0.00913,
    0.01062
   ],
   [
    -0.00586,
    -0.01157,
    0.00383,
    0.00675,
    0.00109,
    -0.00435,
    -0.0093,
    -0.00687,
    -0.00883,
    0.00132,
    0.00657,
    -0.00064,
    -0.00386,
    -0.00868
   ],
   [
    0.00335,
    0.01881,
    0.00399,
    0.00075,
    0.00321,
    0.0015,
    0.00238,
    0.00288,
    0.01518,
    0.00186,
    0.00084,
    0.00247,
    0.00079,
    0.00257
   ],
   [
    0.01579,
    -0.01834,
    0.0066,
    -0.00811,
    -0.00452,
    0.00359,
    0.01405,
    0.01123,
    -0.01393,
    0.00414,
    -0.00755,
    -0.00344,
    -0.00173,
    0.00827
   ],
   [
    0.01248,
    0.00453,
    0.00423,
    -0.00356,
    0.0044,
    -0.00285,
    -0.02101,
    0.01032,
    0.00095,
    0.00044,
    -0.00436,
    0.00281,
    -0.00462,
    -0.01883
   ],
   [
    0.01688,
    0.00879,
    0.00267,
    -0.00962,
    0.00244,
    0.0171,
    0.0114,
    0.01615,
    0.00964,
    0.00212,
    -0.00848,
    0.00122,
    0.01774,
    0.01135
   ],
   [
    -0.01081,
    0.00489,
    -0.00985,
    0.00279,
    -0.00938,
    0.01456,
    0.00909,
    -0.00936,
    0.00727,
    -0.00486,
    0.00408,
    -0.0042,
    0.01638,
    0.01111
   ],
   [
    0.02761,
    -0.00851,
    0.01235,
    0.02369,
    -0.0087,
    -0.01167,
    -0.01766,
    0.02658,
    -0.00962,
    0.00433,
    0.02104,
    -0.01101,
    -0.01323,
    -0.01623
   ],
   [
    0.01068,
    -0.01461,
    0.00617,
    0.00117,
    0.00502,
    0.00161,
    -0.00362,
    0.0046,
    -0.01175,
    0.00411,
    0.00011,
    0.00071,
    0.00056,
    -0.00541
   ],
   [
    0.01174,
    0.03217,
    -0.01081,
    -0.00073,
    0.00438,
    0.00221,
    0.00491,
    0.01129,
    0.02432,
    -0.00691,
    -0.00222,
    0.00504,
    0.00324,
    0.00445
   ],
   [
    0.0087,
    -0.01908,
    -0.00679,
    -0.00792,
    -0.00932,
    0.00383,
    0.00464,
    0.00269,
    -0.01763,
    -0.00668,
    -0.00698,
    -0.00701,
    0.00397,
    0.00434
   ],
   [
    -0.02653,
    0.03145,
    0.00112,
    -0.00531,
    0.03761,
    -0.03234,
    -0.01014,
    -0.02448,
    0.02172,
    -0.00348,
    -0.00739,
    0.02821,
    -0.03083,
    -0.00975
   ],
   [
    -0.00465,
    -0.00325,
    0.00336,
    0.00472,
    0.00541,
    -0.00328,
    -0.00476,
    -0.00428,
    -0.00382,
    0.00629,
    0.00314,
    0.00485,
    -0.00119,
    -0.00291
   ],
   [
    0.01565,
    -0.01192,
    0.00354,
    0.00533,
    -0.00883,
    0.00778,
    0.00491,
    0.00844,
    -0.01017,
    0.0019,
    0.00442,
    -0.00695,
    0.00684,
    0.00448
   ],
   [
    -0.00888,
    0.01512,
    -0.00159,
    0.00628,
    0.01518,
    -0.01541,
    -0.00869,
    -0.0098,
    0.00777,
    -0.0043,
    0.0061,
    0.014,
    -0.01036,
    -0.00526
   ],
   [
    -0.01765,
    -0.01406,
    -0.00879,
    -0.01483,
    -0.00857,
    0.01191,
    -0.00159,
    -0.01557,
    -0.01386,
    -0.00594,
    -0.01623,
    -0.01051,
    0.01009,
    -0.00161
   ],
   [
    0.00492,
    0.00375,
    0.0088,
    -0.00902,
    -0.00147,
    -8e-05,
    -9e-05,
    0.00347,
    0.00042,
    0.00543,
    -0.01086,
    0.00014,
    0.00042,
    0.0012
   ],
   [
    -0.0048,
    -0.02728,
    -0.00035,
    -0.00853,
    -0.00896,
    0.01126,
    0.00034,
    -0.00465,
    -0.02083,
    0.00179,
    -0.00527,
    -0.00765,
    0.01195,
    0.00114
   ],
   [
    0.02689,
    0.00976,
    -0.00369,
    -0.00681,
    -0.001,
    0.0016,
    0.00594,
    0.02637,
    0.01353,
    -0.00302,
    -0.00944,
    -0.00089,
    0.00283,
    0.00558
   ],
   [
    0.00289,
    -0.00986,
    -0.01087,
    0.00676,
    -0.00457,
    -0.00137,
    0.00362,
    0.00164,
    -0.00435,
    -0.00343,
    0.00712,
    -0.00466,
    0.00035,
    0.00336
   ],
   [
    -0.01403,
    -0.00105,
    -0.00291,
    -0.0037,
    0.00957,
    -0.00223,
    0.00911,
    -0.01214,
    0.00044,
    -0.00524,
    -0.00307,
    0.01058,
    -0.00314,
    0.00668
   ],
   [
    -0.01019,
    -0.01826,
    0.00836,
    0.00024,
    -0.00258,
    -0.00022,
    0.01185,
    -0.0079,
    -0.01868,
    0.0014,
    -0.00126,
    -6e-05,
    0.00114,
    0.01137
   ],
   [
    0.00159,
    -0.0034,
    -0.00237,
    0.00473,
    -0.00325,
    0.00609,
    0.01527,
    0.00383,
    -0.00437,
    -0.00168,
    0.00671,
    -0.00202,
    0.0047,
    0.01352
   ],
   [
    0.02027,
    -0.00298,
    0.00438,
    0.00232,
    -0.00423,
    0.0152,
    -0.01084,
    0.01464,
    -0.00471,
    0.00445,
    -0.00173,
    -0.00487,
    0.01356,
    -0.00822
   ],
   [
    0.01037,
    -0.00678,
    -0.00297,
    0.0089,
    0.00073,
    0.00388,
    0.00701,
    0.00952,
    -0.00197,
    -0.00356,
    0.01096,
    0.00228,
    0.00441,
    0.00519
   ],
   [
    0.0152,
    0.01251,
    -0.00055,
    -0.0169,
    0.01295,
    0.00667,
    0.01272,
    0.01428,
    0.01591,
    0.00174,
    -0.01634,
    0.01248,
    0.00846,
    0.01208
   ],
   [
    -0.01836,
    -0.01752,
    -0.00322,
    -0.00591,
    0.00222,
    0.01033,
    -0.01458,
    -0.01773,
    -0.01774,
    -0.00455,
    -0.00313,
    0.00319,
    0.00995,
    -0.01145
   ],
   [
    -0.00549,
    0.00578,
    -0.00054,
    -0.0001,
    0.00818,
    -0.012,
    -0.0102,
    -0.00311,
    -0.00149,
    -0.00444,
    -0.00361,
    0.0038,
    -0.01249,
    -0.01082
   ],
   [
    -0.02626,
    0.00791,
    0.00819,
    -0.00216,
    0.00723,
    0.00288,
    -0.00755,
    -0.02283,
    0.0024,
    0.00646,
    -0.00413,
    0.00589,
    0.00551,
    -0.00445
   ],
   [
    -0.0285,
    -0.01599,
    0.01941,
    0.0027,
    -0.00441,
    0.00236,
    0.00097,
    -0.01914,
    -0.01526,
    0.01378,
    0.00207,
    -0.0037,
    0.004,
    0.00301
   ],
   [
    0.00312,
    0.01661,
    0.00934,
    0.00547,
    -0.00845,
    -0.00241,
    0.00486,
    0.00077,
    0.01337,
    0.00449,
    0.00582,
    -0.00761,
    -0.00249,
    0.00506
   ],
   [
    -0.00411,
    0.00206,
    -0.00139,
    0.00391,
    -0.00955,
    -0.00677,
    0.01103,
    -0.00376,
    0.00127,
    -0.00385,
    0.00442,
    -0.00723,
    -0.00564,
    0.0086
   ],
   [
    0.0023,
    -0.00882,
    -0.00784,
    -0.00344,
    0.0104,
    -0.00064,
    -0.00622,
    0.00063,
    -0.006,
    -0.004,
    -0.00292,
    0.00813,
    -0.00097,
    -0.00591
   ],
   [
    -0.0155,
    -0.00095,
    -0.00812,
    -0.00403,
    0.0096,
    0.00026,
    -0.01176,
    -0.01398,
    -0.00311,
    -0.00542,
    -0.00446,
    0.00738,
    -0.00082,
    -0.01066
   ],
   [
    0.01899,
    -0.0082,
    -0.00285,
    -0.00728,
    0.00962,
    0.00374,
    0.00091,
    0.0069,
    -0.00658,
    -0.00262,
    -0.00318,
    0.00889,
    0.00664,
    0.00343
   ],
   [
    0.01819,
    -0.0144,
    -0.00718,
    0.00441,
    -0.03784,
    0.03258,
    0.01168,
    0.01714,
    -0.01193,
    -0.00099,
    0.00196,
    -0.03153,
    0.02819,
    0.0111
   ],
   [
    -0.00956,
    0.02339,
    -0.01451,
    0.00142,
    0.00946,
    0.01018,
    -0.01332,
    -0.01041,
    0.01576,
    -0.01185,
    0.00015,
    0.00988,
    0.00925,
    -0.01184
   ],
   [
    -0.00556,
    -0.02065,
    -0.01183,
    0.00416,
    -0.00336,
    -0.0075,
    -0.00916,
    -0.00713,
    -0.01724,
    -0.00558,
    0.00288,
    -0.0041,
    -0.00679,
    -0.00827
   ],
   [
    0.03652,
    0.01404,
    0.00876,
    -0.00042,
    0.00271,
    0.00347,
    0.00047,
    0.03258,
    0.01635,
    0.0049,
    -0.00036,
    0.00277,
    0.00411,
    -0.00034
   ],
   [
    -0.00607,
    -0.00709,
    0.01786,
    0.00295,
    -0.01194,
    0.00738,
    -0.00705,
    -0.00427,
    -0.00743,
    0.01305,
    0.00216,
    -0.00984,
    0.00419,
    -0.00479
   ],
   [
    -0.01672,
    0.00205,
    -0.01715,
    -0.00139,
    0.0051,
    -0.00235,
    0.00383,
    -0.01319,
    0.0027,
    -0.01137,
    -0.0011,
    0.00659,
    -0.00214,
    0.00432
   ],
   [
    0.00193,
    0.00872,
    -0.0063,
    -0.00751,
    0.00325,
    0.00706,
    0.00701,
    -0.00053,
    0.00846,
    0.00024,
    -0.00503,
    0.00326,
    0.00786,
    0.00658
   ],
   [
    0.01234,
    0.015,
    0.00055,
    -0.00594,
    -0.0044,
    0.0216,
    0.00528,
    0.01225,
    0.01383,
    0.0017,
    -0.00356,
    -0.00209,
    0.02156,
    0.00729
   ],
   [
    0.00443,
    0.00618,
    -0.0113,
    0.00161,
    0.00481,
    -0.00649,
    -0.00187,
    0.00882,
    0.01011,
    -0.0053,
    0.00317,
    0.003,
    -0.00753,
    -0.00219
   ],
   [
    -0.012,
    0.00755,
    -0.00421,
    0.00101,
    0.01018,
    2e-05,
    0.00232,
    -0.00644,
    0.00667,
    -0.00561,
    0.00197,
    0.01032,
    0.00254,
    0.00287
   ],
   [
    -0.01148,
    -0.02431,
    0.00305,
    0.02346,
    -0.00982,
    -0.01052,
    -0.00927,
    -0.00713,
    -0.01522,
    0.00384,
    0.01814,
    -0.01348,
    -0.01028,
    -0.00815
   ],
   [
    0.00728,
    -0.00642,
    0.00172,
    0.00374,
    0.00089,
    -0.00496,
    -0.00989,
    0.0105,
    -0.00508,
    -0.00378,
    0.00195,
    0.00019,
    -0.00892,
    -0.01018
   ],
   [
    -0.00284,
    -0.01647,
    0.00223,
    0.0031,
    0.00461,
    0.00491,
    0.00802,
    -0.00362,
    -0.00953,
    0.00405,
    0.00246,
    0.00366,
    0.0063,
    0.00716
   ],
   [
    -0.00452,
    0.00511,
    0.00193,
    -0.01897,
    -0.00363,
    0.02149,
    0.01364,
    -0.00707,
    0.00865,
    0.00307,
    -0.01477,
    -0.00126,
    0.02224,
    0.01302
   ],
   [
    0.01509,
    -0.00257,
    0.00102,
    -0.00157,
    3e-05,
    -0.00763,
    0.01199,
    0.01085,
    -0.00167,
    -0.00129,
    -0.00152,
    0.00092,
    -0.00572,
    0.00848
   ],
   [
    0.02474,
    0.00732,
    -0.00522,
    -0.01024,
    0.00757,
    -0.00851,
    -0.00553,
    0.02161,
    0.00241,
    -0.00574,
    -0.01137,
    0.00347,
    -0.00692,
    -0.00448
   ],
   [
    -0.00657,
    -0.01159,
    -0.00019,
    0.00121,
    -0.00146,
    -0.00433,
    -0.01951,
    -0.00993,
    -0.01516,
    -0.00774,
    -0.00062,
    -0.00358,
    -0.00386,
    -0.01767
   ],
   [
    -0.01237,
    -0.00781,
    -0.01368,
    0.00674,
    -0.00735,
    -1e-05,
    0.00142,
    -0.01112,
    -0.01155,
    -0.01291,
    0.00258,
    -0.00777,
    -0.00056,
    0.00195
   ],
   [
    0.00103,
    -0.00624,
    -0.00011,
    -0.0046,
    0.00629,
    -0.00386,
    -0.00133,
    -0.00232,
    -0.00393,
    -0.0003,
    -0.00072,
    0.00709,
    -0.00454,
    -0.00145
   ],
   [
    -0.00208,
    0.00287,
    -0.00233,
    -0.00957,
    -0.0013,
    0.00056,
    -0.00457,
    -0.00301,
    0.00038,
    -0.00213,
    -0.01025,
    -0.00205,
    -0.00208,
    -0.00492
   ],
   [
    0.03095,
    0.00217,
    0.00279,
    0.00046,
    -0.0026,
    -0.00454,
    0.00478,
    0.02815,
    7e-05,
    0.00218,
    0.00143,
    -0.00115,
    -0.0003,
    0.00693
   ],
   [
    0.0232,
    -0.01568,
    -0.01681,
    0.01671,
    -0.00323,
    -0.01,
    -0.00018,
    0.0174,
    -0.0135,
    -0.01014,
    0.01311,
    -0.0062,
    -0.01175,
    -0.00273
   ],
   [
    -0.02854,
    -0.00726,
    0.00012,
    0.00241,
    -0.00396,
    -0.00133,
    -0.00597,
    -0.02362,
    -0.00445,
    0.00027,
    0.00436,
    -0.00156,
    0.00118,
    -0.00327
   ],
   [
    -0.01645,
    -0.00569,
    -0.01504,
    -0.00075,
    -0.00082,
    -0.00313,
    0.00016,
    -0.01258,
    -0.00711,
    -0.00774,
    -0.00138,
    0.00081,
    -0.00221,
    -0.00034
   ],
   [
    0.00118,
    -0.01129,
    0.00454,
    0.00789,
    -0.00338,
    -0.01737,
    0.00014,
    0.00013,
    -0.00825,
    -0.00138,
    0.01059,
    -0.00485,
    -0.01599,
    0.00033
   ],
   [
    0.0201,
    0.02417,
    -7e-05,
    0.00478,
    -0.00359,
    0.00232,
    -0.00751,
    0.01996,
    0.01714,
    0.00244,
    0.00133,
    -0.00416,
    0.00031,
    -0.00753
   ],
   [
    0.01493,
    -0.00939,
    -0.00092,
    -0.00713,
    -0.00019,
    0.00665,
    0.00563,
    0.00972,
    -0.00396,
    -0.00201,
    -0.00368,
    0.00067,
    0.00477,
    0.00446
   ],
   [
    0.02337,
    0.0274,
    -0.01184,
    -0.00568,
    0.00142,
    0.00459,
    -0.00185,
    0.01591,
    0.02239,
    -0.00931,
    -0.00596,
    0.00254,
    0.00716,
    0.0
   ],
   [
    0.01434,
    0.00076,
    -0.00242,
    0.00599,
    -0.00362,
    -0.00282,
    -0.00439,
    0.01276,
    0.00179,
    0.00178,
    0.00385,
    -0.00557,
    -0.00639,
    -0.00687
   ],
   [
    -0.02117,
    0.01683,
    -0.00893,
    -0.02162,
    0.01766,
    0.00686,
    0.00449,
    -0.01537,
    0.00822,
    -0.00283,
    -0.02216,
    0.01586,
    0.00315,
    0.00317
   ],
   [
    -0.00443,
    0.01453,
    -0.006,
    -0.00053,
    0.01665,
    0.00222,
    0.00088,
    -0.00504,
    0.01027,
    -0.00624,
    0.00117,
    0.0155,
    0.00411,
    0.00297
   ],
   [
    -0.01461,
    -0.00426,
    -0.01706,
    0.01237,
    0.00439,
    0.00165,
    -0.01336,
    -0.0165,
    -0.00639,
    -0.01425,
    0.00726,
    -0.00058,
    -0.00119,
    -0.01129
   ],
   [
    -0.00436,
    0.01696,
    -0.00084,
    -0.01577,
    -0.00242,
    0.01364,
    0.00509,
    -0.00317,
    0.01219,
    -0.00095,
    -0.01527,
    -0.00261,
    0.01039,
    0.0042
   ],
   [
    -0.01716,
    0.00414,
    -0.00685,
    -0.00416,
    0.00251,
    0.00906,
    0.00297,
    -0.01423,
    0.00024,
    -0.00409,
    -0.00331,
    0.00327,
    0.00869,
    0.00361
   ],
   [
    -0.00173,
    -0.01074,
    -0.00955,
    0.01423,
    -0.01213,
    -0.00045,
    -0.00176,
    -0.00032,
    -0.00726,
    -0.00812,
    0.01281,
    -0.00818,
    -0.00012,
    -0.00124
   ],
   [
    0.01598,
    0.01032,
    0.0017,
    0.01121,
    0.0012,
    0.00115,
    -0.00981,
    0.02084,
    0.00643,
    0.00251,
    0.00796,
    0.00036,
    -0.0004,
    -0.00849
   ],
   [
    -0.00494,
    0.00506,
    -0.0029,
    -0.0034,
    -0.00354,
    0.01132,
    -0.0003,
    -0.00083,
    0.00538,
    -0.00186,
    -0.00384,
    -0.00357,
    0.01054,
    0.00051
   ],
   [
    -0.02485,
    -0.00218,
    0.00843,
    -0.00616,
    0.01195,
    -0.01148,
    -0.00566,
    -0.01927,
    -0.00528,
    0.00528,
    -0.00672,
    0.01159,
    -0.00959,
    -0.00241
   ],
   [
    -0.0146,
    0.00982,
    0.00142,
    0.01508,
    0.01255,
    -0.00784,
    -0.00413,
    -0.01402,
    0.00319,
    -0.00101,
    0.01246,
    0.00938,
    -0.00852,
    -0.00313
   ],
   [
    0.01484,
    0.01076,
    0.00957,
    0.00553,
    0.0026,
    -0.01063,
    -0.01198,
    0.01455,
    0.00833,
    0.00684,
    0.00414,
    0.00114,
    -0.01066,
    -0.01215
   ],
   [
    -0.01346,
    -0.00347,
    -0.01056,
    0.00333,
    0.00421,
    -0.01145,
    0.01042,
    -0.01276,
    -0.00092,
    -0.00336,
    0.0064,
    0.00459,
    -0.00677,
    0.00846
   ],
   [
    -0.01916,
    -0.01014,
    -0.01198,
    -0.01332,
    7e-05,
    0.00078,
    0.00794,
    -0.01638,
    -0.00846,
    -0.01266,
    -0.01178,
    0.00077,
    -0.00033,
    0.0057
   ],
   [
    -0.02699,
    0.00966,
    -0.00551,
    -0.00713,
    0.0075,
    -0.00997,
    0.00271,
    -0.02204,
    0.00331,
    -0.01271,
    -0.00955,
    0.00492,
    -0.01101,
    0.00142
   ],
   [
    0.00063,
    -0.00656,
    -0.00811,
    -0.00443,
    0.00723,
    -0.00924,
    0.00493,
    -0.00023,
    -0.00135,
    -0.01017,
    -0.00603,
    0.00304,
    -0.00978,
    0.00013
   ],
   [
    0.02644,
    0.00473,
    -0.00122,
    4e-05,
    0.00213,
    -0.0021,
    -0.00054,
    0.02007,
    0.0052,
    0.00164,
    0.00123,
    0.00198,
    -0.00174,
    -0.00076
   ],
   [
    0.02462,
    0.00629,
    -0.00207,
    0.00287,
    -0.00544,
    -0.00038,
    0.01936,
    0.01946,
    0.01146,
    -0.00152,
    0.00433,
    -0.00418,
    -0.00104,
    0.01577
   ],
   [
    -0.00714,
    -0.01134,
    -0.00653,
    -0.00799,
    -0.00158,
    0.01125,
    0.00524,
    -0.0103,
    -0.01202,
    -0.00323,
    -0.00581,
    0.00118,
    0.01046,
    0.00449
   ],
   [
    0.01611,
    0.01143,
    0.00121,
    0.00511,
    -0.00167,
    -0.00613,
    0.00422,
    0.01617,
    0.00889,
    0.0005,
    0.0041,
    -0.00017,
    -0.00264,
    0.00396
   ],
   [
    -0.02276,
    0.01338,
    0.00454,
    -0.00436,
    -0.00228,
    0.00245,
    -0.01243,
    -0.02063,
    0.01243,
    0.00242,
    -0.00466,
    -0.00226,
    0.00279,
    -0.009
   ],
   [
    -0.00464,
    -0.02666,
    -0.01254,
    -0.01239,
    -0.01224,
    -0.00024,
    0.00967,
    -0.0039,
    -0.01837,
    -0.01303,
    -0.0147,
    -0.01176,
    -0.0028,
    0.00526
   ],
   [
    -0.02774,
    0.01655,
    -0.00072,
    0.00096,
    0.00821,
    0.00335,
    -0.00813,
    -0.0223,
    0.01239,
    0.00073,
    0.00318,
    0.00903,
    0.00442,
    -0.00669
   ],
   [
    -0.00032,
    0.00512,
    0.00647,
    -0.01621,
    0.00829,
    0.00326,
    -0.0038,
    0.00562,
    0.00731,
    0.00529,
    -0.01695,
    0.0058,
    0.00429,
    -0.00351
   ],
   [
    -0.01034,
    -0.00567,
    -0.00409,
    0.02239,
    0.00294,
    -0.01966,
    0.00115,
    -0.00916,
    -4e-05,
    -0.00597,
    0.02021,
    0.00054,
    -0.01931,
    -0.00067
   ],
   [
    0.00165,
    0.00131,
    0.00182,
    -0.00422,
    0.00642,
    0.00203,
    -0.0028,
    -0.0015,
    0.00057,
    0.00269,
    -0.00079,
    0.00886,
    0.00392,
    -0.00218
   ],
   [
    0.01183,
    0.01912,
    -0.00515,
    -0.01702,
    0.00762,
    0.00134,
    0.00049,
    0.00977,
    0.01818,
    -0.00404,
    -0.01639,
    0.0061,
    0.00162,
    0.00136
   ],
   [
    -0.00138,
    -0.00459,
    0.00103,
    0.0092,
    0.00658,
    -0.01222,
    -0.01198,
    -0.00258,
    -0.00991,
    -0.0019,
    0.00792,
    0.00282,
    -0.01033,
    -0.01012
   ],
   [
    0.00677,
    0.01797,
    0.00521,
    -0.00515,
    0.00457,
    -0.00139,
    0.00181,
    0.00708,
    0.0183,
    0.0051,
    -0.00438,
    0.00397,
    0.0008,
    0.00252
   ],
   [
    0.00579,
    -0.00624,
    -0.00397,
    0.00516,
    -0.00455,
    0.01182,
    0.00016,
    0.00115,
    -0.00679,
    -0.00727,
    0.00345,
    -0.0045,
    0.00802,
    -0.00236
   ],
   [
    0.0172,
    -0.00021,
    0.01781,
    -0.00415,
    -0.00414,
    0.00036,
    0.01549,
    0.01707,
    0.00275,
    0.01808,
    -0.00541,
    -0.00424,
    -0.0001,
    0.01207
   ],
   [
    0.00345,
    0.00089,
    0.00012,
    0.00018,
    -0.00481,
    0.00545,
    0.00452,
    0.00683,
    -0.00227,
    0.0007,
    -0.00329,
    -0.00559,
    0.00511,
    0.00335
   ],
   [
    0.01752,
    0.02008,
    -0.00103,
    -0.00089,
    -0.00551,
    0.00761,
    -0.00116,
    0.01344,
    0.01589,
    -0.00121,
    -0.0002,
    -0.00019,
    0.0106,
    0.00296
   ],
   [
    0.0069,
    -0.01281,
    0.01471,
    -0.00785,
    -0.00646,
    0.00453,
    0.00077,
    0.00771,
    -0.00715,
    0.01177,
    -0.00593,
    -0.00665,
    0.00506,
    0.00116
   ],
   [
    -0.01455,
    0.00884,
    -0.00089,
    -0.00938,
    -0.00256,
    -0.00053,
    0.00594,
    -0.01249,
    0.00438,
    -0.00272,
    -0.00845,
    -0.00172,
    0.0001,
    0.00398
   ],
   [
    0.008,
    -0.01738,
    -0.00842,
    0.00955,
    -0.00395,
    -0.01432,
    -0.01073,
    0.0037,
    -0.01384,
    -0.00508,
    0.01172,
    -0.00263,
    -0.01268,
    -0.00847
   ],
   [
    -0.01259,
    -0.01688,
    0.00928,
    0.01864,
    -0.00821,
    0.00035,
    -0.01894,
    -0.01004,
    -0.01095,
    0.00658,
    0.01736,
    -0.0075,
    -0.00264,
    -0.01582
   ],
   [
    -0.00839,
    0.01119,
    -0.00517,
    0.00124,
    0.00401,
    -0.00268,
    -0.01515,
    -0.00456,
    0.00673,
    -0.00476,
    0.00157,
    0.00191,
    -0.00312,
    -0.01226
   ],
   [
    0.0057,
    -0.00282,
    0.00264,
    0.01599,
    -0.01326,
    0.00129,
    0.0063,
    0.00852,
    0.0011,
    0.00176,
    0.01405,
    -0.01293,
    0.00058,
    0.00619
   ],
   [
    -0.01047,
    -0.00556,
    -0.01399,
    -0.02277,
    0.00422,
    0.00778,
    0.0217,
    -0.00989,
    0.00072,
    -0.00317,
    -0.02046,
    0.00379,
    0.00587,
    0.01784
   ],
   [
    0.01295,
    0.0152,
    -0.0086,
    0.01781,
    0.00447,
    -0.01228,
    -0.01152,
    0.00956,
    0.00954,
    -0.00669,
    0.01968,
    0.00422,
    -0.01002,
    -0.00884
   ],
   [
    0.00549,
    0.00056,
    0.00574,
    0.00488,
    -0.01222,
    0.00484,
    0.00235,
    0.00945,
    0.00092,
    -0.00278,
    0.00472,
    -0.01125,
    0.00509,
    0.00262
   ],
   [
    0.00323,
    0.02067,
    -0.00313,
    -0.00659,
    0.01179,
    0.00335,
    -0.01017,
    0.00434,
    0.0132,
    -0.00097,
    -0.00751,
    0.00724,
    -0.0008,
    -0.00928
   ],
   [
    0.01887,
    -0.00179,
    0.00499,
    0.01709,
    -0.00677,
    -0.00548,
    -0.0115,
    0.01643,
    -0.00344,
    0.00862,
    0.01771,
    -0.00087,
    -0.00193,
    -0.00731
   ],
   [
    -0.01723,
    -0.00724,
    -0.0061,
    0.01248,
    -0.00123,
    -0.00172,
    -0.00887,
    -0.01404,
    -0.00896,
    -0.00473,
    0.01121,
    -0.0014,
    -0.00071,
    -0.00809
   ],
   [
    0.02092,
    0.01423,
    0.00181,
    -0.0017,
    -0.00204,
    0.00439,
    -0.0009,
    0.01545,
    0.01435,
    0.00151,
    -0.00422,
    -0.00141,
    0.00601,
    2e-05
   ],
   [
    -0.00841,
    0.00096,
    0.00111,
    -0.00248,
    0.00064,
    -0.00494,
    -0.01001,
    -0.00667,
    -0.00664,
    -0.00122,
    -0.00167,
    0.00227,
    -0.00678,
    -0.00831
   ],
   [
    -0.01172,
    0.00305,
    -0.00803,
    -0.00284,
    -0.00197,
    -0.00417,
    -0.00729,
    -0.01019,
    -0.00204,
    -0.01294,
    -0.0036,
    -0.00024,
    -0.00295,
    -0.0055
   ],
   [
    0.00845,
    0.00255,
    -0.0018,
    -0.00067,
    -0.00474,
    -0.00452,
    0.01341,
    0.00526,
    0.00113,
    -0.00168,
    -0.0024,
    -0.00352,
    -0.00154,
    0.01089
   ],
   [
    -0.00119,
    0.00306,
    -0.00648,
    -0.00494,
    0.01029,
    -0.01392,
    0.00279,
    -0.00567,
    0.0021,
    -0.00913,
    -0.00473,
    0.00926,
    -0.01288,
    0.0006
   ],
   [
    0.00493,
    -0.01445,
    -0.00683,
    0.00198,
    -0.00791,
    -0.00475,
    -0.00791,
    0.00078,
    -0.01477,
    -0.00523,
    -0.00163,
    -0.00985,
    -0.00754,
    -0.00917
   ],
   [
    -0.00385,
    -0.00456,
    0.01006,
    0.00477,
    -0.00049,
    0.00399,
    0.00635,
    -0.00109,
    0.00248,
    0.01179,
    0.00726,
    9e-05,
    0.00418,
    0.00665
   ],
   [
    -0.00077,
    0.00694,
    -0.00273,
    -0.00216,
    -0.006,
    0.00258,
    -0.00019,
    0.00333,
    0.00286,
    -0.00269,
    -0.00205,
    -0.00469,
    0.00064,
    5e-05
   ],
   [
    -0.01151,
    0.00818,
    0.00496,
    0.00507,
    -0.01008,
    0.00492,
    0.00035,
    -0.00919,
    -0.00052,
    -0.00269,
    0.00383,
    -0.01078,
    0.00269,
    0.00126
   ],
   [
    -0.00447,
    -0.01272,
    0.00338,
    0.0034,
    -0.00575,
    -0.00419,
    0.00733,
    -0.00623,
    -0.01075,
    0.00339,
    0.00238,
    -0.00595,
    -0.00428,
    0.00601
   ],
   [
    -0.00602,
    0.00888,
    -0.00325,
    -0.02202,
    0.01479,
    0.00812,
    -0.0022,
    -0.00417,
    0.00554,
    -0.00405,
    -0.01868,
    0.01168,
    0.00865,
    -0.00187
   ],
   [
    0.01445,
    -0.00621,
    0.00062,
    0.00102,
    0.00443,
    -0.00064,
    0.00297,
    0.00941,
    -0.00803,
    -0.00648,
    -0.00046,
    0.00372,
    -0.00096,
    0.00259
   ],
   [
    0.00543,
    0.02502,
    0.00163,
    -0.00106,
    0.00033,
    -0.00329,
    0.0335,
    0.00912,
    0.02079,
    0.00367,
    0.00092,
    0.00041,
    -0.00164,
    0.03094
   ],
   [
    -0.00328,
    0.01585,
    -0.0023,
    0.00253,
    0.00469,
    -0.00374,
    -0.00162,
    -0.00187,
    0.01198,
    -0.00398,
    0.0023,
    0.00479,
    -0.00381,
    -0.00285
   ],
   [
    0.01922,
    -0.00682,
    0.00738,
    0.02622,
    -0.01282,
    -0.0081,
    -0.01082,
    0.01923,
    -0.00414,
    0.00644,
    0.02523,
    -0.0067,
    -0.00079,
    -0.00634
   ],
   [
    0.00511,
    0.02803,
    0.00151,
    -0.00692,
    0.01009,
    0.00874,
    -0.00728,
    0.00687,
    0.02051,
    0.00143,
    -0.00821,
    0.00748,
    0.00752,
    -0.00584
   ],
   [
    0.00322,
    0.00442,
    0.01446,
    -0.00938,
    -0.00543,
    0.01102,
    0.00306,
    0.00197,
    6e-05,
    0.01066,
    -0.01,
    -0.00329,
    0.01043,
    0.00402
   ],
   [
    0.01688,
    0.00842,
    0.02003,
    0.00415,
    0.00531,
    -0.00171,
    -7e-05,
    0.01776,
    0.00556,
    0.01514,
    0.0077,
    0.00509,
    0.00019,
    0.00304
   ],
   [
    -0.02384,
    0.0088,
    -0.00605,
    0.004,
    0.00535,
    -0.0127,
    -0.01065,
    -0.01977,
    0.00557,
    -0.0099,
    0.00294,
    0.00305,
    -0.0113,
    -0.00889
   ],
   [
    0.02204,
    -0.00031,
    -0.00846,
    0.00686,
    -0.00463,
    0.01308,
    -0.01827,
    0.01722,
    0.00408,
    -0.00242,
    0.00992,
    -0.00212,
    0.01427,
    -0.01345
   ],
   [
    -0.00549,
    -0.01206,
    0.00044,
    -0.00734,
    0.00443,
    0.00152,
    -0.00211,
    -0.00508,
    -0.00929,
    -0.00057,
    -0.01013,
    0.0019,
    -0.00174,
    -0.00402
   ],
   [
    -0.0041,
    -0.00468,
    -0.00976,
    0.00279,
    0.00108,
    -0.00648,
    0.01177,
    -0.00301,
    -0.00209,
    -0.00565,
    0.00183,
    0.00284,
    -0.0041,
    0.01043
   ],
   [
    0.01709,
    0.02107,
    0.0043,
    0.00894,
    -0.00462,
    -0.00772,
    -0.00458,
    0.00993,
    0.01155,
    0.00027,
    0.00573,
    -0.00504,
    -0.00764,
    -0.00473
   ],
   [
    0.00431,
    -0.02547,
    0.00401,
    0.00241,
    -0.01029,
    0.01158,
    0.01025,
    0.00077,
    -0.01652,
    0.00577,
    0.00109,
    -0.00907,
    0.0106,
    0.00944
   ],
   [
    0.00205,
    0.00247,
    0.00685,
    -0.003,
    0.00769,
    -0.00426,
    0.00178,
    0.00204,
    -0.00528,
    0.00038,
    -0.0067,
    0.00605,
    -0.00582,
    -0.0004
   ],
   [
    0.01679,
    -0.00697,
    0.01383,
    -0.00277,
    0.00266,
    -0.00659,
    -0.00166,
    0.01422,
    -0.00148,
    0.00763,
    -0.00043,
    0.0021,
    -0.0032,
    0.00016
   ],
   [
    -0.00883,
    0.00322,
    -0.00424,
    0.01033,
    0.00486,
    -0.00395,
    -0.0101,
    -0.01105,
    0.00056,
    0.00142,
    0.01189,
    0.00395,
    -0.00686,
    -0.00838
   ],
   [
    0.00794,
    0.01463,
    -0.00562,
    0.00525,
    -0.01472,
    0.00052,
    -0.0068,
    0.0096,
    0.00811,
    -0.00505,
    0.00532,
    -0.01217,
    0.00243,
    -0.004
   ],
   [
    0.01907,
    0.02487,
    0.00242,
    -0.00816,
    0.00841,
    0.00393,
    0.00456,
    0.01873,
    0.02243,
    0.00097,
    -0.00894,
    0.00792,
    0.00112,
    0.00163
   ],
   [
    0.01353,
    -0.03831,
    -0.00139,
    0.00472,
    -0.04635,
    0.00966,
    -0.00658,
    0.00827,
    -0.03453,
    -0.00444,
    0.00338,
    -0.04184,
    0.00538,
    -0.00953
   ],
   [
    -0.02664,
    -0.01736,
    0.00524,
    -0.00964,
    0.00164,
    -0.00099,
    0.00338,
    -0.0224,
    -0.01819,
    0.00332,
    -0.0085,
    0.00196,
    0.00138,
    0.00206
   ],
   [
    -0.00466,
    -0.00349,
    -0.00198,
    -0.00454,
    0.01186,
    -0.01192,
    -0.00751,
    -0.00681,
    -0.00371,
    -0.0074,
    -0.00519,
    0.00933,
    -0.01176,
    -0.00853
   ],
   [
    0.02206,
    0.00644,
    0.01646,
    -0.00175,
    -0.00241,
    -0.00388,
    0.00474,
    0.01861,
    0.01174,
    0.00873,
    0.00119,
    1e-05,
    -0.00363,
    0.00293
   ],
   [
    -0.00316,
    -0.00849,
    -0.00333,
    0.01057,
    0.00378,
    -0.00268,
    -0.02597,
    -0.00177,
    -0.00908,
    -0.00052,
    0.01169,
    0.0043,
    -0.00049,
    -0.02099
   ],
   [
    -0.02783,
    0.00616,
    -0.0033,
    -0.0065,
    0.01854,
    -0.01192,
    0.00512,
    -0.02601,
    0.00465,
    -0.00326,
    -0.00436,
    0.01437,
    -0.01047,
    0.00335
   ],
   [
    0.00774,
    -0.00539,
    -0.00508,
    -0.01044,
    -0.0055,
    0.01192,
    0.01293,
    0.00591,
    -0.00477,
    -0.00665,
    -0.01118,
    -0.00482,
    0.01029,
    0.00874
   ],
   [
    -0.01342,
    0.021,
    0.01198,
    0.00394,
    0.01147,
    0.0033,
    -0.00283,
    -0.00967,
    0.0157,
    0.00976,
    0.00573,
    0.01095,
    0.00683,
    0.00039
   ],
   [
    0.02292,
    0.01762,
    0.00637,
    0.01185,
    -0.0009,
    -0.01086,
    -0.00666,
    0.01791,
    0.0136,
    0.00353,
    0.01358,
    0.00023,
    -0.0105,
    -0.00635
   ],
   [
    0.00942,
    0.01786,
    0.00325,
    -0.00896,
    0.00916,
    -0.00421,
    -0.00658,
    0.0062,
    0.01475,
    0.00132,
    -0.00944,
    0.00555,
    -0.00805,
    -0.00753
   ],
   [
    0.00951,
    0.00297,
    -0.00721,
    -0.00045,
    0.0045,
    0.00244,
    -0.00068,
    0.0066,
    0.00332,
    -0.00412,
    -0.00064,
    0.00412,
    0.00131,
    -0.00308
   ],
   [
    0.00071,
    0.01394,
    0.00118,
    -0.00133,
    0.00831,
    -0.00392,
    0.00473,
    0.00071,
    0.01509,
    -0.00317,
    -0.00265,
    0.00678,
    -0.00403,
    0.00324
   ],
   [
    -0.0138,
    0.01959,
    -0.00317,
    -0.00555,
    0.01386,
    0.00464,
    0.00886,
    -0.00952,
    0.01813,
    -0.00094,
    -0.00494,
    0.01297,
    0.00547,
    0.00934
   ],
   [
    0.00893,
    0.00759,
    -0.01017,
    0.01308,
    -0.00615,
    -0.01548,
    -0.01375,
    0.00779,
    0.0062,
    -0.01064,
    0.01201,
    -0.00682,
    -0.01429,
    -0.01299
   ],
   [
    0.00869,
    0.03702,
    0.00391,
    0.00184,
    0.00543,
    -0.01073,
    0.01283,
    0.00921,
    0.03027,
    0.00315,
    0.00277,
    0.00396,
    -0.01093,
    0.01135
   ],
   [
    -0.00541,
    -0.0035,
    -0.00248,
    -0.00595,
    -0.00764,
    0.00204,
    0.01152,
    -0.00305,
    -0.00108,
    0.00558,
    -0.00761,
    -0.0067,
    0.00213,
    0.00865
   ],
   [
    -0.00152,
    0.00935,
    -0.00438,
    0.0025,
    -0.00383,
    0.01817,
    0.00633,
    0.00099,
    0.00839,
    0.00143,
    0.00066,
    -0.00283,
    0.01665,
    0.00532
   ],
   [
    0.01259,
    -0.01612,
    0.00665,
    -0.00103,
    -0.00435,
    0.00527,
    -0.00761,
    0.01326,
    -0.0127,
    0.00541,
    -0.00161,
    -0.00373,
    0.00467,
    -0.00784
   ],
   [
    -0.01643,
    -0.02386,
    -0.00288,
    0.0091,
    -0.00496,
    -0.01368,
    -0.00629,
    -0.01766,
    -0.021,
    -0.0062,
    0.00934,
    -0.00509,
    -0.01433,
    -0.00704
   ],
   [
    -0.01365,
    -0.00269,
    0.00625,
    0.00268,
    -0.00231,
    0.00458,
    0.00014,
    -0.01018,
    -0.00563,
    0.00283,
    0.00272,
    -0.0013,
    0.00716,
    0.00302
   ],
   [
    0.02475,
    -0.01929,
    0.01612,
    0.01646,
    -0.00126,
    -0.00129,
    -0.02126,
    0.01807,
    -0.01492,
    0.00877,
    0.01803,
    -0.00164,
    -0.00165,
    -0.01777
   ],
   [
    -0.00735,
    0.05283,
    -0.00216,
    -0.008,
    0.0219,
    -0.01452,
    0.0136,
    -0.00914,
    0.05469,
    0.01511,
    0.00096,
    0.02722,
    -0.00203,
    0.01852
   ],
   [
    0.00639,
    0.00115,
    0.00849,
    -0.00025,
    -0.01009,
    -0.0037,
    0.00544,
    0.00392,
    0.00088,
    0.00686,
    -0.00108,
    -0.00981,
    -0.00509,
    0.00312
   ],
   [
    0.03694,
    0.03142,
    0.00208,
    -0.00216,
    0.00528,
    -0.00446,
    -0.00832,
    0.03168,
    0.02992,
    0.0028,
    -0.00276,
    0.00542,
    -0.00336,
    -0.00737
   ],
   [
    0.01178,
    0.01201,
    0.00071,
    -0.00536,
    -0.00322,
    -0.00587,
    -0.00475,
    0.01061,
    0.01031,
    0.00301,
    -0.0097,
    -0.00596,
    -0.00434,
    -0.00375
   ],
   [
    0.00401,
    -0.01243,
    -0.00716,
    0.00784,
    -0.001,
    -0.00499,
    0.0074,
    0.00388,
    -0.01082,
    -0.00597,
    0.00358,
    -0.00391,
    -0.00666,
    0.00439
   ],
   [
    -0.00145,
    0.01776,
    0.00412,
    -0.00974,
    0.00831,
    -0.0028,
    0.01399,
    -0.00152,
    0.0174,
    0.00317,
    -0.00635,
    0.00999,
    -0.00254,
    0.01132
   ],
   [
    0.0098,
    0.00789,
    0.00508,
    0.00453,
    -0.00846,
    0.00524,
    0.00797,
    0.0096,
    0.00729,
    0.00654,
    0.00557,
    -0.00436,
    0.00698,
    0.00897
   ],
   [
    0.00698,
    -0.00225,
    0.00429,
    -0.03555,
    0.00051,
    0.01491,
    0.01136,
    0.00737,
    0.0033,
    0.0038,
    -0.03207,
    0.00176,
    0.01453,
    0.00937
   ],
   [
    -0.00063,
    0.00737,
    0.00304,
    -0.00736,
    0.00057,
    0.01272,
    0.00704,
    -0.00165,
    0.0049,
    0.0017,
    -0.00846,
    -0.00195,
    0.01155,
    0.00562
   ],
   [
    0.02621,
    0.02248,
    0.00433,
    -0.00305,
    0.01009,
    0.00069,
    0.00304,
    0.01999,
    0.0182,
    0.00544,
    -0.0017,
    0.00907,
    0.00096,
    0.00297
   ],
   [
    -0.00631,
    0.00828,
    -0.01199,
    -0.00542,
    0.00109,
    0.00344,
    -0.0051,
    -0.00653,
    0.00645,
    -0.0098,
    -0.00499,
    -0.00049,
    0.00486,
    -0.00219
   ],
   [
    -0.00906,
    0.00524,
    0.00649,
    0.00171,
    0.00102,
    0.00148,
    -0.00522,
    -0.00508,
    0.00536,
    0.00841,
    0.00318,
    0.00196,
    0.0024,
    -0.00331
   ],
   [
    0.00456,
    0.00023,
    0.00548,
    -0.00099,
    0.00615,
    0.0021,
    0.00271,
    -5e-05,
    0.00327,
    0.00369,
    -0.00211,
    0.00318,
    0.00047,
    -0.00024
   ],
   [
    -0.00512,
    0.01279,
    -0.00189,
    -0.00156,
    0.00061,
    -0.0026,
    -0.00685,
    -0.00655,
    0.00756,
    -0.00332,
    -0.0001,
    0.00281,
    -0.00662,
    -0.00763
   ],
   [
    -0.01054,
    -0.01004,
    -0.00569,
    -0.00022,
    -0.00053,
    0.00729,
    0.00343,
    -0.00903,
    -0.01059,
    -0.00558,
    -0.00033,
    -0.00056,
    0.00721,
    0.00423
   ],
   [
    -0.01269,
    -0.01229,
    0.00358,
    -0.0036,
    -0.01008,
    0.0013,
    0.00042,
    -0.00931,
    -0.01064,
    0.00594,
    -0.00326,
    -0.00809,
    0.00403,
    0.00299
   ],
   [
    0.00102,
    -0.00253,
    0.00696,
    0.00324,
    -0.00602,
    0.00908,
    -0.00067,
    0.00192,
    -0.00283,
    0.00448,
    0.00297,
    -0.00675,
    0.00805,
    0.00217
   ],
   [
    0.00148,
    0.00078,
    -0.00505,
    0.00203,
    0.01312,
    -0.00864,
    -0.0068,
    -0.00161,
    -0.00031,
    -0.00336,
    0.00466,
    0.01275,
    -0.00683,
    -0.00473
   ],
   [
    -0.00266,
    0.02603,
    0.01859,
    -0.01472,
    0.01454,
    -0.00235,
    0.01584,
    -0.00039,
    0.02323,
    0.01036,
    -0.01236,
    0.01736,
    0.00156,
    0.01681
   ],
   [
    -0.00386,
    0.00354,
    -0.01098,
    0.00565,
    0.00164,
    0.00248,
    0.00838,
    -0.00508,
    0.00441,
    -0.00462,
    0.00661,
    0.00243,
    0.00345,
    0.00776
   ],
   [
    0.00685,
    -0.01823,
    -0.00734,
    0.01557,
    -0.02282,
    0.00799,
    -0.00298,
    0.004,
    -0.02073,
    -0.01107,
    0.01396,
    -0.01792,
    0.0072,
    -0.00252
   ],
   [
    -0.00805,
    -0.00475,
    0.00783,
    -0.00157,
    -0.00809,
    0.00735,
    0.00913,
    -0.00873,
    -0.00376,
    0.0063,
    -0.00136,
    -0.00646,
    0.00638,
    0.00858
   ],
   [
    0.00564,
    -0.00239,
    -0.00716,
    0.00334,
    -0.00414,
    -0.01103,
    0.00197,
    0.00038,
    -0.00418,
    -0.00666,
    0.00306,
    -0.00713,
    -0.01196,
    6e-05
   ],
   [
    0.0261,
    0.0063,
    0.01194,
    0.01006,
    0.00296,
    -0.00843,
    -0.00156,
    0.02607,
    0.01116,
    0.00717,
    0.00952,
    0.00318,
    -0.00473,
    -0.00026
   ],
   [
    0.01907,
    0.0239,
    -0.00271,
    -0.01483,
    0.00314,
    0.00219,
    0.00088,
    0.01569,
    0.02209,
    -0.00072,
    -0.01414,
    0.00281,
    0.00288,
    0.00189
   ],
   [
    -0.01455,
    -0.00791,
    -0.00742,
    0.00178,
    0.00528,
    -0.00768,
    0.00424,
    -0.01434,
    -0.01155,
    -0.00422,
    0.0013,
    0.0037,
    -0.00786,
    0.00302
   ],
   [
    -0.01357,
    0.001,
    0.00095,
    -0.00136,
    -0.0066,
    0.01419,
    0.00436,
    -0.01269,
    0.00181,
    0.00731,
    1e-05,
    -0.00385,
    0.01255,
    0.00482
   ],
   [
    0.00872,
    0.00425,
    -0.00253,
    0.01159,
    0.00031,
    0.00049,
    -0.00964,
    0.00917,
    0.00224,
    0.00125,
    0.01218,
    0.00086,
    -0.00062,
    -0.00672
   ],
   [
    -0.01096,
    -0.02987,
    -0.00997,
    0.00124,
    -0.00984,
    0.00681,
    0.00421,
    -0.01227,
    -0.0248,
    -0.0088,
    0.00075,
    -0.01169,
    0.00281,
    0.00168
   ],
   [
    0.01729,
    -0.00061,
    0.0078,
    0.00863,
    -0.0121,
    -0.00617,
    -0.00615,
    0.01518,
    0.0033,
    0.01046,
    0.01082,
    -0.0114,
    -0.00447,
    -0.00573
   ],
   [
    0.00924,
    -0.00288,
    -0.00528,
    0.00251,
    0.00317,
    0.00466,
    -0.00052,
    0.00558,
    0.00181,
    -0.00617,
    0.00338,
    0.00433,
    0.00499,
    -0.00033
   ],
   [
    -0.01304,
    -0.00612,
    -0.00126,
    -0.00219,
    -0.00391,
    -0.00208,
    -0.00048,
    -0.01339,
    -0.00693,
    -0.00716,
    -0.00485,
    -0.00588,
    -0.00302,
    -0.00167
   ],
   [
    -0.00316,
    -0.00776,
    0.0058,
    -0.00382,
    -0.00938,
    0.00052,
    -0.00415,
    -0.00081,
    -0.00732,
    0.00297,
    -0.00148,
    -0.00682,
    0.00061,
    -0.00308
   ],
   [
    -0.01346,
    0.00699,
    0.00509,
    -0.01198,
    0.007,
    0.01627,
    -0.00193,
    -0.01019,
    0.0048,
    0.00373,
    -0.01632,
    0.00525,
    0.01336,
    -0.00116
   ],
   [
    -0.00486,
    0.02845,
    -0.00711,
    0.00732,
    0.00838,
    -0.00401,
    -0.00234,
    0.00022,
    0.02899,
    -0.00233,
    0.00614,
    0.00764,
    -0.0044,
    -0.00191
   ],
   [
    0.01636,
    -0.0066,
    0.00501,
    0.00145,
    -0.00378,
    0.00115,
    0.00425,
    0.01557,
    -0.00507,
    0.00276,
    0.00086,
    -0.00279,
    0.00284,
    0.00317
   ],
   [
    -0.03745,
    0.00426,
    -0.01663,
    0.01189,
    -0.01541,
    0.00103,
    -0.00615,
    -0.03481,
    0.00039,
    -0.00583,
    0.01518,
    -0.0117,
    0.00345,
    -0.00227
   ],
   [
    -0.01772,
    -0.00812,
    -0.01147,
    -0.00367,
    -0.0006,
    -0.00967,
    -0.00501,
    -0.015,
    -0.00539,
    -0.00666,
    0.00112,
    0.00286,
    -0.00416,
    -0.00075
   ],
   [
    -0.00156,
    -0.00369,
    -0.00877,
    0.00825,
    0.00148,
    0.00048,
    -0.00122,
    -0.00167,
    -0.00182,
    -0.00359,
    0.00688,
    0.00225,
    -0.00025,
    -0.00152
   ],
   [
    -0.01123,
    0.02113,
    -0.00592,
    -0.00672,
    0.01666,
    -0.02019,
    0.0163,
    -0.01048,
    0.02102,
    -0.00387,
    -0.00838,
    0.01325,
    -0.01532,
    0.01573
   ],
   [
    0.012,
    0.01575,
    0.00649,
    -0.00091,
    0.00746,
    0.00065,
    0.00372,
    0.01165,
    0.01057,
    0.00567,
    -0.00344,
    0.00395,
    -0.00153,
    0.00032
   ],
   [
    -0.00809,
    -0.01081,
    0.00643,
    0.01565,
    0.00019,
    -0.00761,
    -0.02327,
    -0.01051,
    -0.01353,
    0.00067,
    0.01496,
    0.00062,
    -0.00818,
    -0.01993
   ],
   [
    -0.00658,
    0.00642,
    0.0027,
    -0.00852,
    -0.00703,
    0.01404,
    -0.00051,
    -0.00679,
    0.00921,
    0.00762,
    -0.00608,
    -0.00618,
    0.01294,
    0.00059
   ],
   [
    -0.01136,
    0.00298,
    0.00203,
    -0.01874,
    0.01738,
    0.00761,
    0.00114,
    -0.00686,
    0.00087,
    0.00717,
    -0.01697,
    0.01693,
    0.00859,
    0.00227
   ],
   [
    0.01118,
    -0.0064,
    0.00275,
    -0.01033,
    -0.00143,
    0.01512,
    -0.00146,
    0.00872,
    -0.00801,
    0.0005,
    -0.00965,
    -0.00204,
    0.01307,
    0.00022
   ],
   [
    0.00756,
    0.00412,
    0.0067,
    -0.01088,
    -0.00426,
    0.00819,
    0.01444,
    0.00808,
    0.00437,
    0.00519,
    -0.01079,
    -0.00379,
    0.00687,
    0.01191
   ],
   [
    -0.00855,
    -0.01185,
    -0.01668,
    0.01979,
    -0.01049,
    -0.00217,
    -0.02041,
    -0.00933,
    -0.01226,
    -0.01379,
    0.01621,
    -0.01221,
    -0.00368,
    -0.01802
   ],
   [
    0.00839,
    -0.02369,
    -0.00132,
    0.00979,
    -0.00511,
    -0.0161,
    0.00396,
    0.00691,
    -0.0146,
    -0.0035,
    0.00876,
    -0.00501,
    -0.01668,
    0.00152
   ],
   [
    0.00281,
    -0.00182,
    -0.00698,
    0.01457,
    0.01744,
    -0.03279,
    -0.01979,
    0.00107,
    -0.00579,
    -0.00873,
    0.01341,
    0.01282,
    -0.03071,
    -0.0178
   ],
   [
    -0.01842,
    0.00853,
    0.00366,
    0.01081,
    -0.00806,
    -0.00628,
    -0.00249,
    -0.01315,
    -0.00107,
    -0.00024,
    0.00802,
    -0.00779,
    -0.00894,
    -0.00154
   ],
   [
    -0.00087,
    -0.00459,
    -0.01407,
    -0.00517,
    0.00258,
    -0.00239,
    0.00466,
    -0.00041,
    -0.00578,
    -0.01309,
    -0.00449,
    0.00236,
    -0.00353,
    0.0023
   ],
   [
    0.00791,
    -0.00181,
    -0.01657,
    -0.00696,
    0.00203,
    0.00215,
    0.00438,
    0.00571,
    -0.00129,
    -0.01203,
    -0.00812,
    0.00091,
    0.00111,
    0.00348
   ],
   [
    0.01055,
    0.0122,
    -0.0005,
    -0.00054,
    0.01059,
    -0.00288,
    -0.02335,
    0.00771,
    0.01326,
    -0.00502,
    -0.00469,
    0.00728,
    -0.00271,
    -0.01952
   ],
   [
    -0.00647,
    -0.0172,
    -0.0016,
    0.001,
    -0.01319,
    0.00583,
    0.00038,
    -0.00786,
    -0.01192,
    0.00166,
    0.00283,
    -0.01019,
    0.00506,
    0.00181
   ],
   [
    -0.00086,
    0.00281,
    0.01134,
    0.0041,
    0.00339,
    -0.00692,
    0.01011,
    -0.0002,
    0.00065,
    0.00386,
    0.00546,
    0.00418,
    -0.00407,
    0.00922
   ],
   [
    0.01224,
    0.00189,
    -0.00476,
    -0.00082,
    -0.00067,
    -0.00581,
    0.00856,
    0.00596,
    -0.0011,
    -0.00323,
    -0.00037,
    0.00032,
    -0.00516,
    0.00756
   ],
   [
    -0.02136,
    0.01718,
    -0.00398,
    -0.00309,
    0.0001,
    0.00433,
    0.00061,
    -0.01925,
    0.01552,
    -0.00453,
    -0.00395,
    0.00065,
    0.00394,
    0.00062
   ],
   [
    0.0027,
    0.01218,
    0.00612,
    -0.00591,
    0.00651,
    0.00288,
    0.00367,
    0.00614,
    0.0134,
    0.00587,
    -0.00298,
    0.00602,
    0.00107,
    0.00136
   ],
   [
    0.00269,
    0.00125,
    0.00511,
    0.00526,
    -0.00241,
    0.00473,
    -0.00972,
    0.00834,
    0.00105,
    0.00686,
    0.00343,
    -0.00469,
    0.00134,
    -0.00837
   ],
   [
    0.00275,
    -0.00455,
    0.00191,
    -0.00138,
    -0.01014,
    0.00849,
    0.00313,
    0.00504,
    -0.00497,
    0.00521,
    -0.00219,
    -0.00842,
    0.01059,
    0.00577
   ],
   [
    0.00513,
    0.00455,
    0.00901,
    0.0103,
    0.00166,
    0.00171,
    -0.01333,
    0.00539,
    0.0056,
    0.01075,
    0.01055,
    0.00136,
    0.00401,
    -0.01011
   ],
   [
    0.02295,
    -0.00347,
    0.00384,
    0.01082,
    -0.01669,
    -0.00433,
    0.00115,
    0.01934,
    -0.00325,
    -0.00099,
    0.00544,
    -0.01813,
    -0.00857,
    -0.00109
   ],
   [
    0.01147,
    0.00097,
    -0.0011,
    0.00599,
    -0.02194,
    0.01027,
    0.01418,
    0.01003,
    -0.00149,
    -0.0054,
    0.00298,
    -0.01843,
    0.00933,
    0.01275
   ],
   [
    -0.01861,
    0.01403,
    -0.00152,
    -0.00531,
    0.01454,
    -0.00447,
    0.00614,
    -0.01349,
    0.00572,
    -0.002,
    -0.00423,
    0.01381,
    -0.00332,
    0.00739
   ],
   [
    0.00636,
    -0.00085,
    0.00752,
    0.00901,
    -0.00026,
    0.00244,
    -0.00494,
    0.00407,
    -0.00244,
    0.00112,
    0.00658,
    -0.00173,
    0.00013,
    -0.00578
   ],
   [
    -0.00242,
    0.01127,
    -0.00658,
    0.00084,
    0.00399,
    0.00507,
    -0.00177,
    2e-05,
    0.00868,
    -0.00508,
    0.00104,
    0.00348,
    0.00187,
    -0.00266
   ],
   [
    0.01219,
    0.00159,
    -0.00682,
    0.00091,
    -0.00628,
    0.00429,
    0.0132,
    0.00958,
    0.00174,
    -0.00643,
    0.00094,
    -0.00572,
    0.00319,
    0.01131
   ],
   [
    0.00324,
    0.00193,
    -0.0062,
    -0.00909,
    0.00074,
    -0.00353,
    0.0066,
    0.00191,
    0.00706,
    -0.00459,
    -0.00772,
    0.00047,
    -0.00225,
    0.00578
   ],
   [
    -0.00067,
    -0.00228,
    0.00348,
    -0.01057,
    0.01034,
    0.0192,
    -0.01848,
    5e-05,
    -0.00254,
    0.00448,
    -0.012,
    0.0099,
    0.01771,
    -0.01453
   ],
   [
    0.00235,
    -0.00039,
    -0.00506,
    -0.00782,
    0.0036,
    0.00954,
    0.00359,
    0.00042,
    -0.00181,
    -0.00209,
    -0.00772,
    0.00265,
    0.0087,
    0.00253
   ],
   [
    -0.00445,
    -0.0176,
    -0.00054,
    -0.00148,
    -0.01186,
    -0.01182,
    0.00761,
    -0.00613,
    -0.01485,
    -0.00312,
    -0.00259,
    -0.01151,
    -0.01032,
    0.00505
   ],
   [
    0.00415,
    -0.01598,
    -0.00485,
    0.00369,
    -0.01554,
    -0.00414,
    -0.01075,
    0.00106,
    -0.01253,
    -0.00894,
    0.00284,
    -0.01409,
    -0.00665,
    -0.01378
   ],
   [
    -0.01143,
    -0.00309,
    0.00949,
    -0.00066,
    0.00582,
    0.00611,
    -0.00688,
    -0.009,
    -0.00297,
    0.00724,
    -0.00114,
    0.0034,
    0.00593,
    -0.00574
   ],
   [
    0.00656,
    0.01205,
    -0.01199,
    -0.00044,
    -0.01344,
    0.01026,
    0.00877,
    0.00603,
    0.00944,
    -0.01055,
    -0.00111,
    -0.00981,
    0.01072,
    0.01094
   ],
   [
    0.00524,
    0.00223,
    -0.00295,
    -0.01133,
    0.00198,
    0.01188,
    0.00402,
    0.00537,
    0.00279,
    -0.00026,
    -0.01151,
    0.00072,
    0.01306,
    0.00458
   ],
   [
    -0.03601,
    -0.01449,
    0.01719,
    0.00039,
    -0.01582,
    0.01945,
    0.01365,
    -0.0264,
    -0.01168,
    0.01544,
    -0.00095,
    -0.01289,
    0.01503,
    0.0104
   ],
   [
    -0.00815,
    0.00896,
    -0.00299,
    0.01425,
    0.00518,
    -0.01019,
    -0.02374,
    -0.00777,
    0.00451,
    -0.00392,
    0.01514,
    0.00401,
    -0.00735,
    -0.01875
   ],
   [
    0.00544,
    -0.00642,
    -0.01003,
    0.01009,
    -0.00125,
    -0.01194,
    0.00174,
    0.00094,
    -0.00614,
    -0.00383,
    0.00766,
    -0.00369,
    -0.01331,
    0.00048
   ],
   [
    0.00868,
    0.00187,
    -0.00086,
    0.01545,
    -0.00086,
    0.00146,
    -0.00797,
    0.00845,
    0.00553,
    -0.00209,
    0.01346,
    -0.00263,
    0.00071,
    -0.00846
   ],
   [
    -0.00645,
    0.00465,
    -0.00675,
    0.00243,
    0.00625,
    -0.00074,
    0.00294,
    -0.00487,
    0.00546,
    -0.00557,
    0.00309,
    0.00536,
    0.00099,
    0.00304
   ],
   [
    0.01041,
    0.01464,
    0.00568,
    -0.00231,
    0.00723,
    -0.00676,
    0.00164,
    0.01021,
    0.01396,
    0.00558,
    -0.00186,
    0.00464,
    -0.00556,
    0.0013
   ],
   [
    0.01035,
    0.00126,
    -0.0069,
    0.01785,
    -0.004,
    -0.01196,
    -0.00143,
    0.00906,
    0.00053,
    -0.00752,
    0.01615,
    -0.0024,
    -0.00862,
    0.0009
   ],
   [
    0.00557,
    -0.00585,
    0.00548,
    0.00472,
    -0.007,
    -0.00375,
    0.00395,
    0.00432,
    -0.00648,
    0.00576,
    0.00255,
    -0.00605,
    -0.00414,
    0.00092
   ],
   [
    -0.01152,
    0.01189,
    0.00026,
    0.0038,
    0.00475,
    -0.00109,
    -0.00468,
    -0.01095,
    0.00625,
    -0.00438,
    0.00301,
    0.00138,
    -0.00373,
    -0.00528
   ],
   [
    -0.00253,
    0.00194,
    0.00529,
    -0.00306,
    0.00042,
    0.01088,
    0.00214,
    0.00284,
    0.00371,
    0.00593,
    -0.00378,
    0.00074,
    0.01005,
    0.00226
   ],
   [
    -0.01349,
    -0.00323,
    -0.00158,
    0.00278,
    -0.00155,
    0.00468,
    -0.01114,
    -0.01189,
    -0.00237,
    -0.00148,
    0.00415,
    -0.00227,
    0.0035,
    -0.01091
   ],
   [
    0.00321,
    -0.00062,
    -0.00209,
    -0.01345,
    -0.01073,
    0.00841,
    0.01676,
    -0.0003,
    -0.00022,
    -0.00189,
    -0.00944,
    -0.00931,
    0.0102,
    0.01464
   ],
   [
    -0.00045,
    0.0116,
    -0.00507,
    0.00093,
    0.01098,
    -0.00409,
    -0.00912,
    -0.00281,
    0.01294,
    -0.00501,
    0.00299,
    0.01011,
    -0.00426,
    -0.0098
   ],
   [
    -0.00244,
    -0.01344,
    -0.00259,
    0.00168,
    -0.00394,
    0.00046,
    -0.00374,
    -0.00075,
    -0.00635,
    0.00058,
    -0.00204,
    -0.00557,
    -0.00068,
    -0.00434
   ],
   [
    -0.02219,
    0.00914,
    0.0222,
    0.00092,
    -0.00243,
    0.00213,
    -0.00189,
    -0.01738,
    0.00498,
    0.01357,
    -0.00325,
    -0.00468,
    -0.00157,
    -0.00305
   ],
   [
    -0.02669,
    -0.00258,
    0.00304,
    -0.00269,
    0.00275,
    -0.00369,
    -0.00937,
    -0.02429,
    -0.00777,
    0.00107,
    -0.00512,
    0.00068,
    -0.00401,
    -0.00995
   ],
   [
    -0.00367,
    -0.01906,
    0.00393,
    0.0053,
    -0.017,
    0.0093,
    -0.0192,
    -0.00426,
    -0.01535,
    0.00283,
    0.00419,
    -0.01422,
    0.0087,
    -0.01595
   ],
   [
    -0.00168,
    0.02919,
    0.00077,
    0.00876,
    0.01974,
    0.00576,
    -0.01247,
    -0.0021,
    0.02674,
    0.00276,
    0.00882,
    0.01425,
    0.00333,
    -0.01098
   ],
   [
    -0.00205,
    0.00522,
    -0.00976,
    -0.00782,
    0.00854,
    -0.01,
    -0.00022,
    -0.00294,
    0.00746,
    -0.00698,
    -0.00867,
    0.00541,
    -0.00897,
    -0.00102
   ],
   [
    0.01186,
    -0.01034,
    0.00287,
    -0.01016,
    -0.01693,
    0.01628,
    0.01614,
    0.00742,
    -0.00944,
    -0.00273,
    -0.00959,
    -0.01456,
    0.01172,
    0.01205
   ],
   [
    0.00944,
    0.01014,
    0.00621,
    -0.0123,
    0.00315,
    0.00406,
    -0.00026,
    0.00794,
    0.00732,
    0.00361,
    -0.01362,
    0.00111,
    0.00333,
    -0.00171
   ],
   [
    -0.00037,
    -0.01267,
    -0.00512,
    0.00162,
    -0.00334,
    -0.00818,
    -0.00386,
    0.00114,
    -0.01151,
    0.00059,
    -0.00035,
    -0.00439,
    -0.00821,
    -0.00438
   ],
   [
    0.001,
    0.00114,
    -0.00441,
    0.00751,
    -0.00913,
    0.0036,
    -0.001,
    0.00203,
    0.00433,
    -0.00206,
    0.00746,
    -0.0069,
    0.00575,
    0.00182
   ],
   [
    -0.0104,
    -0.0014,
    -0.00299,
    -0.00966,
    -0.00815,
    0.00752,
    0.00586,
    -0.01301,
    -0.00552,
    -0.00241,
    -0.01013,
    -0.00459,
    0.00651,
    0.00318
   ],
   [
    0.0175,
    0.00145,
    0.00372,
    -0.00404,
    -0.01697,
    0.01453,
    0.00272,
    0.01472,
    0.00332,
    0.00608,
    -0.00271,
    -0.01461,
    0.01075,
    0.00093
   ],
   [
    0.01138,
    0.00196,
    -0.00489,
    0.01678,
    0.00418,
    -0.01274,
    -0.0063,
    0.0063,
    -6e-05,
    -0.00297,
    0.01396,
    0.00474,
    -0.01224,
    -0.00586
   ],
   [
    0.01696,
    0.02017,
    -0.00317,
    -0.00375,
    -0.00246,
    -0.00128,
    0.01234,
    0.01209,
    0.0142,
    -0.00343,
    -0.00057,
    -0.0011,
    0.00039,
    0.01147
   ],
   [
    0.01282,
    -0.01634,
    0.00338,
    -0.00534,
    -0.01792,
    0.02208,
    0.00297,
    0.01236,
    -0.00893,
    0.00515,
    -0.00563,
    -0.01813,
    0.01667,
    0.00126
   ],
   [
    -0.00085,
    -0.00302,
    0.00824,
    0.00086,
    0.00099,
    0.00723,
    -0.00847,
    0.00117,
    -0.00675,
    0.00414,
    -0.00228,
    -0.00032,
    0.00578,
    -0.00588
   ],
   [
    -0.00111,
    0.01409,
    -0.00331,
    0.00447,
    0.00296,
    0.00124,
    0.01453,
    0.00242,
    0.01201,
    -0.00061,
    0.00284,
    0.00431,
    0.00351,
    0.01628
   ],
   [
    0.00049,
    -0.00768,
    0.00562,
    0.01308,
    -0.00505,
    -0.00733,
    -0.00778,
    0.0025,
    -0.00263,
    0.00461,
    0.01338,
    -0.00392,
    -0.00794,
    -0.00569
   ],
   [
    0.0098,
    -0.00155,
    0.00395,
    0.00952,
    -0.00023,
    -0.00813,
    -0.00994,
    0.00866,
    -0.00027,
    0.00233,
    0.01037,
    -0.00152,
    -0.00792,
    -0.01019
   ],
   [
    0.00873,
    0.00657,
    -0.00259,
    -0.00935,
    0.00167,
    0.00901,
    0.00614,
    0.0044,
    0.00229,
    0.0006,
    -0.00828,
    0.00186,
    0.0094,
    0.00784
   ],
   [
    0.01394,
    -0.00255,
    0.01382,
    0.00871,
    0.00503,
    -0.00733,
    0.0045,
    0.0129,
    0.00146,
    0.01352,
    0.00912,
    0.00544,
    -0.0034,
    0.00589
   ],
   [
    0.01518,
    -0.00721,
    0.0105,
    -0.00359,
    -0.00835,
    0.00338,
    0.0125,
    0.01295,
    -0.00386,
    0.01108,
    -0.00137,
    -0.00661,
    0.00366,
    0.01118
   ],
   [
    0.00987,
    -0.0052,
    -0.00143,
    0.00032,
    0.00252,
    0.0007,
    -0.00062,
    0.00738,
    -0.00235,
    -0.00328,
    0.00016,
    0.00214,
    0.00051,
    0.00032
   ],
   [
    -0.01116,
    -0.0031,
    0.00637,
    -0.00381,
    0.0019,
    0.00333,
    -0.00999,
    -0.00784,
    -0.00572,
    0.0002,
    -0.00457,
    0.0008,
    0.00231,
    -0.00973
   ],
   [
    -0.00576,
    0.0058,
    -0.01125,
    0.00363,
    0.00951,
    -0.00316,
    -0.01958,
    -0.00236,
    -0.00131,
    -0.00823,
    0.00186,
    0.00806,
    -0.00294,
    -0.01765
   ],
   [
    0.01218,
    0.00904,
    -0.0055,
    0.00889,
    0.00804,
    -0.00864,
    -0.00368,
    0.01037,
    0.00767,
    -0.00593,
    0.00501,
    0.00451,
    -0.009,
    -0.00349
   ],
   [
    -0.01279,
    -0.00133,
    -0.00638,
    0.00917,
    0.01119,
    -0.0155,
    -0.0113,
    -0.01032,
    -0.00398,
    -0.00955,
    0.00449,
    0.00602,
    -0.01975,
    -0.0135
   ],
   [
    0.02696,
    -0.03017,
    -0.00277,
    0.01055,
    -0.0119,
    -0.00693,
    -0.003,
    0.02088,
    -0.02505,
    -0.00228,
    0.01061,
    -0.00923,
    -0.00126,
    -0.00055
   ],
   [
    0.01658,
    -0.0169,
    0.00314,
    0.00379,
    -0.00838,
    0.01073,
    0.00557,
    0.01102,
    -0.01505,
    0.00068,
    0.00163,
    -0.00758,
    0.0076,
    0.00336
   ],
   [
    -0.01146,
    0.00289,
    -0.00395,
    0.00609,
    -0.0091,
    0.00056,
    0.00142,
    -0.00783,
    0.00179,
    -0.00512,
    0.00322,
    -0.00995,
    -0.00261,
    0.00067
   ],
   [
    0.01114,
    -0.01829,
    -0.00454,
    0.00188,
    -0.02109,
    0.00446,
    0.00197,
    0.01136,
    -0.01263,
    -0.00157,
    0.00072,
    -0.01606,
    0.00381,
    0.00299
   ],
   [
    0.01331,
    -0.01563,
    0.01246,
    0.01536,
    -0.01006,
    -0.00848,
    -0.00178,
    0.00977,
    -0.01449,
    0.01222,
    0.01457,
    -0.00992,
    -0.00804,
    -0.00059
   ],
   [
    0.01119,
    -5e-05,
    -0.01037,
    -0.0016,
    -0.00243,
    0.0074,
    0.00258,
    0.00842,
    0.00304,
    -0.00697,
    -0.00332,
    -0.00275,
    0.00556,
    0.00038
   ],
   [
    -0.01127,
    -0.00764,
    0.00546,
    0.00133,
    0.01417,
    -0.01666,
    -0.01021,
    -0.01187,
    0.00058,
    0.00449,
    0.00238,
    0.01043,
    -0.01792,
    -0.01011
   ],
   [
    0.00571,
    -0.02137,
    -0.00445,
    0.00983,
    -0.01259,
    -0.0006,
    0.00189,
    0.00326,
    -0.02236,
    -0.00691,
    0.00785,
    -0.00964,
    0.00118,
    0.0021
   ],
   [
    0.00358,
    -0.01856,
    0.01999,
    -0.01058,
    -0.00106,
    0.0021,
    -0.00996,
    0.00307,
    -0.01629,
    0.01203,
    -0.01016,
    -0.00205,
    0.00278,
    -0.00679
   ],
   [
    0.004,
    -0.00565,
    -0.00327,
    0.00846,
    -0.00559,
    8e-05,
    0.01115,
    0.00325,
    -0.00409,
    0.00087,
    0.00863,
    -0.00499,
    -0.00053,
    0.00963
   ],
   [
    -0.00158,
    0.01964,
    0.00846,
    0.0076,
    -0.00709,
    0.00449,
    -0.00359,
    0.00015,
    0.01557,
    0.01028,
    0.00612,
    -0.00556,
    0.00481,
    -0.00161
   ],
   [
    -0.00677,
    -0.00804,
    0.00092,
    0.00581,
    0.00294,
    -0.01536,
    0.00192,
    -0.00348,
    -0.00774,
    -0.00362,
    0.00459,
    0.00224,
    -0.01456,
    0.00054
   ],
   [
    0.00136,
    0.00464,
    0.00614,
    0.00326,
    -0.00241,
    0.00261,
    0.00531,
    -7e-05,
    0.00256,
    0.00473,
    0.00229,
    -0.00262,
    -0.00195,
    0.00204
   ],
   [
    -0.00151,
    -0.00267,
    0.00047,
    0.00498,
    -0.00165,
    -0.00119,
    0.01278,
    -0.00154,
    0.00358,
    0.00114,
    0.00723,
    0.00045,
    0.00081,
    0.0131
   ],
   [
    -0.02351,
    -0.02134,
    -0.00841,
    0.00642,
    0.00164,
    -0.01315,
    -0.01559,
    -0.0193,
    -0.02038,
    -0.01269,
    0.00423,
    -0.00048,
    -0.01079,
    -0.01316
   ],
   [
    -0.01872,
    -0.01619,
    -0.00145,
    -0.00884,
    -0.01015,
    0.00722,
    -0.00197,
    -0.01454,
    -0.00832,
    -0.00355,
    -0.00622,
    -0.00778,
    0.00804,
    -0.00138
   ],
   [
    -0.00836,
    0.00463,
    -0.00377,
    -0.00613,
    0.01064,
    -0.00221,
    -0.00105,
    -0.00339,
    -0.0002,
    -0.00338,
    -0.00468,
    0.00952,
    -0.00019,
    0.00081
   ],
   [
    -0.00901,
    -0.01796,
    -0.00043,
    0.00241,
    0.00101,
    0.00166,
    -0.00474,
    -0.00638,
    -0.0094,
    0.00189,
    0.0035,
    0.00165,
    0.00297,
    -0.00278
   ],
   [
    0.01171,
    0.01716,
    0.01239,
    0.01338,
    0.0011,
    -0.01125,
    -0.00624,
    0.01111,
    0.01223,
    0.0097,
    0.01297,
    0.00111,
    -0.00966,
    -0.00437
   ],
   [
    0.01254,
    0.01231,
    0.00188,
    -0.00894,
    -0.00409,
    -0.00305,
    0.02286,
    0.00832,
    0.01404,
    0.00383,
    -0.00617,
    -0.00314,
    -0.0002,
    0.02206
   ],
   [
    -0.00529,
    -0.00506,
    -0.00993,
    -0.0071,
    -0.00331,
    0.01402,
    0.01253,
    -0.00184,
    -0.00385,
    -0.00687,
    -0.00758,
    -0.0014,
    0.01334,
    0.01105
   ],
   [
    0.0054,
    0.02169,
    0.00361,
    -0.00851,
    0.01326,
    -0.00174,
    0.00619,
    0.00571,
    0.01749,
    0.00435,
    -0.00733,
    0.01278,
    0.00079,
    0.00746
   ],
   [
    -0.02438,
    0.0044,
    0.00274,
    -0.01119,
    0.01537,
    0.00995,
    -0.00718,
    -0.02028,
    0.00123,
    0.00637,
    -0.00949,
    0.01405,
    0.00771,
    -0.00578
   ],
   [
    -0.00899,
    -0.00551,
    0.00771,
    -0.01062,
    0.00375,
    0.00904,
    -0.00769,
    -0.00697,
    -0.00605,
    0.00527,
    -0.01017,
    0.00224,
    0.00421,
    -0.00956
   ],
   [
    -0.01136,
    -0.00418,
    -0.00472,
    -0.00606,
    -0.00467,
    0.00425,
    -0.0009,
    -0.01066,
    -0.00239,
    -0.00288,
    -0.00602,
    -0.00383,
    0.00404,
    -0.00243
   ],
   [
    -0.00804,
    -0.01085,
    -0.00266,
    -0.01495,
    -0.00339,
    0.01164,
    0.01222,
    -0.00539,
    -0.00355,
    -0.00342,
    -0.0148,
    -0.00393,
    0.00776,
    0.00873
   ],
   [
    0.0063,
    -0.01009,
    -0.00176,
    -0.00362,
    0.00874,
    -0.00538,
    0.00259,
    0.00378,
    -0.00556,
    -0.00085,
    -0.00286,
    0.00457,
    -0.0051,
    0.00038
   ],
   [
    0.00506,
    0.00551,
    0.00403,
    0.0009,
    0.03189,
    -0.01614,
    -0.02258,
    0.00226,
    0.00353,
    0.00262,
    -0.00134,
    0.02493,
    -0.0173,
    -0.02126
   ],
   [
    -0.00392,
    -0.00092,
    0.00945,
    0.00167,
    -0.00777,
    0.00978,
    0.00037,
    -0.00246,
    0.0004,
    0.00635,
    0.00247,
    -0.00766,
    0.00569,
    -0.00259
   ],
   [
    0.00814,
    -0.00347,
    0.00515,
    -0.0028,
    -0.01258,
    0.00822,
    -0.00245,
    0.00437,
    -0.00328,
    0.00861,
    -0.00251,
    -0.00855,
    0.00555,
    -0.00318
   ],
   [
    0.02523,
    0.01755,
    0.00349,
    0.01288,
    -0.00288,
    -0.00823,
    0.00562,
    0.02391,
    0.01461,
    0.00556,
    0.01058,
    -0.00286,
    -0.00987,
    0.00365
   ],
   [
    -0.01603,
    0.0124,
    -0.01041,
    0.00433,
    0.01084,
    -0.00277,
    -0.00322,
    -0.01397,
    0.00601,
    -0.01188,
    0.004,
    0.01082,
    -0.00283,
    -0.0006
   ],
   [
    0.00899,
    -0.00261,
    -0.01297,
    -0.00611,
    0.00485,
    0.00041,
    0.00552,
    0.0089,
    -0.0002,
    -0.01101,
    -0.00371,
    0.00437,
    3e-05,
    0.00483
   ],
   [
    0.00977,
    -0.00951,
    -0.00156,
    0.01008,
    0.00227,
    -0.01334,
    -0.01365,
    0.00742,
    -0.01075,
    -0.00155,
    0.00824,
    0.00024,
    -0.01183,
    -0.01236
   ],
   [
    0.00355,
    -0.00551,
    0.00588,
    -0.01418,
    0.00182,
    0.00055,
    0.02269,
    0.00316,
    0.00113,
    0.00679,
    -0.012,
    0.0022,
    0.00187,
    0.01992
   ],
   [
    0.0074,
    0.00625,
    -0.00459,
    -0.00469,
    -0.0088,
    -0.00544,
    0.0111,
    0.00639,
    0.0059,
    -0.00579,
    -0.0094,
    -0.00781,
    -0.00569,
    0.00905
   ],
   [
    0.01744,
    0.00511,
    -0.00283,
    0.00023,
    0.00099,
    -0.00275,
    0.00372,
    0.01572,
    0.00693,
    0.00262,
    0.00241,
    0.00244,
    -0.00259,
    0.0023
   ],
   [
    -0.00578,
    -0.00623,
    -0.00837,
    -0.00026,
    -0.0148,
    0.00703,
    0.00714,
    -0.00436,
    -0.00511,
    -0.00343,
    0.00257,
    -0.01199,
    0.00435,
    0.0054
   ],
   [
    0.01319,
    -0.00124,
    -0.00915,
    -0.00296,
    0.00305,
    -0.00625,
    -0.00454,
    0.00983,
    -0.00394,
    -0.00881,
    -0.00079,
    0.00343,
    -0.0071,
    -0.00504
   ],
   [
    -0.0095,
    0.00286,
    0.0054,
    0.00387,
    -0.00715,
    0.00867,
    0.01495,
    -0.00669,
    0.00322,
    0.00689,
    0.00351,
    -0.00483,
    0.009,
    0.01489
   ],
   [
    0.02551,
    0.02075,
    -0.01199,
    0.0025,
    0.00219,
    0.00153,
    -0.00338,
    0.01971,
    0.01869,
    -0.01111,
    0.00094,
    2e-05,
    -0.00033,
    -0.00471
   ],
   [
    0.0025,
    -0.00862,
    -0.01412,
    0.00055,
    -0.00291,
    0.00313,
    0.00559,
    0.00066,
    -0.0061,
    -0.01046,
    -0.0002,
    -0.00271,
    0.00303,
    0.0044
   ],
   [
    -0.00213,
    0.01036,
    -0.00022,
    -0.00461,
    -0.00794,
    0.01755,
    0.01672,
    0.00096,
    0.00897,
    0.00155,
    -0.0031,
    -0.00595,
    0.01627,
    0.01551
   ],
   [
    0.00594,
    -0.00724,
    -0.01454,
    0.01561,
    -0.0222,
    -0.00462,
    -0.0184,
    0.00104,
    -0.00989,
    -0.01123,
    0.0135,
    -0.02074,
    -0.00634,
    -0.01745
   ],
   [
    -0.01406,
    -0.03669,
    0.00342,
    -0.00743,
    -0.00575,
    0.00523,
    -0.00229,
    -0.01085,
    -0.03193,
    0.00169,
    -0.008,
    -0.00508,
    0.00557,
    -0.00149
   ],
   [
    -0.01589,
    -0.02457,
    -0.00351,
    0.00965,
    -0.00278,
    -0.00486,
    -0.00214,
    -0.01595,
    -0.02759,
    -0.00644,
    0.00822,
    -0.00155,
    -0.00391,
    -0.0033
   ],
   [
    -0.00998,
    -0.00686,
    -0.00784,
    -0.00193,
    -0.00589,
    0.0055,
    -0.0081,
    -0.00674,
    -0.00684,
    -0.00948,
    -0.00408,
    -0.00498,
    0.00388,
    -0.00691
   ],
   [
    0.01675,
    -0.00454,
    0.01155,
    0.00452,
    -0.00732,
    0.00049,
    0.00026,
    0.01463,
    -0.00679,
    0.01247,
    0.00369,
    -0.00726,
    -0.00116,
    -0.0005
   ],
   [
    0.00898,
    0.00978,
    0.0094,
    0.01133,
    -0.00478,
    -0.00393,
    -0.01579,
    0.00689,
    0.00689,
    0.00437,
    0.00844,
    -0.00541,
    -0.00565,
    -0.0163
   ],
   [
    5e-05,
    0.00494,
    -0.00561,
    0.00615,
    -0.00177,
    0.00431,
    -0.00386,
    -0.00058,
    0.00361,
    -0.00041,
    0.00448,
    -0.00037,
    0.00529,
    -0.003
   ],
   [
    0.00699,
    0.00635,
    0.01111,
    0.00161,
    0.01075,
    -0.00373,
    -0.01051,
    0.00683,
    -0.00187,
    0.00601,
    -0.0003,
    0.00886,
    -0.00305,
    -0.00915
   ],
   [
    -0.00958,
    -0.01095,
    0.00267,
    0.0032,
    -0.0133,
    0.00486,
    0.01161,
    -0.00753,
    -0.00634,
    0.00401,
    0.00534,
    -0.00936,
    0.00363,
    0.00873
   ],
   [
    0.0243,
    0.01006,
    0.00206,
    0.00326,
    0.00929,
    -0.00019,
    -0.001,
    0.02002,
    0.01048,
    0.00226,
    0.0037,
    0.00782,
    -0.00035,
    -3e-05
   ],
   [
    -0.00982,
    0.00093,
    0.0013,
    -0.01822,
    0.00298,
    0.00675,
    -0.00432,
    -0.01156,
    0.00101,
    -0.00056,
    -0.01486,
    0.00152,
    0.00475,
    -0.00398
   ],
   [
    0.00074,
    -0.01272,
    0.00793,
    -0.00464,
    -0.01012,
    -0.00304,
    0.00695,
    8e-05,
    -0.00366,
    0.00532,
    -0.00334,
    -0.00878,
    -0.00259,
    0.00477
   ],
   [
    0.00207,
    -0.02151,
    -0.00293,
    -0.00072,
    -0.0164,
    -0.00263,
    0.01946,
    0.00167,
    -0.01985,
    -0.0071,
    0.00238,
    -0.01252,
    -0.0016,
    0.01815
   ],
   [
    0.01384,
    0.02718,
    0.01424,
    -0.00882,
    0.01562,
    0.00341,
    -0.00497,
    0.01362,
    0.02012,
    0.01202,
    -0.0085,
    0.01371,
    0.00305,
    -0.0034
   ],
   [
    0.00784,
    -0.00144,
    0.00972,
    -0.0024,
    -0.00074,
    -0.00237,
    -0.00703,
    0.00205,
    -0.00142,
    0.00803,
    -0.00119,
    -0.0,
    0.00094,
    -0.00436
   ],
   [
    -0.00861,
    -0.01286,
    -0.00995,
    0.00761,
    -0.00943,
    -0.00824,
    0.00869,
    -0.00801,
    -0.0059,
    -0.01024,
    0.00605,
    -0.00981,
    -0.00819,
    0.0051
   ],
   [
    -0.01969,
    0.00846,
    -0.00343,
    0.00653,
    -0.00298,
    -0.00075,
    0.00096,
    -0.01702,
    0.0052,
    4e-05,
    0.00683,
    0.00077,
    -2e-05,
    0.00284
   ],
   [
    -0.00117,
    0.00288,
    0.00475,
    -0.00541,
    -0.01314,
    0.00748,
    0.01368,
    0.00028,
    0.00578,
    0.00618,
    -0.00242,
    -0.00939,
    0.00618,
    0.0114
   ],
   [
    0.01395,
    0.01035,
    0.01361,
    -0.00805,
    0.00653,
    -0.00971,
    0.00932,
    0.01429,
    0.01223,
    0.00611,
    -0.00887,
    0.00499,
    -0.00728,
    0.00758
   ],
   [
    0.01454,
    -0.00199,
    0.00095,
    0.00013,
    -0.00926,
    0.01209,
    -0.00128,
    0.01038,
    -0.00247,
    5e-05,
    0.00125,
    -0.0075,
    0.00922,
    -0.00202
   ],
   [
    0.0075,
    -0.0005,
    0.00184,
    -0.00607,
    0.00733,
    -0.00333,
    0.00563,
    0.0088,
    -0.00231,
    -0.00089,
    -0.00888,
    0.00609,
    -0.00377,
    0.00492
   ],
   [
    -0.00082,
    -0.00888,
    0.00812,
    0.00807,
    0.00283,
    0.00265,
    -0.00105,
    0.00227,
    -0.00368,
    0.00833,
    0.00762,
    0.00362,
    0.00326,
    -0.00021
   ],
   [
    0.00957,
    -0.01329,
    0.00343,
    -0.00463,
    -0.00787,
    0.00298,
    0.00417,
    0.00946,
    -0.0108,
    0.0052,
    -0.00321,
    -0.00541,
    0.00409,
    0.00307
   ],
   [
    0.00584,
    0.00552,
    0.01392,
    -0.00123,
    -0.00752,
    0.00434,
    0.0099,
    0.00951,
    0.00975,
    0.01146,
    0.00081,
    -0.00399,
    0.00764,
    0.00979
   ],
   [
    0.03447,
    -0.0038,
    0.01691,
    -0.00688,
    -0.00277,
    -0.00491,
    0.0111,
    0.03038,
    4e-05,
    0.0107,
    -0.00608,
    -0.00184,
    -0.00306,
    0.00794
   ],
   [
    0.00029,
    0.01471,
    0.00101,
    -0.00118,
    0.00735,
    -0.00113,
    0.00097,
    0.00186,
    0.01408,
    -0.0014,
    0.00054,
    0.00805,
    0.00137,
    0.00194
   ],
   [
    -0.01336,
    0.00467,
    -0.01819,
    -0.00511,
    0.01345,
    -0.00389,
    -0.00492,
    -0.01021,
    -0.00118,
    -0.01785,
    -0.00447,
    0.01267,
    -0.00277,
    -0.00485
   ],
   [
    -0.01918,
    -0.01843,
    -0.00749,
    0.01349,
    -0.01295,
    -0.00717,
    -0.00779,
    -0.01741,
    -0.0132,
    -0.00767,
    0.0101,
    -0.01533,
    -0.00964,
    -0.00898
   ],
   [
    0.0127,
    0.00474,
    0.00748,
    0.0029,
    -0.00404,
    0.00669,
    -0.00097,
    0.00714,
    -0.00132,
    0.00328,
    0.00314,
    -0.00485,
    0.00459,
    -0.00169
   ],
   [
    -0.00327,
    0.00279,
    -0.01535,
    -0.00894,
    0.0096,
    -0.00326,
    -0.0003,
    -0.00691,
    -0.00433,
    -0.018,
    -0.01099,
    0.00724,
    -0.00413,
    -0.00219
   ],
   [
    0.00256,
    -0.00819,
    0.00055,
    -0.00585,
    -0.00721,
    0.00596,
    0.02447,
    0.00132,
    -0.00534,
    -0.00112,
    -0.00537,
    -0.00647,
    0.00426,
    0.01923
   ],
   [
    0.02056,
    0.00899,
    0.00032,
    -0.0055,
    -0.00713,
    0.00132,
    -0.00237,
    0.01622,
    0.00799,
    5e-05,
    -0.00407,
    -0.00268,
    0.00097,
    -0.00267
   ],
   [
    -0.00768,
    -0.00642,
    -0.00718,
    -0.00124,
    -0.01153,
    0.0097,
    -0.00907,
    -0.00895,
    -0.01009,
    -0.00567,
    -0.00212,
    -0.01054,
    0.00746,
    -0.00766
   ],
   [
    0.01212,
    0.00965,
    0.01641,
    0.01042,
    -0.01069,
    0.00373,
    0.00143,
    0.01019,
    0.00893,
    0.01441,
    0.01277,
    -0.00599,
    0.0049,
    0.00235
   ],
   [
    -0.01892,
    0.00894,
    -0.00307,
    -0.0054,
    0.00125,
    -0.00592,
    -0.00549,
    -0.01734,
    0.0014,
    -0.00596,
    -0.00777,
    -0.00149,
    -0.00679,
    -0.00446
   ],
   [
    0.01292,
    0.00901,
    0.00581,
    0.01372,
    -0.00325,
    0.001,
    -0.00911,
    0.00977,
    0.00483,
    0.00601,
    0.01192,
    -0.00128,
    0.00501,
    -0.0057
   ],
   [
    -0.0063,
    0.00335,
    -0.00337,
    0.00016,
    -0.00404,
    0.00877,
    -0.00787,
    -0.00626,
    0.00077,
    -0.00401,
    -0.00013,
    -0.00473,
    0.00525,
    -0.00653
   ],
   [
    0.0171,
    0.01402,
    0.00025,
    -0.00713,
    -0.0119,
    0.01149,
    0.01493,
    0.01531,
    0.01652,
    0.00018,
    -0.00911,
    -0.01082,
    0.00682,
    0.01008
   ],
   [
    0.01082,
    0.00497,
    0.00521,
    0.00031,
    -0.0014,
    -0.00873,
    -0.00552,
    0.00839,
    0.00553,
    -0.00163,
    -0.0014,
    -0.00504,
    -0.01069,
    -0.00782
   ],
   [
    0.00326,
    0.02482,
    -0.00273,
    0.016,
    0.00513,
    0.0031,
    -0.01993,
    0.00224,
    0.01486,
    -0.00142,
    0.01604,
    0.0039,
    0.00337,
    -0.01529
   ],
   [
    -0.00304,
    0.00597,
    -0.00685,
    0.01053,
    -0.01221,
    0.00407,
    -0.00201,
    1e-05,
    0.00828,
    0.00058,
    0.01013,
    -0.0092,
    0.00408,
    -0.00017
   ],
   [
    0.01668,
    0.00906,
    0.00129,
    0.00888,
    0.00127,
    -0.01168,
    -0.01156,
    0.0095,
    0.01454,
    0.0059,
    0.01191,
    0.00177,
    -0.01248,
    -0.00957
   ],
   [
    0.00395,
    -0.00444,
    0.00354,
    -0.01087,
    0.00592,
    -0.00589,
    0.00534,
    0.00101,
    -0.00353,
    -0.00364,
    -0.01153,
    0.0022,
    -0.00923,
    0.0012
   ],
   [
    -0.03056,
    0.01156,
    0.0068,
    0.00081,
    -0.00385,
    -0.00561,
    -0.00637,
    -0.02664,
    0.00516,
    -0.00079,
    -0.00127,
    -0.00509,
    -0.00677,
    -0.00718
   ],
   [
    -0.01687,
    -0.01366,
    -0.01155,
    0.00681,
    0.01142,
    -0.0077,
    -0.00446,
    -0.01521,
    -0.0093,
    -0.00952,
    0.00641,
    0.00799,
    -0.0079,
    -0.00606
   ],
   [
    0.00408,
    -0.0098,
    0.03641,
    -0.01847,
    -0.031,
    0.02598,
    0.02715,
    0.00457,
    -0.00081,
    0.02384,
    -0.01301,
    -0.02225,
    0.0273,
    0.0267
   ],
   [
    0.00476,
    0.0027,
    0.00897,
    0.00788,
    -0.00636,
    0.00688,
    -0.00464,
    0.00416,
    0.00646,
    0.01026,
    0.00973,
    -0.00269,
    0.00928,
    -0.00179
   ],
   [
    0.01628,
    0.01291,
    -0.00197,
    0.01181,
    0.00877,
    0.00445,
    -0.00958,
    0.01426,
    0.00999,
    0.00047,
    0.01121,
    0.00715,
    0.00333,
    -0.00709
   ],
   [
    0.00751,
    0.0012,
    -0.00788,
    0.00083,
    0.00335,
    -0.00492,
    0.00562,
    0.00529,
    0.00256,
    -0.00934,
    -0.00114,
    0.00186,
    -0.00486,
    0.00394
   ],
   [
    -0.01275,
    -0.01659,
    0.00967,
    -0.00477,
    -0.00081,
    -0.00733,
    0.00451,
    -0.01124,
    -0.01481,
    0.00057,
    -0.0051,
    -0.00091,
    -0.0086,
    0.00294
   ],
   [
    0.00542,
    0.00476,
    -0.00166,
    0.04178,
    -0.00081,
    -0.0316,
    -0.02138,
    0.0096,
    0.00239,
    0.00838,
    0.04563,
    0.00505,
    -0.01891,
    -0.01131
   ],
   [
    0.00424,
    -0.00891,
    0.00763,
    -0.00025,
    -0.00072,
    0.0039,
    0.00709,
    0.00377,
    -0.00161,
    0.00932,
    0.00259,
    -0.0016,
    0.00151,
    0.00479
   ],
   [
    -0.00519,
    -0.00686,
    0.0221,
    -0.00021,
    -0.00066,
    -0.00492,
    -0.00854,
    -0.00351,
    -0.00689,
    0.01886,
    0.00374,
    0.00404,
    -0.0028,
    -0.00467
   ],
   [
    0.01732,
    -0.00402,
    -0.00319,
    0.00857,
    -0.01257,
    -0.00521,
    0.00082,
    0.0112,
    -0.0057,
    -0.00442,
    0.00835,
    -0.01206,
    -0.00617,
    0.00056
   ],
   [
    0.00472,
    0.00423,
    0.00108,
    0.0067,
    0.00427,
    -0.01185,
    -0.01845,
    0.00462,
    0.0037,
    0.00053,
    0.00493,
    0.00358,
    -0.01074,
    -0.01715
   ],
   [
    0.01136,
    -0.00336,
    -0.00825,
    0.00473,
    0.00259,
    -0.00557,
    -0.00854,
    0.00917,
    -0.00181,
    -0.00496,
    0.0061,
    0.00251,
    -0.00413,
    -0.00705
   ],
   [
    -0.00058,
    0.00276,
    0.01124,
    -0.00402,
    0.00029,
    0.00483,
    -0.00224,
    -0.00106,
    0.00167,
    0.005,
    -0.00521,
    -0.00196,
    0.00295,
    -0.00453
   ],
   [
    -0.01761,
    -0.00904,
    -0.002,
    0.00267,
    0.00264,
    -0.00184,
    -0.00597,
    -0.01384,
    -0.00575,
    0.0008,
    0.00043,
    -0.00042,
    -0.00228,
    -0.00629
   ],
   [
    0.02793,
    0.0024,
    0.00303,
    0.01146,
    -0.01805,
    0.00565,
    -0.00481,
    0.0239,
    0.00106,
    0.00087,
    0.00731,
    -0.01587,
    0.00675,
    -0.00411
   ],
   [
    0.00803,
    -0.03299,
    0.00773,
    -0.01557,
    -0.0191,
    0.0129,
    0.01518,
    0.00938,
    -0.0238,
    0.00474,
    -0.01582,
    -0.01806,
    0.01059,
    0.01125
   ],
   [
    -0.01285,
    0.00068,
    0.00371,
    -0.00284,
    -0.00336,
    0.00171,
    -0.00125,
    -0.01218,
    0.00019,
    0.00415,
    -0.00273,
    -0.00136,
    0.00368,
    -0.00055
   ],
   [
    -0.01663,
    -0.0015,
    -0.0067,
    0.00751,
    -0.00655,
    0.00755,
    -0.01233,
    -0.0127,
    -0.00408,
    -0.01056,
    0.00515,
    -0.00695,
    0.00388,
    -0.01236
   ],
   [
    0.00468,
    -0.00355,
    -0.00033,
    -0.0019,
    -0.01666,
    0.02108,
    0.0171,
    0.00402,
    -0.0008,
    0.005,
    0.00011,
    -0.00829,
    0.02431,
    0.01667
   ],
   [
    -0.00066,
    0.02062,
    0.02219,
    -0.00868,
    0.02122,
    -0.0004,
    0.009,
    0.0024,
    0.01864,
    0.0169,
    -0.00699,
    0.01918,
    0.00028,
    0.00869
   ],
   [
    0.0031,
    0.01909,
    -0.00575,
    -0.00855,
    0.02161,
    -0.00975,
    -0.00698,
    0.00094,
    0.01652,
    -0.00475,
    -0.00825,
    0.01579,
    -0.00903,
    -0.00615
   ],
   [
    -0.00634,
    0.00836,
    -0.00101,
    0.00185,
    4e-05,
    -0.00044,
    0.00343,
    -0.00448,
    0.00895,
    -0.0014,
    0.00112,
    -5e-05,
    0.00086,
    0.00374
   ],
   [
    0.01284,
    -0.00043,
    -0.00567,
    -0.00912,
    0.00933,
    0.004,
    0.01004,
    0.01257,
    -0.00193,
    -0.00622,
    -0.01142,
    0.00909,
    0.00413,
    0.00766
   ],
   [
    -0.00016,
    0.00504,
    -0.00314,
    0.01053,
    -0.00666,
    -0.00645,
    -0.0155,
    -0.00259,
    0.00148,
    -0.0051,
    0.00781,
    -0.00627,
    -0.00519,
    -0.01448
   ],
   [
    0.01954,
    -0.00763,
    0.00604,
    -0.00588,
    0.00142,
    0.00118,
    0.00287,
    0.01732,
    -0.00398,
    0.00792,
    -0.00333,
    0.00215,
    0.0033,
    0.0042
   ],
   [
    0.00391,
    0.02646,
    -0.00584,
    -0.00582,
    0.00502,
    0.00191,
    0.00137,
    0.00568,
    0.02398,
    -0.00268,
    -0.00774,
    0.00531,
    0.00335,
    0.002
   ],
   [
    0.02137,
    -0.00978,
    0.0015,
    0.02105,
    -0.00967,
    -0.00385,
    0.00393,
    0.01529,
    -0.01164,
    0.00182,
    0.01755,
    -0.00793,
    -0.00272,
    0.00516
   ],
   [
    -0.00559,
    -0.00901,
    0.00464,
    0.01111,
    0.00461,
    -0.00924,
    0.00594,
    0.00102,
    -0.00291,
    0.00877,
    0.01178,
    0.00354,
    -0.00651,
    0.00809
   ],
   [
    -0.03114,
    -0.01217,
    -0.00335,
    -0.00953,
    0.00178,
    0.00145,
    0.00825,
    -0.02455,
    -0.01211,
    -0.00585,
    -0.01011,
    0.00101,
    0.00045,
    0.00715
   ],
   [
    -0.00276,
    0.00048,
    -0.00361,
    -0.00612,
    0.00476,
    -0.00844,
    0.00661,
    -0.0026,
    0.0029,
    -0.0046,
    -0.00555,
    0.00436,
    -0.00632,
    0.00601
   ],
   [
    -0.01233,
    -0.00889,
    -0.00284,
    -0.00097,
    -0.00255,
    -0.00151,
    0.01232,
    -0.01283,
    -0.0104,
    -0.00257,
    -0.00048,
    -0.00228,
    -0.00266,
    0.00928
   ],
   [
    0.00745,
    -0.00096,
    0.00674,
    0.00284,
    -0.00467,
    -0.00395,
    0.00209,
    0.00747,
    0.00252,
    0.00859,
    0.00342,
    -0.00052,
    3e-05,
    0.00237
   ],
   [
    -0.02128,
    -0.00642,
    0.0044,
    0.00098,
    0.0036,
    -0.00177,
    0.00045,
    -0.0164,
    -0.00813,
    0.00437,
    0.00319,
    0.00316,
    -0.00302,
    -0.0029
   ],
   [
    0.0148,
    0.03222,
    -0.00291,
    0.00402,
    0.00754,
    -0.0068,
    -0.01104,
    0.01473,
    0.02496,
    -0.00232,
    0.00327,
    0.0049,
    -0.00651,
    -0.0103
   ],
   [
    -0.00447,
    0.02838,
    -0.00408,
    -0.00755,
    0.00561,
    -0.00793,
    0.01067,
    -0.00421,
    0.02243,
    -0.00442,
    -0.0103,
    0.00601,
    -0.0061,
    0.00891
   ],
   [
    0.01546,
    0.00687,
    0.00691,
    0.00853,
    -0.01329,
    -0.00506,
    -0.0034,
    0.0164,
    0.00519,
    0.00936,
    0.00373,
    -0.01481,
    -0.0056,
    -0.0037
   ],
   [
    -0.00411,
    -0.00292,
    -0.00409,
    0.00981,
    0.0091,
    -0.00922,
    -0.01445,
    -0.00191,
    -0.00894,
    -0.00941,
    0.00821,
    0.00639,
    -0.01168,
    -0.01601
   ],
   [
    -0.00676,
    0.00249,
    0.00232,
    0.00552,
    0.00381,
    -0.00581,
    0.00125,
    -0.0056,
    0.00077,
    -0.00144,
    0.00511,
    0.00346,
    -0.00326,
    0.00425
   ],
   [
    0.0111,
    -0.0009,
    -0.00381,
    -0.00129,
    0.01285,
    -0.00854,
    5e-05,
    0.00875,
    0.0001,
    -0.00332,
    -0.001,
    0.01073,
    -0.00678,
    0.00049
   ],
   [
    -0.0086,
    -0.03145,
    -0.00065,
    0.00379,
    -0.0002,
    -0.01026,
    0.00111,
    -0.01067,
    -0.02715,
    -0.00597,
    0.00468,
    5e-05,
    -0.01163,
    0.00036
   ],
   [
    -0.01147,
    0.00182,
    -0.00587,
    0.01413,
    0.00834,
    -0.00462,
    -0.00212,
    -0.00683,
    0.00388,
    -0.00719,
    0.01185,
    0.00774,
    -0.00306,
    -0.0022
   ],
   [
    -0.03868,
    -0.02765,
    -0.02434,
    -0.00635,
    0.013,
    -0.00299,
    0.00431,
    -0.03114,
    -0.02355,
    -0.01989,
    -0.00622,
    0.01186,
    -0.00094,
    0.0023
   ],
   [
    -0.00155,
    0.01653,
    -0.00479,
    0.01309,
    0.00532,
    -0.00445,
    -0.01782,
    -0.00099,
    0.01357,
    -0.00025,
    0.01228,
    0.00381,
    -0.00301,
    -0.01362
   ],
   [
    -0.00728,
    -0.00177,
    0.00695,
    0.00422,
    0.00486,
    -0.00193,
    -0.00573,
    -0.00396,
    -0.00315,
    0.00761,
    0.004,
    0.00462,
    -0.00153,
    -0.00436
   ],
   [
    -0.00129,
    -0.0148,
    0.00145,
    0.00691,
    0.00085,
    -0.00508,
    0.00699,
    -0.00285,
    -0.01104,
    -9e-05,
    0.01045,
    0.00122,
    -0.00451,
    0.00566
   ],
   [
    0.02795,
    0.00123,
    -0.00118,
    0.01477,
    -0.01516,
    -0.00565,
    -0.00056,
    0.02319,
    0.0023,
    0.00186,
    0.01266,
    -0.01434,
    -0.00704,
    -0.00072
   ],
   [
    0.00671,
    -0.00082,
    -0.00321,
    -0.00107,
    -0.00096,
    0.00437,
    0.00065,
    0.00537,
    0.00432,
    0.00326,
    0.00141,
    -0.00033,
    0.00239,
    -0.00031
   ],
   [
    0.02071,
    0.00188,
    0.01645,
    -0.0172,
    -0.00752,
    0.02392,
    0.0176,
    0.02053,
    0.00332,
    0.01505,
    -0.01651,
    -0.0036,
    0.02351,
    0.01631
   ],
   [
    0.00224,
    0.00734,
    0.01317,
    -0.00637,
    -0.00903,
    0.01011,
    0.00882,
    0.00525,
    0.0111,
    0.01191,
    -0.00776,
    -0.00729,
    0.00917,
    0.0053
   ],
   [
    0.01148,
    0.02644,
    -0.00426,
    -0.00588,
    0.01121,
    -0.00467,
    0.00403,
    0.01077,
    0.0211,
    0.00081,
    -0.00546,
    0.0089,
    -0.00225,
    0.00398
   ],
   [
    0.03746,
    -0.00586,
    0.00701,
    0.01695,
    -0.01316,
    -0.00773,
    0.00713,
    0.03025,
    -0.00412,
    0.006,
    0.01554,
    -0.01273,
    -0.00784,
    0.00649
   ],
   [
    -0.03216,
    -0.01335,
    0.01352,
    -0.01115,
    0.00387,
    0.00707,
    0.00842,
    -0.02568,
    -0.02026,
    0.00794,
    -0.01329,
    -0.00012,
    0.00588,
    0.00678
   ],
   [
    -0.0076,
    0.01209,
    -0.00541,
    0.00879,
    -0.00108,
    -0.00987,
    -0.01161,
    -0.0066,
    0.00588,
    -0.00132,
    0.00612,
    -0.00157,
    -0.01081,
    -0.01014
   ],
   [
    0.0013,
    0.01105,
    -0.00519,
    -0.00429,
    0.00706,
    -0.00291,
    -0.0063,
    0.00516,
    0.00995,
    -0.00017,
    -0.00285,
    0.00661,
    -0.00108,
    -0.00424
   ],
   [
    -0.00194,
    -0.01552,
    -0.00212,
    0.01042,
    -0.01345,
    0.0121,
    0.0105,
    -0.00287,
    -0.01494,
    -0.00018,
    0.00816,
    -0.01119,
    0.01246,
    0.00814
   ],
   [
    -0.01636,
    -0.00293,
    -0.00073,
    0.0012,
    -0.00088,
    0.01246,
    -0.00266,
    -0.01536,
    -0.00462,
    -0.00164,
    -0.00058,
    -0.00391,
    0.00895,
    -0.00319
   ],
   [
    0.03001,
    0.00126,
    0.01274,
    0.02737,
    -0.01365,
    -0.00197,
    -0.01302,
    0.02534,
    0.00596,
    0.01275,
    0.02702,
    -0.00929,
    -0.00285,
    -0.01029
   ],
   [
    0.00664,
    0.01521,
    -0.00211,
    0.00104,
    0.00538,
    -0.00551,
    -0.00674,
    0.00462,
    0.00775,
    -0.00602,
    -0.0018,
    0.00381,
    -0.00773,
    -0.00791
   ],
   [
    -0.02419,
    0.00943,
    -0.00538,
    -0.00372,
    0.02393,
    -0.01447,
    0.01134,
    -0.01799,
    0.00542,
    -0.0033,
    -0.00271,
    0.01797,
    -0.01515,
    0.00874
   ],
   [
    -0.00461,
    -0.00463,
    0.00144,
    -0.00282,
    0.00093,
    -0.00692,
    -0.00227,
    -0.00351,
    -0.00897,
    -0.00397,
    -0.00163,
    -0.00081,
    -0.00758,
    -0.00325
   ],
   [
    -0.00279,
    0.01122,
    0.00415,
    0.00124,
    -0.00193,
    -0.01189,
    -0.00146,
    -0.00371,
    0.00518,
    -0.00142,
    0.00036,
    -0.00273,
    -0.0131,
    -0.00354
   ],
   [
    -0.00328,
    -0.00575,
    0.00208,
    0.0008,
    0.00619,
    -0.01233,
    -0.00342,
    -0.00454,
    -0.0061,
    7e-05,
    -0.00121,
    0.00411,
    -0.01177,
    -0.00352
   ],
   [
    -0.00105,
    0.00634,
    0.00178,
    -0.0052,
    0.01278,
    -0.00951,
    0.00645,
    -0.00116,
    0.00325,
    -0.00118,
    -0.00563,
    0.00933,
    -0.00957,
    0.00288
   ],
   [
    -0.01171,
    0.00521,
    -0.01317,
    -0.00821,
    -0.0037,
    0.014,
    -0.00275,
    -0.01015,
    0.00184,
    -0.00908,
    -0.00961,
    -0.00312,
    0.01242,
    -0.00138
   ],
   [
    -0.00767,
    -0.01577,
    -0.01245,
    -0.01894,
    -0.01187,
    0.00835,
    0.01915,
    -0.00593,
    -0.01322,
    -0.00707,
    -0.01711,
    -0.00734,
    0.00768,
    0.0159
   ],
   [
    0.00626,
    -0.00586,
    -0.0096,
    -0.01515,
    -0.00058,
    0.01087,
    0.00952,
    0.00624,
    -0.00174,
    -0.00973,
    -0.01638,
    -0.00128,
    0.00574,
    0.00624
   ],
   [
    -0.00858,
    0.00625,
    0.00975,
    0.00123,
    0.01689,
    0.00017,
    0.00891,
    -0.00122,
    0.00561,
    0.00749,
    0.00181,
    0.01728,
    0.00423,
    0.01053
   ],
   [
    -0.00227,
    -0.02972,
    -0.0007,
    -0.0094,
    0.00913,
    0.00394,
    0.00835,
    -0.00018,
    -0.0161,
    0.00499,
    -0.00566,
    0.00795,
    0.00395,
    0.00549
   ],
   [
    -0.00283,
    -0.00699,
    -0.01474,
    0.00155,
    -0.00263,
    -0.00128,
    0.00472,
    -0.00427,
    -0.0077,
    -0.01487,
    0.00086,
    -0.00232,
    -0.00053,
    0.0032
   ],
   [
    -0.0003,
    -0.01362,
    -0.01136,
    -0.00071,
    -0.00527,
    -0.00148,
    -0.00157,
    -0.00252,
    -0.00948,
    -0.00736,
    0.00082,
    -0.00364,
    -0.0027,
    -0.0023
   ],
   [
    0.00486,
    0.01704,
    0.00097,
    1e-05,
    -0.00214,
    -0.00126,
    0.00682,
    0.00718,
    0.01676,
    -0.00223,
    -0.00044,
    -0.0029,
    -0.00573,
    0.00118
   ],
   [
    -0.0113,
    0.00422,
    -0.00933,
    0.00197,
    -0.00598,
    -0.00109,
    0.0038,
    -0.01144,
    0.00508,
    -0.00743,
    0.00156,
    -0.00538,
    -1e-05,
    0.00267
   ],
   [
    0.0163,
    -0.00892,
    0.00705,
    -0.00612,
    -0.0203,
    -0.00304,
    0.00083,
    0.01498,
    -0.01109,
    0.00242,
    -0.01231,
    -0.02134,
    -0.0067,
    -0.00216
   ],
   [
    0.00398,
    -0.01445,
    0.00691,
    0.01155,
    -0.01191,
    -0.00813,
    0.00104,
    0.00486,
    -0.01417,
    0.00429,
    0.01371,
    -0.00915,
    -0.00573,
    0.00235
   ],
   [
    0.00256,
    -0.01402,
    -0.00237,
    -0.00126,
    0.01286,
    -0.02662,
    0.00243,
    -0.00045,
    -0.01181,
    -0.00376,
    1e-05,
    0.01031,
    -0.02422,
    -0.00144
   ],
   [
    0.00127,
    -0.00721,
    0.00163,
    0.00975,
    0.00696,
    -0.002,
    -0.00315,
    -0.00161,
    -0.01011,
    -0.00206,
    0.00861,
    0.00384,
    -0.00428,
    -0.00355
   ],
   [
    0.02618,
    0.00162,
    0.01335,
    -0.0078,
    -0.00743,
    0.01243,
    0.02914,
    0.02658,
    0.00691,
    0.01333,
    -0.00484,
    -0.00401,
    0.01518,
    0.02728
   ],
   [
    -0.0002,
    -0.00835,
    0.00045,
    0.00216,
    0.00082,
    -0.00691,
    -0.00048,
    -0.00285,
    -0.00905,
    -0.00223,
    0.00174,
    0.00052,
    -0.0076,
    -0.00105
   ],
   [
    -0.00058,
    -0.0032,
    0.0014,
    0.01037,
    -0.00739,
    -0.01108,
    0.00364,
    0.00105,
    -0.00154,
    0.00119,
    0.00815,
    -0.00625,
    -0.01079,
    0.00174
   ],
   [
    0.00911,
    -0.01205,
    0.00535,
    0.00383,
    -0.0189,
    3e-05,
    -0.00532,
    0.00992,
    -0.00943,
    0.00048,
    0.00351,
    -0.01483,
    0.00116,
    -0.00385
   ],
   [
    0.00713,
    0.00681,
    0.0051,
    -0.0008,
    -0.00657,
    0.007,
    0.00036,
    0.00639,
    0.00404,
    0.00284,
    -0.00523,
    -0.00726,
    0.00385,
    -0.0004
   ],
   [
    0.01179,
    -0.00917,
    -0.00515,
    -0.00765,
    -0.01016,
    0.00439,
    0.00512,
    0.00863,
    -0.00609,
    -0.00382,
    -0.00454,
    -0.00877,
    0.00149,
    0.00253
   ],
   [
    -0.00979,
    0.00308,
    -0.01261,
    -0.00175,
    -0.00431,
    -0.00595,
    0.00109,
    -0.00754,
    0.00049,
    -0.01173,
    -0.0027,
    -0.00148,
    -0.00191,
    0.00183
   ],
   [
    -0.00585,
    0.00712,
    0.00582,
    -0.00165,
    -0.00383,
    -0.00479,
    -0.00636,
    -0.00332,
    0.00391,
    0.00433,
    -0.00139,
    -0.00287,
    -0.00393,
    -0.0053
   ],
   [
    0.02106,
    -0.00305,
    0.0166,
    0.00963,
    -0.00931,
    0.00674,
    -0.00419,
    0.01976,
    -0.00095,
    0.01887,
    0.01163,
    -0.00561,
    0.0077,
    -0.00114
   ],
   [
    -0.01082,
    0.00438,
    0.0061,
    -0.0149,
    0.01232,
    0.01562,
    1e-05,
    -0.00629,
    0.0032,
    0.00515,
    -0.01088,
    0.01271,
    0.01376,
    0.00212
   ],
   [
    -0.00203,
    0.01722,
    -0.01535,
    -0.00696,
    0.01237,
    0.0074,
    -0.00815,
    -0.00471,
    0.01106,
    -0.01262,
    -0.00844,
    0.01237,
    0.00748,
    -0.00543
   ],
   [
    -0.00973,
    -0.0014,
    0.00538,
    -0.01108,
    0.00165,
    0.0001,
    -0.00019,
    -0.0075,
    -0.00209,
    0.00239,
    -0.00939,
    0.00201,
    -0.00068,
    -0.00017
   ],
   [
    -0.00407,
    -0.0075,
    0.00059,
    7e-05,
    -0.00279,
    0.00047,
    -0.00419,
    -0.00191,
    -0.01026,
    0.00201,
    0.00067,
    -0.00215,
    0.0002,
    -0.00378
   ],
   [
    -0.00192,
    -0.0028,
    0.0074,
    -0.00921,
    0.00785,
    -0.00634,
    -0.0005,
    -0.00217,
    -0.00371,
    0.00454,
    -0.00888,
    0.00713,
    -0.00582,
    -0.00057
   ],
   [
    -0.00566,
    0.00233,
    -0.00015,
    -0.00194,
    0.01361,
    -0.00107,
    -0.00567,
    -0.00322,
    1e-05,
    -0.00108,
    -0.00374,
    0.01133,
    -0.00151,
    -0.00502
   ],
   [
    -0.00153,
    -0.00548,
    -0.00469,
    -0.00467,
    0.00187,
    -0.00037,
    0.00677,
    0.00215,
    -0.00233,
    -0.00413,
    -0.005,
    0.00236,
    0.00031,
    0.0049
   ],
   [
    0.00387,
    0.00669,
    0.00908,
    0.00867,
    -0.00303,
    0.00961,
    -0.0073,
    0.00484,
    0.00418,
    0.00723,
    0.00664,
    -0.0027,
    0.00782,
    -0.00638
   ],
   [
    0.01406,
    0.01196,
    -0.01323,
    0.01447,
    -0.01107,
    -0.00515,
    -0.00629,
    0.01302,
    0.01021,
    -0.00459,
    0.01309,
    -0.00715,
    -0.00075,
    -0.00341
   ],
   [
    0.00641,
    -0.01807,
    -0.00465,
    0.01466,
    -0.01955,
    -0.00432,
    -0.01824,
    0.00301,
    -0.01958,
    -0.00345,
    0.01629,
    -0.01427,
    -0.00618,
    -0.01585
   ],
   [
    0.01734,
    0.02102,
    0.01291,
    0.00115,
    0.00269,
    -0.00481,
    -0.00756,
    0.01599,
    0.01205,
    0.00867,
    0.00249,
    0.00406,
    -0.00265,
    -0.00478
   ],
   [
    0.00863,
    0.02286,
    -0.00327,
    -0.00816,
    0.01356,
    0.00319,
    -0.00298,
    0.0088,
    0.02585,
    0.00111,
    -0.00882,
    0.00958,
    0.00245,
    -0.00236
   ],
   [
    -0.01153,
    -0.01531,
    -0.01849,
    0.00522,
    0.00146,
    -0.00236,
    -0.01074,
    -0.01217,
    -0.00895,
    -0.01275,
    0.00527,
    0.00083,
    -0.00359,
    -0.01098
   ],
   [
    -0.0108,
    0.00828,
    -0.00819,
    -0.00753,
    0.0109,
    -0.0007,
    -0.01535,
    -0.00745,
    7e-05,
    -0.00481,
    -0.00867,
    0.00787,
    -0.00086,
    -0.01357
   ],
   [
    0.00352,
    -0.0022,
    0.00245,
    0.02689,
    -0.00471,
    -0.00922,
    -0.02691,
    0.00062,
    -0.00707,
    0.0062,
    0.03042,
    4e-05,
    -0.00508,
    -0.0201
   ],
   [
    -0.01611,
    -0.01457,
    -0.00651,
    0.00501,
    -0.00839,
    0.00601,
    0.00719,
    -0.01124,
    -0.01268,
    -0.00393,
    0.00508,
    -0.00689,
    0.0068,
    0.00765
   ],
   [
    0.00164,
    0.01307,
    -0.00868,
    -0.00479,
    0.00445,
    -0.00761,
    0.01118,
    0.00609,
    0.00957,
    -0.01259,
    -0.00582,
    0.00473,
    -0.00759,
    0.00783
   ],
   [
    -0.00191,
    -0.00015,
    -0.00293,
    -0.0078,
    0.00891,
    -0.01381,
    0.01093,
    -0.00239,
    0.00249,
    -0.00122,
    -0.00446,
    0.00736,
    -0.01265,
    0.00843
   ],
   [
    0.00414,
    0.01883,
    0.00592,
    0.00144,
    -0.00685,
    -0.00667,
    0.00139,
    0.00467,
    0.01729,
    0.00551,
    0.0014,
    -0.0085,
    -0.00863,
    -0.00186
   ],
   [
    -0.0047,
    -0.01828,
    0.00532,
    -0.00855,
    -0.01046,
    0.01203,
    0.02356,
    -0.00018,
    -0.00809,
    0.00615,
    -0.00586,
    -0.00602,
    0.01249,
    0.02239
   ],
   [
    0.01697,
    0.00116,
    -0.00107,
    -0.01555,
    -0.01113,
    0.01458,
    0.00173,
    0.01241,
    -0.00125,
    -0.00163,
    -0.01737,
    -0.01063,
    0.01057,
    0.00106
   ],
   [
    0.00933,
    -0.01001,
    0.00699,
    -0.01327,
    -0.01694,
    0.01478,
    0.01183,
    0.0093,
    -0.00953,
    0.00429,
    -0.01463,
    -0.01174,
    0.01705,
    0.00986
   ],
   [
    -0.01181,
    -0.01369,
    0.00319,
    0.01213,
    0.00248,
    0.00399,
    -0.0111,
    -0.01029,
    -0.01292,
    0.00653,
    0.01029,
    -0.00043,
    0.00227,
    -0.01028
   ],
   [
    -0.01561,
    -0.01018,
    -0.00675,
    -0.00563,
    0.00365,
    -0.00202,
    -0.01087,
    -0.01278,
    -0.00991,
    -0.00136,
    -0.0026,
    0.00344,
    -0.00094,
    -0.00884
   ],
   [
    0.01802,
    0.00508,
    0.02187,
    -0.00109,
    -0.00296,
    -0.00638,
    -0.00112,
    0.01693,
    0.00711,
    0.01351,
    0.00063,
    -0.00355,
    -0.0057,
    -0.00286
   ],
   [
    0.01008,
    -0.01709,
    -0.00358,
    -0.00576,
    -0.00653,
    -0.00849,
    0.00246,
    0.00994,
    -0.01028,
    0.00112,
    -0.00645,
    -0.00525,
    -0.00613,
    0.0009
   ],
   [
    -0.01885,
    -0.0128,
    0.00851,
    -0.00115,
    -0.00247,
    -0.00655,
    0.00321,
    -0.01711,
    -0.01413,
    0.00789,
    -0.0018,
    -0.00368,
    -0.00713,
    0.00131
   ],
   [
    -0.0027,
    0.00268,
    0.00037,
    0.00849,
    -0.005,
    -0.0075,
    0.00309,
    -0.00432,
    0.00181,
    -0.00692,
    0.00751,
    -0.00247,
    -0.00735,
    0.00144
   ],
   [
    -0.0038,
    -0.00311,
    -0.00469,
    0.00223,
    -0.00912,
    -0.00807,
    -0.00226,
    -0.00537,
    -0.00581,
    -0.00442,
    0.00131,
    -0.01036,
    -0.00939,
    -0.00269
   ],
   [
    -0.00324,
    0.01271,
    -0.00813,
    0.00682,
    -0.00141,
    -0.0056,
    -0.00331,
    -0.00499,
    0.00811,
    -0.00391,
    0.00522,
    -0.00402,
    -0.00702,
    -0.00434
   ],
   [
    0.02883,
    0.00124,
    0.00228,
    0.0002,
    -0.01121,
    0.01818,
    0.00672,
    0.02427,
    -0.00251,
    0.00426,
    0.00049,
    -0.00804,
    0.01536,
    0.00636
   ],
   [
    0.00102,
    0.00104,
    -0.01045,
    -0.00527,
    0.01647,
    -0.00718,
    0.00584,
    -0.0,
    0.00166,
    -0.00742,
    -0.00532,
    0.01319,
    -0.00365,
    0.00614
   ],
   [
    0.0085,
    -0.02242,
    0.00511,
    0.0072,
    -0.00053,
    -0.01471,
    0.01133,
    0.00838,
    -0.01654,
    0.00101,
    0.00808,
    -0.00036,
    -0.01297,
    0.00653
   ],
   [
    -0.02301,
    -0.00815,
    0.00877,
    0.00481,
    0.00194,
    -0.00578,
    -0.00442,
    -0.01929,
    -0.00568,
    0.0035,
    0.00454,
    0.00055,
    -0.00546,
    -0.00446
   ],
   [
    -0.00464,
    -0.00054,
    0.00154,
    0.01021,
    -0.01251,
    0.00263,
    0.00222,
    -0.00673,
    -0.00203,
    0.0028,
    0.00838,
    -0.01011,
    0.0053,
    0.00304
   ],
   [
    -0.00569,
    -0.01075,
    -0.00715,
    0.00793,
    0.00531,
    -0.00804,
    -0.00526,
    -0.00618,
    -0.01587,
    -0.00786,
    0.00534,
    0.00352,
    -0.0081,
    -0.00649
   ],
   [
    0.00709,
    -0.00957,
    0.00058,
    -0.00385,
    -0.00953,
    -0.00053,
    0.00363,
    0.00455,
    -0.00776,
    0.0003,
    -0.00152,
    -0.00726,
    -0.00039,
    0.00287
   ],
   [
    -0.00176,
    -0.00461,
    -0.00362,
    -0.00237,
    -0.00349,
    -0.00875,
    6e-05,
    -0.00227,
    -0.0064,
    -0.00379,
    -0.00315,
    -0.0047,
    -0.00849,
    -0.0011
   ],
   [
    -0.00197,
    -0.0004,
    -0.001,
    -0.00601,
    0.00433,
    -0.00474,
    0.00429,
    -0.00241,
    -0.0008,
    0.00048,
    -0.00435,
    0.00538,
    -0.00069,
    0.00385
   ],
   [
    -0.01628,
    0.00819,
    -0.01173,
    -0.02005,
    0.01574,
    0.00459,
    -0.00939,
    -0.01261,
    0.00693,
    -0.01032,
    -0.02232,
    0.01047,
    0.00078,
    -0.01107
   ],
   [
    0.00574,
    -0.01072,
    0.00656,
    -0.00263,
    -0.00626,
    0.00442,
    0.01823,
    0.00152,
    -0.00491,
    0.00436,
    -0.00213,
    -0.00656,
    0.00376,
    0.01598
   ],
   [
    -0.02396,
    0.00086,
    -0.01249,
    0.00256,
    -0.00541,
    0.00638,
    -0.00362,
    -0.01873,
    -0.00589,
    -0.00989,
    0.00103,
    -0.00559,
    0.0048,
    -0.00139
   ],
   [
    0.00803,
    0.00079,
    -0.01218,
    0.01237,
    0.00682,
    -0.00626,
    -0.00323,
    0.00367,
    0.00254,
    -0.00933,
    0.01056,
    0.00503,
    -0.00378,
    -0.00233
   ],
   [
    0.01216,
    0.00564,
    0.00259,
    0.0091,
    -0.00304,
    0.01068,
    -0.00438,
    0.01342,
    0.0035,
    0.0027,
    0.00775,
    -0.00193,
    0.01002,
    -0.00235
   ],
   [
    0.00994,
    0.01608,
    -0.00705,
    0.00666,
    0.00113,
    -0.00265,
    -0.02353,
    0.01177,
    0.01712,
    -0.00268,
    0.00817,
    0.0014,
    0.00069,
    -0.02021
   ],
   [
    0.00689,
    -0.00227,
    0.00135,
    0.01606,
    0.02224,
    -0.02601,
    -0.01839,
    0.00645,
    0.00316,
    0.00344,
    0.01562,
    0.01873,
    -0.02084,
    -0.01343
   ],
   [
    -0.01076,
    1e-05,
    -0.0071,
    0.00862,
    0.00502,
    -0.0183,
    -0.00326,
    -0.00731,
    0.00096,
    -0.00181,
    0.00759,
    0.00428,
    -0.01369,
    -0.00069
   ],
   [
    -0.01378,
    0.01018,
    0.00628,
    0.00015,
    -0.00082,
    0.00169,
    -0.00186,
    -0.01273,
    0.00379,
    0.00051,
    0.00243,
    0.00163,
    0.00096,
    -0.0012
   ],
   [
    0.00493,
    0.00913,
    -0.00097,
    0.00211,
    0.01235,
    0.00451,
    -0.008,
    0.00674,
    0.00399,
    -0.00512,
    -0.00264,
    0.00603,
    -0.00203,
    -0.00904
   ],
   [
    -0.01678,
    0.00127,
    -0.00167,
    -0.00237,
    0.00038,
    0.00494,
    -0.00137,
    -0.01559,
    -0.00638,
    -0.00482,
    -0.00649,
    -0.00087,
    0.00185,
    -0.00174
   ],
   [
    -0.0059,
    -0.0036,
    0.00509,
    -0.01492,
    0.01158,
    0.00441,
    0.00943,
    -0.00603,
    0.00121,
    0.00286,
    -0.01398,
    0.01101,
    0.00602,
    0.01042
   ],
   [
    0.01165,
    4e-05,
    0.00047,
    0.0006,
    -0.00096,
    0.00671,
    0.00996,
    0.00904,
    0.00323,
    0.00127,
    0.00218,
    0.00125,
    0.00792,
    0.01039
   ],
   [
    -0.00102,
    -0.00384,
    -0.00783,
    0.0066,
    -0.00548,
    0.00714,
    -0.0039,
    -0.00481,
    -0.00439,
    -0.00581,
    0.00401,
    -0.00647,
    0.00275,
    -0.00498
   ],
   [
    0.00426,
    0.01717,
    -0.00226,
    0.00736,
    -0.00222,
    -0.01429,
    -0.00858,
    0.00036,
    0.00941,
    -0.0036,
    0.00661,
    -0.00268,
    -0.01384,
    -0.00799
   ],
   [
    0.00423,
    0.00416,
    -0.00056,
    -0.00234,
    0.01228,
    0.00709,
    0.0097,
    0.00249,
    0.00627,
    0.0007,
    -0.00129,
    0.01223,
    0.00758,
    0.00838
   ],
   [
    -0.00781,
    3e-05,
    0.00109,
    -0.01574,
    0.00267,
    0.01557,
    -0.00182,
    -0.00286,
    -0.00109,
    -0.00275,
    -0.01406,
    0.004,
    0.0129,
    -0.0025
   ],
   [
    0.01063,
    -0.01912,
    0.00421,
    -0.00207,
    -0.00388,
    -0.00346,
    -0.00053,
    0.00806,
    -0.01139,
    0.00418,
    0.00129,
    -0.00164,
    -0.00118,
    0.0012
   ],
   [
    0.01039,
    0.003,
    0.00897,
    0.00521,
    -7e-05,
    -0.0033,
    -0.00716,
    0.00551,
    -0.00105,
    0.00424,
    0.00327,
    -0.00212,
    -0.00515,
    -0.00655
   ],
   [
    0.00712,
    0.00204,
    0.00289,
    -0.00193,
    -0.00597,
    0.00536,
    0.00032,
    0.00643,
    0.00761,
    -0.00023,
    -0.00199,
    -0.00536,
    0.00461,
    1e-05
   ],
   [
    -0.00204,
    0.0027,
    -0.0016,
    0.00263,
    0.00264,
    -0.00345,
    -0.00506,
    -0.00291,
    0.0006,
    -0.00205,
    0.00036,
    -0.00092,
    -0.0052,
    -0.00543
   ],
   [
    -0.00925,
    0.00654,
    0.01601,
    0.00387,
    0.00669,
    0.00411,
    0.0016,
    -0.00292,
    0.00788,
    0.01432,
    0.0054,
    0.00807,
    0.00481,
    0.00334
   ],
   [
    0.0016,
    0.00743,
    0.00929,
    0.00254,
    0.00836,
    -0.0007,
    0.00455,
    0.00517,
    0.00262,
    0.00611,
    6e-05,
    0.00728,
    0.00036,
    0.00615
   ],
   [
    0.0039,
    0.00379,
    -0.01058,
    0.03315,
    0.00457,
    -0.01938,
    -0.03506,
    0.00264,
    0.00259,
    -0.01186,
    0.03293,
    0.00422,
    -0.01616,
    -0.02642
   ],
   [
    0.04195,
    -0.02179,
    -0.00686,
    0.00136,
    -0.0306,
    0.0015,
    -0.01227,
    0.03139,
    -0.01879,
    -0.00688,
    -0.00061,
    -0.03024,
    -0.00652,
    -0.01572
   ],
   [
    0.01063,
    0.011,
    0.00567,
    0.01152,
    -0.00863,
    -0.00093,
    0.0062,
    0.01135,
    0.0091,
    0.01062,
    0.01033,
    -0.00697,
    0.00196,
    0.00815
   ],
   [
    0.01882,
    -0.02126,
    0.00678,
    0.01083,
    -0.02947,
    -0.01125,
    0.02399,
    0.01888,
    -0.01209,
    0.00776,
    0.01301,
    -0.02139,
    -0.00688,
    0.02212
   ],
   [
    -0.001,
    0.00152,
    -0.00473,
    0.00557,
    0.00461,
    -0.00398,
    0.00855,
    4e-05,
    0.00785,
    0.00197,
    0.00585,
    0.00606,
    -0.00385,
    0.00771
   ],
   [
    0.00835,
    0.00653,
    0.00331,
    -0.01164,
    0.00841,
    0.00053,
    0.01037,
    0.00605,
    0.0051,
    0.00582,
    -0.01051,
    0.00855,
    0.0019,
    0.00983
   ],
   [
    -0.00332,
    -0.00479,
    -0.00443,
    -0.00808,
    0.00553,
    -0.00255,
    0.00346,
    -0.00182,
    -0.00164,
    -0.00839,
    -0.00923,
    0.00412,
    -0.00452,
    0.00078
   ],
   [
    0.01609,
    0.00177,
    0.01026,
    0.00243,
    0.00685,
    -0.00931,
    0.00601,
    0.01554,
    0.00054,
    0.00774,
    0.00134,
    0.00598,
    -0.00668,
    0.00669
   ],
   [
    -0.02343,
    0.02076,
    0.00378,
    -0.02445,
    0.00623,
    0.00194,
    0.01892,
    -0.02522,
    0.01773,
    0.0024,
    -0.01857,
    0.01043,
    0.00408,
    0.01846
   ],
   [
    0.00651,
    -0.00315,
    0.00061,
    -0.00054,
    -0.02334,
    -0.00238,
    -0.0011,
    0.0076,
    0.00022,
    0.00029,
    -0.00158,
    -0.02112,
    -0.00444,
    -0.00115
   ],
   [
    0.00217,
    0.00339,
    -0.00132,
    -0.01738,
    0.0043,
    0.00372,
    0.00197,
    0.00039,
    -0.00267,
    -0.00096,
    -0.01609,
    0.00046,
    0.00282,
    0.00145
   ],
   [
    0.00547,
    0.00334,
    -0.00388,
    0.01316,
    -0.01198,
    -0.00315,
    0.0142,
    0.00536,
    0.00134,
    -0.00425,
    0.00949,
    -0.01139,
    -0.00273,
    0.01111
   ],
   [
    0.00664,
    0.00675,
    0.00739,
    0.00953,
    0.00314,
    -0.0017,
    0.00169,
    0.00746,
    0.00406,
    0.00582,
    0.00915,
    0.00259,
    -0.00168,
    0.00298
   ],
   [
    -0.0028,
    -0.014,
    0.00111,
    -0.01152,
    -0.02532,
    0.00985,
    0.02059,
    -0.00111,
    -0.01147,
    0.00169,
    -0.01311,
    -0.02245,
    0.00815,
    0.01374
   ],
   [
    -0.00269,
    0.0143,
    -0.01325,
    -0.00266,
    0.00259,
    0.00699,
    0.00178,
    -0.00522,
    0.00943,
    -0.00449,
    0.00061,
    0.00532,
    0.00581,
    0.00326
   ],
   [
    -0.0007,
    -0.01781,
    0.00472,
    -0.04507,
    -0.01772,
    0.0555,
    0.0057,
    0.00211,
    -0.01862,
    0.00699,
    -0.04458,
    -0.01708,
    0.04728,
    0.00321
   ],
   [
    -0.0138,
    0.01206,
    0.01126,
    -0.00022,
    0.01355,
    -0.00363,
    -0.01233,
    -0.01199,
    0.00957,
    0.00755,
    -0.00219,
    0.00836,
    -0.00479,
    -0.01162
   ],
   [
    -0.00618,
    0.00171,
    0.00063,
    0.00044,
    0.0066,
    0.00556,
    0.00281,
    -0.00394,
    -0.00257,
    -0.0022,
    -0.00105,
    0.00269,
    0.00485,
    0.00414
   ],
   [
    -0.00328,
    -0.01023,
    0.00142,
    -0.00363,
    0.00425,
    -0.00088,
    0.00727,
    -0.00244,
    -0.01003,
    0.00123,
    -0.00256,
    0.00596,
    0.00207,
    0.00994
   ],
   [
    0.00178,
    0.01566,
    -0.00326,
    -0.00586,
    0.00037,
    0.00826,
    0.00896,
    0.00125,
    0.01228,
    0.00381,
    -0.00531,
    0.00099,
    0.00744,
    0.00818
   ],
   [
    -0.01115,
    0.00314,
    -0.00401,
    -0.0102,
    0.00186,
    0.00517,
    0.01628,
    -0.00805,
    -0.00061,
    -0.00471,
    -0.00931,
    0.00438,
    0.00635,
    0.01599
   ],
   [
    -0.00862,
    -0.01902,
    -0.01797,
    -0.00515,
    -0.01086,
    0.01326,
    0.01562,
    -0.00544,
    -0.01895,
    -0.01624,
    -0.00686,
    -0.00951,
    0.01048,
    0.01152
   ],
   [
    -0.00453,
    0.00367,
    0.01416,
    -0.00359,
    0.00149,
    0.00812,
    0.00623,
    -0.00499,
    0.00786,
    0.01458,
    -0.00174,
    0.00294,
    0.00973,
    0.00861
   ],
   [
    -0.00789,
    0.0014,
    -0.01154,
    0.00195,
    9e-05,
    -0.00765,
    -0.00352,
    -0.00317,
    -0.00362,
    -0.00677,
    0.00072,
    -0.00278,
    -0.00816,
    -0.0049
   ],
   [
    0.00736,
    0.00611,
    -0.00394,
    0.00185,
    -0.00362,
    0.00587,
    0.006,
    0.00857,
    0.00584,
    -0.00115,
    0.00155,
    -0.00288,
    0.00816,
    0.00666
   ],
   [
    0.02518,
    0.00834,
    0.00475,
    -0.00229,
    0.00861,
    -0.00887,
    0.00247,
    0.02057,
    0.00654,
    0.00032,
    -0.00105,
    0.0109,
    -0.00545,
    0.00399
   ],
   [
    -0.01107,
    0.00152,
    0.00946,
    -0.00089,
    -0.00881,
    -0.01026,
    -0.0102,
    -0.01009,
    -0.00373,
    0.00568,
    -0.00294,
    -0.0085,
    -0.01147,
    -0.01017
   ],
   [
    -0.00698,
    -0.01041,
    0.00457,
    0.0028,
    -0.00646,
    -0.00121,
    0.00457,
    -0.00894,
    -0.01101,
    0.00033,
    0.00332,
    -0.00404,
    -0.00011,
    0.00429
   ],
   [
    -0.01836,
    -0.01996,
    -0.00767,
    0.01063,
    -0.00469,
    -0.0018,
    -0.00877,
    -0.01248,
    -0.01757,
    -0.00494,
    0.01018,
    -0.00454,
    -0.0015,
    -0.00692
   ],
   [
    -0.01705,
    -0.02656,
    0.00016,
    0.00279,
    -0.00249,
    0.00739,
    -0.00645,
    -0.01683,
    -0.01855,
    -0.00033,
    0.0027,
    -0.00439,
    0.00541,
    -0.00733
   ],
   [
    -0.00197,
    -0.00337,
    0.00799,
    -0.00393,
    -0.00074,
    0.00058,
    0.00953,
    -0.00088,
    -0.00042,
    0.00514,
    -0.00487,
    -0.00077,
    -0.00171,
    0.00506
   ],
   [
    -0.00322,
    0.01152,
    0.00423,
    0.01229,
    0.00893,
    -0.00474,
    -0.00742,
    -0.00268,
    0.01153,
    0.00657,
    0.01114,
    0.00748,
    -0.00454,
    -0.00703
   ],
   [
    -0.02828,
    -0.00196,
    0.01996,
    -0.00023,
    -0.00298,
    0.00077,
    0.02744,
    -0.02251,
    0.00293,
    0.01692,
    0.00351,
    -0.00123,
    0.00264,
    0.02312
   ],
   [
    0.00584,
    -0.00321,
    0.00811,
    -0.00077,
    0.00663,
    -0.00228,
    -0.00555,
    0.00759,
    -0.00089,
    0.00483,
    0.00024,
    0.00701,
    0.00043,
    -0.00298
   ],
   [
    -0.01264,
    -0.00918,
    0.009,
    -0.00031,
    -0.00541,
    0.00496,
    0.02292,
    -0.00825,
    -0.00194,
    0.00674,
    0.00045,
    0.00033,
    0.00996,
    0.02361
   ],
   [
    -0.01013,
    0.00829,
    -0.00475,
    -0.00074,
    0.00453,
    -0.00355,
    0.0039,
    -0.00932,
    0.00599,
    -0.00579,
    -0.00214,
    0.00243,
    -0.00317,
    0.00296
   ],
   [
    0.00354,
    0.00736,
    0.00462,
    0.00118,
    -0.01122,
    0.00132,
    0.00449,
    0.00065,
    0.00648,
    0.00244,
    0.00328,
    -0.01082,
    -0.0011,
    0.003
   ],
   [
    0.02393,
    -0.01363,
    0.00373,
    0.00012,
    -0.00501,
    -0.01074,
    -0.00187,
    0.01726,
    -0.01515,
    -0.00141,
    -0.00104,
    -0.00441,
    -0.01307,
    -0.00443
   ],
   [
    -0.01045,
    -0.00889,
    0.00373,
    0.01018,
    -0.0077,
    -0.00952,
    0.00174,
    -0.00933,
    -0.00859,
    -0.00294,
    0.00972,
    -0.0061,
    -0.00849,
    0.00099
   ],
   [
    -0.02051,
    -0.0176,
    -0.00115,
    0.00252,
    -0.00519,
    -0.01058,
    -0.00699,
    -0.01662,
    -0.01515,
    -0.00547,
    9e-05,
    -0.00539,
    -0.01035,
    -0.00565
   ],
   [
    0.01675,
    0.00809,
    0.01424,
    -0.00176,
    0.0073,
    0.0086,
    0.00146,
    0.01436,
    0.01054,
    0.00851,
    -0.00178,
    0.00858,
    0.0119,
    0.00456
   ],
   [
    -0.01625,
    -0.00368,
    0.00086,
    -0.01588,
    0.0096,
    -0.00196,
    0.00913,
    -0.01154,
    -0.00275,
    -0.00059,
    -0.01433,
    0.00712,
    -0.00355,
    0.00629
   ],
   [
    -0.00911,
    0.0086,
    0.00061,
    -0.00971,
    -0.0035,
    -0.00381,
    0.00817,
    -0.01064,
    0.00294,
    0.00326,
    -0.0102,
    -0.00188,
    -0.00034,
    0.00834
   ],
   [
    -0.00064,
    0.01308,
    0.00156,
    0.00951,
    0.00708,
    0.00087,
    0.00497,
    9e-05,
    0.00545,
    -0.00222,
    0.00518,
    0.00657,
    0.00125,
    0.00403
   ],
   [
    -0.00038,
    -0.00263,
    -0.00813,
    -0.00207,
    -0.00458,
    0.0054,
    0.00067,
    -0.00475,
    -0.00032,
    -0.00705,
    -0.00111,
    -0.00478,
    0.00356,
    -0.00038
   ],
   [
    0.00642,
    -0.00218,
    -0.00432,
    0.01677,
    -0.00425,
    0.00494,
    -0.00731,
    0.00171,
    -0.00673,
    -0.00333,
    0.01641,
    -0.00366,
    0.00463,
    -0.00561
   ],
   [
    0.00987,
    0.0079,
    0.00921,
    0.00998,
    0.00318,
    -0.00609,
    -0.01763,
    0.01035,
    0.00578,
    0.01321,
    0.01143,
    0.00473,
    -0.00353,
    -0.01272
   ],
   [
    -0.00575,
    0.00352,
    -0.00352,
    -0.00611,
    0.0001,
    0.00649,
    0.00326,
    -0.00616,
    0.00462,
    -0.00269,
    -0.00504,
    -0.00016,
    0.00549,
    0.00149
   ],
   [
    0.00498,
    0.00164,
    -0.00132,
    0.00942,
    0.00217,
    0.00036,
    -0.00638,
    0.00471,
    0.0014,
    0.00196,
    0.00687,
    0.00104,
    0.0005,
    -0.00432
   ],
   [
    0.00126,
    0.00958,
    -0.00912,
    -0.00912,
    0.01293,
    -0.0017,
    0.01625,
    -0.00028,
    0.01166,
    -0.00507,
    -0.0084,
    0.01267,
    0.00152,
    0.01446
   ],
   [
    0.00851,
    0.01745,
    0.01355,
    -0.01109,
    -0.00136,
    0.00796,
    0.00575,
    0.00803,
    0.01313,
    0.01117,
    -0.01206,
    -0.0005,
    0.00724,
    0.00439
   ],
   [
    -0.00101,
    0.00654,
    -0.00129,
    -0.00031,
    0.00645,
    0.00523,
    -0.00945,
    -0.00294,
    0.00152,
    -0.00643,
    -0.00252,
    0.00386,
    0.00211,
    -0.00758
   ],
   [
    0.00195,
    0.00363,
    3e-05,
    0.00464,
    0.00113,
    -0.01387,
    -0.00542,
    0.00217,
    0.00031,
    -0.00357,
    0.00366,
    0.00015,
    -0.01494,
    -0.00663
   ],
   [
    0.02145,
    0.01273,
    0.00261,
    0.01192,
    -0.00658,
    0.00024,
    0.00397,
    0.0206,
    0.01161,
    0.00525,
    0.00994,
    -0.00561,
    0.00061,
    0.0031
   ],
   [
    -0.01705,
    -0.01523,
    -0.01064,
    0.0114,
    0.00111,
    -0.00954,
    -0.00165,
    -0.0146,
    -0.02053,
    -0.01219,
    0.00679,
    -0.00339,
    -0.00841,
    -0.00413
   ],
   [
    -0.00032,
    0.0194,
    -0.01264,
    0.00297,
    0.00122,
    -0.00624,
    -0.0113,
    0.00051,
    0.01084,
    -0.00781,
    -0.00029,
    -0.0008,
    -0.00751,
    -0.01154
   ],
   [
    -0.00853,
    0.02847,
    0.00527,
    -0.00819,
    0.01834,
    0.00153,
    -0.00113,
    -0.00323,
    0.0228,
    0.00289,
    -0.00946,
    0.0173,
    0.00279,
    0.00073
   ],
   [
    0.02202,
    0.00669,
    0.00117,
    0.00699,
    -0.00654,
    -0.00741,
    -0.00491,
    0.01732,
    0.00277,
    0.00368,
    0.0057,
    -0.00748,
    -0.00678,
    -0.00529
   ],
   [
    0.0014,
    -0.0136,
    -0.01935,
    0.00565,
    -0.00961,
    0.00257,
    -0.00914,
    -0.00183,
    -0.00513,
    -0.01629,
    0.00679,
    -0.00811,
    0.00266,
    -0.00781
   ],
   [
    0.00589,
    0.02204,
    0.00588,
    -0.00087,
    0.00623,
    -0.01416,
    -0.0071,
    0.00678,
    0.01299,
    0.00301,
    -0.00355,
    0.00225,
    -0.01462,
    -0.00821
   ],
   [
    -0.00036,
    -0.01284,
    -0.00174,
    0.00178,
    -0.00225,
    0.00125,
    0.01539,
    -0.0031,
    -0.00775,
    0.00347,
    0.00215,
    0.0003,
    0.00235,
    0.01471
   ],
   [
    0.00687,
    -0.00774,
    0.0023,
    0.00543,
    -0.00872,
    -0.01088,
    0.01496,
    0.00485,
    -0.00931,
    0.00593,
    0.00595,
    -0.00812,
    -0.00951,
    0.01373
   ],
   [
    0.00036,
    -0.0036,
    0.00281,
    0.00958,
    0.0013,
    0.00062,
    -0.00797,
    -0.00065,
    -0.00633,
    -0.00509,
    0.00483,
    6e-05,
    -0.0008,
    -0.00689
   ],
   [
    -0.01841,
    0.0018,
    -0.00326,
    0.0079,
    -0.00901,
    -0.00011,
    0.00192,
    -0.01581,
    -0.00396,
    -0.0039,
    0.00645,
    -0.00669,
    0.00156,
    0.00297
   ],
   [
    -0.0028,
    0.03019,
    0.00026,
    -0.0077,
    0.01499,
    0.01447,
    0.00308,
    -0.0028,
    0.02995,
    0.00592,
    -0.00497,
    0.01548,
    0.01278,
    0.00346
   ],
   [
    0.00457,
    0.01642,
    0.00171,
    -0.00563,
    0.01269,
    -0.00094,
    -0.00945,
    0.00413,
    0.0137,
    -0.00091,
    -0.00767,
    0.00862,
    -0.00124,
    -0.00872
   ],
   [
    -0.01401,
    -0.01716,
    0.00418,
    -0.00358,
    -0.00256,
    0.00136,
    -0.00541,
    -0.01274,
    -0.0105,
    0.00518,
    -0.00394,
    -0.00179,
    0.00196,
    -0.00429
   ],
   [
    -0.0193,
    0.00176,
    -0.00021,
    -0.01193,
    0.00421,
    -0.00108,
    0.00551,
    -0.01354,
    0.00197,
    -0.00302,
    -0.01186,
    0.00388,
    -0.00219,
    0.00423
   ],
   [
    -0.01698,
    -0.00542,
    -0.01407,
    0.00541,
    0.00196,
    -0.00121,
    -0.01185,
    -0.01254,
    0.00178,
    -0.00713,
    0.00677,
    0.00288,
    0.00039,
    -0.00891
   ],
   [
    -0.00042,
    0.00732,
    -0.00311,
    0.01109,
    0.00234,
    -0.00676,
    -0.01676,
    0.00234,
    0.00369,
    -0.00214,
    0.00507,
    -0.00245,
    -0.0077,
    -0.01563
   ],
   [
    0.0153,
    0.00093,
    -0.00668,
    -0.00252,
    0.0029,
    0.00415,
    0.0042,
    0.01799,
    0.00626,
    0.00127,
    -0.00049,
    0.00491,
    0.00542,
    0.00598
   ],
   [
    0.00593,
    0.00548,
    0.00389,
    -0.01616,
    0.00098,
    -0.00292,
    0.00364,
    0.00921,
    0.00104,
    -0.00032,
    -0.0136,
    0.00244,
    -0.00558,
    0.00102
   ],
   [
    -0.00071,
    0.00109,
    -0.00819,
    -0.005,
    -0.0021,
    -0.00402,
    0.01741,
    0.00143,
    0.00184,
    -0.00534,
    -0.00286,
    -0.00044,
    -0.00204,
    0.01509
   ],
   [
    -0.0105,
    0.0171,
    0.00337,
    0.0033,
    0.01521,
    -0.00026,
    -0.01172,
    -0.00988,
    0.01341,
    6e-05,
    0.00339,
    0.01381,
    0.00113,
    -0.00999
   ],
   [
    -0.01552,
    0.01719,
    -0.00777,
    0.0047,
    -0.00145,
    -0.00125,
    0.00645,
    -0.01434,
    0.01283,
    -0.00627,
    0.00296,
    -0.00033,
    -0.00039,
    0.0069
   ],
   [
    0.01153,
    0.01183,
    -0.00206,
    -0.01119,
    0.01468,
    0.00872,
    -0.00311,
    0.00828,
    0.009,
    -0.00017,
    -0.00856,
    0.01375,
    0.0074,
    -0.00049
   ],
   [
    0.00661,
    0.00097,
    -0.00565,
    -0.00346,
    -0.00179,
    0.00158,
    -0.00225,
    0.0078,
    0.00127,
    -0.00406,
    -0.00363,
    -0.00171,
    -0.00055,
    -0.00281
   ],
   [
    0.01434,
    0.01231,
    0.00563,
    0.00172,
    0.00493,
    0.005,
    -0.00194,
    0.01368,
    0.01192,
    0.00528,
    0.00037,
    0.00309,
    0.00475,
    -0.00199
   ],
   [
    0.00531,
    -0.01682,
    0.00451,
    -0.00579,
    -0.0099,
    0.00657,
    0.01014,
    0.0012,
    -0.01471,
    0.0034,
    -0.00468,
    -0.00943,
    0.00682,
    0.00727
   ],
   [
    -0.01569,
    0.00305,
    -0.01067,
    -0.00422,
    0.00564,
    -0.00759,
    0.00846,
    -0.01468,
    0.00216,
    -0.00911,
    -0.00242,
    0.00761,
    -0.0041,
    0.00981
   ],
   [
    -0.015,
    0.0037,
    -0.00545,
    0.00185,
    -0.00153,
    0.00185,
    -0.00262,
    -0.017,
    0.00045,
    -0.01204,
    0.00016,
    0.00166,
    0.00471,
    0.0004
   ],
   [
    -0.00881,
    -0.00606,
    -0.007,
    0.00574,
    0.01188,
    -0.02569,
    -0.01914,
    -0.00799,
    -0.00836,
    -0.007,
    0.00697,
    0.00964,
    -0.02101,
    -0.01504
   ],
   [
    0.01251,
    0.00075,
    0.00177,
    -0.00658,
    -0.00561,
    0.00429,
    0.00785,
    0.01024,
    0.00071,
    0.002,
    -0.00743,
    -0.00448,
    0.00277,
    0.00552
   ],
   [
    -0.01249,
    -0.00905,
    0.00332,
    -0.00024,
    -0.00365,
    0.00797,
    0.00988,
    -0.01111,
    -0.00503,
    0.00959,
    -0.00034,
    -0.00274,
    0.00775,
    0.00881
   ],
   [
    0.01896,
    0.00408,
    -0.00277,
    0.00663,
    -0.01445,
    -0.0045,
    0.00549,
    0.01528,
    0.00438,
    0.00164,
    0.0071,
    -0.01089,
    -0.00329,
    0.00389
   ],
   [
    0.01638,
    -0.01969,
    0.01317,
    -0.00086,
    -0.00824,
    0.02095,
    -0.01569,
    0.01364,
    -0.02046,
    0.01091,
    0.00187,
    -0.00699,
    0.01911,
    -0.01319
   ],
   [
    0.01081,
    0.00035,
    0.01153,
    0.00291,
    -0.01102,
    0.0026,
    -0.00151,
    0.00938,
    -0.0002,
    0.00607,
    0.00449,
    -0.00963,
    0.00268,
    -0.00025
   ],
   [
    -0.01263,
    0.02257,
    0.01486,
    -0.03178,
    0.00393,
    0.02472,
    0.01436,
    -0.00791,
    0.02258,
    0.00991,
    -0.02879,
    0.00498,
    0.02318,
    0.01311
   ],
   [
    0.00509,
    0.00632,
    -0.0025,
    0.01282,
    -0.01016,
    -0.00037,
    -0.00251,
    0.00517,
    0.00411,
    -0.00752,
    0.01106,
    -0.0093,
    -0.00231,
    -0.0019
   ],
   [
    -0.02764,
    -0.00707,
    0.00089,
    -0.01374,
    -0.00879,
    0.01782,
    0.00101,
    -0.02283,
    -0.00595,
    0.00372,
    -0.01243,
    -0.00657,
    0.01473,
    0.00338
   ],
   [
    0.00462,
    0.00138,
    0.00583,
    0.00095,
    -0.00085,
    -0.00384,
    -0.00161,
    0.00618,
    0.00377,
    0.00334,
    0.00031,
    0.00015,
    -0.00307,
    -0.0011
   ],
   [
    0.01178,
    0.01018,
    0.0041,
    0.00312,
    -0.00183,
    0.00151,
    -0.00667,
    0.01335,
    0.00746,
    0.00452,
    0.00255,
    -0.00207,
    0.00108,
    -0.0061
   ],
   [
    -0.00592,
    -0.00582,
    -0.00171,
    0.01061,
    0.00187,
    -0.01055,
    -0.0105,
    -0.00541,
    -0.00414,
    -0.00271,
    0.0106,
    0.00249,
    -0.00725,
    -0.00774
   ],
   [
    -0.02468,
    -0.01151,
    -0.00043,
    0.00448,
    -7e-05,
    -0.0014,
    -0.01462,
    -0.02123,
    -0.01279,
    -0.0004,
    0.00417,
    -0.00064,
    -0.00033,
    -0.013
   ],
   [
    -0.00877,
    0.00391,
    -0.00736,
    -0.00171,
    0.0102,
    -0.00848,
    0.00485,
    -0.00734,
    0.01018,
    -0.00407,
    -0.00147,
    0.00657,
    -0.00657,
    0.00537
   ],
   [
    -0.01003,
    -0.00714,
    0.00106,
    -0.00237,
    0.00337,
    0.00655,
    -0.00504,
    -0.00849,
    -0.0084,
    -0.00085,
    -0.0035,
    0.00254,
    0.00608,
    -0.00309
   ],
   [
    0.0023,
    -0.01094,
    0.00103,
    0.0047,
    -0.02016,
    0.01482,
    0.00703,
    -0.00053,
    -0.00989,
    -0.0009,
    0.00204,
    -0.01889,
    0.01149,
    0.00419
   ],
   [
    -0.00351,
    -0.0169,
    -0.00175,
    0.01188,
    -0.00863,
    -0.00493,
    -0.00702,
    -0.00648,
    -0.01342,
    -8e-05,
    0.01165,
    -0.00769,
    -0.00492,
    -0.00615
   ],
   [
    -0.01425,
    0.00719,
    0.00231,
    -0.00441,
    0.01022,
    -0.00141,
    -0.0038,
    -0.01016,
    0.00583,
    -0.0004,
    -0.00461,
    0.01011,
    0.00106,
    -0.0013
   ],
   [
    -0.00398,
    -0.00538,
    -0.00878,
    0.00335,
    0.00453,
    -0.00636,
    -0.01224,
    -0.00452,
    -0.00843,
    -0.00964,
    0.00379,
    0.00253,
    -0.00325,
    -0.00789
   ],
   [
    0.00692,
    -0.0136,
    0.0041,
    -0.00525,
    -0.00854,
    0.00027,
    -0.00515,
    0.00639,
    -0.00956,
    0.00475,
    -0.00277,
    -0.00476,
    0.00059,
    -0.00374
   ],
   [
    -0.00314,
    0.01058,
    -0.00937,
    -0.00081,
    0.0096,
    -0.02139,
    0.00274,
    -0.00444,
    0.00169,
    -0.01289,
    -0.00214,
    0.00705,
    -0.02223,
    0.00018
   ],
   [
    0.00481,
    -0.00598,
    0.00125,
    0.00385,
    -0.01028,
    -0.00247,
    0.00747,
    0.00386,
    -0.00068,
    0.00198,
    0.00455,
    -0.01109,
    -0.00328,
    0.00635
   ],
   [
    0.00132,
    -0.02112,
    -0.0023,
    -0.00854,
    -0.00314,
    0.00651,
    0.00017,
    0.00171,
    -0.01262,
    0.00196,
    -0.00773,
    -0.00464,
    0.00405,
    -0.00092
   ],
   [
    -0.00345,
    -0.00905,
    0.00027,
    -0.01183,
    0.00039,
    0.0003,
    0.00831,
    -0.00113,
    -0.00497,
    0.00158,
    -0.00648,
    0.00168,
    0.00087,
    0.00736
   ],
   [
    0.02172,
    0.01401,
    0.0168,
    -0.00677,
    -0.00061,
    0.00575,
    0.00345,
    0.01743,
    0.01602,
    0.01108,
    -0.00505,
    0.00075,
    0.00608,
    0.00242
   ],
   [
    -0.01067,
    -0.00066,
    -0.00728,
    0.01024,
    -0.00591,
    -0.00781,
    -6e-05,
    -0.0088,
    0.0027,
    -0.00804,
    0.00898,
    -0.00488,
    -0.00736,
    -0.00091
   ],
   [
    -0.00941,
    0.00407,
    -0.0024,
    0.00072,
    0.00329,
    -0.0092,
    -0.00572,
    -0.00899,
    -0.0048,
    -0.00318,
    -0.00094,
    0.0003,
    -0.00997,
    -0.00604
   ],
   [
    0.00375,
    -0.01827,
    0.00898,
    -0.01102,
    -0.00911,
    0.00507,
    0.01015,
    0.00489,
    -0.01042,
    0.00801,
    -0.00841,
    -0.0076,
    0.00431,
    0.00748
   ],
   [
    0.00065,
    0.01203,
    -9e-05,
    0.00063,
    0.00281,
    -0.00684,
    -0.00826,
    0.0008,
    0.01186,
    -0.0006,
    0.0007,
    0.00302,
    -0.00507,
    -0.00578
   ],
   [
    0.00708,
    0.00271,
    0.00777,
    -0.00585,
    -0.01094,
    0.0015,
    0.01453,
    0.0049,
    0.00149,
    0.00439,
    -0.00792,
    -0.01207,
    8e-05,
    0.01091
   ],
   [
    0.0021,
    -0.0033,
    -0.00844,
    -0.00321,
    0.00375,
    0.00164,
    0.00458,
    0.00012,
    -0.00191,
    -0.00579,
    -0.00246,
    0.00545,
    0.00373,
    0.00595
   ],
   [
    0.01222,
    0.00081,
    -0.02043,
    0.01754,
    0.02191,
    -0.01577,
    -0.01255,
    0.00554,
    -0.00339,
    -0.01841,
    0.01652,
    0.0192,
    -0.01104,
    -0.00865
   ],
   [
    0.00592,
    0.00399,
    0.00216,
    -0.00764,
    0.00832,
    -0.00041,
    0.00735,
    0.00875,
    0.00512,
    0.00307,
    -0.00697,
    0.00727,
    -0.00155,
    0.00607
   ],
   [
    -0.01551,
    -0.00443,
    0.0022,
    -0.02051,
    0.00023,
    0.00532,
    0.00313,
    -0.01173,
    -0.00553,
    0.00211,
    -0.01923,
    -0.00015,
    0.00591,
    0.00407
   ],
   [
    -0.00034,
    0.01117,
    -0.01583,
    0.01563,
    -0.00481,
    -0.00164,
    -0.00636,
    0.0024,
    0.01276,
    -0.00859,
    0.01573,
    -0.00239,
    0.00101,
    -0.00112
   ],
   [
    -0.00122,
    0.00244,
    -0.00637,
    -0.00279,
    0.003,
    0.00577,
    -0.00358,
    -0.00076,
    0.00227,
    -0.00242,
    -0.00343,
    0.00195,
    0.00565,
    -0.00371
   ],
   [
    0.00724,
    -0.02363,
    0.00345,
    -0.00376,
    -0.0085,
    -0.01077,
    0.00499,
    0.00232,
    -0.02014,
    0.00381,
    -0.00256,
    -0.00781,
    -0.01267,
    0.00126
   ],
   [
    -0.00843,
    -0.01318,
    0.00571,
    -0.00225,
    -0.00803,
    0.00356,
    -0.00124,
    -0.00732,
    -0.0109,
    0.00354,
    -0.00232,
    -0.00901,
    -0.00057,
    -0.00298
   ],
   [
    -0.00076,
    0.00963,
    -0.00147,
    -0.00291,
    0.00301,
    0.0036,
    -0.00967,
    -0.00148,
    0.0052,
    -0.00081,
    -0.00183,
    0.00149,
    0.00106,
    -0.00919
   ],
   [
    0.00829,
    -0.00841,
    0.00805,
    0.00011,
    0.0005,
    0.0094,
    -0.00932,
    0.0064,
    -0.00733,
    0.00191,
    0.00013,
    -0.00283,
    0.00636,
    -0.00823
   ],
   [
    0.01343,
    0.00555,
    0.00275,
    -0.00581,
    0.00677,
    0.01154,
    0.00457,
    0.01363,
    0.00618,
    0.00418,
    -0.00432,
    0.00609,
    0.0104,
    0.0031
   ],
   [
    -0.01843,
    0.00814,
    -0.00177,
    -0.0045,
    0.00213,
    -0.00681,
    0.00349,
    -0.01616,
    0.00186,
    -0.0013,
    -0.00377,
    0.00346,
    -0.00615,
    0.00138
   ],
   [
    -0.00613,
    0.01176,
    -0.00988,
    -0.00742,
    -0.00262,
    -0.00279,
    0.0081,
    -0.00407,
    0.00763,
    -0.0045,
    -0.00867,
    -0.00441,
    -0.00586,
    0.00419
   ],
   [
    -0.00458,
    -0.01708,
    -0.00909,
    -0.00893,
    -0.00294,
    0.0006,
    0.01322,
    -0.00575,
    -0.00876,
    -0.00678,
    -0.00753,
    -0.00266,
    0.00101,
    0.01148
   ],
   [
    0.00011,
    0.00268,
    -0.00319,
    -0.0097,
    0.00884,
    -0.00104,
    0.00804,
    -0.00152,
    0.00023,
    -0.0036,
    -0.00826,
    0.00972,
    -0.00108,
    0.00555
   ],
   [
    0.02992,
    0.00114,
    -0.00362,
    0.01558,
    -0.00321,
    -0.0094,
    0.00039,
    0.02138,
    -0.0001,
    2e-05,
    0.01255,
    -0.00333,
    -0.00742,
    0.00156
   ],
   [
    -0.00521,
    -0.00306,
    0.00836,
    0.00405,
    -0.00173,
    0.00645,
    -0.00169,
    -0.00428,
    -0.00197,
    0.00334,
    0.00216,
    -0.00458,
    0.00476,
    -0.00124
   ],
   [
    0.00955,
    0.00548,
    -0.00139,
    0.00237,
    0.00648,
    -0.00689,
    -0.00633,
    0.01098,
    0.00693,
    0.00154,
    0.00311,
    0.00636,
    -0.00445,
    -0.00441
   ],
   [
    0.00294,
    0.00428,
    -0.00834,
    0.00233,
    -0.00292,
    0.0019,
    -0.01488,
    -0.0032,
    -0.00037,
    -0.00216,
    0.00269,
    -0.00205,
    0.00198,
    -0.01314
   ],
   [
    -0.01151,
    -0.01497,
    0.00146,
    -0.00465,
    -0.01014,
    0.00082,
    0.00295,
    -0.00963,
    -0.01825,
    -0.00205,
    -0.00433,
    -0.00925,
    -1e-05,
    0.00182
   ],
   [
    -0.00823,
    -0.0181,
    0.00201,
    -0.0034,
    -0.00832,
    -0.00228,
    -0.01288,
    -0.00891,
    -0.0196,
    -7e-05,
    -0.00629,
    -0.01163,
    -0.00574,
    -0.0136
   ],
   [
    0.01479,
    0.01417,
    -0.00133,
    -0.00114,
    0.00034,
    -4e-05,
    0.00521,
    0.01091,
    0.01538,
    0.00474,
    0.00012,
    0.00191,
    0.00042,
    0.00501
   ],
   [
    -0.01471,
    0.00236,
    0.00767,
    -0.00783,
    0.00598,
    0.00986,
    -0.01316,
    -0.01156,
    -0.00285,
    0.00409,
    -0.00889,
    0.00498,
    0.00811,
    -0.00967
   ],
   [
    -0.00578,
    -0.01851,
    -0.00481,
    0.00525,
    -0.00667,
    -0.00755,
    -0.00722,
    -0.00582,
    -0.01564,
    -0.00348,
    0.00369,
    -0.00701,
    -0.00796,
    -0.00695
   ],
   [
    -0.00832,
    0.01956,
    0.00198,
    0.00419,
    0.00539,
    -0.00126,
    -0.00427,
    -0.00812,
    0.01791,
    0.00404,
    0.00077,
    0.00161,
    -0.00321,
    -0.00406
   ],
   [
    0.00299,
    0.02838,
    -0.00837,
    -0.00568,
    0.01743,
    -0.0164,
    -0.00784,
    0.00313,
    0.02108,
    -0.00648,
    -0.0038,
    0.01643,
    -0.01544,
    -0.00535
   ],
   [
    -5e-05,
    0.03463,
    0.00777,
    -0.01007,
    0.01665,
    0.00471,
    0.01131,
    0.0006,
    0.03225,
    0.00415,
    -0.00789,
    0.01739,
    0.00537,
    0.01157
   ],
   [
    0.0091,
    0.00895,
    -0.00113,
    -0.00147,
    -0.00966,
    0.00055,
    0.00602,
    0.00784,
    0.0069,
    -0.00089,
    -0.00238,
    -0.00935,
    -0.00264,
    0.00165
   ],
   [
    0.01702,
    -0.00935,
    0.00083,
    -0.01012,
    0.00375,
    0.00976,
    0.00589,
    0.01287,
    -0.00452,
    0.0015,
    -0.00885,
    0.00445,
    0.00876,
    0.00383
   ],
   [
    0.00332,
    0.00688,
    0.01399,
    -0.01,
    0.00587,
    0.01311,
    -0.00177,
    0.00288,
    0.00564,
    0.01017,
    -0.00755,
    0.0075,
    0.01131,
    -0.00098
   ],
   [
    -0.00146,
    0.00543,
    -0.00296,
    -0.00038,
    0.00796,
    0.00071,
    -0.02025,
    -0.00271,
    0.00086,
    -0.00245,
    -0.00157,
    0.00686,
    0.00087,
    -0.01719
   ],
   [
    -0.01145,
    0.00767,
    -0.01357,
    0.00871,
    0.00527,
    -0.003,
    0.00048,
    -0.00824,
    0.0046,
    -0.01191,
    0.00616,
    0.00452,
    -0.00227,
    0.0007
   ],
   [
    -0.01834,
    -0.00688,
    -0.00136,
    0.00903,
    0.00101,
    -0.01038,
    -0.00829,
    -0.01791,
    -0.00495,
    -0.00262,
    0.01066,
    0.0022,
    -0.01011,
    -0.00635
   ],
   [
    -0.01879,
    -0.01475,
    -0.00099,
    -0.00027,
    -0.01053,
    0.01094,
    -0.0006,
    -0.01376,
    -0.01344,
    -0.00119,
    0.00012,
    -0.00831,
    0.01085,
    8e-05
   ],
   [
    -0.01166,
    -0.00039,
    0.00373,
    -0.00103,
    0.00421,
    -0.00076,
    0.00328,
    -0.00558,
    0.00676,
    0.00393,
    0.00085,
    0.00637,
    0.00094,
    0.00519
   ],
   [
    -0.01882,
    0.0217,
    0.01678,
    -0.02284,
    0.00019,
    0.02894,
    0.01335,
    -0.0118,
    0.01701,
    0.00677,
    -0.02211,
    0.00096,
    0.02503,
    0.00936
   ],
   [
    -0.01297,
    -0.00831,
    -0.0058,
    -0.00558,
    0.00848,
    -0.00046,
    -0.00026,
    -0.01093,
    -0.0115,
    -0.00772,
    -0.00672,
    0.00701,
    -0.00152,
    -0.00176
   ],
   [
    -0.00247,
    0.00816,
    0.00919,
    0.01675,
    -0.00276,
    -0.00537,
    -0.00524,
    -0.00278,
    0.00125,
    0.00245,
    0.01558,
    -0.00224,
    -0.00407,
    -0.00277
   ],
   [
    -0.00712,
    0.01163,
    -0.00145,
    0.00156,
    -0.00055,
    0.00868,
    0.00524,
    -0.00749,
    0.01609,
    0.00157,
    0.00039,
    -0.00133,
    0.00538,
    0.00123
   ],
   [
    -0.01831,
    -0.00032,
    0.0037,
    -0.00863,
    0.01157,
    -0.00514,
    -0.00244,
    -0.02148,
    -0.00467,
    0.0026,
    -0.00947,
    0.00738,
    -0.00522,
    -0.00224
   ],
   [
    -0.02016,
    -0.0018,
    -0.01176,
    -0.00822,
    -0.00287,
    0.0092,
    0.01613,
    -0.01708,
    -0.00533,
    -0.00427,
    -0.00639,
    -0.00391,
    0.00737,
    0.01246
   ],
   [
    0.00472,
    0.00388,
    0.00056,
    0.00737,
    -0.00066,
    -0.00482,
    -0.00518,
    0.00308,
    5e-05,
    0.00064,
    0.00498,
    -0.00131,
    -0.00451,
    -0.00359
   ],
   [
    -0.00416,
    0.00767,
    -0.01338,
    -0.00519,
    -0.00782,
    -0.00374,
    0.00612,
    -0.00684,
    0.0074,
    -0.00496,
    -0.00386,
    -0.00512,
    -0.0028,
    0.00371
   ],
   [
    -0.01327,
    -0.02227,
    -0.0008,
    0.00308,
    -0.00505,
    0.0092,
    0.00507,
    -0.00921,
    -0.01537,
    -0.00075,
    0.00246,
    -0.00431,
    0.00988,
    0.00551
   ],
   [
    0.00542,
    0.00446,
    0.00818,
    0.00825,
    -0.00325,
    -0.00225,
    -0.01201,
    0.00705,
    0.00457,
    0.00907,
    0.00515,
    -0.00387,
    -0.00429,
    -0.01129
   ],
   [
    0.00588,
    0.00932,
    0.00871,
    -0.00231,
    -0.00811,
    0.00338,
    0.01288,
    0.00256,
    0.00818,
    0.00515,
    0.00119,
    -0.00264,
    0.00566,
    0.0141
   ],
   [
    -0.00573,
    0.00198,
    -0.01071,
    -0.01115,
    0.00132,
    0.00393,
    -0.00359,
    -0.00592,
    -0.00242,
    -0.01108,
    -0.01062,
    0.00111,
    0.00094,
    -0.00644
   ],
   [
    0.00233,
    -0.01635,
    0.01519,
    0.01005,
    -0.00443,
    0.00631,
    -0.00301,
    0.00226,
    -0.00755,
    0.01259,
    0.01105,
    -0.0034,
    0.00478,
    -0.00375
   ],
   [
    -0.00675,
    -0.00793,
    0.01686,
    0.00511,
    -0.00352,
    -0.01145,
    -0.00817,
    -0.00263,
    -0.0065,
    0.01104,
    0.00588,
    -0.00411,
    -0.01144,
    -0.00605
   ],
   [
    -0.00221,
    0.03045,
    -0.0024,
    -0.00569,
    0.01049,
    0.00834,
    -0.01654,
    -0.00403,
    0.02007,
    -0.00431,
    -0.00701,
    0.00953,
    0.00903,
    -0.01157
   ],
   [
    -0.01084,
    3e-05,
    0.00215,
    0.01238,
    -0.01573,
    0.00077,
    0.00023,
    -0.00863,
    0.00014,
    -0.00113,
    0.01268,
    -0.01324,
    0.00158,
    0.00046
   ],
   [
    0.00501,
    0.01509,
    0.00119,
    0.00205,
    0.0051,
    0.00101,
    -0.00267,
    0.00447,
    0.01235,
    0.00199,
    0.00445,
    0.00428,
    0.00103,
    -0.00347
   ],
   [
    -0.00347,
    -0.01031,
    -0.00252,
    0.00257,
    0.01635,
    -0.01281,
    -0.00319,
    -0.00635,
    -0.00811,
    -0.00073,
    0.00185,
    0.01351,
    -0.00979,
    -0.00302
   ],
   [
    -0.00188,
    0.0036,
    -0.01011,
    0.00106,
    0.00249,
    0.00684,
    -0.00092,
    -0.00374,
    0.00203,
    -0.00625,
    0.00082,
    0.0028,
    0.00391,
    0.0003
   ],
   [
    -0.01248,
    0.0076,
    0.00113,
    0.00739,
    0.00334,
    0.00923,
    0.0016,
    -0.01097,
    0.00973,
    0.0028,
    0.0066,
    0.00461,
    0.00976,
    0.00275
   ],
   [
    -0.00062,
    0.00138,
    -0.01285,
    -0.00044,
    0.00087,
    -0.00128,
    0.01057,
    -0.00063,
    0.00398,
    -0.01127,
    -0.00098,
    0.00015,
    -0.00328,
    0.00833
   ],
   [
    0.01037,
    -0.00036,
    0.00207,
    -0.00661,
    -0.00492,
    0.00722,
    0.00123,
    0.01058,
    0.00218,
    0.00296,
    -0.00769,
    -0.00395,
    0.00569,
    3e-05
   ],
   [
    -0.01346,
    0.00054,
    0.00538,
    -0.00859,
    -0.00197,
    0.01309,
    0.01641,
    -0.01025,
    0.0002,
    0.00463,
    -0.00865,
    -0.0019,
    0.01141,
    0.01283
   ],
   [
    0.01051,
    0.02293,
    -0.00337,
    0.0053,
    0.01598,
    -0.00823,
    0.0058,
    0.0107,
    0.01839,
    -0.00116,
    0.00606,
    0.01469,
    -0.00446,
    0.00694
   ],
   [
    0.02393,
    -0.00431,
    0.00401,
    0.01164,
    -0.01564,
    0.00337,
    -0.00915,
    0.01567,
    -0.00218,
    0.00441,
    0.01115,
    -0.01355,
    0.00079,
    -0.00908
   ],
   [
    0.01156,
    0.00343,
    -0.01126,
    0.00455,
    0.00079,
    -0.00705,
    0.00151,
    0.01019,
    0.00096,
    -0.00773,
    0.00278,
    -0.00207,
    -0.00841,
    0.0011
   ],
   [
    -0.00737,
    0.01999,
    0.00608,
    0.00908,
    0.00778,
    -0.01195,
    -0.01324,
    -0.00221,
    0.01918,
    0.00433,
    0.00761,
    0.0058,
    -0.00969,
    -0.01141
   ],
   [
    -0.01146,
    0.01633,
    -0.0267,
    -0.0115,
    0.02094,
    0.00519,
    -0.01135,
    -0.01235,
    0.00919,
    -0.01838,
    -0.01046,
    0.01976,
    0.00571,
    -0.01067
   ],
   [
    -0.01236,
    -0.01312,
    -0.01123,
    0.0032,
    0.01037,
    -0.00305,
    -0.00993,
    -0.00851,
    -0.00981,
    -0.00933,
    0.00221,
    0.00475,
    -0.00422,
    -0.009
   ],
   [
    -0.01478,
    -0.0162,
    0.00849,
    -0.00656,
    -0.00488,
    -0.00908,
    0.001,
    -0.01252,
    -0.01587,
    0.00125,
    -0.00726,
    -0.0044,
    -0.00929,
    0.00123
   ],
   [
    0.0026,
    -0.03152,
    0.00033,
    0.01285,
    -0.02501,
    -0.00039,
    0.01523,
    0.00125,
    -0.02138,
    -0.0037,
    0.01154,
    -0.02114,
    -0.00168,
    0.01163
   ],
   [
    -0.02054,
    0.01107,
    0.01242,
    -0.01147,
    0.00917,
    -0.00883,
    0.00255,
    -0.01693,
    0.01009,
    0.01063,
    -0.01045,
    0.01076,
    -0.00656,
    0.00506
   ],
   [
    -0.00918,
    0.01419,
    -0.01543,
    -0.01963,
    0.01712,
    0.0002,
    -0.00951,
    -0.00725,
    0.01207,
    -0.0152,
    -0.0192,
    0.01369,
    -0.00217,
    -0.01007
   ],
   [
    -0.00447,
    -0.00745,
    0.00977,
    -0.00537,
    0.00168,
    0.01437,
    -0.00016,
    -0.00076,
    -0.00711,
    0.00957,
    -0.00702,
    0.00086,
    0.01284,
    0.00021
   ],
   [
    0.00782,
    -0.00191,
    0.00378,
    -0.00392,
    0.01027,
    0.00113,
    0.00186,
    0.00558,
    -0.00348,
    -0.00012,
    -0.00253,
    0.00961,
    0.00154,
    0.00111
   ],
   [
    0.01199,
    -0.00931,
    -0.00107,
    0.01041,
    -0.01247,
    -0.0005,
    0.00715,
    0.00769,
    -0.0034,
    0.00601,
    0.01201,
    -0.00651,
    0.00497,
    0.00935
   ],
   [
    0.00437,
    -0.00132,
    -0.00447,
    -0.00192,
    0.0151,
    -0.00348,
    0.0035,
    0.00207,
    0.00762,
    -0.00032,
    0.00119,
    0.01358,
    -0.00216,
    0.00244
   ],
   [
    -0.0299,
    -0.01246,
    0.00557,
    -0.00642,
    -0.00504,
    -0.01026,
    8e-05,
    -0.02808,
    -0.01132,
    -0.00043,
    -0.0035,
    -0.00401,
    -0.01102,
    -0.00152
   ],
   [
    0.0036,
    -0.00311,
    0.00265,
    -0.00023,
    -0.00364,
    -0.01356,
    0.00297,
    0.00149,
    0.00032,
    0.00382,
    0.00065,
    -0.00384,
    -0.01101,
    0.00235
   ],
   [
    0.00053,
    -0.01483,
    0.01187,
    0.0101,
    -0.01066,
    0.0091,
    0.0234,
    -0.00164,
    -0.00887,
    0.01024,
    0.01104,
    -0.00478,
    0.01229,
    0.02101
   ],
   [
    0.01592,
    0.01033,
    0.01836,
    0.01105,
    0.00875,
    -0.00949,
    -0.01225,
    0.01381,
    0.00925,
    0.00971,
    0.00756,
    0.00641,
    -0.01191,
    -0.01283
   ],
   [
    0.01177,
    0.00107,
    -0.00647,
    -0.00265,
    0.00059,
    0.00024,
    -0.00653,
    0.01255,
    0.00038,
    -0.00433,
    -0.00386,
    -0.00103,
    -0.00031,
    -0.005
   ],
   [
    -0.0086,
    0.00226,
    -0.00241,
    -0.02168,
    0.00734,
    -0.00131,
    0.00611,
    -0.00598,
    0.00089,
    -0.00739,
    -0.02127,
    0.00463,
    -0.00282,
    0.00261
   ],
   [
    -0.00757,
    0.011,
    -0.00503,
    -0.0059,
    0.00638,
    -0.00165,
    -0.00098,
    -0.00655,
    0.00467,
    -0.00565,
    -0.00526,
    0.0059,
    -0.00344,
    -0.0012
   ],
   [
    -0.00099,
    0.0065,
    -0.00946,
    0.00409,
    0.01219,
    -0.00192,
    -0.01289,
    0.00201,
    0.00489,
    -0.00723,
    0.00059,
    0.00913,
    -0.00331,
    -0.01199
   ],
   [
    -0.00595,
    0.00592,
    -0.00047,
    -0.00227,
    0.00609,
    -0.01178,
    -0.00681,
    -0.00346,
    0.00732,
    0.00035,
    -0.00196,
    0.00648,
    -0.00925,
    -0.00634
   ],
   [
    -0.01888,
    -0.0061,
    0.00442,
    -0.00409,
    -0.005,
    0.0133,
    0.0161,
    -0.01283,
    -0.0044,
    0.00246,
    -0.00425,
    -0.00038,
    0.01428,
    0.01737
   ],
   [
    -0.00928,
    0.00555,
    -0.00381,
    -0.00519,
    0.00229,
    -0.00499,
    -0.00101,
    -0.00791,
    0.00385,
    -0.00488,
    -0.00584,
    0.00191,
    -0.00693,
    -0.00272
   ],
   [
    -0.01439,
    0.00289,
    -0.00132,
    -0.00877,
    -0.00411,
    0.01072,
    0.01191,
    -0.01215,
    -0.00369,
    -0.00266,
    -0.00964,
    -0.00331,
    0.01084,
    0.01331
   ],
   [
    0.0033,
    -0.00222,
    -0.01537,
    0.0084,
    0.00321,
    -0.01831,
    -0.00558,
    0.00575,
    -0.00111,
    -0.01228,
    0.00599,
    0.0025,
    -0.01389,
    -0.00473
   ],
   [
    0.00279,
    -0.0007,
    -0.00867,
    -0.01016,
    0.00605,
    0.0023,
    -0.00062,
    0.00253,
    -0.00139,
    -0.00933,
    -0.00895,
    0.00457,
    0.00257,
    -0.00149
   ],
   [
    -0.01286,
    0.00752,
    0.00751,
    -0.00411,
    -0.00081,
    0.00463,
    0.00368,
    -0.00878,
    0.00934,
    0.00348,
    -0.00454,
    -0.00193,
    0.00058,
    0.0007
   ],
   [
    0.01011,
    -0.01422,
    0.01489,
    0.01792,
    -0.02147,
    0.01061,
    0.00272,
    0.00986,
    -0.00535,
    0.01287,
    0.01504,
    -0.01457,
    0.01332,
    0.00599
   ],
   [
    0.02273,
    -0.01559,
    0.00693,
    -0.00021,
    -0.00373,
    0.00354,
    0.0092,
    0.01804,
    -0.01195,
    0.00551,
    0.00055,
    -0.00093,
    0.00582,
    0.00898
   ],
   [
    0.00094,
    0.00126,
    -0.00085,
    -0.01353,
    -0.0003,
    0.00825,
    0.00388,
    0.00302,
    0.00581,
    0.00367,
    -0.00972,
    -0.00079,
    0.00764,
    0.00406
   ],
   [
    0.01357,
    0.00476,
    -0.00883,
    -0.01254,
    -0.01673,
    0.00759,
    0.01493,
    0.01588,
    0.00294,
    -0.00516,
    -0.01437,
    -0.01516,
    0.0067,
    0.01226
   ],
   [
    0.0235,
    0.00249,
    0.0013,
    0.00395,
    -0.00892,
    -0.00205,
    -0.00933,
    0.02101,
    0.00326,
    0.00066,
    0.00298,
    -0.00821,
    -0.00284,
    -0.00839
   ],
   [
    -0.02649,
    -0.00601,
    -0.01189,
    0.00163,
    0.00155,
    -0.01163,
    -0.00405,
    -0.02283,
    -0.00666,
    -0.00737,
    0.00185,
    0.0017,
    -0.00972,
    -0.00277
   ],
   [
    0.00457,
    -0.00151,
    0.0086,
    -0.00077,
    -3e-05,
    0.00448,
    -0.00163,
    0.00592,
    0.00074,
    0.00385,
    -0.00085,
    0.00046,
    0.00409,
    -0.00049
   ],
   [
    0.00039,
    -0.01058,
    0.00468,
    -0.00393,
    -0.00349,
    0.00924,
    0.00515,
    0.00104,
    -0.00629,
    0.00512,
    -0.00428,
    -0.00222,
    0.00851,
    0.00366
   ],
   [
    0.00246,
    0.00377,
    0.00291,
    -0.00486,
    0.01136,
    -0.0039,
    0.00474,
    0.00294,
    0.00172,
    -0.00011,
    -0.00427,
    0.01211,
    -0.00237,
    0.0056
   ],
   [
    -0.05895,
    0.01371,
    -0.04295,
    0.01994,
    0.04323,
    -0.07356,
    0.00612,
    -0.06277,
    -0.00692,
    -0.05424,
    0.01668,
    0.03239,
    -0.07907,
    -0.00208
   ],
   [
    0.01389,
    0.00927,
    -0.00688,
    -0.00187,
    0.0143,
    0.00338,
    -0.01937,
    0.01222,
    0.01105,
    -0.00625,
    -0.00188,
    0.01314,
    0.00251,
    -0.0149
   ],
   [
    0.0073,
    -0.02725,
    0.00043,
    0.00588,
    -0.01856,
    -0.01212,
    0.00395,
    0.0047,
    -0.02779,
    0.00067,
    0.00837,
    -0.01908,
    -0.01287,
    0.00126
   ],
   [
    -0.01046,
    0.02004,
    0.01895,
    -0.03787,
    0.03246,
    0.00855,
    0.04511,
    -0.01139,
    0.0236,
    0.02734,
    -0.03628,
    0.0327,
    0.01627,
    0.03969
   ],
   [
    0.00231,
    -0.01942,
    -0.00119,
    -0.00416,
    0.01027,
    0.00485,
    0.01371,
    0.00176,
    -0.01434,
    -0.00492,
    -0.00637,
    0.00688,
    0.00321,
    0.01258
   ],
   [
    -0.0037,
    -0.00507,
    0.00271,
    -0.00393,
    -0.00993,
    0.01467,
    0.00989,
    -0.00273,
    -0.00844,
    -0.00152,
    -0.00394,
    -0.00572,
    0.01739,
    0.01109
   ],
   [
    -0.00861,
    0.00154,
    -0.02009,
    0.00525,
    -0.00067,
    -0.00449,
    0.00079,
    -0.00904,
    -0.00071,
    -0.01035,
    0.00803,
    0.00093,
    -0.00363,
    -0.00071
   ],
   [
    0.01267,
    -0.00013,
    0.00902,
    -0.00039,
    -0.00031,
    0.00296,
    -0.00925,
    0.01095,
    0.00021,
    0.00993,
    0.0005,
    0.00099,
    0.00388,
    -0.00788
   ],
   [
    -0.00805,
    -0.01079,
    -0.01753,
    0.00392,
    -0.001,
    -0.00829,
    -0.00445,
    -0.0077,
    -0.01297,
    -0.01174,
    0.00108,
    -0.00334,
    -0.00968,
    -0.00575
   ],
   [
    -0.01686,
    0.01399,
    0.0072,
    -0.00584,
    0.01056,
    -0.00326,
    0.01178,
    -0.01209,
    0.01097,
    0.0039,
    -0.00579,
    0.00998,
    -0.00286,
    0.01018
   ],
   [
    0.00286,
    0.00745,
    0.00117,
    0.00974,
    0.00416,
    -0.00066,
    -0.00682,
    0.0052,
    0.00359,
    0.00237,
    0.00379,
    0.00065,
    -0.00131,
    -0.00704
   ],
   [
    -0.01284,
    -0.01583,
    -0.01465,
    -0.0047,
    -0.0007,
    -0.005,
    -0.00978,
    -0.01065,
    -0.00962,
    -0.01152,
    -0.00519,
    -0.0026,
    -0.00418,
    -0.008
   ],
   [
    0.0331,
    0.01968,
    0.00301,
    0.00322,
    0.00726,
    -0.00534,
    0.01465,
    0.02558,
    0.02074,
    0.00588,
    0.00437,
    0.00649,
    -0.00222,
    0.01433
   ],
   [
    -0.01006,
    0.00247,
    0.00607,
    -0.01368,
    0.00838,
    -0.00599,
    0.00667,
    -0.00891,
    0.0007,
    0.00026,
    -0.01175,
    0.00902,
    -0.00419,
    0.00643
   ],
   [
    0.00499,
    0.01912,
    -0.00531,
    0.00483,
    0.01077,
    -0.00519,
    -0.00767,
    0.00403,
    0.01687,
    -0.00657,
    0.00304,
    0.00971,
    -0.00131,
    -0.00545
   ],
   [
    0.0218,
    0.00496,
    0.00516,
    0.0082,
    0.00969,
    -0.01139,
    -0.01445,
    0.01615,
    0.00111,
    -0.00086,
    0.00834,
    0.00897,
    -0.00904,
    -0.01341
   ],
   [
    0.02849,
    0.0006,
    -0.00091,
    -0.00922,
    -0.01225,
    0.01302,
    0.01075,
    0.02309,
    0.00224,
    0.00213,
    -0.01,
    -0.01036,
    0.00958,
    0.00808
   ],
   [
    -0.00803,
    0.00884,
    -0.01394,
    0.00028,
    -0.0054,
    0.00045,
    0.00664,
    -0.00438,
    0.00872,
    -0.00994,
    -0.00315,
    -0.00583,
    -0.00017,
    0.00493
   ],
   [
    -0.02129,
    0.00388,
    -0.00459,
    -0.00323,
    0.01191,
    0.00373,
    -0.00918,
    -0.01608,
    -0.00131,
    -0.00217,
    -0.00685,
    0.01072,
    0.00536,
    -0.00642
   ],
   [
    0.00207,
    0.00463,
    0.00218,
    0.0069,
    -0.01196,
    0.01018,
    -0.00691,
    0.00036,
    0.00344,
    -0.00337,
    0.00363,
    -0.00682,
    0.0089,
    -0.00508
   ],
   [
    0.0278,
    -0.00732,
    0.0056,
    -0.00177,
    -0.00488,
    0.00355,
    0.0074,
    0.02655,
    -0.00094,
    0.00494,
    0.00111,
    -0.00238,
    0.00489,
    0.00654
   ],
   [
    0.0028,
    0.01666,
    -0.00115,
    0.00402,
    0.0047,
    -0.00198,
    -0.00181,
    0.00295,
    0.01346,
    -0.00205,
    0.00449,
    0.0062,
    -0.00022,
    -0.00048
   ],
   [
    0.00559,
    0.00157,
    0.00167,
    -0.00698,
    0.00418,
    -0.00388,
    0.01114,
    0.00294,
    0.00262,
    0.00087,
    -0.0065,
    0.00273,
    -0.00376,
    0.00872
   ],
   [
    -0.00154,
    0.00282,
    -0.01096,
    0.00299,
    0.0154,
    -0.01331,
    -0.0167,
    -0.00339,
    -0.00134,
    -0.01009,
    0.00138,
    0.01159,
    -0.01173,
    -0.0131
   ],
   [
    -0.02215,
    0.0046,
    0.00384,
    -0.00406,
    -0.00055,
    0.0079,
    -0.00251,
    -0.01598,
    0.00112,
    0.00343,
    -0.00297,
    -0.00026,
    0.00816,
    -0.00057
   ],
   [
    -0.00346,
    0.00187,
    -0.00178,
    0.00085,
    -3e-05,
    -0.00468,
    0.01097,
    -0.00076,
    0.00569,
    -0.00058,
    0.00089,
    0.0001,
    -0.0028,
    0.00997
   ],
   [
    0.00769,
    0.00708,
    0.00883,
    0.00345,
    -0.01491,
    0.00409,
    -0.0064,
    0.00618,
    0.00459,
    0.00472,
    0.00145,
    -0.01399,
    0.00304,
    -0.00643
   ],
   [
    -0.00318,
    0.00227,
    -0.00844,
    0.00729,
    0.00249,
    -0.00601,
    -0.01057,
    -0.00067,
    0.00225,
    -0.00541,
    0.00238,
    -0.00135,
    -0.01134,
    -0.01196
   ],
   [
    -0.01399,
    -0.00657,
    0.01171,
    0.00167,
    0.00157,
    -0.00765,
    9e-05,
    -0.01391,
    -0.00592,
    0.00619,
    -0.00066,
    0.00014,
    -0.0051,
    0.00082
   ],
   [
    -0.00258,
    -0.01072,
    -0.00314,
    0.00109,
    -0.01256,
    -0.01365,
    -0.00551,
    -0.00104,
    -0.00792,
    -0.00311,
    0.00128,
    -0.0114,
    -0.01281,
    -0.00794
   ],
   [
    -0.00382,
    -0.0151,
    0.00318,
    0.00873,
    -0.01374,
    -0.00284,
    -0.00231,
    -0.0002,
    -0.01002,
    0.00387,
    0.0099,
    -0.01193,
    -0.00423,
    -0.00403
   ],
   [
    0.02086,
    0.02387,
    -0.00647,
    0.00377,
    0.00676,
    0.00412,
    -0.02612,
    0.01103,
    0.01088,
    -0.00803,
    0.00201,
    0.00263,
    -0.00071,
    -0.02317
   ],
   [
    0.00289,
    -0.00848,
    -0.00286,
    0.0099,
    -0.00058,
    0.00463,
    0.00039,
    0.00456,
    -0.00462,
    -0.00594,
    0.01041,
    -0.00085,
    0.00451,
    0.00186
   ],
   [
    0.02288,
    0.02578,
    -0.00267,
    0.00849,
    -0.00509,
    0.00622,
    -0.00995,
    0.02042,
    0.01429,
    -0.00429,
    0.0057,
    -0.0038,
    0.00379,
    -0.01121
   ],
   [
    -0.0148,
    0.00597,
    -0.00437,
    0.00253,
    -0.00459,
    -0.00316,
    0.00199,
    -0.01359,
    0.00421,
    -0.00537,
    0.0006,
    -0.00333,
    -0.00567,
    1e-05
   ],
   [
    0.0005,
    -0.00391,
    0.00195,
    0.00587,
    0.00422,
    -0.0008,
    -0.01104,
    0.00165,
    -0.00146,
    0.00395,
    0.00497,
    0.00236,
    -0.00187,
    -0.01014
   ],
   [
    0.01582,
    0.01378,
    0.01865,
    -0.00167,
    -0.00067,
    0.01175,
    -0.00103,
    0.01384,
    0.01092,
    0.01551,
    -0.00302,
    -0.00222,
    0.0073,
    -0.00146
   ],
   [
    0.0022,
    -0.01244,
    0.00434,
    -0.01148,
    -0.00177,
    0.00866,
    -0.00042,
    0.00125,
    -0.00685,
    0.00869,
    -0.01031,
    -0.00154,
    0.00665,
    -0.00108
   ],
   [
    0.01057,
    0.0024,
    0.00484,
    -0.00188,
    0.00887,
    -0.00791,
    -0.00384,
    0.00906,
    0.00254,
    0.00367,
    -0.0011,
    0.00739,
    -0.00757,
    -0.00266
   ],
   [
    -0.00186,
    -0.00285,
    -0.00131,
    0.00741,
    -0.01375,
    -0.00197,
    -0.01282,
    -0.00327,
    -0.00167,
    -0.00157,
    0.00624,
    -0.01405,
    0.00077,
    -0.00831
   ],
   [
    -0.00214,
    0.01302,
    -0.00819,
    -0.00849,
    0.0135,
    0.01071,
    -0.00364,
    -0.00219,
    0.01041,
    -0.00108,
    -0.00827,
    0.01366,
    0.0113,
    -0.00051
   ],
   [
    0.00097,
    -0.00579,
    0.00526,
    0.01392,
    -0.00575,
    -0.01064,
    -0.00066,
    -0.0019,
    -0.00652,
    0.00576,
    0.01238,
    -0.0052,
    -0.0099,
    -0.00214
   ],
   [
    0.02033,
    0.03415,
    0.00046,
    -0.00773,
    0.00772,
    0.00163,
    -0.00343,
    0.01518,
    0.02486,
    -0.00409,
    -0.00968,
    0.00772,
    0.00239,
    -0.00232
   ],
   [
    -0.00768,
    -0.00964,
    -0.00873,
    0.00126,
    -0.00343,
    -0.00589,
    -0.00128,
    -0.00939,
    -0.01303,
    -0.00847,
    -0.00049,
    -0.00518,
    -0.00443,
    0.00054
   ],
   [
    0.01113,
    -0.00167,
    -0.00283,
    0.00223,
    -0.00295,
    0.00241,
    -0.0054,
    0.00451,
    -0.00017,
    0.00029,
    0.00504,
    -0.00238,
    -0.00054,
    -0.00551
   ],
   [
    0.0187,
    0.00632,
    0.00353,
    -0.01576,
    0.00322,
    0.02003,
    0.00176,
    0.01385,
    0.01028,
    0.00249,
    -0.01417,
    0.00431,
    0.02036,
    0.00237
   ],
   [
    0.00196,
    0.00087,
    -0.00265,
    0.00931,
    0.005,
    -0.00028,
    -0.01257,
    -5e-05,
    -0.00043,
    0.00208,
    0.00817,
    0.00449,
    -0.00079,
    -0.01148
   ],
   [
    0.01638,
    0.00611,
    0.00269,
    -0.02028,
    -0.00258,
    0.01386,
    0.01367,
    0.01299,
    0.01008,
    0.00427,
    -0.0219,
    -0.0038,
    0.01241,
    0.01041
   ],
   [
    -0.00964,
    0.0123,
    -0.00528,
    -0.0062,
    -0.00205,
    -0.00569,
    0.00344,
    -0.00862,
    0.00819,
    -0.0081,
    -0.00758,
    -0.00323,
    -0.00649,
    0.00104
   ],
   [
    -0.01597,
    -0.00515,
    -0.00458,
    0.00921,
    -0.0053,
    -0.00228,
    -0.01532,
    -0.01286,
    -0.00818,
    -0.00567,
    0.00644,
    -0.00849,
    -0.0043,
    -0.01471
   ],
   [
    -0.00791,
    -0.02082,
    0.01503,
    0.0115,
    -0.01509,
    -0.00958,
    0.00525,
    -0.00447,
    -0.01426,
    0.01064,
    0.01131,
    -0.01265,
    -0.00952,
    0.00483
   ],
   [
    -0.00877,
    -0.01014,
    -0.00344,
    -0.00512,
    -0.00668,
    0.014,
    0.0035,
    -0.00571,
    -0.00809,
    -0.00528,
    -0.00291,
    -0.00489,
    0.01192,
    0.00448
   ],
   [
    -0.01137,
    -0.00036,
    -0.00492,
    -0.01037,
    0.01814,
    -0.00276,
    0.00847,
    -0.01131,
    -0.00059,
    -0.00143,
    -0.01047,
    0.01691,
    -9e-05,
    0.00787
   ],
   [
    0.00067,
    0.00435,
    -0.0145,
    0.00566,
    0.00314,
    0.00457,
    -0.00688,
    0.00136,
    0.00189,
    -0.01201,
    0.00359,
    0.00195,
    0.00446,
    -0.00568
   ],
   [
    -0.01297,
    -0.00966,
    -0.00696,
    -0.00976,
    0.00239,
    0.01337,
    0.01283,
    -0.00804,
    -0.00421,
    -0.00451,
    -0.0105,
    0.00269,
    0.01217,
    0.01204
   ],
   [
    -0.00645,
    -0.004,
    0.00724,
    -0.0042,
    0.00431,
    0.00212,
    -0.01213,
    -0.00633,
    -0.00691,
    0.00546,
    -0.00131,
    0.00352,
    0.005,
    -0.00738
   ],
   [
    0.00389,
    0.001,
    0.00594,
    -0.00439,
    -0.01097,
    0.00764,
    0.02665,
    0.00265,
    0.00348,
    0.0003,
    -0.00617,
    -0.01035,
    0.00686,
    0.02069
   ],
   [
    0.01667,
    0.00605,
    0.00983,
    0.00158,
    -0.00135,
    -0.00355,
    0.00992,
    0.01571,
    0.00765,
    0.01106,
    -0.00038,
    -0.0009,
    -0.00163,
    0.00854
   ],
   [
    -0.01514,
    0.01597,
    0.01078,
    0.00245,
    0.00593,
    0.00731,
    -0.00475,
    -0.01046,
    0.00996,
    0.00845,
    0.00014,
    0.00564,
    0.00659,
    -0.00414
   ],
   [
    -0.00152,
    0.0032,
    -0.01339,
    0.01027,
    -0.00032,
    -0.00303,
    -0.00436,
    -0.00305,
    0.00472,
    -0.01108,
    0.00542,
    -0.006,
    -0.00775,
    -0.00624
   ],
   [
    -0.01298,
    -0.00109,
    7e-05,
    -0.00757,
    0.0063,
    0.00011,
    -0.0082,
    -0.01314,
    0.00057,
    0.00301,
    -0.0046,
    0.00777,
    0.00139,
    -0.00524
   ],
   [
    0.00796,
    0.01599,
    -0.01895,
    -0.00636,
    -0.00683,
    0.00781,
    0.00381,
    0.00513,
    0.01403,
    -0.01434,
    -0.01,
    -0.00809,
    0.00513,
    0.00155
   ],
   [
    -0.01138,
    -0.00604,
    -0.00238,
    0.00335,
    0.00134,
    0.0032,
    0.00239,
    -0.00788,
    -0.0066,
    -0.00034,
    0.00468,
    0.0023,
    0.00332,
    0.00231
   ],
   [
    0.00712,
    -0.03503,
    0.017,
    0.00425,
    -0.03311,
    0.01361,
    0.02068,
    0.00247,
    -0.02852,
    0.01264,
    0.00442,
    -0.02969,
    0.00893,
    0.01622
   ],
   [
    -0.00585,
    0.01799,
    0.00142,
    -0.01444,
    0.00663,
    0.00904,
    0.0018,
    -0.00299,
    0.01254,
    0.00246,
    -0.01319,
    0.00868,
    0.01025,
    0.00484
   ],
   [
    -0.01986,
    0.01463,
    -0.015,
    -0.02152,
    0.00291,
    0.00628,
    0.01044,
    -0.01347,
    0.00978,
    -0.0087,
    -0.02242,
    0.00252,
    0.0036,
    0.00757
   ],
   [
    0.0022,
    -0.0047,
    0.00712,
    0.00156,
    0.00694,
    -0.01405,
    -0.01065,
    0.00197,
    -0.0023,
    0.00253,
    0.00507,
    0.0063,
    -0.01064,
    -0.00897
   ],
   [
    -0.00274,
    -0.01136,
    -0.0085,
    -0.00924,
    0.01057,
    -0.00177,
    -0.00339,
    -0.00076,
    -0.01093,
    -0.00549,
    -0.00996,
    0.00659,
    -0.00273,
    -0.00479
   ],
   [
    -0.00634,
    -0.0075,
    0.00747,
    -0.01687,
    -0.00504,
    0.01449,
    0.0039,
    -0.00809,
    -0.00541,
    0.00339,
    -0.01499,
    -0.00578,
    0.01141,
    0.00345
   ],
   [
    0.00615,
    0.01219,
    0.00315,
    -0.0046,
    -0.00031,
    0.01126,
    0.00879,
    0.00624,
    0.01081,
    0.00599,
    -0.00236,
    0.0007,
    0.00977,
    0.00986
   ],
   [
    0.01585,
    -0.00846,
    0.0209,
    0.00083,
    -0.00879,
    0.0042,
    0.00586,
    0.01539,
    -0.00648,
    0.01589,
    0.00143,
    -0.00841,
    0.00527,
    0.00589
   ],
   [
    0.00541,
    -0.00651,
    0.003,
    0.0006,
    0.00189,
    -0.00943,
    0.00523,
    0.00425,
    0.00121,
    0.00518,
    0.00175,
    0.00149,
    -0.00624,
    0.00555
   ],
   [
    -0.01298,
    -0.00542,
    -0.00025,
    0.00346,
    -0.00205,
    -0.00672,
    -0.00806,
    -0.01375,
    -0.00686,
    -0.00489,
    9e-05,
    -0.00537,
    -0.00745,
    -0.00827
   ],
   [
    0.00474,
    0.00323,
    0.00229,
    0.00351,
    -0.00836,
    0.00447,
    -0.00924,
    0.00485,
    0.0001,
    0.0024,
    0.00317,
    -0.0077,
    0.00249,
    -0.0076
   ],
   [
    -0.0083,
    0.00668,
    0.00719,
    0.00821,
    -0.00481,
    -0.00189,
    -0.00209,
    -0.0032,
    -0.00026,
    0.00725,
    0.00694,
    -0.00469,
    -0.00101,
    -0.00065
   ],
   [
    -0.00172,
    0.00324,
    0.01064,
    0.00358,
    0.00437,
    -0.00807,
    -0.00527,
    -0.00114,
    0.00067,
    0.00461,
    0.00177,
    0.00237,
    -0.00817,
    -0.00534
   ],
   [
    -0.01353,
    0.017,
    -0.01116,
    0.00356,
    0.02388,
    -0.01583,
    0.01032,
    -0.00824,
    0.01488,
    -0.00622,
    0.00383,
    0.02329,
    -0.01253,
    0.00942
   ],
   [
    0.00227,
    0.0112,
    -0.01295,
    -0.00164,
    0.01434,
    -0.00939,
    -0.00606,
    0.00123,
    0.00912,
    -0.00818,
    -9e-05,
    0.01206,
    -0.0077,
    -0.00485
   ],
   [
    -0.01939,
    -0.00628,
    0.00763,
    -0.00662,
    -0.00545,
    -0.00385,
    0.00495,
    -0.016,
    -0.00558,
    0.00155,
    -0.00361,
    -0.00382,
    -0.00304,
    0.00477
   ],
   [
    -0.00972,
    0.0079,
    -0.01708,
    0.00734,
    0.01587,
    -0.00513,
    -0.02018,
    -0.00485,
    0.00645,
    -0.01216,
    0.00606,
    0.01392,
    -0.00423,
    -0.01813
   ],
   [
    0.01353,
    0.01858,
    0.00731,
    0.00999,
    0.00704,
    -0.01041,
    0.00075,
    0.01326,
    0.01333,
    0.00859,
    0.01121,
    0.0074,
    -0.00522,
    0.00411
   ],
   [
    -0.02201,
    -0.00219,
    -0.01425,
    0.00563,
    0.0081,
    0.00054,
    -0.00527,
    -0.02044,
    -0.00376,
    -0.01497,
    0.00439,
    0.00869,
    0.00085,
    -0.0006
   ],
   [
    0.00894,
    -0.00677,
    0.01185,
    -0.00872,
    0.00373,
    0.0104,
    0.00026,
    0.00888,
    -0.00319,
    0.01079,
    -0.00767,
    0.00414,
    0.01054,
    0.00153
   ],
   [
    -0.00459,
    -0.00864,
    0.0092,
    0.0131,
    -0.00804,
    0.00402,
    -0.00937,
    -0.00436,
    -0.01106,
    0.00972,
    0.00798,
    -0.00943,
    0.00082,
    -0.00863
   ],
   [
    0.01141,
    -0.0014,
    -0.01175,
    0.0058,
    -0.01223,
    0.0012,
    -0.00165,
    0.00769,
    -0.00049,
    -0.01051,
    0.00479,
    -0.00937,
    0.00126,
    -0.00166
   ],
   [
    -0.01123,
    0.0132,
    -0.01296,
    0.00985,
    0.00617,
    -0.01823,
    -0.00765,
    -0.00923,
    0.01067,
    -0.01005,
    0.0103,
    0.00589,
    -0.01389,
    -0.00546
   ],
   [
    -0.02082,
    -0.00294,
    -0.00819,
    -0.00797,
    0.01203,
    0.00058,
    0.00243,
    -0.02111,
    -0.00563,
    -0.0031,
    -0.00597,
    0.01123,
    0.00346,
    0.00417
   ],
   [
    -0.00225,
    -0.00093,
    -0.00438,
    0.00621,
    -0.00269,
    -0.00538,
    0.0108,
    0.00099,
    -0.0009,
    -7e-05,
    0.00596,
    -0.00188,
    -0.00543,
    0.01006
   ],
   [
    0.01296,
    -0.00803,
    0.01378,
    0.00459,
    -0.0056,
    -0.00203,
    -0.01131,
    0.01218,
    -0.0043,
    0.01215,
    0.00329,
    -0.00619,
    -0.00542,
    -0.01279
   ],
   [
    -0.02214,
    0.00531,
    0.0032,
    -0.00314,
    -0.00041,
    -0.0004,
    -0.00418,
    -0.01956,
    0.00118,
    0.0021,
    -0.0039,
    -0.00211,
    -0.00127,
    -0.00412
   ],
   [
    -0.0147,
    -0.01215,
    -0.00645,
    0.00353,
    0.00028,
    -0.00559,
    -0.00893,
    -0.0137,
    -0.01593,
    -0.00835,
    -0.00046,
    -0.00239,
    -0.00631,
    -0.00652
   ],
   [
    0.0159,
    0.00449,
    0.00049,
    0.00874,
    -0.00273,
    0.0009,
    -0.00559,
    0.00982,
    -0.00086,
    0.00094,
    0.01035,
    -0.00319,
    0.00015,
    -0.00541
   ],
   [
    -0.00485,
    0.00124,
    -0.00954,
    0.00641,
    -0.00137,
    -0.00978,
    -0.01016,
    -0.00308,
    -0.00879,
    -0.01195,
    0.00192,
    -0.00368,
    -0.01189,
    -0.01054
   ],
   [
    -0.01807,
    -0.01266,
    -0.00233,
    0.0046,
    -0.0078,
    -0.00879,
    0.00674,
    -0.01509,
    -0.0123,
    -0.00419,
    0.00513,
    -0.00622,
    -0.00968,
    0.00529
   ],
   [
    -0.00629,
    -0.01023,
    0.00429,
    0.00327,
    -0.00881,
    -0.00519,
    0.00921,
    -0.00348,
    -0.01196,
    -0.0004,
    0.00307,
    -0.00823,
    -0.00321,
    0.00876
   ],
   [
    -0.00492,
    -0.0027,
    -0.00771,
    0.00596,
    -0.00218,
    -0.01075,
    -0.00629,
    -0.002,
    -0.0019,
    -0.00918,
    0.00668,
    -0.00017,
    -0.0092,
    -0.00511
   ],
   [
    0.00746,
    0.00602,
    -0.00055,
    -0.0027,
    0.01491,
    -0.00366,
    -0.00993,
    0.00878,
    0.00494,
    -0.00562,
    -0.00064,
    0.01351,
    -0.00237,
    -0.00846
   ],
   [
    -0.01157,
    0.01464,
    -0.00758,
    0.01048,
    0.01213,
    -0.00624,
    -0.0173,
    -0.0065,
    0.01239,
    -0.00133,
    0.00863,
    0.00963,
    -0.00659,
    -0.01368
   ],
   [
    -0.0085,
    0.00882,
    0.00261,
    -0.00551,
    -0.00294,
    0.00192,
    -0.00286,
    -0.0076,
    0.00247,
    -5e-05,
    -0.00672,
    -0.00431,
    0.00065,
    -0.00447
   ],
   [
    -0.02957,
    -0.00025,
    -0.00567,
    -0.00434,
    0.00416,
    0.00394,
    -0.00548,
    -0.02689,
    0.00016,
    -0.00755,
    -0.00344,
    0.00092,
    0.00098,
    -0.00682
   ],
   [
    -0.00335,
    -0.00392,
    0.00554,
    -0.00786,
    -0.01017,
    0.0113,
    0.00609,
    -0.00207,
    -0.00333,
    0.0024,
    -0.00716,
    -0.00784,
    0.01037,
    0.006
   ],
   [
    0.0167,
    -0.01712,
    0.0061,
    0.00321,
    -0.01928,
    0.0038,
    0.01048,
    0.01482,
    -0.01675,
    0.00578,
    0.00052,
    -0.01903,
    -0.00128,
    0.00488
   ],
   [
    -0.00063,
    -0.01611,
    -0.00756,
    0.0044,
    -0.00943,
    -0.00993,
    0.00256,
    -0.00238,
    -0.01438,
    -0.007,
    0.00386,
    -0.00901,
    -0.00838,
    0.00102
   ],
   [
    0.02688,
    0.00593,
    0.00718,
    0.00812,
    -0.00615,
    -0.00232,
    -0.01381,
    0.02035,
    0.00509,
    0.00672,
    0.01146,
    -0.00383,
    -0.0044,
    -0.0131
   ],
   [
    0.00341,
    0.00524,
    -0.00044,
    0.00237,
    -0.00046,
    0.001,
    0.0063,
    0.00763,
    0.00361,
    0.00328,
    0.00319,
    8e-05,
    0.00287,
    0.00688
   ],
   [
    -0.02396,
    -0.0331,
    -0.00288,
    -0.0029,
    -0.01644,
    0.00582,
    -0.00281,
    -0.02043,
    -0.03226,
    -0.0061,
    -0.00066,
    -0.01162,
    0.00659,
    0.00084
   ],
   [
    -0.00672,
    -0.00642,
    0.00939,
    0.00566,
    -0.00971,
    0.01496,
    0.00517,
    -0.00522,
    -0.00432,
    0.00499,
    0.00644,
    -0.0085,
    0.01476,
    0.00418
   ],
   [
    -0.01489,
    0.00297,
    0.01562,
    -0.00282,
    0.01935,
    -0.01251,
    -0.01908,
    -0.01216,
    0.00192,
    0.0083,
    -0.00188,
    0.01446,
    -0.01379,
    -0.01912
   ],
   [
    -0.03729,
    0.00607,
    -0.00289,
    0.00475,
    -0.01221,
    0.0153,
    0.00071,
    -0.03182,
    -0.00105,
    -0.00424,
    0.00142,
    -0.01192,
    0.01342,
    0.00188
   ],
   [
    0.00625,
    0.00305,
    -0.00257,
    0.00853,
    -0.01477,
    0.01585,
    0.00347,
    0.00939,
    0.00317,
    0.00161,
    0.00686,
    -0.00969,
    0.01343,
    0.00406
   ],
   [
    -0.01522,
    0.00114,
    -0.00336,
    -0.02228,
    0.01093,
    -0.00101,
    0.02069,
    -0.01221,
    0.00139,
    -0.00066,
    -0.01956,
    0.00839,
    -0.00131,
    0.01678
   ],
   [
    0.01344,
    -0.01575,
    0.00639,
    0.00936,
    -0.02192,
    -0.0175,
    0.01731,
    0.00868,
    -0.00999,
    0.00269,
    0.00809,
    -0.01958,
    -0.01559,
    0.01337
   ],
   [
    0.01418,
    0.00478,
    -0.00719,
    -0.00132,
    -0.00589,
    0.0023,
    -0.00648,
    0.01272,
    0.0063,
    -0.00294,
    -0.00234,
    -0.00479,
    0.0019,
    -0.00678
   ],
   [
    -0.00483,
    0.01064,
    0.00729,
    -0.00766,
    -0.00242,
    0.01073,
    0.00881,
    -0.00209,
    0.01061,
    0.00698,
    -0.00596,
    -0.00033,
    0.0146,
    0.01061
   ],
   [
    0.00152,
    -0.00305,
    0.00702,
    -0.00072,
    0.00312,
    0.00659,
    0.00023,
    0.00183,
    0.00133,
    0.00411,
    -8e-05,
    0.00406,
    0.0083,
    7e-05
   ],
   [
    -0.00156,
    0.14475,
    0.2212,
    0.38968,
    0.42083,
    0.40171,
    0.39884,
    0.01459,
    0.12184,
    0.17728,
    0.33748,
    0.35939,
    0.35094,
    0.34063
   ]
  ],
  "inst20": [
   [
    -0.00162,
    -0.01439,
    0.00888,
    0.00029,
    -0.00545,
    0.00321,
    -0.00304,
    -0.00412,
    -0.014,
    0.00968,
    0.00509,
    -0.0033,
    0.005,
    -0.00083
   ],
   [
    0.00035,
    0.01556,
    0.00247,
    -0.00469,
    0.01065,
    0.00142,
    0.00103,
    0.00016,
    0.01243,
    0.00251,
    -0.00201,
    0.01001,
    0.00318,
    0.00282
   ],
   [
    0.0021,
    -0.00087,
    -0.00244,
    -0.00315,
    -0.00275,
    -0.00191,
    0.00344,
    0.0004,
    -0.00259,
    0.00254,
    -0.0031,
    -0.0047,
    -0.00424,
    0.0034
   ],
   [
    -0.01241,
    -0.02273,
    -0.00191,
    -0.00121,
    0.00062,
    0.00688,
    -0.0021,
    -0.01264,
    -0.01803,
    -0.00084,
    8e-05,
    0.00208,
    0.00673,
    -4e-05
   ],
   [
    0.00727,
    -0.01551,
    -0.00298,
    0.01156,
    -0.00909,
    -0.00306,
    0.0127,
    0.00664,
    -0.01042,
    -0.00216,
    0.0153,
    -0.00445,
    -0.002,
    0.01198
   ],
   [
    0.01,
    -0.00064,
    0.0124,
    -0.00243,
    0.00678,
    -0.00199,
    0.0074,
    0.01242,
    -0.00121,
    0.0116,
    0.00038,
    0.00668,
    0.00201,
    0.00915
   ],
   [
    0.0026,
    -0.01063,
    0.00679,
    -0.00151,
    -0.00374,
    0.00038,
    0.00069,
    0.0003,
    -0.0106,
    0.00688,
    -0.00065,
    -0.00399,
    -0.00146,
    -0.00028
   ],
   [
    0.01627,
    -0.00082,
    -0.00747,
    0.01026,
    -0.00781,
    -0.00515,
    0.00921,
    0.01462,
    0.00433,
    -0.00321,
    0.01048,
    -0.00684,
    -0.00519,
    0.00633
   ],
   [
    -0.01201,
    -0.01844,
    -0.00087,
    -0.00226,
    -0.01138,
    0.00896,
    0.00474,
    -0.01123,
    -0.01704,
    -0.00276,
    -0.00361,
    -0.00927,
    0.00865,
    0.00564
   ],
   [
    -0.0184,
    -0.00434,
    -0.02229,
    -0.00925,
    0.01125,
    0.00411,
    -0.00071,
    -0.01434,
    -0.00483,
    -0.01876,
    -0.0065,
    0.01387,
    0.0081,
    0.00397
   ],
   [
    -0.00624,
    -0.01692,
    -0.0084,
    0.01131,
    -0.00801,
    0.00394,
    0.00359,
    -0.00806,
    -0.01095,
    -0.00212,
    0.01198,
    -0.00374,
    0.0048,
    0.00217
   ],
   [
    0.01484,
    0.05255,
    -0.0022,
    0.01833,
    0.04673,
    -0.04971,
    -0.00501,
    0.01156,
    0.03956,
    -0.00508,
    0.02302,
    0.04002,
    -0.04292,
    -0.00195
   ],
   [
    -0.00241,
    -0.00156,
    -0.0021,
    -0.00438,
    -0.00522,
    0.00382,
    -0.00382,
    0.00107,
    0.0021,
    0.00086,
    -0.00572,
    -0.00511,
    0.00452,
    -0.00246
   ],
   [
    -9e-05,
    0.00744,
    0.00875,
    -0.00395,
    0.00591,
    -0.01186,
    -0.00186,
    -0.00204,
    0.00428,
    0.00395,
    -0.00386,
    0.0051,
    -0.01124,
    -0.00198
   ],
   [
    0.00083,
    -0.01162,
    0.00079,
    0.00412,
    -0.0072,
    0.00045,
    0.00127,
    -0.00288,
    -0.0123,
    -0.00089,
    0.00508,
    -0.00475,
    -0.00069,
    0.00117
   ],
   [
    0.01852,
    3e-05,
    0.01744,
    0.00089,
    -0.00484,
    0.00251,
    0.01647,
    0.01841,
    0.00472,
    0.01378,
    0.00524,
    -0.00077,
    0.00432,
    0.01462
   ],
   [
    0.01713,
    0.00657,
    -0.00065,
    0.00456,
    -0.01514,
    -0.00203,
    0.00069,
    0.01301,
    0.00791,
    0.00027,
    0.00321,
    -0.01522,
    -0.00271,
    -0.00048
   ],
   [
    0.00416,
    -0.00248,
    0.0114,
    0.00224,
    0.00286,
    -0.01431,
    -0.00024,
    0.00299,
    -0.00171,
    0.00769,
    0.00342,
    0.00358,
    -0.01312,
    -0.00037
   ],
   [
    -0.00433,
    -0.00634,
    0.01117,
    -0.00812,
    -0.00679,
    0.01082,
    0.00086,
    -0.00627,
    -0.00692,
    0.00943,
    -0.00882,
    -0.00652,
    0.00757,
    0.00022
   ],
   [
    0.0144,
    -0.02775,
    0.00525,
    -0.00321,
    -0.0113,
    0.00193,
    0.01461,
    0.01156,
    -0.01583,
    0.00276,
    -4e-05,
    -0.00787,
    0.0042,
    0.01297
   ],
   [
    -0.02012,
    -0.00713,
    0.00114,
    -0.00353,
    -0.00144,
    0.00102,
    -0.00332,
    -0.0166,
    -0.00865,
    0.00115,
    -0.00343,
    -0.00069,
    0.00424,
    -0.00079
   ],
   [
    -0.00422,
    -0.01762,
    -0.00115,
    -0.00947,
    0.00015,
    -0.01729,
    0.01072,
    -0.00461,
    -0.01175,
    -0.00246,
    -0.00933,
    0.0002,
    -0.01807,
    0.00488
   ],
   [
    -0.00572,
    -0.0058,
    -0.01667,
    -0.00932,
    0.00187,
    -0.00101,
    0.00499,
    -0.009,
    -0.00848,
    -0.01461,
    -0.01128,
    0.00095,
    -0.00109,
    0.00375
   ],
   [
    -0.02094,
    0.01774,
    -0.01046,
    -0.0039,
    0.01273,
    -0.00562,
    -0.01046,
    -0.01327,
    0.01361,
    -0.00879,
    -0.00381,
    0.01071,
    -0.00629,
    -0.01044
   ],
   [
    -0.00151,
    -0.00779,
    -0.00439,
    0.00274,
    -0.00336,
    0.00274,
    0.00408,
    -0.00195,
    -0.00302,
    -0.00159,
    0.00364,
    -0.00224,
    0.00401,
    0.00452
   ],
   [
    -0.00183,
    -0.00733,
    0.00622,
    0.00816,
    -0.00722,
    0.00056,
    -0.00413,
    -0.00197,
    -0.00446,
    0.0059,
    0.01058,
    -0.00454,
    0.00174,
    -0.00094
   ],
   [
    0.00131,
    0.01873,
    -0.00538,
    0.00088,
    0.00899,
    -0.00879,
    -0.00172,
    0.00185,
    0.00972,
    0.00118,
    0.00028,
    0.00714,
    -0.00743,
    -0.00327
   ],
   [
    -0.02134,
    -0.01739,
    0.00266,
    0.00589,
    -0.0178,
    0.00575,
    -0.01567,
    -0.01631,
    -0.01814,
    0.00047,
    0.00586,
    -0.01556,
    0.00673,
    -0.01177
   ],
   [
    -0.03352,
    0.03709,
    -0.00834,
    -0.00653,
    0.04372,
    -0.01304,
    -0.0239,
    -0.02238,
    0.02985,
    -0.00895,
    -0.00624,
    0.03786,
    -0.01296,
    -0.01958
   ],
   [
    -0.00627,
    0.00135,
    -0.0037,
    0.00332,
    0.0017,
    -0.00057,
    -0.00027,
    -0.00427,
    0.00353,
    -0.00059,
    0.00327,
    0.00239,
    0.00129,
    0.00032
   ],
   [
    -0.01389,
    -0.00998,
    -0.00327,
    0.00392,
    -0.00584,
    -0.00205,
    -0.00378,
    -0.00831,
    -0.00785,
    -0.0044,
    0.00559,
    -0.00318,
    0.00124,
    -0.0004
   ],
   [
    -0.00786,
    0.00976,
    0.01127,
    -0.00199,
    -0.00381,
    0.00076,
    0.00576,
    -0.00435,
    0.00683,
    0.00572,
    -0.00229,
    -0.00238,
    0.00111,
    0.00525
   ],
   [
    3e-05,
    -0.0034,
    0.00437,
    0.01458,
    -0.00128,
    -0.0043,
    -0.00909,
    -0.00224,
    -0.00106,
    0.00128,
    0.01401,
    -0.00081,
    -0.00428,
    -0.00676
   ],
   [
    -0.02783,
    -0.03147,
    0.01271,
    -0.02156,
    -0.01576,
    0.02781,
    0.01949,
    -0.02353,
    -0.02446,
    0.0096,
    -0.0152,
    -0.01285,
    0.02314,
    0.01688
   ],
   [
    -0.01344,
    0.0105,
    -0.01018,
    -0.01217,
    0.01217,
    0.00037,
    -0.002,
    -0.01042,
    0.00864,
    -0.00409,
    -0.01247,
    0.01011,
    -0.00122,
    -0.00249
   ],
   [
    0.01299,
    0.00083,
    -0.00014,
    -0.00356,
    -0.00047,
    8e-05,
    0.01169,
    0.01305,
    0.00314,
    0.0011,
    -0.00091,
    0.0018,
    9e-05,
    0.01035
   ],
   [
    0.01016,
    0.00117,
    0.00562,
    0.00568,
    0.00888,
    -0.00714,
    -0.00105,
    0.0093,
    0.00442,
    0.00771,
    0.00617,
    0.00867,
    -0.00233,
    0.00048
   ],
   [
    -0.00582,
    0.00382,
    -0.00929,
    -0.01604,
    0.00057,
    0.01017,
    -0.00337,
    -0.00241,
    0.00059,
    -0.01422,
    -0.01611,
    -0.00092,
    0.00606,
    -0.00286
   ],
   [
    0.00384,
    -0.00493,
    -0.00541,
    0.00988,
    -0.00612,
    -0.00171,
    -0.00396,
    -0.00109,
    -0.0016,
    -0.01124,
    0.00883,
    -0.00309,
    -0.00211,
    -0.00485
   ],
   [
    -0.01035,
    -0.00126,
    -0.01295,
    0.001,
    0.00538,
    -0.00949,
    -0.01112,
    -0.00497,
    -0.00062,
    -0.01077,
    -0.00183,
    0.00552,
    -0.00612,
    -0.00653
   ],
   [
    0.00352,
    -0.00443,
    -0.00627,
    0.00228,
    -0.00277,
    0.00084,
    0.00855,
    0.00233,
    0.00262,
    -0.00086,
    0.00549,
    -0.00101,
    0.00293,
    0.00879
   ],
   [
    0.00904,
    -0.00952,
    0.00068,
    -0.00245,
    -0.00656,
    0.00904,
    0.00264,
    0.00845,
    -0.00616,
    0.00358,
    -0.00107,
    -0.00577,
    0.00754,
    0.00193
   ],
   [
    6e-05,
    -0.01178,
    -0.00531,
    0.00622,
    -0.00713,
    -0.00656,
    -0.01901,
    -0.0027,
    -0.01426,
    -0.00422,
    0.00244,
    -0.01083,
    -0.0072,
    -0.0167
   ],
   [
    -0.00284,
    -0.00599,
    0.00547,
    -0.00383,
    0.00471,
    0.00644,
    0.00195,
    -0.00286,
    -0.00738,
    0.00469,
    -0.00351,
    0.00347,
    0.00563,
    0.00254
   ],
   [
    0.01655,
    0.00946,
    0.00155,
    -0.0155,
    -0.01307,
    0.01251,
    0.0257,
    0.01561,
    0.01421,
    0.00639,
    -0.01661,
    -0.0115,
    0.01143,
    0.02117
   ],
   [
    0.00371,
    0.00623,
    0.00321,
    -0.00636,
    -0.00132,
    0.014,
    -0.00458,
    0.00604,
    0.00339,
    0.00302,
    -0.00619,
    0.00091,
    0.01354,
    -0.00184
   ],
   [
    -0.00367,
    -0.00148,
    -0.00361,
    -0.00745,
    0.01029,
    0.00076,
    -0.00298,
    -0.00129,
    -0.00302,
    -0.00501,
    -0.00711,
    0.00851,
    0.00113,
    -0.00162
   ],
   [
    0.00298,
    0.00033,
    0.00417,
    -0.00286,
    -0.00143,
    0.00336,
    -0.00194,
    0.00308,
    -0.00144,
    0.00249,
    -0.00466,
    -0.00309,
    0.00431,
    -0.00212
   ],
   [
    0.0191,
    0.00413,
    -0.00279,
    -0.00396,
    0.00521,
    -0.00275,
    -0.00076,
    0.01434,
    0.00389,
    -0.00184,
    -0.00301,
    0.00329,
    -0.00258,
    -0.00038
   ],
   [
    0.03387,
    0.00025,
    -0.00563,
    0.00703,
    -0.02214,
    -0.00574,
    0.00294,
    0.02795,
    0.00308,
    -0.00432,
    0.00565,
    -0.01993,
    -0.00858,
    -0.00074
   ],
   [
    -0.02196,
    -0.00561,
    -0.00508,
    -0.00574,
    0.01147,
    -0.00262,
    -0.01404,
    -0.0185,
    -0.00364,
    -0.00628,
    -0.00471,
    0.00928,
    -0.00378,
    -0.01354
   ],
   [
    -0.00582,
    -0.00406,
    -0.00153,
    -0.01107,
    0.00293,
    0.01003,
    -0.00569,
    -0.00197,
    -0.00098,
    0.00099,
    -0.00661,
    0.00405,
    0.01123,
    -0.00382
   ],
   [
    -0.00186,
    -0.00966,
    -0.00621,
    0.00744,
    -0.00652,
    -0.00391,
    -0.00131,
    -0.00313,
    -0.00831,
    -0.00703,
    0.00542,
    -0.00741,
    -0.00483,
    -0.00304
   ],
   [
    0.00064,
    0.0032,
    0.00535,
    -0.00355,
    0.0118,
    0.00351,
    0.00325,
    0.00154,
    0.00129,
    0.00353,
    -0.00307,
    0.00913,
    0.00169,
    0.0027
   ],
   [
    -0.01191,
    -0.00428,
    0.00316,
    -0.00283,
    -0.00167,
    0.00102,
    0.01188,
    -0.01184,
    -0.00704,
    0.0003,
    -0.00263,
    -0.00092,
    0.00117,
    0.01009
   ],
   [
    0.01566,
    0.00427,
    0.00838,
    -0.01609,
    0.00146,
    0.00341,
    -0.00092,
    0.01136,
    0.00074,
    0.00575,
    -0.01198,
    0.00217,
    0.00361,
    0.00024
   ],
   [
    -0.03101,
    -0.02755,
    -0.00092,
    0.01229,
    -0.00898,
    0.00343,
    0.00849,
    -0.02607,
    -0.02231,
    0.00064,
    0.01432,
    -0.00406,
    0.01029,
    0.01192
   ],
   [
    -0.0046,
    -0.01966,
    -0.00627,
    -0.00987,
    -0.00195,
    0.00527,
    0.02949,
    -0.00316,
    -0.01539,
    -0.00718,
    -0.01148,
    0.00029,
    0.00589,
    0.02497
   ],
   [
    -0.00135,
    0.01223,
    -0.00035,
    0.00077,
    0.00896,
    0.00964,
    -0.01482,
    -0.00097,
    0.00421,
    0.0058,
    0.00181,
    0.00898,
    0.00957,
    -0.00873
   ],
   [
    0.01706,
    0.01424,
    -0.00748,
    0.01366,
    -0.00944,
    -0.00046,
    0.00573,
    0.01218,
    0.00522,
    -0.00771,
    0.00973,
    -0.01088,
    -0.00211,
    0.00263
   ],
   [
    -0.01829,
    -0.01217,
    0.00289,
    0.00227,
    6e-05,
    0.00236,
    -0.00653,
    -0.01186,
    -0.00802,
    -0.00242,
    8e-05,
    -0.00334,
    0.00156,
    -0.0062
   ],
   [
    0.01342,
    0.01317,
    -0.00854,
    -0.01013,
    0.01311,
    0.00316,
    -0.00072,
    0.01486,
    0.01061,
    -0.00522,
    -0.01177,
    0.01117,
    0.00469,
    -0.00045
   ],
   [
    0.01224,
    0.0239,
    -0.01978,
    -0.00969,
    0.00499,
    0.00069,
    -0.00514,
    0.01199,
    0.02095,
    -0.01677,
    -0.01165,
    0.00388,
    -0.00211,
    -0.00495
   ],
   [
    -0.00246,
    -0.00233,
    -0.00548,
    -0.00148,
    -0.00255,
    -0.00413,
    0.00491,
    0.00078,
    -0.00412,
    -0.0058,
    -0.00354,
    -0.00223,
    -0.00215,
    0.00517
   ],
   [
    0.01735,
    0.00693,
    0.00834,
    0.00201,
    0.0099,
    -0.01319,
    -0.00625,
    0.01122,
    0.0083,
    0.00605,
    0.00188,
    0.0048,
    -0.01192,
    -0.00651
   ],
   [
    0.01422,
    0.00507,
    0.0028,
    -0.00921,
    0.00474,
    0.00311,
    0.0001,
    0.01385,
    0.00592,
    -0.00035,
    -0.00795,
    0.00277,
    0.00478,
    0.00133
   ],
   [
    0.01236,
    0.00331,
    -0.00305,
    0.00175,
    0.00861,
    0.00903,
    0.00092,
    0.00652,
    0.00317,
    -0.00378,
    0.0005,
    0.00664,
    0.00557,
    -0.0021
   ],
   [
    0.03301,
    0.02151,
    -0.00863,
    0.00169,
    -0.00864,
    0.01831,
    0.0169,
    0.02809,
    0.02516,
    -0.00403,
    0.00275,
    -0.00491,
    0.01543,
    0.01281
   ],
   [
    0.00176,
    0.00985,
    0.00412,
    -0.01118,
    0.00274,
    0.00483,
    -6e-05,
    -0.00018,
    0.01039,
    0.00142,
    -0.01184,
    -0.00011,
    0.0042,
    -0.00021
   ],
   [
    -0.02298,
    -0.00742,
    -0.00173,
    0.0087,
    -0.00916,
    -0.01321,
    -0.00526,
    -0.01704,
    -0.00944,
    -0.00412,
    0.0081,
    -0.008,
    -0.01182,
    -0.00517
   ],
   [
    0.01347,
    -0.00857,
    0.01265,
    0.0033,
    -0.01006,
    0.00345,
    0.00248,
    0.015,
    -0.00601,
    0.00889,
    0.00238,
    -0.00942,
    0.00128,
    0.0028
   ],
   [
    0.01716,
    0.01008,
    0.01139,
    -0.01301,
    -0.00074,
    0.00513,
    0.0,
    0.01589,
    0.01134,
    0.01333,
    -0.00824,
    0.00309,
    0.00844,
    0.0032
   ],
   [
    -0.00878,
    0.00305,
    0.00407,
    -0.00171,
    0.00255,
    0.00336,
    -0.008,
    -0.00825,
    -0.00024,
    0.00411,
    -0.0011,
    0.00546,
    0.00553,
    -0.00468
   ],
   [
    -0.02349,
    -0.0018,
    -0.00446,
    -0.00531,
    -0.00319,
    -0.00997,
    0.01426,
    -0.02093,
    -0.00064,
    -0.00611,
    -0.00537,
    -0.00333,
    -0.00925,
    0.00946
   ],
   [
    -0.00722,
    -0.00036,
    0.0,
    0.0039,
    -0.00309,
    -0.0025,
    -1e-05,
    -0.00767,
    -0.00185,
    -0.00091,
    0.005,
    -0.0032,
    -0.00277,
    -3e-05
   ],
   [
    0.00445,
    -0.00737,
    0.00998,
    -0.00191,
    -0.02131,
    0.01119,
    0.02511,
    0.00674,
    -0.00244,
    0.01315,
    3e-05,
    -0.01756,
    0.00866,
    0.02138
   ],
   [
    -0.0056,
    0.01083,
    -0.00693,
    -0.00345,
    0.00417,
    0.00918,
    -0.0092,
    -0.00258,
    0.01028,
    -0.00251,
    -0.00225,
    0.00434,
    0.00823,
    -0.00693
   ],
   [
    -0.00346,
    -0.00813,
    -0.00853,
    -0.01015,
    0.00635,
    0.00162,
    0.0064,
    -0.00332,
    -0.00879,
    -0.00876,
    -0.01095,
    0.00578,
    0.00054,
    0.00337
   ],
   [
    0.0124,
    0.01744,
    -0.00372,
    -0.00095,
    0.01305,
    -0.00061,
    -0.00271,
    0.00958,
    0.01093,
    -0.00338,
    -0.00053,
    0.01164,
    0.00017,
    -0.0025
   ],
   [
    -0.00807,
    -0.00403,
    -0.00259,
    0.00074,
    -0.00792,
    0.00267,
    -0.00372,
    -0.00803,
    -0.00507,
    0.00394,
    0.00047,
    -0.0071,
    0.00343,
    -0.00223
   ],
   [
    -0.01456,
    -0.01712,
    -0.01231,
    0.00224,
    -0.0024,
    -0.00339,
    0.01134,
    -0.01434,
    -0.01326,
    -0.00794,
    0.00173,
    -0.00297,
    -0.00501,
    0.00718
   ],
   [
    -0.00667,
    -0.04164,
    -0.01189,
    0.00571,
    -0.01709,
    0.00502,
    0.00481,
    -0.00805,
    -0.03011,
    -0.00896,
    0.00666,
    -0.01457,
    0.00686,
    0.00337
   ],
   [
    0.01716,
    0.00249,
    -0.00181,
    0.01222,
    -0.02209,
    -0.0009,
    -0.01568,
    0.01305,
    0.00268,
    0.00186,
    0.0099,
    -0.02182,
    -0.00036,
    -0.01183
   ],
   [
    0.00922,
    -0.00244,
    -0.00163,
    0.00808,
    0.00304,
    -0.00877,
    0.00126,
    0.00881,
    -0.00337,
    -0.00313,
    0.00775,
    0.00228,
    -0.00912,
    -0.00063
   ],
   [
    -0.00759,
    0.00092,
    -0.01145,
    -0.00307,
    -0.00888,
    0.01078,
    0.00597,
    -0.00576,
    0.00087,
    -0.00836,
    -0.00297,
    -0.00673,
    0.0103,
    0.00566
   ],
   [
    -0.00461,
    -0.01274,
    -0.00456,
    0.00571,
    -0.01231,
    0.00168,
    0.00055,
    -0.00386,
    -0.01421,
    -0.00568,
    0.00306,
    -0.01064,
    0.00103,
    0.00022
   ],
   [
    -0.0025,
    0.00679,
    0.00146,
    0.00064,
    0.00293,
    -0.00181,
    -0.01117,
    -0.00338,
    0.00602,
    0.00064,
    9e-05,
    0.00252,
    -0.00254,
    -0.00944
   ],
   [
    -0.00231,
    0.00172,
    0.00062,
    0.00498,
    0.00593,
    -0.01494,
    -0.00049,
    -0.00567,
    -0.00044,
    -0.00358,
    0.00588,
    0.00478,
    -0.01275,
    -0.00053
   ],
   [
    0.00269,
    0.02,
    0.01123,
    0.00138,
    -0.0114,
    0.01528,
    0.00289,
    0.00372,
    0.01486,
    0.01113,
    0.00106,
    -0.00815,
    0.01751,
    0.00487
   ],
   [
    -0.00648,
    0.00069,
    0.00015,
    -0.00094,
    -0.00109,
    -0.00496,
    0.00943,
    -0.00409,
    0.00288,
    0.00055,
    -0.00128,
    -0.00117,
    -0.00464,
    0.00686
   ],
   [
    0.0152,
    -0.00854,
    0.00192,
    -0.01011,
    -0.00596,
    0.0083,
    0.00446,
    0.01238,
    -0.00502,
    0.00147,
    -0.009,
    -0.00645,
    0.00693,
    0.00409
   ],
   [
    -0.00118,
    -0.01127,
    -0.0097,
    0.00867,
    -0.00201,
    0.00067,
    0.00702,
    -0.00214,
    -0.0101,
    -0.00424,
    0.00871,
    -0.00176,
    0.00265,
    0.00749
   ],
   [
    0.01195,
    -0.0069,
    0.01145,
    0.00764,
    -0.00197,
    -0.00691,
    -0.00151,
    0.00685,
    -0.00555,
    0.00143,
    0.00341,
    -0.00372,
    -0.00792,
    -0.00351
   ],
   [
    0.01283,
    0.00033,
    -0.00235,
    -0.00425,
    0.00273,
    0.00223,
    -0.00572,
    0.01146,
    0.00086,
    0.00382,
    -0.00247,
    0.00323,
    0.00228,
    -0.00246
   ],
   [
    -0.00676,
    -0.0113,
    0.00417,
    0.00637,
    -0.00919,
    -0.00221,
    -0.00463,
    -0.0074,
    -0.01029,
    0.00199,
    0.00425,
    -0.00783,
    -0.00313,
    -0.00501
   ],
   [
    -0.02735,
    0.0213,
    -0.01932,
    -0.02741,
    -0.00643,
    0.01167,
    0.02422,
    -0.01933,
    0.0087,
    -0.01519,
    -0.02304,
    -0.00262,
    0.01232,
    0.0238
   ],
   [
    0.00038,
    -0.00322,
    0.01503,
    0.0015,
    -0.00501,
    0.011,
    0.00446,
    0.00165,
    -0.00332,
    0.01462,
    0.00075,
    -0.00614,
    0.00848,
    0.00506
   ],
   [
    -0.00923,
    -0.01242,
    0.00311,
    -0.00421,
    -0.00409,
    0.02009,
    0.00033,
    -0.00895,
    -0.00636,
    0.00925,
    -0.00384,
    -0.00428,
    0.01622,
    0.0
   ],
   [
    -0.01294,
    -0.00815,
    0.00244,
    0.0007,
    -0.00639,
    0.00101,
    -0.00429,
    -0.00636,
    -0.00529,
    0.0028,
    0.00218,
    -0.00513,
    0.00188,
    -0.00407
   ],
   [
    0.02015,
    0.01868,
    -0.00599,
    -0.00404,
    0.00795,
    0.00514,
    0.00598,
    0.01847,
    0.01864,
    0.00263,
    -0.00555,
    0.00619,
    0.00468,
    0.00482
   ],
   [
    -0.00263,
    0.00123,
    -0.00054,
    -0.00348,
    -0.0025,
    0.00839,
    -0.00179,
    -0.00348,
    0.00067,
    -0.00299,
    -0.00345,
    -0.00282,
    0.00605,
    -0.00174
   ],
   [
    -0.00088,
    -0.00662,
    -0.00178,
    0.02604,
    -0.00485,
    -0.0027,
    -0.02583,
    -0.00256,
    -0.01028,
    0.00017,
    0.02354,
    -0.00339,
    -0.00074,
    -0.01916
   ],
   [
    0.01422,
    0.02103,
    0.01359,
    0.00325,
    0.00758,
    -0.00285,
    0.00582,
    0.01077,
    0.01858,
    0.00735,
    0.00271,
    0.00791,
    0.0002,
    0.0091
   ],
   [
    0.00458,
    0.00122,
    -0.0059,
    -0.00331,
    0.00288,
    -0.00158,
    -0.00293,
    0.004,
    0.00103,
    0.00186,
    -0.00355,
    0.00237,
    -0.00017,
    -0.001
   ],
   [
    0.03142,
    0.02638,
    0.00408,
    -0.00114,
    -0.00166,
    -0.00386,
    0.01995,
    0.02631,
    0.02494,
    0.00618,
    0.00012,
    0.00284,
    0.0008,
    0.01934
   ],
   [
    -0.00443,
    -0.00841,
    -0.00595,
    0.00571,
    -0.00012,
    0.00379,
    -0.01339,
    -0.00603,
    -0.00323,
    -0.00474,
    0.00711,
    -0.0001,
    0.00178,
    -0.0131
   ],
   [
    -0.00644,
    0.0016,
    -0.00437,
    -0.00512,
    -5e-05,
    -0.00103,
    0.00304,
    -0.00258,
    -0.00209,
    -0.00307,
    -0.00749,
    -0.00143,
    -0.00521,
    -0.00087
   ],
   [
    0.00978,
    -0.00217,
    -0.00772,
    0.00102,
    -0.0014,
    0.01201,
    -0.00673,
    0.00784,
    -0.00338,
    -0.00632,
    0.00169,
    -0.0016,
    0.00907,
    -0.00484
   ],
   [
    -0.02735,
    -0.01844,
    -0.0059,
    -0.01287,
    0.01613,
    0.01095,
    -0.00503,
    -0.02327,
    -0.01296,
    -0.00629,
    -0.01241,
    0.01209,
    0.00828,
    -0.0052
   ],
   [
    0.00439,
    0.00058,
    0.00209,
    -0.01031,
    0.00509,
    0.00147,
    0.00773,
    0.00361,
    1e-05,
    0.0025,
    -0.00981,
    0.00449,
    0.0017,
    0.00613
   ],
   [
    0.01889,
    0.00756,
    -0.00046,
    0.00771,
    0.0108,
    -0.00791,
    0.00481,
    0.01648,
    0.00411,
    -0.00221,
    0.00714,
    0.01016,
    -0.00692,
    0.00588
   ],
   [
    -0.00255,
    -0.00381,
    0.00281,
    -0.01471,
    -0.0012,
    0.01243,
    0.00454,
    -0.00209,
    -0.00425,
    0.00381,
    -0.01451,
    -0.00095,
    0.01285,
    0.00365
   ],
   [
    -0.01226,
    -0.01728,
    0.00201,
    0.01641,
    -0.0079,
    -0.01038,
    0.00083,
    -0.01202,
    -0.01759,
    -0.00351,
    0.01639,
    -0.00662,
    -0.01215,
    -0.00147
   ],
   [
    -0.02491,
    0.04305,
    0.00083,
    0.01567,
    0.01016,
    -0.02984,
    -0.00587,
    -0.02212,
    0.03202,
    0.00023,
    0.01573,
    0.01078,
    -0.02681,
    -0.0062
   ],
   [
    -0.00869,
    0.00122,
    0.00334,
    -0.01295,
    -0.00342,
    0.0045,
    0.00711,
    -0.00711,
    -0.00074,
    0.00189,
    -0.01245,
    -0.00188,
    0.00534,
    0.00805
   ],
   [
    0.01575,
    0.01515,
    0.01234,
    7e-05,
    0.00523,
    -0.00052,
    0.00483,
    0.01367,
    0.01372,
    0.0076,
    -0.00121,
    0.00469,
    -0.00026,
    0.00394
   ],
   [
    0.01429,
    0.00857,
    -0.00041,
    0.00659,
    0.00654,
    -0.01387,
    -0.01317,
    0.01021,
    0.00672,
    -0.00266,
    0.00685,
    0.00526,
    -0.01339,
    -0.0117
   ],
   [
    -0.00053,
    0.00067,
    -0.00232,
    -0.00218,
    0.00213,
    -0.00211,
    0.0056,
    0.00121,
    -0.00069,
    -0.00316,
    -0.00178,
    0.00191,
    -0.00163,
    0.00499
   ],
   [
    0.00687,
    0.0118,
    -0.00676,
    -0.00295,
    0.00393,
    -0.00091,
    0.00279,
    0.00844,
    0.00409,
    -0.00498,
    -0.0058,
    0.00253,
    -0.00236,
    0.00176
   ],
   [
    -0.01799,
    -0.00998,
    -0.00547,
    -0.00793,
    -0.00709,
    0.01537,
    0.00577,
    -0.01658,
    -0.00658,
    -0.00678,
    -0.00765,
    -0.00474,
    0.01035,
    0.00301
   ],
   [
    -0.00099,
    -0.00357,
    -0.00515,
    -0.01046,
    0.00078,
    0.0069,
    0.00711,
    -0.00182,
    -0.00594,
    -0.00742,
    -0.01143,
    -0.0004,
    0.00573,
    0.00564
   ],
   [
    -0.00432,
    -0.00185,
    0.00194,
    0.01598,
    0.00594,
    -0.02331,
    -0.01061,
    -0.00532,
    -0.00319,
    -0.00524,
    0.01273,
    0.00186,
    -0.02289,
    -0.01041
   ],
   [
    -0.01285,
    -0.00176,
    -8e-05,
    -0.00102,
    0.0099,
    0.00044,
    -0.00332,
    -0.00948,
    -0.00358,
    -0.0031,
    -0.00269,
    0.00958,
    0.00013,
    -0.00385
   ],
   [
    -0.00083,
    -0.01099,
    0.00773,
    -0.0047,
    -0.02021,
    0.00734,
    0.00999,
    -0.00467,
    -0.00901,
    0.00085,
    -0.00743,
    -0.01827,
    0.0037,
    0.00747
   ],
   [
    -0.00838,
    -0.01483,
    0.00076,
    -0.0049,
    0.01014,
    -0.00102,
    0.00223,
    -0.00698,
    -0.01622,
    0.00052,
    -0.00623,
    0.00854,
    -0.00098,
    0.00281
   ],
   [
    -0.00122,
    0.00378,
    -0.00504,
    -0.00677,
    -0.00538,
    0.00049,
    0.00552,
    0.0007,
    0.00381,
    -0.00107,
    -0.00578,
    -0.0045,
    0.00043,
    0.00448
   ],
   [
    0.00662,
    0.01215,
    0.00524,
    0.00459,
    0.00248,
    -0.00284,
    -0.00476,
    0.00703,
    0.00879,
    0.00221,
    0.00314,
    0.001,
    -0.00337,
    -0.00416
   ],
   [
    0.00775,
    0.01601,
    -0.00685,
    0.01919,
    -0.01124,
    0.00457,
    -0.00222,
    0.00817,
    0.01329,
    0.00098,
    0.01607,
    -0.01067,
    0.00436,
    -0.00138
   ],
   [
    0.00162,
    0.00221,
    0.00699,
    0.0053,
    0.00337,
    -0.01061,
    0.00597,
    -0.00124,
    0.00054,
    -0.0009,
    0.00763,
    0.00402,
    -0.00961,
    0.00573
   ],
   [
    -0.00438,
    0.01514,
    -0.00844,
    -0.0038,
    0.0066,
    4e-05,
    0.00398,
    -0.00318,
    0.01013,
    -0.00653,
    -0.00548,
    0.00473,
    -0.00022,
    0.00402
   ],
   [
    0.01826,
    -0.02335,
    0.00799,
    -0.00103,
    -0.01846,
    -0.00255,
    0.01599,
    0.0148,
    -0.01757,
    0.00507,
    -0.00116,
    -0.01754,
    0.00011,
    0.01513
   ],
   [
    -0.00396,
    0.01429,
    0.00895,
    -0.00491,
    -0.00127,
    0.00514,
    0.0028,
    -0.00208,
    0.0099,
    0.00381,
    -0.00506,
    -0.00097,
    0.00523,
    0.00265
   ],
   [
    0.0036,
    0.01221,
    -0.00608,
    0.00291,
    -0.0001,
    -0.00466,
    -0.00933,
    0.00337,
    0.00841,
    -0.00639,
    0.00345,
    0.0006,
    -0.00217,
    -0.00762
   ],
   [
    -0.0027,
    -0.01942,
    -0.00463,
    0.0129,
    -0.011,
    -0.00613,
    -0.00235,
    -0.00602,
    -0.01465,
    -0.00082,
    0.0124,
    -0.0113,
    -0.00703,
    -0.00359
   ],
   [
    0.01278,
    0.01681,
    0.00469,
    0.00273,
    -0.00212,
    0.00913,
    0.00294,
    0.012,
    0.01749,
    0.00648,
    0.00365,
    -0.003,
    0.00805,
    0.00335
   ],
   [
    -0.00211,
    -0.01626,
    0.00427,
    -0.00427,
    -0.0069,
    0.0016,
    0.00468,
    -0.00307,
    -0.01489,
    0.00357,
    -0.00389,
    -0.00774,
    -5e-05,
    0.00249
   ],
   [
    -0.00911,
    -0.01146,
    -0.00812,
    -0.00868,
    0.00205,
    -0.01119,
    0.01089,
    -0.00653,
    -0.01087,
    -0.00526,
    -0.00903,
    0.00233,
    -0.00771,
    0.00904
   ],
   [
    -0.007,
    -0.00033,
    -0.00036,
    0.00314,
    -0.00103,
    0.00066,
    0.00615,
    -0.0044,
    -0.00154,
    -0.0013,
    0.00311,
    -0.00021,
    5e-05,
    0.00594
   ],
   [
    0.0188,
    0.00668,
    0.00917,
    -0.00782,
    0.00247,
    0.00088,
    0.00179,
    0.01081,
    0.00972,
    0.00404,
    -0.00448,
    0.0025,
    7e-05,
    -0.00047
   ],
   [
    0.0044,
    -0.00141,
    -0.01067,
    -0.0037,
    0.00331,
    0.0001,
    -0.01143,
    0.00092,
    -0.00334,
    -0.00662,
    -0.00329,
    0.00147,
    -0.00314,
    -0.01032
   ],
   [
    0.0089,
    0.0148,
    -0.00912,
    0.00599,
    0.03052,
    -0.00611,
    -0.01753,
    0.00564,
    0.00783,
    -0.00522,
    0.00456,
    0.02313,
    -0.00753,
    -0.01277
   ],
   [
    -0.00099,
    -0.0067,
    -0.00301,
    0.00027,
    -0.0093,
    0.00285,
    -0.00073,
    -0.00037,
    -0.0036,
    -0.00236,
    0.00031,
    -0.00695,
    0.00516,
    0.00164
   ],
   [
    0.01909,
    -0.011,
    0.00571,
    -0.00975,
    -0.01071,
    -0.00608,
    0.00354,
    0.01068,
    -0.0051,
    0.00713,
    -0.00991,
    -0.01227,
    -0.0075,
    -0.0
   ],
   [
    0.00014,
    -0.00799,
    0.00836,
    0.00848,
    0.01141,
    -0.00529,
    -0.0145,
    -0.00106,
    -0.00934,
    0.00772,
    0.00689,
    0.00701,
    -0.00601,
    -0.01243
   ],
   [
    -0.00826,
    -0.0126,
    -0.00556,
    -0.00596,
    -0.00415,
    0.01779,
    0.00833,
    -0.0077,
    -0.00972,
    -0.00163,
    -0.00658,
    -0.00328,
    0.01643,
    0.00808
   ],
   [
    -0.02389,
    -0.01464,
    0.01225,
    -0.00888,
    0.00551,
    -0.00135,
    0.00964,
    -0.02058,
    -0.01634,
    0.01186,
    -0.00568,
    0.00914,
    0.00139,
    0.00984
   ],
   [
    0.00911,
    -0.00134,
    0.00794,
    0.00405,
    -0.00674,
    -0.00116,
    -0.0067,
    0.00688,
    -0.00037,
    0.00668,
    0.00593,
    -0.00419,
    -0.00251,
    -0.00728
   ],
   [
    0.01405,
    -0.04059,
    0.00911,
    0.0077,
    -0.01987,
    -0.02528,
    0.01556,
    0.00312,
    -0.02465,
    0.00372,
    0.00982,
    -0.01398,
    -0.01225,
    0.01754
   ],
   [
    -0.00239,
    -0.00265,
    0.00377,
    -0.00747,
    -0.00386,
    0.00642,
    -0.00303,
    -0.00083,
    -0.00128,
    0.00341,
    -0.00621,
    -0.00266,
    0.00669,
    -0.00116
   ],
   [
    0.01201,
    -0.02291,
    0.0108,
    0.00636,
    -0.0026,
    -0.00583,
    -0.00089,
    0.00782,
    -0.01388,
    0.00985,
    0.00664,
    -0.00471,
    -0.00467,
    -0.00301
   ],
   [
    -0.01635,
    0.01285,
    -0.00111,
    -0.00461,
    0.00835,
    -0.01233,
    -0.00808,
    -0.01171,
    0.00579,
    -0.005,
    -0.00379,
    0.00703,
    -0.00941,
    -0.00746
   ],
   [
    0.00315,
    -0.00649,
    -0.00796,
    0.00318,
    2e-05,
    0.00089,
    0.0017,
    0.0013,
    -0.00648,
    -0.00618,
    0.00444,
    0.00176,
    0.00194,
    0.00168
   ],
   [
    0.0072,
    0.00543,
    0.0083,
    -0.00486,
    0.00128,
    -0.00331,
    -0.00368,
    0.00458,
    0.00446,
    0.00919,
    -0.00382,
    0.002,
    -0.0024,
    -0.00127
   ],
   [
    -0.00795,
    -6e-05,
    -0.01291,
    -0.00308,
    0.00872,
    -0.01172,
    0.00582,
    -0.0067,
    -0.0009,
    -0.00711,
    -0.00342,
    0.00693,
    -0.01103,
    0.00407
   ],
   [
    0.00672,
    -0.00715,
    -0.0069,
    -0.01377,
    -0.0057,
    0.00605,
    0.00419,
    0.00724,
    -0.00649,
    0.00139,
    -0.01208,
    -0.00578,
    0.00585,
    0.00387
   ],
   [
    -0.00051,
    -0.01572,
    -0.00099,
    0.01363,
    -0.01005,
    -0.00335,
    0.00245,
    -0.00098,
    -0.01437,
    -0.00261,
    0.0134,
    -0.00817,
    -0.00462,
    0.00398
   ],
   [
    -0.01516,
    -0.00124,
    0.0067,
    -0.00668,
    -0.00538,
    0.00452,
    0.00346,
    -0.01204,
    -0.00459,
    0.00504,
    -0.00707,
    -0.0053,
    0.00411,
    0.00246
   ],
   [
    0.00502,
    -0.00083,
    0.0202,
    0.00762,
    0.00163,
    -0.00792,
    -0.01785,
    0.00718,
    -0.0042,
    0.01265,
    0.00664,
    0.00208,
    -0.00582,
    -0.01307
   ],
   [
    -0.01228,
    0.00875,
    0.00512,
    -0.005,
    0.00916,
    0.00223,
    0.00472,
    -0.01006,
    0.00783,
    0.00754,
    -0.00588,
    0.00764,
    0.00415,
    0.00557
   ],
   [
    0.00231,
    -5e-05,
    0.01088,
    0.01055,
    0.00073,
    -0.00524,
    -0.00314,
    0.00037,
    0.00261,
    0.00607,
    0.01209,
    -0.00108,
    -0.00654,
    -0.00342
   ],
   [
    -0.00686,
    0.00618,
    -0.00792,
    0.01401,
    0.00299,
    0.00718,
    -0.0119,
    -0.00541,
    0.00633,
    -0.00403,
    0.01334,
    0.00187,
    0.00486,
    -0.00916
   ],
   [
    -0.01219,
    -0.01474,
    -0.00198,
    -0.00111,
    0.00944,
    -0.0026,
    -0.00861,
    -0.00998,
    -0.01665,
    0.0013,
    -0.00237,
    0.00706,
    -0.00183,
    -0.00624
   ],
   [
    -0.0067,
    0.00838,
    -0.00709,
    -0.001,
    0.00383,
    -0.00493,
    -0.01051,
    -0.00641,
    0.0052,
    -0.00699,
    -0.00317,
    6e-05,
    -0.00502,
    -0.00747
   ],
   [
    -0.03535,
    -0.00452,
    -0.00758,
    -0.00937,
    0.00253,
    -0.01294,
    0.0193,
    -0.02879,
    -0.00518,
    -0.00694,
    -0.00833,
    0.00233,
    -0.01316,
    0.01629
   ],
   [
    -0.00689,
    -0.00839,
    -0.00983,
    -0.01396,
    0.01004,
    0.00423,
    0.01213,
    -0.00728,
    -0.00889,
    -0.00531,
    -0.01674,
    0.00435,
    0.00301,
    0.00864
   ],
   [
    0.00753,
    0.00572,
    0.00698,
    0.00983,
    -0.00043,
    -0.00467,
    -0.00542,
    0.00794,
    0.00733,
    0.00233,
    0.00817,
    -0.00281,
    -0.003,
    -0.00472
   ],
   [
    -0.00592,
    -0.01167,
    -0.00217,
    0.0174,
    -0.01154,
    -0.00221,
    -0.00427,
    -0.00743,
    -0.00886,
    -0.00457,
    0.01642,
    -0.00994,
    -0.00287,
    -0.00533
   ],
   [
    0.002,
    -0.00115,
    0.00419,
    0.0041,
    -0.00206,
    -0.00138,
    -0.00212,
    0.00061,
    -0.00124,
    0.00254,
    0.00535,
    -0.00248,
    -0.00168,
    -0.00338
   ],
   [
    -0.02536,
    0.00341,
    0.00072,
    0.00224,
    0.00913,
    -0.00016,
    -0.00213,
    -0.02172,
    0.00109,
    -0.00304,
    0.00074,
    0.00629,
    -0.0017,
    -0.00228
   ],
   [
    -0.00252,
    -0.01047,
    0.00128,
    0.00386,
    -0.00454,
    0.00307,
    -0.00846,
    -0.00254,
    -0.01142,
    0.00311,
    0.00304,
    -0.00435,
    0.00415,
    -0.00602
   ],
   [
    -0.00094,
    -0.00454,
    0.01321,
    0.00523,
    -0.00542,
    0.00507,
    -0.0064,
    0.00135,
    -0.00079,
    0.00919,
    0.00444,
    -0.0049,
    0.00503,
    -0.00452
   ],
   [
    -0.00412,
    0.01183,
    -0.00106,
    0.00384,
    0.01035,
    0.0026,
    -0.00848,
    -0.00072,
    0.01087,
    0.00562,
    0.00223,
    0.0086,
    0.00272,
    -0.00657
   ],
   [
    -0.01767,
    -0.01405,
    -0.00463,
    -0.00104,
    -0.00255,
    0.00365,
    -0.00685,
    -0.01395,
    -0.0146,
    -0.00091,
    -0.00184,
    -0.00337,
    0.00233,
    -0.00569
   ],
   [
    0.00712,
    -0.0109,
    -0.00148,
    -0.01194,
    -0.00309,
    0.00396,
    0.00062,
    0.00416,
    -0.0098,
    -0.00203,
    -0.01156,
    -0.00357,
    0.00319,
    0.00056
   ],
   [
    0.01244,
    0.00307,
    -0.00537,
    0.0135,
    -0.01037,
    -0.00068,
    -0.00206,
    0.01379,
    0.00095,
    -0.00267,
    0.01033,
    -0.01011,
    -0.00108,
    -0.00238
   ],
   [
    -0.0048,
    -0.00987,
    -0.0053,
    -0.00688,
    0.00033,
    0.00779,
    0.00145,
    -0.00465,
    -0.00909,
    -0.00618,
    -0.00581,
    0.00203,
    0.00698,
    0.00174
   ],
   [
    -0.00075,
    -0.0115,
    0.00431,
    -0.00597,
    -0.00048,
    0.00415,
    -0.00112,
    0.00035,
    -0.00562,
    0.00421,
    -0.0045,
    0.00129,
    0.0038,
    -0.00217
   ],
   [
    0.0033,
    0.00728,
    -0.01043,
    -1e-05,
    0.00602,
    -0.00125,
    0.00216,
    0.00121,
    0.0064,
    -0.01015,
    -0.0005,
    0.00587,
    -0.00197,
    0.00064
   ],
   [
    0.0058,
    0.01056,
    -0.00931,
    0.00618,
    0.0068,
    0.00094,
    0.00205,
    0.00568,
    0.00441,
    -0.00739,
    0.00482,
    0.00666,
    0.00303,
    0.00233
   ],
   [
    -0.01182,
    -0.01257,
    0.0053,
    -0.00744,
    -0.0025,
    0.00825,
    0.00143,
    -0.00984,
    -0.01112,
    0.00371,
    -0.00568,
    -0.00108,
    0.00954,
    0.00407
   ],
   [
    -0.01419,
    -0.01118,
    -0.0003,
    -0.00711,
    -0.00588,
    0.01044,
    -0.00178,
    -0.0114,
    -0.00862,
    0.00165,
    -0.00796,
    -0.00716,
    0.0085,
    -0.00176
   ],
   [
    0.01255,
    -0.02476,
    0.02132,
    -0.01073,
    -0.00709,
    0.00883,
    0.0024,
    0.01282,
    -0.01367,
    0.01633,
    -0.00836,
    -0.00778,
    0.00932,
    0.00311
   ],
   [
    0.00464,
    0.00433,
    -0.01288,
    0.00409,
    0.00121,
    -0.00261,
    -0.00512,
    0.00523,
    0.00174,
    -0.01174,
    0.00174,
    0.00125,
    -0.00256,
    -0.00485
   ],
   [
    -0.0025,
    0.00538,
    -0.01436,
    -0.0022,
    -0.00779,
    0.01034,
    -0.00087,
    -0.00508,
    -0.00198,
    -0.01084,
    -0.00481,
    -0.00825,
    0.00733,
    0.00022
   ],
   [
    0.00926,
    -0.0042,
    0.00779,
    0.00863,
    0.00611,
    -0.00121,
    -0.00879,
    0.00621,
    -0.00485,
    0.00553,
    0.01015,
    0.00541,
    -0.00339,
    -0.00817
   ],
   [
    -0.03391,
    0.02423,
    0.02544,
    -0.00719,
    0.00711,
    0.01185,
    0.01721,
    -0.02063,
    0.02075,
    0.0113,
    -0.00607,
    0.0078,
    0.01109,
    0.01802
   ],
   [
    -0.01446,
    -0.00013,
    -0.00321,
    0.01159,
    -0.002,
    0.00032,
    -0.00532,
    -0.01462,
    -0.0043,
    -0.00855,
    0.00946,
    -0.00339,
    0.00097,
    -0.00311
   ],
   [
    0.0126,
    0.00631,
    0.00375,
    0.00322,
    -0.00123,
    -0.00654,
    0.00407,
    0.00724,
    0.00723,
    0.00665,
    0.00484,
    -0.00097,
    -0.00505,
    0.00387
   ],
   [
    0.00996,
    -0.00749,
    0.00641,
    0.0024,
    -0.00645,
    0.00434,
    0.00968,
    0.00788,
    -0.00416,
    0.00174,
    0.00288,
    -0.00519,
    0.00907,
    0.01116
   ],
   [
    -0.01626,
    -0.00957,
    0.00095,
    8e-05,
    0.00023,
    -0.00078,
    0.00096,
    -0.01506,
    -0.00947,
    0.00054,
    0.00162,
    0.00263,
    0.00048,
    0.00305
   ],
   [
    0.00615,
    0.00401,
    0.00128,
    0.00042,
    -0.00211,
    0.00362,
    -0.00729,
    0.00564,
    0.00185,
    0.00178,
    -0.00187,
    -0.00332,
    0.002,
    -0.00742
   ],
   [
    0.00851,
    0.01063,
    -0.00377,
    0.00996,
    -0.0016,
    0.00025,
    -0.00289,
    0.00695,
    0.00627,
    -0.00186,
    0.00741,
    0.00015,
    0.00079,
    -0.00136
   ],
   [
    0.00704,
    -0.00174,
    0.00174,
    -7e-05,
    0.00641,
    -0.01126,
    0.00155,
    0.00527,
    -0.00134,
    -0.00185,
    0.00087,
    0.00552,
    -0.00924,
    0.00122
   ],
   [
    0.01441,
    0.01441,
    0.00587,
    0.00466,
    0.00662,
    -0.00353,
    -0.00389,
    0.01166,
    0.01458,
    0.00822,
    0.00593,
    0.0069,
    -0.00201,
    -0.00155
   ],
   [
    0.01697,
    0.00291,
    -0.00492,
    -0.00259,
    0.00615,
    -0.00089,
    0.01682,
    0.01585,
    0.00296,
    -0.00252,
    -0.00276,
    0.00588,
    0.00138,
    0.01398
   ],
   [
    -0.00861,
    0.00016,
    0.00474,
    -0.00575,
    -0.00385,
    0.00822,
    -0.00014,
    -0.00535,
    -0.00326,
    0.00488,
    -0.00598,
    -0.00222,
    0.00962,
    0.00094
   ],
   [
    -0.0222,
    -0.0112,
    -0.01196,
    -0.00214,
    0.00043,
    0.00294,
    -0.00312,
    -0.01947,
    -0.01514,
    -0.00765,
    -0.00585,
    -0.00276,
    0.00202,
    -0.0032
   ],
   [
    0.0254,
    -0.00262,
    0.00806,
    0.00776,
    -0.00692,
    -0.00258,
    0.01846,
    0.02085,
    -0.00028,
    0.00601,
    0.00594,
    -0.00737,
    -0.00355,
    0.01433
   ],
   [
    -0.01129,
    0.00472,
    0.00681,
    -0.00174,
    0.00595,
    -0.00069,
    -0.00325,
    -0.0097,
    0.00499,
    0.00948,
    0.00031,
    0.00548,
    0.00014,
    -0.00123
   ],
   [
    0.00238,
    0.01091,
    -0.00575,
    0.01589,
    0.00074,
    -0.00528,
    -0.00205,
    0.00367,
    0.0083,
    -0.00826,
    0.01283,
    -0.00012,
    -0.0074,
    -0.00458
   ],
   [
    -0.00598,
    0.01611,
    -0.00712,
    -0.00301,
    0.00651,
    -0.00995,
    -0.00739,
    -0.00425,
    0.01161,
    -0.00733,
    -0.00753,
    0.00237,
    -0.01117,
    -0.00816
   ],
   [
    -0.00953,
    0.00295,
    0.00276,
    0.00575,
    0.00146,
    -0.00242,
    -0.00688,
    -0.01056,
    1e-05,
    0.00049,
    0.00609,
    0.00146,
    -0.00298,
    -0.00697
   ],
   [
    -0.0024,
    0.00445,
    0.00865,
    -0.0016,
    -0.00015,
    0.0088,
    0.00388,
    -0.00492,
    0.00909,
    0.00702,
    -0.00042,
    0.00172,
    0.00819,
    0.0038
   ],
   [
    -0.00754,
    -0.02161,
    0.00385,
    -0.00247,
    -0.00706,
    0.00369,
    -0.00473,
    -0.00549,
    -0.01821,
    0.00334,
    -0.00213,
    -0.00643,
    0.00406,
    -0.00406
   ],
   [
    -0.00807,
    0.0058,
    -0.00303,
    0.00506,
    -0.00337,
    0.00435,
    -0.0012,
    -0.00399,
    0.0021,
    -0.00257,
    0.00172,
    -0.00382,
    0.00395,
    0.00067
   ],
   [
    -0.00604,
    0.02284,
    -0.00492,
    -0.01193,
    0.00411,
    0.00963,
    0.00317,
    -0.00191,
    0.02134,
    -0.00193,
    -0.01186,
    0.00485,
    0.00925,
    0.00479
   ],
   [
    0.00819,
    0.00209,
    -0.00083,
    0.00371,
    0.00602,
    0.00117,
    -0.00513,
    0.006,
    0.00437,
    0.00159,
    0.0024,
    0.00406,
    0.00094,
    -0.00517
   ],
   [
    0.00656,
    -0.00449,
    0.00752,
    -0.00623,
    -0.00072,
    0.00397,
    0.0028,
    0.00563,
    -0.00309,
    0.00657,
    -0.00432,
    -0.00078,
    0.00385,
    0.00263
   ],
   [
    -0.00359,
    -0.00521,
    -0.01946,
    0.00498,
    0.01257,
    -0.01127,
    0.00033,
    -0.00252,
    -0.01304,
    -0.01581,
    0.00144,
    0.00652,
    -0.01162,
    -0.00072
   ],
   [
    0.02117,
    -0.00507,
    0.00867,
    0.00422,
    -0.00448,
    0.00469,
    -0.00642,
    0.01489,
    -0.00251,
    0.00611,
    0.00647,
    -0.00381,
    0.00395,
    -0.00608
   ],
   [
    -0.01408,
    0.0048,
    0.00683,
    -0.00941,
    0.00781,
    -0.00578,
    0.00932,
    -0.01175,
    0.00435,
    0.00308,
    -0.00839,
    0.00571,
    -0.0055,
    0.0076
   ],
   [
    0.02222,
    -0.00321,
    -0.00058,
    0.00607,
    -0.01449,
    -0.00294,
    -0.00141,
    0.02094,
    -0.00397,
    0.00263,
    0.00478,
    -0.01277,
    -0.00048,
    -0.00014
   ],
   [
    -0.01898,
    -0.01404,
    0.01471,
    -0.00294,
    -0.0035,
    -0.00749,
    -0.00227,
    -0.01636,
    -0.01346,
    0.00759,
    0.00151,
    -0.00141,
    -0.00411,
    -0.00204
   ],
   [
    0.01926,
    -0.00426,
    -0.01353,
    0.02365,
    -0.00585,
    -0.00095,
    -0.00808,
    0.01441,
    -0.004,
    -0.01076,
    0.02096,
    -0.00332,
    -0.00212,
    -0.00616
   ],
   [
    -0.00798,
    -0.01323,
    0.00138,
    0.00025,
    -0.00813,
    0.00354,
    -0.00069,
    -0.00604,
    -0.01119,
    0.00185,
    0.00231,
    -0.00619,
    0.00432,
    0.00068
   ],
   [
    0.02292,
    0.00157,
    0.01292,
    0.01112,
    -0.01579,
    -0.00135,
    -0.00306,
    0.0193,
    0.00276,
    0.00803,
    0.01364,
    -0.00997,
    0.00181,
    0.00056
   ],
   [
    0.02551,
    0.02322,
    0.00419,
    0.00523,
    0.00057,
    -0.00025,
    0.00707,
    0.02237,
    0.01673,
    0.00407,
    0.00554,
    0.00196,
    -0.00043,
    0.00678
   ],
   [
    -0.01349,
    0.00738,
    -0.00817,
    -0.01096,
    0.00097,
    0.00909,
    0.00304,
    -0.00957,
    0.00608,
    -0.00777,
    -0.01062,
    0.00032,
    0.00967,
    0.00365
   ],
   [
    0.00488,
    -0.02513,
    0.00993,
    -0.00311,
    -0.01247,
    -0.00798,
    0.01013,
    -0.00065,
    -0.01973,
    0.00492,
    -0.002,
    -0.01145,
    -0.007,
    0.00692
   ],
   [
    -0.01059,
    -0.00829,
    0.00429,
    -0.00051,
    -0.00377,
    -0.00988,
    -0.00388,
    -0.00809,
    -0.00793,
    0.0003,
    -0.00148,
    -0.00405,
    -0.00915,
    -0.00342
   ],
   [
    -0.00269,
    0.00374,
    0.00677,
    -0.00651,
    0.00797,
    0.00814,
    -0.00965,
    -0.00037,
    0.00208,
    0.00143,
    -0.00466,
    0.00934,
    0.00637,
    -0.00781
   ],
   [
    -0.00653,
    0.00391,
    0.00484,
    0.00481,
    0.00174,
    -0.00892,
    -0.00073,
    -0.0075,
    0.00231,
    0.00445,
    0.00509,
    -3e-05,
    -0.00944,
    -0.00284
   ],
   [
    0.00878,
    -0.00762,
    0.01081,
    0.00247,
    -0.01807,
    -0.00519,
    0.00033,
    0.00664,
    -0.00572,
    0.00492,
    0.00474,
    -0.01611,
    -0.00574,
    -9e-05
   ],
   [
    -0.00561,
    0.00224,
    -0.03302,
    0.03109,
    0.00253,
    -0.01315,
    -0.02237,
    -0.00304,
    -0.00021,
    -0.02697,
    0.02919,
    0.00381,
    -0.01123,
    -0.01906
   ],
   [
    -0.00498,
    0.00042,
    0.00692,
    0.0135,
    -0.00491,
    -0.01125,
    -0.0145,
    -0.00601,
    -0.00647,
    0.00472,
    0.00894,
    -0.00754,
    -0.00834,
    -0.01193
   ],
   [
    0.01937,
    0.00501,
    0.0042,
    0.00784,
    -0.00524,
    -0.00145,
    0.00191,
    0.01642,
    0.00419,
    0.00568,
    0.00971,
    -0.00303,
    -0.00097,
    0.00188
   ],
   [
    0.01199,
    0.00728,
    -0.00641,
    0.01303,
    -0.00102,
    -0.0016,
    -0.01612,
    0.00763,
    0.00452,
    -0.00245,
    0.00956,
    -0.0015,
    -0.00242,
    -0.01243
   ],
   [
    0.01164,
    0.0084,
    0.00444,
    -0.00836,
    -0.00064,
    0.00581,
    0.01903,
    0.00756,
    0.01351,
    0.00984,
    -0.00353,
    0.00616,
    0.00704,
    0.01773
   ],
   [
    -0.00169,
    0.01577,
    -0.04652,
    0.00474,
    0.0089,
    -0.00965,
    -0.03129,
    -0.00339,
    0.00694,
    -0.03668,
    0.00395,
    0.00371,
    -0.01159,
    -0.02724
   ],
   [
    0.00226,
    0.00725,
    0.00082,
    -0.01073,
    0.01085,
    0.00922,
    0.004,
    0.00255,
    0.00846,
    8e-05,
    -0.00762,
    0.01157,
    0.00815,
    0.00343
   ],
   [
    -0.02032,
    -0.01271,
    0.00482,
    -0.00492,
    -0.00504,
    0.00123,
    -0.00226,
    -0.01553,
    -0.01438,
    0.00338,
    -0.00667,
    -0.00579,
    0.0006,
    -0.00114
   ],
   [
    -0.00024,
    -0.02221,
    0.00612,
    -0.00172,
    -0.01146,
    0.00528,
    0.00861,
    -0.00212,
    -0.0117,
    0.00579,
    2e-05,
    -0.00912,
    0.00522,
    0.00684
   ],
   [
    0.00075,
    -0.02281,
    -0.00822,
    -6e-05,
    -0.01469,
    0.00599,
    0.00379,
    -0.00146,
    -0.02031,
    -0.00664,
    -0.00174,
    -0.01444,
    0.00217,
    0.00029
   ],
   [
    -0.00964,
    -0.01737,
    0.0073,
    -0.01383,
    -0.00064,
    0.00899,
    0.00235,
    -0.00872,
    -0.0159,
    0.00658,
    -0.01296,
    -0.00168,
    0.0075,
    0.00216
   ],
   [
    -0.00371,
    0.01711,
    -0.00266,
    -0.00879,
    0.01559,
    0.00293,
    0.00231,
    -0.00368,
    0.01309,
    -0.00481,
    -0.00838,
    0.01413,
    0.00179,
    0.00154
   ],
   [
    -0.00491,
    0.01,
    0.00449,
    0.01705,
    -0.00467,
    -0.00822,
    -0.00574,
    -0.00402,
    0.00259,
    0.00032,
    0.01432,
    -0.00408,
    -0.0084,
    -0.00481
   ],
   [
    -0.01476,
    -0.02162,
    -0.02218,
    -0.01021,
    0.00543,
    -0.00149,
    0.00558,
    -0.01396,
    -0.01731,
    -0.01567,
    -0.00985,
    0.00447,
    -0.00223,
    0.00449
   ],
   [
    -0.0149,
    0.00659,
    -0.00095,
    -0.00357,
    -0.00351,
    0.00685,
    0.00589,
    -0.01155,
    0.00499,
    0.00197,
    -0.00442,
    -0.00257,
    0.00574,
    0.00636
   ],
   [
    0.00138,
    0.00709,
    0.00392,
    0.00408,
    0.00487,
    -0.00456,
    -0.00067,
    0.00037,
    0.00947,
    0.00326,
    0.00425,
    0.00402,
    -0.00199,
    0.00105
   ],
   [
    0.00877,
    0.00482,
    -0.00262,
    0.00075,
    0.005,
    -0.00795,
    -0.00083,
    0.00885,
    0.00432,
    -0.00479,
    0.00041,
    0.00336,
    -0.00657,
    -0.00182
   ],
   [
    -0.01885,
    -0.02633,
    0.00381,
    -0.00237,
    -0.00867,
    0.00385,
    -0.00394,
    -0.01368,
    -0.02352,
    0.00441,
    -0.00214,
    -0.00644,
    0.00785,
    -0.00165
   ],
   [
    0.00471,
    -0.0143,
    -0.00146,
    0.00477,
    -0.00334,
    -0.00736,
    -0.0097,
    0.00189,
    -0.01145,
    -0.00181,
    0.0044,
    -0.00433,
    -0.00738,
    -0.0091
   ],
   [
    -0.00081,
    -0.00742,
    0.00269,
    8e-05,
    0.00207,
    -0.00391,
    0.00516,
    -0.00102,
    -0.00784,
    -0.0019,
    -0.00361,
    -0.00084,
    -0.00417,
    0.00253
   ],
   [
    0.01188,
    0.0074,
    0.0052,
    0.01083,
    -0.00151,
    -0.00542,
    -0.00024,
    0.00894,
    0.00558,
    0.00487,
    0.01005,
    -0.00038,
    -0.00305,
    0.00018
   ],
   [
    -0.00826,
    -0.00449,
    0.00675,
    0.00375,
    -0.00156,
    0.00476,
    -0.01136,
    -0.00594,
    -0.00429,
    0.0049,
    0.00314,
    -0.0023,
    0.00358,
    -0.00907
   ],
   [
    0.00565,
    -0.00123,
    0.00353,
    -0.00325,
    -0.00942,
    0.00789,
    0.00057,
    0.00503,
    -0.0002,
    0.0033,
    -0.00291,
    -0.00755,
    0.00677,
    0.00052
   ],
   [
    0.00702,
    -0.00455,
    0.0004,
    -0.00145,
    -0.00413,
    -0.00516,
    -0.00042,
    0.00356,
    -0.00265,
    -0.00216,
    -0.00038,
    -0.00353,
    -0.00705,
    -0.00307
   ],
   [
    -0.00172,
    0.00056,
    0.00641,
    0.00661,
    0.00474,
    -0.00546,
    0.00524,
    -0.00487,
    -0.00114,
    0.00428,
    0.0059,
    0.00369,
    -0.00242,
    0.00536
   ],
   [
    0.00914,
    -0.00876,
    0.00265,
    0.01454,
    -0.00586,
    -0.00547,
    0.00917,
    0.00554,
    -0.00077,
    0.00367,
    0.01701,
    -0.00547,
    -0.00492,
    0.00553
   ],
   [
    0.00888,
    -0.00251,
    0.00162,
    3e-05,
    0.00109,
    -0.00469,
    0.01144,
    0.00899,
    -0.00258,
    -0.00022,
    0.00153,
    0.00193,
    -0.00286,
    0.00916
   ],
   [
    0.01347,
    0.00049,
    0.00671,
    -0.00028,
    -0.01002,
    -0.00194,
    -0.00396,
    0.01203,
    0.00321,
    0.00534,
    -0.00057,
    -0.00924,
    -0.00302,
    -0.00437
   ],
   [
    0.02232,
    -0.02275,
    -0.01096,
    0.02361,
    -0.01502,
    -0.02293,
    -0.00111,
    0.01913,
    -0.02094,
    -0.00817,
    0.02149,
    -0.01701,
    -0.0226,
    -0.00204
   ],
   [
    -0.00112,
    0.00754,
    0.00133,
    0.00453,
    0.00346,
    -0.01024,
    -0.00562,
    1e-05,
    0.00152,
    0.00179,
    0.00234,
    0.0016,
    -0.01108,
    -0.00518
   ],
   [
    -0.02971,
    -0.04563,
    0.01911,
    0.00367,
    -0.02992,
    0.02018,
    -0.00713,
    -0.01839,
    -0.04878,
    0.00403,
    0.00227,
    -0.02976,
    0.01444,
    -0.01198
   ],
   [
    -0.00737,
    0.00438,
    -0.00034,
    -0.01997,
    0.0087,
    0.01186,
    0.00279,
    -0.00835,
    0.00299,
    -0.00323,
    -0.01776,
    0.00981,
    0.01201,
    0.00311
   ],
   [
    -0.00467,
    -0.00221,
    -0.0064,
    -0.01163,
    0.00074,
    -0.00774,
    0.01881,
    -0.00424,
    -0.00043,
    -0.00844,
    -0.01324,
    0.00117,
    -0.00723,
    0.01458
   ],
   [
    0.01451,
    -0.01464,
    -0.00034,
    -0.00092,
    -0.01297,
    -0.00482,
    0.01084,
    0.01021,
    -0.01019,
    -0.00177,
    0.00195,
    -0.00761,
    -0.00348,
    0.01016
   ],
   [
    -0.00287,
    -0.01226,
    0.00148,
    -0.00104,
    -0.0012,
    0.00325,
    0.0063,
    -0.00073,
    -0.01029,
    0.00044,
    -0.00045,
    -0.00096,
    0.00215,
    0.00534
   ],
   [
    0.00162,
    -0.01306,
    0.0067,
    0.00475,
    -0.00957,
    -0.00189,
    0.01088,
    -0.00104,
    -0.00776,
    0.00585,
    0.00526,
    -0.0075,
    0.00096,
    0.01002
   ],
   [
    -0.01307,
    -0.00572,
    -0.00795,
    0.00181,
    0.01111,
    -0.00698,
    -0.01301,
    -0.01569,
    -0.01172,
    -0.00715,
    -0.00172,
    0.00872,
    -0.00598,
    -0.01151
   ],
   [
    0.0074,
    -0.00091,
    0.00506,
    -0.00446,
    0.00997,
    0.00124,
    0.01314,
    0.00431,
    0.0043,
    0.00438,
    -0.00268,
    0.00913,
    0.00271,
    0.01203
   ],
   [
    -0.00649,
    -0.00349,
    0.00982,
    0.003,
    -0.00589,
    -0.00121,
    -0.00723,
    -0.00461,
    -0.00734,
    0.00204,
    0.00373,
    -0.00581,
    -0.00203,
    -0.00619
   ],
   [
    -0.00534,
    -0.00075,
    0.00073,
    -0.00056,
    -0.00434,
    0.00477,
    -0.00159,
    -0.00268,
    -0.00019,
    0.00291,
    0.00032,
    -0.00262,
    0.00514,
    0.0001
   ],
   [
    -0.01419,
    0.00961,
    -0.01367,
    0.00105,
    0.02089,
    0.0058,
    -0.00833,
    -0.01141,
    0.00943,
    -0.01089,
    -0.00064,
    0.01508,
    0.00336,
    -0.0074
   ],
   [
    0.01657,
    0.00767,
    -0.00164,
    -0.00652,
    0.00296,
    0.0088,
    0.00978,
    0.01597,
    0.01215,
    0.00376,
    -0.00602,
    0.00069,
    0.00865,
    0.0092
   ],
   [
    0.00191,
    -0.0006,
    0.01149,
    0.0032,
    -0.00198,
    0.01913,
    -0.0099,
    0.00226,
    0.00115,
    0.00989,
    0.00045,
    -0.00333,
    0.01672,
    -0.0072
   ],
   [
    -0.00527,
    -0.01421,
    0.00184,
    -0.006,
    0.00379,
    0.00334,
    0.00374,
    -0.00629,
    -0.00833,
    0.00248,
    -0.00545,
    0.00395,
    0.00395,
    0.00349
   ],
   [
    0.00622,
    0.00379,
    -0.0029,
    0.00166,
    0.00393,
    -0.00027,
    0.004,
    0.00633,
    0.00576,
    0.00045,
    0.00275,
    0.00501,
    0.00215,
    0.00456
   ],
   [
    -0.00385,
    -0.01191,
    0.00397,
    -0.01033,
    -0.01302,
    0.01014,
    0.00076,
    -0.00461,
    -0.01319,
    0.0013,
    -0.01097,
    -0.01144,
    0.00841,
    0.00084
   ],
   [
    0.03084,
    0.01758,
    0.002,
    0.02562,
    0.00263,
    -0.01047,
    -0.00766,
    0.02632,
    0.01534,
    -0.00186,
    0.02406,
    0.00463,
    -0.00681,
    -0.00584
   ],
   [
    -0.00167,
    -0.00511,
    -0.01006,
    0.00213,
    -0.00747,
    -0.00068,
    0.00019,
    -0.00261,
    -0.00414,
    -0.00565,
    0.00291,
    -0.0052,
    -7e-05,
    0.00142
   ],
   [
    0.00994,
    0.01056,
    0.01029,
    0.00036,
    0.01131,
    -0.0014,
    0.00191,
    0.0082,
    0.00862,
    0.00487,
    5e-05,
    0.00945,
    -0.00281,
    0.00146
   ],
   [
    -0.02262,
    -0.00202,
    0.00884,
    0.00618,
    0.00311,
    -0.00683,
    -0.00401,
    -0.01473,
    -0.00318,
    0.00714,
    0.00416,
    0.00201,
    -0.0061,
    -0.00398
   ],
   [
    -0.02814,
    0.00272,
    -0.01308,
    -0.0126,
    0.00967,
    -0.00718,
    0.00085,
    -0.02133,
    -0.00144,
    -0.00739,
    -0.01417,
    0.00759,
    -0.00552,
    0.00057
   ],
   [
    -0.02213,
    -0.00755,
    0.0085,
    -0.00931,
    0.00586,
    0.00631,
    0.00266,
    -0.01592,
    -0.00309,
    0.00507,
    -0.01137,
    0.00291,
    0.00608,
    0.0033
   ],
   [
    -0.01085,
    0.00634,
    0.00864,
    -0.00305,
    0.00891,
    -0.00359,
    -0.0021,
    -0.00751,
    0.00722,
    0.00528,
    -0.0034,
    0.00841,
    -0.00322,
    -0.00189
   ],
   [
    0.00407,
    -0.01229,
    0.01744,
    -0.01277,
    -0.01343,
    0.0107,
    0.0239,
    0.00183,
    -0.00786,
    0.01539,
    -0.01146,
    -0.0124,
    0.00813,
    0.0203
   ],
   [
    0.00328,
    -0.00438,
    -0.00069,
    0.00473,
    -0.01522,
    0.00056,
    0.00915,
    0.00034,
    -0.00364,
    -0.00167,
    0.00485,
    -0.01374,
    -0.00032,
    0.00718
   ],
   [
    0.00277,
    0.01013,
    -0.00465,
    0.01461,
    0.00724,
    -0.00838,
    -0.01834,
    0.00088,
    0.00255,
    -0.00388,
    0.01581,
    0.0058,
    -0.00894,
    -0.0169
   ],
   [
    -0.00173,
    -0.01021,
    -0.01189,
    -0.00119,
    0.00892,
    0.00329,
    -0.0,
    -0.00408,
    -0.00624,
    -0.00715,
    -0.00023,
    0.0087,
    0.00235,
    0.00011
   ],
   [
    0.00788,
    0.00871,
    -0.00565,
    0.00835,
    0.0047,
    -0.00868,
    -0.00489,
    0.00728,
    0.00759,
    -0.00597,
    0.00512,
    0.00384,
    -0.00795,
    -0.00597
   ],
   [
    -0.01683,
    -0.00512,
    0.00297,
    0.00123,
    -0.00854,
    -0.00269,
    -0.00058,
    -0.01289,
    -0.00392,
    -4e-05,
    0.00176,
    -0.00529,
    -0.00245,
    -0.00135
   ],
   [
    0.0162,
    0.0031,
    0.00993,
    0.00146,
    -0.00674,
    0.00562,
    -0.00286,
    0.01191,
    0.00369,
    0.01038,
    0.0024,
    -0.00698,
    0.0058,
    -0.00326
   ],
   [
    -0.00456,
    0.00092,
    0.01933,
    0.00232,
    -0.00142,
    0.01424,
    -0.0001,
    0.00297,
    0.00447,
    0.01938,
    0.00229,
    -0.00163,
    0.01283,
    0.0013
   ],
   [
    0.00697,
    0.00955,
    -0.00042,
    0.00234,
    0.00587,
    -0.00018,
    0.00153,
    0.00241,
    0.00739,
    0.0013,
    0.00359,
    0.00651,
    0.00069,
    0.00296
   ],
   [
    -0.00232,
    -0.00359,
    0.01394,
    -0.00594,
    -0.01389,
    0.00174,
    0.00807,
    -0.00319,
    -0.00279,
    0.01116,
    -0.00296,
    -0.01026,
    0.00345,
    0.00862
   ],
   [
    -0.0133,
    -0.00274,
    -0.01783,
    0.02677,
    0.01482,
    -0.03481,
    -0.01706,
    -0.01644,
    -0.01251,
    -0.01834,
    0.02198,
    0.01244,
    -0.0338,
    -0.01308
   ],
   [
    0.01067,
    0.00587,
    0.0002,
    -0.00353,
    0.00159,
    -0.0075,
    0.00114,
    0.00769,
    0.00246,
    -0.00031,
    -0.00599,
    0.00015,
    -0.00863,
    -0.00062
   ],
   [
    0.00242,
    0.00316,
    0.001,
    -0.0091,
    1e-05,
    0.00539,
    0.00059,
    0.00176,
    -0.00069,
    -0.00224,
    -0.01007,
    0.00056,
    0.00496,
    0.00144
   ],
   [
    -0.00558,
    -0.00886,
    -0.00312,
    -0.00153,
    0.00657,
    0.00081,
    -0.00348,
    -0.00397,
    -0.00438,
    0.00044,
    -0.00078,
    0.00498,
    0.00276,
    -0.00245
   ],
   [
    -0.01642,
    -0.02897,
    -0.00285,
    -0.00212,
    0.00218,
    -0.0022,
    -0.00096,
    -0.01628,
    -0.02279,
    -0.00295,
    -0.00271,
    0.00271,
    0.00024,
    -0.00098
   ],
   [
    -0.00081,
    0.01295,
    0.00901,
    0.00118,
    0.00703,
    -0.00475,
    0.00022,
    0.00056,
    0.01135,
    0.00375,
    0.00141,
    0.00552,
    -0.00528,
    0.00058
   ],
   [
    -0.01518,
    -0.00158,
    -0.00779,
    0.00273,
    0.00042,
    0.00455,
    0.00082,
    -0.01103,
    -0.00083,
    -0.00381,
    0.00191,
    0.00063,
    0.0029,
    0.00014
   ],
   [
    0.00943,
    0.01286,
    0.00426,
    -0.0064,
    0.005,
    0.00308,
    0.00518,
    0.00761,
    0.01196,
    0.00301,
    -0.00387,
    0.00696,
    0.00225,
    0.00357
   ],
   [
    0.0005,
    -0.01892,
    0.00362,
    0.00257,
    -0.01523,
    -0.00131,
    -0.00611,
    -0.0014,
    -0.01824,
    0.00539,
    0.00467,
    -0.0146,
    -0.00219,
    -0.00463
   ],
   [
    -0.00017,
    -0.01369,
    0.00605,
    -0.00602,
    -0.00669,
    -0.00219,
    0.006,
    0.00234,
    -0.00726,
    0.00562,
    -0.00536,
    -0.00549,
    -0.0015,
    0.00526
   ],
   [
    -0.00882,
    0.01022,
    0.00025,
    0.00245,
    -0.0053,
    -0.00191,
    -0.00088,
    -0.00551,
    0.00651,
    -0.00238,
    0.0017,
    -0.0048,
    -0.00249,
    -0.00023
   ],
   [
    -0.02513,
    -0.02613,
    -0.00758,
    -0.0077,
    -0.00199,
    0.01399,
    0.00836,
    -0.0192,
    -0.01914,
    -0.00444,
    -0.00726,
    -0.00068,
    0.01553,
    0.01003
   ],
   [
    -0.00062,
    0.00196,
    0.00024,
    -0.00435,
    0.00916,
    0.00094,
    -0.00345,
    -0.00194,
    0.00076,
    -0.00283,
    -0.00545,
    0.00716,
    -0.0021,
    -0.00461
   ],
   [
    0.00899,
    0.00943,
    -0.00461,
    0.00925,
    0.00163,
    0.00191,
    -0.01445,
    0.0064,
    0.00814,
    -0.00533,
    0.01005,
    0.00161,
    0.00201,
    -0.01069
   ],
   [
    0.00358,
    0.00078,
    -0.00977,
    0.01071,
    -0.01312,
    -0.00561,
    -0.00299,
    0.00293,
    0.00056,
    -0.00578,
    0.0104,
    -0.01098,
    -0.00619,
    -0.00519
   ],
   [
    0.00313,
    0.01357,
    -0.00549,
    -0.00299,
    0.02228,
    -0.00993,
    -0.006,
    0.00552,
    0.01099,
    -0.0084,
    -0.00667,
    0.01307,
    -0.01228,
    -0.0086
   ],
   [
    0.00085,
    -0.0019,
    0.00829,
    -0.00985,
    0.00017,
    0.01007,
    0.00437,
    6e-05,
    -0.00214,
    0.01147,
    -0.00811,
    0.00043,
    0.01137,
    0.00549
   ],
   [
    0.01985,
    0.01636,
    -0.00197,
    0.00778,
    -0.00568,
    -0.00412,
    0.00333,
    0.01903,
    0.01254,
    0.00091,
    0.00732,
    -0.00524,
    -0.00321,
    0.00319
   ],
   [
    0.00141,
    -0.00331,
    0.00039,
    0.01126,
    -0.00863,
    -0.01169,
    -0.0011,
    0.00018,
    -0.0008,
    -0.00094,
    0.01206,
    -0.00868,
    -0.01085,
    -0.00123
   ],
   [
    0.00317,
    0.02452,
    -0.01873,
    0.00051,
    0.00595,
    0.00024,
    -0.00993,
    0.00242,
    0.01659,
    -0.0167,
    -0.00148,
    0.00654,
    -0.00081,
    -0.00983
   ],
   [
    -0.00339,
    0.00761,
    0.00014,
    -0.00374,
    0.00335,
    -0.00119,
    0.01323,
    -0.00289,
    0.00944,
    -0.00179,
    -0.00253,
    0.00377,
    -0.002,
    0.0097
   ],
   [
    -0.00143,
    0.0118,
    -0.00788,
    -0.01836,
    -0.00556,
    0.01353,
    0.00862,
    -0.00303,
    0.01722,
    -0.00948,
    -0.0183,
    -0.00435,
    0.00974,
    0.00392
   ],
   [
    -0.0206,
    -0.0058,
    0.00637,
    0.00857,
    -0.00136,
    -0.00685,
    -0.00443,
    -0.01736,
    -0.00757,
    -0.00179,
    0.00742,
    -0.00135,
    -0.00875,
    -0.00631
   ],
   [
    -0.00568,
    0.00083,
    -0.00364,
    0.00536,
    0.0169,
    -0.01027,
    -0.01125,
    -0.00414,
    0.00155,
    -0.00912,
    0.00582,
    0.01527,
    -0.01028,
    -0.01194
   ],
   [
    -0.00395,
    0.02013,
    -0.01236,
    -0.01347,
    0.00869,
    0.00446,
    -0.01354,
    -0.0041,
    0.0163,
    -0.00574,
    -0.01569,
    0.00709,
    0.00702,
    -0.01007
   ],
   [
    -0.00271,
    0.0019,
    -0.00288,
    -0.00415,
    0.00526,
    -0.0009,
    0.00062,
    0.00015,
    0.00114,
    -0.00453,
    -0.00428,
    0.0036,
    -0.00028,
    0.00064
   ],
   [
    -0.01494,
    0.0061,
    0.00121,
    0.01094,
    0.00316,
    -0.00255,
    -0.01631,
    -0.01709,
    0.00524,
    0.00304,
    0.01149,
    0.00234,
    -0.00388,
    -0.01408
   ],
   [
    -0.00388,
    -0.00166,
    -0.00782,
    -0.00144,
    -0.0033,
    0.01312,
    -0.00128,
    -0.00365,
    -0.00268,
    -0.00524,
    -0.0025,
    -0.00444,
    0.01116,
    -0.00028
   ],
   [
    -0.00119,
    0.0005,
    -0.00061,
    0.00318,
    0.00086,
    -0.00071,
    -0.01193,
    0.0009,
    -0.00253,
    -0.00343,
    0.00026,
    0.00032,
    0.00082,
    -0.00951
   ],
   [
    0.01213,
    0.00235,
    0.0045,
    -0.00211,
    0.00556,
    -0.00473,
    0.01004,
    0.01182,
    0.00347,
    0.00531,
    -0.00247,
    0.00446,
    -0.00268,
    0.00873
   ],
   [
    -0.01359,
    -0.01205,
    -0.0011,
    -0.02435,
    -0.01999,
    0.03445,
    0.02754,
    -0.01432,
    -0.01061,
    0.00439,
    -0.01956,
    -0.01129,
    0.03576,
    0.02838
   ],
   [
    0.00048,
    -0.00726,
    -0.00198,
    0.00983,
    -0.00189,
    -0.00167,
    -0.00793,
    0.00085,
    -0.00576,
    0.0022,
    0.00844,
    -0.00053,
    -0.00068,
    -0.00695
   ],
   [
    0.00585,
    -0.01139,
    -0.0013,
    -0.00627,
    0.00351,
    0.00209,
    0.00167,
    0.00342,
    -0.00704,
    -0.0011,
    -0.00328,
    0.00271,
    0.00046,
    -0.00048
   ],
   [
    -0.0011,
    0.00264,
    0.00753,
    0.00732,
    0.00779,
    -0.00454,
    -0.01659,
    -0.00212,
    0.00405,
    0.00308,
    0.00844,
    0.00828,
    -0.00436,
    -0.01236
   ],
   [
    0.00431,
    0.00992,
    -0.0007,
    -0.00727,
    0.01102,
    0.00231,
    0.00029,
    0.00329,
    0.00863,
    -0.00089,
    -0.00584,
    0.009,
    0.00238,
    -0.00029
   ],
   [
    -0.00304,
    -0.0038,
    -0.0058,
    0.00891,
    0.00554,
    -0.00224,
    -0.00489,
    -0.00103,
    -0.0059,
    -0.00444,
    0.00759,
    0.00539,
    0.00028,
    -0.00132
   ],
   [
    0.0009,
    0.00048,
    0.00373,
    -0.00907,
    0.00426,
    0.00041,
    -0.00483,
    0.00061,
    -0.00099,
    0.00328,
    -0.00605,
    0.00513,
    0.00114,
    -0.0033
   ],
   [
    -0.00347,
    -0.0079,
    -0.0042,
    0.00085,
    -0.00177,
    -0.00016,
    0.00422,
    -0.00349,
    -0.00431,
    -0.00525,
    0.00435,
    -0.00086,
    0.00013,
    0.00176
   ],
   [
    0.00457,
    -0.00417,
    -0.00452,
    0.00626,
    -0.00755,
    -0.01464,
    -0.00353,
    0.00083,
    -0.00106,
    -0.00267,
    0.0071,
    -0.0063,
    -0.01361,
    -0.00329
   ],
   [
    0.00135,
    -0.00195,
    -0.00554,
    0.00418,
    0.00163,
    -0.0051,
    -0.00233,
    -0.00205,
    -0.00185,
    -0.00485,
    0.00365,
    -2e-05,
    -0.00663,
    -0.00384
   ],
   [
    0.02275,
    0.01547,
    -0.0078,
    -0.00719,
    0.01243,
    -0.0018,
    0.01004,
    0.01876,
    0.01555,
    0.00145,
    -0.00631,
    0.01086,
    -0.00194,
    0.00649
   ],
   [
    -0.02527,
    -0.01565,
    0.00252,
    0.01593,
    -0.00514,
    0.00225,
    -0.0096,
    -0.02084,
    -0.01858,
    0.00189,
    0.01359,
    -0.00401,
    0.00027,
    -0.00858
   ],
   [
    -0.00966,
    -0.00394,
    0.00573,
    -0.00679,
    0.00522,
    -0.00017,
    0.00023,
    -0.0111,
    -0.0038,
    0.00543,
    -0.00384,
    0.00385,
    0.00192,
    0.0016
   ],
   [
    -0.00635,
    -4e-05,
    -0.00192,
    -0.01034,
    0.003,
    0.00933,
    0.00946,
    -0.00284,
    -0.00609,
    -0.00364,
    -0.01403,
    0.00132,
    0.00659,
    0.00833
   ],
   [
    0.00174,
    -0.00736,
    0.00453,
    0.01789,
    -0.0127,
    -0.0014,
    0.00215,
    0.00185,
    -0.00665,
    0.00471,
    0.0184,
    -0.00906,
    -0.00154,
    0.00031
   ],
   [
    0.0135,
    0.01636,
    0.00162,
    0.00336,
    0.00169,
    0.00222,
    -0.01014,
    0.01113,
    0.01143,
    0.00021,
    0.00251,
    0.00115,
    0.00217,
    -0.00945
   ],
   [
    -0.00757,
    -0.01668,
    -0.01363,
    0.00683,
    -0.01621,
    -0.00061,
    0.01373,
    -0.00937,
    -0.01562,
    -0.00911,
    0.00518,
    -0.01499,
    -0.00361,
    0.00878
   ],
   [
    -0.03005,
    -0.01652,
    0.01135,
    0.01226,
    -0.0037,
    0.0048,
    -0.0056,
    -0.02729,
    -0.01728,
    0.01033,
    0.01036,
    -0.00302,
    0.00533,
    -0.00458
   ],
   [
    -0.00989,
    -0.00962,
    0.00627,
    0.00713,
    -0.00843,
    0.01489,
    -0.00804,
    -0.00681,
    -0.00488,
    0.00632,
    0.00972,
    -0.00492,
    0.01396,
    -0.00577
   ],
   [
    -0.024,
    -0.00803,
    -0.00402,
    -0.0078,
    0.00128,
    -0.0092,
    -0.00111,
    -0.01762,
    -0.01008,
    -0.00994,
    -0.00785,
    -0.00119,
    -0.0114,
    -0.00518
   ],
   [
    0.00961,
    -0.003,
    0.00028,
    0.0058,
    -0.00951,
    0.00483,
    0.01025,
    0.00955,
    -0.0006,
    0.00208,
    0.00474,
    -0.0069,
    0.00546,
    0.00861
   ],
   [
    0.00761,
    0.00907,
    -0.00031,
    0.0004,
    -0.00259,
    0.01383,
    -0.0049,
    0.00634,
    0.0081,
    0.00345,
    0.00098,
    -0.00333,
    0.01071,
    -0.00567
   ],
   [
    -0.01691,
    -0.00818,
    -0.00278,
    0.00503,
    0.00642,
    0.00067,
    -0.00175,
    -0.0127,
    -0.00801,
    -0.00632,
    0.00691,
    0.00644,
    0.00035,
    -0.00357
   ],
   [
    -0.0024,
    -0.01652,
    -0.00715,
    0.00786,
    -0.01746,
    0.00524,
    0.00861,
    0.00024,
    -0.00856,
    -0.00494,
    0.00861,
    -0.01187,
    0.00743,
    0.00912
   ],
   [
    -0.01813,
    -0.00282,
    -0.00326,
    -0.00433,
    0.00068,
    0.00211,
    -0.00479,
    -0.01837,
    -0.00158,
    -0.00204,
    -0.00284,
    0.00018,
    0.00188,
    -0.00357
   ],
   [
    7e-05,
    -0.01696,
    -0.01013,
    -0.00589,
    -0.00445,
    0.01028,
    0.00567,
    -0.004,
    -0.01502,
    -0.0034,
    -0.00627,
    -0.00287,
    0.00808,
    0.0037
   ],
   [
    -0.00682,
    0.01246,
    -0.00756,
    0.00247,
    0.00638,
    -0.00143,
    -0.00264,
    -0.00645,
    0.00903,
    -0.0033,
    0.00084,
    0.00423,
    -0.00119,
    -0.00263
   ],
   [
    0.00847,
    -0.00385,
    0.00229,
    0.00769,
    0.00068,
    -0.00708,
    -0.00182,
    0.00415,
    -0.0013,
    0.00125,
    0.00876,
    0.00233,
    -0.00707,
    -0.0022
   ],
   [
    -0.00613,
    -0.00016,
    -0.00415,
    0.00128,
    0.00264,
    -0.00154,
    -0.00032,
    -0.00573,
    0.00011,
    -0.00263,
    0.00062,
    0.00267,
    -3e-05,
    0.00032
   ],
   [
    -0.00263,
    0.00147,
    -0.00548,
    -0.00065,
    -0.00306,
    0.00532,
    0.00047,
    -0.0047,
    0.00386,
    0.00015,
    0.00192,
    -0.00241,
    0.00382,
    0.00069
   ],
   [
    0.01769,
    0.01697,
    -0.00659,
    0.00209,
    -0.00397,
    0.00816,
    -0.00192,
    0.01082,
    0.01516,
    -0.00328,
    0.00411,
    -0.00076,
    0.00561,
    -0.00159
   ],
   [
    0.01114,
    0.00924,
    0.0049,
    0.00247,
    0.00722,
    -0.00472,
    -0.01737,
    0.00931,
    0.00889,
    0.00655,
    0.00206,
    0.00494,
    -0.00511,
    -0.01515
   ],
   [
    0.01101,
    0.01968,
    -0.00774,
    -0.00486,
    0.00824,
    -0.00583,
    -0.00347,
    0.00907,
    0.01996,
    -0.00805,
    -0.0028,
    0.00717,
    -0.00623,
    -0.0042
   ],
   [
    -0.00849,
    -0.00227,
    0.00257,
    -0.00471,
    0.00081,
    -0.0048,
    0.00105,
    -0.01016,
    -0.00435,
    -0.00133,
    -0.0056,
    0.00087,
    -0.00543,
    0.0009
   ],
   [
    -0.00288,
    0.02419,
    -0.01103,
    -0.0031,
    0.02524,
    -0.00805,
    -0.00601,
    -0.00338,
    0.02348,
    -0.01176,
    -0.00483,
    0.0197,
    -0.00773,
    -0.00535
   ],
   [
    -0.00411,
    -0.01038,
    0.00599,
    -0.01998,
    0.00443,
    0.01192,
    0.00801,
    -0.00504,
    -0.00728,
    0.0094,
    -0.01492,
    0.0048,
    0.01247,
    0.00854
   ],
   [
    -0.00211,
    -0.024,
    -0.01305,
    0.00188,
    -0.00842,
    0.02117,
    0.01132,
    -0.00064,
    -0.01763,
    -0.00665,
    0.00173,
    -0.00654,
    0.02151,
    0.00879
   ],
   [
    0.00111,
    -0.00522,
    -0.00404,
    -0.00567,
    0.00799,
    -0.00416,
    -0.00124,
    -0.00275,
    -0.00579,
    -0.00016,
    -0.00315,
    0.00797,
    -0.0019,
    -0.0009
   ],
   [
    -0.00513,
    0.015,
    -0.00026,
    0.00564,
    0.01433,
    -0.00454,
    -0.03282,
    -0.00116,
    0.00683,
    -0.00099,
    0.00337,
    0.00857,
    -0.01286,
    -0.03101
   ],
   [
    -0.00719,
    -0.00105,
    0.00132,
    -0.00264,
    0.01051,
    0.00345,
    -0.00986,
    -0.00867,
    -0.00176,
    0.00018,
    -0.00087,
    0.00901,
    0.00297,
    -0.00774
   ],
   [
    0.02148,
    -0.01746,
    -0.01753,
    0.04005,
    0.00351,
    -0.03219,
    -0.02645,
    0.0167,
    -0.00587,
    -0.01459,
    0.03567,
    -0.00321,
    -0.03142,
    -0.02934
   ],
   [
    -0.01705,
    -0.01218,
    0.00756,
    -0.00266,
    -0.00483,
    0.00885,
    -0.01179,
    -0.01719,
    -0.01064,
    0.00415,
    -0.00132,
    -0.00484,
    0.00618,
    -0.00953
   ],
   [
    -0.00658,
    -0.00218,
    -0.00176,
    0.00378,
    -0.00309,
    -0.00349,
    -0.00397,
    -0.00991,
    -0.00262,
    0.00076,
    0.00502,
    -0.00306,
    -0.00357,
    -0.00306
   ],
   [
    -0.01609,
    -0.01562,
    0.00108,
    -0.00623,
    -0.00476,
    0.02067,
    0.00445,
    -0.01221,
    -0.01203,
    0.00168,
    -0.00322,
    -0.00169,
    0.0191,
    0.00541
   ],
   [
    -0.00225,
    0.00636,
    -0.00237,
    -0.00799,
    0.00829,
    -0.00041,
    -0.00555,
    -0.00195,
    0.00657,
    -0.00373,
    -0.00961,
    0.0063,
    -0.00199,
    -0.00759
   ],
   [
    -0.00925,
    -0.01915,
    -0.01019,
    0.0193,
    -0.00114,
    -0.00406,
    -0.00864,
    -0.01033,
    -0.0166,
    -0.00842,
    0.01756,
    -0.00274,
    -0.00178,
    -0.00717
   ],
   [
    -0.01034,
    0.01625,
    0.00011,
    0.00468,
    0.00376,
    0.00223,
    -0.00857,
    -0.00554,
    0.01335,
    -5e-05,
    0.00404,
    0.00317,
    -0.00083,
    -0.00806
   ],
   [
    -0.00658,
    0.01655,
    0.00614,
    0.00653,
    0.00238,
    0.00326,
    0.00837,
    -0.00387,
    0.01213,
    0.00841,
    0.00406,
    0.00138,
    0.00172,
    0.00663
   ],
   [
    -0.00959,
    -0.0174,
    0.00573,
    0.00895,
    -0.0018,
    -0.0039,
    0.00208,
    -0.00892,
    -0.01767,
    0.00263,
    0.00637,
    -0.00289,
    -0.00166,
    0.00222
   ],
   [
    0.01532,
    -0.01516,
    0.01999,
    0.0052,
    0.00203,
    -0.0047,
    0.01259,
    0.01412,
    -0.00956,
    0.01561,
    0.00736,
    0.00282,
    -0.00135,
    0.01318
   ],
   [
    -0.01532,
    -0.00731,
    0.00208,
    -0.00648,
    -0.00136,
    0.00396,
    -0.00257,
    -0.01302,
    -0.00863,
    -0.00073,
    -0.00807,
    -0.00185,
    0.00391,
    -0.00263
   ],
   [
    0.00658,
    0.02842,
    0.00429,
    0.00072,
    0.00329,
    0.01104,
    0.01071,
    0.00827,
    0.0259,
    0.00653,
    0.00489,
    0.00712,
    0.01196,
    0.01103
   ],
   [
    0.00775,
    0.01588,
    -0.00479,
    -0.008,
    -0.0092,
    0.00651,
    0.02018,
    0.007,
    0.01554,
    -0.0016,
    -0.00603,
    -0.00439,
    0.00928,
    0.02077
   ],
   [
    -0.02348,
    -0.01249,
    0.00307,
    -0.00124,
    0.00557,
    -0.00776,
    -0.0041,
    -0.01859,
    -0.00926,
    -0.00022,
    -0.00098,
    0.00423,
    -0.00727,
    -0.0037
   ],
   [
    -0.01332,
    -0.01553,
    -0.00161,
    0.00552,
    -0.01334,
    -0.00711,
    -0.0019,
    -0.01283,
    -0.01641,
    -0.00233,
    0.00301,
    -0.0143,
    -0.00744,
    -0.00176
   ],
   [
    -0.00928,
    -0.00033,
    0.00481,
    -0.00496,
    0.01455,
    -0.01136,
    -0.0083,
    -0.00859,
    0.00046,
    0.00714,
    -0.00365,
    0.01311,
    -0.00746,
    -0.00426
   ],
   [
    0.00317,
    -0.00207,
    -0.00075,
    0.0129,
    -0.00181,
    -0.01108,
    -0.01363,
    0.002,
    -0.00136,
    -0.00412,
    0.0121,
    -0.00091,
    -0.0102,
    -0.01334
   ],
   [
    -0.00893,
    0.0057,
    -0.00967,
    0.00887,
    0.00841,
    -0.00988,
    -0.00939,
    -0.0108,
    0.00092,
    -0.0105,
    0.00944,
    0.006,
    -0.01268,
    -0.0118
   ],
   [
    0.01028,
    0.01234,
    -0.00881,
    0.00762,
    0.01609,
    0.00045,
    -0.02789,
    0.00539,
    0.01164,
    -0.00124,
    0.00599,
    0.00922,
    -0.00031,
    -0.02261
   ],
   [
    -0.01483,
    0.00413,
    -0.00358,
    0.00378,
    0.01429,
    0.00314,
    0.00063,
    -0.01181,
    0.0031,
    0.00153,
    0.00669,
    0.01551,
    0.00556,
    0.00358
   ],
   [
    -0.00298,
    -0.00212,
    0.00937,
    0.00405,
    -0.00209,
    0.00145,
    0.00455,
    7e-05,
    6e-05,
    0.00888,
    0.00408,
    -0.00114,
    0.00287,
    0.0036
   ],
   [
    0.00963,
    -0.00568,
    0.01049,
    0.00242,
    -0.00446,
    -0.00092,
    0.00461,
    0.01112,
    0.00091,
    0.00613,
    0.00533,
    -0.00236,
    -0.00049,
    0.00274
   ],
   [
    0.00605,
    0.01143,
    0.00519,
    0.00111,
    0.00329,
    -0.0054,
    -0.00789,
    0.00567,
    0.00591,
    0.00122,
    0.00184,
    0.00365,
    -0.00476,
    -0.00591
   ],
   [
    0.00393,
    -0.00112,
    -0.00657,
    -0.01114,
    -0.00481,
    0.00702,
    -0.00257,
    0.00567,
    0.00297,
    -0.00335,
    -0.00984,
    -0.0029,
    0.00503,
    -0.00144
   ],
   [
    0.00884,
    0.00958,
    -0.00775,
    -0.00152,
    0.00635,
    0.00181,
    -0.00012,
    0.00865,
    0.00901,
    -0.00542,
    -0.00214,
    0.00477,
    0.00091,
    0.00017
   ],
   [
    -0.0005,
    0.00483,
    0.00325,
    -0.00847,
    0.00554,
    -0.00063,
    0.00124,
    0.00277,
    0.00077,
    0.00372,
    -0.00851,
    0.00351,
    -0.00037,
    0.00147
   ],
   [
    -0.00699,
    -0.00317,
    0.00083,
    -0.00118,
    0.00126,
    -0.01211,
    0.00646,
    -0.00499,
    -0.00436,
    -0.00027,
    -0.00146,
    0.00203,
    -0.00911,
    0.0056
   ],
   [
    -0.01439,
    -0.00098,
    -0.01035,
    3e-05,
    0.00165,
    -0.0008,
    -0.00178,
    -0.01091,
    0.00192,
    -0.00513,
    0.00108,
    0.00289,
    0.00168,
    -0.00064
   ],
   [
    -0.0003,
    0.00286,
    -0.01246,
    -0.00195,
    -0.00247,
    0.01284,
    -0.01083,
    -0.00092,
    0.00134,
    -0.00833,
    -0.00257,
    -0.00168,
    0.01181,
    -0.0095
   ],
   [
    0.00966,
    -0.00242,
    -0.00496,
    0.00153,
    -0.005,
    0.00059,
    -0.00199,
    0.00699,
    -0.00064,
    -0.00332,
    0.00197,
    -0.00451,
    -0.00052,
    -0.00251
   ],
   [
    0.00532,
    0.01558,
    0.00728,
    0.00882,
    -0.00316,
    -0.00389,
    -0.00618,
    0.0049,
    0.01071,
    0.00112,
    0.00645,
    -0.00344,
    -0.00423,
    -0.00805
   ],
   [
    -0.00297,
    -0.00721,
    0.00116,
    0.00342,
    -0.00362,
    -0.00377,
    -0.00071,
    -0.00479,
    -0.00697,
    0.00027,
    0.00423,
    -0.00333,
    -0.00252,
    -7e-05
   ],
   [
    -0.00235,
    0.01348,
    -0.00573,
    -0.00321,
    -0.00335,
    0.01979,
    -0.00142,
    -8e-05,
    0.01162,
    -0.00454,
    -0.00368,
    -0.00034,
    0.0206,
    0.00177
   ],
   [
    -0.00535,
    -0.02033,
    0.00345,
    0.00086,
    -0.00458,
    -0.00177,
    -0.00184,
    -0.00637,
    -0.01742,
    -0.00099,
    -0.00015,
    -0.00599,
    -0.00292,
    -0.00308
   ],
   [
    0.01733,
    0.00674,
    0.01051,
    0.00445,
    0.0032,
    -0.00049,
    -0.00807,
    0.01832,
    0.00786,
    0.00734,
    0.00469,
    0.0036,
    0.00298,
    -0.00415
   ],
   [
    -0.00168,
    -0.01292,
    -0.00814,
    0.01209,
    -0.00773,
    -0.01013,
    -4e-05,
    -0.00393,
    -0.01283,
    -0.00677,
    0.00972,
    -0.00744,
    -0.01097,
    -0.00035
   ],
   [
    0.00432,
    0.01222,
    -0.00296,
    -0.00784,
    0.01515,
    -0.00828,
    0.00031,
    0.00488,
    0.00987,
    -0.00045,
    -0.00705,
    0.01254,
    -0.00896,
    -0.00143
   ],
   [
    0.01708,
    -0.00912,
    0.00234,
    -0.01006,
    0.00244,
    -0.00402,
    0.00746,
    0.01477,
    -0.00533,
    0.00232,
    -0.0091,
    0.00432,
    -0.00064,
    0.00723
   ],
   [
    -0.02171,
    -0.02537,
    -0.01519,
    0.01448,
    -0.02396,
    0.01026,
    -0.00444,
    -0.01526,
    -0.01752,
    -0.01023,
    0.01361,
    -0.01827,
    0.01331,
    -0.00299
   ],
   [
    0.02842,
    -0.00123,
    0.01253,
    -0.01037,
    -0.00613,
    0.01362,
    0.02434,
    0.02943,
    -0.00052,
    0.0144,
    -0.00717,
    -0.00133,
    0.01629,
    0.0234
   ],
   [
    0.01329,
    0.01971,
    -0.00096,
    -0.01101,
    0.00066,
    0.00961,
    0.00656,
    0.01344,
    0.01636,
    -0.00476,
    -0.01283,
    5e-05,
    0.00696,
    0.00514
   ],
   [
    -0.00558,
    0.01629,
    -0.00213,
    -0.0172,
    0.01309,
    -0.00012,
    0.01913,
    -0.00557,
    0.01224,
    -0.0047,
    -0.01549,
    0.01524,
    0.00668,
    0.02009
   ],
   [
    0.00818,
    0.00575,
    0.00039,
    -0.00344,
    0.00844,
    0.00935,
    0.00846,
    0.00772,
    0.00109,
    -0.00053,
    -0.00338,
    0.00768,
    0.00757,
    0.00888
   ],
   [
    0.0026,
    -0.00143,
    -0.00327,
    -0.00426,
    0.00177,
    0.00128,
    0.00878,
    0.00186,
    0.00466,
    -0.00405,
    -0.00208,
    0.00341,
    0.00261,
    0.00799
   ],
   [
    -0.01427,
    0.01043,
    -0.00035,
    -0.0175,
    0.00624,
    0.01202,
    -0.00057,
    -0.01323,
    0.00336,
    -0.00153,
    -0.01436,
    0.00627,
    0.00897,
    -0.00197
   ],
   [
    0.0223,
    0.02405,
    0.00853,
    0.00847,
    0.00634,
    0.00087,
    -0.00469,
    0.02041,
    0.02376,
    0.00501,
    0.00917,
    0.0062,
    0.0029,
    -0.00253
   ],
   [
    0.04477,
    0.02123,
    0.00831,
    -0.00844,
    -0.00146,
    0.00983,
    0.00728,
    0.03408,
    0.02467,
    0.00694,
    -0.01026,
    -0.00372,
    0.00658,
    0.00465
   ],
   [
    0.01089,
    0.00074,
    -0.00917,
    0.00385,
    0.00392,
    -0.00388,
    -0.00704,
    0.01054,
    0.00119,
    -0.00554,
    -0.00155,
    0.0,
    -0.00613,
    -0.0091
   ],
   [
    -0.00643,
    0.0129,
    -0.0005,
    0.00381,
    -0.00493,
    0.00407,
    -0.00463,
    -0.00061,
    0.01278,
    0.0001,
    0.00378,
    -0.00089,
    0.00362,
    -0.00311
   ],
   [
    0.00952,
    -0.01649,
    -4e-05,
    0.0104,
    -0.00981,
    -0.00946,
    0.00145,
    0.00649,
    -0.01163,
    0.00055,
    0.00737,
    -0.00996,
    -0.00789,
    -0.00076
   ],
   [
    -0.00109,
    -0.01159,
    0.00232,
    0.00174,
    -0.00685,
    0.00224,
    -0.00214,
    -0.00207,
    -0.00999,
    -2e-05,
    0.00222,
    -0.00687,
    0.00044,
    -0.00329
   ],
   [
    -0.00387,
    -0.0142,
    0.00379,
    -0.00309,
    0.006,
    -0.00028,
    0.00154,
    -0.00247,
    -0.00827,
    0.00351,
    -0.0007,
    0.00522,
    0.00207,
    0.0029
   ],
   [
    -0.00392,
    -0.01117,
    -0.01752,
    0.00086,
    0.0089,
    -0.0255,
    -0.01083,
    -0.01046,
    -0.01551,
    -0.01656,
    0.00455,
    0.01025,
    -0.01748,
    -0.00587
   ],
   [
    -0.00485,
    0.00086,
    -0.00363,
    0.00524,
    0.00176,
    -0.00249,
    -0.00841,
    -0.00088,
    0.00154,
    -0.00214,
    0.00464,
    0.00166,
    -0.0055,
    -0.00939
   ],
   [
    0.01168,
    0.01901,
    -0.00431,
    -0.00155,
    -0.00786,
    0.01246,
    -0.01507,
    0.0074,
    0.01154,
    -0.00226,
    -0.00116,
    -0.00677,
    0.00815,
    -0.01265
   ],
   [
    0.00107,
    -0.00022,
    -0.00256,
    0.00654,
    -0.00413,
    0.00057,
    -0.00519,
    0.00107,
    0.0012,
    -0.00469,
    0.00671,
    -0.0035,
    0.00183,
    -0.00555
   ],
   [
    -0.0026,
    0.01202,
    0.0031,
    0.00561,
    0.00943,
    -0.00655,
    -0.00997,
    -0.00191,
    0.00601,
    -0.00025,
    0.00707,
    0.00829,
    -0.00448,
    -0.00656
   ],
   [
    -0.00646,
    -0.00818,
    -0.00201,
    0.00415,
    7e-05,
    0.00202,
    -0.00247,
    -0.0077,
    -0.00824,
    -0.00478,
    0.00272,
    0.00026,
    0.00049,
    -0.00245
   ],
   [
    0.00037,
    -0.02005,
    -0.01043,
    0.00133,
    -0.00212,
    -0.00958,
    0.00756,
    -0.00339,
    -0.01713,
    -0.00878,
    0.00312,
    -0.00138,
    -0.00808,
    0.00698
   ],
   [
    -0.00837,
    0.00986,
    -0.00587,
    0.00069,
    -0.00033,
    -0.00999,
    -0.00416,
    -0.00653,
    0.0027,
    -0.00583,
    0.00041,
    -5e-05,
    -0.00786,
    -0.00131
   ],
   [
    0.00167,
    0.00152,
    -0.00552,
    0.00801,
    0.00781,
    -0.00309,
    -0.0013,
    0.00228,
    0.003,
    -0.00623,
    0.00696,
    0.0069,
    -0.00352,
    -0.00102
   ],
   [
    -0.00631,
    -0.01825,
    -0.00287,
    0.00015,
    -0.00558,
    -0.00122,
    0.00352,
    -0.00662,
    -0.01876,
    -0.00202,
    -0.0021,
    -0.00753,
    -0.00257,
    0.00233
   ],
   [
    -0.00607,
    -0.00364,
    -0.00304,
    0.00049,
    0.00914,
    -0.01114,
    0.00518,
    -0.00542,
    -0.00083,
    -0.00222,
    0.00188,
    0.00772,
    -0.01052,
    0.00194
   ],
   [
    -0.01474,
    -0.01981,
    -0.00256,
    -0.00868,
    -0.00482,
    0.00597,
    0.00526,
    -0.01455,
    -0.01578,
    -0.00343,
    -0.00777,
    -0.00344,
    0.00528,
    0.00486
   ],
   [
    -0.00664,
    -0.01533,
    -0.00907,
    0.01183,
    -0.00876,
    -0.00685,
    -0.00616,
    -0.00597,
    -0.01389,
    -0.00506,
    0.0121,
    -0.007,
    -0.00664,
    -0.00531
   ],
   [
    -0.01123,
    -0.00141,
    0.01631,
    0.01872,
    0.01062,
    -0.01227,
    -0.00748,
    -0.00549,
    -0.002,
    0.00324,
    0.0155,
    0.01012,
    -0.01125,
    -0.00929
   ],
   [
    -0.00505,
    0.00579,
    0.00924,
    -0.00686,
    0.00591,
    -0.00959,
    -0.00444,
    -0.00453,
    0.00016,
    0.00367,
    -0.00775,
    0.00506,
    -0.00888,
    -0.00504
   ],
   [
    -0.00093,
    0.00358,
    -0.01209,
    0.01494,
    -0.01476,
    0.01419,
    -0.00885,
    -0.00227,
    0.00436,
    -0.00771,
    0.01176,
    -0.0125,
    0.00677,
    -0.00914
   ],
   [
    -0.00273,
    0.01452,
    0.02298,
    -0.01626,
    -0.00326,
    0.01952,
    0.00667,
    -0.00148,
    0.01133,
    0.01409,
    -0.01196,
    0.00192,
    0.01515,
    0.00265
   ],
   [
    0.00678,
    0.02003,
    -0.00434,
    0.00569,
    -0.00211,
    -0.00053,
    -0.00306,
    0.00501,
    0.01594,
    -0.00413,
    0.00562,
    -0.00147,
    -0.00221,
    -0.00364
   ],
   [
    0.01284,
    -0.00401,
    -0.00107,
    0.00243,
    0.00174,
    0.00367,
    0.00804,
    0.00983,
    0.00217,
    -0.00099,
    0.00583,
    0.00509,
    0.00848,
    0.01032
   ],
   [
    0.01614,
    0.00988,
    -0.01114,
    0.00567,
    0.00081,
    -0.00538,
    0.00739,
    0.0111,
    0.01042,
    -0.00891,
    0.00579,
    0.00136,
    -0.00422,
    0.00567
   ],
   [
    -0.02157,
    0.00393,
    -0.01161,
    2e-05,
    0.00803,
    -0.00558,
    1e-05,
    -0.01792,
    0.00313,
    -0.01357,
    -0.00176,
    0.00592,
    -0.00772,
    -0.00256
   ],
   [
    0.00617,
    0.00339,
    6e-05,
    -0.00166,
    0.00217,
    -0.00504,
    0.00996,
    0.00499,
    0.00264,
    0.00292,
    -0.0023,
    0.00134,
    -0.00396,
    0.0077
   ],
   [
    -0.01061,
    -0.00051,
    -0.00671,
    -0.00442,
    0.00342,
    0.00167,
    -0.00446,
    -0.00837,
    -0.00238,
    -0.0029,
    -0.0059,
    0.00159,
    0.00215,
    -0.00197
   ],
   [
    0.00872,
    0.00264,
    0.00298,
    -0.00041,
    0.00143,
    0.00594,
    0.00606,
    0.00868,
    0.00354,
    0.0002,
    -0.00105,
    0.00033,
    0.0047,
    0.0043
   ],
   [
    -0.00683,
    -0.00012,
    -0.00381,
    -0.01106,
    -0.00104,
    0.00893,
    0.00189,
    -0.00336,
    0.00089,
    -0.00035,
    -0.01151,
    -0.00023,
    0.00607,
    -0.00025
   ],
   [
    0.00629,
    0.00119,
    -0.00488,
    0.00427,
    0.00284,
    -0.00038,
    -0.01127,
    0.00313,
    0.00166,
    -0.0023,
    0.00451,
    0.00079,
    -0.00301,
    -0.01152
   ],
   [
    -0.01244,
    -0.00857,
    0.00094,
    -0.00197,
    0.00426,
    0.00739,
    0.00055,
    -0.00817,
    -0.00366,
    -0.00166,
    -0.00245,
    0.00515,
    0.00851,
    0.00267
   ],
   [
    -0.01744,
    -0.0214,
    -0.00115,
    -0.00666,
    -0.00449,
    0.00757,
    0.00205,
    -0.01489,
    -0.02132,
    0.00084,
    -0.00311,
    -0.00024,
    0.01197,
    0.00622
   ],
   [
    0.0044,
    -0.00504,
    -0.00158,
    -0.00443,
    -0.01881,
    0.01743,
    0.02219,
    0.00565,
    0.00689,
    0.00597,
    -0.00305,
    -0.01512,
    0.01562,
    0.01885
   ],
   [
    -0.01903,
    -0.0059,
    -0.00275,
    -0.00533,
    0.00913,
    0.00192,
    -0.00294,
    -0.01483,
    -0.00681,
    0.00016,
    -0.00707,
    0.00657,
    0.00368,
    -0.00189
   ],
   [
    0.00527,
    0.01756,
    -0.0062,
    0.00554,
    -0.00644,
    0.001,
    -0.01985,
    0.00663,
    0.01009,
    -0.00823,
    0.00547,
    -0.00594,
    0.00149,
    -0.0156
   ],
   [
    0.00546,
    -8e-05,
    0.00959,
    -0.00928,
    -0.01186,
    0.0206,
    0.01568,
    0.00469,
    0.00355,
    0.0106,
    -0.00828,
    -0.00812,
    0.0184,
    0.01305
   ],
   [
    0.01014,
    0.00424,
    -0.00081,
    0.00514,
    -0.00489,
    -0.00281,
    -0.00753,
    0.00803,
    -0.00117,
    -7e-05,
    0.00327,
    -0.00615,
    -0.00231,
    -0.00523
   ],
   [
    0.03099,
    -0.03315,
    -0.02774,
    0.02795,
    -0.02582,
    -0.00851,
    -0.01205,
    0.02427,
    -0.03148,
    -0.0165,
    0.0233,
    -0.02544,
    -0.00791,
    -0.01196
   ],
   [
    -0.01826,
    -0.04465,
    0.00673,
    -0.00359,
    -0.03015,
    0.00142,
    0.01663,
    -0.01483,
    -0.03711,
    -0.00276,
    -0.0066,
    -0.02704,
    -0.0004,
    0.01026
   ],
   [
    0.00931,
    0.00819,
    0.00489,
    0.00441,
    -0.01055,
    -0.00762,
    0.00506,
    0.00667,
    0.002,
    -0.00187,
    0.00503,
    -0.00758,
    -0.00692,
    0.00294
   ],
   [
    0.01123,
    -0.01015,
    -0.00384,
    0.00721,
    -0.00155,
    -0.0039,
    -0.00089,
    0.00901,
    -0.00657,
    -0.00285,
    0.00908,
    -0.00131,
    -0.00479,
    -0.00156
   ],
   [
    0.01102,
    -0.00526,
    -0.00969,
    0.00601,
    -0.01075,
    -0.005,
    -0.00024,
    0.00736,
    -0.00619,
    -0.00791,
    0.0048,
    -0.01073,
    -0.0057,
    0.00012
   ],
   [
    0.00807,
    -0.01934,
    0.0027,
    0.01073,
    -0.00946,
    0.00567,
    0.00065,
    0.00673,
    -0.01,
    0.00119,
    0.01231,
    -0.00668,
    0.00924,
    0.00202
   ],
   [
    -0.00147,
    0.00793,
    -0.01171,
    0.01032,
    -0.00304,
    -0.01575,
    -0.00291,
    0.00062,
    0.00444,
    -0.00598,
    0.0086,
    -0.00381,
    -0.01259,
    -0.00125
   ],
   [
    0.01167,
    0.01697,
    0.01266,
    0.00742,
    -0.00018,
    -0.00571,
    0.00079,
    0.01292,
    0.01581,
    0.01188,
    0.00612,
    -0.00185,
    -0.00427,
    -2e-05
   ],
   [
    -0.002,
    -0.00249,
    0.0036,
    0.00082,
    -0.00509,
    0.00746,
    0.00812,
    -0.00117,
    -0.00057,
    0.00035,
    0.00027,
    -0.00361,
    0.00545,
    0.00741
   ],
   [
    -0.00238,
    -0.00567,
    0.00922,
    0.00613,
    -0.00157,
    0.00039,
    -0.0023,
    -0.00208,
    -0.00671,
    0.00401,
    0.00547,
    -0.00168,
    0.00093,
    -0.00154
   ],
   [
    -0.02053,
    -0.01184,
    0.00217,
    -0.00024,
    0.00011,
    0.00479,
    -7e-05,
    -0.01674,
    -0.01278,
    0.0042,
    -0.00015,
    -0.00041,
    0.00532,
    0.0005
   ],
   [
    -0.02429,
    -0.00525,
    0.00427,
    0.00746,
    -0.01419,
    -0.00611,
    -0.01088,
    -0.0198,
    -0.01003,
    -0.00075,
    0.00525,
    -0.01249,
    -0.00549,
    -0.00897
   ],
   [
    0.01815,
    0.00491,
    0.00088,
    -0.00344,
    -0.00025,
    0.00625,
    0.00614,
    0.01501,
    0.00599,
    0.00139,
    -0.00234,
    0.00114,
    0.00622,
    0.00607
   ],
   [
    -0.00962,
    0.00351,
    -0.01305,
    -0.00031,
    0.01039,
    -0.00578,
    8e-05,
    -0.00891,
    0.00238,
    -0.01206,
    -0.00056,
    0.00868,
    -0.00654,
    -0.0013
   ],
   [
    -0.01005,
    -0.01326,
    -0.00335,
    0.00153,
    -0.01076,
    -0.0036,
    0.0101,
    -0.00732,
    -0.00847,
    -0.0034,
    0.00322,
    -0.01024,
    -0.00346,
    0.00726
   ],
   [
    0.01324,
    0.00033,
    0.0015,
    0.00811,
    -0.005,
    -0.00021,
    0.00811,
    0.01074,
    0.00084,
    0.00131,
    0.00595,
    -0.00631,
    -0.00078,
    0.00584
   ],
   [
    0.00874,
    0.00276,
    -0.00473,
    0.00198,
    -0.00664,
    -0.00898,
    0.00691,
    0.00787,
    0.00205,
    -0.0047,
    0.00132,
    -0.00561,
    -0.00528,
    0.00757
   ],
   [
    -0.01076,
    -0.00753,
    0.00097,
    -0.00104,
    0.00105,
    -0.00768,
    -0.00156,
    -0.01094,
    -0.00566,
    -0.00703,
    0.0008,
    0.00258,
    -0.00616,
    -0.0007
   ],
   [
    -0.00775,
    0.00152,
    -0.00268,
    0.00293,
    0.00739,
    -0.00435,
    -0.00198,
    -0.00743,
    -0.00153,
    -0.00099,
    0.00174,
    0.00614,
    -0.00387,
    -0.00135
   ],
   [
    -0.00683,
    -0.01573,
    -0.01553,
    0.01126,
    -0.01882,
    0.01032,
    0.01498,
    -0.00159,
    -0.00573,
    -0.00985,
    0.01236,
    -0.01316,
    0.01173,
    0.01509
   ],
   [
    0.00947,
    0.00886,
    0.00432,
    0.00721,
    0.00947,
    -0.00617,
    0.00058,
    0.00512,
    0.00523,
    0.00388,
    0.00736,
    0.00681,
    -0.00437,
    0.00032
   ],
   [
    0.00073,
    -0.00507,
    -0.01201,
    -0.00258,
    -0.00077,
    -0.00104,
    2e-05,
    0.0004,
    -0.00313,
    -0.01063,
    -0.00308,
    -0.00373,
    -0.00369,
    -0.00238
   ],
   [
    -0.00828,
    -0.01458,
    -0.01787,
    0.00683,
    0.00068,
    -0.00022,
    -0.01144,
    -0.00835,
    -0.01332,
    -0.01261,
    0.00826,
    0.00264,
    0.0011,
    -0.0075
   ],
   [
    0.03121,
    0.01948,
    -0.00763,
    0.00387,
    -0.00528,
    -0.00792,
    0.00859,
    0.02672,
    0.0126,
    -0.0089,
    0.00328,
    -0.0063,
    -0.00717,
    0.00824
   ],
   [
    0.00155,
    -0.00676,
    0.01109,
    -0.02137,
    0.00038,
    0.00385,
    0.01212,
    0.00238,
    -0.0026,
    0.00841,
    -0.01857,
    -0.00123,
    0.00077,
    0.00879
   ],
   [
    -0.00154,
    -0.02311,
    0.00522,
    0.00776,
    -0.01461,
    0.00225,
    0.00151,
    -0.0033,
    -0.01268,
    0.00329,
    0.00679,
    -0.01152,
    0.00631,
    0.00274
   ],
   [
    -0.00145,
    -0.01167,
    0.01621,
    0.00186,
    -0.01533,
    0.01008,
    -0.00024,
    -0.00089,
    -0.00529,
    0.01455,
    0.00377,
    -0.01322,
    0.0098,
    0.00098
   ],
   [
    0.00895,
    0.02746,
    0.00168,
    -0.00239,
    0.00385,
    -0.00099,
    -0.00131,
    0.00729,
    0.02033,
    0.00315,
    -0.00327,
    0.00337,
    -0.00275,
    -0.00089
   ],
   [
    -0.00058,
    -0.01381,
    0.00342,
    -0.00384,
    0.00222,
    -0.00567,
    0.00396,
    -0.00115,
    -0.00981,
    0.00166,
    -0.00753,
    -0.00112,
    -0.00659,
    0.00189
   ],
   [
    0.00123,
    0.00039,
    0.00673,
    -0.00454,
    -0.00378,
    0.00128,
    0.00755,
    0.00325,
    0.0003,
    0.00405,
    -0.00304,
    -0.00136,
    0.00292,
    0.0076
   ],
   [
    0.00211,
    -0.01087,
    -0.00084,
    -0.01762,
    0.00262,
    0.00683,
    0.00561,
    -0.00123,
    -0.0069,
    0.00094,
    -0.01639,
    -0.00028,
    0.00467,
    0.00119
   ],
   [
    -0.02458,
    0.00229,
    -0.00852,
    0.00453,
    0.00037,
    -0.01377,
    -0.00801,
    -0.02258,
    -0.00294,
    -0.00815,
    0.00683,
    0.00296,
    -0.01149,
    -0.00862
   ],
   [
    -0.01703,
    -0.00238,
    -0.00874,
    -0.0045,
    0.00564,
    -0.00301,
    0.00461,
    -0.01648,
    -0.00251,
    -0.00555,
    -0.00299,
    0.00556,
    -0.00118,
    0.0026
   ],
   [
    0.00071,
    -0.0003,
    0.00879,
    -0.00269,
    0.00751,
    0.00328,
    -0.00481,
    0.00097,
    0.00154,
    0.00619,
    -0.00249,
    0.00743,
    0.00544,
    -0.00193
   ],
   [
    0.00231,
    0.01351,
    0.00427,
    -0.00347,
    0.00602,
    0.00033,
    0.00401,
    0.00335,
    0.01437,
    0.00689,
    -0.00235,
    0.00605,
    0.00081,
    0.00369
   ],
   [
    0.00602,
    -0.00056,
    0.00825,
    -0.00384,
    -0.00073,
    0.01074,
    -0.00169,
    0.00654,
    -0.00145,
    0.00864,
    -0.0048,
    -0.00112,
    0.00914,
    -0.00111
   ],
   [
    0.00308,
    0.00748,
    -0.00278,
    -0.00124,
    0.00062,
    -0.00021,
    -0.00497,
    0.00327,
    0.00582,
    0.00295,
    -0.00011,
    0.00108,
    -0.0005,
    -0.00413
   ],
   [
    -0.00325,
    -0.0025,
    0.01676,
    -0.00331,
    0.00297,
    -0.01335,
    -0.0008,
    -0.00313,
    -0.00194,
    0.00988,
    -0.00305,
    0.00205,
    -0.01371,
    0.00016
   ],
   [
    0.01149,
    -0.01376,
    -0.00217,
    0.0046,
    -0.00502,
    0.00844,
    0.00418,
    0.00806,
    -0.00715,
    -0.00256,
    0.00622,
    -0.00306,
    0.00742,
    0.00408
   ],
   [
    -0.00773,
    0.00418,
    0.01702,
    -0.00474,
    -0.00137,
    0.01008,
    0.00511,
    -0.00571,
    0.00698,
    0.01416,
    -0.003,
    -2e-05,
    0.00964,
    0.00476
   ],
   [
    0.01565,
    0.00728,
    0.00672,
    -0.00329,
    0.00971,
    -0.00364,
    -0.00453,
    0.01374,
    0.0078,
    0.00971,
    -0.00392,
    0.00752,
    -0.00056,
    -0.00457
   ],
   [
    0.00413,
    0.01842,
    0.01237,
    0.00061,
    -0.00464,
    -0.006,
    -0.00371,
    0.00521,
    0.01457,
    0.00933,
    0.00362,
    -0.00267,
    -0.004,
    -7e-05
   ],
   [
    -0.00589,
    -0.00674,
    0.00075,
    0.00028,
    0.00756,
    -0.00879,
    0.00665,
    -0.00496,
    -0.00462,
    0.00331,
    0.00226,
    0.00827,
    -0.0064,
    0.00512
   ],
   [
    -0.00431,
    -0.0077,
    -0.00755,
    0.0133,
    -0.00136,
    -0.00661,
    -0.00966,
    -0.00166,
    -0.00949,
    -0.0097,
    0.00989,
    -0.00402,
    -0.00987,
    -0.00953
   ],
   [
    0.01034,
    -0.00576,
    0.00284,
    -0.0018,
    0.00267,
    -0.0112,
    -0.00246,
    0.00942,
    -0.00592,
    -0.00097,
    -0.00498,
    -8e-05,
    -0.01206,
    -0.00157
   ],
   [
    0.00028,
    0.0052,
    0.00059,
    -0.00118,
    0.0043,
    0.00029,
    0.00349,
    0.0015,
    0.00172,
    0.00042,
    -0.00302,
    0.00328,
    0.00194,
    0.00628
   ],
   [
    0.0146,
    -0.01807,
    -0.01254,
    -0.00011,
    -0.01363,
    0.00728,
    0.01276,
    0.01092,
    -0.00953,
    -0.00589,
    -0.00095,
    -0.01255,
    0.00492,
    0.00871
   ],
   [
    -0.00196,
    0.00289,
    -0.00459,
    -0.00263,
    -0.00679,
    0.0059,
    0.01258,
    2e-05,
    0.0006,
    -0.00363,
    -0.00154,
    -0.00332,
    0.00592,
    0.01085
   ],
   [
    -0.00524,
    0.01238,
    -0.00631,
    0.00611,
    0.0065,
    0.00048,
    -0.00504,
    -0.00364,
    0.00986,
    -0.00363,
    0.00437,
    0.00557,
    0.00081,
    -0.00429
   ],
   [
    -0.00537,
    0.00736,
    -0.00089,
    -0.00846,
    0.00011,
    0.00405,
    -0.01063,
    -0.00138,
    -0.00034,
    -0.00097,
    -0.01064,
    0.00072,
    0.00542,
    -0.00612
   ],
   [
    0.00699,
    0.01862,
    0.00301,
    -0.0067,
    -0.00102,
    0.00707,
    0.00962,
    0.00508,
    0.01755,
    0.00621,
    -0.00392,
    6e-05,
    0.00847,
    0.00905
   ],
   [
    -0.0028,
    -0.00944,
    -0.00169,
    -0.0043,
    -8e-05,
    -0.00089,
    0.00706,
    -0.00302,
    -0.00696,
    -0.0013,
    -0.00289,
    0.00039,
    -0.00157,
    0.00505
   ],
   [
    -1e-05,
    0.02296,
    0.00146,
    -0.00192,
    0.00312,
    -0.00381,
    -0.00023,
    1e-05,
    0.02012,
    0.00225,
    -0.00183,
    0.00214,
    -0.00589,
    -0.00092
   ],
   [
    0.00213,
    0.0093,
    -0.01618,
    -0.00423,
    -0.00484,
    0.00578,
    -0.00114,
    0.00448,
    0.00772,
    -0.00983,
    -0.00345,
    -0.00293,
    0.00683,
    7e-05
   ],
   [
    -0.01504,
    0.00688,
    -0.00761,
    -0.00875,
    0.00059,
    0.01121,
    0.01055,
    -0.01247,
    0.00526,
    -0.00534,
    -0.00734,
    0.00257,
    0.01179,
    0.01092
   ],
   [
    -0.00659,
    -0.00732,
    0.00469,
    0.00105,
    -0.00208,
    -0.00159,
    0.00542,
    -0.00382,
    -0.00594,
    0.001,
    -0.00036,
    -0.00074,
    -0.00119,
    0.00463
   ],
   [
    0.00365,
    -0.00613,
    -0.00635,
    0.00164,
    -0.00227,
    -0.0023,
    -0.00101,
    0.00221,
    -0.0057,
    -0.00499,
    0.00153,
    -0.00301,
    -0.00219,
    -0.00074
   ],
   [
    -0.01136,
    -0.00432,
    -0.00194,
    0.0127,
    -0.0016,
    -0.00663,
    -0.01084,
    -0.00876,
    -0.0048,
    -0.0028,
    0.0108,
    -0.00147,
    -0.00739,
    -0.00866
   ],
   [
    0.01002,
    0.02583,
    0.00072,
    -0.00157,
    -0.011,
    0.0232,
    0.01914,
    0.01071,
    0.02367,
    0.01,
    -0.00039,
    -0.00575,
    0.02426,
    0.01891
   ],
   [
    0.00721,
    -0.00029,
    0.00935,
    0.00948,
    -0.00998,
    -0.00431,
    -0.00355,
    0.01228,
    0.00515,
    0.00915,
    0.01023,
    -0.00992,
    -0.00639,
    -0.00411
   ],
   [
    -0.00314,
    -0.00454,
    -0.0009,
    0.01341,
    -0.00556,
    0.0015,
    -0.00937,
    -0.00226,
    -0.00368,
    -0.00216,
    0.01274,
    -0.00726,
    -0.0016,
    -0.00848
   ],
   [
    0.0111,
    0.0064,
    -0.00364,
    0.00281,
    -0.01027,
    0.0023,
    -0.00586,
    0.0071,
    0.00741,
    -0.00449,
    0.00563,
    -0.0075,
    0.00168,
    -0.00548
   ],
   [
    0.00532,
    0.0019,
    0.0018,
    -0.00122,
    -0.01665,
    0.00195,
    0.00627,
    -0.00028,
    -0.00163,
    0.00238,
    -0.00125,
    -0.01332,
    0.00023,
    0.00513
   ],
   [
    -0.00576,
    -0.01435,
    -0.00743,
    0.00113,
    -0.00904,
    0.00265,
    -0.00164,
    -0.00795,
    -0.01168,
    -0.00495,
    0.00167,
    -0.0069,
    0.00187,
    -0.00197
   ],
   [
    0.0031,
    0.01149,
    0.011,
    -0.00891,
    -0.00945,
    0.01306,
    0.00815,
    0.00285,
    0.01288,
    0.00847,
    -0.00682,
    -0.00561,
    0.01304,
    0.00661
   ],
   [
    -0.01439,
    0.01237,
    -0.00273,
    -0.00434,
    0.00677,
    0.00976,
    0.00243,
    -0.00814,
    0.01212,
    0.00044,
    -0.00429,
    0.00753,
    0.01062,
    0.00436
   ],
   [
    -0.00694,
    -0.02651,
    -0.01917,
    0.00062,
    0.01681,
    -0.01243,
    0.01189,
    -0.01222,
    -0.01696,
    -0.00274,
    0.00353,
    0.01557,
    -0.00467,
    0.01533
   ],
   [
    0.01398,
    0.01281,
    -0.00129,
    0.00039,
    -0.00159,
    -0.00018,
    0.02253,
    0.01174,
    0.01566,
    -0.00601,
    -0.0007,
    -0.00202,
    -0.00208,
    0.01642
   ],
   [
    -0.00189,
    0.00495,
    -0.00057,
    -0.00233,
    0.00328,
    0.00677,
    0.001,
    -0.00208,
    0.00644,
    -0.0016,
    -0.00093,
    0.00411,
    0.00667,
    0.00146
   ],
   [
    0.01432,
    -0.0172,
    0.00933,
    0.00813,
    -0.03209,
    0.01611,
    0.01202,
    0.01257,
    -0.01241,
    0.00916,
    0.01102,
    -0.02175,
    0.02156,
    0.01489
   ],
   [
    0.00956,
    0.01378,
    0.01042,
    0.00111,
    0.00213,
    0.0004,
    0.0005,
    0.00773,
    0.01037,
    0.00642,
    -3e-05,
    0.00255,
    -0.00179,
    -0.00179
   ],
   [
    -0.00567,
    0.00081,
    -0.00813,
    -0.00581,
    0.00152,
    -0.00038,
    -0.00191,
    -0.00292,
    -0.00065,
    -0.0078,
    -0.00824,
    -0.00085,
    -0.00272,
    -0.00261
   ],
   [
    -0.00542,
    -0.00467,
    0.00339,
    -0.00168,
    0.0014,
    0.00042,
    0.00127,
    -0.00381,
    -0.00367,
    0.00612,
    -0.00083,
    0.00172,
    0.00084,
    0.00209
   ],
   [
    2e-05,
    -0.00259,
    -0.00203,
    0.01193,
    -0.00704,
    0.01465,
    -0.00999,
    0.00104,
    -0.00225,
    0.00103,
    0.01015,
    -0.00398,
    0.01111,
    -0.00982
   ],
   [
    0.00198,
    0.00166,
    0.00792,
    0.0032,
    -0.00942,
    -0.00446,
    -0.00256,
    0.00136,
    -0.00313,
    0.00355,
    0.00159,
    -0.00787,
    -0.00417,
    -0.00151
   ],
   [
    0.01318,
    0.01515,
    -0.00158,
    0.00694,
    0.0006,
    -0.00742,
    -0.00125,
    0.01203,
    0.01542,
    7e-05,
    0.00608,
    0.00051,
    -0.00562,
    -0.00339
   ],
   [
    -0.00153,
    -0.011,
    0.00236,
    0.00135,
    -0.00683,
    -0.00137,
    -0.00012,
    -0.00283,
    -0.00813,
    -0.00136,
    0.00214,
    -0.00513,
    -0.0022,
    0.0001
   ],
   [
    0.00804,
    -0.01104,
    -0.00455,
    -0.00443,
    -0.01582,
    0.01399,
    0.01232,
    0.00785,
    -0.00257,
    0.00303,
    -0.00467,
    -0.01158,
    0.01693,
    0.01442
   ],
   [
    0.02501,
    0.01393,
    0.00589,
    0.01671,
    -0.00823,
    0.00914,
    -0.02779,
    0.02336,
    0.00727,
    0.00614,
    0.01257,
    -0.00685,
    0.01135,
    -0.02059
   ],
   [
    0.01844,
    0.00714,
    0.00843,
    0.01618,
    0.00073,
    -0.00857,
    0.0099,
    0.0115,
    0.01162,
    0.00771,
    0.01547,
    -0.00062,
    -0.01082,
    0.00829
   ],
   [
    0.00727,
    -0.01144,
    0.00593,
    -0.00613,
    -0.00035,
    0.00321,
    0.01251,
    0.00782,
    -0.00387,
    0.00237,
    -0.00555,
    -0.00018,
    0.00395,
    0.01144
   ],
   [
    -0.01054,
    -0.00215,
    0.00637,
    -0.01047,
    -0.00362,
    0.01929,
    0.00481,
    -0.00901,
    -0.00146,
    0.00287,
    -0.00838,
    -0.00181,
    0.0158,
    0.00278
   ],
   [
    0.00703,
    -0.00718,
    -0.00769,
    -0.00264,
    0.00795,
    0.01456,
    0.00297,
    0.00614,
    -0.00839,
    -0.00063,
    -0.00338,
    0.00451,
    0.0125,
    0.00295
   ],
   [
    0.00879,
    -0.00065,
    0.00068,
    -0.00272,
    -0.00419,
    0.00828,
    0.00692,
    0.00621,
    -0.00126,
    0.00277,
    -0.00213,
    -0.00327,
    0.0077,
    0.00573
   ],
   [
    0.022,
    -0.00499,
    -0.00596,
    0.00658,
    0.00095,
    -0.00222,
    0.00285,
    0.0174,
    -0.00342,
    -0.00598,
    0.01058,
    0.00403,
    -0.00072,
    0.00372
   ],
   [
    0.004,
    -0.00522,
    0.01411,
    -0.00042,
    -0.00118,
    0.0041,
    -0.00307,
    0.00175,
    -0.00028,
    0.00824,
    0.00118,
    -0.00023,
    0.00332,
    -0.00257
   ],
   [
    0.00089,
    -0.0078,
    0.00923,
    0.00159,
    -0.00109,
    0.00326,
    -0.00089,
    0.00196,
    -0.00735,
    0.00377,
    0.00146,
    -0.00038,
    0.00307,
    -0.00118
   ],
   [
    0.0101,
    0.00123,
    0.00226,
    -0.00363,
    0.00662,
    -0.00254,
    0.0018,
    0.00594,
    0.00266,
    0.00054,
    -0.00378,
    0.00504,
    -0.00192,
    0.00188
   ],
   [
    8e-05,
    -0.00563,
    0.00356,
    0.00291,
    -0.00092,
    0.00073,
    0.00355,
    0.00116,
    -0.00377,
    0.00521,
    0.00529,
    0.00094,
    0.003,
    0.00522
   ],
   [
    0.01201,
    -0.01534,
    0.00533,
    0.01275,
    -0.01554,
    0.01379,
    -0.00742,
    0.01286,
    -0.0095,
    0.0059,
    0.00826,
    -0.01541,
    0.00748,
    -0.00821
   ],
   [
    -0.00559,
    -0.02167,
    0.00556,
    0.00027,
    -0.0024,
    0.00653,
    -0.00712,
    -0.00329,
    -0.02187,
    0.00231,
    0.00132,
    -0.00034,
    0.00762,
    -0.00411
   ],
   [
    0.00094,
    0.00309,
    0.00547,
    -0.01534,
    -0.00392,
    0.01159,
    -0.00074,
    0.00068,
    0.00238,
    -0.00107,
    -0.01196,
    -0.00253,
    0.01042,
    0.0007
   ],
   [
    -0.00635,
    -0.01701,
    0.00263,
    0.00031,
    -0.00708,
    0.0036,
    0.00614,
    -0.00775,
    -0.0142,
    0.00328,
    -0.00061,
    -0.00844,
    0.00353,
    0.00396
   ],
   [
    -0.00704,
    -0.00048,
    0.00604,
    0.00136,
    -0.00359,
    -0.01161,
    0.00218,
    -0.00638,
    -0.00181,
    0.00321,
    0.00101,
    -0.00293,
    -0.00922,
    0.00141
   ],
   [
    -0.00974,
    -0.01308,
    0.00529,
    0.00154,
    -0.00528,
    0.00438,
    -0.00449,
    -0.00949,
    -0.01171,
    0.00366,
    0.00225,
    -0.00348,
    0.00435,
    -0.00299
   ],
   [
    0.0042,
    -0.0059,
    -0.004,
    0.00137,
    5e-05,
    -0.00246,
    0.00013,
    0.00417,
    -0.0058,
    -0.00463,
    0.00078,
    0.00059,
    -0.00246,
    0.00053
   ],
   [
    -0.00406,
    -0.00034,
    -0.00596,
    -0.00026,
    0.01021,
    0.00012,
    -0.01173,
    -0.00225,
    -0.00175,
    -0.00163,
    -0.00291,
    0.00721,
    -0.0011,
    -0.0094
   ],
   [
    -0.00664,
    -0.00071,
    -0.00379,
    -0.00094,
    -0.00741,
    0.00804,
    -0.00483,
    -0.00547,
    0.00168,
    -0.00185,
    0.0007,
    -0.00491,
    0.00905,
    -0.00253
   ],
   [
    0.00836,
    0.01568,
    0.00972,
    0.00345,
    -0.00062,
    -0.0076,
    -0.00872,
    0.00965,
    0.0091,
    0.01099,
    0.00227,
    -0.00287,
    -0.00948,
    -0.00849
   ],
   [
    -0.0073,
    0.00587,
    0.00537,
    -0.01273,
    -0.00629,
    0.01074,
    0.00364,
    -0.00804,
    0.00247,
    0.00421,
    -0.01182,
    -0.0053,
    0.00876,
    0.00192
   ],
   [
    -0.01518,
    0.00722,
    0.00336,
    0.00373,
    -0.0014,
    -0.00597,
    -0.00421,
    -0.00674,
    0.00298,
    -0.00034,
    0.00171,
    -0.00037,
    -0.00476,
    -0.00497
   ],
   [
    0.00394,
    0.00078,
    0.00466,
    -3e-05,
    -0.00634,
    0.0033,
    -0.00475,
    0.00267,
    0.00039,
    0.00542,
    0.00032,
    -0.00665,
    0.00365,
    -0.00418
   ],
   [
    -0.00239,
    0.02462,
    -0.0039,
    0.00098,
    0.01048,
    -0.00567,
    -0.0144,
    0.00318,
    0.01582,
    -0.00234,
    -0.00214,
    0.00703,
    -0.00436,
    -0.01022
   ],
   [
    0.01221,
    0.03366,
    -0.00534,
    0.00955,
    0.00681,
    0.00526,
    -0.0139,
    0.01143,
    0.02268,
    -0.00381,
    0.00824,
    0.00758,
    0.00631,
    -0.00944
   ],
   [
    -0.01085,
    -7e-05,
    -0.01359,
    0.00347,
    0.00456,
    0.00425,
    -2e-05,
    -0.00668,
    -0.00172,
    -0.00988,
    0.00323,
    0.0064,
    0.00506,
    -0.00051
   ],
   [
    -0.00352,
    0.00064,
    -0.01947,
    -0.00995,
    -0.00867,
    0.00206,
    0.01607,
    -0.00564,
    0.00016,
    -0.01573,
    -0.01328,
    -0.011,
    -0.00173,
    0.00922
   ],
   [
    5e-05,
    0.00148,
    -0.00248,
    -0.00165,
    0.00716,
    0.00598,
    0.00142,
    -0.00293,
    0.00254,
    7e-05,
    -0.00175,
    0.0076,
    0.00677,
    0.00131
   ],
   [
    -0.00436,
    0.0163,
    0.00451,
    0.00244,
    0.00375,
    -0.00939,
    -0.01193,
    -0.00293,
    0.00813,
    0.00269,
    -0.00208,
    -0.0011,
    -0.00987,
    -0.01062
   ],
   [
    -0.0138,
    0.01292,
    0.00321,
    -0.0058,
    0.00209,
    0.00594,
    0.00154,
    -0.01009,
    0.0139,
    0.00203,
    -0.00449,
    0.00257,
    0.0057,
    0.00256
   ],
   [
    0.00683,
    0.00567,
    0.00074,
    0.00561,
    0.00327,
    0.00469,
    0.01227,
    0.00859,
    0.00862,
    0.00206,
    0.00518,
    0.00651,
    0.00956,
    0.01406
   ],
   [
    0.01297,
    0.01108,
    -0.00222,
    -0.00589,
    -0.00143,
    4e-05,
    -0.00138,
    0.00833,
    0.00787,
    -0.00239,
    -0.00539,
    -0.00014,
    -0.00021,
    -0.0024
   ],
   [
    0.01586,
    0.00254,
    0.00531,
    -0.00014,
    0.00305,
    0.0107,
    -0.00056,
    0.01548,
    0.00524,
    0.00816,
    0.00109,
    0.00313,
    0.01084,
    0.00208
   ],
   [
    0.05018,
    -0.02453,
    0.00571,
    -0.00472,
    -0.0098,
    -0.00704,
    0.00521,
    0.04096,
    -0.015,
    0.00696,
    -0.00304,
    -0.00835,
    -0.00624,
    0.00444
   ],
   [
    -0.00076,
    -0.00667,
    0.01333,
    -0.00274,
    -0.01312,
    0.00165,
    0.01232,
    0.00227,
    -0.00419,
    0.00586,
    -0.00528,
    -0.01177,
    0.00162,
    0.00934
   ],
   [
    -0.01618,
    -0.01627,
    0.00721,
    -0.00137,
    -0.00423,
    -0.0116,
    0.00344,
    -0.01429,
    -0.01488,
    -0.00106,
    -0.00248,
    -0.00582,
    -0.01206,
    0.00277
   ],
   [
    0.00313,
    -0.01256,
    0.00585,
    0.00419,
    -0.00545,
    -0.00278,
    0.0049,
    0.00039,
    -0.01189,
    0.00535,
    0.0048,
    -0.00417,
    -0.00342,
    0.00338
   ],
   [
    0.00894,
    -0.0088,
    0.00663,
    0.00432,
    -0.00478,
    -0.00015,
    -0.0052,
    0.00753,
    -0.0085,
    0.00686,
    0.00341,
    -0.00508,
    0.00173,
    -0.00306
   ],
   [
    -0.00699,
    0.00555,
    0.00013,
    -0.0094,
    0.00355,
    2e-05,
    -0.00403,
    -0.00287,
    0.00159,
    -0.00126,
    -0.01202,
    0.00156,
    0.00079,
    -0.00379
   ],
   [
    -0.0057,
    0.00242,
    -0.00016,
    -0.0038,
    0.00142,
    -0.0011,
    0.00335,
    -0.004,
    0.00188,
    0.00299,
    -0.0023,
    -0.00037,
    -0.00223,
    0.00263
   ],
   [
    -0.01002,
    -0.01829,
    0.00646,
    0.00701,
    0.0057,
    -0.0049,
    -0.00908,
    -0.00913,
    -0.01597,
    0.00487,
    0.0063,
    0.00412,
    -0.00487,
    -0.00843
   ],
   [
    -0.0165,
    -0.00649,
    0.00599,
    -0.01348,
    -0.00027,
    0.00556,
    -0.00359,
    -0.01313,
    -0.00836,
    0.00085,
    -0.01223,
    0.00029,
    0.00519,
    -0.00255
   ],
   [
    0.00443,
    -0.0003,
    -0.00124,
    0.00867,
    0.00128,
    -0.01046,
    -0.00373,
    0.00369,
    0.00054,
    0.00055,
    0.00918,
    0.003,
    -0.0075,
    -0.00277
   ],
   [
    -7e-05,
    -0.01815,
    0.00875,
    -0.00473,
    -0.01224,
    0.01287,
    0.00821,
    -3e-05,
    -0.01256,
    0.00703,
    -0.00259,
    -0.00934,
    0.0135,
    0.00799
   ],
   [
    0.00752,
    0.00272,
    -0.00368,
    -0.00917,
    0.00667,
    -0.00603,
    0.00167,
    0.00685,
    0.00055,
    -0.00416,
    -0.01008,
    0.00484,
    -0.00652,
    0.00068
   ],
   [
    -0.00528,
    -0.0155,
    -0.00783,
    0.01164,
    0.00157,
    0.01639,
    -0.00198,
    -0.004,
    -0.00933,
    8e-05,
    0.0106,
    0.00011,
    0.01768,
    0.00187
   ],
   [
    0.0122,
    -0.00488,
    -0.00483,
    0.00267,
    0.00959,
    -0.00285,
    -0.00377,
    0.00973,
    -0.00264,
    -0.00208,
    0.00471,
    0.00851,
    -0.00073,
    -0.0021
   ],
   [
    0.00725,
    0.00462,
    -0.00134,
    -0.00666,
    0.00417,
    0.00227,
    0.00465,
    0.00578,
    0.00555,
    0.00053,
    -0.0073,
    0.00513,
    0.00289,
    0.0037
   ],
   [
    -0.02521,
    0.00798,
    -0.00206,
    -0.01103,
    0.00681,
    -0.00028,
    0.00836,
    -0.02148,
    0.00276,
    -0.00197,
    -0.0109,
    0.005,
    -0.00282,
    0.00466
   ],
   [
    0.00412,
    -0.00721,
    -0.00157,
    0.00083,
    -0.00869,
    0.0104,
    0.00368,
    0.00101,
    -0.0061,
    0.00074,
    0.00154,
    -0.00766,
    0.00888,
    0.00359
   ],
   [
    -0.00065,
    0.00799,
    -0.00715,
    0.00082,
    0.01542,
    0.00974,
    -0.02136,
    -0.00378,
    0.00375,
    -0.0051,
    0.00404,
    0.01436,
    0.00796,
    -0.01869
   ],
   [
    0.00356,
    0.00016,
    -0.00288,
    0.00481,
    -0.00459,
    -0.00436,
    -0.00765,
    0.00426,
    -0.00168,
    -0.00343,
    0.00405,
    -0.00385,
    -0.0035,
    -0.00688
   ],
   [
    -0.00592,
    0.00857,
    -0.00199,
    -0.00598,
    0.00032,
    0.00779,
    0.00488,
    -0.00409,
    0.00466,
    -0.00452,
    -0.00536,
    0.00158,
    0.00635,
    0.00416
   ],
   [
    -0.01211,
    0.00228,
    0.00247,
    -0.0036,
    0.0055,
    -0.00366,
    0.00546,
    -0.0093,
    -0.00325,
    -0.00023,
    -0.00444,
    0.00088,
    -0.00661,
    0.00209
   ],
   [
    -0.01331,
    0.00509,
    0.00966,
    -0.02526,
    0.00726,
    0.01858,
    0.01383,
    -0.01033,
    0.01095,
    0.01492,
    -0.0218,
    0.00821,
    0.01691,
    0.01344
   ],
   [
    -0.01396,
    -0.02153,
    -0.00138,
    -0.01019,
    0.00094,
    0.00283,
    0.01184,
    -0.01679,
    -0.01632,
    0.00208,
    -0.00703,
    0.00159,
    0.00294,
    0.01038
   ],
   [
    0.00311,
    -0.00168,
    0.00663,
    0.00788,
    -0.00761,
    0.00039,
    -0.00141,
    0.00211,
    0.00084,
    0.00724,
    0.00802,
    -0.0072,
    0.00151,
    -0.0003
   ],
   [
    -0.00117,
    0.01057,
    0.0032,
    0.00868,
    0.00441,
    -0.00783,
    0.00505,
    -0.00054,
    0.00692,
    0.00292,
    0.00699,
    0.00363,
    -0.00729,
    0.00462
   ],
   [
    0.02398,
    0.00806,
    0.02253,
    -0.00994,
    0.00544,
    0.00309,
    -0.00675,
    0.02146,
    0.00913,
    0.02128,
    -0.00755,
    0.00457,
    0.00256,
    -0.00625
   ],
   [
    -0.00469,
    0.00142,
    0.01397,
    -0.0044,
    -0.0021,
    0.0056,
    -0.00504,
    -0.00567,
    0.00106,
    0.01268,
    -0.00327,
    -0.00122,
    0.00586,
    -0.00395
   ],
   [
    0.03533,
    0.00727,
    -0.0256,
    0.02883,
    -0.01308,
    -0.03599,
    0.00793,
    0.01752,
    -0.00733,
    -0.01976,
    0.02005,
    -0.01416,
    -0.03507,
    0.00212
   ],
   [
    0.00612,
    -0.01121,
    0.00993,
    0.00299,
    0.00438,
    0.00081,
    -0.00177,
    0.00538,
    -0.00981,
    0.01003,
    0.00689,
    0.00447,
    0.00398,
    0.00061
   ],
   [
    -0.00679,
    -0.0075,
    -0.00744,
    -0.01234,
    -0.01513,
    0.00666,
    0.02336,
    -0.00833,
    -0.00838,
    -0.00706,
    -0.00978,
    -0.0101,
    0.0075,
    0.0213
   ],
   [
    0.00839,
    0.00371,
    0.002,
    -0.00072,
    -0.00118,
    0.00619,
    0.00052,
    0.00656,
    -0.00074,
    0.00211,
    -0.00067,
    -0.00246,
    0.00456,
    5e-05
   ],
   [
    0.00197,
    -0.00323,
    0.00946,
    0.0003,
    0.00865,
    -0.00932,
    -0.00043,
    0.00259,
    -0.00211,
    0.00868,
    0.00066,
    0.00757,
    -0.00814,
    -0.00204
   ],
   [
    0.02772,
    0.01746,
    0.00748,
    -0.00777,
    0.00232,
    0.01276,
    0.0048,
    0.02518,
    0.02018,
    0.01306,
    -0.00535,
    0.00285,
    0.01051,
    0.00495
   ],
   [
    0.01726,
    0.01742,
    0.00906,
    0.0114,
    0.01088,
    -0.01489,
    -0.01121,
    0.01684,
    0.01327,
    0.00774,
    0.01056,
    0.00734,
    -0.01641,
    -0.01022
   ],
   [
    -0.0,
    0.02003,
    -0.00225,
    -0.00199,
    0.00965,
    -0.01303,
    -0.00327,
    0.00054,
    0.01247,
    -0.0031,
    -0.00134,
    0.00692,
    -0.01363,
    -0.00417
   ],
   [
    0.00803,
    -0.00742,
    0.00171,
    0.00423,
    -0.01074,
    0.00242,
    0.00371,
    0.00575,
    -0.00293,
    0.00181,
    0.00403,
    -0.00897,
    0.00342,
    0.00345
   ],
   [
    0.00937,
    -0.00696,
    -0.00328,
    0.00767,
    0.00382,
    -0.00383,
    -0.00172,
    0.00857,
    -0.00281,
    -0.0039,
    0.00681,
    0.00135,
    -0.00363,
    -0.00224
   ],
   [
    -0.00249,
    -0.01137,
    0.00193,
    0.00396,
    -0.00756,
    -0.0071,
    -0.00206,
    -0.00301,
    -0.0122,
    0.00287,
    0.0043,
    -0.00714,
    -0.00688,
    -0.00323
   ],
   [
    0.00812,
    0.01645,
    -0.00081,
    -0.00788,
    0.00861,
    0.00463,
    -0.00321,
    0.00717,
    0.01265,
    0.00094,
    -0.00792,
    0.00799,
    0.00481,
    -0.00191
   ],
   [
    -0.01057,
    -0.0108,
    0.00498,
    -0.00685,
    -0.00357,
    0.00909,
    -0.01066,
    -0.01083,
    -0.01099,
    0.00296,
    -0.0088,
    -0.00495,
    0.00774,
    -0.00834
   ],
   [
    0.00462,
    0.00826,
    0.01516,
    0.00163,
    -0.00528,
    -0.00076,
    0.00142,
    0.00167,
    0.00762,
    0.01094,
    0.00401,
    -0.00456,
    0.00055,
    -9e-05
   ],
   [
    0.01701,
    -0.00018,
    0.00907,
    0.00063,
    -0.00753,
    0.00357,
    -0.00361,
    0.01416,
    0.00863,
    0.00923,
    0.00233,
    -0.00526,
    0.00449,
    -0.00496
   ],
   [
    0.0006,
    -0.0077,
    0.00451,
    -0.00215,
    -0.00343,
    0.00366,
    0.00162,
    0.00091,
    -0.00687,
    0.00318,
    -0.00176,
    -0.00207,
    0.00397,
    0.00087
   ],
   [
    0.01703,
    0.0083,
    0.00325,
    0.00314,
    -0.00144,
    0.00686,
    -0.00915,
    0.01361,
    0.00668,
    0.00436,
    0.00409,
    -0.00076,
    0.00743,
    -0.00633
   ],
   [
    0.00576,
    0.00432,
    0.0091,
    0.00134,
    -0.00451,
    0.00191,
    -0.00607,
    0.0068,
    0.00253,
    0.00643,
    0.00098,
    -0.00382,
    0.00407,
    -0.00466
   ],
   [
    -0.00281,
    -0.00793,
    0.00528,
    0.00361,
    0.00257,
    -0.00359,
    -0.00353,
    -0.00176,
    -0.00343,
    0.00258,
    0.00588,
    0.0035,
    -0.00232,
    -0.00146
   ],
   [
    0.02258,
    0.00302,
    -0.00268,
    -0.01211,
    -0.01425,
    -0.00208,
    0.02357,
    0.01844,
    0.00947,
    -0.00054,
    -0.00978,
    -0.00883,
    0.00035,
    0.02044
   ],
   [
    -0.00674,
    -0.00433,
    -0.00894,
    0.00217,
    0.00225,
    -0.00455,
    -0.00431,
    -0.00531,
    -0.00288,
    -0.00738,
    0.00244,
    0.00208,
    -0.00319,
    -0.00443
   ],
   [
    0.00718,
    -0.00324,
    -0.00677,
    -0.00066,
    -0.00482,
    0.00311,
    -0.00514,
    0.00368,
    -0.0029,
    -0.00744,
    0.00159,
    -0.00244,
    0.0014,
    -0.00505
   ],
   [
    -0.00576,
    -0.0022,
    -0.00852,
    0.0047,
    -0.00775,
    0.00554,
    -0.01138,
    -0.00645,
    -0.00179,
    -0.0096,
    0.0046,
    -0.0068,
    0.0032,
    -0.01225
   ],
   [
    -0.00755,
    -0.00218,
    -0.00396,
    -0.02706,
    -0.01973,
    0.02706,
    0.02898,
    -0.00187,
    0.00601,
    0.00436,
    -0.02091,
    -0.01094,
    0.02958,
    0.02861
   ],
   [
    -0.0061,
    0.00163,
    0.00222,
    -0.00579,
    -0.00224,
    0.00585,
    0.00232,
    -0.00546,
    0.00302,
    0.00175,
    -0.00601,
    -0.00349,
    0.00401,
    0.00126
   ],
   [
    0.01064,
    0.004,
    0.01226,
    -0.00799,
    0.01112,
    -0.00474,
    -0.01299,
    0.00831,
    -0.00412,
    0.00532,
    -0.00754,
    0.00993,
    -0.00252,
    -0.01138
   ],
   [
    -0.00702,
    -0.02061,
    -0.00236,
    -0.00728,
    -0.00388,
    0.0006,
    0.00651,
    -0.00656,
    -0.02045,
    -0.00104,
    -0.00476,
    -0.00575,
    0.00041,
    0.0057
   ],
   [
    0.00161,
    0.01304,
    -0.00384,
    -0.00284,
    0.00457,
    0.00167,
    -0.00981,
    0.00017,
    0.00919,
    -0.0055,
    -0.00417,
    0.00274,
    -0.00015,
    -0.00807
   ],
   [
    0.00453,
    -0.00312,
    -0.00531,
    -0.01004,
    -0.01256,
    0.01523,
    0.01672,
    0.00385,
    -0.00344,
    -0.00409,
    -0.01158,
    -0.01225,
    0.01243,
    0.01274
   ],
   [
    0.03723,
    0.01365,
    0.01764,
    -0.00394,
    0.00274,
    0.0036,
    0.02112,
    0.04259,
    0.01734,
    0.00451,
    -0.00398,
    0.00182,
    -0.00136,
    0.01145
   ],
   [
    0.00545,
    -0.01149,
    0.00711,
    0.00717,
    -0.00176,
    -0.00109,
    0.00647,
    0.00417,
    -0.01172,
    0.00388,
    0.00749,
    -0.00151,
    -0.00229,
    0.00424
   ],
   [
    0.0037,
    0.00096,
    -0.00043,
    -0.00251,
    0.00047,
    0.00291,
    0.00095,
    0.00323,
    0.00659,
    -0.00024,
    -0.00142,
    0.00045,
    0.00154,
    -0.00096
   ],
   [
    -0.0047,
    0.00539,
    0.00183,
    -0.00888,
    0.007,
    0.00263,
    -0.0012,
    -0.00706,
    0.00753,
    0.00514,
    -0.00605,
    0.00855,
    0.00379,
    -0.00176
   ],
   [
    -0.0036,
    -0.00125,
    0.01977,
    -0.00212,
    0.00182,
    0.00634,
    -0.00197,
    -0.00167,
    0.00194,
    0.01534,
    0.00105,
    0.00513,
    0.01043,
    0.00321
   ],
   [
    0.01233,
    -0.00261,
    0.01519,
    0.00497,
    0.00212,
    -0.0019,
    -0.01682,
    0.0136,
    0.00512,
    0.01247,
    0.00524,
    0.00177,
    0.00293,
    -0.01018
   ],
   [
    0.00286,
    0.00392,
    -0.00074,
    0.00113,
    0.00574,
    0.00013,
    -0.00304,
    0.00288,
    0.00498,
    0.0023,
    0.00013,
    0.00598,
    0.00061,
    -0.00191
   ],
   [
    0.02429,
    -0.00353,
    0.00673,
    0.00422,
    0.00534,
    -0.00731,
    -0.00348,
    0.0163,
    -0.00357,
    0.00465,
    0.00094,
    -0.00023,
    -0.00921,
    -0.00266
   ],
   [
    -0.00444,
    -0.00701,
    -0.01074,
    0.00847,
    -0.00681,
    -0.0017,
    -0.00486,
    -0.00532,
    -0.00942,
    -0.01038,
    0.00809,
    -0.00479,
    -0.00178,
    -0.00398
   ],
   [
    0.0133,
    0.01562,
    -0.00221,
    0.00617,
    -0.00043,
    -0.00247,
    -0.02192,
    0.01037,
    0.01074,
    -0.00342,
    0.00543,
    0.00072,
    -0.00474,
    -0.02022
   ],
   [
    -0.02209,
    0.00094,
    -0.0035,
    -0.0009,
    0.0113,
    0.00448,
    -0.00657,
    -0.0158,
    0.00212,
    -0.00183,
    -3e-05,
    0.01234,
    0.00644,
    -0.0051
   ],
   [
    0.00745,
    -0.00181,
    -0.0022,
    -0.01147,
    0.00757,
    0.0001,
    -0.01016,
    0.00513,
    -3e-05,
    -0.00233,
    -0.00582,
    0.0096,
    0.00161,
    -0.00852
   ],
   [
    0.002,
    0.00104,
    -0.00509,
    0.00356,
    0.00706,
    -0.00633,
    -0.0094,
    -0.00206,
    0.00093,
    -0.00596,
    0.00463,
    0.00538,
    -0.00342,
    -0.00776
   ],
   [
    -0.01105,
    -0.00653,
    0.0044,
    -0.01372,
    0.00903,
    0.00049,
    0.02398,
    -0.00827,
    -0.00374,
    0.00175,
    -0.01233,
    0.00816,
    -0.00186,
    0.01867
   ],
   [
    0.00364,
    -0.01463,
    -0.00155,
    0.00937,
    -0.00312,
    -0.01145,
    0.00814,
    0.00377,
    -0.00984,
    -0.00087,
    0.00955,
    -0.00324,
    -0.01095,
    0.00566
   ],
   [
    0.00174,
    0.01073,
    0.00281,
    0.00414,
    0.00078,
    0.00197,
    -0.00589,
    0.00019,
    0.01056,
    0.00309,
    0.00492,
    0.00038,
    0.00072,
    -0.00562
   ],
   [
    0.01716,
    0.00227,
    0.0018,
    0.00544,
    -0.0009,
    -0.0021,
    -0.00104,
    0.01603,
    0.00451,
    -1e-05,
    0.00451,
    -0.00058,
    -0.0033,
    -0.00179
   ],
   [
    -0.00669,
    0.00179,
    0.00933,
    -0.01286,
    0.00187,
    0.00516,
    0.00748,
    -0.00497,
    0.0041,
    0.00864,
    -0.0097,
    0.00385,
    0.00684,
    0.00678
   ],
   [
    0.00362,
    0.00108,
    0.00308,
    0.00016,
    -0.00039,
    0.00234,
    -0.00087,
    0.00551,
    0.00141,
    0.005,
    0.00202,
    0.00047,
    0.00268,
    -0.00081
   ],
   [
    -0.00523,
    0.01006,
    -0.00402,
    0.0079,
    0.0026,
    -0.01149,
    0.00033,
    -0.00423,
    0.00686,
    -0.00555,
    0.00685,
    4e-05,
    -0.01067,
    4e-05
   ],
   [
    -0.00462,
    -0.02314,
    0.00097,
    0.0036,
    -0.00232,
    -0.00485,
    0.00464,
    -0.00407,
    -0.01797,
    0.00286,
    0.00309,
    -0.0039,
    -0.00503,
    0.00365
   ],
   [
    0.0069,
    0.00033,
    -0.00516,
    0.00238,
    -0.00283,
    0.00515,
    0.00494,
    0.00543,
    -0.00095,
    -0.00177,
    0.00204,
    -0.00305,
    0.00456,
    0.00459
   ],
   [
    0.00211,
    0.00667,
    -0.00326,
    0.01496,
    -0.00481,
    -0.02016,
    0.01274,
    0.00658,
    0.00511,
    -0.00028,
    0.01432,
    -0.00327,
    -0.01397,
    0.0118
   ],
   [
    -0.02298,
    -0.0172,
    0.01878,
    -0.0178,
    -0.00486,
    0.00884,
    -0.00041,
    -0.01834,
    -0.01538,
    0.01337,
    -0.01745,
    -0.00667,
    0.0044,
    -0.0023
   ],
   [
    -0.01138,
    0.00136,
    -0.00203,
    -0.00997,
    -0.00346,
    0.01687,
    0.0083,
    -0.00778,
    -0.00066,
    -0.00195,
    -0.00999,
    -0.00527,
    0.01497,
    0.00777
   ],
   [
    -0.00962,
    -0.01153,
    -0.00644,
    0.0012,
    -0.01732,
    -0.00251,
    0.00562,
    -0.00509,
    -0.00743,
    -0.00487,
    0.00145,
    -0.01509,
    -0.00139,
    0.00456
   ],
   [
    0.00724,
    -0.00685,
    0.00589,
    -0.00062,
    -0.01227,
    0.00562,
    -0.00189,
    0.0043,
    -0.004,
    0.00234,
    -0.00261,
    -0.0118,
    0.00257,
    -0.00499
   ],
   [
    -0.01345,
    -0.00544,
    -0.0065,
    -0.02061,
    0.00758,
    0.00628,
    0.01112,
    -0.0127,
    -0.00442,
    -0.00366,
    -0.02053,
    0.00674,
    0.00609,
    0.00864
   ],
   [
    -0.00942,
    -0.0247,
    -0.00471,
    -0.00485,
    0.00277,
    -0.00305,
    0.00719,
    -0.00852,
    -0.01622,
    -0.00466,
    -0.00232,
    0.00312,
    -0.00097,
    0.00414
   ],
   [
    0.01161,
    0.00529,
    -0.00195,
    -0.0073,
    0.00208,
    0.00559,
    0.00633,
    0.00949,
    0.00675,
    0.00046,
    -0.00407,
    0.0038,
    0.00568,
    0.0048
   ],
   [
    -0.00487,
    0.00208,
    -0.00236,
    0.00925,
    0.01102,
    -0.02335,
    -0.00656,
    -0.00358,
    -0.00069,
    -0.00342,
    0.00763,
    0.00858,
    -0.02101,
    -0.00565
   ],
   [
    0.01276,
    0.00185,
    -0.00779,
    -0.00236,
    0.00307,
    -0.00886,
    -0.00058,
    0.01156,
    0.00356,
    -0.0086,
    -0.00317,
    0.00226,
    -0.00902,
    -0.00135
   ],
   [
    -0.00268,
    -0.00583,
    0.01057,
    -0.01226,
    -0.01638,
    0.02188,
    0.00711,
    0.00037,
    -0.00748,
    0.01171,
    -0.01081,
    -0.01232,
    0.02117,
    0.00719
   ],
   [
    -0.00592,
    -0.00696,
    0.00684,
    0.00836,
    -0.01568,
    0.00407,
    0.0023,
    -0.00535,
    -0.0047,
    0.00218,
    0.00828,
    -0.01212,
    0.00682,
    0.0033
   ],
   [
    -0.02099,
    0.00694,
    0.00063,
    -0.00683,
    0.00987,
    -0.00768,
    -0.00763,
    -0.017,
    0.0044,
    -0.00113,
    -0.00761,
    0.00606,
    -0.00887,
    -0.00869
   ],
   [
    -0.01067,
    -0.0004,
    -0.0007,
    0.0022,
    -0.00407,
    0.00885,
    0.02082,
    -0.00846,
    0.00423,
    0.00369,
    0.00309,
    -0.00266,
    0.0083,
    0.01801
   ],
   [
    -0.01057,
    0.00478,
    -0.00163,
    0.00573,
    0.00866,
    -0.0077,
    -0.01174,
    -0.00973,
    0.00071,
    -0.00594,
    0.00537,
    0.0081,
    -0.00558,
    -0.00952
   ],
   [
    -0.00347,
    -0.01013,
    0.0047,
    0.00092,
    -0.00155,
    -0.00211,
    0.00885,
    -0.00419,
    -0.00218,
    0.00549,
    0.00206,
    -0.0022,
    -0.00216,
    0.0066
   ],
   [
    -0.00627,
    0.00928,
    -0.00121,
    0.00319,
    0.00757,
    0.00571,
    0.00415,
    -0.00588,
    0.00505,
    0.0006,
    0.00226,
    0.00684,
    0.00499,
    0.00317
   ],
   [
    -0.00248,
    0.01414,
    0.0021,
    0.01467,
    0.00028,
    -0.00513,
    -0.01116,
    0.00059,
    0.01151,
    0.00183,
    0.01205,
    0.00023,
    -0.00668,
    -0.01009
   ],
   [
    0.00699,
    0.00074,
    -0.00868,
    -0.00041,
    0.00075,
    -0.00261,
    -0.00596,
    0.0044,
    0.0015,
    -0.00582,
    -0.00043,
    -8e-05,
    -0.00403,
    -0.00773
   ],
   [
    -0.00631,
    0.00906,
    -8e-05,
    0.00063,
    -0.01381,
    0.01426,
    0.00203,
    -0.00581,
    0.00551,
    -0.00077,
    0.0006,
    -0.01388,
    0.00953,
    0.00143
   ],
   [
    -0.00163,
    0.00109,
    -0.00285,
    0.00977,
    -0.00328,
    -0.00511,
    -0.00669,
    -0.003,
    0.0006,
    -0.00128,
    0.00951,
    -0.00338,
    -0.00435,
    -0.00601
   ],
   [
    0.00758,
    0.00165,
    0.01403,
    -0.00678,
    0.00833,
    -0.0015,
    -0.01412,
    0.00583,
    0.0044,
    0.01078,
    -0.00584,
    0.00353,
    -0.00071,
    -0.01286
   ],
   [
    -0.00125,
    0.00573,
    -0.00446,
    -0.00752,
    -0.00826,
    0.00396,
    -0.00792,
    0.00097,
    0.00688,
    0.0016,
    -0.00837,
    -0.00923,
    0.00351,
    -0.0064
   ],
   [
    0.00143,
    0.01418,
    -0.01262,
    -0.00768,
    0.00773,
    0.00239,
    0.00628,
    0.00272,
    0.00842,
    -0.01045,
    -0.00717,
    0.0086,
    -0.00056,
    0.00363
   ],
   [
    0.0061,
    0.00514,
    -0.00874,
    -0.00271,
    0.00452,
    0.01079,
    0.00691,
    0.00542,
    0.0082,
    -0.00523,
    -0.00122,
    0.00663,
    0.01065,
    0.00621
   ],
   [
    0.00921,
    0.00236,
    -0.00666,
    0.00869,
    -0.00919,
    -0.01087,
    -0.00862,
    0.007,
    0.00358,
    -0.00685,
    0.00825,
    -0.00881,
    -0.01104,
    -0.00777
   ],
   [
    0.00033,
    0.00094,
    -0.00376,
    -0.00263,
    0.00724,
    -0.00812,
    0.00511,
    -0.00024,
    0.00243,
    -0.00014,
    -0.0033,
    0.00604,
    -0.00615,
    0.00329
   ],
   [
    -0.02906,
    -0.03686,
    -0.00731,
    0.00473,
    -0.00656,
    -0.01036,
    0.00186,
    -0.02603,
    -0.03366,
    -0.01033,
    0.0019,
    -0.00677,
    -0.00961,
    0.00318
   ],
   [
    0.00257,
    0.02032,
    0.00698,
    0.00191,
    0.00808,
    -0.00564,
    -0.00523,
    0.00214,
    0.01377,
    0.00645,
    0.00351,
    0.00738,
    -0.00458,
    -0.00229
   ],
   [
    0.02456,
    -0.00057,
    0.00375,
    0.01235,
    -0.0046,
    -0.00408,
    0.00784,
    0.02421,
    0.00169,
    0.00258,
    0.01174,
    -0.00327,
    -0.00416,
    0.00629
   ],
   [
    -0.00307,
    -0.03293,
    0.00433,
    0.02529,
    -0.03203,
    -0.00447,
    0.02883,
    -0.00129,
    -0.02758,
    0.00393,
    0.02417,
    -0.02622,
    -0.00258,
    0.02431
   ],
   [
    -0.0043,
    0.0071,
    0.00077,
    -0.00304,
    0.0041,
    -0.00477,
    0.01057,
    -0.00146,
    0.00805,
    0.00452,
    -0.00503,
    0.00266,
    -0.00224,
    0.00968
   ],
   [
    0.01038,
    -0.01213,
    -0.00228,
    0.0056,
    -0.01018,
    -0.00905,
    0.00039,
    0.00608,
    -0.01123,
    0.00115,
    0.00563,
    -0.00798,
    -0.01003,
    -0.0017
   ],
   [
    0.02059,
    -0.00172,
    -0.00106,
    0.02396,
    0.00587,
    -0.02311,
    -0.00381,
    0.0189,
    0.00374,
    -0.00368,
    0.02463,
    0.00372,
    -0.02316,
    -0.00616
   ],
   [
    0.00288,
    0.00565,
    -0.00203,
    0.01765,
    -0.00238,
    -0.00464,
    -0.00755,
    0.00496,
    0.0013,
    0.00045,
    0.01854,
    0.00047,
    -0.00487,
    -0.00574
   ],
   [
    -0.01317,
    -0.01127,
    -0.0106,
    -0.01071,
    0.01614,
    -0.00114,
    -0.00422,
    -0.00895,
    -0.01234,
    -0.00983,
    -0.01179,
    0.01274,
    -0.00171,
    -0.00616
   ],
   [
    0.00951,
    0.01715,
    0.00784,
    -0.00551,
    0.00218,
    0.00518,
    0.00566,
    0.00769,
    0.0099,
    0.00938,
    -0.00407,
    0.00322,
    0.0059,
    0.00716
   ],
   [
    -0.0096,
    -0.00933,
    -0.02034,
    0.00241,
    -0.00507,
    -0.00553,
    -0.00067,
    -0.00942,
    -0.01157,
    -0.017,
    -0.00151,
    -0.00601,
    -0.0075,
    -0.00353
   ],
   [
    0.02053,
    0.02797,
    0.00602,
    -0.00591,
    0.01253,
    -0.00517,
    -0.01298,
    0.01627,
    0.01881,
    0.00267,
    -0.00528,
    0.00803,
    -0.00644,
    -0.01242
   ],
   [
    -0.00465,
    -0.0027,
    0.01184,
    0.00127,
    0.00889,
    -0.00886,
    -0.00649,
    -0.00444,
    -0.00625,
    0.00089,
    0.00177,
    0.00775,
    -0.00729,
    -0.00695
   ],
   [
    -0.03027,
    -0.01224,
    -0.00838,
    0.00132,
    -0.01604,
    0.00236,
    -0.00193,
    -0.02566,
    -0.01577,
    -0.00513,
    9e-05,
    -0.01469,
    -0.00014,
    -0.00343
   ],
   [
    0.00606,
    -0.00914,
    -0.00066,
    0.00594,
    0.00129,
    -0.00455,
    0.00153,
    0.00419,
    -0.00931,
    0.00032,
    0.00369,
    0.00076,
    -0.00371,
    0.00182
   ],
   [
    -0.00557,
    -0.0136,
    -0.00724,
    0.00142,
    -0.0142,
    0.0126,
    0.00705,
    -0.00433,
    -0.01091,
    -0.00414,
    0.00286,
    -0.012,
    0.01223,
    0.00702
   ],
   [
    0.00169,
    -0.00532,
    -0.00496,
    0.01186,
    -0.0046,
    0.00285,
    -0.00756,
    0.00131,
    -0.00617,
    -0.00724,
    0.01067,
    -0.0039,
    0.0014,
    -0.00598
   ],
   [
    0.00843,
    0.0089,
    5e-05,
    -0.00722,
    0.01387,
    -0.00223,
    0.00575,
    0.00719,
    0.01012,
    0.00577,
    -0.00642,
    0.0128,
    0.00023,
    0.00554
   ],
   [
    -0.00915,
    0.01488,
    0.00519,
    0.01909,
    0.02802,
    -0.01335,
    -0.0265,
    -0.0089,
    0.00659,
    0.00203,
    0.01966,
    0.02228,
    -0.00973,
    -0.02212
   ],
   [
    -0.00085,
    0.00509,
    -0.01009,
    0.01105,
    0.00421,
    -0.01451,
    -0.00222,
    7e-05,
    0.00355,
    -0.00354,
    0.01086,
    0.00519,
    -0.01064,
    0.00048
   ],
   [
    -0.0003,
    -0.01833,
    0.00961,
    -0.00262,
    -0.00444,
    0.00416,
    -0.00321,
    -0.00061,
    -0.01647,
    0.00892,
    -0.00156,
    -0.00318,
    0.00585,
    -0.00342
   ],
   [
    -0.00725,
    0.00947,
    -0.02331,
    -0.02629,
    0.00684,
    0.00609,
    0.00728,
    -0.00857,
    0.00624,
    -0.01416,
    -0.02238,
    0.00776,
    0.00489,
    0.00618
   ],
   [
    0.00297,
    0.02378,
    -0.00293,
    -0.00656,
    0.01438,
    0.0066,
    -0.00713,
    0.00125,
    0.01593,
    -0.00149,
    -0.0046,
    0.01278,
    0.00573,
    -0.00626
   ],
   [
    -0.00067,
    -0.00612,
    0.00568,
    0.0027,
    -0.00432,
    0.00445,
    -0.00092,
    0.00115,
    2e-05,
    0.0075,
    0.00231,
    -0.00402,
    0.00532,
    -0.0007
   ],
   [
    -0.00199,
    -0.0026,
    0.01178,
    -0.00315,
    -0.00337,
    0.01161,
    0.00162,
    -0.00272,
    -0.00209,
    0.01309,
    -0.00398,
    -0.00287,
    0.01161,
    0.0026
   ],
   [
    0.01315,
    0.00382,
    -0.00928,
    -0.00284,
    0.01223,
    -0.00225,
    -0.01158,
    0.00752,
    0.00119,
    -0.00789,
    -0.00279,
    0.01291,
    -0.00094,
    -0.00832
   ],
   [
    0.00581,
    -0.0006,
    0.00589,
    -0.00901,
    -0.00221,
    -0.00476,
    -0.00088,
    0.0063,
    0.00085,
    0.00659,
    -0.00894,
    -0.00325,
    -0.00437,
    -0.00194
   ],
   [
    -0.00026,
    0.00201,
    0.00359,
    0.00203,
    0.00561,
    -0.00178,
    0.00535,
    0.00259,
    0.00161,
    0.00414,
    0.00029,
    0.00336,
    0.00098,
    0.00481
   ],
   [
    -0.00944,
    -0.00875,
    -0.00065,
    0.00204,
    0.00219,
    -0.00155,
    0.00109,
    -0.00601,
    -0.00879,
    0.00048,
    0.00118,
    0.00212,
    -0.00035,
    0.0016
   ],
   [
    0.00406,
    -0.00593,
    -0.0024,
    -0.00936,
    0.00082,
    0.0052,
    0.0022,
    0.0005,
    -0.00687,
    -0.00089,
    -0.00768,
    0.0001,
    0.0042,
    0.00144
   ],
   [
    -0.01368,
    0.00723,
    0.0115,
    -0.00577,
    -0.00273,
    0.01249,
    0.00871,
    -0.01525,
    0.00704,
    0.00955,
    -0.00474,
    -0.00466,
    0.0092,
    0.00651
   ],
   [
    0.00248,
    0.0054,
    0.00111,
    0.00389,
    0.00279,
    -0.00691,
    -0.00638,
    0.00181,
    0.00337,
    -0.00231,
    0.00389,
    0.00224,
    -0.00847,
    -0.00683
   ],
   [
    0.0011,
    0.00684,
    -0.002,
    -0.00385,
    0.00387,
    -0.00388,
    0.00292,
    0.00268,
    0.00519,
    -0.00176,
    -0.00515,
    0.00311,
    -0.00188,
    0.00251
   ],
   [
    0.00197,
    -0.01814,
    -0.00901,
    -0.0129,
    -0.00119,
    0.00629,
    0.01247,
    0.00319,
    -0.01056,
    -0.00015,
    -0.01183,
    -0.00297,
    0.00493,
    0.00994
   ],
   [
    0.00175,
    0.00122,
    0.01212,
    -0.01756,
    -0.00691,
    0.01558,
    0.00624,
    0.0012,
    0.00579,
    0.01174,
    -0.01572,
    -0.00219,
    0.01553,
    0.00738
   ],
   [
    -0.02201,
    0.00434,
    -0.00043,
    -0.01203,
    0.00691,
    0.0028,
    0.00394,
    -0.01416,
    0.00118,
    -0.00241,
    -0.01339,
    0.00512,
    0.00347,
    0.0042
   ],
   [
    -0.02213,
    -0.00259,
    -0.00705,
    0.02149,
    -0.02355,
    -0.00329,
    0.01391,
    -0.01923,
    0.00257,
    -0.0104,
    0.01837,
    -0.02169,
    -0.00347,
    0.00934
   ],
   [
    -0.01011,
    -0.01073,
    0.00326,
    -0.0086,
    -0.01279,
    -0.00281,
    0.01194,
    -0.00865,
    -0.0083,
    0.00275,
    -0.00632,
    -0.01164,
    -0.00373,
    0.00953
   ],
   [
    0.01367,
    -0.02417,
    -0.01614,
    0.00898,
    -0.01236,
    -0.0055,
    -0.00934,
    0.01042,
    -0.02321,
    -0.01324,
    0.00787,
    -0.01312,
    -0.00835,
    -0.01067
   ],
   [
    -0.01408,
    -0.01275,
    -0.00689,
    0.00742,
    5e-05,
    0.00385,
    -0.01856,
    -0.01048,
    -0.0161,
    -0.00413,
    0.00716,
    0.0024,
    0.00505,
    -0.01484
   ],
   [
    -0.00336,
    0.01348,
    -0.00031,
    0.00111,
    0.00269,
    -0.00065,
    -0.00293,
    -0.00249,
    0.01193,
    -0.00111,
    0.00071,
    0.00259,
    -0.00129,
    -0.00255
   ],
   [
    -8e-05,
    -0.00119,
    -0.0006,
    -0.00424,
    0.00422,
    0.00154,
    -0.00221,
    -0.00095,
    0.00141,
    0.00318,
    -0.00162,
    0.00619,
    0.00277,
    -0.00175
   ],
   [
    -0.01689,
    -0.01588,
    0.00621,
    0.00014,
    0.00364,
    -0.00112,
    -0.01025,
    -0.01561,
    -0.01658,
    -0.00218,
    0.0001,
    -0.0008,
    -0.00374,
    -0.01105
   ],
   [
    0.00666,
    0.03444,
    0.00148,
    -0.00743,
    0.00696,
    0.0085,
    -0.00028,
    0.00559,
    0.02435,
    0.00452,
    -0.00964,
    0.00551,
    0.00621,
    -0.00243
   ],
   [
    -0.00753,
    0.00131,
    -0.0041,
    -0.00633,
    0.00339,
    0.01378,
    -0.00716,
    -0.0086,
    -0.00042,
    -0.00021,
    -0.00409,
    0.0024,
    0.01087,
    -0.00578
   ],
   [
    0.00811,
    -0.00021,
    0.00191,
    -0.00877,
    0.00245,
    0.01116,
    0.00138,
    0.00934,
    0.00074,
    0.00414,
    -0.00781,
    0.00059,
    0.01209,
    0.00322
   ],
   [
    0.00399,
    0.0078,
    0.01052,
    -0.00901,
    0.01448,
    0.00524,
    -0.00706,
    0.00544,
    0.01401,
    0.01353,
    -0.00639,
    0.012,
    0.00681,
    -0.00538
   ],
   [
    -0.00079,
    -0.00684,
    -0.00462,
    -0.0003,
    -0.00397,
    0.00098,
    -0.01379,
    0.00151,
    -0.00652,
    -0.00449,
    -0.00251,
    -0.00624,
    -0.0007,
    -0.01262
   ],
   [
    0.01648,
    -0.00062,
    0.00529,
    -0.00261,
    0.00414,
    0.00465,
    0.0008,
    0.01484,
    0.00272,
    0.00573,
    -0.00172,
    0.00373,
    0.00599,
    0.00265
   ],
   [
    -0.00696,
    -0.00422,
    0.0021,
    0.0032,
    0.005,
    -0.00936,
    0.00191,
    -0.00516,
    0.00021,
    0.00273,
    0.00201,
    0.00406,
    -0.00997,
    -0.001
   ],
   [
    -0.00394,
    0.00572,
    -0.00217,
    0.005,
    0.00829,
    -0.00481,
    -0.00265,
    -0.00266,
    0.00183,
    -0.00692,
    0.00269,
    0.00729,
    -0.00476,
    -0.0023
   ],
   [
    0.0092,
    0.01941,
    -0.00503,
    0.003,
    0.00827,
    -0.00678,
    -0.00601,
    0.00817,
    0.0141,
    -0.00575,
    0.00166,
    0.00587,
    -0.00854,
    -0.00601
   ],
   [
    0.00228,
    -0.01951,
    0.00209,
    -0.00471,
    -0.00456,
    -0.0029,
    0.00716,
    -0.00036,
    -0.01323,
    0.00256,
    -0.00444,
    -0.00379,
    -0.00515,
    0.00395
   ],
   [
    0.00854,
    6e-05,
    0.00876,
    -0.00658,
    0.00189,
    -0.00058,
    -0.01804,
    0.00558,
    -0.00396,
    0.00304,
    -0.00712,
    0.00074,
    -0.00129,
    -0.0153
   ],
   [
    0.00497,
    -0.00773,
    -0.00134,
    -0.00403,
    -0.00102,
    0.00287,
    0.00664,
    0.00456,
    -0.00687,
    0.0008,
    -0.00416,
    -0.00143,
    0.00386,
    0.00677
   ],
   [
    0.00326,
    -0.02444,
    -0.00984,
    0.0039,
    -0.00646,
    0.0042,
    0.0055,
    0.00235,
    -0.0138,
    -0.0035,
    0.00508,
    -0.00362,
    0.0055,
    0.00718
   ],
   [
    0.02781,
    0.0011,
    0.01455,
    0.02416,
    -0.00579,
    -0.01373,
    -0.02151,
    0.02659,
    -0.00364,
    0.00709,
    0.01943,
    -0.00968,
    -0.01573,
    -0.02007
   ],
   [
    0.00039,
    0.00241,
    -0.00049,
    0.00285,
    0.00704,
    0.00157,
    -0.00299,
    -0.0013,
    0.00323,
    -0.00087,
    0.00016,
    0.0045,
    0.00022,
    -0.00322
   ],
   [
    -0.00053,
    0.01007,
    -0.01212,
    -0.00384,
    0.00601,
    -0.00131,
    0.00342,
    0.00026,
    0.0057,
    -0.01025,
    -0.00467,
    0.00703,
    -0.00059,
    0.00322
   ],
   [
    0.00134,
    -0.01102,
    0.00358,
    -0.00513,
    -0.00101,
    0.00254,
    -0.00054,
    -0.00232,
    -0.00974,
    0.00089,
    -0.00624,
    0.00014,
    0.00481,
    0.00106
   ],
   [
    -0.03367,
    0.03368,
    -0.01376,
    -0.00927,
    0.03334,
    -0.02839,
    -0.00806,
    -0.0278,
    0.02208,
    -0.01517,
    -0.01374,
    0.02423,
    -0.02879,
    -0.00916
   ],
   [
    -0.00354,
    -0.00976,
    0.0061,
    0.0001,
    0.00393,
    -0.00503,
    -0.00544,
    -0.00224,
    -0.00899,
    0.00729,
    0.00115,
    0.00368,
    -0.00148,
    -0.00234
   ],
   [
    0.00792,
    -0.00978,
    0.00016,
    0.00518,
    -0.01228,
    -0.00081,
    0.00392,
    0.00405,
    -0.00579,
    0.00034,
    0.0042,
    -0.01177,
    -0.0018,
    0.00333
   ],
   [
    -0.01199,
    0.01056,
    -0.00607,
    0.00955,
    0.00967,
    -0.01188,
    -0.01101,
    -0.01293,
    0.00407,
    -0.00827,
    0.01004,
    0.00903,
    -0.00823,
    -0.00745
   ],
   [
    -0.01841,
    0.00609,
    -0.01535,
    -0.007,
    -0.0034,
    0.00423,
    -0.00611,
    -0.01449,
    0.00255,
    -0.01156,
    -0.00865,
    -0.00412,
    0.00334,
    -0.00579
   ],
   [
    0.00924,
    0.00338,
    0.00506,
    -0.00997,
    -0.0032,
    0.0097,
    0.00223,
    0.00736,
    0.00099,
    0.00349,
    -0.01251,
    -0.00129,
    0.0092,
    0.00512
   ],
   [
    0.00119,
    -0.02677,
    -0.00325,
    -0.00879,
    -0.00314,
    0.01028,
    0.01213,
    0.00022,
    -0.0201,
    -0.00341,
    -0.00741,
    -0.00256,
    0.01088,
    0.00996
   ],
   [
    0.01019,
    -0.00207,
    0.00049,
    -0.01017,
    -0.00049,
    0.00109,
    -0.00322,
    0.01115,
    -0.00104,
    0.00346,
    -0.01165,
    1e-05,
    0.00357,
    -0.00287
   ],
   [
    0.00938,
    0.00283,
    -0.00687,
    0.0025,
    -0.00259,
    -0.00389,
    0.00482,
    0.00754,
    0.00705,
    -0.00055,
    0.0034,
    -0.0012,
    -0.00134,
    0.00515
   ],
   [
    -0.00204,
    0.0062,
    -0.00362,
    -0.00947,
    0.00236,
    0.00413,
    0.00526,
    -0.0005,
    0.00376,
    -0.00528,
    -0.00913,
    0.00276,
    0.00287,
    0.0044
   ],
   [
    0.00317,
    0.00555,
    -0.00147,
    -0.0052,
    0.0058,
    0.00028,
    0.00405,
    0.00362,
    0.00289,
    -0.00485,
    -0.0061,
    0.00424,
    -9e-05,
    0.00313
   ],
   [
    0.01668,
    -0.00021,
    0.00902,
    0.00855,
    0.00144,
    0.002,
    0.00606,
    0.01549,
    -0.00041,
    0.00995,
    0.00996,
    0.00083,
    0.00161,
    0.00614
   ],
   [
    0.02749,
    0.01097,
    0.01196,
    0.00545,
    -0.00678,
    0.01359,
    -0.00288,
    0.0221,
    0.01009,
    0.01199,
    0.00431,
    -0.00543,
    0.0139,
    -0.00063
   ],
   [
    0.01158,
    -0.01087,
    0.00347,
    0.00106,
    -0.00564,
    0.01476,
    0.00725,
    0.00877,
    -0.00345,
    0.00224,
    0.00301,
    -0.00348,
    0.01377,
    0.0065
   ],
   [
    0.00583,
    0.00339,
    0.00235,
    -0.01431,
    0.01087,
    -0.00118,
    0.00852,
    0.00726,
    0.01013,
    0.00252,
    -0.01362,
    0.01089,
    0.0008,
    0.00793
   ],
   [
    -0.01901,
    -0.01258,
    -0.0118,
    0.00658,
    0.00405,
    -0.00057,
    -0.01806,
    -0.01959,
    -0.01372,
    -0.01149,
    0.00504,
    0.00251,
    -0.00074,
    -0.01601
   ],
   [
    -0.00909,
    0.00411,
    0.00245,
    0.00432,
    0.00667,
    -0.00468,
    -0.01029,
    -0.00741,
    0.00208,
    0.00206,
    0.00383,
    0.0046,
    -0.00584,
    -0.00903
   ],
   [
    -0.02416,
    -0.00315,
    0.00173,
    -0.01046,
    0.00113,
    0.00467,
    0.00625,
    -0.02137,
    -0.00521,
    0.0019,
    -0.01446,
    -0.00206,
    0.0018,
    0.00317
   ],
   [
    -0.0181,
    -0.01357,
    0.0048,
    0.00239,
    -0.00146,
    -0.00064,
    -0.00062,
    -0.01283,
    -0.01222,
    0.00241,
    0.00335,
    -0.00222,
    -0.00156,
    0.00053
   ],
   [
    -0.00275,
    0.01424,
    0.00078,
    0.00282,
    -0.00255,
    0.00083,
    -0.0043,
    -0.00391,
    0.00867,
    0.00125,
    0.00078,
    -0.00372,
    0.00011,
    -0.00297
   ],
   [
    -0.01467,
    -0.0053,
    0.00137,
    0.00204,
    -0.00481,
    0.00568,
    0.0124,
    -0.00918,
    0.00178,
    0.00131,
    0.00287,
    -0.00211,
    0.00609,
    0.01033
   ],
   [
    0.01392,
    0.00728,
    0.00235,
    -0.00125,
    -0.00201,
    -0.00372,
    0.00202,
    0.0109,
    0.00443,
    -0.00196,
    -0.00233,
    -0.00323,
    -0.00518,
    6e-05
   ],
   [
    -0.00102,
    -0.00787,
    -0.00132,
    -0.00021,
    -0.00932,
    0.00458,
    -0.00639,
    -0.0027,
    -0.00734,
    -0.00062,
    0.00046,
    -0.00785,
    0.00382,
    -0.00535
   ],
   [
    0.01913,
    -0.00468,
    -0.00673,
    -0.00636,
    0.00955,
    0.00791,
    -0.00502,
    0.00716,
    -0.00437,
    -0.00547,
    -0.00355,
    0.00925,
    0.01132,
    -0.00092
   ],
   [
    0.02213,
    -0.01671,
    -0.02656,
    0.00967,
    -0.0388,
    0.03527,
    0.00955,
    0.02302,
    -0.00826,
    -0.00859,
    0.00898,
    -0.0293,
    0.03082,
    0.01105
   ],
   [
    -0.00722,
    0.01326,
    -0.01501,
    0.01302,
    0.00829,
    -0.00228,
    -0.01639,
    -0.00964,
    0.00936,
    -0.01257,
    0.00942,
    0.00693,
    -0.00323,
    -0.01509
   ],
   [
    0.00488,
    -0.00109,
    -0.01102,
    0.00876,
    -0.00615,
    -0.0071,
    -0.00643,
    0.00074,
    0.00101,
    -0.00541,
    0.00788,
    -0.00697,
    -0.00643,
    -0.00507
   ],
   [
    0.01557,
    0.01179,
    0.00654,
    0.00804,
    -0.00522,
    -0.00202,
    -0.00595,
    0.01159,
    0.01298,
    0.00471,
    0.00872,
    -0.00328,
    -0.00157,
    -0.00551
   ],
   [
    -0.00643,
    -0.00094,
    -0.00083,
    -0.00839,
    -0.01116,
    0.00815,
    0.00172,
    -0.00451,
    -0.00319,
    -0.00291,
    -0.00614,
    -0.00766,
    0.00601,
    0.00256
   ],
   [
    -0.03328,
    -0.00179,
    -0.01316,
    -0.01063,
    0.00692,
    -0.00376,
    0.00455,
    -0.02731,
    0.00115,
    -0.01268,
    -0.01034,
    0.00727,
    -0.00386,
    0.00373
   ],
   [
    -0.00197,
    0.00731,
    -0.00478,
    0.00288,
    0.00191,
    0.00246,
    0.0007,
    -0.00272,
    0.00587,
    -0.00227,
    0.00187,
    0.00095,
    0.00242,
    0.0004
   ],
   [
    -0.00284,
    0.00128,
    0.00253,
    0.00503,
    -0.00041,
    0.01052,
    -0.00108,
    -0.00162,
    0.00183,
    0.00392,
    0.00555,
    -0.00051,
    0.00924,
    -0.00021
   ],
   [
    0.00154,
    -0.0019,
    -0.01528,
    0.00452,
    -0.00155,
    -0.01016,
    0.00258,
    0.00347,
    0.00537,
    -0.00673,
    0.00467,
    -0.00456,
    -0.01225,
    -0.00043
   ],
   [
    0.00567,
    -0.00072,
    0.00125,
    0.00249,
    0.00365,
    0.00568,
    -0.00067,
    0.00353,
    -0.00064,
    -0.00024,
    0.00211,
    0.00263,
    0.00424,
    -0.00037
   ],
   [
    0.00272,
    -0.02105,
    -0.01136,
    0.02372,
    -0.01578,
    0.0011,
    -0.01374,
    0.00715,
    -0.01602,
    0.00094,
    0.01828,
    -0.02048,
    0.00027,
    -0.01067
   ],
   [
    0.00822,
    -0.00226,
    0.0054,
    0.00949,
    -0.00067,
    -0.00838,
    -0.0047,
    0.01176,
    -0.00278,
    0.00129,
    0.00673,
    -0.00184,
    -0.01358,
    -0.00648
   ],
   [
    -0.00259,
    0.00421,
    -0.00058,
    -0.00281,
    0.01377,
    0.00222,
    -0.00443,
    -0.00315,
    0.00279,
    0.0001,
    -0.00429,
    0.01164,
    0.00117,
    -0.00563
   ],
   [
    0.00424,
    -0.00607,
    -0.00838,
    -0.01659,
    -0.00075,
    0.0147,
    0.0243,
    -0.00074,
    0.00103,
    -0.00508,
    -0.01389,
    0.0019,
    0.0158,
    0.02038
   ],
   [
    0.01666,
    0.00531,
    0.00137,
    0.00508,
    -0.00198,
    -0.00428,
    0.001,
    0.01631,
    0.00468,
    0.00037,
    0.00683,
    -3e-05,
    -0.0034,
    0.0009
   ],
   [
    0.01891,
    0.01208,
    -0.00441,
    -0.00978,
    -0.00048,
    -0.0089,
    -0.0073,
    0.01428,
    0.00796,
    -0.00569,
    -0.01012,
    -0.0037,
    -0.00757,
    -0.00625
   ],
   [
    -0.00338,
    0.00666,
    0.0038,
    -0.00197,
    0.00039,
    -0.0019,
    -0.00187,
    -0.00554,
    0.0037,
    -0.00542,
    -0.00349,
    0.00126,
    -0.00196,
    -0.00319
   ],
   [
    -0.01324,
    -0.0231,
    -0.00932,
    0.01082,
    -0.00368,
    -0.00215,
    0.00038,
    -0.01156,
    -0.02179,
    -0.00987,
    0.00914,
    -0.00183,
    -0.00091,
    0.00212
   ],
   [
    0.00152,
    0.00069,
    -0.00097,
    -0.00449,
    0.00961,
    0.00356,
    0.00131,
    -0.00026,
    -0.0011,
    -0.00264,
    -0.00272,
    0.00886,
    0.00335,
    0.00073
   ],
   [
    0.00028,
    0.00852,
    -0.00502,
    -0.01443,
    0.00354,
    -0.00596,
    0.00838,
    -0.00315,
    0.00308,
    -0.00605,
    -0.01584,
    0.00181,
    -0.00779,
    0.00563
   ],
   [
    0.02134,
    -0.01035,
    0.00577,
    -0.00532,
    -0.00608,
    -0.00282,
    0.00375,
    0.01592,
    -0.00997,
    0.00438,
    -0.00291,
    -0.00338,
    0.00035,
    0.00699
   ],
   [
    0.01624,
    -0.01079,
    -0.01409,
    0.01831,
    -0.00481,
    -0.00694,
    -0.0007,
    0.01066,
    -0.01108,
    -0.01041,
    0.01501,
    -0.00808,
    -0.00876,
    -0.00265
   ],
   [
    -0.01147,
    -0.01701,
    0.01775,
    -0.00232,
    -0.01186,
    0.00025,
    -0.00435,
    -0.00972,
    -0.01143,
    0.01178,
    0.0016,
    -0.00714,
    0.0044,
    -0.00071
   ],
   [
    -0.01052,
    -0.00576,
    -0.00668,
    -0.00536,
    0.00611,
    -0.00221,
    0.00115,
    -0.00583,
    -0.00797,
    -0.00213,
    -0.00647,
    0.00492,
    -0.00159,
    8e-05
   ],
   [
    0.01078,
    0.00206,
    0.00208,
    0.00633,
    -0.005,
    -0.0037,
    0.00343,
    0.00828,
    0.0007,
    -0.00164,
    0.00586,
    -0.00488,
    -0.00493,
    0.00268
   ],
   [
    0.01162,
    0.03344,
    -0.0005,
    -0.00042,
    0.00848,
    0.00908,
    -0.00419,
    0.01208,
    0.02781,
    0.00403,
    -0.00069,
    0.00718,
    0.00938,
    -0.00332
   ],
   [
    0.00666,
    -0.013,
    0.00133,
    -0.00472,
    0.00155,
    -0.00031,
    0.00129,
    0.00186,
    -0.00469,
    -0.00042,
    0.00049,
    0.00149,
    -0.00202,
    0.00044
   ],
   [
    0.02095,
    0.02515,
    -0.00828,
    -0.00288,
    0.00564,
    0.00111,
    -0.00701,
    0.01479,
    0.02133,
    -0.00535,
    -0.00198,
    0.00577,
    0.00134,
    -0.00502
   ],
   [
    0.01553,
    -0.00026,
    -0.0001,
    0.00945,
    -0.00311,
    -0.00639,
    -0.00943,
    0.0139,
    0.00102,
    0.00689,
    0.00763,
    -0.00556,
    -0.00862,
    -0.01061
   ],
   [
    -0.01302,
    0.00125,
    -0.00308,
    -0.01197,
    0.00886,
    0.00671,
    0.00541,
    -0.01312,
    -0.00147,
    0.00047,
    -0.01092,
    0.00892,
    0.00605,
    0.00515
   ],
   [
    -0.0078,
    0.00734,
    -0.0027,
    -0.0043,
    0.00564,
    0.00661,
    -0.00189,
    -0.00962,
    0.00472,
    -0.00543,
    -0.00326,
    0.00494,
    0.00566,
    -0.00232
   ],
   [
    0.00302,
    -0.00707,
    -0.00688,
    0.01955,
    -0.01387,
    0.00426,
    -0.0074,
    -0.00132,
    -0.00622,
    -0.00352,
    0.0168,
    -0.01384,
    0.00215,
    -0.00577
   ],
   [
    0.0061,
    0.00423,
    0.00037,
    -0.00388,
    -0.01204,
    0.01161,
    0.00173,
    0.00396,
    0.00373,
    0.00141,
    -0.00519,
    -0.0129,
    0.00838,
    0.00152
   ],
   [
    -0.03018,
    -0.01857,
    -0.00416,
    0.00016,
    -0.00598,
    0.01097,
    -0.00239,
    -0.02397,
    -0.02039,
    -0.0051,
    -0.00041,
    -0.00572,
    0.00763,
    -0.00306
   ],
   [
    -0.02254,
    -0.01141,
    -0.00332,
    0.00753,
    -0.00813,
    -0.00258,
    -0.00956,
    -0.01734,
    -0.01106,
    -0.00495,
    0.00534,
    -0.00634,
    -0.00119,
    -0.00798
   ],
   [
    0.02143,
    0.01133,
    0.00635,
    0.00556,
    -0.00163,
    -0.00332,
    -0.01307,
    0.02367,
    0.00412,
    0.00391,
    0.00214,
    -0.00236,
    -0.005,
    -0.01239
   ],
   [
    -0.01228,
    -0.00384,
    -2e-05,
    0.00344,
    -0.00709,
    0.00529,
    -0.00252,
    -0.00875,
    -0.00279,
    -0.00105,
    0.00267,
    -0.00706,
    0.0043,
    -0.00259
   ],
   [
    -0.01193,
    -0.00489,
    0.00377,
    0.00783,
    0.00496,
    -0.01001,
    -0.00853,
    -0.00746,
    -0.00646,
    0.00159,
    0.00674,
    0.0058,
    -0.00632,
    -0.00255
   ],
   [
    -0.01295,
    0.01392,
    0.00475,
    -0.00184,
    0.02658,
    -0.00056,
    0.00446,
    -0.01095,
    0.00862,
    0.00609,
    -0.00284,
    0.0218,
    -0.00199,
    0.00251
   ],
   [
    0.00982,
    0.00629,
    0.00797,
    0.00474,
    -0.00109,
    -0.01081,
    -0.00481,
    0.00881,
    0.00629,
    0.00578,
    0.00419,
    -0.00068,
    -0.0092,
    -0.00498
   ],
   [
    -0.02533,
    -0.01705,
    -0.00816,
    0.00949,
    0.00387,
    -0.00017,
    0.00677,
    -0.02078,
    -0.01398,
    -0.00575,
    0.01042,
    0.00314,
    -0.00139,
    0.00362
   ],
   [
    -0.01442,
    0.00703,
    -0.00825,
    -0.00929,
    0.00589,
    0.00059,
    0.00756,
    -0.01183,
    0.00266,
    -0.00991,
    -0.00808,
    0.0046,
    -0.00027,
    0.00682
   ],
   [
    -0.01481,
    -0.00042,
    -0.01252,
    -0.00146,
    0.02004,
    -0.01961,
    -0.01383,
    -0.0136,
    -0.00524,
    -0.01837,
    -0.00208,
    0.01587,
    -0.02114,
    -0.01516
   ],
   [
    0.0014,
    0.0089,
    -0.00729,
    -0.00601,
    0.01127,
    -0.0238,
    7e-05,
    -0.00045,
    0.00896,
    -0.01299,
    -0.00791,
    0.0061,
    -0.02443,
    -0.00479
   ],
   [
    0.01274,
    0.01482,
    -0.00284,
    0.00492,
    0.0029,
    -0.00033,
    -0.0027,
    0.0127,
    0.01348,
    0.00023,
    0.0071,
    0.00357,
    -0.00046,
    -0.00224
   ],
   [
    0.04054,
    0.01128,
    -0.00434,
    0.00731,
    -0.01281,
    -0.00246,
    0.02201,
    0.03319,
    0.01304,
    -0.00231,
    0.00694,
    -0.01093,
    -0.00215,
    0.01854
   ],
   [
    0.00136,
    -0.0154,
    -0.00743,
    -0.00195,
    -0.00039,
    0.00629,
    -0.00907,
    -0.00321,
    -0.01484,
    -0.00434,
    4e-05,
    0.00186,
    0.0077,
    -0.00739
   ],
   [
    0.00947,
    0.01508,
    7e-05,
    0.00381,
    0.00607,
    -0.01258,
    0.00333,
    0.00838,
    0.01075,
    -0.00091,
    0.00285,
    0.00515,
    -0.01094,
    0.00223
   ],
   [
    -0.02314,
    0.00469,
    -0.00229,
    -0.01107,
    0.00133,
    0.00551,
    -0.00644,
    -0.01929,
    0.00172,
    -0.00272,
    -0.01138,
    -0.00057,
    0.00353,
    -0.00538
   ],
   [
    -0.01399,
    -0.02753,
    -0.01006,
    -0.00608,
    -0.00824,
    0.00299,
    0.00186,
    -0.01291,
    -0.0226,
    -0.00721,
    -0.00811,
    -0.00825,
    0.0007,
    -0.0004
   ],
   [
    -0.02403,
    0.02295,
    -0.00586,
    0.00108,
    0.00958,
    0.00131,
    -0.00405,
    -0.01958,
    0.01706,
    -0.00494,
    0.00208,
    0.00956,
    0.00071,
    -0.00376
   ],
   [
    -0.00181,
    0.01739,
    -0.00301,
    -0.01149,
    0.01393,
    -0.01954,
    -0.00651,
    0.00657,
    0.01143,
    0.00031,
    -0.0172,
    0.00782,
    -0.0187,
    -0.00722
   ],
   [
    -0.0106,
    -0.00581,
    -0.00432,
    0.01282,
    0.00892,
    -0.0262,
    0.00035,
    -0.00851,
    -0.00258,
    -0.00577,
    0.01161,
    0.00501,
    -0.02509,
    -0.00225
   ],
   [
    0.00305,
    0.00951,
    0.00141,
    1e-05,
    0.00312,
    0.00107,
    -0.00348,
    0.0009,
    0.00584,
    0.00153,
    -0.00042,
    0.00391,
    0.00196,
    -0.00357
   ],
   [
    0.0049,
    0.01979,
    -0.00587,
    -0.01786,
    0.00888,
    -0.00381,
    0.01127,
    0.00472,
    0.02215,
    -0.00452,
    -0.01853,
    0.00763,
    -0.0036,
    0.00977
   ],
   [
    0.00274,
    0.0077,
    -0.0116,
    0.01688,
    0.00513,
    -0.0137,
    -0.01637,
    0.0007,
    0.00028,
    -0.00774,
    0.01216,
    0.00279,
    -0.01121,
    -0.01312
   ],
   [
    -0.00682,
    0.00277,
    -0.00668,
    0.00141,
    -0.00301,
    0.00346,
    -0.00221,
    -0.00642,
    -0.00069,
    -0.00583,
    0.00051,
    -0.00351,
    0.00275,
    -0.00244
   ],
   [
    0.00147,
    0.00543,
    -0.00541,
    0.00059,
    -0.00457,
    0.01164,
    -0.00148,
    0.00175,
    0.00405,
    -0.00548,
    0.00124,
    -0.00274,
    0.01076,
    -0.00031
   ],
   [
    0.01767,
    -0.00666,
    0.02015,
    -0.01075,
    -0.00405,
    0.00672,
    0.01742,
    0.01572,
    -0.00112,
    0.02097,
    -0.00943,
    -0.00336,
    0.00836,
    0.01541
   ],
   [
    -0.00588,
    -0.0072,
    -0.00287,
    -0.0026,
    -0.00591,
    9e-05,
    0.00064,
    -0.00353,
    -0.00918,
    0.00036,
    -0.00419,
    -0.00617,
    0.00205,
    0.00172
   ],
   [
    0.02159,
    0.01581,
    -0.0099,
    0.00341,
    -0.00209,
    0.00971,
    -0.00034,
    0.01834,
    0.01185,
    -0.00413,
    0.00383,
    0.00218,
    0.01407,
    0.00307
   ],
   [
    0.00222,
    0.00807,
    0.00569,
    -0.00738,
    0.00584,
    0.00555,
    -0.00527,
    0.00317,
    0.00772,
    0.00531,
    -0.00634,
    0.00346,
    0.00553,
    -0.00348
   ],
   [
    -0.00493,
    0.01081,
    0.00181,
    6e-05,
    0.0023,
    0.00096,
    -0.01248,
    -0.00209,
    0.00274,
    -0.00141,
    -0.00037,
    0.00269,
    0.00123,
    -0.00996
   ],
   [
    0.01304,
    0.00475,
    -0.00682,
    0.00274,
    0.00436,
    -0.00673,
    -0.01186,
    0.0108,
    0.00433,
    -0.00464,
    0.00255,
    0.00377,
    -0.00589,
    -0.01058
   ],
   [
    -0.00191,
    -0.0245,
    0.01509,
    0.00935,
    -0.02134,
    0.00435,
    -0.00341,
    0.00272,
    -0.01999,
    0.00858,
    0.00818,
    -0.02009,
    -0.00019,
    -0.00185
   ],
   [
    -0.02453,
    0.0151,
    -0.0051,
    -0.00059,
    0.00708,
    0.00141,
    -0.01319,
    -0.02,
    0.00636,
    -0.0042,
    0.00024,
    0.00557,
    0.00136,
    -0.01019
   ],
   [
    0.00265,
    -0.00652,
    0.00465,
    0.00871,
    -0.013,
    0.00458,
    0.00709,
    0.00313,
    -0.00119,
    0.0032,
    0.00747,
    -0.01174,
    0.00436,
    0.00647
   ],
   [
    -0.00836,
    0.00055,
    -0.01331,
    -0.01802,
    -0.00323,
    0.00347,
    0.02667,
    -0.00656,
    0.00372,
    -0.00556,
    -0.01609,
    -0.00096,
    0.00319,
    0.0226
   ],
   [
    0.00692,
    0.00054,
    -0.01143,
    0.01278,
    0.00069,
    -0.00927,
    -0.01458,
    0.00338,
    -0.00175,
    -0.00724,
    0.01331,
    0.00036,
    -0.00822,
    -0.01256
   ],
   [
    -0.00031,
    0.00226,
    0.00396,
    0.00367,
    -0.00554,
    -0.0115,
    -0.00364,
    0.00418,
    0.00155,
    -0.00321,
    0.004,
    -0.00465,
    -0.01162,
    -0.0039
   ],
   [
    0.00106,
    -0.00022,
    -0.00323,
    0.00311,
    0.00222,
    -0.00059,
    -0.00902,
    0.00067,
    -0.00256,
    -0.00445,
    0.00417,
    0.00093,
    -0.00461,
    -0.00916
   ],
   [
    0.02063,
    -0.00279,
    0.00305,
    0.01156,
    -0.01136,
    0.00371,
    -0.0114,
    0.01919,
    -0.00113,
    0.01077,
    0.01109,
    -0.00563,
    0.00895,
    -0.00494
   ],
   [
    -0.00976,
    -0.00091,
    -0.01055,
    0.00653,
    0.00068,
    -0.01461,
    -0.0032,
    -0.00771,
    -0.00302,
    -0.00591,
    0.0066,
    0.00139,
    -0.01221,
    -0.00359
   ],
   [
    0.00563,
    0.00095,
    -0.00046,
    0.00022,
    -0.0078,
    0.00703,
    -0.00507,
    0.00388,
    -0.0001,
    -0.00011,
    -0.00195,
    -0.00801,
    0.00657,
    -0.00373
   ],
   [
    -0.01276,
    -0.00986,
    0.00276,
    0.00854,
    -0.00236,
    -0.00097,
    0.00167,
    -0.01012,
    -0.01066,
    -0.00101,
    0.00769,
    -0.0019,
    -0.00198,
    0.00223
   ],
   [
    -0.00862,
    0.0031,
    -0.00298,
    -0.00194,
    -0.00138,
    -0.00157,
    -0.01025,
    -0.00733,
    0.00104,
    -0.00713,
    -0.00254,
    8e-05,
    -0.00041,
    -0.00647
   ],
   [
    -0.00861,
    -0.00362,
    -0.01773,
    0.0034,
    -0.00023,
    -0.00952,
    0.00323,
    -0.00831,
    -0.00368,
    -0.01371,
    0.00336,
    -0.00056,
    -0.00833,
    0.00207
   ],
   [
    -0.00551,
    0.00093,
    -0.0006,
    -0.00598,
    0.00567,
    -0.00312,
    0.00345,
    -0.00749,
    -0.00187,
    -0.00606,
    -0.00525,
    0.00608,
    -0.0038,
    0.00234
   ],
   [
    -0.00773,
    -0.02383,
    -0.01402,
    -0.00148,
    -0.00289,
    5e-05,
    -0.00797,
    -0.00897,
    -0.02078,
    -0.01011,
    -0.00541,
    -0.0065,
    -0.00455,
    -0.00859
   ],
   [
    0.00026,
    -0.00406,
    0.00629,
    -0.0,
    -0.00151,
    0.01272,
    -0.00073,
    0.00122,
    -0.00148,
    0.00205,
    0.00179,
    -0.00034,
    0.01256,
    0.00067
   ],
   [
    -0.00249,
    -0.00022,
    -0.00729,
    -0.0052,
    0.00562,
    -0.00414,
    0.00169,
    -0.00147,
    -0.00207,
    -0.00735,
    -0.00505,
    0.00529,
    -0.00339,
    0.00339
   ],
   [
    -0.01342,
    0.00132,
    0.00282,
    -0.00353,
    -0.00208,
    -0.00122,
    0.001,
    -0.00856,
    -0.00259,
    -0.00116,
    -0.00366,
    -0.00258,
    -0.00028,
    0.002
   ],
   [
    -0.00529,
    -0.00725,
    -0.00206,
    0.0013,
    -0.00405,
    -0.00018,
    0.00178,
    -0.00432,
    -0.00613,
    -0.00102,
    0.0011,
    -0.0036,
    -0.00158,
    0.00095
   ],
   [
    -0.00398,
    0.00501,
    -0.00387,
    -0.01969,
    0.00905,
    0.01462,
    0.00473,
    -0.00288,
    0.0026,
    0.00165,
    -0.01778,
    0.00649,
    0.01386,
    0.00461
   ],
   [
    -0.00173,
    -0.0153,
    -0.00199,
    -0.0011,
    -0.00033,
    0.00089,
    -0.00257,
    -0.00307,
    -0.01751,
    -0.00592,
    -0.0037,
    -0.00113,
    0.00026,
    -0.00164
   ],
   [
    0.00532,
    0.01696,
    -0.00436,
    0.00259,
    -0.00129,
    -0.00318,
    0.02452,
    0.0089,
    0.0138,
    -0.00059,
    0.00188,
    -0.00309,
    -0.00333,
    0.02279
   ],
   [
    0.00623,
    0.01862,
    0.00173,
    0.00579,
    0.00808,
    -0.00501,
    -0.00912,
    0.00672,
    0.01676,
    -0.00108,
    0.00536,
    0.00752,
    -0.00407,
    -0.00768
   ],
   [
    0.0198,
    -0.00697,
    0.00439,
    0.02527,
    -0.00987,
    -0.00736,
    -0.01105,
    0.01575,
    -0.00504,
    0.00215,
    0.02375,
    -0.00437,
    -0.00359,
    -0.00823
   ],
   [
    0.0079,
    0.01577,
    0.00384,
    0.00245,
    0.00539,
    -0.00057,
    0.00591,
    0.00893,
    0.01418,
    0.00362,
    0.00108,
    0.00401,
    -0.00384,
    0.00219
   ],
   [
    0.01745,
    -0.00084,
    0.0089,
    -7e-05,
    -0.00855,
    0.01769,
    0.01196,
    0.01455,
    -0.00071,
    0.00678,
    0.00102,
    -0.00573,
    0.01762,
    0.01332
   ],
   [
    0.02841,
    0.01686,
    0.01863,
    0.00355,
    0.00044,
    0.00432,
    1e-05,
    0.02563,
    0.01378,
    0.01556,
    0.00627,
    0.00285,
    0.0076,
    0.00417
   ],
   [
    -0.01656,
    0.00325,
    -0.00583,
    0.00713,
    0.00455,
    -0.00979,
    -0.01675,
    -0.01346,
    0.00538,
    -0.00758,
    0.00726,
    0.0039,
    -0.00501,
    -0.01181
   ],
   [
    0.0202,
    -0.00487,
    0.00608,
    0.00306,
    -0.01398,
    0.01224,
    -0.00835,
    0.0159,
    -0.00237,
    0.0061,
    0.00637,
    -0.00992,
    0.01382,
    -0.00455
   ],
   [
    -0.00343,
    0.00348,
    0.00447,
    -0.00423,
    0.00542,
    -0.00068,
    0.00064,
    -0.00451,
    0.00281,
    0.00249,
    -0.0045,
    0.00374,
    -0.00437,
    -0.00104
   ],
   [
    -0.0231,
    -0.01615,
    -0.00107,
    0.01022,
    -0.00288,
    -0.00259,
    -0.00032,
    -0.01816,
    -0.01129,
    -0.00125,
    0.00989,
    -0.00182,
    -0.00117,
    0.00072
   ],
   [
    0.0063,
    0.01404,
    -0.00144,
    0.00918,
    0.00789,
    -0.01793,
    -0.00773,
    0.00359,
    0.00614,
    -0.00079,
    0.0082,
    0.0056,
    -0.01702,
    -0.00638
   ],
   [
    0.00872,
    -0.00229,
    0.00013,
    0.0004,
    -0.00191,
    -0.00319,
    -0.00032,
    0.00581,
    0.00099,
    0.00109,
    3e-05,
    -0.00292,
    -0.00303,
    -0.00051
   ],
   [
    0.01086,
    -0.01568,
    0.00533,
    0.00793,
    -0.00385,
    -0.00803,
    -0.00207,
    0.00789,
    -0.01585,
    -0.00148,
    0.00532,
    -0.00525,
    -0.00937,
    -0.00319
   ],
   [
    -0.00485,
    -0.00439,
    0.00709,
    -0.0081,
    0.0039,
    -0.00384,
    0.00027,
    -0.00261,
    -9e-05,
    0.00751,
    -0.00696,
    0.00421,
    -0.0012,
    0.002
   ],
   [
    -0.01111,
    -0.01098,
    0.00113,
    0.01008,
    -0.008,
    0.00471,
    -0.00638,
    -0.01082,
    -0.0102,
    0.00404,
    0.01078,
    -0.0062,
    0.00105,
    -0.00603
   ],
   [
    -0.00395,
    0.0009,
    0.00471,
    0.00269,
    -0.01622,
    0.00328,
    -0.00597,
    -0.00072,
    -0.00381,
    0.00018,
    0.00299,
    -0.01084,
    0.00559,
    -0.0038
   ],
   [
    0.00993,
    0.01055,
    0.0038,
    -0.00585,
    -0.00175,
    -0.00013,
    -0.00169,
    0.0102,
    0.00876,
    0.00251,
    -0.00552,
    -0.00242,
    -0.00185,
    -0.00317
   ],
   [
    0.01856,
    -0.02823,
    -0.00367,
    0.0106,
    -0.0431,
    0.00587,
    -0.00739,
    0.01389,
    -0.02675,
    -0.00642,
    0.00932,
    -0.0375,
    0.00307,
    -0.0091
   ],
   [
    -0.01624,
    -0.00474,
    0.00465,
    0.00186,
    0.00014,
    -0.00405,
    -0.00815,
    -0.01064,
    -0.00764,
    0.00108,
    0.00053,
    0.00014,
    -0.00103,
    -0.00802
   ],
   [
    -0.00742,
    0.00239,
    -0.00667,
    -0.0039,
    0.00427,
    -8e-05,
    -0.00556,
    -0.00681,
    1e-05,
    -0.00648,
    -0.00353,
    0.00284,
    -0.00189,
    -0.00501
   ],
   [
    0.01491,
    0.00264,
    0.01075,
    0.00532,
    -0.00886,
    -0.00478,
    -0.00034,
    0.01134,
    0.00539,
    0.00248,
    0.00528,
    -0.0065,
    -0.00477,
    -0.00045
   ],
   [
    -0.00013,
    -0.00711,
    -0.00471,
    0.0141,
    -0.00418,
    -0.0094,
    -0.00868,
    -0.00124,
    -0.00599,
    -0.00251,
    0.01289,
    -0.00317,
    -0.00712,
    -0.00713
   ],
   [
    -0.03752,
    0.00122,
    -0.00039,
    -0.00964,
    0.01719,
    -0.01162,
    0.00441,
    -0.03512,
    -0.00062,
    -0.00153,
    -0.00673,
    0.01433,
    -0.01067,
    0.00286
   ],
   [
    -0.00133,
    -0.01508,
    -0.00465,
    -0.01434,
    0.00536,
    0.00013,
    0.0107,
    -0.00211,
    -0.0109,
    -0.00498,
    -0.01444,
    0.00294,
    -0.00024,
    0.00805
   ],
   [
    -0.00078,
    0.02901,
    0.00309,
    0.00979,
    0.01159,
    0.00302,
    -0.01268,
    -0.00116,
    0.0221,
    0.00694,
    0.01134,
    0.01127,
    0.00624,
    -0.00727
   ],
   [
    0.01202,
    0.0121,
    -0.00064,
    0.00146,
    0.00434,
    -0.00736,
    0.00234,
    0.00913,
    0.00892,
    -0.0019,
    0.00174,
    0.00392,
    -0.0073,
    0.00053
   ],
   [
    -0.00541,
    0.02709,
    0.0089,
    -0.00398,
    0.01052,
    -0.0038,
    -0.00946,
    -0.00496,
    0.01793,
    0.00637,
    -0.0078,
    0.00475,
    -0.00936,
    -0.01139
   ],
   [
    0.01445,
    -0.00496,
    -0.00044,
    0.00235,
    -0.00778,
    0.00336,
    0.00232,
    0.01056,
    -0.00535,
    0.00155,
    7e-05,
    -0.00768,
    0.0014,
    0.00036
   ],
   [
    -0.00657,
    0.0132,
    -0.00542,
    -0.00595,
    0.01086,
    -0.00417,
    -8e-05,
    -0.00699,
    0.00825,
    -0.00359,
    -0.00613,
    0.00928,
    -0.00495,
    -0.00035
   ],
   [
    -0.01035,
    0.01377,
    -0.00551,
    -0.00294,
    0.00815,
    0.00594,
    0.01087,
    -0.00786,
    0.01314,
    -0.00296,
    -0.00312,
    0.00792,
    0.00708,
    0.01142
   ],
   [
    0.00151,
    0.01202,
    -0.01314,
    0.00757,
    -0.00601,
    0.00324,
    -0.01794,
    0.00121,
    0.0126,
    -0.01072,
    0.00735,
    -0.0054,
    0.00257,
    -0.01571
   ],
   [
    0.02025,
    0.02464,
    0.00549,
    -0.00726,
    0.00889,
    -0.00693,
    0.01448,
    0.01971,
    0.02041,
    0.00411,
    -0.00631,
    0.00622,
    -0.00534,
    0.01375
   ],
   [
    0.00273,
    -0.00232,
    0.00284,
    0.00164,
    -0.00593,
    -0.00194,
    0.00113,
    0.00015,
    -0.00217,
    0.00434,
    -0.0005,
    -0.00608,
    -0.00223,
    -0.0005
   ],
   [
    -0.00505,
    -0.01839,
    -0.00339,
    0.00946,
    -0.00415,
    0.00029,
    0.00686,
    -0.00468,
    -0.01436,
    0.00257,
    0.008,
    -0.00284,
    0.00135,
    0.00591
   ],
   [
    0.00991,
    -0.01514,
    0.00427,
    0.00193,
    -0.00323,
    0.00522,
    -0.00422,
    0.01021,
    -0.01327,
    0.00509,
    0.00013,
    -0.00275,
    0.00504,
    -0.00462
   ],
   [
    -0.01218,
    -0.00975,
    -0.00934,
    0.0087,
    -0.00452,
    -0.00808,
    -0.01057,
    -0.01151,
    -0.0123,
    -0.00833,
    0.00654,
    -0.00621,
    -0.00972,
    -0.01018
   ],
   [
    -0.01743,
    0.00143,
    0.00412,
    -0.00232,
    -0.00414,
    0.0056,
    0.0052,
    -0.0144,
    -0.00094,
    0.0047,
    -0.00339,
    -0.00535,
    0.00412,
    0.0048
   ],
   [
    0.02448,
    -0.02594,
    0.01301,
    0.01828,
    -0.00778,
    0.00047,
    -0.01984,
    0.01709,
    -0.02004,
    0.00906,
    0.01972,
    -0.00747,
    -7e-05,
    -0.01683
   ],
   [
    -0.03981,
    0.07785,
    -0.00426,
    -0.0198,
    0.03258,
    -0.00935,
    0.04369,
    -0.03861,
    0.08695,
    0.01922,
    -0.00681,
    0.04012,
    0.00795,
    0.04834
   ],
   [
    0.01227,
    -0.00497,
    0.01116,
    -0.00892,
    -0.00424,
    0.00483,
    0.0074,
    0.00956,
    -0.00326,
    0.01049,
    -0.00758,
    -0.00289,
    0.00433,
    0.00628
   ],
   [
    0.0415,
    0.02701,
    -0.0007,
    -0.00152,
    0.00902,
    -0.00274,
    -0.00354,
    0.03373,
    0.025,
    0.00155,
    -0.0021,
    0.00818,
    -0.00138,
    -0.00131
   ],
   [
    0.0079,
    -0.01344,
    0.00062,
    -0.00598,
    -0.01614,
    -0.00921,
    0.00985,
    0.00944,
    -0.01036,
    0.00291,
    -0.00891,
    -0.01188,
    -0.00245,
    0.01003
   ],
   [
    0.00831,
    -0.01456,
    0.00772,
    0.00844,
    -0.00871,
    -0.00694,
    -0.00077,
    0.00605,
    -0.01247,
    0.0041,
    0.00352,
    -0.01107,
    -0.00771,
    -0.00252
   ],
   [
    -0.01163,
    0.00759,
    -0.00267,
    -0.00271,
    0.01487,
    -0.01155,
    0.004,
    -0.00971,
    0.00623,
    -0.00319,
    -0.00294,
    0.01266,
    -0.01475,
    -0.00083
   ],
   [
    0.01332,
    0.00691,
    0.00854,
    0.0036,
    -0.00655,
    0.0011,
    0.00298,
    0.0119,
    0.00512,
    0.00602,
    0.00474,
    -0.00408,
    0.00228,
    0.0027
   ],
   [
    -0.00418,
    0.01356,
    0.01787,
    -0.03674,
    0.01075,
    0.01896,
    0.00167,
    -0.00142,
    0.0137,
    0.01282,
    -0.03282,
    0.01047,
    0.01898,
    0.00175
   ],
   [
    0.00224,
    0.00965,
    0.00089,
    -0.00312,
    0.00208,
    0.00813,
    0.00796,
    0.00156,
    0.00935,
    -0.00245,
    -0.00127,
    0.00197,
    0.00669,
    0.00504
   ],
   [
    -0.00136,
    0.01634,
    0.0016,
    -0.0018,
    0.00415,
    0.00396,
    0.0103,
    -0.00051,
    0.01264,
    0.00312,
    -0.00062,
    0.0059,
    0.00346,
    0.00848
   ],
   [
    0.00245,
    0.00924,
    -0.00441,
    -0.00981,
    -0.00402,
    -0.00157,
    0.00306,
    -0.00053,
    0.00636,
    -0.0059,
    -0.01075,
    -0.00543,
    -0.00104,
    0.00306
   ],
   [
    -0.01172,
    0.0217,
    -0.00345,
    -0.00015,
    0.00646,
    0.00157,
    0.00578,
    -0.0085,
    0.01719,
    -0.00185,
    0.0006,
    0.00801,
    -6e-05,
    0.0043
   ],
   [
    -0.00286,
    0.00686,
    -0.00947,
    0.00361,
    0.00609,
    0.00759,
    -0.00878,
    -0.00312,
    0.00743,
    -0.00328,
    0.00157,
    0.00373,
    0.00361,
    -0.0095
   ],
   [
    0.00667,
    0.00295,
    -0.00132,
    -0.0006,
    0.00108,
    -0.00366,
    -0.00423,
    0.00387,
    0.00145,
    -0.00342,
    0.00033,
    0.00056,
    -0.00544,
    -0.00441
   ],
   [
    -0.00143,
    -0.00701,
    0.00426,
    0.00423,
    0.00197,
    0.00061,
    0.00226,
    -0.00358,
    -0.0051,
    0.00525,
    0.00429,
    0.00139,
    0.00153,
    0.00265
   ],
   [
    -0.01339,
    -0.01572,
    0.00129,
    -0.00154,
    -0.00459,
    0.00271,
    -0.00169,
    -0.00939,
    -0.01185,
    0.00298,
    -0.00094,
    -0.00285,
    0.00649,
    0.00131
   ],
   [
    0.00493,
    -0.00167,
    0.00984,
    0.00478,
    -0.00468,
    0.00086,
    -0.00767,
    0.00592,
    0.00095,
    0.00682,
    0.0054,
    -0.00436,
    0.00163,
    -0.00445
   ],
   [
    0.00705,
    -0.00248,
    -0.00537,
    0.007,
    0.00021,
    -0.00301,
    -0.00548,
    0.00139,
    -0.0011,
    -0.00541,
    0.00844,
    0.00106,
    -0.0032,
    -0.00489
   ],
   [
    -0.00205,
    0.02043,
    0.01807,
    -0.00966,
    0.01618,
    -0.00425,
    0.01361,
    -0.00165,
    0.01397,
    0.01049,
    -0.00872,
    0.01742,
    -0.00193,
    0.01345
   ],
   [
    -0.00547,
    0.0043,
    -0.00449,
    0.00268,
    -0.00692,
    0.01568,
    0.00706,
    -0.00679,
    0.00468,
    0.00015,
    0.0034,
    -0.00345,
    0.01566,
    0.00689
   ],
   [
    0.01665,
    0.00332,
    -0.00325,
    0.00684,
    -0.01398,
    0.01738,
    0.00173,
    0.01454,
    -0.00012,
    -0.00632,
    0.00582,
    -0.00902,
    0.01622,
    0.00257
   ],
   [
    -0.00088,
    -0.00227,
    0.00131,
    -0.00305,
    0.0016,
    -0.00391,
    0.0094,
    -0.00412,
    -0.00077,
    0.00337,
    -0.00229,
    0.00201,
    -0.00328,
    0.00801
   ],
   [
    -0.0065,
    0.00206,
    -0.00466,
    0.00414,
    0.01463,
    -0.02277,
    0.01049,
    -0.00589,
    0.00133,
    -0.00438,
    0.00358,
    0.01136,
    -0.02066,
    0.00726
   ],
   [
    0.01335,
    0.01544,
    0.00459,
    0.00429,
    0.00526,
    -0.00819,
    -0.00398,
    0.01267,
    0.01347,
    0.00438,
    0.00406,
    0.00499,
    -0.00714,
    -0.00329
   ],
   [
    0.01775,
    0.01625,
    -0.00097,
    -0.00047,
    -0.00307,
    -0.00473,
    0.00065,
    0.01401,
    0.01573,
    -0.00157,
    0.00014,
    -0.00051,
    -0.0037,
    0.00075
   ],
   [
    -0.01802,
    -0.01365,
    -0.00162,
    -0.00382,
    0.00703,
    -0.00121,
    0.00299,
    -0.01873,
    -0.013,
    0.00133,
    -0.00126,
    0.00594,
    -0.00108,
    0.00281
   ],
   [
    0.00224,
    -0.0052,
    0.00152,
    -0.00672,
    0.00135,
    0.00856,
    0.01327,
    0.00027,
    -0.00093,
    0.00342,
    -0.00391,
    0.00216,
    0.00797,
    0.01162
   ],
   [
    0.00293,
    -0.01239,
    -0.00252,
    0.0154,
    -0.00634,
    -0.00081,
    -0.00127,
    0.00061,
    -0.01249,
    0.00169,
    0.01548,
    -0.00433,
    0.00035,
    0.00081
   ],
   [
    -0.00856,
    -0.00726,
    -0.00346,
    0.00154,
    -0.00153,
    0.00399,
    -0.00198,
    -0.00982,
    -0.00708,
    -0.00427,
    0.00063,
    -0.00356,
    0.0014,
    -0.00311
   ],
   [
    0.00966,
    0.01247,
    0.00489,
    -0.00131,
    0.00518,
    -0.00977,
    -0.00331,
    0.00948,
    0.01462,
    0.00925,
    0.00129,
    0.00605,
    -0.00541,
    -0.00274
   ],
   [
    0.01036,
    0.00787,
    -0.00329,
    0.00396,
    0.00451,
    -0.00475,
    -0.00146,
    0.00876,
    0.00858,
    -0.00336,
    0.00483,
    0.00441,
    -0.00547,
    -0.00306
   ],
   [
    -0.00243,
    0.00357,
    -0.00617,
    0.0006,
    0.00492,
    -0.00081,
    -0.00291,
    -0.00186,
    0.00199,
    -0.00563,
    -0.00212,
    0.00175,
    -0.00118,
    -0.00361
   ],
   [
    -0.01615,
    -0.00213,
    -0.00091,
    -0.00359,
    0.00056,
    -0.00081,
    0.00387,
    -0.01106,
    -0.00349,
    -0.00296,
    -0.00308,
    0.00086,
    -0.00022,
    0.00408
   ],
   [
    -0.01088,
    0.0111,
    0.00198,
    -0.01909,
    0.00581,
    0.01765,
    -0.00224,
    -0.0086,
    0.00972,
    0.0034,
    -0.02178,
    0.00564,
    0.01373,
    -0.00311
   ],
   [
    -0.00126,
    0.03494,
    -0.00742,
    0.0057,
    0.0036,
    -0.00403,
    0.00155,
    0.00273,
    0.0327,
    -0.00461,
    0.0035,
    0.00359,
    -0.00484,
    0.00011
   ],
   [
    0.01743,
    -0.00356,
    0.00355,
    0.01029,
    -0.00437,
    0.00444,
    -0.00269,
    0.01622,
    -0.00598,
    -0.00757,
    0.00748,
    -0.00565,
    0.00198,
    -0.00491
   ],
   [
    -0.03813,
    0.02052,
    -0.01659,
    0.0139,
    -0.03209,
    0.01049,
    0.00865,
    -0.03598,
    0.01508,
    0.00194,
    0.01862,
    -0.02487,
    0.00971,
    0.01031
   ],
   [
    -0.01893,
    0.00553,
    -0.00489,
    -0.00766,
    0.00167,
    -0.00279,
    0.00139,
    -0.01188,
    0.00514,
    -0.00221,
    -0.00326,
    0.0056,
    0.00131,
    0.00323
   ],
   [
    0.00303,
    0.00303,
    -0.00308,
    0.00472,
    0.00537,
    0.00241,
    -0.00313,
    0.00319,
    0.00278,
    -0.00129,
    0.00489,
    0.00758,
    0.00275,
    -0.00051
   ],
   [
    0.01168,
    0.01938,
    -0.00315,
    7e-05,
    0.01072,
    -0.02197,
    0.01612,
    0.00921,
    0.01856,
    -0.00297,
    -0.00306,
    0.0077,
    -0.01859,
    0.0136
   ],
   [
    0.0001,
    0.00349,
    0.00823,
    0.00173,
    -0.00372,
    0.01004,
    0.0008,
    0.00268,
    0.00439,
    0.01091,
    0.00129,
    -0.00293,
    0.00878,
    0.00091
   ],
   [
    0.00418,
    -0.00365,
    0.00264,
    0.02924,
    -0.00171,
    -0.01324,
    -0.02202,
    0.00255,
    -0.00419,
    -0.0011,
    0.02909,
    0.00174,
    -0.0093,
    -0.01685
   ],
   [
    -0.0106,
    0.00995,
    -0.00303,
    -0.00974,
    0.00509,
    0.00611,
    -0.00241,
    -0.00914,
    0.00945,
    -0.00123,
    -0.00887,
    0.00568,
    0.00483,
    -0.00239
   ],
   [
    -0.00519,
    -0.00093,
    0.00536,
    -0.00492,
    0.0106,
    0.00326,
    0.00749,
    -0.00163,
    -0.00014,
    0.00929,
    -0.00427,
    0.01074,
    0.00312,
    0.00681
   ],
   [
    0.02183,
    -0.01362,
    0.00881,
    -0.01141,
    -0.00119,
    0.01746,
    -0.00599,
    0.01833,
    -0.01074,
    0.01126,
    -0.00776,
    -0.00041,
    0.01725,
    -0.00181
   ],
   [
    0.01445,
    0.02231,
    -0.00356,
    -0.00878,
    0.00813,
    0.00102,
    0.00662,
    0.01478,
    0.01726,
    -0.00782,
    -0.00772,
    0.00675,
    0.00021,
    0.00475
   ],
   [
    -0.00523,
    -0.00518,
    -0.01563,
    0.01743,
    -0.01137,
    0.00133,
    -0.0115,
    -0.00822,
    -0.00601,
    -0.01337,
    0.01553,
    -0.01143,
    -5e-05,
    -0.00976
   ],
   [
    0.00671,
    -0.01211,
    -0.00209,
    -0.00018,
    -0.00206,
    -0.0102,
    0.00702,
    0.00597,
    -0.00345,
    -0.00195,
    0.00027,
    -0.00256,
    -0.01127,
    0.00413
   ],
   [
    -0.00728,
    -0.00834,
    -0.00572,
    0.01257,
    0.00524,
    -0.01441,
    -0.01303,
    -0.00817,
    -0.01224,
    -0.00084,
    0.00948,
    0.0012,
    -0.01386,
    -0.00995
   ],
   [
    -0.00088,
    0.00352,
    -0.00596,
    0.01604,
    -0.00692,
    -0.00135,
    5e-05,
    0.00117,
    0.0006,
    -0.00315,
    0.01332,
    -0.00777,
    -0.00275,
    0.00196
   ],
   [
    0.01349,
    0.00037,
    -0.00373,
    0.00079,
    0.00211,
    -0.00163,
    0.00874,
    0.01157,
    -0.00114,
    -0.00429,
    0.00196,
    0.00307,
    -0.00125,
    0.00697
   ],
   [
    -0.00067,
    -0.0032,
    -0.01173,
    -0.00686,
    -0.0032,
    0.00554,
    0.00066,
    0.00085,
    -0.00303,
    -0.00826,
    -0.00689,
    -0.00325,
    0.00427,
    -0.00033
   ],
   [
    0.00686,
    0.02065,
    0.00666,
    -0.00737,
    0.01469,
    0.00366,
    -0.0163,
    0.00498,
    0.02053,
    0.00188,
    -0.01027,
    0.01209,
    0.00365,
    -0.01246
   ],
   [
    0.00091,
    -0.00932,
    -0.00224,
    0.00337,
    -0.0086,
    -0.00432,
    -0.00143,
    8e-05,
    -0.00629,
    -0.00443,
    0.00395,
    -0.00637,
    -0.00348,
    -0.00072
   ],
   [
    -0.00656,
    -0.01052,
    0.00534,
    -0.00033,
    -0.00199,
    -0.00408,
    0.01276,
    -0.00483,
    -0.00906,
    0.00125,
    0.00045,
    -0.00014,
    -0.00171,
    0.01089
   ],
   [
    0.0101,
    -0.00437,
    -0.00013,
    -0.00037,
    -0.00719,
    -0.00348,
    0.00329,
    0.00586,
    -0.00547,
    -0.00197,
    -0.00187,
    -0.00724,
    -0.00402,
    0.00238
   ],
   [
    -0.01689,
    0.00288,
    -0.00206,
    -0.00499,
    0.00299,
    0.00576,
    0.00624,
    -0.01324,
    0.00247,
    -0.00316,
    -0.00744,
    0.00161,
    0.00544,
    0.00485
   ],
   [
    0.00854,
    -0.0001,
    0.00766,
    -0.00679,
    -0.00207,
    -0.00564,
    0.00048,
    0.01004,
    0.00096,
    0.0075,
    -0.00341,
    -0.00096,
    -0.00394,
    0.00072
   ],
   [
    -0.00085,
    0.01139,
    -0.00311,
    0.00608,
    -0.00139,
    0.00784,
    -0.01499,
    0.00339,
    0.00545,
    -0.00079,
    0.00474,
    -0.00203,
    0.00661,
    -0.01227
   ],
   [
    0.00588,
    -0.00949,
    0.00035,
    -0.00284,
    -0.0155,
    0.01847,
    0.00768,
    0.00628,
    -0.00062,
    0.00546,
    0.00059,
    -0.01147,
    0.01952,
    0.01121
   ],
   [
    0.00236,
    0.01252,
    0.00185,
    0.00536,
    0.00173,
    0.00489,
    -0.01012,
    0.00337,
    0.01171,
    0.00388,
    0.00439,
    0.00192,
    0.00517,
    -0.0078
   ],
   [
    0.02157,
    0.00501,
    0.01133,
    0.00816,
    -0.01774,
    0.00216,
    -0.00083,
    0.0181,
    0.00325,
    0.00502,
    0.00283,
    -0.01692,
    0.00062,
    -0.00114
   ],
   [
    0.01778,
    -0.00127,
    -0.00508,
    -0.00013,
    -0.01173,
    0.01132,
    0.00956,
    0.01448,
    -0.00075,
    -0.00505,
    -0.00401,
    -0.00891,
    0.01063,
    0.00827
   ],
   [
    -0.00423,
    0.01713,
    0.00199,
    0.00219,
    0.01983,
    -0.0088,
    -0.00275,
    -0.00286,
    0.00892,
    0.00178,
    0.00354,
    0.01911,
    -0.0077,
    -0.00045
   ],
   [
    0.00899,
    0.00205,
    0.0037,
    0.00732,
    -0.00371,
    -0.00523,
    -0.00758,
    0.00603,
    8e-05,
    -0.00024,
    0.00741,
    -0.003,
    -0.00635,
    -0.00855
   ],
   [
    0.00134,
    0.00376,
    0.00216,
    -0.0028,
    -0.0014,
    0.00731,
    -0.00089,
    0.00299,
    0.0022,
    0.0003,
    -0.00406,
    -0.00342,
    0.00472,
    -0.00139
   ],
   [
    0.01733,
    0.00825,
    0.00086,
    0.00311,
    -0.00275,
    -0.00097,
    0.00942,
    0.0154,
    0.01016,
    0.0003,
    0.00367,
    -0.00172,
    -0.00031,
    0.00827
   ],
   [
    0.00788,
    0.01961,
    0.00095,
    -0.01865,
    0.00774,
    0.00093,
    0.00076,
    0.00897,
    0.01772,
    0.00391,
    -0.01839,
    0.00548,
    0.00245,
    0.00183
   ],
   [
    0.00868,
    0.00126,
    0.00517,
    -0.01294,
    0.01075,
    0.01341,
    -0.01832,
    0.00748,
    0.00102,
    0.00752,
    -0.01318,
    0.01025,
    0.01175,
    -0.01515
   ],
   [
    -0.00051,
    -0.0138,
    -0.00207,
    0.00034,
    -0.00509,
    0.00726,
    0.00295,
    -0.00073,
    -0.01188,
    0.0007,
    -0.0009,
    -0.00483,
    0.00827,
    0.00366
   ],
   [
    0.00095,
    -0.01712,
    0.00226,
    -0.00976,
    -0.01019,
    -0.00694,
    0.01854,
    -0.00031,
    -0.0112,
    -0.00109,
    -0.00971,
    -0.00876,
    -0.00527,
    0.01399
   ],
   [
    -0.00131,
    -0.00672,
    0.00496,
    -0.00125,
    -0.01118,
    0.00373,
    -0.00472,
    -0.00156,
    -0.00621,
    0.00243,
    -0.00037,
    -0.00751,
    0.00522,
    -0.00417
   ],
   [
    -0.01584,
    -0.00539,
    0.00254,
    -0.00809,
    0.00057,
    0.00394,
    0.00512,
    -0.01258,
    -0.00519,
    0.00267,
    -0.00862,
    -0.00215,
    0.0035,
    0.00448
   ],
   [
    0.00951,
    0.00106,
    -0.00949,
    -0.00385,
    -0.01994,
    0.02512,
    0.01427,
    0.0088,
    0.00216,
    -0.00353,
    -0.00541,
    -0.01588,
    0.02164,
    0.01434
   ],
   [
    -0.0018,
    -0.00199,
    -0.00446,
    -0.0007,
    0.00082,
    0.00894,
    -0.00836,
    -0.00025,
    -0.00094,
    -0.00154,
    -0.00035,
    0.00148,
    0.01,
    -0.00612
   ],
   [
    -0.01252,
    -0.00607,
    0.00068,
    0.00262,
    -0.00414,
    0.01446,
    0.00793,
    -0.00735,
    -0.0102,
    -0.00056,
    -0.00179,
    -0.00511,
    0.00975,
    0.00516
   ],
   [
    -0.01312,
    -0.00309,
    -0.00657,
    0.00338,
    -0.00087,
    -0.00437,
    -0.00954,
    -0.01017,
    -0.00444,
    -0.00918,
    0.00431,
    0.00075,
    -0.00185,
    -0.00699
   ],
   [
    -0.00622,
    -0.01609,
    -0.01619,
    0.01452,
    -0.00285,
    -0.00526,
    -0.00652,
    -0.00842,
    -0.01545,
    -0.00727,
    0.0106,
    -0.00572,
    -0.00636,
    -0.00519
   ],
   [
    -0.00046,
    0.00132,
    -0.00461,
    0.00989,
    0.00303,
    0.01018,
    -0.00771,
    -0.00053,
    0.00456,
    -0.00471,
    0.0084,
    0.00139,
    0.00904,
    -0.00702
   ],
   [
    -0.00367,
    0.00188,
    -0.01043,
    0.00918,
    -0.00497,
    0.00684,
    0.0037,
    -0.00223,
    0.00215,
    -0.00777,
    0.0096,
    -0.00192,
    0.00666,
    0.00401
   ],
   [
    0.02345,
    0.02212,
    0.00615,
    -0.00331,
    0.00553,
    0.00111,
    0.0038,
    0.01946,
    0.01953,
    0.00518,
    -0.00157,
    0.00559,
    0.00128,
    0.00271
   ],
   [
    0.01371,
    0.00474,
    -0.00052,
    0.00563,
    -0.00415,
    -0.00551,
    0.00178,
    0.01224,
    0.00605,
    -0.00145,
    0.00667,
    -0.00125,
    -0.00168,
    0.00389
   ],
   [
    0.00805,
    -0.00604,
    0.00916,
    0.00162,
    -0.0103,
    -0.00574,
    0.0087,
    0.00845,
    -0.00518,
    0.00622,
    0.00099,
    -0.00721,
    -0.00724,
    0.00387
   ],
   [
    -0.01418,
    -0.0064,
    0.00226,
    -0.0044,
    -0.00685,
    0.00576,
    0.00054,
    -0.01231,
    -0.00665,
    0.00067,
    -0.00347,
    -0.00617,
    0.0055,
    0.0012
   ],
   [
    0.01742,
    -0.00806,
    -0.00153,
    -0.00255,
    -0.00895,
    0.00402,
    0.00368,
    0.01306,
    -0.00571,
    0.00162,
    -0.00353,
    -0.00908,
    0.00323,
    0.00292
   ],
   [
    0.00246,
    0.00443,
    -0.00101,
    0.01134,
    0.00445,
    -0.01102,
    -0.00686,
    -0.00069,
    0.00209,
    -0.00192,
    0.0119,
    0.00323,
    -0.01002,
    -0.00649
   ],
   [
    -0.00151,
    -0.0098,
    -0.00605,
    -0.00712,
    -0.01101,
    0.00864,
    0.02059,
    -0.00432,
    -0.00561,
    -0.00465,
    -0.00443,
    -0.00974,
    0.00996,
    0.01847
   ],
   [
    -0.01622,
    0.0031,
    0.00566,
    -0.0071,
    0.0061,
    0.00058,
    -0.00239,
    -0.01456,
    0.00732,
    0.00367,
    -0.0052,
    0.00561,
    -1e-05,
    -0.00354
   ],
   [
    0.00082,
    0.00674,
    -0.00758,
    -0.00176,
    -0.00083,
    0.00362,
    -0.00565,
    0.00268,
    0.00494,
    -0.00229,
    -0.00378,
    -0.00237,
    0.0017,
    -0.00618
   ],
   [
    -0.01878,
    0.00858,
    0.0112,
    0.01408,
    -0.00291,
    0.00064,
    -0.01137,
    -0.01478,
    0.00482,
    0.00306,
    0.01067,
    -0.00349,
    -0.00184,
    -0.01124
   ],
   [
    -0.01098,
    0.00206,
    0.00353,
    -0.00678,
    0.00429,
    -0.00264,
    -0.00886,
    -0.01154,
    -0.00078,
    0.00166,
    -0.00815,
    0.00217,
    -0.00274,
    -0.00961
   ],
   [
    0.00462,
    -0.01106,
    0.00499,
    0.00164,
    -0.00626,
    -0.00052,
    -0.00775,
    0.0058,
    -0.01012,
    0.00293,
    -0.00034,
    -0.00548,
    0.00161,
    -0.00584
   ],
   [
    -0.01232,
    0.01189,
    0.00383,
    0.01023,
    0.01117,
    0.00225,
    -0.01589,
    -0.01318,
    0.00933,
    0.00214,
    0.00649,
    0.00253,
    -0.00217,
    -0.0155
   ],
   [
    0.00705,
    7e-05,
    -0.01617,
    -0.00912,
    0.00593,
    0.00145,
    -0.00371,
    0.00486,
    0.00126,
    -0.01323,
    -0.00895,
    0.00442,
    0.00132,
    -0.00501
   ],
   [
    -0.00055,
    -0.01406,
    -0.00591,
    -0.00614,
    -0.0122,
    0.00901,
    0.00918,
    -0.00138,
    -0.01118,
    -0.00609,
    -0.00672,
    -0.01218,
    0.00598,
    0.00691
   ],
   [
    3e-05,
    0.00723,
    0.0018,
    -0.0113,
    -0.00205,
    0.00668,
    0.00297,
    -0.00037,
    0.00478,
    0.00018,
    -0.01216,
    -0.00301,
    0.00441,
    0.00105
   ],
   [
    -0.00078,
    0.00118,
    -0.0071,
    0.00833,
    -0.00284,
    -0.00638,
    -0.00954,
    0.00295,
    -0.00222,
    -0.00362,
    0.00664,
    -0.00315,
    -0.00658,
    -0.00893
   ],
   [
    0.00445,
    -0.00325,
    0.00618,
    0.00187,
    -0.01524,
    0.00268,
    0.00359,
    0.00456,
    -0.00247,
    0.00362,
    0.00318,
    -0.01174,
    0.00427,
    0.00566
   ],
   [
    -0.01808,
    0.00472,
    0.0018,
    -0.00778,
    0.0065,
    0.00301,
    -0.00153,
    -0.01555,
    0.00084,
    0.00068,
    -0.0075,
    0.00701,
    0.00254,
    -0.00288
   ],
   [
    0.02305,
    0.00274,
    -0.00276,
    0.00707,
    -0.01437,
    -0.00224,
    0.0054,
    0.01934,
    0.00569,
    -0.00031,
    0.00872,
    -0.01089,
    -0.00266,
    0.00365
   ],
   [
    0.01283,
    -0.0018,
    0.0046,
    0.01264,
    -0.00063,
    -0.00621,
    -0.00406,
    0.00853,
    -0.00024,
    0.00778,
    0.01182,
    0.00146,
    -0.00704,
    -0.00319
   ],
   [
    0.00479,
    0.01396,
    -0.0011,
    0.00018,
    -0.0014,
    0.00143,
    0.01837,
    0.00392,
    0.01382,
    0.00048,
    0.00165,
    0.00019,
    0.00059,
    0.01627
   ],
   [
    -0.00177,
    -0.00659,
    0.00075,
    -0.01017,
    -0.01575,
    0.01735,
    0.00272,
    0.00174,
    -0.00537,
    0.00185,
    -0.01317,
    -0.01896,
    0.01262,
    0.00037
   ],
   [
    -0.00659,
    0.0102,
    0.00268,
    0.00501,
    0.00546,
    -0.00214,
    -0.00381,
    -0.00267,
    0.00332,
    -0.00314,
    0.00114,
    0.00445,
    -0.00131,
    -0.00133
   ],
   [
    0.00741,
    0.02207,
    -0.00023,
    -0.00656,
    0.01456,
    0.00348,
    0.02765,
    0.00957,
    0.02354,
    0.00209,
    -0.00497,
    0.01449,
    0.00726,
    0.02628
   ],
   [
    0.0116,
    -0.00291,
    0.00313,
    0.01742,
    0.0036,
    -0.01739,
    -0.01414,
    0.01173,
    -0.00203,
    -0.00324,
    0.01738,
    0.0022,
    -0.01753,
    -0.013
   ],
   [
    0.00945,
    0.00758,
    0.00352,
    0.00772,
    -0.00032,
    -0.00342,
    -0.0018,
    0.00664,
    0.00765,
    0.00327,
    0.00698,
    0.0001,
    -0.00282,
    -0.00114
   ],
   [
    0.0045,
    0.00039,
    0.0055,
    0.00538,
    -0.00434,
    -0.00134,
    -0.0011,
    0.00366,
    -0.00283,
    0.00456,
    0.00345,
    -0.00451,
    -0.002,
    -0.00064
   ],
   [
    0.02191,
    -0.0001,
    0.01167,
    0.00375,
    -0.00224,
    -0.00224,
    -0.00041,
    0.01783,
    0.0019,
    0.00958,
    0.00415,
    -0.00322,
    -0.00085,
    0.00143
   ],
   [
    0.01943,
    0.00222,
    0.00698,
    0.00053,
    -0.0068,
    0.0086,
    0.00538,
    0.01636,
    0.00187,
    0.00386,
    0.00023,
    -0.00696,
    0.00742,
    0.00482
   ],
   [
    0.01037,
    -0.00094,
    -0.00885,
    0.00199,
    0.0028,
    0.00404,
    -0.00524,
    0.00729,
    0.0002,
    -0.00818,
    0.00087,
    0.0016,
    0.0025,
    -0.00401
   ],
   [
    -0.00768,
    -0.00762,
    0.00799,
    -0.00021,
    0.00233,
    -0.00107,
    -0.00204,
    -0.00587,
    -0.0073,
    0.00518,
    -0.00093,
    -0.00054,
    -0.00132,
    -0.00253
   ],
   [
    -0.00682,
    2e-05,
    -0.00786,
    0.00527,
    0.00379,
    -0.00075,
    -0.01376,
    -0.00143,
    -0.00299,
    -0.00538,
    0.00562,
    0.00453,
    -0.0006,
    -0.01112
   ],
   [
    0.00715,
    0.00452,
    -0.0084,
    0.00566,
    0.00734,
    -0.01083,
    0.00588,
    0.0067,
    0.00497,
    -0.00644,
    0.0043,
    0.00579,
    -0.01071,
    0.00418
   ],
   [
    -0.00728,
    0.00494,
    -0.011,
    -0.00114,
    0.0006,
    -0.01435,
    -0.00624,
    -0.00726,
    -0.0008,
    -0.01296,
    -0.00442,
    -0.00293,
    -0.01744,
    -0.00826
   ],
   [
    0.0171,
    -0.02601,
    -0.00103,
    0.00966,
    -0.00852,
    -0.00677,
    -0.00911,
    0.01717,
    -0.02076,
    0.00223,
    0.00789,
    -0.0071,
    -0.00218,
    -0.00526
   ],
   [
    0.01723,
    -0.00408,
    -0.00478,
    0.00563,
    -0.00806,
    0.0049,
    0.004,
    0.01386,
    -0.0031,
    -0.00629,
    0.00504,
    -0.00621,
    0.00275,
    0.0016
   ],
   [
    -0.01808,
    -0.00625,
    -0.0007,
    0.00695,
    -0.01419,
    -0.00194,
    0.01083,
    -0.01561,
    -0.00456,
    -0.00338,
    0.00561,
    -0.01319,
    -0.00352,
    0.00955
   ],
   [
    0.0161,
    -0.01186,
    -0.00582,
    0.00925,
    -0.01287,
    -3e-05,
    0.0057,
    0.01672,
    -0.00532,
    -0.00169,
    0.01065,
    -0.00823,
    0.00018,
    0.00611
   ],
   [
    0.00542,
    -0.01201,
    0.00854,
    0.00711,
    -0.00867,
    -0.0039,
    -0.00038,
    0.00265,
    -0.01245,
    0.00775,
    0.00607,
    -0.00952,
    -0.00527,
    0.00097
   ],
   [
    0.00858,
    0.0048,
    -0.00898,
    0.00361,
    -0.01172,
    0.00797,
    -0.00373,
    0.00508,
    0.00629,
    -0.00383,
    0.00021,
    -0.01045,
    0.00376,
    -0.00677
   ],
   [
    0.01873,
    -0.00288,
    0.00684,
    0.00482,
    0.00399,
    -0.02206,
    0.002,
    0.01405,
    0.00216,
    0.00678,
    0.00432,
    0.00191,
    -0.02214,
    -0.00012
   ],
   [
    0.01049,
    -0.00659,
    -0.00848,
    0.00973,
    -0.00992,
    0.0066,
    0.00749,
    0.01021,
    -0.00359,
    -0.00785,
    0.00724,
    -0.00782,
    0.00585,
    0.0081
   ],
   [
    -0.00524,
    0.00166,
    0.02012,
    -0.0038,
    0.00749,
    -0.0046,
    -0.01268,
    -0.0021,
    0.00012,
    0.01225,
    -0.00443,
    0.0037,
    -0.005,
    -0.00995
   ],
   [
    0.0008,
    -0.00363,
    -0.00045,
    0.00503,
    -0.00215,
    0.00101,
    0.00201,
    0.00011,
    -0.00254,
    0.0022,
    0.00432,
    -0.003,
    -0.00028,
    0.00198
   ],
   [
    -0.00194,
    0.01862,
    0.00239,
    0.00741,
    -0.00467,
    0.00451,
    -0.00606,
    -0.00014,
    0.01206,
    0.00335,
    0.00426,
    -0.00605,
    0.00356,
    -0.00508
   ],
   [
    -0.00176,
    -0.00971,
    0.00505,
    0.00545,
    0.00038,
    -0.01473,
    0.00491,
    -0.0009,
    -0.00934,
    0.00067,
    0.00458,
    -0.0006,
    -0.01402,
    0.0037
   ],
   [
    0.004,
    0.00366,
    -0.00079,
    -0.00654,
    0.00452,
    0.01074,
    0.00309,
    0.00269,
    0.00376,
    -2e-05,
    -0.00497,
    0.00402,
    0.00832,
    0.0023
   ],
   [
    -0.01703,
    -0.00559,
    0.00257,
    -0.00675,
    -0.00147,
    0.00414,
    0.0154,
    -0.01344,
    -0.00501,
    0.00202,
    -0.00675,
    0.00135,
    0.00673,
    0.01567
   ],
   [
    -0.01065,
    -0.00837,
    -0.00466,
    -0.0086,
    0.00555,
    -0.00384,
    0.00682,
    -0.00554,
    -0.00636,
    -0.00592,
    -0.00907,
    0.00423,
    -0.0027,
    0.00579
   ],
   [
    -0.02057,
    -0.03424,
    0.00672,
    -0.01317,
    -0.01533,
    0.01188,
    0.0109,
    -0.01545,
    -0.02759,
    0.00058,
    -0.01076,
    -0.01262,
    0.01296,
    0.00911
   ],
   [
    -0.00033,
    0.00761,
    0.00399,
    -0.00503,
    0.00681,
    -0.00269,
    0.00291,
    0.00048,
    0.00344,
    0.00353,
    -0.00354,
    0.00622,
    -0.00055,
    0.00417
   ],
   [
    -0.00403,
    0.00836,
    -0.00432,
    -0.00591,
    0.00645,
    0.00168,
    -0.00911,
    -0.00227,
    0.00916,
    0.00078,
    -0.00536,
    0.00593,
    0.00153,
    -0.0077
   ],
   [
    0.00856,
    0.01256,
    0.0097,
    0.00687,
    0.00626,
    -0.00444,
    -0.00644,
    0.00776,
    0.00753,
    0.00731,
    0.00704,
    0.00415,
    -0.0036,
    -0.00411
   ],
   [
    0.00622,
    0.01708,
    -0.00427,
    -0.01714,
    0.00921,
    -0.00769,
    0.02208,
    0.00709,
    0.01487,
    0.00071,
    -0.01731,
    0.00779,
    -0.00422,
    0.02049
   ],
   [
    -0.00287,
    -0.00938,
    -0.00668,
    -0.0074,
    -0.0006,
    0.0102,
    0.0061,
    -0.00225,
    -0.00782,
    -0.00278,
    -0.00733,
    0.00088,
    0.00904,
    0.00568
   ],
   [
    -0.01069,
    0.02229,
    0.00682,
    -0.00609,
    0.01738,
    -0.00136,
    0.01016,
    -0.00898,
    0.01583,
    0.00158,
    -0.00542,
    0.01573,
    -0.00093,
    0.00863
   ],
   [
    -0.00826,
    0.0073,
    0.00086,
    -0.01323,
    0.00649,
    0.01187,
    0.00143,
    -0.00601,
    0.0057,
    0.00098,
    -0.01313,
    0.0055,
    0.00788,
    0.00019
   ],
   [
    0.00482,
    0.00631,
    0.01175,
    -0.00127,
    0.00235,
    0.00364,
    -0.00769,
    0.00473,
    0.00487,
    0.00753,
    -0.00037,
    0.00224,
    0.00188,
    -0.00702
   ],
   [
    -0.00534,
    0.00593,
    -0.00699,
    -0.00879,
    -0.00291,
    0.01363,
    0.00768,
    -0.00402,
    0.00541,
    -0.00437,
    -0.00787,
    -0.00029,
    0.0128,
    0.00533
   ],
   [
    -0.01709,
    -0.00985,
    -0.01179,
    -0.01086,
    -0.00201,
    0.00116,
    0.00908,
    -0.01185,
    -0.00657,
    -0.00812,
    -0.0112,
    -0.00383,
    -0.00198,
    0.00692
   ],
   [
    0.01238,
    -0.00531,
    -0.00318,
    0.00354,
    0.00733,
    -0.00432,
    -0.00221,
    0.00948,
    -0.00321,
    -0.00133,
    0.00294,
    0.00341,
    -0.00553,
    -0.00406
   ],
   [
    0.02246,
    0.01802,
    0.01339,
    -0.0033,
    0.03613,
    -0.0111,
    -0.01572,
    0.01624,
    0.01317,
    0.01098,
    -0.00377,
    0.03106,
    -0.01353,
    -0.01683
   ],
   [
    0.00212,
    -0.00731,
    -0.00392,
    0.00314,
    -0.00654,
    0.00349,
    0.00227,
    -0.0002,
    -0.00463,
    -0.00193,
    0.00356,
    -0.00678,
    0.00131,
    -0.00051
   ],
   [
    -0.00491,
    -0.02231,
    0.00743,
    -0.00698,
    -0.00821,
    0.00653,
    -0.00323,
    -0.00768,
    -0.01947,
    0.00724,
    -0.0067,
    -0.00675,
    0.00571,
    -0.00322
   ],
   [
    0.02014,
    0.01868,
    0.00506,
    0.01202,
    -0.00372,
    -0.00642,
    0.00525,
    0.01701,
    0.01544,
    0.00499,
    0.01147,
    -0.00242,
    -0.00627,
    0.00463
   ],
   [
    -0.00846,
    0.02372,
    -0.00887,
    -0.00746,
    0.01463,
    0.00372,
    -0.00202,
    -0.00568,
    0.01504,
    -0.00981,
    -0.00708,
    0.01506,
    0.00273,
    0.0015
   ],
   [
    0.00458,
    -0.00412,
    -0.01023,
    -0.0062,
    -0.00234,
    0.00162,
    0.00786,
    0.00478,
    -0.00277,
    -0.00912,
    -0.00393,
    -0.00123,
    0.00264,
    0.00822
   ],
   [
    -0.00245,
    -0.00607,
    0.00415,
    -0.00516,
    0.00092,
    -0.00721,
    0.00297,
    -0.00314,
    -0.00492,
    0.0042,
    -0.00594,
    -0.00157,
    -0.00404,
    0.00289
   ],
   [
    -0.00092,
    0.00041,
    -0.00578,
    -0.00398,
    0.00792,
    -0.0014,
    0.00732,
    -0.00068,
    0.0038,
    -0.00277,
    -0.00356,
    0.00491,
    -0.00237,
    0.00505
   ],
   [
    0.00723,
    0.01811,
    0.0014,
    -0.0098,
    -0.01095,
    -0.00219,
    0.02488,
    0.01107,
    0.01966,
    0.00375,
    -0.01161,
    -0.00821,
    -0.00171,
    0.02063
   ],
   [
    0.00608,
    0.01209,
    -0.001,
    -0.006,
    0.00433,
    0.00345,
    -0.00014,
    0.00602,
    0.01084,
    0.00141,
    -0.00614,
    0.004,
    0.00265,
    -0.00155
   ],
   [
    -0.01164,
    -0.01094,
    -0.00367,
    -0.00292,
    -0.00226,
    -0.00045,
    0.00524,
    -0.011,
    -0.01166,
    -0.00352,
    -0.00212,
    -0.00175,
    -0.0014,
    0.00416
   ],
   [
    0.01897,
    -0.00344,
    -0.00431,
    0.00638,
    0.00332,
    -0.01426,
    -0.00431,
    0.01836,
    -0.00621,
    -0.00938,
    0.00728,
    0.00357,
    -0.01516,
    -0.00484
   ],
   [
    -0.00391,
    0.00373,
    -0.00404,
    0.00157,
    -0.01144,
    0.00767,
    0.01065,
    -0.00394,
    0.00383,
    0.00033,
    0.00119,
    -0.00891,
    0.00888,
    0.01144
   ],
   [
    0.01977,
    0.01124,
    -0.00448,
    0.00266,
    0.00406,
    -0.00454,
    -0.01152,
    0.01313,
    0.00837,
    -0.00577,
    -0.00017,
    0.00121,
    -0.00634,
    -0.01128
   ],
   [
    0.00944,
    -0.0016,
    -0.01233,
    -0.00327,
    0.0063,
    -0.00557,
    0.00416,
    0.00624,
    0.00163,
    -0.00448,
    -0.00204,
    0.00435,
    -0.00405,
    0.00184
   ],
   [
    -0.0046,
    0.00381,
    -0.00588,
    -0.00098,
    -0.00348,
    0.00702,
    0.02175,
    -0.00221,
    0.007,
    0.00216,
    0.00108,
    -0.00255,
    0.00681,
    0.0173
   ],
   [
    0.00608,
    -0.00642,
    -0.00213,
    0.00353,
    -0.01405,
    0.00242,
    -0.01525,
    0.00107,
    -0.00624,
    -0.00037,
    -7e-05,
    -0.01415,
    -0.00024,
    -0.01522
   ],
   [
    -0.01475,
    -0.03004,
    0.00361,
    -0.00483,
    -0.00911,
    0.00142,
    0.00102,
    -0.01207,
    -0.0277,
    0.00056,
    -0.00424,
    -0.00931,
    0.00159,
    0.00157
   ],
   [
    -0.00368,
    -0.02369,
    0.00098,
    -0.00295,
    -0.00318,
    -0.00186,
    0.00787,
    -0.00475,
    -0.0221,
    -0.00246,
    -0.00292,
    -0.00273,
    -0.00156,
    0.00607
   ],
   [
    -0.01153,
    -0.00108,
    -0.00678,
    -0.00492,
    0.00082,
    0.00332,
    -0.00025,
    -0.00612,
    -0.00267,
    -0.00774,
    -0.00771,
    0.00058,
    0.00265,
    -0.00091
   ],
   [
    0.01622,
    0.0003,
    -0.00014,
    0.00796,
    0.00154,
    -0.00072,
    -0.00214,
    0.01527,
    0.00102,
    0.00205,
    0.00579,
    0.00075,
    -0.00144,
    -0.0024
   ],
   [
    0.00131,
    0.0073,
    0.00654,
    0.01729,
    -0.0029,
    -0.0081,
    -0.0128,
    0.00321,
    0.00511,
    0.00354,
    0.01271,
    -0.00562,
    -0.01154,
    -0.0144
   ],
   [
    -0.0017,
    -0.00559,
    0.00263,
    0.00331,
    0.00106,
    0.00143,
    -0.00484,
    -0.00422,
    -0.0053,
    0.00479,
    0.00304,
    0.00129,
    0.00308,
    -0.00258
   ],
   [
    0.00169,
    0.01381,
    0.00491,
    0.00644,
    0.01432,
    0.00043,
    -0.00864,
    0.00438,
    0.01192,
    0.0052,
    0.00613,
    0.01324,
    0.00172,
    -0.00714
   ],
   [
    -0.008,
    -0.0166,
    -0.00225,
    0.00653,
    -0.00607,
    -0.00259,
    0.00648,
    -0.00463,
    -0.01278,
    -0.00048,
    0.00734,
    -0.00413,
    -0.00292,
    0.00537
   ],
   [
    0.01851,
    0.00436,
    0.00141,
    0.00125,
    0.00444,
    -0.00372,
    -0.00419,
    0.01544,
    0.00453,
    0.00074,
    0.00148,
    0.00423,
    -0.00358,
    -0.00382
   ],
   [
    -0.0055,
    -0.00806,
    0.00954,
    -0.01182,
    0.00684,
    0.00153,
    -0.00356,
    -0.00754,
    -0.00656,
    0.00703,
    -0.00914,
    0.00681,
    0.00219,
    -0.00287
   ],
   [
    -0.01161,
    -0.01288,
    0.00596,
    -0.0028,
    -0.00164,
    -0.00233,
    0.00487,
    -0.00952,
    -0.00949,
    0.00269,
    -0.00455,
    -0.00462,
    -0.00324,
    0.00209
   ],
   [
    0.0005,
    -0.01186,
    -0.00868,
    -0.00194,
    -0.00964,
    -0.00365,
    0.01145,
    -0.00028,
    -0.00915,
    -0.00787,
    -0.0016,
    -0.00698,
    -0.00209,
    0.01097
   ],
   [
    -0.01168,
    0.00386,
    0.00422,
    -0.0061,
    0.00151,
    0.00132,
    0.00247,
    -0.00817,
    0.00142,
    0.00427,
    -0.00597,
    0.00194,
    0.00221,
    0.00342
   ],
   [
    0.00979,
    -0.00319,
    0.01185,
    0.00324,
    -0.00366,
    -0.00585,
    -0.00177,
    0.00765,
    5e-05,
    0.00995,
    0.00355,
    -0.00277,
    -0.00408,
    -0.0023
   ],
   [
    -0.01263,
    -0.01001,
    -0.00388,
    -0.00277,
    -0.00692,
    -0.01038,
    0.00543,
    -0.01306,
    -0.00561,
    -0.00553,
    -0.00406,
    -0.01029,
    -0.01488,
    -0.0003
   ],
   [
    -0.01142,
    0.00471,
    -0.00897,
    0.00941,
    -0.00489,
    -0.00708,
    -0.00315,
    -0.00877,
    0.00361,
    -0.00529,
    0.00876,
    -0.0025,
    -0.00665,
    -0.0029
   ],
   [
    0.00451,
    -0.00101,
    0.00221,
    -0.01026,
    -0.01266,
    0.00099,
    0.01071,
    0.00226,
    0.00276,
    0.00866,
    -0.00712,
    -0.0098,
    0.00294,
    0.01052
   ],
   [
    0.00793,
    0.00333,
    0.00561,
    0.0062,
    0.00448,
    -0.01356,
    0.00206,
    0.00964,
    0.00362,
    0.00231,
    0.00383,
    0.00336,
    -0.0129,
    0.00245
   ],
   [
    0.01155,
    0.01191,
    0.00317,
    -0.00518,
    -0.00213,
    0.00907,
    0.00418,
    0.00678,
    0.00817,
    0.00014,
    -0.00427,
    -0.00013,
    0.00655,
    0.003
   ],
   [
    0.00361,
    0.00971,
    0.00928,
    -0.01122,
    0.00494,
    0.00046,
    0.01234,
    0.00351,
    0.0065,
    0.0047,
    -0.0123,
    0.00426,
    -0.0006,
    0.01043
   ],
   [
    0.01178,
    0.00113,
    0.01219,
    0.00754,
    0.00181,
    -0.00218,
    -0.00252,
    0.01019,
    0.00487,
    0.00919,
    0.00882,
    0.00319,
    -0.003,
    -0.00266
   ],
   [
    -0.00402,
    0.00158,
    0.00279,
    -0.00397,
    6e-05,
    -0.00299,
    0.00146,
    -0.00232,
    0.00333,
    0.00275,
    -0.00143,
    0.0014,
    -0.00162,
    0.002
   ],
   [
    0.00821,
    0.00477,
    0.00797,
    0.00338,
    -0.00732,
    -0.003,
    0.01301,
    0.00844,
    0.00785,
    0.00705,
    0.00409,
    -0.00421,
    -0.00081,
    0.01125
   ],
   [
    0.01937,
    0.00318,
    0.00948,
    0.00071,
    -0.00222,
    -0.00799,
    0.00751,
    0.01721,
    0.00463,
    0.00679,
    0.00137,
    -0.00244,
    -0.00664,
    0.00422
   ],
   [
    0.00062,
    0.0093,
    0.00422,
    0.00239,
    -0.00191,
    0.00295,
    -0.00739,
    0.00061,
    0.00774,
    0.00074,
    0.00295,
    0.00051,
    0.00273,
    -0.00557
   ],
   [
    -0.00119,
    0.0021,
    -0.01527,
    0.00043,
    0.01459,
    -0.00447,
    -0.00314,
    -0.00241,
    0.00253,
    -0.01061,
    0.00113,
    0.01346,
    -0.00417,
    -0.00348
   ],
   [
    -0.01128,
    -0.02938,
    -0.01363,
    0.02029,
    -0.01443,
    -0.01083,
    -0.00885,
    -0.01289,
    -0.02397,
    -0.00893,
    0.0167,
    -0.01744,
    -0.01261,
    -0.00988
   ],
   [
    0.0135,
    -0.00422,
    -0.00335,
    0.00546,
    -0.00333,
    0.0055,
    -0.00258,
    0.00878,
    -0.00511,
    -0.0043,
    0.00546,
    -0.00277,
    0.00569,
    -0.00137
   ],
   [
    0.00371,
    0.00854,
    -0.00828,
    -0.00107,
    0.00322,
    -0.00038,
    0.0014,
    0.00286,
    0.00222,
    -0.01032,
    -0.00422,
    0.00113,
    -0.0012,
    0.00129
   ],
   [
    0.00402,
    -0.00076,
    -0.01161,
    -0.00241,
    0.00148,
    -0.0046,
    0.0092,
    0.00233,
    -0.00158,
    -0.01041,
    -0.00156,
    0.00116,
    -0.0038,
    0.00738
   ],
   [
    0.0043,
    0.0079,
    -3e-05,
    -0.0042,
    0.00116,
    -0.00128,
    -0.00011,
    0.0025,
    0.00671,
    -0.0022,
    -0.00458,
    0.00203,
    -0.00192,
    -0.00042
   ],
   [
    0.00362,
    -0.01683,
    0.00057,
    3e-05,
    -0.01596,
    0.00868,
    -0.00892,
    0.00051,
    -0.01535,
    0.0001,
    -0.00027,
    -0.01418,
    0.00777,
    -0.00504
   ],
   [
    0.01716,
    0.02035,
    0.01477,
    0.01218,
    -0.00151,
    0.00231,
    -0.00448,
    0.01377,
    0.01829,
    0.01286,
    0.01403,
    0.00244,
    0.00421,
    -0.00135
   ],
   [
    -0.01388,
    0.003,
    -0.00695,
    -0.00266,
    0.0012,
    -0.01185,
    0.00419,
    -0.01365,
    -0.0017,
    -0.00697,
    -0.00504,
    -0.00085,
    -0.0121,
    0.00314
   ],
   [
    0.00469,
    0.00107,
    0.00181,
    0.00194,
    -0.00037,
    0.00111,
    0.00162,
    0.00345,
    -0.00115,
    0.00212,
    0.00041,
    0.0005,
    0.00351,
    0.00263
   ],
   [
    -0.00455,
    0.0024,
    -0.00074,
    0.00022,
    0.00174,
    0.0025,
    -0.00274,
    -0.00519,
    0.00078,
    -0.00122,
    -0.00018,
    0.00068,
    0.00052,
    -0.00153
   ],
   [
    0.01044,
    0.00518,
    0.00692,
    -0.00637,
    -0.02151,
    0.02202,
    0.01276,
    0.0097,
    0.00734,
    0.00966,
    -0.00979,
    -0.01886,
    0.01637,
    0.00988
   ],
   [
    -0.00309,
    0.00395,
    0.00607,
    -0.00725,
    -0.00096,
    0.00335,
    0.00197,
    -0.00241,
    0.00271,
    0.00588,
    -0.00962,
    -0.00348,
    0.00339,
    0.00216
   ],
   [
    -0.00586,
    0.00607,
    -0.00445,
    0.00291,
    0.00364,
    0.00605,
    -0.00793,
    -0.00585,
    0.00605,
    -0.00182,
    0.00485,
    0.00312,
    0.0056,
    -0.00686
   ],
   [
    -0.00135,
    0.00991,
    -0.00963,
    0.00876,
    -0.0101,
    0.00018,
    -0.01178,
    0.0011,
    0.00955,
    -0.00481,
    0.00787,
    -0.0082,
    -2e-05,
    -0.00937
   ],
   [
    0.03257,
    0.01004,
    0.00053,
    0.00446,
    -0.00426,
    -0.00669,
    -0.00261,
    0.02335,
    0.0168,
    0.00516,
    0.00828,
    -0.00207,
    -0.00606,
    -0.00256
   ],
   [
    0.00581,
    -0.0004,
    0.00279,
    -0.00842,
    -0.00227,
    -0.00019,
    0.01399,
    0.00373,
    0.00402,
    0.00176,
    -0.00648,
    -0.00241,
    -0.00284,
    0.00964
   ],
   [
    -0.02176,
    0.00667,
    0.0012,
    0.00461,
    -0.00058,
    -0.00482,
    -0.00688,
    -0.02003,
    0.00136,
    -0.00383,
    0.00256,
    -0.00227,
    -0.00487,
    -0.00665
   ],
   [
    -0.00932,
    -0.00624,
    -0.00877,
    -0.00097,
    0.01159,
    -0.00075,
    -0.00246,
    -0.00727,
    -0.00535,
    -0.00443,
    -0.00109,
    0.01039,
    -0.0017,
    -0.00287
   ],
   [
    0.01847,
    -0.01787,
    0.02,
    -0.01017,
    -0.03076,
    0.01892,
    0.02806,
    0.0175,
    -0.00549,
    0.01387,
    -0.00602,
    -0.0215,
    0.02088,
    0.02756
   ],
   [
    -0.00477,
    -0.00535,
    0.00633,
    -0.00025,
    -0.00815,
    0.00509,
    -0.00151,
    -0.00218,
    -0.00199,
    0.00584,
    0.00072,
    -0.00547,
    0.00419,
    -0.00176
   ],
   [
    0.01751,
    0.01596,
    0.00977,
    0.0087,
    -0.00463,
    0.00801,
    -0.00263,
    0.015,
    0.01357,
    0.00853,
    0.00908,
    -0.00353,
    0.00746,
    -8e-05
   ],
   [
    0.00932,
    0.01127,
    -0.0089,
    0.00059,
    0.00412,
    -0.00611,
    0.00069,
    0.00694,
    0.00886,
    -0.0064,
    -0.00227,
    0.00196,
    -0.00746,
    -0.00175
   ],
   [
    -0.00589,
    -0.00801,
    0.00407,
    0.0042,
    -0.00576,
    -0.0111,
    0.00083,
    -0.00708,
    -0.00991,
    -0.00129,
    0.00461,
    -0.00491,
    -0.01065,
    -0.00031
   ],
   [
    0.02458,
    0.00415,
    0.0119,
    0.04508,
    -0.00055,
    -0.03223,
    -0.01111,
    0.02614,
    0.00693,
    0.02167,
    0.05196,
    0.00678,
    -0.02141,
    -0.00436
   ],
   [
    0.00477,
    -0.01534,
    0.01031,
    -0.00367,
    0.00134,
    0.00502,
    0.00697,
    0.00343,
    -0.00689,
    0.00975,
    -0.00096,
    0.0006,
    0.00459,
    0.0059
   ],
   [
    0.00192,
    -0.00846,
    0.02165,
    0.00311,
    -0.00576,
    -0.00407,
    -0.00591,
    0.00104,
    -0.00393,
    0.01808,
    0.00581,
    -0.00218,
    -0.00281,
    -0.00355
   ],
   [
    0.02618,
    -0.00022,
    0.00118,
    0.00878,
    -0.00314,
    -0.00768,
    -0.00362,
    0.0212,
    0.0036,
    0.00087,
    0.00841,
    -0.00395,
    -0.00631,
    -0.00295
   ],
   [
    -0.0089,
    0.01364,
    -0.00798,
    0.00956,
    0.0092,
    -0.01058,
    -0.01223,
    -0.0064,
    0.00931,
    -0.0086,
    0.00857,
    0.00722,
    -0.0103,
    -0.01083
   ],
   [
    0.00893,
    0.00984,
    0.0035,
    0.00138,
    0.00285,
    0.00212,
    -0.01777,
    0.01042,
    0.00856,
    0.00469,
    0.00111,
    0.00019,
    0.00063,
    -0.01571
   ],
   [
    0.00638,
    0.01654,
    0.00479,
    -0.00985,
    0.0014,
    0.01008,
    -0.00084,
    0.0073,
    0.01622,
    0.00225,
    -0.01039,
    0.00127,
    0.00833,
    -0.00209
   ],
   [
    -0.0182,
    -0.00758,
    -0.0026,
    0.00294,
    0.00209,
    -0.0001,
    -0.00632,
    -0.01551,
    -0.00664,
    -0.00291,
    0.00211,
    0.00159,
    -0.00012,
    -0.00633
   ],
   [
    0.01303,
    0.00508,
    -0.00261,
    0.00903,
    -0.00651,
    0.00479,
    -0.01382,
    0.01156,
    0.00113,
    0.00136,
    0.00627,
    -0.00587,
    0.00675,
    -0.01166
   ],
   [
    0.00532,
    -0.02183,
    0.00564,
    -0.01247,
    -0.00583,
    0.00753,
    -0.00156,
    0.00495,
    -0.01679,
    0.0013,
    -0.01369,
    -0.00725,
    0.00663,
    -0.00258
   ],
   [
    -0.01633,
    -0.00151,
    0.00577,
    -0.00086,
    -0.00701,
    -0.00089,
    -0.00371,
    -0.01656,
    -0.0039,
    0.00276,
    -0.00042,
    -0.00538,
    -0.00175,
    -0.00386
   ],
   [
    -0.00611,
    -0.00707,
    -0.00707,
    0.00659,
    -0.00557,
    0.00031,
    -0.00202,
    -0.00346,
    -0.00851,
    -0.007,
    0.00553,
    -0.00552,
    -0.00102,
    -0.00297
   ],
   [
    0.00228,
    -0.00971,
    -0.00194,
    0.00129,
    -0.0088,
    0.01186,
    0.00726,
    0.00334,
    -0.00457,
    0.00193,
    0.00379,
    -0.00362,
    0.01465,
    0.00805
   ],
   [
    0.00237,
    -0.00515,
    0.00941,
    -0.00674,
    0.00734,
    0.0048,
    0.01355,
    0.00316,
    -0.00343,
    0.00755,
    -0.00481,
    0.00766,
    0.00455,
    0.01219
   ],
   [
    -0.00224,
    0.01097,
    -0.01845,
    -0.00762,
    0.01993,
    -0.01518,
    -0.01114,
    -0.00382,
    0.00929,
    -0.01597,
    -0.00715,
    0.015,
    -0.01458,
    -0.01071
   ],
   [
    -0.00487,
    0.00117,
    0.0006,
    0.01365,
    0.00432,
    -0.01558,
    0.00618,
    -0.00366,
    0.001,
    0.0005,
    0.01475,
    0.00655,
    -0.01226,
    0.0078
   ],
   [
    0.01513,
    -0.00522,
    -0.01024,
    0.00069,
    0.00182,
    0.00038,
    -8e-05,
    0.01512,
    0.00069,
    -0.00576,
    -0.00269,
    0.00069,
    -0.00087,
    -0.00048
   ],
   [
    -0.00767,
    -0.00431,
    -0.00561,
    0.00962,
    -0.01009,
    0.00065,
    -0.01895,
    -0.00978,
    -0.00208,
    -0.0026,
    0.00982,
    -0.00778,
    -0.0009,
    -0.01743
   ],
   [
    0.01642,
    0.00777,
    0.01521,
    -0.00694,
    0.00102,
    0.00618,
    0.00973,
    0.01676,
    0.01045,
    0.01342,
    -0.00475,
    0.00256,
    0.00664,
    0.00921
   ],
   [
    0.00328,
    0.02029,
    -0.00679,
    -0.00228,
    -0.006,
    -0.00239,
    0.00625,
    0.00385,
    0.01869,
    -0.00237,
    -0.00194,
    -0.00317,
    0.00113,
    0.00685
   ],
   [
    0.01136,
    -0.00271,
    0.00187,
    0.01488,
    -0.00042,
    -0.01708,
    0.01255,
    0.00969,
    -0.00356,
    0.00192,
    0.01266,
    -0.00027,
    -0.01308,
    0.01159
   ],
   [
    -0.00767,
    -0.01189,
    0.01285,
    0.01396,
    0.00372,
    -0.01142,
    -0.00855,
    -0.00272,
    -0.00938,
    0.01411,
    0.01442,
    0.00221,
    -0.01053,
    -0.00583
   ],
   [
    -0.02322,
    -0.02528,
    -0.00828,
    -0.00287,
    -0.00492,
    -0.00313,
    -0.00389,
    -0.01982,
    -0.02232,
    -0.00799,
    -0.00323,
    -0.0067,
    -0.00352,
    -0.00242
   ],
   [
    -0.01417,
    0.00898,
    -0.01335,
    -0.00956,
    0.00614,
    -0.01071,
    0.0042,
    -0.01126,
    0.01077,
    -0.01464,
    -0.01028,
    0.0073,
    -0.00785,
    0.0031
   ],
   [
    -0.00581,
    0.00174,
    0.00341,
    -0.00348,
    0.00295,
    -0.00299,
    -0.00024,
    -0.00742,
    0.00039,
    0.0023,
    -0.004,
    0.00253,
    -0.00423,
    -0.00125
   ],
   [
    0.00732,
    -0.0047,
    0.00361,
    -0.00081,
    -0.00537,
    -0.00087,
    0.00145,
    0.00598,
    -0.0017,
    0.00614,
    -0.00012,
    -0.0019,
    0.00228,
    0.00171
   ],
   [
    -0.01871,
    0.00354,
    0.00284,
    0.00802,
    0.00453,
    -0.00828,
    -0.00436,
    -0.01812,
    0.00132,
    0.00351,
    0.00866,
    0.00216,
    -0.01219,
    -0.00758
   ],
   [
    0.00123,
    0.00665,
    0.00465,
    0.0045,
    -0.00726,
    0.00285,
    -0.00778,
    0.00313,
    0.0069,
    0.00444,
    0.00255,
    -0.00699,
    0.00337,
    -0.00585
   ],
   [
    -0.00749,
    0.01871,
    -0.00808,
    -0.01363,
    0.00576,
    -0.0127,
    0.01663,
    -0.00694,
    0.01433,
    -0.00956,
    -0.01547,
    0.0057,
    -0.01042,
    0.01342
   ],
   [
    0.01125,
    0.0065,
    0.00912,
    0.00313,
    -0.01316,
    -0.00473,
    -0.00513,
    0.01202,
    0.00454,
    0.00951,
    -0.00075,
    -0.01456,
    -0.00509,
    -0.005
   ],
   [
    -0.00058,
    0.00551,
    0.00167,
    0.0103,
    0.00181,
    -0.0139,
    -0.00779,
    0.00166,
    0.00091,
    -0.00366,
    0.00726,
    0.00013,
    -0.01497,
    -0.00867
   ],
   [
    0.00592,
    0.00227,
    0.00261,
    -0.00142,
    -0.00227,
    0.00027,
    0.00347,
    0.00701,
    0.00231,
    0.00417,
    0.00078,
    0.00047,
    0.00459,
    0.00699
   ],
   [
    -0.0147,
    -0.00054,
    -0.00874,
    -0.00329,
    0.0137,
    -0.01599,
    0.0006,
    -0.01664,
    -0.00185,
    -0.0077,
    -0.00144,
    0.01154,
    -0.01405,
    0.00028
   ],
   [
    -0.00279,
    -0.01691,
    0.00038,
    0.00689,
    -0.0036,
    -0.0064,
    -0.00588,
    -0.00441,
    -0.0161,
    -0.00354,
    0.00523,
    -0.00529,
    -0.00896,
    -0.00567
   ],
   [
    -0.00581,
    -0.00236,
    -0.00253,
    0.00633,
    0.00378,
    -0.00047,
    0.00409,
    -0.00458,
    0.00123,
    -0.00202,
    0.00381,
    0.00427,
    0.00144,
    0.0026
   ],
   [
    -0.04616,
    -0.014,
    -0.03003,
    -0.00472,
    0.01597,
    -0.00391,
    0.00674,
    -0.03779,
    -0.01425,
    -0.02562,
    -0.0025,
    0.01615,
    -0.00278,
    0.00396
   ],
   [
    0.00314,
    0.00974,
    -0.00798,
    0.0101,
    0.00632,
    -0.00269,
    -0.0161,
    0.0035,
    0.00921,
    -0.00252,
    0.01013,
    0.00478,
    -0.002,
    -0.01232
   ],
   [
    0.01646,
    0.01466,
    -0.00728,
    0.00243,
    0.01092,
    -0.01128,
    0.00189,
    0.01253,
    0.01109,
    -0.00824,
    0.00298,
    0.00931,
    -0.01176,
    -2e-05
   ],
   [
    0.00481,
    -0.01601,
    -0.00082,
    0.00394,
    0.00467,
    -0.00484,
    0.00279,
    0.0006,
    -0.01489,
    -0.00239,
    0.00873,
    0.00365,
    -0.00525,
    0.00173
   ],
   [
    0.01726,
    -0.00074,
    0.00292,
    0.00995,
    -0.01231,
    -0.0047,
    0.0033,
    0.01496,
    0.0039,
    0.00649,
    0.01006,
    -0.01114,
    -0.00603,
    0.00332
   ],
   [
    -0.00468,
    0.00519,
    -0.00161,
    -0.00417,
    -0.0014,
    0.01152,
    -0.00852,
    -0.00197,
    0.00408,
    0.00429,
    -0.00429,
    -0.00149,
    0.00925,
    -0.00744
   ],
   [
    -0.00417,
    0.00651,
    0.00185,
    -0.01278,
    0.00294,
    0.00312,
    0.00638,
    -0.00169,
    0.00561,
    0.00167,
    -0.01211,
    0.00344,
    0.00158,
    0.00449
   ],
   [
    -0.0047,
    0.00961,
    0.01522,
    -0.00655,
    -0.00433,
    -0.00185,
    0.00179,
    -0.00043,
    0.0091,
    0.01191,
    -0.00504,
    -0.00362,
    -0.00102,
    0.00064
   ],
   [
    0.00528,
    0.02907,
    0.00327,
    -0.00728,
    0.00949,
    0.00056,
    0.01046,
    0.00908,
    0.0236,
    0.00482,
    -0.00872,
    0.00796,
    0.00245,
    0.00893
   ],
   [
    0.03666,
    0.01158,
    0.00619,
    0.00735,
    -0.00553,
    0.00378,
    0.00475,
    0.03048,
    0.01167,
    0.00865,
    0.00788,
    -0.00552,
    0.0027,
    0.00515
   ],
   [
    -0.02702,
    -0.01859,
    0.01462,
    -0.00476,
    -0.00236,
    0.00966,
    0.00825,
    -0.02034,
    -0.02125,
    0.00698,
    -0.00844,
    -0.00589,
    0.00795,
    0.00597
   ],
   [
    -0.00833,
    -0.00467,
    0.00987,
    0.00582,
    -0.00068,
    -0.00246,
    -0.00455,
    -0.00538,
    -0.00436,
    0.00743,
    0.00534,
    9e-05,
    -0.00272,
    -0.00305
   ],
   [
    -0.00352,
    0.005,
    -0.00404,
    -0.00453,
    0.00519,
    0.00671,
    -0.00289,
    -0.00104,
    0.00403,
    0.00051,
    -0.00455,
    0.00432,
    0.00535,
    -0.0019
   ],
   [
    -0.00379,
    -0.01509,
    -0.00184,
    0.01908,
    -0.00709,
    0.00524,
    0.00347,
    -0.00173,
    -0.01211,
    0.00254,
    0.0145,
    -0.00638,
    0.00633,
    0.0028
   ],
   [
    -0.01387,
    -0.01186,
    0.00128,
    -0.00315,
    -0.00896,
    0.0134,
    -0.0037,
    -0.01184,
    -0.00833,
    0.00031,
    -0.00382,
    -0.00997,
    0.00946,
    -0.00443
   ],
   [
    0.00437,
    0.00072,
    0.00321,
    0.01342,
    -0.01892,
    0.0072,
    -0.00468,
    0.00515,
    -0.00125,
    0.00832,
    0.0121,
    -0.01305,
    0.00795,
    -0.00243
   ],
   [
    0.01765,
    0.01575,
    -0.00063,
    0.00349,
    0.00312,
    -0.00831,
    -0.00841,
    0.01166,
    0.01244,
    -0.00011,
    0.00236,
    0.00219,
    -0.00929,
    -0.00815
   ],
   [
    -0.03375,
    0.01153,
    -0.00879,
    -0.0018,
    0.02489,
    -0.01992,
    0.0073,
    -0.02639,
    0.01017,
    -0.00638,
    -0.00144,
    0.0186,
    -0.01961,
    0.00531
   ],
   [
    -0.00347,
    -0.01162,
    0.00383,
    0.00157,
    -0.00279,
    -0.00628,
    0.00075,
    -0.00279,
    -0.01527,
    -0.00088,
    0.00114,
    -0.00341,
    -0.00724,
    -0.00139
   ],
   [
    -0.01778,
    0.01002,
    -0.00372,
    -0.006,
    -0.00679,
    0.00409,
    0.00816,
    -0.01613,
    0.0054,
    -0.00593,
    -0.00518,
    -0.0055,
    0.00262,
    0.00651
   ],
   [
    0.00142,
    -0.00914,
    -0.00136,
    -0.00259,
    0.01057,
    -0.01338,
    0.00065,
    -0.00198,
    -0.00905,
    -0.0042,
    -0.00446,
    0.00715,
    -0.01196,
    -0.00051
   ],
   [
    0.01593,
    0.01849,
    -0.01141,
    0.00077,
    0.00915,
    -0.00371,
    0.01005,
    0.01496,
    0.01734,
    -0.00887,
    0.0025,
    0.00995,
    -0.00282,
    0.00754
   ],
   [
    -0.01278,
    -0.00604,
    -0.01406,
    0.00235,
    -0.01026,
    0.0103,
    0.00462,
    -0.01027,
    -0.00274,
    -0.00901,
    0.0032,
    -0.00701,
    0.0114,
    0.00653
   ],
   [
    0.00796,
    -0.01753,
    -0.00282,
    -0.00252,
    -0.01727,
    0.00508,
    0.00713,
    0.00726,
    -0.01379,
    -0.00111,
    -0.00105,
    -0.01336,
    0.0052,
    0.00637
   ],
   [
    -0.00289,
    0.00169,
    -0.00493,
    -0.01088,
    0.00611,
    -0.00234,
    0.01062,
    -0.00147,
    0.00469,
    -0.00397,
    -0.01119,
    0.00562,
    -0.00544,
    0.00538
   ],
   [
    -0.01,
    0.00054,
    0.00684,
    0.00107,
    0.01078,
    -0.00766,
    0.00718,
    -0.00667,
    0.00232,
    0.009,
    0.00074,
    0.01074,
    -0.00452,
    0.00897
   ],
   [
    -0.00539,
    -0.02021,
    -0.00102,
    -0.01433,
    0.00676,
    0.00609,
    0.00535,
    -0.0053,
    -0.01144,
    0.00689,
    -0.0123,
    0.005,
    0.0064,
    0.00364
   ],
   [
    0.00589,
    -0.00723,
    -0.01491,
    -0.01365,
    -0.00301,
    0.0097,
    0.00863,
    0.0035,
    -0.00813,
    -0.01097,
    -0.01176,
    -0.00241,
    0.00834,
    0.00719
   ],
   [
    -0.00425,
    -0.01413,
    -0.00412,
    -0.00092,
    -0.00131,
    -0.00257,
    0.00016,
    -0.00407,
    -0.01184,
    -0.00156,
    -0.00026,
    -0.00149,
    -0.00283,
    -3e-05
   ],
   [
    0.00214,
    0.00483,
    0.00715,
    0.00085,
    0.0052,
    -0.00459,
    0.00295,
    0.00396,
    0.00441,
    0.00132,
    0.00043,
    0.0035,
    -0.0059,
    0.00072
   ],
   [
    -0.01067,
    -0.01986,
    -0.00319,
    -0.00204,
    -0.00854,
    0.00058,
    6e-05,
    -0.00896,
    -0.01562,
    -0.00123,
    -0.00139,
    -0.00648,
    0.00086,
    -0.00107
   ],
   [
    0.0109,
    -0.01491,
    0.00941,
    -0.00412,
    -0.0157,
    0.00123,
    -0.00828,
    0.00901,
    -0.01503,
    0.00777,
    -0.00915,
    -0.01883,
    -0.00276,
    -0.00887
   ],
   [
    -0.00824,
    -0.01476,
    0.00541,
    0.00454,
    -0.00979,
    -0.01056,
    0.00437,
    -0.00744,
    -0.01699,
    0.00139,
    0.00622,
    -0.00837,
    -0.00859,
    0.00496
   ],
   [
    -0.00616,
    -0.01532,
    0.00477,
    -0.00233,
    0.00012,
    -0.01431,
    0.00307,
    -0.00355,
    -0.01657,
    0.00276,
    -0.00256,
    -0.00143,
    -0.0125,
    0.00186
   ],
   [
    0.00028,
    -0.02121,
    -0.00471,
    0.01273,
    -0.00782,
    -0.00251,
    -0.00244,
    -0.00419,
    -0.0199,
    -0.00422,
    0.01077,
    -0.00893,
    -0.00281,
    -0.00089
   ],
   [
    0.01083,
    -0.01416,
    0.0054,
    0.00074,
    -0.00569,
    -0.00154,
    0.02026,
    0.01255,
    -0.0067,
    0.00502,
    0.00308,
    -0.00461,
    0.00036,
    0.01852
   ],
   [
    0.00445,
    0.00461,
    0.00139,
    -0.00346,
    -0.00747,
    -0.00269,
    0.00155,
    0.0028,
    0.00253,
    -0.00058,
    -0.0012,
    -0.00495,
    -0.00365,
    0.00089
   ],
   [
    0.00301,
    -0.00024,
    0.00241,
    0.00264,
    -0.00191,
    -0.01246,
    0.01077,
    0.00215,
    -0.00034,
    -0.00058,
    0.00122,
    -0.00325,
    -0.01216,
    0.0086
   ],
   [
    0.00522,
    -0.00249,
    0.00705,
    0.00603,
    -0.01058,
    -0.00126,
    -0.00167,
    0.0073,
    -4e-05,
    0.00258,
    0.00591,
    -0.00898,
    -0.00133,
    -0.00026
   ],
   [
    0.00156,
    0.00604,
    -0.00067,
    -0.00634,
    0.00212,
    -0.00122,
    -0.0053,
    -0.00029,
    0.00444,
    -0.00076,
    -0.0075,
    0.00091,
    -0.00149,
    -0.00524
   ],
   [
    0.01338,
    -0.00809,
    -0.00151,
    -0.00023,
    -0.02325,
    0.01576,
    -0.00065,
    0.00911,
    -0.00543,
    -0.00169,
    -0.00024,
    -0.02055,
    0.01148,
    -0.00136
   ],
   [
    -0.01387,
    0.00647,
    -0.01337,
    0.00075,
    0.00584,
    -0.00966,
    -0.00813,
    -0.01407,
    0.00243,
    -0.00894,
    0.00091,
    0.00579,
    -0.00767,
    -0.00765
   ],
   [
    -0.00845,
    0.00897,
    0.00091,
    -0.00449,
    0.00531,
    -0.0002,
    -0.00147,
    -0.00349,
    0.00728,
    0.00381,
    -0.00414,
    0.00554,
    0.00047,
    -0.00093
   ],
   [
    0.01352,
    0.00768,
    0.01501,
    0.00365,
    -0.00026,
    0.0,
    -0.00577,
    0.01185,
    0.00752,
    0.01046,
    0.00452,
    0.00073,
    0.00082,
    -0.00461
   ],
   [
    -0.0073,
    0.00268,
    0.00775,
    -0.00966,
    0.00819,
    0.00802,
    -0.00503,
    -0.00371,
    -0.00137,
    0.0037,
    -0.00794,
    0.00726,
    0.00652,
    -0.00192
   ],
   [
    -0.02038,
    0.01244,
    -0.00823,
    0.00748,
    0.0045,
    0.01025,
    -0.01414,
    -0.01758,
    0.00527,
    -0.00956,
    0.00507,
    0.00739,
    0.01007,
    -0.00977
   ],
   [
    -0.02645,
    -0.01235,
    0.00945,
    -0.00542,
    0.00204,
    -0.00248,
    -0.01028,
    -0.02238,
    -0.01477,
    0.00448,
    -0.00491,
    0.00255,
    -0.00163,
    -0.00739
   ],
   [
    -0.01107,
    -0.00702,
    -0.01211,
    0.0056,
    -0.00276,
    -0.00336,
    -0.00097,
    -0.00749,
    -0.00914,
    -0.00777,
    0.00603,
    -0.00168,
    -0.00273,
    -0.00101
   ],
   [
    -0.00515,
    -0.00536,
    0.00395,
    -0.00103,
    -0.0002,
    0.00303,
    -0.00147,
    -0.00442,
    -0.00573,
    0.00224,
    -0.0005,
    -0.00059,
    0.00151,
    -0.00084
   ],
   [
    0.00106,
    0.00454,
    -0.01018,
    0.00605,
    0.01077,
    -0.00211,
    -0.01065,
    0.00334,
    0.00336,
    -0.00934,
    0.00293,
    0.00799,
    -0.00426,
    -0.00979
   ],
   [
    0.00506,
    -0.00263,
    -0.00388,
    -0.00651,
    0.00265,
    -0.00137,
    0.00078,
    0.00473,
    0.00113,
    -0.00056,
    -0.0071,
    0.00213,
    -0.00121,
    0.00016
   ],
   [
    0.01036,
    -0.01117,
    0.00066,
    0.00142,
    -0.00561,
    0.00048,
    -0.00309,
    0.00938,
    -0.01295,
    0.00132,
    0.00176,
    -0.00566,
    -0.00023,
    -0.00272
   ],
   [
    0.01904,
    0.01194,
    -0.01129,
    0.0161,
    -0.01005,
    -0.0065,
    -0.00927,
    0.01764,
    0.00929,
    -0.00456,
    0.01545,
    -0.00636,
    -0.003,
    -0.0067
   ],
   [
    0.00158,
    -0.02245,
    -0.00181,
    0.00719,
    -0.01885,
    -0.00766,
    -0.00337,
    -0.00222,
    -0.02043,
    -0.00045,
    0.01077,
    -0.01455,
    -0.008,
    -0.00285
   ],
   [
    0.00761,
    0.01322,
    0.00706,
    -0.007,
    0.00581,
    -0.00133,
    0.00037,
    0.00785,
    0.00841,
    0.00233,
    -0.00683,
    0.00657,
    -0.00041,
    0.00104
   ],
   [
    0.00666,
    0.01229,
    0.00172,
    0.00352,
    0.01218,
    -0.00999,
    -0.00729,
    0.00956,
    0.01126,
    0.00692,
    0.00195,
    0.00794,
    -0.01051,
    -0.00637
   ],
   [
    -0.00381,
    -5e-05,
    -0.01451,
    0.00208,
    0.01036,
    -0.00713,
    -0.00982,
    -0.00488,
    0.00072,
    -0.0117,
    0.00265,
    0.00812,
    -0.00859,
    -0.01109
   ],
   [
    -0.01907,
    -0.00979,
    -0.0028,
    -0.00795,
    0.00593,
    -0.00127,
    -0.00579,
    -0.01615,
    -0.01097,
    -0.00418,
    -0.00907,
    0.00344,
    -0.00382,
    -0.00698
   ],
   [
    0.01064,
    0.01052,
    -0.01614,
    0.01495,
    -0.00439,
    -0.01287,
    -0.01968,
    0.00986,
    0.00408,
    -0.00914,
    0.01909,
    -0.00088,
    -0.01156,
    -0.01692
   ],
   [
    -0.00801,
    -0.0017,
    0.0018,
    0.0058,
    -0.00467,
    0.00132,
    0.01311,
    -0.00588,
    -0.00119,
    -0.00184,
    0.00481,
    -0.00285,
    0.00248,
    0.01233
   ],
   [
    0.01545,
    0.00687,
    -0.00822,
    0.00078,
    0.00513,
    -0.00908,
    0.0142,
    0.01473,
    0.00338,
    -0.00802,
    0.00048,
    0.00337,
    -0.00785,
    0.01138
   ],
   [
    -0.00138,
    -0.00439,
    -0.00724,
    -0.00737,
    0.01055,
    -0.01122,
    0.01417,
    -0.00516,
    -0.0007,
    -0.00551,
    -0.00497,
    0.00918,
    -0.01179,
    0.01003
   ],
   [
    0.01875,
    0.02322,
    0.00385,
    -0.01188,
    -0.00647,
    0.00258,
    0.00381,
    0.0169,
    0.02002,
    0.00758,
    -0.01185,
    -0.00605,
    0.0008,
    0.00055
   ],
   [
    -0.01157,
    -0.0219,
    -0.00378,
    -0.00609,
    -0.01075,
    0.00883,
    0.01692,
    -0.00992,
    -0.01519,
    -0.00029,
    -0.006,
    -0.00801,
    0.00785,
    0.01471
   ],
   [
    0.01237,
    0.00858,
    0.00164,
    -0.01421,
    -0.00938,
    0.01565,
    0.00438,
    0.01047,
    0.00897,
    -9e-05,
    -0.01353,
    -0.00643,
    0.01208,
    0.00273
   ],
   [
    0.01642,
    -0.00825,
    0.00674,
    -0.00309,
    -0.01689,
    0.00871,
    -0.0055,
    0.01572,
    -0.00767,
    0.00616,
    -0.00615,
    -0.01471,
    0.01118,
    -0.0056
   ],
   [
    -0.01259,
    0.00378,
    0.01002,
    0.00596,
    0.01057,
    0.00224,
    -0.01606,
    -0.00881,
    -0.0027,
    0.01191,
    0.00455,
    0.00614,
    0.00181,
    -0.01234
   ],
   [
    0.00172,
    -0.02284,
    8e-05,
    -0.00594,
    -0.00588,
    0.00197,
    0.00317,
    -0.0022,
    -0.01756,
    0.00348,
    -0.00358,
    -0.00548,
    0.00059,
    0.0016
   ],
   [
    0.00425,
    0.00359,
    0.00732,
    -0.00045,
    -0.00486,
    -0.01243,
    0.0037,
    0.00557,
    0.00654,
    0.00308,
    0.00209,
    -0.00486,
    -0.01153,
    0.00094
   ],
   [
    -0.00408,
    -0.01698,
    -0.00084,
    -0.00327,
    -0.00327,
    -0.00624,
    0.00803,
    -0.00331,
    -0.01167,
    0.00082,
    -0.00362,
    -0.00262,
    -0.00457,
    0.00513
   ],
   [
    -0.01401,
    -0.00736,
    0.00416,
    0.0054,
    0.00067,
    -0.0153,
    -0.00466,
    -0.01318,
    -0.01037,
    0.00441,
    0.004,
    -0.00111,
    -0.014,
    -0.00529
   ],
   [
    -0.0036,
    0.00952,
    0.00369,
    0.0008,
    0.00565,
    -0.00359,
    0.00118,
    -0.00354,
    0.00827,
    -0.00176,
    0.00128,
    0.00612,
    -0.00294,
    0.0006
   ],
   [
    -0.00099,
    0.00954,
    0.00468,
    -0.00264,
    9e-05,
    0.00725,
    0.00248,
    -0.00311,
    0.00688,
    0.0032,
    -0.00107,
    0.001,
    0.00597,
    0.00301
   ],
   [
    -0.00451,
    0.00407,
    -0.00281,
    0.00107,
    -0.0082,
    0.00185,
    0.0024,
    -0.0067,
    -0.00084,
    -0.00104,
    0.00116,
    -0.00753,
    0.00192,
    0.00226
   ],
   [
    0.03684,
    0.00011,
    0.00664,
    0.00709,
    -0.01352,
    0.01015,
    0.00243,
    0.03025,
    0.00028,
    0.00615,
    0.00744,
    -0.01035,
    0.00908,
    0.00305
   ],
   [
    -0.0053,
    -0.00073,
    -0.00906,
    0.00083,
    0.01185,
    -0.00778,
    -0.00572,
    -0.00469,
    -0.00108,
    -0.00908,
    -0.00048,
    0.00843,
    -0.004,
    -0.00247
   ],
   [
    0.0138,
    -0.01757,
    -0.00201,
    0.01323,
    0.0032,
    -0.01853,
    0.0056,
    0.01335,
    -0.0135,
    -0.00374,
    0.01159,
    0.00247,
    -0.01733,
    0.00158
   ],
   [
    -0.00839,
    0.00153,
    0.00226,
    -0.00137,
    -0.00023,
    0.00528,
    0.00299,
    -0.00432,
    0.00461,
    0.00189,
    -0.00274,
    -0.00154,
    0.00447,
    0.00272
   ],
   [
    -0.00214,
    0.00276,
    -0.00089,
    0.01067,
    -0.00936,
    0.0056,
    0.00173,
    -0.00399,
    0.00011,
    0.0025,
    0.01009,
    -0.00517,
    0.00817,
    0.00307
   ],
   [
    -0.0012,
    -0.00338,
    -0.00848,
    -0.0007,
    0.00866,
    -0.01367,
    -0.00141,
    -0.00114,
    -0.00406,
    -0.00924,
    0.0002,
    0.00875,
    -0.01281,
    -0.00311
   ],
   [
    0.01289,
    0.00798,
    -0.00484,
    0.0053,
    -0.00564,
    -0.00719,
    0.00856,
    0.00827,
    0.01132,
    -0.00116,
    0.00479,
    -0.00521,
    -0.00695,
    0.0056
   ],
   [
    0.00556,
    -0.01016,
    -0.0119,
    -0.01096,
    -0.00402,
    0.00105,
    0.00029,
    0.00321,
    -0.00952,
    -0.00513,
    -0.00953,
    -0.00408,
    0.00105,
    -0.0
   ],
   [
    0.00246,
    -0.00831,
    -0.00091,
    0.00156,
    0.00271,
    -0.00801,
    -0.00142,
    -0.00057,
    -0.00847,
    0.00165,
    0.00308,
    0.00356,
    -0.00658,
    -0.00185
   ],
   [
    -0.00488,
    0.00804,
    -0.00971,
    -0.01203,
    0.00777,
    0.00105,
    0.00289,
    -0.00154,
    0.00772,
    -0.00748,
    -0.01231,
    0.00598,
    0.00103,
    0.0015
   ],
   [
    -0.01473,
    0.01085,
    -0.00614,
    -0.00319,
    0.00947,
    0.00835,
    0.0107,
    -0.01381,
    0.01039,
    -0.00518,
    -0.00475,
    0.00649,
    0.00503,
    0.0102
   ],
   [
    -0.01882,
    -0.00249,
    -0.01124,
    0.00499,
    -0.00384,
    -0.00288,
    0.00299,
    -0.01382,
    -0.00647,
    -0.00703,
    0.00288,
    -0.00367,
    -0.00209,
    0.00285
   ],
   [
    0.01285,
    0.00737,
    -0.00945,
    0.0184,
    0.00512,
    -0.00056,
    -0.00098,
    0.01055,
    0.00932,
    -0.00385,
    0.01606,
    0.00508,
    0.00211,
    2e-05
   ],
   [
    -0.00843,
    -0.00331,
    0.00169,
    0.0029,
    0.00047,
    0.00814,
    0.00334,
    -0.00696,
    -0.0027,
    0.00286,
    0.00235,
    0.00178,
    0.01063,
    0.00457
   ],
   [
    0.01137,
    0.02487,
    -0.00204,
    0.00628,
    -0.00944,
    0.00541,
    -0.0113,
    0.01245,
    0.02532,
    0.0029,
    0.0056,
    -0.00794,
    0.00843,
    -0.0091
   ],
   [
    -0.00134,
    0.00364,
    -0.00348,
    0.01199,
    0.02472,
    -0.02825,
    -0.02015,
    -0.0017,
    0.00631,
    -0.00323,
    0.01143,
    0.01962,
    -0.02488,
    -0.01611
   ],
   [
    -0.01646,
    -0.01896,
    -0.00244,
    0.00981,
    -0.00169,
    -0.01963,
    -0.01746,
    -0.02088,
    -0.02499,
    -0.0003,
    0.00901,
    -0.00232,
    -0.01456,
    -0.0124
   ],
   [
    -0.00878,
    0.0044,
    0.00978,
    0.00543,
    -0.00641,
    0.00388,
    0.00299,
    -0.00754,
    0.00129,
    0.00292,
    0.00766,
    -0.0044,
    0.00266,
    0.00313
   ],
   [
    0.03384,
    -0.01416,
    0.00107,
    0.01328,
    0.01531,
    -0.01981,
    -0.01518,
    0.02912,
    -0.00944,
    -0.00357,
    0.01095,
    0.00805,
    -0.02119,
    -0.01445
   ],
   [
    -0.01103,
    0.00747,
    -0.00301,
    0.00307,
    -0.00084,
    0.00058,
    -0.01513,
    -0.00964,
    0.00315,
    -0.00173,
    0.00116,
    -0.0013,
    0.00076,
    -0.01097
   ],
   [
    -0.00853,
    0.01242,
    -0.00616,
    -0.01007,
    0.00796,
    0.00553,
    0.00198,
    -0.00426,
    0.01156,
    -0.00458,
    -0.00956,
    0.00721,
    0.00572,
    0.00354
   ],
   [
    0.00545,
    0.00212,
    0.00249,
    -0.00285,
    0.01447,
    0.00051,
    -0.00298,
    0.00399,
    0.00033,
    0.00217,
    -0.0035,
    0.01158,
    0.0014,
    -0.001
   ],
   [
    0.01402,
    0.0064,
    -0.00114,
    0.01342,
    -0.00264,
    -0.00138,
    -0.00977,
    0.00709,
    0.00519,
    -0.00303,
    0.01253,
    -0.00331,
    -0.00223,
    -0.00909
   ],
   [
    0.00153,
    -0.00424,
    -0.01135,
    0.00627,
    0.00241,
    -0.01841,
    -0.01159,
    -0.00256,
    -0.00883,
    -0.00969,
    0.00445,
    -0.00057,
    -0.02007,
    -0.01328
   ],
   [
    0.00819,
    0.00256,
    0.00427,
    0.00343,
    0.00259,
    0.00726,
    0.00585,
    0.00759,
    0.00459,
    0.00619,
    0.00593,
    0.00457,
    0.00717,
    0.0054
   ],
   [
    -0.00122,
    -0.01951,
    0.01034,
    -0.00843,
    -0.00875,
    0.01341,
    0.01095,
    0.00054,
    -0.01462,
    0.00806,
    -0.00697,
    -0.00581,
    0.01256,
    0.01103
   ],
   [
    0.01453,
    0.00049,
    0.00597,
    0.00459,
    -0.0032,
    -0.0062,
    -0.00449,
    0.01102,
    0.00538,
    0.00561,
    0.00714,
    -0.00059,
    -0.00456,
    -0.00253
   ],
   [
    0.03044,
    0.00657,
    0.0129,
    0.00729,
    -0.00281,
    -0.0046,
    0.00105,
    0.02088,
    0.00862,
    0.01187,
    0.00678,
    -0.005,
    -0.00597,
    -0.00079
   ],
   [
    0.00556,
    -0.0125,
    0.00363,
    -0.00436,
    0.00023,
    -0.0053,
    -0.0005,
    0.00624,
    -0.00813,
    0.00091,
    -0.00362,
    0.00096,
    -0.0035,
    -0.00095
   ],
   [
    -0.00051,
    -0.00439,
    -0.0082,
    -0.00428,
    -0.00204,
    0.00712,
    -0.00574,
    -0.00164,
    -0.00306,
    -0.00571,
    -0.00416,
    -0.00393,
    0.00559,
    -0.00618
   ],
   [
    -0.00594,
    0.00479,
    0.01148,
    -0.00362,
    0.00592,
    0.00513,
    -0.00578,
    -0.0024,
    0.00129,
    0.00774,
    -0.00282,
    0.00536,
    0.00424,
    -0.00315
   ],
   [
    -0.00135,
    0.00113,
    0.00898,
    0.00577,
    0.00351,
    -0.00198,
    -0.00618,
    0.00213,
    -0.00188,
    0.00357,
    0.0047,
    0.00344,
    -0.00092,
    -0.00397
   ],
   [
    0.01041,
    0.01665,
    -0.00726,
    0.03222,
    0.00737,
    -0.01987,
    -0.02932,
    0.00645,
    0.00943,
    -0.01037,
    0.03001,
    0.00545,
    -0.01778,
    -0.02329
   ],
   [
    0.04585,
    -0.01658,
    0.00456,
    -0.00049,
    -0.0263,
    0.00214,
    -0.01209,
    0.03436,
    -0.01218,
    0.00266,
    -0.00298,
    -0.0252,
    -0.00444,
    -0.01535
   ],
   [
    0.00193,
    -0.00126,
    -0.00297,
    0.00475,
    -0.0119,
    0.00233,
    -2e-05,
    0.00288,
    -0.00208,
    -7e-05,
    0.00401,
    -0.0101,
    0.00322,
    0.00126
   ],
   [
    0.00638,
    -0.01533,
    0.00305,
    -0.0017,
    -0.02308,
    -0.02644,
    0.03185,
    0.0054,
    -0.00852,
    0.00168,
    0.00204,
    -0.01382,
    -0.01892,
    0.02869
   ],
   [
    -0.00117,
    0.00837,
    -0.01263,
    0.00269,
    0.01355,
    -0.00868,
    -0.00311,
    -6e-05,
    0.01299,
    -0.00276,
    0.00295,
    0.01262,
    -0.00952,
    -0.00426
   ],
   [
    0.00218,
    0.01105,
    0.00914,
    -0.01619,
    -0.00481,
    0.0054,
    0.01166,
    0.00326,
    0.0074,
    0.00925,
    -0.01548,
    -0.00298,
    0.00399,
    0.00993
   ],
   [
    -0.00804,
    0.0008,
    -0.00458,
    -0.00496,
    0.00676,
    -0.00172,
    0.00548,
    -0.00567,
    0.00299,
    -0.00718,
    -0.00489,
    0.00726,
    -0.00103,
    0.00423
   ],
   [
    0.00583,
    0.02261,
    0.00072,
    0.00235,
    0.00704,
    -0.00857,
    -0.00068,
    0.00812,
    0.0161,
    -0.00051,
    0.00182,
    0.00696,
    -0.0065,
    -0.00019
   ],
   [
    -0.03991,
    0.02646,
    -0.01498,
    -0.03102,
    0.01407,
    -0.00057,
    0.00822,
    -0.03414,
    0.01785,
    -0.01555,
    -0.02901,
    0.01423,
    -0.00072,
    0.00643
   ],
   [
    -0.00296,
    0.0184,
    -0.00411,
    -0.00797,
    -0.00581,
    0.00027,
    -0.00199,
    -0.00266,
    0.01643,
    -0.00348,
    -0.00883,
    -0.00522,
    -0.0017,
    -0.00135
   ],
   [
    0.02154,
    0.01144,
    0.00574,
    -0.01654,
    0.00279,
    0.01472,
    0.01134,
    0.02074,
    0.01014,
    0.00838,
    -0.01423,
    0.00124,
    0.01393,
    0.01064
   ],
   [
    0.0021,
    0.00401,
    -0.00477,
    0.02955,
    -0.01308,
    -0.0133,
    0.00211,
    0.00191,
    -0.00183,
    -0.00707,
    0.02314,
    -0.0134,
    -0.014,
    -0.00025
   ],
   [
    0.01721,
    -0.00124,
    0.01281,
    0.00537,
    -0.00051,
    0.00104,
    0.00385,
    0.01557,
    0.00168,
    0.00979,
    0.00604,
    6e-05,
    0.00185,
    0.00474
   ],
   [
    -0.00344,
    -0.01253,
    -8e-05,
    -0.00866,
    -0.01567,
    0.00195,
    0.01388,
    -0.00493,
    -0.00958,
    -0.001,
    -0.01017,
    -0.01544,
    0.00161,
    0.00739
   ],
   [
    0.01242,
    0.01348,
    -0.01335,
    -0.00303,
    -0.00054,
    0.01102,
    -0.00412,
    0.00914,
    0.00748,
    -0.00418,
    -0.00285,
    0.00186,
    0.00849,
    -0.00251
   ],
   [
    0.02217,
    -0.01947,
    0.01375,
    -0.05535,
    -0.01122,
    0.05251,
    0.01675,
    0.02136,
    -0.01666,
    0.01629,
    -0.0531,
    -0.00951,
    0.0487,
    0.0133
   ],
   [
    -0.00015,
    0.016,
    0.00321,
    0.00513,
    0.00461,
    -0.00647,
    -0.00668,
    -0.0023,
    0.01102,
    0.00264,
    0.00378,
    0.00269,
    -0.00752,
    -0.00724
   ],
   [
    -0.00229,
    -0.00088,
    -0.00155,
    0.00386,
    -0.00585,
    0.00074,
    -0.00155,
    -0.00169,
    -0.00379,
    -0.00033,
    0.00327,
    -0.00626,
    9e-05,
    4e-05
   ],
   [
    -0.0179,
    -0.01599,
    0.00281,
    -0.00501,
    -0.00246,
    0.01442,
    0.0063,
    -0.01352,
    -0.01137,
    0.00078,
    -0.00263,
    0.00038,
    0.01502,
    0.00868
   ],
   [
    3e-05,
    0.00641,
    -0.0004,
    -0.00397,
    -0.00142,
    0.0022,
    0.00546,
    0.00048,
    0.00404,
    0.00512,
    -0.00461,
    -0.00148,
    0.00192,
    0.00486
   ],
   [
    -0.00836,
    0.00057,
    -0.00138,
    4e-05,
    -0.00578,
    0.00473,
    0.00205,
    -0.00586,
    -0.00091,
    -0.00023,
    1e-05,
    -0.00489,
    0.00558,
    0.00293
   ],
   [
    -0.00853,
    -0.02188,
    -0.01096,
    0.00013,
    -0.008,
    0.01525,
    0.0125,
    -0.00531,
    -0.0247,
    -0.01354,
    -0.00323,
    -0.00897,
    0.01221,
    0.0083
   ],
   [
    -0.01019,
    -0.00279,
    0.00276,
    -0.00726,
    0.00565,
    0.00013,
    0.00725,
    -0.00808,
    0.00049,
    0.00243,
    -0.00721,
    0.00411,
    0.00097,
    0.00642
   ],
   [
    6e-05,
    0.00094,
    -0.00662,
    -0.00433,
    0.00244,
    -0.00732,
    0.00408,
    0.00237,
    -0.00281,
    -0.00192,
    -0.00405,
    0.00082,
    -0.00592,
    0.00195
   ],
   [
    0.00832,
    -0.00453,
    -0.00155,
    0.00226,
    -0.00676,
    0.00762,
    0.00413,
    0.00743,
    -0.00123,
    0.00207,
    0.00297,
    -0.00668,
    0.00956,
    0.00514
   ],
   [
    0.00803,
    0.0016,
    0.01022,
    -0.00012,
    0.01157,
    -0.0014,
    0.00152,
    0.00809,
    -9e-05,
    0.00349,
    0.00058,
    0.0136,
    0.00157,
    0.00335
   ],
   [
    -0.00318,
    -0.00911,
    0.0063,
    -0.00636,
    -0.00823,
    0.0026,
    0.00212,
    -0.00377,
    -0.00879,
    0.00564,
    -0.00655,
    -0.00778,
    -0.00024,
    -0.0005
   ],
   [
    -0.00864,
    -0.00347,
    0.00404,
    -0.00057,
    -0.00458,
    -0.00188,
    0.00586,
    -0.0109,
    -0.00607,
    0.00274,
    -0.00012,
    -0.0038,
    -0.00293,
    0.00493
   ],
   [
    -0.01541,
    -0.01581,
    -0.00166,
    0.00067,
    -0.00781,
    0.00181,
    -0.00341,
    -0.01116,
    -0.01298,
    0.00096,
    -0.00012,
    -0.00845,
    0.00151,
    -0.00268
   ],
   [
    -0.01024,
    -0.00847,
    -0.00081,
    0.00429,
    -0.00067,
    0.00599,
    -0.00708,
    -0.01121,
    -0.00508,
    0.00012,
    0.00483,
    -0.00259,
    0.00396,
    -0.00913
   ],
   [
    -0.00203,
    -0.00032,
    0.00333,
    -0.00876,
    0.00111,
    0.00966,
    0.00136,
    -0.00057,
    0.00063,
    0.00141,
    -0.00815,
    0.00129,
    0.00898,
    0.00076
   ],
   [
    -0.00114,
    -0.00146,
    0.01774,
    0.00867,
    0.00166,
    -0.00314,
    -0.00956,
    -0.00034,
    0.00134,
    0.02172,
    0.00991,
    0.00292,
    0.00075,
    -0.00627
   ],
   [
    -0.03012,
    -0.01336,
    0.0089,
    0.00601,
    -0.00539,
    -0.00584,
    0.01671,
    -0.02668,
    -0.00832,
    0.00967,
    0.00699,
    -0.00388,
    -0.00469,
    0.01421
   ],
   [
    0.00071,
    0.00242,
    5e-05,
    -0.00032,
    0.00613,
    -0.00745,
    -0.00817,
    0.00204,
    0.00186,
    -0.00062,
    0.00031,
    0.00563,
    -0.00532,
    -0.00664
   ],
   [
    -0.0299,
    -0.01345,
    0.00596,
    -0.02122,
    0.00402,
    0.00016,
    0.02847,
    -0.02379,
    -0.00555,
    -0.00199,
    -0.02079,
    0.00857,
    0.00272,
    0.02604
   ],
   [
    -0.01169,
    0.02376,
    -0.00439,
    -0.00492,
    0.00755,
    -0.0021,
    -0.00208,
    -0.00782,
    0.0191,
    -0.00631,
    -0.00654,
    0.00676,
    -0.00046,
    -0.00249
   ],
   [
    0.00594,
    -0.00281,
    -0.00777,
    -0.00153,
    -0.00454,
    -0.0048,
    0.00817,
    0.00382,
    0.00099,
    -0.00794,
    0.00095,
    -0.00437,
    -0.0051,
    0.00628
   ],
   [
    0.02354,
    -0.00701,
    0.00505,
    0.0019,
    -0.00418,
    -0.0134,
    -0.00527,
    0.01703,
    -0.00735,
    -0.00103,
    0.00142,
    -0.0046,
    -0.0161,
    -0.00771
   ],
   [
    -0.00025,
    -0.01007,
    -0.00764,
    0.00545,
    -0.01286,
    -0.0004,
    0.0022,
    0.00079,
    -0.00877,
    -0.00936,
    0.00408,
    -0.01101,
    6e-05,
    0.00172
   ],
   [
    -0.00015,
    -0.01493,
    0.00251,
    -0.00015,
    -0.00603,
    -0.00054,
    -0.00122,
    0.00094,
    -0.01087,
    0.0003,
    0.00066,
    -0.0047,
    -0.00143,
    -0.00086
   ],
   [
    0.00582,
    0.01259,
    0.00661,
    -0.00252,
    0.01333,
    -0.0044,
    -0.00834,
    0.00368,
    0.00834,
    0.0027,
    -0.00385,
    0.0114,
    -0.00289,
    -0.00664
   ],
   [
    -0.01879,
    -0.0138,
    -0.00433,
    -0.0194,
    0.00476,
    0.00393,
    0.00698,
    -0.01459,
    -0.01226,
    -0.00448,
    -0.01826,
    0.00126,
    0.00181,
    0.0051
   ],
   [
    0.01166,
    -0.0048,
    0.00833,
    -0.00508,
    -0.00383,
    -0.00974,
    -0.0015,
    0.00452,
    -0.0058,
    0.00391,
    -0.00535,
    -0.00448,
    -0.00803,
    -0.00033
   ],
   [
    0.00601,
    0.01967,
    -0.00024,
    0.00496,
    0.00917,
    0.00779,
    0.00636,
    0.00881,
    0.01409,
    -0.00174,
    0.0033,
    0.01119,
    0.00823,
    0.00581
   ],
   [
    -0.00333,
    0.00186,
    -0.00682,
    0.00951,
    0.0066,
    -0.00134,
    -0.01499,
    -0.00464,
    2e-05,
    -0.00551,
    0.00871,
    0.00467,
    -0.00288,
    -0.01393
   ],
   [
    0.0078,
    0.00567,
    -0.00161,
    0.01549,
    0.00539,
    0.00508,
    -0.00715,
    0.00421,
    -0.00199,
    -0.00292,
    0.01423,
    0.00411,
    0.00284,
    -0.00542
   ],
   [
    0.0118,
    -0.00812,
    0.01625,
    0.01022,
    0.00889,
    -0.00699,
    -0.01181,
    0.00893,
    -0.00329,
    0.01667,
    0.01292,
    0.00931,
    -0.00663,
    -0.00974
   ],
   [
    0.00502,
    0.00364,
    0.00365,
    -0.00938,
    0.00538,
    0.00095,
    0.00123,
    0.00418,
    0.00453,
    0.00282,
    -0.00595,
    0.00726,
    0.00288,
    0.00108
   ],
   [
    0.00196,
    -0.00274,
    -0.00201,
    0.01469,
    -0.00128,
    0.00189,
    -0.01075,
    0.00213,
    -0.00075,
    0.00167,
    0.01112,
    -0.00259,
    0.00245,
    -0.00744
   ],
   [
    0.00699,
    -0.0092,
    -0.00206,
    -0.00936,
    0.00195,
    -0.00538,
    0.00981,
    0.00563,
    -0.00388,
    -0.00253,
    -0.00984,
    0.00081,
    -0.00443,
    0.00742
   ],
   [
    9e-05,
    0.00657,
    0.00585,
    -0.00304,
    -0.00072,
    0.00088,
    0.00419,
    -0.00218,
    0.00443,
    0.0042,
    -0.00315,
    0.00018,
    0.00185,
    0.00357
   ],
   [
    0.00768,
    0.00602,
    -0.00054,
    -0.00198,
    0.00951,
    0.00726,
    -0.0131,
    0.0048,
    0.00338,
    3e-05,
    -0.00169,
    0.00772,
    0.00549,
    -0.01092
   ],
   [
    0.01067,
    -0.00317,
    -0.00792,
    0.00426,
    -0.00517,
    -0.01208,
    -0.00896,
    0.00642,
    -0.00496,
    -0.00771,
    0.00391,
    -0.0055,
    -0.01438,
    -0.00993
   ],
   [
    0.0258,
    0.02525,
    0.00458,
    0.00936,
    0.00052,
    0.00041,
    -0.00303,
    0.02384,
    0.02129,
    0.0072,
    0.00821,
    0.00143,
    0.00177,
    -0.00219
   ],
   [
    -0.02114,
    -0.02308,
    -0.00485,
    0.01215,
    -0.003,
    -0.01637,
    -0.00888,
    -0.02033,
    -0.02512,
    -0.01278,
    0.00752,
    -0.00763,
    -0.01496,
    -0.01046
   ],
   [
    -0.00535,
    0.0224,
    -0.01389,
    0.00665,
    0.00428,
    -0.00464,
    -0.00634,
    -0.00593,
    0.01627,
    -0.00947,
    0.00686,
    0.0049,
    -0.00413,
    -0.00569
   ],
   [
    -0.01633,
    0.00226,
    -0.00167,
    -0.00993,
    0.00607,
    0.0121,
    -0.00379,
    -0.01267,
    -0.0019,
    -0.00167,
    -0.0118,
    0.00494,
    0.01172,
    -0.002
   ],
   [
    0.01118,
    0.00642,
    0.00976,
    -0.0029,
    -0.00199,
    0.00283,
    -0.00528,
    0.01243,
    0.00444,
    0.0108,
    -0.00438,
    -0.00347,
    0.0037,
    -0.00373
   ],
   [
    -0.00664,
    -0.00762,
    -0.00476,
    -0.0005,
    -0.00784,
    -0.00097,
    0.00131,
    -0.00912,
    -0.00173,
    -0.0046,
    0.00198,
    -0.00555,
    0.00134,
    0.00151
   ],
   [
    -0.00125,
    0.01951,
    0.00649,
    0.00503,
    0.00644,
    -0.00926,
    -0.01315,
    -0.00091,
    0.01182,
    0.00282,
    0.0023,
    0.00235,
    -0.01137,
    -0.01323
   ],
   [
    0.00971,
    -0.01104,
    -0.00092,
    0.00886,
    -0.00224,
    -0.0048,
    0.00753,
    0.0076,
    -0.0066,
    4e-05,
    0.00911,
    -0.00106,
    -0.00257,
    0.00882
   ],
   [
    -0.00159,
    0.00232,
    0.00199,
    0.00103,
    0.00295,
    -0.00426,
    0.00244,
    -0.00176,
    -0.00122,
    0.00475,
    0.00216,
    0.00257,
    -0.00252,
    0.00214
   ],
   [
    0.00732,
    -0.00375,
    0.00573,
    0.00527,
    -0.00488,
    0.00565,
    -0.00601,
    0.00661,
    -0.00386,
    0.00325,
    0.00411,
    -0.00459,
    0.00404,
    -0.00598
   ],
   [
    0.00255,
    0.00555,
    -0.01131,
    0.01579,
    -0.00073,
    -0.01664,
    0.00123,
    0.00154,
    -0.00238,
    -0.01207,
    0.01471,
    0.00186,
    -0.01119,
    0.00358
   ],
   [
    0.00302,
    0.01687,
    -0.00126,
    -0.01283,
    0.01818,
    0.01175,
    0.00427,
    0.00331,
    0.01716,
    0.00473,
    -0.01239,
    0.0174,
    0.01133,
    0.0045
   ],
   [
    0.0034,
    0.01961,
    0.00298,
    -0.00368,
    0.01962,
    0.00176,
    -0.01416,
    0.00033,
    0.02061,
    0.00188,
    -0.00479,
    0.01506,
    0.00076,
    -0.0107
   ],
   [
    -0.00798,
    -0.00927,
    -0.00613,
    -0.00097,
    -0.0047,
    0.00701,
    -0.00471,
    -0.00809,
    -0.00394,
    -0.00234,
    -0.00116,
    -0.00229,
    0.00779,
    -0.00299
   ],
   [
    -0.02784,
    0.00043,
    0.00065,
    -0.01062,
    0.00589,
    -0.0107,
    -0.00673,
    -0.02149,
    -0.00304,
    -0.00325,
    -0.01019,
    0.005,
    -0.01022,
    -0.00691
   ],
   [
    -0.01413,
    0.00767,
    -0.01288,
    0.00042,
    0.00172,
    -0.00075,
    -0.01603,
    -0.00996,
    0.00769,
    -0.00655,
    0.00089,
    0.00213,
    0.00072,
    -0.01242
   ],
   [
    -0.01015,
    -0.00518,
    0.01167,
    0.01476,
    0.01013,
    -0.01943,
    -0.03971,
    -0.0022,
    -0.01125,
    0.01012,
    0.01199,
    0.00552,
    -0.01804,
    -0.03606
   ],
   [
    0.018,
    0.00261,
    0.00075,
    0.00765,
    0.00218,
    -0.00605,
    0.0024,
    0.01731,
    0.00642,
    0.00593,
    0.00772,
    0.00213,
    -0.00546,
    0.00187
   ],
   [
    0.00775,
    -0.00301,
    0.01039,
    -0.01653,
    0.00032,
    0.00453,
    -0.01521,
    0.01238,
    -0.01296,
    -0.0006,
    -0.01708,
    -0.00521,
    -0.00631,
    -0.0182
   ],
   [
    0.0022,
    -0.00093,
    -0.00727,
    -0.00596,
    -0.00383,
    -0.00116,
    0.02083,
    0.00389,
    -0.0004,
    -0.00624,
    -0.00408,
    -0.00257,
    0.00169,
    0.01867
   ],
   [
    -0.0111,
    0.01018,
    -0.00162,
    -9e-05,
    0.0111,
    0.00159,
    0.00087,
    -0.00838,
    0.00718,
    -0.00419,
    0.00211,
    0.01265,
    0.00386,
    0.00049
   ],
   [
    -0.01045,
    0.00942,
    -0.00645,
    0.00435,
    -0.00517,
    -0.00105,
    0.0056,
    -0.00996,
    0.00504,
    -0.00572,
    0.00259,
    -0.0047,
    -0.00027,
    0.00607
   ],
   [
    0.00575,
    0.01564,
    0.00346,
    -0.01045,
    0.01186,
    0.00752,
    -0.00664,
    0.00638,
    0.01414,
    0.00121,
    -0.00778,
    0.01151,
    0.00707,
    -0.00466
   ],
   [
    0.00622,
    0.00796,
    0.00489,
    -0.00096,
    -0.00348,
    0.0089,
    -0.01461,
    0.00671,
    0.00437,
    0.00044,
    -0.0046,
    -0.00564,
    0.00411,
    -0.0134
   ],
   [
    -0.00333,
    0.01794,
    0.0104,
    0.00542,
    0.00273,
    -0.00168,
    0.00158,
    -0.00127,
    0.01517,
    0.00861,
    0.00598,
    0.00378,
    -0.0011,
    0.00153
   ],
   [
    0.00778,
    -0.00906,
    0.00529,
    -0.00023,
    -0.00761,
    0.01109,
    -0.00055,
    0.00606,
    -0.0083,
    0.00415,
    0.00072,
    -0.00598,
    0.01114,
    -0.00124
   ],
   [
    -0.02296,
    -0.01258,
    -0.00416,
    -0.00612,
    0.00276,
    0.00391,
    0.0129,
    -0.01774,
    -0.00814,
    -0.00309,
    -0.00441,
    0.00383,
    0.00418,
    0.01247
   ],
   [
    -0.01095,
    -0.00411,
    0.00133,
    -0.00064,
    -0.00621,
    0.00398,
    0.00124,
    -0.01276,
    -0.00673,
    -0.00469,
    -0.00178,
    -0.00284,
    0.00713,
    0.00231
   ],
   [
    -0.00829,
    -0.01826,
    0.00123,
    0.00344,
    0.01513,
    -0.03821,
    -0.01159,
    -0.0107,
    -0.01687,
    -0.00609,
    0.00709,
    0.01511,
    -0.02986,
    -0.00822
   ],
   [
    -0.00155,
    0.00225,
    0.00307,
    -0.00033,
    0.00111,
    -0.00074,
    0.00231,
    0.00046,
    0.00061,
    0.0008,
    -0.00203,
    0.00218,
    -0.00034,
    0.00344
   ],
   [
    -0.00472,
    -0.00202,
    0.00486,
    0.00505,
    0.00317,
    0.00411,
    0.01166,
    -0.0047,
    0.00287,
    0.00913,
    0.00589,
    0.00398,
    0.00617,
    0.01105
   ],
   [
    0.01658,
    0.00414,
    -0.0008,
    0.00688,
    -0.0125,
    -0.00346,
    0.00095,
    0.01415,
    0.00438,
    0.00064,
    0.00815,
    -0.00914,
    -0.00236,
    0.0002
   ],
   [
    0.00912,
    -0.02976,
    0.00813,
    -0.01776,
    -0.0029,
    0.01895,
    -0.0029,
    0.00463,
    -0.02958,
    0.00334,
    -0.01389,
    -0.00251,
    0.01647,
    -0.00266
   ],
   [
    0.00735,
    0.00815,
    0.00342,
    -0.00311,
    -0.0003,
    -0.00146,
    0.01039,
    0.00699,
    0.0076,
    -0.00056,
    0.00028,
    0.00192,
    -0.00055,
    0.00883
   ],
   [
    -0.03062,
    0.02474,
    0.01291,
    -0.03609,
    0.01726,
    0.02611,
    0.02336,
    -0.02377,
    0.02026,
    0.00943,
    -0.03352,
    0.01736,
    0.02667,
    0.02183
   ],
   [
    0.00577,
    0.0023,
    -0.00219,
    0.00659,
    -0.00771,
    0.00276,
    0.00085,
    0.0033,
    0.00186,
    -0.00292,
    0.00604,
    -0.00671,
    0.00258,
    0.00089
   ],
   [
    -0.02747,
    0.00071,
    0.00883,
    -0.01625,
    -0.01136,
    0.01867,
    0.00546,
    -0.02109,
    0.00097,
    0.00775,
    -0.01589,
    -0.00957,
    0.01441,
    0.00584
   ],
   [
    -0.00153,
    0.00326,
    0.01396,
    0.00158,
    0.00663,
    -0.00486,
    -0.00813,
    0.00047,
    0.00342,
    0.01176,
    0.00136,
    0.00645,
    -0.00289,
    -0.00512
   ],
   [
    0.00714,
    0.01552,
    -0.0027,
    0.00325,
    0.0021,
    0.00471,
    -0.00263,
    0.00711,
    0.01441,
    -0.00091,
    0.00297,
    0.00281,
    0.00344,
    -0.00249
   ],
   [
    -0.00128,
    -0.01183,
    -0.00457,
    0.00946,
    -0.00867,
    6e-05,
    -0.00904,
    -0.0034,
    -0.00909,
    -0.0033,
    0.00889,
    -0.00597,
    0.00139,
    -0.00607
   ],
   [
    -0.01803,
    0.00341,
    0.0066,
    -0.00377,
    0.00421,
    -0.00123,
    -0.01546,
    -0.01644,
    -0.00171,
    0.00296,
    -0.00351,
    0.00349,
    -0.00253,
    -0.0143
   ],
   [
    -0.01224,
    -0.00252,
    -0.00785,
    0.00137,
    0.00929,
    -0.01483,
    -0.00033,
    -0.00908,
    0.00233,
    -0.00745,
    0.00113,
    0.00569,
    -0.01287,
    0.00072
   ],
   [
    -0.01592,
    -0.00488,
    -0.00216,
    -0.00153,
    -0.00679,
    0.00618,
    -0.00089,
    -0.01441,
    -0.00415,
    -0.00207,
    -0.00163,
    -0.00634,
    0.00675,
    0.00024
   ],
   [
    -0.00667,
    0.00938,
    0.00492,
    -0.00085,
    -0.00233,
    0.00814,
    0.00234,
    -0.00576,
    0.00862,
    0.00083,
    -0.00158,
    -0.00117,
    0.00809,
    0.00181
   ],
   [
    0.00169,
    -0.01394,
    0.00125,
    0.00695,
    -0.01306,
    0.00507,
    -0.00424,
    -0.0025,
    -0.01269,
    0.00146,
    0.0079,
    -0.01109,
    0.00547,
    -0.00282
   ],
   [
    0.00374,
    0.00352,
    -0.0066,
    -0.00552,
    0.00082,
    -0.00234,
    -0.00385,
    0.0047,
    0.00529,
    -0.00442,
    -0.00607,
    0.00056,
    -0.00331,
    -0.00432
   ],
   [
    -0.00755,
    0.00352,
    -0.00684,
    -0.00513,
    0.01119,
    -0.00737,
    -0.00765,
    -0.00577,
    0.00338,
    -0.00687,
    -0.00533,
    0.0087,
    -0.00655,
    -0.00586
   ],
   [
    0.00417,
    -0.01217,
    -0.00064,
    0.00517,
    -0.01422,
    -0.00601,
    -0.00946,
    0.0032,
    -0.01107,
    -0.00267,
    0.00545,
    -0.01352,
    -0.00585,
    -0.00799
   ],
   [
    -0.01371,
    0.00161,
    -0.00159,
    -0.00541,
    0.00718,
    -0.0151,
    -0.00242,
    -0.01195,
    -0.00442,
    -0.00552,
    -0.0042,
    0.00475,
    -0.01556,
    -0.00182
   ],
   [
    0.00536,
    -0.00616,
    -0.00493,
    0.00776,
    -0.00632,
    -0.01012,
    -0.00949,
    0.00482,
    -0.00556,
    -0.0052,
    0.00442,
    -0.00982,
    -0.01112,
    -0.01041
   ],
   [
    0.0125,
    -0.03339,
    0.00443,
    0.00058,
    -0.01314,
    0.00027,
    -0.00065,
    0.00901,
    -0.02084,
    0.00679,
    0.00092,
    -0.01232,
    -0.00037,
    -0.00215
   ],
   [
    0.00253,
    0.00763,
    -0.00246,
    -0.00039,
    -0.00057,
    -0.00404,
    0.00464,
    0.00284,
    0.01016,
    0.00378,
    0.0022,
    0.00089,
    -0.00187,
    0.00456
   ],
   [
    0.00271,
    0.01665,
    0.01939,
    -0.00966,
    0.00786,
    0.0032,
    -0.00034,
    0.00091,
    0.01534,
    0.01554,
    -0.00746,
    0.00748,
    0.00259,
    -0.00108
   ],
   [
    -0.01069,
    -0.00573,
    -0.00404,
    0.00716,
    -0.00577,
    -0.00598,
    -0.00475,
    -0.00903,
    -0.00742,
    -0.00469,
    0.00417,
    -0.00655,
    -0.0051,
    -0.00426
   ],
   [
    -0.01495,
    0.00371,
    -0.00255,
    -0.00608,
    0.00497,
    0.00383,
    0.00224,
    -0.01475,
    -0.00023,
    -0.00419,
    -0.008,
    0.00201,
    0.00214,
    0.00121
   ],
   [
    -0.00116,
    -0.01503,
    0.01066,
    -0.00643,
    -0.00169,
    0.00452,
    -0.00256,
    -0.00066,
    -0.01033,
    0.00853,
    -0.00459,
    -0.00222,
    0.00339,
    -0.00205
   ],
   [
    0.0033,
    0.01954,
    -0.0024,
    -0.00579,
    0.00474,
    -0.00623,
    -0.001,
    0.00311,
    0.01455,
    -0.00324,
    -0.00415,
    0.00567,
    -0.00318,
    0.00057
   ],
   [
    -0.00144,
    0.01402,
    -0.00485,
    -0.00486,
    -0.00148,
    -0.00116,
    0.00291,
    -0.00244,
    0.01302,
    -0.0038,
    -0.00577,
    -0.00202,
    -0.00092,
    0.00164
   ],
   [
    0.00209,
    -0.00295,
    -0.00205,
    -0.00145,
    0.00983,
    -8e-05,
    0.00429,
    -0.00117,
    0.00016,
    0.00037,
    -0.00101,
    0.01095,
    0.00024,
    0.00411
   ],
   [
    0.01789,
    0.01244,
    -0.02094,
    0.0073,
    0.02729,
    -0.01718,
    -0.00872,
    0.01262,
    0.00674,
    -0.02029,
    0.00665,
    0.02372,
    -0.01419,
    -0.00659
   ],
   [
    -0.00136,
    0.00933,
    -0.00411,
    0.00331,
    0.01275,
    -0.00286,
    0.00542,
    -0.00018,
    0.00796,
    -0.00112,
    0.0013,
    0.01032,
    -0.00515,
    0.00506
   ],
   [
    -0.00471,
    0.01073,
    0.00549,
    -0.01406,
    0.00345,
    0.00631,
    0.00869,
    -0.00183,
    0.00929,
    0.00668,
    -0.01268,
    0.00479,
    0.00663,
    0.00849
   ],
   [
    -0.01523,
    0.00805,
    -0.00743,
    0.0016,
    -0.00812,
    0.00588,
    0.00837,
    -0.0113,
    0.0042,
    -0.00505,
    0.00039,
    -0.00812,
    0.00542,
    0.00933
   ],
   [
    0.00033,
    0.0106,
    -0.00684,
    -0.00614,
    0.00746,
    -0.00232,
    -0.01655,
    -0.00144,
    0.00947,
    -0.00425,
    -0.00466,
    0.00653,
    -0.00253,
    -0.01387
   ],
   [
    0.00857,
    -0.00937,
    0.00406,
    0.00593,
    -0.00345,
    -0.01332,
    -0.00163,
    0.00546,
    -0.01027,
    -0.00185,
    0.00573,
    -0.00471,
    -0.01732,
    -0.00444
   ],
   [
    -0.01937,
    -0.01784,
    -0.00853,
    -0.00213,
    -0.00243,
    -0.00238,
    0.00322,
    -0.01627,
    -0.01709,
    -0.00665,
    -0.00451,
    -0.00421,
    -0.0047,
    0.00073
   ],
   [
    -0.00844,
    0.00326,
    -0.00736,
    0.00121,
    0.00024,
    0.00297,
    -0.0114,
    -0.00777,
    -0.00108,
    -0.00629,
    -0.00065,
    -0.00207,
    0.0005,
    -0.01098
   ],
   [
    0.00185,
    0.01516,
    0.00522,
    -0.00041,
    0.00734,
    0.00992,
    -0.00931,
    0.00343,
    0.00972,
    0.00256,
    0.00072,
    0.00659,
    0.00905,
    -0.00692
   ],
   [
    0.00506,
    0.00052,
    -0.00493,
    0.00021,
    0.00149,
    -0.00125,
    -0.00671,
    0.00515,
    -0.00068,
    -0.00434,
    -0.00031,
    0.00044,
    -0.00042,
    -0.00522
   ],
   [
    -0.00683,
    0.00515,
    -0.0086,
    0.00026,
    -0.00326,
    -0.00546,
    0.00577,
    -0.00513,
    0.00139,
    -0.00561,
    -0.00098,
    -0.00119,
    -0.00587,
    0.00355
   ],
   [
    -0.00499,
    -0.00261,
    -0.01309,
    -9e-05,
    -0.00277,
    -0.00705,
    0.00165,
    -0.005,
    3e-05,
    -0.00873,
    0.00114,
    -0.00368,
    -0.00835,
    -0.00028
   ],
   [
    -0.01016,
    -0.00927,
    -0.0011,
    -0.01074,
    -0.00442,
    0.00766,
    0.02021,
    -0.01048,
    -0.00433,
    0.00106,
    -0.00858,
    -0.00194,
    0.00802,
    0.01891
   ],
   [
    -0.00363,
    0.00121,
    -0.00476,
    -0.00495,
    0.00267,
    0.00164,
    0.00566,
    -0.00293,
    -0.00085,
    -0.00511,
    -0.00373,
    0.00264,
    0.00081,
    0.00404
   ],
   [
    0.03272,
    -0.00752,
    -0.00359,
    0.0172,
    -0.00653,
    -0.01034,
    0.01002,
    0.02361,
    -0.00336,
    -0.00057,
    0.01565,
    -0.00575,
    -0.00752,
    0.01004
   ],
   [
    0.00253,
    0.00244,
    0.00129,
    0.00077,
    -0.00061,
    0.00401,
    0.00061,
    0.00047,
    0.00289,
    0.00161,
    0.00108,
    -0.00079,
    0.00358,
    0.00188
   ],
   [
    0.00284,
    0.00278,
    0.00142,
    0.0019,
    0.00338,
    0.00395,
    0.00597,
    0.00406,
    0.0069,
    0.00554,
    0.00203,
    0.00396,
    0.00409,
    0.00572
   ],
   [
    0.00328,
    -0.00056,
    -0.01001,
    -0.00416,
    0.00938,
    0.01168,
    -0.01048,
    -0.00035,
    -0.00215,
    -0.00475,
    -0.00386,
    0.00908,
    0.0089,
    -0.01023
   ],
   [
    0.0045,
    0.00304,
    -0.00279,
    -0.00016,
    -0.0084,
    0.00143,
    0.00784,
    0.0035,
    -0.00036,
    -0.00398,
    0.00041,
    -0.00599,
    0.00239,
    0.00636
   ],
   [
    -0.00153,
    -0.00122,
    0.0041,
    0.0016,
    -0.00086,
    0.0028,
    -0.00573,
    -0.0029,
    -0.00487,
    0.0015,
    -0.00153,
    -0.00194,
    0.00068,
    -0.00601
   ],
   [
    5e-05,
    -0.00571,
    -0.00606,
    0.00019,
    -0.00106,
    -0.00339,
    -0.00109,
    -0.00114,
    -0.00177,
    -0.00373,
    0.00233,
    0.00153,
    -0.00294,
    -0.00055
   ],
   [
    -0.01687,
    -0.00464,
    0.00591,
    -0.00615,
    0.00153,
    0.00998,
    -0.0126,
    -0.01656,
    -0.00676,
    0.00349,
    -0.00686,
    0.00046,
    0.008,
    -0.0091
   ],
   [
    -0.02083,
    -0.0356,
    -0.00617,
    -0.00038,
    -0.01831,
    -0.00236,
    -0.00689,
    -0.01926,
    -0.03154,
    -0.00756,
    -0.003,
    -0.01732,
    -0.00517,
    -0.0094
   ],
   [
    -0.00898,
    0.02165,
    -0.00147,
    0.0051,
    0.00789,
    -0.00495,
    0.00229,
    -0.00614,
    0.01717,
    -6e-05,
    0.00488,
    0.00675,
    -0.00531,
    0.00167
   ],
   [
    -0.00794,
    0.02606,
    -0.00409,
    -0.00053,
    0.0164,
    -0.01601,
    -0.00933,
    -0.00739,
    0.02067,
    -0.00735,
    -0.00084,
    0.01567,
    -0.01667,
    -0.00685
   ],
   [
    -0.01267,
    0.04173,
    -0.0027,
    -0.00478,
    0.01471,
    -0.00439,
    0.0063,
    -0.01012,
    0.03846,
    -0.00747,
    -0.00327,
    0.01459,
    -0.00469,
    0.00688
   ],
   [
    -0.00221,
    0.00338,
    -0.01122,
    0.0011,
    -0.00328,
    0.00383,
    -0.0005,
    -0.00309,
    0.00109,
    -0.01077,
    -0.00105,
    -0.00345,
    -0.00087,
    -0.00463
   ],
   [
    0.00763,
    -0.01314,
    0.00093,
    0.00216,
    -0.00446,
    0.00381,
    0.00878,
    0.0051,
    -0.01061,
    0.00097,
    0.00115,
    -0.00354,
    0.00434,
    0.00679
   ],
   [
    0.01303,
    -0.00468,
    0.00636,
    -0.00558,
    -3e-05,
    0.00742,
    0.00278,
    0.01049,
    -0.00386,
    0.00638,
    -0.00531,
    0.00099,
    0.00692,
    0.00277
   ],
   [
    0.01778,
    0.00767,
    0.00378,
    -0.0014,
    0.0019,
    -0.0001,
    -0.00674,
    0.01338,
    0.00611,
    0.00605,
    -0.00329,
    -7e-05,
    0.00032,
    -0.00724
   ],
   [
    -0.01021,
    0.00318,
    -0.00334,
    0.00108,
    0.00252,
    -0.00477,
    -0.00741,
    -0.00759,
    -0.00077,
    -0.0047,
    -0.00073,
    0.00123,
    -0.00316,
    -0.00577
   ],
   [
    -0.00571,
    -0.01103,
    -0.00178,
    0.01185,
    -0.00615,
    -0.01198,
    -0.01577,
    -0.0079,
    -0.00752,
    -0.00382,
    0.0122,
    -0.00644,
    -0.01447,
    -0.01305
   ],
   [
    -0.01714,
    -0.02049,
    0.00078,
    -0.00062,
    -0.01371,
    0.00242,
    -0.00325,
    -0.01412,
    -0.01662,
    0.00153,
    0.00082,
    -0.01153,
    0.0038,
    -0.0019
   ],
   [
    -0.01295,
    -0.0057,
    0.00972,
    0.00531,
    -0.01185,
    0.00298,
    0.0029,
    -0.0075,
    -0.00205,
    0.00841,
    0.00603,
    -0.00799,
    0.00411,
    0.00421
   ],
   [
    -0.03935,
    0.02867,
    0.0299,
    -0.02453,
    0.01735,
    0.03414,
    0.0149,
    -0.02968,
    0.01782,
    0.00855,
    -0.02644,
    0.01589,
    0.02831,
    0.00854
   ],
   [
    -0.00098,
    -0.00262,
    0.00013,
    -1e-05,
    0.00937,
    -0.00385,
    -0.0057,
    -0.00197,
    -0.00232,
    -0.00073,
    0.00347,
    0.00933,
    -0.00199,
    -0.00292
   ],
   [
    -0.01688,
    0.00515,
    0.00385,
    0.00246,
    -0.00187,
    5e-05,
    0.00249,
    -0.01435,
    8e-05,
    -0.00275,
    0.00271,
    -0.00185,
    -0.00056,
    0.00223
   ],
   [
    -0.00323,
    -0.00015,
    -0.01046,
    0.00748,
    -0.00475,
    0.00793,
    -0.00456,
    -0.00309,
    0.00257,
    -0.00766,
    0.0059,
    -0.0049,
    0.00432,
    -0.00686
   ],
   [
    -0.01338,
    -0.00756,
    0.00996,
    -0.01047,
    0.00549,
    -0.00823,
    -0.00028,
    -0.01586,
    -0.00773,
    0.00767,
    -0.01093,
    0.00064,
    -0.00871,
    -0.00064
   ],
   [
    -0.00669,
    -0.0066,
    -0.00531,
    -0.00663,
    -0.01436,
    0.00972,
    0.01256,
    -0.00427,
    -0.00609,
    -0.0025,
    -0.00544,
    -0.01313,
    0.00834,
    0.01057
   ],
   [
    0.00456,
    0.00939,
    0.00345,
    0.00735,
    0.00816,
    -0.00284,
    -0.00955,
    0.00316,
    0.0058,
    0.00314,
    0.00474,
    0.00462,
    -0.00363,
    -0.00833
   ],
   [
    0.00088,
    0.00214,
    -0.01123,
    0.0015,
    -6e-05,
    -0.00442,
    -0.00017,
    -0.00083,
    0.00449,
    -0.00075,
    0.0024,
    0.00071,
    -0.00234,
    -0.00033
   ],
   [
    -0.01586,
    -0.00133,
    -0.00627,
    0.00525,
    -0.00044,
    -0.0029,
    -0.00682,
    -0.01349,
    -0.00047,
    -0.00479,
    0.00483,
    0.00036,
    -0.00083,
    -0.00383
   ],
   [
    0.00722,
    0.00818,
    -0.00254,
    -0.00049,
    -0.00328,
    -0.00746,
    -0.01056,
    0.00776,
    0.007,
    0.00144,
    -0.00319,
    -0.00329,
    -0.00808,
    -0.0094
   ],
   [
    0.02099,
    0.01694,
    0.00434,
    -0.0045,
    -0.0025,
    -0.0042,
    0.00925,
    0.01724,
    0.01525,
    0.00275,
    -0.00102,
    0.00014,
    -0.00365,
    0.00765
   ],
   [
    -0.01287,
    -0.00062,
    -0.01185,
    -0.00937,
    0.00337,
    0.00503,
    -0.00818,
    -0.01155,
    -0.00084,
    -0.00828,
    -0.00864,
    0.00255,
    0.00565,
    -0.00715
   ],
   [
    0.00519,
    0.01329,
    0.00488,
    0.0129,
    0.00166,
    6e-05,
    -4e-05,
    0.00492,
    0.01352,
    0.00354,
    0.01242,
    0.00245,
    0.0001,
    0.0003
   ],
   [
    -0.0033,
    -0.00322,
    0.01228,
    -0.00149,
    -0.00342,
    -0.0057,
    0.00123,
    -0.00238,
    -0.00386,
    0.00526,
    -0.00091,
    -0.00204,
    -0.00613,
    0.0011
   ],
   [
    -0.00889,
    0.01317,
    0.00296,
    -0.0008,
    0.00989,
    0.00486,
    -0.01178,
    -0.00658,
    0.0098,
    0.00136,
    -0.00134,
    0.01052,
    0.00472,
    -0.00858
   ],
   [
    -0.00924,
    -0.00741,
    -0.00258,
    0.00268,
    -0.00269,
    -0.00071,
    -0.00441,
    -0.00925,
    -0.00454,
    -0.00227,
    0.00347,
    -0.00214,
    -0.00056,
    -0.00456
   ],
   [
    0.00414,
    0.00223,
    0.00254,
    0.01127,
    0.00464,
    0.00237,
    -0.00977,
    0.00381,
    0.00194,
    0.00361,
    0.01032,
    0.00317,
    0.00366,
    -0.00866
   ],
   [
    0.00921,
    0.00182,
    -0.00343,
    0.00187,
    0.02097,
    -0.00881,
    -0.01995,
    0.0043,
    -0.00224,
    0.00116,
    7e-05,
    0.01604,
    -0.01051,
    -0.01919
   ],
   [
    0.00596,
    0.01147,
    -0.00886,
    0.01262,
    -0.00097,
    0.00678,
    0.00052,
    0.00724,
    0.01126,
    -0.00472,
    0.01148,
    0.00116,
    0.00522,
    0.0017
   ],
   [
    0.00547,
    -0.00096,
    0.00498,
    -0.00363,
    0.00344,
    0.00913,
    0.00185,
    0.00151,
    -0.00056,
    0.00799,
    -0.00152,
    0.00507,
    0.01064,
    0.00462
   ],
   [
    0.00239,
    -0.00327,
    -0.00388,
    -0.00688,
    0.00364,
    -0.00134,
    0.0165,
    0.0017,
    0.00384,
    -0.00125,
    -0.00519,
    0.00312,
    -0.00187,
    0.01425
   ],
   [
    0.00826,
    0.00072,
    0.00615,
    -0.00332,
    -0.00445,
    0.00993,
    -0.0024,
    0.01013,
    0.00207,
    0.00681,
    -0.00422,
    -0.0029,
    0.00873,
    -0.00273
   ],
   [
    -0.01264,
    -0.00617,
    -0.00343,
    -0.01069,
    0.0002,
    0.01063,
    0.01007,
    -0.01026,
    -0.00925,
    -0.00262,
    -0.01005,
    0.00058,
    0.00981,
    0.00845
   ],
   [
    0.00811,
    0.02112,
    -0.00468,
    0.00745,
    0.01246,
    -0.00719,
    0.01135,
    0.00802,
    0.01634,
    -0.00139,
    0.00775,
    0.01306,
    -0.00379,
    0.01102
   ],
   [
    0.02035,
    -0.00726,
    -0.00149,
    0.01448,
    -0.01994,
    0.00533,
    -0.01279,
    0.01312,
    -0.00496,
    -0.0008,
    0.01473,
    -0.0174,
    0.00218,
    -0.01159
   ],
   [
    -0.00419,
    0.00905,
    -0.01667,
    -0.00245,
    0.00464,
    -0.01587,
    0.00057,
    -0.00278,
    0.00593,
    -0.01564,
    -0.00403,
    0.00012,
    -0.0153,
    0.00035
   ],
   [
    -0.00714,
    0.00967,
    0.00464,
    -0.00099,
    0.00709,
    0.00219,
    -0.00701,
    -0.00155,
    0.00831,
    0.00359,
    -0.00133,
    0.00657,
    0.00588,
    -0.0042
   ],
   [
    0.00094,
    0.01777,
    -0.02548,
    -0.00357,
    0.01269,
    0.00869,
    -0.01408,
    -0.00116,
    0.01044,
    -0.01822,
    -0.00178,
    0.01364,
    0.01018,
    -0.01126
   ],
   [
    -0.01763,
    -0.00528,
    0.0019,
    -0.00159,
    0.00856,
    -0.00119,
    -0.01321,
    -0.0128,
    -0.00259,
    -0.00047,
    -0.00223,
    0.00425,
    -0.00318,
    -0.01261
   ],
   [
    -0.00619,
    0.00582,
    0.00943,
    0.00407,
    -0.00551,
    -0.00434,
    -0.00979,
    -0.00356,
    0.00022,
    0.00485,
    0.00096,
    -0.00594,
    -0.00566,
    -0.00897
   ],
   [
    0.00195,
    -0.03007,
    -0.00365,
    0.00939,
    -0.02095,
    0.00028,
    0.01303,
    0.00139,
    -0.01889,
    -0.00533,
    0.0087,
    -0.0175,
    -0.00163,
    0.01014
   ],
   [
    -0.01568,
    -0.00494,
    0.0104,
    -0.01738,
    0.0061,
    0.00109,
    0.00474,
    -0.01205,
    -0.00136,
    0.014,
    -0.01671,
    0.00818,
    0.00436,
    0.00827
   ],
   [
    -0.0151,
    0.00889,
    -0.01357,
    -0.01003,
    0.0045,
    0.00934,
    -0.00822,
    -0.01247,
    0.00624,
    -0.01382,
    -0.01089,
    0.00378,
    0.00659,
    -0.00811
   ],
   [
    0.00395,
    -0.00718,
    0.01252,
    -0.00475,
    -0.00319,
    0.00703,
    -0.00074,
    0.00342,
    -0.00652,
    0.00911,
    -0.00541,
    -0.00469,
    0.00451,
    -0.00111
   ],
   [
    -0.00659,
    -0.00474,
    0.00268,
    -0.00056,
    0.00137,
    -0.00018,
    0.00177,
    -0.00636,
    -0.00638,
    0.00027,
    -0.00076,
    0.00134,
    -0.00119,
    0.00087
   ],
   [
    0.02062,
    -0.00425,
    -0.00787,
    -0.00233,
    -0.0042,
    0.00988,
    0.01088,
    0.01878,
    0.00606,
    0.00274,
    -0.00221,
    -0.00182,
    0.01272,
    0.00874
   ],
   [
    0.00273,
    0.00797,
    -0.00271,
    -0.00106,
    0.01003,
    -0.00901,
    0.00249,
    0.0031,
    0.0119,
    0.0018,
    0.00041,
    0.01034,
    -0.00481,
    0.00322
   ],
   [
    -0.03051,
    -0.0324,
    -0.00721,
    0.00358,
    -0.01627,
    -0.00229,
    -0.00037,
    -0.02732,
    -0.02594,
    -0.01003,
    0.00532,
    -0.01323,
    -0.00493,
    -0.00229
   ],
   [
    0.00107,
    0.00769,
    -0.00176,
    -0.00529,
    0.00364,
    -0.00515,
    0.00034,
    0.00128,
    0.00757,
    -0.00089,
    -0.0042,
    0.00283,
    -0.00361,
    0.00066
   ],
   [
    0.00012,
    -0.01872,
    0.00607,
    0.01638,
    -0.01512,
    0.00281,
    0.00895,
    -0.00537,
    -0.01377,
    0.00162,
    0.01748,
    -0.01096,
    0.0037,
    0.00662
   ],
   [
    0.00017,
    0.00241,
    0.01394,
    0.00812,
    0.00854,
    -0.00921,
    -0.00939,
    -0.00098,
    -0.00055,
    0.00496,
    0.00497,
    0.00529,
    -0.01111,
    -0.00984
   ],
   [
    0.00595,
    0.0095,
    0.00368,
    -0.00716,
    -0.00448,
    -0.00029,
    -0.00073,
    0.00965,
    0.00639,
    -0.00176,
    -0.00851,
    -0.00553,
    -0.00072,
    0.00011
   ],
   [
    -0.0023,
    -0.00784,
    0.00736,
    -0.00974,
    0.00592,
    -0.00426,
    0.00561,
    -0.00325,
    -0.00341,
    0.00317,
    -0.01014,
    0.00314,
    -0.00421,
    0.00375
   ],
   [
    -0.00195,
    -0.01648,
    0.00062,
    0.00269,
    -0.00291,
    -0.01389,
    0.00365,
    -0.00322,
    -0.01704,
    -0.00128,
    0.00075,
    -0.00391,
    -0.0134,
    0.00257
   ],
   [
    -0.00274,
    0.00103,
    -0.01261,
    0.00907,
    0.00314,
    -0.00309,
    -0.00382,
    -0.0026,
    0.00022,
    -0.0063,
    0.00769,
    0.00296,
    -0.00339,
    -0.00464
   ],
   [
    0.00503,
    -0.0003,
    -0.00202,
    0.00054,
    0.00428,
    -0.00659,
    -0.00609,
    0.00393,
    0.00179,
    -0.00133,
    0.00092,
    0.00385,
    -0.0069,
    -0.00603
   ],
   [
    -0.00901,
    -0.01613,
    0.00084,
    0.00077,
    -0.00876,
    0.00777,
    0.00985,
    -0.00641,
    -0.0149,
    0.00015,
    0.00169,
    -0.00663,
    0.00692,
    0.00909
   ],
   [
    -0.00018,
    0.00333,
    0.0024,
    -0.01069,
    0.00403,
    -0.00115,
    -0.00487,
    -0.00072,
    0.00646,
    0.00041,
    -0.01091,
    0.00397,
    -0.00427,
    -0.00524
   ],
   [
    -0.0073,
    0.00167,
    -0.00277,
    -0.00764,
    -0.00806,
    0.00927,
    0.0111,
    -0.00595,
    -0.00065,
    -0.00247,
    -0.00878,
    -0.00709,
    0.00857,
    0.01088
   ],
   [
    0.004,
    -0.00753,
    -0.00398,
    -0.00242,
    0.00023,
    -0.00227,
    0.00615,
    0.00494,
    -0.00073,
    -0.00242,
    0.00011,
    0.00159,
    -0.00117,
    0.00521
   ],
   [
    0.003,
    0.00687,
    0.00284,
    -0.00315,
    0.00218,
    0.01061,
    -0.00138,
    0.00427,
    0.00561,
    0.00239,
    -0.00196,
    0.00218,
    0.00905,
    -0.00062
   ],
   [
    -0.00912,
    -0.01348,
    -0.00368,
    -0.0025,
    0.00262,
    -0.0,
    -0.00353,
    -0.01046,
    -0.00608,
    -0.00081,
    -0.00312,
    -0.00027,
    -0.00179,
    -0.00336
   ],
   [
    0.00377,
    -0.0237,
    0.01971,
    0.0174,
    -0.02312,
    0.01609,
    0.00171,
    0.00022,
    -0.01303,
    0.01626,
    0.01551,
    -0.01481,
    0.01768,
    0.00505
   ],
   [
    0.01802,
    0.00085,
    -0.00204,
    0.00945,
    0.0018,
    0.00318,
    -0.00109,
    0.0142,
    -0.00028,
    -0.00265,
    0.00863,
    0.0023,
    0.00175,
    -0.00214
   ],
   [
    -0.00164,
    -0.00214,
    -0.00987,
    -0.0091,
    -0.00282,
    0.00593,
    -4e-05,
    -0.00216,
    0.00552,
    -0.00282,
    -0.00293,
    0.00042,
    0.0089,
    0.00321
   ],
   [
    0.01152,
    0.00502,
    -0.01271,
    -0.01751,
    -0.01033,
    0.01115,
    0.01265,
    0.01324,
    0.0007,
    -0.01082,
    -0.02048,
    -0.01038,
    0.00978,
    0.01012
   ],
   [
    0.01014,
    -0.00518,
    0.00441,
    0.01297,
    -0.0141,
    -0.00992,
    -0.01365,
    0.00879,
    -0.00423,
    0.00087,
    0.01097,
    -0.01224,
    -0.00814,
    -0.01203
   ],
   [
    -0.02685,
    -0.00184,
    -0.00887,
    0.00423,
    -0.00194,
    0.00133,
    -0.0076,
    -0.0226,
    -0.0013,
    -0.00511,
    0.0045,
    -0.00104,
    0.00085,
    -0.00633
   ],
   [
    0.00559,
    -0.00633,
    -0.00375,
    0.00362,
    0.00121,
    -0.00697,
    -0.00911,
    0.00603,
    -0.00274,
    -0.00418,
    0.00131,
    0.0007,
    -0.0099,
    -0.009
   ],
   [
    0.0036,
    -0.01135,
    0.00342,
    -0.00976,
    3e-05,
    0.00648,
    0.00707,
    0.00568,
    -0.00676,
    0.00566,
    -0.00949,
    0.00114,
    0.00687,
    0.00674
   ],
   [
    -0.00113,
    -0.00172,
    0.00561,
    -0.00505,
    0.00974,
    -0.00291,
    -0.00014,
    -0.00066,
    -0.00269,
    0.00357,
    -0.00406,
    0.00797,
    -0.00303,
    -0.00022
   ],
   [
    -0.0957,
    -0.00155,
    -0.0313,
    0.01553,
    0.0368,
    -0.03826,
    -0.0266,
    -0.09589,
    -0.0185,
    -0.02474,
    0.00554,
    0.02599,
    -0.03843,
    -0.01951
   ],
   [
    0.0195,
    0.01675,
    -0.00498,
    -0.00801,
    0.01253,
    0.00506,
    -0.00674,
    0.01663,
    0.01477,
    -0.00504,
    -0.00836,
    0.01053,
    0.00339,
    -0.00563
   ],
   [
    0.00542,
    -0.02822,
    0.00454,
    -0.00933,
    -0.01269,
    -0.01138,
    0.00424,
    0.00131,
    -0.02802,
    0.00091,
    -0.0057,
    -0.01359,
    -0.0112,
    0.00093
   ],
   [
    -0.03158,
    -0.00264,
    0.02466,
    -0.04918,
    0.02856,
    0.01355,
    0.04878,
    -0.03013,
    0.01109,
    0.03043,
    -0.04236,
    0.02906,
    0.02135,
    0.04404
   ],
   [
    0.01083,
    -0.02132,
    0.00254,
    -0.00439,
    -0.00879,
    0.00732,
    0.00874,
    0.01022,
    -0.01976,
    -4e-05,
    -0.00565,
    -0.00864,
    0.00784,
    0.00864
   ],
   [
    1e-05,
    -0.00654,
    -0.00305,
    0.00544,
    -0.00781,
    0.00313,
    0.00276,
    -0.00132,
    -0.00681,
    -0.00344,
    0.00669,
    -0.00581,
    0.00483,
    0.00299
   ],
   [
    0.00984,
    0.00028,
    -0.00885,
    0.0093,
    -0.00114,
    -0.00683,
    -0.00598,
    0.0051,
    0.00015,
    -0.00648,
    0.00987,
    -0.00146,
    -0.00722,
    -0.00662
   ],
   [
    -0.00993,
    -0.01423,
    0.00339,
    -0.00136,
    -0.01465,
    0.00931,
    0.0004,
    -0.00671,
    -0.01377,
    0.00463,
    -0.00086,
    -0.0107,
    0.00858,
    0.00099
   ],
   [
    -0.0117,
    0.01202,
    -0.0035,
    0.00022,
    0.01065,
    0.00291,
    -0.00554,
    -0.01041,
    0.00831,
    -0.0033,
    -0.00054,
    0.0092,
    0.00174,
    -0.00502
   ],
   [
    -0.00469,
    0.01917,
    0.00059,
    -0.00931,
    0.00677,
    -0.0023,
    0.00184,
    -0.00254,
    0.01336,
    0.00091,
    -0.00814,
    0.00633,
    -0.00262,
    0.00141
   ],
   [
    0.01419,
    0.02295,
    0.00244,
    0.00859,
    0.00853,
    -0.00806,
    0.00294,
    0.01464,
    0.02049,
    0.00432,
    0.00483,
    0.00621,
    -0.00704,
    0.00019
   ],
   [
    0.0042,
    0.00846,
    -0.00706,
    -0.00121,
    -0.00094,
    -0.00529,
    -0.00046,
    0.00385,
    0.00778,
    -0.00477,
    -0.00165,
    -0.00047,
    -0.00449,
    -0.00019
   ],
   [
    0.02253,
    0.01727,
    0.00212,
    0.00507,
    0.01217,
    -0.01207,
    0.00623,
    0.01788,
    0.01375,
    0.00132,
    0.00629,
    0.01107,
    -0.00736,
    0.00835
   ],
   [
    -0.00927,
    0.00022,
    0.00904,
    -0.01629,
    0.00973,
    -0.00219,
    0.00887,
    -0.00912,
    -0.0008,
    0.0012,
    -0.01473,
    0.00914,
    -0.00195,
    0.00741
   ],
   [
    0.01474,
    0.01721,
    0.00571,
    0.00752,
    0.00077,
    0.00256,
    -0.00153,
    0.01489,
    0.01674,
    0.00215,
    0.00604,
    0.00125,
    0.00449,
    -0.00051
   ],
   [
    0.01262,
    -0.00065,
    0.00216,
    -0.00468,
    0.00336,
    -0.00161,
    -0.00516,
    0.00866,
    -0.00484,
    -0.002,
    -0.0042,
    0.00258,
    -0.00035,
    -0.00434
   ],
   [
    0.01239,
    0.01595,
    0.00454,
    -0.01616,
    -5e-05,
    0.01365,
    0.01484,
    0.01018,
    0.01485,
    0.00779,
    -0.01413,
    0.0015,
    0.0132,
    0.01279
   ],
   [
    -0.005,
    0.00572,
    -0.00726,
    -0.00883,
    0.00161,
    0.00206,
    0.00788,
    -0.00248,
    0.00676,
    -0.0041,
    -0.0079,
    0.00244,
    0.00122,
    0.00515
   ],
   [
    -0.01019,
    -0.00612,
    0.0055,
    0.00821,
    0.00672,
    0.00535,
    -0.00822,
    -0.00756,
    -0.00532,
    0.00449,
    0.00736,
    0.00882,
    0.00821,
    -0.0047
   ],
   [
    0.00841,
    0.01396,
    0.00457,
    0.00669,
    -0.01342,
    0.0196,
    -0.00013,
    0.00603,
    0.01324,
    0.00043,
    0.00398,
    -0.00794,
    0.01722,
    0.00205
   ],
   [
    0.02196,
    0.00106,
    -0.00067,
    -0.00354,
    -0.00691,
    0.00753,
    0.00965,
    0.0216,
    0.00389,
    -0.00068,
    -0.00092,
    -0.00358,
    0.00826,
    0.00859
   ],
   [
    0.01285,
    0.01374,
    -0.00738,
    0.00467,
    0.00411,
    -0.004,
    -0.00792,
    0.0114,
    0.00995,
    -0.00519,
    0.00469,
    0.00376,
    -0.00338,
    -0.00655
   ],
   [
    0.01245,
    0.00194,
    -0.0009,
    0.00329,
    0.00134,
    -0.00673,
    0.00944,
    0.01077,
    0.00369,
    -0.0013,
    0.00396,
    0.00126,
    -0.00589,
    0.0081
   ],
   [
    -0.00286,
    0.00624,
    -0.00152,
    0.00462,
    0.01709,
    -0.00677,
    -0.01468,
    -0.0043,
    0.00158,
    -0.00192,
    0.00352,
    0.01441,
    -0.005,
    -0.01145
   ],
   [
    -0.01722,
    0.01458,
    0.00765,
    -0.00789,
    0.00939,
    0.00232,
    -0.00606,
    -0.01033,
    0.00665,
    0.00133,
    -0.00799,
    0.00689,
    0.00432,
    -0.00307
   ],
   [
    -0.00326,
    0.0023,
    0.00097,
    -0.00283,
    0.00349,
    0.01389,
    0.01051,
    0.0006,
    0.00472,
    0.00454,
    -0.00373,
    0.00285,
    0.0121,
    0.00914
   ],
   [
    0.00424,
    0.00916,
    0.00645,
    -0.00452,
    8e-05,
    0.00452,
    -0.00331,
    0.004,
    0.00742,
    0.00507,
    -0.00556,
    -0.00066,
    0.00343,
    -0.00368
   ],
   [
    0.00493,
    0.00672,
    -0.01088,
    0.00573,
    -0.00427,
    -0.01172,
    -0.01098,
    0.00411,
    0.00287,
    -0.01229,
    0.00077,
    -0.00839,
    -0.01733,
    -0.0133
   ],
   [
    -0.00233,
    0.00655,
    0.01465,
    0.00322,
    0.00568,
    -0.01134,
    -0.00574,
    -0.00097,
    0.00281,
    0.0081,
    0.00114,
    0.00478,
    -0.00987,
    -0.00337
   ],
   [
    0.00854,
    -0.01949,
    0.01015,
    0.00648,
    -0.01447,
    -0.0036,
    -0.00265,
    0.00581,
    -0.01428,
    0.00394,
    0.00766,
    -0.01317,
    -0.00478,
    -0.0046
   ],
   [
    -0.01707,
    -0.02403,
    0.00608,
    0.00261,
    -0.00735,
    -0.00571,
    -0.00269,
    -0.01496,
    -0.02016,
    0.0025,
    0.00253,
    -0.00715,
    -0.0074,
    -0.00382
   ],
   [
    0.02324,
    0.01784,
    -0.00414,
    -0.00296,
    0.00497,
    0.00617,
    -0.02206,
    0.01363,
    0.00855,
    -0.00362,
    -0.00284,
    0.00186,
    0.0035,
    -0.0185
   ],
   [
    -0.0083,
    -0.00353,
    -0.00145,
    -0.00374,
    0.00947,
    0.00652,
    0.00671,
    -0.00585,
    -0.00086,
    -0.00189,
    -0.00391,
    0.00602,
    0.00583,
    0.00587
   ],
   [
    0.02053,
    0.01291,
    0.00169,
    0.00371,
    -0.00136,
    0.00299,
    -0.00823,
    0.01887,
    0.00773,
    -0.0009,
    0.00146,
    -0.00124,
    0.00145,
    -0.00999
   ],
   [
    0.00077,
    0.00757,
    -0.00634,
    0.00055,
    0.00367,
    0.0028,
    0.00041,
    -0.00015,
    0.0049,
    -0.0099,
    -0.002,
    0.00306,
    0.00079,
    -0.0009
   ],
   [
    -0.00287,
    -0.00739,
    0.00492,
    0.00261,
    0.00214,
    -0.00086,
    -0.00383,
    -4e-05,
    -0.00584,
    0.00571,
    0.00349,
    0.00105,
    -0.00292,
    -0.00504
   ],
   [
    0.0071,
    0.00264,
    0.01922,
    -0.00814,
    -0.00252,
    0.01869,
    0.00516,
    0.00635,
    0.00424,
    0.01935,
    -0.00806,
    -0.00116,
    0.01761,
    0.00652
   ],
   [
    0.00488,
    -0.00637,
    -0.00038,
    -0.01782,
    6e-05,
    0.01782,
    -0.00446,
    0.00209,
    -0.0033,
    0.00197,
    -0.01891,
    -0.00035,
    0.01342,
    -0.0059
   ],
   [
    0.00047,
    0.00272,
    -0.0017,
    -0.00288,
    0.00372,
    -0.00678,
    0.00129,
    0.00083,
    0.00135,
    -0.00244,
    -0.00341,
    0.00237,
    -0.00693,
    -1e-05
   ],
   [
    -0.00089,
    0.00919,
    0.00055,
    -0.00102,
    0.00218,
    0.00136,
    -0.01596,
    -0.00303,
    0.00385,
    0.00116,
    -0.00188,
    -0.0009,
    0.00122,
    -0.01367
   ],
   [
    -0.00744,
    0.00777,
    0.00469,
    -0.00704,
    0.01415,
    0.00949,
    0.00379,
    -0.00615,
    0.00785,
    0.00797,
    -0.00537,
    0.01466,
    0.01137,
    0.00532
   ],
   [
    -0.00058,
    0.00679,
    -0.00578,
    0.0177,
    -0.00034,
    -0.00369,
    -0.00603,
    -0.00319,
    0.00433,
    -0.00277,
    0.01699,
    0.0003,
    -0.00154,
    -0.00458
   ],
   [
    0.00255,
    0.03275,
    -0.00576,
    -0.00307,
    0.00922,
    -0.00516,
    -0.00274,
    0.00428,
    0.02536,
    -0.00703,
    -0.00433,
    0.00966,
    -0.00298,
    -0.0016
   ],
   [
    -0.00205,
    -0.01066,
    0.00187,
    -0.00342,
    -0.00178,
    -0.00605,
    -0.00334,
    -0.0045,
    -0.01022,
    0.00098,
    -0.003,
    -0.00154,
    -0.00332,
    -0.00106
   ],
   [
    0.01949,
    -0.00435,
    -0.00379,
    0.00295,
    -0.00128,
    0.00677,
    -0.00491,
    0.01224,
    -0.00156,
    0.00114,
    0.00626,
    -0.00127,
    0.00488,
    -0.00369
   ],
   [
    0.02548,
    0.00861,
    0.01137,
    -0.00702,
    0.00217,
    0.02228,
    0.00541,
    0.01983,
    0.01131,
    0.00784,
    -0.00607,
    0.00292,
    0.02297,
    0.0055
   ],
   [
    0.00122,
    -0.00274,
    -0.00077,
    0.00939,
    0.00448,
    -0.00393,
    -0.00936,
    -0.00109,
    -0.00242,
    -0.00106,
    0.01046,
    0.00418,
    -0.00385,
    -0.00813
   ],
   [
    0.01074,
    0.00541,
    0.0094,
    -0.01778,
    -0.00471,
    0.00922,
    0.013,
    0.00898,
    0.00912,
    0.00869,
    -0.02018,
    -0.00529,
    0.00974,
    0.01038
   ],
   [
    -1e-05,
    0.01863,
    -0.00083,
    -0.01307,
    0.00825,
    -0.00182,
    0.00284,
    0.00114,
    0.01618,
    -0.00591,
    -0.01522,
    0.00523,
    -0.0051,
    -0.0007
   ],
   [
    -0.00662,
    0.0047,
    0.0019,
    0.00073,
    -0.0018,
    0.00065,
    0.00269,
    -0.00479,
    0.00888,
    0.00142,
    0.00114,
    -0.00112,
    0.00136,
    0.00256
   ],
   [
    0.00577,
    -0.02251,
    -0.00055,
    0.01761,
    -0.01745,
    0.00145,
    0.00317,
    0.00625,
    -0.01716,
    -0.00175,
    0.0143,
    -0.0159,
    -0.00087,
    0.00307
   ],
   [
    -0.00968,
    -0.00877,
    -0.00578,
    -0.00562,
    -0.00418,
    0.00561,
    0.00523,
    -0.00719,
    -0.0104,
    -0.00639,
    -0.0048,
    -0.00534,
    0.00389,
    0.00542
   ],
   [
    -0.01819,
    0.00573,
    0.0023,
    -0.00183,
    0.01233,
    0.00541,
    0.00629,
    -0.01344,
    0.00802,
    0.00498,
    -0.00148,
    0.01366,
    0.00773,
    0.00774
   ],
   [
    0.01691,
    -0.00087,
    -0.00658,
    0.00043,
    0.00578,
    -0.00023,
    -0.0138,
    0.01186,
    -0.00443,
    -0.00445,
    0.00031,
    0.00352,
    -0.00038,
    -0.01063
   ],
   [
    -0.01621,
    -0.00595,
    -0.00318,
    -0.0044,
    0.00636,
    0.00648,
    0.01008,
    -0.01077,
    -0.00149,
    -0.00242,
    -0.00571,
    0.00564,
    0.00696,
    0.01001
   ],
   [
    -0.00438,
    0.00352,
    0.00704,
    -0.00394,
    -0.00103,
    0.01174,
    -0.0009,
    -0.00343,
    0.00303,
    0.00432,
    -0.00223,
    -0.00056,
    0.01277,
    0.00142
   ],
   [
    -0.00077,
    -0.01071,
    0.00417,
    -0.00494,
    -0.01724,
    0.01465,
    0.02142,
    -0.002,
    -0.00642,
    0.00123,
    -0.00611,
    -0.01584,
    0.01123,
    0.01394
   ],
   [
    0.01746,
    0.01587,
    0.00538,
    -0.00715,
    0.00322,
    0.00107,
    0.00145,
    0.01528,
    0.01325,
    0.00418,
    -0.00852,
    0.00315,
    0.00283,
    0.00241
   ],
   [
    -0.00633,
    -0.0011,
    0.01009,
    0.00645,
    -0.0029,
    0.00222,
    -0.0074,
    -0.00378,
    0.00081,
    0.00988,
    0.00527,
    -0.00262,
    0.00233,
    -0.00512
   ],
   [
    -0.00626,
    0.01114,
    -0.01719,
    0.01207,
    0.00317,
    -0.00238,
    -0.01493,
    -0.0039,
    0.00906,
    -0.0127,
    0.00672,
    -0.00127,
    -0.00532,
    -0.0134
   ],
   [
    0.01364,
    -0.00399,
    0.00399,
    -0.00637,
    -0.00175,
    -0.0058,
    -0.00261,
    0.00851,
    0.00081,
    0.00579,
    -0.00324,
    -0.00035,
    -0.00371,
    -0.00132
   ],
   [
    0.01135,
    0.00652,
    -0.02153,
    0.00036,
    -0.01077,
    0.00557,
    0.00892,
    0.00741,
    0.00763,
    -0.01435,
    -0.00331,
    -0.0101,
    0.00404,
    0.00568
   ],
   [
    -0.01346,
    -0.00715,
    0.00454,
    0.00017,
    -0.00418,
    0.00577,
    -0.00353,
    -0.01111,
    -0.00519,
    0.00714,
    0.0032,
    -0.00179,
    0.00682,
    -0.00202
   ],
   [
    0.0153,
    -0.01686,
    0.02552,
    0.00591,
    -0.0303,
    0.01396,
    0.02296,
    0.01018,
    -0.01151,
    0.0226,
    0.00694,
    -0.02502,
    0.0113,
    0.01828
   ],
   [
    0.00081,
    0.01613,
    -0.01191,
    -0.0006,
    0.00586,
    0.00742,
    -0.0126,
    0.00095,
    0.01177,
    -0.00762,
    -0.00193,
    0.00515,
    0.00715,
    -0.00895
   ],
   [
    -0.01913,
    0.01513,
    -0.00949,
    -0.01886,
    -0.00242,
    0.01046,
    0.01026,
    -0.0143,
    0.00869,
    -0.00838,
    -0.02019,
    -0.00197,
    0.00848,
    0.0081
   ],
   [
    -0.00211,
    -0.00281,
    0.01252,
    -0.00753,
    0.00342,
    0.00293,
    0.00969,
    -0.00114,
    -0.004,
    0.00929,
    -0.00472,
    0.0041,
    0.00689,
    0.01076
   ],
   [
    0.00479,
    -0.02169,
    -0.00125,
    0.01153,
    0.00617,
    -0.01191,
    -0.00437,
    0.00469,
    -0.02271,
    -0.00022,
    0.00993,
    0.00218,
    -0.01307,
    -0.00543
   ],
   [
    0.01012,
    0.0014,
    0.00931,
    -0.01048,
    -0.00324,
    0.00338,
    0.0028,
    0.0055,
    0.00074,
    0.00297,
    -0.00821,
    -0.00352,
    0.00147,
    0.00212
   ],
   [
    0.00654,
    0.00124,
    0.00696,
    -0.00199,
    -0.00112,
    0.01213,
    -0.00143,
    0.00487,
    0.00289,
    0.00981,
    -0.00088,
    -0.00155,
    0.00947,
    0.00086
   ],
   [
    0.0114,
    -0.00503,
    0.01264,
    0.00403,
    -0.00165,
    -0.00028,
    0.00365,
    0.00951,
    -0.00524,
    0.01071,
    0.00446,
    -0.00012,
    0.0011,
    0.00321
   ],
   [
    0.00169,
    -0.00362,
    0.01352,
    -0.00205,
    -0.00702,
    -0.0019,
    0.00046,
    0.00263,
    0.00179,
    0.01255,
    -0.00023,
    -0.00567,
    -0.00036,
    0.00148
   ],
   [
    -0.02224,
    -0.02127,
    -0.00144,
    -0.00885,
    -0.00327,
    0.00057,
    -0.0048,
    -0.01837,
    -0.0187,
    -0.00227,
    -0.00988,
    -0.00563,
    -0.00039,
    -0.00521
   ],
   [
    -0.00878,
    0.00649,
    -0.00335,
    -0.00013,
    -0.0033,
    0.00186,
    0.00164,
    -0.00557,
    0.00297,
    -0.00408,
    -0.00034,
    -0.00195,
    0.00103,
    0.00256
   ],
   [
    -0.00633,
    0.00349,
    0.00602,
    0.005,
    0.00199,
    0.00159,
    0.00015,
    -0.00191,
    0.00033,
    0.00489,
    0.00509,
    0.00194,
    0.00371,
    0.00038
   ],
   [
    -0.00544,
    -0.00639,
    0.00286,
    0.00425,
    -0.00079,
    -0.00742,
    -0.00531,
    -0.00487,
    -0.0066,
    -0.00193,
    0.0033,
    -0.00052,
    -0.00565,
    -0.00376
   ],
   [
    -0.00779,
    0.00224,
    -0.00259,
    -0.00097,
    0.01698,
    -0.00302,
    0.01118,
    -0.00535,
    0.00394,
    0.00348,
    -0.00099,
    0.01712,
    -0.00056,
    0.01019
   ],
   [
    -0.00607,
    0.00892,
    -0.00999,
    -0.00409,
    0.01614,
    -0.00592,
    0.00096,
    -0.00406,
    0.00997,
    -0.00618,
    -0.00334,
    0.01459,
    -0.00332,
    0.00211
   ],
   [
    -0.03475,
    -0.00353,
    0.0007,
    -0.01229,
    -0.00354,
    0.00259,
    -0.00319,
    -0.02736,
    -0.00501,
    0.00035,
    -0.00924,
    -0.00088,
    0.00501,
    -0.00111
   ],
   [
    -0.01588,
    0.00268,
    -0.01293,
    0.01066,
    0.01321,
    -0.0218,
    -0.02403,
    -0.01595,
    0.00248,
    -0.01182,
    0.01061,
    0.00925,
    -0.02065,
    -0.02238
   ],
   [
    -0.00205,
    0.01035,
    0.00981,
    0.00348,
    0.00477,
    -0.00658,
    0.00334,
    -0.00107,
    0.0067,
    0.00888,
    0.00428,
    0.0048,
    -0.00518,
    0.00343
   ],
   [
    -0.01798,
    -0.00064,
    -0.01354,
    -0.00797,
    0.01092,
    0.00199,
    -0.00077,
    -0.01485,
    -0.00184,
    -0.00942,
    -0.00747,
    0.01273,
    0.00444,
    0.00297
   ],
   [
    0.00707,
    0.0064,
    0.00114,
    0.00319,
    -0.00148,
    -0.00207,
    -0.00189,
    0.00668,
    0.00433,
    -0.00232,
    0.00377,
    -0.00117,
    -0.00333,
    -0.00222
   ],
   [
    -0.01857,
    -0.00357,
    -0.00275,
    0.00487,
    0.00159,
    0.00216,
    -0.00813,
    -0.01675,
    -0.0022,
    -0.00059,
    0.0016,
    -0.00062,
    0.00076,
    -0.00838
   ],
   [
    0.01729,
    0.00595,
    -0.0082,
    0.00513,
    -0.00034,
    0.00084,
    -0.00033,
    0.01369,
    0.00525,
    -0.0072,
    0.00677,
    0.00279,
    0.00284,
    0.0009
   ],
   [
    -0.00045,
    0.00545,
    -0.00661,
    0.0054,
    -0.00173,
    -0.01633,
    -0.00457,
    -0.00011,
    0.00603,
    -0.00286,
    0.00545,
    -0.00098,
    -0.01379,
    -0.00448
   ],
   [
    -0.00482,
    -0.00202,
    0.00256,
    -0.00493,
    0.00894,
    -0.00374,
    0.00093,
    -0.00664,
    -0.00295,
    0.00616,
    -0.00354,
    0.00867,
    0.00079,
    0.00386
   ],
   [
    0.00566,
    0.00575,
    0.0065,
    0.00245,
    -0.00974,
    0.00131,
    0.00527,
    0.00811,
    0.00745,
    0.00729,
    0.00216,
    -0.007,
    0.00197,
    0.00539
   ],
   [
    0.00533,
    -0.00041,
    0.00143,
    0.00011,
    -0.00028,
    -0.00186,
    -0.00207,
    0.0053,
    0.00141,
    0.00611,
    0.00014,
    -0.00012,
    -0.00215,
    -0.0027
   ],
   [
    -0.01165,
    0.00023,
    -7e-05,
    -0.00944,
    0.00547,
    0.00038,
    0.0023,
    -0.00843,
    -0.00234,
    0.00205,
    -0.00835,
    0.00409,
    0.00011,
    0.00174
   ],
   [
    -0.01056,
    -0.01602,
    0.00472,
    -0.00064,
    -0.00567,
    0.00022,
    0.00299,
    -0.00973,
    -0.01471,
    0.00194,
    -0.00174,
    -0.00699,
    -0.00109,
    0.00217
   ],
   [
    0.0141,
    0.01251,
    -0.00404,
    0.00787,
    0.00572,
    0.00355,
    -0.0062,
    0.01054,
    0.00581,
    -0.00329,
    0.00844,
    0.00498,
    0.00019,
    -0.00702
   ],
   [
    -0.01653,
    0.00068,
    -0.01398,
    0.01016,
    0.00627,
    -0.01079,
    -0.01612,
    -0.01216,
    -0.00751,
    -0.01191,
    0.00573,
    0.00292,
    -0.01229,
    -0.01494
   ],
   [
    -0.00337,
    -0.00137,
    0.00507,
    -0.00035,
    -0.00982,
    0.00036,
    0.00887,
    -0.00178,
    -0.00228,
    0.00313,
    -0.00062,
    -0.00795,
    -0.00074,
    0.00735
   ],
   [
    0.01327,
    -0.00766,
    0.00115,
    0.0064,
    -0.00918,
    -0.00737,
    -0.00352,
    0.00936,
    -0.00996,
    -0.00399,
    0.00674,
    -0.0081,
    -0.0071,
    -0.00386
   ],
   [
    -0.01305,
    -0.01059,
    -0.00158,
    -0.00225,
    -0.00759,
    0.00094,
    -0.00969,
    -0.00797,
    -0.00842,
    -0.0019,
    -0.0012,
    -0.00481,
    0.00169,
    -0.00653
   ],
   [
    0.00184,
    0.00636,
    -0.00715,
    0.00691,
    0.0113,
    -0.00386,
    -0.02013,
    0.00452,
    0.0055,
    -0.00572,
    0.00827,
    0.00987,
    -0.00204,
    -0.01634
   ],
   [
    -0.00345,
    0.0177,
    -0.00578,
    0.00587,
    0.0019,
    0.0018,
    -0.02956,
    -0.00135,
    0.01273,
    -0.00112,
    0.00371,
    -0.00041,
    0.00032,
    -0.02387
   ],
   [
    -0.00397,
    0.01281,
    -0.00761,
    -0.00176,
    -0.00058,
    0.00725,
    0.00445,
    -0.00187,
    0.01068,
    -0.00625,
    -0.00275,
    -1e-05,
    0.00544,
    0.00227
   ],
   [
    -0.00801,
    0.01175,
    0.00588,
    -0.00164,
    0.00514,
    0.00639,
    -0.0142,
    -0.00468,
    0.00912,
    0.00343,
    -0.00273,
    0.00245,
    0.00455,
    -0.01234
   ],
   [
    -0.0102,
    -0.01131,
    0.00365,
    -0.0025,
    -0.0109,
    0.00826,
    0.01086,
    -0.00791,
    -0.00666,
    0.00426,
    -0.00181,
    -0.00904,
    0.00934,
    0.01012
   ],
   [
    0.01936,
    -0.01657,
    -0.00386,
    0.00727,
    -0.02805,
    -0.00341,
    0.02493,
    0.01783,
    -0.01443,
    9e-05,
    0.00668,
    -0.02486,
    -0.00645,
    0.01772
   ],
   [
    -0.01177,
    -0.00563,
    -0.00391,
    -0.00013,
    -0.00277,
    0.00107,
    -0.00043,
    -0.00985,
    -0.0065,
    -0.00553,
    -0.00054,
    -0.00191,
    0.00131,
    -0.0002
   ],
   [
    0.01667,
    0.01469,
    0.00547,
    -0.00182,
    -0.00034,
    0.00343,
    -0.00062,
    0.01262,
    0.01318,
    0.00516,
    0.00027,
    0.00181,
    0.00203,
    -0.0022
   ],
   [
    0.00702,
    -0.00059,
    -0.01228,
    0.00481,
    0.00045,
    0.00019,
    0.00603,
    0.00632,
    -0.00266,
    -0.00902,
    0.00414,
    0.00093,
    0.00088,
    0.00556
   ],
   [
    -0.01746,
    -0.03497,
    -0.00609,
    -0.00496,
    -0.01934,
    0.00324,
    -0.01042,
    -0.01451,
    -0.03205,
    -0.00741,
    -0.00244,
    -0.01373,
    0.00492,
    -0.00448
   ],
   [
    -0.01001,
    -0.01784,
    0.01364,
    0.00463,
    -0.01599,
    0.00314,
    0.00275,
    -0.00921,
    -0.01198,
    0.01079,
    0.00486,
    -0.018,
    0.00233,
    0.00169
   ],
   [
    -0.02393,
    -0.00588,
    0.01984,
    -0.00845,
    0.02229,
    -0.01526,
    -0.02094,
    -0.01859,
    -0.00824,
    0.01157,
    -0.00815,
    0.01612,
    -0.01662,
    -0.01973
   ],
   [
    -0.02077,
    0.00573,
    0.00158,
    0.00642,
    -0.00801,
    0.00611,
    -0.0041,
    -0.01816,
    0.00117,
    0.00038,
    0.0051,
    -0.00847,
    0.00559,
    -0.00379
   ],
   [
    -0.01324,
    -0.01364,
    0.00158,
    0.00368,
    -0.01315,
    0.00901,
    0.01176,
    -0.00726,
    -0.00779,
    0.00061,
    0.00304,
    -0.00988,
    0.00664,
    0.0099
   ],
   [
    -0.01204,
    -0.01289,
    -0.00446,
    -0.01049,
    -0.0001,
    0.00057,
    0.01352,
    -0.012,
    -0.01063,
    -0.00161,
    -0.00724,
    0.00034,
    0.00096,
    0.01183
   ],
   [
    0.00854,
    -0.01112,
    0.00421,
    0.00385,
    -0.01478,
    -0.0124,
    0.01231,
    0.00423,
    -0.00518,
    0.00186,
    0.0018,
    -0.01388,
    -0.01212,
    0.00848
   ],
   [
    -0.0128,
    -0.00405,
    -0.00454,
    -0.00458,
    0.00537,
    -0.00533,
    -0.00418,
    -0.01142,
    -0.0052,
    -0.00487,
    -0.00854,
    0.0027,
    -0.00576,
    -0.00536
   ],
   [
    -0.01301,
    0.00878,
    0.00163,
    -0.00945,
    0.00929,
    0.00182,
    0.00234,
    -0.00858,
    0.00763,
    0.00153,
    -0.00957,
    0.00806,
    0.00309,
    0.00275
   ],
   [
    -0.00021,
    0.00614,
    0.00079,
    -0.0042,
    0.00969,
    -0.00598,
    0.00643,
    0.00093,
    0.01068,
    -0.00032,
    -0.00149,
    0.01127,
    -0.00309,
    0.00532
   ],
   [
    0.01321,
    0.1481,
    0.22421,
    0.39223,
    0.42042,
    0.40053,
    0.39994,
    0.02664,
    0.12629,
    0.18034,
    0.34038,
    0.35974,
    0.3504,
    0.34183
   ]
  ]
 },
 "primary": {
  "condition": "real",
  "layer": 14,
  "gauge": "D_BEING",
  "gauge_def": "chat := [unit(h_pool - cent256); 1] @ W; D_BEING := ||chat - being_vec||; D_CENT := ||chat - xbar_pack||"
 },
 "a4_locked_tooth": {
  "x": 0.1992,
  "y": 0.0795,
  "z": 0.0614,
  "e": 0.0818,
  "f": 0.0943,
  "g": 0.0936,
  "h": 0.105,
  "fx": 0.1961,
  "fy": 0.0951,
  "fz": 0.1099,
  "fe": 0.0886,
  "ff": 0.0741,
  "fg": 0.1153,
  "fh": 0.1139
 }
}'''
import hashlib as _h, json as _j
_raw = PAYLOAD_RAW.encode('utf-8')
assert _h.sha256(_raw).hexdigest()[:16] == PAYLOAD_SHA_PIN, 'payload sha mismatch — stale notebook upload'
PAYLOAD = _j.loads(PAYLOAD_RAW)
print('payload OK:', PAYLOAD_SHA_PIN, '| script sha', PAYLOAD['script_sha'], '| R', PAYLOAD['R'])


In [ ]:
# ── Plan + G-SCRIPT + ledger self-test ───────────────────────────────────────
assert script_sha(PAYLOAD['script']) == PAYLOAD['script_sha'], 'G-SCRIPT FAIL'
ledger_selftest()
print('G-SCRIPT pass | thinning-ledger self-test pass')
PLAN, PLAN_SHA = build_plan(PAYLOAD, SMOKE)
MC = mode_consts(SMOKE)
_exp = expected_counts(SMOKE)
_got = {}
for r in PLAN:
    _got[(r['cond'], r['arm'])] = _got.get((r['cond'], r['arm']), 0) + 1
for c in CONDS:
    for a, n in _exp.items():
        assert _got[(c, a)] == n, (c, a, _got[(c, a)], n)
print(f'plan: {len(PLAN)} generations | sha {PLAN_SHA} | per-cond', _exp)
print(f'mode consts: R={MC["R"]} cap={MC["cap"]} perms={MC["n_perm"]}')


In [ ]:
# ── Flight: per-condition function scope -> capture -> inflight ship ─────────
import numpy as _np
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Frozen stimulus machinery (E7-Q/E8-R code path — the same 256-name centroid)
E7Q_SEED = 20260822
rng_dir = _np.random.default_rng(E7Q_SEED)
CENT_NAMES = list(rng_dir.choice([c['name'] for c in pack['concepts']],
                                 size=256, replace=False))

def pooled_reps(m, names, layer, bs=32):
    """Mean-pooled hidden_states[layer] of the E4 text rendering 'NAME: desc'."""
    texts = [f"{n}: {DESC[n]}" if DESC.get(n) else n for n in names]
    reps = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            enc = tok(texts[i:i+bs], padding=True, truncation=True, max_length=64,
                      return_tensors='pt').to(DEV)
            out = m(**enc, output_hidden_states=True)
            h = out.hidden_states[layer]
            mk = enc.attention_mask.unsqueeze(-1).to(h.dtype)
            reps.append(((h * mk).sum(1) / mk.sum(1).clamp(min=1)).float().cpu())
    return torch.cat(reps).numpy()

def compute_dirs(m, layers, names):
    """Directions: pooled rep minus the 256-name centroid, unit-normalized —
    the E7-Q/E8-R formula verbatim. Returns (dirs, cent) per layer."""
    dirs, cents = {}, {}
    for L in layers:
        reps = pooled_reps(m, names, L)
        cent = pooled_reps(m, CENT_NAMES, L).mean(0)
        d = reps - cent
        d = d / _np.linalg.norm(d, axis=1, keepdims=True)
        dirs[L] = {n: d[i] for i, n in enumerate(names)}
        cents[L] = cent
    return dirs, cents

PROBE_KEY = {('base', 14): 'base14', ('real', 14): 'inst14',
             ('real', 20): 'inst20'}

def step_entropy(score_row):
    s = score_row[0].float()
    finite = torch.isfinite(s)
    p = torch.softmax(s[finite], dim=-1)
    return float(-(p * torch.log(p.clamp_min(1e-12))).sum())

def run_turn(m, tok, msgs, seed, cap, layers):
    """One conversation turn: capture s_pre (last template position, BEFORE
    the reply exists), seeded sampled generation, entropy per kept step,
    s_gen (mean over the reply's positions). Pure model mechanics — sliced
    verbatim into the CPU capture suite."""
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_dict=True, return_tensors='pt')
    ids = enc['input_ids'].to(DEV)
    att = enc['attention_mask'].to(DEV)
    with torch.no_grad():
        pre = m(input_ids=ids, attention_mask=att, output_hidden_states=True)
    s_pre = {L: pre.hidden_states[L][0, -1].float().cpu().numpy()
             for L in layers}
    del pre
    torch.manual_seed(seed)
    with torch.no_grad():
        gen = m.generate(input_ids=ids, attention_mask=att,
                         do_sample=True, temperature=GEN_TEMPERATURE,
                         top_p=GEN_TOP_P, top_k=GEN_TOP_K,
                         max_new_tokens=cap,
                         pad_token_id=tok.eos_token_id,
                         return_dict_in_generate=True, output_scores=True)
    new_ids = gen.sequences[0, ids.shape[1]:]
    eos_pos = (new_ids == tok.eos_token_id).nonzero()
    eos_hit = bool(len(eos_pos))
    keep = new_ids[:int(eos_pos[0])] if eos_hit else new_ids
    cap_hit = (not eos_hit) and (len(new_ids) == cap)
    text = tok.decode(keep, skip_special_tokens=True)
    ents = [step_entropy(gen.scores[k]) for k in range(len(keep))]
    del gen
    if len(keep):
        full_ids = torch.cat([ids, keep.unsqueeze(0)], dim=1)
        full_att = torch.ones_like(full_ids)
        with torch.no_grad():
            post = m(input_ids=full_ids, attention_mask=full_att,
                     output_hidden_states=True)
        s_gen = {L: post.hidden_states[L][0, ids.shape[1]:].mean(0)
                 .float().cpu().numpy() for L in layers}
        del post
    else:
        s_gen = None
    return {'text': text, 'n_new': int(len(keep)), 'eos_hit': eos_hit,
            'cap_hit': cap_hit, 'ents': ents, 's_pre': s_pre, 's_gen': s_gen}

def fly_condition(cond):
    """One condition end-to-end in ONE function scope (lane law: model and
    activations die on return)."""
    t0 = time.time()
    layers = [14, 20]
    print(f'[{cond}] loading base model — {ram_report()}')
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                             device_map=DEV, low_cpu_mem_usage=True)
    if cond == 'real':
        print(f'[{cond}] merging E4-real instillation adapter...')
        m = PeftModel.from_pretrained(m, str(ADAPTER_REAL)).merge_and_unload()
    m.eval()
    assert not any(p.requires_grad for p in m.parameters()), 'eval-only flight'

    print(f'[{cond}] centroid + G-DIRS probes...')
    probe_dirs, cents = compute_dirs(m, layers, list(PAYLOAD['probes']))
    gdirs = {}
    for L in layers:
        key = PROBE_KEY.get((cond, L))
        if key is None:
            continue
        resid = max(float(_np.max(_np.abs(
            probe_dirs[L][p] - _np.asarray(PAYLOAD['probe_dirs'][key][p]))))
            for p in PAYLOAD['probes'])
        gdirs[str(L)] = resid
        print(f'  G-DIRS {cond} L{L}: resid {resid:.2e} (tol {DIRS_TOL})')
        assert resid <= DIRS_TOL, (
            f'G-DIRS FAIL {cond} L{L}: {resid:.2e} — wrong adapter/layer/'
            'template; do NOT edit cells in place, re-stage per the VM law')

    enc14 = encoder_for(PAYLOAD, cond, 14)
    enc20 = encoder_for(PAYLOAD, cond, 20)
    cm14 = cloud_mean_for(PAYLOAD, cond, 14)
    rows = []
    plan_rows = [r for r in PLAN if r['cond'] == cond]
    n_done = 0
    hist_key, msgs = None, []
    for r in plan_rows:
        if (r['arm'], r['rep']) != hist_key:
            hist_key, msgs = (r['arm'], r['rep']), []
        msgs.append({'role': 'user', 'content': r['prompt']})
        t = run_turn(m, tok, msgs, r['seed'], MC['cap'], layers)

        def _chat(h, enc_w, L):
            if enc_w is None or h is None:
                return None
            v = h - cents[L]
            v = v / max(float(_np.linalg.norm(v)), 1e-12)
            return [round(float(x), 5) for x in chat_apply(enc_w, v)]

        s_pre, s_gen, ents = t['s_pre'], t['s_gen'], t['ents']
        v14 = s_pre[14] - cents[14]
        r14 = float(_np.linalg.norm(v14))
        u14 = v14 / max(r14, 1e-12)
        row = {'cond': cond, 'arm': r['arm'], 'rep': r['rep'],
               'turn': r['turn'], 'tag': r['tag'], 'text': t['text'],
               'n_new': t['n_new'], 'eos': t['eos_hit'],
               'cap_hit': t['cap_hit'],
               'vis_mass': int(visible_mass(t['text'])),
               'ent_mean': (round(float(_np.mean(ents)), 4) if ents else None),
               'ent_first': (round(ents[0], 4) if ents else None),
               'chat_pre14': _chat(s_pre[14], enc14, 14),
               'chat_pre20': _chat(s_pre[20], enc20, 20),
               'chat_gen14': _chat(s_gen[14] if s_gen else None, enc14, 14),
               'chat_gen20': _chat(s_gen[20] if s_gen else None, enc20, 20),
               'radius_pre14': round(r14, 4),
               'cone_pre14': (round(float(u14 @ cm14), 5)
                              if cm14 is not None else None),
               's_pre14': ([round(float(x), 5) for x in s_pre[14]]
                           if r['keep_state'] else None)}
        rows.append(row)
        msgs.append({'role': 'assistant', 'content': t['text']})
        n_done += 1
        if n_done % 30 == 0:
            print(f'  [{cond}] {n_done}/{len(plan_rows)} rows '
                  f'({time.time()-t0:.0f}s) — {ram_report()}')
    bundle = {'stamp': STAMP, 'mode': MODE, 'cond': cond,
              'plan_sha': PLAN_SHA, 'payload_sha': PAYLOAD_SHA_PIN,
              'gdirs': gdirs,
              'centroid': {str(L): [round(float(x), 5) for x in cents[L]]
                           for L in layers},
              't_sec': round(time.time() - t0, 1), 'rows': rows}
    fn = OUT / f'condition_{cond}.json'
    jdump(bundle, str(fn))
    ship(fn, INFLIGHT)
    print(f'[{cond}] DONE {len(rows)} rows in {bundle["t_sec"]:.0f}s — '
          f'shipped inflight')
    return bundle

BUNDLES, cond_errors = {}, {}
for _cond in CONDS:
    _resume_fn = SEM / INFLIGHT / f'condition_{_cond}.json'
    if RESUME_STAMP and _resume_fn.exists():
        _b = json.load(open(_resume_fn))
        _n_want = sum(expected_counts(SMOKE).values())
        if (not _b.get('cond_error') and len(_b.get('rows', [])) == _n_want
                and _b.get('payload_sha') == PAYLOAD_SHA_PIN):
            BUNDLES[_cond] = _b
            print(f'[{_cond}] RESUMED from Drive ({len(_b["rows"])} rows)')
            continue
        print(f'[{_cond}] resume bundle incomplete/errored — re-flying')
    try:
        BUNDLES[_cond] = fly_condition(_cond)
    except Exception as e:
        # record + STOP FLYING, but fall through to the verdict cell so the
        # banner and every shipped piece reach Drive (smoke-3 lane lesson:
        # the flight ships its evidence even on error; a gate failure on
        # one condition also makes flying the next one worthless)
        import traceback
        cond_errors[_cond] = f'{type(e).__name__}: {e}'
        print(f'[{_cond}] ERROR: {cond_errors[_cond]}')
        traceback.print_exc()
        free_ram()
        break
    finally:
        free_ram()
        print(f'post-{_cond}: {ram_report()}')


In [ ]:
# ── Verdict: gates -> primaries -> secondaries -> riders -> banner -> ship ───
if set(BUNDLES) != set(CONDS):
    missing = sorted(set(CONDS) - set(BUNDLES))
    print('=' * 68)
    print(f'FLIGHT INCOMPLETE — missing condition(s): {missing}')
    for c, err in cond_errors.items():
        print(f'  {c}: {err}')
    print('Shipped inflight pieces are on Drive under', INFLIGHT)
    print('SMOKE: RED — diagnose locally; NEVER edit cells in place '
          '(VM law); fix in the builder, re-stage, re-upload.')
    raise SystemExit('no verdict on a partial flight')
for _c in CONDS:
    BUNDLES[_c].setdefault('cond_error', cond_errors.get(_c))
V = verdict(BUNDLES, PAYLOAD, MODE)
V['stamp'] = STAMP
V['plan_sha'] = PLAN_SHA
V['payload_sha'] = PAYLOAD_SHA_PIN
V['nb_build'] = NB_BUILD

smoke_checks = {
    'no_cond_errors': not any(cond_errors.values()),
    'gates_all_pass': V['gates']['all_pass'],
    'rows_complete': V['gates']['g_plan']['pass'],
    'capture_complete': V['gates']['g_capture']['pass'],
    'masses_present': all(r.get('vis_mass') is not None
                          for c in CONDS for r in BUNDLES[c]['rows']),
}
V['smoke_checks'] = smoke_checks

jdump(V, str(OUT / 'verdict.json'))
ship(OUT / 'verdict.json', INFLIGHT)
final_rel = f'e7bq/{MODE}_{STAMP}'
for f in sorted(OUT.iterdir()):
    ship(f, final_rel)
print('shipped ->', final_rel)

print()
print('=' * 68)
for line in V['banner']:
    print(line)
print('=' * 68)
if SMOKE:
    ok = all(smoke_checks.values())
    print('SMOKE:', 'GREEN — Runtime > Restart runtime, set SMOKE=False, '
          'Run all' if ok else 'RED — do not fly full; diagnose first')
    for k, v in smoke_checks.items():
        print(f'  {k}: {v}')
    if not ok:
        print('  (behavioral checks only — statistics never gate a smoke)')
else:
    print(f'FULL flight complete. RESUME_STAMP for reference: {STAMP}')
    print('Pull semcore/e7bq/ via the Drive integration; recompute locally; '
          'LOCK per protocol.')
